In [ ]:
import itertools
import math

import cirq
import cirq_google
import numpy as np

# Google Quantum AI Challenge Problem

Your code may import from cirq, cirq_google, their dependencies, and python's standard library. Other imports are not allowed.

**Please upload your solutions to [this form](https://docs.google.com/forms/d/e/1FAIpQLScw93lwRb3OihfaHgpobs8TO6nXijwvkRVYjWPpNg7n92FjIA/viewform?usp=publish-editor).** See the form for further instructions.

Consider an $N$-qubit, weight $n$ Dicke state, defined by
$$|D^{(N)}_n\rangle = \begin{pmatrix} N \\ n \end{pmatrix}^{-1/2} \sum_k \mathcal{P}_k \left( |0\rangle^{\otimes(N-n)} \otimes |1\rangle^{\otimes n}\right),$$
where the sum is over all distinct permutations $\mathcal{P}_k$. For $n=1$, this is the $W$ state, given by
$$|W\rangle = \frac{1}{\sqrt N} \left( |100\ldots 0 \rangle + |010\ldots 0 \rangle + \ldots + |00\ldots 01 \rangle \right). $$

Your task is to complete the following function

In [ ]:
# Implementation: the code of willow_dicke.py from the QuickeStates repository. Only differences:
# create_dicke_state keeps the official signature/docstring, and create_dicke_state_adaptive
# lives in the bonus cell below.
"""DuQuantum 2026 - Google Quantum AI Dicke-state challenge submission.
Team: Kai Sandberg, Will, Ben (+ Claude / GPT-Astra).
Exact Dicke-state circuits (CZ + PhasedXZ only) with calibration-aware placement, ALAP scheduling and
dynamical decoupling. Entry points: create_dicke_state(...), create_dicke_state_adaptive(...).
"""
"""Core machinery for the DuQuantum Dicke-state submission (cirq / cirq_google / numpy / stdlib only).

A *template* is an abstract circuit on nodes 0..k-1:
    ops: list of ('u', node, 2x2 unitary) | ('cz', i, j)
    state_nodes: nodes that carry the Dicke state (others are ancillas)
Pipeline: template -> noise-aware embedding onto the hardware graph -> layered schedule
(1q / CZ moments alternate, ALAP so qubits stay in |0> as long as possible) -> dynamical decoupling
on idle superposed qubits -> PhasedXZ conversion.
"""
import math
import itertools

import numpy as np
import cirq

_I2 = np.eye(2, dtype=complex)
_X = np.array([[0, 1], [1, 0]], dtype=complex)
_Y = np.array([[0, -1j], [1j, 0]], dtype=complex)


def ry(t):
    c, s = math.cos(t / 2), math.sin(t / 2)
    return np.array([[c, -s], [s, c]], dtype=complex)


def rz(t):
    return np.array([[np.exp(-0.5j * t), 0], [0, np.exp(0.5j * t)]], dtype=complex)


def is_identity_up_to_phase(U, atol=1e-9):
    ph = U[0, 0] if abs(U[0, 0]) > 0.5 else 1.0
    return abs(abs(U[0, 0]) - 1) < atol and np.allclose(U / ph * abs(ph), _I2, atol=atol) and \
        np.allclose(U, U[0, 0] * _I2, atol=atol)


# ------------------------------------------------------------------ hardware graph
def hw_graph(cz_err, sq_err):
    """adjacency dict: q -> {nbr: cz_error}; only qubits with single-qubit gates allowed."""
    adj = {q: {} for q in sq_err}
    for pair, val in cz_err.items():
        a, b = tuple(pair)
        p = float(val[0])
        if a in adj and b in adj:
            adj[a][b] = p
            adj[b][a] = p
    return adj


def readout_cost(q, n_frac, zero_err, one_err, x_weight=0.5):
    e0 = float(zero_err.get(q, (0.05, 0))[0])
    e1 = float(one_err.get(q, (0.05, 0))[0])
    z = (1 - n_frac) * e0 + n_frac * e1
    x = 0.5 * (e0 + e1)
    return z + x_weight * x


# ------------------------------------------------------------------ embedding
def template_edges(t):
    cnt = {}
    for op in t['ops']:
        if op[0] == 'cz':
            k = (min(op[1], op[2]), max(op[1], op[2]))
            cnt[k] = cnt.get(k, 0) + 1
    return cnt


def node_weights(t):
    """number of 1q gates per node (for sq error weighting) and lifetime proxy."""
    k = t['k']
    n1 = [0] * k
    for op in t['ops']:
        if op[0] == 'u':
            n1[op[1]] += 1
    return n1


def embed(t, adj, node_cost, edge_weight=1.0, beam=3000, exhaustive_limit=200000, return_all=False):
    """Find an injective map template-node -> hardware qubit s.t. every template edge is a hardware edge,
    minimising sum(edge_weight * count * p_cz) + sum(node_cost[node](q)).
    node_cost: function(node, q) -> float.
    Beam search over nodes in BFS order of the template graph."""
    k = t['k']
    edges = template_edges(t)
    nbrs = {i: set() for i in range(k)}
    for (a, b) in edges:
        nbrs[a].add(b)
        nbrs[b].add(a)
    # node order: start from max-degree node, BFS, prefer nodes with most already-placed neighbours
    order = [max(range(k), key=lambda i: len(nbrs[i]))]
    while len(order) < k:
        cand = [i for i in range(k) if i not in order]
        best = max(cand, key=lambda i: (len(nbrs[i] & set(order)), len(nbrs[i])))
        order.append(best)
    qubits = sorted(adj)
    states = []  # (cost, assignment dict)
    first = order[0]
    for q in qubits:
        if len(adj[q]) >= len(nbrs[first]):
            states.append((node_cost(first, q), {first: q}))
    for node in order[1:]:
        placed_nbrs = [m for m in nbrs[node] if True]
        new = []
        for cost, asg in states:
            used = set(asg.values())
            pn = [m for m in nbrs[node] if m in asg]
            if pn:
                cands = set(adj[asg[pn[0]]])
                for m in pn[1:]:
                    cands &= set(adj[asg[m]])
            else:
                cands = set(qubits)
            for q in cands:
                if q in used:
                    continue
                c = cost + node_cost(node, q)
                for m in pn:
                    c += edge_weight * edges[(min(node, m), max(node, m))] * adj[q][asg[m]]
                d = dict(asg)
                d[node] = q
                new.append((c, d))
        if not new:
            return None
        new.sort(key=lambda x: x[0])
        # diversity: dedupe identical qubit sets
        states = new[:beam]
    return states[0] if not return_all else states


# ------------------------------------------------------------------ scheduling
def build_circuit(t, mapping, dd=True, align='alap', eject_z=True):
    """Turn a template + mapping into a layered cirq.Circuit of PhasedXZ and CZ moments."""
    k = t['k']
    q = [mapping[i] for i in range(k)]
    # 1) collapse consecutive 1q gates; produce sequence of 'blocks' per node between CZs
    ops = t['ops']
    # layered ASAP schedule: each CZ gets a CZ-layer index; 1q gates live in 1q-layers between CZ layers.
    # 1q-layer index L means before CZ layer L (layer L=numCZlayers is the final 1q layer).
    pend = [_I2.copy() for _ in range(k)]   # accumulated unitary since last CZ per node
    has_pend = [False] * k
    node_layer = [0] * k                    # next CZ layer index available per node
    cz_layers = []                          # list of list of (i,j)
    oneq = {}                               # (layer, node) -> unitary applied in 1q layer before CZ layer `layer`
    for op in ops:
        if op[0] == 'u':
            pend[op[1]] = op[2] @ pend[op[1]]
            has_pend[op[1]] = True
        else:
            i, j = op[1], op[2]
            L = max(node_layer[i], node_layer[j])
            # 1q gates pending on i/j must be placed in 1q layer L (just before)
            for a in (i, j):
                if has_pend[a]:
                    oneq[(L, a)] = pend[a] @ oneq.get((L, a), _I2)
                    pend[a] = _I2.copy()
                    has_pend[a] = False
            while len(cz_layers) <= L:
                cz_layers.append([])
            cz_layers[L].append((i, j))
            node_layer[i] = node_layer[j] = L + 1
    nL = len(cz_layers)
    for a in range(k):
        if has_pend[a]:
            oneq[(nL, a)] = pend[a] @ oneq.get((nL, a), _I2)
    # remove identities
    oneq = {key: U for key, U in oneq.items() if not is_identity_up_to_phase(U)}
    if align == 'alap':
        cz_layers, oneq = _alap(k, cz_layers, oneq)
    if eject_z:
        oneq = _eject_z(k, nL, oneq)
    # DD
    if dd:
        oneq = _add_dd(k, cz_layers, oneq)
    # emit
    moments = []
    for L in range(nL + 1):
        m1 = [cirq.PhasedXZGate.from_matrix(oneq[(L, a)]).on(q[a]) for a in range(k) if (L, a) in oneq]
        if m1:
            moments.append(cirq.Moment(m1))
        if L < nL and cz_layers[L]:
            moments.append(cirq.Moment([cirq.CZ(q[i], q[j]) for (i, j) in cz_layers[L]]))
    return cirq.Circuit(moments)


def _split_z(U):
    """U = Rz-type diag D @ W with W = Z^a X^x Z^-a (no trailing Z). Returns (W, D)."""
    g = cirq.PhasedXZGate.from_matrix(U)
    W = cirq.unitary(cirq.PhasedXZGate(x_exponent=g.x_exponent, z_exponent=0, axis_phase_exponent=g.axis_phase_exponent))
    D = cirq.unitary(cirq.ZPowGate(exponent=g.z_exponent))
    return W, D


def _eject_z(k, nL, oneq):
    """Push the Z part of every 1q gate forward through CZs (they commute) into the node's next gate.
    The leftover Z after the node's last gate stays in the final 1q layer (it matters for X-basis readout)."""
    out = {}
    for a in range(k):
        slots = sorted(L for (L, b) in oneq if b == a)
        carry = _I2
        for L in slots:
            U = oneq[(L, a)] @ carry
            W, D = _split_z(U)
            carry = D
            if not is_identity_up_to_phase(W):
                out[(L, a)] = W
        if not is_identity_up_to_phase(carry):
            if (nL, a) in out:
                out[(nL, a)] = carry @ out[(nL, a)]
            else:
                out[(nL, a)] = carry
    return out


def _alap(k, cz_layers, oneq):
    """Push CZ layers as late as possible (reverse ASAP), keeping per-node order."""
    nL = len(cz_layers)
    # per node ordered list of events: ('1q', L) and ('cz', L, idx)
    # reverse scheduling: process CZs from last layer to first, assign new layer = min(next layers of both) - 1
    next_free = [nL] * k  # new layer index bound (exclusive) per node
    new_layers = [[] for _ in range(nL)]
    remap = {}
    for L in range(nL - 1, -1, -1):
        for (i, j) in cz_layers[L]:
            nl = min(next_free[i], next_free[j]) - 1
            new_layers[nl].append((i, j))
            remap[(L, i, j)] = nl
            next_free[i] = next_free[j] = nl
    # 1q gates: a 1q gate at layer L on node a sits before the node's first CZ at layer >= L (old indexing).
    # new position = new layer of that CZ (or nL if none).
    node_cz = {a: [] for a in range(k)}
    for L in range(nL):
        for (i, j) in cz_layers[L]:
            node_cz[i].append((L, remap[(L, i, j)]))
            node_cz[j].append((L, remap[(L, i, j)]))
    newq = {}
    for (L, a), U in oneq.items():
        nxt = [nl for (ol, nl) in node_cz[a] if ol >= L]
        nl = nxt[0] if nxt else nL
        newq[(nl, a)] = U @ newq.get((nl, a), _I2) if (nl, a) in newq else U
    return new_layers, newq


def _add_dd(k, cz_layers, oneq):
    """Insert X-X echo pairs on idle windows of qubits that are already 'live' (had a gate).
    Time slots: 1q-layer L sits before CZ layer L. A node idle over CZ layers [L1, L2) with L2-L1 >= 2
    gets X at 1q-layer L1+? ... we use: X in the first free 1q slot after going idle and X in the last
    free 1q slot before the next gate (merged into that gate if one exists)."""
    nL = len(cz_layers)
    busy = [set() for _ in range(k)]
    for L in range(nL):
        for (i, j) in cz_layers[L]:
            busy[i].add(L)
            busy[j].add(L)
    out = dict(oneq)
    for a in range(k):
        events = sorted(set([L for L in busy[a]] + [L for (L, b) in oneq if b == a]))
        if not events:
            continue
        start = min(events)
        # walk CZ layers after `start`: find maximal idle stretches of CZ layers (no CZ on a)
        L = start
        while L < nL:
            if L in busy[a]:
                L += 1
                continue
            s = L
            while L < nL and L not in busy[a]:
                L += 1
            e = L  # idle over CZ layers [s, e); 1q slots s..e available (slot e is before next CZ or final)
            d = e - s
            if d >= 2:
                # echo: X at slot s+a and s+b with a + (d-b) = b - a  (equal free-evolution either side)
                aa = int(round(d / 4))
                bb = aa + int(round(d / 2))
                x1, x2 = s + aa, s + bb
                # slot x1 may hold the gate that preceded the idle window (only if aa == 0): X goes after it
                out[(x1, a)] = _X @ out[(x1, a)] if (x1, a) in out else _X.copy()
                # slot x2 may hold the gate that ends the window (only if bb == d): X goes before it
                out[(x2, a)] = out[(x2, a)] @ _X if (x2, a) in out else _X.copy()
        # nothing else
    return {key: U for key, U in out.items() if not is_identity_up_to_phase(U)}


# ------------------------------------------------------------------ validation helpers
def check_allowed(circuit, cz_err, sq_err):
    allowed = set(frozenset(p) for p in cz_err)
    for op in circuit.all_operations():
        if isinstance(op.gate, cirq.CZPowGate):
            assert op.gate.exponent == 1 and frozenset(op.qubits) in allowed, op
        elif isinstance(op.gate, cirq.PhasedXZGate):
            assert op.qubits[0] in sq_err, op
        elif isinstance(op.gate, cirq.ResetChannel):
            pass
        else:
            raise AssertionError(f'bad op {op}')


# ------------------------------------------------------------------ Bartschi-Eidenbenz, nearest-neighbour path version
_H = np.array([[1, 1], [1, -1]], dtype=complex) / math.sqrt(2)


def _cnot(ops, a, b):
    ops += [('u', b, _H), ('cz', a, b), ('u', b, _H)]


def _gadget2(ops, a, b, th):
    """(a,b): 00->00, 01->cos 01 + sin 10, 11->11 (input 10 unconstrained)."""
    _cnot(ops, a, b)
    ops += [('u', a, ry(-th)), ('cz', b, a), ('u', a, ry(th))]
    _cnot(ops, a, b)


def _gadget3(ops, c, a, b, th):
    """path c-a-b. c=0: (a,b) 00->00, 01->cos 01+sin 10, 11->11;  c=1: 01->10, 11->11."""
    bp = (th + math.pi / 2) / 2
    ap = (th - math.pi / 2) / 2
    _cnot(ops, a, b)
    ops += [('u', a, ry(-bp)), ('cz', c, a), ('u', a, ry(-ap)), ('cz', b, a),
            ('u', a, ry(ap)), ('cz', c, a), ('u', a, ry(bp))]
    _cnot(ops, a, b)


def _gadget3b(ops, c, a, b, th, skip_first_cnot=False):
    """path c-a-b, valid when input 111 is absent: c=0: 00->00, 01->cos 01+sin 10, 11->11; c=1: 01->10.
    4 CZ (3 if a is always 0 on the support, i.e. 011 also absent)."""
    if not skip_first_cnot:
        _cnot(ops, a, b)
    g1 = th                      # b-controlled known-zero rotation: |0> -> Ry(2 th)|0>
    g2 = th + math.pi / 2        # c-controlled: Ry(2 th)|0> -> |1>
    ops += [('u', a, ry(-g1)), ('cz', b, a), ('u', a, ry(g1)),
            ('u', a, ry(-g2)), ('cz', c, a), ('u', a, ry(g2))]
    _cnot(ops, a, b)


def be_path_template(N, n):
    """D(N,n) on path nodes 0..N-1 (node p-1 = B&E position p). Exact, ~5 CZ per gadget."""
    ops = []
    for p in range(N - n, N):
        ops.append(('u', p, _X.copy()))
    for m in range(N, 1, -1):
        k = min(n, m - 1)
        w = [m - k - 1 + i for i in range(k + 1)]  # 0-indexed nodes of window
        for i in range(k):
            th = math.acos(math.sqrt((k - i) / m))
            if i == 0:
                _gadget2(ops, w[0], w[1], th)
            else:
                _gadget3(ops, w[i - 1], w[i], w[i + 1], th)
    return dict(k=N, ops=ops, state_nodes=list(range(N)), N=N, n=n)


def template_state(t):
    """noiseless statevector of a template (node 0 = MSB)."""
    k = t['k']
    psi = np.zeros((2,) * k, dtype=complex)
    psi[(0,) * k] = 1
    for op in t['ops']:
        if op[0] == 'u':
            psi = np.moveaxis(np.tensordot(op[2], psi, axes=([1], [op[1]])), 0, op[1])
        else:
            sl = [slice(None)] * k
            sl[op[1]] = 1
            sl[op[2]] = 1
            psi[tuple(sl)] *= -1
    return psi.reshape(-1)


# ---- support-aware simplification of the B&E-NN construction
def _givens_ops(a, b, th):
    """exact 2-CZ real Givens on (a,b): 01 -> cos 01 + sin 10, 10 -> -sin 01 + cos 10 (node a = first bit)."""
    c, s = math.cos(th), math.sin(th)
    G = np.array([[1, 0, 0, 0], [0, c, -s, 0], [0, s, c, 0], [0, 0, 0, 1]], dtype=complex)
    qa, qb = cirq.LineQubit(0), cirq.LineQubit(1)
    cops = cirq.two_qubit_matrix_to_cz_operations(qa, qb, G, allow_partial_czs=False)
    ops = []
    for op in cops:
        idx = [0 if q == qa else 1 for q in op.qubits]
        if len(idx) == 2:
            ops.append(('cz', a, b))
        else:
            ops.append(('u', a if idx[0] == 0 else b, cirq.unitary(op)))
    return ops


def _classical_synth(support, edges3):
    """support: dict input_bits(tuple)->output_bits(tuple) on 3 local bits (c,a,b). Find min-CNOT circuit
    (CNOTs along edges3 in either direction, X free). Returns list of ('x',q) / ('cnot',ctl,tgt)."""
    from collections import deque
    ins = list(support)
    target = tuple(support[i] for i in ins)
    start = tuple(ins)
    nb = len(ins[0])
    moves = []
    for (u, v) in edges3:
        moves.append(('cnot', u, v))
        moves.append(('cnot', v, u))

    def app(state, mv):
        out = []
        for s in state:
            s = list(s)
            if mv[0] == 'x':
                s[mv[1]] ^= 1
            else:
                s[mv[2]] ^= s[mv[1]]
            out.append(tuple(s))
        return tuple(out)

    def xfix(state):
        # X layer needed to map state->target, if it is a pure X difference
        diff = None
        for s, t in zip(state, target):
            d = tuple(x ^ y for x, y in zip(s, t))
            if diff is None:
                diff = d
            elif d != diff:
                return None
        return diff

    seen = {start: []}
    dq = deque([start])
    while dq:
        st = dq.popleft()
        path = seen[st]
        d = xfix(st)
        if d is not None:
            return path + [('x', q) for q in range(nb) if d[q]]
        if len(path) >= 6:
            continue
        # X moves are free but we fold them in by also exploring X-then-CNOT
        for xs in itertools.product([0, 1], repeat=nb):
            st2 = st
            pre = []
            for q in range(nb):
                if xs[q]:
                    st2 = app(st2, ('x', q))
                    pre.append(('x', q))
            for mv in moves:
                st3 = app(st2, mv)
                if st3 not in seen:
                    seen[st3] = path + pre + [mv]
                    dq.append(st3)
    raise RuntimeError('classical synth failed')


def be_path_template_opt(N, n):
    """Support-aware B&E nearest-neighbour construction on a path 0..N-1."""
    ops = []
    for p in range(N - n, N):
        ops.append(('u', p, _X.copy()))
    for m in range(N, 1, -1):
        k = min(n, m - 1)
        w = [m - k - 1 + i for i in range(k + 1)]
        J = [j for j in range(max(0, n - (N - m)), min(n, m) + 1)]
        for i in range(k):
            # local input support, per branch: c-branch / s-branch patterns
            S = set()
            for j in J:
                p = k - j
                if p < 0:
                    S.add((1, 1, 1) if i >= 1 else (1, 1))
                    continue
                if i < p:
                    S.add((0, 0, 0) if i >= 1 else (0, 0))
                elif i == p:
                    S.add((0, 0, 1) if i >= 1 else (0, 1))
                elif i == p + 1:
                    S.add((0, 1, 1))
                    if m - j > 0:
                        S.add((1, 0, 1))
                else:
                    S.add((1, 1, 1))
                    if m - j > 0:
                        S.add((1, 0, 1))
            th = math.acos(math.sqrt((k - i) / m))
            if i == 0:
                if (0, 1) not in S:
                    continue
                if S == {(0, 1)}:
                    ops += [('u', w[0], ry(2 * th))]
                    _cnot(ops, w[0], w[1])
                else:
                    ops += _givens_ops(w[0], w[1], th)
                continue
            P, Q = (0, 0, 1) in S, (1, 0, 1) in S
            c, a, b = w[i - 1], w[i], w[i + 1]
            if P and Q:
                if (1, 1, 1) not in S:
                    _gadget3b(ops, c, a, b, th, skip_first_cnot=(0, 1, 1) not in S)
                else:
                    _gadget3(ops, c, a, b, th)
            elif P:
                if S <= {(0, 0, 1)}:
                    ops += [('u', a, ry(2 * th))]
                    _cnot(ops, a, b)
                else:
                    ops += _givens_ops(a, b, th)
            elif Q:
                sup = {}
                for s in S:
                    sup[s] = (1, 1, 0) if s == (1, 0, 1) else s
                circ = _classical_synth(sup, [(0, 1), (1, 2)])
                loc = [c, a, b]
                for g in circ:
                    if g[0] == 'x':
                        ops.append(('u', loc[g[1]], _X.copy()))
                    else:
                        _cnot(ops, loc[g[1]], loc[g[2]])
    return dict(k=N, ops=ops, state_nodes=list(range(N)), N=N, n=n)


# ------------------------------------------------------------------ patches & W trees on hardware
def grow_patch(adj, N, root, qcost, lam=0.002, forbid=()):
    """Greedy best-first patch of N qubits from root. qcost(q)->float. Compactness via BFS distance."""
    from collections import deque
    dist = {root: 0}
    dq = deque([root])
    while dq:
        u = dq.popleft()
        for v in adj[u]:
            if v not in dist:
                dist[v] = dist[u] + 1
                dq.append(v)
    patch = [root]
    inP = {root}
    while len(patch) < N:
        best, bc = None, 1e9
        for u in patch:
            for v, p in adj[u].items():
                if v in inP or v in forbid:
                    continue
                c = qcost(v) + p + lam * dist.get(v, 99) ** 2
                if c < bc:
                    best, bc = v, c
        if best is None:
            return None
        patch.append(best)
        inP.add(best)
    return patch


def broadcast_tree(adj, patch, root, prefer_good=True):
    """Rounds of parent->child splits inside patch (each node in <=1 split per round).
    Returns list of rounds, each a list of (parent, child)."""
    P = set(patch)
    informed = {root}
    rounds = []
    # remaining-depth heuristic: farther-from-informed nodes first
    while len(informed) < len(P):
        busy = set()
        rnd = []
        # candidate edges informed->uninformed, prefer child with many uninformed neighbours (spreads), low error
        cands = []
        for u in informed:
            for v, p in adj[u].items():
                if v in P and v not in informed:
                    fan = sum(1 for w in adj[v] if w in P and w not in informed)
                    cands.append((-fan, p if prefer_good else 0, u, v))
        cands.sort(key=lambda x: (x[0], x[1]))
        for _, _, u, v in cands:
            if u in busy or v in busy:
                continue
            rnd.append((u, v))
            busy.add(u)
            busy.add(v)
        for u, v in rnd:
            informed.add(v)
        rounds.append(rnd)
    return rounds


def w_tree_template(patch, rounds):
    """Exact W state on patch via tree splits (2N-3 CZ)."""
    idx = {q: i for i, q in enumerate(patch)}
    N = len(patch)
    children = {q: [] for q in patch}
    for rnd in rounds:
        for u, v in rnd:
            children[u].append(v)
    size = {}

    def sz(q):
        if q not in size:
            size[q] = 1 + sum(sz(c) for c in children[q])
        return size[q]

    root = rounds[0][0][0]
    sz(root)
    remaining = {q: size[q] for q in patch}  # mass still held by node (itself + undistributed subtrees)
    ops = []
    first = True
    for rnd in rounds:
        for u, v in rnd:
            R = remaining[u]
            s = size[v]
            th = math.asin(math.sqrt(s / R))   # amplitude fraction moved to child
            a, b = idx[u], idx[v]
            if first:
                # |00> -> cos th |10> + sin th |01>
                ops.append(('u', a, ry(2 * th)))  # a: cos|0> + sin|1>
                _cnot(ops, a, b)
                ops.append(('u', a, _X.copy()))
                first = False
            else:
                eta = (math.pi - 2 * th) / 2
                ops += [('u', b, ry(eta))]
                _cnot(ops, a, b)
                ops += [('u', b, ry(-eta))]
                _cnot(ops, b, a)
            remaining[u] -= s
    if N == 1:
        ops.append(('u', 0, _X.copy()))
    return dict(k=N, ops=ops, state_nodes=list(range(N)), N=N, n=1)


def be_ops(nodes, k, jmin, jmax):
    """B&E-NN Dicke unitary on path `nodes` (len m0): maps sorted |0..01^j> (ones at the END of `nodes`)
    to D(m0, j) for every j in [jmin, jmax] (j <= k), with consistent phases. Support-pruned."""
    ops = []
    m0 = len(nodes)
    for m in range(m0, 1, -1):
        kk = min(k, m - 1)
        w = [nodes[m - kk - 1 + i] for i in range(kk + 1)]
        J = [j for j in range(max(0, jmin - (m0 - m)), min(jmax, m) + 1)]
        for i in range(kk):
            S = set()
            for j in J:
                p = kk - j
                if p < 0:
                    S.add((1, 1, 1) if i >= 1 else (1, 1))
                    continue
                if i < p:
                    S.add((0, 0, 0) if i >= 1 else (0, 0))
                elif i == p:
                    S.add((0, 0, 1) if i >= 1 else (0, 1))
                elif i == p + 1:
                    S.add((0, 1, 1))
                    if m - j > 0:
                        S.add((1, 0, 1))
                else:
                    S.add((1, 1, 1))
                    if m - j > 0:
                        S.add((1, 0, 1))
            th = math.acos(math.sqrt((kk - i) / m))
            if i == 0:
                if (0, 1) not in S:
                    continue
                if S == {(0, 1)}:
                    ops += [('u', w[0], ry(2 * th))]
                    _cnot(ops, w[0], w[1])
                else:
                    ops += _givens_ops(w[0], w[1], th)
                continue
            P, Q = (0, 0, 1) in S, (1, 0, 1) in S
            c, a, b = w[i - 1], w[i], w[i + 1]
            if P and Q:
                if (1, 1, 1) not in S:
                    _gadget3b(ops, c, a, b, th, skip_first_cnot=(0, 1, 1) not in S)
                else:
                    _gadget3(ops, c, a, b, th)
            elif P:
                if S <= {(0, 0, 1)}:
                    ops += [('u', a, ry(2 * th))]
                    _cnot(ops, a, b)
                else:
                    ops += _givens_ops(a, b, th)
            elif Q:
                sup = {s: ((1, 1, 0) if s == (1, 0, 1) else s) for s in S}
                for g in _classical_synth(sup, [(0, 1), (1, 2)]):
                    loc = [c, a, b]
                    if g[0] == 'x':
                        ops.append(('u', loc[g[1]], _X.copy()))
                    else:
                        _cnot(ops, loc[g[1]], loc[g[2]])
    return ops


def dc2_template(N, n, a=None):
    """Divide & conquer: D(N,n) = sum_j sqrt(C(a,j)C(b,n-j)/C(N,n)) D(a,j) D(b,n-j).
    Node layout: A path = nodes 0..a-1 (tail at a-1), B path = nodes a..N-1 with B position r -> node a+r
    (tail at N-1). Rungs: A[a-r] <-> B[b-n+r-1] for r=1..n (need a>=n, b>=n).
    Returns template incl. 'extra_edges' rungs (they are CZ edges used)."""
    if a is None:
        a = N // 2
    b = N - a
    assert b >= n and a >= 1
    A = list(range(a))
    B = [a + r for r in range(b)]
    p = [math.comb(a, j) * math.comb(b, n - j) / math.comb(N, n) for j in range(n + 1)]
    jmin = min(j for j in range(n + 1) if p[j] > 0)
    jmax = max(j for j in range(n + 1) if p[j] > 0)
    T = [sum(p[r:]) for r in range(n + 2)]
    ops = []
    # unary u_r on A[a-r], r=1..jmax : u_r = 1 iff j >= r
    for r in range(1, jmax + 1):
        q = A[a - r]
        if r <= jmin:
            ops.append(('u', q, _X.copy()))
            continue
        cond = T[r] / T[r - 1]  # P(j>=r | j>=r-1)
        phi = 2 * math.asin(math.sqrt(cond))
        if r == 1 or r - 1 <= jmin and False:
            ops.append(('u', q, ry(phi)))
        elif r - 1 <= jmin:
            ops.append(('u', q, ry(phi)))   # previous bit is deterministically 1
        else:
            # known-zero controlled Ry(phi), control A[a-r+1]
            eta = (math.pi - phi) / 2
            ops.append(('u', q, ry(eta)))
            _cnot(ops, A[a - r + 1], q)
            ops.append(('u', q, ry(-eta)))
    # B starts with n ones at its tail, subtract j: CNOT A[a-r] -> B[b-n+r-1]
    for r in range(1, n + 1):
        bq = B[b - n + r - 1]
        if r <= jmin:
            continue  # u_r = 1 deterministically: X then X cancels -> leave 0
        ops.append(('u', bq, _X.copy()))
        if r <= jmax:
            _cnot(ops, A[a - r], bq)
    ops += be_ops(A, n, jmin, jmax)
    ops += be_ops(B, n, n - jmax, n - jmin)
    return dict(k=N, ops=ops, state_nodes=list(range(N)), N=N, n=n)


# ==================================================================== main entry (inlined into submission.py)
import time as _time

T1_EST = 68e-6
TPHI_EST = 120e-6
LIBRARY = {(2, 1): [{'edges': [[0, 1]], 'seq': [0], 'params': [[[-1.5707963267948966], [-1.5707963267948966]], [[3.5839946445546094], [1.1283943358300803]]], 'mode': 'real', 'shape': 'path', 'L': 1}], (3, 1): [{'edges': [[0, 1], [1, 2]], 'seq': [0, 1, 0], 'params': [[[0.9647388984647173], [2.1655128646107205], [0.6095637510202409]], [[-0.32047720390272355], [-0.41556492137335116], [0.8704998254568896]], [[-0.10683041024328749], [1.7537808879518964], [0.6631725360545844]], [[0.18612742405578603], [-1.5707963267948968], [0.632334933556112]]], 'mode': 'real', 'shape': 'path', 'L': 3}], (4, 1): [{'edges': [[0, 1], [1, 2], [2, 3]], 'seq': [1, 0, 2, 0, 2], 'params': [[[2.7199545744311804], [-1.5707963267942358], [1.5707963267781015], [-1.1907928993461503]], [[1.8576165046566926], [-0.5210452755849042], [-3.0207211112265564], [-0.5041498718556253]], [[-1.0250439686294914], [2.2988535061538964], [-0.02975173938509952], [2.5391537710208882]], [[3.3812384587798032], [0.04814126609378741], [1.4246812278790024], [1.6629335683711233]], [[-3.2999156009234194], [2.3467062203906828], [-0.1839258346054574], [-0.877535404965622]], [[-1.6037213453905859], [2.3656827610826054], [1.5707963270395944], [-0.34965170202795304]]], 'mode': 'real', 'shape': 'path', 'L': 5}, {'edges': [[0, 1], [1, 2], [2, 3], [3, 0]], 'seq': [1, 0, 3, 2, 3], 'params': [[[-1.770600728962932], [2.37002849093598], [-0.7996258678769054], [-2.4673380108430387]], [[3.341397055822803], [-1.5905004115214552], [-1.248466153350983], [2.0607372825589705]], [[-3.14159265245787], [0.6610473992455239], [3.014545343977525], [1.977397055837326]], [[-2.115003230647901], [-1.9510269274471046], [-0.21526532556206346], [-1.5707963267670781]], [[-0.24119125952958684], [-0.5875999290241708], [2.7561724060545654], [-0.7853981634200258]], [[1.5707963294391918], [1.8775794696583936], [0.3854202503241914], [-1.570796325860376]]], 'mode': 'real', 'shape': 'sq', 'L': 5}], (4, 2): [{'edges': [[0, 1], [1, 2], [2, 3]], 'seq': [0, 1, 0, 2, 1, 0], 'params': [[[-2.3416837065745333], [-3.8612230600097015], [2.472269481272324], [0.7296811018585674]], [[1.5071211760433751], [0.8067586968681182], [-0.31829913599443815], [-0.14852627625340528]], [[-2.5223860049334053], [1.2739881882763675], [1.185837568978237], [-0.7471048632643251]], [[-3.159126768330054], [1.2380569855554084], [-3.0352294251605394], [1.7367463644540593]], [[0.36630442809658187], [2.3819489239444622], [-1.1583458400282287], [2.195559439898938]], [[2.633541133431555], [-1.5707963267948966], [0.8867452919364118], [0.4102757769411197]], [[3.453045107678416], [1.8822487808835195], [0.5244551278575348], [-1.6070852838127778]]], 'mode': 'real', 'shape': 'path', 'L': 6}, {'edges': [[1, 2], [2, 3], [0, 1]], 'seq': [1, 0, 2, 0, 0, 2], 'params': [[[-0.6039121422327106, 0.19970756966794445, -1.0233578197645476], [-1.256474366336012, 1.7582868853858806, 0.6889972074290514], [-3.490453674375172, 1.5707963269656442, 4.245637872466751], [1.078265462273148, 4.712388981336991, -2.158463830263483]], [[0.49432835102588724, 0.8960052231295885, 0.018877200901551212], [0.6208478809723648, -0.634055898005935, 0.7736593637878127], [4.309880069577817, 1.7766042768441401, 0.4445720950206556], [4.8172574183436065, 1.279716178250803, 1.1124023071937637]], [[1.243674941761388, 3.1347799740183726, -2.0593875103775825], [-1.3563411002607788, 1.3197393605552585, 1.6592757499519093], [3.9860093565332386, 1.1404603900872041, 2.994909830447573], [-0.7568716826100611, 3.105571455616389, -2.998212599691704]], [[-1.9966759049045804, 2.391842298091698, 2.4161568742472213], [1.120440418611106, 2.266047566874372, 0.14124630335870256], [5.280799816269282, 0.7703639399399251, 0.7283963122731854], [-0.043215791865314226, 0.17836438417617234, 1.1077302889869438]], [[1.411658218018404, 0.3327764563482991, -0.4223245391296947], [-3.422425495286281, 1.5707962502437534, 3.1604699230918376], [-0.7000155309592011, -1.5707963475057787, 0.6187684140343777], [0.8125718896547025, 1.2685919822856584, 0.18328183441941376]], [[1.0204490529787955, 0.9567560216845588, 0.6567130874272844], [1.5519190477555782, 0.5925287929291619, 1.4220482002146846], [3.0232391670244896, 1.6513982764464916, 0.5201202275604562], [-1.1014953437103794, -0.7269891648297138, -0.29675222511075816]], [[-2.5223646924077228, 1.570796274364101, 4.419311109424183], [-2.9928444644685213, 0.6443731957610619, 2.8485147334237366], [1.3119179741886373, 1.09985971652163, 1.0843499768651597], [0.2647567827704094, -0.11423046223982877, 0.9519340156775488]]], 'mode': 'complex', 'shape': 'path', 'L': 6}, {'edges': [[0, 1], [1, 2], [2, 3], [3, 0]], 'seq': [2, 3, 1, 0, 1], 'params': [[[-1.0715377340354926], [-1.2828464643572608], [1.6773254855236024], [1.466666445639058]], [[2.6423340608303882], [-1.8661451259843416], [3.148376786053271], [2.3648261207170034]], [[2.9773950008000805], [2.2540855807311213], [0.764425707266486], [0.7204934375943249]], [[-2.449935350851258], [0.8949060096104815], [-0.4435217910991425], [3.637990115492336]], [[0.2608088753003052], [-1.5707963267948966], [-1.1272745356957536], [3.6602944522199765]], [[-1.0221680499372667], [3.5862811780452515], [2.015484851250354], [-0.3986958960202929]]], 'mode': 'real', 'shape': 'sq', 'L': 5}, {'edges': [[0, 1], [1, 2], [2, 3], [3, 0]], 'seq': [2, 0, 1, 3, 2], 'params': [[[-1.8392262037038425], [2.0253180676598808], [0.8952846349999232], [-1.2455708246958712]], [[0.2684298769089459], [-0.4545217408649841], [-0.09810913742239129], [-0.265649669725784]], [[-2.1600918780266603], [-2.165424532631926], [0.3309796436649954], [-0.631282196660673]], [[0.44215597120112066], [-0.4845840896534056], [2.3372547163996993], [0.07330767049878766]], [[1.5583243722304456], [2.0991436200847584], [2.3751342639849904], [1.5707963267948966]], [[0.19147347966103503], [-0.9880693795283817], [2.794376985851346], [-5.059604648123138]]], 'mode': 'real', 'shape': 'sq', 'L': 5}], (5, 1): [{'edges': [[0, 1], [0, 2], [0, 3], [0, 4]], 'seq': [3, 1, 2, 1, 0, 2, 0], 'params': [[[0.9273109600913177], [1.3328201343530859], [1.5555274533938928], [-1.1451171341909177], [-1.5756556121771155]], [[1.824151273555135], [2.203275125754279], [-2.7259184372009457], [-1.4738778971438047], [-1.1598421345504548]], [[-0.5739404349404124], [1.2717781612578916], [-2.9211313990312333], [-2.0933939490499633], [-0.08917309011651643]], [[3.1415926535897936], [-0.5615361332163207], [-2.838455132550054], [-2.939814740280138], [2.367746983028962]], [[0.6154797086703868], [-2.6755409613540406], [0.1465204703469559], [0.3932059614212497], [-2.309222529008847]], [[9.042635941160197e-17], [-2.3925379880530326], [0.6388915135275444], [-3.121096819650311], [-2.042499698689464]], [[-2.3561944901923444], [-1.5344528289342092], [-1.6587855091961992], [-0.4442655364115883], [-0.06688936862399468]], [[-1.5707963267948963], [4.712388980384689], [-0.23055372848571773], [2.0150618632064887], [-1.4044102767903701]]], 'mode': 'real', 'shape': 'plus', 'L': 7}], (5, 2): [{'edges': [[0, 1], [0, 3], [1, 2], [1, 4], [3, 4]], 'seq': [1, 4, 0, 3, 2, 4, 1, 3, 4, 1], 'params': [[[3.8623485327866955], [2.631635228557763], [1.2951151030569221], [-1.7140013539373784], [0.9489070091398105]], [[1.248871998787486], [-0.6717638728129834], [-0.9464069105244818], [-3.0577664280901113], [-3.1898793442016875]], [[3.8111780670204793], [4.134769548679072], [3.3021127831390125], [0.32131750623567124], [-0.036041692728804677]], [[-4.256058203640799], [1.7870120735379385], [2.352500546917346], [-3.224605172576948], [-0.5663210194281699]], [[-2.241452963765802], [0.06983883732576081], [-1.2909325422040905], [-2.7025165828265956], [0.7011585928371439]], [[-0.22007984893675764], [-0.17212901586690355], [1.5882038712721065], [-1.9305151234432067], [-0.14533179054634188]], [[-0.6555742323404297], [4.3825378372431665], [-1.851741567560954], [-0.6154797086703865], [1.4548406002117762]], [[-4.462946120205325], [0.5019801590084255], [2.289032381413742], [-0.9261719554944098], [-4.132785644800764]], [[1.249293044591401], [3.4424742810220286], [0.0486353486798898], [-0.23701006121047596], [-1.5707963267948966]], [[-2.284134068168216], [0.6789560905898807], [-0.8150825536037867], [1.5707963267948963], [0.9632092654431255]], [[-4.0255396620159005e-16], [-2.5506340448170164], [3.4533415001836865], [1.570796326794896], [0.6075870613517693]]], 'mode': 'real', 'shape': 'g2x3', 'L': 10}, {'edges': [[0, 1], [1, 2], [2, 3], [3, 4]], 'seq': [2, 1, 2, 3, 2, 3, 0, 1, 2, 3], 'params': [[[2.624791018388875], [2.3422677928158224], [1.826369904940376], [0.6678275883300873], [0.2693967957420651]], [[3.3540835096953687], [-3.2905679169064017], [2.4341146637108686], [1.5520025505258572], [-3.3478712591326953]], [[4.160604311214936], [3.7740073463582324], [-3.9334205552738073], [3.1325424016621244], [-0.5004185549639748]], [[-0.15770421966221002], [1.3831856638400888], [-0.6529468725215047], [3.672552996690928], [-0.17817934896004245]], [[-1.7958361900215172], [2.2013597145720785], [0.12934808228622138], [2.1862760738459843], [3.7751541962227755]], [[-3.0446573739493186], [4.164422838776636], [6.50331258526692], [6.898665007443773], [-1.5888781585746214]], [[2.712700574777355], [-1.2469746603064618], [1.0972852831236284], [-3.2358159928612147], [0.7443502054342696]], [[0.98686110820448], [1.5707963255770836], [1.8241801081615323], [3.257583403322666], [0.977539588505094]], [[-4.72371797901878], [3.2079757349273774], [1.570796325927833], [-2.377961931734971], [3.2436499867568194]], [[-0.541154086910362], [2.905934326491898], [0.7142291673316673], [1.5707963290060203], [0.5322473354906517]], [[-0.4343780242445449], [1.740071571550981], [-2.2850254947263777], [-1.5707963308597501], [6.710246426848965e-09]]], 'mode': 'real', 'shape': 'path', 'L': 10}, {'edges': [[0, 1], [0, 2], [0, 3], [0, 4]], 'seq': [3, 0, 2, 0, 3, 0, 1, 2, 0, 3], 'params': [[[-1.7348846171868988], [1.8566941603013687], [-0.7192512651277753], [-0.9478850225221391], [1.2154165503195553]], [[3.575256652904579], [-0.07442805951050258], [2.7403556116320273], [-2.8905446572187254], [-0.3414614122972694]], [[-1.8831842640023553], [-2.9050562733487744], [-0.6314759975951348], [-0.192910011253669], [1.9695208842991967]], [[0.5286143822890906], [2.6442734829865553], [2.2432342292523777], [-0.8980450547647505], [1.68629326722746]], [[-1.3248403830705084], [-1.0399271400964583], [-1.093246666582409], [1.1746919878023308], [-2.0676119303553526]], [[2.6637092315526743], [-0.9764586065571927], [-2.4163987003104093], [-2.0710746544624565], [1.5124189899577178]], [[-1.3262642068983306], [-2.4525795210398837], [-1.6940135380635737], [2.754571586042372], [1.8473470660552997]], [[-1.570796326794897], [-3.0222825634016215], [-3.3120863663630447], [0.19398285680635008], [0.8342016405860311]], [[-3.141592653589794], [-2.81149682216691], [-1.8991466862965312], [-2.9056699358450633], [-0.40471176978083734]], [[-3.141592653589793], [2.1614514493860484], [-0.9804202223642033], [-0.8432368879302086], [-3.694336875843787]], [[1.5707963267948974], [0.9801412042037451], [1.47926429463909], [0.6073141701854794], [-2.0172661996909533e-16]]], 'mode': 'real', 'shape': 'plus', 'L': 10}, {'edges': [[0, 1], [1, 2], [2, 3], [3, 0], [3, 4]], 'seq': [2, 1, 4, 0, 3, 4, 1, 3, 4], 'params': [[[2.7228654999727686], [-1.4117794905168422], [1.4055832995926547], [-2.3628507692880176], [-3.532771005226294]], [[0.12143934882192703], [0.3645819393202445], [1.928705596308489], [-0.044973466594716544], [0.7172249063076739]], [[-0.3327660448714142], [2.374696192990228], [-0.8688193139924637], [-1.967742616707106], [0.2894331539992146]], [[-0.9407424771283848], [2.8612915629927613], [-2.4562683389356583], [-3.7348524149180276], [-2.7671162909950437]], [[1.5707963267948966], [1.2946610663637403], [-2.2787103912827296], [-1.59301627413705], [2.770511867332006]], [[-1.5762044919294893], [2.7905431550135544], [2.017562261577865], [-0.6154797086703868], [2.182880459128321]], [[0.24797902948454814], [3.7687774125971885], [2.015439455838089], [-0.831246630871125], [-0.7975473425102977]], [[-0.24257086434995484], [1.6034181492400663], [-0.3179198391221718], [3.18744112106347], [-1.8854045648083435]], [[-1.2033243663896092], [-1.6513774989676775], [3.504174230913894], [-1.570796326794897], [-1.2440389096686015]], [[-0.3674719604052878], [-1.5228369770672865], [-3.186254391791722], [-1.570796326794897], [3.1415926535897936]]], 'mode': 'real', 'shape': 'sq', 'L': 9}], (6, 1): [{'edges': [[0, 1], [0, 3], [1, 2], [1, 4], [2, 5], [3, 4], [4, 5]], 'seq': [1, 3, 6, 0, 5, 4, 0, 3, 1, 2, 4], 'params': [[[1.1204871953046043], [2.7300047488037724], [-2.718237665143859], [0.833789559095197], [2.719247490006256], [-0.3554121462933529]], [[-2.605865519105164], [-1.159208422008876], [3.1535846691293075], [-1.9556653400351958], [-1.1484511632113599], [-3.093966183989911]], [[-2.67846861244667], [0.08787755905775241], [-1.9100719331086229], [0.8459167937845965], [-0.39960863092028376], [1.8785820034883671]], [[0.8606031815152847], [-1.2555566825915316], [2.119961698952862], [1.2989748644567103], [-0.8327994002346935], [-1.4506234094087938]], [[1.4542720907859865], [0.12631552123731443], [-0.8035763786863864], [-1.7435263149881428], [1.3563981758329924], [-2.4153602533060234]], [[0.3706229422442685], [-3.7033984869813046], [-1.4124567182125864], [-0.24454593774513292], [-1.1456667987238551], [1.6797076272495335]], [[1.3166976205595375], [-2.7061023414355962], [-0.9647464509867855], [1.283524226198704], [-0.7026688780193053], [-2.2751598680173872]], [[-3.1427685769772356], [0.003508572340834102], [0.14514807397073098], [1.5714557189146496], [0.8011381255465629], [1.6997786312159517]], [[0.799740343223147], [-3.133379087570038], [1.135140291075734], [-1.0396376805733243], [3.7894496912318476], [0.01837449572697834]], [[-1.8585883232899705], [-0.9635301841442648], [-1.8863382408545761], [-1.3960214775376047], [-1.920007179347761], [-2.544007657671594]], [[-1.0775657862688044], [-4.070840480888348], [-2.35619448951069], [-2.4777327108063494], [2.125681861197963], [-2.3967727450164333]], [[-0.20543854403101838], [-3.783141153086136], [-1.5707963267948963], [-0.8386347920407352], [3.8588572608924343], [-1.5707963267948966]]], 'mode': 'real', 'shape': 'g2x3', 'L': 11}], (6, 2): [{'edges': [[1, 2], [2, 4], [1, 5], [0, 1], [3, 4], [4, 5]], 'seq': [0, 1, 2, 3, 0, 3, 0, 3, 3, 5, 4, 5, 4, 5, 4], 'params': [[[0.0005451334927713248, -0.12462507729533245, -0.5156839524527694], [3.142035164259196, 0.5156372125745011, -0.019555014994773072], [0.0005253671210067595, 2.1953707144113275, 0.5893535084722639], [0.0006160514465605877, 0.003580968710618697, -0.4470683754588211], [0.0001965923143311429, -0.02914467890619503, -0.434216515085961], [-0.0009455004355393361, -0.02601010149755247, 0.7533862435174229]], [[-0.5156839523412462, -0.1572330987801794, -0.4026445106444522], [3.1220376386210593, 0.44383963713984553, -0.18410946871310857], [0.5893535087582676, 0.060552666405879516, 0.40527619957939887], [-0.44706837548567047, 0.10911848816071233, -0.423976785014694], [-3.575809168084634, 1.5519643061055568, 3.16222565593044], [0.7533862427802998, -0.13590492705526225, 0.334540747881726]], [[-0.40264451085752234, -0.22274620076334845, -0.28736258033665696], [-0.18410946856059487, 0.1000520353717732, -0.13342819053626567], [0.4052761993887638, 0.5414292790632769, 0.10854271491264889], [-0.42397678495605434, 0.016145210238691813, -0.6733294895088533], [3.1622256559314725, 1.5930260236889187, -0.18047661608837673], [-2.8070519053178744, 1.4787204928420592, 3.3608603171812064]], [[1.2834337374725382, 0.9662271883494425, -1.6914508953201013], [-1.7042245859218177, 3.141592650811128, 0.9372895521972113], [0.10854269427744626, 0.5187697331888612, 0.00859011891578323], [-0.6733294145046228, -0.13727625475559257, -0.36295372633455025], [-0.1804765872875444, 0.06573317824617442, -0.1065312167553447], [3.360860312099602, 1.3656479726357011, -0.02443204285927634]], [[1.450141758193758, 0.4968922958223141, -1.733658413170196], [0.9372895521972855, 1.5707960383522919, -1.6098826402605417], [0.008590090369215478, 0.5220480189410125, -0.02635478930010379], [-0.3629537959281678, -0.20941591588970207, -0.27249456065710037], [-0.10653119475933649, 0.04905210129356834, -0.050395211421453395], [-0.024432078662543107, -0.2553006957963141, 0.002285974873133606]], [[-0.11177986157086418, -0.25940261888739713, -0.23015581331104334], [-2.395872289446389, 0.8280070437779489, 3.476593681350572], [3.1463112432737597, 1.7173784549894195, -0.09300094968735846], [-0.30735057373383035, -0.031938421116544324, -0.31089781484157614], [-0.03861271643131182, 0.08324611322980831, -0.010325828208736029], [0.0006750810687083952, -0.3018317974936027, 0.025850587775132057]], [[-0.23015581317418013, 1.3828819934219907, 0.4780940916634994], [0.3350010279083963, 1.5707963277787593, -0.7853981636841352], [-0.09300094970763433, 0.47855439287447926, 0.057327020711657385], [-0.310897814980112, 0.10300078452642862, -0.32641240355782325], [-0.0103258279099308, 0.027755654589610688, 0.02879861359549316], [0.025850587666846742, -0.32202964156243685, 0.05572146982444047]], [[0.4780940919389024, 1.2788328197978034, 0.9867798492939369], [-3.9269908173255357, -1.5707972119552494, 2.356194489784648], [0.05732702093631705, 0.08016382053917699, 0.0621230687110068], [-0.32641240367897423, 0.28309485236746296, -0.35881283292477134], [0.028798613487608392, -0.08341280146105821, 0.10303559237213702], [0.055721470048153345, -0.2913543996109804, 0.08695700988415907]], [[0.9867798491166779, -0.7853981658590107, 0.45803641226444214], [-0.7853981638645994, 0.9450137067155043, -0.2618349776391529], [3.2037157223167805, 1.456111051884416, 0.017127703555698922], [-0.3588128331840169, 0.42499212237827066, -0.3892639693412714], [0.10303559240192126, -0.12098093912749447, 0.1921865438355235], [0.08695700988540267, -0.20909278343888626, 0.11310969594011563]], [[-2.6835562412369334, 1.2386319581937864, 3.079717243610375], [-0.2618349779196463, -0.3310157428961077, 0.47387243193701073], [0.017127703422616877, -0.07705746146948146, 0.005830665977153897], [-0.38926396915799033, 0.5597974969519385, -0.2267687398467541], [0.1921865435830169, -0.03403980769074405, 0.22430648183079338], [0.11310969596209992, -0.08326853175159393, 0.1162033969914322]], [[0.16435832581847926, -0.7704086244568162, 0.14774639000655324], [0.31106947834034027, -0.6791204859585686, 0.26850347373900096], [0.011849871386920887, -0.03284857157145175, 0.001948997075943028], [0.07451689379815198, 0.15410682570159268, -0.004830420142682285], [-3.31743109615013, 1.5707963279188488, 3.9003830526816428], [3.591443241918777, 1.8560917148748657, -0.05311042292924008]], [[0.14774638987161937, -0.7390433308399705, 0.20988670677724325], [0.2685034736700703, -0.6161803478752393, 0.23318871909001077], [0.0019489969805016327, 0.037777299477638006, -0.009328016653811062], [-0.004830419964266176, -0.7455389574654342, -0.06082005572071817], [0.7587903990794047, 0.6154797073282942, 0.47765830818397415], [-0.053110422992386136, 0.43002921816962797, -0.05162789167990372]], [[0.20988670703015355, -0.5645542561671452, 0.2585465526315805], [0.23318871910690964, -0.6644226336053064, 0.18212369926428598], [-0.009328016540280005, 0.04962409846971941, -0.018408810650278804], [-0.060820055694522285, -0.8725441855945641, -0.1442315515797183], [-2.6639343452357727, 1.5707958505124842, 2.3561944055084822], [-0.051627891733973, 0.30371219849146625, 0.10210730077063468]], [[0.258546552587312, -0.3806291309078782, 0.2613706553718031], [0.1821236993690975, -0.7308591220080732, 0.09287864048834607], [-0.018408810532600954, 0.06237984607193152, -0.02575698198473215], [-0.14423155156617906, 0.03538507538720753, -0.19196224490687377], [-0.7853982483163661, 1.5707963326548156, -0.7853981648779989], [3.2436999544077216, 1.2517831571745377, -0.002192975822295138]], [[0.2613706556188825, -0.0030012315808845983, 0.3080617964797402], [0.09287864049539267, -0.7066337682053232, -0.15495766246282627], [-0.025756982195897178, 0.0757266098711351, -0.03184692529629183], [-0.19196224493021002, 1.537985300608433, -0.7624178145499598], [-0.7853981647051502, 3.926990820450957, 0.20334540297948828], [3.1393996781708844, 3.3359507605424707, 0.06308732622607116]], [[0.3323740254646504, 0.2717007448606675, 0.3733250669003039], [-0.1062439855789354, -0.6157978317624687, -0.14709414156236225], [-0.022106923000601894, 0.07928928567584183, -0.02871453450457683], [-3.936719631876191, 1.5707953140803703, 2.7916357759775314], [-3.344938056990631, 1.5707963275471217, 2.791635773220857], [-0.14332667559846984, 1.377048000531219, -0.3344757159057178]]], 'mode': 'complex', 'shape': 'dc2', 'L': 15}, {'edges': [[0, 1], [0, 3], [1, 2], [1, 4], [2, 5], [3, 4], [4, 5]], 'seq': [3, 1, 0, 1, 2, 6, 4, 3, 2, 0, 2, 4, 2, 5, 0, 0], 'params': [[[-1.7893580898396637], [-1.661070730515664], [0.7083430585296906], [-0.4050769286775253], [-1.064384559693149], [1.5303985516252423]], [[2.7179480132113856], [1.1127765766227802], [0.8353658610714314], [-1.511530686868125], [-2.878268103598425], [-3.4311965291459883]], [[-0.2814664651408732], [-1.5567113122325207], [3.6576658911485125], [-3.2356834404612496], [-2.8449386443419202], [0.26140192239676685]], [[1.4761640131656069], [1.9014839071991947], [3.7559028719912435], [-1.4218637923494546], [3.057172483093357], [-0.3265215190749185]], [[-2.1258367312709727], [0.08984655342426663], [4.293812215712716], [0.14349886045141805], [1.3634359598443857], [-1.3840111867380689]], [[0.5658615800947471], [0.6412364415930326], [-2.109859771391749], [-0.7195653679235644], [-0.2932408171873301], [-1.362460226719923]], [[-1.3224646358873413], [-0.9138538684278732], [-1.0317329072718608], [2.075056836809701], [2.981953805701934], [1.5707960877185616]], [[-2.6354944975063552], [-2.3453762747657336], [2.5619513205727813], [2.5505461589436855], [0.1596388597611379], [2.069554528932338]], [[3.5423322917425026], [-3.6651914331350186], [4.364735070394841], [3.799552329448912], [-2.704920598424602], [0.6087533153657795]], [[3.1588016109258277], [-3.1415926190230095], [1.8717408330302336], [0.011511387367138139], [2.3332549134312752], [-0.2971667156918079]], [[1.4155206725186562], [-0.8410685852457723], [-0.3009446096964161], [3.1627683273961], [1.4608764863997559], [-2.4320847600901416]], [[0.633694357353627], [-0.9386595844300332], [-1.8925469031570232], [1.187999369608989], [-1.7638439259154202], [-1.519852968015908]], [[1.2609249589882374], [-2.818412687875604], [1.5707962048087192], [3.5632599484215146], [-1.2885997870806973], [1.325987066293659]], [[-2.474153029401424], [-0.2478025044449754], [2.9294771381882114], [0.553584900501794], [-2.7491561366759956], [0.6808405275825749]], [[1.5202074518511357], [3.389395076054071], [2.625765399966214], [0.26143077439393225], [-1.1210001196304125], [2.536152630062609]], [[3.9269908566894567], [1.570796397342729], [1.859577131536977], [-2.9027608853395708], [-0.7264677994890536], [0.09997984879659966]], [[3.141592541958313], [1.5707962436692893], [-2.7024307876886695], [-0.5002624593861901], [3.4182642725305827], [0.06942893357760634]]], 'mode': 'real', 'shape': 'g2x3', 'L': 16}, {'edges': [[3, 4], [4, 5], [2, 3], [1, 2], [0, 1]], 'seq': [1, 0, 2, 2, 0, 3, 2, 3, 2, 4, 3, 4, 3, 4], 'params': [[[2.436509721304283, -0.18398196133696273, -0.4724559048865304], [0.876455088368926, -0.10352423612156612, 0.4024877281823959], [-1.063940778352714, 1.327766840911121, -1.579022775674419], [0.10615134111449721, 0.408672999757586, -0.6340408781744906], [-4.024077761571401, 1.6109250495366398, 3.696124531320609], [1.1043150785735898, 1.9083469846929682, -3.457695731567039]], [[-1.4328788322425408, 0.03021231885269648, 1.1701941007454961], [-0.540777112392086, -0.2257673534759112, -0.8382227445294269], [0.06114839295898588, 2.925986679666249, -0.820843688023214], [-0.16051781740496143, -1.234434657120727, 0.11930051447823663], [2.58706077922046, 1.6914426164632073, -0.5428813843856394], [3.0794321637058992, 2.1220464701676973, -0.0794525073846699]], [[-1.5074106997946, -0.07699922879725156, -0.6629320667911855], [-0.7717011583407308, 0.13148983799082226, -1.80125081982257], [-0.35026372016173946, 1.136580489625729, 0.19275759477987317], [-0.8321616071747017, 0.9124305431843097, 3.779673540179708], [3.4849816083076752, 0.9100499544234312, 0.20058975906010423], [0.3365981002371761, -1.2017263441500425, 0.3968342046319521]], [[1.1406543205242987, -0.34318385179326183, 0.10028811827093448], [0.05099286469041895, 0.47554338135235424, -0.005130863171262917], [-0.3084484954069188, -0.5408684996678091, 0.9580553004346727], [0.09648941399582135, 2.2797249392304932, 0.8850861775879039], [0.4076479750023255, -0.36569992820055663, 0.600674040263563], [0.24136311144498462, 0.8523336768737003, 0.6507270438619253]], [[0.9630602289843142, -0.18517295361706884, 0.5411431174613663], [-1.4189357745401254, 0.029287281435186413, -0.26930615409147113], [-0.4571706190893122, 0.8114385446706907, -1.7155677059422736], [-0.21129496560639804, 2.0097068037631907, -0.5738052514354826], [4.797547726759012, 0.9354954246792386, 0.46882156179055884], [0.27887505842861265, -0.4965657540021503, 0.7991330477633939]], [[-0.37098172557507525, 1.255318844938623, -0.4795281372438425], [0.1174093731976719, 0.49552181892217273, 1.8022683909965425], [1.1154264768869835, 3.1969777229350345, -0.40325571632330487], [-0.09674189644640288, 2.704985358977246e-09, -0.9005561265283686], [4.963027892666545, 2.3504090965658424, 0.05158835280439592], [0.17785459342268864, -0.9570964924102823, -0.4213785785079439]], [[-0.6819613095737728, -0.000701392811853746, -0.07883589065645245], [-3.1266904691361743, 1.0560377960414387, 2.192476738255283], [-2.4550536464386297, 1.0463407990909188, 0.9147485276509595], [1.1647821771713534, 3.1415926473546274, 0.2552648443924465], [0.11633183644735151, 1.2365412387070525, 1.5442505246010787], [0.7937685901839411, -0.8279386419108669, 0.8146745430743583]], [[-1.4417254551959753, -1.241777562834079, -0.04926429439304695], [0.5396004599429068, -1.189812688281604, 0.2557131239448777], [-7.0895374703268645, -0.5270106864527898, 3.396437001670845], [1.7840287907173076, -1.1179167001283403, -0.7808113835051305], [0.6332140153886954, 0.5064625448043223, 1.3682233892144395], [-1.554039650559028, 0.034652854909606845, 0.4750328090309278]], [[-0.41347900538768456, -0.1223383334048282, 1.0483324021481375], [-0.9652887861932654, 2.7653207088596345, -2.925090415463278], [-0.380091911660285, -4.162187641367355, -0.43588078298848787], [-0.600986074879981, -1.2023388613036854, 2.04315507342446], [-0.9313151197918438, -0.055539605095774415, 0.9821527148287652], [-0.3164039640594598, -1.2363624150985206, 1.2731817648677912]], [[0.8643333972277966, 3.1715086685033405, -2.187216692674866], [-2.981199294367488, -1.912781625708591, 0.7520317921265579], [-2.0829075326410957, -1.829478533460289, -0.18912132375727442], [3.1427243933940683, 2.487689070861481, -0.5712608574809529], [0.33046901279465946, -0.7654720489360406, -0.62428351256036], [0.04544806098648339, -0.22170672552707066, 0.044066397845372564]], [[1.595990664490684, 0.06666209443673321, -3.251995692409556], [2.4459597463960527, 2.521249813064806, -3.3351320149123143], [0.18912132072622864, -1.3121141205370956, -0.00536977875113656], [-0.7279203827106617, -0.07847855549229384, -1.0686269977610554], [-0.11509119447822601, -1.1626073802204855, 0.9987359645332383], [0.4140449098918826, 0.9440683501142845, -0.2616812495140742]], [[-1.107564644226782, 0.6544699497594729, 0.7545825279845393], [-5.205274502319869, 1.145627735332616, 3.0508674752175464], [3.835369268711106, 2.0885542457854736, -0.845098586982752], [-0.6921921710052509, -0.4910386332934258, -0.17207810203158683], [-0.3424970211307682, 0.7837700296036649, -0.8000416100428756], [-0.1314363718324124, -0.5431925772287923, -1.4995824584919974]], [[-0.960537696158997, 1.0024471090278768, -1.2147519652169283], [-1.1541639991162571, 4.8292358476839405, -0.12487645445058569], [0.5699039727560073, -0.5344118261430967, 0.735107169272211], [-1.289637502385441, 1.3470659153080409, -0.5194672161108205], [-0.6271218561276607, 1.3576158446870754, 0.66697342159105], [1.2280210097494675, 0.45288181026482494, -1.0181915370508552]], [[-0.9545718957435388, 0.895456636787502, -0.0428742404625836], [-2.2901443839280096, -1.5707963175430515, 4.016041193449703], [0.9487139897678502, 1.2699215574500613, 1.0359451916202524], [-1.1291073155413356, 0.9070265424033022, -1.3572909323976825], [-0.5502358380327791, -2.2245090837912653, 0.29964473731975433], [-1.3680300837055084, 0.14117171123075614, 0.8460333890986182]], [[-1.5539186943827208, 6.8353625093923e-09, 0.21909999676728253], [2.2671441218574304, 4.712388978912557, 0.3682795609173042], [3.7137687651074094, 1.4511103600983848, -2.232199771978807], [0.37570765580653737, -0.28275678091761686, 0.4572224807658288], [0.24289839144203926, -0.42143256484217706, -0.13991222970520914], [0.279323095556005, 1.7614115936876686, -0.3101729797307747]]], 'mode': 'complex', 'shape': 'path', 'L': 14}, {'edges': [[0, 1], [1, 2], [2, 3], [3, 4], [4, 5]], 'seq': [1, 2, 2, 0, 3, 1, 2, 3, 0, 4, 3, 2, 1, 0], 'params': [[[1.893402453994438], [-1.358752630894669], [-5.324689090014993], [3.2753562805276197], [3.916781002118719], [2.7896427812180984]], [[2.2643873873174476], [2.891139804771109], [0.7929217843982659], [-0.6666372704197181], [3.1597905753751], [1.2417007950225747]], [[0.10325320556176924], [-2.4959160311764075], [1.3929692058770387], [2.0186284563993095], [3.410804358674936], [-0.4727318655369935]], [[-1.7349302905580017], [-0.2300119283947745], [-2.1170264161016017], [-3.6160275951199456], [-1.1466852994778987], [0.7444409253665526]], [[0.44294350330046994], [-2.179939161194875], [-2.6620068053091144], [2.672781784315362], [-3.6729851020394695], [-1.484054856099981]], [[1.665413833073859], [1.2246222567334384], [-2.092154461419412], [1.081482267919913], [3.1795369651185763], [-2.087721075974799]], [[-2.325374496319009], [-2.795846893365648], [3.141591969779358], [4.246386456723975], [0.880116053885941], [-2.7392822821836242]], [[1.1177850535800011], [-0.5466940773419275], [-0.4485586017425943], [2.526112911317482], [-1.873376980713394], [-3.362004492418501]], [[0.05454850501277645], [0.8164280403056939], [3.297424726619327], [-0.02876134910247], [3.7020507131142657], [-0.47759428123645464]], [[1.9811781893296643], [-3.057107502444674], [-2.8632673561378716], [0.437057302017024], [-4.487448936656213], [1.1352153466811334]], [[-1.0773785974814531], [-1.3256105165198433], [1.3031828404313772], [0.3771022811461984], [-1.5707962724955902], [-0.6973517522474663]], [[-0.9404122243751932], [1.7984224087675842], [1.8528112528822929], [1.5707963562129643], [-3.154376431742846], [-1.8228290677329044]], [[0.269029624979035], [1.798897044339205], [1.570796329711792], [-1.707168254591967], [1.4562048988759366], [4.345273637144328]], [[2.12377735914213], [1.5707963365588735], [3.8556694660324093], [1.5255613836302606], [4.603886372540069], [2.1227197950356844]], [[2.4471577787005244e-08], [-1.5707963003411747], [-2.2848731168513177], [-1.389189449882668], [-1.3349185193106352], [-2.377016371876192]]], 'mode': 'real', 'shape': 'path', 'L': 14}], (6, 3): [{'edges': [[1, 2], [0, 1], [2, 3], [1, 4], [0, 5], [3, 4], [4, 5]], 'seq': [0, 2, 3, 4, 1, 0, 1, 0, 1, 5, 6, 5, 6, 5], 'params': [[[-0.4697479101421825, 0.14354036589843083, 0.216971448937442], [3.5342193827324, 1.570796339944114, 0.6812775103480257], [-0.09182838696813195, 4.712388989433568, 0.41472687539139474], [0.3884418704858401, 0.2981614439536927, 0.9206523734745234], [-0.3743826302324516, 0.46510442417215064, -0.034364151423258216], [1.048167274275293, 0.011125975298355809, 0.664276930056111]], [[3.8397483337747427, 0.25523713291127703, 1.2570213026698478], [3.806962255562815, 0.29362557894655084, 0.15101740074706027], [-0.21597743183125, -1.6628406955112611, 1.2989971478230182], [1.451518635460448, -0.2011959176235893, 1.0148191295978792], [-0.6033305150819187, 0.31174033953658087, 0.5451111103630175], [1.4374574165694702, -0.27095597286915435, 0.07854421215245565]], [[-0.11513348620701185, -0.22095459237725634, 0.06240885895035636], [0.3727996752196948, -0.09192202451922124, -0.49131389732048025], [0.6527356269150453, 1.577294514954798, 1.1400765803191701], [3.1956759933273746, 0.4413769296076698, -0.7106960361937337], [-3.403750844981456, 1.1900432865540664, 3.228729719279937], [0.4224261083177043, -0.08271037771609357, 0.02765149695118153]], [[-0.433241651866293, 1.5771425341878251, 1.2475895847415637], [-0.8077697265636465, -1.3577046780376638, -0.9467043913079903], [0.24511308028646198, 0.797281519249375, -0.7408589982728321], [-0.24049018431166713, -0.44846015610607565, -0.02678036600406528], [4.469017159806411, 2.7345662425306796, 0.032580847822934605], [-2.2973630000119027, 1.4502313065981451, 2.991958751603726]], [[1.0653523965378575, -0.39008092302742114, -1.850360020199692], [-1.938703683114272, 4.000051496770453, 1.132492559232741], [-1.4413316858017793, -0.7201339115158631, -1.0887064378140232], [-0.9794419690955897, 0.05799065311713135, 0.5589669360390761], [-0.2968198695921201, 0.9089304465279313, -0.20419473073666086], [2.948905796955601, 2.443226790052348, 0.310356358991686]], [[0.07904622604538394, 1.1215979013819493, -0.5275629593184741], [3.102673775247766, 1.5707963276341408, -2.0899090959187743], [2.73505968185626, 2.772727551249343, 0.04879293319228327], [-1.2822765425807137, 0.47557120728015234, -0.10626051936603376], [-0.6895033203334753, 0.43591391347591657, -0.10316668932756197], [0.6159104766275649, -0.3002828603903151, 1.165475458916266]], [[-1.348028210586758, -1.0139805171464333, -0.26614661612937135], [-4.193276212412864, 1.570796330000439, 3.068305729100778], [3.150944762792256, 0.7024615822244114, 0.19929099663464], [-0.1481085945354065, -0.5035481825816117, -0.0824063430419744], [-1.1054370619322895, -0.3499698800788111, 0.8340244289869396], [0.5669117225033605, 0.36381492462098597, 0.04112114696069479]], [[-0.782871558582991, 0.5631589314078885, -0.48312997614564585], [0.07328690158836208, 1.5707963351722176, -0.06817553615543319], [0.012651488137764457, 0.8794727153976564, 0.4335011615415783], [-0.23065411198420974, -0.5229519528992616, -0.31661162614344773], [-0.45948024696626816, 0.21902330089938563, -0.12618828324560358], [0.19895485432260177, -1.12437280857267, -0.4143905887633831]], [[0.5010635849995209, 1.007709988785061, -1.0585910030218826], [-3.073417118469305, 1.5707963279143664, 2.759627956769439], [0.8424950612334743, 1.041501018019739, -0.08636385786799484], [0.5378932432695936, -1.2221165067101445, 0.7753131208719423], [0.05247038540230047, -0.04553799127009248, 0.24290939242997808], [0.6228944431227869, -0.16912772322693612, 0.6996954257075494]], [[-0.16632312785770775, -0.3411507656059588, -1.3373252243880447], [-0.11160859946158631, 1.5863562290544082, 0.5860852422482622], [3.629370677687296, 1.6943392290803603, -0.835807418146529], [0.1883468121078831, -0.2871194983033381, -0.4814102846520265], [-0.4388510052669365, -0.01646834681882102, -0.7711897082722302], [-0.8724554730959349, -0.9586730985522209, -0.05404491698309201]], [[0.19847407221373675, -0.3711139750372618, 0.3938812947701678], [-0.788363805472659, 0.29675446168657504, -0.1733748583768095], [0.09600508910879216, -0.7753699594274615, 0.8761889036759968], [1.5421665731806253, -0.07297348785916358, -2.262730807000475], [1.729894106021427, 0.6154796985371431, -1.0692799516674394], [-0.6783577278475932, -0.21734078007676544, 0.6435699009844111]], [[-0.07991339063330909, -1.7919142057201367, 0.02751224134476518], [0.8688921373276319, -0.3574063341917124, -0.9192151588677052], [0.41538381161787846, -0.11532787928649982, 0.9727461858317555], [-0.6836399798035789, -1.6423884180429433, 1.8993941545077004], [-4.789857516734782, 1.570796340285785, 2.2052295432088584], [4.2366905623532665, 1.7909334470439344, 1.2629039771527593]], [[0.47845754556653364, -0.2845148731730981, 1.3130730467663336], [0.6619796975599961, 0.022944901597353483, 0.05164943004721586], [0.06207222181350653, 0.4684537853186901, 1.3091454292351161], [0.7497033213686964, 1.609868397798912, 0.28748739771507703], [0.9363631075204141, 1.5707963151271884, -0.17138152712093166], [1.073078572192245, 0.31246182960732516, 0.03400253036641449]], [[0.9714194829101646, -0.34126637147447414, 1.082658074433393], [1.138432896749022, -0.880258433018271, -0.22747538606547937], [0.8290692408783975, -0.5951384068183778, -1.267380020347334], [0.27517251505753204, -0.04618391849937205, -0.8673410015567765], [-2.5461632730970893, 1.5707963229658395, 4.842548827355712], [0.03859362763231639, 0.598690379904494, -1.0013052377947564]], [[0.5717458554507768, -1.365844658314836, 1.5984604289043463], [-1.1113096219350675, 0.13238199958692692, -0.8362229731365327], [0.702125088990648, 0.27157426180862493, 0.6382458931561625], [-2.8364302843888205, 1.5707962936451842, 2.414529934021484], [-0.4818947831305035, 3.141592624705666, 0.4919986730997689], [0.046222901206536555, 1.1954660730052373, 1.3218070747731556]]], 'mode': 'complex', 'shape': 'dc2', 'L': 14}, {'edges': [[0, 1], [0, 3], [1, 2], [1, 4], [2, 5], [3, 4], [4, 5]], 'seq': [3, 2, 4, 3, 6, 0, 4, 1, 3, 6, 5, 5, 0, 2, 4], 'params': [[[2.4636320289916265], [2.133490754134326], [-0.6571009783349802], [0.3146590749055514], [-4.5611763834626124], [2.0222997511537653]], [[-1.2085317602635253], [0.7415575833102286], [3.116477268560992], [1.9929057026959274], [3.7279419641481484], [0.20708735000623493]], [[-0.9433845467022034], [-2.939471889404241], [-0.2446528490600339], [-1.0237736130861346], [-0.359683814193137], [-1.2740704764148463]], [[-3.7594401304339353], [-0.919024941546302], [1.342316366393771], [-2.5514529786214504], [4.32343719333378], [2.2443634903392584]], [[-3.130671680491299], [-0.9850304097742275], [-0.9362419908714198], [-2.1176676300176234], [-3.3025144117736063], [-0.058087459389658036]], [[1.8660071085146357], [1.7704285765638363], [0.345248648053129], [0.5114031973425801], [-3.538798170363096], [-0.6590838746104934]], [[-0.4490446138566135], [0.6621972514017227], [1.604871466004706], [2.5387457731334213], [-0.6846350151301367], [-5.624101428252175]], [[2.0198409406515068], [-1.7969948196293084], [1.8593089101332925], [-1.23561588950062], [-5.675689975211796], [0.9704782157843209]], [[-1.1078839035696348], [-0.4359987585673095], [1.6836905414805652], [0.7432312218373499], [2.0451415246023963], [0.3370392964112028]], [[1.4539020384038912], [2.067466039897508], [-1.117568589374367], [1.6929924920716077], [1.5707963267948968], [-1.3075175071374598]], [[2.5526417407105835], [2.472102178945462], [1.002180071093567], [0.8228034416264709], [-3.1415926535897927], [-3.7859559176111257]], [[1.5190340160693998], [2.7155250715315904], [-0.23497999625102936], [-1.5707963267948968], [3.141592653589793], [-1.9159675284225788]], [[-2.8468975648193426], [2.1696846703948185], [1.1529200035289289], [-0.6474896543929946], [2.227654054804875], [-1.3365690199917084]], [[-1.7508742233688714], [-4.712388989245195], [-3.5601527769496353], [-1.4705789760919428], [1.65257659235498], [-2.8244300544682193]], [[0.07021959757324019], [2.686159329013792], [1.5707963267948968], [0.30051556802999535], [0.48025646899234775], [-1.1326517670706449]], [[1.6806546118366226], [0.45543331777962176], [-3.1415926592250227], [0.3641912837503846], [0.351901868684398], [-1.5707963360454544]]], 'mode': 'real', 'shape': 'g3x3', 'L': 15}], (7, 2): [{'edges': [[1, 2], [2, 5], [1, 6], [0, 1], [4, 5], [5, 6], [3, 4]], 'seq': [0, 1, 2, 3, 0, 3, 0, 3, 3, 4, 5, 4, 5, 6, 4, 6, 4, 6], 'params': [[[3.514334393200789e-05, -0.14230148625046746, -0.33761513113744257], [3.141608849707121, 0.45175754265398305, 0.30893082219801804], [-0.0001250368895881783, 1.9590182667166123, -0.452281971099805], [-0.00014322897816364843, 0.09929771536644925, -0.04556057900869329], [-3.544540105523399e-05, 0.002999570497782332, -0.02962308275101221], [-0.00015724634556808895, -0.0008608246252839226, 0.023524355676889073], [-0.0005697101012550258, 0.0469520037574578, -1.2266223705131478]], [[-0.33761513126833065, -0.19538820654579586, -0.4091817927093358], [3.4505234759615138, 0.5277536883355236, 0.6087921803701046], [-0.45228197099833195, 0.10113899537562183, -0.23449374337788617], [-0.04556057890918054, 0.07853272448700692, 0.05634088655839601], [-0.029623082642244943, -0.022025547574877985, -0.5230439190741256], [-3.1180682977668837, 1.5699364584354314, 2.8524137773487563], [-1.2266223704654329, -0.08611816397448187, -0.5657417614362447]], [[-0.4091817926097234, -0.05507888647743497, -0.1339799031682119], [0.6087921803930283, -0.24437949036970658, 0.17913519283029805], [-0.23449374351069366, 0.10148015154454408, -0.30189593963595507], [0.05634088662852123, 0.013793543948253035, 0.6206262684926408], [-0.5230439188866762, 0.022075810734350037, -0.6062168254418362], [2.8524137774559977, 2.0495114352921266, 0.3582269633780081], [-3.707334414775364, 1.49075233677708, 3.164440443581182]], [[1.4368166318594198, 0.9356723264374378, -1.6762174908316703], [-1.3916611211854346, -2.427416158879793e-09, 1.8945707989856688], [-0.3018960401077659, 0.05382122919747977, -0.2732280579524396], [0.6206263180918266, -0.006668049458193751, 0.17630975813810928], [-0.606216775959603, 0.011386294201886783, -0.2705644453270107], [0.3582270843236659, 0.18580988746232618, 0.32699056343611194], [3.164440427073029, 1.7060464883385584, -0.6608873724568831]], [[-1.8089456587962078, 0.6384134240966632, 1.7160898304365202], [2.3408101089367346, 1.5707963315506883, -1.5710787178689236], [2.870761560716293, 1.6588140523659818, 0.08575023467405612], [0.2706382569660595, 0.039701904507770974, 0.1625017603398678], [-0.29683583984555745, -0.14757253396906897, -0.23976483913923285], [0.31526478557598403, 0.045541640201825107, 0.27478483493763617], [-0.6247850390697274, 0.19536608606146091, -0.6312894185653496]], [[0.14529364271235903, 0.3142034958988559, 0.06671068772839446], [-3.141874865156187, 1.5707963291771652, 2.6792570180504964], [3.227342888231807, 1.0315045558136278, -0.12813014958438065], [0.1625017603307269, 0.0783475087945932, -0.28212377552747303], [-0.23976483910614316, -0.3652464833885134, -0.11838402902814171], [0.27478483484519817, -0.09762530395272777, 0.23404758170471818], [-0.6312894185073316, 0.1800593855659064, -0.5143716142270983]], [[0.3717993321581336, -0.1817916921703556, 0.5724765853448129], [-0.49298082132092486, 1.5714877964433158, 1.318617384281792e-08], [3.0132621450671175, 1.954576835923572, -0.17992604182967353], [-0.43113848230352136, 0.11656036579793323, -0.5746551486698469], [-0.02155372240369196, -0.7168178043963873, 0.053944224557270284], [0.19247393032378998, -0.13087340061846112, 0.09584618973571618], [-0.42292048081855205, 0.3501904337771821, -0.4219326362869777]], [[0.5724765854833296, 1.6465853956666816, -0.30999974237737266], [1.3215290524152304e-08, 1.6421918206894661, -3.4190701487505502e-09], [2.9616666117213133, 1.5379170975271887, 0.1211444188734155], [-0.5746551484655683, 0.15032856352634533, -0.42986904707710943], [0.053944224574835706, -0.9156238647400392, -0.009188334113790166], [0.09584618985976995, -0.19403910690374454, -0.030248416303996464], [-0.42193263642426604, 0.3215564020720291, -0.2786431749815401]], [[-3.4515923959393606, 1.5707963346398746, 3.541901337635944], [-3.3281884065350922e-09, 0.7853981371582315, -0.10293854958120247], [0.12114441889119189, -0.017848449266141528, 0.13405148237146436], [-0.42986904722018277, -0.038935416669436666, -0.25588688472488674], [-0.009188334171686333, -0.9569746587259331, -0.14500770836545004], [-0.03024841633484461, -0.22618999409008994, -0.03141063941734234], [-0.2786431749231663, 0.4134449559766276, -0.14837057299849132]], [[-2.741283969566938, 1.4124696389541878, 3.112776054789329], [-3.244531203140922, 1.662567923923991, 3.0024005484298932], [0.13405148230992742, -0.003079399865136034, 0.1463827351891149], [-0.2558868846300857, -0.09935117967661362, 0.056813234556943225], [-0.14500770838963442, 0.7199413039400707, -0.3731424343678933], [-0.031410639323455586, 1.3449793332489477, 0.4208446272466601], [-0.14837057302765957, 0.4822780977706939, -0.015710540358224164]], [[0.05581519121949817, 0.06646513936012766, 0.16482224870563789], [-0.12970717490748537, 0.054011959770461415, -0.04135190812821674], [0.1637297625591777, 0.013492342652147974, 0.15784730869352578], [0.2901146517206549, 0.14660124194888394, 0.3897718791779084], [0.01426030673839385, -0.6333031272475159, 0.6106184559091417], [0.034504761005054195, 1.5708750489722103, 0.7853981705317902], [0.01080568849903446, 0.8177089552941014, 0.10558727626124854]], [[0.16482224873707596, -0.06090321797192035, 0.2531258432393738], [-0.041351908120652786, 0.1541639350410671, 0.1023526750647593], [0.1578473087706168, 0.04515125049040437, 0.14660554118534122], [0.38977187919061157, 0.1656402296232003, 0.1690608106207114], [0.6106184560103906, -0.3070689205657296, 1.233100859488472], [-2.356194483100858, 1.5706104792801407, 2.617993887216993], [0.10558727630353966, 0.38485898051597695, 0.1257047537540039]], [[0.25312584321729736, -0.23481785211026043, 0.2581270705591224], [0.10235267501889758, 0.04463762419093071, 0.09792071096660983], [0.1466055411701923, 0.04552372058799957, 0.11951013641141586], [0.1690608104825486, -0.10755171328917457, -0.1642980378462206], [1.2331008593627355, 1.1288994534941514, 0.716991376174257], [-0.5235987663777472, 1.5707963130543554, 0.07256345098490519], [3.2672974073439067, 1.418532737739155, 0.05621678676577428]], [[0.2581270705712419, -0.25881722071917607, 0.20211610109454037], [0.09792071097751934, -0.00017204477548375837, 0.11715773844917658], [0.11951013639880212, 0.08732043929355914, 0.08560103860208988], [1.4064982489353872, 1.323685592754783, -1.7729398647702876], [-0.8538048202914956, 0.4455368722197793, 1.5000108074589857], [0.07256348473167772, 0.4198977761957099, 0.03869260160790565], [3.1978094011455918, 1.5923412053641357, 0.028959729515413975]], [[0.060025001261069195, -0.18699027456277728, -0.031085782686096713], [0.14971162687662526, -0.02329825437824258, 0.1517609301454588], [0.11674188988313121, -0.05018270623838658, 0.11195174066132671], [0.30222826821687704, 0.8234845229182023, 0.08458752195635584], [0.2569928386729657, 0.6154796546508969, 1.093138052843461], [-0.0017393538883053482, -0.3356562012176489, -0.06601839867137446], [0.013986374427451837, -0.05031057232411135, -0.007290242122103742]], [[-0.031085782680036082, -0.12712598993559476, -0.04933846111286305], [0.15176093015397266, -0.06464051369850435, 0.08449849888123855], [0.11195174066369676, -0.05139323364994986, 0.11190959528724333], [0.08458752196309881, 0.8059859170944191, 0.08824998447650352], [-2.048454600753806, 1.5707963197001313, 2.356194488332286], [-0.06601839865660218, 0.5952875541624876, -0.01938349422195512], [-0.0072902421452765934, -0.037073676412924625, -0.0180956280354785]], [[-0.04933846115695575, -0.07700721613484582, -0.058505283828077737], [0.08449849891247961, -0.027528069259763262, 0.057490176238450055], [0.11190959521645837, -0.022556047915461865, 0.058679769601922685], [0.0882499844640472, 0.5590893076973317, -0.5589565364510246], [-0.785398165236327, 1.570796167173897, -0.7853981750442783], [3.1222091593367978, 1.5500882089360255, -0.00990716841989444], [-0.01809562797566631, 0.0447969980380188, -0.01751154611645986]], [[-0.05850528380700699, -0.03137499274645992, -0.0650837797656712], [0.0574901762166589, -0.04417682953568323, 0.029280493629061474], [0.05867976960972186, -0.0125572638402896, 0.0292795222130799], [-0.5589565364644861, 1.3035710157996514, 0.05189290260830724], [-0.7853981751079788, 3.9269908756958816, 0.00836196768519713], [3.1316854851969693, 1.6678987898547382, 0.07058115487891728], [-0.01751154609886034, 0.01354591372309916, -0.002335672041183919]], [[-0.14295501430198207, 0.11078684498141975, -0.20174753824092315], [-0.026098051423848505, -0.042383537266058605, 0.05289654273244213], [0.007585655989143303, 0.049414314889450944, -0.01974087901860147], [-4.267108631990265, 1.5707963105800502, 3.1675331008785896], [-3.149954632131081, 1.5707963121386463, 3.167533151540456], [-0.0004916878729186883, -0.09733995944578791, -0.04381819600490318], [0.013026552596016405, 0.007452703289302557, -0.027307373687155564]]], 'mode': 'complex', 'shape': 'dc2', 'L': 18}], (7, 3): [{'edges': [[1, 2], [0, 1], [2, 4], [1, 5], [0, 6], [3, 4], [4, 5], [5, 6]], 'seq': [0, 1, 2, 4, 1, 0, 1, 0, 1, 6, 7, 6, 7, 6, 7, 5, 5, 6, 5, 6, 5], 'params': [[[-0.26348550118064473, -1.0206988418061158, 0.17002245917294215], [3.0852261393154894, 1.8473375452072458, 0.5374575871967219], [-0.006102940253998708, 1.1907402847861976, 1.1208194510346894], [0.03036694774339881, 0.020168308030659117, -0.21859936370923239], [-0.13553173363115567, -0.499174645978553, 0.3218338712397363], [0.5077188737084458, -0.03096077090790489, 0.27226672846968775], [-0.4509399465227487, -0.15662886716919058, -0.7440609497226147]], [[3.4470261075045694, 1.282190902943396, 0.00951176280418267], [3.695618796791894, 1.740575987610019, 0.0373030969776446], [0.8972540805897533, 0.4152825739759224, -0.2875104197594087], [-0.3834353463652349, -0.30709888434237764, -0.6446583932758173], [0.07043056585256206, -0.9053688583732189, 0.6212109835046178], [-1.2824436065644478, 0.1897056312973, 0.13688776721753582], [0.37602718018436926, 0.24765389163976426, -1.0113844472238323]], [[2.9192230194088937, -1.7177214347965846, -0.8717308914716253], [-0.6874739971603697, -1.551235424602963, -0.5328473766264741], [0.13327251179029523, -1.4073068151894303, -0.6264041581557835], [0.7051001397323053, -0.07750438849985833, 0.38411722293947326], [-2.4498741822483923, 1.230005945382917, 3.886208069204923], [0.05430498489690192, 0.45792028556672804, -0.8144806502538489], [-1.5040929796229023, -0.19121211245980735, 0.07875372215060676]], [[-0.6837437687713114, -0.3788180402949414, -1.6339650556038705], [0.18593677027804073, -0.7768017587413087, 0.701275013637262], [0.6310564387981086, -1.567946765738975, -1.078659809453305], [0.005787832547577228, 0.4008399005078065, 0.9406081416599956], [4.261716408817709, 1.7172754146120104, 0.7128500402842609], [-2.7515414228799893, 1.769494594730126, 2.6715079975762], [-0.45448278734550457, -1.3090574066686758, 0.2208062887103735]], [[1.692234087252278, -6.974625960418582e-08, -0.8646643520726237], [-0.3537946864779641, 0.7044299192686054, 1.4155373446220023], [0.39382526299511866, 0.12958926168231635, -0.08114532236852445], [0.1274277898563157, -0.2686769825018557, 0.9515889815461003], [1.0351958658666893, 1.4492163687113093, 0.37099706279590244], [0.057242049898989185, 1.6339036466949384, -0.6740694757916311], [3.9263297742194667, 2.7926529017735393, -0.23478790901481802]], [[1.7988247688657273, 2.3358676007311505, -2.004243666883213], [2.0524904907079518, -0.4765682869196775, -3.158122741859486], [0.1887650277178235, 0.8820638220224671, 0.6990789802336549], [0.8596369576852981, -0.10685587311708683, -0.20118598013604258], [-0.5387775675400884, 0.0948394432459385, -0.07509412001190553], [1.0852165303248824, 1.103137394121791, 0.4154690679774751], [-0.159451695223056, -0.4036238845118594, -0.39808922591164303]], [[-1.0267823680376993, 0.7681315963807629, -0.2374564300455789], [-3.398988625632168, 1.5707963485924255, 2.416321848168761], [3.4793319170955734, 1.5972922367369622, -0.6906195580813448], [0.45293103159982895, -0.10505862288969259, -0.3635841080180736], [-0.41357255263688475, -0.20856384671535302, -0.0027943069866463916], [0.12171110026575821, 0.28965727241161215, 1.9940481082681145], [-0.4789339058568859, 0.4187619224768192, -0.07569106483778525]], [[-0.725600168433081, -0.26402604145767106, 1.2486105937870962], [-0.8455255195942052, 1.5707963836988952, 0.44568891142534817], [3.6762882125610123, 2.370667327078278, 0.5396339940326697], [0.010538772323466533, 0.014178637394680702, 0.19915168396286223], [1.592744632123392, 0.8935327304797527, 1.1050112852887872], [0.8578560707925451, -0.4551527184459854, 0.6890534869313583], [-0.18854169063609177, -0.19758903139970996, -0.0077950933449052635]], [[1.1882128249620774, 4.5092958575642745, -0.9233777053454837], [1.125107438720388, 1.5707963247033816, -0.6086853497515219], [2.9456663246409636, 0.9682459563962468, 1.5237599816934655], [0.18392500910569554, 0.5835042688002354, -0.1845448456858499], [1.4271958616308695, 0.6951897059394333, 1.385314315865462], [-0.1262876198801402, -1.3111633732290469, 0.21445121390518604], [-0.9097543907357827, 0.7551000877434154, -0.24103243471945485]], [[-4.245057975771746, 1.3418764338582645, 2.3587602409863537], [-2.973358977905779, 1.997741903671785, 2.2049563132554337], [-0.7460538890514838, 0.18879939479259902, 1.1417740701421666], [0.36014888317456023, 2.522922305121133, 0.15793526943688613], [-2.7093828157584827, 1.684416270266911, 4.527148366862952], [0.9299983969602711, 0.29999800273025373, 0.6769180534427885], [0.06011540801992104, -0.9355183359465433, -0.13107696649108586]], [[1.1595774010813287, -1.4117456199744414, -0.02522079272041375], [-1.019826316234667, -0.5401429034133896, -0.3659531904960064], [0.412298116538112, 1.2245916769808682, 0.3354141443125156], [0.13729705577529372, 0.2330362666496508, 0.5345600899375715], [-3.2288787010878246, -0.44076139839632367, 4.051520310009035], [2.0151806709238946, -2.4205553079076263, 2.814492761954932], [0.5630678328003632, 1.3379605179073093, 0.028047981953711658]], [[-0.0022185226456561094, 0.3834819009409995, 0.5406048736559452], [-1.1071809539298485, 0.5041131264178942, 0.3903872988961908], [0.4184511418063779, -0.02727152873110674, 1.0748500718962728], [-0.08547441130235103, 0.4067584982212544, 0.47873230344055284], [1.309750222266483, -1.9586707945680542, 1.32557476392811], [1.1179606799044348, 1.370229255420093, 2.728009288902887], [2.059081604653072, 2.0599244343829204, -0.02194052357447921]], [[0.4844112702444055, 0.7877800054444111, -0.6807501481791213], [-0.36557243777853077, -0.23359558031778185, -0.5194833398031721], [-0.021732318269065673, 0.673661159380786, -0.09878197124561706], [0.2044571931223851, 0.11993466144273648, 0.9337965722744884], [-0.09560728108961682, 2.912481453539775, 1.0586082851748186], [-1.7192795942536045, -1.6649130466347049, 2.2275367124191288], [0.0219405265273162, 1.0816682457841171, -0.8981455488943052]], [[-0.10643572597608893, 0.12854786203859028, -0.7428094260014535], [0.9416309815049622, -0.2327709269917364, -0.11986303657979816], [-0.5591890360058629, 0.357985266618456, -0.4441228037459207], [-0.05492899295169904, -0.31711008880038644, -0.1397365188642181], [0.061754029440862314, 1.2590154802419804, -1.6563987065479167], [-1.6934576355849988, 1.2934761801559505, 2.3225107626915746], [0.2962475482132192, -0.29671719901331306, -0.5893788962438223]], [[-0.7374340931137106, -0.21777820545459226, 0.30674647819889994], [-0.3443753912293068, 0.7345349312872571, -0.12195487742185748], [0.059893129679426484, 0.5232940617696746, 0.6087689149656126], [0.06092042929558642, -0.7586427498823662, 0.5724576636575239], [-0.6870409326480157, 4.744375273522538, 0.6492749721299133], [-1.0152983601259609, 2.171700240677934, -0.2095496768491725], [2.5325863449317625, 1.4600035318495417, -0.14915354156451968]], [[-0.5341947984712329, -0.6699760612363124, -0.9061418011403044], [1.3498327358827655, 0.7310118099417731, 0.13140388411102052], [0.21498357313062094, 0.24985289649726689, 1.6573742694736253], [1.3364307710989465, 2.2328167225258304, -3.2268759542019883], [-1.612976197918249, 4.803359757266591, 1.2697492343120849], [-2.2120664467252937, -0.39709537535888684, -0.3738859745486365], [3.188371526274578, 1.8720835482965588, -0.7032895499622404]], [[0.6350704500907706, -0.07734643379033698, 0.2502791733967134], [-0.056405490632667435, 0.9265813769576525, 0.07471872768660687], [-0.8342706230238782, 0.6885867585718788, -1.304189357803026], [0.9943633684512004, -2.1838758107017586, -2.4242850859318934], [2.1237980017392992, 2.5250578898601193, -2.001609507094898], [0.21480253143331238, -0.49778489633832224, 0.15965886731989143], [-0.9387738930453569, 0.5793896799005187, -0.036673811933289]], [[0.6401079705196643, 0.2569512793809154, -0.6157936394876807], [-1.2051572941281, 0.20791476095051414, 0.1383892739838709], [-0.07257423378317661, -0.9717774930099571, 0.655611777101835], [-2.091259249101964, -0.48287913898405377, 2.1884864697521285], [1.997118240838962, 0.7854082435141824, -1.0502448333679204], [3.0607359605133118, 2.2497048355411096, -1.7723308862039386], [-0.6834868312920829, 0.25256451851461464, -0.45114384945102853]], [[-0.6133489885664967, -0.37508606685282686, 0.4392557582557361], [0.19361712269987733, -0.2476966995373518, -0.20000914865000735], [0.5000222227508261, -0.551790313878346, 0.2747480402138197], [-0.34916442081467564, -0.7089791039352653, -0.019900614315109623], [-2.39161767420513, -1.5707963332900805, 3.3394768929991736], [0.8208838545962066, 1.8801473948455314, 0.17130766024425176], [-0.33371829791185464, -1.1433234546178586, -0.8869414549864097]], [[0.7246805907003203, 0.37776580842988805, 0.5066772617764177], [0.15405240702704756, -0.5454224452530352, 0.5514839552772163], [0.1250341894932304, 0.6127966453022318, -0.5829864219935007], [-0.5334656673617482, -0.5966286129280696, -0.6081402822062533], [-0.1978842203682666, -1.570796337283134, -1.7693709619468334], [0.6401601256943016, -0.43464848474040374, -0.7111381450890302], [-1.4853949612411084, -1.1769835235158377, -0.08452584001140433]], [[0.2814444436427497, -0.3364513864548408, 0.621933706824765], [1.1407180000703545, 0.1595805309587136, 1.0398200950722956], [0.6051952284814879, -0.3347310297205185, -0.44464877885466797], [0.03872217935877827, -1.051195934521329, -0.4709949368046586], [-3.249639547948472, 1.570796331505269, 4.589610820890546], [-0.5492290713784035, -0.23949276037098408, 0.30473127412805], [0.6782127937457264, -1.5600544507895968, 0.6731881689583994]], [[0.025556443044610692, 0.04062020511965397, -0.4645248676995128], [-0.6286904755356497, 0.568371498714543, 0.9048772902705304], [-0.6030124319593163, 0.045349582383908495, 0.31990847480809853], [-0.03403138692644361, 5.393318289599633e-08, 0.3662523878097907], [1.6935744515728748, 1.5707963547036572, -0.4465297009781195], [3.3467792727267422, 1.3608218312665141, 1.0082228816371797], [1.219809902410463, -0.8559430178068528, 1.3222630538765108]]], 'mode': 'complex', 'shape': 'dc2', 'L': 21}], (8, 4): [{'edges': [[2, 3], [1, 2], [0, 1], [3, 4], [2, 5], [1, 6], [0, 7], [4, 5], [5, 6], [6, 7]], 'seq': [0, 1, 2, 3, 4, 5, 6, 2, 2, 1, 2, 1, 2, 1, 0, 1, 0, 1, 0, 2, 2, 1, 2, 1, 2, 1, 2, 2, 7, 7, 8, 7, 8, 7, 8, 9, 8, 9, 8, 9, 7, 7, 8, 7, 8, 7, 8, 7, 7], 'params': [[[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [3.141592653589793, 1.0683814272372003, 0.0], [0.0, 2.901974084960706, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [3.141592653589793, 0.6020729202613331, 0.0], [3.141592653589793, 1.0683814272372003, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[3.141592653589793, 0.24497866312686428, 0.0], [3.141592653589793, 0.6020729202613331, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[3.141592653589793, 0.2449786631268643, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [-3.141592653589793, 1.5707963267948966, 3.141592653589793], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [3.141592653589793, 1.5707963267948966, 0.0], [-3.141592653589793, 1.5707963267948966, 3.141592653589793], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [3.141592653589793, 1.5707963267948966, 0.0], [-3.141592653589793, 1.5707963267948966, 3.141592653589793], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [3.141592653589793, 1.5707963267948966, 0.0], [-3.141592653589793, 1.5707963267948966, 3.141592653589793]], [[0.0, 1.5707963267948966, 0.0], [-3.141592653589793, 1.5707963267948966, 3.141592653589793], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [3.141592653589793, 1.5707963267948966, 0.0]], [[0.0, 0.5235987755982989, 0.0], [0.0, 0.5235987755982989, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[-3.141592653589793, 1.5707963267948966, 3.141592653589793], [0.0, 1.5707963267948966, 0.0], [3.141592653589793, 1.5707963267948966, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [-3.141592653589793, 1.1780972450961724, 3.141592653589793], [3.141592653589793, 1.5707963267948966, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.39269908169872414, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [-3.141592653589793, 0.39269908169872414, 3.141592653589793], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 1.1780972450961724, 0.0], [3.141592653589793, 1.5707963267948966, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [3.141592653589793, 1.5707963267948966, 0.0], [3.141592653589793, 1.5707963267948966, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [-3.141592653589793, 1.3089969389957472, 3.141592653589793], [3.141592653589793, 1.5707963267948966, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.26179938779914935, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [-3.141592653589793, 0.26179938779914935, 3.141592653589793], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 1.3089969389957472, 0.0], [3.141592653589793, 1.5707963267948966, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[1.5707963267948966, 1.5707963267948966, -1.5707963267948966], [-1.5707963267948957, 1.5707963267948966, 1.5707963267948957], [0.0, 0.0, 0.0], [3.141592653589793, 1.5707963267948966, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[1.5707963267948966, 0.6154797086703876, -1.5707963267948966], [1.570796326794898, 0.6154797086703868, -1.570796326794898], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[-1.5707963267948966, 1.5707963267948966, 1.5707963267948966], [1.5707963267948981, 1.5707963267948966, -1.5707963267948968], [3.141592653589793, 1.5707963267948966, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [-3.141592653589793, 1.2630564724597029, 3.141592653589793], [3.141592653589793, 1.5707963267948966, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.30773985433519363, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [-3.141592653589793, 0.30773985433519363, 3.141592653589793], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 1.2630564724597029, 0.0], [3.141592653589793, 1.5707963267948966, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 1.5707963267948966, 0.0], [0.0, 1.5707963267948966, 0.0], [3.141592653589793, 1.5707963267948966, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[-3.141592653589793, 0.7853981633974483, 3.141592653589793], [1.2251903831398114e-16, 0.7853981633974483, -1.1496735851465472e-17], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[-3.141592653589793, 1.5707963267948966, 3.141592653589793], [-3.141592653589793, 1.5707963267948966, 3.141592653589793], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 1.5707963267948966, 0.0], [-3.141592653589793, 1.5707963267948966, 3.141592653589793], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.5235987755982989, 0.0], [0.0, 0.5235987755982989, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [-3.141592653589793, 1.5707963267948966, 3.141592653589793], [0.0, 1.5707963267948966, 0.0], [3.141592653589793, 1.5707963267948966, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [-3.141592653589793, 1.1780972450961724, 3.141592653589793], [3.141592653589793, 1.5707963267948966, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.39269908169872414, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [-3.141592653589793, 0.39269908169872414, 3.141592653589793], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 1.1780972450961724, 0.0], [3.141592653589793, 1.5707963267948966, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [3.141592653589793, 1.5707963267948966, 0.0], [3.141592653589793, 1.5707963267948966, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [-3.141592653589793, 1.3089969389957472, 3.141592653589793], [3.141592653589793, 1.5707963267948966, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.26179938779914935, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [-3.141592653589793, 0.26179938779914935, 3.141592653589793], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 1.3089969389957472, 0.0], [3.141592653589793, 1.5707963267948966, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [1.5707963267948966, 1.5707963267948966, -1.5707963267948966], [-1.5707963267948957, 1.5707963267948966, 1.5707963267948957], [0.0, 0.0, 0.0], [3.141592653589793, 1.5707963267948966, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [1.5707963267948966, 0.6154797086703876, -1.5707963267948966], [1.570796326794898, 0.6154797086703868, -1.570796326794898], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [-1.5707963267948966, 1.5707963267948966, 1.5707963267948966], [1.5707963267948981, 1.5707963267948966, -1.5707963267948968], [3.141592653589793, 1.5707963267948966, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [-3.141592653589793, 1.2630564724597029, 3.141592653589793], [3.141592653589793, 1.5707963267948966, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.30773985433519363, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [-3.141592653589793, 0.30773985433519363, 3.141592653589793], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 1.2630564724597029, 0.0], [3.141592653589793, 1.5707963267948966, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 1.5707963267948966, 0.0], [0.0, 1.5707963267948966, 0.0], [3.141592653589793, 1.5707963267948966, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [-3.141592653589793, 0.7853981633974483, 3.141592653589793], [1.2251903831398114e-16, 0.7853981633974483, -1.1496735851465472e-17], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [-3.141592653589793, 1.5707963267948966, 3.141592653589793], [-3.141592653589793, 1.5707963267948966, 3.141592653589793], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]], 'mode': 'complex', 'shape': 'dc2', 'L': 49}]}
PEEP = {(5, 2): [(5, [('u', 2, (0.06417416539079573, 0.2002061048513533, -0.15863876446279643)), ('u', 3, (-0.25230469651845844, 0.11499575798417633, -0.11415213349578218)), ('u', 4, (-0.04641210951175617, 0.047097538848295754, -0.345014388292597)), ('u', 0, (3.4101130654655565, 2.3752905945655853, -0.8270713443163429)), ('u', 1, (0.06653450325499813, 1.6142750764453435, -0.020401255826077887)), ('cz', 0, 1), ('u', 2, (3.0824116174688734, 0.2665572236654895, -2.6718660872937625)), ('u', 3, (2.812749741470863, 1.1504773191668187, -3.291761196786068)), ('u', 4, (-0.3386503216440297, 0.2568185357469526, -0.958397282468898)), ('u', 0, (-1.3503117116976115, 0.059308187734340906, 2.4465534531296043)), ('u', 1, (-3.438206272788822, 1.0030328897431138, 2.834065801596405)), ('cz', 3, 1), ('u', 2, (2.8427869724867643, 0.0107155370572205, -3.3089695261608014)), ('u', 3, (3.662533412840636, 2.0711396814058896, 0.5934087022435817)), ('u', 4, (2.6013466232852736, 1.561574539622621, -3.312151135708441)), ('u', 0, (2.3692144472676833, 1.3200739426623036, -3.0607921888285197)), ('u', 1, (2.959402844346953, 0.3258250188609984, -2.5210423351250766)), ('cz', 4, 0), ('u', 2, (0.060212989652261226, 0.08199952126939657, 0.17983937989010085)), ('u', 3, (-2.0546677704294, 0.41359455675247786, 3.4966300158551316)), ('u', 4, (2.927982661767651, 2.3890840222883973, 0.008562608568400965)), ('u', 0, (0.0851468004887593, 1.5707962365246233, 0.45774770083199323)), ('u', 1, (0.48667563634765765, 1.7207376779086916, 0.06339294494997275)), ('cz', 0, 1), ('u', 2, (1.7682397169090667, 0.5733692770586745, -0.9852974041236715)), ('u', 3, (-3.97598845872157, 3.0958077893047333, -1.2519709441529943)), ('u', 4, (-1.411415665160749, 0.18948186757998975, 3.842454179297896)), ('u', 0, (2.974852292104601, 0.2951590468832118, -2.848831168965828)), ('u', 1, (-2.5984107882735894, 1.583536392500949, 2.9333194961275066)), ('cz', 2, 3), ('u', 2, (1.4379397283244173, 0.5785609573479032, -1.2449756204344027)), ('u', 3, (5.386783270571561, 2.1862762409213947, 0.06473778843364775)), ('u', 4, (0.42563975135858423, 0.890449079408611, -0.3326320131496935)), ('u', 0, (-0.03863288970513888, 0.02728274949310999, 0.06517792955787628)), ('u', 1, (0.17723956207029218, 0.30310176174565606, 0.01594192134885969)), ('cz', 3, 4), ('u', 2, (0.3197897423922124, 0.4995194824989192, 0.4758324674415158)), ('u', 3, (-3.205585019209745, 2.5261128353466136, 1.521770157525988)), ('u', 4, (-3.35713121360027, 1.3501617031512694, 2.3897042561083683)), ('u', 0, (-2.643147556157265, 0.4110150774511253, 3.0323963518084405)), ('u', 1, (-0.27413403963193306, 0.3086820392296995, 0.11691029273789783)), ('cz', 2, 3), ('u', 2, (3.022587830332299, 0.02825914011524722, -3.111678660603214)), ('u', 3, (-1.5208572416193316, 2.356194503198517, -1.630758653677268)), ('u', 4, (2.1805623755906116, 2.134080734985709, 0.38855058949595767)), ('u', 0, (0.0012482574713538336, 0.464629504416325, -0.09528659826040922)), ('u', 1, (0.1930212265602076, 0.2140220406991674, -0.04131447494920497)), ('cz', 3, 4), ('u', 2, (0.18359247568409248, 2.3838301091939544, 0.04108671285385225)), ('u', 3, (-1.5106188129922442, 1.570796368572668, -0.529214806467051)), ('u', 4, (3.939929685673091, 1.1987744112516259, 0.3078061173187987)), ('u', 0, (-0.19845511870835986, 0.3374254387257764, -0.07409905552826768)), ('u', 1, (0.04373643240782611, 0.1505983632221452, -0.15910901580761577)), ('cz', 2, 3), ('u', 2, (3.5794731185322934, 3.141592585539292, -2.59671999775915)), ('u', 3, (-2.612377760845247, 1.570796273340851, 3.2159650695398354)), ('u', 4, (0.017736528180962233, 0.39065518985106523, -0.2283821708560611)), ('u', 0, (2.5103147656811147, 1.6338905350785855, -3.240530077745801)), ('u', 1, (0.19260207987938857, 0.6221358807800801, -0.3705347463748))]), (5, [('u', 1, (0.0, 1.9823131728623846, 0.0)), ('u', 0, (0.0, 1.1831996401397158, 0.0)), ('u', 0, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 0), ('u', 0, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (-3.141592653589793, 1.1831996401397158, 3.141592653589793)), ('u', 3, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 0, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (0.0, 1.5707963267948966, 0.0)), ('u', 1, (0.0, 1.5707963267948966, 0.0)), ('cz', 0, 1), ('u', 1, (1.2251903831398114e-16, 0.7853981633974483, -1.1496735851465472e-17)), ('u', 0, (-3.141592653589793, 0.7853981633974483, 3.141592653589793)), ('cz', 0, 1), ('u', 2, (1.5708446024674354, 0.615479708415582, -1.423395234744073)), ('u', 3, (-4.076420369540545, 3.141592626948408, -1.5120906372603562)), ('u', 4, (1.048234610153547, 0.35262501995127393, 0.5341844404437943)), ('u', 1, (3.13971064238359, 1.5549126675981364, -3.1198619513397956)), ('cz', 2, 3), ('u', 2, (1.7181966918454517, 0.4175592040251194, -1.129679736742519)), ('u', 3, (1.629501736812657, 1.5707963754323724, -1.9634953272520899)), ('u', 4, (0.5341847786752187, 1.395402396652681, -0.08565703082680046)), ('u', 1, (0.02173079729491832, 0.012249787592922284, 0.02192159707382869)), ('cz', 3, 4), ('u', 2, (0.36217946582783367, 0.24400971593032947, 0.21043539161929958)), ('u', 3, (-2.7488937843489896, 1.5707962363946044, 2.6639343464546186)), ('u', 4, (-0.035937284598891436, 0.22412384746199288, -0.08211648071987442)), ('u', 1, (0.02360274576274645, 0.010333005300155922, 0.018529438308069072)), ('cz', 2, 3), ('u', 2, (0.21043497530010674, 0.32477380231996045, 0.05986836762128027)), ('u', 3, (-0.47765830713519264, 1.5707963912970386, -0.7853981304146608)), ('u', 4, (3.0594760908471903, 1.1760553487634595, 1.0711803299696445)), ('u', 1, (0.018529854621483908, 0.003665687734053444, 0.00789648023365892)), ('cz', 3, 4), ('u', 2, (0.05986851767437061, 2.0343149862431438, -0.02696525028711255)), ('u', 3, (-0.7853981304146526, 1.570796374683024, -0.408473893033093)), ('u', 4, (4.212772983559444, 0.866754934231214, 0.34817009268817856)), ('u', 1, (-3.1336961733561233, 0.004371363852426555, 3.1328635209222155)), ('cz', 2, 3), ('u', 2, (3.1146274033026846, 0.7853981561867128, -2.936824902590321)), ('u', 3, (-0.40847389303309467, 1.5707963783916108, 2.981049246342238e-08)), ('u', 4, (0.3481700926881752, 0.34811323690933227, 0.2816367716829552)), ('u', 1, (-3.150321161808917, 0.01372663933785196, 3.1099203377723534)), ('cz', 2, 3), ('u', 2, (0.20476775099948474, 3.14159255591908, -5.638358777533247)), ('u', 3, (3.141592683400285, 1.5707963335529351, -2.906301630238328)), ('u', 4, (0.2816367716829386, 0.8086217290267099, 0.08239309705148327)), ('u', 1, (-3.1732650734801875, 0.024147437792669837, 3.080591167834259)), ('u', 0, (-3.141592653589793, 1.5707963267948966, 3.141592653589793))])], (6, 2): [(6, [('u', 5, (-2.453459795629976, 0.6154108398293135, 3.792701447413792)), ('u', 1, (3.0019054622941077, 0.5152502283591138, -0.033126937133005185)), ('u', 4, (0.2292433038815036, 0.01481289503683034, -1.0177080395201215)), ('u', 2, (0.013018042588220924, 2.1944221470493073, 0.32818112010404515)), ('u', 0, (-2.9787958806291632, 0.0886689973418528, 2.4614610365237706)), ('cz', 1, 2), ('u', 5, (3.3349764885375532, 0.0022967240122862384, -2.8964023772851433)), ('u', 1, (2.9589589148629383, 0.6176586630691273, -0.7274142970197928)), ('u', 4, (2.539613002099871, 1.570072339813219, -2.953840329876965)), ('u', 2, (-0.42430642488693754, 0.02300284793942631, -0.22225922963172645)), ('u', 0, (-0.5928730258605697, 0.13985223781419126, -0.06993066539964493)), ('cz', 4, 2), ('u', 5, (-3.139204532245132, 1.2536185284290928, 2.96666802332593)), ('u', 1, (-3.3075227590289553, 0.14398860885600887, 2.95586682816478)), ('u', 4, (2.780613807479522, 1.4554101551444294, 0.1051697514051142)), ('u', 2, (-2.7876697038606877, 0.1002101766399416, 3.1872120774420005)), ('u', 0, (-3.141880020564839, 0.20856433151600512, 2.704007495148254)), ('cz', 5, 1), ('u', 5, (3.6115223449033067, 1.5090964279556942, -0.6997219565492345)), ('u', 1, (0.0, 3.934340781158519e-13, -1.6892619956600294)), ('u', 4, (3.0937138685259247, 0.25794218728504037, -2.8904607651497876)), ('u', 2, (0.1469691841132235, 0.6052525332913032, -0.3091487519746785)), ('u', 0, (1.083382316755207, 0.7326507291357435, -1.340342122680559)), ('cz', 1, 0), ('u', 5, (-0.24588352327733107, 0.20669328721014976, -0.13976273443047652)), ('u', 1, (1.2596702877841461, 1.5707963267949496, -0.9671876089234042)), ('u', 4, (0.5205856127125582, 0.16359306049893949, -0.45270694994421545)), ('u', 2, (2.7668963848978025, 1.9655136715248875, 0.4670899607319061)), ('u', 0, (-1.49213980074785, 0.766068440906933, 0.8410618941063295)), ('cz', 1, 2), ('u', 5, (-0.03790008073856499, 0.24190180430634445, -0.45231215425760385)), ('u', 1, (-2.1744050446660097, 0.615479708670376, 3.586940755305892)), ('u', 4, (-0.33181423317831465, 0.24053639415852562, 0.1806657260712599)), ('u', 2, (3.394353501268674, 2.2503318610333842, 0.19837895625303315)), ('u', 0, (3.1183009959852743, 0.25342840054172383, -2.9027348844705507)), ('cz', 1, 0), ('u', 5, (0.1530580101984686, 0.07271622055816514, -0.5189376642463033)), ('u', 1, (-0.44534810171638234, 1.5707963267948928, 0.01191361420548298)), ('u', 4, (2.6531340607693896, 0.010161404945631879, -3.3598053328628263)), ('u', 2, (3.7296091757018006, 1.6160485227010506, -0.5714337244232652)), ('u', 0, (0.0, 1.0674903348085818e-13, 0.6006852000182925)), ('cz', 1, 2), ('u', 5, (-0.13358344378767256, 0.8542086362571331, -0.1881003560509148)), ('u', 1, (-3.1535062677953256, 1.570796326794941, 2.764678964026394)), ('u', 4, (1.4963787604077234, 0.0528843622355743, -0.9244454803613732)), ('u', 2, (3.761595471028183, 2.0372422896199693, -0.346298124793583)), ('u', 0, (0.0, 1.198704791825506e-13, 0.8572929198784051)), ('u', 3, (1.8883805791828714, 0.615479708670409, -0.6467094478986308)), ('cz', 3, 4), ('u', 3, (-1.9293815007118043, 0.6039066904739256, 1.4683043598780685)), ('u', 4, (1.8262065010289699, 1.57079632655862, -1.3475066836698089)), ('u', 5, (2.8630429597927805, 1.9992251818045819, 0.10434066042637791)), ('u', 2, (-2.579314349438724, 0.7452325091927046, 2.4866137061923834)), ('cz', 4, 5), ('u', 3, (3.0450181662748643, 0.32108853740524523, -2.9988091323485935)), ('u', 4, (2.918303010546812, 1.5707963267026572, -3.2308762283953705)), ('u', 5, (2.966749132139194, 0.9925184113293671, -0.1258080208303063)), ('u', 2, (-2.5999248213601502, 0.34978155544370026, 2.5401908507714905)), ('cz', 3, 4), ('u', 3, (-0.0530316095262567, 0.23572757035344816, 0.21812206059616673)), ('u', 4, (-0.8660330433189241, 1.5707963264492601, -0.6042175386464284)), ('u', 5, (3.1979447430427776, 1.9455319197599266, 0.1770682281920064)), ('u', 2, (0.16813662917413538, 0.4690709141383835, 0.4382762374570831)), ('cz', 4, 5), ('u', 3, (-0.5633511319579381, 1.3485100752261905, -0.4529163910462173)), ('u', 4, (0.604217538579338, 2.345214810300157, 0.2970227717171343)), ('u', 5, (2.8279125566361265, 1.7856160657121039, 0.23433833243729563)), ('u', 2, (3.0204092877430533, 0.8396472289164091, -2.7838510527484246)), ('cz', 3, 4), ('u', 3, (2.8882297471232, 1.570796326763543, -2.148206467032401)), ('u', 4, (-0.29702277190250914, 0.7853981633974549, 0.2693558231996239)), ('u', 5, (2.576228776662998, 0.18867689737221688, -3.0086311766791254)), ('u', 2, (-0.5736763156694709, 0.2827881667245027, 0.5029686019782795)), ('cz', 3, 4), ('u', 3, (-2.5641825131640075, 1.570796326931336, 2.832019659879205)), ('u', 4, (-3.410948476535545, 1.570796326772328, 2.8320196598791765)), ('u', 5, (3.212369746983524, 0.03635589774106469, -2.535249559059127)), ('u', 2, (-0.8989665873249468, 0.7843530048892474, -0.013499265403156602)), ('u', 0, (0.0, 1.5707963267948966, 0.0)), ('u', 1, (0.0, 1.5707963267948966, 0.0)), ('cz', 0, 1), ('u', 1, (1.2251903831398114e-16, 0.7853981633974483, -1.1496735851465472e-17)), ('u', 0, (-3.141592653589793, 0.7853981633974483, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 0, (-3.141592653589793, 1.5707963267948966, 3.141592653589793))]), (6, [('u', 2, (0.0, 2.214297435588181, 0.0)), ('u', 1, (0.0, 1.0471975511965976, 0.0)), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 1), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.0471975511965976, 3.141592653589793)), ('u', 4, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 5, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 1, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 0, 1), ('u', 1, (1.570796326794898, 0.6154797086703868, -1.570796326794898)), ('u', 0, (1.5707963267948966, 0.6154797086703876, -1.5707963267948966)), ('cz', 0, 1), ('u', 1, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 0, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 1, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 0.9553166181245093, 3.141592653589793)), ('cz', 2, 1), ('u', 1, (0.0, 0.9553166181245093, 0.0)), ('u', 1, (-3.141592653589793, 2.5261129449194057, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 2.5261129449194057, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 3, (1.8883805791828714, 0.615479708670409, -0.6467094478986308)), ('u', 4, (1.7118826188473402, 4.726163444567278e-10, -0.883301315865618)), ('u', 5, (-0.17120323373106677, 0.452290667529392, -0.0696741876668316)), ('u', 2, (4.017129896144521, 1.756817436765106, -0.07916448927145558)), ('cz', 3, 4), ('u', 3, (-1.9293815007118043, 0.6039066904739256, 1.4683043598780685)), ('u', 4, (1.8262065010289699, 1.57079632655862, -1.3475066836698089)), ('u', 5, (2.8630429597927805, 1.9992251818045819, 0.10434066042637791)), ('u', 2, (-2.579314349438724, 0.7452325091927046, 2.4866137061923834)), ('cz', 4, 5), ('u', 3, (3.0450181662748643, 0.32108853740524523, -2.9988091323485935)), ('u', 4, (2.918303010546812, 1.5707963267026572, -3.2308762283953705)), ('u', 5, (2.966749132139194, 0.9925184113293671, -0.1258080208303063)), ('u', 2, (-2.5999248213601502, 0.34978155544370026, 2.5401908507714905)), ('cz', 3, 4), ('u', 3, (-0.0530316095262567, 0.23572757035344816, 0.21812206059616673)), ('u', 4, (-0.8660330433189241, 1.5707963264492601, -0.6042175386464284)), ('u', 5, (3.1979447430427776, 1.9455319197599266, 0.1770682281920064)), ('u', 2, (0.16813662917413538, 0.4690709141383835, 0.4382762374570831)), ('cz', 4, 5), ('u', 3, (-0.5633511319579381, 1.3485100752261905, -0.4529163910462173)), ('u', 4, (0.604217538579338, 2.345214810300157, 0.2970227717171343)), ('u', 5, (2.8279125566361265, 1.7856160657121039, 0.23433833243729563)), ('u', 2, (3.0204092877430533, 0.8396472289164091, -2.7838510527484246)), ('cz', 3, 4), ('u', 3, (2.8882297471232, 1.570796326763543, -2.148206467032401)), ('u', 4, (-0.29702277190250914, 0.7853981633974549, 0.2693558231996239)), ('u', 5, (2.576228776662998, 0.18867689737221688, -3.0086311766791254)), ('u', 2, (-0.5736763156694709, 0.2827881667245027, 0.5029686019782795)), ('cz', 3, 4), ('u', 3, (-2.5641825131640075, 1.570796326931336, 2.832019659879205)), ('u', 4, (-3.410948476535545, 1.570796326772328, 2.8320196598791765)), ('u', 5, (3.212369746983524, 0.03635589774106469, -2.535249559059127)), ('u', 2, (-0.8989665873249468, 0.7843530048892474, -0.013499265403156602)), ('u', 0, (0.0, 1.5707963267948966, 0.0)), ('u', 1, (0.0, 1.5707963267948966, 0.0)), ('cz', 0, 1), ('u', 1, (1.2251903831398114e-16, 0.7853981633974483, -1.1496735851465472e-17)), ('u', 0, (-3.141592653589793, 0.7853981633974483, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 0, (-3.141592653589793, 1.5707963267948966, 3.141592653589793))])], (6, 3): [(6, [('u', 2, (0.0, 2.690565841793531, 0.0)), ('u', 1, (0.0, 0.7590702092666635, 0.0)), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 1), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 0.7590702092666635, 3.141592653589793)), ('u', 0, (0.0, 1.2490457723982544, 0.0)), ('u', 0, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 0), ('u', 0, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (-3.141592653589793, 1.2490457723982544, 3.141592653589793)), ('u', 3, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 4), ('u', 2, (2.572576065009661, 0.0736751545868733, -3.6612768177774417)), ('u', 1, (-3.0957378634480905, 0.017542113885202545, 3.1693247411303176)), ('u', 0, (0.2924246944778046, 2.0144049022332478e-09, 0.22401953854093656)), ('u', 5, (3.141621456214928, 1.570796330465726, -3.09805213094587)), ('cz', 0, 5), ('u', 2, (2.621908489402255, 0.03193775843880457, -3.509769218800873)), ('u', 1, (-1.5430642242387382, 1.57176882783286, 1.5620387653890344)), ('u', 0, (1.79481557405754, 1.5707963063843675, -1.5707962946296408)), ('u', 5, (3.1851331934529603, 1.7388192561033189, 0.1445328327997646)), ('cz', 1, 0), ('u', 2, (2.773416114790225, 0.011712512915930865, -3.487634647530157)), ('u', 1, (-1.5795538882007654, 0.3508795374768818, 2.356194469389992)), ('u', 0, (1.570796358960163, 0.9388818690097547, -2.0807717245217927)), ('u', 5, (0.14453277891283747, 0.17781580492317572, 0.10044140220025762)), ('cz', 1, 0), ('u', 2, (2.795550619992101, 1.6149998495382067, 0.038072395040432916)), ('u', 1, (2.35619446939, 1.5707963346497302, -1.570796372880089)), ('u', 0, (-2.080771724521777, 1.6422561796537511, 1.5129388782153543)), ('u', 5, (0.10044150750248625, 0.1809763328933769, 0.06649110164234236)), ('cz', 2, 1), ('u', 2, (3.1796650317276396, 1.65869340820513, 0.034550296368021094)), ('u', 1, (-3.141592654236178, 0.6621452985233158, 3.141592652492418)), ('u', 0, (3.083735263681261, 0.5549864772909748, -3.1041332876626053)), ('u', 5, (0.0664911016423407, 0.18100721434933958, 0.03954869590947857)), ('cz', 1, 0), ('u', 2, (0.034550296368022856, 0.12533726647302446, -0.014375091750484691)), ('u', 1, (-1.0973828975781453e-09, 1.5707963231684132, 3.53287090559963e-08)), ('u', 0, (3.1790520195169822, 0.17270729538097135, -3.082543531819984)), ('u', 5, (0.03954869590948736, 0.17952451149012152, 0.017037161255502448)), ('cz', 2, 1), ('u', 2, (-0.014375091750492976, 0.009768469000364495, -0.020839153120665917)), ('u', 1, (-3.1415926182610865, 0.631914291686104, 3.141592623127217)), ('u', 0, (3.200641775359596, 0.1787908520679271, -3.0619521477417697)), ('u', 5, (0.017037161255500654, 0.17703034470092435, -0.002895412580624177)), ('cz', 1, 0), ('u', 2, (3.120753500469131, 1.5805564419183182, -0.0003483929136263164)), ('u', 1, (-3.0462575746354565e-08, 1.5707962186264584, -1.5802167536612645e-08)), ('u', 0, (-3.06195214774176, 0.49600372218524175, 3.07762352867769)), ('u', 5, (-0.0028954125806292964, 0.17349234963220875, -0.02153413067381888)), ('cz', 2, 1), ('u', 2, (3.1412442606761646, 1.5776831967573992, -0.021025722803725166)), ('u', 1, (-1.5802168466866894e-08, 2.094395037081264, -3.3851818561640647e-08)), ('u', 0, (-0.06396912491211032, 1.012964249330451, -0.06087199156588615)), ('u', 5, (-0.021534130673809944, 0.1686473636293627, -0.039769651430084436)), ('cz', 1, 0), ('u', 2, (-3.162618376393516, 0.006892905953000969, 3.0823156099915896)), ('u', 1, (-3.385181899379086e-08, 1.570796175154427, -0.1013274575995535)), ('u', 0, (-3.202464645155681, 1.5707962322375566, 3.040265178083402)), ('u', 5, (-0.039769651430092846, 0.16213869678448167, -0.05820906109483628)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 4, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 3, 4), ('u', 4, (1.570796326794898, 0.6154797086703868, -1.570796326794898)), ('u', 3, (1.5707963267948966, 0.6154797086703876, -1.5707963267948966)), ('cz', 3, 4), ('u', 4, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 3, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 4, (0.0, 0.0, 1.3951473992034527e-15)), ('cz', 4, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (-3.141592653589793, 1.2630564724597029, 3.141592653589793)), ('cz', 3, 4), ('u', 4, (0.0, 0.30773985433519363, 0.0)), ('cz', 5, 4), ('u', 4, (-3.141592653589793, 0.30773985433519363, 3.141592653589793)), ('cz', 3, 4), ('u', 4, (0.0, 1.2630564724597029, 0.0)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (0.0, 1.5707963267948966, 0.0)), ('u', 4, (0.0, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (1.2251903831398114e-16, 0.7853981633974483, -1.1496735851465472e-17)), ('u', 3, (-3.141592653589793, 0.7853981633974483, 3.141592653589793)), ('cz', 3, 4), ('u', 4, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 3, (-3.141592653589793, 1.5707963267948966, 3.141592653589793))])], (7, 2): [(7, [('u', 2, (0.0, 2.0137073708685356, 0.0)), ('u', 1, (0.0, 1.1071487177940904, 0.0)), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 1), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.1071487177940904, 3.141592653589793)), ('u', 5, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 6, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 6), ('u', 3, (0.11601578574459524, 0.11445306532186036, -0.05873314765785108)), ('u', 4, (-2.5229814239155903, 0.7853981633929247, 2.672654067515391)), ('u', 5, (0.0, 3.141592653589684, -0.3445112607582286)), ('u', 6, (3.3781982997579023, 1.9938023267735092, -0.07359164259788864)), ('cz', 4, 5), ('u', 3, (-2.9515962975765566, 0.5056755477723608, 2.2200443890006474)), ('u', 4, (3.2149773257678795, 1.1992828892140965, -2.9928042977979965)), ('u', 5, (3.644958195962718, 1.5707963267948784, -2.5163472193680594)), ('u', 6, (3.0487820351105013, 1.9514455601263996, -0.8156097462450673)), ('cz', 5, 6), ('u', 3, (-0.34283879299373743, 0.044686926957994734, 0.07892007325733039)), ('u', 4, (-0.7221058691384219, 0.5262637442364827, -0.5551588226138113)), ('u', 5, (-2.196041761016409, 1.5707963267946825, 2.623777901877183)), ('u', 6, (2.92253223232039, 2.4036930191468513, -0.20277826325117143)), ('cz', 4, 5), ('u', 3, (-0.6305368457106076, 0.7980329266973532, -0.7336226221434942)), ('u', 4, (-0.5167182473551505, 0.18315113902045307, -0.23668899703635093)), ('u', 5, (-0.5293827994841317, 1.5707963267949543, -0.5798174889070856)), ('u', 6, (2.725010306136793, 1.507659020777457, 0.5282031131704192)), ('cz', 5, 6), ('u', 3, (0.6727279757044259, 0.9274976539154787, -1.6936955613998808)), ('u', 4, (-2.1475182100310173, 1.4789871430681827, 1.6303559354483252)), ('u', 5, (-0.2648048008870131, 0.5314794575963966, -0.27660674173470357)), ('u', 6, (3.038008047238889, 1.8625898384981554, -0.5902471651918946)), ('cz', 3, 4), ('u', 3, (1.3756922625147034, 0.6154797086710831, -1.2971086488369805)), ('u', 4, (1.6399186564016064, 0.6499151492340337, -1.919730869296622)), ('u', 5, (-0.1671310696782647, 0.45729015600664225, -0.7313084035710773)), ('u', 6, (2.3024305502496483, 0.05323677740319882, -3.7734166048553166)), ('cz', 3, 4), ('u', 3, (-1.3014447755735246, 1.678142726791292, 2.8032854998997374)), ('u', 4, (1.8150173940969985, 0.6193795682353069, -2.3039289911090957)), ('u', 5, (3.386502475533139, 2.3788032598392967, 0.6418405556062463)), ('u', 6, (0.2903706668900086, 0.2084988360844264, 0.1824124091269326)), ('cz', 4, 5), ('u', 3, (0.21109755371946107, 0.2156690401268879, 0.18913462830811542)), ('u', 4, (2.432294277063316, 0.9553166181254876, -3.76355827912454)), ('u', 5, (3.7507986984912347, 1.570796326794571, -0.3571488549968511)), ('u', 6, (2.9777689850569438, 0.20221729840515162, -2.589186588281091)), ('cz', 4, 5), ('u', 1, (1.5324194940067908, 5.3021376566375703e-08, -1.56266077741119)), ('u', 2, (0.6018106158184183, 0.15980498277650393, -0.6772602058207539)), ('u', 5, (3.017679557250371, 1.3299074850227444, 1.2846107161235554)), ('u', 0, (1.6467528317913072, 0.6154796932346827, -0.6398573753162243)), ('cz', 1, 0), ('u', 1, (3.2155180795844163, 1.5707963354517513, -2.1087019206789432)), ('u', 2, (3.254949555276662, 1.7061877643292427, 0.03301095216222327)), ('u', 5, (0.10046169634961145, 0.6469997841069166, 0.12375403821524343)), ('u', 0, (-1.3225583272613946, 1.174054518294708, 1.1485285047746567)), ('cz', 1, 2), ('u', 1, (3.6794982122158824, 1.5707963588827134, -2.2819746843423734)), ('u', 2, (4.0086681688709564, 0.8720561710448409, -0.4466142483244484)), ('u', 5, (-2.8822710281041393, 0.255541143352538, 3.134271415481891)), ('u', 0, (2.6671680637740542, 1.0254009757252611, -2.1493071334045513)), ('cz', 1, 0), ('u', 1, (-1.8149345872017317, 1.5707963580972752, -1.9944583751605043)), ('u', 2, (2.8394887155613904, 1.5449097868074715, 0.12236943438758852)), ('u', 5, (2.985036412643358, 0.32468604606484874, -3.0385823672053704)), ('u', 0, (-2.4222679250537213, 0.6530276918702874, 3.6461799725817956)), ('cz', 1, 2), ('u', 1, (0.42366197447181075, 1.570796335553448, 0.34961258600754963)), ('u', 2, (2.2835768926992426, 1.8663010908405264, -0.31629292507938644)), ('u', 5, (-0.37412930654170795, 0.4719137211966899, -0.07265702258492518)), ('u', 0, (0.0883085696115993, 1.285424226825953, 0.4271782882932581)), ('cz', 1, 0), ('u', 1, (0.8988429114876053, 1.5707963445850344, -0.3475797548284105)), ('u', 2, (-0.14751240629459822, 0.22395525895596402, 0.5253665284084357)), ('u', 5, (2.763039648099014, 0.12247638867714401, -3.039418776964069)), ('u', 0, (-3.0679695671358878, 0.7853981652638563, 2.8806735599399267)), ('cz', 1, 0), ('u', 1, (-2.794012910794895, 1.57079632381052, 1.849453006574815)), ('u', 2, (2.8025555652395773, 0.516012291944641, -2.9035700835849583)), ('u', 5, (0.48051934449338873, 0.06465831827926592, 0.05791233663936998)), ('u', 0, (-1.099696410105327, 7.545066320794333e-09, 0.06847583918922551)), ('u', 3, (1.366996796440895, 2.101372532601, 0.1847370593476193)), ('u', 4, (-1.0222896891335609, 1.5707963267948746, 1.3277776916500403)), ('u', 6, (-2.995036775581398, 0.17876252601705145, 2.971728660123455)), ('cz', 3, 4), ('u', 3, (-3.0813612244726043, 1.5707963267946186, 2.1469433645606504)), ('u', 4, (0.2430186351454288, 0.7853981633960306, -0.2004559077141073)), ('u', 6, (0.24137550723464346, 0.20561324850047275, 0.5592502294868626)), ('cz', 3, 4), ('u', 3, (2.5654456158246015, 1.5707963267949054, -3.1866241037341707)), ('u', 4, (-2.941136745875393, 1.5707963267949157, 3.0965612032655714)), ('u', 6, (-2.760376582028562, 0.30509340548022884, 3.3325566018123283))]), (7, [('u', 2, (0.0, 2.0137073708685356, 0.0)), ('u', 1, (0.0, 1.1071487177940904, 0.0)), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 1), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.1071487177940904, 3.141592653589793)), ('u', 5, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 6, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 6), ('u', 3, (0.11601578574459524, 0.11445306532186036, -0.05873314765785108)), ('u', 4, (-2.5229814239155903, 0.7853981633929247, 2.672654067515391)), ('u', 5, (0.0, 3.141592653589684, -0.3445112607582286)), ('u', 6, (3.3781982997579023, 1.9938023267735092, -0.07359164259788864)), ('cz', 4, 5), ('u', 3, (-2.9515962975765566, 0.5056755477723608, 2.2200443890006474)), ('u', 4, (3.2149773257678795, 1.1992828892140965, -2.9928042977979965)), ('u', 5, (3.644958195962718, 1.5707963267948784, -2.5163472193680594)), ('u', 6, (3.0487820351105013, 1.9514455601263996, -0.8156097462450673)), ('cz', 5, 6), ('u', 3, (-0.34283879299373743, 0.044686926957994734, 0.07892007325733039)), ('u', 4, (-0.7221058691384219, 0.5262637442364827, -0.5551588226138113)), ('u', 5, (-2.196041761016409, 1.5707963267946825, 2.623777901877183)), ('u', 6, (2.92253223232039, 2.4036930191468513, -0.20277826325117143)), ('cz', 4, 5), ('u', 3, (-0.6305368457106076, 0.7980329266973532, -0.7336226221434942)), ('u', 4, (-0.5167182473551505, 0.18315113902045307, -0.23668899703635093)), ('u', 5, (-0.5293827994841317, 1.5707963267949543, -0.5798174889070856)), ('u', 6, (2.725010306136793, 1.507659020777457, 0.5282031131704192)), ('cz', 5, 6), ('u', 3, (0.6727279757044259, 0.9274976539154787, -1.6936955613998808)), ('u', 4, (-2.1475182100310173, 1.4789871430681827, 1.6303559354483252)), ('u', 5, (-0.2648048008870131, 0.5314794575963966, -0.27660674173470357)), ('u', 6, (3.038008047238889, 1.8625898384981554, -0.5902471651918946)), ('cz', 3, 4), ('u', 3, (1.3756922625147034, 0.6154797086710831, -1.2971086488369805)), ('u', 4, (1.6399186564016064, 0.6499151492340337, -1.919730869296622)), ('u', 5, (-0.1671310696782647, 0.45729015600664225, -0.7313084035710773)), ('u', 6, (2.3024305502496483, 0.05323677740319882, -3.7734166048553166)), ('cz', 3, 4), ('u', 3, (-1.3014447755735246, 1.678142726791292, 2.8032854998997374)), ('u', 4, (1.8150173940969985, 0.6193795682353069, -2.3039289911090957)), ('u', 5, (3.386502475533139, 2.3788032598392967, 0.6418405556062463)), ('u', 6, (0.2903706668900086, 0.2084988360844264, 0.1824124091269326)), ('cz', 4, 5), ('u', 3, (0.21109755371946107, 0.2156690401268879, 0.18913462830811542)), ('u', 4, (2.432294277063316, 0.9553166181254876, -3.76355827912454)), ('u', 5, (3.7507986984912347, 1.570796326794571, -0.3571488549968511)), ('u', 6, (2.9777689850569438, 0.20221729840515162, -2.589186588281091)), ('cz', 4, 5), ('u', 3, (1.366996796440895, 2.101372532601, 0.1847370593476193)), ('u', 4, (-1.0222896891335609, 1.5707963267948746, 1.3277776916500403)), ('u', 5, (3.405589725198712, 1.5476978984160668, 0.26813170011961907)), ('u', 6, (-2.995036775581398, 0.17876252601705145, 2.971728660123455)), ('cz', 3, 4), ('u', 3, (-3.0813612244726043, 1.5707963267946186, 2.1469433645606504)), ('u', 4, (0.2430186351454288, 0.7853981633960306, -0.2004559077141073)), ('u', 5, (-2.6874617869140525, 0.004774720761996742, 3.000213729348172)), ('u', 6, (0.24137550723464346, 0.20561324850047275, 0.5592502294868626)), ('cz', 3, 4), ('u', 3, (2.5654456158246015, 1.5707963267949054, -3.1866241037341707)), ('u', 4, (-2.941136745875393, 1.5707963267949157, 3.0965612032655714)), ('u', 5, (0.7204058193865766, 0.09992579049839889, 0.1522237206849802)), ('u', 6, (-2.760376582028562, 0.30509340548022884, 3.3325566018123283)), ('u', 0, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 1, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 0, 1), ('u', 1, (1.570796326794898, 0.6154797086703868, -1.570796326794898)), ('u', 0, (1.5707963267948966, 0.6154797086703876, -1.5707963267948966)), ('cz', 0, 1), ('u', 1, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 0, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 1, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 0.9553166181245093, 3.141592653589793)), ('cz', 2, 1), ('u', 1, (0.0, 0.9553166181245093, 0.0)), ('u', 1, (-3.141592653589793, 2.5261129449194057, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 2.5261129449194057, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (0.0, 1.5707963267948966, 0.0)), ('u', 1, (0.0, 1.5707963267948966, 0.0)), ('cz', 0, 1), ('u', 1, (1.2251903831398114e-16, 0.7853981633974483, -1.1496735851465472e-17)), ('u', 0, (-3.141592653589793, 0.7853981633974483, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 0, (-3.141592653589793, 1.5707963267948966, 3.141592653589793))])], (7, 3): [(7, [('u', 2, (0.0, 2.451879499619181, 0.0)), ('u', 1, (0.0, 0.866397140683011, 0.0)), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 1), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 0.866397140683011, 3.141592653589793)), ('u', 0, (0.0, 1.2897614252920828, 0.0)), ('u', 0, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 0), ('u', 0, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (-3.141592653589793, 1.2897614252920828, 3.141592653589793)), ('u', 4, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 5, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 6, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 0, 6), ('u', 2, (2.927990914951556, 0.037608684936839026, -2.9067897925621486)), ('u', 1, (0.18682830662460814, 1.570796328805722, 1.5459630175254293)), ('u', 0, (1.82281298683628, 1.570796352722352, -1.4226728678028702)), ('cz', 1, 0), ('u', 2, (0.4418700531653876, 0.22288047860758717, 0.0668089225595277)), ('u', 1, (1.5956295571676744, 0.35087940812151497, -0.5144426080850821)), ('u', 0, (1.4226730203959437, 2.202710638202488, -0.8958445494596178)), ('cz', 1, 0), ('u', 2, (2.8604488232479293, 1.7550979759013636, 0.06030027122532955)), ('u', 1, (2.0852391456306374, 1.5707963250304045, -1.5792191742993054)), ('u', 0, (-1.1924972438880368, 1.427759669397298, 1.2476289875914057)), ('cz', 2, 1), ('u', 2, (3.005610498223304, 1.5164310130476222, 0.09862978373833009)), ('u', 1, (3.150015490793008, 2.4794474210737114, -2.77230333335792)), ('u', 0, (0.07782077051936362, 0.5352106269078157, 0.5416462673949637)), ('cz', 1, 0), ('u', 2, (-0.05524356225504093, 0.2675497095603755, 0.20359478394100908)), ('u', 1, (-0.36928932655915003, 1.5707963636983906, -0.25635923965612184)), ('u', 0, (-1.9930155820684545, 0.2366494479958406, 3.5462227821027197)), ('cz', 2, 1), ('u', 2, (0.1197300595996944, 0.04626078208212085, 0.38579804536576356)), ('u', 1, (-2.885233482113633, 0.6319143150580288, 2.5803068225615586)), ('u', 0, (0.711762447438835, 0.3870087056939558, -0.7547733850526132)), ('cz', 1, 0), ('u', 2, (2.387786870978032, 1.6139637451898206, -0.08929192193052904)), ('u', 1, (0.5612858904313562, 1.570796346954555, -0.19916906104258825)), ('u', 0, (2.488438267223762, 0.6013013961678155, -2.773850216397627)), ('cz', 2, 1), ('u', 2, (3.023045408509128, 1.9646733061472654, -0.05500547276974377)), ('u', 1, (0.19916905562690662, 2.0943951032485066, 0.2126552942333672)), ('u', 0, (0.614458438682712, 0.5568168189701632, -1.1003495014264453)), ('cz', 1, 0), ('u', 2, (2.7294533248225488, 0.4356400138815296, -2.964725961166434)), ('u', 1, (-3.354247960909918, 1.5707962849175021, 2.8895760244168196)), ('u', 0, (2.3260298735930833, 1.5707964715903298, -3.393609304055196)), ('u', 6, (1.7096019472539603, 0.8268800259858878, 0.6226969142284112)), ('u', 5, (0.6775044359672656, 1.5707963248723165, 0.2814876547916655)), ('u', 4, (2.313489117220361, 1.5707963275148005, -2.472571387807654)), ('u', 3, (0.07712786118430925, 3.134153302552414, 1.7620614073562337)), ('cz', 5, 4), ('u', 6, (2.4164750875067043, 1.649752765635693, 0.43070363091364383)), ('u', 5, (2.8601049935471403, 0.7853981633417665, 1.2894281993906118)), ('u', 4, (-2.372974730053756, 0.8644842060716954, 2.125849068145324)), ('u', 3, (-2.0270749703059034, 0.8982487151767372, -0.3821936977835688)), ('cz', 6, 5), ('u', 6, (3.901448919233749, 2.288021044208705, 0.2504639141045335)), ('u', 5, (4.635944698043758, 2.0767669458294863e-10, -0.9985038795557417)), ('u', 4, (-2.9998415836464183, 0.3979549002656172, 1.8593632581251363)), ('u', 3, (-2.6039287213865485, 0.9180809613579872, 2.60890251925452)), ('cz', 5, 4), ('u', 6, (2.593988511809022, 0.23513988322678683, -2.5365709795549267)), ('u', 5, (1.3563162800358772, 1.5707963265867255, -0.1824074250789215)), ('u', 4, (1.2917180113702056, 1.5707963263431968, -1.7516479641056701)), ('u', 3, (-0.6476283685074675, 0.7529085504892408, 0.718844508690945)), ('cz', 5, 4), ('u', 6, (2.5871461198581907, 1.0323099097198227, 0.02409387642085714)), ('u', 5, (-0.7724120834509618, 1.0459822339013305, 0.2834198793904126)), ('u', 4, (-3.086202990435042, 0.784303810403754, 2.877674174726899)), ('u', 3, (-1.0601937628728253, 0.7801694791108086, -0.8228006946101142)), ('cz', 6, 5), ('u', 6, (3.0559736873781795, 1.3347778860466655, -0.31702683437110935)), ('u', 5, (0.2615161709271716, 0.48384369269099137, 1.0627992426627673)), ('u', 4, (-0.046050816455809995, 2.3306961543571387, 1.181660419129296)), ('u', 3, (1.5700812638871322, 1.7664317780705903, -1.0534565184505182)), ('cz', 4, 3), ('u', 6, (2.466756499952858, 0.8138789408014773, -3.391253695161958)), ('u', 5, (2.8241488022799595, 1.9393207387956741, -0.2589588097378437)), ('u', 4, (1.7427773661823036, 1.5707963268846783, -0.7773601156092436)), ('u', 3, (-0.98774642550761, 1.5679244775504066, 3.197838646334241)), ('cz', 5, 4), ('u', 6, (2.3718838348624693, 0.1610554902863146, -3.1451609260925926)), ('u', 5, (3.1812438859274077, 1.2956388035229423, -0.9363594158902351)), ('u', 4, (-2.3642325351621944, 1.5707963268307001, 3.708296298777864)), ('u', 3, (-1.6583297901709708, 0.09154773214686243, 3.9445860178747347)), ('cz', 4, 3), ('u', 6, (2.0833912437352335, 0.4052932148735028, -4.128846981673444)), ('u', 5, (-0.5306514304602161, 1.2190942552290325, -0.5988795404597256)), ('u', 4, (-0.5667036454713823, 0.9553166181638688, 0.9868405769569011)), ('u', 3, (1.3477092368498658, 0.9888417142673291, 0.1453391073528154)), ('cz', 5, 4), ('u', 6, (-1.9734238059199634, 0.9210446919760125, 3.2753461886166804)), ('u', 5, (-0.7250272566644168, 0.13951695534942818, -0.966388415505158)), ('u', 4, (-3.1448769066377587, 4.806953846162392e-11, 3.1151898621906)), ('u', 3, (0.26855113730749347, 0.6231306260987792, -1.6117960646084089)), ('cz', 4, 3), ('u', 6, (3.3847826758333106, 0.00596977314801572, -2.8428600224957288)), ('u', 5, (2.1293157236538565, 0.7214702929695543, -0.14006408794000524)), ('u', 4, (0.6136427935159456, 1.570796326733598, -0.8101638400353446)), ('u', 3, (-2.5464923216353004, 0.8906782349719334, 3.3661107025646615)), ('cz', 5, 4), ('u', 6, (-2.818715732777364, 0.952878782096676, 3.0043417437514988)), ('u', 5, (0.7409371417897517, 2.9556950151821755, -2.2286588731330164)), ('u', 4, (-2.3314288130784124, 1.5707963269612122, -0.7960277889820586)), ('u', 3, (0.6793869157364618, 2.2236498925058483, 0.7190561537490348)), ('cz', 4, 3), ('u', 6, (3.031965526893565, 1.3094348861348473, -1.7242464394092538)), ('u', 5, (-0.12184148553446827, 1.439333407376042, -0.7002989429508656)), ('u', 4, (-0.7747685409843722, 1.5707963280360524, -2.1392781173878195)), ('u', 3, (-0.7249110800378891, 3.141592651784004, -4.3731490908327375))]), (7, [('u', 2, (0.0, 2.451879499619181, 0.0)), ('u', 1, (0.0, 0.866397140683011, 0.0)), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 1), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 0.866397140683011, 3.141592653589793)), ('u', 0, (0.0, 1.2897614252920828, 0.0)), ('u', 0, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 0), ('u', 0, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (-3.141592653589793, 1.2897614252920828, 3.141592653589793)), ('u', 4, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 5, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 6, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 0, 6), ('u', 6, (1.7096019472539603, 0.8268800259858878, 0.6226969142284112)), ('u', 5, (0.6775044359672656, 1.5707963248723165, 0.2814876547916655)), ('u', 4, (2.313489117220361, 1.5707963275148005, -2.472571387807654)), ('u', 1, (0.08830390508230578, 0.002530548629232911, 0.08153420123903503)), ('u', 3, (0.07712786118430925, 3.134153302552414, 1.7620614073562337)), ('cz', 5, 4), ('u', 6, (2.4164750875067043, 1.649752765635693, 0.43070363091364383)), ('u', 5, (2.8601049935471403, 0.7853981633417665, 1.2894281993906118)), ('u', 4, (-2.372974730053756, 0.8644842060716954, 2.125849068145324)), ('u', 1, (-3.0600584523507566, 0.004890692384524863, 3.216147527607153)), ('u', 3, (-2.0270749703059034, 0.8982487151767372, -0.3821936977835688)), ('cz', 6, 5), ('u', 6, (3.901448919233749, 2.288021044208705, 0.2504639141045335)), ('u', 5, (4.635944698043758, 2.0767669458294863e-10, -0.9985038795557417)), ('u', 4, (-2.9998415836464183, 0.3979549002656172, 1.8593632581251363)), ('u', 1, (-3.067037779572436, 0.010194790493636361, 3.2094698535851096)), ('u', 3, (-2.6039287213865485, 0.9180809613579872, 2.60890251925452)), ('cz', 5, 4), ('u', 6, (2.593988511809022, 0.23513988322678683, -2.5365709795549267)), ('u', 5, (1.3563162800358772, 1.5707963265867255, -0.1824074250789215)), ('u', 4, (1.2917180113702056, 1.5707963263431968, -1.7516479641056701)), ('u', 1, (-3.0737154535944806, 0.013494611910459637, 3.2034970868043264)), ('u', 3, (-0.6476283685074675, 0.7529085504892408, 0.718844508690945)), ('cz', 5, 4), ('u', 6, (2.5871461198581907, 1.0323099097198227, 0.02409387642085714)), ('u', 5, (-0.7724120834509618, 1.0459822339013305, 0.2834198793904126)), ('u', 4, (-3.086202990435042, 0.784303810403754, 2.877674174726899)), ('u', 1, (-3.079688220375256, 0.014919767336819259, 3.198529688640101)), ('u', 3, (-1.0601937628728253, 0.7801694791108086, -0.8228006946101142)), ('cz', 6, 5), ('u', 6, (3.0559736873781795, 1.3347778860466655, -0.31702683437110935)), ('u', 5, (0.2615161709271716, 0.48384369269099137, 1.0627992426627673)), ('u', 4, (-0.046050816455809995, 2.3306961543571387, 1.181660419129296)), ('u', 1, (-3.0846556185394896, 0.014604408480013018, 3.1947711245854737)), ('u', 3, (1.5700812638871322, 1.7664317780705903, -1.0534565184505182)), ('cz', 4, 3), ('u', 6, (2.466756499952858, 0.8138789408014773, -3.391253695161958)), ('u', 5, (2.8241488022799595, 1.9393207387956741, -0.2589588097378437)), ('u', 4, (1.7427773661823036, 1.5707963268846783, -0.7773601156092436)), ('u', 1, (-3.0884141825941067, 0.012684299018115628, 3.1923346473525807)), ('u', 3, (-0.98774642550761, 1.5679244775504066, 3.197838646334241)), ('cz', 5, 4), ('u', 6, (2.3718838348624693, 0.1610554902863146, -3.1451609260925926)), ('u', 5, (3.1812438859274077, 1.2956388035229423, -0.9363594158902351)), ('u', 4, (-2.3642325351621944, 1.5707963268307001, 3.708296298777864)), ('u', 1, (-3.0908506598270087, 0.009299651604358205, 3.1912508017852623)), ('u', 3, (-1.6583297901709708, 0.09154773214686243, 3.9445860178747347)), ('cz', 4, 3), ('u', 6, (2.0833912437352335, 0.4052932148735028, -4.128846981673444)), ('u', 5, (-0.5306514304602161, 1.2190942552290325, -0.5988795404597256)), ('u', 4, (-0.5667036454713823, 0.9553166181638688, 0.9868405769569011)), ('u', 1, (3.1912508017852623, 0.0046014604565363295, -3.0917101177485806)), ('u', 3, (1.3477092368498658, 0.9888417142673291, 0.1453391073528154)), ('cz', 5, 4), ('u', 6, (-1.9734238059199634, 0.9210446919760125, 3.2753461886166804)), ('u', 5, (-0.7250272566644168, 0.13951695534942818, -0.966388415505158)), ('u', 4, (-3.1448769066377587, 4.806953846162392e-11, 3.1151898621906)), ('u', 1, (0.04988253584121335, 0.0012411777267747558, 0.05130355500828455)), ('u', 3, (0.26855113730749347, 0.6231306260987792, -1.6117960646084089)), ('cz', 4, 3), ('u', 6, (3.3847826758333106, 0.00596977314801572, -2.8428600224957288)), ('u', 5, (2.1293157236538565, 0.7214702929695543, -0.14006408794000524)), ('u', 4, (0.6136427935159456, 1.570796326733598, -0.8101638400353446)), ('u', 1, (0.0513035550082845, 0.008035083641094912, 0.053750229853381834)), ('u', 3, (-2.5464923216353004, 0.8906782349719334, 3.3661107025646615)), ('cz', 5, 4), ('u', 6, (-2.818715732777364, 0.952878782096676, 3.0043417437514988)), ('u', 5, (0.7409371417897517, 2.9556950151821755, -2.2286588731330164)), ('u', 4, (-2.3314288130784124, 1.5707963269612122, -0.7960277889820586)), ('u', 1, (0.05375022985338495, 0.01555988486698217, 0.05700069010366602)), ('u', 3, (0.6793869157364618, 2.2236498925058483, 0.7190561537490348)), ('cz', 4, 3), ('u', 6, (3.031965526893565, 1.3094348861348473, -1.7242464394092538)), ('u', 5, (-0.12184148553446827, 1.439333407376042, -0.7002989429508656)), ('u', 4, (-0.7747685409843722, 1.5707963280360524, -2.1392781173878195)), ('u', 1, (-1.5137956366912224, 1.6241176501118906, 1.53859742869943)), ('u', 3, (-0.7249110800378891, 3.141592651784004, -4.3731490908327375)), ('u', 0, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('cz', 0, 1), ('u', 1, (1.570796326794898, 0.6154797086703868, -1.570796326794898)), ('u', 0, (1.5707963267948966, 0.6154797086703876, -1.5707963267948966)), ('cz', 0, 1), ('u', 1, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 0, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 1, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.2630564724597029, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.30773985433519363, 0.0)), ('cz', 2, 1), ('u', 1, (-3.141592653589793, 0.30773985433519363, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 1.2630564724597029, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (0.0, 1.5707963267948966, 0.0)), ('u', 1, (0.0, 1.5707963267948966, 0.0)), ('cz', 0, 1), ('u', 1, (1.2251903831398114e-16, 0.7853981633974483, -1.1496735851465472e-17)), ('u', 0, (-3.141592653589793, 0.7853981633974483, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 0, (-3.141592653589793, 1.5707963267948966, 3.141592653589793))])], (8, 2): [(8, [('u', 3, (0.0, 2.1790419057051063, 0.0)), ('u', 2, (0.0, 1.0213290820372691, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.0213290820372691, 3.141592653589793)), ('u', 6, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 6), ('u', 4, (0.37799737698828695, 0.13502176705432672, 0.3214040538289391)), ('u', 5, (0.3985397211331689, 0.00929248085827109, 0.6187820595919282)), ('u', 6, (3.2233516201965586, 1.9067757501890086, -0.05923425999101717)), ('u', 7, (2.62546993154895, 1.5707963142598302, -2.972963826565313)), ('u', 2, (-0.303217546481335, 1.6184756894581043e-08, 0.15983311941825973)), ('cz', 7, 2), ('u', 4, (-2.779877173779888, 0.42155213867689517, 3.331313669718659)), ('u', 5, (0.5589160952636802, 0.7818017314577775, 0.15057489574352603)), ('u', 6, (-2.8387553185480625, 0.3454173956786111, 1.395808581518468)), ('u', 7, (2.0668343681755905, 0.1279373175679867, 0.03892061028899807)), ('u', 2, (-2.812748656454745, 0.19672262791544828, 3.31337968219386)), ('cz', 5, 6), ('u', 4, (-0.1771306243590159, 0.13389517030463718, 0.006932825931703093)), ('u', 5, (3.562087886730953, 0.643980325868593, -2.5821124658608414)), ('u', 6, (1.3916958469779557, 1.5707963158511256, -4.256035537516305)), ('u', 7, (-1.1156796118587728, 0.19288764151068444, 0.732595011970502)), ('u', 2, (3.2541291973852826, 0.030967642953222154, -2.842303518727326)), ('cz', 6, 7), ('u', 4, (3.0498097753948747, 0.2672829582477914, -2.7924000323318396)), ('u', 5, (1.1326269267880138, 0.4031206522779272, 0.28389941516135236)), ('u', 6, (-3.597946013149868, 1.5707964348703332, 0.4449556999661015)), ('u', 7, (-3.3908973224820644, 0.12096164373375765, 2.6080349100269538)), ('u', 2, (-0.09237201813450077, 0.16111360667461563, 0.09151238895854427)), ('cz', 5, 6), ('u', 4, (0.19735485201244668, 0.25948365810477114, 0.010813617398872308)), ('u', 5, (-3.2311828692849645, 1.2051876216157658, 2.416521252848505)), ('u', 6, (-1.4921532509903792, 1.5707964153532405, -0.3490796341036889)), ('u', 7, (1.9753936781527108, 1.058644921011962, 0.16157583510960716)), ('u', 2, (-3.162834510023021, 0.5747347109672598, 2.988971814363478)), ('cz', 6, 7), ('u', 4, (1.0697635739624871, 1.52658235064331, -1.5672771782289299)), ('u', 5, (-4.2127273977581225, 2.0992493467840383, 0.33134519297886533)), ('u', 6, (0.6984144283130203, 1.5252938260335696, -0.21440583620126957)), ('u', 7, (-0.19358815636896123, 0.2559393139967764, 0.47420913599389414)), ('u', 2, (-0.24775721799526773, 0.8788483436124567, 0.6306936253356296)), ('cz', 4, 5), ('u', 4, (-1.582827360163844, 0.21776170604375605, 0.9591539302732023)), ('u', 5, (2.3503014223355794, 0.6154796976043173, -1.7855613183375023)), ('u', 6, (3.3231024420029933, 1.6156397416055137, 0.03364890849704727)), ('u', 7, (2.709376855117922, 0.1955528890284926, -3.2338974877284086)), ('u', 2, (0.8680317109333897, 0.4997448463602015, -0.056827154909853406)), ('cz', 5, 6), ('u', 4, (3.377610500647098, 1.6793898967932746, -2.231024413023696)), ('u', 5, (2.7408778918623415, 1.5707963059636127, -2.1365798542758636)), ('u', 6, (0.58536098355167, 0.6094105028753255, 0.969585707515199)), ('u', 7, (-3.1168755558467276, 0.33665600594107586, 2.811651952041584)), ('u', 2, (0.005221149173498196, 0.19690787579305224, -0.3665845427224871)), ('cz', 4, 5), ('u', 4, (0.29504406680573697, 0.44846474121374397, -0.07571209595121042)), ('u', 5, (0.5657835138432934, 1.570796357313495, 0.8143058310464644)), ('u', 6, (4.55504008542289, 1.7809235481243086, 0.427220452988589)), ('u', 7, (0.3766145100605592, 0.06878691216042872, -0.009096775098752952)), ('u', 2, (3.2696925203060507, 0.7014902173162261, -2.9962649279847104)), ('cz', 5, 6), ('u', 4, (-0.1860683590876881, 1.1357275504580875, 0.8290934330764712)), ('u', 5, (0.7564903845717192, 2.3561944594688504, -0.008435285167640416)), ('u', 6, (3.492481894270857, 1.4998778378919626, 1.1331841437533055)), ('u', 7, (-0.17712062085308095, 0.10840439565034213, 0.5326929757954737)), ('u', 2, (0.02449124975477296, 0.21458275829027432, -0.8363589811847667)), ('cz', 4, 5), ('u', 4, (0.8541514401733135, 1.5707963383368226, 0.028213333138799424)), ('u', 5, (3.1500279094418913, 1.570796348241581, -3.113379339992024)), ('u', 6, (3.8253251949344014, 0.2836996500115563, -1.7985178959753672)), ('u', 7, (2.8333938913609673, 1.8008917306853038, 0.2999084398575791)), ('u', 2, (-3.015514266938172, 0.16650447060293516, 2.5369635946250644)), ('u', 3, (2.0233960134963267, 1.551684614316178, -3.7235890685381516)), ('u', 1, (0.839139008888907, 2.3561944756657436, 0.2504183418438984)), ('u', 0, (-0.4857212906342475, 0.36627051757076884, 0.7052590757840566)), ('cz', 2, 1), ('u', 3, (-0.9925699444130445, 1.3760589052021495, 0.030269165866424308)), ('u', 2, (0.10192704701523331, 1.5707963660171047, 0.9817870956243782)), ('u', 7, (-3.05783079748452, 0.44702950751519543, 3.0591122753422684)), ('u', 1, (-2.401632134400665, 2.011001083856294, 2.7586445799647334)), ('u', 0, (-2.8349759805099564, 0.11167287121304673, 2.4995603415012106)), ('cz', 3, 2), ('u', 3, (2.6528659076669325, 1.4955652126472454, -0.581422947523186)), ('u', 2, (0.5890092258714162, 1.5707964336192224, 0.9543365451447087)), ('u', 7, (-2.4564367081972427, 0.001145535296781138, 2.9803634148299323)), ('u', 1, (2.629943460823213, 0.8604362142849246, -1.384439748694525)), ('u', 0, (-2.7696027145682147, 0.5293282705771406, 2.868909903382035)), ('cz', 2, 1), ('u', 3, (2.7543724602965853, 1.9386575875706562, -0.09457060820058993)), ('u', 2, (1.1400585522478939, 1.5707963842675132, -0.24709348408042153)), ('u', 7, (-0.03540724420330254, 0.36887871489577734, -0.02926433557526944)), ('u', 1, (-2.7464252866341696, 0.1488361173078092, 2.928654164947201)), ('u', 0, (0.09806607900511372, 0.6087747244522421, 1.0134836678818133)), ('cz', 3, 2), ('u', 3, (2.5411515937880274, 2.070356994783111, 0.004172307630795125)), ('u', 2, (-0.43614010449775026, 1.397947603639385, -0.3403222040372)), ('u', 7, (-2.9865958698364845, 0.022585922142579373, 2.6479809788314825)), ('u', 1, (-1.7708976029458179, 1.5767573848468537, 1.0113815559405186)), ('u', 0, (2.081989898519472, 1.78915173207941, -1.7763334126680412)), ('cz', 1, 0), ('u', 3, (0.200114952140458, 0.23801058572618608, 0.38802668476110397)), ('u', 2, (-3.403426308712772, 0.5983450085064305, 2.7416840598397942)), ('u', 7, (0.370147200546696, 0.2863852341889236, 0.8553200698689292)), ('u', 1, (1.99367864708359, 0.6185023678637499, -1.5377647673237942)), ('u', 0, (2.0945541847977083, 0.6154797183900051, -1.6181223319858802)), ('cz', 1, 0), ('u', 3, (3.0225188973485375, 0.33980460339632457, -3.1942827347279366)), ('u', 2, (2.908460845636855, 1.788529110679806, 1.0458870914226317)), ('u', 7, (-0.5566969562654057, 0.12065418147763163, 0.5761517344939163)), ('u', 1, (2.272652938329278, 2.2497998576543905, -0.17164199950911319)), ('u', 0, (-0.8132156765144153, 1.286884569753828, 1.954955932162643)), ('cz', 2, 1), ('u', 3, (3.4756460073539834, 0.7671276329455101, -2.7413935139021897)), ('u', 2, (3.8615107972244402, 1.5707963156913627, -1.1462191953360066)), ('u', 7, (2.5838542391729247, 0.09476825405531618, -3.344643902779925)), ('u', 1, (1.1351466721560337, 0.9553166000479915, -1.3468939314589248)), ('u', 0, (0.12449216004804736, 0.06041407986919991, 0.5463272926004652)), ('cz', 2, 1), ('u', 3, (2.9487294693336192, 0.20656136724811683, -3.1522284666004605)), ('u', 2, (2.7238893579338144, 1.2645000680574323, -0.0010032797489376577)), ('u', 7, (2.840595257658502, 0.42664993018723324, -2.869969576330104)), ('u', 1, (-1.6423778391830317, 1.5707962923311725, -0.7817027492867199)), ('u', 0, (-0.04584132279425973, 0.9891115437703512, -0.43347269804319366)), ('cz', 1, 0), ('u', 3, (-0.33353164980451383, 0.3778770017806244, -0.32176578169817877)), ('u', 2, (-1.8943642451851161, 1.0336000024491467, 3.770440671799113)), ('u', 7, (0.06280046937907335, 0.19132150331072284, -0.5823857950392706)), ('u', 1, (-0.7890937151215436, 0.7853981666124227, -0.26129160392022577)), ('u', 0, (2.2487105993192125, 1.570796405161783, -2.6885381644534028)), ('cz', 1, 0), ('u', 3, (2.8047752638780143, 0.0038477722018873685, -3.247686687003211)), ('u', 2, (3.0859650167633217, 0.6241101156589471, -3.085616678830329)), ('u', 7, (-2.8848664924229377, 0.17511456838923553, 2.654796525999915)), ('u', 1, (-2.880301048039854, 1.5707963283207336, 2.2654738534785963)), ('u', 0, (-2.0238507562988173, 1.5707963070424775, 2.265473890438774))]), (8, [('u', 3, (0.0, 2.1790419057051063, 0.0)), ('u', 2, (0.0, 1.0213290820372691, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.0213290820372691, 3.141592653589793)), ('u', 6, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 6), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 7), ('u', 7, (1.9364614740197674, 0.3995472174128471, -0.408778391043077)), ('u', 6, (0.4970631468127439, 2.8946905580841084e-12, -1.5005067128754823)), ('u', 5, (0.0002449918799918571, 0.7853981633974442, 0.14927729107622487)), ('cz', 6, 5), ('u', 7, (-0.40877839104307756, 0.29611018684241097, 0.5906329498660204)), ('u', 6, (1.641085940714299, 1.5707963267928753, -3.534996487819914)), ('u', 5, (3.290869944666034, 0.9263657552645578, -2.6747629075099306)), ('cz', 7, 6), ('u', 7, (-0.45678950456718925, 0.3862279409890685, -0.5209146414558576)), ('u', 6, (-2.748188819359843, 2.6179938779913856, 1.5707963267949048)), ('u', 5, (0.4503821596144114, 0.2650723925833735, 0.34202380955999967)), ('cz', 6, 5), ('u', 7, (2.620678012133932, 1.2160822408932725, -0.19248102329695715)), ('u', 6, (-1.570796326794875, 1.5707963267950042, 1.0391687510491465e-13)), ('u', 5, (3.483616463149799, 0.13205510060954728, -2.790922164413332)), ('cz', 7, 6), ('u', 4, (1.49256647844945, 0.6154796918608355, -1.8844929670572883)), ('u', 5, (-3.4922631182103436, 3.00953755305233, 0.04060685949754905)), ('u', 6, (0.8929093440695672, 1.6746171378896129, -0.8480474977555372)), ('cz', 4, 5), ('u', 4, (-1.8661064027322336, 0.5855771439992585, 0.9447317300849666)), ('u', 5, (2.8383102399215123, 1.5707963228286488, -1.208114548790991)), ('u', 6, (3.191576520277917, 0.8831819477647298, 0.332112845138266)), ('cz', 5, 6), ('u', 4, (2.5690938753165358, 1.1387393304331621, -2.319003702245468)), ('u', 5, (2.778910873138539, 1.5707963379414593, -2.4279982424653257)), ('u', 6, (0.3141930947397381, 0.3488528338498121, 0.7468902940538816)), ('cz', 4, 5), ('u', 4, (-2.380244555886975, 0.2490873873079936, 3.580992716413699)), ('u', 5, (0.24172220724633975, 1.5707963295175347, 0.9321949381804698)), ('u', 6, (3.9966154250803028, 0.8979612823653983, 0.4082511888815892)), ('cz', 5, 6), ('u', 4, (0.17555151328320606, 0.9799303489539507, 0.014303856419479288)), ('u', 5, (0.6386013883004471, 1.5707963200948742, 0.10528611287434297)), ('u', 6, (3.298804248697997, 2.043171274968978, 0.5241489165152349)), ('cz', 4, 5), ('u', 4, (3.329802219944857, 0.7853981620109924, -2.867304092250632)), ('u', 5, (0.09570264480656124, 1.5707963320520875, 0.02166701053358127)), ('u', 6, (-3.3915051572212875, 0.516402974394895, 2.773526478019853)), ('cz', 4, 5), ('u', 4, (2.9009472042826023, 1.2392423742256807e-09, -2.9332453251897133)), ('u', 5, (3.1199256405272573, 1.5707963166739496, -2.899602255211545)), ('u', 6, (0.1400655483674396, 0.02759067538688454, 0.4273349636324735)), ('u', 3, (2.0233960134963267, 1.551684614316178, -3.7235890685381516)), ('u', 2, (0.12588726673913567, 2.7749175075435377e-08, 0.030011978062980678)), ('u', 7, (2.6373121963296677, 1.7319304813933931, -0.47863191543048367)), ('u', 1, (0.839139008888907, 2.3561944756657436, 0.2504183418438984)), ('u', 0, (-0.4857212906342475, 0.36627051757076884, 0.7052590757840566)), ('cz', 2, 1), ('u', 3, (-0.9925699444130445, 1.3760589052021495, 0.030269165866424308)), ('u', 2, (0.10192704701523331, 1.5707963660171047, 0.9817870956243782)), ('u', 7, (-3.05783079748452, 0.44702950751519543, 3.0591122753422684)), ('u', 1, (-2.401632134400665, 2.011001083856294, 2.7586445799647334)), ('u', 0, (-2.8349759805099564, 0.11167287121304673, 2.4995603415012106)), ('cz', 3, 2), ('u', 3, (2.6528659076669325, 1.4955652126472454, -0.581422947523186)), ('u', 2, (0.5890092258714162, 1.5707964336192224, 0.9543365451447087)), ('u', 7, (-2.4564367081972427, 0.001145535296781138, 2.9803634148299323)), ('u', 1, (2.629943460823213, 0.8604362142849246, -1.384439748694525)), ('u', 0, (-2.7696027145682147, 0.5293282705771406, 2.868909903382035)), ('cz', 2, 1), ('u', 3, (2.7543724602965853, 1.9386575875706562, -0.09457060820058993)), ('u', 2, (1.1400585522478939, 1.5707963842675132, -0.24709348408042153)), ('u', 7, (-0.03540724420330254, 0.36887871489577734, -0.02926433557526944)), ('u', 1, (-2.7464252866341696, 0.1488361173078092, 2.928654164947201)), ('u', 0, (0.09806607900511372, 0.6087747244522421, 1.0134836678818133)), ('cz', 3, 2), ('u', 3, (2.5411515937880274, 2.070356994783111, 0.004172307630795125)), ('u', 2, (-0.43614010449775026, 1.397947603639385, -0.3403222040372)), ('u', 7, (-2.9865958698364845, 0.022585922142579373, 2.6479809788314825)), ('u', 1, (-1.7708976029458179, 1.5767573848468537, 1.0113815559405186)), ('u', 0, (2.081989898519472, 1.78915173207941, -1.7763334126680412)), ('cz', 1, 0), ('u', 3, (0.200114952140458, 0.23801058572618608, 0.38802668476110397)), ('u', 2, (-3.403426308712772, 0.5983450085064305, 2.7416840598397942)), ('u', 7, (0.370147200546696, 0.2863852341889236, 0.8553200698689292)), ('u', 1, (1.99367864708359, 0.6185023678637499, -1.5377647673237942)), ('u', 0, (2.0945541847977083, 0.6154797183900051, -1.6181223319858802)), ('cz', 1, 0), ('u', 3, (3.0225188973485375, 0.33980460339632457, -3.1942827347279366)), ('u', 2, (2.908460845636855, 1.788529110679806, 1.0458870914226317)), ('u', 7, (-0.5566969562654057, 0.12065418147763163, 0.5761517344939163)), ('u', 1, (2.272652938329278, 2.2497998576543905, -0.17164199950911319)), ('u', 0, (-0.8132156765144153, 1.286884569753828, 1.954955932162643)), ('cz', 2, 1), ('u', 3, (3.4756460073539834, 0.7671276329455101, -2.7413935139021897)), ('u', 2, (3.8615107972244402, 1.5707963156913627, -1.1462191953360066)), ('u', 7, (2.5838542391729247, 0.09476825405531618, -3.344643902779925)), ('u', 1, (1.1351466721560337, 0.9553166000479915, -1.3468939314589248)), ('u', 0, (0.12449216004804736, 0.06041407986919991, 0.5463272926004652)), ('cz', 2, 1), ('u', 3, (2.9487294693336192, 0.20656136724811683, -3.1522284666004605)), ('u', 2, (2.7238893579338144, 1.2645000680574323, -0.0010032797489376577)), ('u', 7, (2.840595257658502, 0.42664993018723324, -2.869969576330104)), ('u', 1, (-1.6423778391830317, 1.5707962923311725, -0.7817027492867199)), ('u', 0, (-0.04584132279425973, 0.9891115437703512, -0.43347269804319366)), ('cz', 1, 0), ('u', 3, (-0.33353164980451383, 0.3778770017806244, -0.32176578169817877)), ('u', 2, (-1.8943642451851161, 1.0336000024491467, 3.770440671799113)), ('u', 7, (0.06280046937907335, 0.19132150331072284, -0.5823857950392706)), ('u', 1, (-0.7890937151215436, 0.7853981666124227, -0.26129160392022577)), ('u', 0, (2.2487105993192125, 1.570796405161783, -2.6885381644534028)), ('cz', 1, 0), ('u', 3, (2.8047752638780143, 0.0038477722018873685, -3.247686687003211)), ('u', 2, (3.0859650167633217, 0.6241101156589471, -3.085616678830329)), ('u', 7, (-2.8848664924229377, 0.17511456838923553, 2.654796525999915)), ('u', 1, (-2.880301048039854, 1.5707963283207336, 2.2654738534785963)), ('u', 0, (-2.0238507562988173, 1.5707963070424775, 2.265473890438774))])], (8, 3): [(8, [('u', 3, (0.0, 2.6004931276326473, 0.0)), ('u', 2, (0.0, 0.7468985930690364, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 0.7468985930690364, 3.141592653589793)), ('u', 1, (0.0, 1.183199640139716, 0.0)), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 1), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.183199640139716, 3.141592653589793)), ('u', 5, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 6, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 6), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 7), ('u', 4, (3.3728209205811615, 0.39106167710400747, -2.8687717055019393)), ('u', 5, (-0.267976535731745, 1.3346243562138432e-07, 0.8176961999754309)), ('u', 6, (0.4187445910782187, 1.5707965062407465, -0.08319310590049234)), ('u', 7, (3.222557639729355, 1.5974818362769578, 0.1385678699936217)), ('cz', 5, 6), ('u', 4, (-0.16412447805231375, 0.09470900943880617, 0.5194297750567337)), ('u', 5, (-2.899229607793419, 2.3561944831090047, 2.8887968053752866)), ('u', 6, (3.224785594544601, 1.5707963204610151, 0.6050629969762096)), ('u', 7, (0.3138636053409246, 0.11902835982758336, -0.40969146823104996)), ('cz', 5, 6), ('u', 4, (-3.391915998449925, 0.1158294466557484, 2.5286218601502957)), ('u', 5, (-0.027236169388895837, 1.079027130972998, -0.24536614290042535)), ('u', 6, (-0.4183939664384144, 1.5707961904868029, -3.6094495596543448)), ('u', 7, (3.705920082451782, 1.7564319375656394, -0.17092894557287108)), ('cz', 6, 7), ('u', 4, (2.577118808011268, 0.30839765583209766, -3.2767384268107502)), ('u', 5, (0.4494697052401245, 1.3024393647865542, 0.6675843672387018)), ('u', 6, (2.0386529421286115, 1.570796209493473, -2.200135053185118)), ('u', 7, (0.29050724618378365, 0.25784242113455164, -0.02500257415237106)), ('cz', 5, 6), ('u', 4, (2.7251109785560708, 0.15695392791937654, -3.4302069182176824)), ('u', 5, (-0.061131030905422476, 0.35545862486106106, -0.28617913977166287)), ('u', 6, (1.1529375052313986, 1.5707961932582082, -0.2660537828520958)), ('u', 7, (3.029563323818107, 1.3023766007358066, -0.03784718681305499)), ('cz', 6, 7), ('u', 4, (2.023165182622548, 1.7155532643163143, -1.5341110023713176)), ('u', 5, (-1.6048632094672302, 1.2603414399435735, 1.0688840184512896)), ('u', 6, (-0.5928528095969762, 1.0498745444589463, -0.5844455246568968)), ('u', 7, (3.388646972626647, 1.8617076689373708, 0.4354472908986371)), ('cz', 4, 5), ('u', 4, (1.1816979073348377, 1.5707963789697903, -1.6682862462773675)), ('u', 5, (1.458072891279728, 1.570796386836844, -0.12583102055826956)), ('u', 6, (-0.7194021009941884, 1.2450507217702333, -0.013951281373488833)), ('u', 7, (-2.325339774940423, 0.11129343719290953, 2.7753564553207064)), ('cz', 4, 5), ('u', 4, (1.64271682518899, 3.073442043114022, 4.538690545281303)), ('u', 5, (3.1941662040641745, 1.724911806660808, -1.9410001509872072)), ('u', 6, (2.698541789712482, 0.4479972486841587, -0.6627965288374593)), ('u', 7, (-2.993942680108106, 0.23521611280486165, 3.097752498315438)), ('cz', 5, 6), ('u', 4, (0.11728795446952442, 0.6778994045979737, -0.2951661800459031)), ('u', 5, (-3.217350156074537, 0.6226665641915875, 2.584004647876074)), ('u', 6, (2.5593474395550335, 3.019683074405999, 0.3386062766998035)), ('u', 7, (-0.4266726648541127, 0.09281550326566909, 0.034186827855751084)), ('cz', 4, 5), ('u', 4, (2.5219147368006984, 1.2004686024392561, -3.1671833488467156)), ('u', 5, (-2.584004676052905, 1.570796371905883, 2.6318988603455256)), ('u', 6, (0.37457047092306783, 1.663190656482429, 0.5335023393030396)), ('u', 7, (0.07100987203302439, 0.041022120614617595, -0.3616200218093768)), ('cz', 5, 6), ('u', 4, (-2.7001331184400894, 0.24822547857918786, 3.182386433519112)), ('u', 5, (-0.2808515276680261, 1.5707963220356849, 0.3676041932232932)), ('u', 6, (-2.910272582160462, 0.1449916061870487, 3.3087391888891764)), ('u', 7, (3.0001224361099816, 0.367051690636943, -3.1198527489791386)), ('cz', 4, 5), ('u', 4, (-0.5966271584471116, 0.507432086302431, -0.11837694143314106)), ('u', 5, (1.2031920948608357, 1.5707962973025824, 0.14638436427251433)), ('u', 6, (3.865227906681971, 1.4792373391097866, 0.14001516136296965)), ('u', 7, (2.8617752403302896, 0.3403339271803459, -3.247823365181908)), ('cz', 5, 6), ('u', 2, (0.17019482054623225, 1.5707963164237424, 0.22167593381239736)), ('u', 1, (0.17019485468649442, 1.5707963021933358, 4.616040168497951)), ('u', 0, (0.0022605489924552025, 0.04447500135352692, -1.3393501587254977)), ('u', 3, (-0.8298645381396916, 0.11094846618151442, -0.5972517157377744)), ('u', 6, (2.2120238062657993, 1.9637734623613083, -0.249278751266526)), ('cz', 2, 1), ('u', 2, (-0.22167592189386123, 0.7853981645675656, 0.21898891430049736)), ('u', 1, (3.4761012864725442, 1.339117622575057, -2.7786848898724745)), ('u', 0, (-3.5817562877890836, 0.0531638441594182, 2.5006606863981418)), ('u', 3, (-3.6712572805973256, 0.6313327419861082, 2.299248835676188)), ('u', 6, (-0.48352378709204075, 0.14751049185047072, -0.5142732815555632)), ('cz', 2, 1), ('u', 2, (-0.831132653929176, 3.1415926469123683, -1.8307981270898233)), ('u', 1, (-3.0548966138432574, 1.703265840356079, 3.1790026129297555)), ('u', 0, (-1.1953518813256916, 0.8135945097755325, -2.2144782903357014)), ('u', 3, (2.846945313107482, 1.201742192282055, 0.24909406789456812)), ('u', 6, (-3.0178589908010416, 0.41418460790010425, 2.7033554245375715)), ('cz', 2, 3), ('u', 2, (-1.9229382570990945, 1.5707963290416478, 3.9876628047495686)), ('u', 1, (-1.2146135345598759, 0.3344005973948035, -0.2075894528765152)), ('u', 0, (2.8311582662723764, 0.6607943247008267, -2.9227888201568075)), ('u', 3, (3.3682524746099234, 1.0570166230980986, -0.06205178028258995)), ('u', 6, (-0.4079146951896172, 0.11307885034027033, 0.36102093499718313)), ('cz', 2, 1), ('u', 2, (1.2727790177339775, 1.8558570651771735, -0.4158153352822194)), ('u', 1, (0.949875237522008, 0.6871912870686274, 1.05491839076666)), ('u', 0, (3.7796322374143703, 0.2348284227396344, -2.4546769113537206)), ('u', 3, (-2.8265724101130063, 0.01014374652125673, 2.869560565339029)), ('u', 6, (3.4099064525789466, 0.15348046900051315, -2.5429653211922267)), ('cz', 2, 3), ('u', 2, (-3.723377953773289, 0.5488934566153154, 2.1188625818399984)), ('u', 1, (-2.1981442303549708, 0.3359809558076345, 3.7118480301551884)), ('u', 0, (0.2601192024789832, 0.7567804762796897, -0.653628005679572)), ('u', 3, (-3.051596086358009, 0.646999330693771, 2.751925526860514)), ('u', 6, (-2.850494657567627, 0.31387073000199656, 3.295942863131725)), ('cz', 1, 0), ('u', 2, (-0.27539592757718384, 1.3990719693955174, -0.9872357964566447)), ('u', 1, (1.6516313176946151, 2.3868530285797958, -1.1318144059868365)), ('u', 0, (1.5799187073848207, 1.5707963204526236, -0.9098736884358005)), ('u', 3, (-0.017585970514645693, 0.05100395494315851, 0.4278381579929925)), ('u', 6, (2.677794619512114, 0.23454084676659628, -2.6973212261886648)), ('cz', 1, 0), ('u', 2, (1.83508579342028, 1.0489196739086213, -0.46860092122164476)), ('u', 1, (1.617433129108679, 1.5707963231186162, -0.9841912789155298)), ('u', 0, (-0.6883037262441306, 1.8730241766776343, 1.114845018021535)), ('u', 3, (2.9080536897331903, 0.7424522884930189, -2.740764599761599)), ('u', 6, (3.464127252804771, 0.1676096031474168, -2.6333572906808755)), ('cz', 2, 1), ('u', 2, (4.093737716561863, 1.9146357703355779, 0.4447267859329944)), ('u', 1, (-2.157401373465824, 2.427378425544589, 2.992414488474941)), ('u', 0, (-0.1502087048392144, 0.857195019281663, 0.36540691589954716)), ('u', 3, (0.11480280359866452, 0.05493838074684792, 0.041683955656111576)), ('u', 6, (0.37091179090473403, 0.04141032392649731, 0.25964304423216455)), ('cz', 1, 0), ('u', 2, (1.2908280750009269, 1.1412600605504781, -0.6447271282888243)), ('u', 1, (0.1491781757101821, 1.5707963131232534, 0.7542296017172242)), ('u', 0, (0.8467628977387938, 0.6157647454113345, -0.29417591386695435)), ('u', 3, (2.851518341415716, 0.3231521525718433, -3.2220833280099668)), ('u', 6, (3.0486628701566914, 0.11703664480778077, -3.1755722016658248)), ('cz', 2, 1), ('u', 2, (3.0242614699725854, 0.06841861725852429, -2.7333397051135018)), ('u', 1, (-3.026197618562068, 1.570796315558012, 2.3464466578212173)), ('u', 0, (-0.4142166030401052, 0.5429224848251636, 0.5009461149031439)), ('u', 3, (-3.1952693412540296, 0.16387128079898583, 2.8757264082391285)), ('u', 6, (0.12236014889985145, 0.07368483489706198, -0.47480261029214943)), ('cz', 1, 0), ('u', 2, (3.116353052237453, 1.5073213088663675, -0.10665115514559842)), ('u', 1, (-0.775650340619295, 1.570796314882542, -0.5489102592853109)), ('u', 0, (-3.1315365664496158, 1.6080089822094911e-09, 2.2169944770832615)), ('u', 3, (0.07988145091868482, 0.12412243747944651, 0.05485126932871501)), ('u', 6, (-0.2668207252199987, 0.21757742495350252, -0.37446882952067745)), ('cz', 2, 1), ('u', 2, (3.222691765488767, 1.570796326840022, 0.7040609460816769)), ('u', 1, (0.5489102714949137, 1.5707963415457094, 0.13850967320827018)), ('u', 0, (-1.033353067503298, 5.344050763992638e-09, 0.45510447918870073)), ('u', 3, (-0.07919187134436817, 0.0737716818862282, 0.04330619866210563)), ('u', 6, (-0.004550713039591592, 0.23691724900672498, -0.300794685100584)), ('u', 7, (0.0032436760622015504, 0.6834998443485418, -0.37543550660464076)), ('u', 7, (0.17790794256295062, 0.2778823884636485, 0.06771807429415676)), ('u', 7, (0.03844868731066941, 0.08811718482662821, -0.05371352853264836)), ('u', 7, (3.087879124972279, 0.16187704781250048, -3.1863285152784178)), ('u', 7, (-3.1863285153525838, 0.44829498318534017, 3.047077383201628)), ('u', 4, (0.11837684645650276, 2.634160693045651, 1.050387936672795)), ('u', 5, (-0.14638444338338535, 1.5707962775860798, -0.35482118268032997)), ('u', 7, (-0.5167990477960749, 0.15328110898445052, 0.03839322151376151)), ('u', 7, (-0.2924525804492283, 0.01042098486691196, 0.23685163099643003)), ('u', 7, (-2.6898985381570064, 0.5497250222820544, 3.199934063053756)), ('u', 7, (0.007825319632610211, 0.25680036324009115, -0.2604650333637528)), ('u', 7, (-0.09248169206998845, 0.12472079356709315, 0.10177411824068093)), ('u', 7, (0.1525375994889486, 0.3687417995323103, 0.010712472004470425)), ('u', 7, (2.773136649248701, 0.08129114514751729, -2.9820161047644085)), ('u', 7, (2.65924046167778, 0.07459776920251235, -2.830238065771656)), ('u', 7, (-2.8575393144935695, 0.21072989668142014, 3.326969708625929))]), (8, [('u', 3, (0.0, 2.6004931276326473, 0.0)), ('u', 2, (0.0, 0.7468985930690364, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 0.7468985930690364, 3.141592653589793)), ('u', 1, (0.0, 1.183199640139716, 0.0)), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 1), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.183199640139716, 3.141592653589793)), ('u', 5, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 6, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 6), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 7), ('u', 4, (3.3728209205811615, 0.39106167710400747, -2.8687717055019393)), ('u', 5, (-0.267976535731745, 1.3346243562138432e-07, 0.8176961999754309)), ('u', 6, (0.4187445910782187, 1.5707965062407465, -0.08319310590049234)), ('u', 7, (3.222557639729355, 1.5974818362769578, 0.1385678699936217)), ('cz', 5, 6), ('u', 4, (-0.16412447805231375, 0.09470900943880617, 0.5194297750567337)), ('u', 5, (-2.899229607793419, 2.3561944831090047, 2.8887968053752866)), ('u', 6, (3.224785594544601, 1.5707963204610151, 0.6050629969762096)), ('u', 7, (0.3138636053409246, 0.11902835982758336, -0.40969146823104996)), ('cz', 5, 6), ('u', 4, (-3.391915998449925, 0.1158294466557484, 2.5286218601502957)), ('u', 5, (-0.027236169388895837, 1.079027130972998, -0.24536614290042535)), ('u', 6, (-0.4183939664384144, 1.5707961904868029, -3.6094495596543448)), ('u', 7, (3.705920082451782, 1.7564319375656394, -0.17092894557287108)), ('cz', 6, 7), ('u', 4, (2.577118808011268, 0.30839765583209766, -3.2767384268107502)), ('u', 5, (0.4494697052401245, 1.3024393647865542, 0.6675843672387018)), ('u', 6, (2.0386529421286115, 1.570796209493473, -2.200135053185118)), ('u', 7, (0.29050724618378365, 0.25784242113455164, -0.02500257415237106)), ('cz', 5, 6), ('u', 4, (2.7251109785560708, 0.15695392791937654, -3.4302069182176824)), ('u', 5, (-0.061131030905422476, 0.35545862486106106, -0.28617913977166287)), ('u', 6, (1.1529375052313986, 1.5707961932582082, -0.2660537828520958)), ('u', 7, (3.029563323818107, 1.3023766007358066, -0.03784718681305499)), ('cz', 6, 7), ('u', 4, (2.023165182622548, 1.7155532643163143, -1.5341110023713176)), ('u', 5, (-1.6048632094672302, 1.2603414399435735, 1.0688840184512896)), ('u', 6, (-0.5928528095969762, 1.0498745444589463, -0.5844455246568968)), ('u', 7, (3.388646972626647, 1.8617076689373708, 0.4354472908986371)), ('cz', 4, 5), ('u', 4, (1.1816979073348377, 1.5707963789697903, -1.6682862462773675)), ('u', 5, (1.458072891279728, 1.570796386836844, -0.12583102055826956)), ('u', 6, (-0.7194021009941884, 1.2450507217702333, -0.013951281373488833)), ('u', 7, (-2.325339774940423, 0.11129343719290953, 2.7753564553207064)), ('cz', 4, 5), ('u', 4, (1.64271682518899, 3.073442043114022, 4.538690545281303)), ('u', 5, (3.1941662040641745, 1.724911806660808, -1.9410001509872072)), ('u', 6, (2.698541789712482, 0.4479972486841587, -0.6627965288374593)), ('u', 7, (-2.993942680108106, 0.23521611280486165, 3.097752498315438)), ('cz', 5, 6), ('u', 4, (0.11728795446952442, 0.6778994045979737, -0.2951661800459031)), ('u', 5, (-3.217350156074537, 0.6226665641915875, 2.584004647876074)), ('u', 6, (2.5593474395550335, 3.019683074405999, 0.3386062766998035)), ('u', 7, (-0.4266726648541127, 0.09281550326566909, 0.034186827855751084)), ('cz', 4, 5), ('u', 4, (2.5219147368006984, 1.2004686024392561, -3.1671833488467156)), ('u', 5, (-2.584004676052905, 1.570796371905883, 2.6318988603455256)), ('u', 6, (0.37457047092306783, 1.663190656482429, 0.5335023393030396)), ('u', 7, (0.07100987203302439, 0.041022120614617595, -0.3616200218093768)), ('cz', 5, 6), ('u', 4, (-2.7001331184400894, 0.24822547857918786, 3.182386433519112)), ('u', 5, (-0.2808515276680261, 1.5707963220356849, 0.3676041932232932)), ('u', 6, (-2.910272582160462, 0.1449916061870487, 3.3087391888891764)), ('u', 7, (3.0001224361099816, 0.367051690636943, -3.1198527489791386)), ('cz', 4, 5), ('u', 4, (-0.5966271584471116, 0.507432086302431, -0.11837694143314106)), ('u', 5, (1.2031920948608357, 1.5707962973025824, 0.14638436427251433)), ('u', 6, (3.865227906681971, 1.4792373391097866, 0.14001516136296965)), ('u', 7, (2.8617752403302896, 0.3403339271803459, -3.247823365181908)), ('cz', 5, 6), ('u', 7, (0.0032436760622015504, 0.6834998443485418, -0.37543550660464076)), ('u', 1, (0.0, 3.1415926535890537, 3.88744303690369)), ('u', 6, (2.751834112798575, 2.1012039094847754, -0.1517714776595427)), ('u', 2, (-0.29549666251093637, 0.7050390407397258, -0.0504640633361539)), ('u', 0, (-8.823988715167541e-05, 0.5235987755986036, -0.2713082920613179)), ('cz', 1, 0), ('u', 7, (0.17790794256295062, 0.2778823884636485, 0.06771807429415676)), ('u', 1, (-0.3733786938654473, 1.5707963267944591, -0.3926991012583947)), ('u', 6, (0.05980748153639656, 0.3963037782683509, -0.006982201558379128)), ('u', 2, (-0.05046390721346742, 0.8682743217517934, 0.0020060945442915423)), ('u', 0, (-0.27130828125150197, 0.022777442641511807, 0.5193627464818114)), ('cz', 1, 2), ('u', 7, (0.03844868731066941, 0.08811718482662821, -0.05371352853264836)), ('u', 1, (-2.748893552328643, 0.7853981633375778, 3.1415926763397994)), ('u', 6, (-3.1646736998580014, 0.008360673113919317, 3.1112138433362153)), ('u', 2, (-0.010116346277316357, 0.08552576244548307, 0.04031331707155163)), ('u', 0, (0.044455988032640906, 0.5042175350645443, -0.19952287723631784)), ('cz', 1, 0), ('u', 7, (3.087879124972279, 0.16187704781250048, -3.1863285152784178)), ('u', 1, (-2.2751715630221348e-08, 1.57079632673552, 3.817196586803185e-08)), ('u', 6, (3.111213843333554, 0.3129850431921355, -3.156571838238244)), ('u', 2, (3.1819062965079126, 2.441467427658711, 0.004864965594580273)), ('u', 0, (0.0, 2.6513916189822675e-13, -0.9494262989530413)), ('cz', 1, 2), ('u', 7, (-3.1863285153525838, 0.44829498318534017, 3.047077383201628)), ('u', 1, (3.1415926154163, 1.5707963267948102, -2.769120964141354)), ('u', 6, (-3.1565718382321757, 0.6175309700502564, 3.0883100452959544)), ('u', 2, (3.1464573749932385, 1.570796326794898, 0.3724716894484392)), ('u', 0, (0.0, 9.751213919706676e-14, -1.6445174275305499)), ('u', 4, (0.11837684645650276, 2.634160693045651, 1.050387936672795)), ('u', 5, (-0.14638444338338535, 1.5707962775860798, -0.35482118268032997)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.3089969389957472, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 0.26179938779914935, 0.0)), ('cz', 3, 2), ('u', 2, (-3.141592653589793, 0.26179938779914935, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 1.3089969389957472, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 1, (-1.6785921893219244, 1.570796326932515, 1.6831464583629798)), ('u', 0, (1.4630004612841487, 1.5707963257673239, -1.6507071894784535)), ('u', 7, (-0.5167990477960749, 0.15328110898445052, 0.03839322151376151)), ('u', 2, (-3.1224273432308918, 0.41587665949883285, 2.939095964828157)), ('cz', 1, 0), ('u', 1, (1.4584461959307882, 0.35087940973568627, -0.7353608165593173)), ('u', 0, (1.6507071889773584, 2.2027106391988873, -1.0129019142750488)), ('u', 7, (-0.2924525804492283, 0.01042098486691196, 0.23685163099643003)), ('u', 2, (-2.8993921940537586, 0.11401683106714594, 3.2404143397766507)), ('cz', 1, 0), ('u', 1, (2.306157142595466, 1.5707963256255622, -1.6180373583861483)), ('u', 0, (-0.8987723413484447, 1.6185336315711922, 1.2736150288731265)), ('u', 7, (-2.6898985381570064, 0.5497250222820544, 3.199934063053756)), ('u', 2, (2.8538763397046765, 1.0463658462712422, -0.22826259701931817)), ('cz', 1, 2), ('u', 1, (-3.0943516163981646, 2.479447418865614, 3.0364459711810023)), ('u', 0, (0.43090293811141517, 0.34407505220565543, -0.09433502182989134)), ('u', 7, (0.007825319632610211, 0.25680036324009115, -0.2604650333637528)), ('u', 2, (3.386076292433321, 1.4467275743010095, -0.2690597066109728)), ('cz', 1, 0), ('u', 1, (0.10514668426339173, 1.5707963256835809, 0.13713489223040934)), ('u', 0, (-2.7763584089935467, 0.2296184329001961, 2.4227388217069032)), ('u', 7, (-0.09248169206998845, 0.12472079356709315, 0.10177411824068093)), ('u', 2, (0.45307542307963095, 0.342763381940028, -0.3672684148838083)), ('cz', 1, 2), ('u', 1, (3.0044577596882385, 0.6319143128224841, -3.1236522678810394)), ('u', 0, (2.882192858058798, 0.16702240300304355, -2.6640807431236553)), ('u', 7, (0.1525375994889486, 0.3687417995323103, 0.010712472004470425)), ('u', 2, (3.288950206193982, 0.21976111948157684, -2.8800839393473523)), ('cz', 1, 0), ('u', 1, (-0.01794038353676515, 1.5707963253622785, -0.2665683448864032)), ('u', 0, (3.022013975396951, 0.9319708637721523, -3.180375462917911)), ('u', 7, (2.773136649248701, 0.08129114514751729, -2.9820161047644085)), ('u', 2, (2.9139555600883, 1.3511572433220294, -0.1296179801412065)), ('cz', 1, 2), ('u', 1, (0.26656833931441565, 2.0943951027946675, -0.1617002129103472)), ('u', 0, (-0.2286978311965311, 0.43284174135734543, 0.06931291087109169)), ('u', 7, (2.65924046167778, 0.07459776920251235, -2.830238065771656)), ('u', 2, (3.2400301606384145, 1.4131105162038662, 0.06800845002444533)), ('cz', 1, 0), ('u', 1, (-2.9798924454688773, 1.5707963272351246, 3.2493885166068024)), ('u', 0, (-2.772648015074517, 1.57079632611796, 3.249388518729874)), ('u', 7, (-2.8575393144935695, 0.21072989668142014, 3.326969708625929)), ('u', 2, (0.08242219689306605, 0.15945674688551645, -0.040754500573628616)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0))])], (8, 4): [(8, [('u', 3, (0.0, 2.901974084960706, 0.0)), ('u', 2, (0.0, 0.5024148995576962, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 0.5024148995576962, 3.141592653589793)), ('u', 1, (0.0, 0.9687234065335635, 0.0)), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 1), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 0.9687234065335635, 3.141592653589793)), ('u', 0, (0.0, 1.3258176636680323, 0.0)), ('u', 0, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 0), ('u', 0, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (-3.141592653589793, 1.3258176636680323, 3.141592653589793)), ('u', 4, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 5, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 6, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 6), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 0, 7), ('u', 0, (0.24740256010545428, 1.5707963077782177, 2.347196534357998e-08)), ('u', 1, (3.1242004886093913, 1.5707963371671434, -3.141592665871107)), ('u', 2, (-3.177895552668161, 0.0166128724599478, 3.104620604231258)), ('u', 6, (0.017392164990191028, 0.025281051720125753, 0.01653321709368015)), ('u', 3, (2.9721414483883977, 0.4120439856151028, -2.9410094461947893)), ('u', 4, (0.16945120521196874, 0.0698923834482002, 0.13384585665249996)), ('cz', 0, 1), ('u', 0, (2.347977496886802e-08, 0.5235987762469274, 0.0006192026588912668)), ('u', 1, (-1.2307312022493454e-08, 0.5235987750710543, 4.0957679120355254e-09)), ('u', 2, (-0.03697204936691645, 0.00048686681607281563, -0.03589569340259112)), ('u', 6, (0.01653321712495465, 0.018805607579500085, 0.01511671621412466)), ('u', 3, (3.3421758611358854, 0.29943193629874465, -2.801280139749614)), ('u', 4, (0.13384585669619314, 0.06445488851009958, 0.09366300171440965)), ('cz', 0, 1), ('u', 0, (3.1422118562561323, 1.5765068190908538, -3.03480848239328)), ('u', 1, (4.0618448639284654e-09, 1.570796349677758, -0.038233242362414155)), ('u', 2, (3.1056969602348747, 1.554845145696804, 0.0012595738688832014)), ('u', 6, (0.015116716172206337, 0.013269990713690632, 0.013296369549273422)), ('u', 3, (-2.801280139660277, 0.1072229900875749, 3.4654009918427096)), ('u', 4, (0.09366300138962583, 0.004386940699479065, 0.06148599970898438)), ('cz', 1, 2), ('u', 0, (0.10678417118613182, 0.005843266668250025, 0.10474113038325222)), ('u', 1, (3.1033594113958802, 1.9659883474831268, -3.014698581466032)), ('u', 2, (3.1428522273951844, 1.5677942961108704, -0.023574434816298062)), ('u', 6, (0.013296369425864006, 0.008680015539861844, 0.011198541354675486)), ('u', 3, (0.3238083383413242, 0.08011083568791795, 0.27039913352409645)), ('u', 4, (0.06148600024379006, 0.0037736151387163202, 0.032378317282689306)), ('cz', 0, 1), ('u', 0, (3.2463337840273927, 7.150014946169529e-09, -3.008359840538833)), ('u', 1, (-3.0146985813499216, 0.40689197767059143, 3.141592654109527)), ('u', 2, (-0.023574434888719034, 0.00300537583725997, 0.029476304993231756)), ('u', 6, (0.011198541314167774, 0.0049966365584722125, 0.008926991461794022)), ('u', 3, (0.2703991333973423, 0.2276636401301653, 0.13420829392291284)), ('u', 4, (0.03237831709066832, 0.00028431420947649667, 0.005699356664471493)), ('cz', 1, 2), ('u', 0, (0.13323281300465886, 3.84756981491908e-09, 0.13825310112295322)), ('u', 1, (4.0711090054657006e-10, 0.4068919782469871, -0.12190029762589617)), ('u', 2, (0.029476305361641867, 0.7835378123971821, 3.452390887059842e-09)), ('u', 6, (0.008926991925730862, 0.002147874588835068, 0.0065658296753449965)), ('u', 3, (0.13420829339400075, 0.3277509783372616, 0.13594225576027558)), ('u', 4, (0.00569935647647184, 0.05443235537030334, -0.02529474470543954)), ('cz', 0, 1), ('u', 0, (3.2798457547072815, 0.04042449991685485, -2.9936036274925577)), ('u', 1, (-0.12190029748491468, 1.5594833340583052, -0.011653884343328196)), ('u', 2, (3.1415926568818042, 0.7835378104262972, 0.03488296439235583)), ('u', 6, (0.006565829258824254, 3.877275334941494e-05, 0.0041817744694165205)), ('u', 3, (0.13594225605744475, 0.40343337433406096, 0.10254334770957577)), ('u', 4, (-0.025294744418642594, 0.0061686684257190125, -0.05784649635865514)), ('cz', 2, 3), ('u', 0, (3.2895816799099253, 0.018841485951206053, -2.983977420578603)), ('u', 1, (-0.011653884296412806, 0.4066214082631142, 0.028392030143334628)), ('u', 2, (-3.1067096893045836, 2.8797148839802347, 3.1295315135530695)), ('u', 6, (-3.1374108789156523, 0.001440328390608002, 3.143418905676553)), ('u', 3, (3.24413600129048, 1.561398986569674, -0.009654370797877121)), ('u', 4, (-0.057846496375142484, 0.0026019334314028924, -0.08978871887270674)), ('cz', 1, 2), ('u', 0, (3.299207886364588, 0.005510219285898618, -2.976744077689273)), ('u', 1, (-3.1132006235541954, 1.6154230131070403e-07, 3.1306276257288856)), ('u', 2, (-0.012061140075379756, 1.8326013431221995, -0.04647442187655725)), ('u', 6, (-3.1397664014354603, 0.002406973196539197, 3.1411301860299656)), ('u', 3, (-0.009654370648902611, 0.0093990921605391, 0.0004294948300030358)), ('u', 4, (-3.231381372704636, 0.0008496024159753081, 3.0226345604417624)), ('cz', 2, 3), ('u', 0, (0.1648485760667228, 0.0021809407580145697, 0.16828362485439727)), ('u', 1, (-0.010965027630111281, 1.637056331664681e-07, -0.01269871139026909)), ('u', 2, (3.0951182316139176, 0.26288855875943395, -3.096706221181777)), ('u', 6, (-3.1420551213177568, 0.0029810570217928167, 3.138935623980257)), ('u', 3, (0.00042949486698879753, 0.012491695224934785, -0.0007960693692592453)), ('u', 4, (-3.260550746593288, 0.004700915511354513, 3.0002669274861367)), ('cz', 1, 2), ('u', 0, (0.16828362481483822, 0.00572269746304183, 0.16728867344561255)), ('u', 1, (-3.154291365115813, 0.018826034010998034, 3.128855482149092)), ('u', 2, (0.044886432450014314, 1.30891855727653, -0.045829524379756785)), ('u', 6, (-3.144249683147679, 0.0032804266365954802, 3.136853971042753)), ('u', 3, (3.1407965841908574, 1.5832880257026303, -0.0017197159331123757)), ('u', 4, (-3.2829183797662975, 0.008354769336868528, 2.9894702050321116)), ('cz', 2, 3), ('u', 0, (1.7380847604433431, 1.5107689911037403, -1.571324882275377)), ('u', 1, (-1.5835338264006893, 1.5703167289765865, 1.5613863603126945)), ('u', 2, (3.0957630142659, 0.584451077565623, -3.145699371051266)), ('u', 6, (-3.146331352457625, 0.0034174669627471403, 3.1348976928597034)), ('u', 3, (3.1398728835465652, 1.5376419799231713, 0.009492169126966488)), ('u', 4, (2.989470308545099, 0.01059604756820988, -3.2894516273286976)), ('cz', 0, 1), ('u', 0, (1.5702677713006499, 0.6154797072258071, -1.562243490887305)), ('u', 1, (1.561386360289, 0.6154797110922112, -1.093384343890502)), ('u', 2, (3.1374860576555013, 0.5234099894947305, -3.1390603303721556)), ('u', 6, (-3.1482874525086646, 0.003496569783050107, 3.1330754973688877)), ('u', 3, (-3.132100495325359, 0.027922222706663613, 3.148858360521322)), ('u', 4, (2.9937335297800525, 0.01041795155640386, -3.2694726600341175)), ('cz', 0, 1), ('u', 0, (-1.5622434908862481, 1.5685524404469555, 1.6056213996369189)), ('u', 1, (2.048208309723095, 1.5334813495880195, -1.5575957781587224)), ('u', 2, (3.1441250348259335, 2.0337419782229196, -0.020710776611550408)), ('u', 6, (-3.15010981015512, 0.003612193970001695, 3.131393821723035)), ('u', 3, (-3.1343268765560137, 0.021481875053809187, 3.1468180449301104)), ('u', 4, (3.0137126291018803, 0.0076370837612826535, -3.235962942998284)), ('cz', 1, 2), ('u', 0, (-3.106767784095532, 0.0019089473699859106, 3.172179534140847)), ('u', 1, (3.1547930906241923, 0.9553166197494333, -3.061391390515589)), ('u', 2, (3.120881877022657, 1.5707963169910486, -0.004226921051762478)), ('u', 6, (-3.1517914854691287, 0.003847534019552029, 3.1298599946401118)), ('u', 3, (-3.1363672621309844, 0.01411764137434865, 3.1449914125570997)), ('u', 4, (3.047222364307676, 0.0032825647198644393, -3.192834676468687)), ('cz', 1, 2), ('u', 0, (3.179329951343629, 0.015347223655525247, -3.0987623973207636)), ('u', 1, (-3.086559748241109, 0.7853981587838752, 3.1415926541008123)), ('u', 2, (-0.005908311286144508, 0.006152121796748064, 0.005067712979498832)), ('u', 6, (-3.153444735118384, 0.007543148283208885, 3.128302296756359)), ('u', 3, (0.003157281073491996, 0.004004813866927766, 0.0017976638359951114)), ('u', 4, (3.119194523485116, 0.0018677697291707613, -3.11628581011227)), ('cz', 0, 1), ('u', 0, (-3.0987623970550167, 0.3191079304856438, 3.1810842006744395)), ('u', 1, (5.962494737083578e-10, 0.7853981697864992, 1.5213844843458757e-09)), ('u', 2, (3.146660366552159, 1.576948127007934, -0.02770539087803514)), ('u', 6, (-3.154883010482205, 0.00851161057330634, 3.127035260463243)), ('u', 3, (0.0017976638470731906, 0.012089232256596767, 0.000718599521156016)), ('u', 4, (3.1668994970224933, 0.005817503998311398, -3.072119650548287)), ('cz', 1, 2), ('u', 0, (0.039491546971138984, 1.2732788594102697, 0.015160252314586625)), ('u', 1, (1.4302492379899154e-09, 0.7853981592693322, 7.428270823840504e-10)), ('u', 2, (3.1138872627906964, 2.080222468738006, -0.010864583275173256)), ('u', 6, (-3.156150046624366, 0.009746482137883213, 3.125964659276624)), ('u', 3, (0.00071859940018132, 0.01990596591083898, -6.178743292940315e-05)), ('u', 4, (3.211065656781986, 0.01417286467163173, -3.0358731670780625)), ('cz', 0, 1), ('u', 0, (-3.1264324012417974, 0.7853981574777166, 3.1333786159945234)), ('u', 1, (9.41110679187132e-10, 1.047197554106395, -0.0021205647076725847)), ('u', 2, (-0.010864583312786566, 0.525026676441514, 0.018992696265370222)), ('u', 6, (-3.1572206479508256, 0.011263392747836276, 3.125124278204042)), ('u', 3, (-6.178743661237085e-05, 0.027173244443369486, -0.0005393802257674917)), ('u', 4, (3.247312140084665, 0.026374950718660223, -3.0104594681177015)), ('cz', 0, 1), ('u', 0, (3.107819054907817, 0.6540555043263074, -3.0993775140200084)), ('u', 1, (3.139472091708805, 1.249202282975233e-08, -3.137351524031184)), ('u', 2, (0.014504636230908797, 0.5041748607112927, 0.05400415949506655)), ('u', 6, (3.115152592641828, 0.011939375919032371, -3.136810400210318)), ('u', 3, (-0.0005393626875548407, 0.03361669691817586, -0.0007268792286183329)), ('u', 4, (0.29901481253552936, 1.5134045451940534, 0.1712135923327947)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (3.2967664143484856, 0.37746088244651743, -2.90770693507335)), ('u', 5, (-3.065351981199511, 1.570796325998965, 3.141592810233495)), ('cz', 5, 4), ('u', 0, (-3.1336817276541074, 0.03988000500708097, 3.1477971068631443)), ('u', 7, (3.3754782677989725, 0.2957003307021647, -2.8697445297881767)), ('u', 6, (3.1332705618836747, 0.000670964769903343, -3.14776404325472)), ('u', 5, (-1.2304880342357098e-07, 0.5235987812541554, -2.9738587354984345e-08)), ('u', 2, (3.1384898734794255, 0.03459206355040011, -3.14347395431741)), ('u', 4, (-3.5769262187636676e-08, 0.523598771695477, -0.00014103518936014022)), ('cz', 5, 4), ('u', 0, (-3.1353881481357257, 0.04020044576188537, 3.1458639800638526)), ('u', 7, (3.4134410998710325, 0.18048471998342222, -2.8492079037556315)), ('u', 6, (3.1354213213872644, 1.5700021680961975, -4.475632626022019e-05)), ('u', 5, (-7.312312765017648e-08, 1.5707963100608628, -0.08150001422869849)), ('u', 2, (3.1397114051832484, 0.035398571332497825, -3.1424570006512282)), ('u', 4, (3.141451865879188, 1.5700162024807447, -2.9682222802448033)), ('cz', 6, 5), ('u', 0, (-3.1373213273631406, 0.03904869406383659, 3.1437810459612163)), ('u', 7, (0.29238516947199533, 0.018297901744561286, 0.2766538192545155)), ('u', 6, (3.141547879107957, 1.5729530881038725, -0.023488310377399735)), ('u', 5, (3.0600922833400723, 1.9638558876725143, -3.092184870708789)), ('u', 2, (3.1407285146219714, 0.034780753328794205, -3.1416728739950286)), ('u', 4, (3.3149630865408977, 0.0008295588521163107, -2.966619969343351)), ('cz', 5, 4), ('u', 0, (-3.1394044172879125, 0.036450418837086045, 3.141636995139753)), ('u', 7, (0.27665329513227915, 0.20639358701078347, 0.2227248088732203)), ('u', 6, (3.1181046088741766, 0.0021592102046690117, -3.1285213458022136)), ('u', 5, (-3.0921852441269735, 0.3947927655786377, 3.1415942468560623)), ('u', 2, (3.1415122251017022, 0.03275584928687877, -3.1411353417589982)), ('u', 4, (-2.966619878409117, 3.4215350387617885e-08, 3.2842899804013013)), ('cz', 6, 5), ('u', 0, (-3.141548051954922, 0.03249920709244061, 3.1395296271679576)), ('u', 7, (0.22272481234282604, 0.3511173537588248, 0.15800993946820546)), ('u', 6, (0.013071320724860377, 0.7902924514564892, -2.1749857445456378e-08)), ('u', 5, (1.7116094205398003e-06, 0.3947928587638951, -0.029447614128055424)), ('u', 2, (3.1420500173930836, 0.02940971648979802, -3.1408404266649796)), ('u', 4, (0.14269720984164003, 2.879100755880473e-08, 0.14726041862431766)), ('cz', 5, 4), ('u', 0, (-3.143655783856359, 0.027345703916849064, 3.1375606772273215)), ('u', 7, (0.15800983204431243, 0.43823920309252273, 0.10504175242947242)), ('u', 6, (3.1415925128035482, 0.7902924864061348, 0.021359803985439463)), ('u', 5, (-0.029447717332347988, 1.5600925697278034, -0.04678493351120711)), ('u', 2, (3.1423449846850295, 0.024885924431284862, -3.140768586890696)), ('u', 4, (0.1472604315872025, 0.010083607399992922, 0.15109805594159728)), ('cz', 7, 6), ('u', 0, (-3.1456246299521164, 0.021185628864440732, 3.135835306625308)), ('u', 7, (3.2466346084025846, 1.5796661546578783, 0.011580378728456964)), ('u', 6, (-3.120232697463237, 2.879768739966685, 3.1348440222472)), ('u', 5, (-0.046785070205743386, 0.4056326085404054, 0.002375184506990438)), ('u', 2, (-3.140768795152512, 0.019373565259231323, 3.142300169777424)), ('u', 4, (0.1510978477008815, 0.015762532093672142, 0.15269082699043238)), ('cz', 6, 5), ('u', 0, (-3.1473500005943658, 0.014247799177776082, 3.1344570279934265)), ('u', 7, (-3.130012079446099, 0.008872207960859949, 3.148903141930863)), ('u', 6, (-0.006748918374636249, 1.8325974783381196, -0.02605154015202726)), ('u', 5, (-3.139217322386877, 0.00042255887548605883, 3.1407520214683635)), ('u', 2, (-3.140885241671172, 0.013095025040535484, 3.1420438569013545)), ('u', 4, (0.1526909573242352, 0.004927917258527168, 0.15043588650271156)), ('cz', 7, 6), ('u', 0, (-3.148728435722247, 0.0067827495190096185, 3.1335248220764322)), ('u', 7, (-3.1342821914188344, 0.028129641733904823, 3.139195407234359)), ('u', 6, (3.115541081546385, 0.262140876213786, -3.116429733833928)), ('u', 5, (-0.000840907442600364, 0.0004225930836379312, 0.007189232126876074)), ('u', 2, (-3.141141450070426, 0.006294436631301056, 3.1417061604825034)), ('u', 4, (-2.9911567934896848, 0.0009174423925525192, 3.2859525219007946)), ('cz', 6, 5), ('u', 0, (-0.008067571093408008, 0.000947631652682133, -0.00846216706528116)), ('u', 7, (3.1391956704722497, 1.5426670246695326, 0.0016073258126736434)), ('u', 6, (0.02516320223213957, 1.3089724159489882, -0.00656582071820867)), ('u', 5, (0.007189282899427954, 0.007842517840848119, 0.0028880269470614836)), ('u', 2, (0.00011340286348652562, 0.0007728097840692104, -0.000240879009985278)), ('u', 4, (-2.997232798093956, 0.0029487648706772516, 3.2770051945109633)), ('cz', 7, 6), ('u', 0, (-0.008462270998998336, 0.008679554484677866, -0.008239315058771884)), ('u', 7, (3.1431997551015023, 1.613540236976378, 0.008052968942624084)), ('u', 6, (-3.1481582863499415, 0.5685299684110979, 3.1155206918765828)), ('u', 5, (-1.5679083628424837, 1.5707509915693976, 1.5747175472181811)), ('u', 2, (-0.0002406189394174081, 0.007849405234238405, -0.0005440818251163691)), ('u', 4, (1.7062087148483527, 1.597774835124818, -1.5742161163239792)), ('cz', 5, 4), ('u', 0, (-0.008239206153869909, 0.016155984559545464, -0.007332514493955876)), ('u', 7, (0.008052838513088807, 0.03301089031760844, 0.006835686452027138)), ('u', 6, (3.1155205945275726, 0.5259323273395712, -3.125798377686331)), ('u', 5, (1.5747175471772599, 0.615479715722885, -1.0933194658614573)), ('u', 2, (-0.0005441989674194679, 0.01468457018831093, -0.0007309857091677702)), ('u', 4, (1.5673765373273134, 0.6154797008465961, -1.5400280847058372)), ('cz', 5, 4), ('u', 0, (-0.0073324152878501215, 0.02313481789955032, -0.005691990145514318)), ('u', 7, (0.006835375912833112, 0.02319276961434259, 0.005462251066907252)), ('u', 6, (3.1573866993336663, 2.0475425271728978, -0.0019187775552635955)), ('u', 5, (2.048273187532868, 1.5387685129310567, -1.5594679923212609)), ('u', 2, (-0.0007309271365018807, 0.021041951779091485, -0.0007404893827961025)), ('u', 4, (-1.5400280845256442, 1.5662716165591075, 1.591065342381638)), ('cz', 6, 5), ('u', 0, (-0.005692146265052731, 0.029396965666583372, -0.003285938952843994)), ('u', 7, (0.0054625371585474815, 0.013256196626928542, 0.003982673656713087)), ('u', 6, (3.13967410900957, 1.5707963701698164, -0.014377665440666654)), ('u', 5, (3.1529211212809507, 0.9553166204523242, -3.088479950265933)), ('u', 2, (-0.0007404307098160015, 0.026707380495143162, -0.0005207704482750253)), ('u', 4, (-3.1213234383275608, 0.01206942282101771, 3.153586974922128)), ('cz', 6, 5), ('u', 0, (-0.004193396363146381, 0.03904086961061197, -0.00049550232005148)), ('u', 7, (-3.1392117600713445, 0.0005483262234133876, 3.142488866062188)), ('u', 6, (3.1371272120918303, 0.0037461657141409886, -3.132170972597222)), ('u', 5, (-3.093726690620392, 0.7853981458661417, 3.141592612743864)), ('u', 2, (-0.000993259034395914, 0.035147769751032384, -0.0003747208955972169)), ('u', 4, (3.1703255251510423, 0.0496409716618151, -3.1060708565458612)), ('cz', 5, 4), ('u', 0, (-0.0004954498865382018, 0.042501802368178576, 0.0039263487131322205)), ('u', 7, (-3.140696102630624, 0.011105050147155305, 3.1410754044502793)), ('u', 6, (3.151014190354829, 1.567051569205639, -0.03353177992885881)), ('u', 5, (-6.818412173190677e-08, 0.7853981829662565, -6.261316257286463e-08)), ('u', 2, (-0.00037503346505089865, 0.03793018123688908, 0.0005501239732354355)), ('u', 4, (3.177114138123289, 0.3171364432240032, -3.103697454714576)), ('cz', 6, 5), ('u', 0, (0.003926452593371207, 0.04475061279264097, 0.009034314717243109)), ('u', 7, (-3.1421096424833355, 0.022026519589627508, 3.1398122594858933)), ('u', 6, (3.1080606857057025, 2.1002030280554758, -0.00961579530284773)), ('u', 5, (5.311327925947482e-08, 0.7853981195657265, 3.9747502027850575e-08)), ('u', 2, (0.000550124029179258, 0.03953586933873909, 0.001788159299299279)), ('u', 4, (0.03789506456208645, 1.2714054890423696, 0.014461751057602206)), ('cz', 5, 4), ('u', 0, (0.009034314759717738, 0.04580901009949549, 0.014778175510685176)), ('u', 7, (-3.143373386030864, 0.03338036152990201, 3.138785488245085)), ('u', 6, (-0.009615836838893426, 0.5215262138225559, 0.016704541600747583)), ('u', 5, (-5.127014592034207e-08, 1.0471969548697333, -6.761251225238653e-08)), ('u', 2, (0.0017884197671162879, 0.03998215933670839, 0.0033330064818323925)), ('u', 4, (-3.1271311847527232, 0.7853981645517627, 3.1415925588548226)), ('cz', 5, 4), ('u', 0, (0.014778331645939283, 0.04577123238559829, 0.02110404060006271)), ('u', 7, (-3.1444002354656693, 0.04522320030185381, 3.1380885312288216)), ('u', 6, (0.01670479529149806, 0.5200234341008141, 0.0025332547510830035)), ('u', 5, (-3.1415926097045292, 1.5707962792032464, 3.1294229390228527)), ('u', 2, (0.0033327462044416226, 0.03935447684242358, 0.005167416291726443)), ('u', 4, (-3.1415927722030283, 0.6154806146991095, 3.129423088604382)), ('u', 1, (-3.141592653589793, 1.5707963267948966, 3.141592653589793))]), (8, [('u', 3, (0.0, 2.901974084960706, 0.0)), ('u', 2, (0.0, 0.5024148995576962, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 0.5024148995576962, 3.141592653589793)), ('u', 1, (0.0, 0.9687234065335635, 0.0)), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 1), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 0.9687234065335635, 3.141592653589793)), ('u', 0, (0.0, 1.3258176636680323, 0.0)), ('u', 0, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 0), ('u', 0, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (-3.141592653589793, 1.3258176636680323, 3.141592653589793)), ('u', 4, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 5, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 6, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 6), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 0, 7), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (0.0, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.5235987755982989, 0.0)), ('u', 0, (0.0, 0.5235987755982989, 0.0)), ('cz', 0, 1), ('u', 1, (0.0, 1.5707963267948966, 0.0)), ('u', 0, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.1780972450961724, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.39269908169872414, 0.0)), ('cz', 2, 1), ('u', 1, (-3.141592653589793, 0.39269908169872414, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 1.1780972450961724, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.3089969389957472, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 0.26179938779914935, 0.0)), ('cz', 3, 2), ('u', 2, (-3.141592653589793, 0.26179938779914935, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 1.3089969389957472, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 1, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 0, 1), ('u', 1, (1.570796326794898, 0.6154797086703868, -1.570796326794898)), ('u', 0, (1.5707963267948966, 0.6154797086703876, -1.5707963267948966)), ('cz', 0, 1), ('u', 1, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 0, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 1, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.2630564724597029, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.30773985433519363, 0.0)), ('cz', 2, 1), ('u', 1, (-3.141592653589793, 0.30773985433519363, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 1.2630564724597029, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (0.0, 1.5707963267948966, 0.0)), ('u', 1, (0.0, 1.5707963267948966, 0.0)), ('cz', 0, 1), ('u', 1, (1.2251903831398114e-16, 0.7853981633974483, -1.1496735851465472e-17)), ('u', 0, (-3.141592653589793, 0.7853981633974483, 3.141592653589793)), ('cz', 0, 1), ('u', 0, (3.116033094504285, 1.6089339729985166, -3.133681864957694)), ('u', 7, (3.2967664143484856, 0.37746088244651743, -2.90770693507335)), ('u', 6, (3.131620967463964, 0.00012329162616528996, -3.149914485095441)), ('u', 5, (-3.065351981199511, 1.570796325998965, 3.141592810233495)), ('u', 2, (3.1371045869684653, 0.032420700396367345, -3.1446953815086034)), ('u', 4, (0.1678815913663732, 1.5707964120927065, -3.6217239721136885e-08)), ('cz', 5, 4), ('u', 0, (-3.1336817276541074, 0.03988000500708097, 3.1477971068631443)), ('u', 7, (3.3754782677989725, 0.2957003307021647, -2.8697445297881767)), ('u', 6, (3.1332705618836747, 0.000670964769903343, -3.14776404325472)), ('u', 5, (-1.2304880342357098e-07, 0.5235987812541554, -2.9738587354984345e-08)), ('u', 2, (3.1384898734794255, 0.03459206355040011, -3.14347395431741)), ('u', 4, (-3.5769262187636676e-08, 0.523598771695477, -0.00014103518936014022)), ('cz', 5, 4), ('u', 0, (-3.1353881481357257, 0.04020044576188537, 3.1458639800638526)), ('u', 7, (3.4134410998710325, 0.18048471998342222, -2.8492079037556315)), ('u', 6, (3.1354213213872644, 1.5700021680961975, -4.475632626022019e-05)), ('u', 5, (-7.312312765017648e-08, 1.5707963100608628, -0.08150001422869849)), ('u', 2, (3.1397114051832484, 0.035398571332497825, -3.1424570006512282)), ('u', 4, (3.141451865879188, 1.5700162024807447, -2.9682222802448033)), ('cz', 6, 5), ('u', 0, (-3.1373213273631406, 0.03904869406383659, 3.1437810459612163)), ('u', 7, (0.29238516947199533, 0.018297901744561286, 0.2766538192545155)), ('u', 6, (3.141547879107957, 1.5729530881038725, -0.023488310377399735)), ('u', 5, (3.0600922833400723, 1.9638558876725143, -3.092184870708789)), ('u', 2, (3.1407285146219714, 0.034780753328794205, -3.1416728739950286)), ('u', 4, (3.3149630865408977, 0.0008295588521163107, -2.966619969343351)), ('cz', 5, 4), ('u', 0, (-3.1394044172879125, 0.036450418837086045, 3.141636995139753)), ('u', 7, (0.27665329513227915, 0.20639358701078347, 0.2227248088732203)), ('u', 6, (3.1181046088741766, 0.0021592102046690117, -3.1285213458022136)), ('u', 5, (-3.0921852441269735, 0.3947927655786377, 3.1415942468560623)), ('u', 2, (3.1415122251017022, 0.03275584928687877, -3.1411353417589982)), ('u', 4, (-2.966619878409117, 3.4215350387617885e-08, 3.2842899804013013)), ('cz', 6, 5), ('u', 0, (-3.141548051954922, 0.03249920709244061, 3.1395296271679576)), ('u', 7, (0.22272481234282604, 0.3511173537588248, 0.15800993946820546)), ('u', 6, (0.013071320724860377, 0.7902924514564892, -2.1749857445456378e-08)), ('u', 5, (1.7116094205398003e-06, 0.3947928587638951, -0.029447614128055424)), ('u', 2, (3.1420500173930836, 0.02940971648979802, -3.1408404266649796)), ('u', 4, (0.14269720984164003, 2.879100755880473e-08, 0.14726041862431766)), ('cz', 5, 4), ('u', 0, (-3.143655783856359, 0.027345703916849064, 3.1375606772273215)), ('u', 7, (0.15800983204431243, 0.43823920309252273, 0.10504175242947242)), ('u', 6, (3.1415925128035482, 0.7902924864061348, 0.021359803985439463)), ('u', 5, (-0.029447717332347988, 1.5600925697278034, -0.04678493351120711)), ('u', 2, (3.1423449846850295, 0.024885924431284862, -3.140768586890696)), ('u', 4, (0.1472604315872025, 0.010083607399992922, 0.15109805594159728)), ('cz', 7, 6), ('u', 0, (-3.1456246299521164, 0.021185628864440732, 3.135835306625308)), ('u', 7, (3.2466346084025846, 1.5796661546578783, 0.011580378728456964)), ('u', 6, (-3.120232697463237, 2.879768739966685, 3.1348440222472)), ('u', 5, (-0.046785070205743386, 0.4056326085404054, 0.002375184506990438)), ('u', 2, (-3.140768795152512, 0.019373565259231323, 3.142300169777424)), ('u', 4, (0.1510978477008815, 0.015762532093672142, 0.15269082699043238)), ('cz', 6, 5), ('u', 0, (-3.1473500005943658, 0.014247799177776082, 3.1344570279934265)), ('u', 7, (-3.130012079446099, 0.008872207960859949, 3.148903141930863)), ('u', 6, (-0.006748918374636249, 1.8325974783381196, -0.02605154015202726)), ('u', 5, (-3.139217322386877, 0.00042255887548605883, 3.1407520214683635)), ('u', 2, (-3.140885241671172, 0.013095025040535484, 3.1420438569013545)), ('u', 4, (0.1526909573242352, 0.004927917258527168, 0.15043588650271156)), ('cz', 7, 6), ('u', 0, (-3.148728435722247, 0.0067827495190096185, 3.1335248220764322)), ('u', 7, (-3.1342821914188344, 0.028129641733904823, 3.139195407234359)), ('u', 6, (3.115541081546385, 0.262140876213786, -3.116429733833928)), ('u', 5, (-0.000840907442600364, 0.0004225930836379312, 0.007189232126876074)), ('u', 2, (-3.141141450070426, 0.006294436631301056, 3.1417061604825034)), ('u', 4, (-2.9911567934896848, 0.0009174423925525192, 3.2859525219007946)), ('cz', 6, 5), ('u', 0, (-0.008067571093408008, 0.000947631652682133, -0.00846216706528116)), ('u', 7, (3.1391956704722497, 1.5426670246695326, 0.0016073258126736434)), ('u', 6, (0.02516320223213957, 1.3089724159489882, -0.00656582071820867)), ('u', 5, (0.007189282899427954, 0.007842517840848119, 0.0028880269470614836)), ('u', 2, (0.00011340286348652562, 0.0007728097840692104, -0.000240879009985278)), ('u', 4, (-2.997232798093956, 0.0029487648706772516, 3.2770051945109633)), ('cz', 7, 6), ('u', 0, (-0.008462270998998336, 0.008679554484677866, -0.008239315058771884)), ('u', 7, (3.1431997551015023, 1.613540236976378, 0.008052968942624084)), ('u', 6, (-3.1481582863499415, 0.5685299684110979, 3.1155206918765828)), ('u', 5, (-1.5679083628424837, 1.5707509915693976, 1.5747175472181811)), ('u', 2, (-0.0002406189394174081, 0.007849405234238405, -0.0005440818251163691)), ('u', 4, (1.7062087148483527, 1.597774835124818, -1.5742161163239792)), ('cz', 5, 4), ('u', 0, (-0.008239206153869909, 0.016155984559545464, -0.007332514493955876)), ('u', 7, (0.008052838513088807, 0.03301089031760844, 0.006835686452027138)), ('u', 6, (3.1155205945275726, 0.5259323273395712, -3.125798377686331)), ('u', 5, (1.5747175471772599, 0.615479715722885, -1.0933194658614573)), ('u', 2, (-0.0005441989674194679, 0.01468457018831093, -0.0007309857091677702)), ('u', 4, (1.5673765373273134, 0.6154797008465961, -1.5400280847058372)), ('cz', 5, 4), ('u', 0, (-0.0073324152878501215, 0.02313481789955032, -0.005691990145514318)), ('u', 7, (0.006835375912833112, 0.02319276961434259, 0.005462251066907252)), ('u', 6, (3.1573866993336663, 2.0475425271728978, -0.0019187775552635955)), ('u', 5, (2.048273187532868, 1.5387685129310567, -1.5594679923212609)), ('u', 2, (-0.0007309271365018807, 0.021041951779091485, -0.0007404893827961025)), ('u', 4, (-1.5400280845256442, 1.5662716165591075, 1.591065342381638)), ('cz', 6, 5), ('u', 0, (-0.005692146265052731, 0.029396965666583372, -0.003285938952843994)), ('u', 7, (0.0054625371585474815, 0.013256196626928542, 0.003982673656713087)), ('u', 6, (3.13967410900957, 1.5707963701698164, -0.014377665440666654)), ('u', 5, (3.1529211212809507, 0.9553166204523242, -3.088479950265933)), ('u', 2, (-0.0007404307098160015, 0.026707380495143162, -0.0005207704482750253)), ('u', 4, (-3.1213234383275608, 0.01206942282101771, 3.153586974922128)), ('cz', 6, 5), ('u', 0, (-0.004193396363146381, 0.03904086961061197, -0.00049550232005148)), ('u', 7, (-3.1392117600713445, 0.0005483262234133876, 3.142488866062188)), ('u', 6, (3.1371272120918303, 0.0037461657141409886, -3.132170972597222)), ('u', 5, (-3.093726690620392, 0.7853981458661417, 3.141592612743864)), ('u', 2, (-0.000993259034395914, 0.035147769751032384, -0.0003747208955972169)), ('u', 4, (3.1703255251510423, 0.0496409716618151, -3.1060708565458612)), ('cz', 5, 4), ('u', 0, (-0.0004954498865382018, 0.042501802368178576, 0.0039263487131322205)), ('u', 7, (-3.140696102630624, 0.011105050147155305, 3.1410754044502793)), ('u', 6, (3.151014190354829, 1.567051569205639, -0.03353177992885881)), ('u', 5, (-6.818412173190677e-08, 0.7853981829662565, -6.261316257286463e-08)), ('u', 2, (-0.00037503346505089865, 0.03793018123688908, 0.0005501239732354355)), ('u', 4, (3.177114138123289, 0.3171364432240032, -3.103697454714576)), ('cz', 6, 5), ('u', 0, (0.003926452593371207, 0.04475061279264097, 0.009034314717243109)), ('u', 7, (-3.1421096424833355, 0.022026519589627508, 3.1398122594858933)), ('u', 6, (3.1080606857057025, 2.1002030280554758, -0.00961579530284773)), ('u', 5, (5.311327925947482e-08, 0.7853981195657265, 3.9747502027850575e-08)), ('u', 2, (0.000550124029179258, 0.03953586933873909, 0.001788159299299279)), ('u', 4, (0.03789506456208645, 1.2714054890423696, 0.014461751057602206)), ('cz', 5, 4), ('u', 0, (0.009034314759717738, 0.04580901009949549, 0.014778175510685176)), ('u', 7, (-3.143373386030864, 0.03338036152990201, 3.138785488245085)), ('u', 6, (-0.009615836838893426, 0.5215262138225559, 0.016704541600747583)), ('u', 5, (-5.127014592034207e-08, 1.0471969548697333, -6.761251225238653e-08)), ('u', 2, (0.0017884197671162879, 0.03998215933670839, 0.0033330064818323925)), ('u', 4, (-3.1271311847527232, 0.7853981645517627, 3.1415925588548226)), ('cz', 5, 4), ('u', 0, (0.014778331645939283, 0.04577123238559829, 0.02110404060006271)), ('u', 7, (-3.1444002354656693, 0.04522320030185381, 3.1380885312288216)), ('u', 6, (0.01670479529149806, 0.5200234341008141, 0.0025332547510830035)), ('u', 5, (-3.1415926097045292, 1.5707962792032464, 3.1294229390228527)), ('u', 2, (0.0033327462044416226, 0.03935447684242358, 0.005167416291726443)), ('u', 4, (-3.1415927722030283, 0.6154806146991095, 3.129423088604382)), ('u', 1, (-3.141592653589793, 1.5707963267948966, 3.141592653589793))])], (9, 2): [(9, [('u', 3, (0.0, 2.031350318476219, 0.0)), ('u', 2, (0.0, 1.0697033135295393, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.0697033135295393, 3.141592653589793)), ('u', 7, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 7), ('u', 3, (-1.1751575440460897, 0.26067405538424787, -0.6871036784333111)), ('u', 2, (-2.904130104115435, 1.2877711084423432e-09, 2.934059522144068)), ('u', 1, (-2.9086275139335025, 1.0453477160710642, 3.153321898055499)), ('u', 0, (0.26021582247647235, 0.23328645359957653, 0.5640517095927046)), ('u', 8, (-3.372168548020394, 1.5707965990295842, 2.6266535483509807)), ('cz', 2, 8), ('u', 3, (-4.067784972759647, 0.18118027078371085, 1.9687154621961178)), ('u', 2, (0.6657778834497072, 3.1415924348112156, -0.3848408382298367)), ('u', 1, (0.4128655706052585, 0.29169992513927007, -0.048757596765475836)), ('u', 0, (-0.01538434204453374, 0.2152073564927922, -0.10649516288155722)), ('u', 8, (3.600940580816046, 1.9302884105856912, 0.062230185539184024)), ('cz', 2, 1), ('u', 3, (2.2110636077782657, 1.4366927582046864, 0.05927407796264106)), ('u', 2, (2.8501197239585907, 1.5707961081228536, -2.218612322475263)), ('u', 1, (3.2730549986716997, 0.6886101178919402, -2.575993276237892)), ('u', 0, (0.33564578704341586, 0.34380107009482985, 0.2686060068208995)), ('u', 8, (3.197192101971976, 0.46678376334754723, -2.8464804259837937)), ('cz', 3, 2), ('u', 3, (3.1810500031135835, 2.6883966429925574, -0.38800367167046135)), ('u', 2, (-2.493776771582644, 1.5707962109638427, 2.51030076068191)), ('u', 1, (0.9999694117196286, 0.4176562437414257, 0.20160956604139202)), ('u', 0, (3.6808361043184217, 0.1668222398191245, -2.4812094831857348)), ('u', 8, (0.24504038097329958, 0.047348615295652756, 0.3719166684254209)), ('cz', 2, 1), ('u', 3, (3.230091617416889, 0.6067888110944931, 0.4002632658023271)), ('u', 2, (-1.4631032074158488, 1.57079664710389, -0.14921893023630595)), ('u', 1, (-2.6839994517778503, 1.0495876640777395, 3.460245914604988)), ('u', 0, (0.7187858549182815, 0.037212705319108294, 0.12527987749719643)), ('u', 8, (3.2479004661525797, 0.1907831125143074, -3.0242116567792574)), ('cz', 3, 2), ('u', 3, (3.1985339495323744, 1.2421178686929215, 0.7685533630154877)), ('u', 2, (-2.9302892955456565, 0.40750919585425915, 2.9472548692323572)), ('u', 1, (-1.6084257572840468, 2.0207749134588644, 1.994304461676151)), ('u', 0, (1.8809384577251094, 1.6092895321161782, -1.5211683708909391)), ('u', 8, (0.09200442112359732, 0.16982459029422026, -0.09778032844482153)), ('cz', 1, 0), ('u', 3, (-2.2568501372182834, 0.5240394855151633, 3.002106847942613)), ('u', 2, (-3.072655095440928, 0.3455044194612774, 3.0252096244217404)), ('u', 1, (1.7738837963033383, 0.6154796975274284, -1.3202384912232124)), ('u', 0, (1.2641821205066717, 1.5955335708882186, -1.6258849719860065)), ('u', 8, (0.21689653646156265, 0.20357238920732112, 0.36478729690880035)), ('cz', 2, 1), ('u', 3, (0.1562798261086924, 0.16400185626187547, 0.013634776231117654)), ('u', 2, (3.1243115228579392, 0.13718296059318005, -3.027750293623534)), ('u', 1, (3.5065144559822388, 1.5707965873327723, -2.111596372205712)), ('u', 0, (0.24866742049844764, 0.04413150060343756, -0.35796327032862846)), ('u', 8, (-0.060059179136186525, 0.12055765815950181, 0.5683452214035626)), ('cz', 1, 0), ('u', 3, (-0.28236679701589895, 0.19307427245131378, 0.11749337831775906)), ('u', 2, (3.2175722375293705, 2.051762696906425, 0.20621420836491122)), ('u', 1, (0.5408000122591998, 1.5707964395303073, -0.799879876487902)), ('u', 0, (3.405057351876788, 0.34159515819165936, -2.624131222806333)), ('u', 8, (3.045772691560537, 0.06817211668520294, -2.845739293405762)), ('cz', 2, 1), ('u', 8, (-1.1360710551574318, 1.505239968654346, -4.9016265742176195)), ('u', 7, (-3.1415926600046964, 1.570796324839406, -0.41699610807162846)), ('u', 2, (3.1147814607134094, 1.721597437073412, -0.6333155888908188)), ('u', 6, (-0.32050837989764996, 2.2555155331604326, -0.3438787166564868)), ('u', 5, (0.17174521324562275, 0.031146318806421287, 1.4509660709986316)), ('cz', 7, 6), ('u', 8, (0.4518976427418188, 0.5380965465711559, -0.7076166725272257)), ('u', 7, (2.377563839793169, 1.5707963276492578, -3.5920506340583915)), ('u', 2, (-2.68824546699976, 0.24094033773879892, 3.237162941690255)), ('u', 6, (-2.4810190398761565, 2.000382542730125, 3.1271194163022438)), ('u', 5, (1.5432381968531852, 0.3953680477116938, 0.49343335857450055)), ('cz', 8, 7), ('u', 8, (-3.4986968589449665, 0.42101655926489817, 2.697973728216149)), ('u', 7, (2.613559354625364, 0.7303878192993729, -2.7116651640459857)), ('u', 2, (0.08318659923227986, 0.0589607804817785, -0.1756097494762521)), ('u', 6, (2.4239619553838874, 0.36922021676920536, -2.812224339696096)), ('u', 5, (-3.1398940978223853, 0.33785932587456824, 3.1167451417159167)), ('cz', 7, 6), ('u', 8, (2.9667798658034155, 1.6993058586079814, -0.27691705791021914)), ('u', 7, (0.4062211191842141, 1.5707963256045296, -0.10975701098500565)), ('u', 2, (-3.2830295374537153, 0.560642007824093, 2.996823018344159)), ('u', 6, (-0.7133892649258873, 2.5020118574577993e-08, -0.1758884761136349)), ('u', 5, (-3.0653246916751273, 0.3576152282275567, 3.085728183937486)), ('cz', 8, 7), ('u', 8, (2.933480189689565, 0.27753841802099527, -3.3484508810627425)), ('u', 7, (3.593755397978441, 1.0317532409179107e-08, -2.3057648178684724)), ('u', 2, (-0.5662649656646794, 0.40579970829580525, 0.08065303252572728)), ('u', 6, (2.9916587456425994, 1.4721995737462234e-08, -3.1972097818947)), ('u', 5, (-1.157118088638148, 0.6510021827888648, -0.45425267384030044)), ('cz', 6, 5), ('u', 8, (3.375107406833436, 0.728719169615773, -2.6564227900838295)), ('u', 7, (4.717882284688699, 2.9014631877375603e-08, 0.03499669314220544)), ('u', 2, (3.2566859996186186, 0.13634784137903486, -2.731561020643601)), ('u', 6, (-1.4741812562965175, 1.5707963299940688, 3.8850569601487615)), ('u', 5, (2.944475558010268, 0.7947711174071349, -3.3245539777200115)), ('cz', 7, 6), ('u', 8, (0.3510188461157579, 0.022763173674389754, -0.18008573438180525)), ('u', 7, (-3.0310357172669966, 0.503129628002097, 2.7062472045923314)), ('u', 2, (-0.3376220458028091, 0.09688717601731367, 0.09132756359967983)), ('u', 6, (-2.3142606340546887, 2.1573716490417874, 2.198652747023293)), ('u', 5, (-1.0670745154961903, 0.03110142674163011, -0.5236146847714218)), ('cz', 6, 5), ('u', 8, (0.4671096828828962, 0.008090946775635946, 0.1684230948562603)), ('u', 7, (3.785692462019081, 1.6033891770995943, -0.4677534521033184)), ('u', 2, (-0.011921597402141097, 0.6455752830311845, 0.07236214366382598)), ('u', 6, (-0.10425764881894511, 1.570796340093249, 0.732992731562423)), ('u', 5, (-1.4589561764307943, 0.15632034326211913, -0.4613416507330019)), ('cz', 7, 6), ('u', 8, (0.3054932649597624, 0.7349006554385038, -0.44790492963445244)), ('u', 7, (2.9439525993830005, 1.8998281046082481, -0.9263836084918353)), ('u', 2, (2.7064458565936973, 0.304221399010612, -2.9313223612746593)), ('u', 6, (0.05687524151417256, 0.41419353368705036, -0.6369174662855737)), ('u', 5, (-3.0402980869294605, 1.7172409046188886, 1.5578030228290438)), ('u', 3, (-2.7877311679092527, 0.5131360297009971, 2.4612435316960513)), ('u', 1, (-0.7709163438843996, 0.7853981682079562, -1.3106286904277253)), ('u', 0, (0.2231414472615762, 1.8274158817631607, -0.7523404924869557)), ('cz', 1, 0), ('u', 3, (0.07311385607457008, 0.3503418608868059, 0.31347456825108533)), ('u', 1, (-1.8309640698265242, 1.5707964356112187, 2.4580318925890987)), ('u', 0, (-3.7320491347325393, 1.5707968395486716, 2.4580318218348425)), ('u', 4, (2.6611157981787685, 0.3348491692101108, -3.478146822551501)), ('u', 4, (0.025099278002302938, 0.020289187327979002, 1.0994964759651373)), ('u', 4, (1.1856425386986325, 0.10635862335364886, -0.2028633457911238)), ('u', 4, (-0.502874265909908, 0.20858030829480428, -0.3670399133532716)), ('u', 4, (2.2511667393630503, 1.2655340329567368, -1.360636025018449)), ('cz', 4, 5), ('u', 4, (1.3977632976181833, 0.6525439868142484, -1.121158885295027)), ('u', 5, (1.296487072623602, 0.6154797086700415, -1.2957682456694826)), ('u', 6, (-0.071151212705162, 0.7854015269467203, -0.6100752785286383)), ('u', 7, (2.8446334516331584, 0.7676417279941344, -2.574966225015541)), ('cz', 4, 5), ('u', 4, (-1.861323235743385, 2.3325757957288817, 1.8904113388852222)), ('u', 5, (2.1957840870535623, 1.9503858232632065, -1.7356342087318746)), ('u', 6, (2.363249945498356, 1.3411433722001864, 0.6795535715525908)), ('u', 7, (-3.2164691598207225, 0.08261866080509377, 2.1635538126647655)), ('cz', 5, 6), ('u', 4, (-2.4502383748008967, 0.5066738472749666, 2.658848459855456)), ('u', 5, (-0.1211349652721937, 0.9553166181244168, 0.9193737914269516)), ('u', 6, (4.017787111955051, 1.5707963267950278, -0.14625006961981346)), ('u', 7, (2.5641160141141413, 0.04650871485508044, -3.241997417780418)), ('cz', 5, 6), ('u', 4, (-2.777227076724675, 0.8219273046226232, 2.9033750331038646)), ('u', 5, (3.6497578309532717, 1.5707963267956213, -2.3607616933088007)), ('u', 6, (-0.43303647828505254, 0.23307421193024305, 0.34113733371999544)), ('u', 7, (-3.462869472210422, 0.04369737122806576, 2.659609116246114)), ('cz', 4, 5), ('u', 4, (3.5342993639707627, 1.570796326793336, -1.5745998474870015)), ('u', 5, (-2.3516272870779646, 2.3561944901923084, -2.942884153984175)), ('u', 6, (2.821397764753086, 1.4231330404147982, 0.26950514138026715)), ('u', 7, (2.592944263744413, 0.12303314857564963, -3.235575396438218)), ('cz', 4, 5), ('u', 4, (3.145396174280426, 1.5707963267946714, -2.914376451603585)), ('u', 5, (2.9428841539840684, 1.5707963267948422, -2.9143764516023642)), ('u', 6, (0.768553377240472, 0.663097505060459, 0.7319728725594956)), ('u', 7, (1.6634127072052856, 0.24170758579466498, -3.039721204637301)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0))]), (9, [('u', 3, (0.0, 2.031350318476219, 0.0)), ('u', 2, (0.0, 1.0697033135295393, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.0697033135295393, 3.141592653589793)), ('u', 7, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 7), ('u', 3, (-1.1751575440460897, 0.26067405538424787, -0.6871036784333111)), ('u', 2, (-2.904130104115435, 1.2877711084423432e-09, 2.934059522144068)), ('u', 1, (-2.9086275139335025, 1.0453477160710642, 3.153321898055499)), ('u', 0, (0.26021582247647235, 0.23328645359957653, 0.5640517095927046)), ('u', 8, (-3.372168548020394, 1.5707965990295842, 2.6266535483509807)), ('cz', 2, 8), ('u', 3, (-4.067784972759647, 0.18118027078371085, 1.9687154621961178)), ('u', 2, (0.6657778834497072, 3.1415924348112156, -0.3848408382298367)), ('u', 1, (0.4128655706052585, 0.29169992513927007, -0.048757596765475836)), ('u', 0, (-0.01538434204453374, 0.2152073564927922, -0.10649516288155722)), ('u', 8, (3.600940580816046, 1.9302884105856912, 0.062230185539184024)), ('cz', 2, 1), ('u', 3, (2.2110636077782657, 1.4366927582046864, 0.05927407796264106)), ('u', 2, (2.8501197239585907, 1.5707961081228536, -2.218612322475263)), ('u', 1, (3.2730549986716997, 0.6886101178919402, -2.575993276237892)), ('u', 0, (0.33564578704341586, 0.34380107009482985, 0.2686060068208995)), ('u', 8, (3.197192101971976, 0.46678376334754723, -2.8464804259837937)), ('cz', 3, 2), ('u', 3, (3.1810500031135835, 2.6883966429925574, -0.38800367167046135)), ('u', 2, (-2.493776771582644, 1.5707962109638427, 2.51030076068191)), ('u', 1, (0.9999694117196286, 0.4176562437414257, 0.20160956604139202)), ('u', 0, (3.6808361043184217, 0.1668222398191245, -2.4812094831857348)), ('u', 8, (0.24504038097329958, 0.047348615295652756, 0.3719166684254209)), ('cz', 2, 1), ('u', 3, (3.230091617416889, 0.6067888110944931, 0.4002632658023271)), ('u', 2, (-1.4631032074158488, 1.57079664710389, -0.14921893023630595)), ('u', 1, (-2.6839994517778503, 1.0495876640777395, 3.460245914604988)), ('u', 0, (0.7187858549182815, 0.037212705319108294, 0.12527987749719643)), ('u', 8, (3.2479004661525797, 0.1907831125143074, -3.0242116567792574)), ('cz', 3, 2), ('u', 3, (3.1985339495323744, 1.2421178686929215, 0.7685533630154877)), ('u', 2, (-2.9302892955456565, 0.40750919585425915, 2.9472548692323572)), ('u', 1, (-1.6084257572840468, 2.0207749134588644, 1.994304461676151)), ('u', 0, (1.8809384577251094, 1.6092895321161782, -1.5211683708909391)), ('u', 8, (0.09200442112359732, 0.16982459029422026, -0.09778032844482153)), ('cz', 1, 0), ('u', 3, (-2.2568501372182834, 0.5240394855151633, 3.002106847942613)), ('u', 2, (-3.072655095440928, 0.3455044194612774, 3.0252096244217404)), ('u', 1, (1.7738837963033383, 0.6154796975274284, -1.3202384912232124)), ('u', 0, (1.2641821205066717, 1.5955335708882186, -1.6258849719860065)), ('u', 8, (0.21689653646156265, 0.20357238920732112, 0.36478729690880035)), ('cz', 2, 1), ('u', 3, (0.1562798261086924, 0.16400185626187547, 0.013634776231117654)), ('u', 2, (3.1243115228579392, 0.13718296059318005, -3.027750293623534)), ('u', 1, (3.5065144559822388, 1.5707965873327723, -2.111596372205712)), ('u', 0, (0.24866742049844764, 0.04413150060343756, -0.35796327032862846)), ('u', 8, (-0.060059179136186525, 0.12055765815950181, 0.5683452214035626)), ('cz', 1, 0), ('u', 3, (-0.28236679701589895, 0.19307427245131378, 0.11749337831775906)), ('u', 2, (3.2175722375293705, 2.051762696906425, 0.20621420836491122)), ('u', 1, (0.5408000122591998, 1.5707964395303073, -0.799879876487902)), ('u', 0, (3.405057351876788, 0.34159515819165936, -2.624131222806333)), ('u', 8, (3.045772691560537, 0.06817211668520294, -2.845739293405762)), ('cz', 2, 1), ('u', 3, (-2.7877311679092527, 0.5131360297009971, 2.4612435316960513)), ('u', 2, (2.7309235597960666, 1.9679245926489324, 0.005706386482318138)), ('u', 1, (-0.7709163438843996, 0.7853981682079562, -1.3106286904277253)), ('u', 0, (0.2231414472615762, 1.8274158817631607, -0.7523404924869557)), ('u', 8, (3.4861799926859622, 0.01496747952486012, -2.645700594264318)), ('cz', 1, 0), ('u', 3, (0.07311385607457008, 0.3503418608868059, 0.31347456825108533)), ('u', 2, (2.711866938377728, 0.4312735279598157, -3.435996990272945)), ('u', 1, (-1.8309640698265242, 1.5707964356112187, 2.4580318925890987)), ('u', 0, (-3.7320491347325393, 1.5707968395486716, 2.4580318218348425)), ('u', 8, (2.880664166591831, 1.6294531686526075, 0.17219468854533693)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 6, (0.0, 1.5707963267948966, 0.0)), ('u', 7, (0.0, 1.5707963267948966, 0.0)), ('cz', 6, 7), ('u', 7, (2.983475401827938e-18, 0.886077123792614, -1.140057778643436e-16)), ('u', 6, (-3.141592653589793, 0.886077123792614, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 6, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (-3.141592653589793, 1.1071487177940904, 3.141592653589793)), ('cz', 8, 7), ('u', 7, (0.0, 1.1071487177940904, 0.0)), ('u', 7, (-3.141592653589793, 2.677945044588987, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (0.0, 2.677945044588987, 0.0)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 4, (2.6611157981787685, 0.3348491692101108, -3.478146822551501)), ('u', 5, (-0.6924935761410032, 0.785398163397586, -0.053508259482247134)), ('u', 6, (0.0, 3.2169798970235493e-13, -0.0718275583082407)), ('u', 7, (0.21662944741760443, 0.009419871126993384, 0.2605984439252987)), ('cz', 5, 6), ('u', 4, (0.025099278002302938, 0.020289187327979002, 1.0994964759651373)), ('u', 5, (2.5315883843221796, 1.126852592477393, -3.148371794074219)), ('u', 6, (-1.4998309724843017, 1.5707963267947973, 3.6496660852705918)), ('u', 7, (4.234308233013136, 1.572828927799284, 0.2331611443336068)), ('cz', 6, 7), ('u', 4, (1.1856425386986325, 0.10635862335364886, -0.2028633457911238)), ('u', 5, (-0.8219614493886984, 0.6321024525621387, -0.5878027940943908)), ('u', 6, (-2.0788697584754736, 1.570796326795042, 2.347021439424257)), ('u', 7, (-3.0080223501098153, 0.5096791005494581, 2.4845323181052326)), ('cz', 5, 6), ('u', 4, (-0.502874265909908, 0.20858030829480428, -0.3670399133532716)), ('u', 5, (-0.9707185567135301, 0.13951846091143721, -0.4780125605259198)), ('u', 6, (-0.2526263370310758, 1.5707963267949179, 0.3004834247059334)), ('u', 7, (3.4904687242914436, 1.6114464165634266, -0.4680790966247721)), ('cz', 6, 7), ('u', 4, (2.2511667393630503, 1.2655340329567368, -1.360636025018449)), ('u', 5, (-2.7157115308499935, 1.7101276920868471, 1.6054345492242823)), ('u', 6, (0.13739192132802322, 0.7397973392342709, -0.22411577638408456)), ('u', 7, (3.388780904747323, 2.07951450118411, -0.5454979476806319)), ('cz', 4, 5), ('u', 4, (1.3977632976181833, 0.6525439868142484, -1.121158885295027)), ('u', 5, (1.296487072623602, 0.6154797086700415, -1.2957682456694826)), ('u', 6, (-0.071151212705162, 0.7854015269467203, -0.6100752785286383)), ('u', 7, (2.8446334516331584, 0.7676417279941344, -2.574966225015541)), ('cz', 4, 5), ('u', 4, (-1.861323235743385, 2.3325757957288817, 1.8904113388852222)), ('u', 5, (2.1957840870535623, 1.9503858232632065, -1.7356342087318746)), ('u', 6, (2.363249945498356, 1.3411433722001864, 0.6795535715525908)), ('u', 7, (-3.2164691598207225, 0.08261866080509377, 2.1635538126647655)), ('cz', 5, 6), ('u', 4, (-2.4502383748008967, 0.5066738472749666, 2.658848459855456)), ('u', 5, (-0.1211349652721937, 0.9553166181244168, 0.9193737914269516)), ('u', 6, (4.017787111955051, 1.5707963267950278, -0.14625006961981346)), ('u', 7, (2.5641160141141413, 0.04650871485508044, -3.241997417780418)), ('cz', 5, 6), ('u', 4, (-2.777227076724675, 0.8219273046226232, 2.9033750331038646)), ('u', 5, (3.6497578309532717, 1.5707963267956213, -2.3607616933088007)), ('u', 6, (-0.43303647828505254, 0.23307421193024305, 0.34113733371999544)), ('u', 7, (-3.462869472210422, 0.04369737122806576, 2.659609116246114)), ('cz', 4, 5), ('u', 4, (3.5342993639707627, 1.570796326793336, -1.5745998474870015)), ('u', 5, (-2.3516272870779646, 2.3561944901923084, -2.942884153984175)), ('u', 6, (2.821397764753086, 1.4231330404147982, 0.26950514138026715)), ('u', 7, (2.592944263744413, 0.12303314857564963, -3.235575396438218)), ('cz', 4, 5), ('u', 4, (3.145396174280426, 1.5707963267946714, -2.914376451603585)), ('u', 5, (2.9428841539840684, 1.5707963267948422, -2.9143764516023642)), ('u', 6, (0.768553377240472, 0.663097505060459, 0.7319728725594956)), ('u', 7, (1.6634127072052856, 0.24170758579466498, -3.039721204637301)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0))])], (9, 3): [(9, [('u', 3, (0.0, 2.437045235136948, 0.0)), ('u', 2, (0.0, 0.8259832557410993, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 0.8259832557410993, 3.141592653589793)), ('u', 1, (0.0, 1.220690549027751, 0.0)), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 1), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.220690549027751, 3.141592653589793)), ('u', 6, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 6), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 7), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (0.0, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.5235987755982989, 0.0)), ('u', 0, (0.0, 0.5235987755982989, 0.0)), ('cz', 0, 1), ('u', 1, (0.0, 1.5707963267948966, 0.0)), ('u', 0, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 0.7853981633974484, 3.141592653589793)), ('cz', 2, 1), ('u', 1, (0.0, 0.7853981633974484, 0.0)), ('u', 1, (-3.141592653589793, 2.356194490192345, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 2.356194490192345, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.3089969389957472, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 0.26179938779914935, 0.0)), ('cz', 3, 2), ('u', 2, (-3.141592653589793, 0.26179938779914935, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 1.3089969389957472, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 1, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 0, 1), ('u', 1, (1.570796326794898, 0.6154797086703868, -1.570796326794898)), ('u', 0, (1.5707963267948966, 0.6154797086703876, -1.5707963267948966)), ('cz', 0, 1), ('u', 1, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 0, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 1, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.2630564724597029, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.30773985433519363, 0.0)), ('cz', 2, 1), ('u', 1, (-3.141592653589793, 0.30773985433519363, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 1.2630564724597029, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (0.0, 1.5707963267948966, 0.0)), ('u', 1, (0.0, 1.5707963267948966, 0.0)), ('cz', 0, 1), ('u', 1, (1.2251903831398114e-16, 0.7853981633974483, -1.1496735851465472e-17)), ('u', 0, (-3.141592653589793, 0.7853981633974483, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 0, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 5, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 6, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 5, 6), ('u', 6, (1.570796326794898, 0.6847192030022828, -1.570796326794898)), ('u', 5, (1.5707963267948966, 0.6847192030022828, -1.5707963267948966)), ('cz', 5, 6), ('u', 6, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 5, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 6, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 6, 7), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 6, (-3.141592653589793, 0.8860771237926138, 3.141592653589793)), ('cz', 7, 6), ('u', 6, (0.0, 0.8860771237926138, 0.0)), ('u', 6, (-3.141592653589793, 2.4568734505875103, 3.141592653589793)), ('cz', 5, 6), ('u', 6, (0.0, 2.4568734505875103, 0.0)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 6, 7), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (0.0, 0.23182380450040305, 0.0)), ('cz', 8, 7), ('u', 7, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (0.0, 1.3389725222944935, 0.0)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 4, (-3.72954931593578, 0.8083685852213042, 0.1012430165969731)), ('u', 5, (-1.6916101486795811, 1.5707962345027202, 1.561365268358305)), ('u', 6, (1.449982559454445, 1.5707964073305447, -2.1412378496564703)), ('u', 7, (4.877296674871044, 0.5599439603158379, 0.9016199936380311)), ('cz', 5, 6), ('u', 4, (-0.2607020379710159, 1.8746338942408387, 0.3401729574974718)), ('u', 5, (2.0304752150449596, 2.7163814823405175, -3.284857907484258)), ('u', 6, (-1.0003548225177026, 0.7853981967687228, -0.3816822225789889)), ('u', 7, (1.9062349350608785, 2.105522637512003, -4.350972530764135)), ('cz', 6, 7), ('u', 4, (-0.4512849648783269, 0.08135033979440123, -0.2958884840534576)), ('u', 5, (0.48264868358259083, 1.9622358053620443, 0.21816332608183955)), ('u', 6, (-2.7599104371680427, 1.570796424144148, 2.987079411113781)), ('u', 7, (2.373264586383387, 0.6381935426014714, 0.008766231874767438)), ('cz', 5, 6), ('u', 4, (-0.7922683626848462, 0.535321820802116, -0.6053047073228816)), ('u', 5, (-0.21582878194631516, 0.29491392756227447, -1.908785970763928)), ('u', 6, (-3.1338222889370444, 1.0518638057176164, 2.9995127172346114)), ('u', 7, (2.0596422969937582, 1.2839330821498418, -2.427419892376809)), ('cz', 6, 7), ('u', 4, (-3.5644176982696854, 0.8006553625797103, 2.3521914693749295)), ('u', 5, (-1.4086420991197082, 1.3370697667400062, -0.21687268764352163)), ('u', 6, (3.3568460579956345, 1.570796315412032, -2.7980627980812334)), ('u', 7, (-0.7119361753800968, 5.5809380163208686e-08, 0.9000777535147246)), ('cz', 5, 6), ('u', 4, (2.3257670217661963, 0.841735886985317, -3.0300492534733414)), ('u', 5, (-0.9510123568292165, 1.4123384873657387, 0.3130192043823531)), ('u', 6, (-0.34352987932929163, 1.5707962977350383, 1.058471195503163)), ('u', 7, (0.7265803385858718, 3.119036931642264e-08, 2.7450848409367463)), ('cz', 6, 7), ('u', 4, (2.0000604390750887, 2.574471630733329, -2.0008615944075423)), ('u', 5, (-1.9621601279570933, 2.026228706362223, 1.3519437664298017)), ('u', 6, (2.2650188245937035, 1.9167454764681338, -3.1601552710404825)), ('u', 7, (2.7940623882253157, 0.902045787866622, -0.7115949661478131)), ('cz', 4, 5), ('u', 4, (0.36805322904690185, 0.9580109278616113, -1.9723857468414405)), ('u', 5, (1.6130270152232584, 1.5707963564816858, -1.4491135360350413)), ('u', 6, (3.2040867371190416, 2.0171565420009614, -2.6790084778349907)), ('u', 7, (-0.02663118290327908, 0.4388791870629602, 0.05277341143097515)), ('cz', 5, 6), ('u', 4, (-2.331153110416769, 0.0068119239725258985, 3.775460997396633)), ('u', 5, (-2.4778772623013356, 1.5707964271423986, 3.330005827585186)), ('u', 6, (-0.3483948146131367, 0.018212217767981397, 2.185328320773037)), ('u', 7, (-2.8102024409281654, 0.012305487986607142, 3.174913500381213)), ('cz', 4, 5), ('u', 4, (0.4960106122640332, 0.8602374778048759, 1.3044739321438055)), ('u', 5, (1.350646002028718, 2.355942645811947, 0.7618650638922675)), ('u', 6, (-2.0345367835183605, 0.07979526016554686, -0.9908103006662302)), ('u', 7, (3.1288268434595894, 0.18812271827062169, -2.9452943868082917)), ('cz', 5, 6), ('u', 4, (-0.03177621753540283, 1.477666233413424, 0.21826045474531974)), ('u', 5, (2.357287839630029, 1.5707963176501458, -3.232183345632234)), ('u', 6, (-0.6079955401517958, 1.5707963646129461, -0.23350333310724775)), ('u', 7, (-0.8116259098894907, 0.8587824576924703, 0.30226283836686596)), ('cz', 5, 6), ('u', 4, (0.4775983880143221, 1.871055168017069, -0.0408904809805217)), ('u', 5, (1.661386972022626, 1.4729573351207488, 1.027671342064363)), ('u', 6, (3.3349006072415994, 1.2935519067138632, 1.8780428968163312)), ('u', 7, (-1.6664034980574218, 0.3568481762584665, 3.461868774039142)), ('cz', 4, 5), ('u', 4, (-3.2700559358267576, 0.7853981366823823, 1.6916100964030927)), ('u', 5, (1.328523150757074, 1.570796209305252, 0.12081365177753267)), ('u', 6, (1.2377364450173234, 0.5081222956505452, 1.7584530973296637)), ('u', 7, (-2.849385116261185, 0.36806039682554914, 3.142733708777373)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0))])], (9, 4): [(9, [('u', 3, (0.0, 2.740499937812782, 0.0)), ('u', 2, (0.0, 0.6125547383393389, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 0.6125547383393389, 3.141592653589793)), ('u', 1, (0.0, 1.0365702822269822, 0.0)), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 1), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.0365702822269822, 3.141592653589793)), ('u', 0, (0.0, 1.3508083493994372, 0.0)), ('u', 0, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 0), ('u', 0, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (-3.141592653589793, 1.3508083493994372, 3.141592653589793)), ('u', 5, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 6, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 6), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 7), ('u', 4, (1.6783995219433123, 2.677945059334841, -2.347528943738099)), ('u', 5, (-1.5846090155505541, 3.141592648803949, 0.9739539182276005)), ('u', 6, (-0.043128525936613135, 1.1363142332841882, 0.048276423210969005)), ('u', 7, (3.6131240518873415, 1.1107010877583263, -0.9418354587012361)), ('cz', 4, 5), ('u', 4, (-1.6523744495521213, 0.16800233720120406, 0.9527359405859664)), ('u', 5, (1.0712416136914256, 1.5707963153748048, -2.1972073505033416)), ('u', 6, (-0.768693759082117, 0.5531753246206115, 0.5396942354555023)), ('u', 7, (-2.9148007132765867, 0.2008037332194747, 3.017645632458623)), ('cz', 5, 6), ('u', 4, (0.22916288305642551, 0.37442170013424514, 0.49572334230405124)), ('u', 5, (2.197207529734123, 0.8860771116860425, -3.0442805131145416)), ('u', 6, (2.843357985790216, 0.3495555085185406, 0.4905298831427509)), ('u', 7, (0.2979688948426138, 0.24839385813200238, -0.43803588336134747)), ('cz', 4, 5), ('u', 4, (0.41570487214026375, 0.2787985107045163, 0.2414760855845465)), ('u', 5, (-0.09731230047945474, 1.5707963219500847, 0.18693697618542235)), ('u', 6, (3.369013850140026, 2.7578072190774874, -0.10129045285916272)), ('u', 7, (-0.15950293778007263, 0.6278305605369155, 0.29241996204870857)), ('cz', 5, 6), ('u', 4, (0.19929743529559463, 0.88167062540414, 0.15890588000448685)), ('u', 5, (1.142456664092194, 0.2517907327213283, -0.38435357509787066)), ('u', 6, (3.6071107386496584, 1.5707962929360881, 1.0300524832632352)), ('u', 7, (3.0661863018689504, 1.5717431102987751, -0.1241345695646574)), ('cz', 6, 7), ('u', 4, (-0.3120984854440979, 0.3324426283546392, -0.8392019353490036)), ('u', 5, (-0.9523727258677195, 1.5112007017232427, -1.782143104443146)), ('u', 6, (-2.8107096237423645, 1.2281134369081435, 3.454138659309974)), ('u', 7, (3.3599544039824045, 1.5363198332342687, 0.5738148273621144)), ('cz', 5, 6), ('u', 4, (-0.17407864683084506, 1.1981581532047667, 0.720754058352084)), ('u', 5, (-1.0485857025157856, 0.25106472349746944, -0.026165490427976112)), ('u', 6, (-0.1946523375171709, 0.34489327860458485, -0.577293313523412)), ('u', 7, (0.21409937561655093, 0.048860780375053335, 0.5401001834509469)), ('cz', 6, 7), ('u', 4, (0.08604625009239802, 0.25603321491570213, -0.6234998208627462)), ('u', 5, (-3.115427233337476, 0.25106470278189263, 3.0001440070116105)), ('u', 6, (-2.564299345768496, 0.34489330545421637, 3.14097197578436)), ('u', 7, (-0.7944434936537823, 0.15214440370723623, -0.38700938932873424)), ('cz', 5, 6), ('u', 4, (0.8539870509991602, 0.048184497575099156, 0.3465322584014089)), ('u', 5, (0.2753512792741515, 1.4910808473812084e-07, 0.5142367925003585)), ('u', 6, (-0.11727296544005196, 1.2281134650062855, 0.20497368134664418)), ('u', 7, (3.849810984935723, 1.7152699274463379, 0.6275369163340978)), ('cz', 6, 7), ('u', 4, (-0.03876308831078168, 0.8460430127672541, 0.8580307302996752)), ('u', 5, (2.840620213688528, 2.032147342582146e-07, -3.066629645265795)), ('u', 6, (2.3444126865023827, 1.656357328196517e-09, -2.662531125816978)), ('u', 7, (2.5619230084226126, 1.5707963266568443, -0.061782989421305334)), ('u', 8, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 0, 8), ('u', 2, (0.7253513784390777, 0.6044711094741366, 0.20469392790644375)), ('u', 3, (3.311225398804729, 0.45344431468038326, -2.811379919385577)), ('u', 1, (-2.967218355247499, 1.570796434542743, 2.782623640362077)), ('u', 0, (0.17437422015585258, 1.5707965042184875, 0.0936275536825773)), ('u', 5, (-2.981966217109752, 0.15731121643895493, 2.938590834577346)), ('cz', 1, 0), ('u', 2, (0.35596645780294867, 0.18572047518147186, 0.1993350213934021)), ('u', 3, (0.27426031921868355, 0.23461703245156662, 0.8733942285264623)), ('u', 1, (0.3589689683381108, 0.5235987755137496, 0.808167059420607)), ('u', 0, (-0.0936272595693031, 0.5235987756882645, 0.24164140029433875)), ('u', 5, (-0.3732167230268989, 0.1692158665066253, -0.5138610131235901)), ('cz', 1, 0), ('u', 2, (3.3065392993238065, 2.155371992839475, 0.5333711558032743)), ('u', 3, (0.17598851577546493, 0.15670123378522505, 0.3917956014566585)), ('u', 1, (0.762629607489512, 2.888760746641521, 1.273509717372987)), ('u', 0, (-2.8326884570670465, 1.2221833849116261, 2.9006620162668915)), ('u', 5, (-0.29514789830304344, 0.08656595906264068, -0.2957995850638988)), ('cz', 2, 1), ('u', 2, (3.137649879014554, 1.9634954073118636, -0.6075427985534696)), ('u', 3, (2.641964011958269, 0.22113794763596387, -2.499181790905139)), ('u', 1, (-1.2735092335544311, 1.570796389596646, 2.6537438706729946)), ('u', 0, (2.7190493967038063, 0.4335249098769316, -3.5464227887719253)), ('u', 5, (-0.6391916737665571, 0.1657229044953988, -0.024541329372776033)), ('cz', 2, 1), ('u', 2, (0.17775264740865682, 0.3058307733501329, -0.4417742235515568)), ('u', 3, (-0.24049984202753, 0.0778915304240662, -0.01766321037003693)), ('u', 1, (-3.046442949408996, 1.570796448575322, 2.117105098339699)), ('u', 0, (2.589347008940728, 0.28886655857903776, -3.6589389197843123)), ('u', 5, (0.0508440614483701, 0.07203248788002012, 0.029932533667720446)), ('cz', 1, 0), ('u', 2, (2.424643232393627, 1.6965841706658102, 0.29519056307839975)), ('u', 3, (-2.9031023199264445, 0.06550196451547159, 3.0490689833889255)), ('u', 1, (-0.5463092573212669, 2.748893521815632, -0.6787734947855513)), ('u', 0, (-0.8667873688727257, 0.19392965245174282, 0.26794479272692207)), ('u', 5, (3.093960255811586, 0.3684130596408908, -2.752177636430363)), ('cz', 2, 1), ('u', 2, (0.48622642934218985, 4.965240116914444e-08, 3.1314145386877534)), ('u', 3, (3.4555174636846955, 1.825677016721441, -0.10375945734790903)), ('u', 1, (3.1274454219227774, 0.3987010753799351, -2.7245703058001975)), ('u', 0, (-0.7836119278695097, 0.44642916421938117, -0.010954000022640242)), ('u', 5, (-0.35407245584296543, 0.1009018219745298, 0.12449653021465411)), ('cz', 2, 3), ('u', 2, (-2.95124738442682, 2.872225615416436, 3.168474353976826)), ('u', 3, (2.98864467816622, 1.8770178510291984, 0.03374189545665063)), ('u', 1, (-0.04688921667005741, 0.742010041871667, -0.42431748027242494)), ('u', 0, (2.517451732666387, 0.19291873478940372, -2.9271814483694167)), ('u', 5, (-0.6494532035971243, 0.6081156702407481, 0.4080258856521049)), ('cz', 2, 1), ('u', 2, (0.20730116474291263, 1.8331478973531157, 0.46325541824844596)), ('u', 3, (2.692541925224733, 0.332696125034078, -2.6878777969285363)), ('u', 1, (2.490863927966439, 0.283599555584151, -3.18962913786471)), ('u', 0, (0.17017190346421693, 0.20177925921584974, -0.08704916348372774)), ('u', 5, (2.7417475746980577, 0.5655452491435636, -3.0339198029229153)), ('cz', 2, 3), ('u', 2, (-2.8611849940478558, 0.34985907247470327, 2.4858926099614247)), ('u', 3, (2.87078395521952, 0.16840848054093654, -3.2915600146968)), ('u', 1, (0.048036261016780685, 0.28359965336885995, -0.2578628245055713)), ('u', 0, (-0.04579375400203432, 0.37476347219220135, -0.0019053945699001838)), ('u', 5, (-2.9901835580399307, 0.29429615142429255, 2.5298877934002943)), ('cz', 2, 1), ('u', 2, (-0.056927005043825446, 1.3014292976901551, -0.5532071820595532)), ('u', 3, (3.4785109576898656, 1.40526859119825, -0.254041621662799)), ('u', 1, (2.6370874705266063, 0.27233958144772935, -3.545853570337978)), ('u', 0, (3.105802818900175, 0.21107958032122495, -2.861471392990672)), ('u', 5, (3.0189880563377156, 0.19670772384256013, -3.0445309207076874)), ('cz', 2, 3), ('u', 2, (0.01339029507980527, 0.0416754087342334, -1.347371724509553)), ('u', 3, (3.499319469952347, 1.355519791945654, -0.5991567935202315)), ('u', 1, (-2.1086415347436596, 1.3486442481739653, 1.9995738248248909)), ('u', 0, (1.6776715535216509, 1.3775652646994023, -2.0667194984618944)), ('u', 5, (-0.9230392567029317, 0.3689502539357166, -0.027637551969518004)), ('cz', 1, 0), ('u', 2, (-0.6422086949145018, 0.6564240419907547, 0.3050682210391836)), ('u', 3, (-0.3570995437419664, 0.3230608004096355, -0.4818739743945121)), ('u', 1, (0.9831653765370997, 0.6154796885409349, -2.044379448865771)), ('u', 0, (1.2690644052694249, 0.6154797227608092, -1.450257109340972)), ('u', 5, (0.32441597127291805, 0.35884350817350413, -0.5202647635263868)), ('cz', 1, 0), ('u', 2, (3.0849500536513323, 0.5822022691130421, -0.24393556074215605)), ('u', 3, (-0.07734822951641647, 0.4435772553137907, -0.06596287366372468)), ('u', 1, (1.1222045403932766, 1.2721557575140932, -1.643452931303447)), ('u', 0, (-1.518587141618921, 1.6648415595850694, 1.767035244953014)), ('u', 5, (-2.8821138871253633, 0.6916920960996875, 2.47733528484548)), ('cz', 2, 1), ('u', 2, (2.2401631579399766, 1.57079612890024, -0.6250479515489906)), ('u', 3, (0.019069698632826204, 0.004875598650025819, 0.48157043024513996)), ('u', 1, (2.9948115830243927, 0.9587582103095811, -3.1531878704143894)), ('u', 0, (2.558845213379647, 0.0026409094544454616, -3.144567419915024)), ('u', 5, (-0.24806262907427706, 0.5508837005772087, -0.9339898849771324)), ('cz', 2, 1), ('u', 2, (-2.9673015967720766, 0.08948535665168787, 3.1599312277533493)), ('u', 3, (2.3790348862594284, 0.2712248006470355, -3.2040591439620414)), ('u', 1, (-3.129997892854534, 0.7853981335305925, 3.0269609858112005)), ('u', 0, (3.0359102852588746, 0.1938390933531295, -2.778825432657605)), ('u', 5, (2.705457550215509, 0.48649144979514203, -2.964461900581061)), ('cz', 1, 0), ('u', 2, (3.572868238545496, 1.5598263045480647, -0.19509687224531835)), ('u', 3, (2.998883524589488, 0.15059171477843603, -3.0599404762732667)), ('u', 1, (0.11463202022691882, 0.7853981649586114, 0.17677739302616385)), ('u', 0, (0.25127933585639883, 0.48994533516927113, -0.3547313940345701)), ('u', 5, (-3.157884001950231, 0.5858331264680662, 2.9538041371221344)), ('cz', 2, 1), ('u', 2, (3.3460062868220524, 0.5910844552042714, -0.052680694256529215)), ('u', 3, (-3.2368160309582397, 0.17716874842237512, 2.754312610522317)), ('u', 1, (-0.1767774518436646, 2.356194531997464, 0.14646174769130244)), ('u', 0, (-0.02761925796749795, 1.7818724057866364, 0.1473634861913784)), ('u', 5, (-0.516455160657608, 0.21800904325982232, -0.05002503924043772)), ('cz', 1, 0), ('u', 2, (3.3160876593419366, 0.6613514697864085, -2.9498361401440376)), ('u', 3, (-3.107577927801868, 0.24180153100131335, 2.725398761803262)), ('u', 1, (-0.1464617655355463, 1.1059759781713967, 0.12979889868104255)), ('u', 0, (-3.065137472118039, 0.7853982422009756, 3.1407867514401095)), ('u', 5, (-0.19268113051809058, 0.36691758663500634, -0.12489290891100432)), ('cz', 1, 0), ('u', 2, (0.5529122404706256, 0.10008139403023099, -0.20710168438656099)), ('u', 3, (0.009481441806211063, 0.4136301104545076, 0.1852669385705289)), ('u', 1, (3.0117937696416934, 1.5707962567027152, -2.456961495765487)), ('u', 0, (3.142398533867907, 2.616299151423349, -2.4569616077509973)), ('u', 5, (-2.7450832431691454, 1.5044247017835743, 3.1714501363368495)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (0.0, 0.23182380450040305, 0.0)), ('cz', 8, 7), ('u', 7, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (0.0, 1.3389725222944935, 0.0)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 3, (-0.011998651932780754, 0.035068744618698446, -0.009927966776869981)), ('u', 5, (-3.118408077145403, 8.341254967389955e-09, 3.1300003637502196)), ('u', 6, (3.0799739856600827, 0.04101951625127175, -3.175201664685296)), ('u', 4, (0.15270318509924602, 1.5707963180645836, -3.6543846698711135e-09)), ('u', 7, (-3.6749238119769676, 0.14412996902599418, 2.584848365873525)), ('cz', 5, 4), ('u', 3, (-0.009927966742459823, 0.033442075322526583, -0.008093153569313113)), ('u', 5, (-0.011592289857122701, 0.5235987754951184, -6.505802567675911e-09)), ('u', 6, (3.1079836425593723, 0.00022290334956562985, -3.152959083725753)), ('u', 4, (-3.529107380115216e-09, 0.523598775865786, 6.919391735800735e-05)), ('u', 7, (-0.5567442878288905, 0.09333323753923148, -0.45001944877466343)), ('cz', 5, 4), ('u', 3, (-0.008093153636521204, 0.03100252754737548, -0.006484194519471317)), ('u', 5, (-6.5420543392258335e-09, 1.5707963322971892, -0.03003248731706627)), ('u', 6, (3.1302262234116123, 1.529719592297703, 0.0015703886429401237)), ('u', 4, (3.1416618474185785, 1.5725046920439383, -3.101181087375395)), ('u', 7, (-0.4500194486291905, 0.2781111344697831, -0.2419468717914133)), ('cz', 5, 6), ('u', 3, (-0.006484194433627047, 0.02788255396182956, -0.005084810560235049)), ('u', 5, (3.1115601662193346, 1.9644974755453697, -3.0598092275346036)), ('u', 6, (3.1431630422315986, 1.5756628864443611, -0.0560709050552215)), ('u', 4, (0.040411566141154244, 0.0017139732709513032, 0.02499535504874261)), ('u', 7, (-0.2419468718423906, 0.32065403979562107, -0.1075942805260432)), ('cz', 5, 4), ('u', 3, (-0.005084810613438314, 0.024207170535394285, -0.0038744853192448432)), ('u', 5, (-3.059809227525655, 0.39848399027480996, 3.141592677496497)), ('u', 6, (3.085521748543134, 0.004897338332805505, -3.0994895928797357)), ('u', 4, (0.02499535504225195, 1.3993813386454823e-08, 0.13996816694155428)), ('u', 7, (-0.10759428056327414, 0.30114307748768226, -0.05120056142974744)), ('cz', 5, 6), ('u', 3, (-0.003874485332422318, 0.02009428710042876, -0.002829891589337413)), ('u', 5, (1.2870066012915515e-08, 0.3984839951383076, -0.08631632879508466)), ('u', 6, (0.04210306071591691, 0.80323408105442, -1.854085829658403e-08)), ('u', 4, (3.281560820607734, 8.719845644681125e-09, -2.90502094908558)), ('u', 7, (-0.05120056134528083, 0.29207557255071703, 0.049664416154679626)), ('cz', 5, 4), ('u', 3, (-0.002829891589588122, 0.015654826620762414, -0.0019258915303032504)), ('u', 5, (-0.08631632876698098, 1.5702104351477197, 0.010895163553423598)), ('u', 6, (3.1415926350686307, 0.8032340684189889, 0.0386198062833798)), ('u', 4, (-2.9050209490607397, 0.024789647960802845, 3.3613476276605736)), ('u', 7, (0.04966441621010443, 0.3181230954544873, 0.1275500313587809)), ('cz', 6, 7), ('u', 3, (-0.0019258915260883522, 0.010992808236069998, -0.0011362416795111148)), ('u', 5, (0.01089516367403582, 0.39438583187416293, 0.10283365860826715)), ('u', 6, (-3.102972847332388, 2.8796760336871428, 3.1268429577084658)), ('u', 4, (-2.9218376795000625, 0.019162103192931518, 3.3464864753681027)), ('u', 7, (3.2691426849289273, 1.5993447648568127, -0.010245007543836504)), ('cz', 5, 6), ('u', 3, (-0.0011362416747914867, 0.006205499937120086, -0.00043410405116450464)), ('u', 5, (0.10283365853007384, 0.005250612216072889, 1.229654780485201e-06)), ('u', 6, (-0.014749695915711764, 1.8326041378362559, -0.05675808125253269)), ('u', 4, (-2.936698831881056, 0.013365379715964415, 3.333327885337581)), ('u', 7, (3.1313476460548166, 0.0285544281829088, -3.150346969873966)), ('cz', 6, 7), ('u', 3, (-0.0004341040523492897, 0.0013836895457048843, 0.00020756594846420192)), ('u', 5, (-3.14159142395901, 0.005250595670552266, 3.064537988460174)), ('u', 6, (3.0848345584228554, 0.2634265832319953, -3.0867766015809632)), ('u', 4, (-2.949857421832063, 0.0065639168782822815, 3.3203616507843052)), ('u', 7, (3.1328383373143294, 0.007214997267805406, -3.1278215477172604)), ('cz', 5, 6), ('u', 3, (3.1418002195198937, 0.0033879211136994573, -3.1407769464413704)), ('u', 5, (-0.07705466513291019, 0.012064066316140654, -0.07249966970117298)), ('u', 6, (0.05481605201591043, 1.3088797028288943, -0.05243650089403663)), ('u', 4, (0.17876899717765427, 0.0023099115013583828, 0.16470245329335875)), ('u', 7, (3.155363759457692, 1.5635840616073071, 0.0010188476113417444)), ('cz', 6, 7), ('u', 3, (3.1424083607471442, 0.008030220964960526, -3.140175767869725)), ('u', 5, (-1.643295996502317, 1.5725395677602834, 1.5767650599885832)), ('u', 6, (-3.194029154466275, 0.5109292593725938, 3.069038656875536)), ('u', 4, (1.7354987800932298, 1.5132754165511737, -1.5695222465353704)), ('u', 7, (3.142611501175785, 1.60023235754839, -0.0018191973740395717)), ('cz', 5, 4), ('u', 3, (3.143009539303383, 0.012469131249627013, -3.1395555980269063)), ('u', 5, (1.5767650601072347, 0.6154797050427105, -1.0935137006446563)), ('u', 6, (3.0690386568654087, 0.5191225209576913, -3.1002343444807328)), ('u', 4, (1.5720704071213085, 0.6154797097037269, -1.5675748019018252)), ('u', 7, (-0.0018191974056423884, 0.024172558097420544, -0.0019053133573421854)), ('cz', 5, 4), ('u', 3, (3.1436297091392724, 0.01663502733022202, -3.1388913174911597)), ('u', 5, (2.048078952851612, 1.6168694345030314, -1.587100047237085)), ('u', 6, (3.1829509627314927, 2.1145173881119947, -0.04868326884435881)), ('u', 4, (-1.5675748019755396, 1.5696413567756455, 1.642788204071491)), ('u', 7, (-0.0019053133080495692, 0.018165194301069252, -0.001537521951944066)), ('cz', 5, 6), ('u', 3, (3.1442939896801985, 0.020462181721422237, -3.1381588532690596)), ('u', 5, (3.1252889331316185, 0.9553166185256176, -3.098979656315909)), ('u', 6, (3.0929093847390288, 1.570796343534117, 0.0005513990363770205)), ('u', 4, (-3.0696007762834774, 0.0038697493500351554, 3.1847176674592004)), ('u', 7, (-0.001537521962138161, 0.011414675394315833, -0.0006543839782734821)), ('cz', 5, 6), ('u', 3, (3.145999426008122, 0.024306044078736606, -3.136168317795093)), ('u', 5, (-3.1260808992185045, 0.7853981574050549, 3.141592662162083)), ('u', 6, (-3.13583945440354, 0.014439433724023936, 3.1384400222854185)), ('u', 4, (-3.1031216024398542, 0.0026812704966859917, 3.132629288626685)), ('u', 7, (0.00016945599671113875, 2.7901969406265238e-05, 0.0021370127558946084)), ('cz', 5, 4), ('u', 3, (3.1470169893657323, 0.027215365804092718, -3.1350187258135094)), ('u', 5, (8.588510223293177e-09, 0.7853981661272267, -5.7097120770185845e-09)), ('u', 6, (3.1384400222535827, 1.5563572172741813, -0.08097022914912522)), ('u', 4, (-3.150556018558537, 0.3173188418447972, 3.078454534587191)), ('u', 7, (3.143729666358712, 0.008041009902001532, -3.1368442389081657)), ('cz', 5, 6), ('u', 3, (3.1481665813959263, 0.02958570059649117, -3.133720308378973)), ('u', 5, (-7.224919709996777e-09, 0.7853981552422288, 4.939926870876897e-09)), ('u', 6, (3.060622424441615, 2.092354849933306, 0.003145599496788165)), ('u', 4, (-0.06313811903278108, 1.2697266125414508, -0.024073164687880056)), ('u', 7, (3.1463410682592032, 0.016472200295720375, -3.133585214724122)), ('cz', 5, 4), ('u', 3, (3.149464998810803, 0.03136023613261947, -3.132258540085515)), ('u', 5, (4.926433343212702e-09, 1.0471975265305735, -6.057048844095855e-09)), ('u', 6, (0.003145599502141147, 0.5253013898862602, -0.00545145978210719)), ('u', 4, (3.117519488887346, 0.7853981708664443, -3.141592673128462)), ('u', 7, (3.1496000924388747, 0.025081184697559405, -3.1297020648728147)), ('cz', 5, 4), ('u', 3, (3.15092676707859, 0.03248320803675271, -3.130621577715438)), ('u', 5, (3.141592647523828, 1.5707963173438424, -3.0165452970539737)), ('u', 6, (-0.005451459797216783, 0.523953607673068, 0.02744646447431627)), ('u', 4, (3.141592634091762, 0.6154797504055258, -3.0165452647593365)), ('u', 7, (3.15348324229967, 0.033655719295381435, -3.1252478838979543)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0))]), (9, [('u', 3, (0.0, 2.740499937812782, 0.0)), ('u', 2, (0.0, 0.6125547383393389, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 0.6125547383393389, 3.141592653589793)), ('u', 1, (0.0, 1.0365702822269822, 0.0)), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 1), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.0365702822269822, 3.141592653589793)), ('u', 0, (0.0, 1.3508083493994372, 0.0)), ('u', 0, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 0), ('u', 0, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (-3.141592653589793, 1.3508083493994372, 3.141592653589793)), ('u', 5, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 6, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 6), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 7), ('u', 4, (1.6783995219433123, 2.677945059334841, -2.347528943738099)), ('u', 5, (-1.5846090155505541, 3.141592648803949, 0.9739539182276005)), ('u', 6, (-0.043128525936613135, 1.1363142332841882, 0.048276423210969005)), ('u', 7, (3.6131240518873415, 1.1107010877583263, -0.9418354587012361)), ('cz', 4, 5), ('u', 4, (-1.6523744495521213, 0.16800233720120406, 0.9527359405859664)), ('u', 5, (1.0712416136914256, 1.5707963153748048, -2.1972073505033416)), ('u', 6, (-0.768693759082117, 0.5531753246206115, 0.5396942354555023)), ('u', 7, (-2.9148007132765867, 0.2008037332194747, 3.017645632458623)), ('cz', 5, 6), ('u', 4, (0.22916288305642551, 0.37442170013424514, 0.49572334230405124)), ('u', 5, (2.197207529734123, 0.8860771116860425, -3.0442805131145416)), ('u', 6, (2.843357985790216, 0.3495555085185406, 0.4905298831427509)), ('u', 7, (0.2979688948426138, 0.24839385813200238, -0.43803588336134747)), ('cz', 4, 5), ('u', 4, (0.41570487214026375, 0.2787985107045163, 0.2414760855845465)), ('u', 5, (-0.09731230047945474, 1.5707963219500847, 0.18693697618542235)), ('u', 6, (3.369013850140026, 2.7578072190774874, -0.10129045285916272)), ('u', 7, (-0.15950293778007263, 0.6278305605369155, 0.29241996204870857)), ('cz', 5, 6), ('u', 4, (0.19929743529559463, 0.88167062540414, 0.15890588000448685)), ('u', 5, (1.142456664092194, 0.2517907327213283, -0.38435357509787066)), ('u', 6, (3.6071107386496584, 1.5707962929360881, 1.0300524832632352)), ('u', 7, (3.0661863018689504, 1.5717431102987751, -0.1241345695646574)), ('cz', 6, 7), ('u', 4, (-0.3120984854440979, 0.3324426283546392, -0.8392019353490036)), ('u', 5, (-0.9523727258677195, 1.5112007017232427, -1.782143104443146)), ('u', 6, (-2.8107096237423645, 1.2281134369081435, 3.454138659309974)), ('u', 7, (3.3599544039824045, 1.5363198332342687, 0.5738148273621144)), ('cz', 5, 6), ('u', 4, (-0.17407864683084506, 1.1981581532047667, 0.720754058352084)), ('u', 5, (-1.0485857025157856, 0.25106472349746944, -0.026165490427976112)), ('u', 6, (-0.1946523375171709, 0.34489327860458485, -0.577293313523412)), ('u', 7, (0.21409937561655093, 0.048860780375053335, 0.5401001834509469)), ('cz', 6, 7), ('u', 4, (0.08604625009239802, 0.25603321491570213, -0.6234998208627462)), ('u', 5, (-3.115427233337476, 0.25106470278189263, 3.0001440070116105)), ('u', 6, (-2.564299345768496, 0.34489330545421637, 3.14097197578436)), ('u', 7, (-0.7944434936537823, 0.15214440370723623, -0.38700938932873424)), ('cz', 5, 6), ('u', 4, (0.8539870509991602, 0.048184497575099156, 0.3465322584014089)), ('u', 5, (0.2753512792741515, 1.4910808473812084e-07, 0.5142367925003585)), ('u', 6, (-0.11727296544005196, 1.2281134650062855, 0.20497368134664418)), ('u', 7, (3.849810984935723, 1.7152699274463379, 0.6275369163340978)), ('cz', 6, 7), ('u', 4, (-0.03876308831078168, 0.8460430127672541, 0.8580307302996752)), ('u', 5, (2.840620213688528, 2.032147342582146e-07, -3.066629645265795)), ('u', 6, (2.3444126865023827, 1.656357328196517e-09, -2.662531125816978)), ('u', 7, (2.5619230084226126, 1.5707963266568443, -0.061782989421305334)), ('u', 8, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 0, 8), ('u', 2, (0.7253513784390777, 0.6044711094741366, 0.20469392790644375)), ('u', 3, (3.311225398804729, 0.45344431468038326, -2.811379919385577)), ('u', 1, (-2.967218355247499, 1.570796434542743, 2.782623640362077)), ('u', 0, (0.17437422015585258, 1.5707965042184875, 0.0936275536825773)), ('u', 5, (-2.981966217109752, 0.15731121643895493, 2.938590834577346)), ('cz', 1, 0), ('u', 2, (0.35596645780294867, 0.18572047518147186, 0.1993350213934021)), ('u', 3, (0.27426031921868355, 0.23461703245156662, 0.8733942285264623)), ('u', 1, (0.3589689683381108, 0.5235987755137496, 0.808167059420607)), ('u', 0, (-0.0936272595693031, 0.5235987756882645, 0.24164140029433875)), ('u', 5, (-0.3732167230268989, 0.1692158665066253, -0.5138610131235901)), ('cz', 1, 0), ('u', 2, (3.3065392993238065, 2.155371992839475, 0.5333711558032743)), ('u', 3, (0.17598851577546493, 0.15670123378522505, 0.3917956014566585)), ('u', 1, (0.762629607489512, 2.888760746641521, 1.273509717372987)), ('u', 0, (-2.8326884570670465, 1.2221833849116261, 2.9006620162668915)), ('u', 5, (-0.29514789830304344, 0.08656595906264068, -0.2957995850638988)), ('cz', 2, 1), ('u', 2, (3.137649879014554, 1.9634954073118636, -0.6075427985534696)), ('u', 3, (2.641964011958269, 0.22113794763596387, -2.499181790905139)), ('u', 1, (-1.2735092335544311, 1.570796389596646, 2.6537438706729946)), ('u', 0, (2.7190493967038063, 0.4335249098769316, -3.5464227887719253)), ('u', 5, (-0.6391916737665571, 0.1657229044953988, -0.024541329372776033)), ('cz', 2, 1), ('u', 2, (0.17775264740865682, 0.3058307733501329, -0.4417742235515568)), ('u', 3, (-0.24049984202753, 0.0778915304240662, -0.01766321037003693)), ('u', 1, (-3.046442949408996, 1.570796448575322, 2.117105098339699)), ('u', 0, (2.589347008940728, 0.28886655857903776, -3.6589389197843123)), ('u', 5, (0.0508440614483701, 0.07203248788002012, 0.029932533667720446)), ('cz', 1, 0), ('u', 2, (2.424643232393627, 1.6965841706658102, 0.29519056307839975)), ('u', 3, (-2.9031023199264445, 0.06550196451547159, 3.0490689833889255)), ('u', 1, (-0.5463092573212669, 2.748893521815632, -0.6787734947855513)), ('u', 0, (-0.8667873688727257, 0.19392965245174282, 0.26794479272692207)), ('u', 5, (3.093960255811586, 0.3684130596408908, -2.752177636430363)), ('cz', 2, 1), ('u', 2, (0.48622642934218985, 4.965240116914444e-08, 3.1314145386877534)), ('u', 3, (3.4555174636846955, 1.825677016721441, -0.10375945734790903)), ('u', 1, (3.1274454219227774, 0.3987010753799351, -2.7245703058001975)), ('u', 0, (-0.7836119278695097, 0.44642916421938117, -0.010954000022640242)), ('u', 5, (-0.35407245584296543, 0.1009018219745298, 0.12449653021465411)), ('cz', 2, 3), ('u', 2, (-2.95124738442682, 2.872225615416436, 3.168474353976826)), ('u', 3, (2.98864467816622, 1.8770178510291984, 0.03374189545665063)), ('u', 1, (-0.04688921667005741, 0.742010041871667, -0.42431748027242494)), ('u', 0, (2.517451732666387, 0.19291873478940372, -2.9271814483694167)), ('u', 5, (-0.6494532035971243, 0.6081156702407481, 0.4080258856521049)), ('cz', 2, 1), ('u', 2, (0.20730116474291263, 1.8331478973531157, 0.46325541824844596)), ('u', 3, (2.692541925224733, 0.332696125034078, -2.6878777969285363)), ('u', 1, (2.490863927966439, 0.283599555584151, -3.18962913786471)), ('u', 0, (0.17017190346421693, 0.20177925921584974, -0.08704916348372774)), ('u', 5, (2.7417475746980577, 0.5655452491435636, -3.0339198029229153)), ('cz', 2, 3), ('u', 2, (-2.8611849940478558, 0.34985907247470327, 2.4858926099614247)), ('u', 3, (2.87078395521952, 0.16840848054093654, -3.2915600146968)), ('u', 1, (0.048036261016780685, 0.28359965336885995, -0.2578628245055713)), ('u', 0, (-0.04579375400203432, 0.37476347219220135, -0.0019053945699001838)), ('u', 5, (-2.9901835580399307, 0.29429615142429255, 2.5298877934002943)), ('cz', 2, 1), ('u', 2, (-0.056927005043825446, 1.3014292976901551, -0.5532071820595532)), ('u', 3, (3.4785109576898656, 1.40526859119825, -0.254041621662799)), ('u', 1, (2.6370874705266063, 0.27233958144772935, -3.545853570337978)), ('u', 0, (3.105802818900175, 0.21107958032122495, -2.861471392990672)), ('u', 5, (3.0189880563377156, 0.19670772384256013, -3.0445309207076874)), ('cz', 2, 3), ('u', 2, (0.01339029507980527, 0.0416754087342334, -1.347371724509553)), ('u', 3, (3.499319469952347, 1.355519791945654, -0.5991567935202315)), ('u', 1, (-2.1086415347436596, 1.3486442481739653, 1.9995738248248909)), ('u', 0, (1.6776715535216509, 1.3775652646994023, -2.0667194984618944)), ('u', 5, (-0.9230392567029317, 0.3689502539357166, -0.027637551969518004)), ('cz', 1, 0), ('u', 2, (-0.6422086949145018, 0.6564240419907547, 0.3050682210391836)), ('u', 3, (-0.3570995437419664, 0.3230608004096355, -0.4818739743945121)), ('u', 1, (0.9831653765370997, 0.6154796885409349, -2.044379448865771)), ('u', 0, (1.2690644052694249, 0.6154797227608092, -1.450257109340972)), ('u', 5, (0.32441597127291805, 0.35884350817350413, -0.5202647635263868)), ('cz', 1, 0), ('u', 2, (3.0849500536513323, 0.5822022691130421, -0.24393556074215605)), ('u', 3, (-0.07734822951641647, 0.4435772553137907, -0.06596287366372468)), ('u', 1, (1.1222045403932766, 1.2721557575140932, -1.643452931303447)), ('u', 0, (-1.518587141618921, 1.6648415595850694, 1.767035244953014)), ('u', 5, (-2.8821138871253633, 0.6916920960996875, 2.47733528484548)), ('cz', 2, 1), ('u', 2, (2.2401631579399766, 1.57079612890024, -0.6250479515489906)), ('u', 3, (0.019069698632826204, 0.004875598650025819, 0.48157043024513996)), ('u', 1, (2.9948115830243927, 0.9587582103095811, -3.1531878704143894)), ('u', 0, (2.558845213379647, 0.0026409094544454616, -3.144567419915024)), ('u', 5, (-0.24806262907427706, 0.5508837005772087, -0.9339898849771324)), ('cz', 2, 1), ('u', 2, (-2.9673015967720766, 0.08948535665168787, 3.1599312277533493)), ('u', 3, (2.3790348862594284, 0.2712248006470355, -3.2040591439620414)), ('u', 1, (-3.129997892854534, 0.7853981335305925, 3.0269609858112005)), ('u', 0, (3.0359102852588746, 0.1938390933531295, -2.778825432657605)), ('u', 5, (2.705457550215509, 0.48649144979514203, -2.964461900581061)), ('cz', 1, 0), ('u', 2, (3.572868238545496, 1.5598263045480647, -0.19509687224531835)), ('u', 3, (2.998883524589488, 0.15059171477843603, -3.0599404762732667)), ('u', 1, (0.11463202022691882, 0.7853981649586114, 0.17677739302616385)), ('u', 0, (0.25127933585639883, 0.48994533516927113, -0.3547313940345701)), ('u', 5, (-3.157884001950231, 0.5858331264680662, 2.9538041371221344)), ('cz', 2, 1), ('u', 2, (3.3460062868220524, 0.5910844552042714, -0.052680694256529215)), ('u', 3, (-3.2368160309582397, 0.17716874842237512, 2.754312610522317)), ('u', 1, (-0.1767774518436646, 2.356194531997464, 0.14646174769130244)), ('u', 0, (-0.02761925796749795, 1.7818724057866364, 0.1473634861913784)), ('u', 5, (-0.516455160657608, 0.21800904325982232, -0.05002503924043772)), ('cz', 1, 0), ('u', 2, (3.3160876593419366, 0.6613514697864085, -2.9498361401440376)), ('u', 3, (-3.107577927801868, 0.24180153100131335, 2.725398761803262)), ('u', 1, (-0.1464617655355463, 1.1059759781713967, 0.12979889868104255)), ('u', 0, (-3.065137472118039, 0.7853982422009756, 3.1407867514401095)), ('u', 5, (-0.19268113051809058, 0.36691758663500634, -0.12489290891100432)), ('cz', 1, 0), ('u', 2, (0.5529122404706256, 0.10008139403023099, -0.20710168438656099)), ('u', 3, (0.009481441806211063, 0.4136301104545076, 0.1852669385705289)), ('u', 1, (3.0117937696416934, 1.5707962567027152, -2.456961495765487)), ('u', 0, (3.142398533867907, 2.616299151423349, -2.4569616077509973)), ('u', 5, (-2.7450832431691454, 1.5044247017835743, 3.1714501363368495)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (0.0, 0.23182380450040305, 0.0)), ('cz', 8, 7), ('u', 7, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (0.0, 1.3389725222944935, 0.0)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (0.0, 1.5707963267948966, 0.0)), ('cz', 4, 5), ('u', 5, (0.0, 0.5235987755982989, 0.0)), ('u', 4, (0.0, 0.5235987755982989, 0.0)), ('cz', 4, 5), ('u', 5, (0.0, 1.5707963267948966, 0.0)), ('u', 4, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 6), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 5, (-3.141592653589793, 1.1780972450961724, 3.141592653589793)), ('cz', 4, 5), ('u', 5, (0.0, 0.39269908169872414, 0.0)), ('cz', 6, 5), ('u', 5, (-3.141592653589793, 0.39269908169872414, 3.141592653589793)), ('cz', 4, 5), ('u', 5, (0.0, 1.1780972450961724, 0.0)), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 6), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 6, 7), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 6, (-3.141592653589793, 1.3089969389957472, 3.141592653589793)), ('cz', 5, 6), ('u', 6, (0.0, 0.26179938779914935, 0.0)), ('cz', 7, 6), ('u', 6, (-3.141592653589793, 0.26179938779914935, 3.141592653589793)), ('cz', 5, 6), ('u', 6, (0.0, 1.3089969389957472, 0.0)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 6, 7), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 5, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 4, 5), ('u', 5, (1.570796326794898, 0.6154797086703868, -1.570796326794898)), ('u', 4, (1.5707963267948966, 0.6154797086703876, -1.5707963267948966)), ('cz', 4, 5), ('u', 5, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 4, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 5, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 6), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 5, (-3.141592653589793, 1.2630564724597029, 3.141592653589793)), ('cz', 4, 5), ('u', 5, (0.0, 0.30773985433519363, 0.0)), ('cz', 6, 5), ('u', 5, (-3.141592653589793, 0.30773985433519363, 3.141592653589793)), ('cz', 4, 5), ('u', 5, (0.0, 1.2630564724597029, 0.0)), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 6), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (0.0, 1.5707963267948966, 0.0)), ('u', 5, (0.0, 1.5707963267948966, 0.0)), ('cz', 4, 5), ('u', 5, (1.2251903831398114e-16, 0.7853981633974483, -1.1496735851465472e-17)), ('u', 4, (-3.141592653589793, 0.7853981633974483, 3.141592653589793)), ('cz', 4, 5), ('u', 5, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 4, (-3.141592653589793, 1.5707963267948966, 3.141592653589793))])], (10, 2): [(10, [('u', 4, (0.0, 2.1598272970111703, 0.0)), ('u', 3, (0.0, 1.0068536854342676, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.0068536854342676, 3.141592653589793)), ('u', 8, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 9), ('u', 0, (-0.15456194594996994, 0.4301992741318007, -0.23274622218789695)), ('u', 1, (-0.40170183590150327, 0.28586108595600335, -0.10570445481115337)), ('u', 2, (-2.78781494008481, 0.8860771265740348, 3.098058177545558)), ('u', 3, (-1.2771129659012126, 3.1415926426306906, -3.8698329862184537)), ('u', 4, (0.06455124714868196, 0.15674351119555924, 0.28885302799265855)), ('cz', 2, 3), ('u', 0, (3.2868320324684976, 0.09166576355737018, -2.7550230712349313)), ('u', 1, (0.0911232712051731, 0.01368910055863399, 0.45056206982018426)), ('u', 2, (-3.11743803269269, 0.9442426455972222, 3.1059072654057824)), ('u', 3, (-3.834956751165805, 1.570796328799766, 2.414320365177579)), ('u', 4, (2.933667700131255, 1.7270351552587933, 0.8611115471393367)), ('cz', 3, 4), ('u', 0, (0.5392459371987273, 0.4234279364041615, -0.5568536770895516)), ('u', 1, (3.42675280329011, 0.19504202148645747, -2.469936624299969)), ('u', 2, (-0.21665352692602918, 0.06015829372792343, -0.250870198673664)), ('u', 3, (-0.8435240423228354, 1.5707963259084061, 0.5588680882268506)), ('u', 4, (2.5536936891906237, 2.2077370023760423, 0.639330425498244)), ('cz', 2, 3), ('u', 0, (-0.17556341147665933, 0.29854234430722804, -1.2499813382380576)), ('u', 1, (0.612744997199851, 0.27514680609283165, 0.7704736055176745)), ('u', 2, (-0.4797695508209159, 1.0354512235988662, -1.0189709592004077)), ('u', 3, (0.5482806295361009, 1.5707963250038932, 0.1461042553833234)), ('u', 4, (3.482560418431404, 0.28111978511638774, -0.5912323081025872)), ('cz', 3, 4), ('u', 8, (-0.03394941517093226, 5.385189109187892e-09, -0.0076111932644572795)), ('u', 9, (1.975848093044334, 2.7319147559312325, -1.5292508938419642)), ('u', 4, (2.3036984473829194, 1.4035403496186079, 3.4070213949331376)), ('u', 7, (0.027242781907830027, 2.25551552984548, -0.4026988463482039)), ('cz', 8, 7), ('u', 8, (-3.1000320416522045, 1.570796327378528, 2.8856552605987695)), ('u', 9, (1.5292508480085254, 0.40967789648162206, 0.4005740956185251)), ('u', 4, (3.5892185726162475, 0.49474055051236, -2.6887349409356025)), ('u', 7, (-2.775889780732306, 1.6576170104990753, 2.7150951215683663)), ('cz', 8, 9), ('u', 8, (3.3975300448594044, 0.4636476100548608, -2.877400838491899)), ('u', 9, (-0.3996592118041773, 0.2439904992728105, -0.55119857972043)), ('u', 4, (-2.6028086732454585, 0.32927829913357926, 2.753977253792781)), ('u', 7, (-2.6642458770536543, 0.598835932154401, 2.449836858324313)), ('cz', 8, 7), ('u', 8, (-0.26419180956091604, 1.5707963262608304, 0.28764124343664077)), ('u', 9, (2.3983754899474308, 2.1163988491702015, -0.06831419421822638)), ('u', 4, (2.7746395552011367, 0.0013267737590880525, -3.3960290215859064)), ('u', 7, (-3.5223210155444793, 0.3327712631556224, 2.2693899364712173)), ('cz', 8, 9), ('u', 8, (2.8502382697273334, 1.4984073152988449, -3.3507922936077645)), ('u', 9, (-0.031030283028091757, 1.578003303928968e-09, -0.16351306355662604)), ('u', 4, (-0.3435880939366586, 0.16760042401971526, -0.5583844959970989)), ('u', 7, (0.8722027135706527, 0.33277126751538527, -1.2807911461274981)), ('u', 0, (2.303733656392368, 0.16148543652229524, -3.9497980814397304)), ('u', 1, (0.5442521716505224, 1.6302319785452546, 0.24725989439266866)), ('u', 2, (-0.9535706167161428, 2.1536597079078486, 0.0430441078844051)), ('u', 3, (3.4803502730890328, 0.06325816915873374, -2.3780977071992337)), ('cz', 1, 2), ('u', 0, (-3.5109005070080976, 0.33193395457706537, 2.3789395315381774)), ('u', 1, (3.636835434893288, 1.257269196986108, -2.6156571243980773)), ('u', 2, (-0.19015292244268306, 2.356194493377117, 0.36529779722966144)), ('u', 3, (0.4326275390523066, 0.0296453683871993, 0.23229900119671437)), ('cz', 2, 3), ('u', 0, (-0.43561652999785405, 0.21747443117040807, 0.16312617545616148)), ('u', 1, (-3.8758203981800676, 0.6799436095750659, 2.403543745015792)), ('u', 2, (3.8162357421931343, 8.625763986416733e-08, -1.8133656556594009)), ('u', 3, (-2.416262641035493, 0.4444295281254736, 3.395262614647079)), ('cz', 1, 2), ('u', 0, (-0.23775780442139427, 0.5099565361441982, 0.5355028111200876)), ('u', 1, (2.146789898069154, 0.48624615485680545, -3.406300432885364)), ('u', 2, (0.24982598932370687, 1.5707963244964236, 0.5684170629260561)), ('u', 3, (3.3229594617721228, 1.6964847149577291, 0.4279900434809949)), ('cz', 2, 3), ('u', 0, (-1.1416819268813243, 0.09141455125431372, 1.5396031513100699)), ('u', 1, (-2.5194174598938748, 2.62722496949545, 1.2581226225791136)), ('u', 2, (-2.694818066981105, 0.25953710529805224, 3.1700764326262645)), ('u', 3, (2.5956633643538476, 1.7834987247116252, 0.3136797140268961)), ('cz', 0, 1), ('u', 0, (-1.305340134791441, 1.0554113399907372, 1.6996120596952016)), ('u', 1, (1.2194909709035284, 2.526112946310009, -1.363022137244877)), ('u', 2, (2.8729346238474727, 2.827897909153286, -0.7655164511322208)), ('u', 3, (3.0256913642491283, 0.13734073576096725, -2.6746616762635864)), ('cz', 1, 2), ('u', 0, (3.359957800103702, 0.12131860400987804, -2.651772155709388)), ('u', 1, (-2.733887144360484, 1.570796232525795, 1.3864532688403917)), ('u', 2, (3.4848155449122826, 0.034475470122763266, 0.4975986000164825)), ('u', 3, (2.662002143725979, 0.03342771196927154, -3.4969618370056983)), ('cz', 0, 1), ('u', 0, (2.7579630279105873, 0.3577326610181091, -3.5003133722816817)), ('u', 1, (-2.957249560331902, 1.5707963164193812, 1.2502471578007306)), ('u', 2, (3.828547128199878, 0.9679475657646686, -0.015246983381037804)), ('u', 3, (-3.135687790525723, 0.18663798162941178, 2.98685190044165)), ('cz', 1, 2), ('u', 0, (-0.6037559517700543, 1.7812646868568505, 0.11177292836514038)), ('u', 1, (0.32054916478035084, 2.3561944955839693, -0.21943436013490597)), ('u', 2, (3.2250316244827566, 1.3524252961998304, 0.09760692779412539)), ('u', 3, (-0.23702659924956304, 0.19006214550015305, 0.0472230812371513)), ('cz', 0, 1), ('u', 0, (-5.115607845480742, 1.570796512443898, 0.0012317600150004715)), ('u', 1, (-2.922158288392045, 1.5707963391768094, 3.142824422943174)), ('u', 2, (-0.23106366257657224, 0.2202677448980035, 0.13150084751164126)), ('u', 3, (3.2464809235972143, 0.02434796807129362, -2.8230973252517617)), ('u', 6, (-0.06746225948287628, 0.7853981633844563, -0.2507098911860657)), ('u', 5, (0.18124741655873963, 0.5082353275601084, -0.09434528861527644)), ('cz', 6, 7), ('u', 6, (-3.407068263943925, 0.8047173284387933, 2.7072653381808935)), ('u', 5, (0.39018601072076603, 0.7132339639159315, 0.7221486932429708)), ('u', 7, (-1.688980545053801, 1.570796322522204, 2.312910443969324)), ('u', 8, (2.514393299545078, 1.6165967071055833, -0.2931801504248077)), ('cz', 7, 8), ('u', 6, (-0.9024627771245186, 0.3669632893695555, 0.12425930097755539)), ('u', 5, (0.657901898113797, 0.16434481730991288, 0.17262796406178665)), ('u', 7, (2.399478549066713, 1.5707963530376834, -3.8050245953288666)), ('u', 8, (0.023556419252689184, 0.45832065200257033, 0.25633374574447043)), ('cz', 6, 7), ('u', 6, (0.16682451861780312, 1.028767456233064, -0.17456622506380476)), ('u', 5, (-2.8026493928028087, 0.13242850764647326, 3.366676540834084)), ('u', 7, (-0.38376560945704047, 1.5707963292420288, -0.6154381402983435)), ('u', 8, (3.070245613906753, 1.4968035902350736, -0.035401561467595766)), ('cz', 7, 8), ('u', 6, (-3.4732923274411043, 2.538583612776887, 2.3300369262566836)), ('u', 5, (2.2515963222755877, 2.1741932214083604, -1.7618854422649062)), ('u', 7, (-2.5483192150662166, 1.0060548649181524, 3.180743520861993)), ('u', 8, (3.936811446702922, 0.7107222159088735, -2.2993679180455397)), ('cz', 6, 5), ('u', 6, (-1.423654896855313, 0.6154797135188552, 2.48865937791916)), ('u', 5, (0.6627455564274238, 1.93676385336916, -1.0628427241989613)), ('u', 7, (3.117346054755425, 1.4098279959518074, -2.6379126093690464)), ('u', 8, (-3.1156974871029286, 0.3891490858631498, 3.1537964370543357)), ('cz', 6, 7), ('u', 6, (-0.3023833422967584, 1.5707963184125138, -0.8083946558111244)), ('u', 5, (4.008714976090217, 0.567851090762625, -1.868490146253322)), ('u', 7, (3.017436792437644, 0.09371733484699354, -3.130344922819383)), ('u', 8, (3.009692432138644, 0.2142042447140045, -3.1080323603649433)), ('cz', 6, 5), ('u', 6, (-0.7624016440798572, 1.5707963262994253, -0.8220834678955425)), ('u', 5, (4.394704516591822, 0.6300800986341344, -0.48930112073814214)), ('u', 7, (2.7546503032052136, 2.0994781439389474, -1.0867316679911312)), ('u', 8, (-2.70970511419967, 0.02865364464025868, 3.389901892054621)), ('cz', 6, 7), ('u', 6, (-0.7487128889945406, 0.7853981632650155, -0.4872397876098541)), ('u', 5, (2.1848866991915945, 1.6613132693471457, -0.3488496323362843)), ('u', 7, (1.533176692547352, 1.6902344164007541, -0.7710178789250411)), ('u', 8, (2.8692827720690337, 0.07928377174801848, -2.9014842548452764)), ('cz', 6, 5), ('u', 6, (-2.654352862115446, 1.5707963180330544, 3.250989223727844)), ('u', 5, (-1.8463678080277535, 1.5707963425617313, 3.250989223871215)), ('u', 7, (-1.071322063789216, 0.4207151334470396, -1.1653445219251615)), ('u', 8, (3.7126578350498187, 1.903993313572438, -0.4016492415987214)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0))]), (10, [('u', 4, (0.0, 2.1598272970111703, 0.0)), ('u', 3, (0.0, 1.0068536854342676, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.0068536854342676, 3.141592653589793)), ('u', 8, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 9), ('u', 0, (-0.15456194594996994, 0.4301992741318007, -0.23274622218789695)), ('u', 1, (-0.40170183590150327, 0.28586108595600335, -0.10570445481115337)), ('u', 2, (-2.78781494008481, 0.8860771265740348, 3.098058177545558)), ('u', 3, (-1.2771129659012126, 3.1415926426306906, -3.8698329862184537)), ('u', 4, (0.06455124714868196, 0.15674351119555924, 0.28885302799265855)), ('cz', 2, 3), ('u', 0, (3.2868320324684976, 0.09166576355737018, -2.7550230712349313)), ('u', 1, (0.0911232712051731, 0.01368910055863399, 0.45056206982018426)), ('u', 2, (-3.11743803269269, 0.9442426455972222, 3.1059072654057824)), ('u', 3, (-3.834956751165805, 1.570796328799766, 2.414320365177579)), ('u', 4, (2.933667700131255, 1.7270351552587933, 0.8611115471393367)), ('cz', 3, 4), ('u', 0, (0.5392459371987273, 0.4234279364041615, -0.5568536770895516)), ('u', 1, (3.42675280329011, 0.19504202148645747, -2.469936624299969)), ('u', 2, (-0.21665352692602918, 0.06015829372792343, -0.250870198673664)), ('u', 3, (-0.8435240423228354, 1.5707963259084061, 0.5588680882268506)), ('u', 4, (2.5536936891906237, 2.2077370023760423, 0.639330425498244)), ('cz', 2, 3), ('u', 0, (-0.17556341147665933, 0.29854234430722804, -1.2499813382380576)), ('u', 1, (0.612744997199851, 0.27514680609283165, 0.7704736055176745)), ('u', 2, (-0.4797695508209159, 1.0354512235988662, -1.0189709592004077)), ('u', 3, (0.5482806295361009, 1.5707963250038932, 0.1461042553833234)), ('u', 4, (3.482560418431404, 0.28111978511638774, -0.5912323081025872)), ('cz', 3, 4), ('u', 0, (2.303733656392368, 0.16148543652229524, -3.9497980814397304)), ('u', 1, (0.5442521716505224, 1.6302319785452546, 0.24725989439266866)), ('u', 2, (-0.9535706167161428, 2.1536597079078486, 0.0430441078844051)), ('u', 3, (3.4803502730890328, 0.06325816915873374, -2.3780977071992337)), ('u', 4, (2.9021206040903733, 1.8902987262289384, 0.15952200511756498)), ('cz', 1, 2), ('u', 0, (-3.5109005070080976, 0.33193395457706537, 2.3789395315381774)), ('u', 1, (3.636835434893288, 1.257269196986108, -2.6156571243980773)), ('u', 2, (-0.19015292244268306, 2.356194493377117, 0.36529779722966144)), ('u', 3, (0.4326275390523066, 0.0296453683871993, 0.23229900119671437)), ('u', 4, (0.33432095242720755, 0.07058137719574267, 0.09040953080297819)), ('cz', 2, 3), ('u', 0, (-0.43561652999785405, 0.21747443117040807, 0.16312617545616148)), ('u', 1, (-3.8758203981800676, 0.6799436095750659, 2.403543745015792)), ('u', 2, (3.8162357421931343, 8.625763986416733e-08, -1.8133656556594009)), ('u', 3, (-2.416262641035493, 0.4444295281254736, 3.395262614647079)), ('u', 4, (-3.0138503212098375, 0.4087035163470796, 2.9946048944212804)), ('cz', 1, 2), ('u', 0, (-0.23775780442139427, 0.5099565361441982, 0.5355028111200876)), ('u', 1, (2.146789898069154, 0.48624615485680545, -3.406300432885364)), ('u', 2, (0.24982598932370687, 1.5707963244964236, 0.5684170629260561)), ('u', 3, (3.3229594617721228, 1.6964847149577291, 0.4279900434809949)), ('u', 4, (0.15588009165819533, 0.29653259188363096, 0.16121153774567248)), ('cz', 2, 3), ('u', 0, (-1.1416819268813243, 0.09141455125431372, 1.5396031513100699)), ('u', 1, (-2.5194174598938748, 2.62722496949545, 1.2581226225791136)), ('u', 2, (-2.694818066981105, 0.25953710529805224, 3.1700764326262645)), ('u', 3, (2.5956633643538476, 1.7834987247116252, 0.3136797140268961)), ('u', 4, (2.847319039171421, 0.358051979844267, -2.5046786388170426)), ('cz', 0, 1), ('u', 0, (-1.305340134791441, 1.0554113399907372, 1.6996120596952016)), ('u', 1, (1.2194909709035284, 2.526112946310009, -1.363022137244877)), ('u', 2, (2.8729346238474727, 2.827897909153286, -0.7655164511322208)), ('u', 3, (3.0256913642491283, 0.13734073576096725, -2.6746616762635864)), ('u', 4, (0.20307460207457123, 0.04167185274926202, 0.03990173394962333)), ('cz', 1, 2), ('u', 0, (3.359957800103702, 0.12131860400987804, -2.651772155709388)), ('u', 1, (-2.733887144360484, 1.570796232525795, 1.3864532688403917)), ('u', 2, (3.4848155449122826, 0.034475470122763266, 0.4975986000164825)), ('u', 3, (2.662002143725979, 0.03342771196927154, -3.4969618370056983)), ('u', 4, (0.04705300198168544, 0.10753595499219139, -0.02655288216926022)), ('cz', 0, 1), ('u', 0, (2.7579630279105873, 0.3577326610181091, -3.5003133722816817)), ('u', 1, (-2.957249560331902, 1.5707963164193812, 1.2502471578007306)), ('u', 2, (3.828547128199878, 0.9679475657646686, -0.015246983381037804)), ('u', 3, (-3.135687790525723, 0.18663798162941178, 2.98685190044165)), ('u', 4, (0.15909332685512304, 0.14159030335540024, 0.30778373492233435)), ('cz', 1, 2), ('u', 0, (-0.6037559517700543, 1.7812646868568505, 0.11177292836514038)), ('u', 1, (0.32054916478035084, 2.3561944955839693, -0.21943436013490597)), ('u', 2, (3.2250316244827566, 1.3524252961998304, 0.09760692779412539)), ('u', 3, (-0.23702659924956304, 0.19006214550015305, 0.0472230812371513)), ('u', 4, (-2.44397665074765, 0.21949807884460792, 2.9088773305295414)), ('cz', 0, 1), ('u', 0, (-5.115607845480742, 1.570796512443898, 0.0012317600150004715)), ('u', 1, (-2.922158288392045, 1.5707963391768094, 3.142824422943174)), ('u', 2, (-0.23106366257657224, 0.2202677448980035, 0.13150084751164126)), ('u', 3, (3.2464809235972143, 0.02434796807129362, -2.8230973252517617)), ('u', 4, (0.5585089733958291, 0.19700659298892828, 0.06552985415781196)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (0.0, 1.5707963267948966, 0.0)), ('u', 8, (0.0, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 8, (2.983475401827938e-18, 0.886077123792614, -1.140057778643436e-16)), ('u', 7, (-3.141592653589793, 0.886077123792614, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 7, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (-3.141592653589793, 1.1071487177940904, 3.141592653589793)), ('cz', 9, 8), ('u', 8, (0.0, 1.1071487177940904, 0.0)), ('u', 8, (-3.141592653589793, 2.677945044588987, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 2.677945044588987, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 6, (-0.06746225948287628, 0.7853981633844563, -0.2507098911860657)), ('u', 5, (0.18124741655873963, 0.5082353275601084, -0.09434528861527644)), ('u', 7, (0.7394044437328378, 1.8451179555702133e-09, 0.11433020946755884)), ('u', 8, (0.05115974370160227, 0.07248402517931575, -0.2604938422021681)), ('cz', 6, 7), ('u', 6, (-3.407068263943925, 0.8047173284387933, 2.7072653381808935)), ('u', 5, (0.39018601072076603, 0.7132339639159315, 0.7221486932429708)), ('u', 7, (-1.688980545053801, 1.570796322522204, 2.312910443969324)), ('u', 8, (2.514393299545078, 1.6165967071055833, -0.2931801504248077)), ('cz', 7, 8), ('u', 6, (-0.9024627771245186, 0.3669632893695555, 0.12425930097755539)), ('u', 5, (0.657901898113797, 0.16434481730991288, 0.17262796406178665)), ('u', 7, (2.399478549066713, 1.5707963530376834, -3.8050245953288666)), ('u', 8, (0.023556419252689184, 0.45832065200257033, 0.25633374574447043)), ('cz', 6, 7), ('u', 6, (0.16682451861780312, 1.028767456233064, -0.17456622506380476)), ('u', 5, (-2.8026493928028087, 0.13242850764647326, 3.366676540834084)), ('u', 7, (-0.38376560945704047, 1.5707963292420288, -0.6154381402983435)), ('u', 8, (3.070245613906753, 1.4968035902350736, -0.035401561467595766)), ('cz', 7, 8), ('u', 6, (-3.4732923274411043, 2.538583612776887, 2.3300369262566836)), ('u', 5, (2.2515963222755877, 2.1741932214083604, -1.7618854422649062)), ('u', 7, (-2.5483192150662166, 1.0060548649181524, 3.180743520861993)), ('u', 8, (3.936811446702922, 0.7107222159088735, -2.2993679180455397)), ('cz', 6, 5), ('u', 6, (-1.423654896855313, 0.6154797135188552, 2.48865937791916)), ('u', 5, (0.6627455564274238, 1.93676385336916, -1.0628427241989613)), ('u', 7, (3.117346054755425, 1.4098279959518074, -2.6379126093690464)), ('u', 8, (-3.1156974871029286, 0.3891490858631498, 3.1537964370543357)), ('cz', 6, 7), ('u', 6, (-0.3023833422967584, 1.5707963184125138, -0.8083946558111244)), ('u', 5, (4.008714976090217, 0.567851090762625, -1.868490146253322)), ('u', 7, (3.017436792437644, 0.09371733484699354, -3.130344922819383)), ('u', 8, (3.009692432138644, 0.2142042447140045, -3.1080323603649433)), ('cz', 6, 5), ('u', 6, (-0.7624016440798572, 1.5707963262994253, -0.8220834678955425)), ('u', 5, (4.394704516591822, 0.6300800986341344, -0.48930112073814214)), ('u', 7, (2.7546503032052136, 2.0994781439389474, -1.0867316679911312)), ('u', 8, (-2.70970511419967, 0.02865364464025868, 3.389901892054621)), ('cz', 6, 7), ('u', 6, (-0.7487128889945406, 0.7853981632650155, -0.4872397876098541)), ('u', 5, (2.1848866991915945, 1.6613132693471457, -0.3488496323362843)), ('u', 7, (1.533176692547352, 1.6902344164007541, -0.7710178789250411)), ('u', 8, (2.8692827720690337, 0.07928377174801848, -2.9014842548452764)), ('cz', 6, 5), ('u', 6, (-2.654352862115446, 1.5707963180330544, 3.250989223727844)), ('u', 5, (-1.8463678080277535, 1.5707963425617313, 3.250989223871215)), ('u', 7, (-1.071322063789216, 0.4207151334470396, -1.1653445219251615)), ('u', 8, (3.7126578350498187, 1.903993313572438, -0.4016492415987214)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0))])], (10, 3): [(10, [('u', 4, (0.0, 2.555907110132642, 0.0)), ('u', 3, (0.0, 0.7398807743787406, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 0.7398807743787406, 3.141592653589793)), ('u', 2, (0.0, 1.1502619915109316, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.1502619915109316, 3.141592653589793)), ('u', 7, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 7), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 9), ('u', 5, (-0.38584822121544227, 0.2211811861684394, -0.29244801111232555)), ('u', 6, (1.4827404767000143, 3.043807953410148, -2.426166083706404)), ('u', 7, (0.31310696020471074, 1.5707963264369234, -2.622445565947)), ('u', 8, (1.9494198955258506, 1.570796327428522, 1.7859632426319192)), ('u', 9, (4.795817777973742, 2.215723293597476, -0.22290181124514907)), ('cz', 7, 8), ('u', 5, (-0.45121414565223106, 0.9092493831267439, -1.0278998517941713)), ('u', 6, (-0.7943833925753162, 0.846315370603616, -1.2380036043670477)), ('u', 7, (-2.0899434169371096, 1.570796325998072, 3.022656349667437)), ('u', 8, (1.3556294098946233, 0.8860771203995191, 1.4673473030143767)), ('u', 9, (0.37540989449619083, 0.2855229083849816, -0.28813760081112766)), ('cz', 7, 8), ('u', 5, (1.8784179331081536, 0.8117377452440503, -3.8881253496193318)), ('u', 6, (-0.25314686727403224, 0.5631704631930264, 0.6174402621357059)), ('u', 7, (-0.41862801922666876, 0.895834548341286, 0.4731275368577936)), ('u', 8, (3.245041677772415, 1.5707963282091644, -1.3316488776256254)), ('u', 9, (3.242789340217199, 2.427575908395239, 0.24597426547568135)), ('cz', 8, 9), ('u', 5, (-3.156014573321769, 0.6424434729913439, 3.0135003528388826)), ('u', 6, (-3.6113558126897325, 0.812127133019373, 2.4527436951454162)), ('u', 7, (-0.7060351346072904, 0.6883537371818029, -1.5466435805854484)), ('u', 8, (1.813984879263263, 2.146924794895197, -3.730588958668428)), ('u', 9, (3.801633307771413, 1.5475762429571984, 0.9510298120459608)), ('cz', 7, 8), ('u', 5, (-3.636764826237095, 0.23592658278733264, 2.1767598281029414)), ('u', 6, (2.9385728591527926, 0.0227712554169746, -3.0896209301796334)), ('u', 7, (-1.0196370802563148, 0.01578052085578273, -0.4614215977662744)), ('u', 8, (-0.017841236995828985, 0.13486494043153252, 0.20754517818108564)), ('u', 9, (0.579510060909405, 0.5233836834233824, -0.3661419724292343)), ('cz', 8, 9), ('u', 5, (-0.1018834467788523, 0.20022920659444027, -0.48709114176265966)), ('u', 6, (-0.616810049235738, 0.5430871505857402, 0.4815744936227356)), ('u', 7, (-2.680170678092535, 0.0157805219140349, 2.6793194357676904)), ('u', 8, (0.40357235980117123, 1.68147216031968, -0.1567287388800335)), ('u', 9, (0.38045479537720506, 0.15597240649573132, -0.7292228022826384)), ('cz', 7, 8), ('u', 5, (2.556472761615485, 0.14064238238093768, -3.0864741397069353)), ('u', 6, (2.407487302967441, 0.2474994689584679, -2.4965470777081356)), ('u', 7, (-3.6505033563318907, 1.0489234817020243, 2.2343418875861385)), ('u', 8, (-2.9076383253237834, 0.5761284825610413, 2.91759543033778)), ('u', 9, (2.6056178669444936, 1.6180718649598544, -0.2654152865570627)), ('cz', 8, 9), ('u', 0, (2.4307492535810495, 0.382735021280294, -2.685595804019862)), ('u', 1, (1.5334751529439385, 0.6847192088220965, -1.3118306953989203)), ('u', 2, (-0.4663141351993828, 1.175063987664736e-07, 2.428157695694881)), ('u', 9, (-3.0834345580534372, 2.173894961309956, 2.8232070602932438)), ('u', 3, (0.08074981547483007, 0.10422874954607676, -0.2528179011218028)), ('cz', 1, 2), ('u', 0, (-2.042760743789615, 0.1432778091095904, 2.9255623127093946)), ('u', 1, (-1.6634198584432751, 1.0965282895434323, 1.6047900364446115)), ('u', 2, (-1.9904885761310007, 1.5707961400747807, 1.2550406509284864)), ('u', 9, (-0.1721109199519451, 0.5928349417925829, 0.6945332217951719)), ('u', 3, (3.2844235602375167, 1.6743999038596675, 0.3486460459531253)), ('cz', 2, 3), ('u', 0, (-0.6264929597268192, 0.2943738216382789, 0.08212472745748922)), ('u', 1, (1.7902744505822321, 0.4307999859082922, -3.2523774088318484)), ('u', 2, (1.8865522349542323, 0.6847192023364564, -1.332690106010211)), ('u', 9, (0.38105366621983433, 0.5844334129472873, -0.4563565934645994)), ('u', 3, (-0.4442820407894194, 0.3977351924263156, 0.11037317434032629)), ('cz', 1, 2), ('u', 0, (-0.3904646721552152, 0.33844925518519037, 0.2959481802814933)), ('u', 1, (-1.3191808883785296, 1.991736659093761e-07, -0.24149657205000363)), ('u', 2, (1.3326898789346007, 1.5707963256587345, -1.1704700432653918)), ('u', 9, (3.2452348925771375, 0.36190421640876075, -3.011079690925734)), ('u', 3, (3.099228723952163, 1.2827034888293747, 0.09866860846856218)), ('cz', 2, 3), ('u', 0, (-2.9411605685421436, 0.23633900245369577, 3.026507290011662)), ('u', 1, (-3.312071752997454, 1.6115410786626674e-07, 1.51009503069265)), ('u', 2, (-1.9711224574666883, 1.570796491322578, 0.02864503297721166)), ('u', 9, (-2.9359938106371675, 0.06159871960088969, 2.7958655896942326)), ('u', 3, (3.0769118970815748, 1.5707963275585246, 0.028645064910122642)), ('u', 5, (-0.061979532898852496, 1.8596711848923104, 1.0907368589844846)), ('u', 6, (3.135489257833875, 1.43155634063744, -2.7702454632021674)), ('u', 7, (-3.4021278028029514, 1.0089303321120369, 2.5641201101780604)), ('u', 8, (-0.6566290203415203, 1.3168514397079387, 1.248014187815818)), ('cz', 6, 7), ('u', 5, (-4.345985116749786, 0.6771638494770151, 1.4329239714110154)), ('u', 6, (0.40271283825059473, 0.30821852105401326, -0.08775499054813318)), ('u', 7, (4.987281462389907, 0.7126165553766421, -0.20945527381873896)), ('u', 8, (-1.2480141874323187, 1.824741215200119, -4.545914725382517)), ('cz', 7, 8), ('u', 5, (-3.943875358284748, 0.026429428864854096, 2.177140468424045)), ('u', 6, (-0.377782046032458, 0.5015413742436112, 1.418175130143942)), ('u', 7, (-3.296776313369426, 2.303037799120235, 2.4980094675433824)), ('u', 8, (1.5897032029757472, 0.32081820966660496, 3.257326295688869)), ('cz', 6, 7), ('u', 5, (-1.1123529633057307, 1.0084675472327052, 0.5036375918436751)), ('u', 6, (-2.4092976375572297, 0.29699373484869324, 3.2616132794360393)), ('u', 7, (0.19740019460754038, 1.3424784047298337, 0.03035496642749229)), ('u', 8, (-0.11573364189659674, 0.3208182069601935, -1.0024142315579798)), ('cz', 7, 8), ('u', 5, (-0.9642018244100751, 1.652947106740747, -0.3875912357043)), ('u', 6, (1.1312299549314246, 1.5690002365664493, -3.677624708964337)), ('u', 7, (-0.12041746536495779, 1.9774874745100484, -0.024615485231979907)), ('u', 8, (2.3184507235604643, 1.6004447013299943, -1.9858345388299643)), ('cz', 6, 7), ('u', 5, (-4.189187694543731, 0.01608747047344722, 1.7035009056321235)), ('u', 6, (-1.3311080416588619, 0.8167328732075856, -1.0651392664980694)), ('u', 7, (0.598127639660322, 1.8251455671510737, 1.2832044499062079)), ('u', 8, (3.9825371332741533, 0.07166053457273007, -0.8517301547696945)), ('cz', 7, 8), ('u', 5, (0.10824362465687432, 1.3373721575830104, -2.1472774927977465)), ('u', 6, (-2.3332213860738715, 1.7590475707637074, 2.5347229916048555)), ('u', 7, (3.2073603083679414, 0.012008413303980569, -2.797164814965747)), ('u', 8, (1.576094926696124, 1.2277221046061755, 0.4939019732727694)), ('cz', 5, 6), ('u', 5, (1.822188628376869, 0.6207646234304489, -3.1986642814110504)), ('u', 6, (-3.20133081629888, 0.9562595619669652, 2.7961919463146514)), ('u', 7, (-0.17696748259573425, 0.008489625960196055, 0.23826803226148563)), ('u', 8, (4.155457210185608, 0.7835477934640798, -2.1069012917891996)), ('cz', 5, 6), ('u', 5, (-3.2073805745577353, 0.5369906613011859, 2.420013212130006)), ('u', 6, (-0.43998734787481303, 1.5663005055724715, -1.2126501898138728)), ('u', 7, (2.3648788893246353, 3.137689583356945, 1.4214190228187336)), ('u', 8, (-2.9451564275852777, 0.02426557509894898, 2.961334569336753)), ('cz', 6, 7), ('u', 5, (-2.9884875092409935, 1.1501208166738328, 1.1517467537741446)), ('u', 6, (-1.620985717787917, 1.5707963368374935, 3.477764269126838)), ('u', 7, (3.387223075215565, 1.5850592545989293, -0.5257424707291327)), ('u', 8, (-0.3806827269148584, 0.6485147935772614, 0.8304607696020493)), ('cz', 5, 6), ('u', 5, (-0.5704140322005489, 1.575306984226174, -1.1539225289548274)), ('u', 6, (-0.3361716169029527, 1.5707963226423183, 0.789670426224647)), ('u', 7, (-2.960758163449333, 0.015155354385545243, 2.6575804060988975)), ('u', 8, (-2.7974449019912546, 0.8620425720375742, 3.4122812560149027)), ('cz', 6, 7), ('u', 5, (-0.538845199860207, 0.037056367530225794, 0.9768855445933508)), ('u', 6, (-0.4772177969805509, 1.5707963260645312, -1.109293471308753)), ('u', 7, (-2.4865917150287173, 0.7684770292093597, 3.5918803837020348)), ('u', 8, (3.0767988609039123, 0.10454783373934436, -2.172467962480892)), ('cz', 5, 6), ('u', 5, (1.676559413723329, 3.141592650312581, -3.785176873571088)), ('u', 6, (-2.032299181287651, 1.5707963224417993, 1.5374141667969234)), ('u', 7, (2.9309278567956456, 0.8167961746964582, -3.3406907636955854)), ('u', 8, (3.7605025782840062, 0.5522816535036469, -2.476368423184989)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 0.23182380450040305, 0.0)), ('cz', 4, 3), ('u', 3, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 1.3389725222944935, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 1, (-1.157682930018697, 1.570796232420874, 0.16383364398588174)), ('u', 0, (-0.004215143456711812, 0.27504138738109657, 0.9887877808330129)), ('u', 2, (1.9839099136957876, 1.5707958898611574, -0.35995401416586603)), ('u', 3, (2.8040551642619707, 0.09076996647784621, -3.24839038551328)), ('u', 9, (-0.44068000011409275, 0.7064062699700188, -0.3800634895546903)), ('cz', 1, 2), ('u', 1, (2.8725267744579788, 2.3534209195273648, -1.5997301787773996)), ('u', 0, (1.0217444747852347, 0.18409888134816213, 0.3566268879511885)), ('u', 2, (3.5015464293404586, 0.7853981847844901, 0.5123325325200401)), ('u', 3, (0.11952742055733812, 0.4354958679397325, -0.5836395300747153)), ('u', 9, (3.813160210388557, 0.507635066492853, -2.181363033305418)), ('cz', 1, 2), ('u', 1, (1.5001077884467588, 1.6399766523798136, -0.261746846809328)), ('u', 0, (1.0578373226925986, 0.4083982346122096, 0.1661261193636897)), ('u', 2, (0.6340189923419296, 1.180104230225904e-06, 1.5977212026188932)), ('u', 3, (-0.2790337368679006, 1.3422484339261425, -1.1858342809425126)), ('u', 9, (0.16566156973504104, 0.1727609523662677, 0.2538563847160517)), ('cz', 2, 3), ('u', 1, (3.052653634757833, 0.07364771671056473, -2.324613780016221)), ('u', 0, (-0.03662032701667063, 0.03267549985197457, -0.14556005760455298)), ('u', 2, (-2.7440728649487314, 1.5707964040727442, 2.959122679397897)), ('u', 3, (2.039176874760883, 2.071228241259465, -0.49725998468781574)), ('u', 9, (-0.06847593687362333, 0.13529071764441677, 0.21707377959629157)), ('cz', 1, 2), ('u', 1, (0.32429004005521583, 0.44783869861422226, 0.02850133328139598)), ('u', 0, (2.8430816899146034, 0.7210059170128215, -3.210976252103685)), ('u', 2, (-3.0874157442904195, 0.5283896488131657, 3.1121078342708133)), ('u', 3, (-0.8814937707977086, 0.5155523632050678, 0.11353671590213882)), ('u', 9, (0.2855340876094016, 0.40452197469633216, 0.2753135405950782)), ('cz', 2, 3), ('u', 1, (-2.521460088407784, 1.9364132779344339, 1.3244795355958967)), ('u', 0, (0.7034386955532794, 0.9918026372136939, -0.9333802760400325)), ('u', 2, (-2.0448006079124297, 0.57652473832758, 3.006706170087526)), ('u', 3, (3.368700440849259, 0.8471561412910804, 0.03754047392551629)), ('u', 9, (2.2174760399356357, 0.03974587239353034, -2.4990022170216637)), ('cz', 1, 0), ('u', 1, (2.081778051450047, 1.5707963097276876, -1.2536611395630106)), ('u', 0, (1.4254844448667305, 0.5595360666651668, -0.28406281651026577)), ('u', 2, (2.274758914816376, 1.2337924653890138, -0.7078803759699305)), ('u', 3, (-1.9888289681995333, 0.26870278512744167, 2.608757565038483)), ('u', 9, (3.141122293451506, 0.5806536621427302, -2.526268948429532)), ('cz', 1, 2), ('u', 1, (-1.8879321230535338, 1.570796441383654, 3.4262275638048294)), ('u', 0, (0.2532077664601077, 1.0114745322643588, -0.5667616742641588)), ('u', 2, (4.369346397457467, 1.5039298692274967, -0.9472795294820509)), ('u', 3, (2.928539636167617, 1.1668217913594034, -2.8827373493446755)), ('u', 9, (-0.5229982156127925, 0.4529594874531066, 0.8355675629479081)), ('cz', 1, 0), ('u', 1, (-3.426227945786745, 0.9553167725861532, 1.323734155584405)), ('u', 0, (-2.988002240727035, 1.0839383589893479, 3.2916508455825055)), ('u', 2, (-0.15951717403952737, 0.14853354326381862, -1.1517835159269145)), ('u', 3, (-3.39282150811932, 0.10890998112115866, 2.47733024208187)), ('u', 9, (2.4128514823875644, 0.4035482711993217, -2.4191821168976917)), ('cz', 1, 2), ('u', 1, (1.8178585292618863, 1.5707961930875172, -3.6557030282552176)), ('u', 0, (-3.9629839195608256, 0.5944850995287304, 2.207040579406246)), ('u', 2, (-0.4609003513921515, 0.047224906300792825, -1.255343546578333)), ('u', 3, (-3.323698351573585, 0.016866910769449975, 2.4597035626176598)), ('u', 9, (0.9155195997373274, 0.4257574647214635, 0.9487812735230851)), ('cz', 1, 0), ('u', 1, (-1.056685911573044, 1.5707960819558777, 0.2413029698982394)), ('u', 0, (0.15204261683702225, 0.710945842629969, -0.1932796659376652)), ('u', 2, (2.443823543167665, 0.7668078092958376, -0.15446113057065025)), ('u', 3, (3.152544162081301, 0.06921678313758098, -2.707146757278119)), ('u', 9, (-2.189715179081454, 0.006216121916662596, 3.0544362043273754)), ('cz', 1, 2), ('u', 1, (1.3294933774826518, 1.5707966361997991, -0.24268436150914785)), ('u', 0, (-0.07607960297898715, 0.8779562367770902, -0.18710437024866738)), ('u', 2, (3.274773237736706, 1.5751361791009502, -1.2220784719144766)), ('u', 3, (-2.486283416937551, 0.09965581758383801, 2.0179397778925874)), ('u', 9, (-2.6064816450916135, 0.5092818998925137, 3.450993121391644)), ('cz', 1, 0), ('u', 1, (3.3842769882378647, 1.5707963003167214, -2.0738666647087607)), ('u', 0, (-1.2091543984168034, 1.570796196684069, 4.20931903432824)), ('u', 2, (-0.45488960867013883, 0.04093109952848174, -0.39680989728551797)), ('u', 3, (-0.06045815905080046, 1.0822124657976042, 0.526672739318211)), ('u', 9, (0.1944624620792588, 0.5336235278273715, 0.07405021431980433)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0))]), (10, [('u', 4, (0.0, 2.555907110132642, 0.0)), ('u', 3, (0.0, 0.7398807743787406, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 0.7398807743787406, 3.141592653589793)), ('u', 2, (0.0, 1.1502619915109316, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.1502619915109316, 3.141592653589793)), ('u', 7, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 7), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 9), ('u', 5, (-0.38584822121544227, 0.2211811861684394, -0.29244801111232555)), ('u', 6, (1.4827404767000143, 3.043807953410148, -2.426166083706404)), ('u', 7, (0.31310696020471074, 1.5707963264369234, -2.622445565947)), ('u', 8, (1.9494198955258506, 1.570796327428522, 1.7859632426319192)), ('u', 9, (4.795817777973742, 2.215723293597476, -0.22290181124514907)), ('cz', 7, 8), ('u', 5, (-0.45121414565223106, 0.9092493831267439, -1.0278998517941713)), ('u', 6, (-0.7943833925753162, 0.846315370603616, -1.2380036043670477)), ('u', 7, (-2.0899434169371096, 1.570796325998072, 3.022656349667437)), ('u', 8, (1.3556294098946233, 0.8860771203995191, 1.4673473030143767)), ('u', 9, (0.37540989449619083, 0.2855229083849816, -0.28813760081112766)), ('cz', 7, 8), ('u', 5, (1.8784179331081536, 0.8117377452440503, -3.8881253496193318)), ('u', 6, (-0.25314686727403224, 0.5631704631930264, 0.6174402621357059)), ('u', 7, (-0.41862801922666876, 0.895834548341286, 0.4731275368577936)), ('u', 8, (3.245041677772415, 1.5707963282091644, -1.3316488776256254)), ('u', 9, (3.242789340217199, 2.427575908395239, 0.24597426547568135)), ('cz', 8, 9), ('u', 5, (-3.156014573321769, 0.6424434729913439, 3.0135003528388826)), ('u', 6, (-3.6113558126897325, 0.812127133019373, 2.4527436951454162)), ('u', 7, (-0.7060351346072904, 0.6883537371818029, -1.5466435805854484)), ('u', 8, (1.813984879263263, 2.146924794895197, -3.730588958668428)), ('u', 9, (3.801633307771413, 1.5475762429571984, 0.9510298120459608)), ('cz', 7, 8), ('u', 5, (-3.636764826237095, 0.23592658278733264, 2.1767598281029414)), ('u', 6, (2.9385728591527926, 0.0227712554169746, -3.0896209301796334)), ('u', 7, (-1.0196370802563148, 0.01578052085578273, -0.4614215977662744)), ('u', 8, (-0.017841236995828985, 0.13486494043153252, 0.20754517818108564)), ('u', 9, (0.579510060909405, 0.5233836834233824, -0.3661419724292343)), ('cz', 8, 9), ('u', 5, (-0.1018834467788523, 0.20022920659444027, -0.48709114176265966)), ('u', 6, (-0.616810049235738, 0.5430871505857402, 0.4815744936227356)), ('u', 7, (-2.680170678092535, 0.0157805219140349, 2.6793194357676904)), ('u', 8, (0.40357235980117123, 1.68147216031968, -0.1567287388800335)), ('u', 9, (0.38045479537720506, 0.15597240649573132, -0.7292228022826384)), ('cz', 7, 8), ('u', 5, (2.556472761615485, 0.14064238238093768, -3.0864741397069353)), ('u', 6, (2.407487302967441, 0.2474994689584679, -2.4965470777081356)), ('u', 7, (-3.6505033563318907, 1.0489234817020243, 2.2343418875861385)), ('u', 8, (-2.9076383253237834, 0.5761284825610413, 2.91759543033778)), ('u', 9, (2.6056178669444936, 1.6180718649598544, -0.2654152865570627)), ('cz', 8, 9), ('u', 0, (2.4307492535810495, 0.382735021280294, -2.685595804019862)), ('u', 1, (1.5334751529439385, 0.6847192088220965, -1.3118306953989203)), ('u', 2, (-0.4663141351993828, 1.175063987664736e-07, 2.428157695694881)), ('u', 9, (-3.0834345580534372, 2.173894961309956, 2.8232070602932438)), ('u', 3, (0.08074981547483007, 0.10422874954607676, -0.2528179011218028)), ('cz', 1, 2), ('u', 0, (-2.042760743789615, 0.1432778091095904, 2.9255623127093946)), ('u', 1, (-1.6634198584432751, 1.0965282895434323, 1.6047900364446115)), ('u', 2, (-1.9904885761310007, 1.5707961400747807, 1.2550406509284864)), ('u', 9, (-0.1721109199519451, 0.5928349417925829, 0.6945332217951719)), ('u', 3, (3.2844235602375167, 1.6743999038596675, 0.3486460459531253)), ('cz', 2, 3), ('u', 0, (-0.6264929597268192, 0.2943738216382789, 0.08212472745748922)), ('u', 1, (1.7902744505822321, 0.4307999859082922, -3.2523774088318484)), ('u', 2, (1.8865522349542323, 0.6847192023364564, -1.332690106010211)), ('u', 9, (0.38105366621983433, 0.5844334129472873, -0.4563565934645994)), ('u', 3, (-0.4442820407894194, 0.3977351924263156, 0.11037317434032629)), ('cz', 1, 2), ('u', 0, (-0.3904646721552152, 0.33844925518519037, 0.2959481802814933)), ('u', 1, (-1.3191808883785296, 1.991736659093761e-07, -0.24149657205000363)), ('u', 2, (1.3326898789346007, 1.5707963256587345, -1.1704700432653918)), ('u', 9, (3.2452348925771375, 0.36190421640876075, -3.011079690925734)), ('u', 3, (3.099228723952163, 1.2827034888293747, 0.09866860846856218)), ('cz', 2, 3), ('u', 0, (-2.9411605685421436, 0.23633900245369577, 3.026507290011662)), ('u', 1, (-3.312071752997454, 1.6115410786626674e-07, 1.51009503069265)), ('u', 2, (-1.9711224574666883, 1.570796491322578, 0.02864503297721166)), ('u', 9, (-2.9359938106371675, 0.06159871960088969, 2.7958655896942326)), ('u', 3, (3.0769118970815748, 1.5707963275585246, 0.028645064910122642)), ('u', 5, (-0.061979532898852496, 1.8596711848923104, 1.0907368589844846)), ('u', 6, (3.135489257833875, 1.43155634063744, -2.7702454632021674)), ('u', 7, (-3.4021278028029514, 1.0089303321120369, 2.5641201101780604)), ('u', 8, (-0.6566290203415203, 1.3168514397079387, 1.248014187815818)), ('cz', 6, 7), ('u', 5, (-4.345985116749786, 0.6771638494770151, 1.4329239714110154)), ('u', 6, (0.40271283825059473, 0.30821852105401326, -0.08775499054813318)), ('u', 7, (4.987281462389907, 0.7126165553766421, -0.20945527381873896)), ('u', 8, (-1.2480141874323187, 1.824741215200119, -4.545914725382517)), ('cz', 7, 8), ('u', 5, (-3.943875358284748, 0.026429428864854096, 2.177140468424045)), ('u', 6, (-0.377782046032458, 0.5015413742436112, 1.418175130143942)), ('u', 7, (-3.296776313369426, 2.303037799120235, 2.4980094675433824)), ('u', 8, (1.5897032029757472, 0.32081820966660496, 3.257326295688869)), ('cz', 6, 7), ('u', 5, (-1.1123529633057307, 1.0084675472327052, 0.5036375918436751)), ('u', 6, (-2.4092976375572297, 0.29699373484869324, 3.2616132794360393)), ('u', 7, (0.19740019460754038, 1.3424784047298337, 0.03035496642749229)), ('u', 8, (-0.11573364189659674, 0.3208182069601935, -1.0024142315579798)), ('cz', 7, 8), ('u', 5, (-0.9642018244100751, 1.652947106740747, -0.3875912357043)), ('u', 6, (1.1312299549314246, 1.5690002365664493, -3.677624708964337)), ('u', 7, (-0.12041746536495779, 1.9774874745100484, -0.024615485231979907)), ('u', 8, (2.3184507235604643, 1.6004447013299943, -1.9858345388299643)), ('cz', 6, 7), ('u', 5, (-4.189187694543731, 0.01608747047344722, 1.7035009056321235)), ('u', 6, (-1.3311080416588619, 0.8167328732075856, -1.0651392664980694)), ('u', 7, (0.598127639660322, 1.8251455671510737, 1.2832044499062079)), ('u', 8, (3.9825371332741533, 0.07166053457273007, -0.8517301547696945)), ('cz', 7, 8), ('u', 5, (0.10824362465687432, 1.3373721575830104, -2.1472774927977465)), ('u', 6, (-2.3332213860738715, 1.7590475707637074, 2.5347229916048555)), ('u', 7, (3.2073603083679414, 0.012008413303980569, -2.797164814965747)), ('u', 8, (1.576094926696124, 1.2277221046061755, 0.4939019732727694)), ('cz', 5, 6), ('u', 5, (1.822188628376869, 0.6207646234304489, -3.1986642814110504)), ('u', 6, (-3.20133081629888, 0.9562595619669652, 2.7961919463146514)), ('u', 7, (-0.17696748259573425, 0.008489625960196055, 0.23826803226148563)), ('u', 8, (4.155457210185608, 0.7835477934640798, -2.1069012917891996)), ('cz', 5, 6), ('u', 5, (-3.2073805745577353, 0.5369906613011859, 2.420013212130006)), ('u', 6, (-0.43998734787481303, 1.5663005055724715, -1.2126501898138728)), ('u', 7, (2.3648788893246353, 3.137689583356945, 1.4214190228187336)), ('u', 8, (-2.9451564275852777, 0.02426557509894898, 2.961334569336753)), ('cz', 6, 7), ('u', 5, (-2.9884875092409935, 1.1501208166738328, 1.1517467537741446)), ('u', 6, (-1.620985717787917, 1.5707963368374935, 3.477764269126838)), ('u', 7, (3.387223075215565, 1.5850592545989293, -0.5257424707291327)), ('u', 8, (-0.3806827269148584, 0.6485147935772614, 0.8304607696020493)), ('cz', 5, 6), ('u', 5, (-0.5704140322005489, 1.575306984226174, -1.1539225289548274)), ('u', 6, (-0.3361716169029527, 1.5707963226423183, 0.789670426224647)), ('u', 7, (-2.960758163449333, 0.015155354385545243, 2.6575804060988975)), ('u', 8, (-2.7974449019912546, 0.8620425720375742, 3.4122812560149027)), ('cz', 6, 7), ('u', 5, (-0.538845199860207, 0.037056367530225794, 0.9768855445933508)), ('u', 6, (-0.4772177969805509, 1.5707963260645312, -1.109293471308753)), ('u', 7, (-2.4865917150287173, 0.7684770292093597, 3.5918803837020348)), ('u', 8, (3.0767988609039123, 0.10454783373934436, -2.172467962480892)), ('cz', 5, 6), ('u', 5, (1.676559413723329, 3.141592650312581, -3.785176873571088)), ('u', 6, (-2.032299181287651, 1.5707963224417993, 1.5374141667969234)), ('u', 7, (2.9309278567956456, 0.8167961746964582, -3.3406907636955854)), ('u', 8, (3.7605025782840062, 0.5522816535036469, -2.476368423184989)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 0.23182380450040305, 0.0)), ('cz', 4, 3), ('u', 3, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 1.3389725222944935, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (0.0, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.5235987755982989, 0.0)), ('u', 0, (0.0, 0.5235987755982989, 0.0)), ('cz', 0, 1), ('u', 1, (0.0, 1.5707963267948966, 0.0)), ('u', 0, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 0.7853981633974484, 3.141592653589793)), ('cz', 2, 1), ('u', 1, (0.0, 0.7853981633974484, 0.0)), ('u', 1, (-3.141592653589793, 2.356194490192345, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 2.356194490192345, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.3089969389957472, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 0.26179938779914935, 0.0)), ('cz', 3, 2), ('u', 2, (-3.141592653589793, 0.26179938779914935, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 1.3089969389957472, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 1, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 0, 1), ('u', 1, (1.570796326794898, 0.6154797086703868, -1.570796326794898)), ('u', 0, (1.5707963267948966, 0.6154797086703876, -1.5707963267948966)), ('cz', 0, 1), ('u', 1, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 0, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 1, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.2630564724597029, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.30773985433519363, 0.0)), ('cz', 2, 1), ('u', 1, (-3.141592653589793, 0.30773985433519363, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 1.2630564724597029, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (0.0, 1.5707963267948966, 0.0)), ('u', 1, (0.0, 1.5707963267948966, 0.0)), ('cz', 0, 1), ('u', 1, (1.2251903831398114e-16, 0.7853981633974483, -1.1496735851465472e-17)), ('u', 0, (-3.141592653589793, 0.7853981633974483, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 0, (-3.141592653589793, 1.5707963267948966, 3.141592653589793))])], (10, 4): [(10, [('u', 4, (0.0, 2.8317480138481654, 0.0)), ('u', 3, (0.0, 0.5165288251587206, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 0.5165288251587206, 3.141592653589793)), ('u', 2, (0.0, 0.9326801236225959, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 0.9326801236225959, 3.141592653589793)), ('u', 1, (0.0, 1.2645189576252271, 0.0)), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 1), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.2645189576252271, 3.141592653589793)), ('u', 6, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 6), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 7), ('u', 5, (1.8203294960993182, 2.677945044036175, -1.8726346573026542)), ('u', 6, (-2.8527502421963247, 7.767805834013236e-09, 2.2407057509610437)), ('u', 7, (2.266957442004523, 1.351311450015386, 0.060408582122242205)), ('cz', 5, 6), ('u', 5, (-1.521208529254434, 2.7048584950968224, 1.5534958307607207)), ('u', 6, (1.3222370962136798, 1.570796326674144, -2.4354373568466396)), ('u', 7, (3.6662745746859313, 0.8587857431592025, -0.4029376878622457)), ('cz', 6, 7), ('u', 5, (0.14283356432439562, 0.1127412723768717, -0.1030904432079215)), ('u', 6, (2.4354373511378737, 0.8860771634267568, -3.325249497698744)), ('u', 7, (-0.8074499399882608, 0.4206852718680495, -0.7671415121757957)), ('cz', 5, 6), ('u', 5, (-2.5045227157849266, 4.404126520821067e-10, 3.3023858529270123)), ('u', 6, (0.18365684177050237, 1.5707963727179184, 0.08646004947475114)), ('u', 7, (2.9718455046805703, 0.8679434635999083, -0.2213392088174111)), ('cz', 6, 7), ('u', 5, (0.6212197864437758, 2.924553262063664e-09, -0.8082872630654478)), ('u', 6, (-0.08646003932546953, 1.5707963374248273, -0.7101926010205286)), ('u', 7, (2.8157706154228186, 1.5707963265337945, -0.710192603402009)), ('u', 8, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 8), ('u', 1, (-3.277576698754975, 7.667915939108839e-09, 2.807098992556789)), ('u', 9, (2.6951214583121246, 1.5707983028202153, -3.300060881599923)), ('u', 0, (-0.045599790850416655, 0.8447097053812379, 0.7862056775795166)), ('u', 2, (3.0172215159798648, 1.3529634125633758, -2.4329529547054896)), ('cz', 1, 9), ('u', 1, (1.4011906977627535, 1.303507267794118e-07, -0.36570529884734326)), ('u', 9, (3.156609880557973, 1.41279028949609, 0.5264335997287066)), ('u', 0, (1.936226685581525, 1.150749199392233, -0.21809435359381424)), ('u', 2, (-3.0290054671109217, 0.2508683046628232, 3.085710636402706)), ('cz', 1, 0), ('u', 1, (3.158208863809395, 1.5707958674831153, -2.68487545322985)), ('u', 9, (-2.4881822999656205, 0.024549127519533403, 3.092203692123528)), ('u', 0, (2.7041307174566507, 0.13561179984270652, -2.529497356895207)), ('u', 2, (-3.3523209104498464, 0.0512635207495124, 2.0685113642466306)), ('cz', 1, 2), ('u', 1, (2.684878423606155, 0.8860769696717745, -3.415743496733926)), ('u', 9, (-0.46631050682769754, 0.02008526260421799, 0.2805474477383349)), ('u', 0, (-3.8825407648077097, 0.3288443812267297, 2.0068622427489555)), ('u', 2, (3.137349282156838, 0.03196165307701446, -2.8780016951519514)), ('cz', 1, 0), ('u', 1, (0.274148811122849, 1.5707961629579066, -0.030699682444164586)), ('u', 9, (-0.16382274567119876, 0.07120428779189807, -0.1485482002157888)), ('u', 0, (2.961450157029619, 8.254614541639629e-08, -1.9159123542780216)), ('u', 2, (3.4466498120999933, 0.6576032960830681, -0.14908152563263677)), ('cz', 1, 2), ('u', 1, (0.03069705874365139, 1.5707961872089964, -0.08446776199160602)), ('u', 9, (0.17290348080682968, 0.14436003344923415, -0.2982191305177724)), ('u', 0, (4.288300166784074, 2.8482493215315373e-08, -1.1031273646067892)), ('u', 2, (3.2634617369673267, 1.5707970545676428, -0.08446778525274379)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.2284367252937551, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 0.3423596015011414, 0.0)), ('cz', 3, 2), ('u', 2, (-3.141592653589793, 0.3423596015011414, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 1.2284367252937551, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 0.23182380450040305, 0.0)), ('cz', 4, 3), ('u', 3, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 1.3389725222944935, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 1, (2.5297770035952922, 1.5707960474012908, -2.999175380526384)), ('u', 0, (-0.8002115661892221, 1.5707968672753247, -0.21285780976328283)), ('u', 2, (0.40109833495415903, 0.12126234676163666, 0.06304593419719828)), ('u', 3, (2.768104951433744, 0.06443307435994947, -3.5056053928562583)), ('cz', 1, 0), ('u', 1, (-0.14241705049317946, 0.5235987744435339, -0.4781665227325771)), ('u', 0, (0.21285745354526586, 0.5235987773009165, -0.1454276148765724)), ('u', 2, (3.318052133982163, 0.5787090713207548, -2.8698986478297748)), ('u', 3, (2.915201123157787, 0.06531674177080576, -3.241792720650956)), ('cz', 1, 0), ('u', 1, (0.47816660928190613, 1.5707963849044921, 0.2438014911886236)), ('u', 0, (-2.9293376568311262, 1.4101540322222728, 2.6574316028588103)), ('u', 2, (3.4282617416449543, 1.1578755564601861, 0.048070663519175216)), ('u', 3, (-3.0953814619051707, 0.04296654421080366, 2.7474511530020234)), ('cz', 1, 2), ('u', 1, (2.752811048122835, 1.9650221999203876, -3.0340508691631864)), ('u', 0, (-3.0537108767655052, 0.17387785017613092, 2.7557947319715312)), ('u', 2, (2.894073542017519, 1.4480101743813567, 0.07703832259138021)), ('u', 3, (-3.252118285293735, 0.15568687411688567, 2.646696880242898)), ('cz', 1, 0), ('u', 1, (2.8333296077552648, 0.4014696637362765, -3.337391383240182)), ('u', 0, (-2.810186412266125, 2.0345039928106686e-08, 2.407354915362383)), ('u', 2, (-0.18551821668952811, 0.12350486484258166, -0.13659242858175485)), ('u', 3, (-0.07219843351744573, 0.2496209278297433, -0.335793499065155)), ('cz', 1, 2), ('u', 1, (0.19579873272985326, 0.40146963002301717, 0.2081055999491198)), ('u', 0, (2.689995149549734, 8.647651956455005e-09, -3.3703918972732287)), ('u', 2, (-0.2204782968375812, 0.5859938268240137, 0.13804940451499453)), ('u', 3, (-0.5295408980824935, 0.6733901888155881, -0.28390563780235123)), ('cz', 1, 0), ('u', 1, (-0.11957038329809033, 1.2507101625984693, 0.19261196993027102)), ('u', 0, (0.5136180795664618, 0.2513476196494122, -0.6193938045843215)), ('u', 2, (3.0035434462926025, 0.5859936572535848, -0.8634086278724582)), ('u', 3, (0.18901909928944063, 0.6585548674630286, 0.3040857276816188)), ('cz', 2, 3), ('u', 1, (-0.03568828002761662, 0.7226816041042982, 0.08729068387460608)), ('u', 0, (-2.972883356989559, 0.1784851037199064, 3.0269193005442183)), ('u', 2, (-2.9341209185793464, 2.8777897797457075, 2.9591201873083777)), ('u', 3, (2.7861420202109377, 1.4320446308286061, 0.27994148389994455)), ('cz', 1, 2), ('u', 1, (3.122279831922309, 1.060949377809032e-06, -3.1257317665247153)), ('u', 0, (0.06685573055857483, 0.1584961566505804, -0.020449390348604542)), ('u', 2, (0.060896986109348294, 1.8327401738756184, -0.22513247453254132)), ('u', 3, (-0.09008568493740825, 0.14125711394471396, -0.1655537499952746)), ('cz', 2, 3), ('u', 1, (0.02354831957643501, 1.0578775288306478e-06, 0.23149920866660362)), ('u', 0, (0.3260218135415844, 0.3498124004130844, -0.2159326356930596)), ('u', 2, (2.9258863237971635, 0.28819310012156263, -3.2051393279781903)), ('u', 3, (-2.7939693360820046, 0.17337370173970315, 2.871525460075201)), ('cz', 1, 2), ('u', 1, (0.005653603361540832, 0.1933212257434527, 0.42292299281902623)), ('u', 0, (-0.18379895898202503, 0.0607089196708895, 0.929576528928568)), ('u', 2, (0.4883513926364131, 1.3069934570781006, 0.2815801833525267)), ('u', 3, (3.0489812572669837, 1.4084969307277857, 0.32706925480518434)), ('cz', 2, 3), ('u', 1, (-2.0208292448537675, 1.576102645655354, 1.3526232497460882)), ('u', 0, (0.8961796174728034, 1.4344371525621218, -2.086550490633348)), ('u', 2, (-2.936028502743085, 0.5704194086731823, 2.684231268100344)), ('u', 3, (3.2442293050870274, 1.3829326691433927, -0.37442743058617123)), ('cz', 1, 0), ('u', 1, (1.982218554424752, 0.6154796938201107, -0.9410940479245974)), ('u', 0, (1.4683065309140204, 0.6154797229416431, -2.0742017504789567)), ('u', 2, (-2.7230201481555056, 0.3620863058308674, 2.996718232692891)), ('u', 3, (-3.2025378155258792, 0.2266398275491752, 2.976569052784595)), ('cz', 1, 0), ('u', 1, (1.8471183955923773, 1.9307295523420935, -1.9024235263602252)), ('u', 0, (-0.6805016430036217, 1.468585645126016, 1.6330885743915207)), ('u', 2, (3.173998374295547, 1.8009909271819604, 0.3208123476350919)), ('u', 3, (-0.6464870512981434, 0.26574233093539995, -0.09104193832916918)), ('cz', 1, 2), ('u', 1, (-3.0793137083082125, 0.9988697676803144, 3.108110620284702)), ('u', 0, (2.7101024282115853, 0.05622266783246432, -2.4101210638310118)), ('u', 2, (3.0920241529341173, 1.5707963278535204, 0.4096371518840427)), ('u', 3, (-2.8779261736445028, 0.09638282836868403, 3.119285823976499)), ('cz', 1, 2), ('u', 1, (3.1750743448469554, 0.7853981726696819, -2.972913775447782)), ('u', 0, (-3.171651825736051, 0.35497346512319866, 2.8477607626924053)), ('u', 2, (-3.0996251160540513, 0.15655875606168773, 2.768710965098097)), ('u', 3, (0.6124807641465961, 0.38395253511275673, 0.15018107370056444)), ('cz', 1, 0), ('u', 1, (-0.1686785953210374, 0.7853981548839187, -0.31592857321407725)), ('u', 0, (-2.5567239028142605, 0.05239793879983422, 2.4944277713658103)), ('u', 2, (3.0745259859114658, 1.6707906949523792, 0.7759215312239554)), ('u', 3, (2.9026549691410866, 0.12061377414621048, -2.8437581938669085)), ('cz', 1, 2), ('u', 1, (0.31592853212355, 0.7853981753430412, -0.20828586668670931)), ('u', 0, (-0.4299995069399992, 1.2002998850717423, -0.3424028333118968)), ('u', 2, (2.0245813606797363, 2.379148075921518, -0.28687064248090066)), ('u', 3, (-0.1793500251514863, 0.3994233366556536, -0.08086772397304132)), ('cz', 1, 0), ('u', 1, (0.20828596010464584, 0.8726986982027364, -0.005279231169576862)), ('u', 0, (-2.849177670110597, 0.7853981647499652, 3.2628986769736703)), ('u', 2, (-0.050110980623897405, 0.3969161719200903, -0.3671045520702047)), ('u', 3, (-2.847275653942246, 0.02648888488899852, 3.220616077103796)), ('cz', 1, 0), ('u', 1, (-3.136313398325064, 1.5707963247113212, 2.989653518013588)), ('u', 0, (-3.262898698712991, 0.9955191745762307, 2.9896534088743203)), ('u', 2, (0.9845568707124354, 0.40546800198192223, -0.8914664365959389)), ('u', 3, (-2.869952412194633, 0.23626033890288906, 2.8384181319446804)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (-3.141592653589793, 1.2284367252937551, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (0.0, 0.3423596015011414, 0.0)), ('cz', 8, 7), ('u', 7, (-3.141592653589793, 0.3423596015011414, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (0.0, 1.2284367252937551, 0.0)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 0.23182380450040305, 0.0)), ('cz', 9, 8), ('u', 8, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 1.3389725222944935, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 5, (0.3471646224256862, 1.570794522289403, 3.30687754401815e-08)), ('u', 6, (2.3083508585608032, 1.5707971969908068, -3.1415929711655406)), ('u', 7, (0.4852590363142197, 0.2960800348220305, 0.5264426298207776)), ('u', 8, (-3.2750588778232363, 0.36944524615129887, 2.66601740798002)), ('cz', 5, 6), ('u', 5, (2.993764498110396e-08, 0.5235987834037305, -0.009355810226265544)), ('u', 6, (-3.0977577136307355e-07, 0.5235987672847918, 2.937649408280191e-07)), ('u', 7, (3.6680352793438704, 0.0033194434994307993, -2.5591446240266835)), ('u', 8, (-3.6171678910693568, 0.39873044968366, 2.630966015380148)), ('cz', 5, 6), ('u', 5, (3.1322368427227634, 1.5487958243674722, -2.7891340707829073)), ('u', 6, (2.8756504671934024e-07, 1.5707964869273765, -0.40560830041508344)), ('u', 7, (3.7240406705172173, 1.387771410658388, 0.13230105934677794)), ('u', 8, (-3.652219288547242, 0.2568364332694143, 2.5655595291929925)), ('cz', 6, 7), ('u', 5, (-2.789134072338325, 0.028879061549178794, 3.400175606271247)), ('u', 6, (-3.5472009433145333, 1.1109198635623394, 2.711730268866062)), ('u', 7, (3.2738937159619304, 1.668074876778801, 0.1621456355500266)), ('u', 8, (-0.576033113012597, 0.11389449749845854, -0.4701360273919574)), ('cz', 5, 6), ('u', 5, (-2.8830097114786355, 2.114065558895384e-06, 3.203573677746042)), ('u', 6, (-0.4298623760125103, 0.6491747816288608, 1.10193866753594e-07)), ('u', 7, (3.303738294956433, 0.10259168530766506, -2.972244372661558)), ('u', 8, (-0.47013603715024294, 0.21983120213163293, -0.2086080450479923)), ('cz', 6, 7), ('u', 5, (0.06198102578358969, 2.189585075271447e-06, 0.10713705081417824)), ('u', 6, (3.141592755722514, 0.6491744430103787, -2.710178895261804)), ('u', 7, (-2.972244386484895, 0.6679829337418874, 3.141592491296079)), ('u', 8, (-0.208608033663775, 0.28416941465291, 0.03900748229065894)), ('cz', 5, 6), ('u', 5, (0.10713704918665229, 0.11826870765696305, 0.09961101726030389)), ('u', 6, (0.4314137513695924, 1.1028951580082855, 0.017266287705867356)), ('u', 7, (3.1415925035187056, 2.4736096723188723, -0.016533448149742114)), ('u', 8, (0.03900749204855546, 0.33486196930647083, 0.11332595230920545)), ('cz', 7, 8), ('u', 5, (0.07444086849195077, 0.16947110538964635, 0.04594104253821191)), ('u', 6, (-0.35088717922477014, 0.00849074848499466, 0.027772750613612174)), ('u', 7, (-3.268602471274133, 1.861039483958646, 2.6390802457443003)), ('u', 8, (3.4782119326561904, 1.5737763159765175, 0.0479319702795169)), ('cz', 6, 7), ('u', 5, (0.04594104904345975, 0.21249120190187126, 0.012201798818818228)), ('u', 6, (0.027772721342516314, 8.188510780381645e-07, -0.14475854674329655)), ('u', 7, (-0.5025124019965733, 2.63326541308442, -0.5321655423331498)), ('u', 8, (3.1895246480424513, 0.0029935911710818877, -3.084011826655364)), ('cz', 7, 8), ('u', 5, (0.012201796379474439, 0.25498531651747947, -0.023844804061216522)), ('u', 6, (2.9968341393698146, 1.0741287405356616e-06, -3.1879234097487505)), ('u', 7, (2.609427106601877, 1.8347640400467646, -3.3614141132666715)), ('u', 8, (3.1991734805239895, 0.03355602506866087, -3.012393612995645)), ('cz', 6, 7), ('u', 5, (-0.023844813005371685, 0.2922745340658069, -0.06048427499777201)), ('u', 6, (-0.04633075128034511, 0.08096373709746622, -0.033148510458907016)), ('u', 7, (-3.361414092162013, 0.29024322759170074, 2.7837292907675497)), ('u', 8, (3.2707917051864195, 1.5383535889439617, 0.004209585174630437)), ('cz', 7, 8), ('u', 5, (1.510311992169128, 1.414940509739597, -1.3034653596867245)), ('u', 6, (-1.6039449121093652, 1.5761717469577012, 1.611188633215764)), ('u', 7, (2.7837292531272704, 0.2083117437223183, -3.124156170348651)), ('u', 8, (3.145802254954227, 2.0305539597668667, 0.039685592019989)), ('cz', 5, 6), ('u', 5, (1.8381272939037174, 1.5707967140227468, -1.9240632052824633)), ('u', 6, (1.6111886332147618, 0.6154796099280264, -2.4611683629588645)), ('u', 7, (3.159029140968286, 0.1278557420230431, -3.028648387555657)), ('u', 8, (0.0396856044405423, 0.4603037209953136, 0.026752482328189296)), ('cz', 5, 6), ('u', 5, (-1.9240632052813906, 1.544390146117296, 1.6225754993937935)), ('u', 6, (0.6804242906319378, 3.1415924013049277, -1.3680308549860627)), ('u', 7, (3.2545369467932956, 1.2442169290436433, -0.07091085619142357)), ('u', 8, (0.02675246281234621, 0.46720309659342973, 0.0235353189090746)), ('cz', 6, 7), ('u', 5, (-3.089813467535044, 0.25015137269972104, 3.158835285198214)), ('u', 6, (-2.938827185272749, 0.7853979642520982, 3.0611801600942252)), ('u', 7, (3.0706818056932175, 1.5518108380823337, -0.19552635002955787)), ('u', 8, (0.023535333545869244, 0.4671930015973368, 0.0368260123809082)), ('cz', 5, 6), ('u', 5, (0.017242623477223357, 0.08686238617369406, -0.013785466206849143)), ('u', 6, (-0.08041253315002451, 0.7918923878719472, 0.0567373435042451)), ('u', 7, (2.9460663011080293, 0.02010409362243402, -3.253197898900252)), ('u', 8, (0.03682600750224469, 0.45576083405754597, 0.10280493491130488)), ('cz', 6, 7), ('u', 5, (-0.013785462954395904, 0.01729217330008004, -0.05724525083625284)), ('u', 6, (-3.084855280910327, 1.5707961397900978, 1.5707965506780202)), ('u', 7, (-3.2531979151610826, 1.570795612062177, 2.5130247309297444)), ('u', 8, (0.10280494629558314, 0.4326186853608087, 0.25434315096481114)), ('cz', 6, 7), ('u', 5, (-0.057245252055887486, 1.4676232806376177, 0.3148648375048225)), ('u', 6, (-1.57079613777268, 1.5707969345269466, 1.1455443029426275)), ('u', 7, (2.513024727528941, 1.6576190727520028, 0.3564691478500799)), ('u', 8, (0.2543431200653191, 0.371904814261709, 0.3593322418143746)), ('cz', 5, 6), ('u', 5, (-2.8267278275797882, 1.0471973701411925, 2.6639342667024173)), ('u', 6, (1.1455443629784297, 0.7853979366143886, -7.772983923182153e-08)), ('u', 7, (-2.785123507160726, 0.16444326156339636, 3.390761556434009)), ('u', 8, (-2.7822603955124108, 0.10233602668021283, 3.408357478091854)), ('cz', 5, 6), ('u', 5, (2.6639342816257625, 1.570796674709984, -3.5303753446473247)), ('u', 6, (-3.141592704893204, 1.5707964146551068, 2.752810061545516)), ('u', 7, (-2.8924237507457713, 0.21938290670037167, 3.099230695562456)), ('u', 8, (-2.8748278209568014, 0.41560541446749294, 3.1328441769959032)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0))]), (10, [('u', 4, (0.0, 2.8317480138481654, 0.0)), ('u', 3, (0.0, 0.5165288251587206, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 0.5165288251587206, 3.141592653589793)), ('u', 2, (0.0, 0.9326801236225959, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 0.9326801236225959, 3.141592653589793)), ('u', 1, (0.0, 1.2645189576252271, 0.0)), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 1), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.2645189576252271, 3.141592653589793)), ('u', 6, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 6), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 7), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 8), ('u', 1, (-3.277576698754975, 7.667915939108839e-09, 2.807098992556789)), ('u', 9, (2.6951214583121246, 1.5707983028202153, -3.300060881599923)), ('u', 0, (-0.045599790850416655, 0.8447097053812379, 0.7862056775795166)), ('u', 2, (3.0172215159798648, 1.3529634125633758, -2.4329529547054896)), ('cz', 1, 9), ('u', 1, (1.4011906977627535, 1.303507267794118e-07, -0.36570529884734326)), ('u', 9, (3.156609880557973, 1.41279028949609, 0.5264335997287066)), ('u', 0, (1.936226685581525, 1.150749199392233, -0.21809435359381424)), ('u', 2, (-3.0290054671109217, 0.2508683046628232, 3.085710636402706)), ('cz', 1, 0), ('u', 1, (3.158208863809395, 1.5707958674831153, -2.68487545322985)), ('u', 9, (-2.4881822999656205, 0.024549127519533403, 3.092203692123528)), ('u', 0, (2.7041307174566507, 0.13561179984270652, -2.529497356895207)), ('u', 2, (-3.3523209104498464, 0.0512635207495124, 2.0685113642466306)), ('cz', 1, 2), ('u', 1, (2.684878423606155, 0.8860769696717745, -3.415743496733926)), ('u', 9, (-0.46631050682769754, 0.02008526260421799, 0.2805474477383349)), ('u', 0, (-3.8825407648077097, 0.3288443812267297, 2.0068622427489555)), ('u', 2, (3.137349282156838, 0.03196165307701446, -2.8780016951519514)), ('cz', 1, 0), ('u', 1, (0.274148811122849, 1.5707961629579066, -0.030699682444164586)), ('u', 9, (-0.16382274567119876, 0.07120428779189807, -0.1485482002157888)), ('u', 0, (2.961450157029619, 8.254614541639629e-08, -1.9159123542780216)), ('u', 2, (3.4466498120999933, 0.6576032960830681, -0.14908152563263677)), ('cz', 1, 2), ('u', 1, (0.03069705874365139, 1.5707961872089964, -0.08446776199160602)), ('u', 9, (0.17290348080682968, 0.14436003344923415, -0.2982191305177724)), ('u', 0, (4.288300166784074, 2.8482493215315373e-08, -1.1031273646067892)), ('u', 2, (3.2634617369673267, 1.5707970545676428, -0.08446778525274379)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.2284367252937551, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 0.3423596015011414, 0.0)), ('cz', 3, 2), ('u', 2, (-3.141592653589793, 0.3423596015011414, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 1.2284367252937551, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 0.23182380450040305, 0.0)), ('cz', 4, 3), ('u', 3, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 1.3389725222944935, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 1, (2.5297770035952922, 1.5707960474012908, -2.999175380526384)), ('u', 0, (-0.8002115661892221, 1.5707968672753247, -0.21285780976328283)), ('u', 2, (0.40109833495415903, 0.12126234676163666, 0.06304593419719828)), ('u', 3, (2.768104951433744, 0.06443307435994947, -3.5056053928562583)), ('cz', 1, 0), ('u', 1, (-0.14241705049317946, 0.5235987744435339, -0.4781665227325771)), ('u', 0, (0.21285745354526586, 0.5235987773009165, -0.1454276148765724)), ('u', 2, (3.318052133982163, 0.5787090713207548, -2.8698986478297748)), ('u', 3, (2.915201123157787, 0.06531674177080576, -3.241792720650956)), ('cz', 1, 0), ('u', 1, (0.47816660928190613, 1.5707963849044921, 0.2438014911886236)), ('u', 0, (-2.9293376568311262, 1.4101540322222728, 2.6574316028588103)), ('u', 2, (3.4282617416449543, 1.1578755564601861, 0.048070663519175216)), ('u', 3, (-3.0953814619051707, 0.04296654421080366, 2.7474511530020234)), ('cz', 1, 2), ('u', 1, (2.752811048122835, 1.9650221999203876, -3.0340508691631864)), ('u', 0, (-3.0537108767655052, 0.17387785017613092, 2.7557947319715312)), ('u', 2, (2.894073542017519, 1.4480101743813567, 0.07703832259138021)), ('u', 3, (-3.252118285293735, 0.15568687411688567, 2.646696880242898)), ('cz', 1, 0), ('u', 1, (2.8333296077552648, 0.4014696637362765, -3.337391383240182)), ('u', 0, (-2.810186412266125, 2.0345039928106686e-08, 2.407354915362383)), ('u', 2, (-0.18551821668952811, 0.12350486484258166, -0.13659242858175485)), ('u', 3, (-0.07219843351744573, 0.2496209278297433, -0.335793499065155)), ('cz', 1, 2), ('u', 1, (0.19579873272985326, 0.40146963002301717, 0.2081055999491198)), ('u', 0, (2.689995149549734, 8.647651956455005e-09, -3.3703918972732287)), ('u', 2, (-0.2204782968375812, 0.5859938268240137, 0.13804940451499453)), ('u', 3, (-0.5295408980824935, 0.6733901888155881, -0.28390563780235123)), ('cz', 1, 0), ('u', 1, (-0.11957038329809033, 1.2507101625984693, 0.19261196993027102)), ('u', 0, (0.5136180795664618, 0.2513476196494122, -0.6193938045843215)), ('u', 2, (3.0035434462926025, 0.5859936572535848, -0.8634086278724582)), ('u', 3, (0.18901909928944063, 0.6585548674630286, 0.3040857276816188)), ('cz', 2, 3), ('u', 1, (-0.03568828002761662, 0.7226816041042982, 0.08729068387460608)), ('u', 0, (-2.972883356989559, 0.1784851037199064, 3.0269193005442183)), ('u', 2, (-2.9341209185793464, 2.8777897797457075, 2.9591201873083777)), ('u', 3, (2.7861420202109377, 1.4320446308286061, 0.27994148389994455)), ('cz', 1, 2), ('u', 1, (3.122279831922309, 1.060949377809032e-06, -3.1257317665247153)), ('u', 0, (0.06685573055857483, 0.1584961566505804, -0.020449390348604542)), ('u', 2, (0.060896986109348294, 1.8327401738756184, -0.22513247453254132)), ('u', 3, (-0.09008568493740825, 0.14125711394471396, -0.1655537499952746)), ('cz', 2, 3), ('u', 1, (0.02354831957643501, 1.0578775288306478e-06, 0.23149920866660362)), ('u', 0, (0.3260218135415844, 0.3498124004130844, -0.2159326356930596)), ('u', 2, (2.9258863237971635, 0.28819310012156263, -3.2051393279781903)), ('u', 3, (-2.7939693360820046, 0.17337370173970315, 2.871525460075201)), ('cz', 1, 2), ('u', 1, (0.005653603361540832, 0.1933212257434527, 0.42292299281902623)), ('u', 0, (-0.18379895898202503, 0.0607089196708895, 0.929576528928568)), ('u', 2, (0.4883513926364131, 1.3069934570781006, 0.2815801833525267)), ('u', 3, (3.0489812572669837, 1.4084969307277857, 0.32706925480518434)), ('cz', 2, 3), ('u', 1, (-2.0208292448537675, 1.576102645655354, 1.3526232497460882)), ('u', 0, (0.8961796174728034, 1.4344371525621218, -2.086550490633348)), ('u', 2, (-2.936028502743085, 0.5704194086731823, 2.684231268100344)), ('u', 3, (3.2442293050870274, 1.3829326691433927, -0.37442743058617123)), ('cz', 1, 0), ('u', 1, (1.982218554424752, 0.6154796938201107, -0.9410940479245974)), ('u', 0, (1.4683065309140204, 0.6154797229416431, -2.0742017504789567)), ('u', 2, (-2.7230201481555056, 0.3620863058308674, 2.996718232692891)), ('u', 3, (-3.2025378155258792, 0.2266398275491752, 2.976569052784595)), ('cz', 1, 0), ('u', 1, (1.8471183955923773, 1.9307295523420935, -1.9024235263602252)), ('u', 0, (-0.6805016430036217, 1.468585645126016, 1.6330885743915207)), ('u', 2, (3.173998374295547, 1.8009909271819604, 0.3208123476350919)), ('u', 3, (-0.6464870512981434, 0.26574233093539995, -0.09104193832916918)), ('cz', 1, 2), ('u', 1, (-3.0793137083082125, 0.9988697676803144, 3.108110620284702)), ('u', 0, (2.7101024282115853, 0.05622266783246432, -2.4101210638310118)), ('u', 2, (3.0920241529341173, 1.5707963278535204, 0.4096371518840427)), ('u', 3, (-2.8779261736445028, 0.09638282836868403, 3.119285823976499)), ('cz', 1, 2), ('u', 1, (3.1750743448469554, 0.7853981726696819, -2.972913775447782)), ('u', 0, (-3.171651825736051, 0.35497346512319866, 2.8477607626924053)), ('u', 2, (-3.0996251160540513, 0.15655875606168773, 2.768710965098097)), ('u', 3, (0.6124807641465961, 0.38395253511275673, 0.15018107370056444)), ('cz', 1, 0), ('u', 1, (-0.1686785953210374, 0.7853981548839187, -0.31592857321407725)), ('u', 0, (-2.5567239028142605, 0.05239793879983422, 2.4944277713658103)), ('u', 2, (3.0745259859114658, 1.6707906949523792, 0.7759215312239554)), ('u', 3, (2.9026549691410866, 0.12061377414621048, -2.8437581938669085)), ('cz', 1, 2), ('u', 1, (0.31592853212355, 0.7853981753430412, -0.20828586668670931)), ('u', 0, (-0.4299995069399992, 1.2002998850717423, -0.3424028333118968)), ('u', 2, (2.0245813606797363, 2.379148075921518, -0.28687064248090066)), ('u', 3, (-0.1793500251514863, 0.3994233366556536, -0.08086772397304132)), ('cz', 1, 0), ('u', 1, (0.20828596010464584, 0.8726986982027364, -0.005279231169576862)), ('u', 0, (-2.849177670110597, 0.7853981647499652, 3.2628986769736703)), ('u', 2, (-0.050110980623897405, 0.3969161719200903, -0.3671045520702047)), ('u', 3, (-2.847275653942246, 0.02648888488899852, 3.220616077103796)), ('cz', 1, 0), ('u', 1, (-3.136313398325064, 1.5707963247113212, 2.989653518013588)), ('u', 0, (-3.262898698712991, 0.9955191745762307, 2.9896534088743203)), ('u', 2, (0.9845568707124354, 0.40546800198192223, -0.8914664365959389)), ('u', 3, (-2.869952412194633, 0.23626033890288906, 2.8384181319446804)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 5, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 6, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 5, 6), ('u', 6, (1.570796326794898, 0.4636476090008061, -1.570796326794898)), ('u', 5, (1.5707963267948966, 0.4636476090008061, -1.5707963267948966)), ('cz', 5, 6), ('u', 6, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 5, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 6, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 6, 7), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 6, (-3.141592653589793, 0.6847192030022828, 3.141592653589793)), ('cz', 7, 6), ('u', 6, (0.0, 0.6847192030022828, 0.0)), ('u', 6, (-3.141592653589793, 2.2555155297971794, 3.141592653589793)), ('cz', 5, 6), ('u', 6, (0.0, 2.2555155297971794, 0.0)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 6, 7), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (-3.141592653589793, 1.2284367252937551, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (0.0, 0.3423596015011414, 0.0)), ('cz', 8, 7), ('u', 7, (-3.141592653589793, 0.3423596015011414, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (0.0, 1.2284367252937551, 0.0)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 0.23182380450040305, 0.0)), ('cz', 9, 8), ('u', 8, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 1.3389725222944935, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 5, (0.3471646224256862, 1.570794522289403, 3.30687754401815e-08)), ('u', 6, (2.3083508585608032, 1.5707971969908068, -3.1415929711655406)), ('u', 7, (0.4852590363142197, 0.2960800348220305, 0.5264426298207776)), ('u', 8, (-3.2750588778232363, 0.36944524615129887, 2.66601740798002)), ('cz', 5, 6), ('u', 5, (2.993764498110396e-08, 0.5235987834037305, -0.009355810226265544)), ('u', 6, (-3.0977577136307355e-07, 0.5235987672847918, 2.937649408280191e-07)), ('u', 7, (3.6680352793438704, 0.0033194434994307993, -2.5591446240266835)), ('u', 8, (-3.6171678910693568, 0.39873044968366, 2.630966015380148)), ('cz', 5, 6), ('u', 5, (3.1322368427227634, 1.5487958243674722, -2.7891340707829073)), ('u', 6, (2.8756504671934024e-07, 1.5707964869273765, -0.40560830041508344)), ('u', 7, (3.7240406705172173, 1.387771410658388, 0.13230105934677794)), ('u', 8, (-3.652219288547242, 0.2568364332694143, 2.5655595291929925)), ('cz', 6, 7), ('u', 5, (-2.789134072338325, 0.028879061549178794, 3.400175606271247)), ('u', 6, (-3.5472009433145333, 1.1109198635623394, 2.711730268866062)), ('u', 7, (3.2738937159619304, 1.668074876778801, 0.1621456355500266)), ('u', 8, (-0.576033113012597, 0.11389449749845854, -0.4701360273919574)), ('cz', 5, 6), ('u', 5, (-2.8830097114786355, 2.114065558895384e-06, 3.203573677746042)), ('u', 6, (-0.4298623760125103, 0.6491747816288608, 1.10193866753594e-07)), ('u', 7, (3.303738294956433, 0.10259168530766506, -2.972244372661558)), ('u', 8, (-0.47013603715024294, 0.21983120213163293, -0.2086080450479923)), ('cz', 6, 7), ('u', 5, (0.06198102578358969, 2.189585075271447e-06, 0.10713705081417824)), ('u', 6, (3.141592755722514, 0.6491744430103787, -2.710178895261804)), ('u', 7, (-2.972244386484895, 0.6679829337418874, 3.141592491296079)), ('u', 8, (-0.208608033663775, 0.28416941465291, 0.03900748229065894)), ('cz', 5, 6), ('u', 5, (0.10713704918665229, 0.11826870765696305, 0.09961101726030389)), ('u', 6, (0.4314137513695924, 1.1028951580082855, 0.017266287705867356)), ('u', 7, (3.1415925035187056, 2.4736096723188723, -0.016533448149742114)), ('u', 8, (0.03900749204855546, 0.33486196930647083, 0.11332595230920545)), ('cz', 7, 8), ('u', 5, (0.07444086849195077, 0.16947110538964635, 0.04594104253821191)), ('u', 6, (-0.35088717922477014, 0.00849074848499466, 0.027772750613612174)), ('u', 7, (-3.268602471274133, 1.861039483958646, 2.6390802457443003)), ('u', 8, (3.4782119326561904, 1.5737763159765175, 0.0479319702795169)), ('cz', 6, 7), ('u', 5, (0.04594104904345975, 0.21249120190187126, 0.012201798818818228)), ('u', 6, (0.027772721342516314, 8.188510780381645e-07, -0.14475854674329655)), ('u', 7, (-0.5025124019965733, 2.63326541308442, -0.5321655423331498)), ('u', 8, (3.1895246480424513, 0.0029935911710818877, -3.084011826655364)), ('cz', 7, 8), ('u', 5, (0.012201796379474439, 0.25498531651747947, -0.023844804061216522)), ('u', 6, (2.9968341393698146, 1.0741287405356616e-06, -3.1879234097487505)), ('u', 7, (2.609427106601877, 1.8347640400467646, -3.3614141132666715)), ('u', 8, (3.1991734805239895, 0.03355602506866087, -3.012393612995645)), ('cz', 6, 7), ('u', 5, (-0.023844813005371685, 0.2922745340658069, -0.06048427499777201)), ('u', 6, (-0.04633075128034511, 0.08096373709746622, -0.033148510458907016)), ('u', 7, (-3.361414092162013, 0.29024322759170074, 2.7837292907675497)), ('u', 8, (3.2707917051864195, 1.5383535889439617, 0.004209585174630437)), ('cz', 7, 8), ('u', 5, (1.510311992169128, 1.414940509739597, -1.3034653596867245)), ('u', 6, (-1.6039449121093652, 1.5761717469577012, 1.611188633215764)), ('u', 7, (2.7837292531272704, 0.2083117437223183, -3.124156170348651)), ('u', 8, (3.145802254954227, 2.0305539597668667, 0.039685592019989)), ('cz', 5, 6), ('u', 5, (1.8381272939037174, 1.5707967140227468, -1.9240632052824633)), ('u', 6, (1.6111886332147618, 0.6154796099280264, -2.4611683629588645)), ('u', 7, (3.159029140968286, 0.1278557420230431, -3.028648387555657)), ('u', 8, (0.0396856044405423, 0.4603037209953136, 0.026752482328189296)), ('cz', 5, 6), ('u', 5, (-1.9240632052813906, 1.544390146117296, 1.6225754993937935)), ('u', 6, (0.6804242906319378, 3.1415924013049277, -1.3680308549860627)), ('u', 7, (3.2545369467932956, 1.2442169290436433, -0.07091085619142357)), ('u', 8, (0.02675246281234621, 0.46720309659342973, 0.0235353189090746)), ('cz', 6, 7), ('u', 5, (-3.089813467535044, 0.25015137269972104, 3.158835285198214)), ('u', 6, (-2.938827185272749, 0.7853979642520982, 3.0611801600942252)), ('u', 7, (3.0706818056932175, 1.5518108380823337, -0.19552635002955787)), ('u', 8, (0.023535333545869244, 0.4671930015973368, 0.0368260123809082)), ('cz', 5, 6), ('u', 5, (0.017242623477223357, 0.08686238617369406, -0.013785466206849143)), ('u', 6, (-0.08041253315002451, 0.7918923878719472, 0.0567373435042451)), ('u', 7, (2.9460663011080293, 0.02010409362243402, -3.253197898900252)), ('u', 8, (0.03682600750224469, 0.45576083405754597, 0.10280493491130488)), ('cz', 6, 7), ('u', 5, (-0.013785462954395904, 0.01729217330008004, -0.05724525083625284)), ('u', 6, (-3.084855280910327, 1.5707961397900978, 1.5707965506780202)), ('u', 7, (-3.2531979151610826, 1.570795612062177, 2.5130247309297444)), ('u', 8, (0.10280494629558314, 0.4326186853608087, 0.25434315096481114)), ('cz', 6, 7), ('u', 5, (-0.057245252055887486, 1.4676232806376177, 0.3148648375048225)), ('u', 6, (-1.57079613777268, 1.5707969345269466, 1.1455443029426275)), ('u', 7, (2.513024727528941, 1.6576190727520028, 0.3564691478500799)), ('u', 8, (0.2543431200653191, 0.371904814261709, 0.3593322418143746)), ('cz', 5, 6), ('u', 5, (-2.8267278275797882, 1.0471973701411925, 2.6639342667024173)), ('u', 6, (1.1455443629784297, 0.7853979366143886, -7.772983923182153e-08)), ('u', 7, (-2.785123507160726, 0.16444326156339636, 3.390761556434009)), ('u', 8, (-2.7822603955124108, 0.10233602668021283, 3.408357478091854)), ('cz', 5, 6), ('u', 5, (2.6639342816257625, 1.570796674709984, -3.5303753446473247)), ('u', 6, (-3.141592704893204, 1.5707964146551068, 2.752810061545516)), ('u', 7, (-2.8924237507457713, 0.21938290670037167, 3.099230695562456)), ('u', 8, (-2.8748278209568014, 0.41560541446749294, 3.1328441769959032)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0))])], (10, 5): [(10, [('u', 4, (0.0, 3.0155210212709935, 0.0)), ('u', 3, (0.0, 0.32108595465310086, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 0.32108595465310086, 3.141592653589793)), ('u', 2, (0.0, 0.7277483918509393, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 0.7277483918509393, 3.141592653589793)), ('u', 1, (0.0, 1.0992585722264043, 0.0)), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 1), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.0992585722264043, 3.141592653589793)), ('u', 0, (0.0, 1.3734007669450157, 0.0)), ('u', 0, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 0), ('u', 0, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (-3.141592653589793, 1.3734007669450157, 3.141592653589793)), ('u', 5, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 6, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 6), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 7), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 0, 9), ('u', 5, (1.8945664487800378, 1.5707967518441073, -1.570795846280955)), ('u', 6, (-1.7388988637546396, 1.570796325672544, 1.5707960778855616)), ('u', 7, (0.16676133767055626, 0.06071585380742071, 0.17955809558682237)), ('u', 8, (-0.15421509268626693, 0.06530529457287648, -0.08239156870685518)), ('u', 9, (4.545755374683717, 0.7633799068927589, 1.4257144917397286)), ('cz', 5, 6), ('u', 5, (1.5707968073105743, 0.4636476151656638, -1.5675457432288449)), ('u', 6, (1.5707960778854138, 0.4636476100060951, -1.5707965909074941)), ('u', 7, (0.1795583497470414, 0.0005749801113210117, 0.18452175922968725)), ('u', 8, (-0.08239180887410527, 0.021129446968765873, -0.04257484309447363)), ('u', 9, (-0.11266221521373637, 0.07614745836979754, -0.09251296128145267)), ('cz', 5, 6), ('u', 5, (-1.567545743228623, 1.5690926248656714, 1.6989492553530408)), ('u', 6, (1.5707960626801414, 1.5707964041614257, -2.353349221309959)), ('u', 7, (3.326114125231347, 1.6166756106487763, 0.019448985321698586)), ('u', 8, (-0.042574870563586, 0.0028969593274814597, -0.032544405374719046)), ('u', 9, (-0.09251318524714315, 0.09722052295658065, -0.06806727387788351)), ('cz', 6, 7), ('u', 5, (3.2697454880527808, 0.006721901570050943, -2.9672024156159456)), ('u', 6, (2.3590387684781824, 1.1164401041471685, -3.347577072572264)), ('u', 7, (3.1610416389118186, 1.5767236955179975, -0.13975838932259732)), ('u', 8, (-3.1741370589665054, 0.0018367600320535354, 3.106715161880185)), ('u', 9, (-0.06806727387942839, 0.10925960485138374, -0.040696461377724674)), ('cz', 5, 6), ('u', 5, (-2.9672024156130954, 1.1698399669258657e-07, 3.247518193709247)), ('u', 6, (-0.2059844189818722, 0.4897257742772963, -1.0482337645112416e-07)), ('u', 7, (-3.281351042914826, 0.006166862493138006, 2.9319692396981742)), ('u', 8, (-3.176470145298558, 0.001492486498975132, 3.0997765701636713)), ('u', 9, (-0.040696461378145, 0.11059149155988668, -0.011987254763363315)), ('cz', 6, 7), ('u', 5, (0.10592554011891814, 9.133852450569193e-08, 0.17405312529502553)), ('u', 6, (3.1415925487670613, 0.4897258470238842, -2.9356081745731375)), ('u', 7, (2.9319692396990717, 0.00216951397695742, -3.325280592484795)), ('u', 8, (-3.1834087370162507, 0.0039041928009757114, 3.089141124715767)), ('u', 9, (-0.011987254763881555, 0.1001612496352822, 0.01627629699782284)), ('cz', 5, 6), ('u', 5, (3.315645778882708, 0.06189978738063253, -2.967288758776426)), ('u', 6, (0.20598447901655276, 1.1164401268905617, -0.008022207800787579)), ('u', 7, (2.9579047146971034, 1.5687714658561982, 0.0003892206221305372)), ('u', 8, (-3.194044182465969, 0.0165738826423929, 3.071969130564487)), ('u', 9, (0.01627629699786679, 0.07754490274566826, 0.042063445897705345)), ('cz', 6, 7), ('u', 5, (-2.9672887587742895, 0.00750690928498049, 3.2948529026829627)), ('u', 6, (-0.008022207800504506, 4.447633084727949e-08, -0.0845277613003381)), ('u', 7, (0.00038922062164292726, 1.5707965658968128, -2.944366841624193)), ('u', 8, (3.0719691305689247, 1.6175859669323922, -0.025327798180427408)), ('u', 9, (0.04206344589727465, 0.04293490743296516, 0.06309659901351776)), ('cz', 7, 8), ('u', 5, (0.1532602490942267, 0.06423237825831014, 0.11764263011638738)), ('u', 6, (3.057064892285915, 6.144522794224255e-07, -3.1205756586474482)), ('u', 7, (-2.9443668416238915, 1.9146024384117089, 3.2639940075030487)), ('u', 8, (3.116264855404986, 1.578369789237178, -0.008630980227517515)), ('u', 9, (3.2046892526032327, 0.0028400663396738416, -3.0645429356255116)), ('cz', 6, 7), ('u', 5, (0.11764263011316467, 0.1128584153368073, 0.07104732367082656)), ('u', 6, (-3.120575658639638, 2.410478157379983e-07, 3.094226952319221)), ('u', 7, (-3.019191299674953, 0.3535242690068961, 3.141592522678906)), ('u', 8, (3.1329616733585226, 0.007574589049400573, -3.1421449704821063)), ('u', 9, (3.2186423715566352, 0.058135940368354205, -3.0595598957533223)), ('cz', 7, 8), ('u', 5, (0.07104732366567315, 0.038189033854302924, 0.03919385937824557)), ('u', 6, (-0.04736570128024067, 2.335884784796077e-07, 0.07123480844562785)), ('u', 7, (-3.1415927845017717, 0.3535242544320354, 3.002638027603464)), ('u', 8, (3.141040336697898, 0.7056915298497516, -3.1415929595249756)), ('u', 9, (-3.059559895756642, 0.12011684380768442, 3.218899500433714)), ('cz', 6, 7), ('u', 5, (3.180786512979133, 0.011297013189518592, -3.0901413489082556)), ('u', 6, (-3.070357845138054, 0.009195193806501314, 3.209595512049762)), ('u', 7, (-0.13895462598342365, 1.28433544013007, -0.2484786173182171)), ('u', 8, (3.141592347650575, 2.4359010433382084, 0.1610142376402539)), ('u', 9, (-3.064285806749552, 0.18422515254660257, 3.205643048478841)), ('cz', 8, 9), ('u', 5, (3.193043958267901, 0.0022993544362659886, -3.066018400069809)), ('u', 6, (-3.073589795131183, 0.011182293528819446, 3.206782974790786)), ('u', 7, (2.8931140362703895, 0.06541310894735942, -3.363478157517656)), ('u', 8, (-2.9805784159549935, 2.9094829295361366, 3.1170347314893334)), ('u', 9, (3.205643048483073, 1.5534002427980684, 0.06489723763524902)), ('cz', 7, 8), ('u', 5, (3.2171669071073, 0.0009204183442263046, -3.041385187971427)), ('u', 6, (-3.0764023323921053, 0.007232933428631628, 3.2008280446158444)), ('u', 7, (-0.22188550392773387, 2.84727746576287e-08, -0.2829143883306623)), ('u', 8, (-0.024557922104821085, 1.8026360811260347, -0.10537850908146654)), ('u', 9, (0.064897237635256, 0.017543716167227205, 0.05490428169944987)), ('cz', 8, 9), ('u', 5, (3.2418001192100587, 0.009938705071611332, -3.029004647948375)), ('u', 6, (0.05923539103507921, 0.0007266745212134733, 0.05019001985792796)), ('u', 7, (-3.424507041919481, 5.4686150435648135e-08, 2.776213110096355)), ('u', 8, (3.0362141445081385, 0.23689373941819938, -3.039074288122562)), ('u', 9, (-3.086688371886903, 0.016475712623880815, 3.1697825034520104)), ('cz', 7, 8), ('u', 5, (3.254180659225862, 0.02902822305937778, -3.026995883926962)), ('u', 6, (0.050190019851367396, 0.009471277553238792, 0.0419789041877677)), ('u', 7, (-3.5069721970909766, 0.09792226759275453, 2.774066031215188)), ('u', 8, (0.10251836546663748, 1.3386866001175495, -0.30078105852757575)), ('u', 9, (3.1697825034432414, 1.554346921026769, 0.10831217454680342)), ('cz', 8, 9), ('u', 5, (-0.1989526424035275, 1.4755948231061364, -0.04300026651769265)), ('u', 6, (-3.109650380573552, 1.5560407522094541, 3.1362272286723596)), ('u', 7, (2.9535510338321513, 0.1917314238400673, -3.288014995212577)), ('u', 8, (-0.2565213288976834, 0.7385210428414481, -0.056348456312167144)), ('u', 9, (-3.0332799208769816, 0.009895232768065252, 3.2372102973453596)), ('cz', 5, 6), ('u', 5, (-0.04300026651711933, 0.5235987778907799, -0.0001624016961349721)), ('u', 6, (-0.005365424915997928, 0.5235987763233106, -2.451674302895307e-08)), ('u', 7, (2.9951703119693045, 0.029340688349959933, -3.2541530370516734)), ('u', 8, (-0.056348456315275595, 0.5853481452059874, -0.0025018423016636704)), ('u', 9, (0.09561660322192196, 0.028151317278034053, 0.08159602415206259)), ('cz', 5, 6), ('u', 5, (-3.141755055291686, 1.5723992055769818, 3.0415765817875458)), ('u', 6, (-2.4514472055613135e-08, 1.570796514506075, 0.009768260926048673)), ('u', 7, (3.029032270124341, 1.3267273135169728, 0.09770243618045349)), ('u', 8, (-0.002501842306136662, 0.5559920232839984, 0.20485216634504533)), ('u', 9, (0.08159623226152664, 0.05976476127969489, 0.06638644402500553)), ('cz', 6, 7), ('u', 5, (-0.10001607179837306, 0.0016354289156405124, -0.11612950964937084)), ('u', 6, (3.1513609145112627, 0.46969218153686176, -2.865022956030711)), ('u', 7, (3.239295089775983, 1.5901294014364613, 0.2176069388688684)), ('u', 8, (0.20485216634466413, 0.47925936259400453, 0.2916744479121838)), ('u', 9, (0.0663855074942441, 0.08501739747613762, 0.05025984102553285)), ('cz', 5, 6), ('u', 5, (-3.2577221632422724, 1.0521653711159877e-07, 2.9726134991281166)), ('u', 6, (0.2765696976187939, 1.1631057414837858, -1.92470139548373e-08)), ('u', 7, (3.359199592457004, 0.02132001014074463, -2.852772266169725)), ('u', 8, (0.2916744479132669, 0.4644770856057639, 0.2651561921210803)), ('u', 9, (0.05026098567099667, 0.10367529388336723, 0.033563292599438424)), ('cz', 6, 7), ('u', 5, (-0.16897915446241052, 1.712556877334336e-07, 0.18874854277990988)), ('u', 6, (-3.1415926728366435, 1.163105744367837, 2.9832117552798696)), ('u', 7, (0.2888203874193402, 0.3746973546369369, -0.32248500625610427)), ('u', 8, (0.26515619212113717, 0.5194187059184701, 0.15120583090887615)), ('u', 9, (0.03356287637916325, 0.11531494259045867, 0.01667948267442124)), ('cz', 5, 6), ('u', 5, (0.18874854278052722, 0.022136581261754253, 0.05996157894600253)), ('u', 6, (-0.1583808983092532, 0.761780892814186, 0.38916430069900676)), ('u', 7, (2.819107647333773, 1.8752973431406768, 0.4711040424011319)), ('u', 8, (0.15120583091309805, 0.6138568019398781, 0.06994280597900235)), ('u', 9, (0.016679482669277622, 0.11951208558003339, 3.89954517224754e-05)), ('cz', 7, 8), ('u', 5, (0.006671009854934646, 0.438720937205351, -0.025412188821046667)), ('u', 6, (3.0951561053021592, 0.3208821181104216, -3.1468674127014458)), ('u', 7, (-5.928931353207465, 2.356194500434075, -0.32012510236254954)), ('u', 8, (3.35950234254994, 1.6168011403262426, -0.13216686488645468)), ('u', 9, (-0.0006629716818468429, 0.11626833478702397, -0.01637883152685158)), ('cz', 6, 7), ('u', 5, (-0.025412188817159752, 0.6328556905091125, -0.08156984258741057)), ('u', 6, (-0.005274759110915628, 0.185109892086865, 0.15734514631282612)), ('u', 7, (-0.320125102360336, 0.8948697438541604, 0.2180583477692004)), ('u', 8, (3.0094257887038296, 0.07245778080906697, -3.271081282275234)), ('u', 9, (-0.01637987213022763, 0.10558918138037923, -0.030977447906558424)), ('cz', 7, 8), ('u', 5, (-0.08156984258688584, 0.7578801326318705, -0.019794749914697648)), ('u', 6, (0.15734514631135477, 0.25773927820786874, -0.21992906207449225)), ('u', 7, (-2.923534305822799, 1.5707963431450163, 2.879096297139732)), ('u', 8, (3.0121040249002773, 1.5707963556765756, -3.0568503181275277)), ('u', 9, (-0.030976303248004867, 0.0877407390154386, -0.04378820526334371)), ('cz', 7, 8), ('u', 5, (1.5770444054854393, 2.160061459010524, -0.9483198617204408)), ('u', 6, (-1.9387558667700486, 0.7795399092678185, 0.3231580472634805)), ('u', 7, (-3.7453640500412506, 0.5234700198532182, 2.3690600722094954)), ('u', 8, (3.5385259647743315, 1.4015316135248568, -0.36796415463920895)), ('u', 9, (-0.04525424486893484, 0.062069722209381735, -0.055248130641950666)), ('cz', 5, 6), ('u', 5, (2.1932727918690826, 0.9117383615698841, -2.4400337110120347)), ('u', 6, (0.32315804726410924, 0.6847191956530471, -1.570796188702638)), ('u', 7, (-3.914125234978876, 0.6143282932963846, 2.0451852833894413)), ('u', 8, (-3.5095568082296147, 0.19108711746209564, 2.7675128143144834)), ('u', 9, (-0.0552482347027905, 0.032506139599340456, -0.062101589577161456)), ('cz', 5, 6), ('u', 5, (-2.4400337110078736, 2.6942453487172733, 1.4408573381773138)), ('u', 6, (1.5707964648866628, 1.1071488052222511, -0.7853981736011744)), ('u', 7, (2.0451852833896216, 1.6137129721657928, 0.10241138782877923)), ('u', 8, (2.767512814312614, 0.27751489209214125, -3.498641633586016)), ('u', 9, (-0.06210175867710489, 0.00041093830589513666, -0.06546451005098773)), ('cz', 6, 7), ('u', 5, (-3.2715316422055265, 1.7269275551348893, 3.003459330038761)), ('u', 6, (-2.3561945003957656, 2.1862760602569367, 2.228528098209361)), ('u', 7, (3.244004041412788, 1.780188848595743, -0.08141479557159981)), ('u', 8, (2.7845436735932303, 0.3428335830536826, -3.3001679635875405)), ('u', 9, (3.07612780208225, 0.032319291776130236, -3.2068928830409495)), ('cz', 5, 6), ('u', 5, (-0.13813332355046, 1.2971554461029409, -0.06846191564831489)), ('u', 6, (2.22852809820779, 2.85219093167786e-07, -3.7993244311821375)), ('u', 7, (-0.08141479556971463, 0.21415602858576346, -0.0315739117604345)), ('u', 8, (2.983017343592045, 0.29886214264078376, -3.1279685726939563)), ('u', 9, (3.076293006235819, 0.0638384708317506, -3.203505479855805)), ('cz', 6, 7), ('u', 5, (-0.068461915646121, 1.070283069063573, 0.18684050678292616)), ('u', 6, (2.483860876000405, 2.0344439736929276, -3.1415923313864718)), ('u', 7, (-0.03157391176148561, 0.010059988227004844, 0.11887494656256414)), ('u', 8, (3.1552167344862774, 0.2124921807825797, -3.0148966593973743)), ('u', 9, (3.0796802500644365, 0.09243784117555508, -3.1975016026661045)), ('cz', 5, 6), ('u', 5, (0.18684050678211694, 0.5566518623877553, -0.05286575318484597)), ('u', 6, (3.2220288082029924e-07, 0.5235987525250204, 1.5707959588509708)), ('u', 7, (3.2604676001546253, 1.5805733622889064, -0.5462164614964702)), ('u', 8, (3.2682886477828843, 0.10099498534231649, -3.0017602212067187)), ('u', 9, (3.0856831972259307, 0.11661485626173815, -3.1897108033411654)), ('cz', 6, 7), ('u', 5, (-0.052865753185088196, 2.066556146414766, -0.16968181001095317)), ('u', 6, (1.5707959588520604, 2.0943950969105747, 4.6485461369094594e-08)), ('u', 7, (2.595376192091419, 0.10339962874383246, -1.7348142478113004e-08)), ('u', 8, (-3.0017602212072174, 0.021805535212770306, 3.2132046790382023)), ('u', 9, (3.0934750241333817, 0.13516319747216235, -3.181110747783714)), ('cz', 5, 6), ('u', 5, (-0.16968181001372695, 1.5972593731470857e-07, -1.284495047452395)), ('u', 6, (4.648911666738087e-08, 1.0471974534589155, 1.4059841682778922)), ('u', 7, (-3.1415926709367907, 0.10339947773108651, 2.5007861916883405)), ('u', 8, (3.213204679037897, 0.06844870694314473, -3.0101510717721007)), ('u', 9, (3.102073963545437, 1.7181061008822938, 0.14193028622103854)), ('u', 0, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 1, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 0, 1), ('u', 1, (1.570796326794898, 0.4636476090008061, -1.570796326794898)), ('u', 0, (1.5707963267948966, 0.4636476090008061, -1.5707963267948966)), ('cz', 0, 1), ('u', 1, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 0, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 1, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.1277577648985897, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.44303856189630686, 0.0)), ('cz', 2, 1), ('u', 1, (-3.141592653589793, 0.44303856189630686, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 1.1277577648985897, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.2284367252937551, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 0.3423596015011414, 0.0)), ('cz', 3, 2), ('u', 2, (-3.141592653589793, 0.3423596015011414, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 1.2284367252937551, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 0.23182380450040305, 0.0)), ('cz', 4, 3), ('u', 3, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 1.3389725222944935, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 0, (-0.059244378832045946, 1.5707963686198043, 2.2849472185865194e-08)), ('u', 1, (3.0959677616944874, 1.5707963320749094, -3.141592690729246)), ('u', 2, (0.03160267682213891, 0.0037217677374817297, 0.03959518769352288)), ('u', 3, (2.6452513395795365, 0.2778308265410945, -3.5830012545259042)), ('cz', 0, 1), ('u', 0, (5.4184747727732685e-08, 0.5235987700525918, -0.002707394577367658)), ('u', 1, (-9.15113551737847e-08, 0.5235987803789446, -2.194952384887468e-07)), ('u', 2, (3.181187893567229, 0.0002011244664534999, -3.096939001391856)), ('u', 3, (2.700184257101294, 0.025219118832682155, -3.4742998865127177)), ('cz', 0, 1), ('u', 0, (-3.144299991514342, 1.5861053436489332, 2.971608196384854)), ('u', 1, (9.817352836141019e-08, 1.5707961972709132, 0.010580663399824133)), ('u', 2, (3.1862465489000016, 1.574265066242556, 0.00032172399836749577)), ('u', 3, (-0.3327072298965769, 0.11325079593542514, -0.24122290398308946)), ('cz', 1, 2), ('u', 0, (-0.16998439805345658, 0.01623823502449395, -0.16295208351756255)), ('u', 1, (-3.13101206339036, 1.9635033147499876, 3.1342225610372885)), ('u', 2, (3.1419145596196376, 1.5703524734857628, -0.04200279006060459)), ('u', 3, (-0.24122311625926196, 0.3110251197866584, -0.17765052822101818)), ('cz', 0, 1), ('u', 0, (2.9786406742188127, 2.8787615647615134e-09, -3.2924970407046317)), ('u', 1, (3.134222346575787, 0.39274545637480085, -3.141592653137736)), ('u', 2, (-0.0420027600240184, 0.0004453836566589408, -0.02421870470925712)), ('u', 3, (-0.17765052187495012, 0.3287497476202555, -0.09493277803655581)), ('cz', 1, 2), ('u', 0, (-0.1509042832411338, 2.3527484075697625e-08, -0.15696349934287268)), ('u', 1, (4.4290569112866857e-08, 0.3927454546710999, -0.0032540395693289154)), ('u', 2, (-0.024218522331509958, 0.7913756622189563, -2.837369150569813e-08)), ('u', 3, (-0.0949328850180543, 0.33577845437490766, -0.009797464038533661)), ('cz', 0, 1), ('u', 0, (2.9846292322658776, 0.029524520355996668, -3.2967998400740743)), ('u', 1, (-0.003254029500380281, 1.5507375703321857, 0.024447612496885214)), ('u', 2, (3.1415924582350163, 0.7913757056926529, -0.027561507068230906)), ('u', 3, (-0.009797462507119346, 0.3473459792088328, 0.06627677623101205)), ('cz', 2, 3), ('u', 0, (2.986385389117144, 0.022946351964179473, -3.2951523446491615)), ('u', 1, (0.024447504667370015, 0.4132852785134742, 0.026809655077338304)), ('u', 2, (3.114031314584472, 2.8797473594866156, -3.1323623637347957)), ('u', 3, (3.207869337017283, 1.5722601317634308, -0.008070081249314631)), ('cz', 1, 2), ('u', 0, (2.9880330145133276, 0.013697343511536226, -3.2927487805933464)), ('u', 1, (0.02680981127537451, 1.0248393918223576e-06, 0.011106173552121701)), ('u', 2, (0.009230273691737793, 1.8325990238600085, 0.03560652762476724)), ('u', 3, (-3.1496628300709117, 0.0014639943748108292, 3.1247803582845233)), ('cz', 2, 3), ('u', 0, (2.9904365008940585, 0.0006458388030150457, -3.2891279189823033)), ('u', 1, (-3.130486532245581, 9.799174002920364e-07, 3.145686728798873)), ('u', 2, (-3.10598631294529, 0.2624378293969588, 3.1072013846423765)), ('u', 3, (-3.1584048966251044, 0.02133337740785177, 3.1212174172824656)), ('cz', 1, 2), ('u', 0, (-0.14753547352766716, 0.015126904494877108, -0.14261685724603496)), ('u', 1, (3.1456868850236934, 0.008770113678556457, -3.1355133243054962)), ('u', 2, (-0.03439111998875516, 1.3089510327450513, 0.0035954832017138366)), ('u', 3, (3.1212173932966802, 1.5494805151872955, 0.002375022073694666)), ('cz', 2, 3), ('u', 0, (1.4281795929577539, 1.6282246508203182, -1.5681234699939128)), ('u', 1, (-1.5647173676031776, 1.5709030696723458, 1.566411646038389)), ('u', 2, (-3.137997182510539, 0.535653262644168, 3.1404738960425136)), ('u', 3, (3.1439676725878765, 1.5521759879896702, -0.034947379115504607)), ('cz', 0, 1), ('u', 0, (1.5734691836727512, 0.615479719406373, -1.567385420061161)), ('u', 1, (1.566411646070696, 0.6154796950424387, -1.0937936911580795)), ('u', 2, (3.1404740073970956, 0.5260120118238785, -3.1409385514458856)), ('u', 3, (3.1066453879912563, 0.010402389771937337, -3.171173030152234)), ('cz', 0, 1), ('u', 0, (-1.5673854198818296, 1.5717600334885746, 1.5257621550922562)), ('u', 1, (2.0477989621032746, 1.631634703969574, -1.5923391098991622)), ('u', 2, (3.1422463656611472, 2.0799281279775275, -0.012534978309999723)), ('u', 3, (3.112012745443037, 0.002767074603799922, -3.165109227551853)), ('cz', 1, 2), ('u', 0, (3.0965584549147707, 0.002330141798390607, -3.1819101992136565)), ('u', 1, (-3.1631354250742034, 0.9553166176670005, 3.089548741037573)), ('u', 2, (3.1290575279392767, 1.5707963980926052, -0.0011999714629213898)), ('u', 3, (-0.023516886114037742, 0.004137823126384155, -0.01676046471330975)), ('cz', 1, 2), ('u', 0, (3.105980188556091, 0.00456380398229825, -3.1739318039822426)), ('u', 1, (3.1015837500533294, 0.7853982110452473, -3.141592670907425)), ('u', 2, (-0.004713665764903096, 0.0022060153615032036, 0.0029567986106184756)), ('u', 3, (3.1305006364553165, 0.0028324232637507666, -3.1445910330522864)), ('cz', 0, 1), ('u', 0, (3.109253216864846, 0.3006524441055976, -3.153597231733226)), ('u', 1, (2.0489525315118072e-08, 0.7853981070920558, -5.835236561784377e-08)), ('u', 2, (3.1445494842890565, 1.5730023570193474, 0.03991208356279641)), ('u', 3, (-0.0029981713507351553, 0.002076214866011046, 0.005796698292644474)), ('cz', 1, 2), ('u', 0, (-0.012004233677426507, 1.2558697273250936, -0.004353751175417566)), ('u', 1, (-6.145186754418978e-08, 0.7853981929441164, -7.64717795656315e-08)), ('u', 2, (3.181504902860366, 2.0886942818220944, 0.008331263429352642)), ('u', 3, (0.005796698489485174, 0.006394328085055529, 0.015220097372097233)), ('cz', 0, 1), ('u', 0, (3.1372391493430065, 0.7853982136262638, -3.141592696714178)), ('u', 1, (1.555989188337454e-07, 1.0471975268590015, 1.631771969805461e-08)), ('u', 2, (0.008331617650812851, 0.5254380381340221, -0.014379805162403541)), ('u', 3, (0.015220149331022777, 0.00994263487867583, 0.025110883675483062)), ('cz', 0, 1), ('u', 0, (-3.1415928124946193, 0.6154797173769657, 3.0996313907974367)), ('u', 1, (-3.141592729218556, 1.5707962371061959, 3.0996312249935807)), ('u', 2, (-0.014380117290231684, 0.5275805675161466, -0.03760152574040778)), ('u', 3, (0.02511067542797974, 0.012424574222275513, 0.03517379932621378)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0))]), (10, [('u', 4, (0.0, 3.0155210212709935, 0.0)), ('u', 3, (0.0, 0.32108595465310086, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 0.32108595465310086, 3.141592653589793)), ('u', 2, (0.0, 0.7277483918509393, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 0.7277483918509393, 3.141592653589793)), ('u', 1, (0.0, 1.0992585722264043, 0.0)), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 1), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.0992585722264043, 3.141592653589793)), ('u', 0, (0.0, 1.3734007669450157, 0.0)), ('u', 0, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 0), ('u', 0, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (-3.141592653589793, 1.3734007669450157, 3.141592653589793)), ('u', 5, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 6, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 6), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 7), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 0, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 1, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 0, 1), ('u', 1, (1.570796326794898, 0.4636476090008061, -1.570796326794898)), ('u', 0, (1.5707963267948966, 0.4636476090008061, -1.5707963267948966)), ('cz', 0, 1), ('u', 1, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 0, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 1, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.1277577648985897, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.44303856189630686, 0.0)), ('cz', 2, 1), ('u', 1, (-3.141592653589793, 0.44303856189630686, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 1.1277577648985897, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.2284367252937551, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 0.3423596015011414, 0.0)), ('cz', 3, 2), ('u', 2, (-3.141592653589793, 0.3423596015011414, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 1.2284367252937551, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 0.23182380450040305, 0.0)), ('cz', 4, 3), ('u', 3, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 1.3389725222944935, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 0, (-0.059244378832045946, 1.5707963686198043, 2.2849472185865194e-08)), ('u', 1, (3.0959677616944874, 1.5707963320749094, -3.141592690729246)), ('u', 2, (0.03160267682213891, 0.0037217677374817297, 0.03959518769352288)), ('u', 3, (2.6452513395795365, 0.2778308265410945, -3.5830012545259042)), ('u', 9, (-3.3084517545262417, 0.041974593783753025, 2.9532081700224535)), ('cz', 0, 1), ('u', 0, (5.4184747727732685e-08, 0.5235987700525918, -0.002707394577367658)), ('u', 1, (-9.15113551737847e-08, 0.5235987803789446, -2.194952384887468e-07)), ('u', 2, (3.181187893567229, 0.0002011244664534999, -3.096939001391856)), ('u', 3, (2.700184257101294, 0.025219118832682155, -3.4742998865127177)), ('u', 9, (-3.3299773456715362, 0.04412782070667279, 2.950568590896067)), ('cz', 0, 1), ('u', 0, (-3.144299991514342, 1.5861053436489332, 2.971608196384854)), ('u', 1, (9.817352836141019e-08, 1.5707961972709132, 0.010580663399824133)), ('u', 2, (3.1862465489000016, 1.574265066242556, 0.00032172399836749577)), ('u', 3, (-0.3327072298965769, 0.11325079593542514, -0.24122290398308946)), ('u', 9, (2.9505686933379685, 0.03931485851083587, -3.317651631131255)), ('cz', 1, 2), ('u', 0, (-0.16998439805345658, 0.01623823502449395, -0.16295208351756255)), ('u', 1, (-3.13101206339036, 1.9635033147499876, 3.1342225610372885)), ('u', 2, (3.1419145596196376, 1.5703524734857628, -0.04200279006060459)), ('u', 3, (-0.24122311625926196, 0.3110251197866584, -0.17765052822101818)), ('u', 9, (2.965533677395084, 0.030360448374833766, -3.292799756924407)), ('cz', 0, 1), ('u', 0, (2.9786406742188127, 2.8787615647615134e-09, -3.2924970407046317)), ('u', 1, (3.134222346575787, 0.39274545637480085, -3.141592653137736)), ('u', 2, (-0.0420027600240184, 0.0004453836566589408, -0.02421870470925712)), ('u', 3, (-0.17765052187495012, 0.3287497476202555, -0.09493277803655581)), ('u', 9, (2.990385238138636, 0.020906614734777576, -3.2717732823546615)), ('cz', 1, 2), ('u', 0, (-0.1509042832411338, 2.3527484075697625e-08, -0.15696349934287268)), ('u', 1, (4.4290569112866857e-08, 0.3927454546710999, -0.0032540395693289154)), ('u', 2, (-0.024218522331509958, 0.7913756622189563, -2.837369150569813e-08)), ('u', 3, (-0.0949328850180543, 0.33577845437490766, -0.009797464038533661)), ('u', 9, (3.01141202531293, 0.013744195112854608, -3.26832175137755)), ('cz', 0, 1), ('u', 0, (2.9846292322658776, 0.029524520355996668, -3.2967998400740743)), ('u', 1, (-0.003254029500380281, 1.5507375703321857, 0.024447612496885214)), ('u', 2, (3.1415924582350163, 0.7913757056926529, -0.027561507068230906)), ('u', 3, (-0.009797462507119346, 0.3473459792088328, 0.06627677623101205)), ('u', 9, (-3.2683217505295032, 0.012111420552200406, 2.994763382617767)), ('cz', 2, 3), ('u', 0, (2.986385389117144, 0.022946351964179473, -3.2951523446491615)), ('u', 1, (0.024447504667370015, 0.4132852785134742, 0.026809655077338304)), ('u', 2, (3.114031314584472, 2.8797473594866156, -3.1323623637347957)), ('u', 3, (3.207869337017283, 1.5722601317634308, -0.008070081249314631)), ('u', 9, (-3.288421924540883, 0.02012946498056516, 2.962872104430667)), ('cz', 1, 2), ('u', 0, (2.9880330145133276, 0.013697343511536226, -3.2927487805933464)), ('u', 1, (0.02680981127537451, 1.0248393918223576e-06, 0.011106173552121701)), ('u', 2, (0.009230273691737793, 1.8325990238600085, 0.03560652762476724)), ('u', 3, (-3.1496628300709117, 0.0014639943748108292, 3.1247803582845233)), ('u', 9, (-3.3203130977480133, 0.030502887666218, 2.9352512080385718)), ('cz', 2, 3), ('u', 0, (2.9904365008940585, 0.0006458388030150457, -3.2891279189823033)), ('u', 1, (-3.130486532245581, 9.799174002920364e-07, 3.145686728798873)), ('u', 2, (-3.10598631294529, 0.2624378293969588, 3.1072013846423765)), ('u', 3, (-3.1584048966251044, 0.02133337740785177, 3.1212174172824656)), ('u', 9, (-3.347933893162299, 0.037654313548154614, 2.9198760758168874)), ('cz', 1, 2), ('u', 0, (-0.14753547352766716, 0.015126904494877108, -0.14261685724603496)), ('u', 1, (3.1456868850236934, 0.008770113678556457, -3.1355133243054962)), ('u', 2, (-0.03439111998875516, 1.3089510327450513, 0.0035954832017138366)), ('u', 3, (3.1212173932966802, 1.5494805151872955, 0.002375022073694666)), ('u', 9, (-3.363309124520733, 0.040741909715571976, 2.9195302900122884)), ('cz', 2, 3), ('u', 0, (1.4281795929577539, 1.6282246508203182, -1.5681234699939128)), ('u', 1, (-1.5647173676031776, 1.5709030696723458, 1.566411646038389)), ('u', 2, (-3.137997182510539, 0.535653262644168, 3.1404738960425136)), ('u', 3, (3.1439676725878765, 1.5521759879896702, -0.034947379115504607)), ('u', 9, (2.9195300920867098, 0.0396148955294622, -3.3482563315094893)), ('cz', 0, 1), ('u', 0, (1.5734691836727512, 0.615479719406373, -1.567385420061161)), ('u', 1, (1.566411646070696, 0.6154796950424387, -1.0937936911580795)), ('u', 2, (3.1404740073970956, 0.5260120118238785, -3.1409385514458856)), ('u', 3, (3.1066453879912563, 0.010402389771937337, -3.171173030152234)), ('u', 9, (2.934928977552804, 0.033976558827776594, -3.318827914890996)), ('cz', 0, 1), ('u', 0, (-1.5673854198818296, 1.5717600334885746, 1.5257621550922562)), ('u', 1, (2.0477989621032746, 1.631634703969574, -1.5923391098991622)), ('u', 2, (3.1422463656611472, 2.0799281279775275, -0.012534978309999723)), ('u', 3, (3.112012745443037, 0.002767074603799922, -3.165109227551853)), ('u', 9, (2.9643572747015305, 0.023942444250524355, -3.283287891880807)), ('cz', 1, 2), ('u', 0, (3.0965584549147707, 0.002330141798390607, -3.1819101992136565)), ('u', 1, (-3.1631354250742034, 0.9553166176670005, 3.089548741037573)), ('u', 2, (3.1290575279392767, 1.5707963980926052, -0.0011999714629213898)), ('u', 3, (-0.023516886114037742, 0.004137823126384155, -0.01676046471330975)), ('u', 9, (2.9998975184180243, 0.012798159106612932, -3.2580103254545194)), ('cz', 1, 2), ('u', 0, (3.105980188556091, 0.00456380398229825, -3.1739318039822426)), ('u', 1, (3.1015837500533294, 0.7853982110452473, -3.141592670907425)), ('u', 2, (-0.004713665764903096, 0.0022060153615032036, 0.0029567986106184756)), ('u', 3, (3.1305006364553165, 0.0028324232637507666, -3.1445910330522864)), ('u', 9, (3.0388906020030366, 0.005708487407306541, -3.2378983501360863)), ('cz', 0, 1), ('u', 0, (3.109253216864846, 0.3006524441055976, -3.153597231733226)), ('u', 1, (2.0489525315118072e-08, 0.7853981070920558, -5.835236561784377e-08)), ('u', 2, (3.1445494842890565, 1.5730023570193474, 0.03991208356279641)), ('u', 3, (-0.0029981713507351553, 0.002076214866011046, 0.005796698292644474)), ('u', 9, (-3.2378985598754033, 0.006047132015164575, 3.0353833434364264)), ('cz', 1, 2), ('u', 0, (-0.012004233677426507, 1.2558697273250936, -0.004353751175417566)), ('u', 1, (-6.145186754418978e-08, 0.7853981929441164, -7.64717795656315e-08)), ('u', 2, (3.181504902860366, 2.0886942818220944, 0.008331263429352642)), ('u', 3, (0.005796698489485174, 0.006394328085055529, 0.015220097372097233)), ('u', 9, (-3.2478020660763955, 0.00961654585359189, 3.016187602881469)), ('cz', 0, 1), ('u', 0, (3.1372391493430065, 0.7853982136262638, -3.141592696714178)), ('u', 1, (1.555989188337454e-07, 1.0471975268590015, 1.631771969805461e-08)), ('u', 2, (0.008331617650812851, 0.5254380381340221, -0.014379805162403541)), ('u', 3, (0.015220149331022777, 0.00994263487867583, 0.025110883675483062)), ('u', 9, (-3.266997496248343, 0.012744244361027224, 2.995205213403631)), ('cz', 0, 1), ('u', 0, (-3.1415928124946193, 0.6154797173769657, 3.0996313907974367)), ('u', 1, (-3.141592729218556, 1.5707962371061959, 3.0996312249935807)), ('u', 2, (-0.014380117290231684, 0.5275805675161466, -0.03760152574040778)), ('u', 3, (0.02511067542797974, 0.012424574222275513, 0.03517379932621378)), ('u', 9, (2.9952054061562308, 1.582967171554144, -0.023277656121475587)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 5, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 6, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 5, 6), ('u', 6, (1.570796326794898, 0.4636476090008061, -1.570796326794898)), ('u', 5, (1.5707963267948966, 0.4636476090008061, -1.5707963267948966)), ('cz', 5, 6), ('u', 6, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 5, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 6, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 6, 7), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 6, (-3.141592653589793, 1.1277577648985897, 3.141592653589793)), ('cz', 5, 6), ('u', 6, (0.0, 0.44303856189630686, 0.0)), ('cz', 7, 6), ('u', 6, (-3.141592653589793, 0.44303856189630686, 3.141592653589793)), ('cz', 5, 6), ('u', 6, (0.0, 1.1277577648985897, 0.0)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 6, 7), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (-3.141592653589793, 1.2284367252937551, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (0.0, 0.3423596015011414, 0.0)), ('cz', 8, 7), ('u', 7, (-3.141592653589793, 0.3423596015011414, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (0.0, 1.2284367252937551, 0.0)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 0.23182380450040305, 0.0)), ('cz', 9, 8), ('u', 8, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 1.3389725222944935, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 5, (-0.31354899575689466, 1.570796348248604, 2.1993250581830992e-08)), ('u', 6, (3.1315564526759854, 1.5707963368733475, -3.1415926381074835)), ('u', 7, (-2.962107879717114, 0.1690703254943603, 3.3002840545635106)), ('u', 8, (0.04426013852424536, 0.09461413255901885, 0.015484814177879412)), ('cz', 5, 6), ('u', 5, (2.1993051651549023e-08, 0.5235987738103077, 0.0077199356018875275)), ('u', 6, (1.5482548109131853e-08, 0.5235987774602098, -3.6193629251409554e-08)), ('u', 7, (-2.9829012526145964, 0.028988862479655767, 3.291183650119445)), ('u', 8, (0.01548481417727731, 0.0985894787239515, -0.12462151708972341)), ('cz', 5, 6), ('u', 5, (3.1493125891917515, 1.607214300658221, -2.9410427224524387)), ('u', 6, (-3.6193198604816956e-08, 1.5707963446847064, -0.6803527694078231)), ('u', 7, (3.2911836501182434, 1.4045840328846662, -0.05037345321435982)), ('u', 8, (-0.12462151708813557, 0.1217245995369984, -0.11603566299378823)), ('cz', 6, 7), ('u', 5, (0.200549931138667, 0.039554397728295436, 0.1874516756560237)), ('u', 6, (2.461239884182032, 1.1741162707066524, -2.9837508456155004)), ('u', 7, (3.0912192003761167, 1.5622090936441297, -0.270099088537076)), ('u', 8, (-0.11603566299537041, 0.1377042213935611, -0.06582035526042324)), ('cz', 5, 6), ('u', 5, (-2.954140977934238, 2.071375762486455e-08, 3.2356413228839016)), ('u', 6, (0.15784180797392006, 0.4150603543191284, -2.541361455998814e-08)), ('u', 7, (-0.27009908853803793, 0.01001303545670484, -0.276697681490739)), ('u', 8, (-0.06582035525963252, 0.14627073620069064, 0.013465957460653388)), ('cz', 6, 7), ('u', 5, (0.09404866929363506, 2.343208726496092e-09, 0.36799964248223205)), ('u', 6, (-3.1415926790033843, 0.41506036800121915, 2.983750897469722)), ('u', 7, (-0.2766976814919512, 0.2539961283422237, -0.20478014269006642)), ('u', 8, (0.013465957459560425, 0.15616220618211132, 0.08005305480593586)), ('cz', 5, 6), ('u', 5, (-2.773593011108606, 0.0982665135669904, 3.461287432999035)), ('u', 6, (-0.15784175611850518, 1.1741163093455893, 0.5298295171057543)), ('u', 7, (2.9368125109010474, 1.143357836097282, -0.06336719313121364)), ('u', 8, (0.08005305480657139, 0.17594386496352027, 0.11134308724824167)), ('cz', 6, 7), ('u', 5, (0.31969477940933083, 0.17549678693921383, 0.20700857109975496)), ('u', 6, (-2.611763136483908, 9.161019252172947e-08, 3.4089240359561335)), ('u', 7, (3.0782254604590547, 0.9134778575400373, 0.31425380476085363)), ('u', 8, (3.2529357408379576, 1.3620206329529414, 0.14190091105862912)), ('cz', 7, 8), ('u', 5, (0.20700857109935059, 0.3499024956174044, 0.0924291070745549)), ('u', 6, (-2.874261271223095, 7.460973450095163e-09, 3.367256728172397)), ('u', 7, (-2.827338848827801, 2.107696756854845, 3.0261401607843963)), ('u', 8, (3.283493564649334, 1.5952689200113879, 0.024395455648290376)), ('cz', 6, 7), ('u', 5, (0.09242910707444361, 0.4552116555119754, 0.0020802105457499026)), ('u', 6, (0.22566407458416737, 1.7185357264637598e-09, 0.46839003569813864)), ('u', 7, (-0.11545249280603109, 0.8132164963119031, 0.907428111053003)), ('u', 8, (-3.1171971979412985, 0.0245017495176529, 3.1289963717271054)), ('cz', 7, 8), ('u', 5, (0.0020802105469115498, 0.529149116081907, -0.07083458128890488)), ('u', 6, (0.4683900356961994, 1.731310403571253e-08, 0.14055507025980238)), ('u', 7, (-2.2341645425361687, 0.8192675536808522, 2.54446143664365)), ('u', 8, (3.128996371726612, 0.23153691825200126, -3.152439925086105)), ('cz', 6, 7), ('u', 5, (-0.07083458128950199, 0.5777434449892271, -0.1569770743260479)), ('u', 6, (0.14055507026024464, 0.9257914466412299, -0.32949901939952164)), ('u', 7, (-0.5971312169458483, 1.1959143174845979, -0.20275013649153936)), ('u', 8, (3.1307453820940014, 1.7995735659040661, 0.0041500674825698525)), ('cz', 7, 8), ('u', 5, (1.4138192524690716, 2.2425711928527012, -1.0782176873149583)), ('u', 6, (-1.9002953461945373, 0.7211107812727262, 0.7621310561843645)), ('u', 7, (-0.20275013649084023, 0.780367094265597, -0.27811257707134796)), ('u', 8, (3.145742721071761, 1.6067639117089805, 0.013136437615879348)), ('cz', 5, 6), ('u', 5, (2.063374966274772, 0.9117382850503745, -2.3782810528623877)), ('u', 6, (0.7621310561845498, 0.6847191940449716, -1.5707962723204008)), ('u', 7, (-0.2781125770713845, 0.305832232376102, -0.3289919589435366)), ('u', 8, (0.01313643761552964, 0.012622711159589614, 0.009745414323725476)), ('cz', 5, 6), ('u', 5, (-2.3782810528619525, 2.682020782783948, 1.4942718950784355)), ('u', 6, (1.5707963812688315, 1.107148715188061, -0.7853982199335239)), ('u', 7, (2.812600694644471, 1.7982732275281512, 0.5641126751160201)), ('u', 8, (-3.1318472392662815, 0.002128756528861336, 3.1485124191960616)), ('cz', 6, 7), ('u', 5, (-3.218117085306983, 1.7773575131675587, 2.9638729301408997)), ('u', 6, (-2.356194546728586, 2.1862760309864337, 2.3695870476802496)), ('u', 7, (3.7057053287052035, 1.7961737939818707, 0.003219532604020081)), ('u', 8, (-3.134672887983742, 0.01009300266311791, 3.1454756372242376)), ('cz', 5, 6), ('u', 5, (-0.17771972344973944, 1.22211975038942, -0.04450422874502319)), ('u', 6, (-0.7720056059100122, 4.916641364354111e-08, -0.7987907329668211)), ('u', 7, (0.0032195326047858297, 0.19516115634306297, -0.08156829139487777)), ('u', 8, (-3.137709669955173, 0.01300482316700724, 3.1417634564324404)), ('cz', 6, 7), ('u', 5, (-0.044504228745603086, 1.1388645184527086, 0.16588410546314863)), ('u', 6, (2.342801920624021, 2.034443947884638, -3.1415926496573805)), ('u', 7, (-0.08156829139479031, 0.015618813113860047, 0.0781400338490316)), ('u', 8, (-3.1414218507472964, 0.012264455819588958, 3.1371189241819004)), ('cz', 5, 6), ('u', 5, (0.16588410546334362, 0.5382990883095387, -0.060992309108510494)), ('u', 6, (3.932380532312152e-09, 0.523598768295892, 1.5707963182508151)), ('u', 7, (3.2197326874384977, 1.5862248465027309, -0.60282418648102)), ('u', 8, (-3.146066382997916, 0.00896686088543459, 3.1314243468960785)), ('cz', 6, 7), ('u', 5, (-0.060992309108772756, 2.0863622358785485, -0.14132878173133226)), ('u', 6, (1.5707963182502973, 2.0943950733564867, 1.7712899591515452e-08)), ('u', 7, (2.5387684671090436, 0.35177688547152486, -2.4061335146541296e-09)), ('u', 8, (-3.1517609602834327, 0.004011045196291686, 3.124648487797276)), ('cz', 5, 6), ('u', 5, (-0.1413287817320773, 6.756594294900579e-08, -1.3351601720973787)), ('u', 6, (1.7711736854941762e-08, 1.0471975570507968, 1.3986816241709206)), ('u', 7, (-3.1415926559956278, 0.3517768697553526, 2.6067608898256687)), ('u', 8, (-0.016944165793196114, 0.0017869393062053724, -0.024795398854180275)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0))])], (11, 2): [(11, [('u', 4, (0.0, 2.0426581640745387, 0.0)), ('u', 3, (0.0, 1.0471975511965976, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.0471975511965976, 3.141592653589793)), ('u', 9, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 10), ('u', 0, (-3.1413300529676143, 0.016049892372225413, 2.6310957151665475)), ('u', 1, (-3.141334271329016, 0.10790196324571806, 2.4951626375093583)), ('u', 2, (0.0006102942423147906, 0.8860770992408441, 0.16956806225969473)), ('u', 3, (5.683986992270052, 2.264332608013758e-08, -0.07393694775744386)), ('u', 4, (-0.008070055280788345, 0.16213446703928294, 0.024201808703938066)), ('cz', 2, 3), ('u', 0, (-3.6477065269341726, 0.05672717868556697, 2.573060872959038)), ('u', 1, (-0.7049831267758607, 0.1441761127167599, -0.5150273787798235)), ('u', 2, (2.9421743581226725, 1.2835370959620422, -3.1714125389099683)), ('u', 3, (0.7482698098280074, 1.5707963297419563, -5.497787113241561)), ('u', 4, (3.2016959709882635, 1.7323649322006993, 0.06794890359063821)), ('cz', 3, 4), ('u', 0, (2.573060872966243, 0.1286366251933175, -3.598854991357485)), ('u', 1, (-0.5150273787795427, 0.02680456184655866, -0.33211634244805344)), ('u', 2, (-0.02981988532131627, 0.3983140742529721, -0.057631019209893844)), ('u', 3, (-2.356194459652187, 1.5707963505605989, 2.5880182950437236)), ('u', 4, (3.20954155718086, 2.124637006983497, 0.06404200235888746)), ('cz', 2, 3), ('u', 0, (2.6843303158564105, 0.13016088390058414, -3.305551108470282)), ('u', 1, (-3.473708996035658, 0.1164455944601965, 2.6704345157926572)), ('u', 2, (3.0839616343808935, 0.5282529217968184, -3.1415926717850096)), ('u', 3, (-0.5535743585462398, 1.5707962645420361, -0.6539032631865938)), ('u', 4, (3.205634655949053, 1.0241536819081383, -0.06510830243098908)), ('cz', 3, 4), ('u', 7, (3.3647163380877396, 0.3642904512849049, -2.562503679351548)), ('u', 8, (1.2543194992414728, 0.9553166198640104, -0.8641577840658705)), ('u', 9, (-1.7406080648054003, 3.7473105113670694e-08, 1.8272609241571824)), ('u', 4, (3.1888333656188483, 2.039993577126575, -0.22148998508761997)), ('u', 10, (3.3151230142032393, 1.9679511724629917, 0.1144338419786659)), ('cz', 8, 9), ('u', 7, (0.21745043555831609, 0.09409940264802959, 0.13406035073996295)), ('u', 8, (-1.5853592654492863, 1.3757533784029135, 0.9868128727387029)), ('u', 9, (2.80833638620301, 1.5707963208811606, -1.9644628938921689)), ('u', 4, (-2.7564764829549357, 0.7732517259761388, 3.082057610635956)), ('u', 10, (2.768991399708501, 1.913703732121055, 0.09404797475257887)), ('cz', 9, 10), ('u', 7, (3.2514508499638404, 0.15907520131247813, -2.802036509050106)), ('u', 8, (3.0192057743818763, 0.7544976667580353, -3.2635667387970653)), ('u', 9, (-2.7479261243718427, 1.5707963650141654, 1.8575087912524408)), ('u', 4, (2.5838285084628705, 0.3380246806716943, -3.5860923186797695)), ('u', 10, (2.645859493075393, 0.7833209533319082, 0.19167664126583084)), ('cz', 8, 9), ('u', 7, (1.2604375623510466, 0.08261304144783904, 0.07101523966968715)), ('u', 8, (0.06258553853238874, 0.21072893933328277, 0.024461585062846788)), ('u', 9, (0.13382186909033422, 1.5707964224488247, -0.009104766423228439)), ('u', 4, (-2.824349607315286, 0.21474414320238785, 3.3898980089165027)), ('u', 10, (3.0807267729553134, 1.9273857231501672, 0.20640520391841077)), ('cz', 9, 10), ('u', 5, (-3.141620576518707, 0.0034534167729727734, 2.8513306392987827)), ('u', 6, (7.284772311965683e-05, 0.14885104127511004, 0.0009984636960826008)), ('u', 7, (0.05755570992613236, 1.156521713491745, -0.07734142705372159)), ('u', 8, (0.09844683346637574, 1.3616109993097807, -0.07136076273491827)), ('u', 9, (-3.4169676296579663, 0.6412324806023405, 2.3108022881684347)), ('cz', 7, 8), ('u', 5, (-0.29026243057839735, 0.008396826120885745, -0.3598749614788291)), ('u', 6, (0.000997891303343438, 0.15350342252423685, -0.0072207374961828635)), ('u', 7, (3.1235475953670617, 0.9383977658607346, -2.6140215258277912)), ('u', 8, (-0.31800229155181514, 0.8860771183419569, -0.12722618751874937)), ('u', 9, (-0.2878137473456245, 0.4918513739998731, -0.02240255902715363)), ('cz', 7, 8), ('u', 5, (-0.3598751696272701, 0.012725148487746297, -0.40820315287776404)), ('u', 6, (-0.007220945644116669, 0.15725536451652394, -0.008713730599668512)), ('u', 7, (2.9592184549277616, 1.4347661266917102, -2.9015960016918485)), ('u', 8, (-3.8607070487733304, 2.400889244986718, 2.3242867737036303)), ('u', 9, (2.9339487711445478, 1.881936784726261, -0.6623450174914631)), ('cz', 8, 9), ('u', 5, (-0.40820283903073223, 0.013096430218649806, -0.41058665969708513)), ('u', 6, (-0.008713416752537152, 0.16094805940016083, -0.0026490927572300196)), ('u', 7, (0.1944891654314219, 0.3120939664651444, -0.1952463756651939)), ('u', 8, (-3.3165235982871355, 1.1071487077831936, 2.7132468401082637)), ('u', 9, (3.2240149813164067, 1.5707963147672193, 0.6943220973780453)), ('cz', 8, 9), ('u', 5, (-0.41058614096953294, 0.01612758391725595, -0.39689717040887573)), ('u', 6, (-0.0026485740295479677, 0.164936529940193, 0.009864491142022154)), ('u', 7, (-2.8511032549245563, 0.48405788688310886, 3.25684426550489)), ('u', 8, (-2.7387098010907933, 1.8323524608552493, 2.728479874582392)), ('u', 9, (0.8535780089496929, 0.9988753773021094, 0.15368231094491963)), ('cz', 6, 7), ('u', 5, (-0.12171280384175057, 0.5617733993287385, -0.39040765694525104)), ('u', 6, (3.1469604652503467, 0.8052713369228046, -2.9094526228281765)), ('u', 7, (0.9931922139901161, 2.094395114010817, 1.3146634374864414)), ('u', 8, (3.5547054087512953, 1.3092402003439705, -2.4215721042501652)), ('u', 9, (3.208760366704987, 0.04593263650434926, -3.074332957255127)), ('cz', 7, 8), ('u', 5, (0.4911815346363252, 0.4267308541999764, -0.19494687707366554)), ('u', 6, (-0.43277504600334077, 0.02028821429429574, -0.8045777925590265)), ('u', 7, (-2.6180725871176946, 1.1416160516938738, 3.269319441543259)), ('u', 8, (4.492038488570008, 1.0407302767265691, -0.3727192598059128)), ('u', 9, (3.2088523499244563, 0.02522574139480211, -3.071899824757519)), ('cz', 6, 7), ('u', 5, (-0.00042037645989224937, 0.4141867196081669, -0.09114193661349101)), ('u', 6, (2.1590623532853193, 1.3756466980198663, -3.2020209260932195)), ('u', 7, (0.8608619121364985, 1.570796304912637, 0.47926769844106853)), ('u', 8, (3.383792137762433, 2.6077927368508873, 0.05625539339491281)), ('u', 9, (3.2112854824220696, 0.009572423785491486, -3.0674196907844733)), ('cz', 7, 8), ('u', 5, (1.5731693831487696, 0.6276074107442877, -1.2319234523543967)), ('u', 6, (-1.5795943713543288, 1.2341747964556231, 2.6881354007485765)), ('u', 7, (0.08713039313246476, 1.3250471424965153, -0.4409702738073087)), ('u', 8, (-2.7913101937945304, 0.7705910220259296, 2.921017422192199)), ('u', 9, (0.07417296280532068, 0.0015705586697842672, 0.08020980754772514)), ('cz', 5, 6), ('u', 5, (-0.08089138655997119, 0.4870658682340941, -2.054010845502603)), ('u', 6, (3.7903579498497333, 0.6154797113738913, -1.22040288336344)), ('u', 7, (1.8992421895806388, 1.1498715846688665, 0.6647373510878435)), ('u', 8, (-3.3579674316902137, 0.7030654905871019, 1.8941350235808698)), ('u', 9, (0.0802098075477246, 0.008779778833128941, 0.08720604052005941)), ('cz', 6, 7), ('u', 5, (3.0902833177738613, 1.3176195169910028, -2.32370206556319)), ('u', 6, (3.406678849948572, 1.5707963086153904, -2.18405820600957)), ('u', 7, (-1.9918823651733606, 0.8212476734353905, 3.7619914334384177)), ('u', 8, (-0.7831881340860232, 0.08309165846221452, -1.0268058503012185)), ('u', 9, (0.08720604052006062, 0.012462357801540872, 0.09451486259378711)), ('cz', 5, 6), ('u', 5, (-2.3030082376951473, 0.16213997207803457, 2.823414849459808)), ('u', 6, (0.6132618838334354, 1.57079645741682, -2.232258351024673)), ('u', 7, (3.6283829034732014, 1.992140098878208, -0.5493065063818559)), ('u', 8, (-0.752427267586084, 0.44069221738862063, -0.7905620350588268)), ('u', 9, (0.09451486259379321, 0.012792471898905856, 0.10147504764820364)), ('cz', 6, 7), ('u', 5, (0.6001653464413081, 1.7266376519863316, -0.7571310031890881)), ('u', 6, (-2.480130682760816, 2.356194505706553, -0.6280510011224179)), ('u', 7, (2.8518275785431264, 1.5734764341584064, -0.447547490323724)), ('u', 8, (-0.8699607158278156, 0.2431214198697064, -0.723833371655109)), ('u', 9, (0.10147504764819847, 0.00972354194010656, 0.10743505408843639)), ('cz', 5, 6), ('u', 5, (2.3728628974169568, 1.5707963575829609, -1.3246973414330871)), ('u', 6, (0.628050998006557, 1.5707962946036595, -1.3246973313056483)), ('u', 7, (-1.1524944523161604, 0.09139959694773855, 0.27546714795815513)), ('u', 8, (-1.601299173606403, 0.18754059213334195, -0.10978926979508652)), ('u', 9, (3.2490277076782323, 1.5677712593107498, 0.09780930892878859)), ('u', 4, (-0.2584175918696197, 0.44008158758243593, -0.1343828929941105)), ('u', 10, (3.1753130569771892, 1.0775117594421955, 0.36690208023432946)), ('u', 4, (-2.6405434002386166, 0.5367609955174456, 3.354176478395382)), ('u', 10, (-2.6310917071396447, 0.5126584880277414, 3.112469506986639)), ('u', 4, (-0.10789481536782598, 0.20473900015661003, -0.012773489700559526)), ('u', 10, (-0.7371237683299199, 0.6994109437975933, 0.3057765563577452)), ('u', 4, (-0.695764277746694, 0.22665876018000258, 0.27861988718696995)), ('u', 10, (0.013176368110199704, 0.19321223320151706, -0.4893698785033245)), ('u', 4, (2.3831892344803567, 0.44566932138677856, -2.8556751246459138)), ('u', 10, (-2.925326875891609, 0.1837759718820046, 2.986100091900239)), ('u', 4, (0.6009277643979518, 0.7754535346995289, 0.46341056805523384)), ('u', 10, (0.37447890503462866, 0.14457661874036226, -0.3284944166762539)), ('u', 0, (2.9776341986942363, 0.05353961774145007, -3.0792912619989803)), ('u', 1, (-0.47115813779722227, 0.9050182883328516, -0.026436372384945533)), ('u', 2, (-1.8195526391151517e-08, 0.5282529429708858, -0.17789173461637334)), ('u', 3, (2.487689390403205, 0.7121719611452743, -3.3124530049143206)), ('cz', 1, 2), ('u', 0, (0.06230139158926708, 0.07223886916244122, -0.008243254363708253)), ('u', 1, (3.1151562812051794, 0.6929762432023473, -2.6589581713982926)), ('u', 2, (2.963700918973326, 1.0471975642826303, -3.0700751937505757)), ('u', 3, (2.9707323022655894, 0.8879408364200165, -2.870637794721093)), ('cz', 2, 3), ('u', 0, (0.04322812153185805, 0.12780409715380375, 0.25113226105904746)), ('u', 1, (-2.665059996849431, 0.14745345337493918, 3.1672128137079403)), ('u', 2, (3.181711158327774, 0.5263068880292651, -2.997179730748938)), ('u', 3, (3.400292657495453, 1.0033183184379233, 0.7875653008276615)), ('cz', 1, 2), ('u', 0, (3.6014043972772067, 0.04674257669531842, -2.641906590149284)), ('u', 1, (-2.4506778665896674, 0.5042468211346128, 3.1681203385670695)), ('u', 2, (-0.24104210099365123, 1.5707963786998516, 0.5070956677680156)), ('u', 3, (4.035602919118827, 1.399065735621389, 0.3953281157531867)), ('cz', 2, 3), ('u', 0, (2.0704825035882624, 1.5157985970725332, -1.8956957691017495)), ('u', 1, (-1.5442685443939463, 1.6000518406565911, 1.6993398680714609)), ('u', 2, (0.5070955815609766, 0.005726251377664171, 0.1641478194270985)), ('u', 3, (3.536920536575754, 1.3599659999378026, 0.11335238550196669)), ('cz', 0, 1), ('u', 0, (-1.8334159994714825, 1.6429538596812, 1.4085223086459493)), ('u', 1, (2.4913968774936888, 0.6154796002273588, -1.6446094009796737)), ('u', 2, (2.740093697426668, 1.5763620370737983, -0.6048362170259756)), ('u', 3, (-3.0476852224751756, 0.04168955954147502, 3.186653756608517)), ('cz', 1, 2), ('u', 0, (2.9826382329655985, 0.17909252196273392, -2.9336711441080956)), ('u', 1, (2.5999258626750628, 1.5707963264685767, -2.356194490967865)), ('u', 2, (2.5807354610134063, 0.169860697582233, -3.3256807213984305)), ('u', 3, (0.08347547053706066, 0.04234811948948562, 0.06927926398391966)), ('cz', 0, 1), ('u', 0, (0.20792150948139726, 0.12980755822002127, 0.3564333028740683)), ('u', 1, (-2.356194490967857, 1.5707965008615785, -2.3561944924346694)), ('u', 2, (2.95750458578087, 2.028914775663822, -0.03376405451494735)), ('u', 3, (0.06927926398232259, 0.03274979416540835, 0.10131255442366094)), ('cz', 1, 2), ('u', 0, (0.3564333028739496, 1.4723620300867948, 0.7430191370934366)), ('u', 1, (0.7853981611555738, 0.7853982029947267, 3.1265864341989413e-09)), ('u', 2, (3.1078285990748573, 1.4805012305579615, -1.0665759277685276)), ('u', 3, (0.10131255442407981, 0.120697323349551, 0.11236942837037141)), ('cz', 0, 1), ('u', 0, (-2.3985735164967337, 1.5707963358449135, 3.2914742584669416)), ('u', 1, (3.126049488710869e-09, 1.5707963142799224, 0.14988160470609996)), ('u', 2, (2.075016725821435, 0.16820903197179088, -3.9937206007159616)), ('u', 3, (0.11236942837094595, 0.13825032823993688, 0.10010092994864107))]), (11, [('u', 4, (0.0, 2.0426581640745387, 0.0)), ('u', 3, (0.0, 1.0471975511965976, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.0471975511965976, 3.141592653589793)), ('u', 9, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 10), ('u', 0, (-3.1413300529676143, 0.016049892372225413, 2.6310957151665475)), ('u', 1, (-3.141334271329016, 0.10790196324571806, 2.4951626375093583)), ('u', 2, (0.0006102942423147906, 0.8860770992408441, 0.16956806225969473)), ('u', 3, (5.683986992270052, 2.264332608013758e-08, -0.07393694775744386)), ('u', 4, (-0.008070055280788345, 0.16213446703928294, 0.024201808703938066)), ('cz', 2, 3), ('u', 0, (-3.6477065269341726, 0.05672717868556697, 2.573060872959038)), ('u', 1, (-0.7049831267758607, 0.1441761127167599, -0.5150273787798235)), ('u', 2, (2.9421743581226725, 1.2835370959620422, -3.1714125389099683)), ('u', 3, (0.7482698098280074, 1.5707963297419563, -5.497787113241561)), ('u', 4, (3.2016959709882635, 1.7323649322006993, 0.06794890359063821)), ('cz', 3, 4), ('u', 0, (2.573060872966243, 0.1286366251933175, -3.598854991357485)), ('u', 1, (-0.5150273787795427, 0.02680456184655866, -0.33211634244805344)), ('u', 2, (-0.02981988532131627, 0.3983140742529721, -0.057631019209893844)), ('u', 3, (-2.356194459652187, 1.5707963505605989, 2.5880182950437236)), ('u', 4, (3.20954155718086, 2.124637006983497, 0.06404200235888746)), ('cz', 2, 3), ('u', 0, (2.6843303158564105, 0.13016088390058414, -3.305551108470282)), ('u', 1, (-3.473708996035658, 0.1164455944601965, 2.6704345157926572)), ('u', 2, (3.0839616343808935, 0.5282529217968184, -3.1415926717850096)), ('u', 3, (-0.5535743585462398, 1.5707962645420361, -0.6539032631865938)), ('u', 4, (3.205634655949053, 1.0241536819081383, -0.06510830243098908)), ('cz', 3, 4), ('u', 7, (3.3647163380877396, 0.3642904512849049, -2.562503679351548)), ('u', 8, (1.2543194992414728, 0.9553166198640104, -0.8641577840658705)), ('u', 9, (-1.7406080648054003, 3.7473105113670694e-08, 1.8272609241571824)), ('u', 4, (3.1888333656188483, 2.039993577126575, -0.22148998508761997)), ('u', 10, (3.3151230142032393, 1.9679511724629917, 0.1144338419786659)), ('cz', 8, 9), ('u', 7, (0.21745043555831609, 0.09409940264802959, 0.13406035073996295)), ('u', 8, (-1.5853592654492863, 1.3757533784029135, 0.9868128727387029)), ('u', 9, (2.80833638620301, 1.5707963208811606, -1.9644628938921689)), ('u', 4, (-2.7564764829549357, 0.7732517259761388, 3.082057610635956)), ('u', 10, (2.768991399708501, 1.913703732121055, 0.09404797475257887)), ('cz', 9, 10), ('u', 7, (3.2514508499638404, 0.15907520131247813, -2.802036509050106)), ('u', 8, (3.0192057743818763, 0.7544976667580353, -3.2635667387970653)), ('u', 9, (-2.7479261243718427, 1.5707963650141654, 1.8575087912524408)), ('u', 4, (2.5838285084628705, 0.3380246806716943, -3.5860923186797695)), ('u', 10, (2.645859493075393, 0.7833209533319082, 0.19167664126583084)), ('cz', 8, 9), ('u', 7, (1.2604375623510466, 0.08261304144783904, 0.07101523966968715)), ('u', 8, (0.06258553853238874, 0.21072893933328277, 0.024461585062846788)), ('u', 9, (0.13382186909033422, 1.5707964224488247, -0.009104766423228439)), ('u', 4, (-2.824349607315286, 0.21474414320238785, 3.3898980089165027)), ('u', 10, (3.0807267729553134, 1.9273857231501672, 0.20640520391841077)), ('cz', 9, 10), ('u', 7, (0.015029294259737605, 1.1642565929012174, -0.11099573922631802)), ('u', 8, (-0.1726758776757414, 1.362310977096284, -0.10631142554676284)), ('u', 9, (-3.4268281214637746, 0.6181925943043557, 2.336167933853453)), ('u', 4, (-0.2584175918696197, 0.44008158758243593, -0.1343828929941105)), ('u', 10, (3.1753130569771892, 1.0775117594421955, 0.36690208023432946)), ('cz', 7, 8), ('u', 7, (3.089893283194449, 0.978835646677775, -2.618604998173836)), ('u', 8, (-0.35295295436365076, 0.8860771245648167, -0.16822921599956725)), ('u', 9, (-0.2624481016606052, 0.4680232470868524, -0.041312503674579074)), ('u', 4, (-2.6405434002386166, 0.5367609955174456, 3.354176478395382)), ('u', 10, (-2.6310917071396447, 0.5126584880277414, 3.112469506986639)), ('cz', 7, 8), ('u', 7, (2.954634982581724, 1.5377991035114495, -2.723660439361945)), ('u', 8, (-3.901710077254155, 2.2989524179574996, 2.266041555259831)), ('u', 9, (2.9150388264971214, 1.8541899267816193, -0.5812610008858026)), ('u', 4, (-0.10789481536782598, 0.20473900015661003, -0.012773489700559526)), ('u', 10, (-0.7371237683299199, 0.6994109437975933, 0.3057765563577452)), ('cz', 8, 9), ('u', 7, (0.3724247277613086, 0.25646423985805467, -0.06812546228823857)), ('u', 8, (-3.374768816730933, 1.1071487192481144, 2.5607127233535305)), ('u', 9, (3.305098997922071, 1.5707963236945104, 0.5756102113368464)), ('u', 4, (-0.695764277746694, 0.22665876018000258, 0.27861988718696995)), ('u', 10, (0.013176368110199704, 0.19321223320151706, -0.4893698785033245)), ('cz', 8, 9), ('u', 7, (-2.7239823415475914, 0.49662580170484777, 3.4151917931728795)), ('u', 8, (-2.8912439178455194, 1.570796360232807, 2.9216090443647027)), ('u', 9, (0.7348661229084779, 1.0829423308505814, 0.08651459782972937)), ('u', 4, (2.3831892344803567, 0.44566932138677856, -2.8556751246459138)), ('u', 10, (-2.925326875891609, 0.1837759718820046, 2.986100091900239)), ('cz', 7, 8), ('u', 7, (0.2872491138616414, 6.366639695886986e-09, 0.4495779893801264)), ('u', 8, (-0.5029485103481204, 1.369027357432799, 1.0833010140559345)), ('u', 9, (2.7097760022047455, 0.47638752218908564, -1.0520331566703036)), ('u', 4, (0.6009277643979518, 0.7754535346995289, 0.46341056805523384)), ('u', 10, (0.37447890503462866, 0.14457661874036226, -0.3284944166762539)), ('u', 0, (2.9776341986942363, 0.05353961774145007, -3.0792912619989803)), ('u', 1, (-0.47115813779722227, 0.9050182883328516, -0.026436372384945533)), ('u', 2, (-1.8195526391151517e-08, 0.5282529429708858, -0.17789173461637334)), ('u', 3, (2.487689390403205, 0.7121719611452743, -3.3124530049143206)), ('cz', 1, 2), ('u', 0, (0.06230139158926708, 0.07223886916244122, -0.008243254363708253)), ('u', 1, (3.1151562812051794, 0.6929762432023473, -2.6589581713982926)), ('u', 2, (2.963700918973326, 1.0471975642826303, -3.0700751937505757)), ('u', 3, (2.9707323022655894, 0.8879408364200165, -2.870637794721093)), ('cz', 2, 3), ('u', 0, (0.04322812153185805, 0.12780409715380375, 0.25113226105904746)), ('u', 1, (-2.665059996849431, 0.14745345337493918, 3.1672128137079403)), ('u', 2, (3.181711158327774, 0.5263068880292651, -2.997179730748938)), ('u', 3, (3.400292657495453, 1.0033183184379233, 0.7875653008276615)), ('cz', 1, 2), ('u', 0, (3.6014043972772067, 0.04674257669531842, -2.641906590149284)), ('u', 1, (-2.4506778665896674, 0.5042468211346128, 3.1681203385670695)), ('u', 2, (-0.24104210099365123, 1.5707963786998516, 0.5070956677680156)), ('u', 3, (4.035602919118827, 1.399065735621389, 0.3953281157531867)), ('cz', 2, 3), ('u', 0, (2.0704825035882624, 1.5157985970725332, -1.8956957691017495)), ('u', 1, (-1.5442685443939463, 1.6000518406565911, 1.6993398680714609)), ('u', 2, (0.5070955815609766, 0.005726251377664171, 0.1641478194270985)), ('u', 3, (3.536920536575754, 1.3599659999378026, 0.11335238550196669)), ('cz', 0, 1), ('u', 0, (-1.8334159994714825, 1.6429538596812, 1.4085223086459493)), ('u', 1, (2.4913968774936888, 0.6154796002273588, -1.6446094009796737)), ('u', 2, (2.740093697426668, 1.5763620370737983, -0.6048362170259756)), ('u', 3, (-3.0476852224751756, 0.04168955954147502, 3.186653756608517)), ('cz', 1, 2), ('u', 0, (2.9826382329655985, 0.17909252196273392, -2.9336711441080956)), ('u', 1, (2.5999258626750628, 1.5707963264685767, -2.356194490967865)), ('u', 2, (2.5807354610134063, 0.169860697582233, -3.3256807213984305)), ('u', 3, (0.08347547053706066, 0.04234811948948562, 0.06927926398391966)), ('cz', 0, 1), ('u', 0, (0.20792150948139726, 0.12980755822002127, 0.3564333028740683)), ('u', 1, (-2.356194490967857, 1.5707965008615785, -2.3561944924346694)), ('u', 2, (2.95750458578087, 2.028914775663822, -0.03376405451494735)), ('u', 3, (0.06927926398232259, 0.03274979416540835, 0.10131255442366094)), ('cz', 1, 2), ('u', 0, (0.3564333028739496, 1.4723620300867948, 0.7430191370934366)), ('u', 1, (0.7853981611555738, 0.7853982029947267, 3.1265864341989413e-09)), ('u', 2, (3.1078285990748573, 1.4805012305579615, -1.0665759277685276)), ('u', 3, (0.10131255442407981, 0.120697323349551, 0.11236942837037141)), ('cz', 0, 1), ('u', 0, (-2.3985735164967337, 1.5707963358449135, 3.2914742584669416)), ('u', 1, (3.126049488710869e-09, 1.5707963142799224, 0.14988160470609996)), ('u', 2, (2.075016725821435, 0.16820903197179088, -3.9937206007159616)), ('u', 3, (0.11236942837094595, 0.13825032823993688, 0.10010092994864107)), ('u', 5, (-0.0039034028478323046, 0.2009954616173556, 0.5433774601856451)), ('u', 6, (0.06293478979480127, 0.7853981622804886, -0.00023965737632680034)), ('cz', 6, 7), ('u', 5, (0.2751840277100922, 0.5455127098356944, -0.39490246011352514)), ('u', 6, (3.1370946235900314, 0.8079293275855661, -2.8829253097858887)), ('u', 7, (1.1515397416581001, 1.5707963237366906, 1.1809698504633437)), ('u', 8, (3.747834578533609, 0.8548262628654669, -2.1855551133890923)), ('cz', 7, 8), ('u', 5, (0.48668673146811425, 0.40216557306543516, -0.21445180477973017)), ('u', 6, (-0.406247732961084, 0.022784361696246903, -0.7981818840947825)), ('u', 7, (-2.7517661741407746, 1.5707963180246218, 3.298623867300843)), ('u', 8, (4.728055479431066, 1.2777936317739433, -0.1005011501330264)), ('cz', 6, 7), ('u', 5, (-0.01992530416597569, 0.37930036830615776, -0.11810771628747521)), ('u', 6, (2.16545826174959, 1.3435606178349762, -3.2050974687014886)), ('u', 7, (0.8901663378940949, 1.5707963240037242, 0.4016989676460927)), ('u', 8, (3.6560102474353204, 2.2507379259385147, 0.23337846145166785)), ('cz', 7, 8), ('u', 5, (1.546203603474788, 0.6367923676617261, -1.259131840275079)), ('u', 6, (-1.5826709139626327, 1.2559446998971544, 2.716973490857005)), ('u', 7, (0.009561662337481153, 0.9784096389830532, 0.33137136514128784)), ('u', 8, (-2.614187125737773, 0.7710837794157018, 3.007087536381331)), ('cz', 5, 6), ('u', 5, (-0.10809977448064856, 0.46941007872920354, -2.076947829098388)), ('u', 6, (3.819196039958158, 0.6154797123437021, -1.220402857753761)), ('u', 7, (2.6715838285292337, 1.2935102861188117, 0.5247133188800848)), ('u', 8, (-3.2718973175010837, 0.6873654559671619, 1.9081481783394545)), ('cz', 6, 7), ('u', 5, (3.067346334178069, 1.2873615048690175, -2.3249646014875838)), ('u', 6, (3.406678875558265, 1.5707963172901034, -2.184058206945972)), ('u', 7, (-2.1319063973811123, 0.8012342486613399, 3.6683898242964426)), ('u', 8, (-0.7691749793274529, 0.14761095993996787, -1.0050638494843236)), ('cz', 5, 6), ('u', 5, (-2.3042707736195993, 0.17092288792825708, 2.84575459110712)), ('u', 6, (0.6132618828970325, 1.5707964853079317, -2.232258335783315)), ('u', 7, (3.5347812943312045, 1.792289479885308, -0.5000472790773376)), ('u', 8, (-0.7306852667691774, 0.34271753943463473, -0.8139332741065024)), ('cz', 6, 7), ('u', 5, (0.622505088088614, 1.7328419655259335, -0.7522950000356381)), ('u', 6, (-2.480130667519459, 2.3561944966360993, -0.6280510013562283)), ('u', 7, (2.901086805847627, 1.5765543835679419, -0.515336863135869)), ('u', 8, (-0.8933319548754872, 0.2667016240228794, -0.7186106457810424)), ('cz', 5, 6), ('u', 5, (2.377698900570407, 1.57079634923935, -1.5046612845510263)), ('u', 6, (0.6280509977727411, 1.5707962993535345, -1.5046612820998406)), ('u', 7, (-1.2202838251283077, 0.03507925003870545, 0.23105906917807495)), ('u', 8, (-1.5960764477323917, 0.1500178892607661, -0.13653843164903334)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0))])], (11, 3): [(11, [('u', 4, (0.0, 2.4303874876287264, 0.0)), ('u', 3, (0.0, 0.8026429613782876, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 0.8026429613782876, 3.141592653589793)), ('u', 2, (0.0, 1.183199640139716, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.183199640139716, 3.141592653589793)), ('u', 8, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 9), ('u', 6, (0.01682623351142512, 0.04391712805710915, -0.47763265031377167)), ('u', 7, (-0.08609515546482926, 0.785398163397408, -0.609693948089504)), ('u', 5, (3.2663119417594517, 0.2264768708292619, -2.9788457798992836)), ('u', 8, (-1.1204839222269498, 2.7223960942412306e-11, 3.0632200189647945)), ('u', 9, (2.3100192224675693, 1.4890184220293519, -0.6591343401042042)), ('cz', 7, 8), ('u', 6, (2.759826108660177, 0.008969572383437213, -3.4289665820784894)), ('u', 7, (-3.3142211762161975, 0.5103240291423053, 2.021810115559705)), ('u', 5, (0.04125583432786288, 0.22236326615803606, 0.2598522605512023)), ('u', 8, (-0.0038274956189345843, 1.570796326811965, -1.2712262956802798)), ('u', 9, (-0.3281661146069847, 0.8529202925977987, 0.16806635325247676)), ('cz', 8, 9), ('u', 6, (-0.4944399523883089, 0.05943190214538405, -0.10586677954213958)), ('u', 7, (-0.6339931298978121, 0.5316106735829251, -0.45768142075988366)), ('u', 5, (-3.4364693286101895, 0.07401716947704971, 2.620821342274544)), ('u', 8, (-1.8703663579223766, 0.6154797086702677, 0.15921104145064802)), ('u', 9, (2.748918510940002, 0.07546445493564952, -3.313003968174095)), ('cz', 7, 8), ('u', 6, (0.3689136332099019, 0.10335136665145407, -0.3105690153249634)), ('u', 7, (3.196142270655205, 7.034670549999943e-11, -3.077555519691109)), ('u', 5, (2.6768007666551625, 0.060013446256457235, -2.979393927890725)), ('u', 8, (-0.15921104144285775, 1.5707963267949576, 0.2525372788218364)), ('u', 9, (3.052987493334612, 0.8822468254055224, -0.06353306280132065)), ('cz', 8, 9), ('u', 6, (-2.687501268345167, 0.16299531465056039, 3.0786231057142515)), ('u', 7, (0.1665228714656677, 5.552967984828478e-11, 0.6285091678832561)), ('u', 5, (0.10088600277065606, 0.1444149762132406, 0.14316467669189803)), ('u', 8, (2.889055374760605, 1.570796326800561, -1.9389086011188428)), ('u', 9, (3.2288674649237095, 1.5707963267938034, -1.9389086011188246)), ('u', 10, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 2, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 1, 2), ('u', 2, (1.570796326794898, 0.6847192030022828, -1.570796326794898)), ('u', 1, (1.5707963267948966, 0.6847192030022828, -1.5707963267948966)), ('cz', 1, 2), ('u', 2, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 1, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 2, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 0.8860771237926138, 3.141592653589793)), ('cz', 3, 2), ('u', 2, (0.0, 0.8860771237926138, 0.0)), ('u', 2, (-3.141592653589793, 2.4568734505875103, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 2.4568734505875103, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 0.23182380450040305, 0.0)), ('cz', 4, 3), ('u', 3, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 1.3389725222944935, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (0.0, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.5235987755982989, 0.0)), ('u', 0, (0.0, 0.5235987755982989, 0.0)), ('cz', 0, 1), ('u', 1, (0.0, 1.5707963267948966, 0.0)), ('u', 0, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 0.7853981633974484, 3.141592653589793)), ('cz', 2, 1), ('u', 1, (0.0, 0.7853981633974484, 0.0)), ('u', 1, (-3.141592653589793, 2.356194490192345, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 2.356194490192345, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.3089969389957472, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 0.26179938779914935, 0.0)), ('cz', 3, 2), ('u', 2, (-3.141592653589793, 0.26179938779914935, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 1.3089969389957472, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 1, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 0, 1), ('u', 1, (1.570796326794898, 0.6154797086703868, -1.570796326794898)), ('u', 0, (1.5707963267948966, 0.6154797086703876, -1.5707963267948966)), ('cz', 0, 1), ('u', 1, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 0, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 1, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.2630564724597029, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.30773985433519363, 0.0)), ('cz', 2, 1), ('u', 1, (-3.141592653589793, 0.30773985433519363, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 1.2630564724597029, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (0.0, 1.5707963267948966, 0.0)), ('u', 1, (0.0, 1.5707963267948966, 0.0)), ('cz', 0, 1), ('u', 1, (1.2251903831398114e-16, 0.7853981633974483, -1.1496735851465472e-17)), ('u', 0, (-3.141592653589793, 0.7853981633974483, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 0, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (-3.141592653589793, 1.3605291591529143, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 0.2102671676419825, 0.0)), ('cz', 10, 9), ('u', 9, (-3.141592653589793, 0.2102671676419825, 3.141592653589793)), ('cz', 8, 9), ('u', 5, (0.17237030299209388, 0.07401834352837933, 0.2302146248906372)), ('u', 6, (1.935336656444183, 0.684719202948968, -0.3387998625704308)), ('u', 7, (-2.7093552081192405, 3.1415926535785363, 1.4441670778475313)), ('u', 8, (-2.7871842605895494, 1.1553549616124374, 2.768401448719857)), ('u', 4, (-0.03471795842040962, 0.01596863598018505, 0.06827537298611511)), ('cz', 6, 7), ('u', 5, (3.2887772737216894, 0.09165087812987924, -2.699719827773974)), ('u', 6, (-1.4620280035795141, 0.01456803447928544, 1.3286440031006315)), ('u', 7, (1.7379356624970244, 1.570796326778912, -0.8284162552542631)), ('u', 8, (3.6070309052584166, 0.41701698474443344, -2.5805480678564647)), ('u', 4, (-0.0551668526337184, 0.0646110427144672, -0.4138592726990333)), ('cz', 7, 8), ('u', 5, (-2.314942812618516, 0.06401645002145767, 3.330774995834733)), ('u', 6, (0.48945993140377614, 0.6881634197695576, 0.5439999375645574)), ('u', 7, (-2.3131763983373475, 0.6847192033413388, 3.2749526036898438)), ('u', 8, (-0.1754579276830004, 0.5001601212654382, -0.2671188277728572)), ('u', 4, (-0.038727026917304105, 0.04081734356381132, -0.016032872988218275)), ('cz', 6, 7), ('u', 5, (2.925893780078246, 0.10306468666206371, -3.1207144287871262)), ('u', 6, (-3.2460611759431597, 4.024268674994898e-12, 2.6785675471414656)), ('u', 7, (-0.13335995010354848, 1.5707963271293244, 0.4293308180208435)), ('u', 8, (3.0405403514213294, 1.3450982346019227, -0.23579362469932752)), ('u', 4, (-3.0915225431251727, 0.31494157595476197, 3.0286475037838807)), ('cz', 7, 8), ('u', 5, (-0.07663025449978944, 0.20052742762053993, -0.3685215969714419)), ('u', 6, (2.413366281129644, 2.9936008800581902e-12, -2.803934288793963)), ('u', 7, (-0.4293308180138705, 1.570796326805296, -0.39172735871568043)), ('u', 8, (3.1526736058732405, 1.5707963267947518, -0.39172735871568887)), ('u', 4, (-0.042297243248129834, 0.20279167516186225, 0.5911929838618801)), ('u', 9, (0.0, 1.3605291591529143, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 0.23182380450040305, 0.0)), ('cz', 9, 8), ('u', 8, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 1.3389725222944935, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 5, (2.9797771264554083, 0.6156434406595482, -3.2959398113701592)), ('u', 6, (2.8448498849676023, 1.3754954569633045e-08, -0.4969200214405236)), ('u', 7, (0.1955980511772632, 1.570796353594127, 0.39632816360441553)), ('u', 8, (-0.4128154433303356, 0.04336222396820745, 0.04945306390446427)), ('cz', 6, 7), ('u', 5, (0.5444722531663814, 0.6022887945680928, -0.7441000303590403)), ('u', 6, (-2.553027426689879, 0.7853981611490418, 2.886391985764214)), ('u', 7, (-0.3963281081442811, 1.5707963173578163, -1.4854327280264121)), ('u', 8, (2.993996322505356, 1.4670273726766017, -2.3763770257427037)), ('cz', 6, 7), ('u', 5, (-3.4165925979830494, 0.09641745168924948, 2.168431760231581)), ('u', 6, (-2.845136230365746, 0.6097940315112508, 3.235821653297693)), ('u', 7, (1.6857805328816347, 1.5707962980407193, -0.44468407641865715)), ('u', 8, (3.685023438444097, 0.5258275926246556, -2.0882508467925183)), ('cz', 7, 8), ('u', 5, (-3.483901258398948, 0.1643742599045055, 2.590655089151557)), ('u', 6, (-3.4016906458861857, 0.17756565837059696, 2.229453896345487)), ('u', 7, (2.015863624487668, 1.5701325590428137, -2.6727572939540405)), ('u', 8, (3.51579751160589, 1.4375316286351085, -1.4840358035014785)), ('cz', 6, 7), ('u', 5, (2.399121600158745, 0.22659427900071058, -3.731000504586313)), ('u', 6, (-0.9285033085918732, 0.793027166519653, -0.8934312712158057)), ('u', 7, (-2.5632305856376956, 1.5707963554860795, 2.7905544869920247)), ('u', 8, (-4.248459237144931, 0.780669245096862, 1.6978130071268078)), ('cz', 7, 8), ('u', 5, (1.0946487982463764, 1.8572160872253085, -1.9275557490347839)), ('u', 6, (-2.248161266739471, 0.7930271784092863, 1.6282170444553534)), ('u', 7, (-3.4441976211077674, 1.6247054140397486, 1.7348707006872073)), ('u', 8, (3.4683157764030392, 1.4986417355606603, 0.6095378987560638)), ('cz', 5, 6), ('u', 5, (-0.4062327537123306, 0.34658300580505547, 1.9137621305069374)), ('u', 6, (1.5186541827350228, 1.5706317169950659, -1.1181986081048554)), ('u', 7, (1.6010734520979992, 1.8672463565847248, -0.5614363181944844)), ('u', 8, (2.3388414799390675, 0.11302680077733022, -3.295772330890933)), ('cz', 6, 7), ('u', 5, (0.6603366165874346, 0.1764052871096471, -0.24537789188671552)), ('u', 6, (-2.023910412391709, 0.9553167933103568, 3.8890818675348724)), ('u', 7, (3.3565225298322527, 1.5132422525089877, 0.5424267288786044)), ('u', 8, (0.2971716502089997, 0.20571990308672253, -0.7587416897125508)), ('cz', 5, 6), ('u', 5, (2.7921739739801983, 0.5334219276609302, -3.2416744201323247)), ('u', 6, (0.16018552949341297, 7.189166727258694e-08, -0.40828862542444705)), ('u', 7, (-3.0225438349818323, 0.7468043634962377, 2.7131954865947123)), ('u', 8, (-0.3409604292705519, 0.33037942943634163, -0.618799636425474)), ('cz', 6, 7), ('u', 5, (-2.970211031566412, 0.2524324823144152, 2.662795335532771)), ('u', 6, (2.6436933059870085, 2.3561944091054876, -3.0546341769474874)), ('u', 7, (-3.401545118786185, 0.0019783088680004677, 2.708488933954504)), ('u', 8, (-3.2436142328051916, 0.018878900533301986, 2.91635042226827)), ('cz', 5, 6), ('u', 5, (0.030289380631895013, 0.1893069570447758, 0.6854676813968923)), ('u', 6, (-0.0865395958977434, 1.570997935745412, -0.0032573842647527426)), ('u', 7, (3.4133298623201918, 1.5688437052399191, 0.1639217238077122)), ('u', 8, (2.86786646371811, 0.10238616607992429, -2.9969348899167323)), ('cz', 6, 7), ('u', 5, (-0.28173892200947726, 1.396389473333361, 0.29506277662022373)), ('u', 6, (-2.8187411071775434, 3.1415926294800136, -2.548753194039633)), ('u', 7, (3.7378233191755528, 1.2583877634172406e-07, 0.23774376339484604)), ('u', 8, (-3.203799372993624, 0.6603118215327662, 2.5259189133436)), ('cz', 5, 6), ('u', 5, (0.753141439816695, 0.0004648980650555827, -0.8088204828923403)), ('u', 6, (0.37757886113987604, 3.141592651455384, -5.318984070011139)), ('u', 7, (2.748701599603112, 1.0527296969401629e-07, -3.432902645779708)), ('u', 8, (-0.6251775825298469, 0.62435779176418, 0.11893492814933376)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0))]), (11, [('u', 4, (0.0, 2.4303874876287264, 0.0)), ('u', 3, (0.0, 0.8026429613782876, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 0.8026429613782876, 3.141592653589793)), ('u', 2, (0.0, 1.183199640139716, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.183199640139716, 3.141592653589793)), ('u', 8, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 2, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 1, 2), ('u', 2, (1.570796326794898, 0.6847192030022828, -1.570796326794898)), ('u', 1, (1.5707963267948966, 0.6847192030022828, -1.5707963267948966)), ('cz', 1, 2), ('u', 2, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 1, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 2, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 0.8860771237926138, 3.141592653589793)), ('cz', 3, 2), ('u', 2, (0.0, 0.8860771237926138, 0.0)), ('u', 2, (-3.141592653589793, 2.4568734505875103, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 2.4568734505875103, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 0.23182380450040305, 0.0)), ('cz', 4, 3), ('u', 3, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 1.3389725222944935, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (0.0, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.5235987755982989, 0.0)), ('u', 0, (0.0, 0.5235987755982989, 0.0)), ('cz', 0, 1), ('u', 1, (0.0, 1.5707963267948966, 0.0)), ('u', 0, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 0.7853981633974484, 3.141592653589793)), ('cz', 2, 1), ('u', 1, (0.0, 0.7853981633974484, 0.0)), ('u', 1, (-3.141592653589793, 2.356194490192345, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 2.356194490192345, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.3089969389957472, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 0.26179938779914935, 0.0)), ('cz', 3, 2), ('u', 2, (-3.141592653589793, 0.26179938779914935, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 1.3089969389957472, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 1, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 0, 1), ('u', 1, (1.570796326794898, 0.6154797086703868, -1.570796326794898)), ('u', 0, (1.5707963267948966, 0.6154797086703876, -1.5707963267948966)), ('cz', 0, 1), ('u', 1, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 0, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 1, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.2630564724597029, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.30773985433519363, 0.0)), ('cz', 2, 1), ('u', 1, (-3.141592653589793, 0.30773985433519363, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 1.2630564724597029, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (0.0, 1.5707963267948966, 0.0)), ('u', 1, (0.0, 1.5707963267948966, 0.0)), ('cz', 0, 1), ('u', 1, (1.2251903831398114e-16, 0.7853981633974483, -1.1496735851465472e-17)), ('u', 0, (-3.141592653589793, 0.7853981633974483, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 0, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 7, (0.0, 1.5707963267948966, 0.0)), ('u', 8, (0.0, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 8, (1.2251903831398114e-16, 0.7853981633974483, -1.1496735851465472e-17)), ('u', 7, (-3.141592653589793, 0.7853981633974483, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 7, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (-3.141592653589793, 0.9553166181245093, 3.141592653589793)), ('cz', 9, 8), ('u', 8, (0.0, 0.9553166181245093, 0.0)), ('u', 8, (-3.141592653589793, 2.5261129449194057, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 2.5261129449194057, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (-3.141592653589793, 1.3605291591529143, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 0.2102671676419825, 0.0)), ('cz', 10, 9), ('u', 9, (-3.141592653589793, 0.2102671676419825, 3.141592653589793)), ('cz', 8, 9), ('u', 5, (0.17237030299209388, 0.07401834352837933, 0.2302146248906372)), ('u', 6, (1.935336656444183, 0.684719202948968, -0.3387998625704308)), ('u', 7, (-2.7093552081192405, 3.1415926535785363, 1.4441670778475313)), ('u', 8, (-2.7871842605895494, 1.1553549616124374, 2.768401448719857)), ('u', 4, (-0.03471795842040962, 0.01596863598018505, 0.06827537298611511)), ('cz', 6, 7), ('u', 5, (3.2887772737216894, 0.09165087812987924, -2.699719827773974)), ('u', 6, (-1.4620280035795141, 0.01456803447928544, 1.3286440031006315)), ('u', 7, (1.7379356624970244, 1.570796326778912, -0.8284162552542631)), ('u', 8, (3.6070309052584166, 0.41701698474443344, -2.5805480678564647)), ('u', 4, (-0.0551668526337184, 0.0646110427144672, -0.4138592726990333)), ('cz', 7, 8), ('u', 5, (-2.314942812618516, 0.06401645002145767, 3.330774995834733)), ('u', 6, (0.48945993140377614, 0.6881634197695576, 0.5439999375645574)), ('u', 7, (-2.3131763983373475, 0.6847192033413388, 3.2749526036898438)), ('u', 8, (-0.1754579276830004, 0.5001601212654382, -0.2671188277728572)), ('u', 4, (-0.038727026917304105, 0.04081734356381132, -0.016032872988218275)), ('cz', 6, 7), ('u', 5, (2.925893780078246, 0.10306468666206371, -3.1207144287871262)), ('u', 6, (-3.2460611759431597, 4.024268674994898e-12, 2.6785675471414656)), ('u', 7, (-0.13335995010354848, 1.5707963271293244, 0.4293308180208435)), ('u', 8, (3.0405403514213294, 1.3450982346019227, -0.23579362469932752)), ('u', 4, (-3.0915225431251727, 0.31494157595476197, 3.0286475037838807)), ('cz', 7, 8), ('u', 5, (-0.07663025449978944, 0.20052742762053993, -0.3685215969714419)), ('u', 6, (2.413366281129644, 2.9936008800581902e-12, -2.803934288793963)), ('u', 7, (-0.4293308180138705, 1.570796326805296, -0.39172735871568043)), ('u', 8, (3.1526736058732405, 1.5707963267947518, -0.39172735871568887)), ('u', 4, (-0.042297243248129834, 0.20279167516186225, 0.5911929838618801)), ('u', 9, (0.0, 1.3605291591529143, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 0.23182380450040305, 0.0)), ('cz', 9, 8), ('u', 8, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 1.3389725222944935, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 5, (2.9797771264554083, 0.6156434406595482, -3.2959398113701592)), ('u', 6, (2.8448498849676023, 1.3754954569633045e-08, -0.4969200214405236)), ('u', 7, (0.1955980511772632, 1.570796353594127, 0.39632816360441553)), ('u', 8, (-0.4128154433303356, 0.04336222396820745, 0.04945306390446427)), ('cz', 6, 7), ('u', 5, (0.5444722531663814, 0.6022887945680928, -0.7441000303590403)), ('u', 6, (-2.553027426689879, 0.7853981611490418, 2.886391985764214)), ('u', 7, (-0.3963281081442811, 1.5707963173578163, -1.4854327280264121)), ('u', 8, (2.993996322505356, 1.4670273726766017, -2.3763770257427037)), ('cz', 6, 7), ('u', 5, (-3.4165925979830494, 0.09641745168924948, 2.168431760231581)), ('u', 6, (-2.845136230365746, 0.6097940315112508, 3.235821653297693)), ('u', 7, (1.6857805328816347, 1.5707962980407193, -0.44468407641865715)), ('u', 8, (3.685023438444097, 0.5258275926246556, -2.0882508467925183)), ('cz', 7, 8), ('u', 5, (-3.483901258398948, 0.1643742599045055, 2.590655089151557)), ('u', 6, (-3.4016906458861857, 0.17756565837059696, 2.229453896345487)), ('u', 7, (2.015863624487668, 1.5701325590428137, -2.6727572939540405)), ('u', 8, (3.51579751160589, 1.4375316286351085, -1.4840358035014785)), ('cz', 6, 7), ('u', 5, (2.399121600158745, 0.22659427900071058, -3.731000504586313)), ('u', 6, (-0.9285033085918732, 0.793027166519653, -0.8934312712158057)), ('u', 7, (-2.5632305856376956, 1.5707963554860795, 2.7905544869920247)), ('u', 8, (-4.248459237144931, 0.780669245096862, 1.6978130071268078)), ('cz', 7, 8), ('u', 5, (1.0946487982463764, 1.8572160872253085, -1.9275557490347839)), ('u', 6, (-2.248161266739471, 0.7930271784092863, 1.6282170444553534)), ('u', 7, (-3.4441976211077674, 1.6247054140397486, 1.7348707006872073)), ('u', 8, (3.4683157764030392, 1.4986417355606603, 0.6095378987560638)), ('cz', 5, 6), ('u', 5, (-0.4062327537123306, 0.34658300580505547, 1.9137621305069374)), ('u', 6, (1.5186541827350228, 1.5706317169950659, -1.1181986081048554)), ('u', 7, (1.6010734520979992, 1.8672463565847248, -0.5614363181944844)), ('u', 8, (2.3388414799390675, 0.11302680077733022, -3.295772330890933)), ('cz', 6, 7), ('u', 5, (0.6603366165874346, 0.1764052871096471, -0.24537789188671552)), ('u', 6, (-2.023910412391709, 0.9553167933103568, 3.8890818675348724)), ('u', 7, (3.3565225298322527, 1.5132422525089877, 0.5424267288786044)), ('u', 8, (0.2971716502089997, 0.20571990308672253, -0.7587416897125508)), ('cz', 5, 6), ('u', 5, (2.7921739739801983, 0.5334219276609302, -3.2416744201323247)), ('u', 6, (0.16018552949341297, 7.189166727258694e-08, -0.40828862542444705)), ('u', 7, (-3.0225438349818323, 0.7468043634962377, 2.7131954865947123)), ('u', 8, (-0.3409604292705519, 0.33037942943634163, -0.618799636425474)), ('cz', 6, 7), ('u', 5, (-2.970211031566412, 0.2524324823144152, 2.662795335532771)), ('u', 6, (2.6436933059870085, 2.3561944091054876, -3.0546341769474874)), ('u', 7, (-3.401545118786185, 0.0019783088680004677, 2.708488933954504)), ('u', 8, (-3.2436142328051916, 0.018878900533301986, 2.91635042226827)), ('cz', 5, 6), ('u', 5, (0.030289380631895013, 0.1893069570447758, 0.6854676813968923)), ('u', 6, (-0.0865395958977434, 1.570997935745412, -0.0032573842647527426)), ('u', 7, (3.4133298623201918, 1.5688437052399191, 0.1639217238077122)), ('u', 8, (2.86786646371811, 0.10238616607992429, -2.9969348899167323)), ('cz', 6, 7), ('u', 5, (-0.28173892200947726, 1.396389473333361, 0.29506277662022373)), ('u', 6, (-2.8187411071775434, 3.1415926294800136, -2.548753194039633)), ('u', 7, (3.7378233191755528, 1.2583877634172406e-07, 0.23774376339484604)), ('u', 8, (-3.203799372993624, 0.6603118215327662, 2.5259189133436)), ('cz', 5, 6), ('u', 5, (0.753141439816695, 0.0004648980650555827, -0.8088204828923403)), ('u', 6, (0.37757886113987604, 3.141592651455384, -5.318984070011139)), ('u', 7, (2.748701599603112, 1.0527296969401629e-07, -3.432902645779708)), ('u', 8, (-0.6251775825298469, 0.62435779176418, 0.11893492814933376)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0))])], (12, 2): [(12, [('u', 5, (0.0, 2.1477276720313534, 0.0)), ('u', 4, (0.0, 0.9975930176940415, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (-3.141592653589793, 0.9975930176940415, 3.141592653589793)), ('u', 10, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 11, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 11), ('u', 9, (1.8245794479006703, 0.9553166181386613, -2.078558746612874)), ('u', 10, (0.9863720226311673, 1.6344049434792755e-08, -2.045264618917613)), ('u', 11, (2.9455895851782827, 1.523114476083327, 0.11801642924932976)), ('cz', 9, 10), ('u', 9, (-1.3746684868833494, 1.1436540920547587, 1.7551245585270916)), ('u', 10, (1.8436326022391494, 1.5707963181909397, -0.8425121944931349)), ('u', 11, (3.803880599527135, 1.3987611036756726, -0.3326290910917571)), ('cz', 10, 11), ('u', 9, (0.49825045105010424, 0.32851450845057334, 0.31094012256553905)), ('u', 10, (-2.2990804579072113, 0.4205343352839644, 3.0975567240251745)), ('u', 11, (3.2995893389209634, 2.333529923445325, 0.16848681265493504)), ('cz', 9, 10), ('u', 9, (-2.8354698589175946, 0.2250564019062336, 2.9737888515919195)), ('u', 10, (0.044035929386599566, 1.5707963267949108, -0.15804863241042671)), ('u', 11, (2.6998492685651336, 1.210944353686906, 0.45799205771299567)), ('cz', 10, 11), ('u', 4, (-0.08739429996964554, 3.770566875693945e-07, 1.3818351836367677)), ('u', 3, (1.0621870409342855, 2.1862760967408863, -2.4376487379766525)), ('u', 5, (-4.3246283296972825, 0.14098712523764986, 1.676463844839981)), ('u', 2, (-2.4380607461559687, 0.5636736635997478, 2.7734838210202137)), ('u', 11, (2.6559887233975585, 1.7477124136035582, -0.7708095823481558)), ('cz', 4, 3), ('u', 4, (2.26633897882659, 1.5707961365819603, -0.25770469446822064)), ('u', 3, (-1.1950609808247181, 2.311806112307064, 1.8333393649016978)), ('u', 5, (3.0670401109686716, 1.5663810982760182, -0.5721613654727093)), ('u', 2, (-2.9273457536053873, 0.411075073705774, 3.0858523625226666)), ('u', 11, (2.760981216232155, 0.14389170244801167, -3.3377044494980668)), ('cz', 4, 5), ('u', 4, (0.2577064147860374, 0.4205343223099434, -0.18437629627354923)), ('u', 3, (-0.10782563732126993, 0.40033203181216903, 0.9885545393373191)), ('u', 5, (3.3516170674882653, 2.371821135610537, -0.3802753629698179)), ('u', 2, (2.56617846257456, 0.405295153743926, -2.8967223246246627)), ('u', 11, (0.3134044958102739, 0.043246487680743705, -0.07325222053425023)), ('cz', 4, 3), ('u', 4, (0.18437469045907176, 1.570796329963336, -0.2169783929862323)), ('u', 3, (-2.96483418213719, 2.6039295658524844, 2.6439595160455007)), ('u', 5, (2.533973075357986, 0.5552406473182457, -0.2960420316968957)), ('u', 2, (3.1537870536913273, 0.1660600088389333, -2.004637798517467)), ('u', 11, (-0.16136840290499485, 0.041861796153663616, -0.1170153926304364)), ('cz', 4, 5), ('u', 4, (-2.9305534995805145, 1.4711092301861604, 3.177919666207508)), ('u', 3, (-2.6439602752942646, 0.5376628012542504, 3.327845942424548)), ('u', 5, (4.384000773717635, 1.5222872263968898, 0.2746123560904179)), ('u', 2, (0.09353931177752386, 1.2498212153447752, 0.4271012680066425)), ('u', 11, (0.3149696919722872, 0.2845096145004271, -0.11010321114970048)), ('cz', 3, 2), ('u', 4, (-3.856685375573164, 1.6447626951259098, 2.115741831376012)), ('u', 3, (0.5520959145828285, 1.5707961071903869, -0.16501310741108152)), ('u', 5, (-0.13382052595585606, 0.0347649644431061, 0.5714917704788464)), ('u', 2, (-2.7066939792859572, 0.19183946124963602, 3.3781509277886834)), ('u', 11, (-3.0408593233294945, 0.0344634786947446, 3.107614005494057)), ('cz', 4, 3), ('u', 4, (-3.5871727288380653, 0.07905998896083763, 1.990599784404098)), ('u', 3, (3.306604679423938, 0.46364761170446767, -2.3598547698218604)), ('u', 5, (-3.054888432941452, 0.14568143705086917, 3.167028322747415)), ('u', 2, (0.4824452915261087, 1.024666237011512, 0.9137195235514831)), ('u', 11, (3.2100250435183404, 0.28005067231072994, -2.852112271213469)), ('cz', 3, 2), ('u', 4, (2.594964059684078, 2.0460065481502467, 0.08801715269467336)), ('u', 3, (-0.7817365961172004, 1.5707963231933404, -0.46366150072383194)), ('u', 5, (-2.9996365986486477, 0.13630065438001115, 2.3755769890699785)), ('u', 2, (1.3123230981334415, 0.4487410260874321, 0.027619848122773738)), ('u', 11, (2.770160523991431, 0.02912147709530635, -3.162886833927296)), ('cz', 4, 3), ('u', 8, (0.1786622367446048, 0.8860771222146757, 1.3451962488787408)), ('u', 9, (-2.9737888984643126, 2.9165361441400752, 1.9739559036381784)), ('u', 10, (2.9627536090401354, 2.89174810058659, -2.755130633467782)), ('u', 5, (-0.8877690983742033, 0.3252592857367499, 0.45329638350825513)), ('u', 4, (2.844445258565836, 1.2780251676903092, 0.7442806883824118)), ('cz', 8, 9), ('u', 8, (-1.395897301498704, 0.9314327105617929, 2.598247965208892)), ('u', 9, (0.5591567620381078, 1.5707962662540775, -0.503208557822333)), ('u', 10, (3.268161193085032, 0.3524166889750484, -2.3954528358410996)), ('u', 5, (-0.002164040305630388, 0.22595343574918667, -0.6804704359496352)), ('u', 4, (0.4294497428002402, 0.40548447873812804, 0.258033558063514)), ('cz', 9, 10), ('u', 8, (-0.16382478072080364, 0.060460046855557094, 1.2453610660517418)), ('u', 9, (3.6448011135187794, 0.463647601653386, -2.624910709686496)), ('u', 10, (-0.007123893697464939, 0.46249248653943115, -0.47413767632059745)), ('u', 5, (-3.034468608478265, 0.021566387630053908, 3.215811786292058)), ('u', 4, (-2.791004748282556, 0.506470833390148, 3.4722093367749167)), ('cz', 8, 9), ('u', 8, (-0.2493487920660744, 0.2526746490968153, 1.3950114574430015)), ('u', 9, (-0.5166818344622964, 1.5707963216957628, 0.09969628797425945)), ('u', 10, (2.2673198865151702, 1.1605038264075578, 0.19761438594261715)), ('u', 5, (0.5395359132196164, 0.027363874547082872, -0.8034265162234285)), ('u', 4, (-0.1908095116195652, 0.32573474862362917, -0.4047714522421887)), ('cz', 9, 10), ('u', 6, (3.0729180895231414, 0.7703963202047999, -2.2747767904551246)), ('u', 7, (0.34971677933587875, 0.7853982169362412, 0.7485904919305114)), ('u', 8, (1.7465814153484196, 0.25267454118412014, -0.48832924042987946)), ('u', 9, (0.1448453907995484, 2.7545933198760517, -1.527415730283212)), ('cz', 7, 8), ('u', 6, (2.617281110176182, 0.7501352364135964, -3.1460940814608653)), ('u', 7, (2.564270165730943, 0.7961807635948261, -2.3837299461290713)), ('u', 8, (0.5124872597205522, 1.5707963201843353, -1.0298517273110335)), ('u', 9, (2.259639618000099, 0.41547225323858794, -3.146696884144622)), ('cz', 8, 9), ('u', 6, (-0.09825286696006773, 0.06219137011559926, -0.14171324472108007)), ('u', 7, (0.6645203309344397, 0.12215669317552058, 0.7558540216494837)), ('u', 8, (2.600647988037975, 1.570796227483977, -2.7554947634553337)), ('u', 9, (3.600803299235137, 1.339934505663914, -0.8655710653645785)), ('cz', 7, 8), ('u', 6, (2.64263795754161, 0.04065684258431491, -2.7612086353323013)), ('u', 7, (0.8497756294053884, 0.531120836778257, -0.4122229634451662)), ('u', 8, (1.708297214135094, 1.5707962349102484, -0.282195154597755)), ('u', 9, (1.942549154256038, 2.2733860326815, 0.7630406963540899)), ('cz', 8, 9), ('u', 6, (1.759785103902497, 1.0064845027987224, -1.1816697873674566)), ('u', 7, (-0.8797522368559194, 1.4105110198932376, -1.4461563484466164)), ('u', 8, (-3.0592136509462255, 0.8877058781777308, 2.258433876440418)), ('u', 9, (-0.06372347609670492, 0.03234210700908735, 0.5767655238724242)), ('cz', 6, 7), ('u', 6, (-1.7784686250150483, 0.995334565742242, 2.125412655029514)), ('u', 7, (0.38290946498143064, 0.6154797248040618, -0.04142724227838704)), ('u', 8, (2.1464928718945244, 2.9714730995592804, -0.5773794218832196)), ('u', 9, (0.7404195733000527, 0.15245812925698993, -0.030309092493204293)), ('cz', 7, 8), ('u', 6, (-1.5926201578718944, 0.6464044895298017, 3.9143971790543137)), ('u', 7, (-4.055481895845448, 1.5707963012750934, 0.7235817964538493)), ('u', 8, (3.19226817816393, 1.6864254912856216, -0.2484527860768222)), ('u', 9, (-3.178613834712625, 0.8381937918220419, 2.9027230039692893)), ('cz', 6, 7), ('u', 6, (1.0460066413145173, 0.9457956460520283, -1.5817010304445587)), ('u', 7, (0.8472145387414913, 1.5707963060851882, 0.2879796070346632)), ('u', 8, (2.4262379090631727, 1.7169827963883288, -0.33164076544850474)), ('u', 9, (3.121281589705624, 0.45649343857425556, -2.7266574319023325)), ('cz', 7, 8), ('u', 6, (0.06822781043509762, 1.4915551474282889, -0.6901140215187032)), ('u', 7, (1.2828163028297934, 0.7853981856526834, 0.6563413163792482)), ('u', 8, (2.5996962065134834, 2.334823944330995, 0.29192459848258223)), ('u', 9, (-2.7373331055993857, 0.23807778334502255, 3.5035415103749292)), ('cz', 6, 7), ('u', 6, (3.2044326654343482, 1.57079634528553, -2.196236159053842)), ('u', 7, (-0.6563412980824703, 1.5707963435604955, 0.9453564892681035)), ('u', 8, (-2.515013869271413, 1.0060900864202655, 3.475794876085521)), ('u', 9, (1.0299438387533502, 0.5981626073937075, -0.13746226822960883)), ('u', 10, (2.4358303682382068, 1.5707963381883066, -0.9652234444357511)), ('u', 5, (-2.755378607261519, 0.3547137923473315, 3.274915054801085)), ('u', 4, (-0.7667957453207299, 0.17652314735968777, -0.42058196591711083)), ('u', 1, (0.7412288640492811, 2.356194492522506, 0.005325255942831286)), ('u', 0, (-0.23611034651145957, 0.05460943144052282, 0.062131626198132325)), ('u', 2, (3.1139726462766344, 0.44874089346963825, -0.8242875459363077)), ('u', 3, (3.891759158580755, 2.343917301619611, -1.5606800315847211)), ('cz', 1, 2), ('u', 1, (3.4529824667772404, 2.4663873874313933, -2.5314292644599767)), ('u', 0, (0.6108058655348234, 0.45495423125868817, -0.4384011630597971)), ('u', 2, (2.3312756670857295, 1.5707965508270174, -3.8276684175089004)), ('u', 3, (2.1718466930158047, 0.7680512702229998, -0.19617745603489445)), ('cz', 2, 3), ('u', 1, (0.6021284812466575, 0.23740178356767752, -0.18588911772280456)), ('u', 0, (1.6871011769849407, 0.10238334285005367, -2.4880006549617857)), ('u', 2, (2.2568719533639485, 1.5707960613977707, -3.7084629760377505)), ('u', 3, (2.7453737460722, 2.3441106054491216, -0.712014026153942)), ('cz', 1, 2), ('u', 1, (-1.0210886526316207, 0.9696918640684943, 0.845745649536796)), ('u', 0, (-0.2838464731583833, 0.488933341693604, -0.5683712329295803)), ('u', 2, (2.661265436376154, 1.5707961647420012, 3.2022954507084993)), ('u', 3, (3.4675959597720576, 1.3574288237948446, 0.8205496086856936)), ('cz', 2, 3), ('u', 1, (-1.269224929178692, 0.6449112768296513, 1.0388610852574673)), ('u', 0, (0.3988417994000745, 1.2115451619846414, -0.5997089009882102)), ('u', 2, (1.3771898487456442, 0.42345056929911445, -3.5465983846204674)), ('u', 3, (4.154106670624198, 1.1867790854940272, 0.3228407698256428)), ('cz', 1, 0), ('u', 1, (2.257687245505903, 0.6154797072094761, -1.525446724867419)), ('u', 0, (1.4612193675508625, 0.6406953460642455, -1.1112554615939598)), ('u', 2, (-0.3904512366070922, 0.2696760942085726, 0.9147661958179247)), ('u', 3, (-3.1136085898339463, 0.8393308151959621, 2.94584554244833)), ('cz', 1, 0), ('u', 1, (2.426405592111868, 1.19420166762454, -0.7011719601240354)), ('u', 0, (-0.5253316201079938, 1.529100526555119, 1.8135189284230102)), ('u', 2, (-0.04810161635688748, 1.8227188996772763, -3.9270175412922788)), ('u', 3, (2.4978889690238044, 0.8746941754436426, -2.410170961632853)), ('cz', 1, 2), ('u', 1, (-3.7277988119148175, 0.9553166179972424, 2.116222008320426)), ('u', 0, (-2.285343353295458, 0.2247132452497797, 3.4381262481984995)), ('u', 2, (2.1987949279867105, 1.5707964448504583, -3.450709743862259)), ('u', 3, (0.4744718836591277, 0.19902801420265745, 0.13114850552618545)), ('cz', 1, 2), ('u', 1, (-1.3173268598098806, 1.5707965354512161, 0.6861715732073628)), ('u', 0, (0.40112964916438487, 0.6693734434533707, -0.908977999871106)), ('u', 2, (3.3774735267636906, 1.7155129942662726, -0.7314425530107427)), ('u', 3, (-0.0028962095988690102, 0.10099194513739455, -0.03437243163684296)), ('cz', 1, 0), ('u', 1, (0.8846245871808652, 0.7853981667224689, 0.691414405068067)), ('u', 0, (-3.2651085505340642, 1.570796154248359, 2.989384697769773)), ('u', 2, (-1.7031106831616103, 0.2774301045937453, 3.8480021541327947)), ('u', 3, (0.334526450943965, 0.5264093201347806, 0.40273496129147873)), ('cz', 1, 0), ('u', 1, (2.450178246254839, 1.5707963275409877, -2.5337739413160785)), ('u', 0, (1.7230044409829273, 1.5707963116195378, -2.533773942969272)), ('u', 2, (-0.09102238709159846, 0.26116851983517, 0.09983258276432336)), ('u', 3, (0.5609503820762544, 0.7728032743817536, -0.5935934538716358)), ('u', 11, (-2.871320587684953, 0.18270763755063463, 3.2126447403495293))]), (12, [('u', 5, (0.0, 2.1477276720313534, 0.0)), ('u', 4, (0.0, 0.9975930176940415, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (-3.141592653589793, 0.9975930176940415, 3.141592653589793)), ('u', 10, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 11, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 11), ('u', 9, (1.8245794479006703, 0.9553166181386613, -2.078558746612874)), ('u', 10, (0.9863720226311673, 1.6344049434792755e-08, -2.045264618917613)), ('u', 11, (2.9455895851782827, 1.523114476083327, 0.11801642924932976)), ('cz', 9, 10), ('u', 9, (-1.3746684868833494, 1.1436540920547587, 1.7551245585270916)), ('u', 10, (1.8436326022391494, 1.5707963181909397, -0.8425121944931349)), ('u', 11, (3.803880599527135, 1.3987611036756726, -0.3326290910917571)), ('cz', 10, 11), ('u', 9, (0.49825045105010424, 0.32851450845057334, 0.31094012256553905)), ('u', 10, (-2.2990804579072113, 0.4205343352839644, 3.0975567240251745)), ('u', 11, (3.2995893389209634, 2.333529923445325, 0.16848681265493504)), ('cz', 9, 10), ('u', 9, (-2.8354698589175946, 0.2250564019062336, 2.9737888515919195)), ('u', 10, (0.044035929386599566, 1.5707963267949108, -0.15804863241042671)), ('u', 11, (2.6998492685651336, 1.210944353686906, 0.45799205771299567)), ('cz', 10, 11), ('u', 4, (-0.08739429996964554, 3.770566875693945e-07, 1.3818351836367677)), ('u', 3, (1.0621870409342855, 2.1862760967408863, -2.4376487379766525)), ('u', 5, (-4.3246283296972825, 0.14098712523764986, 1.676463844839981)), ('u', 2, (-2.4380607461559687, 0.5636736635997478, 2.7734838210202137)), ('u', 11, (2.6559887233975585, 1.7477124136035582, -0.7708095823481558)), ('cz', 4, 3), ('u', 4, (2.26633897882659, 1.5707961365819603, -0.25770469446822064)), ('u', 3, (-1.1950609808247181, 2.311806112307064, 1.8333393649016978)), ('u', 5, (3.0670401109686716, 1.5663810982760182, -0.5721613654727093)), ('u', 2, (-2.9273457536053873, 0.411075073705774, 3.0858523625226666)), ('u', 11, (2.760981216232155, 0.14389170244801167, -3.3377044494980668)), ('cz', 4, 5), ('u', 4, (0.2577064147860374, 0.4205343223099434, -0.18437629627354923)), ('u', 3, (-0.10782563732126993, 0.40033203181216903, 0.9885545393373191)), ('u', 5, (3.3516170674882653, 2.371821135610537, -0.3802753629698179)), ('u', 2, (2.56617846257456, 0.405295153743926, -2.8967223246246627)), ('u', 11, (0.3134044958102739, 0.043246487680743705, -0.07325222053425023)), ('cz', 4, 3), ('u', 4, (0.18437469045907176, 1.570796329963336, -0.2169783929862323)), ('u', 3, (-2.96483418213719, 2.6039295658524844, 2.6439595160455007)), ('u', 5, (2.533973075357986, 0.5552406473182457, -0.2960420316968957)), ('u', 2, (3.1537870536913273, 0.1660600088389333, -2.004637798517467)), ('u', 11, (-0.16136840290499485, 0.041861796153663616, -0.1170153926304364)), ('cz', 4, 5), ('u', 4, (-2.9305534995805145, 1.4711092301861604, 3.177919666207508)), ('u', 3, (-2.6439602752942646, 0.5376628012542504, 3.327845942424548)), ('u', 5, (4.384000773717635, 1.5222872263968898, 0.2746123560904179)), ('u', 2, (0.09353931177752386, 1.2498212153447752, 0.4271012680066425)), ('u', 11, (0.3149696919722872, 0.2845096145004271, -0.11010321114970048)), ('cz', 3, 2), ('u', 4, (-3.856685375573164, 1.6447626951259098, 2.115741831376012)), ('u', 3, (0.5520959145828285, 1.5707961071903869, -0.16501310741108152)), ('u', 5, (-0.13382052595585606, 0.0347649644431061, 0.5714917704788464)), ('u', 2, (-2.7066939792859572, 0.19183946124963602, 3.3781509277886834)), ('u', 11, (-3.0408593233294945, 0.0344634786947446, 3.107614005494057)), ('cz', 4, 3), ('u', 4, (-3.5871727288380653, 0.07905998896083763, 1.990599784404098)), ('u', 3, (3.306604679423938, 0.46364761170446767, -2.3598547698218604)), ('u', 5, (-3.054888432941452, 0.14568143705086917, 3.167028322747415)), ('u', 2, (0.4824452915261087, 1.024666237011512, 0.9137195235514831)), ('u', 11, (3.2100250435183404, 0.28005067231072994, -2.852112271213469)), ('cz', 3, 2), ('u', 4, (2.594964059684078, 2.0460065481502467, 0.08801715269467336)), ('u', 3, (-0.7817365961172004, 1.5707963231933404, -0.46366150072383194)), ('u', 5, (-2.9996365986486477, 0.13630065438001115, 2.3755769890699785)), ('u', 2, (1.3123230981334415, 0.4487410260874321, 0.027619848122773738)), ('u', 11, (2.770160523991431, 0.02912147709530635, -3.162886833927296)), ('cz', 4, 3), ('u', 8, (0.1786622367446048, 0.8860771222146757, 1.3451962488787408)), ('u', 9, (-2.9737888984643126, 2.9165361441400752, 1.9739559036381784)), ('u', 10, (2.9627536090401354, 2.89174810058659, -2.755130633467782)), ('u', 5, (-0.8877690983742033, 0.3252592857367499, 0.45329638350825513)), ('u', 4, (2.844445258565836, 1.2780251676903092, 0.7442806883824118)), ('cz', 8, 9), ('u', 8, (-1.395897301498704, 0.9314327105617929, 2.598247965208892)), ('u', 9, (0.5591567620381078, 1.5707962662540775, -0.503208557822333)), ('u', 10, (3.268161193085032, 0.3524166889750484, -2.3954528358410996)), ('u', 5, (-0.002164040305630388, 0.22595343574918667, -0.6804704359496352)), ('u', 4, (0.4294497428002402, 0.40548447873812804, 0.258033558063514)), ('cz', 9, 10), ('u', 8, (-0.16382478072080364, 0.060460046855557094, 1.2453610660517418)), ('u', 9, (3.6448011135187794, 0.463647601653386, -2.624910709686496)), ('u', 10, (-0.007123893697464939, 0.46249248653943115, -0.47413767632059745)), ('u', 5, (-3.034468608478265, 0.021566387630053908, 3.215811786292058)), ('u', 4, (-2.791004748282556, 0.506470833390148, 3.4722093367749167)), ('cz', 8, 9), ('u', 8, (-0.2493487920660744, 0.2526746490968153, 1.3950114574430015)), ('u', 9, (-0.5166818344622964, 1.5707963216957628, 0.09969628797425945)), ('u', 10, (2.2673198865151702, 1.1605038264075578, 0.19761438594261715)), ('u', 5, (0.5395359132196164, 0.027363874547082872, -0.8034265162234285)), ('u', 4, (-0.1908095116195652, 0.32573474862362917, -0.4047714522421887)), ('cz', 9, 10), ('u', 8, (0.5134801990157082, 1.6561312916366313, -0.2381189128655575)), ('u', 9, (-0.09969620677482222, 1.5707962854720336, -0.9652234169654113)), ('u', 10, (2.4358303682382068, 1.5707963381883066, -0.9652234444357511)), ('u', 5, (-2.755378607261519, 0.3547137923473315, 3.274915054801085)), ('u', 4, (-0.7667957453207299, 0.17652314735968777, -0.42058196591711083)), ('u', 1, (0.7412288640492811, 2.356194492522506, 0.005325255942831286)), ('u', 0, (-0.23611034651145957, 0.05460943144052282, 0.062131626198132325)), ('u', 2, (3.1139726462766344, 0.44874089346963825, -0.8242875459363077)), ('u', 3, (3.891759158580755, 2.343917301619611, -1.5606800315847211)), ('cz', 1, 2), ('u', 1, (3.4529824667772404, 2.4663873874313933, -2.5314292644599767)), ('u', 0, (0.6108058655348234, 0.45495423125868817, -0.4384011630597971)), ('u', 2, (2.3312756670857295, 1.5707965508270174, -3.8276684175089004)), ('u', 3, (2.1718466930158047, 0.7680512702229998, -0.19617745603489445)), ('cz', 2, 3), ('u', 1, (0.6021284812466575, 0.23740178356767752, -0.18588911772280456)), ('u', 0, (1.6871011769849407, 0.10238334285005367, -2.4880006549617857)), ('u', 2, (2.2568719533639485, 1.5707960613977707, -3.7084629760377505)), ('u', 3, (2.7453737460722, 2.3441106054491216, -0.712014026153942)), ('cz', 1, 2), ('u', 1, (-1.0210886526316207, 0.9696918640684943, 0.845745649536796)), ('u', 0, (-0.2838464731583833, 0.488933341693604, -0.5683712329295803)), ('u', 2, (2.661265436376154, 1.5707961647420012, 3.2022954507084993)), ('u', 3, (3.4675959597720576, 1.3574288237948446, 0.8205496086856936)), ('cz', 2, 3), ('u', 1, (-1.269224929178692, 0.6449112768296513, 1.0388610852574673)), ('u', 0, (0.3988417994000745, 1.2115451619846414, -0.5997089009882102)), ('u', 2, (1.3771898487456442, 0.42345056929911445, -3.5465983846204674)), ('u', 3, (4.154106670624198, 1.1867790854940272, 0.3228407698256428)), ('cz', 1, 0), ('u', 1, (2.257687245505903, 0.6154797072094761, -1.525446724867419)), ('u', 0, (1.4612193675508625, 0.6406953460642455, -1.1112554615939598)), ('u', 2, (-0.3904512366070922, 0.2696760942085726, 0.9147661958179247)), ('u', 3, (-3.1136085898339463, 0.8393308151959621, 2.94584554244833)), ('cz', 1, 0), ('u', 1, (2.426405592111868, 1.19420166762454, -0.7011719601240354)), ('u', 0, (-0.5253316201079938, 1.529100526555119, 1.8135189284230102)), ('u', 2, (-0.04810161635688748, 1.8227188996772763, -3.9270175412922788)), ('u', 3, (2.4978889690238044, 0.8746941754436426, -2.410170961632853)), ('cz', 1, 2), ('u', 1, (-3.7277988119148175, 0.9553166179972424, 2.116222008320426)), ('u', 0, (-2.285343353295458, 0.2247132452497797, 3.4381262481984995)), ('u', 2, (2.1987949279867105, 1.5707964448504583, -3.450709743862259)), ('u', 3, (0.4744718836591277, 0.19902801420265745, 0.13114850552618545)), ('cz', 1, 2), ('u', 1, (-1.3173268598098806, 1.5707965354512161, 0.6861715732073628)), ('u', 0, (0.40112964916438487, 0.6693734434533707, -0.908977999871106)), ('u', 2, (3.3774735267636906, 1.7155129942662726, -0.7314425530107427)), ('u', 3, (-0.0028962095988690102, 0.10099194513739455, -0.03437243163684296)), ('cz', 1, 0), ('u', 1, (0.8846245871808652, 0.7853981667224689, 0.691414405068067)), ('u', 0, (-3.2651085505340642, 1.570796154248359, 2.989384697769773)), ('u', 2, (-1.7031106831616103, 0.2774301045937453, 3.8480021541327947)), ('u', 3, (0.334526450943965, 0.5264093201347806, 0.40273496129147873)), ('cz', 1, 0), ('u', 1, (2.450178246254839, 1.5707963275409877, -2.5337739413160785)), ('u', 0, (1.7230044409829273, 1.5707963116195378, -2.533773942969272)), ('u', 2, (-0.09102238709159846, 0.26116851983517, 0.09983258276432336)), ('u', 3, (0.5609503820762544, 0.7728032743817536, -0.5935934538716358)), ('u', 11, (-2.871320587684953, 0.18270763755063463, 3.2126447403495293)), ('u', 7, (0.0, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 8, (1.2251903831398114e-16, 0.7853981633974483, -1.1496735851465472e-17)), ('u', 7, (-3.141592653589793, 0.7853981633974483, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 7, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (-3.141592653589793, 1.0471975511965979, 3.141592653589793)), ('cz', 9, 8), ('u', 8, (0.0, 1.0471975511965979, 0.0)), ('u', 8, (-3.141592653589793, 2.6179938779914944, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 2.6179938779914944, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 6, (2.0647644523158686, 0.6154797086703749, -1.5791895412191075)), ('u', 7, (0.0, 1.483538959592185e-12, -0.25027598782607496)), ('u', 8, (-3.1335320307099703, 0.1656920318711231, 2.2245652046732305)), ('cz', 6, 7), ('u', 6, (-1.5751645393309863, 0.9663926645733115, 1.8958815458794114)), ('u', 7, (2.0585845923856336, 1.570796326793952, -1.553631715003139)), ('u', 8, (2.7142399639708263, 1.533274132359459, 0.08436264384289682)), ('cz', 7, 8), ('u', 6, (1.224283325054985, 0.35102546868142687, 0.2322075716689106)), ('u', 7, (3.1244280417992076, 1.5707963267945744, -3.062894868320912)), ('u', 8, (3.7981088130622664, 1.1809541342523935, -0.21769268471836156)), ('cz', 6, 7), ('u', 6, (0.22247292361988538, 0.30424216356323164, -0.4342390040675236)), ('u', 7, (-1.0340144033933893, 1.5707963267930358, -0.13589419264094255)), ('u', 8, (2.846283009808967, 1.9960299837750193, -0.2688994952353383)), ('cz', 7, 8), ('u', 6, (0.010627713700761182, 1.2920134711950644, -0.29724720212618594)), ('u', 7, (0.13589419264107394, 1.5618612720601845, 0.26509911239244055)), ('u', 8, (2.8722638134653433, 1.9175120526392753, 0.21078353526792482)), ('cz', 6, 7), ('u', 6, (2.776894944526565, 1.5707963267950549, -2.746675253363822)), ('u', 7, (-0.2650991123935444, 0.7853981633974457, -0.22506355113432194)), ('u', 8, (3.5833692017202323, 0.640094968798964, -2.507522059706495)), ('cz', 6, 7), ('u', 6, (-1.9657137270205, 1.570796326795341, 2.8951453210904075)), ('u', 7, (-2.916529102454747, 1.5707963267960128, 2.8951453210904226)), ('u', 8, (-0.15693365901497597, 0.36820738813035225, 0.26630621549441696)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0))])], (12, 3): [(12, [('u', 5, (0.0, 2.5290379152504547, 0.0)), ('u', 4, (0.0, 0.7353144528166685, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (-3.141592653589793, 0.7353144528166685, 3.141592653589793)), ('u', 3, (0.0, 1.1302856637901981, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.1302856637901981, 3.141592653589793)), ('u', 9, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 10), ('u', 7, (0.0, 0.0, 0.0007585389249996382)), ('u', 6, (0.0, 0.0, 0.0007646114002879502)), ('u', 8, (-3.141223188201763, 0.7853981643166987, 3.1338230837535828)), ('u', 9, (0.5646181035578604, 3.1415926529198557, -4.640169288520284)), ('u', 10, (4.2318857907724166, 2.392043869971481, 0.2428099616940813)), ('cz', 8, 9), ('u', 7, (0.0, 0.0, 0.0008745315367836417)), ('u', 6, (0.0, 0.0, 0.001008973286350238)), ('u', 8, (3.1338230837536436, 0.539595176176044, -3.1190245466440363)), ('u', 9, (1.6430160186593528, 1.5707963256511739, -1.5707963275117758)), ('u', 10, (0.24280996169408903, 0.41584551219021093, -1.1057637098596882)), ('cz', 9, 10), ('u', 7, (0.0, 0.0, 0.0008454228356342076)), ('u', 6, (0.0, 0.0, 0.0007691376676996627)), ('u', 8, (3.1641607605355353, 0.24598317514134324, -3.0321037025224933)), ('u', 9, (-1.5707963275117827, 0.6154797052461431, 2.510064418016416)), ('u', 10, (2.0358289437300985, 0.3071278490292182, -0.16309141234544422)), ('cz', 8, 9), ('u', 7, (0.0, 0.0, 0.0008798751866400383)), ('u', 6, (0.0, 0.0, 0.0008301693344043811)), ('u', 8, (0.1673231707381715, 0.19773294414027243, -1.6445152184463296e-10)), ('u', 9, (0.6315282368998028, 1.5707963234645563, 2.65654054310005e-10)), ('u', 10, (2.9650082736656307, 2.4804727895611136, -0.06184066750305961)), ('cz', 9, 10), ('u', 7, (0.0, 0.0, 0.0008840590547666084)), ('u', 6, (0.0, 0.0, 0.0006162030446476306)), ('u', 8, (-3.14159265375426, 0.19773294374178838, 2.411810782538944)), ('u', 9, (-3.141592653324135, 1.570796327982228, 2.721413933535757)), ('u', 10, (3.07975198608668, 1.5707963267608613, -0.42017871992501865)), ('u', 11, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 11), ('u', 1, (3.0481212327661855, 0.07899252828333087, -2.486482291858509)), ('u', 0, (2.5187445813633853, 0.11791131910415209, -3.14424337125469)), ('u', 2, (0.3482477386355402, 0.7853981633994663, -0.11589222537981132)), ('u', 3, (0.05109338077661091, 1.4248038534026443e-10, -0.006141365651658379)), ('u', 4, (2.5363257468024205, 0.23225208308317713, -2.7274639676522234)), ('cz', 2, 3), ('u', 1, (-2.618867952525948, 0.22996590376829276, 3.0257537211319625)), ('u', 0, (0.6654841593360534, 0.15840591342512744, -0.1537638971389868)), ('u', 2, (2.6000482599469654, 1.398584732268289, -3.2453289143890247)), ('u', 3, (3.120221131491702, 1.570796326877936, -3.018315376489242)), ('u', 4, (3.369310993675936, 1.3835646811128859, 0.10137182641452158)), ('cz', 3, 4), ('u', 1, (0.16232889162309366, 0.366448432598935, 0.4801456292221619)), ('u', 0, (-1.159471014814055, 0.23671083816633937, -1.1654166140951678)), ('u', 2, (-0.5198238601537504, 0.8330660040272931, -0.32544035337245986)), ('u', 3, (3.01831537657018, 0.6154797086702457, -2.9151366262893266)), ('u', 4, (2.392346799094065, 1.0034076270875687, 0.5564515720449603)), ('cz', 2, 3), ('u', 1, (-2.495629777608992, 0.15512082082141004, 2.1847347268341557)), ('u', 0, (-3.389951681520682, 0.05979369921099443, 2.2677925801779106)), ('u', 2, (-0.5983313169168214, 0.14685075521309557, 0.11596571278414872)), ('u', 3, (-0.22645602737219225, 1.5707963267947438, -0.07480043119723165)), ('u', 4, (3.0145402823079914, 1.8658281870890274, 0.3343117114397236)), ('cz', 3, 4), ('u', 1, (2.0636268206154726, 0.06944650835790743, -2.1616011296673348)), ('u', 0, (-0.174785448527092, 0.2233208972446901, 0.3265068923856165)), ('u', 2, (-3.2575583671464816, 0.14685075523943447, 1.850482973666618)), ('u', 3, (0.07480043115310891, 1.570796326760786, -0.023580493037617167)), ('u', 4, (3.251768346274295, 1.5707963268008613, -0.023580493035072703)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (-3.141592653589793, 1.3605291591529143, 3.141592653589793)), ('cz', 3, 4), ('u', 4, (0.0, 0.2102671676419825, 0.0)), ('cz', 5, 4), ('u', 4, (-3.141592653589793, 0.2102671676419825, 3.141592653589793)), ('cz', 3, 4), ('u', 4, (0.0, 1.3605291591529143, 0.0)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 5), ('u', 2, (-4.020944855313366, 1.3674918153663328e-08, 1.8955327837722808)), ('u', 3, (2.571644335889275, 0.256970710801668, -3.520937661994353)), ('u', 1, (1.5878369240669779, 0.6847191637192789, -1.1610675576311817)), ('u', 11, (2.744561547199597, 0.276973804982832, -3.2081383694490966)), ('u', 4, (0.6073665053653187, 0.09512717122877642, 0.669819621177511)), ('cz', 2, 1), ('u', 2, (1.2257506151869664, 1.5707963623076528, -0.5892141577502872)), ('u', 3, (3.1152168653535672, 1.3338831539054212, -0.004750437392734064)), ('u', 1, (-2.061500513358623, 1.319286726616464, 2.2784627704389457)), ('u', 11, (-0.6098168233575867, 0.27349318912962395, -0.32154922078452197)), ('u', 4, (0.23545546303267909, 0.29088353976677306, 0.2400897384178685)), ('cz', 2, 3), ('u', 2, (-2.552378193033693, 0.6847192037149724, 3.534188492066675)), ('u', 3, (2.663659373863285, 2.044380114102642, 0.10725649851591035)), ('u', 1, (0.7771239277655451, 0.6379455799453707, -0.04430849432874845)), ('u', 11, (0.10018178164395003, 0.054483567215899406, -0.2956813466462815)), ('u', 4, (2.8644636354367297, 0.07248934047529751, -3.1947283813428777)), ('cz', 2, 1), ('u', 2, (-0.3925960872757029, 1.5707963263247418, -0.31668776022739037)), ('u', 3, (2.5728587843569435, 1.279740337640615, 0.3687463737795711)), ('u', 1, (0.4135014659345674, 0.1490687485613123, -0.031873805594377136)), ('u', 11, (-0.10784844663278795, 0.4336845541511144, -0.4671784105975467)), ('u', 4, (2.900792186158851, 0.15172119210965154, -2.4415982131295397)), ('cz', 2, 3), ('u', 2, (3.21612579275289, 0.9267683648318735, -3.039069684015713)), ('u', 3, (3.3098637971475, 1.5707962534739768, 0.5808828881238006)), ('u', 1, (-2.7892990152841994, 0.2824052913658417, 3.3284945151104157)), ('u', 11, (-3.294960137381061, 0.4156669728264371, 2.8476196436551744)), ('u', 4, (3.9037784766223416, 1.6059007022404186, 0.025109451776498926)), ('cz', 3, 4), ('u', 2, (-2.8538372621125303, 0.6819392616147583, 3.0462678730100246)), ('u', 3, (2.8743595354937237, 1.3388582157640125, -3.2886661447487247)), ('u', 1, (0.11613136297566654, 0.022900293474337974, -0.13298252405310815)), ('u', 11, (-3.137957320280787, 0.024717994101340363, 2.6678239275881888)), ('u', 4, (3.276242102729564, 1.4329530245118294, -0.1270903014724345)), ('cz', 2, 3), ('u', 2, (-0.03658921574775821, 0.0015976739573187707, -0.008785917296887248)), ('u', 3, (0.016251868542596797, 0.23386530471664435, 0.04490820419687531)), ('u', 1, (3.2113158536099276, 0.20001440788947106, -3.0249039080386986)), ('u', 11, (-2.863057464753309, 0.14510214003147293, 2.634658140142464)), ('u', 4, (-0.08096014718112254, 0.1408418031506233, -0.0737499454495722)), ('cz', 3, 4), ('u', 2, (3.1504417516570307, 0.0015976667802821717, -2.7572642523810478)), ('u', 3, (-3.186500230085183, 0.23386527281654632, 2.9708554763154242)), ('u', 1, (-0.2751328421252311, 0.2396077092505456, 0.15705602281063527)), ('u', 11, (2.9495314267766437, 0.08756063794739287, -2.777182133211986)), ('u', 4, (2.605563542589809, 0.08761920952538847, -3.3706785025452812)), ('cz', 2, 3), ('u', 2, (0.6770370008163178, 0.20683220117293072, -0.09865403177437976)), ('u', 3, (0.3015582031333537, 1.3388581972162237, -0.44994585485476)), ('u', 1, (-0.4222741113636511, 0.19391679094835615, 0.16218295123845616)), ('u', 11, (2.536817680816707, 0.18135324926096447, -3.2842701648958172)), ('u', 4, (2.550552038477692, 1.5109466523370723, -0.08444534854770769)), ('cz', 3, 4), ('u', 2, (-0.24409885125609895, 1.3756834103474782, 0.2610307721274085)), ('u', 3, (1.7890434390071097, 0.10458729264649885, -2.517511837248301)), ('u', 1, (0.36428816122109975, 1.4960864949794048, 2.985193607848061)), ('u', 11, (-3.1917937328852704, 0.05276146455278438, 1.9683483305235738)), ('u', 4, (-0.3802797539376923, 0.0033018073279211285, -0.1689753008428753)), ('cz', 2, 1), ('u', 2, (2.949634724268638, 2.356194489409649, -2.264338203869037)), ('u', 3, (0.20015999778740734, 1.641962926441273, 1.4679462293106675)), ('u', 1, (0.7042723127615393, 2.913184957143721, -5.522634500204262)), ('u', 11, (-1.369142292977996, 0.07735686448561044, 0.4359589943516575)), ('u', 4, (-0.22850821613786854, 0.6259106761271263, 0.15726865284501423)), ('cz', 2, 3), ('u', 2, (0.17691447429567653, 3.1415916481073167, -4.999907078957155)), ('u', 3, (2.7495377977680016, 2.4425735913049627, -0.023910210690165634)), ('u', 1, (-0.19301314707278194, 0.9714510067065284, -0.9547337000725561)), ('u', 11, (-0.29999492932348704, 0.4583471483608021, 0.49638850576941007)), ('u', 4, (0.06584730429261001, 0.11780561008315198, 0.1503144419158602)), ('cz', 2, 1), ('u', 2, (-0.22910938798676903, 1.5707963547216357, -0.2203140210857498)), ('u', 3, (0.6170557504765648, 0.1315022481529303, -0.168565996701113)), ('u', 1, (-2.01041584235678, 1.5707967079771865, 0.12378704669720808)), ('u', 11, (-2.838183263999435, 0.5646118523601609, 3.1235905670108832)), ('u', 4, (0.05768723562376743, 0.30931197823406537, 0.4388505825555788)), ('cz', 2, 1), ('u', 2, (0.21434314248014422, 2.6179835628317303, -0.27583515673825165)), ('u', 3, (3.5408321119432693, 1.7802471772629684, 0.18608246580639864)), ('u', 1, (-2.839376430846758, 0.4545611282306032, 3.1345604281878883)), ('u', 11, (-0.44258434653442885, 0.42426212338702035, -0.17956403511121405)), ('u', 4, (-2.972207621205507, 0.13439300980723798, 3.2097616628759673)), ('cz', 2, 3), ('u', 2, (3.0008857498331762, 0.029918614553072514, -2.7118310298595496)), ('u', 3, (2.881413594241195, 1.76716728691908, 0.4340436007150823)), ('u', 1, (-1.3965942747620073, 1.6519319389361602, 1.3798504999113494)), ('u', 11, (3.145226116705863, 0.3594808447199273, -3.0286167249610427)), ('u', 4, (-2.7253754032788313, 0.5767389877493958, 2.845009730404505)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (-0.491208675658117, 0.4831254884701491, 0.04922378503399025)), ('u', 0, (3.248498512208629, 0.7903103989318482, -2.8417652948614194)), ('u', 0, (-2.2720067530307118, 0.4648169523841723, 3.1647442435988973)), ('u', 0, (0.6609440677860969, 0.42100900700268823, 0.8568434699238094)), ('u', 0, (0.1033952911302645, 1.1765315178603903, -0.08383046798637486)), ('u', 0, (2.151159514349068, 0.6237318407597543, -1.0135397177261316)), ('cz', 1, 0), ('u', 3, (0.27154518203854794, 0.156574284638506, -0.19904746728795425)), ('u', 2, (3.1216419000335165, 1.5433537783775642, 0.18145163589752533)), ('u', 11, (-0.2587427784826737, 0.04212299304800157, -0.11439294212521267)), ('u', 1, (-0.7306382578180514, 2.406864166747237e-07, 0.29645097108066454)), ('u', 0, (-0.9821421222497059, 2.5119044341170866, 0.8721996017651316)), ('cz', 2, 1), ('u', 3, (-2.5086987830304714, 0.6763891249815223, 3.5140692013478008)), ('u', 2, (1.9828302471775878, 0.8347493912911903, -0.19312797575563256)), ('u', 11, (-0.07450254369386232, 0.17645071611579927, -0.09563564617294044)), ('u', 1, (1.6020433307882922, 0.9553167253432009, -1.4651191620910062)), ('u', 0, (-2.5071882129766125, 0.00995959553926695, 3.1313095881852684)), ('cz', 1, 0), ('u', 3, (0.3554334064827065, 0.3181995032142394, 0.1697781864760842)), ('u', 2, (-2.948465788547482, 0.8347500461186684, 2.7752921415493192)), ('u', 11, (0.10009300027086443, 0.24185526979528194, -0.9034560608612274)), ('u', 1, (-2.4618715052845315, 1.5707963748150602, 2.92062862862977)), ('u', 0, (2.43846687704859, 1.7899438485698647, -2.366256135535081)), ('cz', 2, 1), ('u', 3, (3.489971466866175, 0.2063310477483668, -2.4336483332402223)), ('u', 2, (2.5858756306273016, 0.09173639560952442, -3.2934713883461835)), ('u', 11, (3.2016616357651038, 0.8309069796238984, -2.8400200157407496)), ('u', 1, (0.45437702550792736, 1.5707962923084495, 1.23585343535074)), ('u', 0, (-2.292114815324766, 1.3329696509088158, 2.7853678344382478)), ('cz', 1, 0), ('u', 3, (0.10654053620958025, 0.04057293724764386, -0.01869337593786062)), ('u', 2, (2.34312266255703, 1.5173618374423803, 0.1103850443224883)), ('u', 11, (-2.701487648028792, 0.9043926109893675, 3.2884725169943327)), ('u', 1, (1.9057388052947186, 1.5707965058818665, -1.5649332101251299)), ('u', 0, (0.10686658053214887, 0.28764522783237956, 0.5644688708179588)), ('cz', 2, 1), ('u', 3, (0.8373653154688827, 0.7099563492196652, -0.027972786369606595)), ('u', 2, (3.2656815472120604, 1.5346627789921687, -0.27826749186542843)), ('u', 11, (-0.09728025144447311, 0.12193921462821987, -0.04054872395287614)), ('u', 1, (-1.810072511769857, 1.5707964383975355, -0.6567471740069314)), ('u', 0, (0.8050479835661133, 1.5117181066612224, 0.49258342033831326)), ('cz', 1, 0), ('u', 3, (-3.2202078613998246, 0.6014823782931357, 2.4333355310556968)), ('u', 2, (-1.0721280643830893, 0.1638694885173495, -0.5321234434239689)), ('u', 11, (3.1062471101427542, 1.4741250065503686, 0.035703721946902967)), ('u', 1, (-2.4848455271739787, 1.5707964723378247, 2.8327647643966762)), ('u', 0, (4.061860192304446, 4.1519342329353984e-08, -2.0033566590882907)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 10, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (-3.141592653589793, 1.3605291591529143, 3.141592653589793)), ('cz', 9, 10), ('u', 10, (0.0, 0.2102671676419825, 0.0)), ('cz', 11, 10), ('u', 10, (-3.141592653589793, 0.2102671676419825, 3.141592653589793)), ('cz', 9, 10), ('u', 10, (0.0, 1.3605291591529143, 0.0)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 10, 11), ('u', 6, (0.00020663823424371675, 0.0457231615349091, 1.0096266719987605)), ('u', 7, (1.5707472836055427, 1.5707963253991875, -1.5245230883262415)), ('u', 8, (-1.5227746953744032, 1.5707963278482338, 1.5328697902830697)), ('u', 9, (-0.006353961847695994, 0.02416500538156028, 0.014892265155584866)), ('u', 10, (-0.14764531105557424, 0.46531074621646273, -0.2572634986839013)), ('cz', 7, 8), ('u', 6, (1.009626701941372, 0.006269047909349947, 0.5654175377429311)), ('u', 7, (1.617069565265616, 0.6882333588934038, -1.5975526953330097)), ('u', 8, (1.5328697903344901, 0.6847192029590976, -3.141592653781001)), ('u', 9, (0.014892265315672079, 0.005128884657443164, 0.02675956829582319)), ('u', 10, (-0.25726349869508697, 0.33974543325781076, -0.30288595631869536)), ('cz', 7, 8), ('u', 6, (-2.576174881984309, 0.04329075591782931, 3.647505179304552)), ('u', 7, (-1.5975526954762391, 1.5885705616934935, 1.46535189894401)), ('u', 8, (-1.16165965735604e-10, 1.5707963305271444, -1.471469396832998)), ('u', 9, (3.168352221773913, 1.599999123301306, 0.0011423893762105841)), ('u', 10, (-0.30288595632369447, 0.1545986345521885, -0.30741210614652664)), ('cz', 8, 9), ('u', 6, (-2.6356800756197245, 0.019957751015754324, 3.611033723048523)), ('u', 7, (3.0361482150159294, 0.051020547972946084, -3.241874264317337)), ('u', 8, (-3.042265732012571, 2.255515529525348, 3.1415926493495174)), ('u', 9, (3.14273504285698, 1.5707963268539142, 0.13440701843686087)), ('u', 10, (2.834180547435486, 0.20677188986582806, -3.3929670717691476)), ('cz', 8, 9), ('u', 6, (0.4694409648291979, 0.009359905107940195, 0.24425377070261806)), ('u', 7, (-3.241874267022193, 0.017667137229213245, 3.041030817805053)), ('u', 8, (-3.1415926506427083, 1.5707963285851194, 3.1402616437536905)), ('u', 9, (-3.00718563507565, 0.7839642561619297, 3.1415926510651127)), ('u', 10, (2.8902182353407264, 0.36563049554442895, -3.3658033214613505)), ('cz', 7, 8), ('u', 6, (0.2442535092229099, 0.0179496359739968, 0.054326499919879104)), ('u', 7, (-0.10056184364293307, 0.06798093607384323, -0.04963982148556047)), ('u', 8, (-0.0013309942829948102, 2.451498680523748, 0.15173472440682573)), ('u', 9, (3.141592650960565, 2.357628399177859, -0.1381564396572239)), ('u', 10, (2.91738198572791, 0.5151524185048744, -3.3105430110746634)), ('cz', 9, 10), ('u', 6, (0.054326134014471054, 0.015623615776086655, -0.036146625830046804)), ('u', 7, (-3.1912324934703933, 0.0232105439509164, 3.058262639186251)), ('u', 8, (0.1517347197502425, 0.0056339422549575545, 0.10189937227969903)), ('u', 9, (3.003436213993115, 0.23392342405649574, -3.0752642086076425)), ('u', 10, (2.972642296113425, 1.573403724742435, -0.01830708263865244)), ('cz', 8, 9), ('u', 6, (3.1054461321657953, 0.0017796589770213483, -3.1706211787554865)), ('u', 7, (-3.2249226655572847, 0.043913946360584356, 3.0142322399465136)), ('u', 8, (0.1018993758280302, 6.113487075145362e-09, 0.07587918130131695)), ('u', 9, (3.2079210984470308, 1.8027377195878327, -2.8786095588314495)), ('u', 10, (-3.1598997361630423, 0.002609144383916954, 3.077252783930476)), ('cz', 9, 10), ('u', 6, (3.112564076293773, 0.03356022252694113, -3.095867964039239)), ('u', 7, (-3.268953069777305, 0.05078533303322996, 2.897804362259922)), ('u', 8, (3.2174718339981148, 5.357181973731769e-09, -2.880012881448579)), ('u', 9, (-2.8786095589335123, 0.2666037998208392, 3.0753614418759345)), ('u', 10, (3.0772527837717916, 0.01811788444445302, -3.1905449535555435)), ('cz', 8, 9), ('u', 6, (0.04578652195562127, 0.4475520493295309, 0.4904631700164719)), ('u', 7, (3.150793602987614, 3.0845488839212947, 0.12545036752858718)), ('u', 8, (0.3397918935143327, 0.6202872101386329, 0.3514372199044056)), ('u', 9, (-0.06623121167117405, 0.04108488370169745, -0.059147746086104985)), ('u', 10, (3.092640353721128, 0.0023624851229679627, -3.1108719339139155)), ('cz', 6, 7), ('u', 6, (0.4904625893892679, 0.16995904180718713, -0.9081998972709736)), ('u', 7, (0.12545035447696834, 1.5707963244349294, 0.8712334056516684)), ('u', 8, (0.3514372158877993, 1.071773487096535, 0.13314599702987048)), ('u', 9, (-0.05914774614290811, 0.02375615393747697, -0.05084463471850991)), ('u', 10, (0.030720719831308207, 0.0018717948879952927, 0.05394598680157092)), ('cz', 7, 8), ('u', 6, (-0.2014923037514699, 0.4268005233661855, 0.25511660776087236)), ('u', 7, (-3.055757157487303, 1.048271933166876, 2.8334151433735055)), ('u', 8, (-3.1487368157572755, 0.26352062225447836, 2.9327808659906234)), ('u', 9, (-0.05084463467423744, 0.0011952489665043515, -0.043222830581381835)), ('u', 10, (0.053945986491619524, 0.0052121663685944924, 0.08124946773231354)), ('cz', 6, 7), ('u', 6, (0.25511681790344093, 0.09299936124112074, 1.128429544169301)), ('u', 7, (-0.30817752508229174, 1.5707963288694706, -0.3752000003904286)), ('u', 8, (2.9327808667588418, 2.1200552734865115, -0.3421571198228124)), ('u', 9, (3.0983698230116117, 0.0275975162074632, -3.1799879515181537)), ('u', 10, (0.08124946789153831, 0.00652177010189773, 0.11299766601375297)), ('cz', 7, 8), ('u', 6, (1.1284296141652224, 0.062293876538084, 1.2695267606828162)), ('u', 7, (2.7663926537140275, 1.5397471685127067, -3.224077393754685)), ('u', 8, (2.799435534778861, 1.5919143791517385, -0.7479031372916201)), ('u', 9, (-0.03839529799516117, 1.3021246866573928, -0.12285004561036661)), ('u', 10, (3.254590319619691, 1.5653822410333713, -0.028115098479905676)), ('cz', 9, 10), ('u', 6, (-3.03504001982043, 0.07309508079128234, 3.018901376884088)), ('u', 7, (3.2598672159837987, 0.04416579841505668, -3.0079714756133447)), ('u', 8, (-0.5471438319283586, 0.2187796940235946, -0.10944399256069004)), ('u', 9, (-0.12285004560160728, 0.8570719423916445, 3.1415926454392897)), ('u', 10, (3.113477555207084, 1.4843998799533307, 0.044253817043743204)), ('cz', 8, 9), ('u', 6, (-3.264283981393828, 0.07613661885048449, 2.8846789444513656)), ('u', 7, (3.2752138346976207, 0.03303254685421155, -2.9969009783724743)), ('u', 8, (3.032148662016482, 1.9321634517806854, -3.1164499330425315)), ('u', 9, (3.1415926455347836, 1.5707963264365103, -0.04037272564948413)), ('u', 10, (-3.0973388366285635, 0.07371941131320663, 3.183945936471396)), ('cz', 8, 9), ('u', 6, (-0.3224759102381275, 0.06616322789909065, -0.34217188336677573)), ('u', 7, (0.37762944290348693, 0.06574280511917258, 0.18800714765143728)), ('u', 8, (3.116449941407185, 0.785398163465744, -3.1415926578762003)), ('u', 9, (-0.033587766537974205, 0.16704016987731776, 0.035087214375447436)), ('u', 10, (-3.102219227807077, 0.04228280473821831, 3.176485134497014)), ('cz', 7, 8), ('u', 6, (-0.3421718949520949, 0.15792329265758215, -0.3740301535241837)), ('u', 7, (0.18800715245523983, 0.2118550011088309, 0.18133631156722144)), ('u', 8, (-5.236712497663776e-09, 0.7853981640888559, -0.01897398583496425)), ('u', 9, (3.176679868087319, 1.3498085218606124, -0.02007277407464203)), ('u', 10, (-3.1067001726317995, 0.02591614501415039, 3.170686031954766)), ('cz', 8, 9), ('u', 6, (1.1967661426828256, 1.5686016018787456, -1.3078992879554734)), ('u', 7, (-1.3894600162525077, 1.4946381796557338, 1.2761098021635167)), ('u', 8, (3.12261866777336, 0.7760616403836724, -3.1487225678300796)), ('u', 9, (3.1215198795698953, 1.274781628110235, -0.011669731358020297)), ('u', 10, (-3.1124992752539415, 0.009023678312353437, 3.1637915676879858)), ('cz', 6, 7), ('u', 6, (1.8336933655399732, 0.9566322584167632, -1.861410887626854)), ('u', 7, (1.2761098021031225, 0.6154797092189657, -2.0340211099566172)), ('u', 8, (-3.1487225678351254, 0.7834517421688932, 3.122333136933048)), ('u', 9, (3.1299229222464513, 0.2758294280037525, -3.1500759368762656)), ('u', 10, (0.022198914092677623, 0.00756752393486162, 0.014481610731526623)), ('cz', 6, 7), ('u', 6, (-1.8614108876229754, 1.5476460375749002, 1.5998778532511135)), ('u', 7, (1.107571543548716, 1.8503922055304418, -1.4685745838680946)), ('u', 8, (3.122333136909429, 2.36636284057493, 0.02884537539009946)), ('u', 9, (3.1331093701687522, 0.24505547805241654, -3.1342865810756053)), ('u', 10, (0.014481610746405887, 0.022797251634206567, 0.006189292913471925)), ('cz', 7, 8), ('u', 6, (-3.112511191663781, 0.2615124841522484, 3.13626341178677)), ('u', 7, (-3.039370915866766, 0.9553166180914445, 3.063106499388604)), ('u', 8, (3.1704380290308953, 1.5707963265711409, -0.00024991941079388624)), ('u', 9, (3.148898726359738, 0.20452705603326007, -3.117630836274382)), ('u', 10, (0.006189292905740272, 0.035517056468296754, -0.0024996203552288924)), ('cz', 7, 8), ('u', 6, (-3.1634251739360537, 0.20663879733053145, 3.119357510385685)), ('u', 7, (3.067870804228894, 0.7853981633266898, -3.14159265097917)), ('u', 8, (-3.13408354875839, 0.029673366808064105, 3.1379614606673214)), ('u', 9, (-3.115933861798746, 0.1438554085860551, 3.154467081776227)), ('u', 10, (-0.010480646488991493, 0.044056986284693254, -0.020068026447170588)), ('cz', 6, 7), ('u', 6, (-0.02223515910588095, 0.2959491786699587, 0.021100921762980156)), ('u', 7, (2.564204410891336e-09, 0.7853981654143503, 2.4224197534672933e-09)), ('u', 8, (3.137961460587512, 1.5411237331203873, -0.06944019192013995)), ('u', 9, (-3.128718225141107, 0.11132646239649657, 3.1344208404990037)), ('u', 10, (-0.020068026482461674, 0.04610809780819416, -0.02981526170276449)), ('cz', 7, 8), ('u', 6, (0.021100906850089838, 1.8906287889648599, -0.007535173259034228)), ('u', 7, (2.457048822810036e-09, 2.356194490004182, -4.5262644613857766e-09)), ('u', 8, (3.0721524616733773, 1.0230828668750294, -0.0011447520003269318)), ('u', 9, (3.134420840499627, 0.0997000859410616, -3.1457264413875854)), ('u', 10, (-0.029815261649792594, 0.04248209501241417, -0.039623702374196164)), ('cz', 6, 7), ('u', 6, (3.134057480495983, 0.785398163312766, -3.1415926541580825)), ('u', 7, (-4.570539703333689e-09, 1.0471975440745136, -3.1599689872515833e-10)), ('u', 8, (3.1404479015927005, 0.5145185342670128, -3.1395855812423235)), ('u', 9, (-3.1457264415203, 0.0970989005701676, 3.122502996454589)), ('u', 10, (-0.03962370242207246, 0.032658044205827415, -0.049254267926738635)), ('cz', 6, 7), ('u', 6, (-3.1415926542088455, 2.5261129338996686, 3.0751962333179197)), ('u', 7, (-3.1415926538844747, 1.5707963273578327, 3.0751962329597813)), ('u', 8, (-3.1395855813722644, 0.508566592425987, 3.059770192603981)), ('u', 9, (-3.1606823107275073, 0.0980606959949402, 3.1012414353482995)), ('u', 10, (-0.04925426793017285, 0.016203780141577806, -0.05827346509839334)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0))]), (12, [('u', 5, (0.0, 2.5290379152504547, 0.0)), ('u', 4, (0.0, 0.7353144528166685, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (-3.141592653589793, 0.7353144528166685, 3.141592653589793)), ('u', 3, (0.0, 1.1302856637901981, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.1302856637901981, 3.141592653589793)), ('u', 9, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 10), ('u', 7, (0.0, 0.0, 0.0007585389249996382)), ('u', 6, (0.0, 0.0, 0.0007646114002879502)), ('u', 8, (-3.141223188201763, 0.7853981643166987, 3.1338230837535828)), ('u', 9, (0.5646181035578604, 3.1415926529198557, -4.640169288520284)), ('u', 10, (4.2318857907724166, 2.392043869971481, 0.2428099616940813)), ('cz', 8, 9), ('u', 7, (0.0, 0.0, 0.0008745315367836417)), ('u', 6, (0.0, 0.0, 0.001008973286350238)), ('u', 8, (3.1338230837536436, 0.539595176176044, -3.1190245466440363)), ('u', 9, (1.6430160186593528, 1.5707963256511739, -1.5707963275117758)), ('u', 10, (0.24280996169408903, 0.41584551219021093, -1.1057637098596882)), ('cz', 9, 10), ('u', 7, (0.0, 0.0, 0.0008454228356342076)), ('u', 6, (0.0, 0.0, 0.0007691376676996627)), ('u', 8, (3.1641607605355353, 0.24598317514134324, -3.0321037025224933)), ('u', 9, (-1.5707963275117827, 0.6154797052461431, 2.510064418016416)), ('u', 10, (2.0358289437300985, 0.3071278490292182, -0.16309141234544422)), ('cz', 8, 9), ('u', 7, (0.0, 0.0, 0.0008798751866400383)), ('u', 6, (0.0, 0.0, 0.0008301693344043811)), ('u', 8, (0.1673231707381715, 0.19773294414027243, -1.6445152184463296e-10)), ('u', 9, (0.6315282368998028, 1.5707963234645563, 2.65654054310005e-10)), ('u', 10, (2.9650082736656307, 2.4804727895611136, -0.06184066750305961)), ('cz', 9, 10), ('u', 7, (0.0, 0.0, 0.0008840590547666084)), ('u', 6, (0.0, 0.0, 0.0006162030446476306)), ('u', 8, (-3.14159265375426, 0.19773294374178838, 2.411810782538944)), ('u', 9, (-3.141592653324135, 1.570796327982228, 2.721413933535757)), ('u', 10, (3.07975198608668, 1.5707963267608613, -0.42017871992501865)), ('u', 11, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 11), ('u', 1, (3.0481212327661855, 0.07899252828333087, -2.486482291858509)), ('u', 0, (2.5187445813633853, 0.11791131910415209, -3.14424337125469)), ('u', 2, (0.3482477386355402, 0.7853981633994663, -0.11589222537981132)), ('u', 3, (0.05109338077661091, 1.4248038534026443e-10, -0.006141365651658379)), ('u', 4, (2.5363257468024205, 0.23225208308317713, -2.7274639676522234)), ('cz', 2, 3), ('u', 1, (-2.618867952525948, 0.22996590376829276, 3.0257537211319625)), ('u', 0, (0.6654841593360534, 0.15840591342512744, -0.1537638971389868)), ('u', 2, (2.6000482599469654, 1.398584732268289, -3.2453289143890247)), ('u', 3, (3.120221131491702, 1.570796326877936, -3.018315376489242)), ('u', 4, (3.369310993675936, 1.3835646811128859, 0.10137182641452158)), ('cz', 3, 4), ('u', 1, (0.16232889162309366, 0.366448432598935, 0.4801456292221619)), ('u', 0, (-1.159471014814055, 0.23671083816633937, -1.1654166140951678)), ('u', 2, (-0.5198238601537504, 0.8330660040272931, -0.32544035337245986)), ('u', 3, (3.01831537657018, 0.6154797086702457, -2.9151366262893266)), ('u', 4, (2.392346799094065, 1.0034076270875687, 0.5564515720449603)), ('cz', 2, 3), ('u', 1, (-2.495629777608992, 0.15512082082141004, 2.1847347268341557)), ('u', 0, (-3.389951681520682, 0.05979369921099443, 2.2677925801779106)), ('u', 2, (-0.5983313169168214, 0.14685075521309557, 0.11596571278414872)), ('u', 3, (-0.22645602737219225, 1.5707963267947438, -0.07480043119723165)), ('u', 4, (3.0145402823079914, 1.8658281870890274, 0.3343117114397236)), ('cz', 3, 4), ('u', 1, (2.0636268206154726, 0.06944650835790743, -2.1616011296673348)), ('u', 0, (-0.174785448527092, 0.2233208972446901, 0.3265068923856165)), ('u', 2, (-3.2575583671464816, 0.14685075523943447, 1.850482973666618)), ('u', 3, (0.07480043115310891, 1.570796326760786, -0.023580493037617167)), ('u', 4, (3.251768346274295, 1.5707963268008613, -0.023580493035072703)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (-3.141592653589793, 1.3605291591529143, 3.141592653589793)), ('cz', 3, 4), ('u', 4, (0.0, 0.2102671676419825, 0.0)), ('cz', 5, 4), ('u', 4, (-3.141592653589793, 0.2102671676419825, 3.141592653589793)), ('cz', 3, 4), ('u', 4, (0.0, 1.3605291591529143, 0.0)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 2, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 1, 2), ('u', 2, (1.570796326794898, 0.6847192030022828, -1.570796326794898)), ('u', 1, (1.5707963267948966, 0.6847192030022828, -1.5707963267948966)), ('cz', 1, 2), ('u', 2, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 1, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 2, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 0.8860771237926138, 3.141592653589793)), ('cz', 3, 2), ('u', 2, (0.0, 0.8860771237926138, 0.0)), ('u', 2, (-3.141592653589793, 2.4568734505875103, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 2.4568734505875103, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 0.23182380450040305, 0.0)), ('cz', 4, 3), ('u', 3, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 1.3389725222944935, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 3, (-0.8844383883781579, 0.15461951980193148, 0.4077681607359882)), ('u', 2, (-0.015357436798632329, 1.570796812492685, 0.27077369932017253)), ('u', 11, (-3.2098263381589156, 0.09287850884125283, 2.6479773673862903)), ('u', 1, (-0.015357430813252027, 1.5707967894868313, 3.3211439827152405)), ('u', 0, (-0.491208675658117, 0.4831254884701491, 0.04922378503399025)), ('cz', 2, 1), ('u', 3, (0.467223655818865, 1.4712177995839404, 0.6981893533938204)), ('u', 2, (2.870819045267668, 2.3561944879565426, -2.633942760816981)), ('u', 11, (-1.0502586325838734, 0.20892461812751373, 0.7389124574425363)), ('u', 1, (-5.713925627527281, 2.8916181984880023, 0.5515112442866319)), ('u', 0, (3.248498512208629, 0.7903103989318482, -2.8417652948614194)), ('cz', 3, 2), ('u', 3, (2.871395683221043, 2.3066219428850427, 0.04515518473946756)), ('u', 2, (0.4033361789673706, 3.1415918518349835, -5.289755193561309)), ('u', 11, (-0.6257503698478103, 0.24490845449449444, 0.6258129688038535)), ('u', 1, (0.009098086642510106, 0.9907828773947474, -1.1296353304436193)), ('u', 0, (-2.2720067530307118, 0.4648169523841723, 3.1647442435988973)), ('cz', 2, 1), ('u', 3, (-2.7526697548852153, 0.01329079925442627, 2.826003808194034)), ('u', 2, (-0.08244412187423705, 1.5707963635248265, -0.44755345936690955)), ('u', 11, (0.36713282506686173, 0.308376205706903, 0.1232262158346361)), ('u', 1, (-1.7759094412785768, 1.5707964702201629, -0.2075825807948034)), ('u', 0, (0.6609440677860969, 0.42100900700268823, 0.8568434699238094)), ('cz', 2, 1), ('u', 3, (3.501920426362306, 1.3430067274140598, 0.18831065667900893)), ('u', 2, (0.36252214468949207, 2.615899067449905, -0.6627693203040838)), ('u', 11, (-0.007599469961123359, 0.5577372180501146, 0.11443093896114555)), ('u', 1, (-2.7833369800472383, 0.10585753814380736, 3.3812678616213763)), ('u', 0, (0.1033952911302645, 1.1765315178603903, -0.08383046798637486)), ('cz', 3, 2), ('u', 3, (3.0203844143048926, 1.7588309002972864, 0.4507052249020116)), ('u', 2, (2.8718881856967235, 0.04085948487966964, -2.2869780797919286)), ('u', 11, (3.090453441212571, 0.0445005310767158, -2.8286729280331078)), ('u', 1, (-1.315125181955235, 1.621259841033651, 1.7345159717028364)), ('u', 0, (2.151159514349068, 0.6237318407597543, -1.0135397177261316)), ('cz', 1, 0), ('u', 3, (0.27154518203854794, 0.156574284638506, -0.19904746728795425)), ('u', 2, (3.1216419000335165, 1.5433537783775642, 0.18145163589752533)), ('u', 11, (-0.2587427784826737, 0.04212299304800157, -0.11439294212521267)), ('u', 1, (-0.7306382578180514, 2.406864166747237e-07, 0.29645097108066454)), ('u', 0, (-0.9821421222497059, 2.5119044341170866, 0.8721996017651316)), ('cz', 2, 1), ('u', 3, (-2.5086987830304714, 0.6763891249815223, 3.5140692013478008)), ('u', 2, (1.9828302471775878, 0.8347493912911903, -0.19312797575563256)), ('u', 11, (-0.07450254369386232, 0.17645071611579927, -0.09563564617294044)), ('u', 1, (1.6020433307882922, 0.9553167253432009, -1.4651191620910062)), ('u', 0, (-2.5071882129766125, 0.00995959553926695, 3.1313095881852684)), ('cz', 1, 0), ('u', 3, (0.3554334064827065, 0.3181995032142394, 0.1697781864760842)), ('u', 2, (-2.948465788547482, 0.8347500461186684, 2.7752921415493192)), ('u', 11, (0.10009300027086443, 0.24185526979528194, -0.9034560608612274)), ('u', 1, (-2.4618715052845315, 1.5707963748150602, 2.92062862862977)), ('u', 0, (2.43846687704859, 1.7899438485698647, -2.366256135535081)), ('cz', 2, 1), ('u', 3, (3.489971466866175, 0.2063310477483668, -2.4336483332402223)), ('u', 2, (2.5858756306273016, 0.09173639560952442, -3.2934713883461835)), ('u', 11, (3.2016616357651038, 0.8309069796238984, -2.8400200157407496)), ('u', 1, (0.45437702550792736, 1.5707962923084495, 1.23585343535074)), ('u', 0, (-2.292114815324766, 1.3329696509088158, 2.7853678344382478)), ('cz', 1, 0), ('u', 3, (0.10654053620958025, 0.04057293724764386, -0.01869337593786062)), ('u', 2, (2.34312266255703, 1.5173618374423803, 0.1103850443224883)), ('u', 11, (-2.701487648028792, 0.9043926109893675, 3.2884725169943327)), ('u', 1, (1.9057388052947186, 1.5707965058818665, -1.5649332101251299)), ('u', 0, (0.10686658053214887, 0.28764522783237956, 0.5644688708179588)), ('cz', 2, 1), ('u', 3, (0.8373653154688827, 0.7099563492196652, -0.027972786369606595)), ('u', 2, (3.2656815472120604, 1.5346627789921687, -0.27826749186542843)), ('u', 11, (-0.09728025144447311, 0.12193921462821987, -0.04054872395287614)), ('u', 1, (-1.810072511769857, 1.5707964383975355, -0.6567471740069314)), ('u', 0, (0.8050479835661133, 1.5117181066612224, 0.49258342033831326)), ('cz', 1, 0), ('u', 3, (-3.2202078613998246, 0.6014823782931357, 2.4333355310556968)), ('u', 2, (-1.0721280643830893, 0.1638694885173495, -0.5321234434239689)), ('u', 11, (3.1062471101427542, 1.4741250065503686, 0.035703721946902967)), ('u', 1, (-2.4848455271739787, 1.5707964723378247, 2.8327647643966762)), ('u', 0, (4.061860192304446, 4.1519342329353984e-08, -2.0033566590882907)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 10, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (-3.141592653589793, 1.3605291591529143, 3.141592653589793)), ('cz', 9, 10), ('u', 10, (0.0, 0.2102671676419825, 0.0)), ('cz', 11, 10), ('u', 10, (-3.141592653589793, 0.2102671676419825, 3.141592653589793)), ('cz', 9, 10), ('u', 10, (0.0, 1.3605291591529143, 0.0)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 10, 11), ('u', 6, (0.00020663823424371675, 0.0457231615349091, 1.0096266719987605)), ('u', 7, (1.5707472836055427, 1.5707963253991875, -1.5245230883262415)), ('u', 8, (-1.5227746953744032, 1.5707963278482338, 1.5328697902830697)), ('u', 9, (-0.006353961847695994, 0.02416500538156028, 0.014892265155584866)), ('u', 10, (-0.14764531105557424, 0.46531074621646273, -0.2572634986839013)), ('cz', 7, 8), ('u', 6, (1.009626701941372, 0.006269047909349947, 0.5654175377429311)), ('u', 7, (1.617069565265616, 0.6882333588934038, -1.5975526953330097)), ('u', 8, (1.5328697903344901, 0.6847192029590976, -3.141592653781001)), ('u', 9, (0.014892265315672079, 0.005128884657443164, 0.02675956829582319)), ('u', 10, (-0.25726349869508697, 0.33974543325781076, -0.30288595631869536)), ('cz', 7, 8), ('u', 6, (-2.576174881984309, 0.04329075591782931, 3.647505179304552)), ('u', 7, (-1.5975526954762391, 1.5885705616934935, 1.46535189894401)), ('u', 8, (-1.16165965735604e-10, 1.5707963305271444, -1.471469396832998)), ('u', 9, (3.168352221773913, 1.599999123301306, 0.0011423893762105841)), ('u', 10, (-0.30288595632369447, 0.1545986345521885, -0.30741210614652664)), ('cz', 8, 9), ('u', 6, (-2.6356800756197245, 0.019957751015754324, 3.611033723048523)), ('u', 7, (3.0361482150159294, 0.051020547972946084, -3.241874264317337)), ('u', 8, (-3.042265732012571, 2.255515529525348, 3.1415926493495174)), ('u', 9, (3.14273504285698, 1.5707963268539142, 0.13440701843686087)), ('u', 10, (2.834180547435486, 0.20677188986582806, -3.3929670717691476)), ('cz', 8, 9), ('u', 6, (0.4694409648291979, 0.009359905107940195, 0.24425377070261806)), ('u', 7, (-3.241874267022193, 0.017667137229213245, 3.041030817805053)), ('u', 8, (-3.1415926506427083, 1.5707963285851194, 3.1402616437536905)), ('u', 9, (-3.00718563507565, 0.7839642561619297, 3.1415926510651127)), ('u', 10, (2.8902182353407264, 0.36563049554442895, -3.3658033214613505)), ('cz', 7, 8), ('u', 6, (0.2442535092229099, 0.0179496359739968, 0.054326499919879104)), ('u', 7, (-0.10056184364293307, 0.06798093607384323, -0.04963982148556047)), ('u', 8, (-0.0013309942829948102, 2.451498680523748, 0.15173472440682573)), ('u', 9, (3.141592650960565, 2.357628399177859, -0.1381564396572239)), ('u', 10, (2.91738198572791, 0.5151524185048744, -3.3105430110746634)), ('cz', 9, 10), ('u', 6, (0.054326134014471054, 0.015623615776086655, -0.036146625830046804)), ('u', 7, (-3.1912324934703933, 0.0232105439509164, 3.058262639186251)), ('u', 8, (0.1517347197502425, 0.0056339422549575545, 0.10189937227969903)), ('u', 9, (3.003436213993115, 0.23392342405649574, -3.0752642086076425)), ('u', 10, (2.972642296113425, 1.573403724742435, -0.01830708263865244)), ('cz', 8, 9), ('u', 6, (3.1054461321657953, 0.0017796589770213483, -3.1706211787554865)), ('u', 7, (-3.2249226655572847, 0.043913946360584356, 3.0142322399465136)), ('u', 8, (0.1018993758280302, 6.113487075145362e-09, 0.07587918130131695)), ('u', 9, (3.2079210984470308, 1.8027377195878327, -2.8786095588314495)), ('u', 10, (-3.1598997361630423, 0.002609144383916954, 3.077252783930476)), ('cz', 9, 10), ('u', 6, (3.112564076293773, 0.03356022252694113, -3.095867964039239)), ('u', 7, (-3.268953069777305, 0.05078533303322996, 2.897804362259922)), ('u', 8, (3.2174718339981148, 5.357181973731769e-09, -2.880012881448579)), ('u', 9, (-2.8786095589335123, 0.2666037998208392, 3.0753614418759345)), ('u', 10, (3.0772527837717916, 0.01811788444445302, -3.1905449535555435)), ('cz', 8, 9), ('u', 6, (0.04578652195562127, 0.4475520493295309, 0.4904631700164719)), ('u', 7, (3.150793602987614, 3.0845488839212947, 0.12545036752858718)), ('u', 8, (0.3397918935143327, 0.6202872101386329, 0.3514372199044056)), ('u', 9, (-0.06623121167117405, 0.04108488370169745, -0.059147746086104985)), ('u', 10, (3.092640353721128, 0.0023624851229679627, -3.1108719339139155)), ('cz', 6, 7), ('u', 6, (0.4904625893892679, 0.16995904180718713, -0.9081998972709736)), ('u', 7, (0.12545035447696834, 1.5707963244349294, 0.8712334056516684)), ('u', 8, (0.3514372158877993, 1.071773487096535, 0.13314599702987048)), ('u', 9, (-0.05914774614290811, 0.02375615393747697, -0.05084463471850991)), ('u', 10, (0.030720719831308207, 0.0018717948879952927, 0.05394598680157092)), ('cz', 7, 8), ('u', 6, (-0.2014923037514699, 0.4268005233661855, 0.25511660776087236)), ('u', 7, (-3.055757157487303, 1.048271933166876, 2.8334151433735055)), ('u', 8, (-3.1487368157572755, 0.26352062225447836, 2.9327808659906234)), ('u', 9, (-0.05084463467423744, 0.0011952489665043515, -0.043222830581381835)), ('u', 10, (0.053945986491619524, 0.0052121663685944924, 0.08124946773231354)), ('cz', 6, 7), ('u', 6, (0.25511681790344093, 0.09299936124112074, 1.128429544169301)), ('u', 7, (-0.30817752508229174, 1.5707963288694706, -0.3752000003904286)), ('u', 8, (2.9327808667588418, 2.1200552734865115, -0.3421571198228124)), ('u', 9, (3.0983698230116117, 0.0275975162074632, -3.1799879515181537)), ('u', 10, (0.08124946789153831, 0.00652177010189773, 0.11299766601375297)), ('cz', 7, 8), ('u', 6, (1.1284296141652224, 0.062293876538084, 1.2695267606828162)), ('u', 7, (2.7663926537140275, 1.5397471685127067, -3.224077393754685)), ('u', 8, (2.799435534778861, 1.5919143791517385, -0.7479031372916201)), ('u', 9, (-0.03839529799516117, 1.3021246866573928, -0.12285004561036661)), ('u', 10, (3.254590319619691, 1.5653822410333713, -0.028115098479905676)), ('cz', 9, 10), ('u', 6, (-3.03504001982043, 0.07309508079128234, 3.018901376884088)), ('u', 7, (3.2598672159837987, 0.04416579841505668, -3.0079714756133447)), ('u', 8, (-0.5471438319283586, 0.2187796940235946, -0.10944399256069004)), ('u', 9, (-0.12285004560160728, 0.8570719423916445, 3.1415926454392897)), ('u', 10, (3.113477555207084, 1.4843998799533307, 0.044253817043743204)), ('cz', 8, 9), ('u', 6, (-3.264283981393828, 0.07613661885048449, 2.8846789444513656)), ('u', 7, (3.2752138346976207, 0.03303254685421155, -2.9969009783724743)), ('u', 8, (3.032148662016482, 1.9321634517806854, -3.1164499330425315)), ('u', 9, (3.1415926455347836, 1.5707963264365103, -0.04037272564948413)), ('u', 10, (-3.0973388366285635, 0.07371941131320663, 3.183945936471396)), ('cz', 8, 9), ('u', 6, (-0.3224759102381275, 0.06616322789909065, -0.34217188336677573)), ('u', 7, (0.37762944290348693, 0.06574280511917258, 0.18800714765143728)), ('u', 8, (3.116449941407185, 0.785398163465744, -3.1415926578762003)), ('u', 9, (-0.033587766537974205, 0.16704016987731776, 0.035087214375447436)), ('u', 10, (-3.102219227807077, 0.04228280473821831, 3.176485134497014)), ('cz', 7, 8), ('u', 6, (-0.3421718949520949, 0.15792329265758215, -0.3740301535241837)), ('u', 7, (0.18800715245523983, 0.2118550011088309, 0.18133631156722144)), ('u', 8, (-5.236712497663776e-09, 0.7853981640888559, -0.01897398583496425)), ('u', 9, (3.176679868087319, 1.3498085218606124, -0.02007277407464203)), ('u', 10, (-3.1067001726317995, 0.02591614501415039, 3.170686031954766)), ('cz', 8, 9), ('u', 6, (1.1967661426828256, 1.5686016018787456, -1.3078992879554734)), ('u', 7, (-1.3894600162525077, 1.4946381796557338, 1.2761098021635167)), ('u', 8, (3.12261866777336, 0.7760616403836724, -3.1487225678300796)), ('u', 9, (3.1215198795698953, 1.274781628110235, -0.011669731358020297)), ('u', 10, (-3.1124992752539415, 0.009023678312353437, 3.1637915676879858)), ('cz', 6, 7), ('u', 6, (1.8336933655399732, 0.9566322584167632, -1.861410887626854)), ('u', 7, (1.2761098021031225, 0.6154797092189657, -2.0340211099566172)), ('u', 8, (-3.1487225678351254, 0.7834517421688932, 3.122333136933048)), ('u', 9, (3.1299229222464513, 0.2758294280037525, -3.1500759368762656)), ('u', 10, (0.022198914092677623, 0.00756752393486162, 0.014481610731526623)), ('cz', 6, 7), ('u', 6, (-1.8614108876229754, 1.5476460375749002, 1.5998778532511135)), ('u', 7, (1.107571543548716, 1.8503922055304418, -1.4685745838680946)), ('u', 8, (3.122333136909429, 2.36636284057493, 0.02884537539009946)), ('u', 9, (3.1331093701687522, 0.24505547805241654, -3.1342865810756053)), ('u', 10, (0.014481610746405887, 0.022797251634206567, 0.006189292913471925)), ('cz', 7, 8), ('u', 6, (-3.112511191663781, 0.2615124841522484, 3.13626341178677)), ('u', 7, (-3.039370915866766, 0.9553166180914445, 3.063106499388604)), ('u', 8, (3.1704380290308953, 1.5707963265711409, -0.00024991941079388624)), ('u', 9, (3.148898726359738, 0.20452705603326007, -3.117630836274382)), ('u', 10, (0.006189292905740272, 0.035517056468296754, -0.0024996203552288924)), ('cz', 7, 8), ('u', 6, (-3.1634251739360537, 0.20663879733053145, 3.119357510385685)), ('u', 7, (3.067870804228894, 0.7853981633266898, -3.14159265097917)), ('u', 8, (-3.13408354875839, 0.029673366808064105, 3.1379614606673214)), ('u', 9, (-3.115933861798746, 0.1438554085860551, 3.154467081776227)), ('u', 10, (-0.010480646488991493, 0.044056986284693254, -0.020068026447170588)), ('cz', 6, 7), ('u', 6, (-0.02223515910588095, 0.2959491786699587, 0.021100921762980156)), ('u', 7, (2.564204410891336e-09, 0.7853981654143503, 2.4224197534672933e-09)), ('u', 8, (3.137961460587512, 1.5411237331203873, -0.06944019192013995)), ('u', 9, (-3.128718225141107, 0.11132646239649657, 3.1344208404990037)), ('u', 10, (-0.020068026482461674, 0.04610809780819416, -0.02981526170276449)), ('cz', 7, 8), ('u', 6, (0.021100906850089838, 1.8906287889648599, -0.007535173259034228)), ('u', 7, (2.457048822810036e-09, 2.356194490004182, -4.5262644613857766e-09)), ('u', 8, (3.0721524616733773, 1.0230828668750294, -0.0011447520003269318)), ('u', 9, (3.134420840499627, 0.0997000859410616, -3.1457264413875854)), ('u', 10, (-0.029815261649792594, 0.04248209501241417, -0.039623702374196164)), ('cz', 6, 7), ('u', 6, (3.134057480495983, 0.785398163312766, -3.1415926541580825)), ('u', 7, (-4.570539703333689e-09, 1.0471975440745136, -3.1599689872515833e-10)), ('u', 8, (3.1404479015927005, 0.5145185342670128, -3.1395855812423235)), ('u', 9, (-3.1457264415203, 0.0970989005701676, 3.122502996454589)), ('u', 10, (-0.03962370242207246, 0.032658044205827415, -0.049254267926738635)), ('cz', 6, 7), ('u', 6, (-3.1415926542088455, 2.5261129338996686, 3.0751962333179197)), ('u', 7, (-3.1415926538844747, 1.5707963273578327, 3.0751962329597813)), ('u', 8, (-3.1395855813722644, 0.508566592425987, 3.059770192603981)), ('u', 9, (-3.1606823107275073, 0.0980606959949402, 3.1012414353482995)), ('u', 10, (-0.04925426793017285, 0.016203780141577806, -0.05827346509839334)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0))])], (12, 4): [(12, [('u', 5, (0.0, 2.791654562258416, 0.0)), ('u', 4, (0.0, 0.5235987755982989, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (-3.141592653589793, 0.5235987755982989, 3.141592653589793)), ('u', 3, (0.0, 0.9117382909684877, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 0.9117382909684877, 3.141592653589793)), ('u', 2, (0.0, 1.2309594173407745, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.2309594173407745, 3.141592653589793)), ('u', 8, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 10), ('u', 0, (0.0, 0.0, 0.00045385381602142686)), ('u', 1, (-3.141417673305193, 0.5619785451015195, 1.755762792016481)), ('u', 2, (4.750545825083805, 4.894991541664164e-09, -1.2467983469537571)), ('u', 3, (-0.39885316352674083, 0.29250323815455703, -0.5329452114966307)), ('u', 11, (3.1419583075230575, 1.570796379894862, -2.91411540406028)), ('cz', 2, 11), ('u', 0, (0.0, 0.0, 0.0005864400751959172)), ('u', 1, (0.18496555721113828, 0.569613000935121, -1.216118518546765)), ('u', 2, (0.3239974683516449, 1.666901012969065e-08, 3.090440667007608)), ('u', 3, (-0.532945114173757, 0.2972955495177567, -0.7047629298703767)), ('u', 11, (3.3690699031193065, 1.3608147589159625, 0.30667378977101345)), ('cz', 1, 2), ('u', 0, (0.0, 0.0, 0.0004227024855079249)), ('u', 1, (-1.2161170261986696, 0.5363506161282531, 1.2199822496581336)), ('u', 2, (3.09044083783316, 1.5707965313705847, -2.402522628104958)), ('u', 3, (2.4368297170583153, 1.364941020430264, 0.1548888402683941)), ('u', 11, (3.4482664934618774, 0.1639737821371882, -2.7363474607371736)), ('cz', 2, 3), ('u', 0, (0.0, 0.0, 0.0006519017966065135)), ('u', 1, (2.7675598044591725, 0.8625178622721962, -2.8760946734686454)), ('u', 2, (2.402522619906311, 0.7853981426186574, -3.141592651063212)), ('u', 3, (-3.2297463326302203, 0.43808958256232255, 2.942168103316346)), ('u', 11, (-2.7459511899214766, 0.17340911120839114, 3.531467527302679)), ('cz', 1, 2), ('u', 0, (0.0, 0.0, 0.0005219558878084403)), ('u', 1, (-2.876094673468617, 1.8790445107580537e-08, 3.27721024835792)), ('u', 2, (2.52656444069094e-09, 1.5707963055090137, 1.4833835660970393e-07)), ('u', 3, (2.942168086883374, 1.9626307548860298, 0.1175731566088789)), ('u', 11, (-2.7517178305984737, 0.2054198283286131, 3.518766230208988)), ('cz', 2, 3), ('u', 0, (0.0, 0.0, 0.00035603016299620867)), ('u', 1, (0.1356176280203025, 8.187765337271877e-09, -0.2573030607473791)), ('u', 2, (-3.1415927993303154, 1.570796640166298, 2.962271370229391)), ('u', 3, (3.259165810198772, 1.570796326255513, -0.17932124353781242)), ('u', 11, (3.518766695095171, 0.22182393541317091, -2.6816514974783887)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.2630564724597029, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 0.30773985433519363, 0.0)), ('cz', 4, 3), ('u', 3, (-3.141592653589793, 0.30773985433519363, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 1.2630564724597029, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (-3.141592653589793, 1.3605291591529143, 3.141592653589793)), ('cz', 3, 4), ('u', 4, (0.0, 0.2102671676419825, 0.0)), ('cz', 5, 4), ('u', 4, (-3.141592653589793, 0.2102671676419825, 3.141592653589793)), ('cz', 3, 4), ('u', 1, (-1.1016724271811895, 2.5574649132652786e-09, 1.7898550009481864)), ('u', 0, (0.492436999637897, 0.46364761525294385, -1.6816064035924678)), ('u', 2, (0.30196721313550523, 1.32434149825627, 1.3718818156479662)), ('u', 3, (3.130347103302131, 0.15583663848895907, -2.714915839976067)), ('cz', 1, 0), ('u', 1, (-1.367099146386174, 1.5707963165602687, 2.4696154336645475)), ('u', 0, (-1.3054011689183693, 0.5099635913065507, 1.4789061636793646)), ('u', 2, (-2.823328442708983, 1.1287393129341152, 2.630683480757448)), ('u', 3, (0.6078559467018646, 0.27216399250372597, -0.3478222767450082)), ('cz', 1, 2), ('u', 1, (-2.469615428131421, 0.8860771166547974, 3.50488669949602)), ('u', 0, (2.5964209002136824, 0.08576512345534301, -3.2666422677176383)), ('u', 2, (3.3201612531932185, 1.2966149584802662, -0.627654755184281)), ('u', 3, (3.1129239955373142, 0.16267881320005392, -2.83935881579118)), ('cz', 1, 0), ('u', 1, (-0.36329406285149735, 1.570796319297545, 0.0421360315609646)), ('u', 0, (0.03475439363401456, 0.49200093788901084, 0.4282780611812027)), ('u', 2, (3.0733018184492344, 1.3961032537147713, -1.1096675221148782)), ('u', 3, (-2.45762294378278, 0.27159753425019095, 3.1358780209388786)), ('cz', 1, 2), ('u', 1, (-0.5339367917262093, 0.4328088727224011, 0.1820815867386366)), ('u', 0, (3.695557676601006, 0.6667683376977227, -2.4052535566239235)), ('u', 2, (2.900808765005412, 1.5707963535098508, 0.28830100446119244)), ('u', 3, (2.9974287708912692, 1.426067758045666, -0.09448372835273555)), ('cz', 2, 3), ('u', 1, (0.35103960871098244, 1.1920951042601695, -0.5500084230729227)), ('u', 0, (-0.07576803723940528, 0.38421595152640814, 1.01865818719142)), ('u', 2, (-2.5874581356675836, 1.227747145803723, 3.220046396392778)), ('u', 3, (3.1615349967554534, 1.4983077846307333, 0.11252073545068297)), ('cz', 1, 2), ('u', 1, (-0.34035791872224164, 1.882513655029134e-08, -0.2720754280049721)), ('u', 0, (0.0016156368761822137, 0.056178455501631086, -0.374752782608482)), ('u', 2, (-0.24958689277728624, 0.347737012694652, 0.5460941173622462)), ('u', 3, (-0.6112718240968465, 0.08250119589651433, -0.5839190409582741)), ('cz', 2, 3), ('u', 1, (2.0795344138269165, 4.001127514044458e-10, -2.815942107665937)), ('u', 0, (0.3374098642378107, 0.4400437218054845, -0.8555257012929747)), ('u', 2, (2.59549568543325, 0.3477371443355821, -2.796152892796439)), ('u', 3, (-3.5412491074347523, 0.300585177628234, 2.6095270134107267)), ('cz', 1, 2), ('u', 1, (-0.6051170231370915, 0.23170920799201136, -0.19352585580802978)), ('u', 0, (-0.7106275890922221, 0.38646721213968077, 0.7315202922914065)), ('u', 2, (-0.17430419590148566, 1.2277471141063765, -0.10002252923312116)), ('u', 3, (2.868166862299735, 1.3592695643869925, 0.21623953664446738)), ('cz', 2, 3), ('u', 1, (-2.9480665269496917, 0.23170920321285257, 2.968073234694831)), ('u', 0, (3.241534213077716, 0.44748641734593503, -1.9930910874731798)), ('u', 2, (0.19446457534791317, 5.5130786652326745e-08, 0.7532316485939478)), ('u', 3, (3.1405335988123566, 1.5707963325768293, 0.7726821454964128)), ('u', 4, (0.0, 1.3605291591529143, 0.0)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 0.23182380450040305, 0.0)), ('cz', 4, 3), ('u', 3, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 1.3389725222944935, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 1, (-3.0826921333831248, 1.5707963334751514, 3.1415926528874536)), ('u', 0, (0.058900550053380546, 1.5707963575143529, -8.63503485382111e-09)), ('u', 2, (-0.0019510831109328275, 0.014376303182436052, -0.004305090713767894)), ('u', 3, (0.09265607477527761, 0.009466066771348586, 0.06885852248287296)), ('cz', 1, 0), ('u', 1, (-7.186814283079957e-10, 0.5235987759203711, -9.651482007874934e-09)), ('u', 0, (-8.657536741327188e-09, 0.5235987747677111, -0.015177899732086506)), ('u', 2, (-0.0043050907097357954, 0.0016065739921317383, -0.00571042854094426)), ('u', 3, (0.06885852250968953, 0.007279218845029685, 0.03897063224364024)), ('cz', 1, 0), ('u', 1, (-9.676216911058688e-09, 1.5707964014962306, -0.06788249364230205)), ('u', 0, (3.1264147538575164, 1.5412899854085447, -2.932410474816722)), ('u', 2, (3.1358822250129386, 1.5867762299986898, -0.00015315250393621582)), ('u', 3, (0.03897063223836868, 0.0037022790165427673, 0.00507307486164962)), ('cz', 1, 2), ('u', 1, (3.073710159952543, 1.1780971158515567, -3.1415926717628224)), ('u', 0, (-2.932410474378462, 0.0007103469192460386, 3.335488042539339)), ('u', 2, (3.1414395010729192, 1.5707963852692206, -0.04076925033497414)), ('u', 3, (-3.136519578727483, 0.00015074922551734117, 3.1108287335403975)), ('cz', 1, 2), ('u', 1, (-3.1415926717786338, 1.9634952757086144, 3.1415926349746597)), ('u', 0, (-2.9476972628885987, 0.04167158829807069, 3.3274662837868325)), ('u', 2, (3.100823403278013, 0.7897348064105751, -3.1415926343168072)), ('u', 3, (-3.1723565736490853, 0.003639310118667049, 3.075134855695109)), ('cz', 1, 0), ('u', 1, (-1.8615455690818017e-08, 2.748893441017929, -0.0008925942561315802)), ('u', 0, (-2.9557190245652807, 0.08050125978206948, 3.3131477008087495)), ('u', 2, (3.1415926728534456, 2.351857846542888, 0.02835202782950974)), ('u', 3, (-3.208050451470034, 0.006652533605909226, 3.04199986158802)), ('cz', 1, 2), ('u', 1, (3.1407000593176715, 0.2064611020199604, -3.1397479141288764)), ('u', 0, (-2.970037606956515, 0.0540259424448958, 3.3022818236656724)), ('u', 2, (3.169944680885602, 3.1415925902990307, 0.04352197044932149)), ('u', 3, (3.0419998615767048, 1.5804041787190217, -0.0012676214703097877)), ('cz', 2, 3), ('u', 1, (3.143437393030641, 0.18623852297545937, -3.1369442302934076)), ('u', 0, (-2.9809034846880524, 0.03621737249292138, 3.297060955221808)), ('u', 2, (-3.0980706831158735, 2.8794394418002374, 3.115978198773106)), ('u', 3, (3.140325031984367, 1.5649280572380082, -0.005113199710227612)), ('cz', 1, 2), ('u', 1, (-3.136944230269357, 0.01715749606225689, 3.1415939081735074)), ('u', 0, (-2.9861243519203353, 0.024445331181859993, 3.2959920171760237)), ('u', 2, (-0.025614454800490738, 1.832621146533895, -0.09777814682290584)), ('u', 3, (-0.005113199708506954, 0.00586857886423324, -0.0027199540200543637)), ('cz', 2, 3), ('u', 1, (1.2545791676002297e-06, 0.017157497698696213, 0.0008938879255108202)), ('u', 0, (3.2959920194991814, 0.014825856537526326, -2.9860486453782555)), ('u', 2, (3.0438145067873195, 0.2666765505644305, -3.047187772706772)), ('u', 3, (3.1388726995489815, 0.011714929699201501, -3.1189797409697446)), ('cz', 1, 2), ('u', 1, (-3.1406987656411247, 0.0037468355926052036, 3.1402839319002376)), ('u', 0, (3.297136659494896, 0.0037848611924673143, -2.9831878631134883)), ('u', 2, (0.09440488087045854, 1.3086431159403837, 0.018879768007723986)), ('u', 3, (3.1642055662386737, 1.5590933884695546, 0.001748857214283639)), ('cz', 2, 3), ('u', 1, (-1.5721050484941315, 1.5707865649368478, 1.56892292710091)), ('u', 0, (1.7292011189324623, 1.3770072129599067, -1.563730746037875)), ('u', 2, (0.018879768002981405, 0.5810370190019523, -0.020306516875134177)), ('u', 3, (3.1433415107485194, 1.6081338230702829, 0.028439015773600307)), ('cz', 1, 0), ('u', 1, (1.5689229270698437, 0.6154797146445743, -2.0452677126331746)), ('u', 0, (1.5778619075442688, 0.6154797154017753, -1.5748609964779445)), ('u', 2, (-0.02030651687731161, 0.579349440882157, 0.01829161854099471)), ('u', 3, (0.028439015778306227, 0.0316740836365834, 0.021306838709220868)), ('cz', 1, 0), ('u', 1, (1.096324941068762, 1.4371796893872175, -1.6183919581220318)), ('u', 0, (-1.574860996480543, 1.5704900357606484, 1.5495549774468305)), ('u', 2, (3.159884272142458, 0.9928607001248485, -0.03375352484996741)), ('u', 3, (0.02130683869916065, 0.0252724294071901, 0.013880702551721368)), ('cz', 1, 2), ('u', 1, (-3.189188284923943, 0.9619530601374069, 3.0745558809947378)), ('u', 0, (-0.021241349367459993, 0.003783775735988992, -0.03865858416157714)), ('u', 2, (3.1078391288090037, 1.570796333994149, 0.006134673029376092)), ('u', 3, (0.01388070253319873, 0.018138410824286827, 0.0064856668224291605)), ('cz', 1, 2), ('u', 1, (-3.0745560831508625, 0.7853981818164019, 3.141592592061559)), ('u', 0, (0.04768857613836599, 0.00435145842382782, -0.03056591650099847)), ('u', 2, (-3.1318483624011475, 0.041461833554270876, 3.1336781036540917)), ('u', 3, (-3.1373232773103856, 0.0048814553558966895, 3.1395586236367823)), ('cz', 1, 0), ('u', 1, (-6.152601044608126e-08, 0.7853981536866611, 4.6589120071597154e-08)), ('u', 0, (-0.03056591647351238, 0.3231681776359888, -0.02884375484050347)), ('u', 2, (3.1336781036377515, 1.6258022391926685, 0.11082272277844041)), ('u', 3, (-3.1436266835577813, 0.01369225649162358, 3.1339435262723)), ('cz', 1, 2), ('u', 1, (4.657757475881855e-08, 2.356194480312736, 1.6295778389805275e-08)), ('u', 0, (-0.028843754850249554, 1.9213049193774079, 0.011709165929577648)), ('u', 2, (3.2524153763578556, 1.0716444691637244, 0.009739355024963814)), ('u', 3, (-3.1492417809413253, 0.022612223686060758, 3.1293074968996515)), ('cz', 1, 0), ('u', 1, (1.6287097330102835e-08, 1.129762392544152, -5.647100742843332e-09)), ('u', 0, (-3.1298834876426187, 0.7853981666666218, 3.141592640085412)), ('u', 2, (-3.131853298563487, 0.5294978794991645, 3.125022636643347)), ('u', 3, (-3.153877810232764, 0.03144992356975638, 3.12599492077843)), ('cz', 1, 0), ('u', 1, (3.1415926479604073, 1.5707963234655797, -2.990706321975372)), ('u', 0, (3.1415926401135312, 2.649984465268439, -2.9907063395592246)), ('u', 2, (-3.158162670543663, 0.5423096448977569, 3.0905552965946272)), ('u', 3, (-3.157190386416209, 0.04000933418859771, 3.1244620860259964)), ('u', 6, (3.057868776099681, 0.05724484913262502, -2.9033776801798727)), ('u', 7, (1.4862818435911178, 0.6154796928887615, -2.3140382840338285)), ('u', 8, (0.39579705201273097, 4.116944113737925e-09, -1.7541780698269656)), ('u', 9, (3.2522356995007558, 0.39132071997339096, -2.7027640088096607)), ('u', 4, (3.124573395276648, 1.553209204066543, -0.1460152294792283)), ('cz', 7, 8), ('u', 6, (3.539958444290356, 0.5141134315749432, -2.537305937427602)), ('u', 7, (-1.4861691676314916, 0.4278784446668978, 1.4591226047803207)), ('u', 8, (1.652372250155047, 1.5707963180169806, -0.5107049919451496)), ('u', 9, (2.900731050381868, 1.1863815046476784, 0.3737351403897131)), ('u', 4, (3.251737941242348, 0.15717831319804074, -2.957496960474519)), ('cz', 8, 9), ('u', 6, (0.1108205562829287, 0.2336308779787492, 0.4839292991049608)), ('u', 7, (-0.1018890248795074, 0.36994597984753486, -0.6236626236341223)), ('u', 8, (-2.630887630263892, 0.7853981753340872, 2.874456241900612)), ('u', 9, (2.670049329300877, 2.416552856995307, -0.15404404025000984)), ('u', 4, (-2.822513361223004, 0.13037732784991188, 2.9723117844274882)), ('cz', 7, 8), ('u', 6, (-1.7751976671176934, 0.2656615077747423, 3.2856338647777776)), ('u', 7, (2.232187276569363, 0.20609087495514802, -1.5849435883772183)), ('u', 8, (0.26713639305777215, 1.5707963428469054, -0.10493502908089006)), ('u', 9, (3.279540122814995, 1.6312759713558698, 0.024857009497030624)), ('u', 4, (-0.09533309472786339, 0.3378653869050729, -0.334903992866626)), ('cz', 8, 9), ('u', 6, (-2.8987052442483154, 0.16636639308819504, 3.169872799315324)), ('u', 7, (-1.5566490378474866, 0.2060908739493186, 3.6019446138854296)), ('u', 8, (-3.036657650287216, 1.5707963232823523, 2.847601423602299)), ('u', 9, (3.1318318485460965, 1.5707963213328997, -0.29399122710006864)), ('u', 4, (-3.0435496248985334, 0.03976424917099071, 3.17121333009913)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (-3.141592653589793, 1.2630564724597029, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 0.30773985433519363, 0.0)), ('cz', 10, 9), ('u', 9, (-3.141592653589793, 0.30773985433519363, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 1.2630564724597029, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 10, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (-3.141592653589793, 1.3605291591529143, 3.141592653589793)), ('cz', 9, 10), ('u', 10, (0.0, 0.2102671676419825, 0.0)), ('cz', 11, 10), ('u', 10, (-3.141592653589793, 0.2102671676419825, 3.141592653589793)), ('cz', 9, 10), ('u', 8, (0.1547915861858089, 0.870197231041285, -0.05211586217363003)), ('u', 7, (-1.007395850699157, 7.889570568224631e-09, -0.9858327498391138)), ('u', 6, (1.5710448231724112, 0.4636476152020498, -0.5201818243004968)), ('u', 9, (3.128244063954049, 0.11699248646068149, -3.1372656829310883)), ('u', 4, (0.015636983980430716, 0.02151581187648193, 0.012262514570495387)), ('cz', 7, 6), ('u', 8, (-0.05211561296926502, 0.7032821677354445, -0.232410450370457)), ('u', 7, (2.1557599037504738, 1.5707962770946264, -2.237246729502395)), ('u', 6, (2.62141082928903, 0.16502330197726106, -1.7490838336287073)), ('u', 9, (3.145919436683136, 0.066844117957369, -3.1220514279218587)), ('u', 4, (0.012262599671876123, 0.017201897835546213, 0.009019852583398259)), ('cz', 8, 7), ('u', 8, (3.2011121721905083, 0.38403407614316193, -3.050532818711849)), ('u', 7, (2.2372467129518245, 0.8860771495374434, -3.141592617178442)), ('u', 6, (3.1247536746693765, 0.40413472822319985, -3.13461525037419)), ('u', 9, (0.016175775967075763, 0.005271669669242575, 0.01622031323043413)), ('u', 4, (0.008825016566130392, 0.011729236883970906, 0.005734890897182529)), ('cz', 7, 6), ('u', 8, (3.2326522708141305, 2.081431939784217, -0.05374910253038534)), ('u', 7, (-1.8255190289973022e-08, 1.5707963525364104, -0.06987729161151385)), ('u', 6, (3.148570317004371, 0.027371927493456533, -3.0720122043141385)), ('u', 9, (0.01622000098207704, 0.04462282281745182, 0.00703149314816991)), ('u', 4, (0.005735125070307745, 0.007100538104441932, 0.0027798349750419644)), ('cz', 8, 7), ('u', 8, (3.0878435198040632, 1.570796273406513, -0.050796095394392005)), ('u', 7, (3.071715518051413, 0.8731653689595862, -3.033805688941762)), ('u', 6, (3.2111731540926196, 0.007330282162403942, -2.9892752253797217)), ('u', 9, (3.148624353793796, 1.4375396006026198, -0.0077754614818257295)), ('u', 4, (0.0027800951627738714, 0.002386187719455812, -8.053288326428216e-05)), ('cz', 8, 9), ('u', 8, (3.090796593661109, 1.228217405918825, -3.1902309247191436)), ('u', 7, (-3.0338057149615607, 0.7091826877526773, 3.166766741995233)), ('u', 6, (3.2939101866940135, 0.001253031860605077, -2.8765443875689316)), ('u', 9, (3.133817315612175, 1.667528443801526, -0.002027120789804826)), ('u', 4, (-3.1416731604545074, 0.0023971328340698437, 3.1386873950647676)), ('cz', 8, 7), ('u', 8, (-0.04863817562076912, 0.3440810322845707, 4.825265707518456e-07)), ('u', 7, (0.025174179474799538, 6.225748057642174e-07, 0.0836769055433744)), ('u', 6, (3.406640814716529, 0.006578554862773865, -2.7291532677430888)), ('u', 9, (-3.1436198205360024, 0.09673297655059457, 3.1269021993604516)), ('u', 4, (-3.1444978080410473, 0.0072142595798596115, 3.1358190805832833)), ('cz', 8, 9), ('u', 8, (3.14159304170576, 0.34408100348017867, -3.092954816086462)), ('u', 7, (-3.0579158391158074, 6.341439391294435e-07, 3.1824769873549617)), ('u', 6, (3.554031934542796, 0.019449193760943912, -2.557398729135583)), ('u', 9, (3.126902088773573, 0.06659247750936963, -3.13688890290397)), ('u', 4, (-3.147366278632977, 0.012029750068164051, 3.132804229891488)), ('cz', 8, 7), ('u', 8, (0.048637869388173494, 1.2282174050133376, 0.023978839768609955)), ('u', 7, (-3.1007084889518217, 1.7731215380101888e-07, 2.966907740738773)), ('u', 6, (3.725786317032761, 0.03218484816845054, -2.3224558293697504)), ('u', 9, (3.146296525839081, 1.5042066792324484, -0.00031307414856707183)), ('u', 4, (-3.1503808431242746, 0.01680008971571288, 3.1295164594537948)), ('cz', 8, 9), ('u', 8, (0.023978758557944596, 1.455473606693179e-07, -0.057848960710595004)), ('u', 7, (-0.17468486081227602, 1.8587545797242927e-07, -0.277306922135008)), ('u', 6, (3.9607298420882433, 0.0316895145287728, -2.0099953226455796)), ('u', 9, (3.1412796745875893, 1.5707964516605788, -0.05220015129309452)), ('u', 4, (-3.1536689257801025, 0.0214922602347508, 3.125803332161244)), ('u', 10, (0.0, 1.3605291591529143, 0.0)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 10, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 0.23182380450040305, 0.0)), ('cz', 10, 9), ('u', 9, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 1.3389725222944935, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 6, (-0.6144015668824097, 1.5707962656546597, 0.05407076882570355)), ('u', 7, (2.584985717685074, 1.5707963334461452, -3.3696830723855196)), ('u', 8, (-2.2162209881795683, 0.07680661622028434, 3.1992478692747373)), ('u', 5, (2.8410849427121896, 0.030146749559514963, -2.940110668162239)), ('u', 9, (-2.848628801181663, 0.2909980722970483, 3.2580211705172353)), ('cz', 6, 7), ('u', 6, (-0.05407074472123739, 0.5235987798095147, 0.08426475843237556)), ('u', 7, (0.22809041258847643, 0.5235987876001257, 0.4341619676753321)), ('u', 8, (3.4704443019049407, 0.00424674517151328, -2.794448472688792)), ('u', 5, (0.17582584960036074, 0.1727717365844082, 0.06726767749374088)), ('u', 9, (3.601791366936722, 0.0389412543034744, -2.2638269658036934)), ('cz', 6, 7), ('u', 6, (3.0583857339857077, 1.5966033513208107, -2.973113850615029)), ('u', 7, (-0.43416208287936564, 1.570796384254059, -0.1753692836763128)), ('u', 8, (3.7997457771391154, 1.5548160004909632, 0.10180374552695048)), ('u', 5, (-0.03126549411761934, 0.12673289017882486, -0.08981645037239747)), ('u', 9, (0.4407475818569301, 0.3080072789374719, -0.23203821074102635)), ('cz', 7, 8), ('u', 6, (-0.12750692926289814, 0.025828764782617055, 0.045377645580731124)), ('u', 7, (-3.184891133959278, 1.177953949143171, 2.982601937756189)), ('u', 8, (3.0119144128604516, 1.476483543649625, -0.3294826379193012)), ('u', 5, (-3.127520325609462, 0.37643518034688117, 2.739315427074166)), ('u', 9, (3.2360660550713143, 0.09422171765893309, -2.9572198393550946)), ('cz', 6, 7), ('u', 6, (-0.5030382994531521, 0.13883642206688063, 0.09244279759207008)), ('u', 7, (0.22145264976688656, 0.39353326587212223, -0.08374699844158472)), ('u', 8, (-0.1695852312072813, 0.10731986350311833, 0.1766789855016746)), ('u', 5, (-0.19766761034480812, 0.2879861074567509, -0.11205569059419403)), ('u', 9, (0.1292464678907173, 0.32901193723170363, -0.4507385704261851)), ('cz', 7, 8), ('u', 6, (-3.234035468729402, 0.13883640468426028, 2.9998061175589554)), ('u', 7, (-3.057845713771119, 0.3935331721825652, 3.207618643621114)), ('u', 8, (0.06363404955724347, 0.9647249338675202, 0.016061774452762127)), ('u', 5, (-0.04490966875089235, 0.2241120881509651, -0.21928248451703397)), ('u', 9, (0.33576657071032695, 0.4480102748295067, -0.26928960282374753)), ('cz', 6, 7), ('u', 6, (-3.2564530128754634, 0.30057259282255955, 3.0216600479163493)), ('u', 7, (-0.10603645721727831, 1.1486178315451279, -0.17441136170131602)), ('u', 8, (3.125530745627095, 0.9647250811300286, 0.3084504340760761)), ('u', 5, (3.1990131656054266, 0.28380544981295347, -3.064802645513263)), ('u', 9, (0.35450893218270774, 0.4565831733023154, -0.01718681321477486)), ('cz', 8, 9), ('u', 6, (2.870802712132601, 0.09205516206207547, -2.9167347935532035)), ('u', 7, (-0.4425748601739352, 0.035853115130758575, -0.25721435632547685)), ('u', 8, (3.0811280742605893, 2.8797733263263128, -3.1552210037979878)), ('u', 5, (-0.5009569101086845, 0.2928547043842979, 0.011703495934647812)), ('u', 9, (3.4237591982883533, 1.5385918869070603, -0.3303003024004454)), ('cz', 7, 8), ('u', 6, (0.09200086153833811, 0.1449064383152162, -0.3897522872175848)), ('u', 7, (-0.036144957857610446, 5.589587942604881e-07, 0.03216825147518594)), ('u', 8, (0.0014654781692366448, 1.8325971275669297, 0.12315056543219902)), ('u', 5, (-0.3212438703587194, 0.27481478768866596, 0.04330004351292674)), ('u', 9, (0.2770223656924996, 0.032250211298216876, 0.3575939828866548)), ('cz', 8, 9), ('u', 6, (-2.8873292864334257, 0.038823617538546684, 3.23262401940441)), ('u', 7, (3.051763818493768, 5.211298924584333e-07, -3.2215344170802673)), ('u', 8, (2.971480688792345, 0.26207672464309156, -2.9077273588689923)), ('u', 5, (-0.20866396555366085, 0.5916077191255947, 0.09601874654235278)), ('u', 9, (-0.11712411927329239, 0.13615258303286296, -0.3244964849636601)), ('cz', 7, 8), ('u', 6, (0.30971620633886254, 0.33859875108997955, -0.09237075366486151)), ('u', 7, (-0.15939989100049914, 0.4697770453550152, 0.2516723953768456)), ('u', 8, (-0.18850519766525273, 1.3089770173802489, -0.04119589529579018)), ('u', 5, (-2.8205094348390043, 0.6187058838927584, 3.222983951059847)), ('u', 9, (3.273511124158989, 1.7044622707733743, 0.07250067277144479)), ('cz', 8, 9), ('u', 6, (1.7296345355785148, 1.4927985709020386, -1.5427191615038351)), ('u', 7, (-1.7794321602967098, 1.5489575797954507, 1.82688348557566)), ('u', 8, (-3.5044839772037415, 0.7498619318161707, 2.2036263019671383)), ('u', 5, (-2.9382201730369024, 0.343385868746656, 3.138329547507708)), ('u', 9, (3.0434075569979826, 1.6414602415478101, 0.35543752176801746)), ('cz', 6, 7), ('u', 6, (1.4678646897337246, 0.6154797208499309, -1.4671133594454078)), ('u', 7, (1.784016278556056, 0.6154797043861712, -2.4457437336011774)), ('u', 8, (-3.3389279406602164, 0.6519422145201983, 2.669773490860587)), ('u', 5, (-0.8439421144669875, 0.3537846450000773, 0.07225207665278)), ('u', 9, (-3.0203462356507105, 0.4999747327032258, 2.7286921610713524)), ('cz', 6, 7), ('u', 6, (-1.5809990783846237, 1.6239662616260102, 1.3922842113808078)), ('u', 7, (0.8749472633675701, 2.485667779960095, -2.1881868636191544)), ('u', 8, (2.855983252273615, 1.4146241218591151, -0.6456583555520354)), ('u', 5, (-0.3088816496391717, 0.10078358281663227, -0.550218761003495)), ('u', 9, (0.7920486005451626, 0.12832752805055064, 0.2846819747753943)), ('cz', 7, 8), ('u', 6, (2.4157975017911455, 0.05140965298419192, -2.846701970095228)), ('u', 7, (3.1435034847415784, 1.5707963227426673, -2.8501139729796074)), ('u', 8, (2.4762253256526963, 1.9913305164213122, 0.06744725770568438)), ('u', 5, (-0.2539086638550363, 0.1067877263922323, -0.439590009427577)), ('u', 9, (0.1463526001507095, 0.09331853182998076, 0.13203174160015424)), ('cz', 7, 8), ('u', 6, (-2.734516514663416, 0.06300360268141958, 2.067371928377759)), ('u', 7, (2.0647158325924138, 1.5707963206696085, -2.158059640182459)), ('u', 8, (0.5297683891738632, 0.5046295600113868, 1.046192617495422)), ('u', 5, (-0.19994473725627626, 0.2852763887256859, -0.31578003184129366)), ('u', 9, (-2.6196575382093195, 0.2653380868172664, 2.699618311795301)), ('cz', 6, 7), ('u', 6, (-0.2208156022084172, 0.4852860205384801, -0.44966242931948364)), ('u', 7, (0.5872632384695814, 0.7853981482572642, 0.8543981501822199)), ('u', 8, (4.121456852757532, 1.3325516979413137, -0.18224695293372584)), ('u', 5, (-0.29014454208719537, 0.2610813591436803, -0.4390560477320831)), ('u', 9, (-0.33119523340248913, 0.1286820460334461, 0.4426306908478681)), ('cz', 7, 8), ('u', 6, (-0.15326434497243957, 1.1610195447318261, -0.3474686132028678)), ('u', 7, (0.7804523012624992, 1.5068732193940002, -0.6159726449811246)), ('u', 8, (3.265279674847954, 2.2568806676258504, -0.11230689900506485)), ('u', 5, (-0.04923426562691985, 0.5790297353560863, 0.4315288207232498)), ('u', 9, (-0.0038234161274624257, 0.09625799440574824, -0.30488376656223803)), ('cz', 6, 7), ('u', 6, (2.9713584270696165, 1.5707965407710982, -2.5675474729983945)), ('u', 7, (-0.17147417257313669, 0.7853981330980636, 0.12577887579850477)), ('u', 8, (0.06804073485366016, 0.5081509277460858, 0.12313976159056872)), ('u', 5, (-3.566811523106617, 0.4071490919045857, 2.450433210357753)), ('u', 9, (-0.5465540655414279, 0.1942030631313397, 0.37801042183726796)), ('cz', 6, 7), ('u', 6, (-2.1448413099440824, 1.5707963828844467, 3.1893972481646484)), ('u', 7, (3.015813702670151, 1.5707963841645742, -3.0937880859905187)), ('u', 8, (-0.03013030524858923, 0.3775626646589626, -0.42792385609191846)), ('u', 5, (-0.47334119714031, 0.7180627881399289, 0.1218153740432236)), ('u', 9, (-3.0482665213420925, 0.06063876221735686, 2.7356329008928015)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0))]), (12, [('u', 5, (0.0, 2.791654562258416, 0.0)), ('u', 4, (0.0, 0.5235987755982989, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (-3.141592653589793, 0.5235987755982989, 3.141592653589793)), ('u', 3, (0.0, 0.9117382909684877, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 0.9117382909684877, 3.141592653589793)), ('u', 2, (0.0, 1.2309594173407745, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.2309594173407745, 3.141592653589793)), ('u', 8, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 10), ('u', 0, (0.0, 0.0, 0.00045385381602142686)), ('u', 1, (-3.141417673305193, 0.5619785451015195, 1.755762792016481)), ('u', 2, (4.750545825083805, 4.894991541664164e-09, -1.2467983469537571)), ('u', 3, (-0.39885316352674083, 0.29250323815455703, -0.5329452114966307)), ('u', 11, (3.1419583075230575, 1.570796379894862, -2.91411540406028)), ('cz', 2, 11), ('u', 0, (0.0, 0.0, 0.0005864400751959172)), ('u', 1, (0.18496555721113828, 0.569613000935121, -1.216118518546765)), ('u', 2, (0.3239974683516449, 1.666901012969065e-08, 3.090440667007608)), ('u', 3, (-0.532945114173757, 0.2972955495177567, -0.7047629298703767)), ('u', 11, (3.3690699031193065, 1.3608147589159625, 0.30667378977101345)), ('cz', 1, 2), ('u', 0, (0.0, 0.0, 0.0004227024855079249)), ('u', 1, (-1.2161170261986696, 0.5363506161282531, 1.2199822496581336)), ('u', 2, (3.09044083783316, 1.5707965313705847, -2.402522628104958)), ('u', 3, (2.4368297170583153, 1.364941020430264, 0.1548888402683941)), ('u', 11, (3.4482664934618774, 0.1639737821371882, -2.7363474607371736)), ('cz', 2, 3), ('u', 0, (0.0, 0.0, 0.0006519017966065135)), ('u', 1, (2.7675598044591725, 0.8625178622721962, -2.8760946734686454)), ('u', 2, (2.402522619906311, 0.7853981426186574, -3.141592651063212)), ('u', 3, (-3.2297463326302203, 0.43808958256232255, 2.942168103316346)), ('u', 11, (-2.7459511899214766, 0.17340911120839114, 3.531467527302679)), ('cz', 1, 2), ('u', 0, (0.0, 0.0, 0.0005219558878084403)), ('u', 1, (-2.876094673468617, 1.8790445107580537e-08, 3.27721024835792)), ('u', 2, (2.52656444069094e-09, 1.5707963055090137, 1.4833835660970393e-07)), ('u', 3, (2.942168086883374, 1.9626307548860298, 0.1175731566088789)), ('u', 11, (-2.7517178305984737, 0.2054198283286131, 3.518766230208988)), ('cz', 2, 3), ('u', 0, (0.0, 0.0, 0.00035603016299620867)), ('u', 1, (0.1356176280203025, 8.187765337271877e-09, -0.2573030607473791)), ('u', 2, (-3.1415927993303154, 1.570796640166298, 2.962271370229391)), ('u', 3, (3.259165810198772, 1.570796326255513, -0.17932124353781242)), ('u', 11, (3.518766695095171, 0.22182393541317091, -2.6816514974783887)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.2630564724597029, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 0.30773985433519363, 0.0)), ('cz', 4, 3), ('u', 3, (-3.141592653589793, 0.30773985433519363, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 1.2630564724597029, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (-3.141592653589793, 1.3605291591529143, 3.141592653589793)), ('cz', 3, 4), ('u', 4, (0.0, 0.2102671676419825, 0.0)), ('cz', 5, 4), ('u', 4, (-3.141592653589793, 0.2102671676419825, 3.141592653589793)), ('cz', 3, 4), ('u', 1, (-1.1016724271811895, 2.5574649132652786e-09, 1.7898550009481864)), ('u', 0, (0.492436999637897, 0.46364761525294385, -1.6816064035924678)), ('u', 2, (0.30196721313550523, 1.32434149825627, 1.3718818156479662)), ('u', 3, (3.130347103302131, 0.15583663848895907, -2.714915839976067)), ('cz', 1, 0), ('u', 1, (-1.367099146386174, 1.5707963165602687, 2.4696154336645475)), ('u', 0, (-1.3054011689183693, 0.5099635913065507, 1.4789061636793646)), ('u', 2, (-2.823328442708983, 1.1287393129341152, 2.630683480757448)), ('u', 3, (0.6078559467018646, 0.27216399250372597, -0.3478222767450082)), ('cz', 1, 2), ('u', 1, (-2.469615428131421, 0.8860771166547974, 3.50488669949602)), ('u', 0, (2.5964209002136824, 0.08576512345534301, -3.2666422677176383)), ('u', 2, (3.3201612531932185, 1.2966149584802662, -0.627654755184281)), ('u', 3, (3.1129239955373142, 0.16267881320005392, -2.83935881579118)), ('cz', 1, 0), ('u', 1, (-0.36329406285149735, 1.570796319297545, 0.0421360315609646)), ('u', 0, (0.03475439363401456, 0.49200093788901084, 0.4282780611812027)), ('u', 2, (3.0733018184492344, 1.3961032537147713, -1.1096675221148782)), ('u', 3, (-2.45762294378278, 0.27159753425019095, 3.1358780209388786)), ('cz', 1, 2), ('u', 1, (-0.5339367917262093, 0.4328088727224011, 0.1820815867386366)), ('u', 0, (3.695557676601006, 0.6667683376977227, -2.4052535566239235)), ('u', 2, (2.900808765005412, 1.5707963535098508, 0.28830100446119244)), ('u', 3, (2.9974287708912692, 1.426067758045666, -0.09448372835273555)), ('cz', 2, 3), ('u', 1, (0.35103960871098244, 1.1920951042601695, -0.5500084230729227)), ('u', 0, (-0.07576803723940528, 0.38421595152640814, 1.01865818719142)), ('u', 2, (-2.5874581356675836, 1.227747145803723, 3.220046396392778)), ('u', 3, (3.1615349967554534, 1.4983077846307333, 0.11252073545068297)), ('cz', 1, 2), ('u', 1, (-0.34035791872224164, 1.882513655029134e-08, -0.2720754280049721)), ('u', 0, (0.0016156368761822137, 0.056178455501631086, -0.374752782608482)), ('u', 2, (-0.24958689277728624, 0.347737012694652, 0.5460941173622462)), ('u', 3, (-0.6112718240968465, 0.08250119589651433, -0.5839190409582741)), ('cz', 2, 3), ('u', 1, (2.0795344138269165, 4.001127514044458e-10, -2.815942107665937)), ('u', 0, (0.3374098642378107, 0.4400437218054845, -0.8555257012929747)), ('u', 2, (2.59549568543325, 0.3477371443355821, -2.796152892796439)), ('u', 3, (-3.5412491074347523, 0.300585177628234, 2.6095270134107267)), ('cz', 1, 2), ('u', 1, (-0.6051170231370915, 0.23170920799201136, -0.19352585580802978)), ('u', 0, (-0.7106275890922221, 0.38646721213968077, 0.7315202922914065)), ('u', 2, (-0.17430419590148566, 1.2277471141063765, -0.10002252923312116)), ('u', 3, (2.868166862299735, 1.3592695643869925, 0.21623953664446738)), ('cz', 2, 3), ('u', 1, (-2.9480665269496917, 0.23170920321285257, 2.968073234694831)), ('u', 0, (3.241534213077716, 0.44748641734593503, -1.9930910874731798)), ('u', 2, (0.19446457534791317, 5.5130786652326745e-08, 0.7532316485939478)), ('u', 3, (3.1405335988123566, 1.5707963325768293, 0.7726821454964128)), ('u', 4, (0.0, 1.3605291591529143, 0.0)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 0.23182380450040305, 0.0)), ('cz', 4, 3), ('u', 3, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 1.3389725222944935, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 6, (3.057868776099681, 0.05724484913262502, -2.9033776801798727)), ('u', 7, (1.4862818435911178, 0.6154796928887615, -2.3140382840338285)), ('u', 8, (0.39579705201273097, 4.116944113737925e-09, -1.7541780698269656)), ('u', 9, (3.2522356995007558, 0.39132071997339096, -2.7027640088096607)), ('u', 4, (3.124573395276648, 1.553209204066543, -0.1460152294792283)), ('cz', 7, 8), ('u', 6, (3.539958444290356, 0.5141134315749432, -2.537305937427602)), ('u', 7, (-1.4861691676314916, 0.4278784446668978, 1.4591226047803207)), ('u', 8, (1.652372250155047, 1.5707963180169806, -0.5107049919451496)), ('u', 9, (2.900731050381868, 1.1863815046476784, 0.3737351403897131)), ('u', 4, (3.251737941242348, 0.15717831319804074, -2.957496960474519)), ('cz', 8, 9), ('u', 6, (0.1108205562829287, 0.2336308779787492, 0.4839292991049608)), ('u', 7, (-0.1018890248795074, 0.36994597984753486, -0.6236626236341223)), ('u', 8, (-2.630887630263892, 0.7853981753340872, 2.874456241900612)), ('u', 9, (2.670049329300877, 2.416552856995307, -0.15404404025000984)), ('u', 4, (-2.822513361223004, 0.13037732784991188, 2.9723117844274882)), ('cz', 7, 8), ('u', 6, (-1.7751976671176934, 0.2656615077747423, 3.2856338647777776)), ('u', 7, (2.232187276569363, 0.20609087495514802, -1.5849435883772183)), ('u', 8, (0.26713639305777215, 1.5707963428469054, -0.10493502908089006)), ('u', 9, (3.279540122814995, 1.6312759713558698, 0.024857009497030624)), ('u', 4, (-0.09533309472786339, 0.3378653869050729, -0.334903992866626)), ('cz', 8, 9), ('u', 6, (-2.8987052442483154, 0.16636639308819504, 3.169872799315324)), ('u', 7, (-1.5566490378474866, 0.2060908739493186, 3.6019446138854296)), ('u', 8, (-3.036657650287216, 1.5707963232823523, 2.847601423602299)), ('u', 9, (3.1318318485460965, 1.5707963213328997, -0.29399122710006864)), ('u', 4, (-3.0435496248985334, 0.03976424917099071, 3.17121333009913)), ('u', 0, (0.0, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.5235987755982989, 0.0)), ('u', 0, (0.0, 0.5235987755982989, 0.0)), ('cz', 0, 1), ('u', 1, (0.0, 1.5707963267948966, 0.0)), ('u', 0, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.1780972450961724, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.39269908169872414, 0.0)), ('cz', 2, 1), ('u', 1, (-3.141592653589793, 0.39269908169872414, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 1.1780972450961724, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.3089969389957472, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 0.26179938779914935, 0.0)), ('cz', 3, 2), ('u', 2, (-3.141592653589793, 0.26179938779914935, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 1.3089969389957472, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 1, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 0, 1), ('u', 1, (1.570796326794898, 0.6154797086703868, -1.570796326794898)), ('u', 0, (1.5707963267948966, 0.6154797086703876, -1.5707963267948966)), ('cz', 0, 1), ('u', 1, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 0, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 1, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.2630564724597029, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.30773985433519363, 0.0)), ('cz', 2, 1), ('u', 1, (-3.141592653589793, 0.30773985433519363, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 1.2630564724597029, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (0.0, 1.5707963267948966, 0.0)), ('u', 1, (0.0, 1.5707963267948966, 0.0)), ('cz', 0, 1), ('u', 1, (1.2251903831398114e-16, 0.7853981633974483, -1.1496735851465472e-17)), ('u', 0, (-3.141592653589793, 0.7853981633974483, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 0, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (-3.141592653589793, 1.2630564724597029, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 0.30773985433519363, 0.0)), ('cz', 10, 9), ('u', 9, (-3.141592653589793, 0.30773985433519363, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 1.2630564724597029, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 10, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (-3.141592653589793, 1.3605291591529143, 3.141592653589793)), ('cz', 9, 10), ('u', 10, (0.0, 0.2102671676419825, 0.0)), ('cz', 11, 10), ('u', 10, (-3.141592653589793, 0.2102671676419825, 3.141592653589793)), ('cz', 9, 10), ('u', 8, (0.1547915861858089, 0.870197231041285, -0.05211586217363003)), ('u', 7, (-1.007395850699157, 7.889570568224631e-09, -0.9858327498391138)), ('u', 6, (1.5710448231724112, 0.4636476152020498, -0.5201818243004968)), ('u', 9, (3.128244063954049, 0.11699248646068149, -3.1372656829310883)), ('u', 4, (0.015636983980430716, 0.02151581187648193, 0.012262514570495387)), ('cz', 7, 6), ('u', 8, (-0.05211561296926502, 0.7032821677354445, -0.232410450370457)), ('u', 7, (2.1557599037504738, 1.5707962770946264, -2.237246729502395)), ('u', 6, (2.62141082928903, 0.16502330197726106, -1.7490838336287073)), ('u', 9, (3.145919436683136, 0.066844117957369, -3.1220514279218587)), ('u', 4, (0.012262599671876123, 0.017201897835546213, 0.009019852583398259)), ('cz', 8, 7), ('u', 8, (3.2011121721905083, 0.38403407614316193, -3.050532818711849)), ('u', 7, (2.2372467129518245, 0.8860771495374434, -3.141592617178442)), ('u', 6, (3.1247536746693765, 0.40413472822319985, -3.13461525037419)), ('u', 9, (0.016175775967075763, 0.005271669669242575, 0.01622031323043413)), ('u', 4, (0.008825016566130392, 0.011729236883970906, 0.005734890897182529)), ('cz', 7, 6), ('u', 8, (3.2326522708141305, 2.081431939784217, -0.05374910253038534)), ('u', 7, (-1.8255190289973022e-08, 1.5707963525364104, -0.06987729161151385)), ('u', 6, (3.148570317004371, 0.027371927493456533, -3.0720122043141385)), ('u', 9, (0.01622000098207704, 0.04462282281745182, 0.00703149314816991)), ('u', 4, (0.005735125070307745, 0.007100538104441932, 0.0027798349750419644)), ('cz', 8, 7), ('u', 8, (3.0878435198040632, 1.570796273406513, -0.050796095394392005)), ('u', 7, (3.071715518051413, 0.8731653689595862, -3.033805688941762)), ('u', 6, (3.2111731540926196, 0.007330282162403942, -2.9892752253797217)), ('u', 9, (3.148624353793796, 1.4375396006026198, -0.0077754614818257295)), ('u', 4, (0.0027800951627738714, 0.002386187719455812, -8.053288326428216e-05)), ('cz', 8, 9), ('u', 8, (3.090796593661109, 1.228217405918825, -3.1902309247191436)), ('u', 7, (-3.0338057149615607, 0.7091826877526773, 3.166766741995233)), ('u', 6, (3.2939101866940135, 0.001253031860605077, -2.8765443875689316)), ('u', 9, (3.133817315612175, 1.667528443801526, -0.002027120789804826)), ('u', 4, (-3.1416731604545074, 0.0023971328340698437, 3.1386873950647676)), ('cz', 8, 7), ('u', 8, (-0.04863817562076912, 0.3440810322845707, 4.825265707518456e-07)), ('u', 7, (0.025174179474799538, 6.225748057642174e-07, 0.0836769055433744)), ('u', 6, (3.406640814716529, 0.006578554862773865, -2.7291532677430888)), ('u', 9, (-3.1436198205360024, 0.09673297655059457, 3.1269021993604516)), ('u', 4, (-3.1444978080410473, 0.0072142595798596115, 3.1358190805832833)), ('cz', 8, 9), ('u', 8, (3.14159304170576, 0.34408100348017867, -3.092954816086462)), ('u', 7, (-3.0579158391158074, 6.341439391294435e-07, 3.1824769873549617)), ('u', 6, (3.554031934542796, 0.019449193760943912, -2.557398729135583)), ('u', 9, (3.126902088773573, 0.06659247750936963, -3.13688890290397)), ('u', 4, (-3.147366278632977, 0.012029750068164051, 3.132804229891488)), ('cz', 8, 7), ('u', 8, (0.048637869388173494, 1.2282174050133376, 0.023978839768609955)), ('u', 7, (-3.1007084889518217, 1.7731215380101888e-07, 2.966907740738773)), ('u', 6, (3.725786317032761, 0.03218484816845054, -2.3224558293697504)), ('u', 9, (3.146296525839081, 1.5042066792324484, -0.00031307414856707183)), ('u', 4, (-3.1503808431242746, 0.01680008971571288, 3.1295164594537948)), ('cz', 8, 9), ('u', 8, (0.023978758557944596, 1.455473606693179e-07, -0.057848960710595004)), ('u', 7, (-0.17468486081227602, 1.8587545797242927e-07, -0.277306922135008)), ('u', 6, (3.9607298420882433, 0.0316895145287728, -2.0099953226455796)), ('u', 9, (3.1412796745875893, 1.5707964516605788, -0.05220015129309452)), ('u', 4, (-3.1536689257801025, 0.0214922602347508, 3.125803332161244)), ('u', 10, (0.0, 1.3605291591529143, 0.0)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 10, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 0.23182380450040305, 0.0)), ('cz', 10, 9), ('u', 9, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 1.3389725222944935, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 6, (-0.6144015668824097, 1.5707962656546597, 0.05407076882570355)), ('u', 7, (2.584985717685074, 1.5707963334461452, -3.3696830723855196)), ('u', 8, (-2.2162209881795683, 0.07680661622028434, 3.1992478692747373)), ('u', 5, (2.8410849427121896, 0.030146749559514963, -2.940110668162239)), ('u', 9, (-2.848628801181663, 0.2909980722970483, 3.2580211705172353)), ('cz', 6, 7), ('u', 6, (-0.05407074472123739, 0.5235987798095147, 0.08426475843237556)), ('u', 7, (0.22809041258847643, 0.5235987876001257, 0.4341619676753321)), ('u', 8, (3.4704443019049407, 0.00424674517151328, -2.794448472688792)), ('u', 5, (0.17582584960036074, 0.1727717365844082, 0.06726767749374088)), ('u', 9, (3.601791366936722, 0.0389412543034744, -2.2638269658036934)), ('cz', 6, 7), ('u', 6, (3.0583857339857077, 1.5966033513208107, -2.973113850615029)), ('u', 7, (-0.43416208287936564, 1.570796384254059, -0.1753692836763128)), ('u', 8, (3.7997457771391154, 1.5548160004909632, 0.10180374552695048)), ('u', 5, (-0.03126549411761934, 0.12673289017882486, -0.08981645037239747)), ('u', 9, (0.4407475818569301, 0.3080072789374719, -0.23203821074102635)), ('cz', 7, 8), ('u', 6, (-0.12750692926289814, 0.025828764782617055, 0.045377645580731124)), ('u', 7, (-3.184891133959278, 1.177953949143171, 2.982601937756189)), ('u', 8, (3.0119144128604516, 1.476483543649625, -0.3294826379193012)), ('u', 5, (-3.127520325609462, 0.37643518034688117, 2.739315427074166)), ('u', 9, (3.2360660550713143, 0.09422171765893309, -2.9572198393550946)), ('cz', 6, 7), ('u', 6, (-0.5030382994531521, 0.13883642206688063, 0.09244279759207008)), ('u', 7, (0.22145264976688656, 0.39353326587212223, -0.08374699844158472)), ('u', 8, (-0.1695852312072813, 0.10731986350311833, 0.1766789855016746)), ('u', 5, (-0.19766761034480812, 0.2879861074567509, -0.11205569059419403)), ('u', 9, (0.1292464678907173, 0.32901193723170363, -0.4507385704261851)), ('cz', 7, 8), ('u', 6, (-3.234035468729402, 0.13883640468426028, 2.9998061175589554)), ('u', 7, (-3.057845713771119, 0.3935331721825652, 3.207618643621114)), ('u', 8, (0.06363404955724347, 0.9647249338675202, 0.016061774452762127)), ('u', 5, (-0.04490966875089235, 0.2241120881509651, -0.21928248451703397)), ('u', 9, (0.33576657071032695, 0.4480102748295067, -0.26928960282374753)), ('cz', 6, 7), ('u', 6, (-3.2564530128754634, 0.30057259282255955, 3.0216600479163493)), ('u', 7, (-0.10603645721727831, 1.1486178315451279, -0.17441136170131602)), ('u', 8, (3.125530745627095, 0.9647250811300286, 0.3084504340760761)), ('u', 5, (3.1990131656054266, 0.28380544981295347, -3.064802645513263)), ('u', 9, (0.35450893218270774, 0.4565831733023154, -0.01718681321477486)), ('cz', 8, 9), ('u', 6, (2.870802712132601, 0.09205516206207547, -2.9167347935532035)), ('u', 7, (-0.4425748601739352, 0.035853115130758575, -0.25721435632547685)), ('u', 8, (3.0811280742605893, 2.8797733263263128, -3.1552210037979878)), ('u', 5, (-0.5009569101086845, 0.2928547043842979, 0.011703495934647812)), ('u', 9, (3.4237591982883533, 1.5385918869070603, -0.3303003024004454)), ('cz', 7, 8), ('u', 6, (0.09200086153833811, 0.1449064383152162, -0.3897522872175848)), ('u', 7, (-0.036144957857610446, 5.589587942604881e-07, 0.03216825147518594)), ('u', 8, (0.0014654781692366448, 1.8325971275669297, 0.12315056543219902)), ('u', 5, (-0.3212438703587194, 0.27481478768866596, 0.04330004351292674)), ('u', 9, (0.2770223656924996, 0.032250211298216876, 0.3575939828866548)), ('cz', 8, 9), ('u', 6, (-2.8873292864334257, 0.038823617538546684, 3.23262401940441)), ('u', 7, (3.051763818493768, 5.211298924584333e-07, -3.2215344170802673)), ('u', 8, (2.971480688792345, 0.26207672464309156, -2.9077273588689923)), ('u', 5, (-0.20866396555366085, 0.5916077191255947, 0.09601874654235278)), ('u', 9, (-0.11712411927329239, 0.13615258303286296, -0.3244964849636601)), ('cz', 7, 8), ('u', 6, (0.30971620633886254, 0.33859875108997955, -0.09237075366486151)), ('u', 7, (-0.15939989100049914, 0.4697770453550152, 0.2516723953768456)), ('u', 8, (-0.18850519766525273, 1.3089770173802489, -0.04119589529579018)), ('u', 5, (-2.8205094348390043, 0.6187058838927584, 3.222983951059847)), ('u', 9, (3.273511124158989, 1.7044622707733743, 0.07250067277144479)), ('cz', 8, 9), ('u', 6, (1.7296345355785148, 1.4927985709020386, -1.5427191615038351)), ('u', 7, (-1.7794321602967098, 1.5489575797954507, 1.82688348557566)), ('u', 8, (-3.5044839772037415, 0.7498619318161707, 2.2036263019671383)), ('u', 5, (-2.9382201730369024, 0.343385868746656, 3.138329547507708)), ('u', 9, (3.0434075569979826, 1.6414602415478101, 0.35543752176801746)), ('cz', 6, 7), ('u', 6, (1.4678646897337246, 0.6154797208499309, -1.4671133594454078)), ('u', 7, (1.784016278556056, 0.6154797043861712, -2.4457437336011774)), ('u', 8, (-3.3389279406602164, 0.6519422145201983, 2.669773490860587)), ('u', 5, (-0.8439421144669875, 0.3537846450000773, 0.07225207665278)), ('u', 9, (-3.0203462356507105, 0.4999747327032258, 2.7286921610713524)), ('cz', 6, 7), ('u', 6, (-1.5809990783846237, 1.6239662616260102, 1.3922842113808078)), ('u', 7, (0.8749472633675701, 2.485667779960095, -2.1881868636191544)), ('u', 8, (2.855983252273615, 1.4146241218591151, -0.6456583555520354)), ('u', 5, (-0.3088816496391717, 0.10078358281663227, -0.550218761003495)), ('u', 9, (0.7920486005451626, 0.12832752805055064, 0.2846819747753943)), ('cz', 7, 8), ('u', 6, (2.4157975017911455, 0.05140965298419192, -2.846701970095228)), ('u', 7, (3.1435034847415784, 1.5707963227426673, -2.8501139729796074)), ('u', 8, (2.4762253256526963, 1.9913305164213122, 0.06744725770568438)), ('u', 5, (-0.2539086638550363, 0.1067877263922323, -0.439590009427577)), ('u', 9, (0.1463526001507095, 0.09331853182998076, 0.13203174160015424)), ('cz', 7, 8), ('u', 6, (-2.734516514663416, 0.06300360268141958, 2.067371928377759)), ('u', 7, (2.0647158325924138, 1.5707963206696085, -2.158059640182459)), ('u', 8, (0.5297683891738632, 0.5046295600113868, 1.046192617495422)), ('u', 5, (-0.19994473725627626, 0.2852763887256859, -0.31578003184129366)), ('u', 9, (-2.6196575382093195, 0.2653380868172664, 2.699618311795301)), ('cz', 6, 7), ('u', 6, (-0.2208156022084172, 0.4852860205384801, -0.44966242931948364)), ('u', 7, (0.5872632384695814, 0.7853981482572642, 0.8543981501822199)), ('u', 8, (4.121456852757532, 1.3325516979413137, -0.18224695293372584)), ('u', 5, (-0.29014454208719537, 0.2610813591436803, -0.4390560477320831)), ('u', 9, (-0.33119523340248913, 0.1286820460334461, 0.4426306908478681)), ('cz', 7, 8), ('u', 6, (-0.15326434497243957, 1.1610195447318261, -0.3474686132028678)), ('u', 7, (0.7804523012624992, 1.5068732193940002, -0.6159726449811246)), ('u', 8, (3.265279674847954, 2.2568806676258504, -0.11230689900506485)), ('u', 5, (-0.04923426562691985, 0.5790297353560863, 0.4315288207232498)), ('u', 9, (-0.0038234161274624257, 0.09625799440574824, -0.30488376656223803)), ('cz', 6, 7), ('u', 6, (2.9713584270696165, 1.5707965407710982, -2.5675474729983945)), ('u', 7, (-0.17147417257313669, 0.7853981330980636, 0.12577887579850477)), ('u', 8, (0.06804073485366016, 0.5081509277460858, 0.12313976159056872)), ('u', 5, (-3.566811523106617, 0.4071490919045857, 2.450433210357753)), ('u', 9, (-0.5465540655414279, 0.1942030631313397, 0.37801042183726796)), ('cz', 6, 7), ('u', 6, (-2.1448413099440824, 1.5707963828844467, 3.1893972481646484)), ('u', 7, (3.015813702670151, 1.5707963841645742, -3.0937880859905187)), ('u', 8, (-0.03013030524858923, 0.3775626646589626, -0.42792385609191846)), ('u', 5, (-0.47334119714031, 0.7180627881399289, 0.1218153740432236)), ('u', 9, (-3.0482665213420925, 0.06063876221735686, 2.7356329008928015)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0))])], (12, 6): [(12, [('u', 5, (0.0, 3.0757856104947443, 0.0)), ('u', 4, (0.0, 0.19879928334601105, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (-3.141592653589793, 0.19879928334601105, 3.141592653589793)), ('u', 3, (0.0, 0.5278194054589851, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 0.5278194054589851, 3.141592653589793)), ('u', 2, (0.0, 0.8903978130278803, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 0.8903978130278803, 3.141592653589793)), ('u', 1, (0.0, 1.185542550497017, 0.0)), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 1), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.185542550497017, 3.141592653589793)), ('u', 0, (0.0, 1.4056476493802696, 0.0)), ('u', 0, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 0), ('u', 0, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (-3.141592653589793, 1.4056476493802696, 3.141592653589793)), ('u', 6, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 6), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 7), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 11, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 0, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (0.0, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (-6.026816351730587e-17, 0.4205343352839649, 4.75701228604804e-18)), ('u', 0, (-6.026816351730587e-17, 0.4205343352839649, 4.75701228604804e-18)), ('cz', 0, 1), ('u', 1, (0.0, 1.5707963267948966, 0.0)), ('u', 0, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.093138017732642, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.4776583090622546, 0.0)), ('cz', 2, 1), ('u', 1, (-3.141592653589793, 0.4776583090622546, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 1.093138017732642, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.1780972450961724, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 0.39269908169872414, 0.0)), ('cz', 3, 2), ('u', 2, (-3.141592653589793, 0.39269908169872414, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 1.1780972450961724, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.2630564724597029, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 0.30773985433519363, 0.0)), ('cz', 4, 3), ('u', 3, (-3.141592653589793, 0.30773985433519363, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 1.2630564724597029, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (-3.141592653589793, 1.3605291591529143, 3.141592653589793)), ('cz', 3, 4), ('u', 4, (0.0, 0.2102671676419825, 0.0)), ('cz', 5, 4), ('u', 4, (-3.141592653589793, 0.2102671676419825, 3.141592653589793)), ('cz', 3, 4), ('u', 4, (0.0, 1.3605291591529143, 0.0)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 1, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 0, 1), ('u', 1, (1.570796326794898, 0.4636476090008061, -1.570796326794898)), ('u', 0, (1.5707963267948966, 0.4636476090008061, -1.5707963267948966)), ('cz', 0, 1), ('u', 1, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 0, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 1, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.1277577648985897, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.44303856189630686, 0.0)), ('cz', 2, 1), ('u', 1, (-3.141592653589793, 0.44303856189630686, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 1.1277577648985897, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.2284367252937551, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 0.3423596015011414, 0.0)), ('cz', 3, 2), ('u', 2, (-3.141592653589793, 0.3423596015011414, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 1.2284367252937551, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 0.23182380450040305, 0.0)), ('cz', 4, 3), ('u', 3, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 1.3389725222944935, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 1, (1.363845864587723, 1.5707963267672458, -2.9993854688414716)), ('u', 0, (1.363845864586009, 1.5707963268378884, 1.2639026291583169)), ('u', 2, (-3.2071888465277136, 0.184904712775633, 2.8610451944514357)), ('u', 3, (0.5385123367231761, 0.06467488581738873, 0.04937591114067358)), ('u', 11, (1.6992100207502157, 0.23360903143076647, 0.5579594806284142)), ('cz', 1, 0), ('u', 1, (-0.14220718475653546, 0.5235987755985089, -0.41745071061119066)), ('u', 0, (1.877690024460379, 2.6179938779918035, -0.03580353792904112)), ('u', 2, (0.4800735699666825, 0.2897401439513171, 0.14878724963518253)), ('u', 3, (-3.013747770360676, 0.03451614731288748, 3.003756469721795)), ('u', 11, (0.2185318996508333, 0.1328640734067659, -0.2804221915090717)), ('cz', 1, 0), ('u', 1, (0.4174507106123065, 1.5707963268071967, -0.47126800775294303)), ('u', 0, (3.1470202037463095, 1.6882687876873737, -2.7231476408232136)), ('u', 2, (3.238386628155749, 1.6848016324293131, -0.06463740846983468)), ('u', 3, (-0.08956681406336006, 0.06641496875959405, -0.00173379117006673)), ('u', 11, (3.290510076704486, 0.3843153515279157, -2.2473537186376977)), ('cz', 1, 2), ('u', 1, (2.5760504070058876, 0.39269908170299855, -2.9988382870440558)), ('u', 0, (-2.7625746621013945, 0.0950651554024133, 2.371617425516931)), ('u', 2, (3.1971735290592127, 1.5707963268013454, -0.21746618356704195)), ('u', 3, (2.817035250472036, 0.4816552162375767, -2.8307390933588885)), ('u', 11, (-3.03777683654344, 0.21866970396518934, 3.043424696251818)), ('cz', 1, 2), ('u', 1, (2.998838287025793, 0.39269908169903683, -3.0143314903138596)), ('u', 0, (0.1715106168685091, 0.18755420994158523, 0.4161264827006588)), ('u', 2, (0.05982400444235608, 1.0485062235908116e-10, 0.28710524803901055)), ('u', 3, (2.8908380088425445, 0.12446905701880381, -2.8730332913397967)), ('u', 11, (-0.17878558894341473, 0.05048690051415294, 0.14584352443348808)), ('cz', 1, 0), ('u', 1, (-0.1272611631371841, 1.178097245110616, 0.043673916629391046)), ('u', 0, (0.003818572825657514, 0.2467105513567965, -0.06941740237412158)), ('u', 2, (2.644326099731402, 6.954531877379686e-11, -1.0548234819637168)), ('u', 3, (0.18491647072024606, 0.24436291923884504, -0.25004419633888014)), ('u', 11, (-0.03550929641403387, 0.4030732709508788, 0.2678597573031803)), ('cz', 1, 2), ('u', 1, (-3.1228504119408154, 1.030966922621024, 2.9453401058764124)), ('u', 0, (0.1179962239684767, 0.23548435679138233, 0.1527200841677004)), ('u', 2, (2.449394360105412, 3.1415926535678182, -0.0922889975238399)), ('u', 3, (2.540224590607155, 1.2430644716230643, 0.34042883892006603)), ('u', 11, (-2.8597740722484075, 0.28847312567924255, 2.6216845541657174)), ('cz', 2, 3), ('u', 1, (-3.017053450194865, 0.9344486059450692, 2.4728430001260175)), ('u', 0, (0.5054443303789188, 0.16415946838028975, 0.530326491748032)), ('u', 2, (-2.6307929107371804, 2.87907570840622, 3.2359900602319955)), ('u', 3, (2.7974402917130727, 1.4766533627026013, -0.20887776650500722)), ('u', 11, (-2.9456829708982157, 0.316226895673598, 2.651044412584663)), ('cz', 1, 2), ('u', 1, (3.0878665036375486, 2.1702215788789444e-10, -3.0036321220042597)), ('u', 0, (-2.6810618737305143, 0.15689068523105396, 2.926997184274815)), ('u', 2, (-0.16730747595939177, 1.8326473121711249, -0.38983431787754164)), ('u', 3, (-0.00923206423481826, 0.09641354025270232, -0.2687390618669404)), ('u', 11, (0.004171154571400146, 0.3169631030215196, -0.035171437866080006)), ('cz', 2, 3), ('u', 1, (0.0205047673468016, 1.7716992026397947e-10, -0.01402221768342051)), ('u', 0, (0.10901292728483414, 0.11595455849628988, 0.28503099895241835)), ('u', 2, (3.256426824750352, 0.27158740610627147, -2.7011709482222255)), ('u', 3, (-0.2014546693702715, 0.21615536475340474, 0.44973465481447467)), ('u', 11, (3.1168657275130505, 0.009097770850434048, -2.8631452225525758)), ('cz', 1, 2), ('u', 1, (0.08936923612351227, 0.0006266391039143163, -0.6256947727937483)), ('u', 0, (0.3127772484991065, 0.4425334180787224, -0.7738530006778277)), ('u', 2, (-0.17502644640991238, 1.3082793816134326, -0.4711828139334281)), ('u', 3, (3.4163064623247226, 1.733777998216237, 0.01266097499274732)), ('u', 11, (-0.34596199000980177, 0.14386738279590947, 0.6250437041635831)), ('cz', 2, 3), ('u', 1, (-2.0399542004731384, 1.5713533217248365, 1.7013097330827514)), ('u', 0, (1.6166016491329345, 1.9347304860345607, -1.773316789719498)), ('u', 2, (-0.4386484124822272, 0.0015522740078519823, 0.12839069158283042)), ('u', 3, (3.375586976992392, 1.7580043569266424, 0.13266390002743167)), ('u', 11, (3.0497594033307784, 0.40527541662488026, -2.967944020089303)), ('cz', 1, 0), ('u', 1, (1.4405700323809594, 2.5261129449188844, -1.0736099945134443)), ('u', 0, (-1.5307563786235483, 0.7083937952823107, 1.094342241382531)), ('u', 2, (3.1386558134130316, 1.5723364014655843, 0.6539020735549244)), ('u', 3, (2.7090304069013973, 0.0018120668535631193, -2.798423756554544)), ('u', 11, (-3.1526955155722445, 0.4136623979181174, 2.912043609045781)), ('cz', 1, 2), ('u', 1, (-3.023299277215143, 1.5707963267863896, 2.6032780596287983)), ('u', 0, (2.9921942707562117, 0.18474840197135983, -2.805109016014484)), ('u', 2, (3.255975628587858, 1.719373693642963, -0.28321952549538576)), ('u', 3, (0.07153529109902992, 0.3132596387815738, 0.037483751637371485)), ('u', 11, (0.3289736344727156, 0.5185160451997999, -0.052674441438991804)), ('cz', 1, 0), ('u', 1, (-1.0324817328297038, 1.5707963267998148, 0.3639224763010943)), ('u', 0, (0.05440250129871599, 0.5293683032878496, -0.35793546408076754)), ('u', 2, (0.007885513401241365, 0.2833965772722029, 0.03910361915813132)), ('u', 3, (2.845935794416537, 0.23002941200652896, -3.096456826830062)), ('u', 11, (0.04657467538722733, 0.10198091863099273, 0.18825217824805818)), ('cz', 1, 2), ('u', 1, (0.0, 1.1383257248941486e-12, 0.233665301559387)), ('u', 0, (2.415982008321575, 0.92714299809076, -2.995458349540593)), ('u', 2, (-0.34092652257087946, 0.018534898805743556, 0.0986872907939854)), ('u', 3, (0.17707817274062912, 0.4845914864344751, 0.4828603407009331)), ('u', 11, (0.14622383230920563, 0.3504973020598406, 0.5319535697949473)), ('cz', 1, 0), ('u', 1, (0.18781038554835428, 1.5707963268007776, 0.7290114095506618)), ('u', 0, (0.4613575044770586, 1.5387056026239314, 0.9892258188675678)), ('u', 2, (3.180073602453756, 1.5891571698882074, 0.08911935625609946)), ('u', 3, (-0.007173783717025234, 0.13554058107026465, -0.39180659805914264)), ('u', 11, (0.2792148056763726, 0.2254854997367548, 0.4127664392821797)), ('cz', 1, 2), ('u', 1, (3.5354308443674016, 3.141592653565993, 2.727856506850605)), ('u', 0, (0.6216410374626734, 2.4660318495354776, 0.16017059567987135)), ('u', 2, (3.4070925274539876, 0.9889427263735244, -0.07346448659884208)), ('u', 3, (2.7846538835367625, 0.17954065757908608, -3.0869740348848307)), ('u', 11, (-2.5556941338854244, 0.3325180963010587, 2.7358645841653395)), ('cz', 1, 0), ('u', 1, (-0.034210579714706625, 1.0471975511877258, -0.10685196646117553)), ('u', 0, (3.032753438631378, 0.7853981633986965, -3.0753693796873662)), ('u', 2, (2.9667356164544696, 0.7904971409785349, -2.6131984669867947)), ('u', 3, (3.19564422079325, 0.11531413262527708, -2.9040522501596)), ('u', 11, (0.26968276724919915, 0.5316085498556282, 0.42426397548867156)), ('cz', 1, 0), ('u', 1, (-3.0347406871394482, 1.57079632680026, 2.614152936642021)), ('u', 0, (-3.2078159275137983, 2.526112944922686, 2.6141529366064606)), ('u', 2, (-2.8165042172229344, 0.27596906765413426, 3.1369904505263846)), ('u', 3, (-2.8218267864127635, 0.6587119422794867, 2.80649796296318)), ('u', 11, (3.4160566868134237, 1.6723986827024986, 0.25117053936860545)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 6, (0.0, 1.5707963267948966, 0.0)), ('u', 7, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (-6.026816351730587e-17, 0.4205343352839649, 4.75701228604804e-18)), ('u', 6, (-6.026816351730587e-17, 0.4205343352839649, 4.75701228604804e-18)), ('cz', 6, 7), ('u', 7, (0.0, 1.5707963267948966, 0.0)), ('u', 6, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (-3.141592653589793, 1.093138017732642, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (0.0, 0.4776583090622546, 0.0)), ('cz', 8, 7), ('u', 7, (-3.141592653589793, 0.4776583090622546, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (0.0, 1.093138017732642, 0.0)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (-3.141592653589793, 1.1780972450961724, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 0.39269908169872414, 0.0)), ('cz', 9, 8), ('u', 8, (-3.141592653589793, 0.39269908169872414, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 1.1780972450961724, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (-3.141592653589793, 1.2630564724597029, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 0.30773985433519363, 0.0)), ('cz', 10, 9), ('u', 9, (-3.141592653589793, 0.30773985433519363, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 1.2630564724597029, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 10, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (-3.141592653589793, 1.3605291591529143, 3.141592653589793)), ('cz', 9, 10), ('u', 10, (0.0, 0.2102671676419825, 0.0)), ('cz', 11, 10), ('u', 10, (-3.141592653589793, 0.2102671676419825, 3.141592653589793)), ('cz', 9, 10), ('u', 10, (0.0, 1.3605291591529143, 0.0)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 10, 11), ('u', 8, (-3.3616956317625375, 0.0039025995926633074, 2.902737059849745)), ('u', 7, (-1.8398229777199921, 1.5707960232532314, 1.5707962546177183)), ('u', 9, (-0.12906788908650108, 0.04578873476250403, -0.11604755197690657)), ('u', 10, (3.0428786886420847, 0.13545291161950676, -3.0261753243807634)), ('u', 6, (0.8679903566181906, 1.5707906309898687, -1.5707889469954732)), ('cz', 7, 6), ('u', 8, (-3.3804482473286686, 0.009924520520796851, 2.8958184391473565)), ('u', 7, (1.5707962546201286, 0.46364759641455194, -1.5708013254767899)), ('u', 9, (-0.11604755198141795, 0.03774799811526014, -0.1033234987376338)), ('u', 10, (0.11541732920874666, 0.03198075418302716, 0.11534099377988617)), ('u', 6, (1.5708037065959612, 0.4636476216847983, -1.5650281579996224)), ('cz', 7, 6), ('u', 8, (2.895818439144895, 1.5598568467563203, 0.0038364657985918704)), ('u', 7, (1.5707913281098738, 1.5707929981959943, -1.7331065517780861)), ('u', 9, (-0.1033234987368545, 0.04526744111726166, -0.09360845081939316)), ('u', 10, (0.1153409937792085, 0.0363283934257337, 0.04320833115246539)), ('u', 6, (-1.565028157997609, 1.574078060718253, 1.4321880634934594)), ('cz', 8, 7), ('u', 8, (3.145429119388768, 1.566289540396074, 0.026029970715103712)), ('u', 7, (2.9792824286060386, 1.1189901810498026, -2.957031669585208)), ('u', 9, (-3.2352011044145392, 0.010733493041444131, 3.045550197439411)), ('u', 10, (0.04320833115135897, 0.03840627777527113, -0.037635627331219015)), ('u', 6, (3.0029843902853877, 0.012008086508496617, -3.26112331728915)), ('cz', 7, 6), ('u', 8, (0.02602997071261816, 0.004513970290364402, 0.045673007519336756)), ('u', 7, (0.1845609840041862, 0.47978010719223435, 1.4814137756780088e-07)), ('u', 9, (-3.2376351097367766, 0.016736599373341834, 3.0384856163742726)), ('u', 10, (-0.03763562733066405, 0.04404069064629421, -0.09604932914749834)), ('u', 6, (3.0220619898888565, 1.7719525310387264e-07, -2.954812737053833)), ('cz', 8, 7), ('u', 8, (-3.0959196460633858, 0.00305631116949349, 3.167334409245106)), ('u', 7, (-3.14159250544717, 0.47978058229153203, 2.957031631929024)), ('u', 9, (-3.2446996908038086, 0.016549755426092796, 3.0288083253542935)), ('u', 10, (-0.09604932914829628, 0.041676868215172455, -0.12806783452783213)), ('u', 6, (-2.9548127370537993, 1.0698913694347639e-07, 3.069244957059103)), ('cz', 7, 6), ('u', 8, (3.1673344092429017, 1.5677294516704396, -8.083696201843793e-05)), ('u', 7, (-0.18456102165971902, 1.1189903908168635, -0.022232579678741593)), ('u', 9, (-3.2543769818273494, 0.009907990163093456, 3.0187862681471365)), ('u', 10, (3.0135248190622534, 0.00831377776835061, -3.2392172041523577)), ('u', 6, (3.0692449570575775, 0.06500962406178507, -3.2025812648154197)), ('cz', 8, 7), ('u', 8, (3.141511816629895, 1.5707909975421381, 0.2795097751339344)), ('u', 7, (3.1193600739139855, 0.0038773567669587576, -3.143290458412774)), ('u', 9, (3.0187862681466413, 1.5636697132372983, -0.04650499316495793)), ('u', 10, (3.0439681030273817, 0.016583435574176644, -3.217121673046057)), ('u', 6, (3.080604042362413, 0.004485672376198623, -3.2011727302665123)), ('cz', 8, 9), ('u', 8, (-2.8620828784616825, 1.2282940888725038, 3.1808721047960575)), ('u', 7, (-0.0016978048442320426, 0.003888989001008372, -0.1348019816267435)), ('u', 9, (3.0950876604220294, 1.5665272369139556, 0.06875979257009135)), ('u', 10, (3.0660636341350878, 0.01691657118941096, -3.1980165558293714)), ('u', 6, (-0.05958007667562662, 0.013360953812156214, -0.058805102870934114)), ('cz', 8, 7), ('u', 8, (0.03927945120649929, 0.3434805014602192, -2.640530083544701e-07)), ('u', 7, (-0.13480198162427626, 0.027094233027520797, 1.6388245246690136e-05)), ('u', 9, (0.06875979256601772, 0.004301554704185576, 0.12891887819345363)), ('u', 10, (3.0851687513520285, 0.010079346674583382, -3.178405289074981)), ('u', 6, (-0.05880510287110147, 0.016240194270530382, -0.059196495329496585)), ('cz', 8, 9), ('u', 8, (-3.1415929176424457, 0.34348064036200293, 3.102313453583718)), ('u', 7, (-3.1415762653376214, 0.02709345261626353, 3.0505835608842955)), ('u', 9, (0.12891887819755127, 0.01013010130220086, 0.1216228584180174)), ('u', 10, (-0.03681263548924178, 0.0025202763610329714, -0.016404787978367717)), ('u', 6, (-0.05919649533829383, 0.01053532679040274, -0.06137444854302273)), ('cz', 8, 7), ('u', 8, (-0.03927920000837837, 1.2282940124357071, 0.07878175113281488)), ('u', 7, (-3.2326017463036396, 0.07722318387032373, 3.0447093928887488)), ('u', 9, (3.2632155120108504, 1.5806177012213178, 0.0012158864986266948)), ('u', 10, (-0.016404787975790074, 0.018348675980638433, 0.0010685150898290452)), ('u', 6, (-0.061374448537893374, 0.006804978319595123, -0.0643754609622237)), ('cz', 8, 9), ('u', 8, (-3.06281090244338, 0.240706890636655, 3.141565133298758)), ('u', 7, (-3.238475914286812, 0.07084984151046538, 3.0385486390618226)), ('u', 9, (3.142808540084078, 1.5707827303287538, 0.21075944922806578)), ('u', 10, (3.14266116867586, 1.5385196004839299, -0.03491889911013457)), ('u', 6, (-0.06437546096512091, 0.0054045980623675886, -0.06736450547171578)), ('cz', 9, 10), ('u', 8, (-2.7520291481966663e-05, 0.24070602798931603, -0.05997165814907577)), ('u', 7, (-3.244636668109622, 0.05598245495502534, 3.0320741887932416)), ('u', 9, (-2.9308332043634375, 1.3385498632407555, 3.265353558096892)), ('u', 10, (3.106673754481762, 1.560541883503742, -0.0005960049060806938)), ('u', 6, (-0.06736450546594043, 0.004020741557122551, -0.07003693041619181)), ('cz', 8, 9), ('u', 8, (3.0816209954444678, 1.6702349862793216e-06, -3.151948482655484)), ('u', 7, (-3.251111118393032, 0.03175284464352435, 3.0254859312522413)), ('u', 9, (0.12376090451497594, 0.23927558486023273, -3.062663233299734e-07)), ('u', 10, (-0.000596004904864884, 0.01025434933108818, 0.0038151881137930034)), ('u', 6, (-0.07003693041811267, 0.0017968556903600957, -0.07238659593900958)), ('cz', 9, 10), ('u', 8, (-0.01035582905640281, 1.6571755945843642e-06, -0.26452570087329547)), ('u', 7, (-3.2576993759273543, 0.00032258283848635226, 3.0193708150795637)), ('u', 9, (-3.1415929598712458, 0.23927570036938825, 3.0178320803715932)), ('u', 10, (0.003815188114607915, 0.004999239888433819, 0.0003175146299575066)), ('u', 6, (-3.21397924952885, 0.0017199534340491375, 3.0669741613455583)), ('cz', 8, 9), ('u', 8, (-0.2645257008822089, 0.022698070588217982, -0.2683201712919627)), ('u', 7, (-0.12222183851135454, 0.03453443679232455, -0.12705353820514548)), ('u', 9, (-0.12376057320591063, 1.3385499787135362, -0.017193318234947096)), ('u', 10, (3.141910168223556, 1.575823938467278, 0.037139376734341756)), ('u', 6, (-3.2162111458374176, 0.006962294321765245, 3.0645101870333398)), ('cz', 9, 10), ('u', 8, (-0.2683201712850233, 0.02588619674058842, -0.27405974044826253)), ('u', 7, (3.0145391153815564, 1.5012651227117857, -3.042494962187664)), ('u', 9, (-0.017193318243465552, 0.2274733924350092, 0.027012934042615122)), ('u', 10, (3.178732030327292, 1.6054043772816553, 0.026355978482590192)), ('u', 6, (-0.07708246655565573, 1.5565325809083093, -0.01157463050711708)), ('cz', 7, 6), ('u', 8, (-0.2740597404429681, 0.0050791591696530336, -0.2798497704482895)), ('u', 7, (0.09909769140352187, 0.5235986704000041, 4.934548634913383e-07)), ('u', 9, (0.027012934048280552, 0.23223570148007708, 0.02057562114681377)), ('u', 10, (0.026355978480794698, 0.03291981374402586, 0.02871457744481702)), ('u', 6, (-0.011574630507208597, 0.5235988584712555, 0.003755531543405243)), ('cz', 7, 6), ('u', 8, (2.86174288313786, 1.5849930139053003, -0.02772880601938077)), ('u', 7, (4.934559445501918e-07, 1.570799985605745, -0.3672017901622837)), ('u', 9, (0.020575621150727856, 0.2347947214593049, 0.015443935593118692)), ('u', 10, (0.028714577445230784, 0.03078005594798939, 0.030420880016502092)), ('u', 6, (-3.137837122050208, 1.5084297217180156, 3.081684438802766)), ('cz', 8, 7), ('u', 8, (3.11386384757129, 1.5858597004508448, -0.21648083033204935)), ('u', 7, (2.7743908634293213, 1.1765537238359582, -3.2424815135233667)), ('u', 9, (0.0154439355960923, 0.2327901548221136, 0.006755768213924533)), ('u', 10, (0.030420880017375692, 0.028224759810729885, 0.031458108518774945)), ('u', 6, (-3.201500868373799, 0.062809903304435, 3.072788440835523)), ('cz', 7, 6), ('u', 8, (-3.358073483933465, 0.016594588685602412, 2.922541544690592)), ('u', 7, (-0.10088885993272834, 0.4015641044850999, 1.9146506219161474e-06)), ('u', 9, (0.0067557682212195805, 0.22548235911651435, -0.003989299684354418)), ('u', 10, (0.03145810851650309, 0.025280110402835204, 0.03181947270364757)), ('u', 6, (-0.06880421275127409, 1.604476694557179e-06, 0.05357468562342515)), ('cz', 8, 7), ('u', 8, (-0.21905110889413099, 0.7557955267801929, -2.7925791038357906e-06)), ('u', 7, (3.1415945682440825, 0.40156381598093055, -3.0405994709292026)), ('u', 9, (-0.003989299666667046, 0.2147681910777935, -0.014293947919701181)), ('u', 10, (0.031819472703085036, 0.021961191043929674, 0.031507818962971446)), ('u', 6, (3.195167339215446, 1.2763621866968949e-06, -2.850626142844065)), ('cz', 7, 6), ('u', 8, (3.141589861007258, 0.7557972359659272, -0.17651100844701206)), ('u', 7, (0.10099318265882568, 1.1979374092082964, -0.03355076945371302)), ('u', 9, (-0.01429394792598595, 0.20530927933180743, -0.022490857279512437)), ('u', 10, (0.03150781896208828, 0.01827092495024244, 0.030535130309331398)), ('u', 6, (0.29096651073712837, 0.30021305291261163, 0.2778840351660003)), ('cz', 8, 9), ('u', 8, (-2.0711141832239552, 1.8522518302064177, 2.676226797705731)), ('u', 7, (3.184285301393627, 0.02138436038075681, -3.0981379303176886)), ('u', 9, (3.1387580868246863, 1.5637723959116658, -0.024014521939939915)), ('u', 10, (-3.1107917586702616, 0.046950571881594114, 3.1708570956443003)), ('u', 6, (0.26867859365444346, 0.3360642572215557, 0.2506566488391688)), ('cz', 8, 7), ('u', 8, (2.6762267977114638, 0.4508518802864524, -2.6514295007963016)), ('u', 7, (0.043454723268409115, 4.352468814837479e-06, 0.08926924916257063)), ('u', 9, (-0.024014521941247213, 0.00703190605988138, -0.029926439855007595)), ('u', 10, (-3.1123282115368607, 0.04923057455422213, 3.168630680023086)), ('u', 6, (0.25065664883982414, 0.37868904030764705, 0.23552140500318813)), ('cz', 8, 9), ('u', 8, (-2.651429500805082, 1.3075256080222397, 1.3856145361709284)), ('u', 7, (3.2308619027580914, 4.722053095840896e-06, -2.927243181464406)), ('u', 9, (3.1116662137350612, 0.04562313341921808, -3.084781858848838)), ('u', 10, (-3.1145546271570494, 0.04995151609940563, 3.165689276599569)), ('u', 6, (0.23552140499680949, 0.42272800858487664, 0.2253678638517245)), ('cz', 8, 7), ('u', 8, (-1.755978117384596, 2.860137193418854, -0.39118296614422066)), ('u', 7, (0.2143494721257993, 0.7809464399597905, 0.4181283424285072)), ('u', 9, (3.1984034483290094, 1.525476192498973, 0.017549776005519213)), ('u', 10, (-3.117496030579368, 0.0490400341303214, 3.162003580322241)), ('u', 6, (0.22536786385532717, 0.4459511938364855, 0.22304256245171158)), ('cz', 8, 9), ('u', 8, (-0.3911829661374032, 0.04667667107490827, -0.44663640869297494)), ('u', 7, (-1.152667984363594, 0.9366009367620469, 1.8164739573447128)), ('u', 9, (3.1591424295921255, 1.6121812251712162, -0.03274870374482575)), ('u', 10, (-3.121181726856949, 0.04656733683275216, 3.1575188605983557)), ('u', 6, (1.793838889248835, 1.3501582727139965, -1.1403975452439905)), ('cz', 7, 6), ('u', 8, (-0.4466364087047528, 0.1096364448358437, -0.47525664164906617)), ('u', 7, (1.8164739573437514, 1.5707960189765107, -1.6426218990624604)), ('u', 9, (-0.032748703747504104, 0.02640103660169595, -0.011999429642412445)), ('u', 10, (-3.125666446584757, 0.04307213498177388, 3.1521045306332796)), ('u', 6, (2.0011951083475314, 0.6154796888855657, -1.50066293622409)), ('cz', 7, 6), ('u', 8, (2.6663360119497153, 1.622115658151251, 0.3924302047674202)), ('u', 7, (1.4989707545388609, 1.57079758646911, -1.2630565997990424)), ('u', 9, (-0.011999429636831817, 0.01432335642396587, 0.008515071949799209)), ('u', 10, (0.010511877046058654, 0.008595592443085652, 0.007209941166385776)), ('u', 6, (-1.5006629362271944, 0.5786264974028342, 1.7600359216969468)), ('cz', 8, 7), ('u', 8, (3.5340228583608244, 1.7032000880946896, -0.020534934410332273)), ('u', 7, (-2.8338529265939436, 0.7853986878687494, 3.0587967431639114)), ('u', 9, (0.008515071950168087, 0.005700369894339191, 0.02708467108704314)), ('u', 10, (0.007209941165085509, 0.015431279072740337, 0.0039356021134373115)), ('u', 6, (-2.9523530586865037, 0.086974843088867, 3.2935099651206152)), ('cz', 7, 6), ('u', 8, (-0.020534934410238084, 0.3743921736986888, -0.15981395480196686)), ('u', 7, (-0.08279591042911418, 0.7922834412819855, 0.05841184965368405)), ('u', 9, (0.027084671089494264, 0.001288378251789532, 0.042435565818684115)), ('u', 10, (0.0039356021140504694, 0.014405475603402189, 0.0004565804200185068)), ('u', 6, (3.293509965118046, 0.0014199029078546372, -2.961141930049238)), ('cz', 8, 7), ('u', 8, (-0.1598139548054538, 1.570796410795173, 0.7223714329487047)), ('u', 7, (0.05841184964908008, 1.570796017402631, -1.309320706403827e-07)), ('u', 9, (0.04243556581174311, 0.00300930997654793, 0.05368591160956045)), ('u', 10, (0.00045658041965513643, 0.012410543352531113, -0.003246868178306708)), ('u', 6, (0.18045072353912656, 0.003278365791523672, 0.205751444790659)), ('cz', 8, 7), ('u', 8, (3.86396408653314, 1.571145798537709, 0.15440661523178512)), ('u', 7, (-1.309308895400263e-07, 1.5707961359952263, -0.10609942251410948)), ('u', 9, (0.05368591161727363, 0.01752948364162591, 0.06012814820463553)), ('u', 10, (-0.0032468681786684424, 0.009955138947101028, -0.007150595866979236)), ('u', 6, (0.20575144479313678, 1.5688086081645831, -0.3079005763286621)), ('cz', 7, 6), ('u', 8, (0.1544066152372917, 0.002491062245472753, 0.25803183635411187)), ('u', 7, (-0.10609942253662326, 0.7853984864413228, 3.768912299195737e-07)), ('u', 9, (3.2017208017915415, 0.06456139051322682, -3.0766973042554477)), ('u', 10, (-0.007150595867724202, 0.006754673855885729, -0.011211928098680534)), ('u', 6, (2.833692077263284, 1.0471982243125375, -2.66393339973847)), ('cz', 7, 6), ('u', 8, (0.2580318363550479, 0.16979115008052928, 0.29310648646187865)), ('u', 7, (-3.1415922767012914, 1.5707946913660074, 2.696338100072989)), ('u', 9, (3.206488002925403, 0.05199856364242917, -3.0766828637509698)), ('u', 10, (-0.011211928097410468, 0.0025805312124761734, -0.015374066321883307)), ('u', 6, (-2.663933399730657, 1.5707943453258024, 2.696338058138845)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0))]), (12, [('u', 5, (0.0, 3.0757856104947443, 0.0)), ('u', 4, (0.0, 0.19879928334601105, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (-3.141592653589793, 0.19879928334601105, 3.141592653589793)), ('u', 3, (0.0, 0.5278194054589851, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 0.5278194054589851, 3.141592653589793)), ('u', 2, (0.0, 0.8903978130278803, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 0.8903978130278803, 3.141592653589793)), ('u', 1, (0.0, 1.185542550497017, 0.0)), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 1), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.185542550497017, 3.141592653589793)), ('u', 0, (0.0, 1.4056476493802696, 0.0)), ('u', 0, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 0), ('u', 0, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (-3.141592653589793, 1.4056476493802696, 3.141592653589793)), ('u', 6, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 6), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 7), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 11, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 0, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (0.0, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (-6.026816351730587e-17, 0.4205343352839649, 4.75701228604804e-18)), ('u', 0, (-6.026816351730587e-17, 0.4205343352839649, 4.75701228604804e-18)), ('cz', 0, 1), ('u', 1, (0.0, 1.5707963267948966, 0.0)), ('u', 0, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.093138017732642, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.4776583090622546, 0.0)), ('cz', 2, 1), ('u', 1, (-3.141592653589793, 0.4776583090622546, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 1.093138017732642, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.1780972450961724, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 0.39269908169872414, 0.0)), ('cz', 3, 2), ('u', 2, (-3.141592653589793, 0.39269908169872414, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 1.1780972450961724, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.2630564724597029, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 0.30773985433519363, 0.0)), ('cz', 4, 3), ('u', 3, (-3.141592653589793, 0.30773985433519363, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 1.2630564724597029, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (-3.141592653589793, 1.3605291591529143, 3.141592653589793)), ('cz', 3, 4), ('u', 4, (0.0, 0.2102671676419825, 0.0)), ('cz', 5, 4), ('u', 4, (-3.141592653589793, 0.2102671676419825, 3.141592653589793)), ('cz', 3, 4), ('u', 4, (0.0, 1.3605291591529143, 0.0)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 1, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 0, 1), ('u', 1, (1.570796326794898, 0.4636476090008061, -1.570796326794898)), ('u', 0, (1.5707963267948966, 0.4636476090008061, -1.5707963267948966)), ('cz', 0, 1), ('u', 1, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 0, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 1, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.1277577648985897, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.44303856189630686, 0.0)), ('cz', 2, 1), ('u', 1, (-3.141592653589793, 0.44303856189630686, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 1.1277577648985897, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.2284367252937551, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 0.3423596015011414, 0.0)), ('cz', 3, 2), ('u', 2, (-3.141592653589793, 0.3423596015011414, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 1.2284367252937551, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 0.23182380450040305, 0.0)), ('cz', 4, 3), ('u', 3, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 1.3389725222944935, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 1, (1.363845864587723, 1.5707963267672458, -2.9993854688414716)), ('u', 0, (1.363845864586009, 1.5707963268378884, 1.2639026291583169)), ('u', 2, (-3.2071888465277136, 0.184904712775633, 2.8610451944514357)), ('u', 3, (0.5385123367231761, 0.06467488581738873, 0.04937591114067358)), ('u', 11, (1.6992100207502157, 0.23360903143076647, 0.5579594806284142)), ('cz', 1, 0), ('u', 1, (-0.14220718475653546, 0.5235987755985089, -0.41745071061119066)), ('u', 0, (1.877690024460379, 2.6179938779918035, -0.03580353792904112)), ('u', 2, (0.4800735699666825, 0.2897401439513171, 0.14878724963518253)), ('u', 3, (-3.013747770360676, 0.03451614731288748, 3.003756469721795)), ('u', 11, (0.2185318996508333, 0.1328640734067659, -0.2804221915090717)), ('cz', 1, 0), ('u', 1, (0.4174507106123065, 1.5707963268071967, -0.47126800775294303)), ('u', 0, (3.1470202037463095, 1.6882687876873737, -2.7231476408232136)), ('u', 2, (3.238386628155749, 1.6848016324293131, -0.06463740846983468)), ('u', 3, (-0.08956681406336006, 0.06641496875959405, -0.00173379117006673)), ('u', 11, (3.290510076704486, 0.3843153515279157, -2.2473537186376977)), ('cz', 1, 2), ('u', 1, (2.5760504070058876, 0.39269908170299855, -2.9988382870440558)), ('u', 0, (-2.7625746621013945, 0.0950651554024133, 2.371617425516931)), ('u', 2, (3.1971735290592127, 1.5707963268013454, -0.21746618356704195)), ('u', 3, (2.817035250472036, 0.4816552162375767, -2.8307390933588885)), ('u', 11, (-3.03777683654344, 0.21866970396518934, 3.043424696251818)), ('cz', 1, 2), ('u', 1, (2.998838287025793, 0.39269908169903683, -3.0143314903138596)), ('u', 0, (0.1715106168685091, 0.18755420994158523, 0.4161264827006588)), ('u', 2, (0.05982400444235608, 1.0485062235908116e-10, 0.28710524803901055)), ('u', 3, (2.8908380088425445, 0.12446905701880381, -2.8730332913397967)), ('u', 11, (-0.17878558894341473, 0.05048690051415294, 0.14584352443348808)), ('cz', 1, 0), ('u', 1, (-0.1272611631371841, 1.178097245110616, 0.043673916629391046)), ('u', 0, (0.003818572825657514, 0.2467105513567965, -0.06941740237412158)), ('u', 2, (2.644326099731402, 6.954531877379686e-11, -1.0548234819637168)), ('u', 3, (0.18491647072024606, 0.24436291923884504, -0.25004419633888014)), ('u', 11, (-0.03550929641403387, 0.4030732709508788, 0.2678597573031803)), ('cz', 1, 2), ('u', 1, (-3.1228504119408154, 1.030966922621024, 2.9453401058764124)), ('u', 0, (0.1179962239684767, 0.23548435679138233, 0.1527200841677004)), ('u', 2, (2.449394360105412, 3.1415926535678182, -0.0922889975238399)), ('u', 3, (2.540224590607155, 1.2430644716230643, 0.34042883892006603)), ('u', 11, (-2.8597740722484075, 0.28847312567924255, 2.6216845541657174)), ('cz', 2, 3), ('u', 1, (-3.017053450194865, 0.9344486059450692, 2.4728430001260175)), ('u', 0, (0.5054443303789188, 0.16415946838028975, 0.530326491748032)), ('u', 2, (-2.6307929107371804, 2.87907570840622, 3.2359900602319955)), ('u', 3, (2.7974402917130727, 1.4766533627026013, -0.20887776650500722)), ('u', 11, (-2.9456829708982157, 0.316226895673598, 2.651044412584663)), ('cz', 1, 2), ('u', 1, (3.0878665036375486, 2.1702215788789444e-10, -3.0036321220042597)), ('u', 0, (-2.6810618737305143, 0.15689068523105396, 2.926997184274815)), ('u', 2, (-0.16730747595939177, 1.8326473121711249, -0.38983431787754164)), ('u', 3, (-0.00923206423481826, 0.09641354025270232, -0.2687390618669404)), ('u', 11, (0.004171154571400146, 0.3169631030215196, -0.035171437866080006)), ('cz', 2, 3), ('u', 1, (0.0205047673468016, 1.7716992026397947e-10, -0.01402221768342051)), ('u', 0, (0.10901292728483414, 0.11595455849628988, 0.28503099895241835)), ('u', 2, (3.256426824750352, 0.27158740610627147, -2.7011709482222255)), ('u', 3, (-0.2014546693702715, 0.21615536475340474, 0.44973465481447467)), ('u', 11, (3.1168657275130505, 0.009097770850434048, -2.8631452225525758)), ('cz', 1, 2), ('u', 1, (0.08936923612351227, 0.0006266391039143163, -0.6256947727937483)), ('u', 0, (0.3127772484991065, 0.4425334180787224, -0.7738530006778277)), ('u', 2, (-0.17502644640991238, 1.3082793816134326, -0.4711828139334281)), ('u', 3, (3.4163064623247226, 1.733777998216237, 0.01266097499274732)), ('u', 11, (-0.34596199000980177, 0.14386738279590947, 0.6250437041635831)), ('cz', 2, 3), ('u', 1, (-2.0399542004731384, 1.5713533217248365, 1.7013097330827514)), ('u', 0, (1.6166016491329345, 1.9347304860345607, -1.773316789719498)), ('u', 2, (-0.4386484124822272, 0.0015522740078519823, 0.12839069158283042)), ('u', 3, (3.375586976992392, 1.7580043569266424, 0.13266390002743167)), ('u', 11, (3.0497594033307784, 0.40527541662488026, -2.967944020089303)), ('cz', 1, 0), ('u', 1, (1.4405700323809594, 2.5261129449188844, -1.0736099945134443)), ('u', 0, (-1.5307563786235483, 0.7083937952823107, 1.094342241382531)), ('u', 2, (3.1386558134130316, 1.5723364014655843, 0.6539020735549244)), ('u', 3, (2.7090304069013973, 0.0018120668535631193, -2.798423756554544)), ('u', 11, (-3.1526955155722445, 0.4136623979181174, 2.912043609045781)), ('cz', 1, 2), ('u', 1, (-3.023299277215143, 1.5707963267863896, 2.6032780596287983)), ('u', 0, (2.9921942707562117, 0.18474840197135983, -2.805109016014484)), ('u', 2, (3.255975628587858, 1.719373693642963, -0.28321952549538576)), ('u', 3, (0.07153529109902992, 0.3132596387815738, 0.037483751637371485)), ('u', 11, (0.3289736344727156, 0.5185160451997999, -0.052674441438991804)), ('cz', 1, 0), ('u', 1, (-1.0324817328297038, 1.5707963267998148, 0.3639224763010943)), ('u', 0, (0.05440250129871599, 0.5293683032878496, -0.35793546408076754)), ('u', 2, (0.007885513401241365, 0.2833965772722029, 0.03910361915813132)), ('u', 3, (2.845935794416537, 0.23002941200652896, -3.096456826830062)), ('u', 11, (0.04657467538722733, 0.10198091863099273, 0.18825217824805818)), ('cz', 1, 2), ('u', 1, (0.0, 1.1383257248941486e-12, 0.233665301559387)), ('u', 0, (2.415982008321575, 0.92714299809076, -2.995458349540593)), ('u', 2, (-0.34092652257087946, 0.018534898805743556, 0.0986872907939854)), ('u', 3, (0.17707817274062912, 0.4845914864344751, 0.4828603407009331)), ('u', 11, (0.14622383230920563, 0.3504973020598406, 0.5319535697949473)), ('cz', 1, 0), ('u', 1, (0.18781038554835428, 1.5707963268007776, 0.7290114095506618)), ('u', 0, (0.4613575044770586, 1.5387056026239314, 0.9892258188675678)), ('u', 2, (3.180073602453756, 1.5891571698882074, 0.08911935625609946)), ('u', 3, (-0.007173783717025234, 0.13554058107026465, -0.39180659805914264)), ('u', 11, (0.2792148056763726, 0.2254854997367548, 0.4127664392821797)), ('cz', 1, 2), ('u', 1, (3.5354308443674016, 3.141592653565993, 2.727856506850605)), ('u', 0, (0.6216410374626734, 2.4660318495354776, 0.16017059567987135)), ('u', 2, (3.4070925274539876, 0.9889427263735244, -0.07346448659884208)), ('u', 3, (2.7846538835367625, 0.17954065757908608, -3.0869740348848307)), ('u', 11, (-2.5556941338854244, 0.3325180963010587, 2.7358645841653395)), ('cz', 1, 0), ('u', 1, (-0.034210579714706625, 1.0471975511877258, -0.10685196646117553)), ('u', 0, (3.032753438631378, 0.7853981633986965, -3.0753693796873662)), ('u', 2, (2.9667356164544696, 0.7904971409785349, -2.6131984669867947)), ('u', 3, (3.19564422079325, 0.11531413262527708, -2.9040522501596)), ('u', 11, (0.26968276724919915, 0.5316085498556282, 0.42426397548867156)), ('cz', 1, 0), ('u', 1, (-3.0347406871394482, 1.57079632680026, 2.614152936642021)), ('u', 0, (-3.2078159275137983, 2.526112944922686, 2.6141529366064606)), ('u', 2, (-2.8165042172229344, 0.27596906765413426, 3.1369904505263846)), ('u', 3, (-2.8218267864127635, 0.6587119422794867, 2.80649796296318)), ('u', 11, (3.4160566868134237, 1.6723986827024986, 0.25117053936860545)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 6, (0.0, 1.5707963267948966, 0.0)), ('u', 7, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (-6.026816351730587e-17, 0.4205343352839649, 4.75701228604804e-18)), ('u', 6, (-6.026816351730587e-17, 0.4205343352839649, 4.75701228604804e-18)), ('cz', 6, 7), ('u', 7, (0.0, 1.5707963267948966, 0.0)), ('u', 6, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (-3.141592653589793, 1.093138017732642, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (0.0, 0.4776583090622546, 0.0)), ('cz', 8, 7), ('u', 7, (-3.141592653589793, 0.4776583090622546, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (0.0, 1.093138017732642, 0.0)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (-3.141592653589793, 1.1780972450961724, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 0.39269908169872414, 0.0)), ('cz', 9, 8), ('u', 8, (-3.141592653589793, 0.39269908169872414, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 1.1780972450961724, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (-3.141592653589793, 1.2630564724597029, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 0.30773985433519363, 0.0)), ('cz', 10, 9), ('u', 9, (-3.141592653589793, 0.30773985433519363, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 1.2630564724597029, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 10, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (-3.141592653589793, 1.3605291591529143, 3.141592653589793)), ('cz', 9, 10), ('u', 10, (0.0, 0.2102671676419825, 0.0)), ('cz', 11, 10), ('u', 10, (-3.141592653589793, 0.2102671676419825, 3.141592653589793)), ('cz', 9, 10), ('u', 10, (0.0, 1.3605291591529143, 0.0)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 10, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 6, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 7, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 6, 7), ('u', 7, (1.570796326794898, 0.4636476090008061, -1.570796326794898)), ('u', 6, (1.5707963267948966, 0.4636476090008061, -1.5707963267948966)), ('cz', 6, 7), ('u', 7, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 6, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 7, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (-3.141592653589793, 1.1277577648985897, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (0.0, 0.44303856189630686, 0.0)), ('cz', 8, 7), ('u', 7, (-3.141592653589793, 0.44303856189630686, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (0.0, 1.1277577648985897, 0.0)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (-3.141592653589793, 1.2284367252937551, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 0.3423596015011414, 0.0)), ('cz', 9, 8), ('u', 8, (-3.141592653589793, 0.3423596015011414, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 1.2284367252937551, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 0.23182380450040305, 0.0)), ('cz', 10, 9), ('u', 9, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 1.3389725222944935, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 6, (0.0, 1.5707963267948966, 0.0)), ('u', 7, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (0.0, 0.5235987755982989, 0.0)), ('u', 6, (0.0, 0.5235987755982989, 0.0)), ('cz', 6, 7), ('u', 7, (0.0, 1.5707963267948966, 0.0)), ('u', 6, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (-3.141592653589793, 1.1780972450961724, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (0.0, 0.39269908169872414, 0.0)), ('cz', 8, 7), ('u', 7, (-3.141592653589793, 0.39269908169872414, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (0.0, 1.1780972450961724, 0.0)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (-3.141592653589793, 1.3089969389957472, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 0.26179938779914935, 0.0)), ('cz', 9, 8), ('u', 8, (-3.141592653589793, 0.26179938779914935, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 1.3089969389957472, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 6, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 7, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 6, 7), ('u', 7, (1.570796326794898, 0.6154797086703868, -1.570796326794898)), ('u', 6, (1.5707963267948966, 0.6154797086703876, -1.5707963267948966)), ('cz', 6, 7), ('u', 7, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 6, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 7, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (-3.141592653589793, 1.2630564724597029, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (0.0, 0.30773985433519363, 0.0)), ('cz', 8, 7), ('u', 7, (-3.141592653589793, 0.30773985433519363, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (0.0, 1.2630564724597029, 0.0)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 6, (0.0, 1.5707963267948966, 0.0)), ('u', 7, (0.0, 1.5707963267948966, 0.0)), ('cz', 6, 7), ('u', 7, (1.2251903831398114e-16, 0.7853981633974483, -1.1496735851465472e-17)), ('u', 6, (-3.141592653589793, 0.7853981633974483, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 6, (-3.141592653589793, 1.5707963267948966, 3.141592653589793))])], (13, 2): [(13, [('u', 5, (0.0, 2.0505249644706494, 0.0)), ('u', 4, (0.0, 1.0321328607141094, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (-3.141592653589793, 1.0321328607141094, 3.141592653589793)), ('u', 11, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 12, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 12), ('u', 5, (-0.026488601017550822, 0.33817761953207987, -0.03727149628245448)), ('u', 4, (0.5261809395945654, 4.583733762573752e-09, -2.075678192631724)), ('u', 3, (1.7279839243247377, 0.9553166181244952, -2.31897812556857)), ('cz', 4, 3), ('u', 5, (3.2069423080124477, 1.9088505746848605, 0.35741556707447986)), ('u', 4, (1.5494972529733677, 1.570796323704034, -0.9700524480599615)), ('u', 3, (-1.2917068473260913, 1.0057498368542224, 1.6663688863848016)), ('cz', 5, 4), ('u', 5, (2.745693454732362, 1.8819958982276148, -0.16966288443458488)), ('u', 4, (-2.1715402047728465, 0.4205343352838054, 3.204649700748775)), ('u', 3, (0.16302187254096098, 0.391758611081972, 0.1327063423358217)), ('cz', 4, 3), ('u', 5, (3.246038069634937, 0.7332181270315529, 0.1516148239645223)), ('u', 4, (-0.06305704776105525, 1.5707963267948473, 0.24332075105407625)), ('u', 3, (0.02391047167407853, 1.2080095905811189e-09, 0.24946083236423147)), ('cz', 5, 4), ('u', 11, (0.014123007862392137, 1.838960753766552e-11, -0.05362749491799243)), ('u', 10, (0.06390942585002227, 1.0068536854342445, 0.36449757668352667)), ('u', 5, (2.9188198307459565, 1.4903817262192667, -0.062301176161434935)), ('u', 12, (4.683811108606976, 1.068272411818259, -0.04449585264636946)), ('cz', 11, 10), ('u', 11, (3.181097140646071, 1.5707963268046399, -2.977912352841801)), ('u', 10, (-2.7783866533775874, 1.5301679536926232, 3.3123900907738446)), ('u', 5, (-3.529945311495741, 0.017594718243649072, 2.226200412042302)), ('u', 12, (3.186088506240345, 1.0682724118189153, 0.6886119259411965)), ('cz', 11, 12), ('u', 11, (2.9779123528412828, 0.3875966866550018, -3.2240578855384574)), ('u', 10, (0.6647276179542967, 0.8602210086009686, 0.4186675249851015)), ('u', 5, (3.2244590785210248, 0.2362746996268105, -2.8213393477873385)), ('u', 12, (-2.8772912855650215, 0.10488861972121248, 3.2268380432672266)), ('cz', 11, 10), ('u', 11, (0.0824652319492355, 1.5707963267947394, 0.07074018970629348)), ('u', 10, (0.29894490710826455, 1.3106055848784151e-11, 0.2579669040575551)), ('u', 5, (-0.06139703677532954, 0.28729079492035065, 0.5356523381062686)), ('u', 12, (3.7322981803303117, 1.1003784426097099, 0.23088062023803424)), ('cz', 11, 12), ('u', 11, (-0.20692861776948868, 1.7822613829830467, -0.5560139334631086)), ('u', 10, (3.667829686926611, 2.2078489058424875e-11, -0.9778502517120493)), ('u', 5, (3.381408492134356, 0.47255173869772243, -2.8720850392364534)), ('u', 12, (0.0, 1.5294403672763525e-12, -0.3016902364815227)), ('u', 4, (-3.554819720556316, 1.8175894909190264, 2.2355366836874895)), ('u', 3, (-1.7850661071595355, 3.5701273649817347e-09, 2.990269384106037)), ('u', 2, (-0.27113878386298496, 0.8860771237922334, 0.2832335905683325)), ('u', 1, (1.0724667860992239, 0.3135441280757749, 0.3477481597921714)), ('cz', 2, 3), ('u', 2, (2.4400313446652238, 0.4674334689778797, -2.7557448911498277)), ('u', 3, (-1.2466744714517015, 1.5707963269126817, -0.3474883157693149)), ('u', 1, (1.2610848555455991, 0.5154430607864812, 0.4649785201028739)), ('u', 4, (-0.3487503182872954, 1.8124312541247014, -0.029227653884645405)), ('cz', 3, 4), ('u', 2, (3.4924690019924354, 0.487420075135062, -2.0561402443611527)), ('u', 3, (-2.794104337117351, 2.6779450445890216, 2.73086955979565)), ('u', 1, (4.139468277336655, 0.301183724873278, -1.6188205903909119)), ('u', 4, (3.6409272527688463, 0.10783953278100211, -0.27102555731550226)), ('cz', 2, 3), ('u', 2, (1.7718525496675044, 0.3070472199392804, -0.4995159680649769)), ('u', 3, (0.41072309437235316, 1.570796326794958, -0.2380438412443237)), ('u', 1, (4.128297871557185, 0.08416766952170567, -0.9881778941580865)), ('u', 4, (2.793806547308158, 1.1941283984595739, 0.5571280141976567)), ('cz', 3, 4), ('u', 0, (3.1415466473015825, 0.3472761449629368, -2.775502885699414)), ('u', 1, (-1.943048849865654, 0.0866519139605691, 3.3796548616430324)), ('u', 2, (0.49951723708076523, 2.834545501634763, 0.2477359093599764)), ('u', 3, (-2.900546912689433, 0.0220330108239607, 2.9517078164711683)), ('cz', 1, 2), ('u', 0, (-2.7755028856994066, 0.2557486829034849, 3.4795316578682636)), ('u', 1, (-2.287986882890206, 0.8387533584115957, 2.608378634138784)), ('u', 2, (-3.440321172504869, 1.5707963601966048, 2.4810079929535647)), ('u', 3, (1.628741394809413, 0.16973412222193243, 0.14557562970134652)), ('cz', 2, 3), ('u', 0, (-2.8036536493112556, 0.1072802106185393, 3.4534547002776566)), ('u', 1, (-0.42968280486432603, 0.09900030019283891, -0.5979464612539381)), ('u', 2, (-0.9102116414123469, 1.570796303280397, 0.5896891607222703)), ('u', 3, (-1.1994459353604168, 0.8540493077491784, 2.8966776766920646)), ('cz', 1, 2), ('u', 0, (0.3118622548531915, 0.06450072200815428, 0.3158295634276317)), ('u', 1, (4.0056453750975125, 1.5710093821180986, -2.25306084708445)), ('u', 2, (0.457508391465305, 1.5707961085469244, 1.9756186289307065)), ('u', 3, (-0.8946606362517096, 2.7743788682406283, -3.296909735624784)), ('cz', 2, 3), ('u', 0, (1.8861358465261668, 1.0886309990492427, -1.247457185490609)), ('u', 1, (-5.444254626341652, 3.1402271545080267, -0.46524964391451995)), ('u', 2, (0.8866780470332694, 2.7806992103177426, -0.027947407209413355)), ('u', 3, (-0.33059098576245977, 0.23605648596041864, -0.344797406280728)), ('cz', 0, 1), ('u', 0, (1.8941354680991402, 0.5435018820283635, -1.265545217593394)), ('u', 1, (1.5103199473301145, 2.52611291578785, -0.8703574349521468)), ('u', 2, (-0.12699537993299054, 0.7405798026131227, -0.3377410900233119)), ('u', 3, (3.126455392394852, 0.014708899727501449, -3.155938522155425)), ('cz', 1, 2), ('u', 0, (0.5720960810390555, 0.3997299720525652, 0.5480258762238771)), ('u', 1, (-0.08495921531061223, 1.5707965524889658, 0.7853981576711249)), ('u', 2, (-0.1307648484010647, 0.05425977598078749, 0.15618413163899017)), ('u', 3, (3.1272467850241585, 0.03145786231116345, -3.1537851270176698)), ('cz', 0, 1), ('u', 0, (0.5480258241825835, 0.06872222509928691, 0.3748642475219047)), ('u', 1, (0.7853981232512715, 1.570796315327001, -0.7853981651137552)), ('u', 2, (3.297776796581335, 1.006857237795784, 0.017799187962781016)), ('u', 3, (3.129400180161916, 0.04582839097485164, -3.1501000422518555)), ('cz', 1, 2), ('u', 0, (0.3748643116489141, 1.520463564085735, -0.6833998961474932)), ('u', 1, (2.3561944734590226, 0.7853981619176521, -2.993151367895617)), ('u', 2, (3.159391795646975, 1.5652872587775164, -0.6482539739882052)), ('u', 3, (3.1330852649277396, 0.057291232680230515, -3.144741673716754)), ('cz', 0, 1), ('u', 0, (-4.075799266061252, 1.5707959008369972, 1.4311782578899523)), ('u', 1, (2.993151235150358, 1.570796346808479, -1.7104143349925467)), ('u', 2, (-0.5745315317617433, 0.016153120923974765, -0.48767068019668275)), ('u', 3, (3.1384436334628316, 1.63618109680531, 0.9488551956938394)), ('u', 4, (3.269071830129811, 1.479831739000346, 0.17271472025738444)), ('u', 4, (2.443603264821166, 0.26906696464229146, -3.4844556820946466)), ('u', 4, (-0.5656839107888291, 0.4324551396795177, 0.5672956822916545)), ('u', 4, (3.4718819387260575, 0.02947903331610299, -2.6838157074667235)), ('u', 4, (-1.091406699900337, 0.5339080579637239, -0.9189972948725385)), ('u', 9, (1.7374314583008175, 0.9553166181245347, -1.7307364819736202)), ('cz', 10, 9), ('u', 10, (1.5836501549235944, 1.5707963267948863, -2.3155200955541257)), ('u', 11, (3.7332420256656658, 1.7773523945178165, -0.03386081663131968)), ('u', 5, (0.21538661477591506, 0.4113744600180603, -0.34624254293845436)), ('u', 9, (-1.6242586207680705, 1.0389838633050952, 2.215966057121418)), ('cz', 10, 11), ('u', 10, (5.4571127491452325, 2.721058318305919, -0.28916993170208904)), ('u', 11, (2.2592387795429465, 0.21428637202578618, 0.20294686331677614)), ('u', 5, (2.2248492570551557, 0.006326700559579445, -3.4638318082638264)), ('u', 9, (-0.15190330840597188, 0.19761040578288908, 0.6375881451188443)), ('cz', 10, 9), ('u', 10, (-2.8524227218865676, 1.5707963267950193, -3.1219919943559007)), ('u', 11, (3.9282494804920547, 1.261933072103709, -0.16726105952307035)), ('u', 5, (0.37581495582603064, 0.061602788949830845, 0.5200417645274498)), ('u', 9, (-2.87127622666875, 0.49923142964298683, 2.590526912552226)), ('cz', 10, 11), ('u', 7, (-2.3826838273729773, 0.3614621303677486, 3.6679233168629457)), ('u', 6, (-2.117311825287107, 0.4784707764587197, 3.3257605396880416)), ('u', 8, (-0.19460765679991376, 2.2555155276143646, -1.0745545039727589)), ('u', 9, (3.692658410843422, 2.64236123131327, 0.577789156914825)), ('u', 10, (1.4216573156780532, 1.5972087127785883, -0.3421645986013977)), ('cz', 8, 9), ('u', 7, (-2.803839419189235, 0.38681601458849474, 3.030512126583069)), ('u', 6, (2.1790864001006938, 0.105280931757105, -3.372111972727743)), ('u', 8, (1.9305927607055218, 1.7060931551459226, -3.9038779750360924)), ('u', 9, (-0.12943900089258248, 1.5707963302372876, 0.15441406876819885)), ('u', 10, (3.370316226114566, 0.8703701847898042, -2.3484747106824795)), ('cz', 9, 10), ('u', 7, (-1.9331757959275457, 0.06497460334715786, 4.021415571172954)), ('u', 6, (0.5146623142675849, 0.5960956973341891, 0.5600591075222124)), ('u', 8, (3.0950235961014423, 0.9417594179099568, -3.0601212492126724)), ('u', 9, (1.4606981058539503, 1.4824244060327743, -2.791005604486636)), ('u', 10, (3.67967266783973, 1.2012399760177852, -0.026828124057411085)), ('cz', 8, 9), ('u', 7, (0.7967178521823098, 0.5595590435546649, 0.5584872677456035)), ('u', 6, (0.5869587804260354, 0.3425174746571694, 0.4998663754048011)), ('u', 8, (0.28416134322110775, 0.5859304775207309, 0.7155658887746387)), ('u', 9, (0.7546019348860217, 1.5707963294074894, 0.1080025331873643)), ('u', 10, (3.6312340204756466, 2.218822206250899, 0.22082375566249723)), ('cz', 9, 10), ('u', 7, (-0.29739162334068575, 1.1569188106046167, -0.34567342919107275)), ('u', 6, (-2.17176373648691, 0.26950397007034277, 4.013742294558004)), ('u', 8, (2.42602669763317, 0.585930440726457, -2.9642780198222116)), ('u', 9, (-0.4025764192654687, 0.8602507361691653, -0.6541050707181572)), ('u', 10, (0.33482194106024116, 0.7988678606778703, -0.9123262952112081)), ('cz', 7, 8), ('u', 7, (1.961376259614753, 0.28771737062356734, -2.439160426997573)), ('u', 6, (3.928509160265549, 0.5403453096396793, -2.164956555426372)), ('u', 8, (-2.50220024419078, 1.570796289267466, 1.9697475212726812)), ('u', 9, (3.7956976517838354, 0.8602504911481038, 2.0903003390181096)), ('u', 10, (0.1667422854318628, 0.4212244323259071, 0.6319477160100364)), ('cz', 8, 9), ('u', 7, (0.25389867024469126, 0.9259588591766047, -0.0033468833353068883)), ('u', 6, (0.02989321524621308, 0.4853535656584459, 0.8848862865935332)), ('u', 8, (2.742641265988369, 0.09883509613255445, -1.242834981752615)), ('u', 9, (-2.2487204471435374, 0.34974952553137933, 2.8484456380868894)), ('u', 10, (0.6533556135922449, 0.0408816143639005, 1.1392550996501511)), ('cz', 7, 8), ('u', 7, (0.044449890330617525, 1.178811477410957, 1.8989588279867933)), ('u', 6, (0.24642226044923082, 0.8205789587523448, 0.4712993552843266)), ('u', 8, (0.19563762230327547, 1.5707963430040812, -0.29476571945863106)), ('u', 9, (-2.450987244661805, 1.7751203341616022, -3.563423005253667)), ('u', 10, (2.783757987575507, 0.6398276150017117, -3.0023055832099788)), ('cz', 8, 9), ('u', 7, (-0.05441389382570283, 2.1499677737520138, 1.5024083017818404)), ('u', 6, (1.0728977528509231, 1.8046682301943775, -1.0990769046686113)), ('u', 8, (-0.3253254327713744, 0.9330732489977708, 0.6196832846765372)), ('u', 9, (2.984469113624575, 1.4322566449347487, 0.40671568017166804)), ('u', 10, (2.9722577060140005, 2.126901131843333, -0.3368647683408992)), ('cz', 7, 6), ('u', 7, (2.068657648059694, 2.526112944241145, -0.7749197163832823)), ('u', 6, (-1.2012507143700044, 1.1557911820253304, 1.7892681735085718)), ('u', 8, (2.8659235425059952, 1.4615199429400298, -0.04315161935545353)), ('u', 9, (0.05969772829934855, 0.2842366423413663, -0.3533574922917047)), ('u', 10, (-0.7849133792153977, 0.5722439810019387, -0.4983751188245813)), ('cz', 7, 8), ('u', 7, (2.961195751559055, 1.5707963255300226, -2.267827886209706)), ('u', 6, (-2.6436835893988873, 0.5407065145299464, 3.429194885428971)), ('u', 8, (3.3149981599564637, 0.1765276993088318, -2.6792082723886588)), ('u', 9, (-2.276095927088909, 0.20638257090595896, 2.7444029347836474)), ('u', 10, (-0.6342088195135234, 0.17137020199951494, -0.15903202768470232)), ('cz', 7, 6), ('u', 7, (0.697031559527799, 1.5707963233180409, 0.7864359358088141)), ('u', 6, (-2.7610863076506127, 0.18956993182958573, 3.173317482760791)), ('u', 8, (3.0030134992637594, 2.019853837753595, 0.060693144736520654)), ('u', 9, (1.5649760761455802, 0.12665166797350452, -2.8465585663904074)), ('u', 10, (-0.027665143099708446, 0.008886920788653758, 0.08953748550762855)), ('cz', 7, 8), ('u', 7, (0.7843603929785936, 0.7853981585720565, 0.3185833349092088)), ('u', 6, (0.8311290249110079, 1.6949250865635417, -0.12159482477920797)), ('u', 8, (3.037067324808663, 1.638393158057371, -0.21503575782517714)), ('u', 9, (2.432588857655999, 0.13984589747431292, -3.2962093419990723)), ('u', 10, (2.6471207837048043, 0.43740011475780394, -3.595422832462038)), ('cz', 7, 6), ('u', 7, (2.823009325410358, 1.5707963279927293, -3.057555881429602)), ('u', 6, (-1.3055524596742032, 1.570796329757634, 3.225629439883212)), ('u', 8, (-2.5765071880059374, 0.07194626556227154, 2.87641337309422)), ('u', 9, (-3.493549131745561, 0.23904733019859017, 2.650540293878046)), ('u', 10, (-0.23764470671062649, 0.4767222271515354, -0.719002478628656)), ('u', 11, (3.504804013006078, 1.5707963267948155, -0.03520372206012401)), ('u', 5, (0.2566753560197081, 0.03854440189796176, 0.2555286240630516)), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0))]), (13, [('u', 5, (0.0, 2.0505249644706494, 0.0)), ('u', 4, (0.0, 1.0321328607141094, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (-3.141592653589793, 1.0321328607141094, 3.141592653589793)), ('u', 11, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 12, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 12), ('u', 5, (-0.026488601017550822, 0.33817761953207987, -0.03727149628245448)), ('u', 4, (0.5261809395945654, 4.583733762573752e-09, -2.075678192631724)), ('u', 3, (1.7279839243247377, 0.9553166181244952, -2.31897812556857)), ('cz', 4, 3), ('u', 5, (3.2069423080124477, 1.9088505746848605, 0.35741556707447986)), ('u', 4, (1.5494972529733677, 1.570796323704034, -0.9700524480599615)), ('u', 3, (-1.2917068473260913, 1.0057498368542224, 1.6663688863848016)), ('cz', 5, 4), ('u', 5, (2.745693454732362, 1.8819958982276148, -0.16966288443458488)), ('u', 4, (-2.1715402047728465, 0.4205343352838054, 3.204649700748775)), ('u', 3, (0.16302187254096098, 0.391758611081972, 0.1327063423358217)), ('cz', 4, 3), ('u', 5, (3.246038069634937, 0.7332181270315529, 0.1516148239645223)), ('u', 4, (-0.06305704776105525, 1.5707963267948473, 0.24332075105407625)), ('u', 3, (0.02391047167407853, 1.2080095905811189e-09, 0.24946083236423147)), ('cz', 5, 4), ('u', 5, (2.863033136373312, 4.4607539425156097e-10, -2.9466331615999555)), ('u', 4, (-3.554819720556316, 1.8175894909190264, 2.2355366836874895)), ('u', 3, (-1.7850661071595355, 3.5701273649817347e-09, 2.990269384106037)), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-0.27113878386298496, 0.8860771237922334, 0.2832335905683325)), ('u', 1, (1.0724667860992239, 0.3135441280757749, 0.3477481597921714)), ('cz', 2, 3), ('u', 2, (2.4400313446652238, 0.4674334689778797, -2.7557448911498277)), ('u', 3, (-1.2466744714517015, 1.5707963269126817, -0.3474883157693149)), ('u', 1, (1.2610848555455991, 0.5154430607864812, 0.4649785201028739)), ('u', 4, (-0.3487503182872954, 1.8124312541247014, -0.029227653884645405)), ('cz', 3, 4), ('u', 2, (3.4924690019924354, 0.487420075135062, -2.0561402443611527)), ('u', 3, (-2.794104337117351, 2.6779450445890216, 2.73086955979565)), ('u', 1, (4.139468277336655, 0.301183724873278, -1.6188205903909119)), ('u', 4, (3.6409272527688463, 0.10783953278100211, -0.27102555731550226)), ('cz', 2, 3), ('u', 2, (1.7718525496675044, 0.3070472199392804, -0.4995159680649769)), ('u', 3, (0.41072309437235316, 1.570796326794958, -0.2380438412443237)), ('u', 1, (4.128297871557185, 0.08416766952170567, -0.9881778941580865)), ('u', 4, (2.793806547308158, 1.1941283984595739, 0.5571280141976567)), ('cz', 3, 4), ('u', 0, (3.1415466473015825, 0.3472761449629368, -2.775502885699414)), ('u', 1, (-1.943048849865654, 0.0866519139605691, 3.3796548616430324)), ('u', 2, (0.49951723708076523, 2.834545501634763, 0.2477359093599764)), ('u', 3, (-2.900546912689433, 0.0220330108239607, 2.9517078164711683)), ('cz', 1, 2), ('u', 0, (-2.7755028856994066, 0.2557486829034849, 3.4795316578682636)), ('u', 1, (-2.287986882890206, 0.8387533584115957, 2.608378634138784)), ('u', 2, (-3.440321172504869, 1.5707963601966048, 2.4810079929535647)), ('u', 3, (1.628741394809413, 0.16973412222193243, 0.14557562970134652)), ('cz', 2, 3), ('u', 0, (-2.8036536493112556, 0.1072802106185393, 3.4534547002776566)), ('u', 1, (-0.42968280486432603, 0.09900030019283891, -0.5979464612539381)), ('u', 2, (-0.9102116414123469, 1.570796303280397, 0.5896891607222703)), ('u', 3, (-1.1994459353604168, 0.8540493077491784, 2.8966776766920646)), ('cz', 1, 2), ('u', 0, (0.3118622548531915, 0.06450072200815428, 0.3158295634276317)), ('u', 1, (4.0056453750975125, 1.5710093821180986, -2.25306084708445)), ('u', 2, (0.457508391465305, 1.5707961085469244, 1.9756186289307065)), ('u', 3, (-0.8946606362517096, 2.7743788682406283, -3.296909735624784)), ('cz', 2, 3), ('u', 0, (1.8861358465261668, 1.0886309990492427, -1.247457185490609)), ('u', 1, (-5.444254626341652, 3.1402271545080267, -0.46524964391451995)), ('u', 2, (0.8866780470332694, 2.7806992103177426, -0.027947407209413355)), ('u', 3, (-0.33059098576245977, 0.23605648596041864, -0.344797406280728)), ('cz', 0, 1), ('u', 0, (1.8941354680991402, 0.5435018820283635, -1.265545217593394)), ('u', 1, (1.5103199473301145, 2.52611291578785, -0.8703574349521468)), ('u', 2, (-0.12699537993299054, 0.7405798026131227, -0.3377410900233119)), ('u', 3, (3.126455392394852, 0.014708899727501449, -3.155938522155425)), ('cz', 1, 2), ('u', 0, (0.5720960810390555, 0.3997299720525652, 0.5480258762238771)), ('u', 1, (-0.08495921531061223, 1.5707965524889658, 0.7853981576711249)), ('u', 2, (-0.1307648484010647, 0.05425977598078749, 0.15618413163899017)), ('u', 3, (3.1272467850241585, 0.03145786231116345, -3.1537851270176698)), ('cz', 0, 1), ('u', 0, (0.5480258241825835, 0.06872222509928691, 0.3748642475219047)), ('u', 1, (0.7853981232512715, 1.570796315327001, -0.7853981651137552)), ('u', 2, (3.297776796581335, 1.006857237795784, 0.017799187962781016)), ('u', 3, (3.129400180161916, 0.04582839097485164, -3.1501000422518555)), ('cz', 1, 2), ('u', 0, (0.3748643116489141, 1.520463564085735, -0.6833998961474932)), ('u', 1, (2.3561944734590226, 0.7853981619176521, -2.993151367895617)), ('u', 2, (3.159391795646975, 1.5652872587775164, -0.6482539739882052)), ('u', 3, (3.1330852649277396, 0.057291232680230515, -3.144741673716754)), ('cz', 0, 1), ('u', 0, (-4.075799266061252, 1.5707959008369972, 1.4311782578899523)), ('u', 1, (2.993151235150358, 1.570796346808479, -1.7104143349925467)), ('u', 2, (-0.5745315317617433, 0.016153120923974765, -0.48767068019668275)), ('u', 3, (3.1384436334628316, 1.63618109680531, 0.9488551956938394)), ('u', 4, (3.269071830129811, 1.479831739000346, 0.17271472025738444)), ('u', 4, (2.443603264821166, 0.26906696464229146, -3.4844556820946466)), ('u', 4, (-0.5656839107888291, 0.4324551396795177, 0.5672956822916545)), ('u', 4, (3.4718819387260575, 0.02947903331610299, -2.6838157074667235)), ('u', 4, (-1.091406699900337, 0.5339080579637239, -0.9189972948725385)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (0.0, 1.5707963267948966, 0.0)), ('u', 11, (0.0, 1.5707963267948966, 0.0)), ('cz', 10, 11), ('u', 11, (-5.039500729646646e-17, 1.0068536854342682, 5.039500729646646e-17)), ('u', 10, (3.141592653589793, 1.0068536854342682, -3.141592653589793)), ('cz', 10, 11), ('u', 11, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 10, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 11, 12), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 11, (-3.141592653589793, 1.183199640139716, 3.141592653589793)), ('cz', 12, 11), ('u', 11, (0.0, 1.183199640139716, 0.0)), ('u', 11, (-3.141592653589793, 2.7539959669346126, 3.141592653589793)), ('cz', 10, 11), ('u', 11, (0.0, 2.7539959669346126, 0.0)), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 11, 12), ('u', 10, (0.0, 1.8078517587411687e-13, 1.5931462207262648)), ('u', 11, (-0.5639284404090444, 0.25097236529063677, 0.02239008214857119)), ('u', 5, (-3.100615969558219, 0.47086568349310587, 3.075883716107122)), ('u', 9, (1.7374314583008175, 0.9553166181245347, -1.7307364819736202)), ('cz', 10, 9), ('u', 10, (1.5836501549235944, 1.5707963267948863, -2.3155200955541257)), ('u', 11, (3.7332420256656658, 1.7773523945178165, -0.03386081663131968)), ('u', 5, (0.21538661477591506, 0.4113744600180603, -0.34624254293845436)), ('u', 9, (-1.6242586207680705, 1.0389838633050952, 2.215966057121418)), ('cz', 10, 11), ('u', 10, (5.4571127491452325, 2.721058318305919, -0.28916993170208904)), ('u', 11, (2.2592387795429465, 0.21428637202578618, 0.20294686331677614)), ('u', 5, (2.2248492570551557, 0.006326700559579445, -3.4638318082638264)), ('u', 9, (-0.15190330840597188, 0.19761040578288908, 0.6375881451188443)), ('cz', 10, 9), ('u', 10, (-2.8524227218865676, 1.5707963267950193, -3.1219919943559007)), ('u', 11, (3.9282494804920547, 1.261933072103709, -0.16726105952307035)), ('u', 5, (0.37581495582603064, 0.061602788949830845, 0.5200417645274498)), ('u', 9, (-2.87127622666875, 0.49923142964298683, 2.590526912552226)), ('cz', 10, 11), ('u', 7, (-2.3826838273729773, 0.3614621303677486, 3.6679233168629457)), ('u', 6, (-2.117311825287107, 0.4784707764587197, 3.3257605396880416)), ('u', 8, (-0.19460765679991376, 2.2555155276143646, -1.0745545039727589)), ('u', 9, (3.692658410843422, 2.64236123131327, 0.577789156914825)), ('u', 10, (1.4216573156780532, 1.5972087127785883, -0.3421645986013977)), ('cz', 8, 9), ('u', 7, (-2.803839419189235, 0.38681601458849474, 3.030512126583069)), ('u', 6, (2.1790864001006938, 0.105280931757105, -3.372111972727743)), ('u', 8, (1.9305927607055218, 1.7060931551459226, -3.9038779750360924)), ('u', 9, (-0.12943900089258248, 1.5707963302372876, 0.15441406876819885)), ('u', 10, (3.370316226114566, 0.8703701847898042, -2.3484747106824795)), ('cz', 9, 10), ('u', 7, (-1.9331757959275457, 0.06497460334715786, 4.021415571172954)), ('u', 6, (0.5146623142675849, 0.5960956973341891, 0.5600591075222124)), ('u', 8, (3.0950235961014423, 0.9417594179099568, -3.0601212492126724)), ('u', 9, (1.4606981058539503, 1.4824244060327743, -2.791005604486636)), ('u', 10, (3.67967266783973, 1.2012399760177852, -0.026828124057411085)), ('cz', 8, 9), ('u', 7, (0.7967178521823098, 0.5595590435546649, 0.5584872677456035)), ('u', 6, (0.5869587804260354, 0.3425174746571694, 0.4998663754048011)), ('u', 8, (0.28416134322110775, 0.5859304775207309, 0.7155658887746387)), ('u', 9, (0.7546019348860217, 1.5707963294074894, 0.1080025331873643)), ('u', 10, (3.6312340204756466, 2.218822206250899, 0.22082375566249723)), ('cz', 9, 10), ('u', 7, (-0.29739162334068575, 1.1569188106046167, -0.34567342919107275)), ('u', 6, (-2.17176373648691, 0.26950397007034277, 4.013742294558004)), ('u', 8, (2.42602669763317, 0.585930440726457, -2.9642780198222116)), ('u', 9, (-0.4025764192654687, 0.8602507361691653, -0.6541050707181572)), ('u', 10, (0.33482194106024116, 0.7988678606778703, -0.9123262952112081)), ('cz', 7, 8), ('u', 7, (1.961376259614753, 0.28771737062356734, -2.439160426997573)), ('u', 6, (3.928509160265549, 0.5403453096396793, -2.164956555426372)), ('u', 8, (-2.50220024419078, 1.570796289267466, 1.9697475212726812)), ('u', 9, (3.7956976517838354, 0.8602504911481038, 2.0903003390181096)), ('u', 10, (0.1667422854318628, 0.4212244323259071, 0.6319477160100364)), ('cz', 8, 9), ('u', 7, (0.25389867024469126, 0.9259588591766047, -0.0033468833353068883)), ('u', 6, (0.02989321524621308, 0.4853535656584459, 0.8848862865935332)), ('u', 8, (2.742641265988369, 0.09883509613255445, -1.242834981752615)), ('u', 9, (-2.2487204471435374, 0.34974952553137933, 2.8484456380868894)), ('u', 10, (0.6533556135922449, 0.0408816143639005, 1.1392550996501511)), ('cz', 7, 8), ('u', 7, (0.044449890330617525, 1.178811477410957, 1.8989588279867933)), ('u', 6, (0.24642226044923082, 0.8205789587523448, 0.4712993552843266)), ('u', 8, (0.19563762230327547, 1.5707963430040812, -0.29476571945863106)), ('u', 9, (-2.450987244661805, 1.7751203341616022, -3.563423005253667)), ('u', 10, (2.783757987575507, 0.6398276150017117, -3.0023055832099788)), ('cz', 8, 9), ('u', 7, (-0.05441389382570283, 2.1499677737520138, 1.5024083017818404)), ('u', 6, (1.0728977528509231, 1.8046682301943775, -1.0990769046686113)), ('u', 8, (-0.3253254327713744, 0.9330732489977708, 0.6196832846765372)), ('u', 9, (2.984469113624575, 1.4322566449347487, 0.40671568017166804)), ('u', 10, (2.9722577060140005, 2.126901131843333, -0.3368647683408992)), ('cz', 7, 6), ('u', 7, (2.068657648059694, 2.526112944241145, -0.7749197163832823)), ('u', 6, (-1.2012507143700044, 1.1557911820253304, 1.7892681735085718)), ('u', 8, (2.8659235425059952, 1.4615199429400298, -0.04315161935545353)), ('u', 9, (0.05969772829934855, 0.2842366423413663, -0.3533574922917047)), ('u', 10, (-0.7849133792153977, 0.5722439810019387, -0.4983751188245813)), ('cz', 7, 8), ('u', 7, (2.961195751559055, 1.5707963255300226, -2.267827886209706)), ('u', 6, (-2.6436835893988873, 0.5407065145299464, 3.429194885428971)), ('u', 8, (3.3149981599564637, 0.1765276993088318, -2.6792082723886588)), ('u', 9, (-2.276095927088909, 0.20638257090595896, 2.7444029347836474)), ('u', 10, (-0.6342088195135234, 0.17137020199951494, -0.15903202768470232)), ('cz', 7, 6), ('u', 7, (0.697031559527799, 1.5707963233180409, 0.7864359358088141)), ('u', 6, (-2.7610863076506127, 0.18956993182958573, 3.173317482760791)), ('u', 8, (3.0030134992637594, 2.019853837753595, 0.060693144736520654)), ('u', 9, (1.5649760761455802, 0.12665166797350452, -2.8465585663904074)), ('u', 10, (-0.027665143099708446, 0.008886920788653758, 0.08953748550762855)), ('cz', 7, 8), ('u', 7, (0.7843603929785936, 0.7853981585720565, 0.3185833349092088)), ('u', 6, (0.8311290249110079, 1.6949250865635417, -0.12159482477920797)), ('u', 8, (3.037067324808663, 1.638393158057371, -0.21503575782517714)), ('u', 9, (2.432588857655999, 0.13984589747431292, -3.2962093419990723)), ('u', 10, (2.6471207837048043, 0.43740011475780394, -3.595422832462038)), ('cz', 7, 6), ('u', 7, (2.823009325410358, 1.5707963279927293, -3.057555881429602)), ('u', 6, (-1.3055524596742032, 1.570796329757634, 3.225629439883212)), ('u', 8, (-2.5765071880059374, 0.07194626556227154, 2.87641337309422)), ('u', 9, (-3.493549131745561, 0.23904733019859017, 2.650540293878046)), ('u', 10, (-0.23764470671062649, 0.4767222271515354, -0.719002478628656)), ('u', 11, (3.504804013006078, 1.5707963267948155, -0.03520372206012401)), ('u', 5, (0.2566753560197081, 0.03854440189796176, 0.2555286240630516)), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0))])], (13, 5): [(13, [('u', 5, (0.0, 2.8854165799347697, 0.0)), ('u', 4, (0.0, 0.4194735802213585, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (-3.141592653589793, 0.4194735802213585, 3.141592653589793)), ('u', 3, (0.0, 0.7825572390207782, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 0.7825572390207782, 3.141592653589793)), ('u', 2, (0.0, 1.0959425247457042, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.0959425247457042, 3.141592653589793)), ('u', 1, (0.0, 1.3361538466137453, 0.0)), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 1), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.3361538466137453, 3.141592653589793)), ('u', 8, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 11, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 12, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 12), ('u', 0, (0.2895396804257915, 0.42053432963085197, 0.2186344537665792)), ('u', 1, (-2.172596698081646, 2.7771176198164215e-09, 2.5876963239152198)), ('u', 2, (-3.228353500519151, 0.14952614669551412, 3.045051894219899)), ('cz', 0, 1), ('u', 0, (2.9792540206708886, 0.8208288587674605, -3.2394362427156973)), ('u', 1, (-0.5189128758343072, 1.5707963278808252, 0.28787753873452127)), ('u', 2, (3.2208886381194444, 1.4212920830783036, 0.02240296332127567)), ('cz', 1, 2), ('u', 0, (0.15665480505572835, 0.40150716667742453, -0.24764615828620704)), ('u', 1, (2.85371509660119, 0.955316603180468, -3.1457179085982)), ('u', 2, (2.87772559676124, 1.3299879385880082, -0.18603273356959749)), ('cz', 0, 1), ('u', 0, (0.5338595447223321, 0.019280386943774562, -0.026246824335227292)), ('u', 1, (0.00412526566754981, 1.5707963120452608, 0.197486676731702)), ('u', 2, (3.119123150084535, 0.7395037326590929, 0.17542817108495523)), ('cz', 1, 2), ('u', 0, (-3.1153459688867526, 0.019280386430582032, 2.8536491653853044)), ('u', 1, (2.944105989056777, 1.5707963266274152, -3.03777941513755)), ('u', 2, (3.3217938815324324, 1.5707963268105951, 0.10381323861495151)), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.1780972450961724, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 0.39269908169872414, 0.0)), ('cz', 3, 2), ('u', 2, (-3.141592653589793, 0.39269908169872414, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 1.1780972450961724, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.2630564724597029, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 0.30773985433519363, 0.0)), ('cz', 4, 3), ('u', 3, (-3.141592653589793, 0.30773985433519363, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 1.2630564724597029, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (-3.141592653589793, 1.3605291591529143, 3.141592653589793)), ('cz', 3, 4), ('u', 4, (0.0, 0.2102671676419825, 0.0)), ('cz', 5, 4), ('u', 4, (-3.141592653589793, 0.2102671676419825, 3.141592653589793)), ('cz', 3, 4), ('u', 4, (0.0, 1.3605291591529143, 0.0)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 5), ('u', 6, (-2.7956955245943567, 0.006638396074130232, 1.9839531812880271)), ('u', 7, (3.3060693547306053, 2.57765001221808, 2.7878048943093074)), ('u', 8, (-0.9802959536225644, 3.1415926212758114, -5.141306813857595)), ('u', 5, (3.1836423183974443, 1.368608423331714, -0.35706132391778844)), ('u', 9, (3.282137172694394, 0.22185546077212015, -2.37617977051081)), ('cz', 7, 8), ('u', 6, (-0.863687536887122, 0.2036612508775408, -0.49096140311386094)), ('u', 7, (2.8866631605901194, 2.0572489031345156, -2.832204402499025)), ('u', 8, (0.8352656017295159, 1.5707963294313407, 0.3463688831494805)), ('u', 5, (-2.899488852373683, 0.023818332706368017, 2.986727851963161)), ('u', 9, (2.4208820653955025, 1.3491553116568324, 0.8806290474140755)), ('cz', 8, 9), ('u', 6, (2.832707855812823, 0.15840937628477464, -3.115598061390125)), ('u', 7, (-2.937596754155037, 0.6717279144270676, 3.0266444728300304)), ('u', 8, (2.7952252909671804, 0.8570719478274053, -3.0533719187797725)), ('u', 5, (0.12071465626978571, 0.013907386342634304, 0.332968572639534)), ('u', 9, (0.9569479039919327, 1.094241866608657, 0.41378535469459476)), ('cz', 7, 8), ('u', 6, (-0.1186231744977771, 0.12081883678314975, 0.8833194869947525)), ('u', 7, (2.8448936708527257, 2.0093937785142094e-08, -3.234615357195483)), ('u', 8, (-0.08822174150423814, 1.5707963267868776, 0.47942481146490246)), ('u', 5, (-0.4232632365402874, 0.07836685227309081, 0.06556582823325244)), ('u', 9, (3.4238520709664293, 1.3817784233230703, 0.7518585830252498)), ('cz', 8, 9), ('u', 6, (3.227019319836882, 0.17251920947071164, -3.001264902666509)), ('u', 7, (-0.4519555042207237, 2.180643064547514e-08, -0.17549316113966626)), ('u', 8, (-0.4794260024013855, 1.5707963376176648, -0.18415260485211554)), ('u', 5, (-0.004473691871499111, 0.13802087244552758, 0.17356360142769778)), ('u', 9, (3.4455507137660275, 1.5707963267535443, -0.1841526048505968)), ('u', 0, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 1, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 0, 1), ('u', 1, (1.570796326794898, 0.4636476090008061, -1.570796326794898)), ('u', 0, (1.5707963267948966, 0.4636476090008061, -1.5707963267948966)), ('cz', 0, 1), ('u', 1, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 0, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 1, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.1277577648985897, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.44303856189630686, 0.0)), ('cz', 2, 1), ('u', 1, (-3.141592653589793, 0.44303856189630686, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 1.1277577648985897, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.2284367252937551, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 0.3423596015011414, 0.0)), ('cz', 3, 2), ('u', 2, (-3.141592653589793, 0.3423596015011414, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 1.2284367252937551, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 0.23182380450040305, 0.0)), ('cz', 4, 3), ('u', 3, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 1.3389725222944935, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (0.0, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.5235987755982989, 0.0)), ('u', 0, (0.0, 0.5235987755982989, 0.0)), ('cz', 0, 1), ('u', 1, (0.0, 1.5707963267948966, 0.0)), ('u', 0, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.1780972450961724, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.39269908169872414, 0.0)), ('cz', 2, 1), ('u', 1, (-3.141592653589793, 0.39269908169872414, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 1.1780972450961724, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.3089969389957472, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 0.26179938779914935, 0.0)), ('cz', 3, 2), ('u', 2, (-3.141592653589793, 0.26179938779914935, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 1.3089969389957472, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 0, (1.5975031134487676, 1.5707963272139616, -1.4583260377988119)), ('u', 1, (-1.54408954149832, 1.5707963198077501, 1.3521023127170535)), ('u', 2, (0.4163910916243424, 0.12604905616327075, 0.1796067758230245)), ('u', 12, (2.7155853457469954, 0.5678988615986397, -3.4592333738342464)), ('u', 11, (-3.2540656422843393, 0.45729772108327515, 2.7432750747445382)), ('cz', 0, 1), ('u', 0, (-1.683266610466608, 0.35087941271618284, 2.248154716411485)), ('u', 1, (1.7894903408157568, 0.9388820122407656, -2.399799194875527)), ('u', 2, (-0.12953338009534995, 0.04267977738907076, -0.21110581637379025)), ('u', 12, (3.1294795206033554, 0.5247634475974824, -2.850546001145028)), ('u', 11, (-0.18611450997874712, 0.04196935913749408, 0.297059632354847)), ('cz', 0, 1), ('u', 0, (-0.781764349030515, 1.3911468260832656, 1.1366236004817467)), ('u', 1, (0.8290028681389747, 1.570796327653262, -1.6679283120759159)), ('u', 2, (2.90762869647199, 1.7258965962616892, 0.35594949100433304)), ('u', 12, (-0.29436611229319126, 0.06336129390386466, 0.12660173736263364)), ('u', 11, (-0.052560879925793835, 0.2925393227860578, -0.34812362747375786)), ('cz', 1, 2), ('u', 0, (-0.606238074805997, 0.20750166415997245, -0.1413497091421183)), ('u', 1, (3.2387246426597587, 1.783726566066143, -2.927107105507331)), ('u', 2, (2.674332934444953, 2.667134756416933, 0.18377461966606456)), ('u', 12, (-0.02517377623569074, 0.22848098844014691, -0.23743011823468702)), ('u', 11, (2.791075631089582, 0.028550721842786404, -2.7857303526618726)), ('cz', 0, 1), ('u', 0, (0.2870163991718453, 0.631329916390911, -1.1575766205601583)), ('u', 1, (-0.2144855484903, 1.5707963257469955, 0.12450393900161129)), ('u', 2, (2.81117069228005, 0.18918583546733062, -3.367085730403926)), ('u', 12, (0.06305385120595786, 0.3166404995005163, -0.2825686347518802)), ('u', 11, (0.45240242421825433, 0.08856480643722661, -0.0622323368282614)), ('cz', 1, 2), ('u', 0, (2.6432153823022575, 0.689887690248816, -2.9753932597393926)), ('u', 1, (3.0170887146218646, 1.219916916051761, -2.907691461433919)), ('u', 2, (0.347432482153075, 0.17008197917746487, 0.10933617373952358)), ('u', 12, (-3.1705340249581955, 0.15694014973217746, 2.83666678471503)), ('u', 11, (3.0697715655147766, 0.13513130316745625, -3.1050068501656094)), ('cz', 0, 1), ('u', 0, (2.883426082382281, 0.4299000998455679, -3.307858478448408)), ('u', 1, (-0.23390119273605664, 1.5707963268173502, 0.10404184919224967)), ('u', 2, (3.0189204714027347, 1.7408634733534913, 0.3533525795306547)), ('u', 12, (0.03740488863901909, 0.08416531328752182, 0.04261917041794456)), ('u', 11, (-0.1945646232439777, 0.10244344759876793, -0.07270488527152388)), ('cz', 1, 2), ('u', 0, (-0.2752404986777236, 0.8877076644874999, -0.06943074545980613)), ('u', 1, (-0.10404184921858331, 1.0471975512176015, -0.0468480720739655)), ('u', 2, (2.771496108773465, 1.3600215597856466, -0.015972367652567376)), ('u', 12, (0.48261676861561076, 0.49026342000038897, 0.6205389806682916)), ('u', 11, (0.17733583642481887, 0.013820878347285374, 0.3781592630021231)), ('cz', 0, 1), ('u', 0, (2.8468449871901704, 1.5707963271498533, -3.1682994374420446)), ('u', 1, (-3.09474457775648, 1.5707963272724914, 3.114885866951368)), ('u', 2, (0.08510928202367114, 0.21126464460018773, -0.09431127670325377)), ('u', 12, (0.8467525967881754, 0.23954980321235764, -0.6368784966259559)), ('u', 11, (3.255697208449639, 1.460876724969107, 0.11052641477944514)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (-3.141592653589793, 1.2139341373225139, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 0.35686218947238274, 0.0)), ('cz', 10, 9), ('u', 9, (-3.141592653589793, 0.35686218947238274, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 1.2139341373225139, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 10, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (-3.141592653589793, 1.2888250061145825, 3.141592653589793)), ('cz', 9, 10), ('u', 10, (0.0, 0.2819713206803144, 0.0)), ('cz', 11, 10), ('u', 10, (-3.141592653589793, 0.2819713206803144, 3.141592653589793)), ('cz', 9, 10), ('u', 7, (-0.0010222009133052934, 1.3041008747344632e-09, 0.2003370551542191)), ('u', 8, (0.023930023575935412, 0.2953173773967398, -0.12864126913934845)), ('u', 9, (-0.030708083689677756, 0.12471016231533803, -0.5936932650946682)), ('u', 6, (0.07121035154428908, 0.4205343267071202, -0.2832141972956279)), ('u', 4, (0.07530003087270047, 0.16373930134914844, -0.26222930989690885)), ('cz', 7, 6), ('u', 7, (-0.12687198917430412, 1.5707963360271298, 0.2038898277475962)), ('u', 8, (3.2812013072679873, 1.866096954858985, 0.30184449475142205)), ('u', 9, (-0.2750715762601466, 0.1444293365187298, -0.5619473575075802)), ('u', 6, (-3.0846318122760255, 0.7274540347580297, 2.685237625937257)), ('u', 4, (-2.6959001663774766, 0.023390897362289824, 3.3774526330675045)), ('cz', 7, 8), ('u', 7, (2.937702797807351, 0.9553165545708295, -3.0503475368792787)), ('u', 8, (3.3207345893489655, 1.8839026748016603, -0.007233571707222719)), ('u', 9, (0.7067021023968258, 0.1861341116737401, 0.6824799267543968)), ('u', 6, (0.16900917346851702, 0.3290496173131335, 0.14955942962615904)), ('u', 4, (0.13632596908237446, 0.5047675908256389, 0.19005678333059065)), ('cz', 7, 6), ('u', 7, (-0.0912450930625181, 1.5707962664208395, 0.03309400916940823)), ('u', 8, (3.425558647277073, 1.3322673600128185, -0.027285494896139628)), ('u', 9, (2.9343637010964203, 0.3849224661080576, -3.181083530011608)), ('u', 6, (3.0294580093363934, 0.13808492223951388, -3.0008784410797094)), ('u', 4, (2.808193939953363, 0.34629612135864385, -3.327594214191119)), ('cz', 7, 8), ('u', 7, (-3.0107057097410745, 1.034782101861369, 3.2649574529238565)), ('u', 8, (2.701874333399131, 1.5707963333587613, 0.01699495886877811)), ('u', 9, (3.0218224318300324, 1.613517291586055, 0.20240124091333955)), ('u', 6, (-0.19260732367119549, 0.0506388373239212, 0.49460918963778)), ('u', 4, (3.2519730805535927, 0.002414860976536661, -2.897569812225165)), ('cz', 8, 9), ('u', 7, (2.704908443716258, 0.5581851719223989, -3.2239761040741026)), ('u', 8, (-2.7715068751111733, 1.1780352387189146, 2.8119235453642313)), ('u', 9, (3.2650506749779575, 1.7643530058585124, 0.13265112018338687)), ('u', 6, (0.524846134475885, 0.13013042238544234, 0.6969355712505834)), ('u', 4, (0.31766617169078154, 0.29627032380597074, -0.25692668698661764)), ('cz', 7, 8), ('u', 7, (2.9242210810494496, 0.07895765790608032, -3.0403033743301244)), ('u', 8, (0.28854329030454917, 0.39306029653695296, -0.018521965452702627)), ('u', 9, (-2.4666500872262773, 0.27631055933330684, 3.652088429466448)), ('u', 6, (-2.8645903242081086, 0.00811891627662618, 2.960010214724399)), ('u', 4, (-3.0255760348888474, 0.3172366772250132, 2.742544186481971)), ('cz', 8, 9), ('u', 7, (-0.10128924503841866, 0.07895765916161333, -0.54812026936145)), ('u', 8, (-3.1230706552997467, 0.39306028706200213, 2.9768939074869922)), ('u', 9, (0.22834942953139625, 0.16951111093992396, -0.46976825774972053)), ('u', 6, (-0.05157190861832458, 0.2831035587339365, 0.040556641699272526)), ('u', 4, (-2.9585527433127945, 0.18789853193368702, 3.150001059291019)), ('cz', 7, 8), ('u', 7, (0.30452840482299254, 1.497244522071447e-07, -0.21779698434427083)), ('u', 8, (0.20582453884073654, 1.1780352293228151, 0.36255569092721357)), ('u', 9, (2.5401593790964863, 1.6526078360053855, 0.15843675001271862)), ('u', 6, (-3.2179817957037806, 0.2740507804104168, 2.703112265700859)), ('u', 4, (-3.275513452054124, 0.15846558147607348, 2.9939277750514792)), ('cz', 8, 9), ('u', 7, (3.459547899859897, 1.5115771072655002e-07, -2.5026524551797933)), ('u', 8, (2.92342169170971, 2.5221885901529127e-09, -3.245476641550241)), ('u', 9, (2.834527942346747, 1.5707963245157934, 0.01822865285915709)), ('u', 6, (-2.6012871529086037, 0.10687131265079321, 1.7398627461175855)), ('u', 4, (0.05015276160771126, 0.10355943414250254, -0.4269578465944131)), ('u', 10, (0.0, 1.2888250061145825, 0.0)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 10, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 11, 12), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 11, (-3.141592653589793, 1.3769979834673063, 3.141592653589793)), ('cz', 10, 11), ('u', 11, (0.0, 0.1937983433275903, 0.0)), ('cz', 12, 11), ('u', 11, (-3.141592653589793, 0.1937983433275903, 3.141592653589793)), ('cz', 10, 11), ('u', 11, (0.0, 1.3769979834673063, 0.0)), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 11, 12), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (-3.141592653589793, 1.2630564724597029, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 0.30773985433519363, 0.0)), ('cz', 10, 9), ('u', 9, (-3.141592653589793, 0.30773985433519363, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 1.2630564724597029, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 10, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (-3.141592653589793, 1.3605291591529143, 3.141592653589793)), ('cz', 9, 10), ('u', 10, (0.0, 0.2102671676419825, 0.0)), ('cz', 11, 10), ('u', 10, (-3.141592653589793, 0.2102671676419825, 3.141592653589793)), ('cz', 9, 10), ('u', 10, (0.0, 1.3605291591529143, 0.0)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 10, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 6, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 7, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 6, 7), ('u', 7, (1.570796326794898, 0.4636476090008061, -1.570796326794898)), ('u', 6, (1.5707963267948966, 0.4636476090008061, -1.5707963267948966)), ('cz', 6, 7), ('u', 7, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 6, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 7, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (-3.141592653589793, 1.1277577648985897, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (0.0, 0.44303856189630686, 0.0)), ('cz', 8, 7), ('u', 7, (-3.141592653589793, 0.44303856189630686, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (0.0, 1.1277577648985897, 0.0)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (-3.141592653589793, 1.2284367252937551, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 0.3423596015011414, 0.0)), ('cz', 9, 8), ('u', 8, (-3.141592653589793, 0.3423596015011414, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 1.2284367252937551, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 0.23182380450040305, 0.0)), ('cz', 10, 9), ('u', 9, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 1.3389725222944935, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 6, (-0.10337358284045067, 1.5707963975858181, 1.015835984878588e-06)), ('u', 7, (3.038218917454895, 1.5707961616258277, -3.1415926581399134)), ('u', 8, (0.019828976040683857, 0.013724809664014377, 0.01783890191797663)), ('u', 9, (0.0035274908852903856, 0.07434633330967455, 0.0010581438542694391)), ('cz', 6, 7), ('u', 6, (5.959103240659553e-07, 0.5235987669981246, 0.0008503728370569114)), ('u', 7, (-7.864374259467977e-07, 0.5235987850137847, 2.1514876648522244e-08)), ('u', 8, (0.017838485853208828, 3.491322085471117e-05, 0.015992763508435617)), ('u', 9, (0.0010573111958433138, 0.055537721042183685, -0.00015720448849646705)), ('cz', 6, 7), ('u', 6, (-3.1407408931064547, 1.561653060584754, 3.108194496565501)), ('u', 7, (1.2678335777061012e-07, 1.5707963436427994, 0.1330778157276038)), ('u', 8, (3.1575855108403723, 1.5845243547676091, 0.00046389001286240017)), ('u', 9, (-0.00015720450130723025, 0.03568220976022034, 0.0007906627104473196)), ('cz', 7, 8), ('u', 6, (-0.03339896159371908, 0.0009456606723681459, -0.053307384901516736)), ('u', 7, (-3.008514747676956, 1.1780973866952744, 3.1415924849124677)), ('u', 8, (3.1420579084520557, 1.5707964334536435, 0.013241139958231063)), ('u', 9, (0.0007889975112366238, 0.012161875644356494, 0.005072193771559766)), ('cz', 7, 8), ('u', 6, (-3.1948983668400106, 0.01024058544541974, 3.0631718212986123)), ('u', 7, (3.141592626777089, 1.9634956698150956, -3.141592497081211)), ('u', 8, (0.013241406399454505, 0.811795812113978, -5.1901048898732316e-08)), ('u', 9, (3.14666567994145, 0.0183872739501306, -3.1274161263819162)), ('cz', 6, 7), ('u', 6, (-0.07842081277498814, 0.0636763764592665, -0.11265482417680511)), ('u', 7, (-1.5174894674825287e-07, 2.748894824926096, 0.004717116185436816)), ('u', 8, (3.1415931115099625, 0.8117960196269799, 0.0024698603439405886)), ('u', 9, (3.155767515687325, 0.057705323320175354, -3.1121926183807496)), ('cz', 7, 8), ('u', 6, (-0.11265400467920675, 0.036843437586738984, -0.14917897433769264)), ('u', 7, (0.004716613199078894, 0.21246252000648394, -0.010068637593983236)), ('u', 8, (0.0024699010658877363, 3.723199423905932e-07, 3.156587806026825)), ('u', 9, (3.1709922127383923, 1.6717308694221615, 0.006357761268223072)), ('cz', 8, 9), ('u', 6, (-0.149179819856092, 0.0073166448209931205, -0.18873465895337813)), ('u', 7, (-0.010068429561566238, 0.180256711353873, -0.025384936550044934)), ('u', 8, (-3.1265982994149426, 2.879792808347136, 3.14262416800779)), ('u', 9, (3.147950783718318, 1.5810194225609866, 0.0051794225141676975)), ('cz', 7, 8), ('u', 6, (-3.3303273125335644, 0.012528338239947763, 2.9115537090683197)), ('u', 7, (-0.02538514458468043, 8.543096018313841e-07, -0.020319696249047118)), ('u', 8, (0.0010318785259384909, 1.8325959332673774, 0.00398736224364416)), ('u', 9, (-3.1364136051101927, 0.010223660278832961, 3.135896011527404)), ('cz', 8, 9), ('u', 6, (-3.371630765613361, 0.01443821671985917, 2.871845946781179)), ('u', 7, (-0.02032094450144711, 1.8712083349506447e-07, -0.01330079721428616)), ('u', 8, (-3.137607417068711, 0.26180723510489173, 3.137741517763965)), ('u', 9, (-0.005695809571485449, 0.00403625252390588, 0.011325767365440383)), ('cz', 7, 8), ('u', 6, (-0.2697483848758344, 0.0015061147350338621, -0.341535392261783)), ('u', 7, (-3.1548932427496057, 0.006187151643163442, 3.1248315042568446)), ('u', 8, (-0.003849809993246951, 1.308996287788808, 0.01065630030694045)), ('u', 9, (3.1529182975762176, 1.5748316101961628, 0.0014652491448243765)), ('cz', 8, 9), ('u', 6, (1.2292585095449935, 1.5295686302113742, -1.5278527879141417)), ('u', 7, (-1.5875594496229066, 1.5705884974312958, 1.5677044976095087)), ('u', 8, (-3.1309342313072106, 0.49019375451821573, 3.0891773474553403)), ('u', 9, (3.1430586698649674, 1.543729458425524, 0.05079693796372453)), ('cz', 6, 7), ('u', 6, (1.613738941174663, 0.6154797193336008, -1.5693249595122236)), ('u', 7, (1.5677041944582901, 0.6154796712782338, -1.0941551216460745)), ('u', 8, (3.089176372066575, 0.5197091220145889, -3.1124591545135134)), ('u', 9, (-3.0907953812292854, 0.02205876862949908, 3.1797309234268294)), ('cz', 6, 7), ('u', 6, (-1.5693234376800447, 1.5710268263607212, 1.5320474481219526)), ('u', 7, (2.0474392804567048, 1.6465095989641034, -1.5976301429120348)), ('u', 8, (3.170724942066729, 2.133202830195918, 0.04435021715472276)), ('u', 9, (-3.103454591900695, 0.015664100610430897, 3.16660790256985)), ('cz', 7, 8), ('u', 6, (3.1028434940412866, 0.0029342574022471267, -3.170238681167906)), ('u', 7, (3.1147605899518007, 0.9553166913707994, -3.159526019221204)), ('u', 8, (3.1859433528692485, 1.5707966240942846, -0.00899140872696691)), ('u', 9, (-3.116576572034678, 0.00825150306986584, 3.153170208668008)), ('cz', 7, 8), ('u', 6, (3.1063913060080957, 1.881494231841266e-05, -3.1706637323774425)), ('u', 7, (3.124081332256533, 0.7853981594575237, -3.1415930460131394)), ('u', 8, (3.084263402421209, 0.022685295311754557, -3.10842371493674)), ('u', 9, (-0.0019941521657814727, 0.0006813777759123335, -0.01574928663732262)), ('cz', 6, 7), ('u', 6, (3.1125213667304164, 0.3092041646086194, -3.12625345487787)), ('u', 7, (-4.080796685731872e-08, 0.7853981558924876, 7.772173463874602e-07)), ('u', 8, (3.1747613891209787, 1.5481628904529872, -0.009805768630532175)), ('u', 9, (-0.015748454043375226, 0.008431297425970982, -0.0295910929262283)), ('cz', 7, 8), ('u', 6, (0.01533891114478319, 1.2643549581204578, 0.0057153799668915855)), ('u', 7, (-5.018874719283185e-07, 0.7853980462868956, 6.318653877145332e-07)), ('u', 8, (3.131787219964067, 2.0895288501316824, 0.0025644477643140817)), ('u', 9, (-0.029591092924892516, 0.015601263069330702, -0.04344267924801568)), ('cz', 6, 7), ('u', 6, (-3.1358763000417835, 0.7853983234787361, 3.141594348345165)), ('u', 7, (-7.034785056979964e-07, 1.0471960345973994, 3.6104177568044136e-07)), ('u', 8, (0.0025637937964505214, 0.5245194617938411, -0.00442326163518103)), ('u', 9, (-0.043441430274428344, 0.02175985552611398, -0.05718132719321954)), ('cz', 6, 7), ('u', 6, (3.14159139058198, 0.6154825200535677, -3.1045753624373456)), ('u', 7, (3.141592370965353, 1.570796391465584, -3.1045751262997925)), ('u', 8, (-0.004423053508179188, 0.5275553257727209, -0.011433642825968456)), ('u', 9, (-0.057182576164625784, 0.026436670338707852, -0.07062164023534258)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0))]), (13, [('u', 5, (0.0, 2.8854165799347697, 0.0)), ('u', 4, (0.0, 0.4194735802213585, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (-3.141592653589793, 0.4194735802213585, 3.141592653589793)), ('u', 3, (0.0, 0.7825572390207782, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 0.7825572390207782, 3.141592653589793)), ('u', 2, (0.0, 1.0959425247457042, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.0959425247457042, 3.141592653589793)), ('u', 1, (0.0, 1.3361538466137453, 0.0)), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 1), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.3361538466137453, 3.141592653589793)), ('u', 8, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 11, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 12, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 12), ('u', 0, (0.2895396804257915, 0.42053432963085197, 0.2186344537665792)), ('u', 1, (-2.172596698081646, 2.7771176198164215e-09, 2.5876963239152198)), ('u', 2, (-3.228353500519151, 0.14952614669551412, 3.045051894219899)), ('cz', 0, 1), ('u', 0, (2.9792540206708886, 0.8208288587674605, -3.2394362427156973)), ('u', 1, (-0.5189128758343072, 1.5707963278808252, 0.28787753873452127)), ('u', 2, (3.2208886381194444, 1.4212920830783036, 0.02240296332127567)), ('cz', 1, 2), ('u', 0, (0.15665480505572835, 0.40150716667742453, -0.24764615828620704)), ('u', 1, (2.85371509660119, 0.955316603180468, -3.1457179085982)), ('u', 2, (2.87772559676124, 1.3299879385880082, -0.18603273356959749)), ('cz', 0, 1), ('u', 0, (0.5338595447223321, 0.019280386943774562, -0.026246824335227292)), ('u', 1, (0.00412526566754981, 1.5707963120452608, 0.197486676731702)), ('u', 2, (3.119123150084535, 0.7395037326590929, 0.17542817108495523)), ('cz', 1, 2), ('u', 0, (-3.1153459688867526, 0.019280386430582032, 2.8536491653853044)), ('u', 1, (2.944105989056777, 1.5707963266274152, -3.03777941513755)), ('u', 2, (3.3217938815324324, 1.5707963268105951, 0.10381323861495151)), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.1780972450961724, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 0.39269908169872414, 0.0)), ('cz', 3, 2), ('u', 2, (-3.141592653589793, 0.39269908169872414, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 1.1780972450961724, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.2630564724597029, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 0.30773985433519363, 0.0)), ('cz', 4, 3), ('u', 3, (-3.141592653589793, 0.30773985433519363, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 1.2630564724597029, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (-3.141592653589793, 1.3605291591529143, 3.141592653589793)), ('cz', 3, 4), ('u', 4, (0.0, 0.2102671676419825, 0.0)), ('cz', 5, 4), ('u', 4, (-3.141592653589793, 0.2102671676419825, 3.141592653589793)), ('cz', 3, 4), ('u', 4, (0.0, 1.3605291591529143, 0.0)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 5), ('u', 6, (-2.7956955245943567, 0.006638396074130232, 1.9839531812880271)), ('u', 7, (3.3060693547306053, 2.57765001221808, 2.7878048943093074)), ('u', 8, (-0.9802959536225644, 3.1415926212758114, -5.141306813857595)), ('u', 5, (3.1836423183974443, 1.368608423331714, -0.35706132391778844)), ('u', 9, (3.282137172694394, 0.22185546077212015, -2.37617977051081)), ('cz', 7, 8), ('u', 6, (-0.863687536887122, 0.2036612508775408, -0.49096140311386094)), ('u', 7, (2.8866631605901194, 2.0572489031345156, -2.832204402499025)), ('u', 8, (0.8352656017295159, 1.5707963294313407, 0.3463688831494805)), ('u', 5, (-2.899488852373683, 0.023818332706368017, 2.986727851963161)), ('u', 9, (2.4208820653955025, 1.3491553116568324, 0.8806290474140755)), ('cz', 8, 9), ('u', 6, (2.832707855812823, 0.15840937628477464, -3.115598061390125)), ('u', 7, (-2.937596754155037, 0.6717279144270676, 3.0266444728300304)), ('u', 8, (2.7952252909671804, 0.8570719478274053, -3.0533719187797725)), ('u', 5, (0.12071465626978571, 0.013907386342634304, 0.332968572639534)), ('u', 9, (0.9569479039919327, 1.094241866608657, 0.41378535469459476)), ('cz', 7, 8), ('u', 6, (-0.1186231744977771, 0.12081883678314975, 0.8833194869947525)), ('u', 7, (2.8448936708527257, 2.0093937785142094e-08, -3.234615357195483)), ('u', 8, (-0.08822174150423814, 1.5707963267868776, 0.47942481146490246)), ('u', 5, (-0.4232632365402874, 0.07836685227309081, 0.06556582823325244)), ('u', 9, (3.4238520709664293, 1.3817784233230703, 0.7518585830252498)), ('cz', 8, 9), ('u', 6, (3.227019319836882, 0.17251920947071164, -3.001264902666509)), ('u', 7, (-0.4519555042207237, 2.180643064547514e-08, -0.17549316113966626)), ('u', 8, (-0.4794260024013855, 1.5707963376176648, -0.18415260485211554)), ('u', 5, (-0.004473691871499111, 0.13802087244552758, 0.17356360142769778)), ('u', 9, (3.4455507137660275, 1.5707963267535443, -0.1841526048505968)), ('u', 0, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 1, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 0, 1), ('u', 1, (1.570796326794898, 0.4636476090008061, -1.570796326794898)), ('u', 0, (1.5707963267948966, 0.4636476090008061, -1.5707963267948966)), ('cz', 0, 1), ('u', 1, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 0, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 1, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.1277577648985897, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.44303856189630686, 0.0)), ('cz', 2, 1), ('u', 1, (-3.141592653589793, 0.44303856189630686, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 1.1277577648985897, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.2284367252937551, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 0.3423596015011414, 0.0)), ('cz', 3, 2), ('u', 2, (-3.141592653589793, 0.3423596015011414, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 1.2284367252937551, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 0.23182380450040305, 0.0)), ('cz', 4, 3), ('u', 3, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 1.3389725222944935, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (0.0, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.5235987755982989, 0.0)), ('u', 0, (0.0, 0.5235987755982989, 0.0)), ('cz', 0, 1), ('u', 1, (0.0, 1.5707963267948966, 0.0)), ('u', 0, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.1780972450961724, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.39269908169872414, 0.0)), ('cz', 2, 1), ('u', 1, (-3.141592653589793, 0.39269908169872414, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 1.1780972450961724, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.3089969389957472, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 0.26179938779914935, 0.0)), ('cz', 3, 2), ('u', 2, (-3.141592653589793, 0.26179938779914935, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 1.3089969389957472, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 1, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 0, 1), ('u', 1, (1.570796326794898, 0.6154797086703868, -1.570796326794898)), ('u', 0, (1.5707963267948966, 0.6154797086703876, -1.5707963267948966)), ('cz', 0, 1), ('u', 1, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 0, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 1, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.2630564724597029, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.30773985433519363, 0.0)), ('cz', 2, 1), ('u', 1, (-3.141592653589793, 0.30773985433519363, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 1.2630564724597029, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (0.0, 1.5707963267948966, 0.0)), ('u', 1, (0.0, 1.5707963267948966, 0.0)), ('cz', 0, 1), ('u', 1, (1.2251903831398114e-16, 0.7853981633974483, -1.1496735851465472e-17)), ('u', 0, (-3.141592653589793, 0.7853981633974483, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 0, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (-3.141592653589793, 1.2139341373225139, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 0.35686218947238274, 0.0)), ('cz', 10, 9), ('u', 9, (-3.141592653589793, 0.35686218947238274, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 1.2139341373225139, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 10, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (-3.141592653589793, 1.2888250061145825, 3.141592653589793)), ('cz', 9, 10), ('u', 10, (0.0, 0.2819713206803144, 0.0)), ('cz', 11, 10), ('u', 10, (-3.141592653589793, 0.2819713206803144, 3.141592653589793)), ('cz', 9, 10), ('u', 7, (-0.0010222009133052934, 1.3041008747344632e-09, 0.2003370551542191)), ('u', 8, (0.023930023575935412, 0.2953173773967398, -0.12864126913934845)), ('u', 9, (-0.030708083689677756, 0.12471016231533803, -0.5936932650946682)), ('u', 6, (0.07121035154428908, 0.4205343267071202, -0.2832141972956279)), ('u', 4, (0.07530003087270047, 0.16373930134914844, -0.26222930989690885)), ('cz', 7, 6), ('u', 7, (-0.12687198917430412, 1.5707963360271298, 0.2038898277475962)), ('u', 8, (3.2812013072679873, 1.866096954858985, 0.30184449475142205)), ('u', 9, (-0.2750715762601466, 0.1444293365187298, -0.5619473575075802)), ('u', 6, (-3.0846318122760255, 0.7274540347580297, 2.685237625937257)), ('u', 4, (-2.6959001663774766, 0.023390897362289824, 3.3774526330675045)), ('cz', 7, 8), ('u', 7, (2.937702797807351, 0.9553165545708295, -3.0503475368792787)), ('u', 8, (3.3207345893489655, 1.8839026748016603, -0.007233571707222719)), ('u', 9, (0.7067021023968258, 0.1861341116737401, 0.6824799267543968)), ('u', 6, (0.16900917346851702, 0.3290496173131335, 0.14955942962615904)), ('u', 4, (0.13632596908237446, 0.5047675908256389, 0.19005678333059065)), ('cz', 7, 6), ('u', 7, (-0.0912450930625181, 1.5707962664208395, 0.03309400916940823)), ('u', 8, (3.425558647277073, 1.3322673600128185, -0.027285494896139628)), ('u', 9, (2.9343637010964203, 0.3849224661080576, -3.181083530011608)), ('u', 6, (3.0294580093363934, 0.13808492223951388, -3.0008784410797094)), ('u', 4, (2.808193939953363, 0.34629612135864385, -3.327594214191119)), ('cz', 7, 8), ('u', 7, (-3.0107057097410745, 1.034782101861369, 3.2649574529238565)), ('u', 8, (2.701874333399131, 1.5707963333587613, 0.01699495886877811)), ('u', 9, (3.0218224318300324, 1.613517291586055, 0.20240124091333955)), ('u', 6, (-0.19260732367119549, 0.0506388373239212, 0.49460918963778)), ('u', 4, (3.2519730805535927, 0.002414860976536661, -2.897569812225165)), ('cz', 8, 9), ('u', 7, (2.704908443716258, 0.5581851719223989, -3.2239761040741026)), ('u', 8, (-2.7715068751111733, 1.1780352387189146, 2.8119235453642313)), ('u', 9, (3.2650506749779575, 1.7643530058585124, 0.13265112018338687)), ('u', 6, (0.524846134475885, 0.13013042238544234, 0.6969355712505834)), ('u', 4, (0.31766617169078154, 0.29627032380597074, -0.25692668698661764)), ('cz', 7, 8), ('u', 7, (2.9242210810494496, 0.07895765790608032, -3.0403033743301244)), ('u', 8, (0.28854329030454917, 0.39306029653695296, -0.018521965452702627)), ('u', 9, (-2.4666500872262773, 0.27631055933330684, 3.652088429466448)), ('u', 6, (-2.8645903242081086, 0.00811891627662618, 2.960010214724399)), ('u', 4, (-3.0255760348888474, 0.3172366772250132, 2.742544186481971)), ('cz', 8, 9), ('u', 7, (-0.10128924503841866, 0.07895765916161333, -0.54812026936145)), ('u', 8, (-3.1230706552997467, 0.39306028706200213, 2.9768939074869922)), ('u', 9, (0.22834942953139625, 0.16951111093992396, -0.46976825774972053)), ('u', 6, (-0.05157190861832458, 0.2831035587339365, 0.040556641699272526)), ('u', 4, (-2.9585527433127945, 0.18789853193368702, 3.150001059291019)), ('cz', 7, 8), ('u', 7, (0.30452840482299254, 1.497244522071447e-07, -0.21779698434427083)), ('u', 8, (0.20582453884073654, 1.1780352293228151, 0.36255569092721357)), ('u', 9, (2.5401593790964863, 1.6526078360053855, 0.15843675001271862)), ('u', 6, (-3.2179817957037806, 0.2740507804104168, 2.703112265700859)), ('u', 4, (-3.275513452054124, 0.15846558147607348, 2.9939277750514792)), ('cz', 8, 9), ('u', 7, (3.459547899859897, 1.5115771072655002e-07, -2.5026524551797933)), ('u', 8, (2.92342169170971, 2.5221885901529127e-09, -3.245476641550241)), ('u', 9, (2.834527942346747, 1.5707963245157934, 0.01822865285915709)), ('u', 6, (-2.6012871529086037, 0.10687131265079321, 1.7398627461175855)), ('u', 4, (0.05015276160771126, 0.10355943414250254, -0.4269578465944131)), ('u', 10, (0.0, 1.2888250061145825, 0.0)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 10, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 11, 12), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 11, (-3.141592653589793, 1.3769979834673063, 3.141592653589793)), ('cz', 10, 11), ('u', 11, (0.0, 0.1937983433275903, 0.0)), ('cz', 12, 11), ('u', 11, (-3.141592653589793, 0.1937983433275903, 3.141592653589793)), ('cz', 10, 11), ('u', 11, (0.0, 1.3769979834673063, 0.0)), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 11, 12), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (-3.141592653589793, 1.2630564724597029, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 0.30773985433519363, 0.0)), ('cz', 10, 9), ('u', 9, (-3.141592653589793, 0.30773985433519363, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 1.2630564724597029, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 10, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (-3.141592653589793, 1.3605291591529143, 3.141592653589793)), ('cz', 9, 10), ('u', 10, (0.0, 0.2102671676419825, 0.0)), ('cz', 11, 10), ('u', 10, (-3.141592653589793, 0.2102671676419825, 3.141592653589793)), ('cz', 9, 10), ('u', 10, (0.0, 1.3605291591529143, 0.0)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 10, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 6, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 7, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 6, 7), ('u', 7, (1.570796326794898, 0.4636476090008061, -1.570796326794898)), ('u', 6, (1.5707963267948966, 0.4636476090008061, -1.5707963267948966)), ('cz', 6, 7), ('u', 7, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 6, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 7, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (-3.141592653589793, 1.1277577648985897, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (0.0, 0.44303856189630686, 0.0)), ('cz', 8, 7), ('u', 7, (-3.141592653589793, 0.44303856189630686, 3.141592653589793)), ('cz', 6, 7), ('u', 7, (0.0, 1.1277577648985897, 0.0)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (-3.141592653589793, 1.2284367252937551, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 0.3423596015011414, 0.0)), ('cz', 9, 8), ('u', 8, (-3.141592653589793, 0.3423596015011414, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 1.2284367252937551, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 0.23182380450040305, 0.0)), ('cz', 10, 9), ('u', 9, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 1.3389725222944935, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 6, (-0.10337358284045067, 1.5707963975858181, 1.015835984878588e-06)), ('u', 7, (3.038218917454895, 1.5707961616258277, -3.1415926581399134)), ('u', 8, (0.019828976040683857, 0.013724809664014377, 0.01783890191797663)), ('u', 9, (0.0035274908852903856, 0.07434633330967455, 0.0010581438542694391)), ('cz', 6, 7), ('u', 6, (5.959103240659553e-07, 0.5235987669981246, 0.0008503728370569114)), ('u', 7, (-7.864374259467977e-07, 0.5235987850137847, 2.1514876648522244e-08)), ('u', 8, (0.017838485853208828, 3.491322085471117e-05, 0.015992763508435617)), ('u', 9, (0.0010573111958433138, 0.055537721042183685, -0.00015720448849646705)), ('cz', 6, 7), ('u', 6, (-3.1407408931064547, 1.561653060584754, 3.108194496565501)), ('u', 7, (1.2678335777061012e-07, 1.5707963436427994, 0.1330778157276038)), ('u', 8, (3.1575855108403723, 1.5845243547676091, 0.00046389001286240017)), ('u', 9, (-0.00015720450130723025, 0.03568220976022034, 0.0007906627104473196)), ('cz', 7, 8), ('u', 6, (-0.03339896159371908, 0.0009456606723681459, -0.053307384901516736)), ('u', 7, (-3.008514747676956, 1.1780973866952744, 3.1415924849124677)), ('u', 8, (3.1420579084520557, 1.5707964334536435, 0.013241139958231063)), ('u', 9, (0.0007889975112366238, 0.012161875644356494, 0.005072193771559766)), ('cz', 7, 8), ('u', 6, (-3.1948983668400106, 0.01024058544541974, 3.0631718212986123)), ('u', 7, (3.141592626777089, 1.9634956698150956, -3.141592497081211)), ('u', 8, (0.013241406399454505, 0.811795812113978, -5.1901048898732316e-08)), ('u', 9, (3.14666567994145, 0.0183872739501306, -3.1274161263819162)), ('cz', 6, 7), ('u', 6, (-0.07842081277498814, 0.0636763764592665, -0.11265482417680511)), ('u', 7, (-1.5174894674825287e-07, 2.748894824926096, 0.004717116185436816)), ('u', 8, (3.1415931115099625, 0.8117960196269799, 0.0024698603439405886)), ('u', 9, (3.155767515687325, 0.057705323320175354, -3.1121926183807496)), ('cz', 7, 8), ('u', 6, (-0.11265400467920675, 0.036843437586738984, -0.14917897433769264)), ('u', 7, (0.004716613199078894, 0.21246252000648394, -0.010068637593983236)), ('u', 8, (0.0024699010658877363, 3.723199423905932e-07, 3.156587806026825)), ('u', 9, (3.1709922127383923, 1.6717308694221615, 0.006357761268223072)), ('cz', 8, 9), ('u', 6, (-0.149179819856092, 0.0073166448209931205, -0.18873465895337813)), ('u', 7, (-0.010068429561566238, 0.180256711353873, -0.025384936550044934)), ('u', 8, (-3.1265982994149426, 2.879792808347136, 3.14262416800779)), ('u', 9, (3.147950783718318, 1.5810194225609866, 0.0051794225141676975)), ('cz', 7, 8), ('u', 6, (-3.3303273125335644, 0.012528338239947763, 2.9115537090683197)), ('u', 7, (-0.02538514458468043, 8.543096018313841e-07, -0.020319696249047118)), ('u', 8, (0.0010318785259384909, 1.8325959332673774, 0.00398736224364416)), ('u', 9, (-3.1364136051101927, 0.010223660278832961, 3.135896011527404)), ('cz', 8, 9), ('u', 6, (-3.371630765613361, 0.01443821671985917, 2.871845946781179)), ('u', 7, (-0.02032094450144711, 1.8712083349506447e-07, -0.01330079721428616)), ('u', 8, (-3.137607417068711, 0.26180723510489173, 3.137741517763965)), ('u', 9, (-0.005695809571485449, 0.00403625252390588, 0.011325767365440383)), ('cz', 7, 8), ('u', 6, (-0.2697483848758344, 0.0015061147350338621, -0.341535392261783)), ('u', 7, (-3.1548932427496057, 0.006187151643163442, 3.1248315042568446)), ('u', 8, (-0.003849809993246951, 1.308996287788808, 0.01065630030694045)), ('u', 9, (3.1529182975762176, 1.5748316101961628, 0.0014652491448243765)), ('cz', 8, 9), ('u', 6, (1.2292585095449935, 1.5295686302113742, -1.5278527879141417)), ('u', 7, (-1.5875594496229066, 1.5705884974312958, 1.5677044976095087)), ('u', 8, (-3.1309342313072106, 0.49019375451821573, 3.0891773474553403)), ('u', 9, (3.1430586698649674, 1.543729458425524, 0.05079693796372453)), ('cz', 6, 7), ('u', 6, (1.613738941174663, 0.6154797193336008, -1.5693249595122236)), ('u', 7, (1.5677041944582901, 0.6154796712782338, -1.0941551216460745)), ('u', 8, (3.089176372066575, 0.5197091220145889, -3.1124591545135134)), ('u', 9, (-3.0907953812292854, 0.02205876862949908, 3.1797309234268294)), ('cz', 6, 7), ('u', 6, (-1.5693234376800447, 1.5710268263607212, 1.5320474481219526)), ('u', 7, (2.0474392804567048, 1.6465095989641034, -1.5976301429120348)), ('u', 8, (3.170724942066729, 2.133202830195918, 0.04435021715472276)), ('u', 9, (-3.103454591900695, 0.015664100610430897, 3.16660790256985)), ('cz', 7, 8), ('u', 6, (3.1028434940412866, 0.0029342574022471267, -3.170238681167906)), ('u', 7, (3.1147605899518007, 0.9553166913707994, -3.159526019221204)), ('u', 8, (3.1859433528692485, 1.5707966240942846, -0.00899140872696691)), ('u', 9, (-3.116576572034678, 0.00825150306986584, 3.153170208668008)), ('cz', 7, 8), ('u', 6, (3.1063913060080957, 1.881494231841266e-05, -3.1706637323774425)), ('u', 7, (3.124081332256533, 0.7853981594575237, -3.1415930460131394)), ('u', 8, (3.084263402421209, 0.022685295311754557, -3.10842371493674)), ('u', 9, (-0.0019941521657814727, 0.0006813777759123335, -0.01574928663732262)), ('cz', 6, 7), ('u', 6, (3.1125213667304164, 0.3092041646086194, -3.12625345487787)), ('u', 7, (-4.080796685731872e-08, 0.7853981558924876, 7.772173463874602e-07)), ('u', 8, (3.1747613891209787, 1.5481628904529872, -0.009805768630532175)), ('u', 9, (-0.015748454043375226, 0.008431297425970982, -0.0295910929262283)), ('cz', 7, 8), ('u', 6, (0.01533891114478319, 1.2643549581204578, 0.0057153799668915855)), ('u', 7, (-5.018874719283185e-07, 0.7853980462868956, 6.318653877145332e-07)), ('u', 8, (3.131787219964067, 2.0895288501316824, 0.0025644477643140817)), ('u', 9, (-0.029591092924892516, 0.015601263069330702, -0.04344267924801568)), ('cz', 6, 7), ('u', 6, (-3.1358763000417835, 0.7853983234787361, 3.141594348345165)), ('u', 7, (-7.034785056979964e-07, 1.0471960345973994, 3.6104177568044136e-07)), ('u', 8, (0.0025637937964505214, 0.5245194617938411, -0.00442326163518103)), ('u', 9, (-0.043441430274428344, 0.02175985552611398, -0.05718132719321954)), ('cz', 6, 7), ('u', 6, (3.14159139058198, 0.6154825200535677, -3.1045753624373456)), ('u', 7, (3.141592370965353, 1.570796391465584, -3.1045751262997925)), ('u', 8, (-0.004423053508179188, 0.5275553257727209, -0.011433642825968456)), ('u', 9, (-0.057182576164625784, 0.026436670338707852, -0.07062164023534258)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0))])], (14, 7): [(14, [('u', 6, (0.0, 3.107451558374362, 0.0)), ('u', 5, (0.0, 0.11979173921120091, 0.0)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 6, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 5, (-3.141592653589793, 0.11979173921120091, 3.141592653589793)), ('u', 4, (0.0, 0.36945167713654126, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (-3.141592653589793, 0.36945167713654126, 3.141592653589793)), ('u', 3, (0.0, 0.7015304354513633, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 0.7015304354513633, 3.141592653589793)), ('u', 2, (0.0, 1.0063930812163366, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.0063930812163366, 3.141592653589793)), ('u', 1, (0.0, 1.2460031075107123, 0.0)), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 1), ('u', 1, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.2460031075107123, 3.141592653589793)), ('u', 0, (0.0, 1.4288992721907325, 0.0)), ('u', 0, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 0), ('u', 0, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (-3.141592653589793, 1.4288992721907325, 3.141592653589793)), ('u', 7, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 6, 7), ('u', 7, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 8), ('u', 8, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 11, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 12, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 12), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 13, (0.0, 3.141592653589793, -3.141592653589793)), ('u', 13, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 0, 13), ('u', 13, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 1, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 0, 1), ('u', 1, (1.5707963267948981, 0.3875966866551809, -1.5707963267948981)), ('u', 0, (1.5707963267948968, 0.3875966866551808, -1.5707963267948968)), ('cz', 0, 1), ('u', 1, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 0, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 1, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.0673694840777628, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.5034268427171338, 0.0)), ('cz', 2, 1), ('u', 1, (-3.141592653589793, 0.5034268427171338, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 1.0673694840777628, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.142260352869831, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 0.4285359739250654, 0.0)), ('cz', 3, 2), ('u', 2, (-3.141592653589793, 0.4285359739250654, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 1.142260352869831, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.2139341373225139, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 0.35686218947238274, 0.0)), ('cz', 4, 3), ('u', 3, (-3.141592653589793, 0.35686218947238274, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 1.2139341373225139, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (-3.141592653589793, 1.2888250061145825, 3.141592653589793)), ('cz', 3, 4), ('u', 4, (0.0, 0.2819713206803144, 0.0)), ('cz', 5, 4), ('u', 4, (-3.141592653589793, 0.2819713206803144, 3.141592653589793)), ('cz', 3, 4), ('u', 4, (0.0, 1.2888250061145825, 0.0)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 6), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 5, (-3.141592653589793, 1.3769979834673063, 3.141592653589793)), ('cz', 4, 5), ('u', 5, (0.0, 0.1937983433275903, 0.0)), ('cz', 6, 5), ('u', 5, (-3.141592653589793, 0.1937983433275903, 3.141592653589793)), ('cz', 4, 5), ('u', 5, (0.0, 1.3769979834673063, 0.0)), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 5, 6), ('u', 6, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (0.0, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (-6.026816351730587e-17, 0.4205343352839649, 4.75701228604804e-18)), ('u', 0, (-6.026816351730587e-17, 0.4205343352839649, 4.75701228604804e-18)), ('cz', 0, 1), ('u', 1, (0.0, 1.5707963267948966, 0.0)), ('u', 0, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.093138017732642, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.4776583090622546, 0.0)), ('cz', 2, 1), ('u', 1, (-3.141592653589793, 0.4776583090622546, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 1.093138017732642, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.1780972450961724, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 0.39269908169872414, 0.0)), ('cz', 3, 2), ('u', 2, (-3.141592653589793, 0.39269908169872414, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 1.1780972450961724, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.2630564724597029, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 0.30773985433519363, 0.0)), ('cz', 4, 3), ('u', 3, (-3.141592653589793, 0.30773985433519363, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 1.2630564724597029, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (-3.141592653589793, 1.3605291591529143, 3.141592653589793)), ('cz', 3, 4), ('u', 4, (0.0, 0.2102671676419825, 0.0)), ('cz', 5, 4), ('u', 4, (-3.141592653589793, 0.2102671676419825, 3.141592653589793)), ('cz', 3, 4), ('u', 4, (0.0, 1.3605291591529143, 0.0)), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 4, 5), ('u', 5, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 0, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 1, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 0, 1), ('u', 1, (1.570796326794898, 0.4636476090008061, -1.570796326794898)), ('u', 0, (1.5707963267948966, 0.4636476090008061, -1.5707963267948966)), ('cz', 0, 1), ('u', 1, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 0, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 1, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 1, (-3.141592653589793, 1.1277577648985897, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 0.44303856189630686, 0.0)), ('cz', 2, 1), ('u', 1, (-3.141592653589793, 0.44303856189630686, 3.141592653589793)), ('cz', 0, 1), ('u', 1, (0.0, 1.1277577648985897, 0.0)), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 1, 2), ('u', 2, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 2, (-3.141592653589793, 1.2284367252937551, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 0.3423596015011414, 0.0)), ('cz', 3, 2), ('u', 2, (-3.141592653589793, 0.3423596015011414, 3.141592653589793)), ('cz', 1, 2), ('u', 2, (0.0, 1.2284367252937551, 0.0)), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 2, 3), ('u', 3, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 3, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 0.23182380450040305, 0.0)), ('cz', 4, 3), ('u', 3, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 2, 3), ('u', 3, (0.0, 1.3389725222944935, 0.0)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 3, 4), ('u', 1, (1.2318296534985738, 1.5707963891411776, -2.9993859084934416)), ('u', 0, (1.231829587763898, 1.570796581687041, 1.2639027086785746)), ('u', 2, (-2.998939587119123, 0.17406219884173368, 3.1280437744357883)), ('u', 3, (0.5563549679974619, 0.017533046989716777, 0.09519835774965771)), ('u', 13, (2.111310488313956, 0.3002695778440715, 0.5374705280188435)), ('cz', 1, 0), ('u', 1, (-0.14220762440805534, 0.5235987649055251, -0.417450337944417)), ('u', 0, (1.8776901039842242, 2.617993868063009, -0.051040734168147694)), ('u', 2, (0.7470721499495174, 0.2519516282357102, 0.3931338367988535)), ('u', 3, (-2.967925323752319, 0.09026725305039957, 3.07718582964398)), ('u', 13, (0.19804294703857667, 0.1219381728828627, -0.28810827849922405)), ('cz', 1, 0), ('u', 1, (0.41745108327705743, 1.5707962245343299, -0.3514604716396726)), ('u', 0, (3.131783007511179, 1.7299604937881123, -2.797132241397772)), ('u', 2, (3.482733215321075, 1.7390590478911996, -0.06262408084911009)), ('u', 3, (-0.016137454141659763, 0.022472542731438203, 0.09118399060295301)), ('u', 13, (3.282823989717301, 0.392077688292537, -2.28767195062971)), ('cz', 1, 2), ('u', 1, (2.695857943117891, 0.3926990677539391, -2.9988383969163634)), ('u', 0, (-2.8365592626780307, 0.06688847676932218, 2.2961611469634144)), ('u', 2, (3.1991868566815893, 1.5707962634949266, -0.04979100007713422)), ('u', 3, (2.90995303224707, 0.4554029842153294, -2.850779839770056)), ('u', 13, (-3.0780950685349153, 0.2370134958481819, 3.0194613971532536)), ('cz', 1, 2), ('u', 1, (2.9988381771539276, 0.39269909714723195, -3.014331388103631)), ('u', 0, (0.09605433831342877, 0.2131656390353723, 0.3815550173079302)), ('u', 2, (3.369091841522241, 3.954788457970534e-07, -2.7026795753949453)), ('u', 3, (2.870797262432834, 0.09640409380963649, -2.9295607053882513)), ('u', 13, (-0.20274888804284125, 0.03332621495886792, 0.127740381167612)), ('cz', 1, 0), ('u', 1, (-0.1272610609285276, 1.1780970955078343, -0.020546244808281815)), ('u', 0, (-0.03075289256538069, 0.04791846691333017, -0.3149937200418014)), ('u', 2, (-0.3454587237016874, 2.402146058625508e-07, 2.1358064526366203)), ('u', 3, (0.12838905667016132, 0.2668679937160416, -0.26756517904437843)), ('u', 13, (-0.05361243968074117, 0.383073285426385, 0.2511860695253595)), ('cz', 1, 2), ('u', 1, (-3.187070573375776, 1.0239150860474089, 3.018925365874875)), ('u', 0, (-0.12758009369927642, 0.09179113075420134, -0.19098110689493691)), ('u', 2, (-0.6431610124732767, 3.141592599069518, -3.1787012420657215)), ('u', 3, (2.5227036079005973, 1.2329583370851056, 0.4095446247138912)), ('u', 13, (-2.8764477600269793, 0.26146006638783154, 2.61759090085264)), ('cz', 2, 3), ('u', 1, (-2.9434681901979216, 0.9417084160904503, 2.5932207409054513)), ('u', 0, (0.1617431393159768, 0.03087153381860342, 0.18476822007363228)), ('u', 2, (-2.575612501687395, 2.8793653837677833, 3.2442807226076766)), ('u', 3, (2.8665560775078163, 1.4861932540056146, -0.2018631088110705)), ('u', 13, (-2.9497766242116676, 0.3383924025969984, 2.6558189689394793)), ('cz', 1, 2), ('u', 1, (3.2082442444177657, 8.854600263747566e-08, -2.837066790918172)), ('u', 0, (-3.0266201454039527, 0.4220618310238916, 2.646088980425962)), ('u', 2, (-0.1590168135832456, 1.8326264230225109, -0.359580711423065)), ('u', 3, (-0.002217406540528266, 0.0863871510416005, -0.20351779904120118)), ('u', 13, (0.008945710926425836, 0.29510897735962904, -0.030314185840352045)), ('cz', 2, 3), ('u', 1, (0.18707009843392597, 2.387600001512695e-07, 0.15426325877376323)), ('u', 0, (-0.17189527656230552, 0.08160716647730763, 0.032266886977778284)), ('u', 2, (3.2866804312054043, 0.2676847885115938, -2.730331429951794)), ('u', 3, (-0.13623340654444996, 0.20336580375167437, 0.45325952197254704)), ('u', 13, (3.1217229795383017, 0.02906914229557036, -2.8544954724331615)), ('cz', 1, 2), ('u', 1, (0.2576547125796726, 0.08224191400875382, -0.46285370645133733)), ('u', 0, (0.06001313652140583, 0.18539668713978025, -0.9389421652454829)), ('u', 2, (-0.20418692813982034, 1.3085690422466387, -0.27796227214114966)), ('u', 3, (3.41983132948364, 1.7230695619174563, -0.011797196196519533)), ('u', 13, (-0.33731223988950343, 0.12408544738613886, 0.6351091935005295)), ('cz', 2, 3), ('u', 1, (-1.8771131341299276, 1.628065443803098, 1.672032844130754)), ('u', 0, (1.4515124845650895, 1.5987740503514825, -1.7557543807569043)), ('u', 2, (-0.2454278706897291, 0.6757879212245423, 0.20744621368957772)), ('u', 3, (3.351128805801928, 1.6870543191792902, 0.10233693036285052)), ('u', 13, (3.0598248926673244, 0.42498648216852647, -2.9515116502787992)), ('cz', 1, 0), ('u', 1, (1.5286170778356827, 0.6154797346583963, -2.4179809513697834)), ('u', 0, (1.5791362219019225, 0.6154796863858842, -1.4637268702511077)), ('u', 2, (-0.4007914716380826, 0.3644558829743909, 0.0291651293900482)), ('u', 3, (-0.1269832251616172, 0.23726504445725444, -0.3387665440288648)), ('u', 13, (-0.5256656034937417, 0.2061764494836852, 0.617615395513069)), ('cz', 1, 0), ('u', 1, (1.6101699148574042, 2.1526787200873216, -1.1394315996518978)), ('u', 0, (-1.4695262479411109, 1.4524884326329035, 1.76325794047411)), ('u', 2, (3.224297070832188, 1.0150454860164617, 0.7091458479271178)), ('u', 3, (2.6734492380441623, 0.05576016973613418, -2.832713631890208)), ('u', 13, (-3.125717236341968, 0.3886371855720659, 2.940252474553878)), ('cz', 1, 2), ('u', 1, (3.1940644248245924, 0.9555526735669784, -2.717143186933664)), ('u', 0, (-2.622075337330641, 0.4215995202677075, 3.6576511083759367)), ('u', 2, (3.3112194029611124, 1.5707964069991887, -0.19183885530663503)), ('u', 3, (0.037245415763906595, 0.2619310031916806, 0.011058115923989652)), ('u', 13, (0.3571824999803529, 0.5350154638498804, -0.03290237736833232)), ('cz', 1, 2), ('u', 1, (2.7171421003950074, 0.7853981457269307, -3.1538508910431964)), ('u', 0, (-0.2647995621850458, 0.19307735715437033, -0.22354545216698082)), ('u', 2, (-0.21661284388131616, 0.054280967956542386, 0.23479534218005307)), ('u', 3, (0.1524800577944141, 0.4326415435654515, 0.4640436595178439)), ('u', 13, (0.1641580652072659, 0.3911384995510761, 0.5559238278649419)), ('cz', 1, 0), ('u', 1, (0.012258975814176634, 0.7853981657233824, 0.16758657095592588)), ('u', 0, (0.09167774826056166, 0.4993291335189328, 0.14493445704339158)), ('u', 2, (3.3161816538402658, 1.6388735654419, -0.2924996512895628)), ('u', 3, (-0.025990464898288518, 0.0747982236462298, -0.3972286462536099)), ('u', 13, (0.3031850637481034, 0.2112776202861276, 0.412914410955572)), ('cz', 1, 2), ('u', 1, (-0.16758664781666482, 2.35619449222481, -0.18976283262494215)), ('u', 0, (-0.22265032436016655, 1.7006401613031743, 0.15747352373133458)), ('u', 2, (3.025473519909344, 1.0022276380685784, -0.011782665036470963)), ('u', 3, (2.7792318353410996, 0.22851223571038026, -3.084664205793418)), ('u', 13, (-2.555546162212931, 0.3170585380205804, 2.6661131314005853)), ('cz', 1, 0), ('u', 1, (0.18976273439918814, 1.062244362731298, -0.10685191335948799)), ('u', 0, (3.030056366683838, 0.7853981018688468, -3.075369291716216)), ('u', 2, (3.028417438017049, 0.7707942062452784, -2.8212914341166253)), ('u', 3, (3.197954049883649, 0.16727794183362407, -2.8931995882323167)), ('u', 13, (0.19993131448410484, 0.5210794318920908, 0.34833827069640055)), ('cz', 1, 0), ('u', 1, (-3.0347406340381085, 1.5707963344891147, 2.697431585124785)), ('u', 0, (-3.207815839542296, 2.5502705989992873, 2.6974314518757634)), ('u', 2, (-3.024597184352242, 0.25061148351499507, 2.7699357206750657)), ('u', 3, (-2.8109741244848476, 0.7027851768752893, 2.8130256307694546)), ('u', 13, (3.340130982021578, 1.6847140816560398, 0.22640386793002842)), ('u', 4, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 8, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 7, 8), ('u', 8, (1.5707963267948981, 0.3875966866551809, -1.5707963267948981)), ('u', 7, (1.5707963267948968, 0.3875966866551808, -1.5707963267948968)), ('cz', 7, 8), ('u', 8, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 7, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 8, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (-3.141592653589793, 1.0673694840777628, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 0.5034268427171338, 0.0)), ('cz', 9, 8), ('u', 8, (-3.141592653589793, 0.5034268427171338, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 1.0673694840777628, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (-3.141592653589793, 1.142260352869831, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 0.4285359739250654, 0.0)), ('cz', 10, 9), ('u', 9, (-3.141592653589793, 0.4285359739250654, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 1.142260352869831, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 10, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (-3.141592653589793, 1.2139341373225139, 3.141592653589793)), ('cz', 9, 10), ('u', 10, (0.0, 0.35686218947238274, 0.0)), ('cz', 11, 10), ('u', 10, (-3.141592653589793, 0.35686218947238274, 3.141592653589793)), ('cz', 9, 10), ('u', 10, (0.0, 1.2139341373225139, 0.0)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 10, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 11, 12), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 11, (-3.141592653589793, 1.2888250061145825, 3.141592653589793)), ('cz', 10, 11), ('u', 11, (0.0, 0.2819713206803144, 0.0)), ('cz', 12, 11), ('u', 11, (-3.141592653589793, 0.2819713206803144, 3.141592653589793)), ('cz', 10, 11), ('u', 11, (0.0, 1.2888250061145825, 0.0)), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 11, 12), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 12, 13), ('u', 13, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 12, (-3.141592653589793, 1.3769979834673063, 3.141592653589793)), ('cz', 11, 12), ('u', 12, (0.0, 0.1937983433275903, 0.0)), ('cz', 13, 12), ('u', 12, (-3.141592653589793, 0.1937983433275903, 3.141592653589793)), ('cz', 11, 12), ('u', 12, (0.0, 1.3769979834673063, 0.0)), ('u', 13, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 12, 13), ('u', 13, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (0.0, 1.5707963267948966, 0.0)), ('u', 8, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (-6.026816351730587e-17, 0.4205343352839649, 4.75701228604804e-18)), ('u', 7, (-6.026816351730587e-17, 0.4205343352839649, 4.75701228604804e-18)), ('cz', 7, 8), ('u', 8, (0.0, 1.5707963267948966, 0.0)), ('u', 7, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (-3.141592653589793, 1.093138017732642, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 0.4776583090622546, 0.0)), ('cz', 9, 8), ('u', 8, (-3.141592653589793, 0.4776583090622546, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 1.093138017732642, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (-3.141592653589793, 1.1780972450961724, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 0.39269908169872414, 0.0)), ('cz', 10, 9), ('u', 9, (-3.141592653589793, 0.39269908169872414, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 1.1780972450961724, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 10, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (-3.141592653589793, 1.2630564724597029, 3.141592653589793)), ('cz', 9, 10), ('u', 10, (0.0, 0.30773985433519363, 0.0)), ('cz', 11, 10), ('u', 10, (-3.141592653589793, 0.30773985433519363, 3.141592653589793)), ('cz', 9, 10), ('u', 10, (0.0, 1.2630564724597029, 0.0)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 10, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 11, 12), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 11, (-3.141592653589793, 1.3605291591529143, 3.141592653589793)), ('cz', 10, 11), ('u', 11, (0.0, 0.2102671676419825, 0.0)), ('cz', 12, 11), ('u', 11, (-3.141592653589793, 0.2102671676419825, 3.141592653589793)), ('cz', 10, 11), ('u', 11, (0.0, 1.3605291591529143, 0.0)), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 11, 12), ('u', 12, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 8, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 7, 8), ('u', 8, (1.570796326794898, 0.4636476090008061, -1.570796326794898)), ('u', 7, (1.5707963267948966, 0.4636476090008061, -1.5707963267948966)), ('cz', 7, 8), ('u', 8, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 7, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 8, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (-3.141592653589793, 1.1277577648985897, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 0.44303856189630686, 0.0)), ('cz', 9, 8), ('u', 8, (-3.141592653589793, 0.44303856189630686, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 1.1277577648985897, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (-3.141592653589793, 1.2284367252937551, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 0.3423596015011414, 0.0)), ('cz', 10, 9), ('u', 9, (-3.141592653589793, 0.3423596015011414, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 1.2284367252937551, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 10, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (-3.141592653589793, 1.3389725222944935, 3.141592653589793)), ('cz', 9, 10), ('u', 10, (0.0, 0.23182380450040305, 0.0)), ('cz', 11, 10), ('u', 10, (-3.141592653589793, 0.23182380450040305, 3.141592653589793)), ('cz', 9, 10), ('u', 10, (0.0, 1.3389725222944935, 0.0)), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 10, 11), ('u', 11, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (0.0, 1.5707963267948966, 0.0)), ('u', 8, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 0.5235987755982989, 0.0)), ('u', 7, (0.0, 0.5235987755982989, 0.0)), ('cz', 7, 8), ('u', 8, (0.0, 1.5707963267948966, 0.0)), ('u', 7, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (-3.141592653589793, 1.1780972450961724, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 0.39269908169872414, 0.0)), ('cz', 9, 8), ('u', 8, (-3.141592653589793, 0.39269908169872414, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 1.1780972450961724, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 9, (-3.141592653589793, 1.3089969389957472, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 0.26179938779914935, 0.0)), ('cz', 10, 9), ('u', 9, (-3.141592653589793, 0.26179938779914935, 3.141592653589793)), ('cz', 8, 9), ('u', 9, (0.0, 1.3089969389957472, 0.0)), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 9, 10), ('u', 10, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (1.5707963267948966, 1.5707963267948966, -1.5707963267948966)), ('u', 8, (-1.5707963267948957, 1.5707963267948966, 1.5707963267948957)), ('cz', 7, 8), ('u', 8, (1.570796326794898, 0.6154797086703868, -1.570796326794898)), ('u', 7, (1.5707963267948966, 0.6154797086703876, -1.5707963267948966)), ('cz', 7, 8), ('u', 8, (1.5707963267948981, 1.5707963267948966, -1.5707963267948981)), ('u', 7, (-1.5707963267948966, 1.5707963267948966, 1.5707963267948966)), ('u', 8, (0.0, 0.0, 1.3951473992034527e-15)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 8, (-3.141592653589793, 1.2630564724597029, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 0.30773985433519363, 0.0)), ('cz', 9, 8), ('u', 8, (-3.141592653589793, 0.30773985433519363, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (0.0, 1.2630564724597029, 0.0)), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('cz', 8, 9), ('u', 9, (3.141592653589793, 1.5707963267948966, 0.0)), ('u', 7, (0.0, 1.5707963267948966, 0.0)), ('u', 8, (0.0, 1.5707963267948966, 0.0)), ('cz', 7, 8), ('u', 8, (1.2251903831398114e-16, 0.7853981633974483, -1.1496735851465472e-17)), ('u', 7, (-3.141592653589793, 0.7853981633974483, 3.141592653589793)), ('cz', 7, 8), ('u', 8, (-3.141592653589793, 1.5707963267948966, 3.141592653589793)), ('u', 7, (-3.141592653589793, 1.5707963267948966, 3.141592653589793))])]}


def _lib_template(d, N, n):
    edges, seq, P, mode = d['edges'], d['seq'], d['params'], d['mode']
    ops = []

    def layer(Pl):
        for q in range(N):
            if mode == 'real':
                ops.append(('u', q, ry(Pl[q][0])))
            else:
                a, t, b = Pl[q]
                ops.append(('u', q, rz(b) @ ry(t) @ rz(a)))

    layer(P[0])
    for t, e in enumerate(seq):
        a, b = edges[e]
        ops.append(('cz', a, b))
        layer(P[t + 1])
    return dict(k=N, ops=ops, state_nodes=list(range(N)), N=N, n=n)


def estimate_error(circuit, state_qubits, n, cz_err, sq_err, zero_err, one_err, dd=True):
    """Crude log-infidelity proxy for choosing between exact candidates."""
    czp = {}
    for pair, val in cz_err.items():
        czp[frozenset(pair)] = float(val[0])
    N = len(state_qubits)
    e = 0.0
    t = 0.0
    first = {}
    for mi, m in enumerate(circuit):
        has2 = any(len(op.qubits) == 2 for op in m.operations)
        dt = 42e-9 if has2 else 25e-9
        for op in m.operations:
            for q in op.qubits:
                first.setdefault(q, t)
            if len(op.qubits) == 2:
                e += 0.8 * czp[frozenset(op.qubits)]
            else:
                e += 0.67 * float(sq_err[op.qubits[0]][0])
        t += dt
    T = t
    expo = _exposure(circuit) if len(circuit.all_qubits()) <= 14 else None
    if expo is None:
        # excitation exposure: n excitations from (roughly) the first gate on any state qubit to the end
        expo = n * (T - min(first.values())) if first else 0.0
    e += expo / T1_EST
    # dephasing exposure of live qubits (DD suppresses much of it)
    e += (0.1 if dd else 0.2) * sum(T - first[q] for q in first) / TPHI_EST
    for q in state_qubits:
        e += readout_cost(q, n / N, zero_err, one_err, x_weight=0.5)
    # small-M penalty: readout-asymmetry spread -> systematic shell non-uniformity (raises chi2)
    M = math.comb(N, n)
    if 1 < M <= 200:
        d = [float(one_err.get(q, (0, 0))[0]) - float(zero_err.get(q, (0, 0))[0]) for q in state_qubits]
        var = float(np.var(d))
        delta2 = n * (N - n) / (N - 1) * var
        K = 1e4 * math.exp(-e)
        e += math.log(1 + K * delta2 / (M - 1))
    return e


def _exposure(circuit):
    """sum over moments of (sum_q P(q=1)) * duration, from the exact noiseless state."""
    order = sorted(circuit.all_qubits())
    idx = {q: i for i, q in enumerate(order)}
    k = len(order)
    psi = np.zeros((2,) * k, dtype=complex)
    psi[(0,) * k] = 1
    tot = 0.0
    for mo in circuit:
        two = False
        for op in mo.operations:
            if len(op.qubits) == 1:
                a = idx[op.qubits[0]]
                psi = np.moveaxis(np.tensordot(cirq.unitary(op), psi, axes=([1], [a])), 0, a)
            else:
                two = True
                a, b = idx[op.qubits[0]], idx[op.qubits[1]]
                sl = [slice(None)] * k
                sl[a] = 1
                sl[b] = 1
                psi[tuple(sl)] *= -1
        p = np.abs(psi) ** 2
        tot += sum(float(p.take(1, axis=i).sum()) for i in range(k)) * (42e-9 if two else 25e-9)
    return tot


def _verify(circuit, state_qubits, n, atol=1e-6):
    sq = sorted(state_qubits)
    anc = sorted(set(circuit.all_qubits()) - set(sq))
    N = len(sq)
    if len(sq) + len(anc) > 18:
        return True  # too large to check here; constructions are proven exact
    psi = cirq.final_state_vector(circuit, qubit_order=sq + anc, dtype=np.complex128).reshape(2 ** N, -1)
    tgt = np.zeros(2 ** N)
    for c in itertools.combinations(range(N), n):
        tgt[sum(1 << (N - 1 - i) for i in c)] = 1
    tgt /= np.linalg.norm(tgt)
    F = float(np.sum(np.abs(tgt @ psi) ** 2))
    return abs(F - 1) < atol


def _diverse(states, k):
    """top-k embeddings with distinct qubit sets"""
    out, seen = [], set()
    for st in states:
        key = frozenset(st[1].values())
        if key in seen:
            continue
        seen.add(key)
        out.append(st)
        if len(out) >= k:
            break
    return out


def _circuit_to_template(c, outs, N, n):
    """unitary cirq circuit (CZ + 1q only) -> template with state nodes 0..N-1 (outputs first)."""
    order = sorted(outs) + sorted(set(c.all_qubits()) - set(outs))
    idx = {q: i for i, q in enumerate(order)}
    ops = []
    for m in c:
        for op in m.operations:
            if len(op.qubits) == 2:
                ops.append(('cz', idx[op.qubits[0]], idx[op.qubits[1]]))
            else:
                ops.append(('u', idx[op.qubits[0]], cirq.unitary(op)))
    return dict(k=len(order), ops=ops, state_nodes=list(range(N)), N=N, n=n)


def _bonus_templates(N, n):
    out = []
    try:
        srcs = []
        try:
            srcs.append(_bonus_mod.create_dicke_state_adaptive(N, n))
        except Exception:
            pass
        if (N, n) in BRICK:
            srcs.append(_brick_circuit(BRICK[(N, n)]))
        for c, qs in srcs:
            ok = all(isinstance(op.gate, (cirq.CZPowGate, cirq.PhasedXZGate)) or
                     (len(op.qubits) == 1 and cirq.has_unitary(op) and not cirq.is_measurement(op))
                     for op in c.all_operations())
            ok = ok and not any(cirq.is_measurement(op) or isinstance(op, cirq.ClassicallyControlledOperation)
                                for op in c.all_operations())
            ok = ok and all(op.gate == cirq.CZ for op in c.all_operations() if len(op.qubits) == 2)
            if ok and len(c.all_qubits()) == N:
                out.append(_circuit_to_template(c, qs, N, n))
    except Exception:
        pass
    return out


def _candidates(N, n, adj, cz_err, sq_err, zero_err, one_err, deadline):
    qcost = lambda q: readout_cost(q, n / N, zero_err, one_err) + 2 * float(sq_err[q][0])
    out = []
    good = sorted(adj, key=lambda q: qcost(q) + min(adj[q].values(), default=1))
    # --- W trees
    if n == 1 and N >= 2:
        for root in good[:10]:
            for lam in (0.0005, 0.003, 0.02):
                if _time.time() > deadline:
                    break
                patch = grow_patch(adj, N, root, qcost, lam=lam)
                if patch is None:
                    continue
                rounds = broadcast_tree(adj, patch, root)
                t = w_tree_template(patch, rounds)
                out.append(('wtree', t, {i: q for i, q in enumerate(patch)}))
    # --- library templates (numerically synthesised, exact)
    for d in LIBRARY.get((N, n), []):
        t = _lib_template(d, N, n)
        nw = node_weights(t)
        res = embed(t, adj, lambda node, q: qcost(q) + nw[node] * 0.0, beam=2000, return_all=True)
        if res:
            for r in _diverse(res, 6 if math.comb(N, n) <= 200 else 2):
                out.append(('lib', t, r[1]))
    # --- peephole-optimised exact op lists (state-aware re-synthesis, offline)
    for k, pops in PEEP.get((N, n), []):
        ops = [('u', o[1], rz(o[2][2]) @ ry(o[2][1]) @ rz(o[2][0])) if o[0] == 'u' else o for o in pops]
        t = dict(k=k, ops=ops, state_nodes=list(range(N)), N=N, n=n)
        res = embed(t, adj, lambda node, q: qcost(q), beam=1500, return_all=True)
        if res:
            for r in _diverse(res, 4 if math.comb(N, n) <= 200 else 2):
                out.append(('peep', t, r[1]))
    # --- depth-optimised bonus circuits (measurement-free ones are legal here): re-placed on calibrated qubits
    if N <= 15 and _time.time() < deadline - 20:
        for t in _bonus_templates(N, n):
            res = embed(t, adj, lambda node, q: qcost(q), beam=1500, return_all=True)
            if res:
                for r in _diverse(res, 3):
                    out.append(('bonus', t, r[1]))
    # --- divide & conquer (weight-distribution seed + two parallel B&E blocks)
    if n >= 2 and N >= 2 * n:
        splits = sorted({N // 2, (N + 1) // 2}, key=lambda a: abs(a - N / 2))
        for a in splits:
            if a < n or N - a < n or _time.time() > deadline:
                continue
            t = dc2_template(N, n, a)
            res = embed(t, adj, lambda node, q: qcost(q), beam=600 if N > 12 else 1500)
            if res is not None:
                out.append(('dc2', t, res[1]))
    # --- B&E nearest-neighbour path (support-pruned)
    if N >= 2:
        t = be_path_template_opt(N, n)
        nw = node_weights(t)
        res = embed(t, adj, lambda node, q: qcost(q), beam=600 if N > 12 else 2000)
        if res is not None:
            out.append(('bepath', t, res[1]))
    return out


def create_dicke_state(
    num_qubits: int,
    num_excitations: int,
    cz_pauli_error_rate: dict[
        tuple[cirq.GridQubit, cirq.GridQubit], tuple[float, float]
    ],
    sq_pauli_error_rate: dict[cirq.GridQubit, tuple[float, float]],
    zero_errors: dict[cirq.GridQubit, tuple[float, float]],
    one_errors: dict[cirq.GridQubit, tuple[float, float]],
) -> tuple[cirq.Circuit, set[cirq.GridQubit]]:
    """Create a Cirq circuit that produces a Dicke state.

    Args:
        num_qubits: The number, N, of qubits in the Dicke state.
        num_excitations: The number, n, of 1s in the state.
        cz_pauli_error_rate: Measured CZ Pauli error rates and their statistical uncertainties.
        sq_pauli_error_rate: Measured single-qubit Pauli error rates and their statistical uncertainties.
        zero_errors: Measured parallel 0 -> 1 readout error rates and their statistical uncertainties.
        one_errors: Measured 1 -> 0 readout error rates and their statistical uncertainties.

    Returns:
        A Cirq circuit to produce Dicke state and a set indicating which qubits the resulting state lives on.

    Raises:
        NotImplementedError: The requested (N, n) combination is not implemented.
    """
    t0 = _time.time()
    deadline = t0 + 35.0
    N, n = int(num_qubits), int(num_excitations)
    if N < 1 or n < 0 or n > N:
        raise NotImplementedError
    flip = False
    if n > N - n:
        n, flip = N - n, True
    adj = hw_graph(cz_pauli_error_rate, sq_pauli_error_rate)
    if n == 0:
        qs = sorted(adj, key=lambda q: readout_cost(q, 1.0 if flip else 0.0, zero_errors, one_errors))[:N]
        if flip:
            c = cirq.Circuit([cirq.Moment([cirq.PhasedXZGate(x_exponent=1, z_exponent=0, axis_phase_exponent=0)(q)
                                           for q in qs])])
        else:
            c = cirq.Circuit([cirq.Moment([cirq.PhasedXZGate(x_exponent=0, z_exponent=0, axis_phase_exponent=0)(q)
                                           for q in qs])])
        return c, set(qs)
    cands = _candidates(N, n, adj, cz_pauli_error_rate, sq_pauli_error_rate, zero_errors, one_errors, deadline)
    scored = []
    for name, t, mapping in cands:
        for dd in (True, False):
            c = build_circuit(t, mapping, dd=dd)
            if flip:
                c = _append_flip(c, [mapping[i] for i in t['state_nodes']])
            qs = {mapping[i] for i in t['state_nodes']}
            e = estimate_error(c, qs, num_excitations, cz_pauli_error_rate, sq_pauli_error_rate,
                               zero_errors, one_errors, dd=dd)
            scored.append((e, name, c, qs))
    scored.sort(key=lambda x: x[0])
    for e, name, c, qs in scored:
        if len(c) > 1750:
            continue
        if _verify(c, qs, num_excitations):
            return c, qs
    raise NotImplementedError


def _append_flip(c, qubits):
    c = c.copy()
    last = c[-1] if len(c) else None
    if last is not None and all(len(op.qubits) == 1 for op in last.operations):
        ops = {op.qubits[0]: op for op in last.operations}
        new = []
        for q in qubits:
            U = cirq.unitary(ops[q]) if q in ops else _I2
            new.append(cirq.PhasedXZGate.from_matrix(_X @ U).on(q))
        for q, op in ops.items():
            if q not in set(qubits):
                new.append(op)
        c[-1] = cirq.Moment(new)
    else:
        c.append(cirq.Moment([cirq.PhasedXZGate.from_matrix(_X).on(q) for q in qubits]))
    return c


# ==================================================================== BONUS: adaptive circuits
import types as _types
_BONUS_SRC = '"""Willow105 Dicke preparation, verified to complex128 numerical precision.\n\nAnalytic recurrences are supplemented by independently synthesized, embedded\nnative seeds. No fitting, calibration maps, external files, or hardware\nsubmission imports are needed at runtime. The public function returns a fresh\ncircuit and output set. See verify_bonus.py for full-state and all-branch checks.\n"""\nfrom __future__ import annotations\n\nimport functools\nimport collections\nimport math\nimport random\n\nimport cirq\nimport cirq_google\nimport numpy as np\nimport sympy\n\n\nH = cirq.unitary(cirq.H)\nX = cirq.unitary(cirq.X)\nI = np.eye(2, dtype=np.complex128)\n\n\ndef ry(theta):\n    c, s = math.cos(theta / 2), math.sin(theta / 2)\n    return np.array([[c, -s], [s, c]], dtype=np.complex128)\n\n\ndef _u(ops, q, matrix):\n    ops.append((q, matrix))\n\n\ndef _cx(ops, a, b):\n    ops.extend([(b, H), (a, b), (b, H)])\n\n\ndef _identity(matrix):\n    return np.linalg.norm(matrix - np.trace(matrix) * I / 2) < 2e-14\n\n\ndef _compile(ops):\n    """Fuse local matrices between CZs and ASAP-pack mixed native moments.\n\n    A CZ flushes only its two operands, so disjoint rotations and entanglers\n    share moments.  Local identity up to phase can be dropped because these\n    are unconditional operations.  Never apply that rule to feedforward gates.\n    """\n    pending = {}\n    emitted = []\n\n    def flush(q):\n        matrix = pending.pop(q, None)\n        if matrix is not None and not _identity(matrix):\n            emitted.append(cirq.PhasedXZGate.from_matrix(matrix).on(q))\n\n    for a, b in ops:\n        if isinstance(b, np.ndarray):\n            pending[a] = b @ pending.get(a, I)\n        else:\n            flush(a)\n            flush(b)\n            emitted.append(cirq.CZ(a, b))\n    for q in list(pending):\n        flush(q)\n    return cirq.Circuit(emitted, strategy=cirq.InsertStrategy.EARLIEST)\n\n\ndef _optimize_frames(circuit):\n    """Commute exact Pauli/Z frames through CZs, then refold local layers."""\n    best = circuit\n    for _ in range(3):\n        c = cirq.eject_z(best, atol=1e-13)\n        c = cirq.eject_phased_paulis(c, atol=1e-13)\n        c = cirq.merge_single_qubit_gates_to_phxz(c, atol=1e-13)\n        c = cirq.drop_negligible_operations(c, atol=1e-13)\n        ops = []\n        for op in c.all_operations():\n            if not op.qubits:\n                continue\n            if len(op.qubits) == 1:\n                ops.append((op.qubits[0], cirq.unitary(op)))\n            else:\n                assert op.gate == cirq.CZ\n                ops.append(op.qubits)\n        c = _compile(ops)\n        if len(c) >= len(best):\n            break\n        best = c\n    return best\n\n\ndef _schedule_commuting(circuit, trials=6):\n    """Resource-constrained scheduling of an exact commutation DAG.\n\n    Diagonal gates commute even when they share a wire. Every other shared\n    wire ordering is retained, so this never approximates the unitary.\n    """\n    ops = list(circuit.all_operations())\n    successors = [set() for _ in ops]\n    predecessors = [set() for _ in ops]\n    diagonal = []\n    for op in ops:\n        matrix = cirq.unitary(op) if len(op.qubits) == 1 else None\n        diagonal.append(matrix is None or abs(matrix[0,1])+abs(matrix[1,0]) < 1e-13)\n    by_qubit = {}\n    for j,op in enumerate(ops):\n        for q in op.qubits:\n            for i in by_qubit.get(q,[]):\n                if not (diagonal[i] and diagonal[j]):\n                    predecessors[j].add(i)\n                    successors[i].add(j)\n            by_qubit.setdefault(q,[]).append(j)\n    heights = [0]*len(ops)\n    for i in range(len(ops)-1,-1,-1):\n        heights[i] = 1+max((heights[j] for j in successors[i]),default=0)\n    best = circuit\n    for seed in range(trials):\n        rng = random.Random(seed)\n        ranks = [rng.random() for _ in ops]\n        counts = [len(p) for p in predecessors]\n        ready = {i for i,n in enumerate(counts) if n == 0}\n        moments = []\n        while ready:\n            busy,chosen = set(),[]\n            for i in sorted(ready,key=lambda i:(-heights[i],ranks[i])):\n                if not busy.intersection(ops[i].qubits):\n                    chosen.append(i)\n                    busy.update(ops[i].qubits)\n            moments.append(cirq.Moment(ops[i] for i in chosen))\n            ready.difference_update(chosen)\n            for i in chosen:\n                for j in successors[i]:\n                    counts[j] -= 1\n                    if counts[j] == 0:\n                        ready.add(j)\n        candidate = cirq.Circuit(moments)\n        assert sum(len(m) for m in candidate) == len(ops)\n        if len(candidate) < len(best):\n            best = candidate\n    return best\n\n\ndef _optimize_depth(circuit):\n    best = _optimize_frames(circuit)\n    current = circuit\n    # Equal-depth frame changes can unlock a later saving; do not stop early.\n    for _ in range(4):\n        current = cirq.eject_z(current,atol=1e-13)\n        current = cirq.eject_phased_paulis(current,atol=1e-13)\n        current = cirq.merge_single_qubit_gates_to_phxz(current,atol=1e-13)\n        current = cirq.drop_negligible_operations(current,atol=1e-13)\n        current = _schedule_commuting(current)\n        if len(current) < len(best):\n            best = current\n    return best\n\n\n@functools.lru_cache(maxsize=1)\ndef _graph():\n    metadata = cirq_google.Willow105.metadata\n    graph = {q: set() for q in metadata.qubit_set}\n    for a, b in metadata.qubit_pairs:\n        graph[a].add(b)\n        graph[b].add(a)\n    return graph\n\n\ndef _givens(ops, a, b, theta):\n    """Full real excitation-preserving rotation: 10 -> cos*10 + sin*01.\n\n    Conjugating opposite Ry rotations by H.H CZ gives this exact two-CZ\n    formula, retaining the common phase on the vacuum and occupied branches.\n    """\n    ops.extend([(a,H),(b,H),(a,b),(a,ry(theta)),(b,ry(-theta)),\n                (a,b),(a,H),(b,H)])\n\n\n@functools.lru_cache(maxsize=256)\ndef _givens_native(theta):\n    c, s = math.cos(theta), math.sin(theta)\n    matrix = np.array([[1, 0, 0, 0], [0, c, s, 0],\n                       [0, -s, c, 0], [0, 0, 0, 1]], dtype=complex)\n    a, b = cirq.LineQubit.range(2)\n    # A single omitted global phase is common to every input weight. It is\n    # therefore harmless for this unconditional subroutine. A local scalar is\n    # harmless for the whole unitary, unlike a weight-dependent phase.\n    return tuple(cirq.two_qubit_matrix_to_cz_operations(\n        a, b, matrix, allow_partial_czs=False, atol=1e-12))\n\n\ndef _reflect3(ops, a, b, c, m, ell):\n    """Core five-CZ restricted isometry on path a-b-c.\n\n    Reverse of the core module\'s c-a-b convention. Allowed inputs:\n    000->000, 100->sqrt(ell/m)100+sqrt(1-ell/m)010,\n    101->011, 110->110, 111->111. All phases are identical.\n    """\n    theta = math.acos(math.sqrt(ell / m))\n    bp, ap = (theta + math.pi / 2) / 2, (theta - math.pi / 2) / 2\n    _cx(ops, b, a)\n    ops.extend([(b, ry(-bp)), (c, b), (b, ry(-ap)), (a, b),\n                (b, ry(ap)), (c, b), (b, ry(bp))])\n    _cx(ops, b, a)\n\n\ndef _unary3(ops, q):\n    """Analytic six-CZ isometry, valid ONLY for input weights 0,1,2.\n\n    Adapted read-only from astra/research/exact_u3.py; its domain restriction\n    matters: this is not a U(3,3). The general routine handles weight three.\n    """\n    a = math.asin(1 / math.sqrt(3))\n    angles = [-math.pi/2, math.pi, a-math.pi, -math.pi/2, a-math.pi,\n              math.pi/2+a, a, -math.pi/2, -math.pi/2, -math.pi/2, 0,\n              -math.pi/4, math.pi/4, math.pi/2, math.pi/2]\n    edges = [(1, 2), (0, 1), (1, 2), (0, 1), (1, 2), (1, 2)]\n    for i in range(3):\n        _u(ops, q[i], ry(angles[i]))\n    for j, (u, v) in enumerate(edges):\n        ops.append((q[u], q[v]))\n        _u(ops, q[u], ry(angles[3+2*j]))\n        _u(ops, q[v], ry(angles[4+2*j]))\n\n\ndef _unary_legacy(ops, q, k, small=True):\n    """Coherently decode 1^j 0^(m-j) for every 0<=j<=k.\n\n    Baertschi--Eidenbenz SCS with wire movement absorbed in the reverse\n    staircase. No restoration swaps; disjoint blocks pipeline automatically.\n    """\n    if len(q) == 3 and k == 2 and small:\n        _unary3(ops, q)\n        return\n    for start in range(len(q)-1):\n        m = len(q)-start\n        if m == 3 and k == 2 and small:\n            _unary3(ops, q[start:])\n            break\n        width = min(k, m-1)\n        if width == 0:\n            break\n        a, b = q[start+width-1], q[start+width]\n        theta = math.acos(math.sqrt(width/m))\n        if m > k+1:\n            # This is the first operation reaching b: it is still zero.\n            eta = math.pi/2-theta\n            _u(ops,b,ry(eta))\n            _cx(ops,a,b)\n            _u(ops,b,ry(-eta))\n            _cx(ops,b,a)\n        else:\n            _givens(ops, a, b, theta)\n        for ell in range(width-1, 0, -1):\n            _reflect3(ops, *q[start+ell-1:start+ell+2], m, ell)\n\n\ndef _unary3_full_forward(ops, q):\n    """Eight-CZ coherent U(3,3), independently synthesized and analyticized.\n\n    All four unary columns, including 111, have the same phase.\n    """\n    a = math.asin(1/math.sqrt(3))\n    h = math.pi/2\n    angles = [h,h,h,h+a,math.pi-a,h,h+a/2,-h+a/2,h,0,\n              h+a/2,-h+a/2,-h,3*h,-3*math.pi/4,h,-5*math.pi/4,h,-h]\n    for i in range(3):\n        _u(ops,q[i],ry(angles[i]))\n    for j in range(8):\n        u,v = q[j%2],q[1+j%2]\n        ops.append((u,v))\n        _u(ops,u,ry(angles[3+2*j]))\n        _u(ops,v,ry(angles[4+2*j]))\n\n\ndef _unary3_full(ops,q):\n    # Spin complement plus wire reversal preserves every unary-to-Dicke column.\n    # This orientation lets the suffix start earlier at a decoder boundary.\n    ops.extend((x,X) for x in q)\n    _unary3_full_forward(ops,q[::-1])\n    ops.extend((x,X) for x in q)\n\n\ndef _scs_pair(ops, q, m, ell):\n    """Four-CZ fused SCS window, input columns 000,100,110.\n\n    100 -> sqrt(ell/m)100 + sqrt(1-ell/m)010;\n    110 -> sqrt((ell+1)/m)110 + sqrt(1-(ell+1)/m)011.\n    Independently derived by real-isometry synthesis then analyticized.\n    """\n    a,b = math.asin(math.sqrt(ell/m)),math.asin(math.sqrt((ell+1)/m))\n    h = math.pi/2\n    angles = [-h,math.pi,h+b,-h,h+b,h+a,a-math.pi,h,0,-h,h]\n    for i in range(3):\n        _u(ops,q[i],ry(angles[i]))\n    for j,(u,v) in enumerate([(1,2),(0,1),(1,2),(0,1)]):\n        ops.append((q[u],q[v]))\n        _u(ops,q[u],ry(angles[3+2*j]))\n        _u(ops,q[v],ry(angles[4+2*j]))\n\n\ndef _unary(ops, q, k, small=True):\n    if small == \'fused\':\n        return _unary_fused(ops,q,k)\n    if small == \'pruned\':\n        return _unary_pruned(ops,q,k)\n    return _unary_legacy(ops,q,k,small)\n\n\ndef _unary_fused(ops,q,k):\n    for start in range(len(q)-1):\n        m = len(q)-start\n        if m == 3 and k >= 3:\n            _unary3_full(ops,q[start:])\n            break\n        width = min(k,m-1)\n        if m > k and width >= 2:\n            _scs_pair(ops,q[start+width-2:start+width+1],m,width-1)\n            for ell in range(width-2,0,-1):\n                _reflect3(ops,*q[start+ell-1:start+ell+2],m,ell)\n        else:\n            a,b = q[start+width-1],q[start+width]\n            _givens(ops,a,b,math.acos(math.sqrt(width/m)))\n            for ell in range(width-1,0,-1):\n                _reflect3(ops,*q[start+ell-1:start+ell+2],m,ell)\n\n\ndef _reflect4(ops, a, b, c, theta, skip=False):\n    """Claude\'s four-CZ isometry in our reversed wire convention."""\n    if not skip:\n        _cx(ops, b, a)\n    ops.extend([(b, ry(-theta)), (a, b), (b, ry(-math.pi/2)),\n                (c, b), (b, ry(theta+math.pi/2))])\n    _cx(ops, b, a)\n\n\n@functools.lru_cache(maxsize=64)\ndef _classical_map(source, target):\n    """Shortest adjacent-CX/X realization on a promised three-bit support."""\n    queue = collections.deque([(source, ())])\n    seen = {source}\n    gates = [(a,b) for a,b in [(0,1),(1,0),(1,2),(2,1)]]\n    gates += [(a,a) for a in range(3)]\n    while queue:\n        state, path = queue.popleft()\n        if state == target:\n            return path\n        for a,b in gates:\n            nxt = tuple(s ^ ((1 << (2-b)) if a == b or s & (1 << (2-a)) else 0)\n                        for s in state)\n            if nxt not in seen:\n                seen.add(nxt)\n                queue.append((nxt, path+((a,b),)))\n    raise ValueError(\'No affine support realization\')\n\n\ndef _unary_pruned(ops, q, k, jmin=0, jmax=None):\n    """Track exact combinatorial support, with no amplitudes or state fitting."""\n    jmax = min(k,len(q)) if jmax is None else jmax\n    support = {tuple([1]*j+[0]*(len(q)-j)) for j in range(jmin,jmax+1)}\n\n    def apply(indices, mapping):\n        nonlocal support\n        changed = set()\n        for state in support:\n            local = tuple(state[i] for i in indices)\n            for output in mapping.get(local, [local]):\n                result = list(state)\n                for i,v in zip(indices,output):\n                    result[i] = v\n                changed.add(tuple(result))\n        support = changed\n\n    for start in range(len(q)-1):\n        m = len(q)-start\n        if m == 3:\n            if k == 2:\n                _unary3(ops,q[start:])\n                break\n            if k >= 3:\n                _unary3_full(ops,q[start:])\n                break\n        width = min(k,m-1)\n        indices = [start+width-1,start+width]\n        a,b = [q[i] for i in indices]\n        s = {tuple(state[i] for i in indices) for state in support}\n        theta = math.acos(math.sqrt(width/m))\n        if (1,0) in s:\n            if s == {(1,0)}:\n                _u(ops,b,ry(2*theta))\n                _cx(ops,b,a)\n            elif (1,1) not in s:\n                eta = math.pi/2-theta\n                _u(ops,b,ry(eta))\n                _cx(ops,a,b)\n                _u(ops,b,ry(-eta))\n                _cx(ops,b,a)\n            else:\n                _givens(ops,a,b,theta)\n        apply(indices,{(1,0):[(1,0),(0,1)]})\n        for ell in range(width-1,0,-1):\n            indices = list(range(start+ell-1,start+ell+2))\n            a,b,c = [q[i] for i in indices]\n            s = {tuple(state[i] for i in indices) for state in support}\n            theta = math.acos(math.sqrt(ell/m))\n            p,r = (1,0,0) in s,(1,0,1) in s\n            if p and r:\n                if (1,1,1) not in s:\n                    _reflect4(ops,a,b,c,theta,skip=(1,1,0) not in s)\n                else:\n                    _reflect3(ops,a,b,c,m,ell)\n            elif p:\n                _givens(ops,a,b,theta)\n            elif r:\n                source = tuple(sorted(4*x+2*y+z for x,y,z in s))\n                target = tuple(3 if v == 5 else v for v in source)\n                for u,v in _classical_map(source,target):\n                    if u == v:\n                        _u(ops,[a,b,c][u],X)\n                    else:\n                        _cx(ops,[a,b,c][u],[a,b,c][v])\n            apply(indices,{(1,0,0):[(1,0,0),(0,1,0)],(1,0,1):[(0,1,1)]})\n\n\ndef _ladder(a, b, k):\n    """Find k rungs with two nonintersecting path extensions on Willow."""\n    graph = _graph()\n\n    def paths(prefix, length, occupied):\n        if len(prefix) == length:\n            yield prefix\n        else:\n            for nxt in sorted(graph[prefix[-1]] - occupied):\n                yield from paths(prefix+[nxt], length, occupied | {nxt})\n\n    roots = sorted(graph, key=lambda q: ((q.row-6)**2+(q.col-7)**2, q))\n    for root in roots:\n        for dr, dc in [(0, 1), (1, 0), (0, -1), (-1, 0)]:\n            for sr, sc in [(-dc, dr), (dc, -dr)]:\n                top = [cirq.GridQubit(root.row+i*dr, root.col+i*dc) for i in range(k)]\n                bottom = [cirq.GridQubit(root.row+(k-1-i)*dr+sr,\n                                         root.col+(k-1-i)*dc+sc) for i in range(k)]\n                core = set(top+bottom)\n                if len(core) != 2*k or not core <= graph.keys():\n                    continue\n                edges = list(zip(top, top[1:])) + list(zip(bottom, bottom[1:]))\n                edges += [(top[j], bottom[k-1-j]) for j in range(k)]\n                if any(y not in graph[x] for x, y in edges):\n                    continue\n                for full_top in paths(top, a, core):\n                    full_bottom = next(paths(bottom, b, core | set(full_top)), None)\n                    if full_bottom is not None:\n                        return full_top, full_bottom\n    raise RuntimeError(\'No Willow ladder found\')\n\n\ndef _divide(n, k, a=None, small=True, pivot=0, left_first=False):\n    a = n//2 if a is None else a\n    b = n-a\n    top, bottom = _ladder(a, b, k)\n    weights = [math.comb(a, j)*math.comb(b, k-j) for j in range(k+1)]\n    tail = [sum(weights[j:]) for j in range(k+1)]\n    ops = [(q, X) for q in bottom[:k]]\n    prefix = [sum(weights[:j+1]) for j in range(k+1)]\n    _u(ops,top[pivot],ry(2*math.asin(math.sqrt(tail[pivot+1]/tail[0]))))\n    def expand_right():\n        for j in range(pivot+1,k):\n            eta = math.pi/2-math.asin(math.sqrt(tail[j+1]/tail[j]))\n            _u(ops,top[j],ry(eta))\n            _cx(ops,top[j-1],top[j])\n            _u(ops,top[j],ry(-eta))\n    def expand_left():\n        for j in range(pivot-1,-1,-1):\n            eta = math.pi/2-math.asin(math.sqrt(prefix[j]/prefix[j+1]))\n            _u(ops,top[j+1],X)\n            _u(ops,top[j],ry(eta))\n            _cx(ops,top[j+1],top[j])\n            _u(ops,top[j],ry(-eta))\n            _u(ops,top[j+1],X)\n            _u(ops,top[j],X)\n    if left_first:\n        expand_left();expand_right()\n    else:\n        expand_right();expand_left()\n    for j in range(k):\n        _cx(ops, top[j], bottom[k-1-j])\n    _unary(ops, top, k, small)\n    _unary(ops, bottom, k, small)\n    return _compile(ops), set(top+bottom)\n\n\ndef _square():\n    q = [cirq.GridQubit(4,4), cirq.GridQubit(4,5),\n         cirq.GridQubit(5,5), cirq.GridQubit(5,4)]\n    a = math.acos(1/math.sqrt(3))\n    ops = []\n    layers = [[ry(a-math.pi/2), H, ry(-a), H],\n              [ry(math.pi/2), ry(-math.pi/2)@H, ry(3*math.pi/2), ry(-math.pi-a)@H],\n              [I, ry(-math.pi), ry(-math.pi/2), ry(-math.pi/2)],\n              [ry(a-math.pi), ry(2*a), ry(a), ry(a-math.pi/2)]]\n    for i, layer in enumerate(layers):\n        ops.extend(zip(q, layer))\n        if i < 3:\n            ops.extend([(q[0],q[1]),(q[2],q[3])] if i%2 == 0\n                       else [(q[1],q[2]),(q[3],q[0])])\n    return _compile(ops), set(q)\n\n\n# INDEPENDENT NATIVE SEEDS START\n_NATIVE_SEEDS = {(5, 2): {\'angles\': [1.5901770943121394,\n                     -2.209960070660441,\n                     -0.7091601230882575,\n                     -0.3043315333387232,\n                     -2.598142427345299,\n                     -0.917169399671696,\n                     0.41842715016396437,\n                     1.7496392757141055,\n                     2.1664957579788107,\n                     -3.741609447121856,\n                     -2.7336266023998292,\n                     2.0446189031881654,\n                     -4.316243675422165,\n                     3.110303759135296,\n                     1.627362894082465,\n                     -2.4206903968601123,\n                     1.701620864075715,\n                     -1.5707963267948968,\n                     -2.8410295874377143,\n                     -1.484634915615345,\n                     -2.607216994232227,\n                     0.6798184144094802,\n                     -3.38439815282079e-16,\n                     -1.4656517427965308e-16,\n                     3.313868466630986,\n                     1.5707963267948966,\n                     1.6596137066175907,\n                     -1.5707963267948966,\n                     -1.171515403577853,\n                     2.8831554293690487,\n                     -1.5707963267948968,\n                     3.1415926535897927,\n                     3.141592653589794,\n                     -1.9700772500119397,\n                     -3.859892801317504e-17],\n          \'coords\': [[0, 0], [0, 1], [1, 0], [1, 1], [2, 0]],\n          \'layers\': [[[0, 1], [2, 3]],\n                     [[0, 2], [1, 3]],\n                     [[2, 4], [0, 1]],\n                     [[0, 1], [2, 3]],\n                     [[0, 2], [1, 3]],\n                     [[2, 4], [0, 1]]]},\n (6, 3): {\'angles\': [-1.1587325261959078,\n                     4.054291264522096,\n                     -0.832318319265232,\n                     -0.05529000552069274,\n                     1.945349057346249,\n                     2.449356900559118,\n                     -1.2273715213816687,\n                     -3.0740608433392755,\n                     -0.22411234242881936,\n                     -1.218563474982311,\n                     -1.2682497606296208,\n                     1.7028864809395576,\n                     -0.5583156428328178,\n                     -0.45406327603201657,\n                     2.642354447205798,\n                     1.3654962842988456,\n                     0.7989304988580569,\n                     -0.696406337546169,\n                     -0.6296843788676697,\n                     2.0042367834475625,\n                     0.4465084180019968,\n                     0.06506701652677492,\n                     1.397654299316236,\n                     -1.0305224066812717,\n                     -1.0596798458932037,\n                     0.9355058269456195,\n                     4.71238898038469,\n                     1.5707963267948966,\n                     -1.5707963267948963,\n                     4.71238898038469,\n                     -1.5707963267948966,\n                     1.5707963267948966,\n                     -9.244868821011682e-17,\n                     -3.141592653589793,\n                     1.5707963267948968,\n                     -1.5707963267948966,\n                     1.5707963267948968,\n                     -5.2202401900754905e-17,\n                     1.5707963267948963,\n                     3.141592653589793,\n                     1.5707963267948966,\n                     3.3919127285705133e-16],\n          \'coords\': [[0, 0], [0, 1], [1, 0], [1, 1], [2, 0], [2, 1]],\n          \'layers\': [[[0, 2], [1, 3], [4, 5]],\n                     [[0, 1], [2, 3], [4, 5]],\n                     [[0, 2], [1, 3], [4, 5]],\n                     [[0, 1], [2, 4], [3, 5]],\n                     [[0, 2], [1, 3], [4, 5]],\n                     [[0, 1], [2, 3], [4, 5]]]},\n (8, 3): {\'angles\': [1.5583187045626021,\n                     -3.1415926535897927,\n                     5.208491071435463,\n                     0.7818320292060451,\n                     0.5313519625682895,\n                     -2.670820321126655,\n                     -3.139913987986567,\n                     -1.0149708664277908,\n                     1.5583187045626021,\n                     -3.6039317335405306e-16,\n                     -4.405158511086932,\n                     -3.191424733971771,\n                     -1.0972527734774649,\n                     3.0933092733224776,\n                     -1.9960076199541927,\n                     2.3396128956288824,\n                     3.094494000534572,\n                     1.3625073655517543,\n                     1.581929853746623,\n                     2.4763086992746093,\n                     -3.175404704938736,\n                     -6.184957546348496,\n                     -2.8580372420390145,\n                     -0.9901588318738902,\n                     2.8329988416935614,\n                     -2.0774516709018056,\n                     1.750194454777398,\n                     -2.7238527237224184,\n                     1.1504270471575575,\n                     -2.878694326002739,\n                     -0.5251095513406993,\n                     -0.5630258708367256,\n                     -8.473365917931746,\n                     -1.2031086345590554,\n                     3.3840571385390796,\n                     0.42989943402514597,\n                     -0.20514828979913344,\n                     1.2792023071464371,\n                     0.6655837745142688,\n                     0.6547237454781646,\n                     0.864662744791886,\n                     2.3379665699681538,\n                     3.1415926535897936,\n                     1.1589338251128961e-16,\n                     -0.7054971262004875,\n                     -1.8171405310405533,\n                     4.4930901681199105,\n                     -1.0374989643280963,\n                     1.9547930623865346,\n                     1.4022020809967617,\n                     3.1415926535897936,\n                     2.2113504909352253,\n                     1.2158289201545691,\n                     -1.3746655614494785,\n                     -1.3944749412557864,\n                     -0.37269481654733194,\n                     -1.2131507666444596,\n                     -0.2929603566478166,\n                     -3.141592653589793,\n                     2.7360054115101713,\n                     1.462659293282678,\n                     -0.030978369796198283,\n                     -4.593270659377416,\n                     2.146158857081619,\n                     3.8432714931494187,\n                     2.226107556515043,\n                     -3.1415926535897936,\n                     -0.7510787657431719,\n                     2.818927827293758,\n                     -0.2796952785212738,\n                     1.7159293330234586,\n                     -1.658158213158069,\n                     2.0606115785894383,\n                     1.6894075654110476,\n                     -2.0183266607644494e-17,\n                     -1.097561054985494e-17,\n                     -1.170027477180718,\n                     3.141592653589793,\n                     -1.5250599141234784,\n                     2.1893776337589133,\n                     1.4686008662970813,\n                     1.610009484128973,\n                     -3.141592653589793,\n                     -3.141592653589793,\n                     1.0591793861554955,\n                     3.141592653589793,\n                     1.28697448178118,\n                     -0.5402432982433211,\n                     0.9578020357652939,\n                     -0.5605245247508785,\n                     3.141592653589793,\n                     -3.141592653589793,\n                     0.11651114501944934,\n                     3.141592653589793,\n                     -2.557858682256592,\n                     2.787982139225623,\n                     -1.7589980729426002,\n                     3.8518281175833535,\n                     -1.5707963267948966,\n                     -2.997539058395352,\n                     1.6333908462806441,\n                     -2.999902776521562,\n                     -1.6668292415473602,\n                     -2.3416814810735547,\n                     -0.3190545626931916,\n                     -1.0022370355752628,\n                     -1.1758108535150003e-16,\n                     -1.5707963267948966,\n                     -1.49688668473229,\n                     0.1092276667572707,\n                     -1.116635076589909,\n                     0.7273158990165092,\n                     -3.5514854616344877e-16,\n                     -3.141592653589793,\n                     1.5707963267948966,\n                     1.5707963267948966,\n                     1.3145263286586932,\n                     -1.1789709192162374,\n                     -3.2629847494128574,\n                     -1.6246852449622058],\n          \'coords\': [[0, 0], [0, 1], [1, 0], [1, 1], [2, 0], [2, 1], [3, 0], [3, 1]],\n          \'layers\': [[[0, 1], [2, 3], [4, 5], [6, 7]],\n                     [[0, 2], [1, 3], [4, 6], [5, 7]],\n                     [[2, 4], [3, 5], [0, 1], [6, 7]],\n                     [[0, 1], [2, 3], [4, 5], [6, 7]],\n                     [[0, 2], [1, 3], [4, 6], [5, 7]],\n                     [[2, 4], [3, 5], [0, 1], [6, 7]],\n                     [[0, 1], [2, 3], [4, 5], [6, 7]],\n                     [[0, 2], [1, 3], [4, 6], [5, 7]],\n                     [[2, 4], [3, 5], [0, 1], [6, 7]],\n                     [[0, 1], [2, 3], [4, 5], [6, 7]],\n                     [[0, 2], [1, 3], [4, 6], [5, 7]],\n                     [[2, 4], [3, 5], [0, 1], [6, 7]],\n                     [[0, 1], [2, 3], [4, 5], [6, 7]],\n                     [[0, 2], [1, 3], [4, 6], [5, 7]]]},\n (8, 4): {\'angles\': [1.342148489334907,\n                     -4.612114008872736,\n                     -0.5055213333918288,\n                     -2.442351980254898,\n                     1.5707963267948963,\n                     -1.5707963267948966,\n                     4.71238898038469,\n                     -1.5707963267948966,\n                     5.5195889672377545,\n                     -3.532078850472962,\n                     -3.4582841638009136,\n                     1.4180146845055248,\n                     3.8087564804381864,\n                     10.091941787617772,\n                     -3.746255035781058,\n                     2.1754587089861612,\n                     2.8032559155511527,\n                     4.1695980573432525,\n                     0.5168572971379731,\n                     2.6134153217969067,\n                     0.36603547559631594,\n                     -1.3742753164431483,\n                     3.8746911286404777,\n                     1.5707963267948968,\n                     -1.1067256021843812,\n                     1.4773640088256055,\n                     1.5707963267948968,\n                     8.813522445852739e-17,\n                     1.8299966339145368,\n                     -2.6841558389605695,\n                     1.0017651766069884,\n                     -1.2710065158619162,\n                     5.23695538276118,\n                     -2.005376070877285,\n                     1.5707963267948963,\n                     1.5707963267948972,\n                     -3.1415926535897936,\n                     -1.5707963267948968,\n                     6.223319612899869,\n                     1.5707963267948966,\n                     2.043808448093259,\n                     -1.5707963267948968,\n                     -3.141592653589793,\n                     1.5707963267948972,\n                     9.902629713708122e-18,\n                     -2.75723953456677e-16,\n                     -0.1736976233072595,\n                     5.0730971112734995,\n                     1.5707963267948966,\n                     -1.5707963267948968,\n                     -1.1028159302606837,\n                     1.4411394796797814,\n                     -3.141592653589793,\n                     -1.5707963267948966,\n                     -9.666891048842117e-17,\n                     -2.254966393059075e-17,\n                     3.141592653589794,\n                     0.2996808628723877,\n                     3.0597028088390537,\n                     -1.2150944469848233,\n                     3.9575497334626766e-16,\n                     -1.5707963267948966,\n                     -1.397098703487637,\n                     1.2100881959060872,\n                     4.712388980384929,\n                     -1.043298218363021,\n                     3.141592653589793,\n                     0.2701453725531202,\n                     -3.0722598631232356e-16,\n                     2.156970577508139,\n                     4.05288834663716,\n                     -1.8374115464844574,\n                     1.5707963267948968,\n                     -0.0033703839439966905,\n                     -1.5707963267948966,\n                     -3.4566197594509463e-18,\n                     6.298744508512385e-17,\n                     2.172950900540702,\n                     3.141592653589793,\n                     0.9400908425031144,\n                     -4.712388980384929,\n                     1.570796326794898,\n                     -4.71238898038469,\n                     -3.0523149439161976,\n                     2.186276035465284,\n                     1.5707963267948966,\n                     0.19061272694848302,\n                     -3.4082078732793537,\n                     -1.5707963267948968,\n                     9.138459623426163e-16,\n                     9.415552633385868e-17,\n                     -6.93392713101534e-17,\n                     -1.5707963267948966,\n                     -1.3005854598447509e-17,\n                     -2.0844629258405236,\n                     1.3876251559129937e-16,\n                     1.941890635282795,\n                     0.7327953705992574,\n                     -1.5707963267948966,\n                     -3.438080472071161,\n                     -3.1415926535897936,\n                     -1.7640017699633897e-16,\n                     -2.3561944901923453,\n                     -1.9817112991944992e-16,\n                     -3.1415926535897922,\n                     4.2264584775420873e-17,\n                     1.4655737820522332e-16,\n                     -1.5707963267948963,\n                     -5.699837118170861e-17,\n                     -2.819849061273205e-17,\n                     0.5136665990456272,\n                     5.496428860628118e-17,\n                     -1.1076322260409452e-16,\n                     -1.432987450681155e-16,\n                     -1.5707963267948966,\n                     1.5707963267948963,\n                     1.6102316138157857e-16,\n                     4.3418420262843594e-16,\n                     2.920790813687381e-16,\n                     -1.570796326794897],\n          \'coords\': [[0, 0], [0, 1], [1, 0], [1, 1], [2, 0], [2, 1], [3, 0], [3, 1]],\n          \'layers\': [[[0, 1], [2, 3], [4, 5], [6, 7]],\n                     [[0, 2], [1, 3], [4, 6], [5, 7]],\n                     [[2, 4], [3, 5], [0, 1], [6, 7]],\n                     [[0, 1], [2, 3], [4, 5], [6, 7]],\n                     [[0, 2], [1, 3], [4, 6], [5, 7]],\n                     [[2, 4], [3, 5], [0, 1], [6, 7]],\n                     [[0, 1], [2, 3], [4, 5], [6, 7]],\n                     [[0, 2], [1, 3], [4, 6], [5, 7]],\n                     [[2, 4], [3, 5], [0, 1], [6, 7]],\n                     [[0, 1], [2, 3], [4, 5], [6, 7]],\n                     [[0, 2], [1, 3], [4, 6], [5, 7]],\n                     [[2, 4], [3, 5], [0, 1], [6, 7]],\n                     [[0, 1], [2, 3], [4, 5], [6, 7]],\n                     [[0, 2], [1, 3], [4, 6], [5, 7]]]}}\n# INDEPENDENT NATIVE SEEDS END\n\n# THREE ARM SEEDS START\n# _MOONSHOT_BRICK_UPDATE: collaborator architecture, independently certified.\n_NATIVE_SEEDS.update({(8, 4): {\'angles\': [3.0683488265486365,\n                     2.0755939850993692e-15,\n                     1.4486402990198601,\n                     -1.8857631568219275,\n                     3.1415926535897936,\n                     -1.8140777569447513,\n                     -2.4127891465319204,\n                     1.1053494133402593,\n                     0.9532266634319345,\n                     0.4421773679406813,\n                     -0.0728315423816641,\n                     3.245438424338353,\n                     -2.8193399466089497,\n                     -1.814077756944751,\n                     -2.2778084578107265,\n                     0.11120389607995111,\n                     0.047075150401809034,\n                     -3.029904300938692,\n                     0.7971701110055627,\n                     0.38994529549444507,\n                     1.9733653001006002,\n                     3.141592653589793,\n                     0.25860583208901083,\n                     -1.370268392949318,\n                     -0.053993258037614594,\n                     0.5092226733252898,\n                     -1.9361584503424931,\n                     -0.2924684781910074,\n                     1.1718968029157897,\n                     3.1415926535897936,\n                     0.07196075597576868,\n                     1.8574827944870789,\n                     3.141592653589793,\n                     0.04391814927894965,\n                     1.1493381996151466,\n                     -2.2111147974218808,\n                     -0.006527751211751996,\n                     9.349448168705862e-17,\n                     -3.9023000280859135,\n                     -2.43717435516382,\n                     3.1183487213799967,\n                     -0.043918149278949874,\n                     -3.319605082403784,\n                     2.08582431918424,\n                     -1.5134905509276575,\n                     -3.15430176418747e-17,\n                     -0.5146621268288687,\n                     2.6635671534005425,\n                     -2.163032103255488,\n                     -0.8529107517645595,\n                     -0.7142634710338088,\n                     0.6458172444226458,\n                     0.2667753656834031,\n                     4.71238898038469,\n                     1.3133685484405802,\n                     2.4653967933320993,\n                     1.4510097771769122,\n                     -2.356194490192345,\n                     2.4835526647752313,\n                     1.5081061182749478,\n                     -2.822154553695377,\n                     0.31943809989441674,\n                     3.3795954683371865,\n                     -3.4909121500850655,\n                     1.7717991241598479,\n                     -1.5707963267948968,\n                     -1.0578325061141673,\n                     1.9083795474720673,\n                     1.744333584917592,\n                     -1.5707963267948961,\n                     -0.12569214617404678,\n                     1.114190278288815,\n                     -1.7717991241598479,\n                     -1.5707963267948966,\n                     1.5707963267948968,\n                     0.10031409916439452,\n                     -0.17353725812269502,\n                     -7.337219333560441e-17,\n                     1.5707963267948966,\n                     0.04196993879583287,\n                     1.5707963267948963,\n                     -1.5707963267948966,\n                     7.810639069778934e-17,\n                     -1.5707963267948966,\n                     1.4804104797636398e-16,\n                     3.141592653589793,\n                     -3.141592653589793,\n                     1.5707963267948966,\n                     3.141592653589793,\n                     1.5707963267948963,\n                     -1.5707963267948966,\n                     1.5707963267948966,\n                     -1.5707963267948966,\n                     4.1646736352431126e-16,\n                     -1.6804808442211747e-16,\n                     1.248528971343653e-17],\n          \'coords\': [[0, 0], [0, 1], [0, 2], [0, 3], [1, 0], [1, 1], [1, 2], [1, 3]],\n          \'layers\': [[[0, 1], [2, 3], [4, 5], [6, 7]],\n                     [[0, 4], [1, 5], [2, 6], [3, 7]],\n                     [[1, 2], [5, 6]],\n                     [[0, 1], [2, 3], [4, 5], [6, 7]],\n                     [[0, 4], [1, 5], [2, 6], [3, 7]],\n                     [[1, 2], [5, 6]],\n                     [[0, 1], [2, 3], [4, 5], [6, 7]],\n                     [[0, 4], [1, 5], [2, 6], [3, 7]],\n                     [[1, 2], [5, 6]],\n                     [[0, 1], [2, 3], [4, 5], [6, 7]],\n                     [[0, 4], [1, 5], [2, 6], [3, 7]]],\n          \'source\': \'Claude data/brick/8_4_2x4_d11.json; amplitude-refined under bonus/\'}})\n\n_THREE_ARM_SEEDS = {12: {\'N\': 6,\n      \'angles\': [-1.7753612101505012,\n                 -1.5031115642308233,\n                 -1.2974973838927273,\n                 4.795027544633506,\n                 1.5707963267948966,\n                 1.5707963267948974,\n                 0.5896530708477706,\n                 -2.258350892005411,\n                 -3.7687001756936302,\n                 2.255551546582445,\n                 -0.10784931344318781,\n                 2.699974722275354,\n                 -3.6945708360575424,\n                 3.141592653589793,\n                 -1.5707963267948966,\n                 3.141592653589793,\n                 -2.5921254088321657,\n                 2.877531407632771e-17,\n                 -4.604551122942012e-17,\n                 -0.5494672447576275,\n                 -2.356194490192345,\n                 2.356194490192345,\n                 1.570796326794896,\n                 -5.497787143782138,\n                 -1.5707963267948966,\n                 4.837678183550181,\n                 -3.9269908169872414,\n                 1.5707963267948981,\n                 -6.3665077052936025,\n                 1.5707963267948966,\n                 1.5707963267948966,\n                 -9.166390104959884e-18,\n                 -3.141592653589793,\n                 2.2979787286487854e-16,\n                 -1.5707963267948968,\n                 1.1788520049107612e-16,\n                 -3.1415926535897936,\n                 -4.987980590979463e-19,\n                 4.71238898038469,\n                 -3.141592653589793,\n                 7.9373040320885,\n                 3.1415926535897936],\n      \'layers\': [[[0, 2], [1, 3], [4, 5]],\n                 [[0, 1], [2, 4], [3, 5]],\n                 [[0, 2], [1, 3], [4, 5]],\n                 [[0, 1], [2, 4], [3, 5]],\n                 [[0, 2], [1, 3], [4, 5]],\n                 [[0, 1], [2, 3], [4, 5]]],\n      \'n\': 2,\n      \'residual\': 5.086359306078474e-14,\n      \'sizes\': [4, 4, 4],\n      \'total\': 12},\n 15: {\'N\': 6,\n      \'angles\': [-1.440790703919356,\n                 -2.287067197151006,\n                 -1.255914435443566,\n                 4.751237396643934,\n                 1.5707963267948963,\n                 1.5707963267948954,\n                 0.184716530316489,\n                 -3.1861799918514793,\n                 -2.7628019666403714,\n                 3.200721579119603,\n                 0.2754057996269368,\n                 2.302244212602228,\n                 -3.9299654461729254,\n                 3.1415926535897936,\n                 -1.5707963267948968,\n                 3.141592653589793,\n                 -2.5776500122291646,\n                 -5.4397938530322016e-17,\n                 1.71019330737766e-17,\n                 -0.5639426413606287,\n                 -2.356194490192345,\n                 2.356194490192345,\n                 1.570796326794897,\n                 -5.497787143782138,\n                 -1.5707963267948968,\n                 3.8214782192759196,\n                 -3.926990816987242,\n                 1.5707963267948948,\n                 -9.17147783224243,\n                 1.5707963267948968,\n                 1.5707963267948966,\n                 -1.0650113191615857e-16,\n                 -3.141592653589793,\n                 -3.913842428212354e-16,\n                 -1.570796326794897,\n                 -1.2239910749985933e-16,\n                 -3.141592653589793,\n                 -3.587376619827427e-17,\n                 4.71238898038469,\n                 -3.141592653589793,\n                 10.742274159037327,\n                 3.141592653589793],\n      \'layers\': [[[0, 2], [1, 3], [4, 5]],\n                 [[0, 1], [2, 4], [3, 5]],\n                 [[0, 2], [1, 3], [4, 5]],\n                 [[0, 1], [2, 4], [3, 5]],\n                 [[0, 2], [1, 3], [4, 5]],\n                 [[0, 1], [2, 3], [4, 5]]],\n      \'n\': 2,\n      \'residual\': 7.5645951984681e-14,\n      \'sizes\': [5, 5, 5],\n      \'total\': 15}}\n# THREE ARM SEEDS END\n\n\n_BINARY_ARM_SEEDS = {(8, 3): {\'angles\': [6.791309984074567,\n                     1.2970547065149216,\n                     1.937332664604971,\n                     2.7834673545082906,\n                     -2.235106378101703,\n                     1.0795729687353997,\n                     5.952062206164358,\n                     -4.3562717395437955,\n                     -0.447001966756547,\n                     -1.3281782668235351,\n                     -1.8434726546602784,\n                     -2.0960139450373805,\n                     1.7894665295130796,\n                     -4.3901811048856265,\n                     0.38201307796666556,\n                     -0.8619878890763916,\n                     -2.9487620692886285,\n                     -2.3558441111272512,\n                     -2.211929714542513,\n                     -4.806157693182201,\n                     3.141592653589793,\n                     -2.972200846183414,\n                     -1.5707963267948968,\n                     -2.356194490192345,\n                     -2.3512195480212044,\n                     -1.7069561021640238,\n                     -3.141592653589793,\n                     -2.9196286445262265,\n                     -3.7401296016226724e-17,\n                     2.356194490192345,\n                     -1.5195852074316494,\n                     2.695370369756649,\n                     1.5707963267948966,\n                     1.5707963267948963,\n                     3.141592653589793,\n                     2.356194490192345,\n                     3.141592653589793,\n                     -3.141592653589793,\n                     1.5707963267948963,\n                     1.5707963267948963,\n                     1.4203330146806535e-16,\n                     2.3561944901923444],\n          \'layers\': [[[0, 2], [1, 3], [4, 5]],\n                     [[0, 1], [2, 3], [4, 5]],\n                     [[0, 1], [2, 4], [3, 5]],\n                     [[0, 2], [1, 3], [4, 5]],\n                     [[0, 1], [2, 4], [3, 5]],\n                     [[0, 2], [1, 3], [4, 5]]],\n          \'sizes\': [3, 3, 2]},\n (9, 3): {\'angles\': [-1.5605130654690498,\n                     0.698251699395309,\n                     -1.8978291563021943,\n                     0.8850510741172648,\n                     0.9679511880150917,\n                     -1.8432408752148859,\n                     1.6027982189446965,\n                     -3.196172371652146,\n                     -0.030305247274619578,\n                     0.3330189380002476,\n                     -1.208557913419086,\n                     1.1506010609947217,\n                     -1.5707963267948966,\n                     -3.8057649774272164,\n                     -3.1415926535897936,\n                     -0.7375999863581832,\n                     -2.747094528374111,\n                     -1.898471098709491,\n                     3.141592653589793,\n                     3.141592653589793,\n                     1.5707963267948966,\n                     2.8255073745347694,\n                     0.5253750960912921,\n                     0.719256628749094,\n                     1.6705144083207144,\n                     -3.141592653589793,\n                     -2.3315082489859402,\n                     3.1584661232366282,\n                     1.8338857474303258,\n                     2.910958849568834,\n                     0.984557075556895,\n                     4.91099063381433,\n                     -1.3261584796698107,\n                     -3.811836802723338,\n                     -1.0023296119794952e-16,\n                     -2.678080165767608,\n                     1.5707963267948972,\n                     2.410598786651478,\n                     -1.5707963267948966,\n                     -1.4175050507827627,\n                     -3.1415926535897936,\n                     2.032191403412983,\n                     4.4884693000189997e-17,\n                     -0.060072296456705414,\n                     3.1415926535897936,\n                     1.8491372902408565,\n                     -3.141592653589793,\n                     1.9214387432720879,\n                     -3.1415926535897936,\n                     2.6097005068389114,\n                     8.314721472399146e-17,\n                     -3.1759782938351253,\n                     3.141592653589793,\n                     4.512081912020453],\n          \'layers\': [[[0, 2], [1, 3], [4, 5]],\n                     [[0, 1], [2, 4], [3, 5]],\n                     [[0, 1], [2, 3], [4, 5]],\n                     [[0, 2], [1, 3], [4, 5]],\n                     [[0, 1], [2, 4], [3, 5]],\n                     [[0, 2], [1, 3], [4, 5]],\n                     [[0, 1], [2, 4], [3, 5]],\n                     [[0, 1], [2, 3], [4, 5]]],\n          \'sizes\': [3, 3, 3]},\n (9, 4): {\'angles\': [5.446533361244669,\n                     1.816855686565328,\n                     3.4333108555744603,\n                     3.9537876184428287,\n                     -5.573165589969889,\n                     4.888584556268617,\n                     -0.21257814578303638,\n                     2.8178332445898975,\n                     -1.2036408873599573,\n                     3.654115121680013,\n                     -4.492343658847489,\n                     -3.0195825494883906,\n                     3.1721293219645634,\n                     0.2732503330659304,\n                     1.9135041785781997,\n                     -0.9279896265945153,\n                     -0.36060829857775817,\n                     -3.604994304092107,\n                     0.47327846501523,\n                     2.2774503016628445,\n                     -1.107944959809961,\n                     -2.1814846481848518,\n                     -1.0738922086700067,\n                     -4.681032628243082,\n                     -1.1181941721570279,\n                     0.7833371522714115,\n                     -2.911983156673193,\n                     -1.4528253510720253,\n                     3.5359002136470434,\n                     3.1415926535897927,\n                     -1.8902896828165667,\n                     3.5735596687426647,\n                     2.614993197684157,\n                     0.30220159060840796,\n                     0.49690411812489,\n                     4.741340534009008,\n                     3.0422113891738065,\n                     -2.2327135855039772,\n                     2.459422430766712,\n                     2.3537550864095262,\n                     3.1295481985623166,\n                     2.565835246362213,\n                     1.8780927446584088,\n                     -2.881147877479258,\n                     1.5707963267948966,\n                     -1.6998346876794364,\n                     -1.5707963267948966,\n                     -1.1082749922049109,\n                     -3.141592653589793,\n                     3.1224621015894884,\n                     -4.71238898038469,\n                     -1.9869504600968506,\n                     3.1415926535897936,\n                     3.3646249530542116],\n          \'layers\': [[[0, 2], [1, 3], [4, 5]],\n                     [[0, 1], [2, 3], [4, 5]],\n                     [[0, 1], [2, 4], [3, 5]],\n                     [[0, 2], [1, 3], [4, 5]],\n                     [[0, 1], [2, 3], [4, 5]],\n                     [[0, 2], [1, 3], [4, 5]],\n                     [[0, 1], [2, 4], [3, 5]],\n                     [[0, 2], [1, 3], [4, 5]]],\n          \'sizes\': [3, 3, 3]}}\n\n_BINARY_DECODER3 = {\'angles\': [2.082506231059624e-16,\n            1.5707963267948966,\n            1.5707963267948966,\n            -1.5707963267948963,\n            -0.9553166181245092,\n            -3.141592653589793,\n            2.5261129449194057,\n            2.356194490192345,\n            1.5707963267948966,\n            1.5707963267948966,\n            -1.5707963267948968,\n            0.7853981633974482,\n            3.1415926535897936,\n            1.5707963267948966,\n            -1.5707963267948966,\n            1.5707963267948968,\n            -7.873629922416639e-17],\n \'edges\': [[1, 2], [0, 1], [1, 2], [0, 1], [1, 2], [0, 1], [1, 2]]}\n\ndef _binary_decoder3(ops, q):\n    """Coherently map binary (low, high, 0) to D(3,j), j=0..3."""\n    data = _BINARY_DECODER3\n    for i in range(3):\n        _u(ops,q[i],ry(data[\'angles\'][i]))\n    for j,(a,b) in enumerate(data[\'edges\']):\n        ops.append((q[a],q[b]))\n        _u(ops,q[a],ry(data[\'angles\'][3+2*j]))\n        _u(ops,q[b],ry(data[\'angles\'][4+2*j]))\n\n\ndef _binary_three_arm(n,k,open_center=False):\n    data = _BINARY_ARM_SEEDS.get((n,k))\n    if data is None:\n        return None\n    arms = [[cirq.GridQubit(4+i,4+j) for j in range(size)]\n            for i,size in enumerate(data[\'sizes\'])]\n    roots = [q for arm in arms for q in arm[:2]]\n    outputs = {q for arm in arms for q in arm}\n    ops = []\n    for layer in range(len(data[\'layers\'])+1):\n        if layer:\n            ops.extend((roots[a],roots[b]) for a,b in data[\'layers\'][layer-1])\n        for i,q in enumerate(roots):\n            _u(ops,q,ry(data[\'angles\'][6*layer+i]))\n    for arm in arms:\n        if len(arm) == 3:\n            _binary_decoder3(ops,arm)\n        else:\n            assert len(arm) == 2\n            _cx(ops,arm[1],arm[0])\n            _givens(ops,*arm,math.pi/4)\n    if open_center:\n        # Free the enclosed central output for the exterior-cat layout. Each\n        # two-CNOT move has a known-zero destination and restores its source.\n        assert n == 9\n        center,neighbor,bridge,fresh = [cirq.GridQubit(5,c) for c in [5,6,7,8]]\n        for source,dest in [(neighbor,bridge),(bridge,fresh),(center,neighbor)]:\n            _cx(ops,source,dest); _cx(ops,dest,source)\n        outputs.remove(center); outputs.add(fresh)\n    return _optimize_depth(_compile(ops)),outputs\n\n\ndef _three_arm(n):\n    data = _THREE_ARM_SEEDS.get(n)\n    if data is None:\n        return None\n    arms = [[cirq.GridQubit(4+i,4+j) for j in range(size)]\n            for i,size in enumerate(data[\'sizes\'])]\n    roots = [q for arm in arms for q in arm[:2]]\n    ops = []\n    for layer in range(len(data[\'layers\'])+1):\n        if layer:\n            ops.extend((roots[a],roots[b]) for a,b in data[\'layers\'][layer-1])\n        for i,q in enumerate(roots):\n            _u(ops,q,ry(data[\'angles\'][6*layer+i]))\n    for arm in arms:\n        _unary_fused(ops,arm,2)\n    return _optimize_depth(_compile(ops)),{q for arm in arms for q in arm}\n\n\ndef _native_seed(n,k):\n    data = _NATIVE_SEEDS.get((n,k))\n    if data is None:\n        return None\n    q = [cirq.GridQubit(4+r,4+c) for r,c in data[\'coords\']]\n    ops = []\n    for layer in range(len(data[\'layers\'])+1):\n        if layer:\n            ops.extend((q[a],q[b]) for a,b in data[\'layers\'][layer-1])\n        for i in range(n):\n            _u(ops,q[i],ry(data[\'angles\'][layer*n+i]))\n    return _optimize_depth(_compile(ops)),set(q)\n\n\ndef _subtract_balanced_seed(n,k):\n    """Measure an isolated output diagonal, then complement the zero branch.\n\n    A final monomial gate on the measured bit is absorbed into the readout\'s\n    invert mask. Its phase is branch-global. Full diagonal measurements and\n    remaining data gates are then scheduled together with causal feedforward.\n    """\n    if n != 2*k+1:\n        return None\n    seeds = [v for v in (_native_seed(n+1,k+1), _round4_unitary(n+1,k+1), _round6_unitary(n+1,k+1), _round7_unitary(n+1,k+1), _round8_unitary(n+1,k+1), _round9_unitary(n+1,k+1), _round10_unitary(n+1,k+1), _round11_unitary(n+1,k+1)) if v is not None]\n    if not seeds:\n        return None\n    physical = _graph()\n    candidates = []\n    for circuit,outputs in seeds:\n        choices = [q for q in outputs\n                   if sum(x.row+x.col == q.row+q.col for x in outputs) == 1]\n        for q in sorted(choices):\n            candidate = cirq.Circuit(circuit)\n            index = max(i for i,m in enumerate(candidate) if q in m.qubits)\n            op = candidate[index].operation_at(q)\n            invert = False\n            if len(op.qubits) == 1:\n                matrix = cirq.unitary(op)\n                diagonal_gate = abs(matrix[0,1])+abs(matrix[1,0]) < 1e-13\n                antidiagonal_gate = abs(matrix[0,0])+abs(matrix[1,1]) < 1e-13\n                if diagonal_gate or antidiagonal_gate:\n                    candidate[index] = candidate[index].without_operations_touching([q])\n                    invert = antidiagonal_gate\n            diagonal = sorted(x for x in physical if x.row+x.col == q.row+q.col)\n            candidate.append(cirq.Moment(cirq.measure(x,key=f\'subtract_{x.row}_{x.col}\',\n                            invert_mask=(bool(invert and x == q),)) for x in diagonal))\n            retained = outputs-{q}\n            condition = cirq.SympyCondition(sympy.Eq(sympy.Symbol(f\'subtract_{q.row}_{q.col}\'),0))\n            candidate.append(cirq.Moment(cirq.X(x).with_classical_controls(condition) for x in retained))\n            candidate = _schedule_adaptive(candidate)\n            candidates.append((candidate,retained))\n    return min(candidates,key=lambda item:len(item[0])) if candidates else None\n\n\ndef _w_from_tree(rounds):\n    edges = [edge for rnd in rounds for edge in rnd]\n    root = edges[0][0]\n    children = {}\n    for a, b in edges:\n        children.setdefault(a, []).append(b)\n    sizes = {}\n\n    def size(q):\n        sizes[q] = 1+sum(size(c) for c in children.get(q, []))\n        return sizes[q]\n\n    size(root)\n    remaining = dict(sizes)\n    ops = []\n    for i, (a, b) in enumerate(edges):\n        theta = math.asin(math.sqrt(sizes[b]/remaining[a]))\n        if i == 0:\n            _u(ops, a, ry(2*theta))\n            _cx(ops, a, b)\n            _u(ops, a, X)\n        else:\n            eta = math.pi/2-theta\n            _u(ops, b, ry(eta))\n            _cx(ops, a, b)\n            _u(ops, b, ry(-eta))\n            _cx(ops, b, a)\n        remaining[a] -= sizes[b]\n    return _compile(ops), set(sizes)\n\n\ndef _w(n):\n    graph = _graph()\n    rng = random.Random(83419+n)\n    best = None\n    finalists = []\n    # Fast deterministic offline-style tree search, bounded independently of\n    # statevector size. Both geometry and asynchronous native depths matter.\n    root = cirq.GridQubit(6,7)\n    for trial in range(96):\n        used = {root}\n        rounds = []\n        while len(used) < n:\n            candidates = [(a,b) for a in sorted(used) for b in sorted(graph[a]-used)]\n            rng.shuffle(candidates)\n            busy, rnd = set(), []\n            for a, b in candidates:\n                if a in busy or b in busy:\n                    continue\n                rnd.append((a,b))\n                busy.update((a,b))\n                if len(used)+len(rnd) == n:\n                    break\n            rounds.append(rnd)\n            used.update(b for _,b in rnd)\n        candidate = _w_from_tree(rounds)\n        candidate = (_optimize_frames(candidate[0]), candidate[1])\n        finalists.append(candidate)\n        if best is None or len(candidate[0]) < len(best[0]):\n            best = candidate\n    for candidate,outputs in sorted(finalists,key=lambda item:len(item[0]))[:16]:\n        candidate = _optimize_depth(candidate)\n        if len(candidate) < len(best[0]):\n            best = candidate,outputs\n    return best\n\n\ndef _schedule_adaptive(circuit, trials=16):\n    """ASAP/list schedule with causal keys and indivisible readout diagonals.\n\n    Shared-wire commutation is used only for unconditional diagonal unitaries.\n    Every physical readout diagonal remains one scheduling block. Measurements,\n    native CZs, local gates and conditional corrections can share a moment.\n    """\n    blocks = []\n    for moment in circuit:\n        measured = collections.defaultdict(list)\n        for op in moment:\n            if cirq.is_measurement(op):\n                assert len(op.qubits) == 1\n                measured[sum((op.qubits[0].row, op.qubits[0].col))].append(op)\n            else:\n                blocks.append((op,))\n        blocks.extend(tuple(ops) for _, ops in sorted(measured.items()))\n    support = [set(q for op in block for q in op.qubits) for block in blocks]\n    diagonal = []\n    for block in blocks:\n        op = block[0]\n        matrix = cirq.unitary(op, default=None) if len(block) == 1 else None\n        diagonal.append(matrix is not None and not op.classical_controls and\n                        (op.gate == cirq.CZ or\n                         (len(op.qubits) == 1 and abs(matrix[0,1])+abs(matrix[1,0]) < 1e-13)))\n    pred = [set() for _ in blocks]\n    succ = [set() for _ in blocks]\n    by_qubit, key_owner = {}, {}\n    for j, block in enumerate(blocks):\n        for q in support[j]:\n            for i in by_qubit.get(q, []):\n                if not (diagonal[i] and diagonal[j]):\n                    pred[j].add(i)\n            by_qubit.setdefault(q, []).append(j)\n        for op in block:\n            for key in cirq.control_keys(op):\n                pred[j].add(key_owner[key])\n        for op in block:\n            if cirq.is_measurement(op):\n                for key in cirq.measurement_key_objs(op):\n                    key_owner[key] = j\n        for i in pred[j]:\n            succ[i].add(j)\n    heights = [0]*len(blocks)\n    for i in range(len(blocks)-1, -1, -1):\n        heights[i] = 1+max((heights[j] for j in succ[i]), default=0)\n    best = cirq.Circuit(circuit)\n    for seed in range(trials):\n        rng = random.Random(983+seed)\n        ranks = [rng.random() for _ in blocks]\n        counts = [len(p) for p in pred]\n        ready = {i for i, count in enumerate(counts) if not count}\n        moments = []\n        while ready:\n            busy, chosen = set(), []\n            for i in sorted(ready, key=lambda i:(-heights[i], ranks[i])):\n                if not busy.intersection(support[i]):\n                    chosen.append(i); busy.update(support[i])\n            moments.append(cirq.Moment(op for i in chosen for op in blocks[i]))\n            ready.difference_update(chosen)\n            for i in chosen:\n                for j in succ[i]:\n                    counts[j] -= 1\n                    if counts[j] == 0:\n                        ready.add(j)\n        assert sum(len(m) for m in moments) == sum(len(b) for b in blocks)\n        if len(moments) < len(best):\n            best = cirq.Circuit(moments)\n    return best\n\n\n@functools.lru_cache(maxsize=64)\ndef _best(n, k):\n    prebuilt = _round11_native(n,k) or _round10_unitary(n,k) or _round9_unitary(n,k) or _round8_unitary(n,k) or _round7_unitary(n,k) or _round6_unitary(n,k) or _round5_unitary(n,k) or _round4_unitary(n,k)\n    if prebuilt is not None:\n        c,q = prebuilt\n        method = _ROUND11_VERIFIED[(n,k)][\'method\'] if (n,k) in _ROUND11_VERIFIED else _ROUND10_VERIFIED.get((n,k), _ROUND9_VERIFIED.get((n,k), _ROUND8_VERIFIED.get((n,k), _ROUND7_VERIFIED.get((n,k), _ROUND6_VERIFIED.get((n,k), _ROUND5_VERIFIED.get((n,k), _ROUND4_VERIFIED.get((n,k))))))))[\'method\']\n        subtraction = _subtract_balanced_seed(n,k)\n        if subtraction is not None and len(subtraction[0]) < len(c):\n            c,q = subtraction\n            method = \'verified balanced seed + deterministic measurement/complement\'\n        if n == 2*k:\n            adaptive = balanced_candidate(n)\n            if len(adaptive[\'circuit\']) < len(c):\n                c,q = cirq.Circuit(adaptive[\'circuit\']),set(adaptive[\'outputs\'])\n                method = \'adaptive cat complement + even-site teleportation\'\n        for op in c.all_operations():\n            cirq_google.Willow105.validate_operation(op.without_classical_controls())\n        if not c.has_measurements():\n            cirq_google.Willow105.validate_circuit(c)\n        return cirq.FrozenCircuit(c),frozenset(q),method\n    if k == 1:\n        c, q = _w(n)\n        method = \'parallel W tree\'\n    elif (n, k) == (4, 2):\n        c, q = _square()\n        method = \'analytic square\'\n    else:\n        candidates = []\n        for a in sorted({n//2, (n+1)//2}):\n            for small in ([\'fused\', True, False, \'pruned\'] if k == 2 else [\'fused\', False, \'pruned\']):\n                candidate, outputs = _divide(n, k, a, small)\n                candidate = _optimize_depth(candidate)\n                candidates.append((len(candidate), candidate, outputs,\n                                   f\'hypergeometric {a}+{n-a}\' +\n                                   (\' / \'+small if isinstance(small,str) else \' / U3\' if small else \'\')))\n            if n%2 == 0 and k >= 3:\n                candidate,outputs = _divide(n,k,a,small=\'fused\',pivot=k//2)\n                candidate = _optimize_depth(candidate)\n                candidates.append((len(candidate),candidate,outputs,\n                                   f\'hypergeometric {a}+{n-a} / center-out seed\'))\n        _, c, q, method = min(candidates, key=lambda item:item[0])\n        seed = _native_seed(n,k)\n        if seed is not None and len(seed[0]) < len(c):\n            c,q = seed\n            method = \'independent native grid seed\'\n        arms = _three_arm(n) if k == 2 else None\n        if arms is not None and len(arms[0]) < len(c):\n            c,q = arms\n            method = \'three-arm unary distribution + parallel decoders\'\n        binary = _binary_three_arm(n,k)\n        if binary is not None and len(binary[0]) < len(c):\n            c,q = binary\n            method = \'three-arm binary distribution + coherent decoders\'\n        subtraction = _subtract_balanced_seed(n,k)\n        if subtraction is not None and len(subtraction[0]) < len(c):\n            c,q = subtraction\n            method = \'native balanced seed + deterministic measurement/complement\'\n        if n == 2*k and n >= 6:\n            adaptive = balanced_candidate(n)\n            if len(adaptive[\'circuit\']) < len(c):\n                c, q = cirq.Circuit(adaptive[\'circuit\']), set(adaptive[\'outputs\'])\n                method = \'adaptive cat complement + even-site teleportation\'\n    for op in c.all_operations():\n        cirq_google.Willow105.validate_operation(op.without_classical_controls())\n    if not c.has_measurements():\n        cirq_google.Willow105.validate_circuit(c)\n    return cirq.FrozenCircuit(c), frozenset(q), method\n\n\ndef create_dicke_state_adaptive(num_qubits, num_excitations):\n    """Return an exact D(N,n) for all confirmed bonus instances.\n\n    The interface permits feedforward; the portfolio may choose a shallower\n    unitary circuit. N=2..15, n=1..floor(N/2). Build time is bounded and cached.\n    """\n    if (isinstance(num_qubits, bool) or isinstance(num_excitations, bool)\n            or not isinstance(num_qubits, int) or not isinstance(num_excitations, int)\n            or not 2 <= num_qubits <= 15 or not 1 <= num_excitations <= num_qubits//2):\n        raise NotImplementedError(\'Bonus domain: integer 2<=N<=15, 1<=n<=N//2\')\n    circuit, qubits, _ = _best(num_qubits, num_excitations)\n    return cirq.Circuit(circuit), set(qubits)\n\n\ndef method_for(num_qubits, num_excitations):\n    return _best(num_qubits, num_excitations)[2]\n\n\n# Diagonal-safe adaptive complement; self-contained submission support.\ndef _layout(core_outputs, reserved=None):\n    graph = _graph()\n    core = set(core_outputs)\n    reserved = core if reserved is None else set(reserved) | core\n    odd = {q for q in graph if (q.row+q.col)%2}\n    even = set(graph)-odd\n    external_cats = odd-reserved\n    assignment = {}\n    # A core wire restored to zero can be a later teleportation destination.\n    # It is still excluded from the concurrently prepared cat resource below.\n    occupied = set(core)\n    # Retained even sites need an outside neighboring cat.\n    destinations = {q:q for q in core if q in even}\n    for q in sorted(core & odd):\n        candidates = [v for v in graph[q]-occupied if v in even\n                      and graph[v] & external_cats]\n        if not candidates:\n            raise ValueError(\'No fresh even teleportation destination\')\n        v = min(candidates, key=lambda v: (-len(graph[v]&external_cats), v))\n        destinations[q] = v\n        occupied.add(v)\n    usage = collections.Counter()\n    for q, dest in sorted(destinations.items()):\n        choices = graph[dest] & external_cats\n        if not choices:\n            raise ValueError(\'No outside cat neighbor for an even output\')\n        cat = min(choices, key=lambda c:(usage[c], c))\n        usage[cat] += 1\n        assignment[dest] = cat\n\n    # Distance-two cat graph. Every chosen even mediator is outside the core;\n    # a mediator may also be a future output, since it is restored to |0>.\n    cat_graph = {q:{} for q in external_cats}\n    for mediator in sorted(even-reserved):\n        neighbors = sorted(graph[mediator] & external_cats)\n        for a in neighbors:\n            for b in neighbors:\n                if a != b:\n                    cat_graph[a].setdefault(b, mediator)\n    terminals = set(assignment.values())\n    best = None\n    for root in sorted(cat_graph):\n        parent = {root:None}\n        distance = {root:0}\n        queue = collections.deque([root])\n        while queue:\n            a = queue.popleft()\n            for b in sorted(cat_graph[a]):\n                if b not in parent:\n                    parent[b] = a\n                    distance[b] = distance[a]+1\n                    queue.append(b)\n        if not terminals <= parent.keys():\n            continue\n        included = {root}\n        for terminal in terminals:\n            v = terminal\n            while v not in included:\n                included.add(v)\n                v = parent[v]\n        edges = [(parent[q], q, cat_graph[parent[q]][q])\n                 for q in sorted(included-{root}, key=lambda q:(distance[q],q))]\n        ops = [(root,H)]\n        for a,b,m in edges:\n            _cx(ops,a,m)\n            _cx(ops,m,b)\n            _cx(ops,a,m)\n        cat_circuit = _compile(ops)\n        root_options = [(cat, target) for cat in included\n                        for target in sorted(graph[cat]-occupied) if target in even]\n        if not root_options:\n            continue\n        root_cat, output_root = min(root_options, key=lambda ct:(usage[ct[0]],ct))\n        candidate = (len(cat_circuit), len(included), cat_circuit, included,\n                     root_cat, output_root, edges)\n        if best is None or candidate[:2] < best[:2]:\n            best = candidate\n    if best is None:\n        raise ValueError(\'External cat graph is disconnected\')\n    _,_,cat_circuit,cats,root_cat,output_root,edges = best\n    return dict(destinations=destinations, assignment=assignment,\n                cat_circuit=cat_circuit, cats=cats, root_cat=root_cat,\n                root=output_root, cat_edges=edges)\n\n\ndef _key(q):\n    return f\'odd_{q.row}_{q.col}\'\n\n\ndef _parity(keys):\n    return cirq.SympyCondition(sympy.Eq(sympy.Mod(sum(sympy.Symbol(k) for k in keys),2),1))\n\n\n@functools.lru_cache(maxsize=8)\ndef balanced_candidate(n):\n    if n%2 or not 6 <= n <= 14:\n        raise ValueError(\'Use even N=6..14\')\n    k = n//2\n    cores = []\n    round4_core = _round4_unitary(n-1,k-1)\n    if round4_core is not None:\n        try:\n            _layout(round4_core[1],reserved=round4_core[0].all_qubits())\n            cores.append(round4_core)\n        except ValueError:\n            pass\n    if not cores:\n        for a in sorted({(n-1)//2,n//2}):\n            for mode in [True,\'pruned\',\'fused\']:\n                candidate,output = _divide(n-1,k-1,a=a,small=mode)\n                cores.append((_optimize_depth(candidate),output))\n    seed = _native_seed(n-1,k-1)\n    if seed is not None:\n        cores.append(seed)\n    binary = _binary_three_arm(n-1,k-1,open_center=True) if n == 10 else None\n    if binary is not None:\n        cores.append(binary)\n    viable = []\n    for candidate,output in sorted(cores,key=lambda pair:len(pair[0])):\n        try:\n            placement = _layout(output,reserved=candidate.all_qubits())\n        except ValueError:\n            continue\n        viable.append((candidate,output,placement))\n        break\n    assert viable, \'No diagonal-safe cat/core layout\'\n    core,core_outputs,layout = viable[0]\n    cat = layout[\'cat_circuit\']\n    # Nontrivial core and cat operations have disjoint supports. Preparing\n    # the resource does not add serial depth until it exceeds core duration.\n    assert not (core.all_qubits() & cat.all_qubits())\n    prefix = cirq.Circuit(core.all_operations(), cat.all_operations())\n    suffix_ops = []\n    for old,dest in sorted(layout[\'destinations\'].items()):\n        if old != dest:\n            _cx(suffix_ops,old,dest)\n    for dest,control in sorted(layout[\'assignment\'].items()):\n        _cx(suffix_ops,control,dest)\n    _cx(suffix_ops,layout[\'root_cat\'],layout[\'root\'])\n    # Move the final X before measurement/correction. Its anticommutation\n    # with the conditional Z contributes only a branch-global phase.\n    _u(suffix_ops,layout[\'root\'],X)\n    odd = sorted(q for q in _graph() if (q.row+q.col)%2)\n    measured_active = layout[\'cats\'] | {q for q in core_outputs if (q.row+q.col)%2}\n    for q in sorted(measured_active):\n        _u(suffix_ops,q,H)\n    suffix_unitary = _compile(suffix_ops)\n    # Fuse every boundary and pack gates without ever moving across readout.\n    prep_ops = []\n    for op in itertools_ops(prefix, suffix_unitary):\n        if len(op.qubits) == 1:\n            prep_ops.append((op.qubits[0],cirq.unitary(op)))\n        else:\n            assert op.gate == cirq.CZ\n            prep_ops.append(op.qubits)\n    unscheduled_prefix = _compile(prep_ops)\n    circuit = cirq.Circuit(_schedule_commuting(unscheduled_prefix, trials=16))\n    circuit.append(cirq.Moment(cirq.measure(q,key=_key(q)) for q in odd))\n    corrections = []\n    correction_keys = {}\n    for old,dest in sorted(layout[\'destinations\'].items()):\n        if old != dest:\n            keys = [_key(old)]\n            corrections.append(cirq.Z(dest).with_classical_controls(_key(old)))\n            correction_keys[dest] = keys\n    keys = sorted(_key(q) for q in measured_active)\n    corrections.append(cirq.Z(layout[\'root\']).with_classical_controls(_parity(keys)))\n    correction_keys[layout[\'root\']] = keys\n    circuit.append(cirq.Moment(corrections))\n    unscheduled_circuit = cirq.FrozenCircuit(circuit)\n    # Full readout diagonals and feedforward are indivisible causal blocks;\n    # independent final data gates may share their moments.\n    circuit = _schedule_adaptive(circuit)\n    outputs = set(layout[\'destinations\'].values()) | {layout[\'root\']}\n    assert len(outputs) == n and all((q.row+q.col)%2 == 0 for q in outputs)\n    return dict(circuit=cirq.FrozenCircuit(circuit), outputs=frozenset(outputs),\n                core=cirq.FrozenCircuit(core), core_outputs=frozenset(core_outputs),\n                cat=cirq.FrozenCircuit(cat), suffix=cirq.FrozenCircuit(suffix_unitary),\n                unscheduled_prefix=cirq.FrozenCircuit(unscheduled_prefix),\n                unscheduled_circuit=unscheduled_circuit,\n                correction_keys=correction_keys, **layout)\n\n\ndef itertools_ops(*circuits):\n    for circuit in circuits:\n        yield from circuit.all_operations()\n\n\n\n\n# ROUND 4 VERIFIED NATIVE CIRCUITS START\n# Generated only from full complex128 statevector and physical-device acceptance.\n_ROUND4_VERIFIED = {(10, 3): {\'method\': \'binary count seed + moving six-CZ decoders\',\n           \'moments\': [[(0, 0.5075250292358409, 0.0, -0.5),\n                        (4, 0.5640582364224308, 0.0, -0.5),\n                        (1, 0.21521876042258375, 0.0, 0.5),\n                        (5, 0.40607625324767044, 0.0, 0.5),\n                        (7, 0.3268287043101393, 0.0, 0.5),\n                        (8, 0.538298819430637, 0.0, -0.5),\n                        (2, 0.4135610417387456, 0.0, -0.5),\n                        (3, 0.5, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (9, 0.5, 0.0, 0.5)],\n                       [(0, 4), (1, 5), (7, 8)],\n                       [(0, 0.4625211535683372, 0.0, 0.5),\n                        (1, 0.9604308353955838, 0.0, -0.5),\n                        (4, 0.03671905871867652, 0.0, 0.5),\n                        (7, 0.3128234467263022, 0.0, -0.5),\n                        (5, 0.02134966799353345, 0.0, 0.5),\n                        (8, 0.37131330816679287, 0.0, 0.5)],\n                       [(0, 1), (4, 7), (5, 8)],\n                       [(0, 0.5000000000000001, 0.0, -0.5),\n                        (1, 0.7834904832650229, 0.0, 0.5),\n                        (4, 1.0, 0.0, 0.5),\n                        (5, 0.2724041902391714, 0.0, -0.5),\n                        (7, 0.9144066141716028, 0.0, -0.5),\n                        (8, 0.5505652661300525, 0.0, -0.5)],\n                       [(0, 1), (4, 5), (7, 8)],\n                       [(0, 1.0, 0.0, -0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (1, 1.0, 0.0, -0.5),\n                        (5, 0.969751333521469, 0.0, 0.5),\n                        (7, 0.2007184326188422, 0.0, 0.5),\n                        (8, 0.2368582275889551, 0.0, 0.5)],\n                       [(0, 4), (1, 5), (7, 8)],\n                       [(0, 0.5548771086812294, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5),\n                        (4, 0.7379536328460687, 0.0, -0.5),\n                        (7, 0.7215353403057394, 0.0, 0.5),\n                        (5, 0.9900120526976681, 2.220446049250313e-16, -0.5000000000000002),\n                        (8, 0.892328487053231, 0.0, 0.5)],\n                       [(0, 1), (4, 7), (5, 8)],\n                       [(0, 0.30279448144254484, 0.0, 0.5),\n                        (4, 0.5229082281297596, 0.0, -0.5),\n                        (1, 0.49205017325274075, 0.0, -0.5),\n                        (5, 0.8763813388802872, 0.0, 0.5),\n                        (8, 0.810070564137761, 0.0, -0.5)],\n                       [(0, 4), (1, 5), (7, 8)],\n                       [(0, 0.4999999999999998, 0.0, 0.5),\n                        (1, 0.8644526864271973, 0.0, 0.5),\n                        (4, 0.5000000000000001, 0.0, -0.5),\n                        (7, 1.0, 0.0, 0.5),\n                        (5, 0.4338494543433774, 0.0, -0.5),\n                        (8, 0.6436424188213695, 0.0, 0.5)],\n                       [(0, 1), (4, 7), (5, 8)],\n                       [(1, 0.04081793246062402, 0.0, -0.5000000000000002),\n                        (4, 1.0, 0.0, -0.5),\n                        (5, 0.6299089726522493, 0.0, 0.5),\n                        (7, 1.0, 0.0, 0.5),\n                        (8, 0.6270991898233952, 0.0, 0.5)],\n                       [(0, 1), (4, 5), (7, 8)],\n                       [(1, 0.8436690440138603, 0.0, -0.5),\n                        (0, 0.9999999999999997, 0.0, -0.5),\n                        (5, 0.49696336328397284, 0.0, -0.5),\n                        (8, 0.047368593789405784, 0.0, -0.5),\n                        (7, 1.0, 0.0, -0.5)],\n                       [(1, 2), (5, 6), (8, 9)],\n                       [(1, 0.3819668494768557, 0.0, 0.5),\n                        (2, 0.6338602364006151, 0.0, -0.5),\n                        (5, 0.49999999999999994, 0.0, -0.5),\n                        (6, 0.3040867239846963, 0.0, -0.5),\n                        (8, 0.49999999999999994, 0.0, -0.5),\n                        (9, 0.3040867239846963, 0.0, -0.5)],\n                       [(0, 1), (4, 5), (7, 8)],\n                       [(1, 0.7376490383100945, 0.0, 0.5),\n                        (0, 1.0, 0.0, 0.5),\n                        (5, 0.8040867239846964, 0.0, 0.5),\n                        (4, 1.0, 0.0, 0.5),\n                        (8, 0.8040867239846964, 0.0, 0.5),\n                        (7, 1.0, 0.0, 0.5)],\n                       [(1, 2), (5, 6), (8, 9)],\n                       [(1, 0.16018463696253962, 0.0, 0.5),\n                        (2, 0.18023932987586394, 0.0, -0.5000000000000002),\n                        (5, 0.75, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (8, 0.75, 0.0, 0.5),\n                        (9, 0.5, 0.0, 0.5)],\n                       [(0, 1), (4, 5), (7, 8)],\n                       [(1, 0.4422317453870271, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (5, 0.5000000000000001, 0.0, -0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (8, 0.5000000000000001, 0.0, -0.5),\n                        (7, 0.5, 0.0, 0.5)],\n                       [(1, 2), (5, 6), (8, 9)],\n                       [(1, 0.6666666666666669, 0.0, 0.5),\n                        (2, 0.8040867239846964, 2.220446049250313e-16, -0.5000000000000002),\n                        (5, 0.25, 0.0, 0.5),\n                        (6, 1.0, 0.0, -0.5),\n                        (8, 0.25, 0.0, 0.5),\n                        (9, 1.0, 0.0, -0.5)],\n                       [(0, 1), (2, 3), (4, 5), (7, 8)],\n                       [(1, 1.0, 0.0, 0.5),\n                        (2, 0.49999999999999994, 0.0, -0.5),\n                        (3, 0.3040867239846963, 0.0, -0.5),\n                        (5, 0.5, 0.0, -0.5),\n                        (8, 0.5, 0.0, -0.5),\n                        (0, 0.5, 0.0, -0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (7, 0.5, 0.0, 0.5)],\n                       [(1, 2), (5, 6), (8, 9)],\n                       [(2, 0.8040867239846964, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5),\n                        (5, 0.5000000000000001, 0.0, 0.5),\n                        (8, 0.5000000000000001, 0.0, 0.5)],\n                       [(2, 3)],\n                       [(2, 0.75, 0.0, 0.5), (3, 0.5, 0.0, 0.5)],\n                       [(1, 2)],\n                       [(2, 0.5000000000000001, 0.0, -0.5), (1, 0.5, 0.0, 0.5)],\n                       [(2, 3)],\n                       [(2, 0.25, 0.0, 0.5), (3, 1.0, 0.0, -0.5)],\n                       [(1, 2)],\n                       [(2, 0.5, 0.0, -0.5), (1, 0.5, 0.0, 0.5)],\n                       [(2, 3)],\n                       [(2, 0.5000000000000001, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],\n           \'qubits\': [(3, 4), (3, 5), (3, 6), (3, 7), (4, 4), (4, 5), (4, 6), (5, 4), (5, 5), (5, 6)]},\n (10, 4): {\'method\': \'paired binary counts + recursive coherent decoders\',\n           \'moments\': [[(0, 0.576169350429717, 0.0, 0.5),\n                        (1, 0.24654597436413472, 0.0, -0.5),\n                        (2, 0.33300299859750804, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (7, 0.4999999999999998, 0.0, -0.5000000000000002),\n                        (5, 0.5, 1.0, -0.5),\n                        (8, 0.5, 0.0, -0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (9, 0.5, 0.0, 0.5)],\n                       [(0, 1), (3, 2), (6, 7)],\n                       [(1, 0.9274578370544355, 0.0, 0.5),\n                        (6, 0.5335957289217554, 0.0, -0.5),\n                        (2, 0.8657722407926058, 0.0, 0.5),\n                        (7, 0.03359572892175544, 0.0, -0.5000000000000002),\n                        (0, 0.8952958399493551, 1.0, 0.5),\n                        (3, 0.4999999999999998, 1.0, 0.5)],\n                       [(1, 6), (2, 7)],\n                       [(1, 0.4999999999999998, 0.0, -0.5000000000000002),\n                        (2, 0.5, 0.0, 0.5),\n                        (6, 0.44472707612688994, 0.0, 0.5),\n                        (7, 0.9447270761268902, 0.0, -0.5000000000000002)],\n                       [(1, 2), (6, 7)],\n                       [(0, 1), (3, 2), (6, 0.4999999999999998, 1.0, 0.5), (7, 0.5, 1.0, -0.5)],\n                       [(1, 0.7239752345042922, 0.0, 0.5),\n                        (2, 0.5894561875110336, 0.0, 0.5),\n                        (0, 0.5, 1.0, -0.5),\n                        (3, 0.5, 1.0, 0.5)],\n                       [(1, 6), (2, 7)],\n                       [(1, 0.8040867239846965, 5.551115123125783e-17, 0.49999999999999994),\n                        (2, 0.4105438124889665, 0.0, -0.5),\n                        (7, 0.1338602364006152, 1.0, 0.5)],\n                       [(1, 2), (6, 7)],\n                       [(1, 0.5, 0.0, 0.5), (2, 0.5, 0.0, 0.5), (7, 0.5, 1.0, 0.5)],\n                       [(0, 1), (3, 2)],\n                       [(1, 6), (2, 0.5000000000000003, 0.0, 0.5), (0, 0.5, 1.0, -0.5), (3, 0.5, 1.0, -0.5)],\n                       [(2, 7), (1, 0.5420140480044957, 0.0, -0.5), (6, 0.318767202525581, 0.0, -0.5)],\n                       [(0, 1),\n                        (6, 5),\n                        (7, 0.9526274389261962, 0.0, 0.5),\n                        (2, 0.9579859519955054, 2.220446049250313e-16, -0.5000000000000002)],\n                       [(7, 8), (3, 2), (1, 0.21639595347017165, 0.0, -0.5), (5, 0.0, 1.0, 0)],\n                       [(1, 6), (5, 0), (2, 0.21639595347017165, 0.0, 0.5), (8, 0.0, 1.0, 0)],\n                       [(2, 7),\n                        (8, 3),\n                        (0, 0.44250075386119647, 0.0, 0.5),\n                        (1, 0.8655339613566024, 0.0, -0.5000000000000002),\n                        (6, 0.5, 0.0, 0.5),\n                        (5, 0.5, 0.0, 0.5)],\n                       [(0, 1),\n                        (6, 5),\n                        (3, 0.44250075386119647, 0.0, 0.5),\n                        (2, 0.8655339613566024, 0.0, -0.5000000000000002),\n                        (7, 0.5, 0.0, -0.5),\n                        (8, 0.5, 1.0, 0.5)],\n                       [(3, 2),\n                        (7, 8),\n                        (1, 0.7362067632071183, 0.0, -0.5),\n                        (6, 0.5, 1.0, -0.5),\n                        (5, 0.020330766314526457, 1.0, 0.5),\n                        (0, 0.7628573843657716, 0.0, -0.5)],\n                       [(1, 6),\n                        (5, 0),\n                        (2, 0.7362067632071183, 0.0, -0.5),\n                        (7, 0.5, 0.0, 0.5),\n                        (8, 0.020330766314526457, 1.0, 0.5),\n                        (3, 0.7628573843657716, 0.0, -0.5)],\n                       [(2, 7),\n                        (8, 3),\n                        (0, 0.20741313285212096, 0.0, 0.5),\n                        (1, 0.8453403754074111, 0.0, -0.5),\n                        (5, 0.08250711583968151, 0.0, 0.5)],\n                       [(0, 1),\n                        (6, 5),\n                        (3, 0.20741313285212096, 0.0, 0.5),\n                        (2, 0.8453403754074111, 0.0, 0.5),\n                        (8, 0.08250711583968151, 1.0, 0.5)],\n                       [(3, 2),\n                        (7, 8),\n                        (1, 0.8626157748423866, 0.0, 0.5),\n                        (5, 0.8491679204856472, 1.0, 0.5),\n                        (0, 0.5846516053950286, 0.0, -0.5)],\n                       [(1, 6),\n                        (5, 0),\n                        (2, 0.8626157748423866, 0.0, -0.5),\n                        (7, 0.0, 1.0, 0),\n                        (8, 0.8491679204856472, 1.0, 0.5),\n                        (3, 0.5846516053950286, 0.0, -0.5)],\n                       [(2, 7),\n                        (8, 3),\n                        (0, 0.7281308634155641, 0.0, 0.5),\n                        (1, 0.15189628146893905, 0.0, 0.5),\n                        (5, 0.7775493667322889, 1.0, 0.49999999999999994)],\n                       [(0, 1),\n                        (6, 5),\n                        (3, 0.7281308634155641, 0.0, 0.5),\n                        (2, 0.15189628146893905, 0.0, 0.5),\n                        (8, 0.7775493667322889, 0.0, 0.5)],\n                       [(3, 2),\n                        (7, 8),\n                        (1, 0.17966697347952792, 0.0, 0.5),\n                        (5, 0.7487068121713458, 1.0, -0.5),\n                        (0, 0.04683006583111981, 4.440892098500626e-16, 0.49999999999999956)],\n                       [(1, 6),\n                        (5, 0),\n                        (2, 0.17966697347952792, 0.0, 0.5),\n                        (8, 0.7487068121713458, 1.0, -0.5),\n                        (3, 0.04683006583111981, 4.440892098500626e-16, 0.49999999999999956)],\n                       [(2, 7),\n                        (8, 3),\n                        (0, 0.2359340761074103, 0.0, -0.5),\n                        (1, 0.773538838069632, 0.0, 0.5),\n                        (5, 0.5556977655654309, 1.0, 0.5)],\n                       [(0, 1),\n                        (6, 5),\n                        (3, 0.2359340761074103, 0.0, -0.5),\n                        (2, 0.773538838069632, 0.0, -0.5000000000000002),\n                        (7, 0.0, 1.0, 0),\n                        (8, 0.5556977655654309, 0.0, 0.5)],\n                       [(3, 2),\n                        (7, 8),\n                        (1, 0.6370237375638905, 2.220446049250313e-16, -0.5000000000000002),\n                        (6, 0.0, 1.0, 0),\n                        (5, 0.01539729570142323, 1.0, -0.5000000000000002),\n                        (0, 0.4429438206750263, 0.0, 0.5)],\n                       [(1, 6),\n                        (5, 0),\n                        (2, 0.6370237375638905, 0.0, 0.5),\n                        (8, 0.01539729570142323, 1.0, -0.5000000000000002),\n                        (3, 0.4429438206750263, 0.0, 0.5)],\n                       [(2, 7),\n                        (8, 3),\n                        (0, 0.6131686327461048, 0.0, -0.5),\n                        (1, 0.7717884632990302, 0.0, 0.5),\n                        (5, 0.7676256178984648, 0.0, 0.5)],\n                       [(0, 1),\n                        (6, 5),\n                        (3, 0.6131686327461048, 0.0, -0.5),\n                        (2, 0.7717884632990302, 0.0, 0.5),\n                        (8, 0.7676256178984648, 1.0, 0.5)],\n                       [(3, 2),\n                        (7, 8),\n                        (1, 0.8699038911668677, 0.0, -0.5000000000000002),\n                        (6, 0.5, 1.0, -0.5),\n                        (5, 0.22313271290680284, 1.0, -0.5),\n                        (0, 0.14080285459112307, 0.0, 0.5)],\n                       [(1, 6),\n                        (5, 0),\n                        (2, 0.8699038911668677, 0.0, -0.5000000000000002),\n                        (7, 0.5, 0.0, 0.5),\n                        (8, 0.22313271290680284, 1.0, -0.5),\n                        (3, 0.14080285459112307, 0.0, 0.5)],\n                       [(2, 7),\n                        (8, 3),\n                        (0, 0.004810136259751463, 0.0, -0.4999999999999991),\n                        (1, 0.5, 0.0, -0.5),\n                        (5, 0.40834871822043467, 0.0, 0.5),\n                        (6, 0.5, 0.0, -0.5)],\n                       [(0, 1),\n                        (5, 4),\n                        (3, 0.004810136259751463, 0.0, -0.4999999999999991),\n                        (2, 0.5, 0.0, 0.5),\n                        (8, 0.40834871822043467, 0.0, 0.5),\n                        (7, 0.5, 0.0, -0.5)],\n                       [(3, 2),\n                        (8, 9),\n                        (0, 0.5, 0.0, -0.5),\n                        (5, 0.4999999999999998, 0.0, -0.5000000000000002),\n                        (4, 0.3040867239846962, 0.0, -0.5000000000000002)],\n                       [(1, 0),\n                        (6, 5),\n                        (3, 0.5, 0.0, -0.5),\n                        (8, 0.5, 1.0, -0.5),\n                        (9, 0.3040867239846962, 0.0, -0.5000000000000002)],\n                       [(2, 3),\n                        (7, 8),\n                        (1, 0.25000000000000006, 0.0, -0.5),\n                        (0, 0.25000000000000006, 0.0, -0.5),\n                        (5, 0.8040867239846965, 5.551115123125783e-17, 0.49999999999999994)],\n                       [(1, 0),\n                        (5, 4),\n                        (2, 0.25000000000000006, 0.0, 0.5),\n                        (3, 0.25000000000000006, 0.0, -0.5),\n                        (8, 0.8040867239846965, 5.551115123125783e-17, 0.49999999999999994)],\n                       [(2, 3),\n                        (8, 9),\n                        (5, 0.75, 1.0, 0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (0, 0.5, 0.0, -0.5),\n                        (1, 0.5, 1.0, 0.5)],\n                       [(6, 5),\n                        (8, 0.75, 0.0, 0.5),\n                        (9, 0.5, 0.0, 0.5),\n                        (2, 0.5, 0.0, -0.5),\n                        (3, 0.5, 0.0, -0.5)],\n                       [(7, 8), (5, 0.5, 0.0, -0.5), (6, 0.5, 0.0, 0.5)],\n                       [(5, 4), (8, 0.5, 0.0, -0.5), (7, 0.5, 0.0, 0.5)],\n                       [(8, 9), (5, 0.25000000000000006, 1.0, 0.5)],\n                       [(6, 5), (8, 0.25000000000000006, 0.0, 0.5)],\n                       [(7, 8), (5, 0.5, 1.0, -0.5), (6, 0.5, 0.0, -0.5)],\n                       [(5, 4), (8, 0.5, 1.0, -0.5), (7, 0.5, 4.440892098500626e-16, 0.5)],\n                       [(8, 9), (4, 1.0, 0.0, 0.5), (5, 0.5, 0.0, 0.5)],\n                       [(8, 0.5, 0.0, 0.5), (9, 1.0, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],\n           \'qubits\': [(4, 4), (4, 5), (4, 6), (4, 7), (5, 3), (5, 4), (5, 5), (5, 6), (5, 7), (5, 8)]},\n (11, 3): {\'method\': \'binary count seed + moving six-CZ decoders\',\n           \'moments\': [[(0, 0.4977095531171784, 0.0, -0.5),\n                        (4, 0.47999676525351215, 0.0, -0.5),\n                        (1, 0.13289535826303486, 0.0, 0.5),\n                        (5, 0.30529911596742315, 0.0, 0.5),\n                        (7, 0.31211681879788, 0.0, 0.5),\n                        (8, 0.46080081683211016, 0.0, -0.5),\n                        (2, 0.4135610417387456, 0.0, -0.5),\n                        (3, 0.5, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (9, 0.4135610417387456, 0.0, -0.5),\n                        (10, 0.5, 0.0, 0.5)],\n                       [(0, 4), (1, 5), (7, 8)],\n                       [(0, 0.4636860383182841, 0.0, 0.5),\n                        (1, 0.8349015140950895, 0.0, -0.5),\n                        (4, 0.03624196952874488, 0.0, 0.5),\n                        (7, 0.2137341063327851, 0.0, -0.5),\n                        (5, 0.22378717952558877, 0.0, 0.5),\n                        (8, 0.4933349703475507, 0.0, 0.5)],\n                       [(0, 1), (4, 7), (5, 8)],\n                       [(0, 0.5000000000000001, 0.0, -0.5),\n                        (1, 0.7311025414991926, 0.0, 0.5),\n                        (4, 1.0, 0.0, 0.5),\n                        (5, 0.32175092945542666, 0.0, -0.5),\n                        (7, 0.9241027775512922, 0.0, -0.5),\n                        (8, 0.4344976034821523, 0.0, -0.5)],\n                       [(0, 1), (4, 5), (7, 8)],\n                       [(0, 1.0, 0.0, -0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (1, 1.0, 0.0, -0.5),\n                        (5, 0.9674838768770302, 2.220446049250313e-16, -0.5000000000000002),\n                        (7, 0.3185254465017618, 0.0, 0.5),\n                        (8, 0.17074045794894652, 0.0, 0.5)],\n                       [(0, 4), (1, 5), (7, 8)],\n                       [(0, 0.5881898098972081, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5),\n                        (4, 0.7631294770516314, 0.0, -0.5),\n                        (7, 0.6932783553567452, 0.0, 0.5),\n                        (5, 0.9641022893365128, 0.0, 0.5),\n                        (8, 0.7708477377925105, 0.0, 0.5)],\n                       [(0, 1), (4, 7), (5, 8)],\n                       [(0, 0.48176156561702044, 0.0, 0.5),\n                        (4, 0.5996540067178372, 0.0, -0.5),\n                        (1, 0.5044593869341634, 0.0, -0.5),\n                        (5, 0.7543939581801827, 0.0, 0.5),\n                        (8, 0.9081527187719957, 0.0, -0.5)],\n                       [(0, 4), (1, 5), (7, 8)],\n                       [(0, 0.49999999999999994, 0.0, 0.5),\n                        (1, 0.9975529633914461, 0.0, 0.5),\n                        (4, 0.5, 0.0, -0.5),\n                        (7, 1.0, 0.0, 0.5),\n                        (5, 0.526039179709408, 0.0, -0.5),\n                        (8, 0.5470533185096936, 0.0, 0.5)],\n                       [(0, 1), (4, 7), (5, 8)],\n                       [(1, 0.2352186315018946, 0.0, -0.5),\n                        (4, 1.0, 0.0, -0.5),\n                        (5, 0.5728585954068288, 0.0, 0.5),\n                        (7, 1.0, 0.0, 0.5),\n                        (8, 0.7360395305448575, 0.0, 0.5)],\n                       [(0, 1), (4, 5), (7, 8)],\n                       [(1, 0.9934520461966211, 0.0, -0.5),\n                        (0, 1.0, 0.0, 0.5),\n                        (5, 0.4174153969225333, 0.0, -0.5),\n                        (8, 0.24825141824218944, 0.0, -0.5),\n                        (7, 1.0, 0.0, -0.5)],\n                       [(1, 2), (5, 6), (8, 9)],\n                       [(1, 0.3819668494768557, 0.0, 0.5),\n                        (2, 0.6338602364006151, 0.0, -0.5),\n                        (5, 0.49999999999999994, 0.0, -0.5),\n                        (6, 0.3040867239846963, 0.0, -0.5),\n                        (8, 0.3819668494768557, 0.0, 0.5),\n                        (9, 0.6338602364006151, 0.0, -0.5)],\n                       [(0, 1), (4, 5), (7, 8)],\n                       [(1, 0.7376490383100945, 0.0, 0.5),\n                        (0, 1.0, 0.0, 0.5),\n                        (5, 0.8040867239846964, 0.0, 0.5),\n                        (4, 1.0, 0.0, 0.5),\n                        (8, 0.7376490383100945, 0.0, 0.5),\n                        (7, 1.0, 0.0, 0.5)],\n                       [(1, 2), (5, 6), (8, 9)],\n                       [(1, 0.16018463696253962, 0.0, 0.5),\n                        (2, 0.18023932987586394, 0.0, -0.5000000000000002),\n                        (5, 0.75, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (8, 0.16018463696253962, 0.0, 0.5),\n                        (9, 0.18023932987586394, 0.0, -0.5000000000000002)],\n                       [(0, 1), (4, 5), (7, 8)],\n                       [(1, 0.4422317453870271, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (5, 0.5000000000000001, 0.0, -0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (8, 0.4422317453870271, 0.0, 0.5),\n                        (7, 0.5, 0.0, 0.5)],\n                       [(1, 2), (5, 6), (8, 9)],\n                       [(1, 0.6666666666666669, 0.0, 0.5),\n                        (2, 0.8040867239846964, 2.220446049250313e-16, -0.5000000000000002),\n                        (5, 0.25, 0.0, 0.5),\n                        (6, 1.0, 0.0, -0.5),\n                        (8, 0.6666666666666669, 0.0, 0.5),\n                        (9, 0.8040867239846964, 2.220446049250313e-16, -0.5000000000000002)],\n                       [(0, 1), (2, 3), (4, 5), (7, 8), (9, 10)],\n                       [(1, 1.0, 0.0, 0.5),\n                        (2, 0.49999999999999994, 0.0, -0.5),\n                        (3, 0.3040867239846963, 0.0, -0.5),\n                        (5, 0.5, 0.0, -0.5),\n                        (8, 1.0, 0.0, 0.5),\n                        (9, 0.49999999999999994, 0.0, -0.5),\n                        (10, 0.3040867239846963, 0.0, -0.5),\n                        (0, 0.5, 0.0, -0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (7, 0.5, 0.0, -0.5)],\n                       [(1, 2), (5, 6), (8, 9)],\n                       [(2, 0.8040867239846964, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5),\n                        (9, 0.8040867239846964, 0.0, 0.5),\n                        (8, 1.0, 0.0, 0.5),\n                        (5, 0.5000000000000001, 0.0, 0.5)],\n                       [(2, 3), (9, 10)],\n                       [(2, 0.75, 0.0, 0.5), (3, 0.5, 0.0, 0.5), (9, 0.75, 0.0, 0.5), (10, 0.5, 0.0, 0.5)],\n                       [(1, 2), (8, 9)],\n                       [(2, 0.5000000000000001, 0.0, -0.5),\n                        (1, 0.5, 0.0, 0.5),\n                        (9, 0.5000000000000001, 0.0, -0.5),\n                        (8, 0.5, 0.0, 0.5)],\n                       [(2, 3), (9, 10)],\n                       [(2, 0.25, 0.0, 0.5), (3, 1.0, 0.0, -0.5), (9, 0.25, 0.0, 0.5), (10, 1.0, 0.0, -0.5)],\n                       [(1, 2), (8, 9)],\n                       [(2, 0.5, 0.0, -0.5), (9, 0.5, 0.0, -0.5), (1, 0.5, 0.0, 0.5), (8, 0.5, 0.0, 0.5)],\n                       [(2, 3), (9, 10)],\n                       [(2, 0.5000000000000001, 0.0, 0.5), (9, 0.5000000000000001, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (3, 6),\n                      (3, 7),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7)]},\n (11, 4): {\'method\': \'paired binary counts + recursive coherent decoders\',\n           \'moments\': [[(1, 0.6804183583624992, 0.0, 0.5),\n                        (2, 0.2715149105420544, 0.0, -0.5000000000000002),\n                        (3, 0.3910132305016676, 0.0, 0.5),\n                        (7, 0.5, 0.0, 0.5),\n                        (8, 0.4999999999999998, 0.0, -0.5000000000000002),\n                        (6, 0.5, 1.0, -0.5),\n                        (9, 0.23850197504578502, 0.0, 0.6723366713104197),\n                        (5, 0.5, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (10, 0.5, 0.0, 0.5)],\n                       [(1, 2), (4, 3), (7, 8)],\n                       [(2, 0.8245125290280035, 0.0, 0.5),\n                        (7, 0.3399162669016432, 0.0, -0.5000000000000002),\n                        (3, 0.7367750224499274, 0.0, 0.5),\n                        (8, 0.16008373309835672, 0.0, 0.5),\n                        (1, 0.7552309029870035, 1.0, 0.5),\n                        (4, 0.5, 1.0, 0.5)],\n                       [(2, 7), (3, 8)],\n                       [(2, 0.5, 0.0, -0.5),\n                        (3, 0.5, 0.0, 0.5),\n                        (7, 0.45750956125153186, 0.0, 0.5),\n                        (8, 0.9575095612515319, 0.0, -0.5000000000000002)],\n                       [(2, 3), (7, 8)],\n                       [(1, 2), (4, 3), (7, 0.5, 1.0, 0.5), (8, 0.5, 1.0, -0.5)],\n                       [(2, 0.7356809418135472, 0.0, 0.5),\n                        (3, 0.6015434872071732, 0.0, 0.5),\n                        (1, 0.5, 1.0, -0.5),\n                        (4, 0.5, 1.0, 0.5)],\n                       [(2, 7), (3, 8)],\n                       [(2, 0.7820471084244873, 0.0, 0.5),\n                        (3, 0.3984565127928268, 0.0, -0.5),\n                        (8, 0.17490085614052142, 1.0, 0.5)],\n                       [(2, 3), (7, 8)],\n                       [(2, 0.5, 0.0, 0.5), (3, 0.5, 0.0, 0.5), (8, 0.5, 1.0, 0.5)],\n                       [(1, 2), (4, 3)],\n                       [(2, 7), (3, 0.4999999999999998, 0.0, 0.5), (1, 0.5, 1.0, -0.5), (4, 0.5, 1.0, -0.5)],\n                       [(3, 8), (2, 0.5420140480044957, 0.0, -0.5), (7, 0.318767202525581, 0.0, -0.5)],\n                       [(1, 2),\n                        (7, 6),\n                        (8, 0.958834343779025, 0.0, -0.014901640704749752),\n                        (3, 0.8595522677647842, 0.0, -0.693699434815511)],\n                       [(8, 9), (4, 3), (2, 0.21639595347017165, 0.0, -0.5), (6, 0.0, 1.0, 0)],\n                       [(2, 7),\n                        (6, 1),\n                        (3, 0.9549431959582095, 0.0, 0.565089921313934),\n                        (8, 0.8439229341407215, 1.0, -0.96924798035922),\n                        (9, 0.5683578105795405, 0.0, -0.878625289122293),\n                        (4, 0.0, 1.0, 0)],\n                       [(3, 8),\n                        (1, 0.44250075386119647, 0.0, 0.5),\n                        (2, 0.8655339613566024, 0.0, -0.5000000000000002),\n                        (7, 0.5, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5)],\n                       [(1, 2),\n                        (7, 6),\n                        (8, 0.6798126636824355, 0.0, -0.3508043320382481),\n                        (3, 0.3636111207350248, 0.0, -0.7302394438090745)],\n                       [(8, 9),\n                        (4, 3),\n                        (2, 0.7362067632071183, 0.0, -0.5),\n                        (7, 0.5, 1.0, -0.5),\n                        (6, 0.020330766314526457, 1.0, 0.5),\n                        (1, 0.7628573843657716, 0.0, -0.5)],\n                       [(2, 7),\n                        (6, 1),\n                        (3, 0.5747492210962788, 5.551115123125783e-17, 0.3581345590961306),\n                        (8, 0.4197402158362804, 1.0, -0.03709055302262865),\n                        (9, 0.42068674081551494, 0.0, -0.8412061211954147),\n                        (4, 2.2204460492503128e-16, 1.0, -0.36923311612343257)],\n                       [(3, 8),\n                        (1, 0.20741313285212096, 0.0, 0.5),\n                        (2, 0.8453403754074111, 0.0, -0.5),\n                        (6, 0.08250711583968151, 0.0, 0.5)],\n                       [(1, 2),\n                        (7, 6),\n                        (8, 0.3406064361562491, 0.0, 0.9776918147274036),\n                        (3, 0.05618598641592863, 0.0, -0.8318915358055705)],\n                       [(8, 9),\n                        (4, 3),\n                        (2, 0.8626157748423866, 0.0, 0.5),\n                        (6, 0.8491679204856472, 1.0, 0.5),\n                        (1, 0.5846516053950286, 0.0, -0.5)],\n                       [(2, 7),\n                        (6, 1),\n                        (3, 0.5474162491089649, 0.0, 0.9234947857011742),\n                        (8, 0.185987156616219, 1.0, 0.004371383477020652),\n                        (9, 0.46193265931565103, 1.1102230246251565e-16, -0.24595652224492492),\n                        (4, 0.0, 1.0, 0)],\n                       [(3, 8),\n                        (1, 0.7281308634155641, 0.0, 0.5),\n                        (2, 0.15189628146893905, 0.0, 0.5),\n                        (6, 0.7775493667322889, 1.0, 0.49999999999999994)],\n                       [(1, 2),\n                        (7, 6),\n                        (8, 0.30175441391238306, 0.0, 0.676222190721804),\n                        (3, 0.7392621381122315, 0.0, -0.2845905710342014)],\n                       [(8, 9),\n                        (4, 3),\n                        (2, 0.17966697347952792, 0.0, 0.5),\n                        (6, 0.7487068121713458, 1.0, -0.5),\n                        (1, 0.04683006583111981, 4.440892098500626e-16, 0.49999999999999956)],\n                       [(2, 7),\n                        (6, 1),\n                        (3, 0.27259688752465094, 0.0, 0.26026183666291547),\n                        (8, 0.053622665477313765, 1.0, -0.2870813295451675),\n                        (9, 0.8029611387137003, 0.0, 0.16269843839489745),\n                        (4, 0.0, 1.0, 0)],\n                       [(3, 8),\n                        (1, 0.2359340761074103, 0.0, -0.5),\n                        (2, 0.773538838069632, 0.0, 0.5),\n                        (6, 0.5556977655654309, 1.0, 0.5)],\n                       [(1, 2),\n                        (7, 6),\n                        (8, 0.6852836122352371, 0.0, 0.9412962844644133),\n                        (3, 0.5582935812841525, 0.0, 0.9390205235244355)],\n                       [(8, 9),\n                        (4, 3),\n                        (2, 0.6370237375638905, 2.220446049250313e-16, -0.5000000000000002),\n                        (7, 0.0, 1.0, 0),\n                        (6, 0.01539729570142323, 1.0, -0.5000000000000002),\n                        (1, 0.4429438206750263, 0.0, 0.5)],\n                       [(2, 7),\n                        (6, 1),\n                        (3, 0.8681490545286747, 0.0, -0.11410887725068442),\n                        (8, 0.4277380566342639, 1.0, -0.798012049875509),\n                        (9, 0.881708884870647, 0.0, -0.12980371729230367),\n                        (4, 0.5, 1.0, -0.9281148643661068)],\n                       [(3, 8),\n                        (1, 0.6131686327461048, 0.0, -0.5),\n                        (2, 0.7717884632990302, 0.0, 0.5),\n                        (6, 0.7676256178984648, 0.0, 0.5)],\n                       [(1, 2),\n                        (7, 6),\n                        (8, 0.11927728230983443, 0.0, -0.9375586526131592),\n                        (3, 0.6515066280755871, 8.881784197001252e-16, -0.9708832139278183)],\n                       [(8, 9),\n                        (4, 3),\n                        (2, 0.8699038911668677, 0.0, -0.5000000000000002),\n                        (7, 0.5, 1.0, -0.5),\n                        (6, 0.22313271290680284, 1.0, -0.5),\n                        (1, 0.14080285459112307, 0.0, 0.5)],\n                       [(2, 7),\n                        (6, 1),\n                        (3, 0.0, -0.6195272304439268, 0),\n                        (8, 0.7621154011768825, -2.220446049250313e-16, -0.3319937117990528),\n                        (4, 0.5000000000000001, 8.881784197001252e-16, -0.9281148643661046),\n                        (9, 0.6174170730099287, 1.1102230246251565e-16, -0.20873821911732637)],\n                       [(3, 8),\n                        (1, 0.004810136259751463, 0.0, -0.4999999999999991),\n                        (2, 0.5, 0.0, -0.5),\n                        (6, 0.40834871822043467, 0.0, 0.5),\n                        (7, 0.5, 0.0, -0.5)],\n                       [(1, 2),\n                        (6, 5),\n                        (3, 0.5, 0.0, 0.3648404284027391),\n                        (8, 0.7571708757236936, 0.0, -0.582013698707923)],\n                       [(4, 3),\n                        (8, 9),\n                        (1, 0.5, 0.0, -0.5),\n                        (6, 0.4999999999999998, 0.0, -0.5000000000000002),\n                        (5, 0.3040867239846962, 0.0, -0.5000000000000002)],\n                       [(2, 1),\n                        (7, 6),\n                        (3, 0.041748308476002194, 0.0, -0.13515957159726089),\n                        (8, 0.5323582538647059, 0.0, 0.6622106619575865),\n                        (4, 0.6218015137737606, 0.0, -0.7153710350390152),\n                        (9, 0.24679677437537367, 0.0, -0.7565306828698788)],\n                       [(3, 8),\n                        (4, 0),\n                        (9, 10),\n                        (2, 0.25000000000000006, 0.0, -0.5),\n                        (1, 0.25000000000000006, 0.0, -0.5),\n                        (6, 0.8040867239846965, 5.551115123125783e-17, 0.49999999999999994)],\n                       [(2, 1),\n                        (6, 5),\n                        (3, 0.5, 0.0, 0.3648404284027391),\n                        (4, 0.4999999999999998, 0.0, -0.35127986527513944),\n                        (9, 0.5, 0.0, -0.8523760134876583),\n                        (0, 0.3040867239846962, 0.0, -0.5000000000000002),\n                        (10, 0.3040867239846962, 0.0, -0.5000000000000002)],\n                       [(3, 4),\n                        (8, 9),\n                        (6, 0.75, 1.0, 0.5),\n                        (5, 0.5, 0.0, 0.5),\n                        (1, 0.5, 0.0, -0.5),\n                        (2, 0.5, 1.0, 0.5)],\n                       [(7, 6),\n                        (4, 0.8040867239846965, 0.0, 0.6487201347248606),\n                        (9, 0.8040867239846965, 5.551115123125783e-17, 0.14762398651234138)],\n                       [(4, 0), (9, 10), (6, 0.5, 0.0, -0.5), (7, 0.5, 0.0, 0.5)],\n                       [(6, 5),\n                        (4, 0.7499999999999999, 1.0, 0.6487201347248606),\n                        (9, 0.7499999999999999, 1.0, 0.14762398651234143),\n                        (0, 0.5, 0.0, 0.5),\n                        (10, 0.5, 0.0, 0.5)],\n                       [(3, 4), (8, 9), (6, 0.25000000000000006, 1.0, 0.5)],\n                       [(7, 6),\n                        (4, 0.5, -2.220446049250313e-16, -0.3512798652751392),\n                        (9, 0.5, 0.0, -0.8523760134876583),\n                        (3, 0.5, 0.0, 0.9572426028315615),\n                        (8, 0.5, 0.0, -0.9584612760937317)],\n                       [(4, 0), (9, 10), (6, 0.5, 1.0, -0.5), (7, 0.5, 0.0, -0.5)],\n                       [(6, 5),\n                        (4, 0.25, 1.0, 0.6487201347248606),\n                        (9, 0.25000000000000006, 1.0, 0.14762398651234143)],\n                       [(3, 4), (8, 9), (5, 1.0, 0.0, 0.5), (6, 0.5, 0.0, 0.5)],\n                       [(4, 0.5, 1.0, -0.35127986527513944),\n                        (9, 0.5, 1.0, -0.8523760134876583),\n                        (3, 0.5, -0.45724260283156104, -0.04275739716843874),\n                        (8, 0.5, -0.5415387239062683, 0.041538723906268504)],\n                       [(4, 0), (9, 10)],\n                       [(0, 1.0, 0.0, 0.5),\n                        (4, 0.5, -0.14872013472486056, 0.6487201347248606),\n                        (9, 0.5, 0.35237601348765857, 0.14762398651234143),\n                        (10, 1.0, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],\n           \'qubits\': [(3, 7),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 3),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8)]},\n (11, 5): {\'method\': \'paired binary counts + recursive coherent decoders\',\n           \'moments\': [[(1, 0.4907018809473783, 0.0, 0.5),\n                        (2, 0.5550422610798844, 0.0, -0.5),\n                        (4, 0.5008134412882038, 0.0, -0.5),\n                        (3, 0.4930157801733349, 0.0, -0.5),\n                        (7, 1.0, 0.0, -0.5),\n                        (8, 0.7705976301927333, 0.0, -0.5),\n                        (6, 0.9110494354557895, 0.0, 0.5),\n                        (5, 0.5, 0.0, 0.5),\n                        (9, 0.4652411656670723, 0.25343700566029476, -0.020153833182791514),\n                        (0, 0.5, 0.0, 0.5),\n                        (10, 0.5, 0.0, 0.5)],\n                       [(1, 2), (4, 3), (7, 8)],\n                       [(2, 0.9472614134659207, 0.0, 0.5),\n                        (7, 0.42521108825630477, 0.0, -0.5),\n                        (3, 0.37419172055761374, 0.0, 0.5),\n                        (8, 0.2294023698072668, 0.0, 0.5),\n                        (1, 0.9464554303254016, 0.0, 0.5),\n                        (4, 0.39921335209975817, 0.0, -0.5)],\n                       [(2, 7), (3, 8)],\n                       [(2, 0.5, 0.0, -0.5),\n                        (3, 0.47495427477567476, 0.0, 0.5),\n                        (7, 0.5801769884442538, 0.0, 0.5),\n                        (8, 1.0, 0.0, -0.5)],\n                       [(2, 3), (7, 8)],\n                       [(2, 0.49999999999999994, 0.0, -0.5),\n                        (3, 1.0, 0.0, 0.5),\n                        (7, 0.9422205038637325, 2.220446049250313e-16, -0.5000000000000002),\n                        (8, 1.0, 0.0, -0.5)],\n                       [(1, 2), (4, 3)],\n                       [(2, 0.7421439232860106, 0.0, 0.5),\n                        (3, 0.29667120537234515, 0.0, 0.5),\n                        (1, 0.5000000000000001, 0.0, -0.5),\n                        (4, 0.001401470241944226, 0.0, 0.5)],\n                       [(2, 7), (3, 8)],\n                       [(2, 0.3558842447906469, 0.0, -0.5),\n                        (3, 0.7966712053723455, 0.0, 0.5),\n                        (7, 0.5192241398420848, 0.0, -0.5),\n                        (8, 0.5, 0.0, 0.5)],\n                       [(2, 3), (7, 8)],\n                       [(2, 0.5000000000000001, 0.0, 0.5),\n                        (3, 0.5, 0.0, 0.5),\n                        (7, 0.5, 0.0, -0.5),\n                        (8, 0.5000000000000001, 0.0, 0.5)],\n                       [(1, 2), (4, 3)],\n                       [(2, 0.5000000000000001, 0.0, 0.5),\n                        (3, 8),\n                        (1, 0.07630690887472924, 0.0, 0.5),\n                        (4, 5.991769793960435e-17, 0.3490145657789867, 0.6380694274027181)],\n                       [(2, 7),\n                        (3, 0.5000000000000003, -0.43425977376051383, -0.7167835017790345),\n                        (8, 0.49525143929027055, -0.10624679471070642, -0.5185288144886113)],\n                       [(2, 0.16036638351148846, 0.0, 0.5),\n                        (7, 0.7792024120153752, 0.0, -0.5),\n                        (4, 3),\n                        (8, 9)],\n                       [(1, 2),\n                        (7, 6),\n                        (3, 1.0, 0.0, -0.6232499615788845),\n                        (8, 0.6205477744107245, -0.4346056454709941, 0.8578762318707984),\n                        (4, 1.2494398795541513e-16, 0.5088739665680895, -0.29960421858491126),\n                        (9, 0.5, -0.856181737698472, -0.2667168275224967)],\n                       [(2, 0.6909473652914381, 0.0, -0.5),\n                        (7, 0.1613403886518739, 0.0, -0.5000000000000002),\n                        (6, 0.47661372723901413, 0.0, -0.5),\n                        (1, 0.1946254454100686, 0.0, 0.5),\n                        (3, 8)],\n                       [(2, 7),\n                        (6, 1),\n                        (3, 0.7299176310680837, -0.3050481039924364, -0.5182595887762778),\n                        (8, 0.5, -0.6658378647241441, 0.9418578425861992)],\n                       [(1, 0.5151931900534683, 0.0, 0.5),\n                        (2, 0.27247647704799166, 0.0, -0.5),\n                        (7, 0.2990888634227144, 0.0, -0.5),\n                        (6, 0.7078007758797716, 0.0, 0.5),\n                        (4, 3),\n                        (8, 9)],\n                       [(1, 2),\n                        (7, 6),\n                        (3, 0.5708700815399964, 0.26118898246468064, -0.1126183862557657),\n                        (8, 0.1305686818715648, -0.03124526675740369, -0.9628239507997705),\n                        (4, 0.5, -0.8045387544356146, 0.5040898256465949),\n                        (9, 0.4479507936854727, 0.08888756127965092, -0.2593046523693947)],\n                       [(2, 0.779877236337489, 0.0, 0.5),\n                        (7, 0.5434242741585822, 0.0, 0.5),\n                        (6, 0.593356271964612, 0.0, 0.5),\n                        (1, 0.9992939137354855, 0.0, -0.5),\n                        (3, 8)],\n                       [(2, 7),\n                        (6, 1),\n                        (3, 0.35440396373816807, 0.34038417172181856, -0.5500095398253386),\n                        (8, 0.12328076625869655, -0.16201040862540284, -0.24145169939543676)],\n                       [(1, 0.4783976744291458, 0.0, -0.5),\n                        (2, 0.18496049343470256, 0.0, -0.5000000000000002),\n                        (7, 0.1727949182206442, 0.0, -0.5000000000000002),\n                        (6, 0.6547361219059727, 0.0, 0.5),\n                        (4, 3),\n                        (8, 9)],\n                       [(1, 2),\n                        (7, 6),\n                        (3, 1.0, 0.0, -0.07161798016617316),\n                        (8, 0.3127752765816518, -0.3519819185096116, -0.9034621080208396),\n                        (4, 0.5, 0.9642984800108094, 0.6995510712109803),\n                        (9, 0.23061776402485326, -0.8837906797874904, -0.9291499272477464)],\n                       [(2, 0.2679946667017772, 0.0, 0.5),\n                        (7, 0.8573378967739099, 2.220446049250313e-16, -0.5000000000000002),\n                        (6, 0.3513992031752148, 0.0, -0.5),\n                        (1, 0.294791916986395, 0.0, 0.5),\n                        (3, 8)],\n                       [(2, 7),\n                        (6, 1),\n                        (3, 0.5000000000000001, -0.942055553389656, -0.6299694009536578),\n                        (8, 0.7520450774165834, 0.3701270555753081, -0.7554440265304516)],\n                       [(1, 0.32984723036136065, 0.0, -0.5),\n                        (2, 0.7954980125962142, 0.0, -0.5),\n                        (7, 0.20771309087310677, 0.0, -0.5),\n                        (6, 0.8566361096480111, 0.0, -0.5),\n                        (4, 3),\n                        (8, 9)],\n                       [(1, 2),\n                        (7, 6),\n                        (3, 0.5000000000000001, -0.7732025334068713, 0.42797504565668615),\n                        (8, 0.22588638999651064, -0.6398236112927398, 0.11468302904485689),\n                        (4, 0.6321947838533056, -0.23105582111469314, -0.20861488773052983),\n                        (9, 0.3922258392328339, 0.8041885989453146, 0.5524219776258632)],\n                       [(2, 0.10037764173127965, 0.0, -0.5000000000000002),\n                        (7, 0.5046746026666172, 0.0, -0.5),\n                        (6, 0.5613608170725523, 0.0, 0.5),\n                        (1, 0.5036872975675825, 0.0, 0.5),\n                        (3, 8)],\n                       [(2, 7),\n                        (6, 1),\n                        (3, 2.2424398256382823e-17, -0.7755362444289533, -0.8500384573380648),\n                        (8, 0.696715986124177, 0.6526700604597154, -0.5251405822478832)],\n                       [(1, 0.3386567848293327, 0.0, 0.5),\n                        (2, 0.4847166034965695, 0.0, -0.5),\n                        (7, 0.023419071447328108, 0.0, 0.5),\n                        (6, 0.9247297487704056, 0.0, 0.5),\n                        (4, 3),\n                        (8, 9)],\n                       [(1, 2),\n                        (7, 6),\n                        (3, 1.0, 0.0, 0.18571993763011035),\n                        (8, 0.583609737771011, 0.4134418537745024, 0.6275294782118321),\n                        (4, 0.6771139237982055, 0.30216423803247405, -0.8514322105251233),\n                        (9, 0.6050120054676126, -0.15962079999151868, -0.12944477520904973)],\n                       [(2, 0.331132663133044, 0.0, -0.5),\n                        (7, 0.5511011783263742, 0.0, 0.5),\n                        (6, 0.6060273733042101, 0.0, 0.5),\n                        (1, 0.02615359895253171, 0.0, 0.5),\n                        (3, 8)],\n                       [(2, 7),\n                        (6, 1),\n                        (3, 0.5, 0.09173623835416944, -0.2746456563519435),\n                        (8, 0.06412882081148266, -0.6727975586871568, 0.04097133198633657)],\n                       [(1, 0.41378349954053234, 0.0, -0.5),\n                        (2, 0.21420722619256916, 0.0, -0.5),\n                        (7, 0.6548079112547879, 0.0, 0.5),\n                        (6, 0.04799438820222998, 0.0, -0.5000000000000002),\n                        (4, 3),\n                        (8, 9)],\n                       [(1, 2),\n                        (7, 6),\n                        (3, 0.5, -0.8513483079611661, -0.18290941799777416),\n                        (8, 0.49999999999999956, -0.5956415375973685, -0.6318262267008221),\n                        (4, 0.9999999999999999, 1.0, -0.47542901700248974),\n                        (9, 0.6668726728064026, -0.9393404410203794, -0.9846423446928667)],\n                       [(2, 0.02748200107701255, 0.0, 0.5),\n                        (7, 0.8744494561950673, 0.0, -0.5),\n                        (6, 0.9589738619662713, 0.0, -0.5),\n                        (1, 0.6921765407203413, 0.0, 0.5),\n                        (3, 8)],\n                       [(2, 7),\n                        (6, 1),\n                        (3, 0.5, 0.06464945840902381, -0.5847231978914305),\n                        (8, 0.4842223867225731, 0.2838310093130295, -0.6765883601291904)],\n                       [(1, 0.508361350659906, 0.0, -0.5),\n                        (2, 0.4261437927486822, 0.0, -0.5),\n                        (7, 0.823383043981599, 2.220446049250313e-16, -0.5000000000000002),\n                        (6, 0.19113236200572387, 0.0, 0.5),\n                        (4, 3),\n                        (8, 9)],\n                       [(1, 2),\n                        (7, 6),\n                        (3, 0.2714423079803394, 0.6073005506143939, -0.02007373948240687),\n                        (8, 0.5, 0.7440799782712231, -0.8517520477653158),\n                        (4, 0.7832164982209657, 0.34659116286393377, -0.34659116286393377),\n                        (9, 0.7832164982209655, 0.1774493007693972, -0.17744930076939713)],\n                       [(2, 0.4879960926011882, 0.0, -0.5),\n                        (7, 0.4892315910172894, 0.0, -0.5),\n                        (6, 0.46301818768630826, 0.0, 0.5),\n                        (1, 0.5593748655417544, 0.0, -0.5),\n                        (3, 8),\n                        (4, 0),\n                        (9, 10)],\n                       [(2, 7),\n                        (6, 1),\n                        (3, 0.49999999999999994, -0.4456185620215045, 0.08722681113198716),\n                        (4, 0.49999999999999994, 0.0, -0.5),\n                        (0, 0.3040867239846963, 0.0, -0.5),\n                        (8, 6.648603149363958e-17, 0.24928031860457545, 0.22900049579505016),\n                        (9, 0.49999999999999994, 0.0, -0.5),\n                        (10, 0.3040867239846963, 0.0, -0.5)],\n                       [(1, 0.5985688464425616, 0.0, -0.5),\n                        (2, 0.06716492138222008, 0.0, -0.5000000000000002),\n                        (7, 0.44772119461374327, 0.0, -0.5),\n                        (6, 0.655585885549357, 0.0, 0.5),\n                        (3, 4),\n                        (8, 9)],\n                       [(1, 2),\n                        (7, 6),\n                        (4, 0.8040867239846964, 0.0, 0.5),\n                        (3, 1.0, 0.0, 0.5),\n                        (9, 0.8040867239846964, 0.0, 0.5),\n                        (8, 1.0, 0.0, 0.5)],\n                       [(2, 0.4314290570480122, 0.0, 0.5),\n                        (7, 0.630801946857221, 0.0, -0.5),\n                        (6, 0.13991873508696726, 0.0, -0.5000000000000002),\n                        (1, 0.742364948917508, 0.0, -0.5),\n                        (4, 0),\n                        (9, 10)],\n                       [(2, 7),\n                        (6, 1),\n                        (4, 0.75, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (9, 0.75, 0.0, 0.5),\n                        (10, 0.5, 0.0, 0.5)],\n                       [(1, 0.5852091969275525, 0.0, -0.5),\n                        (2, 0.975819462445198, 2.220446049250313e-16, -0.5000000000000002),\n                        (7, 0.6828010596214558, 0.0, 0.5),\n                        (6, 0.6827556138673159, 0.0, -0.5),\n                        (3, 4),\n                        (8, 9)],\n                       [(1, 2),\n                        (7, 6),\n                        (4, 0.5000000000000001, 0.0, -0.5),\n                        (3, 0.5, 0.0, 0.5),\n                        (9, 0.5000000000000001, 0.0, -0.5),\n                        (8, 0.5, 0.0, 0.5)],\n                       [(2, 0.5619508420614284, 0.0, 0.5),\n                        (7, 0.7914256191019993, 0.0, -0.5),\n                        (6, 0.10515249299745623, 0.0, 0.5),\n                        (1, 0.271262519959959, 0.0, -0.5),\n                        (4, 0),\n                        (9, 10)],\n                       [(2, 7),\n                        (6, 1),\n                        (4, 0.25, 0.0, 0.5),\n                        (0, 1.0, 0.0, -0.5),\n                        (9, 0.25, 0.0, 0.5),\n                        (10, 1.0, 0.0, -0.5)],\n                       [(1, 0.7093561802708562, 0.0, -0.5),\n                        (2, 0.21899278445405918, 1.0, -0.5),\n                        (6, 0.07472977972152471, 0.0, -0.5000000000000002),\n                        (7, 0.24013802082480246, 0.0, 0.5),\n                        (3, 4),\n                        (8, 9)],\n                       [(1, 2),\n                        (6, 5),\n                        (4, 0.5, 0.0, -0.5),\n                        (9, 0.5, 0.0, -0.5),\n                        (3, 0.5, 0.0, 0.5),\n                        (8, 0.5, 0.0, 0.5)],\n                       [(1, 0.5, 1.0, -0.5),\n                        (6, 0.49999999999999994, 0.0, -0.5),\n                        (5, 0.3040867239846963, 0.0, -0.5),\n                        (4, 0),\n                        (9, 10)],\n                       [(2, 1), (7, 6), (4, 0.5000000000000001, 0.0, 0.5), (9, 0.5000000000000001, 0.0, 0.5)],\n                       [(2, 0.25000000000000006, 0.0, 0.5),\n                        (1, 0.25000000000000006, 0.0, -0.5),\n                        (6, 0.8040867239846964, 0.0, 0.5),\n                        (7, 1.0, 0.0, 0.5)],\n                       [(2, 1), (6, 5)],\n                       [(6, 0.75, 0.0, 0.5), (5, 0.5, 0.0, 0.5), (2, 0.5, 1.0, -0.5), (1, 0.5, 1.0, -0.5)],\n                       [(7, 6)],\n                       [(6, 0.5000000000000001, 0.0, -0.5), (7, 0.5, 0.0, 0.5)],\n                       [(6, 5)],\n                       [(6, 0.25, 0.0, 0.5), (5, 1.0, 0.0, -0.5)],\n                       [(7, 6)],\n                       [(6, 0.5, 0.0, -0.5), (7, 0.5, 0.0, 0.5)],\n                       [(6, 5)],\n                       [(6, 0.5000000000000001, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],\n           \'qubits\': [(3, 7),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 3),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8)]},\n (12, 3): {\'method\': \'binary count seed + moving six-CZ decoders\',\n           \'moments\': [[(0, 0.5111628543845956, 0.0, -0.5),\n                        (4, 0.5967584468007902, 0.0, -0.5),\n                        (1, 0.2224043775590581, 0.0, 0.5),\n                        (5, 0.31262506132141626, 0.0, 0.5),\n                        (8, 0.3396990389404374, 0.0, 0.5),\n                        (9, 0.6434041384784817, 0.0, -0.5),\n                        (2, 0.4135610417387456, 0.0, -0.5),\n                        (3, 0.5, 0.0, 0.5),\n                        (6, 0.4135610417387456, 0.0, -0.5),\n                        (7, 0.5, 0.0, 0.5),\n                        (10, 0.4135610417387456, 0.0, -0.5),\n                        (11, 0.5, 0.0, 0.5)],\n                       [(0, 4), (1, 5), (8, 9)],\n                       [(0, 0.46285961629392974, 0.0, 0.5),\n                        (1, 0.9970670680638652, 0.0, 0.5),\n                        (4, 0.03543040876372844, 0.0, 0.5),\n                        (8, 0.3046427360537705, 0.0, -0.5),\n                        (5, 0.09447303316706054, 0.0, 0.5),\n                        (9, 0.4682994853843922, 0.0, 0.5)],\n                       [(0, 1), (4, 8), (5, 9)],\n                       [(0, 0.5000000000000001, 0.0, -0.5),\n                        (1, 0.7812473274387313, 0.0, 0.5),\n                        (4, 1.0, 0.0, 0.5),\n                        (5, 0.2419971333999616, 0.0, -0.5),\n                        (8, 0.8642923742789275, 0.0, -0.5),\n                        (9, 0.5867995296243195, 0.0, -0.5)],\n                       [(0, 1), (4, 5), (8, 9)],\n                       [(0, 1.0, 0.0, -0.5),\n                        (4, 0.49999999999999994, 0.0, 0.5),\n                        (1, 1.0, 0.0, -0.5),\n                        (5, 0.9094455581581089, 0.0, 0.5),\n                        (8, 0.17914367281967578, 0.0, 0.5),\n                        (9, 0.22308999977550756, 0.0, 0.5)],\n                       [(0, 4), (1, 5), (8, 9)],\n                       [(0, 0.5198213526971043, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5),\n                        (4, 0.6638125975824298, 0.0, -0.5),\n                        (8, 0.5573697908245031, 0.0, 0.5),\n                        (5, 0.9965940589703396, 2.220446049250313e-16, -0.5000000000000002),\n                        (9, 0.9265082889398296, 0.0, 0.5)],\n                       [(0, 1), (4, 8), (5, 9)],\n                       [(0, 0.38058459439193304, 0.0, 0.5),\n                        (4, 0.4306438005457498, 0.0, -0.5),\n                        (1, 0.4244993749336057, 0.0, -0.5),\n                        (5, 0.7808187343107386, 0.0, 0.5),\n                        (9, 0.8774444190135595, 0.0, -0.5)],\n                       [(0, 4), (1, 5), (8, 9)],\n                       [(0, 0.5000000000000018, 0.0, 0.5),\n                        (1, 0.9585893018165611, 2.220446049250313e-16, -0.5000000000000002),\n                        (4, 0.5, 0.0, -0.5),\n                        (8, 1.0, 0.0, 0.5),\n                        (5, 0.4506337752796546, 0.0, -0.5),\n                        (9, 0.6298653732211198, 0.0, 0.5)],\n                       [(0, 1), (4, 8), (5, 9)],\n                       [(1, 0.015649167467986832, 0.0, -0.5000000000000002),\n                        (4, 1.0, 0.0, -0.5),\n                        (5, 0.5910387482582845, 0.0, 0.5),\n                        (8, 1.0, 0.0, 0.5),\n                        (9, 0.6266087885988807, 0.0, 0.5)],\n                       [(0, 1), (4, 5), (8, 9)],\n                       [(1, 0.993634966006078, 0.0, -0.5),\n                        (0, 0.9999999999999983, 0.0, 0.5),\n                        (5, 0.5728311794330377, 0.0, -0.5),\n                        (9, 0.11907828187925305, 0.0, -0.5000000000000002),\n                        (8, 1.0, 0.0, -0.5)],\n                       [(1, 2), (5, 6), (9, 10)],\n                       [(1, 0.3819668494768557, 0.0, 0.5),\n                        (2, 0.6338602364006151, 0.0, -0.5),\n                        (5, 0.3819668494768557, 0.0, 0.5),\n                        (6, 0.6338602364006151, 0.0, -0.5),\n                        (9, 0.3819668494768557, 0.0, 0.5),\n                        (10, 0.6338602364006151, 0.0, -0.5)],\n                       [(0, 1), (4, 5), (8, 9)],\n                       [(1, 0.7376490383100945, 0.0, 0.5),\n                        (0, 1.0, 0.0, 0.5),\n                        (5, 0.7376490383100945, 0.0, 0.5),\n                        (4, 1.0, 0.0, 0.5),\n                        (9, 0.7376490383100945, 0.0, 0.5),\n                        (8, 1.0, 0.0, 0.5)],\n                       [(1, 2), (5, 6), (9, 10)],\n                       [(1, 0.16018463696253962, 0.0, 0.5),\n                        (2, 0.18023932987586394, 0.0, -0.5000000000000002),\n                        (5, 0.16018463696253962, 0.0, 0.5),\n                        (6, 0.18023932987586394, 0.0, -0.5000000000000002),\n                        (9, 0.16018463696253962, 0.0, 0.5),\n                        (10, 0.18023932987586394, 0.0, -0.5000000000000002)],\n                       [(0, 1), (4, 5), (8, 9)],\n                       [(1, 0.4422317453870271, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (5, 0.4422317453870271, 0.0, 0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (9, 0.4422317453870271, 0.0, 0.5),\n                        (8, 0.5, 0.0, 0.5)],\n                       [(1, 2), (5, 6), (9, 10)],\n                       [(1, 0.6666666666666669, 0.0, 0.5),\n                        (2, 0.8040867239846964, 2.220446049250313e-16, -0.5000000000000002),\n                        (5, 0.6666666666666669, 0.0, 0.5),\n                        (6, 0.8040867239846964, 2.220446049250313e-16, -0.5000000000000002),\n                        (9, 0.6666666666666669, 0.0, 0.5),\n                        (10, 0.8040867239846964, 2.220446049250313e-16, -0.5000000000000002)],\n                       [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9), (10, 11)],\n                       [(1, 1.0, 0.0, 0.5),\n                        (2, 0.49999999999999994, 0.0, -0.5),\n                        (3, 0.3040867239846963, 0.0, -0.5),\n                        (5, 1.0, 0.0, 0.5),\n                        (6, 0.49999999999999994, 0.0, -0.5),\n                        (7, 0.3040867239846963, 0.0, -0.5),\n                        (9, 1.0, 0.0, 0.5),\n                        (10, 0.49999999999999994, 0.0, -0.5),\n                        (11, 0.3040867239846963, 0.0, -0.5),\n                        (0, 0.5, 0.0, -0.5),\n                        (4, 0.5, 0.0, -0.5),\n                        (8, 0.5, 0.0, -0.5)],\n                       [(1, 2), (5, 6), (9, 10)],\n                       [(2, 0.8040867239846964, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5),\n                        (6, 0.8040867239846964, 0.0, 0.5),\n                        (5, 1.0, 0.0, 0.5),\n                        (10, 0.8040867239846964, 0.0, 0.5),\n                        (9, 1.0, 0.0, 0.5)],\n                       [(2, 3), (6, 7), (10, 11)],\n                       [(2, 0.75, 0.0, 0.5),\n                        (3, 0.5, 0.0, 0.5),\n                        (6, 0.75, 0.0, 0.5),\n                        (7, 0.5, 0.0, 0.5),\n                        (10, 0.75, 0.0, 0.5),\n                        (11, 0.5, 0.0, 0.5)],\n                       [(1, 2), (5, 6), (9, 10)],\n                       [(2, 0.5000000000000001, 0.0, -0.5),\n                        (1, 0.5, 0.0, 0.5),\n                        (6, 0.5000000000000001, 0.0, -0.5),\n                        (5, 0.5, 0.0, 0.5),\n                        (10, 0.5000000000000001, 0.0, -0.5),\n                        (9, 0.5, 0.0, 0.5)],\n                       [(2, 3), (6, 7), (10, 11)],\n                       [(2, 0.25, 0.0, 0.5),\n                        (3, 1.0, 0.0, -0.5),\n                        (6, 0.25, 0.0, 0.5),\n                        (7, 1.0, 0.0, -0.5),\n                        (10, 0.25, 0.0, 0.5),\n                        (11, 1.0, 0.0, -0.5)],\n                       [(1, 2), (5, 6), (9, 10)],\n                       [(2, 0.5, 0.0, -0.5),\n                        (6, 0.5, 0.0, -0.5),\n                        (10, 0.5, 0.0, -0.5),\n                        (1, 0.5, 0.0, 0.5),\n                        (5, 0.5, 0.0, 0.5),\n                        (9, 0.5, 0.0, 0.5)],\n                       [(2, 3), (6, 7), (10, 11)],\n                       [(2, 0.5000000000000001, 0.0, 0.5),\n                        (6, 0.5000000000000001, 0.0, 0.5),\n                        (10, 0.5000000000000001, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (3, 6),\n                      (3, 7),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7)]},\n (12, 4): {\'method\': \'paired binary counts + recursive coherent decoders\',\n           \'moments\': [[(3, 0.7957376214201458, 0.0, 0.5),\n                        (8, 0.4687737961889322, 0.0, -0.5),\n                        (9, 0.8272675342562875, 0.0, 0.5),\n                        (5, 0.31506567447794065, 0.0, 0.5),\n                        (10, 0.6262559764502733, 0.0, -0.5),\n                        (2, 0.23850197504578505, 0.6836536622471767, 0.6723366713104197),\n                        (1, 0.5, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (11, 0.23850197504578505, 0.6836536622471767, 0.6723366713104197),\n                        (7, 0.5, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5)],\n                       [(3, 8), (4, 9), (5, 10)],\n                       [(3, 0.33008608643604, 0.0, 0.5),\n                        (4, 0.8518152381372813, 0.0, -0.5),\n                        (8, 0.4608533292880442, 0.0, 0.5),\n                        (9, 0.8272675342562876, 2.220446049250313e-16, -0.5000000000000002),\n                        (5, 0.39852685394517157, 0.0, 0.5),\n                        (10, 0.8032714945010344, 0.0, 0.5)],\n                       [(3, 4), (8, 9)],\n                       [(4, 0.7961552209828021, 0.0, -0.5),\n                        (9, 1.0, 0.0, -0.5),\n                        (3, 0.5954307508624413, 0.0, -0.5),\n                        (8, 0.9532370736381507, 0.0, 0.5)],\n                       [(4, 5), (9, 10), (3, 8)],\n                       [(4, 0.5660272926594455, 0.0, -0.5),\n                        (9, 0.5000000000000001, 0.0, 0.5),\n                        (5, 0.29794465064146197, 0.0, 0.5),\n                        (10, 0.07527312065996314, 0.0, 0.5),\n                        (3, 0.44883755131908326, 0.0, 0.5),\n                        (8, 0.9170566730838476, 2.220446049250313e-16, -0.5000000000000002)],\n                       [(4, 9), (5, 10)],\n                       [(4, 1.0, 0.0, -0.5),\n                        (9, 0.5000000000000001, 0.0, 0.5),\n                        (5, 0.5000000000000001, 0.0, -0.5),\n                        (10, 0.263501550499697, 0.0, -0.5)],\n                       [(3, 4), (8, 9)],\n                       [(4, 0.5000000000000001, 0.0, 0.5),\n                        (9, 0.5000000000000001, 0.0, 0.5),\n                        (3, 0.9302553973796315, 0.0, 0.5),\n                        (8, 0.49999999999999994, 0.0, -0.5)],\n                       [(4, 5), (9, 10), (3, 8)],\n                       [(4, 0.75, 0.0, -0.5),\n                        (9, 0.25000000000000006, 0.0, -0.5),\n                        (5, 0.9999999276861041, 2.220446049250313e-16, -0.5000000000000002),\n                        (10, 0.9999999276861041, 0.0, 0.5),\n                        (3, 0.22116248200228986, 0.0, -0.5),\n                        (8, 0.49999999999999994, 0.0, -0.5)],\n                       [(4, 9), (5, 10)],\n                       [(4, 0.07613840799480427, 0.0, -0.5000000000000002),\n                        (9, 0.0761384079948043, 0.0, 0.5),\n                        (5, 0.32863614774161637, 0.1559079978302702, 0.45205715403443725),\n                        (10, 0.3286360047511991, 0.1559079855137125, 0.4520571410745993)],\n                       [(3, 4), (8, 9), (5, 2), (10, 11)],\n                       [(3, 5.94669025668831e-17, 0.1242305927826205, -0.27615314601879404),\n                        (4, 0.6871467979306279, 0.07253891415043512, -0.5947841296358423),\n                        (5, 0.8439229341407215, -0.7193557375644737, 0.1933650113406229),\n                        (2, 0.5683578105795406, 0.8902564540277604, -0.19497162687511627),\n                        (8, 5.94669025668831e-17, 0.1242305927826205, -0.27615314601879404),\n                        (9, 0.45972651481726245, -0.144866084003354, 0.5791473316978477),\n                        (10, 0.8439229341407215, -0.7193557375644737, 0.1933650113406229),\n                        (11, 0.5683578105795406, 0.8902564540277604, -0.19497162687511627)],\n                       [(3, 4), (8, 9)],\n                       [(4, 0.9549431959582095, 0.2926407819379362, 0.9863479589464894),\n                        (3, 5.217193789602451e-17, 0.5290892819342421, 0.3632841892336514),\n                        (9, 0.9549431959582095, 0.2926407819379362, 0.9863479589464894),\n                        (8, 5.217193789602451e-17, 0.5290892819342421, 0.3632841892336514)],\n                       [(4, 5), (9, 10)],\n                       [(4, 0.36361112073502466, -0.9676259037637183, 0.574318106007434),\n                        (5, 0.6798126636824355, -0.6534603880148313, 0.09245292209712119),\n                        (9, 0.36361112073502466, -0.9676259037637183, 0.574318106007434),\n                        (10, 0.6798126636824355, -0.6534603880148313, 0.09245292209712119)],\n                       [(3, 4), (5, 2), (8, 9), (10, 11)],\n                       [(4, 0.5747492210962789, -0.2289794130554328, 0.5183181993385106),\n                        (5, 0.4197402158362804, 0.34982652381078727, -0.24729368690209075),\n                        (3, 1.4234525758301107e-16, 0.6580126527760145, 0.28408675859343013),\n                        (2, 0.4206867408155152, -0.2496719329425492, 0.7327039950795224),\n                        (9, 0.5747492210962789, -0.2289794130554328, 0.5183181993385106),\n                        (10, 0.4197402158362804, 0.34982652381078727, -0.24729368690209075),\n                        (8, 1.4234525758301107e-16, 0.6580126527760145, 0.28408675859343013),\n                        (11, 0.4206867408155152, -0.2496719329425492, 0.7327039950795224)],\n                       [(4, 5), (9, 10)],\n                       [(4, 0.05618598641592854, 0.6467852559054132, -0.520635118815221),\n                        (5, 0.34060643615624925, 0.12899040145588803, -0.8826847953412706),\n                        (9, 0.05618598641592854, 0.6467852559054132, -0.520635118815221),\n                        (10, 0.34060643615624925, 0.12899040145588803, -0.8826847953412706)],\n                       [(3, 4), (5, 2), (8, 9), (10, 11)],\n                       [(4, 0.5474162491089649, 0.0553311168193632, 0.3707638155834475),\n                        (5, 0.18598715661621895, 0.9227362769616874, 0.272985174864234),\n                        (3, 8.000318775988365e-17, -0.4282343752250468, -0.32110609986475414),\n                        (2, 0.46193265931565114, -0.07081953356643589, -0.9217183389125367),\n                        (9, 0.5474162491089649, 0.0553311168193632, 0.3707638155834475),\n                        (10, 0.18598715661621895, 0.9227362769616874, 0.272985174864234),\n                        (8, 8.000318775988365e-17, -0.4282343752250468, -0.32110609986475414),\n                        (11, 0.46193265931565114, -0.07081953356643589, -0.9217183389125367)],\n                       [(4, 5), (9, 10)],\n                       [(4, 0.7392621381122315, -0.953532617984139, -0.36581971086181353),\n                        (5, 0.30175441391238317, 0.29530563640419083, -0.13242774092929488),\n                        (9, 0.7392621381122315, -0.953532617984139, -0.36581971086181353),\n                        (10, 0.30175441391238317, 0.29530563640419083, -0.13242774092929488)],\n                       [(3, 4), (5, 2), (8, 9), (10, 11)],\n                       [(4, 0.272596887524651, -0.6964840875439884, 0.13579526345693066),\n                        (5, 0.053622665477313765, -0.8471744523039595, -0.800425624792076),\n                        (3, 3.1835063145372436e-17, 0.8968480336579149, 0.7702390463042106),\n                        (2, 0.8029611387137005, 0.5442766657737075, -0.5838829118391504),\n                        (9, 0.272596887524651, -0.6964840875439884, 0.13579526345693066),\n                        (10, 0.053622665477313765, -0.8471744523039595, -0.800425624792076),\n                        (8, 3.1835063145372436e-17, 0.8968480336579149, 0.7702390463042106),\n                        (11, 0.8029611387137005, 0.5442766657737075, -0.5838829118391504)],\n                       [(4, 5), (9, 10)],\n                       [(4, 0.5582935812841525, 0.37603640487764345, 0.7605524890514219),\n                        (5, 0.6852836122352374, 0.407932545353454, -0.4192224630864543),\n                        (9, 0.5582935812841525, 0.37603640487764345, 0.7605524890514219),\n                        (10, 0.6852836122352374, 0.407932545353454, -0.4192224630864543)],\n                       [(3, 4), (5, 2), (8, 9), (10, 11)],\n                       [(4, 0.8681490545286747, -0.7995069070705199, 0.1897182947041854),\n                        (5, 0.4277380566342639, 0.5625320856099076, 0.24940174792707698),\n                        (3, 0.49999999999999994, -0.2296343063497157, 0.8518313215596385),\n                        (2, 0.8817088848706472, -0.558722158731753, -0.33210840175264433),\n                        (9, 0.8681490545286747, -0.7995069070705199, 0.1897182947041854),\n                        (10, 0.4277380566342639, 0.5625320856099076, 0.24940174792707698),\n                        (8, 0.49999999999999994, -0.2296343063497157, 0.8518313215596385),\n                        (11, 0.8817088848706472, -0.558722158731753, -0.33210840175264433)],\n                       [(4, 5), (9, 10)],\n                       [(4, 0.6515066280755871, -0.4397086566230084, 0.24698572431079874),\n                        (5, 0.11927728230983459, -0.5193100186065933, 0.672387230799335),\n                        (9, 0.6515066280755871, -0.4397086566230084, 0.24698572431079874),\n                        (10, 0.11927728230983459, -0.5193100186065933, 0.672387230799335)],\n                       [(3, 4), (5, 2), (8, 9), (10, 11)],\n                       [(4, 1.0, 0.0, -0.4733697614619907),\n                        (5, 0.7621154011768825, 0.23269196247418478, 0.7586421530068475),\n                        (3, 0.5000000000000001, -0.16285755051025763, 0.6221970152099252),\n                        (2, 0.6174170730099287, 0.2292823704785909, -0.9697650623094198),\n                        (9, 1.0, 0.0, -0.4733697614619907),\n                        (10, 0.7621154011768825, 0.23269196247418478, 0.7586421530068475),\n                        (8, 0.5000000000000001, -0.16285755051025763, 0.6221970152099252),\n                        (11, 0.6174170730099287, 0.2292823704785909, -0.9697650623094198)],\n                       [(4, 5), (9, 10)],\n                       [(4, 0.5, -0.09365115211300723, -0.7987657178372884),\n                        (5, 0.7571708757236933, 0.9757235844718755, 0.7413141285721623),\n                        (9, 0.5, -0.09365115211300723, -0.7987657178372884),\n                        (10, 0.7571708757236933, 0.9757235844718755, 0.7413141285721623)],\n                       [(3, 4), (5, 2), (8, 9), (10, 11)],\n                       [(4, 0.04174830847600226, 0.27717979478307964, 0.6075831300497043),\n                        (5, 0.5323582538647063, -0.7524850549651096, 0.9612620737095474),\n                        (3, 0.6218015137737606, 0.4638255362093675, 0.6720832940267569),\n                        (2, 0.24679677437537356, 0.884120486201161, 0.7117248444166187),\n                        (9, 0.04174830847600226, 0.27717979478307964, 0.6075831300497043),\n                        (10, 0.5323582538647063, -0.7524850549651096, 0.9612620737095474),\n                        (8, 0.6218015137737606, 0.4638255362093675, 0.6720832940267569),\n                        (11, 0.24679677437537356, 0.884120486201161, 0.7117248444166187)],\n                       [(4, 5), (3, 1), (2, 0), (9, 10), (8, 7), (11, 6)],\n                       [(4, 0.5, 0.5228349007383934, -0.6152370751672163),\n                        (3, 0.49999999999999994, 0.0, -0.5),\n                        (1, 0.3040867239846963, 0.0, -0.5),\n                        (5, 3.977568099314147e-17, -0.0881050806931194, 0.4882323777908926),\n                        (2, 0.49999999999999994, 0.0, -0.5),\n                        (0, 0.3040867239846963, 0.0, -0.5),\n                        (9, 0.5, 0.5228349007383934, -0.6152370751672163),\n                        (8, 0.49999999999999994, 0.0, -0.5),\n                        (7, 0.3040867239846963, 0.0, -0.5),\n                        (10, 3.977568099314147e-17, -0.0881050806931194, 0.4882323777908926),\n                        (11, 0.49999999999999994, 0.0, -0.5),\n                        (6, 0.3040867239846963, 0.0, -0.5)],\n                       [(4, 3), (5, 2), (9, 8), (10, 11)],\n                       [(3, 0.8040867239846964, 0.0, 0.5),\n                        (4, 1.0, 0.0, 0.5),\n                        (2, 0.8040867239846964, 0.0, 0.5),\n                        (5, 1.0, 0.0, 0.5),\n                        (8, 0.8040867239846964, 0.0, 0.5),\n                        (9, 1.0, 0.0, 0.5),\n                        (11, 0.8040867239846964, 0.0, 0.5),\n                        (10, 1.0, 0.0, 0.5)],\n                       [(3, 1), (2, 0), (8, 7), (11, 6)],\n                       [(3, 0.75, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5),\n                        (2, 0.75, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (8, 0.75, 0.0, 0.5),\n                        (7, 0.5, 0.0, 0.5),\n                        (11, 0.75, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5)],\n                       [(4, 3), (5, 2), (9, 8), (10, 11)],\n                       [(3, 0.5000000000000001, 0.0, -0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (2, 0.5000000000000001, 0.0, -0.5),\n                        (5, 0.5, 0.0, 0.5),\n                        (8, 0.5000000000000001, 0.0, -0.5),\n                        (9, 0.5, 0.0, 0.5),\n                        (11, 0.5000000000000001, 0.0, -0.5),\n                        (10, 0.5, 0.0, 0.5)],\n                       [(3, 1), (2, 0), (8, 7), (11, 6)],\n                       [(3, 0.25, 0.0, 0.5),\n                        (1, 1.0, 0.0, -0.5),\n                        (2, 0.25, 0.0, 0.5),\n                        (0, 1.0, 0.0, -0.5),\n                        (8, 0.25, 0.0, 0.5),\n                        (7, 1.0, 0.0, -0.5),\n                        (11, 0.25, 0.0, 0.5),\n                        (6, 1.0, 0.0, -0.5)],\n                       [(4, 3), (5, 2), (9, 8), (10, 11)],\n                       [(3, 0.5, 0.0, -0.5),\n                        (2, 0.5, 0.0, -0.5),\n                        (8, 0.5, 0.0, -0.5),\n                        (11, 0.5, 0.0, -0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (5, 0.5, 0.0, 0.5),\n                        (9, 0.5, 0.0, 0.5),\n                        (10, 0.5, 0.0, 0.5)],\n                       [(3, 1), (2, 0), (8, 7), (11, 6)],\n                       [(3, 0.5000000000000001, 0.0, 0.5),\n                        (2, 0.5000000000000001, 0.0, 0.5),\n                        (8, 0.5000000000000001, 0.0, 0.5),\n                        (11, 0.5000000000000001, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],\n           \'qubits\': [(2, 6),\n                      (3, 4),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 3),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7)]},\n (12, 5): {\'method\': \'paired binary counts + recursive coherent decoders\',\n           \'moments\': [[(4, 0.59761258847203, -1.0, -0.5),\n                        (3, 0.4427087847307891, 0.0, 0.5),\n                        (5, 0.24038975071939683, 0.0, 0.5),\n                        (10, 0.7461391400083337, 0.0, -0.5),\n                        (11, 0.4652411656670723, -1.0, -0.020153833182791514),\n                        (2, 0.4652411656670723, 0.0, -0.020153833182791514),\n                        (7, 0.5, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (1, 0.5, -1.0, 0.5),\n                        (0, 0.5, 0.0, 0.5)],\n                       [(4, 9), (3, 8), (5, 10)],\n                       [(4, 0.8488151281372129, 0.0, -0.5),\n                        (3, 0.8942958367340017, 0.0, 0.5),\n                        (5, 0.5468684988152348, 0.0, 0.5),\n                        (8, 0.03561776170707942, 0.0, -0.5000000000000002),\n                        (10, 0.856748677172571, -1.0, 0.5)],\n                       [(3, 4), (9, 10)],\n                       [(4, 0.9924012037726202, 0.0, 0.5),\n                        (8, 9),\n                        (3, 0.21437093213337244, 0.0, 0.5),\n                        (10, 0.6319964862414964, 0.0, 0.5)],\n                       [(4, 5),\n                        (8, 0.46438223829292036, 0.0, -0.5),\n                        (3, 0.0, -0.6509854342210133, -0.8490145657789869)],\n                       [(4, 0.8463306209568837, 0.0, 0.5), (5, 0.3601142899420795, 0.0, 0.5), (3, 8)],\n                       [(4, 9), (5, 10), (8, 0.5000000000000009, 0.0, 0.5)],\n                       [(4, 0.7079554715118657, 0.0, -0.5),\n                        (9, 0.5, 0.0, 0.5),\n                        (5, 0.34166512137921257, 0.0, -0.5),\n                        (10, 0.5821612098820304, 0.0, -0.5),\n                        (3, 8)],\n                       [(3, 4), (8, 9)],\n                       [(4, 0.21235401309341517, 0.0, 0.5), (9, 0.5, 0.0, 0.5), (8, 0.0, -1.0, -0.5)],\n                       [(4, 5), (9, 10)],\n                       [(5, 0.34166512137921257, -1.0, 0.5), (9, 0.5, -1.0, -0.5), (3, 4)],\n                       [(5, 10), (4, 9)],\n                       [(10, 0.5047485607097295, 0.0, 0.4814711855113889),\n                        (5, 0.3467039870302344, 0.0, 0.5209076984246561),\n                        (8, 9),\n                        (4, 0.5, 0.0, -0.7167835017790345)],\n                       [(10, 11), (5, 2), (9, 0.2167835017790345, -1.0, -1.0), (3, 4)],\n                       [(10, 0.6205477744107246, 0.0, -0.0011806555587272616),\n                        (5, 0.6205477744107246, 0.0, 0.9935875903678559),\n                        (4, 0.0, 0.48915363147939694, 0.010846368520603056),\n                        (8, 9),\n                        (2, 0.5, 0.0, -0.5201538331827915),\n                        (11, 0.5, -1.0, -0.5201538331827915)],\n                       [(9, 10), (4, 5)],\n                       [(10, 0.5, 0.0, 0.48023208825487806),\n                        (5, 0.5, 0.0, -0.4878251534457494),\n                        (9, 0.7299176310680837, 0.0, 0.9228029411580569),\n                        (4, 0.7299176310680837, 0.0, 0.34956718854230506)],\n                       [(5, 2), (3, 4), (8, 9), (10, 11)],\n                       [(4, 0.5708700815399965, 0.0, -0.9397435049447465),\n                        (9, 0.5708700815399963, 0.0, -0.7878863653548915),\n                        (5, 0.13056868187156478, 0.0, 0.27333091789242525),\n                        (10, 0.13056868187156478, 0.0, -0.28092398308329647),\n                        (2, 0.44795079368547275, 0.0, 0.3434400796687824),\n                        (11, 0.44795079368547275, -1.0, 0.3434400796687824),\n                        (3, 0.5, -1.1102230246251565e-16, -0.004784140921494595),\n                        (8, 0.5, 0.0, 0.3537987067004813)],\n                       [(9, 10), (4, 5)],\n                       [(5, 0.12328076625869655, 0.0, -0.9740515639458376),\n                        (10, 0.12328076625869655, 0.0, -0.03354150124503397),\n                        (4, 0.35440396373816796, 0.0, 0.3616763590209999),\n                        (9, 0.35440396373816796, 0.0, 0.9106937706793623)],\n                       [(10, 11), (5, 2), (9, 0.0, -0.4897861375627026, 0.9897861375627026), (3, 4)],\n                       [(5, 0.31277527658165183, -1.0, 0.5259484360541624),\n                        (10, 0.31277527658165183, 0.0, 0.46645849875496603),\n                        (8, 9),\n                        (3, 0.5, -1.0, 0.9952158590785052),\n                        (2, 0.23061776402485323, 0.0, -0.41529275648922015),\n                        (11, 0.23061776402485323, -1.0, -0.41529275648922015)],\n                       [(4, 5), (9, 10), (8, 0.5, 0.0, 0.3537987067004813)],\n                       [(9, 0.5, -1.0, -0.27545117560817256),\n                        (5, 0.7520450774165834, 0.0, -0.9740515639458378),\n                        (10, 0.7520450774165834, -1.1102230246251565e-16, -0.03354150124503352),\n                        (4, 0.5, 4.440892098500626e-16, -0.9419648322541685)],\n                       [(5, 2), (8, 9), (3, 4), (10, 11)],\n                       [(10, 0.22588638999651067, 0.0, -0.5335415012450335),\n                        (4, 0.5, 4.440892098500626e-16, -0.9419648322541685),\n                        (5, 0.22588638999651067, -1.0, -0.4740515639458377),\n                        (9, 0.5, -1.0, -0.2754511756081728),\n                        (2, 0.39222583923283394, 0.0, -0.04993017182812021),\n                        (11, 0.39222583923283394, -1.0, -0.04993017182812021),\n                        (3, 0.6321947838533057, -1.0, -0.8772485798738141),\n                        (8, 0.6321947838533055, 0.0, 0.2262631456528008)],\n                       [(4, 5), (9, 10)],\n                       [(10, 0.696715986124177, 0.0, -0.5335415012450335),\n                        (5, 0.696715986124177, 0.0, -0.4740515639458379),\n                        (8, 9),\n                        (4, 0.0, -0.5037355019564718, -0.9962644980435282)],\n                       [(10, 11),\n                        (5, 2),\n                        (9, 0.0, -1.0, -0.7802621451960521),\n                        (3, 4),\n                        (8, 0.6771139237982053, 0.0, 0.6380246473327011)],\n                       [(5, 0.583609737771011, 0.0, 0.025948436054161972),\n                        (10, 0.583609737771011, 0.0, 0.9664584987549663),\n                        (11, 0.6050120054676125, -1.0, 0.4640144763916523),\n                        (3, 0.6771139237982053, 0.0, 0.7109899184462858),\n                        (2, 0.6050120054676125, 0.0, 0.4640144763916523),\n                        (8, 0.0, -0.1656564789136512, 0.6656564789136512)],\n                       [(9, 10), (4, 5)],\n                       [(5, 0.06412882081148275, 0.0, -0.974051563945836),\n                        (4, 0.5, 4.440892098500626e-16, 0.787450401998057),\n                        (10, 0.06412882081148275, 4.440892098500626e-16, 0.9664584987549643),\n                        (9, 0.5, 0.0, 0.4913980881831299)],\n                       [(8, 9), (10, 11), (3, 4), (5, 2)],\n                       [(4, 0.5, 4.440892098500626e-16, 0.787450401998057),\n                        (5, 0.49999999999999956, 0.0, -0.9740515639458378),\n                        (9, 0.5, 0.0, -0.5086019118168701),\n                        (10, 0.49999999999999956, 0.0, 0.9664584987549663),\n                        (11, 0.6668726728064027, -1.0, -0.23156229310064602),\n                        (2, 0.6668726728064027, 0.0, -0.23156229310064602)],\n                       [(4, 5), (9, 10)],\n                       [(5, 0.48422238672257323, 0.0, -0.42317215977683753),\n                        (4, 0.5, -1.0, -0.7630150699344329),\n                        (9, 0.5, 0.0, 0.04186356011561987),\n                        (10, 0.48422238672257323, 0.0, -0.5844209054140341)],\n                       [(10, 11), (5, 2), (8, 9), (3, 4)],\n                       [(11, 0.7832164982209655, -1.1102230246251565e-16, -0.48502880815679716),\n                        (8, 0.7832164982209657, 2.220446049250313e-16, -0.47263293293255326),\n                        (9, 0.2714423079803394, 0.0, -0.4581364398843797),\n                        (3, 0.7832164982209657, -1.1102230246251565e-16, -0.344008980202111),\n                        (2, 0.7832164982209655, -1.1102230246251565e-16, -0.48502880815679716),\n                        (5, 0.5, 0.0, -0.8821668567259924),\n                        (4, 0.2714423079803394, 0.0, -0.26301506993443335),\n                        (10, 0.5, -1.1102230246251565e-16, -0.12542620846487917)],\n                       [(4, 5), (11, 6), (3, 1), (9, 10), (2, 0), (8, 7)],\n                       [(3, 0.5, -1.1102230246251565e-16, 0.15599101979788887),\n                        (8, 0.5, -1.0, -0.9726329329325536),\n                        (4, 0.5, -1.0, -0.7630150699344334),\n                        (11, 0.5, -1.0, -0.985028808156797),\n                        (9, 0.5, 4.440892098500626e-16, -0.9581364398843799),\n                        (2, 0.5, 0.0, -0.985028808156797),\n                        (0, 0.30408672398469627, 0.0, -0.5000000000000002),\n                        (6, 0.30408672398469627, 0.0, -0.5000000000000002),\n                        (7, 0.30408672398469627, 0.0, -0.5000000000000002),\n                        (1, 0.30408672398469627, -1.0, -0.5000000000000002),\n                        (10, 0.0, 0.9676359187087931, -0.46763591870879295)],\n                       [(9, 8), (10, 11), (5, 2), (4, 3)],\n                       [(3, 0.8040867239846965, 4.440892098500626e-16, -0.8440089802021113),\n                        (2, 0.8040867239846965, 0.0, 0.014971191843202503),\n                        (11, 0.8040867239846965, 0.0, 0.014971191843202503),\n                        (8, 0.8040867239846965, 0.0, 0.02736706706744685),\n                        (9, 0.0, -0.30487863005005345, 0.8048786300500534)],\n                       [(8, 7), (2, 0), (3, 1), (11, 6)],\n                       [(3, 0.75, 0.0, -0.8440089802021111),\n                        (8, 0.75, 0.0, 0.02736706706744685),\n                        (2, 0.75, -1.0, 0.014971191843202503),\n                        (11, 0.75, 0.0, 0.014971191843202503),\n                        (7, 0.5, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (1, 0.5, -1.0, 0.5),\n                        (0, 0.5, 0.0, 0.5)],\n                       [(5, 2), (10, 11), (4, 3), (9, 8)],\n                       [(11, 0.5000000000000002, 0.0, -0.985028808156797),\n                        (8, 0.5, 0.0, -0.9726329329325536),\n                        (2, 0.5000000000000002, 0.0, -0.985028808156797),\n                        (3, 0.5, 4.440892098500626e-16, -0.8440089802021113),\n                        (10, 0.5, 0.0, -0.5161820406456039),\n                        (5, 0.5, 0.0, -0.5237751058364752),\n                        (4, 0.5, 4.440892098500626e-16, -0.9046233190449158),\n                        (9, 0.5, -1.1102230246251565e-16, -0.12140682082395071)],\n                       [(11, 6), (3, 1), (2, 0), (8, 7)],\n                       [(2, 0.25, -1.0, 0.014971191843202503),\n                        (8, 0.25, 0.0, 0.02736706706744685),\n                        (3, 0.25, 0.0, 0.15599101979788887),\n                        (11, 0.25, 0.0, 0.014971191843202503)],\n                       [(10, 11), (5, 2), (9, 8), (4, 3)],\n                       [(11, 0.5, -1.0, -0.985028808156797),\n                        (8, 0.5, -1.0, -0.9726329329325536),\n                        (3, 0.5, 0.0, 0.15599101979788887),\n                        (2, 0.5, -1.0, -0.985028808156797),\n                        (5, 0.5, -0.9762248941635248, 0.47622489416352476),\n                        (10, 0.5, -0.9838179593543961, -0.5161820406456039),\n                        (9, 0.5, 0.6214068208239507, -0.12140682082395071),\n                        (4, 0.5, 0.40462331904491533, -0.9046233190449153)],\n                       [(11, 6), (3, 1), (2, 0), (8, 7)],\n                       [(11, 0.5, 0.4850288081567973, 0.014971191843202725),\n                        (3, 0.4999999999999998, 0.3440089802021111, -0.8440089802021111),\n                        (8, 0.5, 0.47263293293255315, 0.02736706706744685),\n                        (6, 1, 0, 0.5),\n                        (1, 1, 0, -1.0),\n                        (0, 1, 0, 0.5),\n                        (2, 0.5, 0.4850288081567973, 0.014971191843202725),\n                        (7, 1, 0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],\n           \'qubits\': [(2, 6),\n                      (3, 4),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 3),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7)]},\n (12, 6): {\'method\': \'paired binary counts + recursive coherent decoders\',\n           \'moments\': [[(8, 0.5669640766356541, 0.0, -0.5),\n                        (5, 0.750678442494626, 0.0, -0.5),\n                        (10, 0.5593258217574768, 0.0, -0.5),\n                        (4, 0.42469929883596297, 1.0, -0.5),\n                        (2, 0.4956530884318588, 5.551115123125783e-17, 0.259121498867743),\n                        (11, 0.4956530884318588, 5.551115123125783e-17, 0.259121498867743),\n                        (7, 0.5, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (1, 0.5, 1.0, 0.5)],\n                       [(3, 8), (5, 10), (4, 9)],\n                       [(3, 0.8369228832741187, 0.0, 0.5),\n                        (4, 0.132995388754664, 0.0, 0.5),\n                        (8, 0.5669640766356541, 1.0, 0.5),\n                        (9, 0.6769921099054629, 0.0, -0.5),\n                        (5, 0.34268205803875595, 0.0, -0.5),\n                        (10, 0.7783634189838007, 1.0, 0.49999999999999994)],\n                       [(3, 4), (8, 9)],\n                       [(4, 0.9161407826255356, 0.0, 0.5),\n                        (3, 0.25552846935171464, 1.0, 0.5),\n                        (9, 0.676992109905463, 0.0, 0.5)],\n                       [(4, 5), (3, 8), (9, 10)],\n                       [(4, 0.06757546116177693, 0.0, 0.5),\n                        (5, 0.547103915718318, 0.0, -0.5),\n                        (10, 0.05223886011573531, 0.0, -0.5000000000000002),\n                        (3, 0.20748441349360713, 0.0, -0.5000000000000002)],\n                       [(4, 9), (5, 10)],\n                       [(4, 0.4238941469781188, 0.0, 0.5),\n                        (9, 0.045907404435637345, 0.0, 0.5),\n                        (5, 0.2486733075491454, 0.0, -0.5),\n                        (10, 0.44702827401477485, 0.0, 0.5)],\n                       [(3, 4), (8, 9)],\n                       [(3, 0.9467104513876794, 0.0, 0.5),\n                        (8, 0.4999999999999998, 0.0, -0.5000000000000002),\n                        (4, 0.5933742595437735, 0.0, -0.5),\n                        (9, 0.04590740443563756, 0.0, -0.5000000000000002)],\n                       [(3, 8), (4, 5), (9, 10)],\n                       [(4, 0.9908214978311425, 0.0, 0.5),\n                        (9, 0.4999999999999994, 0.0, -0.5000000000000002),\n                        (5, 0.2500218371421519, 0.0, -0.5),\n                        (10, 0.9824676864795339, 0.0, 0.5),\n                        (8, 0.5, 0.0, -0.5)],\n                       [(4, 9), (5, 10)],\n                       [(8, 9),\n                        (3, 4),\n                        (5, 0.6914957373254336, 0.0, -0.8818821053569152),\n                        (10, 0.4395527261108638, 0.0, 0.7859954024356863)],\n                       [(5, 2),\n                        (10, 11),\n                        (9, 0.43201051140706537, 1.0, 0.5348918558833398),\n                        (3, 0.0, 0.37030651952631866, 0),\n                        (4, 0.9237017269641554, 0.0, -0.6486906977905211)],\n                       [(8, 9),\n                        (3, 4),\n                        (10, 0.34488295206841396, 0.0, 0.6955036446005987),\n                        (5, 0.344882952068414, 0.9999999999999999, -0.20654670951644594),\n                        (11, 0.4925777780083309, 0.0, 0.5072045568580279),\n                        (2, 0.4925777780083309, 0.0, 0.5072045568580279)],\n                       [(9, 0.33274203901299115, 0.0, -0.3575101930019611),\n                        (4, 0.33274203901299115, 0.0, 0.09127634640611276),\n                        (8, 0.0, 0.5279921173015927, 0)],\n                       [(9, 10), (4, 5)],\n                       [(9, 0.9294880475869709, -4.440892098500626e-16, -0.14866550202694473),\n                        (10, 0.9291764376708231, 0.0, 0.7904601964239721),\n                        (4, 0.9294880475869709, 0.0, -0.11756834456890308),\n                        (5, 0.9291764376708231, 0.0, -0.11159015769307246)],\n                       [(8, 9), (10, 11), (3, 4), (5, 2)],\n                       [(9, 0.9764886499665415, 0.0, -0.549121066616858),\n                        (10, 0.2914905715516678, 0.0, 0.6838436799133809),\n                        (4, 0.9764886499665415, 4.440892098500626e-16, -0.7171127799789903),\n                        (5, 0.29149057155166774, 1.0, -0.2182066742036637),\n                        (11, 0.41571419491182243, 0.0, 0.41360830056593145),\n                        (2, 0.41571419491182243, 0.0, 0.41360830056593145)],\n                       [(9, 10), (4, 5)],\n                       [(9, 0.322056849702119, 0.0, -0.07126249800721896),\n                        (10, 0.4183604728916831, 0.0, -0.4569501433304397),\n                        (4, 0.32205684970211895, 0.0, 0.8050286514113707),\n                        (5, 0.4183604728916831, 0.0, 0.6409995025525159)],\n                       [(8, 9), (10, 11), (3, 4), (5, 2)],\n                       [(9, 0.5910955657739062, 0.0, 0.9971346861599337),\n                        (10, 0.5658106523525602, 0.0, -0.15169167422835095),\n                        (4, 0.5910955657739059, 0.0, 0.736631467244218),\n                        (5, 0.5658106523525602, 1.0, 0.9462579716546045),\n                        (11, 0.30256758326839167, 0.0, -0.5706246524937872),\n                        (2, 0.30256758326839167, 0.0, -0.5706246524937872)],\n                       [(9, 10), (4, 5)],\n                       [(9, 0.5222942734869765, 0.0, 0.2870475766703844),\n                        (10, 0.6380247240052519, 0.0, -0.8038597110957453),\n                        (4, 0.5222942734869767, 0.0, -0.5532814232662329),\n                        (5, 0.6380247240052519, 0.0, 0.29408993478721035)],\n                       [(8, 9), (10, 11), (3, 4), (5, 2)],\n                       [(9, 0.3636045284494889, 5.551115123125783e-17, 0.1795160645210389),\n                        (10, 0.017085511701252946, 0.0, -0.9325450233945605),\n                        (4, 0.36360452844948893, 0.0, 0.5542500888831123),\n                        (5, 0.01708551170125294, 1.0, 0.16540462248839555),\n                        (11, 0.32590008584322616, 0.0, -0.5618902727160411),\n                        (2, 0.32590008584322616, 0.0, -0.5618902727160411),\n                        (3, 0.0, -0.4256760795576473, 0)],\n                       [(9, 10), (4, 5)],\n                       [(10, 0.7382217283247945, 0.0, 0.059999120854666854),\n                        (9, 0.9940835612187666, -0.9999999999999996, -0.7719408053629284),\n                        (5, 0.7382217283247945, 0.0, -0.8420512332623777),\n                        (4, 0.9940835612187666, -4.440892098500626e-16, -0.49429304123291984)],\n                       [(10, 11), (8, 9), (5, 2), (3, 4)],\n                       [(9, 0.5303663346437925, 0.0, -0.34568388158448515),\n                        (10, 0.3903186307590596, 0.0, 0.05078007579537891),\n                        (4, 0.5303663346437923, 0.0, -0.9205499650113635),\n                        (5, 0.3903186307590596, 1.0, -0.8512702783216659),\n                        (8, 0.0, 0.9989100529131245, 0),\n                        (11, 0.5100876690078296, 0.0, 0.3944365573706427),\n                        (2, 0.5100876690078296, 0.0, 0.3944365573706427)],\n                       [(9, 10), (4, 5)],\n                       [(9, 0.6025816719136099, 0.0, 0.2876488803785353),\n                        (10, 0.6856477314180824, 0.0, -0.3836781185455438),\n                        (4, 0.6025816719136099, 0.0, 0.44611727302561643),\n                        (5, 0.6856477314180824, 0.0, 0.7142715273374114)],\n                       [(8, 9), (10, 11), (3, 4), (5, 2)],\n                       [(9, 0.8921121044926871, 0.0, -0.2907819069706692),\n                        (10, 0.3385860029956734, 0.0, -0.5682054022541256),\n                        (4, 0.8921121044926871, 0.0, 0.024548060374820935),\n                        (5, 0.3385860029956734, 0.9999999999999999, 0.52974424362883),\n                        (11, 0.23477498989560155, 0.0, -0.3542327425401759),\n                        (2, 0.23477498989560155, 0.0, -0.3542327425401759)],\n                       [(9, 10), (4, 5)],\n                       [(9, 0.5928759364551912, 0.0, -0.7114679379649376),\n                        (10, 0.44925295079759087, 0.0, 0.3298158659654795),\n                        (4, 0.5928759364551911, 0.0, 0.4452340913690893),\n                        (5, 0.44925295079759087, 0.0, -0.5722344881515646)],\n                       [(8, 9), (10, 11), (3, 4), (5, 2)],\n                       [(9, 0.8105365413512657, 1.1102230246251565e-16, -0.21628174461018368),\n                        (10, 0.6433165441464112, 0.0, 0.8408221915366021),\n                        (4, 0.8105365413512657, 0.0, 0.9500478980143345),\n                        (5, 0.6433165441464112, 1.0, -0.06122816258044228),\n                        (8, 0.4999999999999997, 0.0, -0.5883923056830007),\n                        (11, 0.34497587576931366, 0.0, -0.28963794555439826),\n                        (3, 0.5, 0.0, -0.9400750841336103),\n                        (2, 0.34497587576931366, 0.0, -0.28963794555439826)],\n                       [(9, 10), (4, 5)],\n                       [(9, 0.3843844916022807, 0.0, -0.7453686465075835),\n                        (10, 0.5260557718525964, 0.0, 0.9214491736636061),\n                        (4, 0.3843844916022807, 0.0, -0.5208652000882656),\n                        (5, 0.5260557718525966, 0.0, 0.019398819546561974)],\n                       [(8, 9), (10, 11), (3, 4), (5, 2)],\n                       [(10, 0.7720515191095958, 0.0, -0.5698283093864791),\n                        (5, 0.7720515191095958, 1.0, 0.5281213364964763),\n                        (8, 0.5, 0.0, -0.5883923056830003),\n                        (11, 0.4355845015328854, 0.0, -0.8111689735837979),\n                        (3, 0.5, 5.551115123125783e-17, 0.05992491586638943),\n                        (2, 0.4355845015328854, 0.0, -0.8111689735837979)],\n                       [(9, 10), (4, 5)],\n                       [(9, 0.5, 0.0, -0.5382756120315335),\n                        (10, 0.13993480154011695, 0.0, -0.28285243405355254),\n                        (4, 0.5000000000000001, 0.0, 0.27204176543568437),\n                        (5, 0.13993480154011695, 4.440892098500626e-16, 0.8150972118294026)],\n                       [(8, 9), (10, 11), (3, 4), (5, 2)],\n                       [(8, 0.45461121091321155, 0.0, -0.2859199864658297),\n                        (9, 0.22995692914698024, 0.0, -0.03827561203153351),\n                        (10, 0.893580808387734, 8.881784197001252e-16, -0.9895855299559677),\n                        (11, 0.3416013957299424, 0.0, -0.7695375446745221),\n                        (2, 0.3416013957299424, 0.0, -0.7695375446745221),\n                        (3, 0.45461121091321166, 0.0, 0.7574525966492187),\n                        (4, 0.22995692914698024, 0.0, 0.7720417654356844),\n                        (5, 0.893580808387734, -0.9999999999999996, 0.10836411592698758)],\n                       [(8, 7), (9, 10), (11, 6), (2, 0), (3, 1), (4, 5)],\n                       [(9, 0.4999999999999998, 0.0, -0.538275612031534),\n                        (8, 0.5000000000000001, 0.0, -0.4988086435756469),\n                        (11, 0.4999999999999998, 0.9999999999999999, -0.18230509551025165),\n                        (4, 0.4999999999999998, 0.0, -0.7279582345643147),\n                        (3, 0.5, 0.0, 0.9703412537590361),\n                        (2, 0.4999999999999998, 0.9999999999999999, -0.18230509551025165),\n                        (7, 0.3040867239846962, 0.0, -0.5000000000000002),\n                        (6, 0.3040867239846962, 0.0, -0.5000000000000002),\n                        (0, 0.3040867239846962, 0.0, -0.5000000000000002),\n                        (1, 0.3040867239846962, 1.0, -0.5000000000000002)],\n                       [(9, 8), (10, 11), (4, 3), (5, 2)],\n                       [(8, 0.8040867239846965, 0.0, 0.5011913564243531),\n                        (11, 0.8040867239846965, 4.440892098500626e-16, 0.8176949044897481),\n                        (2, 0.8040867239846965, 4.440892098500626e-16, 0.8176949044897481),\n                        (3, 0.8040867239846965, 0.0, 0.9703412537590361),\n                        (10, 0.0, 0.9505424333846237, 0),\n                        (5, 0.0, 0.9505424333846242, 0),\n                        (4, 0.0, -0.07477424670013, 0)],\n                       [(8, 7), (11, 6), (2, 0), (3, 1)],\n                       [(11, 0.75, 0.0, 0.8176949044897481),\n                        (8, 0.75, 1.0, 0.5011913564243531),\n                        (2, 0.75, 0.0, 0.8176949044897481),\n                        (3, 0.75, 0.0, 0.9703412537590361),\n                        (7, 0.5, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (1, 0.5, 1.0, 0.5)],\n                       [(10, 11), (9, 8), (5, 2), (4, 3)],\n                       [(8, 0.5000000000000002, 0.0, -0.4988086435756469),\n                        (11, 0.5000000000000002, 0.0, -0.18230509551025165),\n                        (2, 0.5000000000000002, 0.0, -0.18230509551025165),\n                        (3, 0.5, 0.0, -0.02965874624096365),\n                        (10, 0.5, 4.440892098500626e-16, 0.9632960183125248),\n                        (9, 0.5, 0.0, 0.1963790300540857),\n                        (5, 0.5, 0.0, 0.06124566419548033),\n                        (4, 0.5, 1.0, 0.4626128766499342)],\n                       [(8, 7), (11, 6), (2, 0), (3, 1)],\n                       [(11, 0.25000000000000006, 4.440892098500626e-16, 0.8176949044897481),\n                        (8, 0.25000000000000006, 1.0, 0.5011913564243531),\n                        (2, 0.25000000000000006, 4.440892098500626e-16, 0.8176949044897481),\n                        (3, 0.25, 0.0, 0.9703412537590361)],\n                       [(10, 11), (9, 8), (5, 2), (4, 3)],\n                       [(8, 0.5, 1.0, -0.4988086435756469),\n                        (11, 0.5, 0.9999999999999999, -0.18230509551025165),\n                        (3, 0.5, 0.0, -0.02965874624096365),\n                        (2, 0.5, 0.9999999999999999, -0.18230509551025165),\n                        (10, 0.5, -0.46329601831252454, 0.9632960183125248),\n                        (9, 0.5, 0.30362096994591425, -0.8036209699459145),\n                        (4, 0.5, 0.03738712335006561, 0.4626128766499344),\n                        (5, 0.5, 0.43875433580452006, 0.06124566419548039)],\n                       [(8, 7), (11, 6), (3, 1), (2, 0)],\n                       [(6, 1.0, 0.0, 0.5),\n                        (7, 1.0, 0.0, 0.5),\n                        (8, 0.5, -0.001191356424353085, 0.5011913564243531),\n                        (11, 0.5, -0.31769490448974813, 0.8176949044897481),\n                        (1, 1.0, 0.0, 0.0),\n                        (0, 1.0, 0.0, 0.5),\n                        (2, 0.5, -0.31769490448974813, 0.8176949044897481),\n                        (3, 0.5, 0.5296587462409641, 0.9703412537590359)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],\n           \'qubits\': [(2, 6),\n                      (3, 4),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 3),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7)]},\n (13, 3): {\'method\': \'binary count seed + moving six-CZ decoders\',\n           \'moments\': [[(0, 0.5011286333212674, 0.0, -0.5),\n                        (5, 0.5648042837558233, 0.0, -0.5),\n                        (1, 0.22639989947272104, 0.0, 0.5),\n                        (6, 0.37901296827077646, 0.0, 0.5),\n                        (9, 0.3584365439544183, 0.0, 0.5),\n                        (10, 0.6115265567555555, 0.0, -0.5),\n                        (2, 0.412184131733575, 0.0, -0.5),\n                        (3, 0.4135610417387456, 0.0, -0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (7, 0.4135610417387456, 0.0, -0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (11, 0.4135610417387456, 0.0, -0.5),\n                        (12, 0.5, 0.0, 0.5)],\n                       [(0, 5), (1, 6), (9, 10)],\n                       [(0, 0.4944183722198826, 0.0, 0.5),\n                        (1, 0.9887937899601499, 0.0, -0.5),\n                        (5, 0.0054663407070543735, 0.0, 0.5),\n                        (9, 0.20817138739784288, 0.0, -0.5),\n                        (6, 0.05896518616556724, 0.0, 0.5),\n                        (10, 0.5109898938650242, 0.0, 0.5)],\n                       [(0, 1), (5, 9), (6, 10)],\n                       [(0, 0.5000000000000001, 0.0, -0.5),\n                        (1, 0.7787066990539395, 0.0, 0.5),\n                        (5, 1.0, 0.0, 0.5),\n                        (6, 0.2666010547086309, 0.0, -0.5),\n                        (9, 0.9044411812600366, 0.0, -0.5),\n                        (10, 0.5357339946022933, 0.0, -0.5)],\n                       [(0, 1), (5, 6), (9, 10)],\n                       [(0, 1.0, 0.0, -0.5),\n                        (5, 0.5, 0.0, 0.5),\n                        (1, 1.0, 0.0, -0.5),\n                        (6, 0.9592381571300939, 0.0, 0.5),\n                        (9, 0.20484824042867295, 0.0, 0.5),\n                        (10, 0.22637759711444339, 0.0, 0.5)],\n                       [(0, 5), (1, 6), (9, 10)],\n                       [(0, 0.540825634749762, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5),\n                        (5, 0.6799453846918434, 0.0, -0.5),\n                        (9, 0.6796033860670321, 0.0, 0.5),\n                        (6, 0.9913262427252412, 2.220446049250313e-16, -0.5000000000000002),\n                        (10, 0.8874597674119914, 0.0, 0.5)],\n                       [(0, 1), (5, 9), (6, 10)],\n                       [(0, 0.35408611734683687, 0.0, 0.5),\n                        (5, 0.5122437211025593, 0.0, -0.5),\n                        (1, 0.47699444027673116, 0.0, -0.5),\n                        (6, 0.8492358345397026, 0.0, 0.5),\n                        (10, 0.8351725982640147, 0.0, -0.5)],\n                       [(0, 5), (1, 6), (9, 10)],\n                       [(0, 0.4999999999999993, 0.0, 0.5),\n                        (1, 0.8130648608503948, 2.220446049250313e-16, -0.5000000000000002),\n                        (5, 0.5, 0.0, -0.5),\n                        (9, 1.0, 0.0, 0.5),\n                        (6, 0.43942608628445445, 0.0, -0.5),\n                        (10, 0.6237193013742914, 0.0, 0.5)],\n                       [(0, 1), (5, 9), (6, 10)],\n                       [(1, 0.03811502306613775, 0.0, -0.5000000000000002),\n                        (5, 1.0, 0.0, -0.5),\n                        (6, 0.6322754934663775, 0.0, 0.5),\n                        (9, 1.0, 0.0, 0.5),\n                        (10, 0.6383902508124725, 0.0, 0.5)],\n                       [(0, 1), (5, 6), (9, 10)],\n                       [(1, 0.8118240558323843, 0.0, 0.5),\n                        (0, 0.9999999999999994, 0.0, -0.5),\n                        (6, 0.5591170679959561, 0.0, -0.5),\n                        (10, 0.10047114062159844, 0.0, -0.5000000000000002),\n                        (9, 1.0, 0.0, -0.5)],\n                       [(1, 2), (6, 7), (10, 11)],\n                       [(1, 0.5903120017947893, 0.0, -0.5),\n                        (2, 0.6117890887690124, 0.0, -0.5),\n                        (6, 0.3819668494768557, 0.0, 0.5),\n                        (7, 0.6338602364006151, 0.0, -0.5),\n                        (10, 0.3819668494768557, 0.0, 0.5),\n                        (11, 0.6338602364006151, 0.0, -0.5)],\n                       [(0, 1), (5, 6), (9, 10)],\n                       [(1, 0.7075436162554982, 0.0, 0.5),\n                        (0, 1.0, 0.0, 0.5),\n                        (6, 0.7376490383100945, 0.0, 0.5),\n                        (5, 1.0, 0.0, 0.5),\n                        (10, 0.7376490383100945, 0.0, 0.5),\n                        (9, 1.0, 0.0, 0.5)],\n                       [(1, 2), (6, 7), (10, 11)],\n                       [(1, 0.13376589621282342, 0.0, 0.5),\n                        (2, 0.1417314999479891, 0.0, 0.5),\n                        (6, 0.16018463696253962, 0.0, 0.5),\n                        (7, 0.18023932987586394, 0.0, -0.5000000000000002),\n                        (10, 0.16018463696253962, 0.0, 0.5),\n                        (11, 0.18023932987586394, 0.0, -0.5000000000000002)],\n                       [(0, 1), (5, 6), (9, 10)],\n                       [(1, 0.4425626624366138, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (6, 0.4422317453870271, 0.0, 0.5),\n                        (5, 0.5, 0.0, 0.5),\n                        (10, 0.4422317453870271, 0.0, 0.5),\n                        (9, 0.5, 0.0, 0.5)],\n                       [(1, 2), (6, 7), (10, 11)],\n                       [(1, 0.35241638234956674, 0.0, -0.5),\n                        (2, 0.8746983728132804, 0.0, -0.5),\n                        (6, 0.6666666666666669, 0.0, 0.5),\n                        (7, 0.8040867239846964, 2.220446049250313e-16, -0.5000000000000002),\n                        (10, 0.6666666666666669, 0.0, 0.5),\n                        (11, 0.8040867239846964, 2.220446049250313e-16, -0.5000000000000002)],\n                       [(0, 1), (2, 3), (5, 6), (7, 8), (9, 10), (11, 12)],\n                       [(1, 1.0, 0.0, 0.5),\n                        (2, 0.3819668494768557, 0.0, 0.5),\n                        (3, 0.6338602364006151, 0.0, -0.5),\n                        (6, 1.0, 0.0, 0.5),\n                        (7, 0.49999999999999994, 0.0, -0.5),\n                        (8, 0.3040867239846963, 0.0, -0.5),\n                        (10, 1.0, 0.0, 0.5),\n                        (11, 0.49999999999999994, 0.0, -0.5),\n                        (12, 0.3040867239846963, 0.0, -0.5),\n                        (0, 0.5, 0.0, -0.5),\n                        (5, 0.5, 0.0, -0.5),\n                        (9, 0.5, 0.0, -0.5)],\n                       [(1, 2), (6, 7), (10, 11)],\n                       [(2, 0.7376490383100945, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5),\n                        (7, 0.8040867239846964, 0.0, 0.5),\n                        (6, 1.0, 0.0, 0.5),\n                        (11, 0.8040867239846964, 0.0, 0.5),\n                        (10, 1.0, 0.0, 0.5)],\n                       [(2, 3), (7, 8), (11, 12)],\n                       [(2, 0.16018463696253962, 0.0, 0.5),\n                        (3, 0.18023932987586394, 0.0, -0.5000000000000002),\n                        (7, 0.75, 0.0, 0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (11, 0.75, 0.0, 0.5),\n                        (12, 0.5, 0.0, 0.5)],\n                       [(1, 2), (6, 7), (10, 11)],\n                       [(2, 0.4422317453870271, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5),\n                        (7, 0.5000000000000001, 0.0, -0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (11, 0.5000000000000001, 0.0, -0.5),\n                        (10, 0.5, 0.0, 0.5)],\n                       [(2, 3), (7, 8), (11, 12)],\n                       [(2, 0.6666666666666669, 0.0, 0.5),\n                        (3, 0.8040867239846964, 2.220446049250313e-16, -0.5000000000000002),\n                        (7, 0.25, 0.0, 0.5),\n                        (8, 1.0, 0.0, -0.5),\n                        (11, 0.25, 0.0, 0.5),\n                        (12, 1.0, 0.0, -0.5)],\n                       [(1, 2), (3, 4), (6, 7), (10, 11)],\n                       [(2, 1.0, 0.0, 0.5),\n                        (3, 0.49999999999999994, 0.0, -0.5),\n                        (4, 0.3040867239846963, 0.0, -0.5),\n                        (7, 0.5, 0.0, -0.5),\n                        (11, 0.5, 0.0, -0.5),\n                        (1, 0.5, 0.0, -0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (10, 0.5, 0.0, 0.5)],\n                       [(2, 3), (7, 8), (11, 12)],\n                       [(3, 0.8040867239846964, 0.0, 0.5),\n                        (2, 1.0, 0.0, 0.5),\n                        (7, 0.5000000000000001, 0.0, 0.5),\n                        (11, 0.5000000000000001, 0.0, 0.5)],\n                       [(3, 4)],\n                       [(3, 0.75, 0.0, 0.5), (4, 0.5, 0.0, 0.5)],\n                       [(2, 3)],\n                       [(3, 0.5000000000000001, 0.0, -0.5), (2, 0.5, 0.0, 0.5)],\n                       [(3, 4)],\n                       [(3, 0.25, 0.0, 0.5), (4, 1.0, 0.0, -0.5)],\n                       [(2, 3)],\n                       [(3, 0.5, 0.0, -0.5), (2, 0.5, 0.0, 0.5)],\n                       [(3, 4)],\n                       [(3, 0.5000000000000001, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (3, 6),\n                      (3, 7),\n                      (3, 8),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7)]},\n (13, 4): {\'method\': \'paired binary counts + recursive coherent decoders\',\n           \'moments\': [[(8, 0.4255771808057458, 0.0, -0.5),\n                        (6, 0.2859133439469217, 0.0, 0.5),\n                        (10, 0.7530877427391665, 0.0, 0.5),\n                        (5, 0.13672592290297383, 1.0, -0.5000000000000002),\n                        (2, 0.23850197504578502, 0.0, 0.6723366713104197),\n                        (11, 0.4258875989061941, 1.0, 0.5918062908230985),\n                        (7, 0.23850197504578505, 1.0, 0.6723366713104195),\n                        (0, 0.5, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5),\n                        (3, 0.5, 0.0, 0.5),\n                        (12, 0.5, 1.0, 0.5)],\n                       [(4, 8), (6, 10), (5, 9)],\n                       [(4, 0.994057967253313, 0.0, 0.5),\n                        (5, 0.8632740770970266, 0.0, 0.5),\n                        (8, 0.1318497038270321, 1.0, 0.5),\n                        (9, 0.17750083138414635, 0.0, 0.5),\n                        (6, 0.2676309800973877, 0.0, -0.5),\n                        (10, 0.674589841540707, 1.0, 0.5)],\n                       [(4, 5), (8, 9)],\n                       [(5, 6),\n                        (4, 0.4940579672533128, 0.0, 0.5),\n                        (8, 0.3291523396328097, 0.0, 0.5),\n                        (9, 0.9877837715850091, -6.661338147750939e-16, 0.5)],\n                       [(4, 8),\n                        (9, 10),\n                        (5, 0.5, 1.0, 0.5),\n                        (6, 0.000257731393613625, 0.0, -0.5000000000000002)],\n                       [(5, 9), (10, 0.49999999999999956, 0.0, 0.5), (4, 0.0, 1.0, 0)],\n                       [(6, 10), (9, 0.1871672777696409, 0.0, -0.5000000000000002), (4, 5)],\n                       [(8, 9),\n                        (4, 0.4995965996049643, 1.0, -0.5),\n                        (5, 0.9995965996049643, -2.220446049250313e-14, 0.5000000000000111),\n                        (6, 0.5000000000000003, 1.0, -0.5),\n                        (10, 0.18450505978277265, 1.0, 0.5)],\n                       [(4, 8), (5, 6), (9, 0.18716727776964115, 0.0, 0.5)],\n                       [(9, 10),\n                        (5, 0.9732593325653318, 0.0, 0.5),\n                        (6, 0.4999999999999994, 1.0, 0.5),\n                        (8, 0.4990118679928877, 0.0, 0.5),\n                        (4, 0.025933860165469717, 1.0, 0.5)],\n                       [(5, 9), (6, 10)],\n                       [(8, 9),\n                        (6, 0.1773057395529836, 0.0, -0.4216452752443307),\n                        (5, 0.5008068072691987, 0.0, 0.5),\n                        (10, 0.602764719237668, -2.220446049250313e-16, 0.24105859635036891)],\n                       [(6, 2),\n                        (4, 5),\n                        (10, 11),\n                        (8, 0.4990118679928877, 0.0, -0.5),\n                        (9, 0.2398990199947708, 0.0, -0.5789541976490555)],\n                       [(8, 9),\n                        (4, 0.5, 1.0, -0.5),\n                        (5, 0.6139518098922928, 4.440892098500626e-16, 0.583937470962544),\n                        (10, 0.7812646514021274, 0.0, 0.9595069682307824),\n                        (6, 0.8439229341407215, -0.9999999999999998, 0.07215057771742672),\n                        (11, 0.34615445441753034, 0.0, -0.7912592244739658),\n                        (2, 0.5683578105795405, 0.0, -0.878625289122293)],\n                       [(4, 5),\n                        (9, 0.26010688228278145, 0.0, -0.5762100968175856),\n                        (8, 0.0, 0.88045028870981, 0)],\n                       [(9, 10), (5, 0.9549431959582095, 0.0, -0.9483011140282682), (4, 0.0, 1.0, 0)],\n                       [(5, 6),\n                        (10, 0.0, -0.6085778157287907, 0),\n                        (9, 0.3793257511260211, 0.0, 0.004785426884094335)],\n                       [(10, 11),\n                        (8, 9),\n                        (6, 0.6798126636824355, 0.0, 0.6905942260383984),\n                        (5, 0.3636111207350248, 0.0, -0.24363047915127667)],\n                       [(6, 2),\n                        (4, 5),\n                        (9, 0.1728700707127322, 0.0, -0.16027504198763243),\n                        (10, 0.4999999999999997, 1.0, -0.6809969084851055),\n                        (11, 0.565849320070786, 0.0, 0.8639427624792817),\n                        (8, 0.4999999999999998, 1.0, -0.011037258340573564)],\n                       [(9, 10),\n                        (5, 0.5747492210962788, 0.0, 0.8447435237539285),\n                        (6, 0.4197402158362804, 1.0, -0.9956919949459819),\n                        (2, 0.42068674081551494, 0.0, -0.8412061211954147),\n                        (4, 2.2204460492503128e-16, 1.0, -0.36923311612343257)],\n                       [(5, 6),\n                        (10, 0.49999999999999845, 0.0, -0.18099690848510486),\n                        (9, 0.9245508650264845, 0.0, -0.5071351431892186)],\n                       [(10, 11),\n                        (8, 9),\n                        (6, 0.3406064361562491, 0.0, 0.01909037280405096),\n                        (5, 0.0561859864159284, 0.0, -0.34528257114777294)],\n                       [(6, 2),\n                        (4, 5),\n                        (9, 0.5000000000000001, 0.0, 0.34333371597516305),\n                        (10, 0.4999999999999998, 1.0, -0.878749331436373),\n                        (8, 0.5, 1.0, 0.9889627416594262)],\n                       [(9, 10),\n                        (5, 0.547416249108965, 0.0, -0.5898962496410278),\n                        (6, 0.18598715661621895, 1.0, -0.9542300584463328),\n                        (2, 0.46193265931565103, 1.1102230246251565e-16, -0.24595652224492492),\n                        (4, 0.0, 1.0, 0)],\n                       [(5, 6),\n                        (10, 0.5, 0.0, -0.6287493314363717),\n                        (9, 0.3485609400244275, 0.0, 0.843333715975163)],\n                       [(10, 11),\n                        (8, 9),\n                        (6, 0.30175441391238306, 0.0, -0.2823792512015486),\n                        (5, 0.7392621381122314, 0.0, 0.2020183936235962)],\n                       [(6, 2),\n                        (4, 5),\n                        (9, 0.7500000000000004, 0.0, 0.843333715975163),\n                        (8, 0.49999999999999956, 1.0, -0.6605683991761877)],\n                       [(9, 10),\n                        (5, 0.272596887524651, 0.0, 0.7468708013207133),\n                        (6, 0.053622665477313765, 1.0, 0.7543172285314794),\n                        (2, 0.8029611387137003, 0.0, 0.16269843839489745),\n                        (4, 0.0, 1.0, 0)],\n                       [(5, 6),\n                        (10, 0.620275804748211, -2.220446049250313e-16, 0.3712506685636283),\n                        (9, 0.23117431994059623, 0.0, 0.3433337159751631)],\n                       [(10, 11),\n                        (8, 9),\n                        (6, 0.6852836122352373, 0.0, -0.01730515745893957),\n                        (5, 0.5582935812841525, 0.0, -0.5743705118177669)],\n                       [(6, 2),\n                        (4, 5),\n                        (9, 0.6245911673988008, 0.0, -0.656666284024837),\n                        (10, 0.5, 0.0, -0.12874933143637168),\n                        (11, 0.6713639237535922, 0.0, 0.9367827016359593),\n                        (8, 0.0, 0.39456853286863147, 0)],\n                       [(9, 10),\n                        (11, 7),\n                        (5, 0.8681490545286747, 0.0, 0.37250008740711316),\n                        (6, 0.42773805663426384, 1.0, 0.24338650820113772),\n                        (2, 0.881708884870647, 0.0, -0.12980371729230367),\n                        (4, 0.5, 1.0, -0.9281148643661068)],\n                       [(5, 6),\n                        (9, 0.5, 0.0, -0.15666628402483695),\n                        (10, 0.14044773223521692, 0.0, -0.9323335712933396),\n                        (11, 0.8439229341407215, 0.0, 0.3513828295218477),\n                        (7, 0.5683578105795405, 1.0, -0.878625289122293)],\n                       [(9, 10),\n                        (6, 0.11927728230983443, 0.0, 0.1038399054634882),\n                        (5, 0.6515066280755871, 0.0, -0.4842742492700203)],\n                       [(6, 2), (4, 5), (10, 0.9549431959582095, -2.220446049250313e-16, 0.3264557848361065)],\n                       [(10, 11),\n                        (5, 0.0, 0.40725484024047837, 0),\n                        (6, 0.7621154011768825, 0.0, 0.7094048462775937),\n                        (4, 0.5000000000000001, 8.881784197001252e-16, -0.9281148643661046),\n                        (2, 0.6174170730099287, 1.1102230246251565e-16, -0.20873821911732637)],\n                       [(5, 6),\n                        (10, 0.3636111207350248, 0.0, 0.031126419713097542),\n                        (11, 0.6798126636824355, 0.0, 0.7329391812008756)],\n                       [(9, 10),\n                        (11, 7),\n                        (5, 0.5, 0.0, -0.1217685362550589),\n                        (6, 0.7571708757236936, 0.0, 0.459384859368724)],\n                       [(4, 5),\n                        (6, 2),\n                        (10, 0.5747492210962788, 0.0, 0.11950042261830335),\n                        (11, 0.4197402158362804, 5.551115123125783e-17, 0.41922540218525656),\n                        (7, 0.4206867408155152, 1.0, -0.8412061211954147)],\n                       [(10, 11),\n                        (2, 0.24679677437537367, 0.0, -0.7565306828698788),\n                        (4, 0.6218015137737606, 0.0, -0.7153710350390152),\n                        (5, 0.041748308476002194, 0.0, -0.6217685362550589),\n                        (6, 0.5323582538647063, 0.0, -0.29639077996576635)],\n                       [(2, 0),\n                        (4, 1),\n                        (5, 6),\n                        (10, 0.056185986415928424, 0.0, -0.07052567228339868),\n                        (11, 0.3406064361562493, 0.0, 0.4044430344352232)],\n                       [(9, 10),\n                        (11, 7),\n                        (5, 0.5, 0.0, -0.1217685362550589),\n                        (4, 0.4999999999999998, 0.0, -0.35127986527513944),\n                        (2, 0.5, 0.0, -0.8523760134876583),\n                        (0, 0.3040867239846962, 0.0, -0.5000000000000002),\n                        (1, 0.3040867239846962, 0.0, -0.5000000000000002)],\n                       [(5, 4),\n                        (6, 2),\n                        (10, 0.547416249108965, 0.0, 0.6848606492233458),\n                        (11, 0.185987156616219, 0.0, -0.6222365343143934),\n                        (7, 0.46193265931565114, 1.0, -0.2459565222449247)],\n                       [(10, 11),\n                        (2, 0.8040867239846965, 5.551115123125783e-17, 0.14762398651234138),\n                        (4, 0.8040867239846965, 0.0, 0.6487201347248606)],\n                       [(2, 0),\n                        (4, 1),\n                        (10, 0.7392621381122315, -2.220446049250313e-16, 0.47677529248797157),\n                        (11, 0.3017544139123831, 0.0, -0.2940873415591767)],\n                       [(9, 10),\n                        (11, 7),\n                        (4, 0.7499999999999999, 1.0, 0.6487201347248606),\n                        (2, 0.7499999999999999, 1.0, 0.14762398651234143),\n                        (0, 0.5, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5)],\n                       [(5, 4),\n                        (6, 2),\n                        (10, 0.27259688752465094, -2.220446049250313e-16, 0.021627700185087306),\n                        (11, 0.053622665477313765, 0.0, 0.6692161787077946),\n                        (7, 0.8029611387137003, 1.0, 0.1626984383948975)],\n                       [(10, 11),\n                        (2, 0.5, 0.0, -0.8523760134876583),\n                        (4, 0.5, -2.220446049250313e-16, -0.3512798652751392),\n                        (5, 0.5, 0.0, 0.4706336381737639),\n                        (6, 0.5, 0.0, 0.08293728198291522)],\n                       [(2, 0),\n                        (4, 1),\n                        (10, 0.5582935812841526, -2.220446049250313e-16, -0.29961361295339195),\n                        (11, 0.6852836122352373, 0.0, 0.44083856469821375)],\n                       [(9, 10),\n                        (11, 7),\n                        (4, 0.25, 1.0, 0.6487201347248606),\n                        (2, 0.25000000000000006, 1.0, 0.14762398651234143)],\n                       [(5, 4),\n                        (6, 2),\n                        (10, 0.8681490545286747, 0.0, -0.3527430137285119),\n                        (11, 0.4277380566342639, 0.0, 0.18014689903813697),\n                        (9, 0.5, 0.0, -0.35129294709245507),\n                        (7, 0.881708884870647, 1.0, -0.12980371729230367)],\n                       [(10, 11),\n                        (2, 0.5, 1.0, -0.8523760134876583),\n                        (4, 0.5, 1.0, -0.35127986527513944),\n                        (5, 0.5, 0.029366361826235643, -0.5293663618262361),\n                        (6, 0.5, 0.41706271801708494, -0.9170627180170849)],\n                       [(2, 0),\n                        (4, 1),\n                        (10, 0.6515066280755871, 0.0, -0.20951735040564579),\n                        (11, 0.11927728230983443, 0.0, -0.680306498224214)],\n                       [(9, 10),\n                        (11, 7),\n                        (0, 1.0, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5),\n                        (2, 0.5, 0.35237601348765857, 0.14762398651234143),\n                        (4, 0.5, -0.14872013472486056, 0.6487201347248606)],\n                       [(10, 0.0, 0.5515581011636237, 0),\n                        (11, 0.7621154011768825, 0.0, 0.7141285609616803),\n                        (9, 0.5000000000000001, 4.440892098500626e-16, 0.6487070529075425),\n                        (7, 0.6174170730099287, 1.0, -0.20873821911732615)],\n                       [(10, 11)],\n                       [(10, 0.5, 1.0, 0.2972916235324614),\n                        (11, 0.7571708757236936, 0.0, 0.9641485478705505)],\n                       [(9, 10), (11, 7)],\n                       [(7, 0.24679677437537367, 0.0, -0.7565306828698788),\n                        (9, 0.6218015137737605, 0.0, 0.4359632235804547),\n                        (10, 0.041748308476002194, -0.9999999999999998, -0.20270837646753836),\n                        (11, 0.5323582538647061, 0.0, -0.2800758127949594)],\n                       [(7, 3), (9, 12), (10, 11)],\n                       [(10, 0.5, 1.0, 0.2972916235324614),\n                        (9, 0.5, 0.0, 0.07187205381657874),\n                        (7, 0.4999999999999998, 1.0, -0.8523760134876583),\n                        (3, 0.3040867239846962, 0.0, -0.5000000000000002),\n                        (12, 0.3040867239846962, 1.0, -0.5000000000000002)],\n                       [(10, 9), (11, 7)],\n                       [(7, 0.8040867239846965, 5.551115123125783e-17, 0.14762398651234138),\n                        (9, 0.8040867239846965, 4.440892098500626e-16, -0.9281279461834213),\n                        (11, 0.0, 0.8953840330325828, 0)],\n                       [(7, 3), (9, 12)],\n                       [(9, 0.75, 4.440892098500626e-16, -0.9281279461834213),\n                        (7, 0.75, 5.551115123125783e-17, 0.14762398651234138),\n                        (3, 0.5, 0.0, 0.5),\n                        (12, 0.5, 1.0, 0.5)],\n                       [(10, 9), (11, 7)],\n                       [(7, 0.5, 0.0, -0.8523760134876583),\n                        (9, 0.5, 4.440892098500626e-16, -0.9281279461834213),\n                        (10, 0.5, 0.0, -0.110306202038716),\n                        (11, 0.5, 0.0, 0.23598015828894137)],\n                       [(7, 3), (9, 12)],\n                       [(9, 0.25000000000000006, 0.0, 0.07187205381657868),\n                        (7, 0.25, 0.0, 0.14762398651234143)],\n                       [(10, 9), (11, 7)],\n                       [(9, 0.5, -2.220446049250313e-16, 0.07187205381657874),\n                        (7, 0.5, 1.0, -0.8523760134876583),\n                        (11, 0.5, 0.26401984171105863, 0.23598015828894137),\n                        (10, 0.5, -0.3896937979612838, -0.110306202038716)],\n                       [(9, 12), (7, 3)],\n                       [(12, 1.0, 0.0, 0.0),\n                        (3, 1.0, 0.0, 0.5),\n                        (7, 0.5, 0.35237601348765857, 0.14762398651234143),\n                        (9, 0.4999999999999998, 0.42812794618342176, -0.9281279461834218)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],\n           \'qubits\': [(2, 6),\n                      (3, 4),\n                      (3, 6),\n                      (3, 7),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 5)]},\n (13, 5): {\'method\': \'paired binary counts + recursive coherent decoders\',\n           \'moments\': [[(4, 0.569332469519003, 0.0, 0.5),\n                        (6, 0.1695457497514563, 0.0, 0.5),\n                        (10, 0.8851758861264409, 0.0, 0.5),\n                        (5, 0.5893103323535331, 1.0, -0.5),\n                        (2, 0.4652411656670723, 0.0, -0.020153833182791514),\n                        (11, 0.34609627978820595, 1.0, -0.5573948904901638),\n                        (0, 0.5, 0.0, 0.5),\n                        (1, 0.5, 1.0, 0.5),\n                        (7, 0.4652411656670723, 0.0, -0.020153833182791514),\n                        (3, 0.5, 0.0, 0.5),\n                        (12, 0.5, 1.0, 0.5)],\n                       [(4, 8), (6, 10), (5, 9)],\n                       [(4, 0.8644613585144227, 0.0, 0.5),\n                        (5, 0.8562023004789405, 0.0, -0.5),\n                        (8, 0.34796383624416055, 0.0, 0.5),\n                        (6, 0.583132062570996, 0.0, 0.5),\n                        (10, 0.9217652461188455, 1.0, -0.5000000000000002)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.1027320913617671, 0.0, 0.5),\n                        (8, 0.8479638362441597, 0.0, -0.5000000000000002),\n                        (5, 0.9241355121124453, 0.0, -0.5),\n                        (9, 10)],\n                       [(4, 8), (5, 6), (10, 0.464092919728691, 0.0, 0.5)],\n                       [(5, 0.9406843667808629, 0.0, 0.5),\n                        (6, 0.3728714828738406, 0.0, 0.5),\n                        (8, 0.5000000000000027, 0.0, 0.5)],\n                       [(5, 9), (6, 10)],\n                       [(5, 0.8264941771064476, 0.0, -0.5000000000000002),\n                        (9, 0.5, 0.0, 0.5),\n                        (6, 0.5543622980045821, 0.0, -0.5),\n                        (10, 0.47680910051200187, 0.0, -0.5)],\n                       [(4, 5), (8, 9)],\n                       [(4, 8), (5, 0.18806105625269726, 0.0, 0.5), (9, 0.5, 0.0, 0.5)],\n                       [(5, 6), (9, 10), (8, 0.0, 1.0, 0)],\n                       [(9, 0.5, 1.0, -0.5), (6, 0.5543622980045824, 1.0, 0.5)],\n                       [(5, 9), (6, 10)],\n                       [(4, 5),\n                        (8, 9),\n                        (6, 0.5590176409888397, 8.881784197001252e-16, 0.5188501950123972),\n                        (10, 0.08255256792708421, 0.0, 0.6924045197900228)],\n                       [(6, 2),\n                        (10, 11),\n                        (4, 0.0, -0.9423463578432312, 0),\n                        (5, 0.5, 0.0, -0.7167835017790345),\n                        (9, 0.47722440058003673, 1.0, 0.48961966994850625)],\n                       [(4, 5),\n                        (8, 9),\n                        (6, 0.6205477744107246, 0.0, -0.9952957373516547),\n                        (10, 0.4653211182942605, 0.0, 0.23541166555792326),\n                        (2, 0.5, 0.0, -0.5201538331827915),\n                        (11, 0.5295069129729826, 1.0, -0.5893209314959036)],\n                       [(5, 0.0, 0.48915363147939694, 0), (9, 0.48619895433086263, 1.0, 0.6796753570576566)],\n                       [(5, 6), (9, 10)],\n                       [(5, 0.7299176310680837, 0.0, 0.34956718854230506),\n                        (6, 0.5, 2.220446049250313e-16, -0.4767084811652602),\n                        (9, 0.44396062653016743, 1.0, -0.4905293179699852),\n                        (10, 0.6827527082828264, 0.0, 0.8771001587767535)],\n                       [(4, 5), (6, 2), (8, 9), (10, 11)],\n                       [(5, 0.5708700815399964, 0.0, -0.9397435049447465),\n                        (6, 0.13056868187156476, 0.0, 0.28444759017291465),\n                        (9, 0.37803029493452756, 1.0, -0.9905293179699852),\n                        (10, 0.10695351794258778, 0.0, 0.37710015877675884),\n                        (4, 0.5, 0.0, -0.2961450645437129),\n                        (2, 0.4479507936854727, 0.0, 0.3434400796687824),\n                        (8, 0.0, -0.8237733106920035, 0),\n                        (11, 0.4066809502355106, 1.0, 0.8713162318815462)],\n                       [(5, 6), (9, 10)],\n                       [(5, 0.354403963738168, 0.0, 0.3616763590209999),\n                        (6, 0.12328076625869655, 0.0, -0.9629348916653482),\n                        (9, 0.6406239893192525, 0.0, 0.5094706820300199),\n                        (10, 0.46644240507378737, 0.0, -0.5136380272512597)],\n                       [(4, 5), (6, 2), (8, 9), (10, 11)],\n                       [(6, 0.31277527658165183, 1.0, 0.5370651083346518),\n                        (9, 0.44253549868521186, -0.9999999999999998, 0.009470682030020161),\n                        (10, 0.19782660459849952, 0.0, -0.9976275523079159),\n                        (2, 0.2306177640248532, 0.0, -0.41529275648922015),\n                        (4, 0.5, 1.0, 0.7038549354562866),\n                        (11, 0.5492710367616964, 1.0, 0.4063504694497735)],\n                       [(5, 6), (9, 10)],\n                       [(6, 0.7520450774165834, 0.0, -0.9629348916653484),\n                        (9, 0.18474661797413353, 0.0, -0.44652690098876446),\n                        (5, 0.5, 8.881784197001252e-16, -0.9419648322541685),\n                        (10, 0.7684689111399088, -4.440892098500626e-16, -0.49845363090599304)],\n                       [(6, 2), (8, 9), (4, 5), (10, 11)],\n                       [(5, 0.5, 8.881784197001252e-16, -0.9419648322541685),\n                        (6, 0.22588638999651067, 1.0, -0.4629348916653484),\n                        (9, 0.5432475928713613, 1.0, -0.4441510317020585),\n                        (10, 0.5501391504896241, 0.0, 0.23018891641671718),\n                        (2, 0.39222583923283394, 0.0, -0.04993017182812021),\n                        (8, 0.4999999999999998, 0.0, 0.9136666417234083),\n                        (4, 0.6321947838533057, 1.0, 0.8313904965039676),\n                        (11, 0.8902926131088993, 1.0, 0.025169754041003356)],\n                       [(5, 6), (9, 10)],\n                       [(6, 0.696715986124177, 0.0, -0.4629348916653486),\n                        (9, 0.15727698463740847, 0.0, -0.25721349193504905),\n                        (4, 5),\n                        (10, 0.7052739125587255, 8.881784197001252e-16, -0.9087455915201457)],\n                       [(6, 2),\n                        (8, 9),\n                        (10, 11),\n                        (5, 0.0, -0.5037355019564718, 0),\n                        (4, 0.6771139237982055, 5.551115123125783e-17, 0.41962899482406746)],\n                       [(6, 0.5836097377710109, 0.0, 0.03706510833465143),\n                        (9, 0.5, 1.0, -0.4528996572966504),\n                        (10, 0.25375773065927315, 4.440892098500626e-16, 0.5912544084798552),\n                        (2, 0.6050120054676126, 0.0, 0.4640144763916523),\n                        (8, 0.6680110440239018, 0.0, -0.5863333582765917),\n                        (11, 0.3280609017568143, 0.0, 0.47350959105691115)],\n                       [(5, 6), (9, 10)],\n                       [(5, 0.5, 4.440892098500626e-16, 0.787450401998057),\n                        (6, 0.06412882081148276, 0.0, -0.9629348916653471),\n                        (9, 0.5, 0.0, 0.5471003427033498),\n                        (10, 0.0, -0.5954473892522909, 0)],\n                       [(4, 5), (6, 2), (8, 9), (10, 11)],\n                       [(5, 0.5, 4.440892098500626e-16, 0.787450401998057),\n                        (6, 0.49999999999999956, 0.0, -0.9629348916653484),\n                        (9, 0.6272475297087556, 0.0, -0.7906784058408134),\n                        (2, 0.6668726728064027, 0.0, -0.23156229310064602),\n                        (8, 0.5000000000000003, 0.0, -0.08633335827659172),\n                        (11, 0.6783645703167935, 0.0, 0.9735095910569107)],\n                       [(5, 6), (9, 10)],\n                       [(6, 0.48422238672257323, 1.1102230246251565e-16, -0.41205548749634824),\n                        (9, 0.5879904199402892, 0.0, 0.2820503711541178),\n                        (10, 0.5, 0.0, 0.9605253834400624),\n                        (5, 0.5, 1.0, -0.7630150699344329)],\n                       [(6, 2), (8, 9), (10, 11), (4, 5)],\n                       [(2, 0.7832164982209655, 0.0, -0.4850288081567973),\n                        (5, 0.2714423079803394, 0.0, -0.26301506993443335),\n                        (6, 0.5, 0.0, -0.8710501844455032),\n                        (9, 0.7259252541215333, 0.0, -0.9949922828664874),\n                        (10, 0.25066573639280026, 0.0, -0.5394746165599364),\n                        (4, 0.7832164982209657, 0.0, -0.635369903824329),\n                        (8, 0.4999999999999998, 0.0, 0.4610950397257303)],\n                       [(2, 0), (5, 6), (9, 10), (4, 1)],\n                       [(2, 0.5, 0.0, -0.985028808156797),\n                        (9, 0.5, 0.0, -0.8388903703815074),\n                        (10, 0.7552770370149398, 0.0, 0.4605253834400624),\n                        (5, 0.5, -0.9999999999999998, -0.7630150699344334),\n                        (4, 0.5, 0.0, -0.135369903824329),\n                        (0, 0.3040867239846962, 0.0, -0.5000000000000002),\n                        (1, 0.3040867239846962, 1.0, -0.5000000000000002)],\n                       [(6, 2), (8, 9), (10, 11), (5, 4)],\n                       [(2, 0.8040867239846965, 5.551115123125783e-17, 0.014971191843202447),\n                        (9, 0.45694135055250906, 0.0, 0.16110962961849234),\n                        (10, 0.5, 0.0, 0.9605253834400618),\n                        (11, 0.4952514392902705, 0.0, 0.47579762381668517),\n                        (4, 0.8040867239846965, 0.0, 0.864630096175671),\n                        (8, 0.5, -0.6989342207984972, 0.9610950397257303)],\n                       [(2, 0), (9, 10), (11, 7), (4, 1)],\n                       [(4, 0.75, 0.0, 0.864630096175671),\n                        (9, 0.5, 0.0, 0.6611096296184924),\n                        (10, 0.3552017031280632, 1.1102230246251565e-16, 0.4341360325646554),\n                        (2, 0.75, 1.0, 0.014971191843202447),\n                        (0, 0.5, 0.0, 0.5),\n                        (1, 0.5, 1.0, 0.5),\n                        (11, 0.6205477744107246, 1.0, -0.04155053511319906),\n                        (7, 0.5, 0.0, -0.5201538331827915)],\n                       [(5, 4), (9, 10), (6, 2)],\n                       [(2, 0.5000000000000001, 0.0, -0.9850288081567968),\n                        (4, 0.5, 0.0, 0.864630096175671),\n                        (10, 11),\n                        (5, 0.5, 4.440892098500626e-16, -0.9046233190449158),\n                        (6, 0.5, 0.0, -0.5126584335559858),\n                        (9, 2.2204460492503128e-16, 1.0, -0.7590058516477056)],\n                       [(2, 0),\n                        (4, 1),\n                        (11, 0.5, 5.551115123125783e-17, 0.4770367210731962),\n                        (10, 0.7299176310680837, 4.440892098500626e-16, 0.5160877402092579)],\n                       [(4, 0.25000000000000006, 0.0, -0.135369903824329),\n                        (11, 7),\n                        (2, 0.25000000000000006, 1.0, 0.014971191843202447),\n                        (9, 10)],\n                       [(5, 4),\n                        (6, 2),\n                        (10, 0.5708700815399963, 0.0, -0.19460156630369063),\n                        (11, 0.13056868187156476, 1.0, -0.7618072075886293),\n                        (7, 0.4479507936854727, 0.0, 0.3434400796687824),\n                        (9, 0.5, 1.0, -0.46418577398428873)],\n                       [(4, 0.5, 0.0, -0.135369903824329),\n                        (2, 0.5, 1.0, -0.985028808156797),\n                        (10, 11),\n                        (5, 0.5, 0.40462331904491533, -0.9046233190449153),\n                        (6, 0.5, -0.9873415664440142, 0.4873415664440141)],\n                       [(4, 1),\n                        (2, 0),\n                        (11, 0.12328076625869651, 0.0, -0.00918968942689169),\n                        (10, 0.35440396373816796, 0.0, 0.5039785697305632)],\n                       [(11, 7),\n                        (9, 10),\n                        (1, 1.0, 0.0, 0.0),\n                        (0, 1.0, 0.0, 0.5),\n                        (2, 0.5, 0.4850288081567973, 0.014971191843202725),\n                        (4, 0.4999999999999998, 0.635369903824329, 0.864630096175671)],\n                       [(10, 0.0, 0.5102138624372969, 0),\n                        (11, 0.31277527658165183, 0.0, -0.5091896894268917),\n                        (9, 0.5, 0.0, 0.5358142260157113),\n                        (7, 0.2306177640248532, 0.0, -0.41529275648922015)],\n                       [(10, 11)],\n                       [(10, 0.5, 0.0, 0.3178336234430276),\n                        (11, 0.7520450774165834, 0.0, -0.009189689426892578)],\n                       [(9, 10), (11, 7)],\n                       [(10, 0.5, 0.0, 0.31783362344302835),\n                        (11, 0.22588638999651067, 0.0, 0.4908103105731081),\n                        (9, 0.6321947838533057, 0.0, 0.6633497870633913),\n                        (7, 0.39222583923283394, 0.0, -0.04993017182812021)],\n                       [(10, 11)],\n                       [(9, 10), (11, 0.696715986124177, 0.0, 0.4908103105731081)],\n                       [(11, 7), (9, 0.6771139237982056, 1.0, 0.2515882853834913)],\n                       [(11, 0.583609737771011, 1.0, 0.9908103105731074),\n                        (7, 0.6050120054676126, 0.0, 0.4640144763916523)],\n                       [(10, 11)],\n                       [(11, 0.06412882081148273, 0.0, -0.009189689426890357),\n                        (10, 0.5, 0.0, 0.08468288723432948)],\n                       [(11, 7), (9, 10)],\n                       [(10, 0.5, 0.0, 0.08468288723432948),\n                        (11, 0.49999999999999956, 1.0, -0.009189689426891912),\n                        (7, 0.6668726728064027, 0.0, -0.23156229310064602)],\n                       [(10, 11)],\n                       [(10, 0.5, 0.0, -0.3648516408331792),\n                        (11, 0.48422238672257323, 0.0, 0.5416897147421078)],\n                       [(9, 10), (11, 7)],\n                       [(7, 0.7832164982209655, 0.0, -0.4850288081567973),\n                        (9, 0.7832164982209657, 5.551115123125783e-17, 0.1965893867350954),\n                        (10, 0.27144230798033936, 0.0, 0.13514835916682166),\n                        (11, 0.5, 1.0, 0.08269501779295335)],\n                       [(7, 3), (9, 12), (10, 11)],\n                       [(10, 0.5, 0.0, 0.6351483591668199),\n                        (9, 0.5, 0.0, 0.696589386735095),\n                        (7, 0.5, 0.0, -0.985028808156797),\n                        (3, 0.3040867239846962, 0.0, -0.5000000000000002),\n                        (12, 0.3040867239846962, 1.0, -0.5000000000000002)],\n                       [(10, 9), (11, 7)],\n                       [(7, 0.8040867239846965, 5.551115123125783e-17, 0.014971191843202447),\n                        (9, 0.8040867239846965, 0.0, 0.696589386735095),\n                        (10, 0.0, 0.6951213699499467, 0)],\n                       [(7, 3), (9, 12)],\n                       [(9, 0.7499999999999999, 0.0, 0.696589386735095),\n                        (7, 0.75, 1.0, 0.014971191843202447),\n                        (3, 0.5, 0.0, 0.5),\n                        (12, 0.5, 1.0, 0.5)],\n                       [(10, 9), (11, 7)],\n                       [(7, 0.5000000000000001, 0.0, -0.9850288081567968),\n                        (9, 0.5, 0.0, -0.3034106132649046),\n                        (10, 0.5, -0.9999999999999998, 0.47187797822724936),\n                        (11, 0.5, 0.0, 0.4410867686824704)],\n                       [(7, 3), (9, 12)],\n                       [(9, 0.25, 0.0, 0.696589386735095),\n                        (7, 0.25000000000000006, 1.0, 0.014971191843202447)],\n                       [(10, 9), (11, 7)],\n                       [(9, 0.5, 0.0, -0.3034106132649046),\n                        (7, 0.5, 1.0, -0.985028808156797),\n                        (10, 0.5, 0.028122021772750916, 0.4718779782272491),\n                        (11, 0.5, 0.05891323131753001, -0.5589132313175298)],\n                       [(9, 12), (7, 3)],\n                       [(12, 1.0, 0.0, 0.0),\n                        (3, 1.0, 0.0, 0.5),\n                        (7, 0.5, 0.4850288081567973, 0.014971191843202725),\n                        (9, 0.5, 0.803410613264905, 0.696589386735095)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],\n           \'qubits\': [(2, 6),\n                      (3, 4),\n                      (3, 6),\n                      (3, 7),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 5)]},\n (13, 6): {\'method\': \'paired binary counts + recursive coherent decoders\',\n           \'moments\': [[(4, 0.4501939478585324, 0.0, 0.5),\n                        (5, 0.44772204076930416, 0.0, 0.5),\n                        (9, 0.49890701820126315, 0.0, -0.5),\n                        (6, 0.2968525986784217, 0.0, 0.5),\n                        (10, 0.32688298887505485, 0.0, -0.5000000000000002),\n                        (2, 0.4956530884318588, 5.551115123125783e-17, 0.259121498867743),\n                        (11, 0.2961278032423518, 0.0, -0.37072357629548036),\n                        (7, 0.4956530884318588, 5.551115123125783e-17, 0.259121498867743),\n                        (0, 0.5, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5),\n                        (3, 0.5, 0.0, 0.5),\n                        (12, 0.5, 0.0, 0.5)],\n                       [(4, 8), (5, 9), (6, 10)],\n                       [(4, 0.5498060521414676, 0.0, 0.5),\n                        (5, 0.7514161356970753, 0.0, -0.5),\n                        (9, 0.528747685665836, 0.0, 0.5),\n                        (6, 0.4099536350801986, 0.0, 0.5),\n                        (10, 0.6956068135166766, 0.0, 0.5)],\n                       [(4, 5), (8, 9)],\n                       [(8, 0.9786577811306381, 0.0, 0.5),\n                        (5, 0.4179130342668791, 0.0, -0.5000000000000002),\n                        (9, 0.8442678290297327, 0.0, -0.5)],\n                       [(4, 8), (5, 6), (9, 10)],\n                       [(5, 0.7215147296373279, 0.0, 0.5),\n                        (9, 0.6742556384235301, 0.0, 0.5),\n                        (6, 0.278710499924302, 0.0, 0.5),\n                        (10, 0.45282643258460314, 0.0, 0.5),\n                        (8, 0.47865778113063806, 0.0, 0.5)],\n                       [(5, 9), (6, 10)],\n                       [(5, 0.783830865607837, 0.0, -0.5),\n                        (9, 0.11983442054539162, 0.0, -0.5),\n                        (10, 0.7137922747629629, 0.0, -0.5),\n                        (6, 0.3033248226376897, 0.0, -0.5000000000000002)],\n                       [(4, 5), (8, 9)],\n                       [(9, 10), (4, 0.5, 0.0, 0.5), (8, 0.5, 0.0, -0.5), (5, 0.08774248146714281, 0.0, 0.5)],\n                       [(4, 8), (5, 6), (9, 0.5, 0.0, -0.5), (10, 0.9726170148740435, 0.0, -0.5)],\n                       [(5, 0.5323399770960722, 0.0, -0.5),\n                        (6, 0.4058078462486393, 0.0, 0.5),\n                        (4, 0.5, 0.0, 0.5)],\n                       [(5, 9), (6, 10)],\n                       [(8, 9),\n                        (5, 0.5, 0.0, -0.5),\n                        (6, 0.5761969458326828, 0.0, -0.7903707519798902),\n                        (10, 0.14168689371770468, 0.0, 0.4999999999999998)],\n                       [(4, 5),\n                        (6, 2),\n                        (10, 11),\n                        (8, 0.5051488971691418, 5.551115123125783e-17, 0.38516684131825535),\n                        (9, 0.8701833787352258, 0.0, 1.0)],\n                       [(8, 9),\n                        (5, 0.5679894885929349, 0.0, -0.53489185588334),\n                        (10, 0.4999999999999998, 0.0, -0.2596332425295489),\n                        (6, 0.344882952068414, 0.0, 0.09860863229426092),\n                        (11, 0.5, 0.0, -0.8707235762954806),\n                        (2, 0.4925777780083309, 0.0, 0.5072045568580279)],\n                       [(4, 5), (9, 10), (8, 0.22783776220036114, 0.0, 0.8851668413182554)],\n                       [(5, 0.33274203901299115, 0.0, 0.5727060952313592),\n                        (9, 0.4949708987340448, 0.0, 0.067114646892253),\n                        (10, 0.550755412437784, 0.0, -0.09655868968129022)],\n                       [(5, 6), (8, 9), (10, 11)],\n                       [(5, 0.9294880475869709, 0.0, 0.7815507862063753),\n                        (6, 0.9291764376708231, 5.551115123125783e-17, 0.1935651841176344),\n                        (9, 0.5033793022373012, 0.0, -0.4328853531077468),\n                        (10, 0.4999999999999998, 0.0, -0.6947798655138218),\n                        (11, 0.340073632931319, 0.0, -0.2109036333971741),\n                        (8, 0.2906009388508253, 0.0, 0.3851668413182551)],\n                       [(4, 5), (6, 2), (9, 10)],\n                       [(10, 11),\n                        (5, 0.9764886499665415, 0.0, 0.3810952216164627),\n                        (6, 0.29149057155166774, 0.0, 0.08694866760704295),\n                        (8, 9),\n                        (2, 0.41571419491182243, 0.0, 0.41360830056593145)],\n                       [(5, 6),\n                        (9, 0.5188671153526303, 0.0, 0.12409642752196826),\n                        (10, 0.33147541570683337, 0.0, -0.19477986551382176),\n                        (8, 0.5063547328191556, 0.0, -0.11483315868174504),\n                        (11, 0.3751563089816202, 0.0, -0.5976229662630799)],\n                       [(5, 0.322056849702119, 0.0, 0.8589537902261015),\n                        (6, 0.418360472891683, 0.0, 0.946154844363222),\n                        (9, 10)],\n                       [(4, 5),\n                        (6, 2),\n                        (9, 0.6548369278761568, 0.0, 0.31340147450262945),\n                        (10, 0.8111326152692739, 0.0, 0.305220134486178)],\n                       [(5, 0.5910955657739062, 0.0, -0.0726490256067458),\n                        (6, 0.5658106523525603, 0.0, -0.7485866865346891),\n                        (8, 9),\n                        (10, 11),\n                        (2, 0.30256758326839167, 0.0, -0.5706246524937872)],\n                       [(5, 6),\n                        (9, 0.06166130247734847, 0.0, 0.18105551804554004),\n                        (10, 0.015313232122055268, 0.0, 0.805220134486176),\n                        (11, 0.5129607164516408, 0.0, -0.7562970243690965)],\n                       [(5, 0.5222942734869767, 0.0, -0.7827361350962951),\n                        (6, 0.6380247240052519, 4.440892098500626e-16, 0.5992452765979166),\n                        (9, 10)],\n                       [(4, 5),\n                        (6, 2),\n                        (9, 0.5518863484824029, 0.0, -0.3433864477493547),\n                        (10, 0.8714469607225119, 0.0, -0.6947798655138229)],\n                       [(5, 0.36360452844948893, 0.0, -0.8902676472456401),\n                        (6, 0.01708551170125294, 0.0, 0.47055996429910185),\n                        (8, 9),\n                        (10, 11),\n                        (2, 0.32590008584322616, 0.0, -0.5618902727160411)],\n                       [(5, 6),\n                        (9, 0.8128647203031829, 0.0, 0.3997508087077936),\n                        (10, 0.8531775343812674, 1.1102230246251565e-16, -0.19477986551382287),\n                        (11, 0.6608132099476296, 0.0, 0.7425767458343033)],\n                       [(5, 0.9940835612187666, -2.220446049250313e-16, 0.15827548287039261),\n                        (6, 0.7382217283247945, 0.0, -0.536895891451671),\n                        (9, 10)],\n                       [(4, 5),\n                        (6, 2),\n                        (9, 0.4788736599175649, 0.0, -0.14457875337894954),\n                        (10, 0.03620506650561751, 0.0, 0.3052201344861767)],\n                       [(5, 0.5303663346437927, 0.0, -0.4154675933511647),\n                        (6, 0.39031863075905954, 0.0, -0.5461149365109592),\n                        (8, 9),\n                        (10, 11),\n                        (2, 0.5100876690078296, 0.0, 0.3944365573706427)],\n                       [(5, 6),\n                        (9, 0.3202664377244359, 5.551115123125783e-17, 0.2934673993833527),\n                        (10, 0.5000000000000002, 0.0, -0.6947798655138229),\n                        (8, 0.5, 0.0, -0.8149083106323869),\n                        (11, 0.4253414728603863, 0.0, -0.7091324900840312)],\n                       [(5, 0.6025816719136099, 0.0, 0.21786516861185573),\n                        (6, 0.6856477314180824, 0.0, -0.9805731308518819),\n                        (9, 10)],\n                       [(4, 5),\n                        (6, 2),\n                        (9, 0.5, -2.220446049250313e-16, -0.34232310803451704),\n                        (10, 0.4457705204849698, 0.0, 0.7712481686278343)],\n                       [(5, 0.8921121044926871, 0.0, -0.3605656187373487),\n                        (6, 0.3385860029956733, 0.0, 0.834899585439536),\n                        (8, 9),\n                        (10, 11),\n                        (2, 0.2347749898956015, 0.0, -0.3542327425401759)],\n                       [(5, 6),\n                        (9, 10),\n                        (11, 0.47411281038999165, 0.0, 0.6259258041344009),\n                        (8, 0.5, 0.6298166212647747, 0.1850916893676126)],\n                       [(5, 0.592875936455191, 0.0, -0.7812516497316166),\n                        (6, 0.44925295079759087, 0.0, -0.2670791463408584),\n                        (11, 7),\n                        (10, 0.5486314612204101, 0.0, -0.9851782296221445)],\n                       [(4, 5),\n                        (6, 2),\n                        (9, 10),\n                        (11, 0.344882952068414, 8.881784197001252e-16, -0.9595787953219059),\n                        (7, 0.4925777780083309, 0.0, 0.5072045568580279)],\n                       [(5, 0.8105365413512657, 0.0, -0.28606545637686276),\n                        (6, 0.6433165441464112, 0.0, 0.24392717923026397),\n                        (4, 0.4999999999999998, 0.0, -0.11529447589771813),\n                        (2, 0.34497587576931366, 0.0, -0.28963794555439826),\n                        (10, 0.33274203901299115, 0.0, 0.037826546397841554)],\n                       [(5, 6), (10, 11)],\n                       [(5, 0.3843844916022808, 0.0, -0.8151523582742632),\n                        (6, 0.5260557718525964, 0.0, 0.324554161357268),\n                        (10, 0.9294880475869709, 0.0, -0.17101814457717412),\n                        (11, 0.9291764376708231, 5.551115123125783e-17, 0.13537775650146772)],\n                       [(4, 5), (6, 2), (9, 10), (11, 7)],\n                       [(6, 0.7720515191095958, 0.0, 0.8332766783071828),\n                        (4, 0.5, 0.0, -0.11529447589771702),\n                        (2, 0.4355845015328854, 0.0, -0.8111689735837979),\n                        (10, 0.9764886499665415, 0.0, 0.22943742001273848),\n                        (11, 0.2914905715516678, 0.0, 0.028761239990876486),\n                        (7, 0.41571419491182243, 0.0, 0.41360830056593145)],\n                       [(5, 6), (10, 11)],\n                       [(5, 0.5, 0.0, -0.6080593237982133),\n                        (6, 0.13993480154011698, 0.0, -0.8797474463598904),\n                        (10, 0.322056849702119, 0.0, -0.24842114859690057),\n                        (11, 0.4183604728916834, 0.0, -0.11203258325294407)],\n                       [(4, 5), (6, 2), (9, 10), (11, 7)],\n                       [(2, 0.3416013957299424, 0.0, -0.7695375446745221),\n                        (4, 0.4546112109132116, 5.551115123125783e-17, 0.18717784331945325),\n                        (5, 0.2299569291469802, 0.0, -0.10805932379821304),\n                        (6, 0.893580808387734, 0.0, 0.41351945773769394),\n                        (10, 0.5910955657739062, 0.0, 0.6831816672359468),\n                        (11, 0.5658106523525602, 5.551115123125783e-17, 0.19322588584914446),\n                        (7, 0.30256758326839167, 0.0, -0.5706246524937872)],\n                       [(2, 0), (4, 1), (5, 6), (10, 11)],\n                       [(5, 0.4999999999999997, 0.0, -0.6080593237982135),\n                        (4, 0.5, 0.0, -0.025710813790364107),\n                        (2, 0.4999999999999998, 0.0, -0.18230509551025165),\n                        (0, 0.30408672398469605, 0.0, -0.5000000000000002),\n                        (1, 0.30408672398469605, 0.0, -0.5000000000000002),\n                        (10, 0.5222942734869767, 0.0, -0.6067312232745037),\n                        (11, 0.6380247240052519, 0.0, 0.5410578489817497)],\n                       [(5, 4), (6, 2), (9, 10), (11, 7)],\n                       [(2, 0.8040867239846965, 1.1102230246251565e-16, -0.1823050955102521),\n                        (4, 0.8040867239846965, 0.0, 0.9742891862096354),\n                        (10, 0.36360452844948893, 0.0, -0.4991997111251587),\n                        (11, 0.017085511701252946, 0.0, 0.4123725366829347),\n                        (7, 0.32590008584322616, 0.0, -0.5618902727160411)],\n                       [(2, 0), (4, 1), (10, 11)],\n                       [(4, 0.75, 0.0, 0.9742891862096354),\n                        (2, 0.75, 1.1102230246251565e-16, -0.18230509551025254),\n                        (0, 0.5, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5),\n                        (10, 0.9940835612187666, 0.0, 0.4522571587588089),\n                        (11, 0.7382217283247945, 0.0, 0.40491668093216243)],\n                       [(5, 4), (6, 2), (9, 10), (11, 7)],\n                       [(2, 0.5, 0.0, 0.8176949044897479),\n                        (4, 0.5, 4.440892098500626e-16, 0.974289186209637),\n                        (5, 0.5, 0.0, 0.12659531828740622),\n                        (6, 0.5, 0.0, -0.5841414273784373),\n                        (10, 0.5303663346437927, 0.0, -0.9739997650196348),\n                        (11, 0.39031863075905965, 0.0, 0.3956976358728742),\n                        (7, 0.5100876690078296, 0.0, 0.3944365573706427)],\n                       [(2, 0), (4, 1), (10, 11)],\n                       [(4, 0.25000000000000006, 0.0, -0.025710813790362996),\n                        (2, 0.25000000000000006, 0.0, -0.1823050955102523),\n                        (10, 0.6025816719136099, 0.0, 0.39266747301734495),\n                        (11, 0.6856477314180824, 4.440892098500626e-16, 0.961239441531951)],\n                       [(5, 4), (6, 2), (9, 10), (11, 7)],\n                       [(2, 0.5, 8.881784197001252e-16, 0.817694904489747),\n                        (4, 0.5, 0.0, -0.025710813790362774),\n                        (10, 0.8921121044926871, 0.0, 0.9710982603665494),\n                        (11, 0.3385860029956733, 4.440892098500626e-16, 0.7767121578233691),\n                        (7, 0.2347749898956015, 0.0, -0.3542327425401759),\n                        (5, 0.5, 0.3734046817125938, -0.873404681712594),\n                        (6, 0.5, -0.9158585726215631, -0.5841414273784373)],\n                       [(2, 0), (4, 1), (10, 11)],\n                       [(10, 0.5928759364551912, 0.0, -0.6082157086391826),\n                        (11, 0.44925295079759087, 0.0, 0.6747334260429754),\n                        (0, 1.0, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5),\n                        (2, 0.5, -0.31769490448974746, 0.8176949044897481),\n                        (4, 0.5, 0.5257108137903632, -0.02571081379036322)],\n                       [(9, 10), (11, 7)],\n                       [(10, 0.8105365413512657, 0.0, 0.8965980980060635),\n                        (11, 0.6433165441464112, 0.0, -0.8142602483859023),\n                        (9, 0.5, 0.0, 0.3572907267001517),\n                        (7, 0.34497587576931366, 0.0, -0.28963794555439826)],\n                       [(10, 11)],\n                       [(10, 0.3843844916022807, 0.0, -0.5743150000965365),\n                        (11, 0.5260557718525964, 0.0, 0.2663667337411011)],\n                       [(9, 10), (11, 7)],\n                       [(11, 0.7720515191095958, 8.881784197001252e-16, 0.775089250691015),\n                        (9, 0.5, 0.0, -0.6427092732998481),\n                        (7, 0.4355845015328854, 0.0, -0.8111689735837979)],\n                       [(10, 11)],\n                       [(10, 0.5, 0.0, -0.7814080345725862),\n                        (11, 0.13993480154011695, 0.0, 0.062065126023943096)],\n                       [(9, 10), (11, 7)],\n                       [(7, 0.3416013957299424, 0.0, -0.7695375446745221),\n                        (9, 0.4546112109132118, 0.0, 0.6597630459173223),\n                        (10, 0.22995692914698007, 0.0, 0.7185919654274138),\n                        (11, 0.893580808387734, -2.220446049250313e-16, -0.6446679698784725)],\n                       [(7, 3), (9, 12), (10, 11)],\n                       [(10, 0.4999999999999998, 0.0, -0.781408034572586),\n                        (9, 0.4999999999999998, 0.0, 0.4468743888075053),\n                        (7, 0.4999999999999998, 0.0, -0.18230509551025165),\n                        (3, 0.30408672398469605, 0.0, -0.5000000000000002),\n                        (12, 0.30408672398469605, 0.0, -0.5000000000000002)],\n                       [(10, 9), (11, 7)],\n                       [(7, 0.8040867239846965, 1.1102230246251565e-16, -0.1823050955102521),\n                        (9, 0.8040867239846965, 0.0, 0.4468743888075053)],\n                       [(7, 3), (9, 12)],\n                       [(9, 0.75, 0.0, 0.4468743888075055),\n                        (7, 0.75, 1.1102230246251565e-16, -0.18230509551025254),\n                        (3, 0.5, 0.0, 0.5),\n                        (12, 0.5, 0.0, 0.5)],\n                       [(10, 9), (11, 7)],\n                       [(7, 0.5, 0.0, 0.8176949044897479),\n                        (9, 0.5, 0.0, -0.5531256111924947),\n                        (10, 0.5, -2.220446049250313e-16, 0.4839373233417934),\n                        (11, 0.5, 0.0, -0.642328854994604)],\n                       [(7, 3), (9, 12)],\n                       [(9, 0.25000000000000006, 0.0, 0.44687438880750546),\n                        (7, 0.25000000000000006, 0.0, -0.1823050955102523)],\n                       [(10, 9), (11, 7)],\n                       [(7, 0.5, 8.881784197001252e-16, 0.817694904489747),\n                        (9, 0.5, 0.0, -0.5531256111924947),\n                        (10, 0.5, 0.0160626766582066, 0.4839373233417934),\n                        (11, 0.5, -0.857671145005396, -0.642328854994604)],\n                       [(7, 3), (9, 12)],\n                       [(3, 1.0, 0.0, 0.5),\n                        (7, 0.5, -0.31769490448974746, 0.8176949044897481),\n                        (9, 0.5, -0.9468743888075051, -0.5531256111924949),\n                        (12, 1.0, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],\n           \'qubits\': [(2, 6),\n                      (3, 4),\n                      (3, 6),\n                      (3, 7),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 5)]},\n (14, 3): {\'method\': \'binary count seed + moving six-CZ decoders\',\n           \'moments\': [[(0, 0.500654098647404, 0.0, -0.5),\n                        (5, 0.5460924801300725, 0.0, -0.5),\n                        (1, 0.2952676223543462, 0.0, 0.5),\n                        (6, 0.30754615199876284, 0.0, 0.5),\n                        (10, 0.4118208540116386, 0.0, 0.5),\n                        (11, 0.7990860045271903, 0.0, -0.5),\n                        (2, 0.412184131733575, 0.0, -0.5),\n                        (3, 0.4135610417387456, 0.0, -0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (7, 0.412184131733575, 0.0, -0.5),\n                        (8, 0.4135610417387456, 0.0, -0.5),\n                        (9, 0.5, 0.0, 0.5),\n                        (12, 0.4135610417387456, 0.0, -0.5),\n                        (13, 0.5, 0.0, 0.5)],\n                       [(0, 5), (1, 6), (10, 11)],\n                       [(0, 0.49546733790847824, 0.0, 0.5),\n                        (1, 0.8558465753189465, 0.0, -0.5),\n                        (5, 0.004485221166433276, 0.0, 0.5),\n                        (10, 0.34793618080734534, 0.0, -0.5),\n                        (6, 0.08992166537861615, 0.0, 0.5),\n                        (11, 0.6026574392905439, 0.0, 0.5)],\n                       [(0, 1), (5, 10), (6, 11)],\n                       [(0, 0.5, 0.0, -0.5),\n                        (1, 0.9531281587107934, 0.0, 0.5),\n                        (5, 1.0, 0.0, 0.5),\n                        (6, 0.29370516922839646, 0.0, -0.5),\n                        (10, 0.9226247843065278, 0.0, -0.5),\n                        (11, 0.7928065501808954, 0.0, -0.5)],\n                       [(0, 1), (5, 6), (10, 11)],\n                       [(0, 0.9999999999999999, 2.220446049250313e-16, -0.5000000000000002),\n                        (5, 0.5, 0.0, 0.5),\n                        (1, 1.0, 0.0, -0.5),\n                        (6, 0.949586832724987, 0.0, 0.5),\n                        (10, 0.2809293943810169, 0.0, 0.5),\n                        (11, 0.3008399165007728, 0.0, 0.5)],\n                       [(0, 5), (1, 6), (10, 11)],\n                       [(0, 0.4203607417436471, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5),\n                        (5, 0.6510554317367806, 0.0, -0.5),\n                        (10, 0.5658284044979385, 0.0, 0.5),\n                        (6, 0.9897711558721369, 2.220446049250313e-16, -0.5000000000000002),\n                        (11, 0.9632739550369707, 2.220446049250313e-16, -0.5000000000000002)],\n                       [(0, 1), (5, 10), (6, 11)],\n                       [(0, 0.32994625881989303, 0.0, 0.5),\n                        (5, 0.34169619932071577, 0.0, -0.5),\n                        (1, 0.33533707352777786, 0.0, -0.5),\n                        (6, 0.8906281992022789, 0.0, 0.5),\n                        (11, 0.9449650288787829, 0.0, -0.5)],\n                       [(0, 5), (1, 6), (10, 11)],\n                       [(0, 0.49999999999999994, 0.0, 0.5),\n                        (1, 0.7633065265106983, 0.0, -0.5),\n                        (5, 0.5, 0.0, -0.5),\n                        (10, 1.0, 0.0, 0.5),\n                        (6, 0.31746242475777553, 0.0, -0.5),\n                        (11, 0.784375585596912, 0.0, 0.5)],\n                       [(0, 1), (5, 10), (6, 11)],\n                       [(1, 0.2389850714577341, 0.0, 0.5),\n                        (5, 1.0, 0.0, -0.5),\n                        (6, 0.46129367301539215, 0.0, 0.5),\n                        (10, 1.0, 0.0, 0.5),\n                        (11, 0.6256142167541747, 0.0, 0.5)],\n                       [(0, 1), (5, 6), (10, 11)],\n                       [(1, 0.8967343446268049, 0.0, 0.5),\n                        (0, 1.0, 0.0, 0.5),\n                        (6, 0.4870092274336755, 0.0, -0.5),\n                        (11, 0.2521996563046033, 0.0, -0.5),\n                        (10, 1.0, 0.0, -0.5)],\n                       [(1, 2), (6, 7), (11, 12)],\n                       [(1, 0.5903120017947893, 0.0, -0.5),\n                        (2, 0.6117890887690124, 0.0, -0.5),\n                        (6, 0.5903120017947893, 0.0, -0.5),\n                        (7, 0.6117890887690124, 0.0, -0.5),\n                        (11, 0.3819668494768557, 0.0, 0.5),\n                        (12, 0.6338602364006151, 0.0, -0.5)],\n                       [(0, 1), (5, 6), (10, 11)],\n                       [(1, 0.7075436162554982, 0.0, 0.5),\n                        (0, 1.0, 0.0, 0.5),\n                        (6, 0.7075436162554982, 0.0, 0.5),\n                        (5, 1.0, 0.0, 0.5),\n                        (11, 0.7376490383100945, 0.0, 0.5),\n                        (10, 1.0, 0.0, 0.5)],\n                       [(1, 2), (6, 7), (11, 12)],\n                       [(1, 0.13376589621282342, 0.0, 0.5),\n                        (2, 0.1417314999479891, 0.0, 0.5),\n                        (6, 0.13376589621282342, 0.0, 0.5),\n                        (7, 0.1417314999479891, 0.0, 0.5),\n                        (11, 0.16018463696253962, 0.0, 0.5),\n                        (12, 0.18023932987586394, 0.0, -0.5000000000000002)],\n                       [(0, 1), (5, 6), (10, 11)],\n                       [(1, 0.4425626624366138, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (6, 0.4425626624366138, 0.0, 0.5),\n                        (5, 0.5, 0.0, 0.5),\n                        (11, 0.4422317453870271, 0.0, 0.5),\n                        (10, 0.5, 0.0, 0.5)],\n                       [(1, 2), (6, 7), (11, 12)],\n                       [(1, 0.35241638234956674, 0.0, -0.5),\n                        (2, 0.8746983728132804, 0.0, -0.5),\n                        (6, 0.35241638234956674, 0.0, -0.5),\n                        (7, 0.8746983728132804, 0.0, -0.5),\n                        (11, 0.6666666666666669, 0.0, 0.5),\n                        (12, 0.8040867239846964, 2.220446049250313e-16, -0.5000000000000002)],\n                       [(0, 1), (2, 3), (5, 6), (7, 8), (10, 11), (12, 13)],\n                       [(1, 1.0, 0.0, 0.5),\n                        (2, 0.3819668494768557, 0.0, 0.5),\n                        (3, 0.6338602364006151, 0.0, -0.5),\n                        (6, 1.0, 0.0, 0.5),\n                        (7, 0.3819668494768557, 0.0, 0.5),\n                        (8, 0.6338602364006151, 0.0, -0.5),\n                        (11, 1.0, 0.0, 0.5),\n                        (12, 0.49999999999999994, 0.0, -0.5),\n                        (13, 0.3040867239846963, 0.0, -0.5),\n                        (0, 0.5, 0.0, -0.5),\n                        (5, 0.5, 0.0, -0.5),\n                        (10, 0.5, 0.0, -0.5)],\n                       [(1, 2), (6, 7), (11, 12)],\n                       [(2, 0.7376490383100945, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5),\n                        (7, 0.7376490383100945, 0.0, 0.5),\n                        (6, 1.0, 0.0, 0.5),\n                        (12, 0.8040867239846964, 0.0, 0.5),\n                        (11, 1.0, 0.0, 0.5)],\n                       [(2, 3), (7, 8), (12, 13)],\n                       [(2, 0.16018463696253962, 0.0, 0.5),\n                        (3, 0.18023932987586394, 0.0, -0.5000000000000002),\n                        (7, 0.16018463696253962, 0.0, 0.5),\n                        (8, 0.18023932987586394, 0.0, -0.5000000000000002),\n                        (12, 0.75, 0.0, 0.5),\n                        (13, 0.5, 0.0, 0.5)],\n                       [(1, 2), (6, 7), (11, 12)],\n                       [(2, 0.4422317453870271, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5),\n                        (7, 0.4422317453870271, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (12, 0.5000000000000001, 0.0, -0.5),\n                        (11, 0.5, 0.0, 0.5)],\n                       [(2, 3), (7, 8), (12, 13)],\n                       [(2, 0.6666666666666669, 0.0, 0.5),\n                        (3, 0.8040867239846964, 2.220446049250313e-16, -0.5000000000000002),\n                        (7, 0.6666666666666669, 0.0, 0.5),\n                        (8, 0.8040867239846964, 2.220446049250313e-16, -0.5000000000000002),\n                        (12, 0.25, 0.0, 0.5),\n                        (13, 1.0, 0.0, -0.5)],\n                       [(1, 2), (3, 4), (6, 7), (8, 9), (11, 12)],\n                       [(2, 1.0, 0.0, 0.5),\n                        (3, 0.49999999999999994, 0.0, -0.5),\n                        (4, 0.3040867239846963, 0.0, -0.5),\n                        (7, 1.0, 0.0, 0.5),\n                        (8, 0.49999999999999994, 0.0, -0.5),\n                        (9, 0.3040867239846963, 0.0, -0.5),\n                        (12, 0.5, 0.0, -0.5),\n                        (1, 0.5, 0.0, -0.5),\n                        (6, 0.5, 0.0, -0.5),\n                        (11, 0.5, 0.0, 0.5)],\n                       [(2, 3), (7, 8), (12, 13)],\n                       [(3, 0.8040867239846964, 0.0, 0.5),\n                        (2, 1.0, 0.0, 0.5),\n                        (8, 0.8040867239846964, 0.0, 0.5),\n                        (7, 1.0, 0.0, 0.5),\n                        (12, 0.5000000000000001, 0.0, 0.5)],\n                       [(3, 4), (8, 9)],\n                       [(3, 0.75, 0.0, 0.5), (4, 0.5, 0.0, 0.5), (8, 0.75, 0.0, 0.5), (9, 0.5, 0.0, 0.5)],\n                       [(2, 3), (7, 8)],\n                       [(3, 0.5000000000000001, 0.0, -0.5),\n                        (2, 0.5, 0.0, 0.5),\n                        (8, 0.5000000000000001, 0.0, -0.5),\n                        (7, 0.5, 0.0, 0.5)],\n                       [(3, 4), (8, 9)],\n                       [(3, 0.25, 0.0, 0.5), (4, 1.0, 0.0, -0.5), (8, 0.25, 0.0, 0.5), (9, 1.0, 0.0, -0.5)],\n                       [(2, 3), (7, 8)],\n                       [(3, 0.5, 0.0, -0.5), (8, 0.5, 0.0, -0.5), (2, 0.5, 0.0, 0.5), (7, 0.5, 0.0, 0.5)],\n                       [(3, 4), (8, 9)],\n                       [(3, 0.5000000000000001, 0.0, 0.5), (8, 0.5000000000000001, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (3, 6),\n                      (3, 7),\n                      (3, 8),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (4, 8),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7)]},\n (14, 4): {\'method\': \'paired binary counts + recursive coherent decoders\',\n           \'moments\': [[(5, 0.1492208351548694, 0.0, 0.5),\n                        (9, 0.49364150917817384, 0.0, -0.5000000000000002),\n                        (6, 0.1304149928257203, 0.0, 0.5),\n                        (10, 0.8109564595691234, 0.0, -0.5000000000000002),\n                        (7, 0.4212522357058561, 1.0, 0.5),\n                        (3, 0.4258875989061941, 1.0, 0.5918062908230985),\n                        (12, 0.4258875989061941, 1.0, 0.5918062908230985),\n                        (1, 0.23850197504578505, 1.0, 0.6723366713104195),\n                        (8, 0.23850197504578505, 1.0, 0.6723366713104195),\n                        (0, 0.5, 0.0, 0.5),\n                        (2, 0.5, 0.0, 0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (13, 0.5, 1.0, 0.5)],\n                       [(5, 9), (6, 10), (7, 11)],\n                       [(5, 0.23246105279082324, 0.0, 0.5),\n                        (6, 0.920204551693893, 0.0, 0.5),\n                        (9, 0.9565608181455243, 0.0, 0.5),\n                        (10, 0.5818292564517651, 0.0, 0.5),\n                        (7, 0.07874776429414387, 0.0, -0.5000000000000002)],\n                       [(5, 6), (9, 10)],\n                       [(5, 0.4458120464876009, 0.0, -0.5),\n                        (9, 0.09583805393818556, 0.0, 0.5),\n                        (6, 0.06033211708887333, 0.0, 0.5),\n                        (10, 0.18282003036844155, 0.0, 0.5)],\n                       [(5, 9), (6, 7), (10, 11)],\n                       [(10, 0.9727870547104276, 0.0, -0.5),\n                        (7, 0.5, 0.0, 0.5),\n                        (11, 0.75, 0.0, -0.5),\n                        (9, 0.8943608728307322, 0.0, 0.5),\n                        (5, 0.8666380306139951, 1.0, 0.5)],\n                       [(6, 10), (7, 11)],\n                       [(10, 0.19566498588353884, 0.0, -0.5000000000000002), (5, 6), (11, 0.75, 0.0, -0.5)],\n                       [(9, 10), (5, 0.4229501978568093, 0.0, -0.5000000000000002), (6, 0.5, 0.0, 0.5)],\n                       [(9, 0.24042206046407322, 0.0, -0.5), (6, 7), (10, 0.4941988651660729, 0.0, 0.5)],\n                       [(5, 9), (10, 11), (6, 0.0, 1.0, 0), (7, 0.5, 0.0, -0.5)],\n                       [(6, 10),\n                        (7, 11),\n                        (5, 0.11574275384819255, 0.0, 0.5),\n                        (9, 0.9170234463620259, 1.0, 0.49999999999999994)],\n                       [(5, 6),\n                        (7, 0.2741498684187644, 0.0, 0.13733448444442953),\n                        (9, 10),\n                        (11, 0.602764719237668, 0.0, -0.7589414036496311)],\n                       [(7, 3),\n                        (11, 12),\n                        (6, 0.7308754207007917, 4.440892098500626e-16, 0.5720804852700498),\n                        (9, 0.31225344562382357, 1.0, -0.5),\n                        (10, 0.23989901999477103, 0.0, 0.4210458023509449)],\n                       [(5, 6),\n                        (9, 10),\n                        (7, 0.7812646514021274, 0.0, -0.7369685784544937),\n                        (11, 0.7812646514021274, 0.0, -0.04049303176921759),\n                        (3, 0.34615445441753034, 0.0, -0.7912592244739658),\n                        (12, 0.34615445441753034, 0.0, -0.7912592244739658)],\n                       [(6, 0.26010688228278145, 1.0, -0.3242586920561654),\n                        (10, 0.26010688228278145, 0.0, -0.5762100968175852)],\n                       [(6, 7), (10, 11)],\n                       [(7, 0.0, -0.21562672235823777, 0),\n                        (6, 0.3793257511260211, 1.0, 0.09474578424215485),\n                        (10, 0.3793257511260211, -2.220446049250313e-16, 0.004785426884094779),\n                        (11, 0.0, 0.3914221842712094, 0)],\n                       [(7, 3), (5, 6), (9, 10), (11, 12)],\n                       [(6, 0.17287007071273214, 0.0, 0.2598062531138816),\n                        (7, 0.4999999999999998, 0.0, -0.9845213617998294),\n                        (10, 0.1728700707127321, 0.0, 0.8397249580123674),\n                        (11, 0.4999999999999997, 1.0, -0.6809969084851055),\n                        (3, 0.565849320070786, 0.0, 0.8639427624792817),\n                        (5, 0.4999999999999998, 0.0, -0.10851245294961642),\n                        (9, 0.4999999999999998, 0.0, 0.8914875470503834),\n                        (12, 0.565849320070786, 0.0, 0.8639427624792817)],\n                       [(6, 7), (10, 11)],\n                       [(7, 0.4999999999999985, 0.0, -0.48452136179982874),\n                        (6, 0.9245508650264845, 1.0, 0.6066663543154673),\n                        (10, 0.9245508650264845, -2.220446049250313e-16, 0.4928648568107823),\n                        (11, 0.49999999999999845, 0.0, -0.18099690848510486)],\n                       [(7, 3), (5, 6), (9, 10), (11, 12)],\n                       [(6, 0.5, 0.0, -0.24380250484891386),\n                        (7, 0.4999999999999998, 4.440892098500626e-16, 0.8177262152489031),\n                        (10, 0.5000000000000001, 0.0, 0.34333371597516305),\n                        (11, 0.4999999999999998, 1.0, -0.878749331436373),\n                        (5, 0.5, 0.0, 0.8914875470503836),\n                        (9, 0.5, 0.0, 0.8914875470503836)],\n                       [(6, 7), (10, 11)],\n                       [(7, 0.5, 4.440892098500626e-16, -0.9322737847510956),\n                        (6, 0.3485609400244275, 0.9999999999999999, -0.7438025048489139),\n                        (10, 0.3485609400244275, 0.0, 0.843333715975163),\n                        (11, 0.5, 0.0, -0.6287493314363717)],\n                       [(7, 3), (5, 6), (9, 10), (11, 12)],\n                       [(6, 0.7500000000000004, 1.0, -0.7438025048489139),\n                        (10, 0.7500000000000004, 1.1102230246251565e-16, -0.15666628402483695),\n                        (5, 0.4999999999999995, 0.0, 0.5410186878859977),\n                        (9, 0.49999999999999956, 0.0, -0.4589813121140027)],\n                       [(6, 7), (10, 11)],\n                       [(7, 0.620275804748211, 0.0, -0.9322737847510956),\n                        (6, 0.231174319940596, 1.0, 0.7561974951510861),\n                        (10, 0.23117431994059623, 0.0, -0.656666284024837),\n                        (11, 0.620275804748211, -2.220446049250313e-16, 0.3712506685636283)],\n                       [(7, 3), (5, 6), (9, 10), (11, 12)],\n                       [(3, 0.6713639237535922, 0.0, 0.9367827016359593),\n                        (6, 0.6245911673988008, -0.9999999999999998, -0.24380250484891386),\n                        (7, 0.5, 0.0, 0.5677262152489044),\n                        (10, 0.6245911673988008, 0.0, -0.656666284024837),\n                        (11, 0.5, 0.0, -0.12874933143637168),\n                        (12, 0.6713639237535922, 0.0, 0.9367827016359593),\n                        (5, 1.0, 0.0, 0.13750941078922074),\n                        (9, 1.0, 0.0, 0.13750941078922052)],\n                       [(3, 1), (6, 7), (10, 11), (12, 8)],\n                       [(6, 0.5, -0.9999999999999998, 0.25619749515108614),\n                        (7, 0.14044773223521692, 0.0, 0.7641419753919367),\n                        (10, 0.5, 0.0, -0.15666628402483695),\n                        (11, 0.14044773223521692, 0.0, -0.9323335712933396),\n                        (3, 0.8439229341407215, 0.0, 0.3513828295218477),\n                        (12, 0.8439229341407215, 0.0, 0.3513828295218477),\n                        (1, 0.5683578105795405, 1.0, -0.878625289122293),\n                        (8, 0.5683578105795405, 1.0, -0.878625289122293)],\n                       [(6, 7), (10, 11)],\n                       [(7, 0.9549431959582095, 0.0, -0.9770686684786174),\n                        (11, 0.9549431959582095, -2.220446049250313e-16, 0.3264557848361065),\n                        (6, 0.0, 1.0, 0)],\n                       [(7, 3), (11, 12)],\n                       [(3, 0.6798126636824355, 0.0, 0.7329391812008756),\n                        (7, 0.3636111207350248, 0.0, 0.7276019663983737),\n                        (11, 0.3636111207350248, 0.0, 0.031126419713097542),\n                        (12, 0.6798126636824355, 0.0, 0.7329391812008756)],\n                       [(3, 1), (6, 7), (10, 11), (12, 8)],\n                       [(7, 0.5747492210962788, 4.440892098500626e-16, -0.18402403069642115),\n                        (3, 0.4197402158362804, 5.551115123125783e-17, 0.41922540218525656),\n                        (11, 0.5747492210962788, 0.0, 0.11950042261830335),\n                        (12, 0.4197402158362804, 5.551115123125783e-17, 0.41922540218525656),\n                        (1, 0.4206867408155152, 1.0, -0.8412061211954147),\n                        (6, 2.2204460492503128e-16, 1.0, 0.009705906461377867),\n                        (8, 0.4206867408155152, 1.0, -0.8412061211954147)],\n                       [(7, 3), (11, 12)],\n                       [(3, 0.3406064361562493, 0.0, 0.4044430344352232),\n                        (7, 0.05618598641592863, 0.0, -0.3740501255981221),\n                        (11, 0.056185986415928424, 0.0, -0.07052567228339868),\n                        (12, 0.3406064361562493, 0.0, 0.4044430344352232)],\n                       [(3, 1), (6, 7), (10, 11), (12, 8)],\n                       [(7, 0.547416249108965, 0.0, -0.6186638040913772),\n                        (3, 0.185987156616219, 0.0, -0.6222365343143934),\n                        (11, 0.547416249108965, 0.0, 0.6848606492233458),\n                        (12, 0.185987156616219, 0.0, -0.6222365343143934),\n                        (1, 0.46193265931565114, 1.0, -0.2459565222449247),\n                        (6, 0.0, 1.0, 0),\n                        (8, 0.46193265931565114, 1.0, -0.2459565222449247)],\n                       [(7, 3), (11, 12)],\n                       [(3, 0.3017544139123831, 0.0, -0.2940873415591767),\n                        (7, 0.7392621381122315, 0.0, -0.8267491608267532),\n                        (11, 0.7392621381122315, -2.220446049250313e-16, 0.47677529248797157),\n                        (12, 0.3017544139123831, 0.0, -0.2940873415591767)],\n                       [(3, 1), (6, 7), (10, 11), (12, 8)],\n                       [(7, 0.27259688752465094, 0.0, -0.2818967531296366),\n                        (3, 0.053622665477313765, 0.0, 0.6692161787077946),\n                        (11, 0.27259688752465094, -2.220446049250313e-16, 0.021627700185087306),\n                        (12, 0.053622665477313765, 0.0, 0.6692161787077946),\n                        (1, 0.8029611387137003, 1.0, 0.1626984383948975),\n                        (6, 0.0, 1.0, 0),\n                        (8, 0.8029611387137003, 1.0, 0.1626984383948975)],\n                       [(7, 3), (11, 12)],\n                       [(3, 0.6852836122352373, 0.0, 0.44083856469821375),\n                        (7, 0.5582935812841525, 0.0, -0.6031380662681158),\n                        (11, 0.5582935812841526, -2.220446049250313e-16, -0.29961361295339195),\n                        (12, 0.6852836122352373, 0.0, 0.44083856469821375)],\n                       [(3, 1), (6, 7), (10, 11), (12, 8)],\n                       [(7, 0.8681490545286747, 0.0, 0.3437325329567642),\n                        (3, 0.4277380566342639, 0.0, 0.18014689903813697),\n                        (11, 0.8681490545286747, 0.0, -0.3527430137285119),\n                        (12, 0.4277380566342639, 0.0, 0.18014689903813697),\n                        (1, 0.881708884870647, 1.0, -0.12980371729230367),\n                        (6, 0.5, -0.9999999999999998, -0.5491758417812964),\n                        (10, 0.5, 0.0, -0.35129294709245507),\n                        (8, 0.881708884870647, 1.0, -0.12980371729230367)],\n                       [(7, 3), (11, 12)],\n                       [(3, 0.11927728230983443, 0.0, -0.680306498224214),\n                        (7, 0.6515066280755871, 0.0, 0.48695819627963033),\n                        (11, 0.6515066280755871, 0.0, -0.20951735040564579),\n                        (12, 0.11927728230983443, 0.0, -0.680306498224214)],\n                       [(3, 1), (6, 7), (10, 11), (12, 8)],\n                       [(7, 0.0, 0.5515581011636232, 0),\n                        (3, 0.7621154011768825, 0.0, 0.7141285609616803),\n                        (11, 0.0, 0.5515581011636237, 0),\n                        (12, 0.7621154011768825, 0.0, 0.7141285609616803),\n                        (6, 0.5, 0.0, -0.549175841781294),\n                        (1, 0.6174170730099287, 1.0, -0.20873821911732615),\n                        (10, 0.5000000000000001, 4.440892098500626e-16, 0.6487070529075425),\n                        (8, 0.6174170730099287, 1.0, -0.20873821911732615)],\n                       [(7, 3), (11, 12)],\n                       [(7, 0.5, 0.0, -0.006232829782262694),\n                        (3, 0.7571708757236936, 0.0, 0.9641485478705505),\n                        (11, 0.5, 1.0, 0.2972916235324614),\n                        (12, 0.7571708757236936, 0.0, 0.9641485478705505)],\n                       [(6, 7), (3, 1), (10, 11), (12, 8)],\n                       [(1, 0.24679677437537367, 0.0, -0.7565306828698788),\n                        (6, 0.6218015137737606, 2.220446049250313e-16, -0.3364320124542046),\n                        (7, 0.0417483084760022, 1.0, -0.5062328297822625),\n                        (3, 0.5323582538647061, 0.0, -0.2800758127949594),\n                        (8, 0.24679677437537367, 0.0, -0.7565306828698788),\n                        (10, 0.6218015137737605, 0.0, 0.4359632235804547),\n                        (11, 0.041748308476002194, -0.9999999999999998, -0.20270837646753836),\n                        (12, 0.5323582538647061, 0.0, -0.2800758127949594)],\n                       [(1, 0), (6, 2), (7, 3), (8, 4), (10, 13), (11, 12)],\n                       [(7, 0.5, 0.0, -0.006232829782262694),\n                        (6, 0.5, -2.220446049250313e-16, 0.027659157309670945),\n                        (1, 0.4999999999999998, 1.0, -0.8523760134876583),\n                        (11, 0.5, 1.0, 0.2972916235324614),\n                        (10, 0.5, 0.0, 0.07187205381657874),\n                        (8, 0.4999999999999998, 1.0, -0.8523760134876583),\n                        (0, 0.3040867239846962, 0.0, -0.5000000000000002),\n                        (2, 0.3040867239846962, 0.0, -0.5000000000000002),\n                        (4, 0.3040867239846962, 0.0, -0.5000000000000002),\n                        (13, 0.3040867239846962, 1.0, -0.5000000000000002)],\n                       [(7, 6), (3, 1), (11, 10), (12, 8)],\n                       [(1, 0.8040867239846965, 5.551115123125783e-17, 0.14762398651234138),\n                        (6, 0.8040867239846965, 0.0, -0.972340842690329),\n                        (8, 0.8040867239846965, 5.551115123125783e-17, 0.14762398651234138),\n                        (10, 0.8040867239846965, 4.440892098500626e-16, -0.9281279461834213),\n                        (3, 0.0, 0.8953840330325828, 0),\n                        (12, 0.0, 0.8953840330325828, 0)],\n                       [(1, 0), (6, 2), (8, 4), (10, 13)],\n                       [(1, 0.75, 5.551115123125783e-17, 0.14762398651234138),\n                        (6, 0.75, 1.0, -0.972340842690329),\n                        (10, 0.75, 4.440892098500626e-16, -0.9281279461834213),\n                        (8, 0.75, 5.551115123125783e-17, 0.14762398651234138),\n                        (0, 0.5, 0.0, 0.5),\n                        (2, 0.5, 0.0, 0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (13, 0.5, 1.0, 0.5)],\n                       [(3, 1), (7, 6), (11, 10), (12, 8)],\n                       [(1, 0.5, 0.0, -0.8523760134876583),\n                        (6, 0.5, 0.0, 0.027659157309670945),\n                        (8, 0.5, 0.0, -0.8523760134876583),\n                        (10, 0.5, 4.440892098500626e-16, -0.9281279461834213),\n                        (3, 0.5, 0.0, 0.23598015828894137),\n                        (7, 0.5, 4.440892098500626e-16, 0.5861693446465599),\n                        (11, 0.5, 0.0, -0.110306202038716),\n                        (12, 0.5, 0.0, 0.23598015828894137)],\n                       [(1, 0), (6, 2), (8, 4), (10, 13)],\n                       [(1, 0.25, 0.0, 0.14762398651234143),\n                        (6, 0.25000000000000006, 1.0, -0.972340842690329),\n                        (10, 0.25000000000000006, 0.0, 0.07187205381657868),\n                        (8, 0.25, 0.0, 0.14762398651234143)],\n                       [(3, 1), (7, 6), (11, 10), (12, 8)],\n                       [(1, 0.5, 1.0, -0.8523760134876583),\n                        (6, 0.5, 1.0, 0.027659157309671),\n                        (10, 0.5, 1.1102230246251565e-16, 0.0718720538165768),\n                        (8, 0.5, 1.0, -0.8523760134876583),\n                        (3, 0.5, 0.26401984171105863, 0.23598015828894137),\n                        (7, 0.5, -0.08616934464656056, -0.4138306553534399),\n                        (12, 0.5, 0.26401984171105863, 0.23598015828894137),\n                        (11, 0.5, -0.3896937979612838, -0.110306202038716)],\n                       [(1, 0), (6, 2), (10, 13), (8, 4)],\n                       [(0, 1.0, 0.0, 0.5),\n                        (1, 0.5, 0.35237601348765857, 0.14762398651234143),\n                        (2, 1.0, 0.0, 0.5),\n                        (6, 0.5, -0.527659157309671, -0.972340842690329),\n                        (13, 1.0, 0.0, 0.0),\n                        (4, 1.0, 0.0, 0.5),\n                        (8, 0.5, 0.35237601348765857, 0.14762398651234143),\n                        (10, 0.4999999999999998, 0.42812794618342176, -0.9281279461834218)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],\n           \'qubits\': [(1, 6),\n                      (2, 6),\n                      (3, 5),\n                      (3, 6),\n                      (3, 7),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 5)]},\n (14, 5): {\'method\': \'paired binary counts + recursive coherent decoders\',\n           \'moments\': [[(6, 0.6649284327846141, -1.0, -0.5),\n                        (5, 0.3778099659784764, 0.0, 0.5),\n                        (11, 0.9999404304359252, 0.0, -0.5),\n                        (7, 0.20369585066947749, 0.0, 0.5),\n                        (3, 0.3460962797882059, 0.0, -0.5573948904901638),\n                        (12, 0.3460962797882059, -1.0, -0.5573948904901638),\n                        (8, 0.4652411656670723, 0.0, -0.020153833182791514),\n                        (1, 0.4652411656670723, 0.0, -0.020153833182791514),\n                        (0, 0.5, 0.0, 0.5),\n                        (13, 0.5, -1.0, 0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (2, 0.5, 0.0, 0.5)],\n                       [(6, 10), (5, 9), (7, 11)],\n                       [(6, 0.9297740209165868, 2.220446049250313e-16, -0.5000000000000002),\n                        (5, 0.9765810720440469, 0.0, 0.5),\n                        (9, 0.37339124773516286, 0.0, 0.5),\n                        (7, 0.5740941455895439, 0.0, 0.5),\n                        (11, 0.9999259074782365, -1.0, 0.5)],\n                       [(5, 6), (9, 10)],\n                       [(6, 0.9999943381762997, 0.0, 0.5),\n                        (9, 0.8733912477351629, 0.0, -0.5000000000000002),\n                        (5, 0.20093748739029382, 0.0, 0.5),\n                        (10, 11)],\n                       [(6, 7), (5, 9), (11, 0.4999890307436661, 0.0, 0.5)],\n                       [(6, 0.9999948690424976, 0.0, -0.5),\n                        (9, 0.5, 0.0, 0.5),\n                        (7, 0.31868390382857603, 0.0, 0.5),\n                        (5, 0.0, -0.9651293313589813, -0.534870668641019)],\n                       [(6, 10), (7, 11), (5, 9)],\n                       [(6, 0.8562952046824337, 0.0, -0.5000000000000002),\n                        (10, 0.4999999999999545, 0.0, 0.5),\n                        (7, 0.49996737843785244, 0.0, -0.5000000000000002),\n                        (11, 0.5000208919712605, 0.0, -0.5),\n                        (9, 0.0, -1.0, -0.5),\n                        (5, 0.0, -1.0, 0.7941425110563254)],\n                       [(9, 10), (5, 6)],\n                       [(10, 0.5, 0.0, 0.5),\n                        (6, 0.23084142329666135, 0.0, 0.5),\n                        (9, 0.0, -0.8040131261619035, -0.6959868738380965)],\n                       [(6, 7), (10, 11)],\n                       [(10, 0.5, -1.0, -0.5), (7, 0.4999673784378522, -1.0, 0.5), (5, 6)],\n                       [(7, 11), (6, 10)],\n                       [(9, 10),\n                        (6, 0.9749741089553319, 4.440892098500626e-16, -0.36410496470755027),\n                        (7, 0.08252572858156038, 0.0, 0.3075255823905667),\n                        (11, 0.08255256792708421, 0.0, 0.6924045197900228)],\n                       [(10, 0.4772244005800368, -1.0, 0.48961966994850625), (7, 3), (5, 6), (11, 12)],\n                       [(9, 10),\n                        (11, 0.4653211182942605, 0.0, 0.23541166555792303),\n                        (6, 0.4861989543308627, -1.1102230246251565e-16, -0.41789406021725417),\n                        (7, 0.4653211182942605, -1.0, 0.7645907520498305),\n                        (3, 0.5295069129729826, 0.0, -0.5893209314959036),\n                        (12, 0.5295069129729826, -1.0, -0.5893209314959036)],\n                       [(10, 0.4861989543308627, -1.0, 0.6796753570576561), (6, 7)],\n                       [(10, 11),\n                        (7, 0.6827527082828264, 0.0, 0.12290225883100048),\n                        (6, 0.4439606265301672, 0.0, 0.7523106148103873)],\n                       [(11, 0.6827527082828264, 0.0, 0.8771001587767528),\n                        (10, 0.44396062653016743, -1.0, -0.49052931796998567),\n                        (5, 6),\n                        (7, 3)],\n                       [(11, 12),\n                        (9, 10),\n                        (7, 0.10695351794258778, -1.0, 0.6229022588309947),\n                        (6, 0.3780302949345278, 0.0, -0.7476893851896125),\n                        (3, 0.40668095023551065, 0.0, 0.8713162318815462)],\n                       [(10, 0.37803029493452756, -1.0, -0.9905293179699857),\n                        (11, 0.10695351794258778, 0.0, 0.3771001587767584),\n                        (6, 7),\n                        (12, 0.40668095023551065, -1.0, 0.8713162318815462)],\n                       [(10, 11),\n                        (7, 0.4664424050737874, 0.0, -0.4863595551409866),\n                        (6, 0.6406239893192525, 2.220446049250313e-16, -0.24768938518961792)],\n                       [(10, 0.6406239893192525, 0.0, 0.5094706820300192),\n                        (11, 0.4664424050737874, 0.0, -0.5136380272512602),\n                        (5, 6),\n                        (7, 3)],\n                       [(9, 10),\n                        (11, 12),\n                        (7, 0.19782660459849954, -1.0, -0.0023700300843298994),\n                        (6, 0.4425354986852117, 0.0, -0.7476893851896178),\n                        (3, 0.5492710367616964, 0.0, 0.4063504694497735)],\n                       [(10, 0.4425354986852117, -1.0, 0.009470682030019661),\n                        (11, 0.19782660459849954, 0.0, -0.9976275523079168),\n                        (6, 7),\n                        (12, 0.5492710367616964, -1.0, 0.4063504694497735)],\n                       [(10, 11),\n                        (6, 0.1847466179741335, 0.0, -0.29169180217083346),\n                        (7, 0.7684689111399088, 0.0, -0.5015439514862527)],\n                       [(11, 0.7684689111399088, -1.1102230246251565e-16, -0.4984536309059936),\n                        (10, 0.1847466179741335, 0.0, -0.4465269009887648),\n                        (5, 6),\n                        (7, 3)],\n                       [(11, 12),\n                        (9, 10),\n                        (6, 0.5432475928713614, 0.0, 0.7059323285424606),\n                        (7, 0.550139150489624, -1.0, 0.7698135011910368),\n                        (5, 0.4999999999999998, 0.0, -0.702569283774394),\n                        (3, 0.8902926131088993, 0.0, 0.02516975404100341)],\n                       [(11, 0.550139150489624, 0.0, 0.2301889164167168),\n                        (10, 0.5432475928713614, -1.0, -0.44415103170205883),\n                        (6, 7),\n                        (9, 0.4999999999999998, 0.0, 0.9334268262535086),\n                        (12, 0.8902926131088993, -1.0, 0.02516975404100341)],\n                       [(10, 11),\n                        (7, 0.7052739125587255, 0.0, -0.09125199087210067),\n                        (6, 0.15727698463740847, 0.0, 0.5189947887754507)],\n                       [(11, 0.7052739125587255, 4.440892098500626e-16, -0.9087455915201457),\n                        (10, 0.15727698463740847, 0.0, -0.2572134919350494),\n                        (5, 6),\n                        (7, 3)],\n                       [(11, 12),\n                        (9, 10),\n                        (7, 0.2537577306592731, -1.0, 0.40874800912789944),\n                        (6, 0.5, 0.0, -0.28531904586294765),\n                        (5, 0.6680110440239018, 0.0, -0.20256928377439354),\n                        (3, 0.32806090175681435, -1.0, 0.47350959105691115)],\n                       [(11, 0.2537577306592731, 4.440892098500626e-16, 0.5912544084798552),\n                        (10, 0.5000000000000002, -1.0, -0.4528996572966507),\n                        (6, 7),\n                        (9, 0.6680110440239018, 0.0, -0.5665731737464916),\n                        (12, 0.32806090175681435, 0.0, 0.47350959105691115)],\n                       [(10, 11), (6, 0.5, 0.0, 0.7146809541370525), (7, 3)],\n                       [(10, 0.5, 4.440892098500626e-16, 0.5471003427033492),\n                        (5, 6),\n                        (11, 12),\n                        (3, 0.6783645703167935, -1.0, 0.9735095910569109)],\n                       [(9, 10),\n                        (6, 0.6272475297087556, 0.0, -0.9475402973187848),\n                        (11, 0.0, -0.5954473892522909, -0.9045526107477091),\n                        (12, 0.6783645703167935, 0.0, 0.9735095910569109),\n                        (5, 0.5000000000000004, 0.0, 0.29743071622560624),\n                        (3, 0.0, -1.0, -0.08479870631144637)],\n                       [(10, 0.6272475297087556, 0.0, -0.7906784058408136),\n                        (6, 7),\n                        (9, 0.5000000000000004, 0.0, -0.06657317374649141)],\n                       [(10, 11),\n                        (6, 0.5879904199402892, 0.0, 0.9797309256862847),\n                        (7, 0.5, 0.0, 0.44402964491540153)],\n                       [(11, 0.5, 0.0, 0.9605253834400624),\n                        (10, 0.5879904199402892, -1.1102230246251565e-16, 0.28205037115411735),\n                        (5, 6),\n                        (7, 3)],\n                       [(11, 12),\n                        (9, 10),\n                        (7, 0.2506657363928002, 0.0, -0.055970355084599355),\n                        (6, 0.7259252541215333, 0.0, -0.7432264202931096),\n                        (5, 0.4999999999999998, 0.0, 0.7500023182232844)],\n                       [(10, 0.7259252541215333, 0.0, -0.9949922828664883),\n                        (11, 0.2506657363928002, 0.0, -0.5394746165599364),\n                        (6, 7),\n                        (9, 0.4999999999999998, 0.0, 0.4808552242558304)],\n                       [(10, 11),\n                        (7, 0.7552770370149398, 0.0, -0.05597035508459847),\n                        (6, 0.5, 0.0, 0.1006716672219099)],\n                       [(11, 0.7552770370149398, 0.0, 0.46052538344006244),\n                        (10, 0.5, 0.0, -0.8388903703815078),\n                        (7, 3),\n                        (5, 6)],\n                       [(11, 12),\n                        (9, 10),\n                        (3, 0.4952514392902705, 0.0, 0.4757976238166852),\n                        (7, 0.5, 0.0, -0.5559703550845985),\n                        (6, 0.45694135055250906, 0.0, 0.1006716672219099),\n                        (5, 0.5, -0.9878414992960507, 0.250002318223284)],\n                       [(12, 0.4952514392902705, 0.0, 0.4757976238166852),\n                        (11, 0.5, 0.0, 0.9605253834400618),\n                        (10, 0.45694135055250906, 0.0, 0.16110962961849218),\n                        (6, 7),\n                        (3, 1),\n                        (9, 0.5, -0.7186944053285966, 0.9808552242558302)],\n                       [(12, 8),\n                        (10, 11),\n                        (3, 0.6205477744107246, 0.0, -0.04155053511319895),\n                        (7, 0.35520170312806343, 0.0, 0.9704189957908083),\n                        (6, 0.5, 0.0, 0.6006716672219097),\n                        (1, 0.5, 0.0, -0.5201538331827915)],\n                       [(10, 0.5, 0.0, 0.6611096296184922),\n                        (12, 0.6205477744107246, -1.0, -0.04155053511319906),\n                        (11, 0.3552017031280632, 0.0, 0.4341360325646555),\n                        (7, 3),\n                        (8, 0.5, 0.0, -0.5201538331827915)],\n                       [(11, 12),\n                        (6, 7),\n                        (3, 0.5, 0.0, 0.4770367210731963),\n                        (10, 2.220446049250313e-16, -1.0, -0.7590058516477058)],\n                       [(10, 11),\n                        (7, 0.0, 0.8376036152760205, -0.33760361527602045),\n                        (12, 0.5, 0.0, 0.4770367210731963),\n                        (3, 1)],\n                       [(7, 0.7299176310680837, -1.0, 0.7260709034222252),\n                        (11, 0.7299176310680837, 4.440892098500626e-16, 0.5160877402092579),\n                        (12, 8),\n                        (3, 0.13056868187156478, 0.0, -0.7618072075886293),\n                        (1, 0.44795079368547275, 0.0, 0.3434400796687824)],\n                       [(6, 7),\n                        (10, 11),\n                        (12, 0.13056868187156478, -1.0, -0.7618072075886293),\n                        (8, 0.44795079368547275, 0.0, 0.3434400796687824)],\n                       [(11, 0.5708700815399963, 0.0, -0.19460156630369063),\n                        (7, 0.5708700815399963, 0.0, -0.5632397900648256),\n                        (10, 0.5, -1.0, -0.4641857739842893),\n                        (6, 0.5, 0.0, 0.7259670708246908)],\n                       [(7, 3), (11, 12)],\n                       [(3, 0.12328076625869655, 0.0, -0.00918968942689169),\n                        (12, 0.12328076625869655, 0.0, -0.00918968942689169),\n                        (11, 0.35440396373816796, 0.0, 0.5039785697305632),\n                        (7, 0.35440396373816796, -1.0, 0.7381800739009203)],\n                       [(12, 8), (3, 1), (6, 7), (11, 0.0, 0.5102138624372969, -0.010213862437296939)],\n                       [(3, 0.31277527658165183, -1.0, -0.5091896894268917),\n                        (12, 0.31277527658165183, 0.0, -0.5091896894268917),\n                        (10, 11),\n                        (1, 0.23061776402485323, 0.0, -0.41529275648922015),\n                        (8, 0.23061776402485323, 0.0, -0.41529275648922015),\n                        (6, 0.5, 0.0, -0.2740329291753092)],\n                       [(11, 12), (7, 3), (10, 0.5, 0.0, 0.5358142260157108)],\n                       [(11, 0.5, 0.0, 0.3178336234430277),\n                        (12, 0.7520450774165834, 0.0, -0.009189689426892467),\n                        (3, 0.7520450774165834, 0.0, -0.009189689426892467),\n                        (7, 0.5, 0.0, -0.5654611173742472)],\n                       [(12, 8), (6, 7), (10, 11), (3, 1)],\n                       [(12, 0.22588638999651067, 0.0, 0.4908103105731081),\n                        (11, 0.5, 0.0, 0.31783362344302835),\n                        (3, 0.22588638999651067, -1.0, 0.4908103105731081),\n                        (7, 0.5, 0.0, 0.4345388826257528),\n                        (1, 0.39222583923283394, 0.0, -0.04993017182812021),\n                        (8, 0.39222583923283394, 0.0, -0.04993017182812021),\n                        (6, 0.6321947838533055, 0.0, 0.5984315097770114),\n                        (10, 0.6321947838533057, 0.0, 0.6633497870633913)],\n                       [(7, 3), (11, 12)],\n                       [(12, 0.696715986124177, 0.0, 0.4908103105731081),\n                        (3, 0.696715986124177, 0.0, 0.4908103105731081),\n                        (10, 11),\n                        (7, 0.0, -0.5037355019564731, -0.9962644980435269)],\n                       [(3, 1), (12, 8), (6, 7), (10, 0.6771139237982053, -1.0, 0.251588285383491)],\n                       [(3, 0.583609737771011, 4.440892098500626e-16, 0.9908103105731074),\n                        (12, 0.583609737771011, -1.0, 0.9908103105731074),\n                        (6, 0.6771139237982053, -1.1102230246251565e-16, 0.010193011456911094),\n                        (8, 0.6050120054676125, 0.0, 0.4640144763916523),\n                        (1, 0.6050120054676125, 0.0, 0.4640144763916523)],\n                       [(11, 12), (7, 3), (6, 0.0, 0.1272918880664975, 0.3727081119335025)],\n                       [(7, 0.5, -1.0, -0.8360458831220228),\n                        (12, 0.06412882081148275, 0.0, -0.009189689426890246),\n                        (11, 0.5, -1.1102230246251565e-16, 0.08468288723432948),\n                        (3, 0.06412882081148275, 0.0, -0.009189689426890246)],\n                       [(10, 11), (12, 8), (3, 1), (6, 7)],\n                       [(12, 0.49999999999999956, -1.0, -0.009189689426891912),\n                        (11, 0.5, -1.1102230246251565e-16, 0.08468288723432948),\n                        (7, 0.5, 0.0, -0.8360458831220228),\n                        (3, 0.49999999999999956, 0.0, -0.009189689426891912),\n                        (1, 0.6668726728064027, 0.0, -0.23156229310064602),\n                        (8, 0.6668726728064027, 0.0, -0.23156229310064602)],\n                       [(7, 3), (11, 12)],\n                       [(7, 0.5, 0.0, -0.3865113550545133),\n                        (11, 0.5, 0.0, -0.36485164083317934),\n                        (12, 0.48422238672257323, 0.0, 0.541689714742108),\n                        (3, 0.48422238672257323, 0.0, 0.541689714742108)],\n                       [(6, 7), (12, 8), (10, 11), (3, 1)],\n                       [(8, 0.7832164982209655, -1.1102230246251565e-16, -0.48502880815679716),\n                        (7, 0.2714423079803394, 0.0, 0.11348864494548727),\n                        (1, 0.7832164982209655, -1.1102230246251565e-16, -0.48502880815679716),\n                        (10, 0.7832164982209657, 0.0, 0.19658938673509452),\n                        (6, 0.7832164982209657, 0.0, -0.8075162018281958),\n                        (12, 0.5, -1.0, 0.0826950177929533),\n                        (3, 0.5, 0.0, 0.0826950177929533),\n                        (11, 0.2714423079803394, 0.0, 0.13514835916682166)],\n                       [(10, 13), (11, 12), (8, 4), (6, 2), (1, 0), (7, 3)],\n                       [(11, 0.5, 0.0, 0.6351483591668199),\n                        (6, 0.5, 0.0, 0.6924837981718042),\n                        (7, 0.5, 0.0, -0.3865113550545133),\n                        (8, 0.5, 0.0, -0.985028808156797),\n                        (10, 0.4999999999999998, 0.0, 0.6965893867350945),\n                        (1, 0.5, 0.0, -0.985028808156797),\n                        (13, 0.30408672398469627, -1.0, -0.5000000000000002),\n                        (4, 0.30408672398469627, 0.0, -0.5000000000000002),\n                        (0, 0.30408672398469627, 0.0, -0.5000000000000002),\n                        (2, 0.30408672398469627, 0.0, -0.5000000000000002)],\n                       [(12, 8), (7, 6), (3, 1), (11, 10)],\n                       [(1, 0.8040867239846965, 0.0, 0.014971191843202503),\n                        (8, 0.8040867239846965, 0.0, 0.014971191843202503),\n                        (6, 0.8040867239846965, -1.1102230246251565e-16, -0.3075162018281954),\n                        (10, 0.8040867239846965, 4.440892098500626e-16, 0.6965893867350945),\n                        (11, 0.0, 0.6951213699499466, -0.19512136994994667)],\n                       [(6, 2), (8, 4), (10, 13), (1, 0)],\n                       [(8, 0.75, -1.0, 0.014971191843202503),\n                        (10, 0.75, 0.0, 0.6965893867350945),\n                        (6, 0.75, -1.0, -0.3075162018281955),\n                        (1, 0.75, -1.0, 0.014971191843202503),\n                        (0, 0.5, 0.0, 0.5),\n                        (13, 0.5, -1.0, 0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (2, 0.5, 0.0, 0.5)],\n                       [(11, 10), (3, 1), (12, 8), (7, 6)],\n                       [(6, 0.5000000000000002, 0.0, 0.6924837981718042),\n                        (1, 0.5000000000000002, 0.0, -0.985028808156797),\n                        (10, 0.5, 0.0, -0.3034106132649055),\n                        (8, 0.5000000000000002, 0.0, -0.985028808156797),\n                        (7, 0.5, 0.0, 0.4718803958350044),\n                        (11, 0.5, -1.0, 0.4718779782272493),\n                        (12, 0.5, 0.0, 0.44108676868247043),\n                        (3, 0.5, 0.0, 0.44108676868247043)],\n                       [(8, 4), (10, 13), (6, 2), (1, 0)],\n                       [(10, 0.25, 0.0, 0.6965893867350945),\n                        (8, 0.25, -1.0, 0.014971191843202503),\n                        (6, 0.25, -1.0, -0.3075162018281955),\n                        (1, 0.25, -1.0, 0.014971191843202503)],\n                       [(3, 1), (12, 8), (11, 10), (7, 6)],\n                       [(6, 0.5, -1.0, 0.6924837981718042),\n                        (1, 0.5, -1.0, -0.985028808156797),\n                        (10, 0.5, 0.0, -0.3034106132649055),\n                        (8, 0.5, -1.0, -0.985028808156797),\n                        (12, 0.5, 0.05891323131752957, -0.5589132313175298),\n                        (7, 0.5, 0.02811960416499515, -0.5281196041649956),\n                        (3, 0.5, 0.05891323131752957, -0.5589132313175298),\n                        (11, 0.5, 0.028122021772750916, 0.4718779782272491)],\n                       [(6, 2), (8, 4), (10, 13), (1, 0)],\n                       [(0, 1, 0, 0.5),\n                        (13, 1, 0, -1.0),\n                        (10, 0.4999999999999998, 0.8034106132649055, 0.6965893867350945),\n                        (1, 0.5, 0.4850288081567973, 0.014971191843202725),\n                        (4, 1, 0, 0.5),\n                        (6, 0.5, 0.8075162018281956, -0.3075162018281955),\n                        (8, 0.5, 0.4850288081567973, 0.014971191843202725),\n                        (2, 1, 0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],\n           \'qubits\': [(1, 6),\n                      (2, 6),\n                      (3, 5),\n                      (3, 6),\n                      (3, 7),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 5)]},\n (14, 6): {\'method\': \'paired binary counts + recursive coherent decoders\',\n           \'moments\': [[(5, 0.40199709739981815, 0.0, 0.5),\n                        (6, 0.22525566744065986, 0.0, 0.5),\n                        (10, 0.8081443303828094, 0.0, -0.5000000000000002),\n                        (7, 0.11355748076805527, 0.0, 0.5),\n                        (11, 0.674339930903129, 0.0, -0.5),\n                        (3, 0.2961278032423518, 0.0, -0.37072357629548036),\n                        (12, 0.2961278032423518, 0.0, -0.37072357629548036),\n                        (1, 0.4956530884318588, 5.551115123125783e-17, 0.259121498867743),\n                        (8, 0.4956530884318588, 5.551115123125783e-17, 0.259121498867743),\n                        (0, 0.5, 0.0, 0.5),\n                        (2, 0.5, 0.0, 0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (13, 0.5, 0.0, 0.5)],\n                       [(5, 9), (6, 10), (7, 11)],\n                       [(5, 0.5980029026001819, 0.0, 0.5),\n                        (6, 0.6043434208926137, 0.0, -0.5),\n                        (9, 0.4144920760550428, 0.0, 0.5),\n                        (10, 0.3587134164814194, 0.0, 0.5),\n                        (11, 0.900710470819486, 0.0, -0.5000000000000002),\n                        (7, 0.5058014844940542, 0.0, 0.5)],\n                       [(5, 6), (9, 10)],\n                       [(10, 11), (9, 0.7655924148737219, 0.0, 0.5), (6, 0.45023304981067797, 0.0, -0.5)],\n                       [(5, 9), (6, 7), (11, 0.39247591807286747, 0.0, -0.5)],\n                       [(6, 0.9380911750102539, 0.0, 0.5),\n                        (7, 0.29972159246484953, 0.0, 0.5),\n                        (9, 0.2655924148737218, 0.0, 0.5)],\n                       [(6, 10), (7, 11)],\n                       [(6, 0.9558748665699874, 0.0, 0.5),\n                        (10, 0.2620726188484401, 0.0, 0.5),\n                        (7, 0.3163737216172757, 0.0, -0.5),\n                        (11, 0.8721307074113489, 0.0, -0.5)],\n                       [(5, 6), (9, 10)],\n                       [(5, 0.5, 0.0, -0.5),\n                        (9, 0.8775228500880285, 0.0, 0.5),\n                        (6, 0.3809431308306502, 0.0, 0.5),\n                        (10, 0.25000000000000006, 0.0, 0.5)],\n                       [(5, 9), (6, 7), (10, 11)],\n                       [(6, 0.6896352224558817, 0.0, -0.5),\n                        (10, 0.5, 0.0, -0.5),\n                        (7, 0.4465012045786749, 0.0, 0.5),\n                        (11, 0.5, 0.0, 0.5),\n                        (5, 0.5, 0.0, 0.5)],\n                       [(6, 10), (7, 11)],\n                       [(7, 3),\n                        (6, 0.08450095726395301, 0.0, -0.5000000000000002),\n                        (10, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (11, 0.4999999999999998, 0.0, -0.5000000000000002)],\n                       [(5, 6),\n                        (9, 10),\n                        (11, 12),\n                        (7, 0.4999999999999998, 0.0, -0.2596332425295489),\n                        (3, 0.5, 0.0, -0.8707235762954806)],\n                       [(5, 0.5051488971691418, 5.551115123125783e-17, 0.38516684131825535),\n                        (6, 0.4225762562831474, 0.0, 0.6339996202613476),\n                        (9, 0.5051488971691418, 0.0, 0.6148331586817446),\n                        (10, 0.7248594368839316, 0.0, -0.3254129288985448),\n                        (11, 0.4999999999999998, 0.0, 0.7403667574704509),\n                        (12, 0.5, 0.0, -0.8707235762954806)],\n                       [(5, 6), (9, 10)],\n                       [(6, 7),\n                        (10, 11),\n                        (5, 0.22783776220036114, 0.0, 0.8851668413182554),\n                        (9, 0.22783776220036114, 0.0, 0.1148331586817446)],\n                       [(6, 0.4949708987340448, 4.440892098500626e-16, -0.8330784399099445),\n                        (7, 0.550755412437784, 0.0, -0.09655868968129022),\n                        (10, 0.4949708987340448, 0.0, -0.6381081553247452),\n                        (11, 0.550755412437784, 0.0, 0.9034413103187096)],\n                       [(5, 6), (7, 3), (9, 10), (11, 12)],\n                       [(6, 0.5033793022373013, 0.0, 0.666921560090056),\n                        (7, 0.4999999999999998, 0.0, -0.6947798655138213),\n                        (10, 0.5033793022373012, 0.0, -0.13810815532474496),\n                        (11, 0.4999999999999998, 0.0, 0.305220134486178),\n                        (3, 0.340073632931319, 0.0, -0.2109036333971741),\n                        (12, 0.340073632931319, 0.0, -0.2109036333971741),\n                        (5, 0.2906009388508253, 0.0, 0.3851668413182551),\n                        (9, 0.2906009388508253, 0.0, 0.6148331586817446)],\n                       [(6, 7), (10, 11)],\n                       [(7, 3), (11, 12), (5, 6), (9, 10)],\n                       [(6, 0.5188671153526303, -2.220446049250313e-16, -0.7760966592802288),\n                        (7, 0.33147541570683337, 0.0, -0.19477986551382154),\n                        (10, 0.5188671153526303, 0.0, -0.5811263746950295),\n                        (11, 0.33147541570683337, 4.440892098500626e-16, 0.8052201344861782),\n                        (5, 0.5063547328191559, 0.0, -0.11483315868174548),\n                        (3, 0.3751563089816202, 0.0, -0.5976229662630799),\n                        (9, 0.5063547328191556, 0.0, -0.8851668413182552),\n                        (12, 0.3751563089816202, 0.0, -0.5976229662630799)],\n                       [(6, 7), (10, 11)],\n                       [(6, 0.6548369278761568, 0.0, -0.5867916122995678),\n                        (7, 0.8111326152692739, 0.0, 0.305220134486178),\n                        (10, 0.6548369278761568, 0.0, -0.39182132771436784),\n                        (11, 0.8111326152692739, 0.0, -0.6947798655138218)],\n                       [(5, 6), (7, 3), (9, 10), (11, 12)],\n                       [(6, 0.06166130247734848, 0.0, -0.7191375687566577),\n                        (7, 0.015313232122055268, 0.0, 0.805220134486176),\n                        (10, 0.06166130247734846, 0.0, 0.475832715828542),\n                        (11, 0.01531323212205527, 0.0, -0.1947798655138242),\n                        (3, 0.5129607164516408, 0.0, -0.7562970243690965),\n                        (12, 0.5129607164516408, 0.0, -0.7562970243690965)],\n                       [(6, 7), (10, 11)],\n                       [(6, 0.5518863484824029, 4.440892098500626e-16, 0.7564204654484481),\n                        (7, 0.8714469607225119, 0.0, -0.6947798655138229),\n                        (10, 0.5518863484824028, 0.0, -0.048609249966351964),\n                        (11, 0.8714469607225119, 1.1102230246251565e-16, 0.305220134486177)],\n                       [(5, 6), (7, 3), (9, 10), (11, 12)],\n                       [(6, 0.8128647203031829, 0.0, -0.5004422780944038),\n                        (7, 0.8531775343812674, 1.1102230246251565e-16, -0.19477986551382287),\n                        (10, 0.8128647203031829, 0.0, -0.30547199350920407),\n                        (11, 0.8531775343812674, 4.440892098500626e-16, 0.8052201344861771),\n                        (3, 0.6608132099476296, 0.0, 0.7425767458343033),\n                        (12, 0.6608132099476296, 0.0, 0.7425767458343033)],\n                       [(6, 7), (10, 11)],\n                       [(6, 0.47887365991756475, 4.440892098500626e-16, 0.9552281598188521),\n                        (7, 0.03620506650561751, 0.0, 0.3052201344861767),\n                        (10, 0.4788736599175647, 0.0, -0.8498015555959482),\n                        (11, 0.03620506650561751, 0.0, -0.6947798655138233)],\n                       [(5, 6), (7, 3), (9, 10), (11, 12)],\n                       [(6, 0.320266437724436, 0.0, -0.6067256874188456),\n                        (7, 0.5000000000000002, 0.0, -0.6947798655138229),\n                        (10, 0.320266437724436, 0.0, 0.5882445971663532),\n                        (11, 0.5, 0.0, 0.30522013448617735),\n                        (5, 0.5, 0.0, -0.8149083106323878),\n                        (3, 0.4253414728603863, 0.0, -0.7091324900840312),\n                        (9, 0.5, 0.0, -0.1850916893676131),\n                        (12, 0.4253414728603863, 0.0, -0.7091324900840312)],\n                       [(6, 7), (10, 11)],\n                       [(6, 0.5, 0.0, 0.7574838051632844),\n                        (7, 0.4457705204849698, 0.0, 0.7712481686278343),\n                        (10, 0.5, 0.0, -0.04754591025151611),\n                        (11, 0.4457705204849698, 0.0, -0.2287518313721657)],\n                       [(5, 6), (7, 3), (9, 10), (11, 12)],\n                       [(3, 0.47411281038999165, 0.0, 0.6259258041344009),\n                        (6, 7),\n                        (10, 11),\n                        (12, 0.47411281038999165, 0.0, 0.6259258041344009),\n                        (5, 0.5, 0.6298166212647756, 0.18509168936761172),\n                        (9, 0.4999999999999998, 1.1102230246251565e-16, -0.1850916893676129)],\n                       [(3, 1),\n                        (12, 8),\n                        (7, 0.5486314612204101, 0.0, -0.9851782296221445),\n                        (11, 0.5486314612204102, 5.551115123125783e-17, 0.01482177037785476)],\n                       [(6, 7),\n                        (10, 11),\n                        (3, 0.344882952068414, 8.881784197001252e-16, -0.9595787953219059),\n                        (12, 0.344882952068414, 8.881784197001252e-16, -0.9595787953219059),\n                        (1, 0.4925777780083309, 0.0, 0.5072045568580279),\n                        (8, 0.4925777780083309, 0.0, 0.5072045568580279)],\n                       [(7, 0.33274203901299115, 0.0, 0.037826546397841554),\n                        (11, 0.33274203901299115, 0.0, -0.9621734536021593)],\n                       [(7, 3), (11, 12)],\n                       [(3, 0.9291764376708231, 5.551115123125783e-17, 0.13537775650146772),\n                        (7, 0.9294880475869709, 0.0, -0.17101814457717412),\n                        (11, 0.9294880475869709, 0.0, 0.8289818554228245),\n                        (12, 0.9291764376708231, 5.551115123125783e-17, 0.13537775650146772)],\n                       [(3, 1), (6, 7), (10, 11), (12, 8)],\n                       [(7, 0.9764886499665415, 0.0, 0.22943742001273848),\n                        (3, 0.2914905715516678, 0.0, 0.028761239990876486),\n                        (11, 0.9764886499665415, 0.0, -0.7705625799872613),\n                        (12, 0.2914905715516678, 0.0, 0.028761239990876486),\n                        (1, 0.41571419491182243, 0.0, 0.41360830056593145),\n                        (8, 0.41571419491182243, 0.0, 0.41360830056593145)],\n                       [(7, 3), (11, 12)],\n                       [(3, 0.4183604728916834, 0.0, -0.11203258325294407),\n                        (7, 0.322056849702119, 0.0, -0.24842114859690057),\n                        (11, 0.322056849702119, 0.0, 0.7515788514030981),\n                        (12, 0.4183604728916834, 0.0, -0.11203258325294407)],\n                       [(3, 1), (6, 7), (10, 11), (12, 8)],\n                       [(7, 0.5910955657739062, 0.0, 0.6831816672359468),\n                        (3, 0.5658106523525602, 5.551115123125783e-17, 0.19322588584914446),\n                        (11, 0.5910955657739062, 0.0, -0.3168183327640539),\n                        (12, 0.5658106523525602, 5.551115123125783e-17, 0.19322588584914446),\n                        (1, 0.30256758326839167, 0.0, -0.5706246524937872),\n                        (8, 0.30256758326839167, 0.0, -0.5706246524937872)],\n                       [(7, 3), (11, 12)],\n                       [(3, 0.6380247240052519, 0.0, 0.5410578489817497),\n                        (7, 0.5222942734869767, 0.0, -0.6067312232745037),\n                        (11, 0.5222942734869767, 5.551115123125783e-17, 0.39326877672549565),\n                        (12, 0.6380247240052519, 0.0, 0.5410578489817497)],\n                       [(3, 1), (6, 7), (10, 11), (12, 8)],\n                       [(7, 0.36360452844948893, 0.0, -0.4991997111251587),\n                        (3, 0.017085511701252946, 0.0, 0.4123725366829347),\n                        (11, 0.36360452844948893, 0.0, 0.5008002888748404),\n                        (12, 0.017085511701252946, 0.0, 0.4123725366829347),\n                        (1, 0.32590008584322616, 0.0, -0.5618902727160411),\n                        (8, 0.32590008584322616, 0.0, -0.5618902727160411)],\n                       [(7, 3), (11, 12)],\n                       [(3, 0.7382217283247945, 0.0, 0.40491668093216243),\n                        (7, 0.9940835612187666, 0.0, 0.4522571587588089),\n                        (11, 0.9940835612187666, 0.0, -0.547742841241192),\n                        (12, 0.7382217283247945, 0.0, 0.40491668093216243)],\n                       [(3, 1), (6, 7), (10, 11), (12, 8)],\n                       [(7, 0.5303663346437927, 0.0, -0.9739997650196348),\n                        (3, 0.39031863075905965, 0.0, 0.3956976358728742),\n                        (11, 0.5303663346437927, 0.0, 0.026000234980365133),\n                        (12, 0.39031863075905965, 0.0, 0.3956976358728742),\n                        (1, 0.5100876690078296, 0.0, 0.3944365573706427),\n                        (8, 0.5100876690078296, 0.0, 0.3944365573706427)],\n                       [(7, 3), (11, 12)],\n                       [(3, 0.6856477314180824, 4.440892098500626e-16, 0.961239441531951),\n                        (7, 0.6025816719136099, 0.0, 0.39266747301734495),\n                        (11, 0.6025816719136099, 0.0, -0.6073325269826559),\n                        (12, 0.6856477314180824, 4.440892098500626e-16, 0.961239441531951)],\n                       [(3, 1), (6, 7), (10, 11), (12, 8)],\n                       [(7, 0.8921121044926871, 0.0, 0.9710982603665494),\n                        (3, 0.3385860029956733, 4.440892098500626e-16, 0.7767121578233691),\n                        (11, 0.8921121044926871, 1.1102230246251565e-16, -0.028901739633451662),\n                        (12, 0.3385860029956733, 4.440892098500626e-16, 0.7767121578233691),\n                        (1, 0.2347749898956015, 0.0, -0.3542327425401759),\n                        (8, 0.2347749898956015, 0.0, -0.3542327425401759)],\n                       [(7, 3), (11, 12)],\n                       [(3, 0.44925295079759087, 0.0, 0.6747334260429754),\n                        (7, 0.5928759364551912, 0.0, -0.6082157086391826),\n                        (11, 0.5928759364551911, 0.0, 0.39178429136081694),\n                        (12, 0.44925295079759087, 0.0, 0.6747334260429754)],\n                       [(3, 1), (6, 7), (10, 11), (12, 8)],\n                       [(7, 0.8105365413512657, 0.0, 0.8965980980060635),\n                        (3, 0.6433165441464112, 0.0, -0.8142602483859023),\n                        (11, 0.8105365413512657, 0.0, -0.10340190199393717),\n                        (12, 0.6433165441464112, 0.0, -0.8142602483859023),\n                        (1, 0.34497587576931366, 0.0, -0.28963794555439826),\n                        (6, 0.4999999999999998, 0.0, -0.5429023601020468),\n                        (10, 0.4999999999999998, 2.220446049250313e-16, -0.34793207551684535),\n                        (8, 0.34497587576931366, 0.0, -0.28963794555439826)],\n                       [(7, 3), (11, 12)],\n                       [(3, 0.5260557718525964, 0.0, 0.2663667337411011),\n                        (7, 0.3843844916022807, 0.0, -0.5743150000965365),\n                        (11, 0.3843844916022807, 0.0, 0.4256849999034631),\n                        (12, 0.5260557718525964, 0.0, 0.2663667337411011)],\n                       [(3, 1), (6, 7), (10, 11), (12, 8)],\n                       [(3, 0.7720515191095958, 8.881784197001252e-16, 0.775089250691015),\n                        (12, 0.7720515191095958, 8.881784197001252e-16, 0.775089250691015),\n                        (1, 0.4355845015328854, 0.0, -0.8111689735837979),\n                        (6, 0.5000000000000001, 0.0, 0.45709763989795316),\n                        (10, 0.5, 4.440892098500626e-16, 0.6520679244831535),\n                        (8, 0.4355845015328854, 0.0, -0.8111689735837979)],\n                       [(7, 3), (11, 12)],\n                       [(3, 0.13993480154011695, 0.0, 0.062065126023943096),\n                        (7, 0.5, 0.0, -0.7814080345725862),\n                        (11, 0.5, 5.551115123125783e-17, 0.21859196542741305),\n                        (12, 0.13993480154011695, 0.0, 0.062065126023943096)],\n                       [(3, 1), (6, 7), (10, 11), (12, 8)],\n                       [(1, 0.3416013957299424, 0.0, -0.7695375446745221),\n                        (6, 0.4546112109132118, 0.0, -0.2404300408848754),\n                        (7, 0.22995692914698007, 0.0, 0.7185919654274138),\n                        (3, 0.893580808387734, -2.220446049250313e-16, -0.6446679698784725),\n                        (8, 0.3416013957299424, 0.0, -0.7695375446745221),\n                        (10, 0.4546112109132119, 0.0, -0.045459756299676135),\n                        (11, 0.22995692914698007, 0.0, -0.28140803457258734),\n                        (12, 0.893580808387734, -2.220446049250313e-16, -0.6446679698784725)],\n                       [(1, 0), (6, 2), (7, 3), (8, 4), (10, 13), (11, 12)],\n                       [(1, 0.4999999999999998, 0.0, -0.18230509551025165),\n                        (7, 0.4999999999999998, 0.0, -0.781408034572586),\n                        (6, 0.5, 0.0, -0.45331869799469215),\n                        (11, 0.4999999999999998, 0.0, 0.2185919654274131),\n                        (10, 0.5, 0.0, -0.2583484134094918),\n                        (8, 0.4999999999999998, 0.0, -0.18230509551025165),\n                        (0, 0.30408672398469605, 0.0, -0.5000000000000002),\n                        (2, 0.30408672398469605, 0.0, -0.5000000000000002),\n                        (4, 0.30408672398469605, 0.0, -0.5000000000000002),\n                        (13, 0.30408672398469605, 0.0, -0.5000000000000002)],\n                       [(3, 1), (7, 6), (11, 10), (12, 8)],\n                       [(1, 0.8040867239846965, 1.1102230246251565e-16, -0.1823050955102521),\n                        (6, 0.8040867239846965, 0.0, -0.45331869799469215),\n                        (8, 0.8040867239846965, 1.1102230246251565e-16, -0.1823050955102521),\n                        (10, 0.8040867239846965, 0.0, -0.2583484134094918)],\n                       [(1, 0), (6, 2), (8, 4), (10, 13)],\n                       [(1, 0.75, 1.1102230246251565e-16, -0.18230509551025254),\n                        (6, 0.75, 0.0, -0.45331869799469215),\n                        (10, 0.7499999999999999, 0.0, -0.2583484134094918),\n                        (8, 0.75, 1.1102230246251565e-16, -0.18230509551025254),\n                        (0, 0.5, 0.0, 0.5),\n                        (2, 0.5, 0.0, 0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (13, 0.5, 0.0, 0.5)],\n                       [(3, 1), (7, 6), (11, 10), (12, 8)],\n                       [(1, 0.5, 0.0, 0.8176949044897479),\n                        (6, 0.5, 0.0, 0.5466813020053076),\n                        (8, 0.5, 0.0, 0.8176949044897479),\n                        (10, 0.5, 8.881784197001252e-16, 0.7416515865905067),\n                        (3, 0.5, 0.0, -0.642328854994604),\n                        (7, 0.5, -2.220446049250313e-16, 0.4839373233417934),\n                        (11, 0.5, 0.0, -0.5160626766582066),\n                        (12, 0.5, 0.0, -0.642328854994604)],\n                       [(1, 0), (6, 2), (8, 4), (10, 13)],\n                       [(1, 0.25000000000000006, 0.0, -0.1823050955102523),\n                        (6, 0.25000000000000006, 0.0, -0.45331869799469215),\n                        (10, 0.25, 0.0, -0.2583484134094918),\n                        (8, 0.25000000000000006, 0.0, -0.1823050955102523)],\n                       [(3, 1), (7, 6), (11, 10), (12, 8)],\n                       [(1, 0.5, 8.881784197001252e-16, 0.817694904489747),\n                        (6, 0.5, 0.0, 0.5466813020053076),\n                        (8, 0.5, 8.881784197001252e-16, 0.817694904489747),\n                        (10, 0.5, 8.881784197001252e-16, 0.7416515865905067),\n                        (3, 0.5, -0.857671145005396, -0.642328854994604),\n                        (7, 0.5, 0.0160626766582066, 0.4839373233417934),\n                        (11, 0.5, -0.9839373233417934, -0.5160626766582066),\n                        (12, 0.5, -0.857671145005396, -0.642328854994604)],\n                       [(1, 0), (6, 2), (8, 4), (10, 13)],\n                       [(0, 1.0, 0.0, 0.5),\n                        (1, 0.5, -0.31769490448974746, 0.8176949044897481),\n                        (2, 1.0, 0.0, 0.5),\n                        (4, 1.0, 0.0, 0.5),\n                        (6, 0.5, -0.0466813020053074, 0.5466813020053076),\n                        (8, 0.5, -0.31769490448974746, 0.8176949044897481),\n                        (10, 0.5, -0.2416515865905069, 0.7416515865905067),\n                        (13, 1.0, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],\n           \'qubits\': [(1, 6),\n                      (2, 6),\n                      (3, 5),\n                      (3, 6),\n                      (3, 7),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 5)]},\n (14, 7): {\'method\': \'paired binary counts + recursive coherent decoders\',\n           \'moments\': [[(9, 0.0726903407232826, 0.0, 0.5),\n                        (6, 0.2105043071720354, 0.0, -0.5),\n                        (11, 0.8442009589751782, 0.0, -0.5),\n                        (3, 0.5134368213488532, 0.0, -0.5),\n                        (12, 0.5134368213488532, 0.0, -0.5),\n                        (8, 0.4956530884318588, 5.551115123125783e-17, 0.259121498867743),\n                        (1, 0.4956530884318588, 5.551115123125783e-17, 0.259121498867743),\n                        (4, 0.5, 0.0, 0.5),\n                        (13, 0.5, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (2, 0.5, 0.0, 0.5)],\n                       [(5, 9), (6, 10), (7, 11)],\n                       [(5, 0.8160265687841695, 0.0, 0.5),\n                        (6, 0.7894956928279646, 0.0, -0.5),\n                        (9, 0.9028620606102902, 0.0, -0.5),\n                        (10, 0.6627376333571398, 0.0, 0.5),\n                        (7, 0.19627939477576395, 0.0, 0.5),\n                        (11, 0.3442009589751782, 0.0, 0.5)],\n                       [(5, 6), (9, 10)],\n                       [(5, 0.18397343121583082, 0.0, -0.5000000000000002),\n                        (9, 0.7510481991484306, 0.0, 0.5),\n                        (6, 0.3688802223287273, 0.0, -0.5000000000000002),\n                        (10, 0.18381582001445085, 0.0, 0.5)],\n                       [(5, 9), (6, 7), (10, 11)],\n                       [(6, 0.41021400009546194, 0.0, 0.5),\n                        (10, 0.29743870814587847, 0.0, -0.5),\n                        (7, 0.4665031381347729, 0.0, 0.5),\n                        (11, 0.5, 0.0, -0.5),\n                        (5, 0.3378226236080215, 0.0, 0.5),\n                        (9, 0.338503073915604, 0.0, -0.5)],\n                       [(6, 10), (7, 11)],\n                       [(5, 6), (10, 0.4605424294276723, 0.0, 0.5)],\n                       [(6, 7), (9, 10), (5, 0.8551748142380378, 0.0, 0.5)],\n                       [(9, 0.5326595405526414, 0.0, 0.5),\n                        (10, 0.122534303185607, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (7, 0.5, 0.0, 0.5)],\n                       [(5, 9), (10, 11)],\n                       [(6, 10),\n                        (7, 11),\n                        (5, 0.09492149604478904, 0.0, 0.5),\n                        (9, 0.5949214960447889, 0.0, 0.5)],\n                       [(6, 0.5578271339686174, 0.0, -0.5),\n                        (7, 0.03971873938181481, 0.0, -0.5000000000000002),\n                        (10, 0.905078503955211, 0.0, -0.5),\n                        (11, 0.9373970476727265, 0.0, -0.5000000000000002)],\n                       [(5, 6), (7, 3), (9, 10), (11, 12)],\n                       [(5, 0.5, 0.0, 0.5),\n                        (6, 0.4891369935076435, 0.0, 0.5),\n                        (9, 0.4999999999999998, 0.0, -0.5000000000000002),\n                        (10, 0.760846669639411, 0.0, -0.5),\n                        (7, 0.7941033541984499, 0.0, -0.5),\n                        (11, 0.7941033541984499, 0.0, -0.5),\n                        (3, 0.09503601272046969, 0.0, 0.4999999999999998),\n                        (12, 0.09503601272046969, 0.0, 0.4999999999999998)],\n                       [(5, 6), (9, 10)],\n                       [(6, 0.32526722636904526, 0.0, 0.5), (10, 0.32526722636904526, 0.0, 0.5)],\n                       [(6, 7), (10, 11)],\n                       [(6, 0.8041266624310963, 0.0, -0.5),\n                        (7, 0.7244331475506502, 0.0, 0.5),\n                        (10, 0.8041266624310963, 0.0, -0.5),\n                        (11, 0.7244331475506502, 0.0, -0.5)],\n                       [(5, 6), (7, 3), (9, 10), (11, 12)],\n                       [(6, 0.23444009082657646, 0.0, -0.5),\n                        (7, 0.8613042944089135, 0.0, 0.5),\n                        (10, 0.23444009082657646, 0.0, 0.5),\n                        (11, 0.8613042944089135, 0.0, -0.5),\n                        (3, 0.2829411270406133, 0.0, -0.5),\n                        (12, 0.2829411270406133, 0.0, -0.5)],\n                       [(6, 7), (10, 11)],\n                       [(6, 0.25071987449186994, 0.0, 0.5),\n                        (7, 0.5449608651484803, 0.0, -0.5),\n                        (10, 0.25071987449186994, 0.0, -0.5000000000000002),\n                        (11, 0.5449608651484803, 0.0, -0.5)],\n                       [(5, 6), (7, 3), (9, 10), (11, 12)],\n                       [(6, 0.5623146345847853, 0.0, 0.5),\n                        (7, 0.45182722069768927, 0.0, 0.5),\n                        (10, 0.5623146345847853, 0.0, 0.5),\n                        (11, 0.45182722069768927, 0.0, 0.5),\n                        (3, 0.3648597074431628, 0.0, 0.5),\n                        (12, 0.3648597074431628, 0.0, 0.5)],\n                       [(6, 7), (10, 11)],\n                       [(6, 0.7475099191326353, 0.0, -0.5),\n                        (7, 0.43170540271778, 0.0, 0.5),\n                        (10, 0.7475099191326353, 0.0, -0.5),\n                        (11, 0.43170540271778, 0.0, -0.5)],\n                       [(5, 6), (7, 3), (9, 10), (11, 12)],\n                       [(6, 0.1863554871185897, 0.0, 0.5),\n                        (7, 0.600006446168567, 0.0, 0.5),\n                        (10, 0.1863554871185897, 0.0, -0.5),\n                        (11, 0.600006446168567, 0.0, -0.5),\n                        (3, 0.5948087170803427, 0.0, -0.5),\n                        (12, 0.5948087170803427, 0.0, -0.5)],\n                       [(6, 7), (10, 11)],\n                       [(6, 0.18736031327399977, 0.0, 0.5),\n                        (7, 0.08545923160647084, 0.0, 0.5),\n                        (10, 0.18736031327399977, 0.0, -0.5),\n                        (11, 0.08545923160647084, 0.0, 0.5)],\n                       [(5, 6), (7, 3), (9, 10), (11, 12)],\n                       [(6, 0.3095273135966239, 0.0, 0.5),\n                        (7, 0.2853676947061159, 0.0, 0.5),\n                        (10, 0.3095273135966239, 0.0, 0.5),\n                        (11, 0.2853676947061159, 0.0, 0.5),\n                        (3, 0.5609325442841979, 0.0, -0.5),\n                        (12, 0.5609325442841979, 0.0, -0.5)],\n                       [(6, 7), (10, 11)],\n                       [(6, 0.7120975782091497, 0.0, -0.5),\n                        (7, 0.19227288083003605, 0.0, 0.5),\n                        (10, 0.7120975782091497, 0.0, -0.5),\n                        (11, 0.19227288083003605, 0.0, -0.5)],\n                       [(5, 6), (7, 3), (9, 10), (11, 12)],\n                       [(6, 0.3728883046820175, 0.0, -0.5000000000000002),\n                        (7, 0.21224365634312453, 0.0, -0.5),\n                        (10, 0.3728883046820175, 0.0, 0.5),\n                        (11, 0.21224365634312453, 0.0, 0.5),\n                        (5, 0.5, 0.0, -0.5),\n                        (3, 0.1206646036111223, 0.0, 0.5),\n                        (9, 0.5, 0.0, 0.5),\n                        (12, 0.1206646036111223, 0.0, 0.5)],\n                       [(6, 7), (10, 11)],\n                       [(6, 0.07381131043005106, 0.0, -0.5000000000000002),\n                        (7, 0.5972481824285067, 0.0, -0.5),\n                        (10, 0.07381131043005106, 0.0, 0.5),\n                        (11, 0.5972481824285067, 0.0, -0.5)],\n                       [(5, 6), (7, 3), (9, 10), (11, 12)],\n                       [(6, 0.5, 0.0, -0.5),\n                        (7, 0.3633470223178652, 0.0, -0.5000000000000002),\n                        (10, 0.5, 0.0, -0.5),\n                        (11, 0.3633470223178652, 0.0, -0.5000000000000002),\n                        (3, 0.40745450984385334, 0.0, -0.5),\n                        (12, 0.40745450984385334, 0.0, -0.5),\n                        (5, 0.5000000000000003, 0.0, 0.5),\n                        (9, 0.5000000000000003, 0.0, 0.5)],\n                       [(6, 7), (10, 11)],\n                       [(7, 3), (11, 12), (6, 0.5, 0.0, -0.5), (10, 0.5, 0.0, -0.5)],\n                       [(5, 6), (9, 10), (3, 0.900298441988697, 0.0, 0.5), (12, 0.900298441988697, 0.0, 0.5)],\n                       [(6, 0.5, 0.0, 0.5), (10, 0.5, 0.0, -0.5), (5, 0.5, 0.0, 0.5), (9, 0.5, 0.0, -0.5)],\n                       [(6, 7), (10, 11)],\n                       [(5, 6),\n                        (9, 10),\n                        (7, 0.1210845387975159, 0.0, 0.5),\n                        (11, 0.1210845387975159, 0.0, 0.5)],\n                       [(7, 3),\n                        (11, 12),\n                        (6, 0.22413225269763748, 0.0, 0.5),\n                        (10, 0.22413225269763748, 0.0, 0.5),\n                        (5, 0.1399779753714987, 0.0, 0.4999999999999998),\n                        (9, 0.1399779753714987, 0.0, 0.5)],\n                       [(7, 0.1852829437333603, 0.0, -0.5),\n                        (11, 0.1852829437333603, 0.0, -0.5),\n                        (12, 0.5316467170069789, 0.0, -0.5),\n                        (3, 0.5316467170069789, 0.0, -0.5)],\n                       [(6, 7), (10, 11)],\n                       [(11, 12),\n                        (6, 0.45333192403000727, 0.0, 0.5),\n                        (7, 3),\n                        (10, 0.4533319240300076, 0.0, 0.5)],\n                       [(5, 6),\n                        (9, 10),\n                        (12, 0.6205120316187871, -2.220446049250313e-16, 0.19011722026833855),\n                        (3, 0.6205120316187871, -2.220446049250313e-16, 0.19011722026833855)],\n                       [(10, 11),\n                        (12, 8),\n                        (3, 1),\n                        (6, 7),\n                        (5, 0.5, -0.9999999999999993, -0.5),\n                        (9, 0.5, 0.0, -0.5)],\n                       [(10, 0.5, 0.0, 0.5),\n                        (11, 0.3496441065798774, 0.0, 0.5383077432439662),\n                        (6, 0.5, 0.0, -0.5),\n                        (7, 0.3496441065798774, 0.0, 0.5383077432439662),\n                        (12, 0.3448829520684143, 0.0, -0.9847174208946683),\n                        (3, 0.3448829520684143, 0.0, -0.9847174208946683),\n                        (8, 0.4925777780083309, 0.0, 0.5072045568580279),\n                        (1, 0.4925777780083309, 0.0, 0.5072045568580279)],\n                       [(10, 11), (6, 7)],\n                       [(11, 0.33274203901299115, 0.0, -0.3641661882066014),\n                        (7, 0.33274203901299115, 0.0, 0.6358338117933988)],\n                       [(11, 12), (7, 3)],\n                       [(11, 0.9294880475869709, 0.0, -0.1553214972315855),\n                        (12, 0.9291764376708231, 0.0, -0.8897608690712939),\n                        (3, 0.9291764376708231, 0.0, -0.8897608690712939),\n                        (7, 0.9294880475869709, 0.0, 0.844678502768414)],\n                       [(10, 11), (12, 8), (3, 1), (6, 7)],\n                       [(11, 0.9764886499665415, 0.0, 0.4442229381785019),\n                        (12, 0.29149057155166774, 0.0, -0.9963773855818847),\n                        (7, 0.9764886499665415, 0.0, 0.4442229381785019),\n                        (3, 0.29149057155166774, 0.0, -0.9963773855818847),\n                        (8, 0.41571419491182243, 0.0, 0.41360830056593145),\n                        (1, 0.41571419491182243, 0.0, 0.41360830056593145)],\n                       [(11, 12), (7, 3)],\n                       [(11, 0.322056849702119, 0.0, 0.9220815067881405),\n                        (12, 0.41836047289168304, 0.0, -0.13717120882570644),\n                        (3, 0.41836047289168304, 0.0, -0.13717120882570644),\n                        (7, 0.322056849702119, 0.0, 0.9220815067881405)],\n                       [(10, 11), (12, 8), (3, 1), (6, 7)],\n                       [(11, 0.5910955657739063, 0.0, 0.9904786909552934),\n                        (12, 0.5658106523525603, 0.0, 0.16808726027638188),\n                        (7, 0.5910955657739061, 1.1102230246251565e-16, -0.009521309044706827),\n                        (3, 0.5658106523525603, 0.0, 0.16808726027638188),\n                        (8, 0.30256758326839167, 0.0, -0.5706246524937872),\n                        (1, 0.30256758326839167, 0.0, -0.5706246524937872)],\n                       [(11, 12), (7, 3)],\n                       [(11, 0.5222942734869767, 0.0, 0.2803915814657436),\n                        (12, 0.6380247240052519, 1.1102230246251565e-16, -0.484080776591012),\n                        (3, 0.6380247240052519, 1.1102230246251565e-16, -0.484080776591012),\n                        (7, 0.5222942734869767, -2.220446049250313e-16, -0.719608418534256)],\n                       [(10, 11), (12, 8), (3, 1), (6, 7)],\n                       [(11, 0.36360452844948893, 0.0, -0.8271399306836011),\n                        (12, 0.01708551170125294, 0.0, -0.612766088889827),\n                        (7, 0.3636045284494889, 0.0, -0.8271399306836014),\n                        (3, 0.01708551170125294, 0.0, -0.612766088889827),\n                        (8, 0.32590008584322616, 0.0, -0.5618902727160411),\n                        (1, 0.32590008584322616, 0.0, -0.5618902727160411)],\n                       [(11, 12), (7, 3)],\n                       [(11, 0.9940835612187666, 5.551115123125783e-17, 0.22140319943243209),\n                        (12, 0.7382217283247945, 0.0, 0.37977805535940057),\n                        (3, 0.7382217283247945, 0.0, 0.37977805535940057),\n                        (7, 0.9940835612187666, 0.0, 0.22140319943243147)],\n                       [(10, 11), (12, 8), (3, 1), (6, 7)],\n                       [(11, 0.5303663346437925, 8.881784197001252e-16, 0.6476601232108743),\n                        (12, 0.39031863075905965, 0.0, 0.37055901030011285),\n                        (7, 0.5303663346437923, 0.0, -0.3523398767891255),\n                        (3, 0.39031863075905965, 0.0, 0.37055901030011285),\n                        (8, 0.5100876690078296, 0.0, 0.3944365573706427),\n                        (1, 0.5100876690078296, 0.0, 0.3944365573706427)],\n                       [(11, 12), (7, 3)],\n                       [(11, 0.6025816719136099, 0.0, -0.7190071148261046),\n                        (12, 0.6856477314180824, 0.0, -0.06389918404080985),\n                        (3, 0.6856477314180824, 0.0, -0.06389918404080985),\n                        (7, 0.6025816719136099, 0.0, 0.2809928851738954)],\n                       [(10, 11), (12, 8), (3, 1), (6, 7)],\n                       [(11, 0.8921121044926871, 2.220446049250313e-16, -0.2974379021753091),\n                        (12, 0.3385860029956733, 0.0, -0.2484264677493928),\n                        (7, 0.8921121044926871, 2.220446049250313e-16, -0.2974379021753091),\n                        (3, 0.3385860029956733, 0.0, -0.2484264677493928),\n                        (8, 0.2347749898956015, 0.0, -0.3542327425401759),\n                        (1, 0.2347749898956015, 0.0, -0.3542327425401759)],\n                       [(11, 12), (7, 3)],\n                       [(11, 0.5928759364551912, 0.0, -0.7181239331695775),\n                        (12, 0.44925295079759087, 0.0, 0.6495948004702141),\n                        (3, 0.44925295079759087, 0.0, 0.6495948004702141),\n                        (7, 0.5928759364551912, 0.0, -0.7181239331695775)],\n                       [(10, 11), (12, 8), (3, 1), (6, 7)],\n                       [(11, 0.8105365413512657, 0.0, 0.7770622601851762),\n                        (12, 0.6433165441464112, 0.0, -0.8393988739586637),\n                        (7, 0.8105365413512657, 0.0, -0.22293773981482357),\n                        (3, 0.6433165441464112, 0.0, -0.8393988739586637),\n                        (10, 0.5, 0.0, -0.8847055241022825),\n                        (8, 0.34497587576931366, 0.0, -0.28963794555439826),\n                        (1, 0.34497587576931366, 0.0, -0.28963794555439826),\n                        (6, 0.4999999999999998, 0.0, 0.8847055241022816)],\n                       [(11, 12), (7, 3)],\n                       [(11, 0.3843844916022806, 0.0, 0.24797535828777706),\n                        (12, 0.5260557718525964, 0.0, -0.7587718918316595),\n                        (3, 0.5260557718525964, 0.0, -0.7587718918316595),\n                        (7, 0.3843844916022808, 0.0, -0.7520246417122236)],\n                       [(10, 11), (12, 8), (3, 1), (6, 7)],\n                       [(12, 0.7720515191095958, 1.1102230246251565e-16, -0.25004937488174583),\n                        (3, 0.7720515191095958, 1.1102230246251565e-16, -0.25004937488174583),\n                        (10, 0.5, 0.0, -0.8847055241022825),\n                        (8, 0.4355845015328854, 0.0, -0.8111689735837979),\n                        (1, 0.4355845015328854, 0.0, -0.8111689735837979),\n                        (6, 0.5, 0.0, 0.8847055241022843)],\n                       [(11, 12), (7, 3)],\n                       [(11, 0.5, 0.0, -0.5449316072361736),\n                        (12, 0.13993480154011698, 0.0, 0.03692650045118023),\n                        (3, 0.13993480154011698, 0.0, 0.03692650045118023),\n                        (7, 0.5, 0.0, -0.5449316072361734)],\n                       [(10, 11), (12, 8), (3, 1), (6, 7)],\n                       [(8, 0.3416013957299424, 0.0, -0.7695375446745221),\n                        (10, 0.4546112109132118, 0.0, 0.8128221566805471),\n                        (11, 0.2299569291469802, 0.0, 0.9550683927638275),\n                        (12, 0.893580808387734, 0.0, -0.6698065954512338),\n                        (1, 0.3416013957299424, 0.0, -0.7695375446745221),\n                        (6, 0.4546112109132117, 0.0, -0.8128221566805471),\n                        (7, 0.22995692914698024, 0.0, -0.0449316072361734),\n                        (3, 0.893580808387734, 0.0, -0.6698065954512338)],\n                       [(8, 4), (10, 13), (11, 12), (1, 0), (6, 2), (7, 3)],\n                       [(11, 0.4999999999999997, 0.0, 0.45506839276382594),\n                        (10, 0.5, 0.0, -0.9742891862096363),\n                        (8, 0.4999999999999998, 0.0, -0.18230509551025165),\n                        (1, 0.4999999999999998, 0.0, -0.18230509551025165),\n                        (7, 0.4999999999999998, 0.0, -0.5449316072361738),\n                        (6, 0.4999999999999998, 0.0, 0.9742891862096354),\n                        (4, 0.30408672398469605, 0.0, -0.5000000000000002),\n                        (13, 0.3040867239846962, 0.0, 0.4999999999999998),\n                        (0, 0.30408672398469605, 0.0, -0.5000000000000002),\n                        (2, 0.30408672398469605, 0.0, -0.5000000000000002)],\n                       [(11, 10), (12, 8), (3, 1), (7, 6)],\n                       [(8, 0.8040867239846965, 1.1102230246251565e-16, -0.1823050955102521),\n                        (10, 0.8040867239846965, 0.0, 0.025710813790364107),\n                        (1, 0.8040867239846965, 1.1102230246251565e-16, -0.1823050955102521),\n                        (6, 0.8040867239846965, 0.0, -0.025710813790363662)],\n                       [(8, 4), (10, 13), (1, 0), (6, 2)],\n                       [(10, 0.75, -2.220446049250313e-16, 0.025710813790364107),\n                        (8, 0.75, 1.1102230246251565e-16, -0.18230509551025254),\n                        (1, 0.75, 1.1102230246251565e-16, -0.18230509551025254),\n                        (6, 0.75, 0.0, -0.025710813790363662),\n                        (4, 0.5, 0.0, 0.5),\n                        (13, 0.5, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (2, 0.5, 0.0, 0.5)],\n                       [(11, 10), (12, 8), (3, 1), (7, 6)],\n                       [(8, 0.5, 0.0, 0.8176949044897479),\n                        (10, 0.5, 0.0, 0.025710813790363884),\n                        (1, 0.5, 0.0, 0.8176949044897479),\n                        (6, 0.5, 0.0, -0.02571081379036344),\n                        (11, 0.5, 0.0, -0.8102769651505541),\n                        (12, 0.5, 5.551115123125783e-17, 0.33253251943263223),\n                        (3, 0.5, 5.551115123125783e-17, 0.33253251943263223),\n                        (7, 0.5, 0.0, 0.18972303484944586)],\n                       [(8, 4), (10, 13), (1, 0), (6, 2)],\n                       [(10, 0.25000000000000006, 0.0, -0.9742891862096366),\n                        (8, 0.25000000000000006, 0.0, -0.1823050955102523),\n                        (1, 0.25000000000000006, 0.0, -0.1823050955102523),\n                        (6, 0.25000000000000006, 4.440892098500626e-16, 0.9742891862096352)],\n                       [(11, 10), (12, 8), (3, 1), (7, 6)],\n                       [(8, 0.5, 8.881784197001252e-16, 0.817694904489747),\n                        (10, 0.5, 8.881784197001252e-16, -0.9742891862096372),\n                        (1, 0.5, 8.881784197001252e-16, 0.817694904489747),\n                        (6, 0.5, 0.0, 0.9742891862096354),\n                        (11, 0.5, 0.31027696515055414, -0.8102769651505541),\n                        (12, 0.5, 0.1674674805673675, 0.3325325194326325),\n                        (3, 0.5, 0.1674674805673675, 0.3325325194326325),\n                        (7, 0.5, 0.31027696515055414, -0.8102769651505544)],\n                       [(8, 4), (10, 13), (1, 0), (6, 2)],\n                       [(4, 1.0, 0.0, 0.5),\n                        (8, 0.5, -0.31769490448974746, 0.8176949044897481),\n                        (10, 0.4999999999999998, -0.5257108137903634, 0.025710813790363884),\n                        (13, 1.0, 0.0, 0.5),\n                        (0, 1.0, 0.0, 0.5),\n                        (1, 0.5, -0.31769490448974746, 0.8176949044897481),\n                        (2, 1.0, 0.0, 0.5),\n                        (6, 0.5, -0.4742891862096361, 0.9742891862096361)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],\n           \'qubits\': [(1, 6),\n                      (2, 6),\n                      (3, 5),\n                      (3, 6),\n                      (3, 7),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 5)]},\n (15, 3): {\'method\': \'binary count seed + moving six-CZ decoders\',\n           \'moments\': [[(0, 0.5004931579194969, 0.0, -0.5),\n                        (5, 0.5700348769048169, 0.0, -0.5),\n                        (1, 0.24138580830535591, 0.0, 0.5),\n                        (6, 0.3116198483700336, 0.0, 0.5),\n                        (10, 0.3762212833232641, 0.0, 0.5),\n                        (11, 0.7973888376918397, 0.0, -0.5),\n                        (2, 0.412184131733575, 0.0, -0.5),\n                        (3, 0.4135610417387456, 0.0, -0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (7, 0.412184131733575, 0.0, -0.5),\n                        (8, 0.4135610417387456, 0.0, -0.5),\n                        (9, 0.5, 0.0, 0.5),\n                        (12, 0.412184131733575, 0.0, -0.5),\n                        (13, 0.4135610417387456, 0.0, -0.5),\n                        (14, 0.5, 0.0, 0.5)],\n                       [(0, 5), (1, 6), (10, 11)],\n                       [(0, 0.49774043690417774, 0.0, 0.5),\n                        (1, 0.8436377750014457, 0.0, -0.5),\n                        (5, 0.0022050905969901694, 0.0, 0.5),\n                        (10, 0.3316948803903533, 0.0, -0.5),\n                        (6, 0.1042859540604348, 0.0, 0.5),\n                        (11, 0.6121001089933615, 0.0, 0.5)],\n                       [(0, 1), (5, 10), (6, 11)],\n                       [(0, 0.5, 0.0, -0.5),\n                        (1, 0.9171447574980371, 0.0, 0.5),\n                        (5, 1.0, 0.0, 0.5),\n                        (6, 0.34568332170776583, 0.0, -0.5),\n                        (10, 0.8865593269978381, 0.0, -0.5),\n                        (11, 0.8137793568788922, 0.0, -0.5)],\n                       [(0, 1), (5, 6), (10, 11)],\n                       [(0, 1.0, 0.0, -0.5),\n                        (5, 0.5, 0.0, 0.5),\n                        (1, 1.0, 0.0, -0.5),\n                        (6, 0.9056685254861114, 0.0, 0.5),\n                        (10, 0.2971845674170174, 0.0, 0.5),\n                        (11, 0.24312932209862245, 0.0, 0.5)],\n                       [(0, 5), (1, 6), (10, 11)],\n                       [(0, 0.4065882770911243, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5),\n                        (5, 0.6594265589016369, 0.0, -0.5),\n                        (10, 0.5135612905851601, 0.0, 0.5),\n                        (6, 0.9866805895423636, 0.0, 0.5),\n                        (11, 0.9185973208306801, 2.220446049250313e-16, -0.5000000000000002)],\n                       [(0, 1), (5, 10), (6, 11)],\n                       [(0, 0.3707227555445639, 0.0, 0.5),\n                        (5, 0.3508492030086968, 0.0, -0.5),\n                        (1, 0.329288506053183, 0.0, -0.5),\n                        (6, 0.9117177107952472, 0.0, 0.5),\n                        (11, 0.9874078125129342, 0.0, -0.5)],\n                       [(0, 5), (1, 6), (10, 11)],\n                       [(0, 0.5000000000000001, 0.0, 0.5),\n                        (1, 0.8969972054842089, 2.220446049250313e-16, -0.5000000000000002),\n                        (5, 0.49999999999999994, 0.0, -0.5),\n                        (10, 1.0, 0.0, 0.5),\n                        (6, 0.4014516477690309, 0.0, -0.5),\n                        (11, 0.781287702544128, 0.0, 0.5)],\n                       [(0, 1), (5, 10), (6, 11)],\n                       [(1, 0.16304050725178096, 0.0, 0.5),\n                        (5, 1.0, 0.0, -0.5),\n                        (6, 0.4587625645746655, 0.0, 0.5),\n                        (10, 1.0, 0.0, 0.5),\n                        (11, 0.6233735693344237, 0.0, 0.5)],\n                       [(0, 1), (5, 6), (10, 11)],\n                       [(1, 0.9933638321833571, 0.0, -0.5),\n                        (0, 1.0, 0.0, 0.5),\n                        (6, 0.5170591124614515, 0.0, -0.5),\n                        (11, 0.266571724956362, 0.0, -0.5),\n                        (10, 1.0, 0.0, -0.5)],\n                       [(1, 2), (6, 7), (11, 12)],\n                       [(1, 0.5903120017947893, 0.0, -0.5),\n                        (2, 0.6117890887690124, 0.0, -0.5),\n                        (6, 0.5903120017947893, 0.0, -0.5),\n                        (7, 0.6117890887690124, 0.0, -0.5),\n                        (11, 0.5903120017947893, 0.0, -0.5),\n                        (12, 0.6117890887690124, 0.0, -0.5)],\n                       [(0, 1), (5, 6), (10, 11)],\n                       [(1, 0.7075436162554982, 0.0, 0.5),\n                        (0, 1.0, 0.0, 0.5),\n                        (6, 0.7075436162554982, 0.0, 0.5),\n                        (5, 1.0, 0.0, 0.5),\n                        (11, 0.7075436162554982, 0.0, 0.5),\n                        (10, 1.0, 0.0, 0.5)],\n                       [(1, 2), (6, 7), (11, 12)],\n                       [(1, 0.13376589621282342, 0.0, 0.5),\n                        (2, 0.1417314999479891, 0.0, 0.5),\n                        (6, 0.13376589621282342, 0.0, 0.5),\n                        (7, 0.1417314999479891, 0.0, 0.5),\n                        (11, 0.13376589621282342, 0.0, 0.5),\n                        (12, 0.1417314999479891, 0.0, 0.5)],\n                       [(0, 1), (5, 6), (10, 11)],\n                       [(1, 0.4425626624366138, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (6, 0.4425626624366138, 0.0, 0.5),\n                        (5, 0.5, 0.0, 0.5),\n                        (11, 0.4425626624366138, 0.0, 0.5),\n                        (10, 0.5, 0.0, 0.5)],\n                       [(1, 2), (6, 7), (11, 12)],\n                       [(1, 0.35241638234956674, 0.0, -0.5),\n                        (2, 0.8746983728132804, 0.0, -0.5),\n                        (6, 0.35241638234956674, 0.0, -0.5),\n                        (7, 0.8746983728132804, 0.0, -0.5),\n                        (11, 0.35241638234956674, 0.0, -0.5),\n                        (12, 0.8746983728132804, 0.0, -0.5)],\n                       [(0, 1), (2, 3), (5, 6), (7, 8), (10, 11), (12, 13)],\n                       [(1, 1.0, 0.0, 0.5),\n                        (2, 0.3819668494768557, 0.0, 0.5),\n                        (3, 0.6338602364006151, 0.0, -0.5),\n                        (6, 1.0, 0.0, 0.5),\n                        (7, 0.3819668494768557, 0.0, 0.5),\n                        (8, 0.6338602364006151, 0.0, -0.5),\n                        (11, 1.0, 0.0, 0.5),\n                        (12, 0.3819668494768557, 0.0, 0.5),\n                        (13, 0.6338602364006151, 0.0, -0.5),\n                        (0, 0.5, 0.0, -0.5),\n                        (5, 0.5, 0.0, -0.5),\n                        (10, 0.5, 0.0, -0.5)],\n                       [(1, 2), (6, 7), (11, 12)],\n                       [(2, 0.7376490383100945, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5),\n                        (7, 0.7376490383100945, 0.0, 0.5),\n                        (6, 1.0, 0.0, 0.5),\n                        (12, 0.7376490383100945, 0.0, 0.5),\n                        (11, 1.0, 0.0, 0.5)],\n                       [(2, 3), (7, 8), (12, 13)],\n                       [(2, 0.16018463696253962, 0.0, 0.5),\n                        (3, 0.18023932987586394, 0.0, -0.5000000000000002),\n                        (7, 0.16018463696253962, 0.0, 0.5),\n                        (8, 0.18023932987586394, 0.0, -0.5000000000000002),\n                        (12, 0.16018463696253962, 0.0, 0.5),\n                        (13, 0.18023932987586394, 0.0, -0.5000000000000002)],\n                       [(1, 2), (6, 7), (11, 12)],\n                       [(2, 0.4422317453870271, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5),\n                        (7, 0.4422317453870271, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (12, 0.4422317453870271, 0.0, 0.5),\n                        (11, 0.5, 0.0, 0.5)],\n                       [(2, 3), (7, 8), (12, 13)],\n                       [(2, 0.6666666666666669, 0.0, 0.5),\n                        (3, 0.8040867239846964, 2.220446049250313e-16, -0.5000000000000002),\n                        (7, 0.6666666666666669, 0.0, 0.5),\n                        (8, 0.8040867239846964, 2.220446049250313e-16, -0.5000000000000002),\n                        (12, 0.6666666666666669, 0.0, 0.5),\n                        (13, 0.8040867239846964, 2.220446049250313e-16, -0.5000000000000002)],\n                       [(1, 2), (3, 4), (6, 7), (8, 9), (11, 12), (13, 14)],\n                       [(2, 1.0, 0.0, 0.5),\n                        (3, 0.49999999999999994, 0.0, -0.5),\n                        (4, 0.3040867239846963, 0.0, -0.5),\n                        (7, 1.0, 0.0, 0.5),\n                        (8, 0.49999999999999994, 0.0, -0.5),\n                        (9, 0.3040867239846963, 0.0, -0.5),\n                        (12, 1.0, 0.0, 0.5),\n                        (13, 0.49999999999999994, 0.0, -0.5),\n                        (14, 0.3040867239846963, 0.0, -0.5),\n                        (1, 0.5, 0.0, -0.5),\n                        (6, 0.5, 0.0, -0.5),\n                        (11, 0.5, 0.0, -0.5)],\n                       [(2, 3), (7, 8), (12, 13)],\n                       [(3, 0.8040867239846964, 0.0, 0.5),\n                        (2, 1.0, 0.0, 0.5),\n                        (8, 0.8040867239846964, 0.0, 0.5),\n                        (7, 1.0, 0.0, 0.5),\n                        (13, 0.8040867239846964, 0.0, 0.5),\n                        (12, 1.0, 0.0, 0.5)],\n                       [(3, 4), (8, 9), (13, 14)],\n                       [(3, 0.75, 0.0, 0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (8, 0.75, 0.0, 0.5),\n                        (9, 0.5, 0.0, 0.5),\n                        (13, 0.75, 0.0, 0.5),\n                        (14, 0.5, 0.0, 0.5)],\n                       [(2, 3), (7, 8), (12, 13)],\n                       [(3, 0.5000000000000001, 0.0, -0.5),\n                        (2, 0.5, 0.0, 0.5),\n                        (8, 0.5000000000000001, 0.0, -0.5),\n                        (7, 0.5, 0.0, 0.5),\n                        (13, 0.5000000000000001, 0.0, -0.5),\n                        (12, 0.5, 0.0, 0.5)],\n                       [(3, 4), (8, 9), (13, 14)],\n                       [(3, 0.25, 0.0, 0.5),\n                        (4, 1.0, 0.0, -0.5),\n                        (8, 0.25, 0.0, 0.5),\n                        (9, 1.0, 0.0, -0.5),\n                        (13, 0.25, 0.0, 0.5),\n                        (14, 1.0, 0.0, -0.5)],\n                       [(2, 3), (7, 8), (12, 13)],\n                       [(3, 0.5, 0.0, -0.5),\n                        (8, 0.5, 0.0, -0.5),\n                        (13, 0.5, 0.0, -0.5),\n                        (2, 0.5, 0.0, 0.5),\n                        (7, 0.5, 0.0, 0.5),\n                        (12, 0.5, 0.0, 0.5)],\n                       [(3, 4), (8, 9), (13, 14)],\n                       [(3, 0.5000000000000001, 0.0, 0.5),\n                        (8, 0.5000000000000001, 0.0, 0.5),\n                        (13, 0.5000000000000001, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (3, 6),\n                      (3, 7),\n                      (3, 8),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (4, 8),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8)]},\n (15, 4): {\'method\': \'paired binary counts + recursive coherent decoders\',\n           \'moments\': [[(6, 0.5050311694609142, 0.0, 0.5),\n                        (10, 0.7754283788865404, 0.0, -0.5),\n                        (7, 0.31203074057342906, 0.0, -0.5),\n                        (11, 0.6647424953468708, 0.0, -0.5),\n                        (8, 0.4019600678849752, 0.0, 0.5),\n                        (12, 0.9860634487343363, 0.0, 0.5),\n                        (4, 0.42588759890619404, 0.0, 0.5918062908230985),\n                        (13, 0.75, 0.0, -0.07606118876167),\n                        (9, 0.42588759890619404, 0.0, 0.5918062908230985),\n                        (1, 0.23850197504578505, 1.0, 0.6723366713104195),\n                        (5, 0.23850197504578505, 1.0, 0.6723366713104195),\n                        (0, 0.5, 0.0, 0.5),\n                        (3, 0.5, 0.0, 0.5),\n                        (2, 0.5, 0.0, 0.5),\n                        (14, 0.5, 0.0, 0.5)],\n                       [(6, 10), (7, 11), (8, 12)],\n                       [(6, 0.5066084360344099, 0.0, 0.5),\n                        (7, 0.7976060787857662, 0.0, 0.5),\n                        (10, 0.004284890859710177, 0.0, 0.5),\n                        (11, 0.7535395929271749, 0.0, 0.5),\n                        (8, 0.04960829035671744, 0.0, 0.5),\n                        (12, 0.9957726141502476, 0.0, -0.5000000000000002)],\n                       [(6, 7), (10, 11)],\n                       [(6, 0.3881710437984986, 0.0, 0.5),\n                        (10, 0.8887989454065368, 0.0, 0.5),\n                        (7, 8),\n                        (11, 0.3000520402754841, 0.0, -0.5)],\n                       [(6, 10),\n                        (11, 12),\n                        (7, 0.49999999999999223, 0.0, 0.5),\n                        (8, 0.1993382800735905, 0.0, 0.5)],\n                       [(11, 0.20007974046133922, 0.0, -0.5),\n                        (12, 0.4999999999999998, 0.0, 0.5),\n                        (6, 0.09428404233770227, 0.0, 0.5),\n                        (10, 0.8168684096225483, 0.0, 0.5)],\n                       [(7, 11), (8, 12)],\n                       [(7, 0.6272655967963714, 2.220446049250313e-16, -0.5000000000000002),\n                        (11, 0.139052622364793, 0.0, -0.5000000000000002),\n                        (12, 0.5, 0.0, 0.5),\n                        (8, 0.9921928118257514, 0.0, -0.5)],\n                       [(6, 7), (10, 11)],\n                       [(11, 12), (7, 0.8727344032036237, 0.0, 0.5), (10, 0.5, 1.0, -0.5)],\n                       [(7, 8), (6, 10), (12, 0.8148309319180607, 0.0, 0.5)],\n                       [(7, 0.7233498104809428, 0.0, 0.5),\n                        (8, 0.6959132760153035, 0.0, 0.5),\n                        (10, 0.5, 0.0, -0.5)],\n                       [(7, 11), (8, 12)],\n                       [(10, 11),\n                        (7, 0.0, 1.0, 0),\n                        (8, 0.27414986841876443, 0.0, 0.8626655155555705),\n                        (12, 0.5000000000000009, 0.0, -0.5)],\n                       [(6, 7),\n                        (8, 4),\n                        (12, 13),\n                        (10, 0.5777417536294465, 0.0, -0.8698433920623254),\n                        (11, 0.8009489577761006, 0.0, 0.9999999999999998)],\n                       [(10, 11),\n                        (7, 0.7308754207007768, 1.0, 0.572080485270047),\n                        (8, 0.7812646514021274, 0.0, -0.26303142154550585),\n                        (12, 0.5, -0.9999999999999998, -0.39810208444779915),\n                        (13, 0.2500000000000002, 0.0, 0.9239388112383322),\n                        (4, 0.34615445441753034, 1.0, -0.7912592244739658)],\n                       [(6, 7), (11, 12), (10, 0.12935406511725, 1.0, -0.5318618508996764)],\n                       [(7, 0.2601068822827814, 0.0, -0.3242586920561732),\n                        (6, 0.0, 0.8804502887098096, 0),\n                        (12, 0.37023281907812394, 0.0, -0.7314354177811329),\n                        (11, 0.5, 0.0, 0.30454264866352565)],\n                       [(7, 8), (12, 13), (10, 11)],\n                       [(7, 0.3793257511260212, 0.0, 0.09474578424214686),\n                        (8, 4),\n                        (11, 0.949194148682035, 0.0, 0.3045426486635256),\n                        (12, 0.29863959134576673, 1.0, 0.26856458221886725),\n                        (10, 0.9052420397088912, 1.0, 0.9681381491003223)],\n                       [(6, 7),\n                        (8, 0.4999999999999998, 0.0, -0.23110536055840925),\n                        (11, 12),\n                        (4, 0.5658493200707859, 1.0, 0.8639427624792817)],\n                       [(7, 0.17287007071273214, 1.0, 0.25980625311387395),\n                        (6, 0.4999999999999998, 0.0, -0.011037258340573564),\n                        (12, 0.5, -2.220446049250313e-16, 0.1622120753992698),\n                        (11, 0.5, 0.0, -0.19545735133647457)],\n                       [(7, 8), (12, 13), (10, 11)],\n                       [(7, 0.9245508650264845, 0.0, 0.6066663543154596),\n                        (8, 0.49999999999999845, 0.0, -0.7311053605584099),\n                        (11, 0.5842177387066689, 0.0, -0.6954573513364743),\n                        (12, 2.220446049250313e-16, -0.9999999999999998, 0.5027084528360495),\n                        (10, 0.6063525068195972, 1.0, 0.9681381491003216)],\n                       [(6, 7), (8, 4), (11, 12)],\n                       [(7, 0.5, 1.0, -0.24380250484892163),\n                        (8, 0.4999999999999998, 0.0, -0.033352937607141975),\n                        (6, 0.5, 0.0, 0.9889627416594262),\n                        (12, 0.6297671809218761, 0.0, -0.33778792460073026),\n                        (4, 0.0, 1.0, 0),\n                        (11, 0.5, 0.0, 0.30454264866352565)],\n                       [(7, 8), (12, 13), (10, 11)],\n                       [(7, 0.3485609400244275, 0.0, -0.7438025048489216),\n                        (8, 0.5, 1.1102230246251565e-16, -0.2833529376071433),\n                        (13, 0.6027647192376683, 5.551115123125783e-17, 0.08098229933576112),\n                        (11, 0.8311275895761097, 0.0, -0.6954573513364739),\n                        (12, 0.5, -0.9999999999999998, 0.1622120753992698),\n                        (10, 0.4999999999999971, -0.1186126279883708, -0.03186185089967952)],\n                       [(6, 7), (8, 4), (13, 9), (11, 12)],\n                       [(7, 0.7500000000000004, -2.220446049250313e-16, -0.7438025048489214),\n                        (8, 0.0, 0.30239487438068663, 0),\n                        (6, 0.49999999999999956, 0.0, -0.6605683991761879),\n                        (11, 0.5, 0.0, 0.8045426486635261),\n                        (12, 0.6448425857543856, 1.0, 0.33499248560989875),\n                        (13, 0.7812646514021274, 0.0, -0.637466072544653),\n                        (9, 0.34615445441753034, 1.0, -0.7912592244739658)],\n                       [(7, 8), (11, 12)],\n                       [(7, 0.231174319940596, 4.440892098500626e-16, 0.7561974951510784),\n                        (8, 0.620275804748211, 1.0, 0.01904193677354321),\n                        (12, 0.2601068822827814, 0.0, -0.34232808921666313),\n                        (11, 0.0, 0.3405361655781928, 0)],\n                       [(6, 7), (8, 4), (12, 13)],\n                       [(7, 0.6245911673988008, 0.0, -0.2438025048489212),\n                        (8, 0.5, -2.220446049250313e-16, -0.48095806322645673),\n                        (4, 0.6713639237535922, -2.220446049250313e-16, -0.06321729836404066),\n                        (12, 0.3793257511260208, 0.0, 0.07667638708165692),\n                        (13, 9),\n                        (6, 0.0, 0.39456853286863147, 0)],\n                       [(7, 8),\n                        (4, 1),\n                        (11, 12),\n                        (13, 0.4999999999999998, 0.0, -0.6055400115575558),\n                        (9, 0.5658493200707859, 1.0, 0.8639427624792817)],\n                       [(7, 0.5, 5.551115123125783e-17, 0.25619749515107854),\n                        (8, 0.14044773223521692, 0.0, -0.677373823369489),\n                        (4, 0.8439229341407215, 0.0, -0.6486171704781523),\n                        (12, 0.1728700707127322, 1.0, 0.24173685595338384),\n                        (11, 0.4999999999999998, 0.0, -0.5959342539206143),\n                        (1, 0.5683578105795405, 1.0, -0.878625289122293)],\n                       [(7, 8), (12, 13)],\n                       [(8, 0.9549431959582095, 1.0, -0.9361631794989349),\n                        (12, 0.9245508650264845, 4.440892098500626e-16, 0.5885969571549694),\n                        (13, 0.49999999999999845, 0.0, 0.8944599884424438)],\n                       [(8, 4), (11, 12), (13, 9)],\n                       [(8, 0.3636111207350248, 0.0, 0.35916618562407376),\n                        (4, 0.6798126636824355, 0.0, 0.7329391812008756),\n                        (12, 0.5, 1.0, -0.2618719020094116),\n                        (13, 0.4999999999999998, -2.220446049250313e-16, -0.4077875886062883),\n                        (11, 0.5, 0.0, 0.4040657460793855),\n                        (9, 0.0, 1.0, 0)],\n                       [(7, 8), (4, 1), (12, 13)],\n                       [(8, 0.5747492210962788, 1.0, -0.7292078172811314),\n                        (4, 0.4197402158362804, 0.0, 0.4192254021852564),\n                        (12, 0.3485609400244275, 0.0, -0.7618719020094116),\n                        (1, 0.4206867408155152, 1.0, -0.8412061211954147),\n                        (13, 0.5, 0.0, -0.6577875886062898)],\n                       [(8, 4), (11, 12), (13, 9)],\n                       [(8, 0.05618598641592864, 0.0, 0.4608182776205698),\n                        (4, 0.34060643615624925, 0.0, -0.5955569655647766),\n                        (12, 0.7500000000000004, 0.0, -0.761871902009412),\n                        (13, 0.0, 0.30239487438068663, 0),\n                        (11, 0.49999999999999956, 0.0, 0.7545346052437711)],\n                       [(7, 8), (4, 1), (12, 13)],\n                       [(8, 0.5474162491089652, 1.0, 0.7054319561138249),\n                        (4, 0.185987156616219, 0.0, 0.3777634656856068),\n                        (12, 0.231174319940596, 0.0, 0.7381280979905884),\n                        (1, 0.46193265931565114, 1.0, -0.2459565222449247),\n                        (13, 0.620275804748211, 1.0, -0.35539271422560326)],\n                       [(8, 4), (11, 12), (13, 9)],\n                       [(8, 0.7392621381122315, 0.0, -0.0864826871507991),\n                        (4, 0.30175441391238306, 0.0, -0.29408734155917693),\n                        (12, 0.6245911673988008, 0.0, -0.26187190200941135),\n                        (13, 0.5, 0.0, -0.855392714225603),\n                        (9, 0.6713639237535922, -2.220446049250313e-16, -0.06321729836404066),\n                        (11, 0.0, 0.979465528448672, 0)],\n                       [(7, 8), (4, 1), (12, 13), (9, 5)],\n                       [(8, 0.27259688752465094, 1.0, -0.6313350948479157),\n                        (4, 0.053622665477313765, 0.0, 0.669216178707795),\n                        (12, 0.5, 0.0, 0.23812809799058837),\n                        (13, 0.14044773223521692, 0.0, 0.9481915256313649),\n                        (1, 0.8029611387137003, 1.0, 0.1626984383948975),\n                        (9, 0.8439229341407215, 0.0, -0.6486171704781523),\n                        (5, 0.5683578105795405, 1.0, -0.878625289122293)],\n                       [(8, 4), (12, 13)],\n                       [(8, 0.5582935812841525, 4.440892098500626e-16, 0.6899062182905635),\n                        (4, 0.6852836122352373, 0.0, -0.5591614353017862),\n                        (13, 0.9549431959582095, 0.9999999999999999, 0.6894021695019188)],\n                       [(7, 8), (4, 1), (13, 9)],\n                       [(8, 0.8681490545286747, 1.0, -0.2569643809343165),\n                        (4, 0.4277380566342639, 0.0, -0.8198531009618637),\n                        (7, 0.5, 0.0, -0.549175841781304),\n                        (13, 0.3636111207350248, 0.0, -0.015268465375072537),\n                        (1, 0.881708884870647, 1.0, -0.12980371729230367),\n                        (9, 0.6798126636824355, 0.0, 0.7329391812008756)],\n                       [(8, 4), (12, 13), (9, 5)],\n                       [(8, 0.6515066280755871, 4.440892098500626e-16, 0.5998099557428174),\n                        (4, 0.11927728230983443, 0.0, -0.680306498224214),\n                        (13, 0.5747492210962788, 1.0, 0.8963575317197221),\n                        (9, 0.4197402158362804, 0.0, 0.4192254021852564),\n                        (5, 0.4206867408155152, 1.0, -0.8412061211954147)],\n                       [(7, 8), (4, 1), (13, 9)],\n                       [(4, 0.7621154011768825, 0.0, 0.7141285609616803),\n                        (13, 0.05618598641592864, 0.0, 0.08638362662142329),\n                        (1, 0.6174170730099287, 1.0, -0.20873821911732615),\n                        (7, 0.5, -0.9999999999999998, -0.5491758417813015),\n                        (9, 0.34060643615624925, 0.0, -0.5955569655647766)],\n                       [(8, 4), (12, 13), (9, 5)],\n                       [(4, 0.7571708757236936, 0.0, -0.03585145212944951),\n                        (8, 0.5, 0.0, -0.35544091703166614),\n                        (13, 0.5474162491089651, -0.9999999999999998, 0.33099730511467906),\n                        (9, 0.185987156616219, 0.0, 0.3777634656856068),\n                        (5, 0.46193265931565114, 1.0, -0.2459565222449247)],\n                       [(4, 1), (7, 8), (13, 9)],\n                       [(1, 0.24679677437537367, 0.0, -0.7565306828698788),\n                        (7, 0.6218015137737606, 0.0, -0.33643201245421217),\n                        (8, 0.041748308476002194, 0.0, 0.14455908296833364),\n                        (4, 0.5323582538647063, 0.0, 0.7199241872050406),\n                        (13, 0.7392621381122315, 0.0, -0.4609173381499456),\n                        (9, 0.30175441391238306, 0.0, -0.29408734155917693)],\n                       [(1, 0), (7, 3), (8, 4), (12, 13), (9, 5)],\n                       [(1, 0.4999999999999998, 1.0, -0.8523760134876583),\n                        (8, 0.5, 0.0, 0.6445590829683334),\n                        (7, 0.4999999999999998, 1.0, 0.02765915730966345),\n                        (0, 0.3040867239846962, 0.0, -0.5000000000000002),\n                        (3, 0.3040867239846962, 0.0, -0.5000000000000002),\n                        (13, 0.272596887524651, 1.0, 0.9942302541529373),\n                        (9, 0.053622665477313765, 0.0, 0.669216178707795),\n                        (5, 0.8029611387137003, 1.0, 0.1626984383948975)],\n                       [(4, 1), (8, 7), (13, 9)],\n                       [(1, 0.8040867239846965, 5.551115123125783e-17, 0.14762398651234138),\n                        (7, 0.8040867239846965, 0.0, -0.9723408426903366),\n                        (4, 0.0, 0.8953840330325824, 0),\n                        (8, 0.0, 0.5340124361070497, 0),\n                        (13, 0.5582935812841525, 0.0, 0.31547156729141723),\n                        (9, 0.6852836122352373, 0.0, -0.5591614353017862)],\n                       [(1, 0), (7, 3), (12, 13), (9, 5)],\n                       [(1, 0.75, 5.551115123125783e-17, 0.14762398651234138),\n                        (7, 0.7499999999999999, 0.0, -0.9723408426903366),\n                        (0, 0.5, 0.0, 0.5),\n                        (3, 0.5, 0.0, 0.5),\n                        (13, 0.8681490545286747, 1.0, -0.6313990319334628),\n                        (9, 0.4277380566342639, 0.0, -0.8198531009618637),\n                        (12, 0.4999999999999997, 0.0, -0.5672452389417941),\n                        (5, 0.881708884870647, 1.0, -0.12980371729230367)],\n                       [(4, 1), (8, 7), (13, 9)],\n                       [(1, 0.5, 0.0, -0.8523760134876583),\n                        (7, 0.5, 0.0, 0.027659157309663396),\n                        (4, 0.5, 0.0, 0.23598015828894137),\n                        (8, 0.5, 4.440892098500626e-16, 0.5861693446465608),\n                        (13, 0.6515066280755871, 5.551115123125783e-17, 0.22537530474367035),\n                        (9, 0.11927728230983443, 0.0, -0.680306498224214)],\n                       [(1, 0), (7, 3), (12, 13), (9, 5)],\n                       [(1, 0.25, 0.0, 0.14762398651234143),\n                        (7, 0.25000000000000006, 0.0, -0.9723408426903366),\n                        (9, 0.7621154011768825, 0.0, 0.7141285609616803),\n                        (5, 0.6174170730099287, 1.0, -0.20873821911732615),\n                        (12, 0.5, 1.0, -0.5672452389417917)],\n                       [(4, 1), (8, 7), (13, 9)],\n                       [(1, 0.5, 1.0, -0.8523760134876583),\n                        (7, 0.5, 1.0, 0.027659157309663396),\n                        (9, 0.7571708757236936, 0.0, -0.03585145212944951),\n                        (13, 0.5, 0.0, -0.7298755680308129),\n                        (4, 0.5, 0.26401984171105863, 0.23598015828894137),\n                        (8, 0.5, -0.08616934464656101, 0.5861693446465608)],\n                       [(1, 0), (7, 3), (9, 5), (12, 13)],\n                       [(5, 0.24679677437537367, 0.0, -0.7565306828698788),\n                        (12, 0.6218015137737606, 0.0, -0.35450140961470233),\n                        (13, 0.041748308476002194, 0.0, -0.2298755680308131),\n                        (9, 0.5323582538647063, 0.0, 0.7199241872050406),\n                        (0, 1.0, 0.0, 0.5),\n                        (1, 0.5, 0.35237601348765857, 0.14762398651234143),\n                        (3, 1.0, 0.0, 0.5),\n                        (7, 0.5, -0.5276591573096634, -0.9723408426903366)],\n                       [(5, 2), (12, 14), (13, 9)],\n                       [(5, 0.4999999999999998, 1.0, -0.8523760134876583),\n                        (13, 0.5, 0.0, 0.27012443196918756),\n                        (12, 0.5, 1.0, 0.00958976014917351),\n                        (2, 0.3040867239846962, 0.0, -0.5000000000000002),\n                        (14, 0.3040867239846962, 0.0, -0.5000000000000002)],\n                       [(9, 5), (13, 12)],\n                       [(5, 0.8040867239846965, 5.551115123125783e-17, 0.14762398651234138),\n                        (12, 0.8040867239846965, 4.440892098500626e-16, -0.990410239850827),\n                        (9, 0.0, 0.8953840330325824, 0),\n                        (13, 0.0, 0.5340124361070497, 0)],\n                       [(5, 2), (12, 14)],\n                       [(5, 0.75, 5.551115123125783e-17, 0.14762398651234138),\n                        (12, 0.75, 4.440892098500626e-16, -0.990410239850827),\n                        (2, 0.5, 0.0, 0.5),\n                        (14, 0.5, 0.0, 0.5)],\n                       [(9, 5), (13, 12)],\n                       [(5, 0.5, 0.0, -0.8523760134876583),\n                        (12, 0.5000000000000002, 0.0, 0.00958976014917351),\n                        (9, 0.5, 0.0, 0.23598015828894137),\n                        (13, 0.5, 1.1102230246251565e-16, 0.2117346936474142)],\n                       [(5, 2), (12, 14)],\n                       [(5, 0.25, 0.0, 0.14762398651234143),\n                        (12, 0.25000000000000006, 4.440892098500626e-16, -0.990410239850827)],\n                       [(9, 5), (13, 12)],\n                       [(5, 0.5, 1.0, -0.8523760134876583),\n                        (12, 0.5, 1.0, 0.00958976014917351),\n                        (9, 0.5, 0.26401984171105863, 0.23598015828894137),\n                        (13, 0.5, 0.2882653063525858, 0.2117346936474142)],\n                       [(5, 2), (12, 14)],\n                       [(2, 1.0, 0.0, 0.5),\n                        (5, 0.5, 0.35237601348765857, 0.14762398651234143),\n                        (12, 0.5, -0.5095897601491735, -0.9904102398508265),\n                        (14, 1.0, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],\n           \'qubits\': [(1, 6),\n                      (2, 6),\n                      (2, 7),\n                      (3, 5),\n                      (3, 6),\n                      (3, 7),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 6)]},\n (15, 5): {\'method\': \'paired binary counts + recursive coherent decoders\',\n           \'moments\': [[(7, 0.6275002275345876, 0.0, -0.5),\n                        (6, 0.5489789454250178, 0.0, 0.5),\n                        (10, 0.16462440927981215, 0.0, -0.5000000000000002),\n                        (8, 0.15428447475915785, 0.0, 0.5),\n                        (12, 0.7654787523830568, 0.0, 0.5),\n                        (13, 0.4218790180730516, 0.0, 0.5),\n                        (4, 0.3460962797882059, 0.0, -0.5573948904901638),\n                        (9, 0.3460962797882059, 0.0, -0.5573948904901638),\n                        (5, 0.4652411656670723, 0.0, -0.020153833182791514),\n                        (1, 0.4652411656670723, 0.0, -0.020153833182791514),\n                        (3, 0.5, 0.0, 0.5),\n                        (2, 0.5, 0.0, 0.5),\n                        (14, 0.5, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5)],\n                       [(7, 11), (6, 10), (8, 12)],\n                       [(7, 0.8944734479102134, 0.0, 0.5),\n                        (6, 0.9463453303367797, 0.0, 0.5),\n                        (10, 0.2732330958832385, 0.0, 0.5),\n                        (8, 0.5236972827102715, 0.0, 0.5),\n                        (12, 0.8346328303977621, 0.0, -0.5000000000000002)],\n                       [(6, 7), (10, 11)],\n                       [(7, 0.8480064173628117, 0.0, 0.5), (10, 0.7996673274560251, 0.0, -0.5), (11, 12)],\n                       [(7, 8), (6, 10), (12, 0.4196168908097533, 0.0, -0.5)],\n                       [(7, 0.834181082212214, 0.0, -0.5),\n                        (10, 0.33750642686758914, 0.0, 0.5),\n                        (8, 0.35521166835174856, 0.0, 0.5)],\n                       [(7, 11), (8, 12), (6, 10)],\n                       [(7, 0.7110369344122653, 0.0, 0.5),\n                        (11, 0.5, 0.0, 0.5),\n                        (8, 0.5945881689715726, 0.0, -0.5),\n                        (12, 0.45214815790160445, 0.0, 0.4999999999999998)],\n                       [(6, 7), (10, 11)],\n                       [(11, 0.4999999999999998, 0.0, 0.5), (7, 0.20908557661949767, 0.0, -0.5)],\n                       [(7, 8), (11, 12)],\n                       [(11, 0.5, 0.0, -0.5), (8, 0.5945881689715726, 0.0, 0.5), (6, 7)],\n                       [(7, 11), (8, 12)],\n                       [(10, 11),\n                        (12, 0.24689718107237368, 0.0, -0.5),\n                        (8, 0.1689528890801011, 0.0, -0.592961532105009),\n                        (7, 0.9749741089553319, 0.0, 0.6358950352924495)],\n                       [(11, 0.24999999999999734, 0.0, -0.5), (12, 13), (6, 7), (8, 4)],\n                       [(10, 11),\n                        (12, 0.3433738949856535, 0.0, -0.5),\n                        (13, 0.5815495709767187, 0.0, -0.5),\n                        (7, 0.4861989543308627, 0.0, 0.5821059397827446),\n                        (8, 0.4653211182942605, -1.1102230246251565e-16, -0.2425887145221114),\n                        (4, 0.5295069129729826, 0.0, -0.5893209314959036)],\n                       [(11, 0.7500000000000027, 0.0, 0.5), (7, 8)],\n                       [(11, 12),\n                        (8, 0.6827527082828264, 0.0, 0.11572279225905824),\n                        (7, 0.44396062653016743, 0.0, -0.24768938518961336)],\n                       [(12, 0.48490630783530553, 0.0, -0.5), (11, 0.5, 0.0, -0.5), (6, 7), (8, 4)],\n                       [(12, 13),\n                        (10, 11),\n                        (7, 0.3780302949345278, 0.0, 0.25231061481038664),\n                        (8, 0.10695351794258778, 0.0, 0.6157227922590531),\n                        (4, 0.40668095023551065, 0.0, 0.8713162318815462)],\n                       [(11, 0.5000000000000004, 0.0, -0.5),\n                        (12, 0.7722728776982923, 0.0, -0.5),\n                        (13, 0.807120789296266, 0.0, 0.5),\n                        (10, 0.4999999999999998, 0.0, -0.5000000000000002),\n                        (7, 8)],\n                       [(11, 12),\n                        (7, 0.6406239893192525, 0.0, 0.7523106148103818),\n                        (8, 0.4664424050737874, 4.440892098500626e-16, 0.5064609782870713)],\n                       [(12, 0.980444474956224, 0.0, 0.5000000000000002), (10, 11), (6, 7), (8, 4)],\n                       [(12, 13),\n                        (10, 0.4999999999999998, 0.0, 0.5),\n                        (8, 0.19782660459849954, 0.0, 0.9904505033437283),\n                        (7, 0.4425354986852117, 0.0, 0.2523106148103811),\n                        (4, 0.5492710367616964, 0.0, 0.4063504694497735)],\n                       [(12, 0.8433224590410755, 0.0, 0.5), (13, 0.3107641817847775, 0.0, 0.5), (7, 8)],\n                       [(11, 12),\n                        (7, 0.1847466179741335, 0.0, 0.7083081978291652),\n                        (8, 0.7684689111399088, 0.0, -0.508723418058195)],\n                       [(11, 0.5, 0.0, 0.5),\n                        (12, 0.35655562966777365, 0.0, 0.4999999999999998),\n                        (6, 7),\n                        (8, 4)],\n                       [(12, 13),\n                        (10, 11),\n                        (7, 0.5432475928713614, 0.0, -0.2940676714575402),\n                        (8, 0.550139150489624, 0.0, 0.7626340346190945),\n                        (6, 0.4999999999999998, 0.0, -0.7374399524154125),\n                        (4, 0.8902926131088993, 0.0, 0.02516975404100341)],\n                       [(11, 0.5, 0.0, 0.5),\n                        (12, 0.774978129060625, 0.0, -0.5),\n                        (13, 0.7062146922843116, 0.0, 0.5),\n                        (7, 8)],\n                       [(11, 12),\n                        (7, 0.15727698463740847, 0.0, -0.4810052112245493),\n                        (8, 0.7052739125587255, 0.0, 0.9015685425559576)],\n                       [(12, 0.2415487281680624, 0.0, 0.5), (11, 0.5, 0.0, 0.5), (8, 4), (6, 7)],\n                       [(12, 13),\n                        (10, 11),\n                        (8, 0.2537577306592731, 0.0, -0.598431457444043),\n                        (7, 0.5, 0.0, 0.7146809541370516),\n                        (6, 0.6680110440239018, 0.0, -0.23743995241541216),\n                        (4, 0.32806090175681435, 0.0, 0.47350959105691115)],\n                       [(11, 0.5, 0.0, -0.5),\n                        (12, 0.7422868369579267, 0.0, -0.5),\n                        (13, 0.20234964811607892, 0.0, 0.5),\n                        (10, 0.4999999999999998, 0.0, 0.5),\n                        (7, 8)],\n                       [(11, 12), (7, 0.5, 0.0, -0.2853190458629483), (8, 4)],\n                       [(12, 0.496112015128797, 0.0, -0.5),\n                        (10, 11),\n                        (6, 7),\n                        (4, 0.6783645703167935, -1.1102230246251565e-16, -0.026490408943089072)],\n                       [(12, 13),\n                        (11, 0.773996237406807, 0.0, -0.5),\n                        (10, 0.5, 0.0, -0.5),\n                        (7, 0.6272475297087556, 0.0, 0.05245970268121458),\n                        (6, 0.5000000000000004, 0.0, 0.2625600475845873)],\n                       [(12, 0.5267416122468935, 0.0, -0.5), (13, 0.9222701747459348, 0.0, 0.5), (7, 8)],\n                       [(11, 12),\n                        (7, 0.5879904199402892, 0.0, -0.020269074313716162),\n                        (8, 0.5, 0.0, 0.43685017834345885)],\n                       [(11, 0.8373851781383239, 0.0, -0.5),\n                        (12, 0.5595564916996747, 0.0, -0.5),\n                        (8, 4),\n                        (6, 7)],\n                       [(10, 11),\n                        (12, 13),\n                        (8, 0.2506657363928002, 0.0, -0.06314982165654204),\n                        (7, 0.7259252541215333, 0.0, 0.25677357970688974),\n                        (6, 0.4999999999999998, 0.0, 0.7151316495822655)],\n                       [(11, 0.33780377348990176, 0.0, 0.5), (10, 0.5, 0.0, 0.5), (7, 8)],\n                       [(11, 12),\n                        (7, 0.5, 0.0, -0.8993283327780908),\n                        (8, 0.7552770370149398, -1.1102230246251565e-16, -0.06314982165654126)],\n                       [(12, 0.5007250294616208, 0.0, 0.5),\n                        (11, 0.32479478834301734, 0.0, 0.5),\n                        (6, 7),\n                        (8, 4)],\n                       [(12, 13),\n                        (10, 11),\n                        (4, 0.4952514392902705, 0.0, -0.5242023761833143),\n                        (8, 0.5, 0.0, -0.5631498216565411),\n                        (7, 0.45694135055250906, 0.0, -0.8993283327780908),\n                        (6, 0.5, -0.9529708306550324, 0.21513164958226572)],\n                       [(12, 0.5000000000000011, 0.0, 0.5),\n                        (13, 0.08255256792708421, 0.0, -0.6924045197900222),\n                        (7, 8),\n                        (4, 1),\n                        (10, 0.5, -1.0, -0.5)],\n                       [(11, 12),\n                        (13, 9),\n                        (7, 0.5, 0.0, -0.3993283327780912),\n                        (8, 0.3552017031280632, 0.0, 0.9632395292188658),\n                        (4, 0.6205477744107246, 0.0, 0.9584494648868007),\n                        (1, 0.5, 0.0, -0.5201538331827915)],\n                       [(12, 0.7884989418971764, 4.440892098500626e-16, 0.5167958107250736),\n                        (13, 0.4653211182942605, 0.0, -0.23541166555792326),\n                        (9, 0.5295069129729826, 0.0, -0.5893209314959036),\n                        (8, 4)],\n                       [(11, 12), (7, 8), (4, 0.5, 0.0, -0.5229632789268042)],\n                       [(12, 0.4861989543308627, 0.0, -0.28066359205895897),\n                        (8, 0.7299176310680837, 0.0, -0.11871217842573656),\n                        (4, 1)],\n                       [(12, 13),\n                        (7, 8),\n                        (4, 0.13056868187156478, 0.0, 0.23819279241137092),\n                        (1, 0.44795079368547275, 0.0, 0.3434400796687824)],\n                       [(13, 0.6827527082828264, 0.0, -0.8771001587767537),\n                        (12, 0.44396062653016743, 4.440892098500626e-16, 0.5491317329133991),\n                        (8, 0.5708700815399963, 0.0, -0.40802287191278863),\n                        (7, 0.5, 0.0, -0.27403292917530986)],\n                       [(11, 12), (13, 9), (8, 4)],\n                       [(12, 0.37803029493452756, 0.0, 0.04913173291339934),\n                        (13, 0.10695351794258778, 0.0, -0.37710015877675995),\n                        (9, 0.40668095023551065, 0.0, 0.8713162318815462),\n                        (4, 0.12328076625869655, 4.440892098500626e-16, 0.9908103105731083),\n                        (8, 0.35440396373816796, 0.0, 0.8933969920529579)],\n                       [(12, 13), (4, 1), (7, 8)],\n                       [(13, 0.4664424050737874, 0.0, 0.5136380272512591),\n                        (12, 0.6406239893192525, 0.0, -0.45086826708659566),\n                        (4, 0.31277527658165183, 0.0, 0.4908103105731083),\n                        (7, 0.5, 0.0, 0.7259670708246899),\n                        (1, 0.23061776402485323, 0.0, -0.41529275648922015)],\n                       [(13, 9), (11, 12), (8, 4)],\n                       [(12, 0.4425354986852119, -1.1102230246251565e-16, 0.04913173291340467),\n                        (13, 0.19782660459849954, 0.0, 0.9976275523079166),\n                        (9, 0.5492710367616964, 0.0, 0.4063504694497735),\n                        (4, 0.7520450774165834, 0.0, -0.009189689426892467),\n                        (8, 0.5, 0.0, 0.5897558007777901)],\n                       [(12, 13), (7, 8), (4, 1)],\n                       [(13, 0.7684689111399088, 0.0, 0.4984536309059937),\n                        (12, 0.1847466179741335, 0.0, -0.4068658501053799),\n                        (4, 0.22588638999651067, 0.0, 0.4908103105731081),\n                        (8, 0.5, 0.0, -0.4102441992222099),\n                        (1, 0.39222583923283394, 0.0, -0.04993017182812021),\n                        (7, 0.6321947838533055, -1.1102230246251565e-16, -0.40156849022298924)],\n                       [(13, 9), (11, 12), (8, 4)],\n                       [(12, 0.5432475928713614, 0.0, 0.5955100191813256),\n                        (13, 0.550139150489624, -1.1102230246251565e-16, -0.2301889164167169),\n                        (11, 0.4999999999999998, 0.0, -0.7374399524154129),\n                        (9, 0.8902926131088993, 0.0, 0.02516975404100341),\n                        (4, 0.696715986124177, 0.0, -0.5091896894268917),\n                        (7, 8)],\n                       [(12, 13), (4, 1), (7, 0.6771139237982053, 0.0, -0.9898069885430896)],\n                       [(12, 0.15727698463740847, 0.0, 0.7824475589483355),\n                        (13, 0.7052739125587255, 0.0, 0.9087455915201457),\n                        (4, 0.583609737771011, 0.0, -0.009189689426893466),\n                        (1, 0.6050120054676125, 0.0, 0.4640144763916523)],\n                       [(13, 9), (11, 12), (8, 4)],\n                       [(12, 0.5, 0.0, -0.4132386064132658),\n                        (13, 0.2537577306592731, 0.0, -0.5912544084798561),\n                        (11, 0.6680110440239018, 0.0, 0.7625600475845877),\n                        (9, 0.32806090175681435, 0.0, 0.47350959105691115),\n                        (8, 0.5, 0.0, 0.8229065369864876),\n                        (4, 0.06412882081148275, 0.0, 0.9908103105731101)],\n                       [(12, 13), (4, 1), (7, 8)],\n                       [(12, 0.5, 4.440892098500626e-16, 0.5867613935867337),\n                        (13, 9),\n                        (8, 0.5, 0.0, -0.17709346301351248),\n                        (4, 0.49999999999999956, 0.0, 0.9908103105731083),\n                        (1, 0.6668726728064027, 0.0, -0.23156229310064602)],\n                       [(11, 12),\n                        (9, 0.6783645703167935, -1.1102230246251565e-16, -0.026490408943089072),\n                        (8, 4)],\n                       [(12, 0.6272475297087556, 0.0, 0.24898264504257117),\n                        (11, 0.5000000000000004, 0.0, 0.26256004758458706),\n                        (8, 0.5, 0.0, 0.2724410650539959),\n                        (4, 0.48422238672257323, 0.0, -0.4583102852578924)],\n                       [(12, 13), (4, 1), (7, 8)],\n                       [(13, 0.5, 0.0, 0.444027227307646),\n                        (12, 0.5879904199402892, 0.0, 0.32171142203750214),\n                        (8, 0.2714423079803394, 0.0, 0.7724410650539966),\n                        (4, 0.5, 0.0, -0.9173049822070471),\n                        (1, 0.7832164982209655, -1.1102230246251565e-16, -0.48502880815679716),\n                        (7, 0.7832164982209657, 0.0, 0.06519191010530578)],\n                       [(13, 9), (11, 12), (7, 3), (8, 4), (1, 0)],\n                       [(12, 0.7259252541215333, 0.0, 0.04466876801689601),\n                        (13, 0.2506657363928002, 0.0, -0.05597277269235479),\n                        (11, 0.4999999999999998, -1.1102230246251565e-16, -0.2848683504177342),\n                        (1, 0.5, 4.440892098500626e-16, -0.985028808156797),\n                        (7, 0.5, 0.0, -0.43480808989469444),\n                        (8, 0.4999999999999998, 0.0, 0.2724410650539981),\n                        (0, 0.30408672398469627, 0.0, -0.5000000000000002),\n                        (3, 0.30408672398469627, 0.0, -0.5000000000000002)],\n                       [(12, 13), (8, 7), (4, 1)],\n                       [(13, 0.7552770370149398, 0.0, 0.9440272273076464),\n                        (12, 0.5, 0.0, -0.799229319498123),\n                        (7, 0.8040867239846965, 0.0, 0.5651919101053062),\n                        (1, 0.8040867239846965, 0.0, 0.014971191843202503)],\n                       [(11, 12), (13, 9), (1, 0), (7, 3)],\n                       [(9, 0.4952514392902705, 0.0, -0.5242023761833143),\n                        (13, 0.5, 0.0, 0.444027227307646),\n                        (12, 0.45694135055250906, 0.0, -0.799229319498125),\n                        (7, 0.75, 0.0, 0.5651919101053062),\n                        (1, 0.75, 0.0, 0.014971191843202503),\n                        (3, 0.5, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (11, 0.5, -0.9529708306550329, 0.21513164958226616)],\n                       [(12, 13), (9, 5), (4, 1), (8, 7)],\n                       [(12, 0.5, 0.0, 0.7007706805018767),\n                        (9, 0.6205477744107246, 0.0, 0.9584494648868007),\n                        (13, 0.3552017031280632, 0.0, 0.9704165781830534),\n                        (5, 0.5, 0.0, -0.5201538331827915),\n                        (7, 0.5, 4.440892098500626e-16, 0.5651919101053058),\n                        (1, 0.5, 0.0, 0.014971191843202947),\n                        (8, 0.5, 0.0, -0.8691671840564845),\n                        (4, 0.5, 0.0, -0.5589132313175298)],\n                       [(12, 13), (1, 0), (7, 3)],\n                       [(13, 9), (1, 0.25, 0.0, -0.9850288081567977), (7, 0.25, 0.0, -0.4348080898946939)],\n                       [(13, 0.7299176310680837, 0.0, 0.8884648705384504),\n                        (9, 0.5, 0.0, -0.5229632789268042),\n                        (8, 7),\n                        (4, 1)],\n                       [(9, 5),\n                        (12, 13),\n                        (1, 0.5, 4.440892098500626e-16, -0.985028808156797),\n                        (7, 0.5, 0.0, -0.4348080898946941),\n                        (8, 0.5, -0.630832815943517, 0.13083281594351548),\n                        (4, 0.5, -0.9410867686824709, 0.44108676868246977)],\n                       [(9, 0.13056868187156478, 0.0, 0.23819279241137092),\n                        (13, 0.5708700815399963, 0.0, -0.4008458229486005),\n                        (5, 0.44795079368547275, 0.0, 0.3434400796687824),\n                        (12, 0.5, 0.0, 0.5754752768990936),\n                        (7, 3),\n                        (1, 0)],\n                       [(13, 9),\n                        (0, 1, 0, 0.5),\n                        (1, 0.5, -0.5149711918432023, -0.9850288081567977),\n                        (7, 0.5, 0.9348080898946942, -0.4348080898946941),\n                        (3, 1, 0, 0.5)],\n                       [(9, 0.12328076625869655, 4.440892098500626e-16, 0.9908103105731083),\n                        (13, 0.35440396373816796, 0.0, 0.9005740410171459)],\n                       [(9, 5), (12, 13)],\n                       [(9, 0.31277527658165183, 0.0, 0.4908103105731083),\n                        (12, 0.5, 0.0, -0.4245247231009053),\n                        (5, 0.23061776402485323, 0.0, -0.41529275648922015)],\n                       [(13, 9)],\n                       [(13, 0.5, 0.0, -0.40306715025802187),\n                        (9, 0.7520450774165834, 0.0, -0.009189689426892467)],\n                       [(9, 5), (12, 13)],\n                       [(13, 0.5, 0.0, -0.40306715025802187),\n                        (9, 0.22588638999651067, 0.0, 0.4908103105731081),\n                        (5, 0.39222583923283394, 0.0, -0.04993017182812021),\n                        (12, 0.6321947838533057, 0.0, 0.7030108379467759)],\n                       [(13, 9)],\n                       [(9, 0.696715986124177, 0.0, -0.5091896894268917), (12, 13)],\n                       [(9, 5), (12, 0.6771139237982053, 0.0, -0.7087506637331247)],\n                       [(9, 0.583609737771011, 0.0, -0.009189689426893466),\n                        (5, 0.6050120054676125, 0.0, 0.4640144763916523)],\n                       [(13, 9)],\n                       [(9, 0.06412882081148275, 0.0, 0.9908103105731101),\n                        (13, 0.5, 0.0, -0.16991641404932323)],\n                       [(9, 5), (12, 13)],\n                       [(13, 0.5, 0.0, -0.16991641404932323),\n                        (9, 0.49999999999999956, 0.0, 0.9908103105731083),\n                        (5, 0.6668726728064027, 0.0, -0.23156229310064602)],\n                       [(13, 9)],\n                       [(9, 0.48422238672257323, 0.0, -0.4583102852578924),\n                        (13, 0.5, 0.0, 0.2796181140181839)],\n                       [(9, 5), (12, 13)],\n                       [(12, 0.7832164982209657, 0.0, 0.23625043761847886),\n                        (5, 0.7832164982209655, -1.1102230246251565e-16, -0.48502880815679716),\n                        (13, 0.2714423079803394, 0.0, -0.2203818859818154),\n                        (9, 0.5, 0.0, -0.9173049822070471)],\n                       [(13, 9), (12, 14), (5, 2)],\n                       [(5, 0.5, 4.440892098500626e-16, -0.985028808156797),\n                        (13, 0.5, 0.0, -0.720381885981815),\n                        (12, 0.5, 0.0, 0.7362504376184795),\n                        (2, 0.30408672398469627, 0.0, -0.5000000000000002),\n                        (14, 0.30408672398469627, 0.0, 0.4999999999999998)],\n                       [(9, 5), (13, 12)],\n                       [(5, 0.8040867239846965, 0.0, 0.014971191843202503),\n                        (12, 0.8040867239846965, -1.1102230246251565e-16, -0.26374956238151936)],\n                       [(12, 14), (5, 2)],\n                       [(5, 0.75, 0.0, 0.014971191843202503),\n                        (12, 0.75, 0.0, -0.26374956238151936),\n                        (2, 0.5, 0.0, 0.5),\n                        (14, 0.5, 0.0, 0.5)],\n                       [(13, 12), (9, 5)],\n                       [(5, 0.5, 0.0, 0.014971191843202947),\n                        (12, 0.5, 0.0, -0.26374956238151936),\n                        (9, 0.5, 0.0, -0.5589132313175298),\n                        (13, 0.5, -1.1102230246251565e-16, 0.13800986490770262)],\n                       [(12, 14), (5, 2)],\n                       [(12, 0.25, 0.0, 0.7362504376184795), (5, 0.25, 0.0, -0.9850288081567977)],\n                       [(13, 12), (9, 5)],\n                       [(5, 0.5, 4.440892098500626e-16, -0.985028808156797),\n                        (12, 0.5, 0.0, 0.7362504376184795),\n                        (13, 0.5, -0.6380098649077031, 0.13800986490770284),\n                        (9, 0.5, -0.9410867686824709, 0.44108676868246977)],\n                       [(5, 2), (12, 14)],\n                       [(2, 1, 0, 0.5),\n                        (12, 0.4999999999999998, -0.23625043761847975, -0.26374956238152014),\n                        (14, 1, 0, 0.5),\n                        (5, 0.5, -0.5149711918432023, -0.9850288081567977)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],\n           \'qubits\': [(1, 6),\n                      (2, 6),\n                      (2, 7),\n                      (3, 5),\n                      (3, 6),\n                      (3, 7),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 6)]},\n (15, 6): {\'method\': \'paired binary counts + recursive coherent decoders\',\n           \'moments\': [[(10, 0.08137430420962379, 0.0, 0.5),\n                        (7, 0.2609018821509444, 0.0, -0.5),\n                        (11, 0.7283497085085496, 0.0, -0.5),\n                        (8, 0.429605463982615, 0.0, -0.5000000000000002),\n                        (12, 0.33039219284883115, 0.0, -0.5),\n                        (4, 0.2961278032423518, 0.0, -0.37072357629548036),\n                        (1, 0.4956530884318588, 5.551115123125783e-17, 0.259121498867743),\n                        (9, 0.2961278032423518, 0.0, -0.37072357629548036),\n                        (5, 0.4956530884318588, 5.551115123125783e-17, 0.259121498867743),\n                        (0, 0.5, 0.0, 0.5),\n                        (3, 0.5, 0.0, 0.5),\n                        (2, 0.5, 0.0, 0.5),\n                        (14, 0.5, 0.0, 0.5)],\n                       [(6, 10), (7, 11), (8, 12)],\n                       [(7, 0.3914883276806753, 0.0, 0.5),\n                        (10, 0.08137430420962379, 0.0, -0.5000000000000002),\n                        (11, 0.55825036779021, 0.0, -0.5),\n                        (8, 0.9159528977943725, 0.0, -0.5),\n                        (12, 0.8476150074274553, 0.0, 0.5)],\n                       [(6, 7), (10, 11)],\n                       [(6, 0.5603562169092022, 0.0, 0.5),\n                        (7, 0.8778237702158793, 0.0, 0.5),\n                        (11, 0.9699203028584522, 0.0, -0.5000000000000002)],\n                       [(6, 10), (7, 8), (11, 12)],\n                       [(7, 0.38153361927704305, 0.0, -0.5000000000000002),\n                        (11, 0.911219943745337, 0.0, 0.5),\n                        (8, 0.5324384000347786, 0.0, -0.5),\n                        (12, 0.6569792924734914, 0.0, -0.5),\n                        (6, 0.06035621690920246, 0.0, -0.5000000000000002)],\n                       [(7, 11), (8, 12)],\n                       [(7, 0.1413838676644081, 0.0, -0.5),\n                        (11, 0.25886131971141557, 0.0, 0.5),\n                        (8, 0.7533172719466821, 0.0, -0.5),\n                        (12, 0.6166998193136692, 0.0, 0.5)],\n                       [(6, 7), (10, 11)],\n                       [(7, 8),\n                        (6, 0.4999999999999998, 0.0, 0.5),\n                        (10, 0.5, 0.0, 0.5),\n                        (11, 0.33007922527760614, 0.0, 0.5)],\n                       [(6, 10),\n                        (11, 12),\n                        (7, 0.5, 0.0, 0.5),\n                        (8, 0.06737977302766152, 0.0, 0.4999999999999998)],\n                       [(11, 0.5236808529404309, 0.0, -0.5000000000000002),\n                        (12, 0.28800009122243897, 0.0, -0.5),\n                        (10, 0.5, 0.0, -0.5)],\n                       [(7, 11), (8, 12)],\n                       [(10, 11),\n                        (7, 0.5, 0.0, 0.5),\n                        (8, 0.6804500291091764, 0.0, 0.5),\n                        (12, 0.1528591556586767, 0.0, 0.5)],\n                       [(6, 7), (8, 4), (12, 13), (11, 0.4492578077627285, 0.0, -0.5)],\n                       [(10, 11),\n                        (6, 0.5051488971691416, 5.551115123125783e-17, 0.38516684131825535),\n                        (7, 0.5, 0.0, -0.3701833787352258),\n                        (12, 0.2032333906758557, 0.0, 0.5),\n                        (8, 0.4999999999999998, 4.440892098500626e-16, 0.7403667574704509),\n                        (13, 0.5077243736649955, 0.0, -0.5),\n                        (4, 0.5, 0.0, -0.8707235762954806)],\n                       [(6, 7), (11, 0.006502424823503717, 0.0, 0.5)],\n                       [(11, 12), (7, 8), (6, 0.22783776220036117, 0.0, -0.1148331586817446)],\n                       [(11, 0.49349757517649645, 0.0, -0.5000000000000002),\n                        (12, 0.7832993633104575, 0.0, 0.5),\n                        (7, 0.4949708987340448, 5.551115123125783e-17, 0.06270197437252095),\n                        (8, 0.550755412437784, 0.0, -0.09655868968129022)],\n                       [(10, 11), (12, 13), (6, 7), (8, 4)],\n                       [(11, 0.0507421922372715, 0.0, 0.5),\n                        (12, 0.46738638831939167, 0.0, 0.5),\n                        (7, 0.5033793022373013, 0.0, 0.562701974372521),\n                        (8, 0.4999999999999998, 0.0, -0.6947798655138215),\n                        (10, 0.5, 0.0, -0.5),\n                        (6, 0.2906009388508253, 0.0, 0.3851668413182551),\n                        (4, 0.340073632931319, 0.0, -0.2109036333971741),\n                        (13, 0.39724376413581114, 0.0, 0.5)],\n                       [(11, 12), (7, 8)],\n                       [(10, 11), (6, 7), (8, 4), (12, 0.8474396067906351, 0.0, -0.5)],\n                       [(12, 13),\n                        (7, 0.5188671153526303, 0.0, 0.00572019374280508),\n                        (8, 0.33147541570683337, 0.0, 0.8052201344861782),\n                        (6, 0.5063547328191557, 0.0, 0.8851668413182552),\n                        (4, 0.3751563089816202, 0.0, -0.5976229662630799),\n                        (10, 0.49999999999999956, 0.0, 0.5)],\n                       [(7, 8), (12, 0.6637573241896284, 0.0, -0.5), (13, 0.7167838474737436, 0.0, -0.5)],\n                       [(11, 12),\n                        (7, 0.6548369278761568, -2.220446049250313e-16, -0.18358485323785656),\n                        (8, 0.8111326152692739, 0.0, 0.305220134486178)],\n                       [(6, 7), (8, 4), (11, 0.5, 0.0, 0.5), (12, 0.6086116775370498, 0.0, -0.5)],\n                       [(10, 11),\n                        (12, 13),\n                        (7, 0.06166130247734847, 0.0, -0.05123889678076665),\n                        (8, 0.015313232122055268, 0.0, 0.805220134486176),\n                        (4, 0.5129607164516408, 0.0, -0.7562970243690965)],\n                       [(7, 8),\n                        (11, 0.5, 0.0, 0.5),\n                        (12, 0.6066862377617777, 0.0, -0.5),\n                        (10, 0.49942367498501183, 0.0, 0.5),\n                        (13, 0.8498386969951102, 0.0, 0.5)],\n                       [(11, 12),\n                        (7, 0.5518863484824026, 0.0, 0.4732030690141271),\n                        (8, 0.8714469607225119, 0.0, 0.3052201344861769)],\n                       [(6, 7), (8, 4), (11, 0.5, 0.0, 0.5), (12, 13)],\n                       [(10, 11),\n                        (7, 0.8128647203031829, 1.1102230246251565e-16, -0.2699341874430212),\n                        (8, 0.8531775343812674, 0.0, 0.8052201344861769),\n                        (12, 0.4869842508446618, 0.0, 0.5),\n                        (4, 0.6608132099476296, 0.0, 0.7425767458343033),\n                        (13, 0.10434522657386716, 0.0, -0.49999999999999956)],\n                       [(7, 8), (11, 0.5, 0.0, -0.5), (10, 0.4999999999999998, 0.0, 0.5)],\n                       [(11, 12),\n                        (7, 0.478873659917565, 5.551115123125783e-17, 0.2743953746437233),\n                        (8, 0.03620506650561751, 0.0, 0.3052201344861767)],\n                       [(10, 11), (6, 7), (8, 4), (12, 0.6754970381762204, 0.0, -0.5)],\n                       [(12, 13),\n                        (7, 0.320266437724436, 0.0, -0.16365077811857942),\n                        (8, 0.5, 0.0, -0.6947798655138229),\n                        (11, 0.6900233870891572, 0.0, -0.5),\n                        (6, 0.5, 0.0, -0.8149083106323871),\n                        (4, 0.4253414728603863, 0.0, -0.7091324900840312),\n                        (10, 0.5, 0.0, -0.5)],\n                       [(7, 8), (12, 0.37468764539728827, 0.0, -0.5)],\n                       [(11, 12),\n                        (7, 0.5, 0.0, 0.47213972929929104),\n                        (8, 0.4457705204849698, 0.0, -0.2287518313721657)],\n                       [(6, 7),\n                        (8, 4),\n                        (11, 0.9264366009426301, 0.0, -0.5),\n                        (12, 0.6635344255240376, 0.0, 0.5)],\n                       [(10, 11),\n                        (12, 13),\n                        (7, 8),\n                        (4, 0.47411281038999165, 0.0, 0.6259258041344009),\n                        (6, 0.5, -0.37018337873522533, -0.8149083106323871)],\n                       [(4, 1),\n                        (11, 0.6067784046198255, 0.0, 0.5),\n                        (12, 0.102662853759375, 0.0, 0.5),\n                        (8, 0.5486314612204102, 0.0, -0.9851782296221454),\n                        (10, 0.4999999999999998, 0.0, 0.5)],\n                       [(11, 12),\n                        (7, 8),\n                        (4, 0.344882952068414, 8.881784197001252e-16, -0.9595787953219059),\n                        (1, 0.4925777780083309, 0.0, 0.5072045568580279)],\n                       [(11, 0.2644015547734993, 0.0, 0.5),\n                        (12, 0.12854213255721647, 0.0, 0.5),\n                        (8, 0.33274203901299115, 0.0, -0.9621734536021593)],\n                       [(10, 11), (12, 13), (8, 4)],\n                       [(12, 0.2967739639205334, 0.0, 0.5),\n                        (13, 9),\n                        (4, 0.9291764376708231, 5.551115123125783e-17, 0.13537775650146772),\n                        (8, 0.9294880475869709, 0.0, 0.8289818554228245),\n                        (10, 0.5, 1.0, -0.5000000000000002)],\n                       [(11, 12),\n                        (4, 1),\n                        (7, 8),\n                        (13, 0.4999999999999998, 0.0, -0.25963324252954934),\n                        (9, 0.5, 0.0, -0.8707235762954806)],\n                       [(11, 0.5051488971691418, 0.0, -0.3851668413182554),\n                        (12, 0.6518568970095876, 0.0, 0.6473192883601617),\n                        (8, 0.9764886499665415, 0.0, 0.22943742001273876),\n                        (4, 0.2914905715516678, 0.0, 0.028761239990876486),\n                        (1, 0.41571419491182243, 0.0, 0.41360830056593145)],\n                       [(11, 12), (8, 4)],\n                       [(12, 13),\n                        (4, 0.4183604728916834, 0.0, -0.11203258325294407),\n                        (8, 0.3220568497021188, 0.0, -0.24842114859690034),\n                        (11, 0.2278377622003611, 0.0, -0.8851668413182554)],\n                       [(4, 1),\n                        (7, 8),\n                        (12, 0.4949708987340448, 0.0, 0.28612578071562367),\n                        (13, 0.550755412437784, 0.0, -0.09655868968129022)],\n                       [(11, 12),\n                        (13, 9),\n                        (8, 0.5910955657739062, 0.0, -0.3168183327640539),\n                        (4, 0.5658106523525602, 5.551115123125783e-17, 0.19322588584914446),\n                        (1, 0.30256758326839167, 0.0, -0.5706246524937872)],\n                       [(8, 4),\n                        (12, 0.5033793022373012, 0.0, 0.7861257807156217),\n                        (13, 0.4999999999999998, 0.0, -0.6947798655138218),\n                        (9, 0.340073632931319, 0.0, -0.2109036333971741),\n                        (11, 0.2906009388508253, 0.0, -0.3851668413182554)],\n                       [(12, 13),\n                        (4, 0.6380247240052519, 0.0, 0.5410578489817497),\n                        (8, 0.5222942734869767, 5.551115123125783e-17, 0.39326877672549565)],\n                       [(13, 9), (4, 1), (7, 8), (11, 12)],\n                       [(8, 0.36360452844948893, 0.0, -0.4991997111251596),\n                        (4, 0.017085511701252946, 0.0, 0.4123725366829347),\n                        (12, 0.5188671153526302, 0.0, 0.3431075613453358),\n                        (13, 0.33147541570683337, 0.0, -0.19477986551382198),\n                        (1, 0.32590008584322616, 0.0, -0.5618902727160411),\n                        (11, 0.5063547328191556, 5.551115123125783e-17, 0.1148331586817461),\n                        (9, 0.3751563089816202, 0.0, -0.5976229662630799)],\n                       [(8, 4), (12, 13)],\n                       [(4, 0.7382217283247945, 0.0, 0.40491668093216243),\n                        (8, 0.9940835612187666, 0.0, 0.45225715875880823),\n                        (12, 0.6548369278761568, 0.0, 0.5324126083259995),\n                        (13, 0.8111326152692739, 0.0, 0.305220134486178)],\n                       [(4, 1), (7, 8), (11, 12), (13, 9)],\n                       [(8, 0.5303663346437927, 0.0, 0.026000234980365133),\n                        (4, 0.39031863075905965, 0.0, 0.3956976358728742),\n                        (12, 0.06166130247734848, 0.0, -0.5999333481310916),\n                        (13, 0.015313232122055268, 0.0, 0.805220134486176),\n                        (1, 0.5100876690078296, 0.0, 0.3944365573706427),\n                        (9, 0.5129607164516408, 0.0, -0.7562970243690965)],\n                       [(8, 4), (12, 13)],\n                       [(4, 0.6856477314180824, 4.440892098500626e-16, 0.961239441531951),\n                        (8, 0.6025816719136099, 0.0, -0.607332526982657),\n                        (12, 0.5518863484824028, 0.0, 0.8756246860740133),\n                        (13, 0.8714469607225119, 0.0, -0.6947798655138229)],\n                       [(4, 1), (7, 8), (11, 12), (13, 9)],\n                       [(8, 0.8921121044926871, 0.0, 0.9710982603665479),\n                        (4, 0.3385860029956733, 4.440892098500626e-16, 0.7767121578233691),\n                        (12, 0.8128647203031829, 0.0, 0.6187619425311628),\n                        (13, 0.8531775343812674, 0.0, -0.19477986551382287),\n                        (1, 0.2347749898956015, 0.0, -0.3542327425401759),\n                        (9, 0.6608132099476296, 0.0, 0.7425767458343033)],\n                       [(8, 4), (12, 13)],\n                       [(4, 0.44925295079759087, 0.0, 0.6747334260429754),\n                        (8, 0.5928759364551912, 0.0, -0.6082157086391844),\n                        (12, 0.4788736599175647, 1.1102230246251565e-16, 0.0744323804444188),\n                        (13, 0.03620506650561751, 1.1102230246251565e-16, 0.3052201344861768)],\n                       [(4, 1), (7, 8), (11, 12), (13, 9)],\n                       [(8, 0.8105365413512657, 0.0, -0.10340190199393806),\n                        (4, 0.6433165441464112, 0.0, -0.8142602483859023),\n                        (12, 0.320266437724436, 0.0, -0.4875214667932788),\n                        (13, 0.5, 0.0, -0.6947798655138229),\n                        (1, 0.34497587576931366, 0.0, -0.28963794555439826),\n                        (7, 0.5, 0.0, -0.2274741054353795),\n                        (11, 0.5, 4.440892098500626e-16, 0.8149083106323882),\n                        (9, 0.4253414728603863, 0.0, -0.7091324900840312)],\n                       [(8, 4), (12, 13)],\n                       [(4, 0.5260557718525964, 0.0, 0.2663667337411011),\n                        (8, 0.3843844916022808, 0.0, 0.425684999903462),\n                        (12, 0.5, 0.0, 0.8766880257888532),\n                        (13, 0.4457705204849698, 4.440892098500626e-16, 0.7712481686278343)],\n                       [(4, 1), (7, 8), (11, 12), (13, 9)],\n                       [(4, 0.7720515191095958, 8.881784197001252e-16, 0.775089250691015),\n                        (9, 0.47411281038999165, 0.0, 0.6259258041344009),\n                        (12, 13),\n                        (1, 0.4355845015328854, 0.0, -0.8111689735837979),\n                        (7, 0.5, 0.0, 0.772525894564621),\n                        (11, 0.4999999999999998, 0.9999999999999988, 0.8149083106323882)],\n                       [(8, 4),\n                        (9, 5),\n                        (13, 0.5486314612204102, -2.220446049250313e-16, -0.9851782296221445)],\n                       [(4, 0.13993480154011695, 0.0, 0.062065126023943096),\n                        (8, 0.5, 0.0, -0.7814080345725882),\n                        (12, 13),\n                        (9, 0.344882952068414, 8.881784197001252e-16, -0.9595787953219059),\n                        (5, 0.4925777780083309, 0.0, 0.5072045568580279)],\n                       [(4, 1), (7, 8), (13, 0.33274203901299115, 0.0, 0.037826546397841554)],\n                       [(1, 0.3416013957299424, 0.0, -0.7695375446745221),\n                        (7, 0.4546112109132116, 0.0, -0.52994642465255),\n                        (8, 0.22995692914698027, 0.0, -0.2814080345725891),\n                        (4, 0.893580808387734, -2.220446049250313e-16, -0.6446679698784725),\n                        (13, 9)],\n                       [(1, 0),\n                        (7, 3),\n                        (8, 4),\n                        (9, 0.9291764376708231, 5.551115123125783e-17, 0.13537775650146772),\n                        (13, 0.9294880475869709, 0.0, -0.17101814457717412)],\n                       [(1, 0.4999999999999998, 0.0, -0.18230509551025165),\n                        (8, 0.4999999999999998, 0.0, 0.2185919654274131),\n                        (7, 0.5, 0.0, -0.3170577675427335),\n                        (9, 5),\n                        (12, 13),\n                        (0, 0.30408672398469605, 0.0, -0.5000000000000002),\n                        (3, 0.3040867239846962, 0.0, 0.4999999999999998)],\n                       [(4, 1),\n                        (8, 7),\n                        (13, 0.9764886499665415, 0.0, 0.22943742001273848),\n                        (9, 0.2914905715516678, 0.0, 0.028761239990876486),\n                        (5, 0.41571419491182243, 0.0, 0.41360830056593145)],\n                       [(1, 0.8040867239846965, 1.1102230246251565e-16, -0.1823050955102521),\n                        (7, 0.8040867239846965, 1.1102230246251565e-16, -0.31705776754273396),\n                        (13, 9)],\n                       [(1, 0),\n                        (7, 3),\n                        (9, 0.4183604728916834, 0.0, -0.11203258325294407),\n                        (13, 0.32205684970211895, 0.0, -0.24842114859690057)],\n                       [(1, 0.75, 1.1102230246251565e-16, -0.18230509551025254),\n                        (7, 0.75, 2.220446049250313e-16, -0.3170577675427344),\n                        (9, 5),\n                        (12, 13),\n                        (0, 0.5, 0.0, 0.5),\n                        (3, 0.5, 0.0, 0.5)],\n                       [(4, 1),\n                        (8, 7),\n                        (13, 0.5910955657739059, 0.0, 0.6831816672359465),\n                        (9, 0.5658106523525602, 5.551115123125783e-17, 0.19322588584914446),\n                        (5, 0.30256758326839167, 0.0, -0.5706246524937872)],\n                       [(1, 0.5, 0.0, 0.8176949044897479),\n                        (7, 0.5, 4.440892098500626e-16, 0.6829422324572652),\n                        (13, 9),\n                        (4, 0.5, 0.0, -0.642328854994604),\n                        (8, 0.5, 0.0, -0.5160626766582084)],\n                       [(1, 0),\n                        (7, 3),\n                        (9, 0.6380247240052519, 0.0, 0.5410578489817497),\n                        (13, 0.5222942734869767, 0.0, -0.6067312232745039)],\n                       [(1, 0.25000000000000006, 0.0, -0.1823050955102523),\n                        (7, 0.25000000000000006, 0.0, -0.3170577675427353),\n                        (9, 5),\n                        (12, 13)],\n                       [(4, 1),\n                        (8, 7),\n                        (13, 0.36360452844948893, 0.0, -0.49919971112515893),\n                        (9, 0.017085511701252946, 0.0, 0.4123725366829347),\n                        (5, 0.32590008584322616, 0.0, -0.5618902727160411)],\n                       [(1, 0.5, 8.881784197001252e-16, 0.817694904489747),\n                        (7, 0.5, 0.0, 0.682942232457264),\n                        (13, 9),\n                        (4, 0.5, -0.857671145005396, -0.642328854994604),\n                        (8, 0.5, 0.016062676658208375, 0.4839373233417916)],\n                       [(1, 0),\n                        (7, 3),\n                        (9, 0.7382217283247945, 0.0, 0.40491668093216243),\n                        (13, 0.9940835612187666, 0.0, 0.45225715875880845)],\n                       [(9, 5),\n                        (12, 13),\n                        (0, 1.0, 0.0, 0.5),\n                        (1, 0.5, -0.31769490448974746, 0.8176949044897481),\n                        (3, 1.0, 0.0, 0.5),\n                        (7, 0.4999999999999998, -0.1829422324572645, -0.3170577675427355)],\n                       [(13, 0.5303663346437927, 0.0, -0.973999765019635),\n                        (9, 0.39031863075905965, 0.0, 0.3956976358728742),\n                        (5, 0.5100876690078296, 0.0, 0.3944365573706427)],\n                       [(13, 9)],\n                       [(9, 0.6856477314180824, 4.440892098500626e-16, 0.961239441531951),\n                        (13, 0.6025816719136099, 0.0, 0.39266747301734406)],\n                       [(9, 5), (12, 13)],\n                       [(13, 0.8921121044926871, 0.0, 0.971098260366549),\n                        (9, 0.3385860029956733, 4.440892098500626e-16, 0.7767121578233691),\n                        (5, 0.2347749898956015, 0.0, -0.3542327425401759)],\n                       [(13, 9)],\n                       [(9, 0.44925295079759087, 0.0, 0.6747334260429754),\n                        (13, 0.5928759364551912, 0.0, -0.6082157086391833)],\n                       [(9, 5), (12, 13)],\n                       [(13, 0.8105365413512657, 0.0, 0.8965980980060624),\n                        (9, 0.6433165441464112, 0.0, -0.8142602483859023),\n                        (5, 0.34497587576931366, 0.0, -0.28963794555439826),\n                        (12, 0.4999999999999998, 0.0, 0.5763018605235184)],\n                       [(13, 9)],\n                       [(9, 0.5260557718525964, 0.0, 0.2663667337411011),\n                        (13, 0.3843844916022808, 0.0, -0.5743150000965371)],\n                       [(9, 5), (12, 13)],\n                       [(9, 0.7720515191095958, 8.881784197001252e-16, 0.775089250691015),\n                        (5, 0.4355845015328854, 0.0, -0.8111689735837979),\n                        (12, 0.5, -2.220446049250313e-16, -0.4236981394764805)],\n                       [(13, 9)],\n                       [(9, 0.13993480154011695, 0.0, 0.062065126023943096),\n                        (13, 0.5, 0.0, -0.7814080345725871)],\n                       [(9, 5), (12, 13)],\n                       [(5, 0.3416013957299424, 0.0, -0.7695375446745221),\n                        (12, 0.4546112109132116, 0.0, 0.8787741797406896),\n                        (13, 0.22995692914698024, 0.0, 0.7185919654274129),\n                        (9, 0.893580808387734, -2.220446049250313e-16, -0.6446679698784725)],\n                       [(5, 2), (12, 14), (13, 9)],\n                       [(5, 0.4999999999999998, 0.0, -0.18230509551025165),\n                        (13, 0.49999999999999983, 0.0, -0.7814080345725871),\n                        (12, 0.4999999999999998, 0.0, 0.6658855226308735),\n                        (2, 0.30408672398469605, 0.0, -0.5000000000000002),\n                        (14, 0.30408672398469605, 0.0, -0.5000000000000002)],\n                       [(9, 5), (13, 12)],\n                       [(5, 0.8040867239846965, 1.1102230246251565e-16, -0.1823050955102521),\n                        (12, 0.8040867239846965, 8.881784197001252e-16, 0.6658855226308735)],\n                       [(5, 2), (12, 14)],\n                       [(5, 0.75, 1.1102230246251565e-16, -0.18230509551025254),\n                        (12, 0.75, 0.0, 0.6658855226308735),\n                        (2, 0.5, 0.0, 0.5),\n                        (14, 0.5, 0.0, 0.5)],\n                       [(9, 5), (13, 12)],\n                       [(5, 0.5, 0.0, 0.8176949044897479),\n                        (12, 0.5, 0.0, -0.3341144773691269),\n                        (9, 0.5, 0.0, -0.642328854994604),\n                        (13, 0.5, -2.220446049250313e-16, 0.4839373233417934)],\n                       [(5, 2), (12, 14)],\n                       [(5, 0.25000000000000006, 0.0, -0.1823050955102523),\n                        (12, 0.25000000000000006, 0.0, 0.6658855226308735)],\n                       [(9, 5), (13, 12)],\n                       [(5, 0.5, 8.881784197001252e-16, 0.817694904489747),\n                        (12, 0.5, 0.0, -0.3341144773691269),\n                        (9, 0.5, -0.857671145005396, -0.642328854994604),\n                        (13, 0.5, 0.0160626766582066, 0.4839373233417934)],\n                       [(5, 2), (12, 14)],\n                       [(2, 1.0, 0.0, 0.5),\n                        (5, 0.5, -0.31769490448974746, 0.8176949044897481),\n                        (12, 0.5, 0.8341144773691263, -0.33411447736912625),\n                        (14, 1.0, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],\n           \'qubits\': [(1, 6),\n                      (2, 6),\n                      (2, 7),\n                      (3, 5),\n                      (3, 6),\n                      (3, 7),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 6)]},\n (15, 7): {\'method\': \'paired binary counts + recursive coherent decoders\',\n           \'moments\': [[(3, 0.05380912957041928, 0.0, 0.5),\n                        (6, 0.31630918818247244, 0.0, -0.5),\n                        (5, 0.4134881505743113, 0.0, -0.5000000000000002),\n                        (10, 0.9749914697119215, 0.0, -0.5000000000000002),\n                        (11, 0.9138626762504471, 0.0, 0.5),\n                        (12, 0.2670195212955466, 0.0, 0.5),\n                        (13, 0.5134368213488532, 0.0, -0.5),\n                        (9, 0.5134368213488532, 0.0, -0.5),\n                        (8, 0.49565308843185885, 0.0, 0.25912149886774305),\n                        (7, 0.49565308843185885, 0.0, 0.25912149886774305),\n                        (2, 0.5, 0.0, 0.5),\n                        (14, 0.5, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5)],\n                       [(6, 5), (3, 4), (10, 11)],\n                       [(10, 0.9758980292752844, 2.220446049250313e-16, -0.5000000000000002),\n                        (4, 0.3572761584752988, 0.0, 0.5),\n                        (11, 0.08588433755655989, 0.0, -0.5000000000000002),\n                        (5, 0.6503211074889919, 0.0, -0.5),\n                        (6, 0.3755027911010411, 0.0, -0.5),\n                        (3, 0.4461908704295805, 0.0, 0.5)],\n                       [(5, 11), (4, 10)],\n                       [(4, 0.8589803893445076, 0.0, -0.5),\n                        (5, 0.49999999999999867, 0.0, -0.5000000000000002),\n                        (11, 0.49999999999999956, 0.0, -0.5000000000000002)],\n                       [(4, 5), (10, 11)],\n                       [(5, 0.5000000000000011, 0.0, 0.5),\n                        (3, 4),\n                        (10, 0.8278647016465945, 0.0, -0.5000000000000004),\n                        (11, 0.5000000000000004, 0.0, -0.5)],\n                       [(6, 5), (4, 0.8457372208504039, 0.0, 0.5), (3, 0.4999999999999989, 0.0, 0.5)],\n                       [(5, 0.12844403112522307, 0.0, -0.5), (4, 10), (6, 0.5000000000000016, 0.0, -0.5)],\n                       [(5, 11),\n                        (4, 0.6486931696176113, 0.0, 0.4999999999999998),\n                        (10, 0.5674640172984227, 0.0, 0.4999999999999998)],\n                       [(5, 0.31579113051767504, 0.0, -0.5000000000000002), (11, 0.5, 0.0, -0.5)],\n                       [(4, 5), (10, 11)],\n                       [(5, 0.1706789593772302, 0.0, 0.4999999999999998),\n                        (11, 0.5, 0.0, -0.5),\n                        (4, 0.35112917909082464, 0.0, 0.49999999999999956)],\n                       [(6, 5), (3, 4)],\n                       [(5, 0.44784353982792924, 0.0, 0.5),\n                        (4, 0.031841076455933814, 0.0, 0.49999999999999956)],\n                       [(5, 11), (4, 10)],\n                       [(11, 0.22155304972464673, 0.0, 0.5),\n                        (5, 0.5896919606012836, 0.0, 0.5),\n                        (4, 0.16558135883786695, 0.0, -0.5000000000000002),\n                        (10, 0.9373970476727265, 0.0, -0.5000000000000004)],\n                       [(6, 5), (11, 12), (3, 4), (10, 9)],\n                       [(12, 0.5636230767879025, 0.0, 0.5),\n                        (11, 0.9388370446631149, 0.0, 0.5),\n                        (5, 0.9710379810239014, 0.0, 0.5),\n                        (6, 0.5, 0.0, -0.5),\n                        (10, 0.7941033541984499, 0.0, -0.5000000000000004),\n                        (4, 0.32526722636904526, 0.0, 0.5),\n                        (9, 0.09503601272046969, 0.0, 0.4999999999999998)],\n                       [(5, 11), (12, 6), (4, 10)],\n                       [(5, 0.7138078209573102, 0.0, 0.5),\n                        (6, 0.5, 0.0, -0.5),\n                        (11, 0.9311061771381746, 0.0, -0.5000000000000002),\n                        (4, 0.8041266624310963, 0.0, -0.5),\n                        (10, 0.7244331475506502, 0.0, 0.4999999999999998)],\n                       [(11, 12), (6, 5), (3, 4), (10, 9)],\n                       [(6, 0.5, 0.0, 0.5),\n                        (12, 0.5, 0.0, -0.5),\n                        (5, 0.6065488447006204, 0.0, -0.5),\n                        (11, 0.9766369503471868, 0.0, -0.5000000000000002),\n                        (4, 0.23444009082657646, 0.0, 0.5),\n                        (10, 0.8613042944089135, 0.0, 0.4999999999999998),\n                        (9, 0.28294112704061325, 0.0, -0.5)],\n                       [(12, 6), (5, 11), (4, 10)],\n                       [(11, 0.908164340798375, 0.0, -0.5000000000000002),\n                        (12, 0.09634627307113086, 0.0, 0.5),\n                        (6, 0.5909884901079994, 0.0, -0.5),\n                        (5, 0.12127286283001393, 0.0, 0.5),\n                        (4, 0.25071987449186994, 0.0, -0.5000000000000002),\n                        (10, 0.5449608651484801, 0.0, -0.5000000000000004)],\n                       [(11, 12), (6, 5), (3, 4), (10, 9)],\n                       [(5, 0.5, 0.0, 0.5),\n                        (6, 0.6462376519630171, 0.0, 0.5),\n                        (12, 0.21899912015833123, 0.0, 0.5),\n                        (10, 0.4518272206976892, 0.0, 0.49999999999999956),\n                        (4, 0.5623146345847854, 0.0, 0.5),\n                        (9, 0.36485970744316276, 0.0, 0.5)],\n                       [(5, 11), (12, 6), (4, 10)],\n                       [(11, 0.5009307501989091, 0.0, 0.5),\n                        (5, 0.5, 0.0, 0.5),\n                        (12, 0.5, 0.0, 0.5),\n                        (4, 0.7475099191326353, 0.0, -0.5),\n                        (10, 0.43170540271778, 0.0, 0.4999999999999998)],\n                       [(11, 12), (6, 5), (3, 4), (10, 9)],\n                       [(11, 0.9166708423793262, 0.0, 0.5),\n                        (5, 0.7804346776838513, 2.220446049250313e-16, -0.5000000000000002),\n                        (12, 6),\n                        (10, 0.600006446168567, 0.0, 0.4999999999999998),\n                        (4, 0.18635548711858974, 0.0, -0.5),\n                        (9, 0.5948087170803427, 0.0, -0.5)],\n                       [(5, 11), (6, 0.5, 0.0, 0.5), (12, 0.890971554367487, 0.0, -0.5), (4, 10)],\n                       [(5, 0.9436047013024957, 0.0, 0.5),\n                        (11, 0.04820527690334275, 0.0, 0.5),\n                        (10, 0.08545923160647084, 0.0, 0.4999999999999998),\n                        (4, 0.18736031327399982, 0.0, -0.5000000000000002)],\n                       [(6, 5), (11, 12), (3, 4), (10, 9)],\n                       [(12, 0.13352204480907548, 0.0, -0.4999999999999999),\n                        (5, 0.44901981863615203, 0.0, 0.5),\n                        (11, 0.694402647712888, 0.0, -0.5),\n                        (10, 0.2853676947061159, 0.0, 0.4999999999999998),\n                        (4, 0.3095273135966239, 0.0, 0.5),\n                        (9, 0.5609325442841979, 0.0, -0.5)],\n                       [(5, 11), (12, 6), (4, 10)],\n                       [(12, 0.4003523938564044, 0.0, -0.5),\n                        (5, 0.9044215738187418, 0.0, 0.5),\n                        (11, 0.7312496663403019, 0.0, 0.5),\n                        (10, 0.19227288083003602, 0.0, 0.4999999999999998),\n                        (4, 0.7120975782091497, 0.0, -0.5)],\n                       [(11, 12), (6, 5), (10, 9), (3, 4)],\n                       [(5, 0.10682962567577325, 0.0, -0.5),\n                        (12, 0.7925773419100788, 0.0, -0.5),\n                        (11, 0.5996937779346163, 2.220446049250313e-16, -0.5000000000000002),\n                        (10, 0.21224365634312448, 0.0, -0.5000000000000004),\n                        (4, 0.3728883046820175, 0.0, 0.5),\n                        (9, 0.1206646036111223, 0.0, 0.5),\n                        (3, 0.5, 0.0, -0.5)],\n                       [(5, 11), (12, 6), (4, 10)],\n                       [(12, 0.09548414467068067, 0.0, 0.5),\n                        (5, 0.07276045906068562, 0.0, -0.5000000000000002),\n                        (11, 0.423334840900496, 0.0, -0.5),\n                        (10, 0.5972481824285067, 0.0, -0.5000000000000004),\n                        (4, 0.07381131043005107, 0.0, 0.5)],\n                       [(6, 5), (11, 12), (3, 4), (10, 9)],\n                       [(5, 0.4142546780340952, 0.0, 0.5),\n                        (11, 0.8235550199062609, 0.0, -0.5),\n                        (12, 0.01729597549951878, 0.0, -0.49999999999999967),\n                        (4, 0.5, 0.0, -0.5),\n                        (10, 0.3633470223178652, 0.0, -0.5000000000000004),\n                        (3, 0.5000000000000004, 0.0, 0.5),\n                        (9, 0.40745450984385334, 0.0, -0.5)],\n                       [(5, 11), (12, 6), (4, 10)],\n                       [(5, 0.00242025674892421, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (12, 0.5625850197644766, 0.0, -0.5),\n                        (11, 0.6011648116854427, 0.0, -0.5),\n                        (4, 0.5, 0.0, -0.5),\n                        (10, 9)],\n                       [(11, 12), (6, 5), (3, 4), (9, 0.900298441988697, 0.0, 0.5)],\n                       [(11, 0.17327297830743582, 0.0, -0.5),\n                        (12, 0.40681614262679, 0.0, -0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (4, 0.5, 0.0, -0.5),\n                        (3, 0.5, 0.0, 0.5)],\n                       [(5, 11), (12, 6), (4, 10)],\n                       [(12, 0.8300853461329338, 0.0, -0.5),\n                        (11, 0.44426621558009605, 0.0, 0.5),\n                        (10, 0.1210845387975159, 0.0, 0.4999999999999998),\n                        (3, 4),\n                        (6, 1, 0, -1.0)],\n                       [(5, 11),\n                        (12, 13),\n                        (10, 9),\n                        (4, 0.22413225269763748, 0.0, 0.5),\n                        (3, 0.1399779753714987, 0.0, 0.5)],\n                       [(12, 0.7941033541984499, 0.0, -0.5),\n                        (11, 0.32526722636904526, 0.0, 0.5),\n                        (13, 0.09503601272046969, 0.0, 0.4999999999999998),\n                        (10, 0.1852829437333603, 0.0, -0.5000000000000004),\n                        (9, 0.531646717006979, 0.0, -0.5)],\n                       [(11, 12), (4, 10)],\n                       [(12, 0.7244331475506502, 0.0, -0.5),\n                        (11, 0.8041266624310963, 0.0, -0.5),\n                        (4, 0.45333192403000755, 0.0, 0.5),\n                        (10, 9)],\n                       [(5, 11), (12, 13), (3, 4), (9, 0.6205120316187869, 0.0, 0.19011722026833766)],\n                       [(11, 0.23444009082657646, 0.0, -0.5),\n                        (12, 0.8613042944089135, 0.0, -0.5),\n                        (13, 0.28294112704061325, 0.0, -0.5),\n                        (4, 10),\n                        (9, 8),\n                        (3, 0.5, -1.0, 0.4999999999999998)],\n                       [(11, 12),\n                        (10, 0.34964410657987743, 0.0, 0.5383077432439662),\n                        (4, 0.5, 0.0, 0.5),\n                        (9, 0.344882952068414, 0.0, -0.984717420894669),\n                        (8, 0.4925777780083309, 0.0, 0.5072045568580279)],\n                       [(11, 0.25071987449186994, 0.0, 0.5), (12, 0.5449608651484803, 0.0, -0.5), (4, 10)],\n                       [(12, 13), (5, 11), (10, 0.33274203901299115, 0.0, 0.6358338117933988)],\n                       [(12, 0.4518272206976892, 0.0, 0.5),\n                        (11, 0.5623146345847854, 0.0, 0.5),\n                        (13, 0.36485970744316276, 0.0, 0.5),\n                        (10, 9)],\n                       [(11, 12),\n                        (10, 0.9294880475869709, 4.440892098500626e-16, 0.844678502768414),\n                        (9, 0.9291764376708231, 4.440892098500626e-16, -0.8897608690712948)],\n                       [(11, 0.7475099191326353, 0.0, -0.5),\n                        (12, 0.43170540271778, 0.0, -0.5),\n                        (9, 8),\n                        (4, 10)],\n                       [(12, 13),\n                        (5, 11),\n                        (9, 0.2914905715516678, 4.440892098500626e-16, -0.9963773855818856),\n                        (10, 0.9764886499665415, 0.0, 0.4442229381785012),\n                        (8, 0.4157141949118224, 0.0, 0.41360830056593145)],\n                       [(12, 0.600006446168567, 0.0, -0.5),\n                        (11, 0.18635548711858974, 0.0, 0.5),\n                        (13, 0.5948087170803427, 0.0, -0.5),\n                        (10, 9)],\n                       [(11, 12),\n                        (9, 0.418360472891683, 2.220446049250313e-16, -0.1371712088257072),\n                        (10, 0.322056849702119, 0.0, 0.9220815067881403)],\n                       [(11, 0.18736031327399982, 0.0, 0.5),\n                        (12, 0.08545923160647084, 0.0, 0.5),\n                        (9, 8),\n                        (4, 10)],\n                       [(5, 11),\n                        (12, 13),\n                        (10, 0.5910955657739061, 0.0, -0.009521309044706605),\n                        (9, 0.5658106523525603, -1.1102230246251565e-16, 0.1680872602763812),\n                        (8, 0.3025675832683916, 0.0, -0.5706246524937872)],\n                       [(11, 0.3095273135966239, 0.0, 0.5),\n                        (12, 0.2853676947061159, 0.0, 0.5),\n                        (13, 0.5609325442841979, 0.0, -0.5),\n                        (10, 9)],\n                       [(11, 12),\n                        (9, 0.6380247240052519, 0.0, -0.4840807765910127),\n                        (10, 0.5222942734869767, 0.0, -0.7196084185342562)],\n                       [(11, 0.7120975782091497, 0.0, -0.5),\n                        (12, 0.19227288083003602, 0.0, -0.5),\n                        (4, 10),\n                        (9, 8)],\n                       [(5, 11),\n                        (12, 13),\n                        (9, 0.017085511701252942, 0.0, -0.6127660888898279),\n                        (10, 0.3636045284494889, 4.440892098500626e-16, -0.8271399306836016),\n                        (8, 0.32590008584322616, 0.0, -0.5618902727160411)],\n                       [(12, 0.21224365634312448, 0.0, 0.5),\n                        (11, 0.3728883046820175, 0.0, -0.5000000000000002),\n                        (13, 0.1206646036111223, 0.0, 0.5),\n                        (5, 0.4999999999999998, 0.0, -0.5000000000000002),\n                        (10, 9)],\n                       [(11, 12),\n                        (9, 0.7382217283247945, 0.0, 0.3797780553593999),\n                        (10, 0.9940835612187666, -1.1102230246251565e-16, 0.22140319943243147)],\n                       [(12, 0.5972481824285067, 0.0, -0.5),\n                        (11, 0.07381131043005107, 0.0, -0.5000000000000002),\n                        (9, 8),\n                        (4, 10)],\n                       [(12, 13),\n                        (5, 11),\n                        (10, 0.5303663346437923, 0.0, -0.3523398767891256),\n                        (9, 0.3903186307590596, 0.0, 0.3705590103001124),\n                        (8, 0.5100876690078295, 0.0, 0.39443655737064276)],\n                       [(11, 0.5, 0.0, -0.5),\n                        (12, 0.3633470223178652, 0.0, -0.5000000000000002),\n                        (5, 0.5000000000000004, 0.0, -0.5),\n                        (13, 0.40745450984385334, 0.0, -0.5),\n                        (10, 9)],\n                       [(11, 12),\n                        (10, 0.6025816719136099, 0.0, 0.2809928851738954),\n                        (9, 0.6856477314180824, 0.0, -0.06389918404081052)],\n                       [(11, 0.5, 0.0, -0.5), (12, 13), (9, 8), (4, 10)],\n                       [(5, 11),\n                        (13, 0.900298441988697, 0.0, 0.5),\n                        (10, 0.892112104492687, 0.0, -0.29743790217530897),\n                        (9, 0.3385860029956733, 0.0, -0.24842646774939348),\n                        (8, 0.23477498989560153, 0.0, -0.3542327425401759)],\n                       [(11, 0.5, 0.0, 0.5), (5, 0.5, 0.0, 0.5), (10, 9)],\n                       [(11, 12),\n                        (10, 0.5928759364551912, 0.0, -0.7181239331695775),\n                        (9, 0.44925295079759087, 4.440892098500626e-16, 0.6495948004702132)],\n                       [(12, 0.1210845387975159, 0.0, 0.5), (5, 11), (9, 8), (4, 10)],\n                       [(12, 13),\n                        (11, 0.22413225269763748, 0.0, 0.5),\n                        (5, 0.1399779753714987, 0.0, -0.5000000000000002),\n                        (10, 0.8105365413512657, 0.0, -0.22293773981482357),\n                        (9, 0.643316544146411, 4.440892098500626e-16, -0.8393988739586646),\n                        (8, 0.34497587576931377, 0.0, -0.28963794555439826),\n                        (4, 0.4999999999999998, -1.1102230246251565e-16, -0.11529447589771835)],\n                       [(12, 0.1852829437333603, 0.0, -0.5), (13, 0.531646717006979, 0.0, -0.5), (10, 9)],\n                       [(11, 12),\n                        (9, 0.5260557718525964, 0.0, -0.7587718918316604),\n                        (10, 0.3843844916022807, 0.0, -0.7520246417122236)],\n                       [(11, 0.45333192403000755, 0.0, 0.5), (12, 13), (9, 8), (4, 10)],\n                       [(5, 11),\n                        (13, 0.6205120316187869, 0.0, 0.19011722026833766),\n                        (9, 0.7720515191095958, 0.0, -0.2500493748817465),\n                        (4, 0.5, 0.0, -0.11529447589771835),\n                        (8, 0.4355845015328854, 0.0, -0.8111689735837979)],\n                       [(11, 12), (13, 7), (10, 9), (5, 0.5, -1.0, -0.5)],\n                       [(12, 0.34964410657987743, 0.0, 0.5383077432439662),\n                        (11, 0.5, 0.0, -0.5),\n                        (13, 0.344882952068414, 0.0, -0.984717420894669),\n                        (7, 0.4925777780083309, 0.0, 0.5072045568580279),\n                        (10, 0.5, 0.0, -0.5449316072361734),\n                        (9, 0.13993480154011695, 0.0, 0.0369265004511794)],\n                       [(11, 12), (4, 10), (9, 8)],\n                       [(12, 0.33274203901299115, 0.0, -0.3641661882066013),\n                        (9, 0.8935808083877341, 4.440892098500626e-16, -0.6698065954512344),\n                        (8, 0.3416013957299424, 0.0, -0.7695375446745221),\n                        (4, 0.4546112109132119, 0.0, 0.18717784331945309),\n                        (10, 0.2299569291469803, 0.0, -0.04493160723617351)],\n                       [(12, 13), (8, 2), (4, 0), (10, 9)],\n                       [(12, 0.9294880475869709, 0.0, -0.1553214972315855),\n                        (13, 0.9291764376708231, 4.440892098500626e-16, -0.8897608690712948),\n                        (4, 0.4999999999999998, -1.1102230246251565e-16, -0.025710813790364884),\n                        (8, 0.4999999999999998, 0.0, -0.18230509551025154),\n                        (10, 0.4999999999999998, 0.0, -0.5449316072361738),\n                        (0, 0.30408672398469627, 0.0, -0.5000000000000002),\n                        (2, 0.30408672398469627, 0.0, -0.5000000000000002)],\n                       [(11, 12), (13, 7), (9, 8), (10, 4)],\n                       [(12, 0.9764886499665415, 0.0, 0.44422293817850145),\n                        (13, 0.2914905715516678, 4.440892098500626e-16, -0.9963773855818856),\n                        (7, 0.4157141949118224, 0.0, 0.41360830056593145),\n                        (4, 0.8040867239846965, 4.440892098500626e-16, 0.9742891862096339),\n                        (8, 0.8040867239846965, -1.1102230246251565e-16, -0.18230509551025187)],\n                       [(12, 13), (8, 2), (4, 0)],\n                       [(12, 0.322056849702119, 0.0, 0.9220815067881403),\n                        (13, 0.418360472891683, 2.220446049250313e-16, -0.1371712088257072),\n                        (4, 0.75, 0.0, 0.9742891862096341),\n                        (8, 0.75, 0.0, -0.1823050955102523),\n                        (2, 0.5, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5)],\n                       [(13, 7), (11, 12), (10, 4), (9, 8)],\n                       [(12, 0.5910955657739061, 0.0, 0.9904786909552934),\n                        (13, 0.5658106523525603, -1.1102230246251565e-16, 0.1680872602763812),\n                        (7, 0.3025675832683916, 0.0, -0.5706246524937872),\n                        (4, 0.5, 4.440892098500626e-16, 0.9742891862096339),\n                        (8, 0.5, 0.0, 0.8176949044897479),\n                        (10, 0.5, 0.0, 0.18972303484944586),\n                        (9, 0.5, 0.0, 0.3325325194326316)],\n                       [(12, 13), (8, 2), (4, 0)],\n                       [(13, 0.6380247240052519, 0.0, -0.4840807765910127),\n                        (12, 0.5222942734869767, 0.0, 0.2803915814657436),\n                        (4, 0.25, 0.0, -0.025710813790364995),\n                        (8, 0.25, 0.0, -0.18230509551025254)],\n                       [(13, 7), (11, 12), (10, 4), (9, 8)],\n                       [(13, 0.017085511701252942, 0.0, -0.6127660888898279),\n                        (12, 0.3636045284494889, 4.440892098500626e-16, -0.827139930683602),\n                        (7, 0.32590008584322616, 0.0, -0.5618902727160411),\n                        (8, 0.5, 0.0, 0.8176949044897477),\n                        (4, 0.5, 0.0, -0.025710813790364884),\n                        (9, 0.5, 0.16746748056736704, 0.33253251943263207),\n                        (10, 0.5, 0.31027696515055414, -0.8102769651505544)],\n                       [(12, 13), (8, 2), (4, 0)],\n                       [(12, 0.9940835612187666, 0.0, 0.2214031994324317),\n                        (13, 0.7382217283247945, 0.0, 0.3797780553593999),\n                        (0, 1, 0, 0.5),\n                        (4, 0.5, 0.5257108137903654, -0.025710813790365328),\n                        (2, 1, 0, 0.5),\n                        (8, 0.5, -0.31769490448974746, 0.8176949044897481)],\n                       [(13, 7), (11, 12)],\n                       [(12, 0.5303663346437926, 0.0, 0.6476601232108743),\n                        (13, 0.3903186307590596, 0.0, 0.3705590103001124),\n                        (7, 0.5100876690078295, 0.0, 0.39443655737064276)],\n                       [(12, 13)],\n                       [(13, 0.6856477314180824, 0.0, -0.06389918404081052),\n                        (12, 0.6025816719136099, 0.0, -0.7190071148261055)],\n                       [(13, 7), (11, 12)],\n                       [(13, 0.3385860029956733, 0.0, -0.24842646774939348),\n                        (12, 0.892112104492687, 0.0, -0.29743790217530897),\n                        (7, 0.23477498989560153, 0.0, -0.3542327425401759)],\n                       [(12, 13)],\n                       [(12, 0.5928759364551912, 0.0, -0.7181239331695775),\n                        (13, 0.44925295079759087, 4.440892098500626e-16, 0.6495948004702132)],\n                       [(11, 12), (13, 7)],\n                       [(13, 0.643316544146411, 4.440892098500626e-16, -0.8393988739586646),\n                        (12, 0.8105365413512657, 4.440892098500626e-16, 0.7770622601851747),\n                        (7, 0.34497587576931377, 0.0, -0.28963794555439826),\n                        (11, 0.5, 0.0, 0.11529447589771702)],\n                       [(12, 13)],\n                       [(13, 0.5260557718525964, 0.0, -0.7587718918316604),\n                        (12, 0.3843844916022807, 0.0, 0.2479753582877764)],\n                       [(13, 7), (11, 12)],\n                       [(13, 0.7720515191095958, 0.0, -0.2500493748817465),\n                        (11, 0.5, 0.0, 0.11529447589771702),\n                        (7, 0.4355845015328854, 0.0, -0.8111689735837979)],\n                       [(12, 13)],\n                       [(13, 0.13993480154011695, 0.0, 0.0369265004511794),\n                        (12, 0.5, 0.0, -0.5449316072361736)],\n                       [(13, 7), (11, 12)],\n                       [(11, 0.4546112109132119, 2.220446049250313e-16, -0.18717784331945397),\n                        (7, 0.3416013957299424, 0.0, -0.7695375446745221),\n                        (12, 0.2299569291469803, 0.0, 0.9550683927638244),\n                        (13, 0.8935808083877341, 4.440892098500626e-16, -0.6698065954512344)],\n                       [(12, 13), (7, 1), (11, 14)],\n                       [(12, 0.4999999999999998, 0.0, 0.45506839276382616),\n                        (11, 0.5, -1.1102230246251565e-16, 0.025710813790362996),\n                        (7, 0.4999999999999998, 0.0, -0.18230509551025154),\n                        (1, 0.30408672398469627, 0.0, -0.5000000000000002),\n                        (14, 0.30408672398469627, 0.0, 0.4999999999999998)],\n                       [(13, 7), (12, 11)],\n                       [(7, 0.8040867239846965, -1.1102230246251565e-16, -0.18230509551025187),\n                        (11, 0.8040867239846965, 4.440892098500626e-16, -0.9742891862096366)],\n                       [(11, 14), (7, 1)],\n                       [(11, 0.75, 4.440892098500626e-16, -0.9742891862096366),\n                        (7, 0.75, 0.0, -0.1823050955102523),\n                        (1, 0.5, 0.0, 0.5),\n                        (14, 0.5, 0.0, 0.5)],\n                       [(12, 11), (13, 7)],\n                       [(11, 0.5, 0.0, -0.9742891862096366),\n                        (7, 0.5, 0.0, 0.8176949044897479),\n                        (13, 0.5, 0.0, 0.3325325194326316),\n                        (12, 0.5, 0.0, -0.8102769651505541)],\n                       [(7, 1), (11, 14)],\n                       [(11, 0.25, 0.0, 0.025710813790362774), (7, 0.25, 0.0, -0.18230509551025254)],\n                       [(12, 11), (13, 7)],\n                       [(7, 0.5, 0.0, 0.8176949044897477),\n                        (11, 0.5, -1.1102230246251565e-16, 0.025710813790362996),\n                        (13, 0.5, 0.16746748056736704, 0.33253251943263207),\n                        (12, 0.5, 0.3102769651505546, -0.8102769651505546)],\n                       [(7, 1), (11, 14)],\n                       [(7, 0.5, -0.31769490448974746, 0.8176949044897481),\n                        (14, 1, 0, 0.5),\n                        (1, 1, 0, 0.5),\n                        (11, 0.4999999999999998, 0.4742891862096372, -0.9742891862096372)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],\n           \'qubits\': [(3, 5),\n                      (3, 8),\n                      (4, 3),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (4, 8),\n                      (5, 3),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8),\n                      (6, 6)]}}\n\ndef _round4_unitary(n, k):\n    data = _ROUND4_VERIFIED.get((n, k))\n    if data is None:\n        return None\n    q = [cirq.GridQubit(*v) for v in data[\'qubits\']]\n    moments = []\n    for layer in data[\'moments\']:\n        ops = []\n        for gate in layer:\n            if len(gate) == 2:\n                ops.append(cirq.CZ(q[gate[0]], q[gate[1]]))\n            else:\n                i, x, z, a = gate\n                ops.append(cirq.PhasedXZGate(x_exponent=x, z_exponent=z,\n                                           axis_phase_exponent=a).on(q[i]))\n        moments.append(cirq.Moment(ops))\n    return cirq.Circuit(moments), {q[i] for i in data[\'outputs\']}\n# ROUND 4 VERIFIED NATIVE CIRCUITS END\n\n\n# ROUND 5 VERIFIED NATIVE CIRCUITS START\n# Frozen only after full-state, output-purity, native-gate and coupler checks.\n_ROUND5_VERIFIED = {(4, 1): {\'method\': \'boundary-compatible W tree\',\n          \'moments\': [[(1, 0.5, 0.0, 0.5),\n                       (2, 0.5, 0.0, -0.5),\n                       (0, 0.25000000000000006, 1.0, -0.5),\n                       (3, 0.5, 1.0, -0.5)],\n                      [(1, 2)],\n                      [(1, 0), (2, 3)],\n                      [(0, 0.25000000000000006, 1.0, 0.5),\n                       (1, 0.5, 0.0, 0.5),\n                       (2, 0.75, 0.0, -0.5),\n                       (3, 0.25000000000000006, 1.0, 0.5)],\n                      [(0, 1), (2, 3)],\n                      [(3, 0.5, 0.0, -0.5), (1, 0.5, 1.0, 0.5), (2, 0.5, 0.0, -0.5)]],\n          \'outputs\': [0, 1, 2, 3],\n          \'qubits\': [(6, 6), (6, 7), (7, 7), (8, 7)]},\n (6, 1): {\'method\': \'boundary-compatible W tree\',\n          \'moments\': [[(4, 0.6081734479693928, 0.0, 0.5),\n                       (3, 0.5, 0.0, -0.5),\n                       (0, 0.5, 0.0, -0.5),\n                       (5, 0.5, 0.0, 0.5),\n                       (1, 0.5, 0.0, 0.5),\n                       (2, 0.5, 0.0, -0.5)],\n                      [(4, 3)],\n                      [(3, 0), (4, 0.5, 0.0, 0.5)],\n                      [(4, 5),\n                       (3, 0.25000000000000006, 0.0, -0.5000000000000002),\n                       (0, 0.25000000000000006, 0.0, 0.5)],\n                      [(3, 0),\n                       (4, 0.25000000000000006, 0.0, -0.5000000000000002),\n                       (5, 0.25000000000000006, 0.0, -0.5000000000000002)],\n                      [(4, 5), (0, 1), (3, 2)],\n                      [(0, 0.25000000000000006, 0.0, 0.5),\n                       (1, 0.25000000000000006, 0.0, 0.5),\n                       (3, 0.25000000000000006, 0.0, -0.5000000000000002),\n                       (2, 0.25000000000000006, 0.0, 0.5),\n                       (4, 0.5, 1.0, 0.5),\n                       (5, 0.5, 0.0, -0.5)],\n                      [(0, 1), (3, 2)],\n                      [(0, 0.5, 0.0, 0.5), (1, 0.5, 0.0, -0.5), (2, 0.5, 0.0, 0.5), (3, 0.5, 0.0, 0.5)]],\n          \'outputs\': [0, 1, 2, 3, 4, 5],\n          \'qubits\': [(5, 6), (5, 7), (6, 5), (6, 6), (6, 7), (6, 8)]},\n (7, 1): {\'method\': \'boundary-compatible W tree\',\n          \'moments\': [[(4, 0.5456289483429899, 0.0, 0.5),\n                       (3, 0.5, 0.0, -0.5),\n                       (0, 0.5, 0.0, -0.5),\n                       (6, 0.1959132760153035, 1.0, 0.5),\n                       (1, 0.5, 0.0, 0.5),\n                       (2, 0.5, 0.0, -0.5),\n                       (5, 0.5, 1.0, 0.5)],\n                      [(4, 3)],\n                      [(3, 0), (4, 6)],\n                      [(3, 0.25000000000000006, 0.0, -0.5000000000000002),\n                       (0, 0.25000000000000006, 0.0, 0.5),\n                       (4, 0.5, 0.0, -0.5),\n                       (6, 0.1959132760153035, 1.0, -0.5000000000000002)],\n                      [(3, 0), (4, 6)],\n                      [(0, 1), (3, 2), (4, 5)],\n                      [(0, 0.25000000000000006, 0.0, 0.5),\n                       (1, 0.25000000000000006, 0.0, 0.5),\n                       (3, 0.25000000000000006, 0.0, -0.5000000000000002),\n                       (2, 0.25000000000000006, 0.0, 0.5),\n                       (4, 0.25000000000000006, 0.0, -0.5000000000000002),\n                       (5, 0.25000000000000006, 1.0, -0.5)],\n                      [(0, 1), (3, 2), (4, 5)],\n                      [(0, 0.5, 0.0, 0.5),\n                       (1, 0.5, 0.0, -0.5),\n                       (2, 0.5, 0.0, 0.5),\n                       (3, 0.5, 0.0, 0.5),\n                       (5, 0.5, 0.0, -0.5),\n                       (4, 0.5, 1.0, -0.5)]],\n          \'outputs\': [0, 1, 2, 3, 4, 5, 6],\n          \'qubits\': [(5, 6), (5, 7), (6, 5), (6, 6), (6, 7), (6, 8), (7, 7)]},\n (8, 1): {\'method\': \'boundary-compatible W tree\',\n          \'moments\': [[(3, 0.5, 0.0, 0.5),\n                       (5, 0.5, 0.0, -0.5),\n                       (0, 0.25000000000000006, 1.0, -0.5),\n                       (4, 0.5, 1.0, 0.5),\n                       (1, 0.75, 0.0, -0.5),\n                       (7, 0.5, 0.0, 0.5),\n                       (2, 0.5, 1.0, 0.5),\n                       (6, 0.5, 1.0, -0.5)],\n                      [(3, 5)],\n                      [(3, 0), (5, 4)],\n                      [(3, 0.5, 0.0, 0.5),\n                       (0, 0.25000000000000006, 1.0, 0.5),\n                       (5, 0.25000000000000006, 0.0, 0.5),\n                       (4, 0.25000000000000006, 1.0, -0.5)],\n                      [(3, 0), (5, 4)],\n                      [(0, 1), (4, 7), (3, 2), (5, 6)],\n                      [(0, 0.5, 0.0, -0.5),\n                       (1, 0.25000000000000006, 0.0, -0.5000000000000002),\n                       (4, 0.25000000000000006, 0.0, 0.5),\n                       (7, 0.25000000000000006, 0.0, -0.5000000000000002),\n                       (3, 0.75, 0.0, -0.5),\n                       (2, 0.25000000000000006, 1.0, -0.5),\n                       (5, 0.25000000000000006, 0.0, -0.5000000000000002),\n                       (6, 0.25000000000000006, 1.0, -0.5)],\n                      [(0, 1), (4, 7), (3, 2), (5, 6)],\n                      [(1, 1.0, 0.0, 0.5),\n                       (0, 0.5, 0.0, -0.5),\n                       (2, 0.5, 0.0, 0.5),\n                       (4, 0.5, 0.0, -0.5),\n                       (6, 0.5, 0.0, 0.5),\n                       (7, 0.5, 0.0, -0.5),\n                       (3, 0.5, 1.0, -0.5),\n                       (5, 0.5, 0.0, 0.5)]],\n          \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7],\n          \'qubits\': [(4, 7), (4, 8), (5, 6), (5, 7), (6, 6), (6, 7), (6, 8), (7, 6)]},\n (9, 1): {\'method\': \'boundary-compatible W tree\',\n          \'moments\': [[(4, 0.5, 0.0, -0.5),\n                       (0, 0.2820471084244876, 1.0, -0.5),\n                       (1, 0.46455905439753997, 0.0, 0.5),\n                       (3, 0.3040867239846965, 0.0, -0.5),\n                       (5, 0.5, 1.0, -0.5),\n                       (7, 0.5, 1.0, -0.5),\n                       (8, 0.5, 1.0, -0.5),\n                       (6, 0.25000000000000006, 0.0, 0.5),\n                       (2, 0.5, 0.0, -0.5)],\n                      [(1, 0), (4, 5)],\n                      [(1, 4), (0, 0.2820471084244876, 1.0, 0.5), (5, 0.25000000000000006, 1.0, -0.5)],\n                      [(1, 0.5, 0.0, 0.5), (0, 3), (4, 0.25000000000000006, 0.0, -0.5), (5, 8)],\n                      [(0, 1),\n                       (3, 0.3040867239846965, 0.0, 0.5),\n                       (4, 7),\n                       (8, 0.25000000000000006, 1.0, -0.5)],\n                      [(0, 0.5, 0.0, -0.5), (1, 2), (4, 5), (3, 6), (7, 0.25000000000000006, 1.0, 0.5)],\n                      [(3, 0),\n                       (5, 0.25000000000000006, 0.0, -0.5),\n                       (4, 0.75, 0.0, -0.5),\n                       (1, 0.75, 0.0, 0.5),\n                       (2, 0.25000000000000006, 0.0, 0.5),\n                       (6, 0.25000000000000006, 0.0, -0.5000000000000002)],\n                      [(3, 0.5, 0.0, 0.5), (4, 7), (5, 8), (1, 2), (0, 0.5, 0.0, 0.5)],\n                      [(3, 6),\n                       (7, 0.5, 0.0, -0.5),\n                       (8, 0.5, 0.0, 0.5),\n                       (1, 0.5, 1.0, -0.5),\n                       (4, 0.5, 0.0, -0.5),\n                       (2, 0.5, 0.0, -0.5),\n                       (5, 0.5, 0.0, 0.5)],\n                      [(3, 0.5, 0.0, -0.5)]],\n          \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8],\n          \'qubits\': [(6, 6), (6, 7), (6, 8), (7, 6), (7, 7), (7, 8), (8, 6), (8, 7), (8, 8)]},\n (10, 1): {\'method\': \'boundary-compatible W tree\',\n           \'moments\': [[(4, 0.5, 0.0, -0.5),\n                        (0, 0.2820471084244876, 1.0, -0.5),\n                        (5, 0.5, 1.0, -0.5),\n                        (1, 0.5, 0.0, 0.5),\n                        (3, 0.3040867239846965, 0.0, -0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (7, 0.25000000000000006, 0.0, 0.5),\n                        (8, 0.5, 1.0, -0.5),\n                        (2, 0.5, 0.0, -0.5),\n                        (9, 0.5, 0.0, -0.5)],\n                       [(4, 5), (1, 0)],\n                       [(1, 4), (0, 0.2820471084244876, 1.0, 0.5), (5, 0.2820471084244876, 1.0, -0.5)],\n                       [(4, 0.2820471084244876, 0.0, -0.5), (1, 0.5, 0.0, 0.5), (5, 6), (0, 3)],\n                       [(4, 5),\n                        (0, 1),\n                        (3, 0.3040867239846965, 0.0, 0.5),\n                        (6, 0.1959132760153035, 0.0, -0.5000000000000002)],\n                       [(5, 0.8040867239846965, 5.551115123125783e-17, 0.49999999999999994),\n                        (0, 0.5, 0.0, -0.5),\n                        (1, 2),\n                        (3, 7),\n                        (4, 8)],\n                       [(5, 6),\n                        (3, 0),\n                        (4, 0.75, 0.0, -0.5),\n                        (2, 0.25000000000000006, 0.0, 0.5),\n                        (7, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (8, 0.25000000000000006, 1.0, 0.5),\n                        (1, 0.75, 0.0, 0.5)],\n                       [(5, 9), (3, 0.5, 0.0, 0.5), (1, 2), (4, 8), (0, 0.5, 0.0, 0.5), (6, 0.5, 0.0, 0.5)],\n                       [(9, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (5, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (3, 7),\n                        (4, 0.5, 0.0, -0.5),\n                        (1, 0.5, 1.0, -0.5),\n                        (8, 0.5, 0.0, -0.5),\n                        (2, 0.5, 0.0, -0.5)],\n                       [(5, 9), (3, 0.5, 0.0, -0.5)],\n                       [(9, 0.5, 0.0, 0.5), (5, 0.5, 0.0, -0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],\n           \'qubits\': [(6, 6), (6, 7), (6, 8), (7, 6), (7, 7), (7, 8), (7, 9), (8, 6), (8, 7), (8, 8)]},\n (11, 1): {\'method\': \'boundary-compatible W tree\',\n           \'moments\': [[(0, 0.2820471084244876, 1.0, -0.5),\n                        (5, 0.5, 0.0, -0.5),\n                        (6, 0.5, 1.0, -0.5),\n                        (1, 0.5289772698358562, 0.0, 0.5),\n                        (4, 0.3040867239846965, 0.0, -0.5),\n                        (7, 0.5, 0.0, 0.5),\n                        (10, 0.5, 0.0, -0.5),\n                        (3, 0.5, 0.0, -0.5),\n                        (2, 0.5, 0.0, -0.5),\n                        (8, 0.25000000000000006, 0.0, 0.5),\n                        (9, 0.5, 1.0, -0.5)],\n                       [(1, 0), (5, 6)],\n                       [(1, 5), (0, 0.2820471084244876, 1.0, 0.5), (6, 0.3040867239846965, 1.0, -0.5)],\n                       [(5, 0.30408672398469627, 0.0, -0.5), (1, 0.5, 0.0, 0.5), (0, 4), (6, 7)],\n                       [(0, 1),\n                        (5, 6),\n                        (4, 0.3040867239846965, 0.0, 0.5),\n                        (7, 0.25000000000000006, 0.0, -0.5000000000000002)],\n                       [(6, 0.75, 0.0, 0.5), (0, 0.5, 0.0, -0.5), (4, 8), (1, 2), (7, 3), (5, 9)],\n                       [(4, 0),\n                        (6, 7),\n                        (8, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (3, 0.25000000000000006, 0.0, 0.5),\n                        (5, 0.75, 0.0, -0.5),\n                        (1, 0.75, 0.0, 0.5),\n                        (2, 0.25000000000000006, 0.0, 0.5),\n                        (9, 0.25000000000000006, 1.0, 0.5)],\n                       [(6, 10),\n                        (7, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (4, 0.5, 0.0, 0.5),\n                        (5, 9),\n                        (1, 2),\n                        (0, 0.5, 0.0, 0.5)],\n                       [(6, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (10, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (7, 3),\n                        (4, 8),\n                        (9, 0.5, 0.0, -0.5),\n                        (2, 0.5, 0.0, -0.5),\n                        (5, 0.5, 0.0, -0.5),\n                        (1, 0.5, 1.0, -0.5)],\n                       [(6, 10), (7, 0.5, 0.0, 0.5), (3, 0.5, 0.0, 0.5), (4, 0.5, 0.0, -0.5)],\n                       [(10, 0.5, 0.0, 0.5), (6, 0.5, 0.0, -0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],\n           \'qubits\': [(6, 6),\n                      (6, 7),\n                      (6, 8),\n                      (6, 9),\n                      (7, 6),\n                      (7, 7),\n                      (7, 8),\n                      (7, 9),\n                      (8, 6),\n                      (8, 7),\n                      (8, 8)]},\n (12, 1): {\'method\': \'boundary-compatible W tree\',\n           \'moments\': [[(1, 0.5533003790381135, 0.0, 0.5),\n                        (5, 0.5, 0.0, -0.5),\n                        (0, 0.2820471084244876, 1.0, -0.5),\n                        (6, 0.5, 1.0, -0.5),\n                        (4, 0.3040867239846965, 0.0, -0.5),\n                        (7, 0.5, 0.0, 0.5),\n                        (9, 0.5, 1.0, -0.5),\n                        (2, 0.5, 0.0, -0.5),\n                        (8, 0.25000000000000006, 0.0, 0.5),\n                        (10, 0.5, 0.0, -0.5),\n                        (3, 0.5, 0.0, -0.5),\n                        (11, 0.5, 0.0, 0.5)],\n                       [(1, 5)],\n                       [(1, 0), (5, 6)],\n                       [(0, 0.2820471084244876, 1.0, 0.5),\n                        (1, 0.5, 0.0, 0.5),\n                        (5, 0.27281447417149485, 0.0, -0.5000000000000002),\n                        (6, 0.27281447417149507, 1.0, -0.5)],\n                       [(0, 1), (5, 6)],\n                       [(0, 4), (6, 7), (5, 9), (1, 2)],\n                       [(4, 0.3040867239846965, 0.0, 0.5),\n                        (0, 0.5, 0.0, -0.5),\n                        (6, 0.75, 0.0, 0.5),\n                        (7, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (5, 0.6959132760153035, 0.0, -0.5),\n                        (9, 0.3040867239846965, 1.0, 0.5),\n                        (1, 0.75, 0.0, 0.5),\n                        (2, 0.25000000000000006, 0.0, 0.5)],\n                       [(4, 0), (6, 7), (5, 9), (1, 2)],\n                       [(4, 8),\n                        (6, 10),\n                        (7, 3),\n                        (9, 11),\n                        (0, 0.5, 0.0, 0.5),\n                        (5, 0.5, 0.0, -0.5),\n                        (1, 0.5, 1.0, -0.5),\n                        (2, 0.5, 0.0, -0.5)],\n                       [(4, 0.5, 0.0, 0.5),\n                        (8, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (6, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (10, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (7, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (3, 0.25000000000000006, 0.0, 0.5),\n                        (9, 0.25000000000000006, 0.0, 0.5),\n                        (11, 0.25000000000000006, 0.0, -0.5000000000000002)],\n                       [(4, 8), (6, 10), (7, 3), (9, 11)],\n                       [(3, 0.5, 0.0, 0.5),\n                        (4, 0.5, 0.0, -0.5),\n                        (6, 0.5, 0.0, -0.5),\n                        (7, 0.5, 0.0, 0.5),\n                        (9, 0.5, 0.0, -0.5),\n                        (10, 0.5, 0.0, 0.5),\n                        (11, 0.5, 0.0, -0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],\n           \'qubits\': [(6, 6),\n                      (6, 7),\n                      (6, 8),\n                      (6, 9),\n                      (7, 6),\n                      (7, 7),\n                      (7, 8),\n                      (7, 9),\n                      (8, 6),\n                      (8, 7),\n                      (8, 8),\n                      (9, 7)]},\n (13, 1): {\'method\': \'boundary-compatible W tree\',\n           \'moments\': [[(6, 0.5, 0.0, -0.5),\n                        (2, 0.5245095876987231, 0.0, 0.5),\n                        (1, 0.3040867239846965, 1.0, -0.5),\n                        (7, 0.5, 1.0, -0.5),\n                        (5, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (10, 0.5, 1.0, -0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (4, 0.5, 0.0, -0.5),\n                        (9, 0.25000000000000006, 0.0, 0.5),\n                        (3, 0.5, 0.0, -0.5),\n                        (11, 0.5, 0.0, -0.5),\n                        (12, 0.5, 0.0, 0.5),\n                        (0, 0.25000000000000006, 0.0, -0.5000000000000002)],\n                       [(2, 1), (6, 7)],\n                       [(2, 6), (1, 0.3040867239846965, 1.0, 0.5), (7, 0.27281447417149507, 1.0, -0.5)],\n                       [(2, 0.5, 0.0, 0.5),\n                        (1, 5),\n                        (6, 0.27281447417149485, 0.0, -0.5000000000000002),\n                        (7, 8)],\n                       [(1, 2),\n                        (5, 0.25000000000000006, 0.0, 0.5),\n                        (6, 10),\n                        (8, 0.25000000000000006, 0.0, -0.5000000000000002)],\n                       [(1, 0.5, 0.0, -0.5),\n                        (6, 7),\n                        (10, 0.3040867239846965, 1.0, 0.5),\n                        (2, 3),\n                        (8, 4),\n                        (5, 9)],\n                       [(5, 1),\n                        (6, 0.6959132760153035, 0.0, -0.5),\n                        (7, 0.75, 0.0, 0.5),\n                        (10, 12),\n                        (2, 0.75, 0.0, 0.5),\n                        (9, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (4, 0.25000000000000006, 0.0, 0.5),\n                        (3, 0.25000000000000006, 0.0, 0.5)],\n                       [(1, 0.5, 0.0, 0.5),\n                        (7, 11),\n                        (6, 10),\n                        (5, 0.5, 0.0, 0.5),\n                        (12, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (2, 3)],\n                       [(7, 8),\n                        (1, 0),\n                        (10, 0.25000000000000006, 0.0, 0.5),\n                        (11, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (5, 9),\n                        (3, 0.5, 0.0, -0.5),\n                        (6, 0.5, 0.0, -0.5),\n                        (2, 0.5, 1.0, -0.5)],\n                       [(1, 0.5, 0.0, -0.5),\n                        (8, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (0, 0.25000000000000006, 0.0, 0.5),\n                        (7, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (10, 12),\n                        (5, 0.5, 0.0, -0.5)],\n                       [(0, 1), (8, 4), (7, 11), (12, 0.5, 0.0, -0.5), (10, 0.5, 0.0, -0.5)],\n                       [(1, 0.5, 0.0, 0.5),\n                        (11, 0.5, 0.0, 0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (7, 0.5, 0.0, -0.5),\n                        (4, 0.5, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],\n           \'qubits\': [(5, 6),\n                      (6, 6),\n                      (6, 7),\n                      (6, 8),\n                      (6, 9),\n                      (7, 6),\n                      (7, 7),\n                      (7, 8),\n                      (7, 9),\n                      (8, 6),\n                      (8, 7),\n                      (8, 8),\n                      (9, 7)]},\n (14, 1): {\'method\': \'boundary-compatible W tree\',\n           \'moments\': [[(8, 0.5, 0.0, 0.5),\n                        (5, 0.5, 0.0, -0.5),\n                        (6, 0.5, 1.0, -0.5),\n                        (9, 0.27281447417149507, 1.0, -0.5),\n                        (10, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (1, 0.5, 1.0, 0.5),\n                        (2, 0.3040867239846965, 0.0, 0.5),\n                        (12, 0.3040867239846965, 1.0, -0.5),\n                        (7, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (0, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (3, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (11, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (4, 0.75, 1.0, -0.5),\n                        (13, 0.5, 1.0, -0.5)],\n                       [(8, 5)],\n                       [(5, 6), (8, 9)],\n                       [(5, 0.22718552582850493, 0.0, -0.5),\n                        (6, 0.22718552582850488, 1.0, -0.5),\n                        (9, 0.2728144741714952, 1.0, 0.5),\n                        (8, 0.5, 0.0, 0.5)],\n                       [(5, 6), (9, 8)],\n                       [(9, 10), (5, 1), (6, 0.5, 0.0, 0.5), (8, 0.5, 0.0, -0.5)],\n                       [(6, 2),\n                        (8, 12),\n                        (10, 0.25000000000000006, 0.0, 0.5),\n                        (9, 0.5, 0.0, -0.5),\n                        (5, 0.25000000000000006, 0.0, 0.5),\n                        (1, 0.25000000000000006, 1.0, -0.5)],\n                       [(10, 9),\n                        (5, 1),\n                        (6, 0.5, 0.0, 0.5),\n                        (2, 0.3040867239846965, 0.0, -0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (12, 0.3040867239846965, 1.0, 0.5)],\n                       [(6, 2), (10, 7), (8, 12), (1, 0.5, 0.0, -0.5), (5, 0.5, 0.0, -0.5), (9, 13)],\n                       [(1, 0),\n                        (2, 3),\n                        (12, 11),\n                        (5, 4),\n                        (7, 0.25000000000000006, 0.0, 0.5),\n                        (10, 0.5, 0.0, -0.5),\n                        (9, 0.75, 0.0, -0.5),\n                        (13, 0.25000000000000006, 1.0, 0.5),\n                        (6, 0.5, 0.0, -0.5),\n                        (8, 0.5, 1.0, 0.5)],\n                       [(7, 10),\n                        (0, 0.25000000000000006, 0.0, 0.5),\n                        (1, 0.5, 0.0, -0.5),\n                        (3, 0.25000000000000006, 0.0, 0.5),\n                        (2, 0.5, 0.0, -0.5),\n                        (11, 0.25000000000000006, 0.0, 0.5),\n                        (12, 0.5, 0.0, -0.5),\n                        (5, 0.5, 0.0, -0.5),\n                        (4, 0.25000000000000006, 1.0, -0.5),\n                        (9, 13)],\n                       [(0, 1),\n                        (3, 2),\n                        (11, 12),\n                        (5, 4),\n                        (10, 0.5, 0.0, 0.5),\n                        (13, 0.5, 0.0, -0.5),\n                        (9, 0.5, 0.0, -0.5)],\n                       [(1, 0.5, 0.0, 0.5),\n                        (2, 0.5, 0.0, 0.5),\n                        (4, 1.0, 0.0, 0.5),\n                        (12, 0.5, 0.0, 0.5),\n                        (5, 0.5, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],\n           \'qubits\': [(3, 7),\n                      (4, 7),\n                      (4, 8),\n                      (4, 9),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8),\n                      (5, 9),\n                      (6, 7),\n                      (6, 8),\n                      (6, 9),\n                      (7, 6),\n                      (7, 7),\n                      (7, 8)]},\n (15, 1): {\'method\': \'boundary-compatible W tree\',\n           \'moments\': [[(4, 0.5212364096070798, 0.0, 0.5),\n                        (5, 0.5, 0.0, -0.5),\n                        (10, 0.5, 0.0, 0.5),\n                        (9, 0.2728144741714952, 1.0, 0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (8, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (13, 0.5, 0.0, 0.5),\n                        (1, 0.3040867239846965, 1.0, -0.5),\n                        (0, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (2, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (3, 0.75, 0.0, -0.5),\n                        (7, 0.5, 0.0, 0.5),\n                        (11, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (14, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (12, 0.5, 1.0, -0.5)],\n                       [(4, 5)],\n                       [(5, 10), (4, 9)],\n                       [(5, 0.75, 0.0, 0.5),\n                        (10, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (4, 0.5, 0.0, -0.5),\n                        (9, 0.27281447417149507, 1.0, -0.5)],\n                       [(5, 10), (4, 9)],\n                       [(5, 6), (9, 8), (10, 13), (4, 0.5, 0.0, 0.5)],\n                       [(4, 1),\n                        (5, 0.25000000000000006, 0.0, 0.5),\n                        (6, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (9, 0.5, 0.0, -0.5),\n                        (8, 0.25000000000000006, 0.0, 0.5),\n                        (10, 0.75, 0.0, 0.5),\n                        (13, 0.25000000000000006, 0.0, -0.5000000000000002)],\n                       [(5, 6), (9, 8), (10, 13), (1, 0.3040867239846965, 1.0, 0.5), (4, 0.5, 0.0, 0.5)],\n                       [(1, 4),\n                        (5, 0.5, 0.0, -0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (10, 0.5, 0.0, -0.5),\n                        (13, 0.5, 0.0, 0.5),\n                        (9, 12)],\n                       [(1, 0),\n                        (5, 2),\n                        (6, 3),\n                        (8, 7),\n                        (10, 11),\n                        (13, 14),\n                        (9, 0.75, 0.0, -0.5),\n                        (12, 0.25000000000000006, 1.0, 0.5),\n                        (4, 0.5, 1.0, 0.5)],\n                       [(0, 0.25000000000000006, 0.0, 0.5),\n                        (1, 0.5, 0.0, -0.5),\n                        (2, 0.25000000000000006, 0.0, 0.5),\n                        (5, 0.5, 0.0, -0.5),\n                        (6, 0.5, 0.0, -0.5),\n                        (3, 0.25000000000000006, 0.0, 0.5),\n                        (8, 0.75, 0.0, 0.5),\n                        (7, 0.25000000000000006, 0.0, 0.5),\n                        (11, 0.25000000000000006, 0.0, 0.5),\n                        (10, 0.5, 0.0, -0.5),\n                        (14, 0.25000000000000006, 0.0, 0.5),\n                        (13, 0.5, 0.0, -0.5),\n                        (9, 12)],\n                       [(0, 1),\n                        (2, 5),\n                        (6, 3),\n                        (8, 7),\n                        (11, 10),\n                        (14, 13),\n                        (12, 0.5, 0.0, -0.5),\n                        (9, 0.5, 0.0, -0.5)],\n                       [(3, 1.0, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5),\n                        (5, 0.5, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (7, 0.5, 0.0, 0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (10, 0.5, 0.0, 0.5),\n                        (13, 0.5, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],\n           \'qubits\': [(4, 7),\n                      (5, 7),\n                      (5, 8),\n                      (5, 9),\n                      (6, 7),\n                      (6, 8),\n                      (6, 9),\n                      (7, 5),\n                      (7, 6),\n                      (7, 7),\n                      (7, 8),\n                      (7, 9),\n                      (8, 7),\n                      (8, 8),\n                      (8, 9)]}}\n\ndef _round5_unitary(n, k):\n    data = _ROUND5_VERIFIED.get((n, k))\n    if data is None:\n        return None\n    q = [cirq.GridQubit(*v) for v in data[\'qubits\']]\n    moments = []\n    for layer in data[\'moments\']:\n        ops = []\n        for gate in layer:\n            if len(gate) == 2:\n                ops.append(cirq.CZ(q[gate[0]], q[gate[1]]))\n            else:\n                i, x, z, a = gate\n                ops.append(cirq.PhasedXZGate(x_exponent=x, z_exponent=z,\n                                           axis_phase_exponent=a).on(q[i]))\n        moments.append(cirq.Moment(ops))\n    return cirq.Circuit(moments), {q[i] for i in data[\'outputs\']}\n# ROUND 5 VERIFIED NATIVE CIRCUITS END\n\n\n# ROUND 6 VERIFIED NATIVE CIRCUITS START\n# Each entry passed full-state, purity and native Willow checks before freezing.\n_ROUND6_VERIFIED = {(11, 4): {\'method\': \'reachable-support count kernels + native scheduling\',\n           \'moments\': [[(1, 0.6804183583624992, 0.0, 0.5),\n                        (2, 0.2715149105420544, 0.0, -0.5000000000000002),\n                        (3, 0.3910132305016676, 0.0, 0.5),\n                        (7, 0.5, 0.0, 0.5),\n                        (8, 0.4999999999999998, 0.0, -0.5000000000000002),\n                        (6, 0.5, 1.0, -0.5),\n                        (9, 0.4999999999999998, 0.0, 0.5937872305331664),\n                        (5, 0.5, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (10, 0.5, 0.0, 0.5)],\n                       [(1, 2), (4, 3), (7, 8)],\n                       [(2, 0.8245125290280035, 0.0, 0.5),\n                        (7, 0.3399162669016432, 0.0, -0.5000000000000002),\n                        (3, 0.7367750224499274, 0.0, 0.5),\n                        (8, 0.16008373309835672, 0.0, 0.5),\n                        (1, 0.7552309029870035, 1.0, 0.5),\n                        (4, 0.5, 1.0, 0.5)],\n                       [(2, 7), (3, 8)],\n                       [(2, 0.5, 0.0, -0.5),\n                        (3, 0.5, 0.0, 0.5),\n                        (7, 0.45750956125153186, 0.0, 0.5),\n                        (8, 0.9575095612515319, 0.0, -0.5000000000000002)],\n                       [(2, 3), (7, 8)],\n                       [(1, 2), (4, 3), (7, 0.5, 1.0, 0.5), (8, 0.5, 1.0, -0.5)],\n                       [(2, 0.7356809418135472, 0.0, 0.5),\n                        (3, 0.6015434872071732, 0.0, 0.5),\n                        (1, 0.5, 1.0, -0.5),\n                        (4, 0.5, 1.0, 0.5)],\n                       [(2, 7), (3, 8)],\n                       [(2, 0.7820471084244873, 0.0, 0.5),\n                        (3, 0.3984565127928268, 0.0, -0.5),\n                        (8, 0.17490085614052142, 1.0, 0.5)],\n                       [(2, 3), (7, 8)],\n                       [(2, 0.5, 0.0, 0.5), (3, 0.5, 0.0, 0.5), (8, 0.5, 1.0, 0.5)],\n                       [(1, 2), (4, 3)],\n                       [(2, 7), (3, 0.4999999999999998, 0.0, 0.5), (1, 0.5, 1.0, -0.5), (4, 0.5, 1.0, -0.5)],\n                       [(3, 8), (2, 0.5420140480044957, 0.0, -0.5), (7, 0.318767202525581, 0.0, -0.5)],\n                       [(1, 2),\n                        (7, 6),\n                        (8, 0.4831265102867419, 0.0, 0.010344370100573597),\n                        (3, 0.5, 0.0, -0.49010529949480985)],\n                       [(8, 9), (4, 3), (2, 0.21639595347017168, 0.0, -0.5), (6, 0.0, 1.0, 0.0)],\n                       [(2, 7),\n                        (6, 1),\n                        (3, 0.0, 0.8450770868598165, 0.0),\n                        (8, 0.9416885504219519, 0.0, 0.7843691475830337),\n                        (9, 0.6085354469534892, 8.881784197001252e-16, -0.9311758863092963)],\n                       [(3, 8),\n                        (1, 0.44250075386119647, 0.0, 0.5),\n                        (2, 0.8655339613566024, 0.0, -0.5000000000000002),\n                        (7, 0.5, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5)],\n                       [(1, 2),\n                        (7, 6),\n                        (3, 0.09666409909802184, 0.0, 0.806714584995337),\n                        (8, 0.5, 0.0, -0.2883600848339505)],\n                       [(4, 3),\n                        (8, 9),\n                        (2, 0.7362067632071183, 0.0, -0.5),\n                        (7, 0.5, 1.0, -0.5),\n                        (6, 0.020330766314526457, 1.0, 0.5),\n                        (1, 0.7628573843657716, 0.0, -0.5)],\n                       [(2, 7),\n                        (6, 1),\n                        (3, 0.5812745787265813, 0.0, 0.6008125536694957),\n                        (8, 0.5150139304348057, 0.0, -0.2883600848339505),\n                        (9, 0.16723314080476154, 0.0, -0.7374666288921621)],\n                       [(3, 8),\n                        (1, 0.20741313285212096, 0.0, 0.5),\n                        (2, 0.8453403754074111, 0.0, -0.5),\n                        (6, 0.08250711583968151, 0.0, 0.5)],\n                       [(1, 2),\n                        (7, 6),\n                        (3, 0.2955394861408327, 0.0, -0.426593528114672),\n                        (8, 0.08858950606155513, 5.551115123125783e-17, 0.21163991516604946)],\n                       [(4, 3),\n                        (8, 9),\n                        (2, 0.8626157748423866, 0.0, 0.5),\n                        (6, 0.8491679204856472, 1.0, 0.5),\n                        (1, 0.5846516053950286, 0.0, -0.5)],\n                       [(2, 7),\n                        (6, 1),\n                        (3, 0.2943011721717049, 0.0, -0.40438048920731884),\n                        (8, 0.3017505280831282, 0.0, -0.2883600848339507),\n                        (4, 0.49999999999999967, 0.0, 0.7450526497474059),\n                        (9, 0.6303550437629813, 2.220446049250313e-16, -0.18158983536649442)],\n                       [(3, 8),\n                        (1, 0.7281308634155641, 0.0, 0.5),\n                        (2, 0.15189628146893905, 0.0, 0.5),\n                        (6, 0.7775493667322889, 1.0, 0.4999999999999999)],\n                       [(1, 2),\n                        (7, 6),\n                        (3, 0.302862294871528, 0.0, 0.586014145652751),\n                        (8, 0.199548964435123, 0.0, 0.21163991516604902)],\n                       [(4, 3),\n                        (8, 9),\n                        (2, 0.1796669734795279, 0.0, 0.5),\n                        (6, 0.7487068121713458, 1.0, -0.5),\n                        (1, 0.04683006583111981, 4.440892098500626e-16, 0.49999999999999956)],\n                       [(2, 7),\n                        (6, 1),\n                        (3, 0.5000000000000001, 0.0, -0.6455351803962504),\n                        (8, 0.22177219480665555, 0.0, 0.7116399151660491),\n                        (4, 0.33347826094116906, 0.0, 0.24505264974740465),\n                        (9, 0.8315920243625678, 1.1102230246251565e-16, -0.4157702878189209)],\n                       [(3, 8),\n                        (1, 0.2359340761074103, 0.0, -0.5),\n                        (2, 0.773538838069632, 0.0, 0.5),\n                        (6, 0.5556977655654309, 1.0, 0.5)],\n                       [(1, 2),\n                        (7, 6),\n                        (3, 0.5, 0.0, -0.6455351803962508),\n                        (8, 0.5, 0.0, -0.8013226122314938)],\n                       [(4, 3),\n                        (8, 9),\n                        (2, 0.6370237375638905, 2.220446049250313e-16, -0.5000000000000002),\n                        (7, 0.0, 1.0, 0.0),\n                        (6, 0.01539729570142323, 1.0, -0.5000000000000002),\n                        (1, 0.4429438206750263, 0.0, 0.5)],\n                       [(2, 7),\n                        (6, 1),\n                        (3, 0.5519231616233388, 4.440892098500626e-16, 0.5722611361353818),\n                        (8, 0.5, 0.0, -0.8013226122314938),\n                        (4, 0.5276550549544043, 0.0, 0.2450526497474046),\n                        (9, 0.5, 0.0, 0.46178536847925344)],\n                       [(3, 8),\n                        (1, 0.6131686327461048, 0.0, -0.5),\n                        (2, 0.7717884632990302, 0.0, 0.5),\n                        (6, 0.7676256178984648, 0.0, 0.5)],\n                       [(1, 2),\n                        (7, 6),\n                        (3, 0.49999999999999994, 0.0, -0.06188561777021251),\n                        (8, 0.5, 0.0, -0.1008012743636435)],\n                       [(4, 3),\n                        (8, 9),\n                        (2, 0.8699038911668677, 0.0, -0.5000000000000002),\n                        (7, 0.5, 1.0, -0.5),\n                        (6, 0.2231327129068029, 1.0, -0.5),\n                        (1, 0.14080285459112307, 0.0, 0.5)],\n                       [(2, 7),\n                        (6, 1),\n                        (4, 0.5000000000000002, 0.0, -0.25494735025259585),\n                        (3, 0.5, 0.0, 0.9381143822297875),\n                        (9, 0.5, 0.0, -0.038214631520746334)],\n                       [(4, 3),\n                        (8, 9),\n                        (1, 0.004810136259751463, 0.0, -0.4999999999999991),\n                        (2, 0.5, 0.0, -0.5),\n                        (6, 0.40834871822043467, 0.0, 0.5),\n                        (7, 0.5, 0.0, -0.5)],\n                       [(1, 2),\n                        (6, 5),\n                        (4, 0.4901053274689138, 0.0, -0.2549238179692832),\n                        (9, 0.5068340379253959, 0.0, -0.5310585926382718),\n                        (3, 0.5, 4.440892098500626e-16, 0.9381143822297868),\n                        (8, 0.5, 0.0, -0.6008012743636435)],\n                       [(4, 0),\n                        (9, 10),\n                        (1, 0.5, 0.0, -0.5),\n                        (6, 0.4999999999999998, 0.0, -0.5000000000000002),\n                        (5, 0.30408672398469616, 0.0, -0.5000000000000002)],\n                       [(2, 1),\n                        (7, 6),\n                        (4, 0.5, 0.0, 0.24431903208484224),\n                        (9, 0.5, 0.0, -0.7736945421001282),\n                        (0, 0.30408672398469616, 0.0, -0.5000000000000002),\n                        (10, 0.30408672398469616, 0.0, -0.5000000000000002)],\n                       [(3, 4),\n                        (8, 9),\n                        (2, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (1, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (6, 0.8040867239846965, 1.1102230246251565e-16, 0.4999999999999999)],\n                       [(2, 1),\n                        (6, 5),\n                        (4, 0.8040867239846965, 8.881784197001252e-16, -0.7556809679151584),\n                        (9, 0.8040867239846965, 0.0, 0.22630545789987178)],\n                       [(4, 0),\n                        (9, 10),\n                        (6, 0.75, 1.0, 0.5),\n                        (5, 0.5, 0.0, 0.5),\n                        (1, 0.5, 0.0, -0.5),\n                        (2, 0.5, 1.0, 0.5)],\n                       [(7, 6),\n                        (4, 0.7499999999999999, 1.0, -0.7556809679151579),\n                        (9, 0.75, 1.0, 0.22630545789987166),\n                        (0, 0.5, 0.0, 0.5),\n                        (10, 0.5, 0.0, 0.5)],\n                       [(3, 4), (8, 9), (6, 0.5, 0.0, -0.5), (7, 0.5, 0.0, 0.5)],\n                       [(6, 5),\n                        (4, 0.5000000000000002, 0.0, 0.24431903208484212),\n                        (9, 0.5, 0.0, -0.7736945421001282),\n                        (3, 0.5, 0.0, -0.34448802297721204),\n                        (8, 0.5, 0.0, -0.3687111520137809)],\n                       [(4, 0), (9, 10), (6, 0.25000000000000006, 1.0, 0.5)],\n                       [(7, 6),\n                        (4, 0.25000000000000006, -0.9999999999999996, -0.7556809679151579),\n                        (9, 0.25000000000000006, 1.0, 0.22630545789987178)],\n                       [(3, 4), (8, 9), (6, 0.5, 1.0, -0.5), (7, 0.5, 0.0, -0.5)],\n                       [(6, 5),\n                        (4, 0.5, 1.0, 0.24431903208484224),\n                        (9, 0.5, 1.0, -0.7736945421001282),\n                        (3, 0.5, 0.8444880229772123, 0.6555119770227877),\n                        (8, 0.5, 0.8687111520137814, 0.6312888479862191)],\n                       [(4, 0), (9, 10), (5, 1.0, 0.0, 0.5), (6, 0.5, 0.0, 0.5)],\n                       [(0, 1.0, 0.0, 0.5),\n                        (4, 0.5, -0.7443190320848423, -0.7556809679151579),\n                        (9, 0.5, 0.2736945421001282, 0.22630545789987178),\n                        (10, 1.0, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],\n           \'qubits\': [(3, 7),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 3),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8)]},\n (12, 4): {\'method\': \'reachable-support count kernels + native scheduling\',\n           \'moments\': [[(2, 0.7547213446998079, 0.0, 0.5),\n                        (3, 0.4812539258587223, 0.0, -0.5000000000000002),\n                        (4, 0.20882481497877967, 0.0, 0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (9, 0.4999999999999998, 0.0, -0.5000000000000002),\n                        (7, 0.4127792040228902, -0.9999999999999998, -0.9262332254895425),\n                        (10, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (0, 0.5, 0.0, 0.5),\n                        (6, 0.5, 1.0, 0.5),\n                        (1, 0.5, 0.0, 0.5),\n                        (11, 0.5, 1.0, 0.5)],\n                       [(2, 3), (5, 4), (8, 9)],\n                       [(3, 0.5260979312786449, 0.0, 0.5),\n                        (8, 0.6896632272499081, 0.0, -0.5),\n                        (4, 0.9801549305958261, 0.0, 0.5),\n                        (9, 0.18966322724990792, 0.0, -0.5000000000000002),\n                        (2, 0.5181677870940664, 1.0, 0.5),\n                        (5, 0.5, 1.0, 0.5)],\n                       [(3, 8), (4, 9)],\n                       [(3, 0.5, 0.0, -0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (8, 0.4375661304670748, 0.0, 0.5),\n                        (9, 0.9375661304670747, 0.0, -0.5000000000000002)],\n                       [(3, 4), (8, 9)],\n                       [(2, 3), (5, 4), (8, 0.5, 1.0, 0.5), (9, 0.5, 1.0, -0.5)],\n                       [(3, 0.7208339391878029, 0.0, 0.5),\n                        (4, 0.5980681240477937, 0.0, 0.5),\n                        (2, 0.5, 1.0, -0.5),\n                        (5, 0.5, 1.0, 0.5)],\n                       [(3, 8), (4, 9)],\n                       [(3, 0.7992192552039086, 1.6653345369377348e-16, 0.49999999999999983),\n                        (4, 0.40193187595220636, 0.0, -0.5),\n                        (9, 0.14351493264378545, 1.0, 0.5)],\n                       [(3, 4), (8, 9)],\n                       [(3, 0.5, 0.0, 0.5), (4, 0.5, 0.0, 0.5), (9, 0.5, 1.0, 0.5)],\n                       [(2, 3), (5, 4)],\n                       [(3, 8),\n                        (4, 0.5, 0.0, 0.5),\n                        (2, 0.3005269769394423, 1.0, -4.440892098500626e-16),\n                        (5, 0.3005269769394423, 1.0, 0.0)],\n                       [(4, 9),\n                        (3, 0.45891109526391904, 1.1102230246251565e-16, 0.0),\n                        (8, 0.7780272931625696, 0.0, -0.5821778094721619)],\n                       [(2, 3),\n                        (8, 7),\n                        (9, 0.5721853640985723, 0.0, -0.5538166093874051),\n                        (4, 0.5, 0.0, 0.958911095263919)],\n                       [(9, 10),\n                        (5, 4),\n                        (7, 0.85928593962011, 0.0, -0.9262332254895425),\n                        (2, 0.5, 0.0, 0.49999999999999956),\n                        (3, 0.0, 0.45487938783478143, 0.0),\n                        (8, 0.5, 0.0, 0.9178221905278381)],\n                       [(7, 2),\n                        (3, 8),\n                        (4, 0.0, -0.4629428026930569, 0.0),\n                        (9, 0.5, 0.0, 0.870400557392476),\n                        (10, 0.85928593962011, 0.0, -0.9262332254895425),\n                        (5, 0.5, 0.0, 0.49999999999999956)],\n                       [(4, 9),\n                        (10, 5),\n                        (3, 0.5000000000000001, 0.9999999999999996, 0.7532307017118327),\n                        (8, 0.6714165234362537, 0.0, 0.8464060987968449),\n                        (7, 0.7485956033393342, 1.0, -0.42623322548954246)],\n                       [(2, 3),\n                        (8, 7),\n                        (9, 0.6714165234362537, 4.440892098500626e-16, 0.9418166491234687),\n                        (10, 0.7485956033393342, -4.440892098500626e-16, -0.42623322548954246),\n                        (4, 0.5, 1.0, 0.2943196064479139)],\n                       [(9, 10),\n                        (5, 4),\n                        (3, 0.5, 1.0, -0.24676929828816707),\n                        (8, 0.3935294720438233, 8.881784197001252e-16, 0.5673589831832238),\n                        (7, 0.9961572718865779, 1.0, -0.9262332254895478),\n                        (2, 0.0528060550044127, 0.0, 0.6736003437869695)],\n                       [(3, 8),\n                        (7, 2),\n                        (4, 0.5, 0.0, -0.7056803935520861),\n                        (9, 0.39352947204382344, 0.0, -0.7791362352629096),\n                        (10, 0.9961572718865779, 1.0, -0.9262332254895478),\n                        (5, 0.0528060550044127, 0.0, 0.6736003437869695)],\n                       [(4, 9),\n                        (10, 5),\n                        (2, 0.24963665794604498, 0.0, -0.8090370713032549),\n                        (8, 0.3104369525321424, 0.0, 0.9858802335834231),\n                        (7, 0.5655532211464794, 1.0, -0.4262332254895531)],\n                       [(2, 3),\n                        (8, 7),\n                        (5, 0.24963665794604517, 0.0, -0.8090370713032553),\n                        (9, 0.31043695253214243, 0.0, 0.8023425143368912),\n                        (10, 0.5655532211464794, 0.0, -0.4262332254895531)],\n                       [(5, 4),\n                        (9, 10),\n                        (8, 0.6193705712509701, 0.0, 0.11291454448282767),\n                        (7, 0.4999999999999998, 1.0, -0.4262332254895531),\n                        (2, 0.32701156232958106, 0.0, 0.7987432874086429)],\n                       [(3, 8),\n                        (7, 2),\n                        (9, 0.6193705712509703, 0.9999999999999999, -0.32469179656251335),\n                        (10, 0.4999999999999998, 1.0, -0.4262332254895531),\n                        (5, 0.32701156232958106, 0.0, 0.7987432874086426)],\n                       [(4, 9),\n                        (10, 5),\n                        (3, 0.0, 0.9739798652145688, 0.0),\n                        (8, 0.6256105257587086, 4.440892098500626e-16, 0.6296441764731986),\n                        (7, 0.5960614607313279, 1.0, -0.11684358694965402),\n                        (2, 0.5, 0.0, 0.04610781888751925)],\n                       [(3, 8),\n                        (7, 2),\n                        (4, 0.0, 0.9739798652145693, 0.0),\n                        (9, 0.6256105257587086, 1.1102230246251565e-16, 0.1585785714471154),\n                        (10, 0.5960614607313279, 1.0, -0.11684358694965402),\n                        (5, 0.5, 0.0, 0.04610781888751925)],\n                       [(4, 9),\n                        (10, 5),\n                        (2, 0.5, 0.0, 0.5461078188875192),\n                        (3, 0.5000000000000001, 1.0, -0.5938608623810806),\n                        (8, 0.9764747842833552, 8.881784197001252e-16, -0.9681400656061936),\n                        (7, 0.14949404734505523, 1.0, 0.2730671118828738)],\n                       [(2, 3),\n                        (8, 7),\n                        (9, 0.9764747842833552, 2.220446049250313e-16, -0.2436371864734923),\n                        (10, 0.14949404734505523, 0.0, 0.2730671118828738),\n                        (5, 0.5, 4.440892098500626e-16, 0.5461078188875186),\n                        (4, 0.5000000000000001, 1.0, 0.9472280423550004)],\n                       [(9, 10),\n                        (5, 4),\n                        (3, 0.5000000000000002, 1.0, 0.40613913761891934),\n                        (8, 0.5, 0.0, 0.7888816512119323),\n                        (2, 0.0, 0.08503202047153685, 0.0)],\n                       [(3, 8),\n                        (7, 2),\n                        (4, 0.5, 0.0, -0.052771957644999645),\n                        (9, 0.5, 0.0, -0.000658903291617996),\n                        (5, 0.0, 0.08503202047153624, 0.0)],\n                       [(4, 9),\n                        (10, 5),\n                        (3, 0.5, 0.0, 0.47332799163755634),\n                        (8, 0.14731871270290628, 0.0, -0.21111834878806768),\n                        (7, 0.4247761378557593, 0.0, 0.9776181484214663)],\n                       [(2, 3),\n                        (8, 7),\n                        (4, 0.5000000000000002, 0.0, 0.014416896373637744),\n                        (9, 0.14731871270290633, 0.9999999999999999, 0.999341096708382),\n                        (10, 0.4247761378557593, 0.0, 0.9776181484214663)],\n                       [(5, 4),\n                        (9, 10),\n                        (3, 0.0, 1.0, 0.0),\n                        (8, 0.9433458052208397, 4.440892098500626e-16, -0.711118348788067),\n                        (7, 0.4500869528994624, 0.0, -0.33653668617336185),\n                        (2, 0.42020075164346243, -0.9999999999999998, 0.3377021977675534)],\n                       [(3, 8),\n                        (7, 2),\n                        (9, 0.9433458052208397, 0.0, 0.4993410967083811),\n                        (10, 0.45008695289946266, 0.0, 0.6634633138266384),\n                        (5, 0.42020075164346243, -0.9999999999999998, 0.3377021977675534)],\n                       [(4, 9),\n                        (10, 5),\n                        (2, 0.5410889047360808, 0.0, -0.7574936873931724),\n                        (7, 0.5410889047360808, 1.1102230246251565e-16, 0.21035484945072336),\n                        (3, 0.5, 0.0, 0.4733279916375562)],\n                       [(2, 0),\n                        (7, 6),\n                        (5, 0.5410889047360808, 0.0, -0.7574936873931726),\n                        (10, 0.541088904736081, 0.0, -0.7896451505492768),\n                        (4, 0.5, 0.0, 0.014416896373637023),\n                        (9, 0.0, 1.0, 0.0)],\n                       [(5, 1),\n                        (10, 11),\n                        (2, 0.4999999999999998, -4.440892098500626e-16, -0.2574936873931717),\n                        (7, 0.5, 0.0, -0.28964515054927653),\n                        (0, 0.30408672398469616, 0.0, -0.5000000000000002),\n                        (6, 0.3040867239846962, 1.0, -0.5000000000000002)],\n                       [(3, 2),\n                        (8, 7),\n                        (5, 0.5000000000000001, 2.220446049250313e-16, -0.25749368739317235),\n                        (10, 0.5, 0.0, 0.710354849450723),\n                        (1, 0.30408672398469616, 0.0, -0.5000000000000002),\n                        (11, 0.3040867239846962, 1.0, -0.5000000000000002)],\n                       [(4, 5),\n                        (9, 10),\n                        (2, 0.8040867239846965, 0.0, 0.7425063126068279),\n                        (7, 0.8040867239846965, 0.0, -0.28964515054927653),\n                        (8, 0.0, 0.46332560231221276, 0.0)],\n                       [(2, 0),\n                        (7, 6),\n                        (5, 0.8040867239846965, 4.440892098500626e-16, 0.7425063126068276),\n                        (10, 0.8040867239846965, 0.0, -0.28964515054927653)],\n                       [(5, 1),\n                        (10, 11),\n                        (2, 0.7499999999999999, 0.9999999999999999, 0.7425063126068279),\n                        (7, 0.7499999999999999, 1.1102230246251565e-16, -0.28964515054927675),\n                        (0, 0.5, 0.0, 0.5),\n                        (6, 0.5, 1.0, 0.5)],\n                       [(3, 2),\n                        (8, 7),\n                        (10, 0.7499999999999999, 1.1102230246251565e-16, -0.28964515054927675),\n                        (5, 0.7499999999999999, 0.9999999999999999, 0.7425063126068279),\n                        (1, 0.5, 0.0, 0.5),\n                        (11, 0.5, 1.0, 0.5)],\n                       [(9, 10),\n                        (4, 5),\n                        (2, 0.5, 2.220446049250313e-16, -0.2574936873931719),\n                        (7, 0.5, 0.0, 0.710354849450723),\n                        (3, 0.5, 0.0, 0.9938724440055964),\n                        (8, 0.5, 1.0, -0.7683371988438938)],\n                       [(2, 0),\n                        (7, 6),\n                        (5, 0.5000000000000002, 0.0, -0.25749368739317235),\n                        (10, 0.5, 0.0, -0.28964515054927653),\n                        (9, 0.5, 0.0, 0.019885549076420705),\n                        (4, 0.5, 0.0, 0.5349613487416782)],\n                       [(5, 1),\n                        (10, 11),\n                        (2, 0.25000000000000006, 1.0, 0.7425063126068276),\n                        (7, 0.25000000000000006, 0.0, -0.28964515054927653)],\n                       [(3, 2),\n                        (8, 7),\n                        (10, 0.25000000000000006, 0.0, 0.7103548494507228),\n                        (5, 0.25000000000000006, 0.9999999999999998, 0.7425063126068276)],\n                       [(9, 10),\n                        (4, 5),\n                        (7, 0.5, 0.0, 0.710354849450723),\n                        (2, 0.5, 1.0, -0.2574936873931719),\n                        (8, 0.5, -0.7316628011561062, -0.7683371988438938),\n                        (3, 0.5, -0.4938724440055964, -0.006127555994403178)],\n                       [(7, 6),\n                        (2, 0),\n                        (10, 0.5, 0.0, 0.710354849450723),\n                        (5, 0.5, -0.9999999999999996, -0.25749368739317235),\n                        (4, 0.5, -0.03496134874167822, -0.4650386512583218),\n                        (9, 0.5, -0.5198855490764207, 0.019885549076420705)],\n                       [(10, 11),\n                        (5, 1),\n                        (6, 1.0, 0.0, 0.0),\n                        (0, 1.0, 0.0, 0.5),\n                        (2, 0.5, -0.2425063126068281, 0.7425063126068279),\n                        (7, 0.4999999999999998, -0.21035484945072325, -0.28964515054927675)],\n                       [(11, 1.0, 0.0, 0.0),\n                        (1, 1.0, 0.0, 0.5),\n                        (5, 0.5, -0.24250631260682765, 0.7425063126068276),\n                        (10, 0.4999999999999998, -0.21035484945072325, -0.28964515054927675)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],\n           \'qubits\': [(3, 4),\n                      (3, 7),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 3),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8)]},\n (12, 5): {\'method\': \'reachable-support count kernels + native scheduling\',\n           \'moments\': [[(2, 0.22481261212979553, 0.0, 0.5),\n                        (3, 0.35141000394669586, 0.0, -0.5),\n                        (5, 0.5, 0.0, -0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (8, 0.8363229499359709, 0.0, 0.5),\n                        (9, 0.9238916086653385, 0.0, -0.5),\n                        (7, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (10, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (0, 0.5, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (11, 0.5, 0.0, 0.5)],\n                       [(2, 3), (5, 4), (8, 9)],\n                       [(3, 0.6862285077038865, 0.0, -0.5),\n                        (8, 0.8402275817110916, 0.0, 0.5),\n                        (4, 0.700930528722115, 0.0, -0.5),\n                        (9, 0.433429197856543, 0.0, -0.5),\n                        (2, 0.3833200746081822, 0.0, 0.5),\n                        (5, 0.2990694712778854, 0.0, 0.5)],\n                       [(3, 8), (4, 9)],\n                       [(3, 0.4999999999999998, 0.0, 0.5), (8, 0.5, 0.0, 0.5)],\n                       [(3, 4), (8, 9)],\n                       [(3, 0.5, 0.0, -0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (8, 0.5773465114555592, 0.0, 0.5),\n                        (9, 0.8616248505549822, 0.0, 0.5)],\n                       [(2, 3), (5, 4)],\n                       [(3, 0.12013217332780404, 0.0, -0.5),\n                        (4, 0.06107035858744703, 0.0, -0.5000000000000002),\n                        (2, 0.5, 0.0, -0.5),\n                        (5, 0.9389296414125533, 0.0, -0.5)],\n                       [(3, 8), (4, 9)],\n                       [(3, 4), (8, 0.4296141178581542, 0.0, -0.5)],\n                       [(8, 9), (3, 0.5, 0.0, -0.5), (5, 4)],\n                       [(2, 3),\n                        (8, 0.4999999999999998, 0.0, 0.5),\n                        (4, 0.5, 0.0, -0.5),\n                        (9, 0.5, 0.0, 0.5),\n                        (5, 0.6994730230605577, 0.0, 1.0)],\n                       [(3, 8), (4, 9), (2, 0.5000000000000004, 4.440892098500626e-16, 0.8005269769394421)],\n                       [(3, 0.4589110952639191, 0.0, 1.0),\n                        (4, 0.4589110952639191, 0.0, 4.440892098500626e-16),\n                        (8, 0.22197270683743037, 0.0, -0.5821778094721621),\n                        (9, 0.28676557846249295, 0.0, -0.4329272144072267)],\n                       [(2, 3), (5, 4), (8, 7), (9, 10)],\n                       [(8, 0.5, 0.0, 0.9178221905278379),\n                        (9, 0.5, 0.0, -0.03852674713281079),\n                        (7, 0.85928593962011, 0.0, -0.9262332254895425),\n                        (2, 0.5, 0.0, -0.19947302306055814),\n                        (10, 0.85928593962011, 0.0, -0.9262332254895425),\n                        (5, 0.5, 0.0, 0.4999999999999998)],\n                       [(3, 8), (4, 9), (7, 2), (10, 5)],\n                       [(3, 0.5, 0.0, -0.2983513138770517),\n                        (4, 0.5, 0.0, 0.7016486861229474),\n                        (8, 0.6714165234362537, 0.0, -0.010761717741169186),\n                        (7, 0.7485956033393342, 0.0, -0.42623322548954246),\n                        (9, 0.6714165234362537, 0.0, -0.9671106554018183),\n                        (10, 0.7485956033393342, 0.0, -0.42623322548954246)],\n                       [(2, 3), (5, 4), (8, 7), (9, 10)],\n                       [(3, 0.5, 0.0, -0.2983513138770517),\n                        (8, 0.3935294720438233, 0.0, 0.26828539787245176),\n                        (4, 0.5, 0.0, 0.7016486861229474),\n                        (9, 0.3935294720438233, -2.220446049250313e-16, -0.6880635397881967),\n                        (7, 0.9961572718865779, 0.0, -0.9262332254895478),\n                        (2, 0.0528060550044127, 0.0, 0.9741273207264114),\n                        (10, 0.9961572718865779, 0.0, -0.9262332254895478),\n                        (5, 0.0528060550044127, 0.0, -0.32639965621303046)],\n                       [(3, 8), (4, 9), (7, 2), (10, 5)],\n                       [(2, 0.24963665794604517, 0.0, -0.5085100943638139),\n                        (5, 0.24963665794604517, 0.0, 0.19096292869674492),\n                        (8, 0.31043695253214243, 0.0, 0.8497641474722528),\n                        (7, 0.5655532211464794, 0.0, 0.5737667745104471),\n                        (9, 0.3104369525321424, 0.0, -0.10658479018839673),\n                        (10, 0.5655532211464794, 0.0, 0.5737667745104471)],\n                       [(2, 3), (5, 4), (8, 7), (9, 10)],\n                       [(8, 0.6193705712509703, 0.0, -0.2772701634271515),\n                        (9, 0.6193705712509703, 0.0, 0.7663808989121987),\n                        (7, 0.4999999999999997, 0.0, 0.5737667745104471),\n                        (2, 0.32701156232958084, 0.0, 0.09927026434808439),\n                        (10, 0.4999999999999997, 0.0, 0.5737667745104471),\n                        (5, 0.32701156232958084, 0.0, 0.7987432874086429)],\n                       [(3, 8), (4, 9), (7, 2), (10, 5)],\n                       [(8, 0.6256105257587086, 5.551115123125783e-17, 0.20600020458247753),\n                        (9, 0.6256105257587086, 0.0, -0.7503487330781726),\n                        (7, 0.5960614607313279, 0.0, -0.11684358694965402),\n                        (2, 0.5, 0.0, -0.6533652041730398),\n                        (10, 0.5960614607313279, 0.0, -0.11684358694965402),\n                        (5, 0.5, 0.0, 0.04610781888751925)],\n                       [(3, 8), (4, 9), (7, 2), (10, 5)],\n                       [(2, 0.5, 0.0, -0.15336520417304),\n                        (3, 0.5, 8.881784197001252e-16, -0.9772798845695692),\n                        (5, 0.5, 0.0, 0.5461078188875186),\n                        (4, 0.5, 0.0, 0.022720115430429733),\n                        (8, 0.9764747842833552, 8.881784197001252e-16, 0.8037844466618682),\n                        (7, 0.14949404734505523, 0.0, -0.7269328881171262),\n                        (9, 0.9764747842833552, 0.0, -0.15256449099877933),\n                        (10, 0.14949404734505523, 0.0, -0.7269328881171262)],\n                       [(2, 3), (5, 4), (8, 7), (9, 10)],\n                       [(3, 0.5, 8.881784197001252e-16, -0.9772798845695692),\n                        (8, 0.5, 0.0, -0.9532372701562573),\n                        (4, 0.5, 0.0, 0.022720115430429733),\n                        (9, 0.5, 0.0, 0.09041379218309448),\n                        (7, 2),\n                        (10, 5)],\n                       [(3, 8),\n                        (4, 9),\n                        (7, 0.4247761378557593, 0.0, -0.022381851578533674),\n                        (10, 0.4247761378557593, 0.0, -0.022381851578533674)],\n                       [(3, 0.5, 0.0, 0.9555312614117943),\n                        (4, 0.5, 0.0, -0.04446873858820721),\n                        (8, 0.14731871270290633, 0.0, -0.9532372701562561),\n                        (9, 0.14731871270290625, 0.0, 0.09041379218309409)],\n                       [(2, 3), (5, 4), (8, 7), (9, 10)],\n                       [(8, 0.9433458052208397, 0.0, -0.4532372701562577),\n                        (9, 0.9433458052208397, 0.0, 0.5904137921830941),\n                        (7, 0.45008695289946266, 0.0, -0.3365366861733614),\n                        (2, 0.4202007516434625, 0.0, -0.4468028457645421),\n                        (10, 0.45008695289946266, 0.0, -0.3365366861733614),\n                        (5, 0.4202007516434625, 0.0, 0.2526701772960167)],\n                       [(3, 8), (4, 9), (7, 2), (10, 5)],\n                       [(2, 0.541088904736081, 0.0, -0.5419987309252676),\n                        (5, 0.541088904736081, 0.0, 0.15747429213529096),\n                        (7, 0.541088904736081, 0.0, 0.21035484945072325),\n                        (10, 0.541088904736081, 0.0, 0.21035484945072325),\n                        (3, 0.5, 0.0, 0.9555312614117943),\n                        (4, 0.5, 0.0, -0.04446873858820721)],\n                       [(2, 0), (5, 1), (7, 6), (10, 11)],\n                       [(2, 0.4999999999999998, 0.0, -0.041998730925267846),\n                        (5, 0.5, 0.0, 0.6574742921352912),\n                        (7, 0.5, 0.0, -0.2896451505492772),\n                        (10, 0.5, 0.0, -0.2896451505492772),\n                        (0, 0.30408672398469605, 0.0, -0.5000000000000002),\n                        (1, 0.30408672398469605, 0.0, -0.5000000000000002),\n                        (6, 0.3040867239846962, 0.0, 0.4999999999999998),\n                        (11, 0.3040867239846962, 0.0, 0.4999999999999998)],\n                       [(3, 2), (4, 5), (8, 7), (9, 10)],\n                       [(2, 0.8040867239846965, 1.1102230246251565e-16, -0.04199873092526851),\n                        (5, 0.8040867239846965, 0.0, 0.6574742921352901),\n                        (7, 0.8040867239846965, 0.0, 0.7103548494507237),\n                        (10, 0.8040867239846965, 0.0, 0.7103548494507237)],\n                       [(2, 0), (5, 1), (7, 6), (10, 11)],\n                       [(2, 0.75, 0.0, -0.041998730925268735),\n                        (5, 0.75, 0.0, 0.6574742921352912),\n                        (7, 0.75, 0.0, 0.710354849450723),\n                        (10, 0.75, 0.0, 0.710354849450723),\n                        (0, 0.5, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (11, 0.5, 0.0, 0.5)],\n                       [(3, 2), (4, 5), (8, 7), (9, 10)],\n                       [(2, 0.5, 0.0, 0.958001269074731),\n                        (5, 0.5, 0.0, -0.3425257078647088),\n                        (7, 0.5, 0.0, 0.7103548494507237),\n                        (10, 0.5, 0.0, 0.7103548494507237),\n                        (3, 0.5, 0.0, 0.4349868090437538),\n                        (4, 0.5, 0.0, -0.5650131909562477),\n                        (8, 0.5, 0.0, -0.9326928177882177),\n                        (9, 0.5, 0.0, 0.11095824455113362)],\n                       [(2, 0), (5, 1), (7, 6), (10, 11)],\n                       [(2, 0.25000000000000006, 0.0, -0.04199873092526918),\n                        (5, 0.25000000000000006, 0.0, 0.6574742921352912),\n                        (7, 0.25000000000000006, 0.0, -0.2896451505492772),\n                        (10, 0.25000000000000006, 0.0, -0.2896451505492772)],\n                       [(3, 2), (4, 5), (8, 7), (9, 10)],\n                       [(2, 0.5, 4.440892098500626e-16, 0.9580012690747302),\n                        (5, 0.5000000000000001, 0.0, -0.3425257078647088),\n                        (7, 0.5, 0.0, -0.2896451505492772),\n                        (10, 0.5, 0.0, -0.2896451505492772),\n                        (3, 0.5, 0.06501319095624614, 0.43498680904375386),\n                        (4, 0.5, -0.9349868090437523, -0.5650131909562479),\n                        (8, 0.5, 0.4326928177882188, -0.9326928177882188),\n                        (9, 0.5, -0.6109582445511332, 0.11095824455113312)],\n                       [(2, 0), (5, 1), (7, 6), (10, 11)],\n                       [(0, 1.0, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5),\n                        (2, 0.5, -0.45800126907472993, 0.9580012690747302),\n                        (5, 0.5, 0.8425257078647082, -0.34252570786470815),\n                        (6, 1.0, 0.0, 0.5),\n                        (7, 0.4999999999999998, 0.7896451505492768, 0.7103548494507228),\n                        (10, 0.4999999999999998, 0.7896451505492768, 0.7103548494507228),\n                        (11, 1.0, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],\n           \'qubits\': [(3, 4),\n                      (3, 7),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 3),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8)]},\n (12, 6): {\'method\': \'reachable-support count kernels + native scheduling\',\n           \'moments\': [[(2, 0.9752354245236108, 0.0, 0.5),\n                        (3, 1.0, 0.0, 0.5),\n                        (5, 0.6179050039211448, 0.0, 0.5),\n                        (4, 0.6788698597100533, 0.0, -0.5),\n                        (8, 0.5385006094314906, 0.0, -0.5),\n                        (9, 0.5126546109165118, 0.0, 0.5),\n                        (7, 0.4127792040228902, 0.905374889856748, -0.9262332254895422),\n                        (6, 0.5, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (10, 0.4127792040228902, 0.905374889856748, -0.9262332254895422),\n                        (11, 0.5, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5)],\n                       [(2, 3), (5, 4), (8, 9)],\n                       [(3, 0.5144272659930226, 0.0, 0.5),\n                        (8, 0.40232584282186046, 0.0, -0.5),\n                        (4, 0.8339146082303962, 0.0, -0.5000000000000002),\n                        (9, 0.5966305683806078, 0.0, -0.5),\n                        (2, 0.47523542452361084, 0.0, 0.5),\n                        (5, 0.2004740902710799, 0.0, 0.5)],\n                       [(3, 8), (4, 9)],\n                       [(3, 0.3333411944998191, 0.0, -0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (8, 0.9370445056241755, 0.0, 0.5),\n                        (9, 0.5596530855663964, 0.0, -0.5)],\n                       [(3, 4), (8, 9)],\n                       [(3, 1.0, 0.0, 0.5),\n                        (4, 0.5, 0.0, -0.5),\n                        (8, 0.4442611216857431, 0.0, 0.5),\n                        (9, 0.27623206912931403, 0.0, 0.5)],\n                       [(2, 3), (5, 4)],\n                       [(3, 1.0, 0.0, 0.5),\n                        (4, 0.5767952821882669, 0.0, 0.5),\n                        (2, 0.5, 0.0, -0.5),\n                        (5, 0.5000000000000001, 0.0, -0.5)],\n                       [(3, 8), (4, 9)],\n                       [(3, 0.5, 0.0, 0.5),\n                        (4, 0.5759739595870296, 0.0, 0.5),\n                        (8, 0.4887934518361713, 0.0, -0.5),\n                        (9, 0.17745216697145247, 0.0, 0.5)],\n                       [(3, 4), (8, 9)],\n                       [(3, 1.0, 0.0, 0.5),\n                        (4, 0.5000000000000001, 0.0, -0.5),\n                        (8, 0.5763455522833301, 0.0, 0.5),\n                        (9, 0.8544758093340572, 0.0, -0.5)],\n                       [(2, 3), (5, 4)],\n                       [(3, 0.5, 0.0, -0.5),\n                        (4, 0.5, 0.0, -0.5),\n                        (2, 0.49999999999999994, 0.9280633834860437, 0.19947302306055764),\n                        (5, 0.5000000000000002, 0.32700942960715884, -0.1994730230605577)],\n                       [(3, 8), (4, 9)],\n                       [(3, 0.4999999999999998, 0.5282691788511853, 0.9589110952639188),\n                        (8, 0.48723979758966546, -0.5604981159152982, 0.5524684660050749),\n                        (4, 0.5000000000000001, -0.5539086306209766, -0.958911095263919),\n                        (9, 0.3166801131013955, -0.48194237681401564, -0.5626301404463834)],\n                       [(2, 3), (8, 7), (5, 4), (9, 10)],\n                       [(3, 1.0, 0.0, 0.21461996803249472),\n                        (8, 0.5, 0.7022844924594797, 0.4262761990482463),\n                        (7, 0.85928593962011, -0.4875608597182275, -0.02085833563279449),\n                        (2, 0.5000000000000001, 0.12925995895837872, -0.8724635934533991),\n                        (4, 1.0, 0.0, 0.21461996803249472),\n                        (9, 0.5, 0.7022844924594797, 0.4262761990482463),\n                        (10, 0.85928593962011, -0.4875608597182275, -0.02085833563279449),\n                        (5, 0.5000000000000001, 0.12925995895837872, -0.8724635934533991)],\n                       [(3, 8), (7, 2), (4, 9), (10, 5)],\n                       [(2, 1.0, 0.0, -0.28654568860164664),\n                        (3, 0.5000000000000001, -0.07216288375695012, -0.25958902417306295),\n                        (8, 0.6714165234362537, 0.7430251514267326, -0.8000232167612813),\n                        (7, 0.7485956033393342, -0.10058310030841655, -0.008419195351022202),\n                        (5, 1.0, 0.0, -0.28654568860164664),\n                        (4, 0.5000000000000001, -0.07216288375695012, -0.25958902417306295),\n                        (9, 0.6714165234362537, 0.7430251514267326, -0.8000232167612813),\n                        (10, 0.7485956033393342, -0.10058310030841655, -0.008419195351022202)],\n                       [(2, 3), (8, 7), (5, 4), (9, 10)],\n                       [(3, 0.5000000000000001, -0.47921561052476913, 0.6682480920699869),\n                        (8, 0.39352947204382344, -0.0400166187933757, 0.22204905027907273),\n                        (7, 0.996157271886578, -0.38731562230521455, -0.6090022956594439),\n                        (2, 0.05280605500441269, -0.7140339826902271, -0.0034880864952426816),\n                        (4, 0.5000000000000001, -0.47921561052476913, 0.6682480920699869),\n                        (9, 0.39352947204382344, -0.0400166187933757, 0.22204905027907273),\n                        (10, 0.996157271886578, -0.38731562230521455, -0.6090022956594439),\n                        (5, 0.05280605500441269, -0.7140339826902271, -0.0034880864952426816)],\n                       [(3, 8), (7, 2), (4, 9), (10, 5)],\n                       [(2, 0.24963665794604506, 0.8669932073346845, 0.7651153459047553),\n                        (3, 1.0, 0.0, 0.21763245964208),\n                        (8, 0.3104369525321425, 0.8085371278988395, -0.23648881891450202),\n                        (7, 0.5655532211464792, -0.7603717865514881, -0.49631791796466374),\n                        (5, 0.24963665794604506, 0.8669932073346845, 0.7651153459047553),\n                        (4, 1.0, 0.0, 0.21763245964208),\n                        (9, 0.3104369525321425, 0.8085371278988395, -0.23648881891450202),\n                        (10, 0.5655532211464792, -0.7603717865514881, -0.49631791796466374)],\n                       [(2, 3), (8, 7), (5, 4), (9, 10)],\n                       [(3, 1.3678757607663618e-17, 0.19456334695121486, 0.24274381274574858),\n                        (8, 0.6193705712509703, -0.4502913348689348, -0.5549860019150674),\n                        (7, 0.49999999999999994, 0.914005305656897, 0.7433102954838477),\n                        (2, 0.32701156232958095, 0.6955438578648168, -0.9756718054724585),\n                        (4, 1.3678757607663618e-17, 0.19456334695121486, 0.24274381274574858),\n                        (9, 0.6193705712509703, -0.4502913348689348, -0.5549860019150674),\n                        (10, 0.49999999999999994, 0.914005305656897, 0.7433102954838477),\n                        (5, 0.32701156232958095, 0.6955438578648168, -0.9756718054724585)],\n                       [(3, 8), (7, 2), (4, 9), (10, 5)],\n                       [(3, 1.0, 0.0, 0.3991857392005793),\n                        (8, 0.6256105257587085, 0.9452263271866326, -0.5220069687743731),\n                        (7, 0.5960614607313279, 0.2573629292943688, -0.033294760319355854),\n                        (2, 0.5000000000000001, 0.3922892154072385, 0.472507520913482),\n                        (4, 1.0, 0.0, 0.3991857392005793),\n                        (9, 0.6256105257587085, 0.9452263271866326, -0.5220069687743731),\n                        (10, 0.5960614607313279, 0.2573629292943688, -0.033294760319355854),\n                        (5, 0.5000000000000001, 0.3922892154072385, 0.472507520913482)],\n                       [(3, 8), (7, 2), (4, 9), (10, 5)],\n                       [(2, 0.5, -0.2739726829496021, 0.3647967363207205),\n                        (3, 0.5000000000000001, -0.2267887021096202, 0.036504264403519004),\n                        (8, 0.9764747842833552, 0.005682454151557326, 0.021003600491651175),\n                        (7, 0.14949404734505523, 0.6915804042354776, 0.6139788678075409),\n                        (5, 0.5, -0.2739726829496021, 0.3647967363207205),\n                        (4, 0.5000000000000001, -0.2267887021096202, 0.036504264403519004),\n                        (9, 0.9764747842833552, 0.005682454151557326, 0.021003600491651175),\n                        (10, 0.14949404734505523, 0.6915804042354776, 0.6139788678075409)],\n                       [(2, 3), (8, 7), (5, 4), (9, 10)],\n                       [(3, 0.5, -0.9528264593460201, 0.8097155622938987),\n                        (8, 0.5, 0.47939827321355255, 0.2696643378250829),\n                        (7, 1.0, 0.0, -0.31991326341917814),\n                        (2, 1.0, 0.0, -0.27721006339896803),\n                        (4, 0.5, -0.9528264593460201, 0.8097155622938987),\n                        (9, 0.5, 0.47939827321355255, 0.2696643378250829),\n                        (10, 1.0, 0.0, -0.31991326341917814),\n                        (5, 1.0, 0.0, -0.27721006339896803)],\n                       [(3, 8), (7, 2), (4, 9), (10, 5)],\n                       [(2, 2.6911488363979655e-17, -0.10657650605920965, 0.3199678821356995),\n                        (3, 0.5000000000000001, 0.6065964482612792, -0.07592204303348415),\n                        (8, 0.14731871270290642, 0.47763908808934363, -0.2509373889613644),\n                        (7, 0.42477613785575913, 0.24565143911589105, -0.6499368354199673),\n                        (5, 2.6911488363979655e-17, -0.10657650605920965, 0.3199678821356995),\n                        (4, 0.5000000000000001, 0.6065964482612792, -0.07592204303348415),\n                        (9, 0.14731871270290642, 0.47763908808934363, -0.2509373889613644),\n                        (10, 0.42477613785575913, 0.24565143911589105, -0.6499368354199673)],\n                       [(2, 3), (8, 7), (5, 4), (9, 10)],\n                       [(3, 2.183989817369057e-17, 0.04089633285515237, 0.7520884815410067),\n                        (8, 0.9433458052208397, -0.437164462020319, -0.27329830087202156),\n                        (7, 0.4500869528994625, 0.5858004296037851, -0.09013056170924827),\n                        (2, 0.42020075164346243, -0.13244587453752787, -0.04525832781976602),\n                        (4, 2.183989817369057e-17, 0.04089633285515237, 0.7520884815410067),\n                        (9, 0.9433458052208397, -0.437164462020319, -0.27329830087202156),\n                        (10, 0.4500869528994625, 0.5858004296037851, -0.09013056170924827),\n                        (5, 0.42020075164346243, -0.13244587453752787, -0.04525832781976602)],\n                       [(3, 8), (7, 2), (4, 9), (10, 5)],\n                       [(7, 0.5410889047360808, -0.9487783322704517, -0.05122166772954828),\n                        (8, 5.822447387112414e-17, 0.6899183105243011, -0.1202404731065354),\n                        (2, 0.5410889047360808, 0.2729000875180194, 0.7270999124819801),\n                        (3, 0.5000000000000002, -0.5921151904509876, 0.5715707380829471),\n                        (10, 0.5410889047360808, -0.9487783322704517, -0.05122166772954828),\n                        (9, 5.822447387112414e-17, 0.6899183105243011, -0.1202404731065354),\n                        (5, 0.5410889047360808, 0.2729000875180194, 0.7270999124819801),\n                        (4, 0.5000000000000002, -0.5921151904509876, 0.5715707380829471)],\n                       [(7, 6), (2, 0), (10, 11), (5, 1)],\n                       [(7, 0.49999999999999983, 0.0, -0.5),\n                        (6, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (2, 0.49999999999999983, 0.0, -0.5),\n                        (0, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (10, 0.49999999999999983, 0.0, -0.5),\n                        (11, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (5, 0.49999999999999983, 0.0, -0.5),\n                        (1, 0.3040867239846963, 0.0, -0.5000000000000002)],\n                       [(8, 7), (3, 2), (9, 10), (4, 5)],\n                       [(7, 0.8040867239846964, 0.0, 0.5),\n                        (8, 1.0, 0.0, 0.5),\n                        (2, 0.8040867239846964, 0.0, 0.5),\n                        (3, 1.0, 0.0, 0.5),\n                        (10, 0.8040867239846964, 0.0, 0.5),\n                        (9, 1.0, 0.0, 0.5),\n                        (5, 0.8040867239846964, 0.0, 0.5),\n                        (4, 1.0, 0.0, 0.5)],\n                       [(7, 6), (2, 0), (10, 11), (5, 1)],\n                       [(7, 0.75, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (2, 0.75, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (10, 0.75, 0.0, 0.5),\n                        (11, 0.5, 0.0, 0.5),\n                        (5, 0.75, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5)],\n                       [(8, 7), (3, 2), (9, 10), (4, 5)],\n                       [(7, 0.5000000000000001, 0.0, -0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (2, 0.5000000000000001, 0.0, -0.5),\n                        (3, 0.5, 0.0, 0.5),\n                        (10, 0.5000000000000001, 0.0, -0.5),\n                        (9, 0.5, 0.0, 0.5),\n                        (5, 0.5000000000000001, 0.0, -0.5),\n                        (4, 0.5, 0.0, 0.5)],\n                       [(7, 6), (2, 0), (10, 11), (5, 1)],\n                       [(7, 0.25000000000000006, 0.0, 0.5),\n                        (6, 1.0, 0.0, 0.5),\n                        (2, 0.25000000000000006, 0.0, 0.5),\n                        (0, 1.0, 0.0, 0.5),\n                        (10, 0.25000000000000006, 0.0, 0.5),\n                        (11, 1.0, 0.0, 0.5),\n                        (5, 0.25000000000000006, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5)],\n                       [(8, 7), (3, 2), (9, 10), (4, 5)],\n                       [(7, 0.5, 0.0, -0.5),\n                        (2, 0.5, 0.0, -0.5),\n                        (10, 0.5, 0.0, -0.5),\n                        (5, 0.5, 0.0, -0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (3, 0.5, 0.0, 0.5),\n                        (9, 0.5, 0.0, 0.5),\n                        (4, 0.5, 0.0, 0.5)],\n                       [(7, 6), (2, 0), (10, 11), (5, 1)],\n                       [(7, 0.5000000000000001, 0.0, 0.5),\n                        (2, 0.5000000000000001, 0.0, 0.5),\n                        (10, 0.5000000000000001, 0.0, 0.5),\n                        (5, 0.5000000000000001, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],\n           \'qubits\': [(3, 4),\n                      (3, 7),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 3),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8)]},\n (13, 3): {\'method\': \'reachable-support count kernels + native scheduling\',\n           \'moments\': [[(0, 0.5011286333212674, 0.0, -0.5),\n                        (5, 0.5648042837558234, 0.0, -0.5),\n                        (1, 0.22639989947272104, 0.0, 0.5),\n                        (6, 0.37901296827077646, 0.0, 0.5),\n                        (9, 0.3584365439544183, 0.0, 0.5),\n                        (10, 0.6115265567555555, 0.0, -0.5),\n                        (2, 0.41218411310679315, 4.973524073736968e-05, -0.5000093746463457),\n                        (3, 0.4535018303190155, 3.2138086280997236e-05, -0.4999866117568501),\n                        (4, 0.6538196916638768, 0.0012975466470504515, 0.4992166768114052),\n                        (7, 0.4135610417387456, 0.0, -0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (11, 0.4135610417387456, 0.0, -0.5),\n                        (12, 0.5, 0.0, 0.5)],\n                       [(0, 5), (1, 6), (9, 10)],\n                       [(0, 0.4944183722198826, 0.0, 0.5),\n                        (1, 0.9887937899601499, 2.220446049250313e-16, -0.5000000000000002),\n                        (5, 0.0054663407070543735, 0.0, 0.5),\n                        (9, 0.20817138739784288, 0.0, -0.5),\n                        (6, 0.05896518616556724, 0.0, 0.5),\n                        (10, 0.5109898938650242, 0.0, 0.5)],\n                       [(0, 1), (5, 9), (6, 10)],\n                       [(0, 0.5000000000000001, 0.0, -0.5),\n                        (1, 0.7787066990539395, 0.0, 0.5),\n                        (5, 1.0, 0.0, 0.5),\n                        (6, 0.2666010547086309, 0.0, -0.5),\n                        (9, 0.9044411812600366, 2.220446049250313e-16, -0.5000000000000002),\n                        (10, 0.5357339946022933, 0.0, -0.5)],\n                       [(0, 1), (5, 6), (9, 10)],\n                       [(0, 1.0, 0.0, 0.5),\n                        (5, 0.5, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5),\n                        (6, 0.9592381571300939, 0.0, 0.5),\n                        (9, 0.20484824042867295, 0.0, 0.5),\n                        (10, 0.22637759711444339, 0.0, 0.5)],\n                       [(0, 5), (1, 6), (9, 10)],\n                       [(0, 0.540825634749762, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5),\n                        (5, 0.6799453846918434, 0.0, -0.5),\n                        (9, 0.6796033860670321, 0.0, 0.5),\n                        (6, 0.9913262427252412, 0.0, -0.5000000000000002),\n                        (10, 0.8874597674119914, 0.0, 0.5)],\n                       [(0, 1), (5, 9), (6, 10)],\n                       [(0, 0.35408611734683687, 0.0, 0.5),\n                        (5, 0.5122437211025594, 0.0, -0.5),\n                        (1, 0.47699444027673116, 0.0, -0.5),\n                        (6, 0.8492358345397026, 0.0, 0.5),\n                        (10, 0.8351725982640147, 0.0, -0.5)],\n                       [(0, 5), (1, 6), (9, 10)],\n                       [(0, 0.4999999999999993, 0.0, 0.5),\n                        (1, 0.8130648608503948, 0.0, -0.5000000000000002),\n                        (5, 0.5, 0.0, -0.5),\n                        (9, 1.0, 0.0, 0.5),\n                        (6, 0.43942608628445445, 0.0, -0.5),\n                        (10, 0.6237193013742914, 0.0, 0.5)],\n                       [(0, 1), (5, 9), (6, 10)],\n                       [(1, 0.03811502306613776, 0.0, -0.5000000000000002),\n                        (5, 1.0, 0.0, 0.5),\n                        (6, 0.6322754934663775, 0.0, 0.5),\n                        (9, 1.0, 0.0, 0.5),\n                        (10, 0.6383902508124725, 0.0, 0.5)],\n                       [(0, 1), (5, 6), (9, 10)],\n                       [(1, 0.8118240687303543, 1.1101809104463456e-05, 0.5000147959624823),\n                        (0, 0.9999999999999994, -3.907090327559537e-05, -0.5000140183935871),\n                        (6, 0.5591170679959561, 0.0, -0.5),\n                        (10, 0.10047114062159843, 0.0, -0.5000000000000002),\n                        (9, 1.0, 0.0, 0.5)],\n                       [(1, 2), (6, 7), (10, 11)],\n                       [(1, 0.5125153233564552, 6.816602735071342e-05, -0.5000700309596928),\n                        (2, 0.6117890444840384, 0.00023906418654384698, -0.5000335859224758),\n                        (6, 0.3819668494768557, 0.0, 0.5),\n                        (7, 0.6338602364006151, 0.0, -0.5),\n                        (10, 0.3819668494768557, 0.0, 0.5),\n                        (11, 0.6338602364006151, 0.0, -0.5)],\n                       [(0, 1), (5, 6), (9, 10)],\n                       [(1, 0.7075435896451812, -0.00015441380199909815, 0.5000167573434053),\n                        (0, 1.0, 0.0, -0.4999840678705958),\n                        (6, 0.7376490383100945, 0.0, 0.5),\n                        (5, 1.0, 0.0, 0.5),\n                        (10, 0.7376490383100945, 0.0, 0.5),\n                        (9, 1.0, 0.0, 0.5)],\n                       [(1, 2), (6, 7), (10, 11)],\n                       [(1, 0.13376593564282768, -3.928146360454576e-05, 0.5001322711129306),\n                        (2, 0.16507290691754561, -4.4512723523348896e-05, 0.49992633361577),\n                        (6, 0.16018463696253962, 0.0, 0.5),\n                        (7, 0.18023932987586394, 0.0, -0.5000000000000002),\n                        (10, 0.16018463696253962, 0.0, 0.5),\n                        (11, 0.18023932987586394, 0.0, -0.5000000000000002)],\n                       [(0, 1), (5, 6), (9, 10)],\n                       [(1, 0.4425626628501562, 0.00019430062546132998, 0.49988424829215317),\n                        (0, 0.5, -2.699936250127877e-07, 0.50000374169819),\n                        (6, 0.4422317453870271, 0.0, 0.5),\n                        (5, 0.5, 0.0, 0.5),\n                        (10, 0.4422317453870271, 0.0, 0.5),\n                        (9, 0.5, 0.0, 0.5)],\n                       [(1, 2), (6, 7), (10, 11)],\n                       [(1, 0.28336083660654154, 6.297169447666562e-05, -0.500078436406141),\n                        (2, 0.874698411133806, -5.5613993600545086e-05, -0.4999329159564543),\n                        (6, 0.6666666666666669, 0.0, 0.5),\n                        (7, 0.8040867239846964, 0.0, -0.5000000000000002),\n                        (10, 0.6666666666666669, 0.0, 0.5),\n                        (11, 0.8040867239846964, 0.0, -0.5000000000000002)],\n                       [(0, 1), (2, 3), (5, 6), (7, 8), (9, 10), (11, 12)],\n                       [(1, 0.9999999999999999, 1.0626804806790346e-05, 0.5000138488409602),\n                        (2, 0.5195218910218442, -2.492251341545959e-05, 0.49999809046902643),\n                        (3, 0.6338602251592093, -4.516549831157235e-05, -0.5000105128978589),\n                        (6, 1.0, 0.0, 0.5),\n                        (7, 0.49999999999999983, 0.0, -0.5),\n                        (8, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (10, 1.0, 0.0, 0.5),\n                        (11, 0.49999999999999983, 0.0, -0.5),\n                        (12, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (0, 0.5000000000000002, -9.01601680149966e-05, -0.4999965282954304),\n                        (5, 0.5, 0.0, -0.5),\n                        (9, 0.5, 0.0, -0.5)],\n                       [(1, 2), (6, 7), (10, 11)],\n                       [(2, 0.7465546320890828, -6.819337841124096e-05, 0.5000383284289129),\n                        (1, 1.0, 0.0, 0.4999738833942602),\n                        (7, 0.8040867239846964, 0.0, 0.5),\n                        (6, 1.0, 0.0, 0.5),\n                        (11, 0.8040867239846964, 0.0, 0.5),\n                        (10, 1.0, 0.0, 0.5)],\n                       [(2, 3), (7, 8), (11, 12)],\n                       [(2, 0.1601846384780278, -4.503868777239184e-05, 0.5000255168882044),\n                        (3, 0.288907983885649, 8.07695182105661e-05, -0.500167316940584),\n                        (7, 0.75, 0.0, 0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (11, 0.75, 0.0, 0.5),\n                        (12, 0.5, 0.0, 0.5)],\n                       [(1, 2), (6, 7), (10, 11)],\n                       [(2, 0.4422317455429899, -3.658965661768043e-05, 0.5000154601436346),\n                        (1, 0.5, -1.2757522787731901e-05, 0.5000119394225819),\n                        (7, 0.5000000000000001, 0.0, -0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (11, 0.5000000000000001, 0.0, -0.5),\n                        (10, 0.5, 0.0, 0.5)],\n                       [(2, 3), (7, 8), (11, 12)],\n                       [(2, 0.7312715381493196, -2.9719479089695966e-05, 0.5000244754386074),\n                        (3, 0.9620769699850723, -0.00013852518429713356, 0.500054648470984),\n                        (7, 0.25000000000000006, 0.0, 0.5),\n                        (8, 1.0, 0.0, 0.5),\n                        (11, 0.25000000000000006, 0.0, 0.5),\n                        (12, 1.0, 0.0, 0.5)],\n                       [(1, 2), (3, 4), (6, 7), (10, 11)],\n                       [(3, 0.37351126548912544, -7.546439919492443e-05, 0.5000773294001128),\n                        (4, 0.3040867239846963, -0.0005776526930145742, -0.49948577654154436),\n                        (2, 5.255863519135624e-17, -1.712579946921977e-05, 0.5012001664600749),\n                        (7, 0.5, 0.0, -0.5),\n                        (11, 0.5, 0.0, -0.5),\n                        (1, 0.5, -4.5164637105132854e-05, -0.5000008181002027),\n                        (6, 0.5, 0.0, 0.5),\n                        (10, 0.5, 0.0, 0.5)],\n                       [(3, 4), (7, 8), (11, 12)],\n                       [(3, 0.5000000000000004, -1.4577800550252817e-05, -0.4999981349990823),\n                        (4, 0.20227432911242674, -0.00018048654702673694, 0.4999365707654403),\n                        (7, 0.5000000000000001, 0.0, 0.5),\n                        (11, 0.5000000000000001, 0.0, 0.5)],\n                       [(2, 3)],\n                       [(3, 0.25000000000000006, -2.019130418995374e-05, -0.49998019675610816),\n                        (2, 0.5, -3.842212173532644e-06, 0.5000044520738967)],\n                       [(3, 4)],\n                       [(3, 0.25, 4.345730322263819e-07, 0.4999996119397021),\n                        (4, 0.5000000000000004, 0.00010607496466885635, -0.5002764318251103)],\n                       [(2, 3)],\n                       [(3, 0.5, -4.246260076889996e-06, -0.4999999534872659),\n                        (2, 0.5, -2.4189174229860555e-05, 0.5000006098617227)],\n                       [(3, 4)],\n                       [(3, 0.5000000000000001, -1.937956516417927e-05, 0.4999958002526569),\n                        (4, 6.271301173889318e-17, 0.00014677754793412934, 0.4999292001208716)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (3, 6),\n                      (3, 7),\n                      (3, 8),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7)]},\n (13, 4): {\'method\': \'reachable-support count kernels + native scheduling\',\n           \'moments\': [[(2, 0.5578274214919605, 0.0, 0.5),\n                        (3, 0.23366943453940836, 0.0, -0.5),\n                        (4, 0.41031150239878506, 0.0, 0.5),\n                        (9, 0.5, 0.0, 0.5),\n                        (10, 0.4999999999999998, 0.0, -0.5000000000000002),\n                        (8, 0.4999999999999997, 0.9999999999999999, 0.5937872305331664),\n                        (11, 0.7728144741714948, 0.0, 0.0805732800089835),\n                        (12, 0.4999999999999998, 0.0, 0.5937872305331664),\n                        (0, 0.5, 0.0, 0.5),\n                        (7, 0.5, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5)],\n                       [(2, 3), (5, 4), (9, 10)],\n                       [(3, 0.9485603490041102, 0.0, 0.5),\n                        (9, 0.4588991616887064, 0.0, -0.5),\n                        (4, 0.7216746256016107, 0.0, 0.5),\n                        (10, 0.04110083831129362, 0.0, 0.5),\n                        (2, 0.9227936295542016, 1.0, 0.5),\n                        (5, 0.5, 1.0, 0.5)],\n                       [(3, 9), (4, 10)],\n                       [(3, 0.5, 0.0, -0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (9, 0.4496124391437314, 5.551115123125783e-17, 0.49999999999999994),\n                        (10, 0.9496124391437313, 0.0, -0.5000000000000002)],\n                       [(3, 4), (9, 10)],\n                       [(2, 3), (5, 4), (9, 0.5, 1.0, 0.5), (10, 0.5, 1.0, -0.5)],\n                       [(3, 0.73179775895637, 0.0, 0.5),\n                        (4, 0.6066435204495901, 0.0, 0.5),\n                        (2, 0.5, 1.0, -0.5),\n                        (5, 0.5, 1.0, 0.5)],\n                       [(3, 9), (4, 10)],\n                       [(3, 0.7820471084244874, 0.0, 0.5),\n                        (4, 0.3933564795504099, 0.0, -0.5000000000000002),\n                        (10, 0.17490085614052123, 1.0, 0.5)],\n                       [(3, 4), (9, 10)],\n                       [(3, 0.5, 0.0, 0.5), (4, 0.5, 0.0, 0.5), (10, 0.4999999999999998, 1.0, 0.5)],\n                       [(2, 3), (5, 4)],\n                       [(3, 9), (4, 0.49999999999999956, 0.0, 0.5), (2, 0.5, 1.0, -0.5), (5, 0.5, 1.0, 0.5)],\n                       [(4, 10),\n                        (3, 0.9901052994948099, 8.881784197001252e-16, 0.999999999999996),\n                        (9, 0.5, 0.0, 0.019789401010380736)],\n                       [(2, 3),\n                        (9, 8),\n                        (10, 0.32509914385947836, 0.0, -0.5),\n                        (4, 0.5423492035105278, 0.0, -0.2865947927371191)],\n                       [(10, 11),\n                        (5, 4),\n                        (3, 0.0, 0.8648664878702104, 0.0),\n                        (9, 0.9416885504219519, 0.0, 0.5705895034222981),\n                        (8, 0.6085354469534892, -0.9999999999999996, -0.9311758863092963)],\n                       [(3, 9),\n                        (11, 0.7728144741714948, 0.0, -0.9194267199910164),\n                        (5, 0.5, 0.0, 0.9231677647409815),\n                        (4, 0.2630328282816796, 0.0, -0.020064248512857485),\n                        (10, 0.791814249190367, -0.9999999999999996, 0.3871439398384573)],\n                       [(11, 5),\n                        (4, 10),\n                        (3, 0.09666409909802187, 0.0, -0.1833907144994662),\n                        (9, 0.5, 0.0, -0.3566812641607169)],\n                       [(2, 3),\n                        (9, 8),\n                        (5, 0.5, 1.0, -0.07683223525901872),\n                        (4, 0.791370059531975, 0.0, -0.027626445409131506),\n                        (10, 0.6714475580169545, 1.0, -0.1541927339008855)],\n                       [(5, 4),\n                        (10, 11),\n                        (3, 0.5812745787265814, -0.9999999999999996, -0.38929274582530704),\n                        (9, 0.5150139304348057, 0.0, -0.3566812641607169),\n                        (8, 0.16723314080476154, 1.0, -0.7374666288921616)],\n                       [(3, 9),\n                        (11, 0.5, 0.0, 0.2456489329819702),\n                        (4, 0.6715105414343046, 0.0, -0.9636727402756899),\n                        (10, 0.5751637920695032, 1.0, -0.8183844063186687)],\n                       [(11, 5),\n                        (4, 10),\n                        (3, 0.2955394861408327, 0.0, 0.5833011723905248),\n                        (9, 0.08858950606155513, 0.0, -0.8566812641607169)],\n                       [(2, 3),\n                        (9, 8),\n                        (4, 0.8057807413978859, 0.0, 0.19411818156114946),\n                        (10, 0.3240470566789824, 0.9999999999999999, 0.9612835579157248),\n                        (11, 0.5, 0.0, 0.24564893298197044)],\n                       [(5, 4),\n                        (10, 11),\n                        (3, 0.2943011721717048, 1.0, 0.6055142112978777),\n                        (9, 0.3017505280831282, 0.0, -0.3566812641607169),\n                        (2, 0.49999999999999967, 0.0, 0.7450526497474059),\n                        (8, 0.6303550437629813, 1.0, -0.18158983536649442)],\n                       [(3, 9),\n                        (4, 0.6831600707545022, -2.220446049250313e-16, 0.1987927050404905),\n                        (10, 0.5461136927787811, -0.9999999999999998, -0.48189054808852116),\n                        (11, 0.5, 0.0, 0.6159985518275233),\n                        (5, 0.0, 0.35939939477765753, 0.0)],\n                       [(4, 10),\n                        (11, 5),\n                        (3, 0.30286229487152805, 2.220446049250313e-16, -0.4040911538420522),\n                        (9, 0.19954896443512293, 0.0, -0.8566812641607167)],\n                       [(2, 3),\n                        (9, 8),\n                        (5, 0.5, 0.9999999999999998, -0.35622594245338646),\n                        (4, 0.5852312538430753, 0.0, -0.670924474199621),\n                        (10, 0.7893393807194902, 0.9999999999999998, 0.5580609028067265),\n                        (11, 0.5, 0.0, 0.6159985518275233)],\n                       [(5, 4),\n                        (10, 11),\n                        (3, 0.5, 1.0, 0.36435952010894634),\n                        (9, 0.22177219480665555, 0.0, 0.6433187358392833),\n                        (2, 0.33347826094116906, 0.0, 0.24505264974740465),\n                        (8, 0.8315920243625678, 1.0, -0.4157702878189209)],\n                       [(3, 9),\n                        (4, 0.0, 0.08324666599350922, 0.0),\n                        (10, 0.7114915454728947, 0.0, 0.46709466269247035),\n                        (11, 0.0, -0.9935579880705372, 0.0),\n                        (5, 0.324430705358977, 0.0, -0.856225942453386)],\n                       [(4, 10),\n                        (11, 5),\n                        (3, 0.5, 0.0, 0.364359520108946),\n                        (9, 0.5000000000000002, 0.0, 0.156281263236826)],\n                       [(2, 3),\n                        (9, 8),\n                        (10, 0.6122002465840692, 4.440892098500626e-16, -0.9844781776159088),\n                        (11, 0.5, 0.0, -0.6896229878078779),\n                        (5, 0.5, 0.37129927578369415, 0.6437740575466129)],\n                       [(4, 10),\n                        (11, 12),\n                        (3, 0.5519231616233388, 1.0, -0.4178441633594212),\n                        (9, 0.5, 0.0, 0.15628126323682645),\n                        (2, 0.5276550549544043, 0.0, 0.2450526497474046),\n                        (8, 0.5, 0.9999999999999999, 0.46178536847925333)],\n                       [(3, 9),\n                        (11, 0.9416885504219519, 0.9999999999999998, -0.2404230902197959),\n                        (12, 0.6085354469534892, 8.881784197001252e-16, -0.9311758863092963),\n                        (4, 0.0, 1.0, 0.0)],\n                       [(10, 11),\n                        (3, 0.5, 4.440892098500626e-16, 0.9480090827349839),\n                        (9, 0.5, 0.0, -0.5442400746310239)],\n                       [(2, 3),\n                        (9, 8),\n                        (11, 0.5, 0.0, 0.6868476773632199),\n                        (10, 0.09666409909802187, 0.0, 0.8295970343555588)],\n                       [(11, 12),\n                        (4, 10),\n                        (2, 0.5000000000000002, 0.0, -0.25494735025259585),\n                        (3, 0.5, 0.0, -0.051990917265015923),\n                        (8, 0.5, 1.0, -0.038214631520746334)],\n                       [(2, 3),\n                        (9, 8),\n                        (10, 0.5812745787265814, 0.0, -0.9645009343186002),\n                        (11, 0.5150139304348058, 0.9999999999999998, 0.6868476773632199),\n                        (12, 0.16723314080476154, 0.0, -0.7374666288921621),\n                        (4, 2.220446049250313e-16, 1.0, 0.2435452314781319)],\n                       [(10, 11),\n                        (2, 0.4901053274689138, 0.0, -0.2549238179692832),\n                        (8, 0.5068340379253959, 0.0, -0.5310585926382718),\n                        (3, 0.5, 0.0, -0.05199091726501637),\n                        (9, 0.5, 0.0, -0.044240074631024306)],\n                       [(2, 0),\n                        (8, 7),\n                        (11, 0.08858950606155513, 0.0, -0.8131523226367803),\n                        (10, 0.2955394861408327, 0.0, 0.06290514746556775)],\n                       [(11, 12),\n                        (4, 10),\n                        (2, 0.5, 0.0, 0.24431903208484224),\n                        (8, 0.5, 1.0, -0.7736945421001282),\n                        (0, 0.30408672398469616, 0.0, -0.5000000000000002),\n                        (7, 0.30408672398469616, 0.0, -0.5000000000000002)],\n                       [(3, 2),\n                        (9, 8),\n                        (10, 0.2943011721717049, 0.0, 0.04069210855821448),\n                        (11, 0.30175052808312813, 0.9999999999999998, 0.6868476773632197),\n                        (12, 0.6303550437629813, 2.220446049250313e-16, -0.18158983536649442),\n                        (4, 0.49999999999999956, 0.9999999999999999, 0.6441131210935778)],\n                       [(10, 11),\n                        (2, 0.8040867239846965, 8.881784197001252e-16, -0.7556809679151584),\n                        (8, 0.8040867239846965, 0.0, 0.22630545789987178),\n                        (9, 0.0, 0.5526603939617729, 0.0)],\n                       [(2, 0),\n                        (8, 7),\n                        (11, 0.19954896443512302, 0.0, -0.8131523226367807),\n                        (10, 0.30286229487152805, 8.881784197001252e-16, -0.9497025263018555)],\n                       [(11, 12),\n                        (4, 10),\n                        (8, 0.7499999999999999, 5.551115123125783e-17, 0.22630545789987172),\n                        (2, 0.7499999999999999, 1.0, -0.7556809679151579),\n                        (0, 0.5, 0.0, 0.5),\n                        (7, 0.5, 0.0, 0.5)],\n                       [(9, 8),\n                        (3, 2),\n                        (10, 0.5, 0.0, 0.2818467997471461),\n                        (11, 0.22177219480665555, 1.0, -0.3131523226367805),\n                        (12, 0.8315920243625678, 1.1102230246251565e-16, -0.4157702878189209),\n                        (4, 0.3334782609411692, -0.9999999999999996, 0.1441131210935767)],\n                       [(10, 11),\n                        (2, 0.5000000000000002, 0.0, 0.24431903208484212),\n                        (8, 0.5, 0.0, -0.7736945421001282),\n                        (9, 0.5, 0.0, 0.27633019698088646),\n                        (3, 0.5, 0.0, 0.6654066775279848)],\n                       [(2, 0),\n                        (8, 7),\n                        (11, 0.5, -2.220446049250313e-16, 0.17388514996567728),\n                        (10, 0.5, 0.0, 0.2818467997471465)],\n                       [(11, 12),\n                        (4, 10),\n                        (8, 0.25000000000000006, 5.551115123125783e-17, 0.22630545789987166),\n                        (2, 0.25000000000000006, -0.9999999999999996, -0.7556809679151579)],\n                       [(9, 8),\n                        (3, 2),\n                        (10, 0.5519231616233388, 0.0, -0.9359495167844865),\n                        (11, 0.5, 1.0, 0.17388514996567667),\n                        (12, 0.5, 0.0, 0.46178536847925344),\n                        (4, 0.5276550549544043, 1.0, 0.14411312109357727)],\n                       [(10, 11),\n                        (2, 0.5, 1.0, 0.24431903208484224),\n                        (8, 0.5, 1.0, -0.7736945421001282),\n                        (9, 0.5, 0.22366980301911354, 0.27633019698088646),\n                        (3, 0.5, -0.16540667752798477, -0.33459332247201523)],\n                       [(2, 0),\n                        (8, 7),\n                        (11, 0.5, 0.0, 0.8744064878335265),\n                        (10, 0.5, 0.0, -0.3018027628788915)],\n                       [(11, 12),\n                        (4, 10),\n                        (0, 1.0, 0.0, 0.5),\n                        (2, 0.5, -0.7443190320848423, -0.7556809679151579),\n                        (7, 1.0, 0.0, 0.5),\n                        (8, 0.5, 0.2736945421001282, 0.22630545789987178)],\n                       [(12, 0.5, 0.0, -0.038214631520746334),\n                        (4, 0.5000000000000002, 1.0, -0.3558868789064227),\n                        (10, 0.5, 0.0, 0.6981972371211085)],\n                       [(11, 12), (4, 10)],\n                       [(4, 0.4901053274689138, 0.0, -0.3558633466231107),\n                        (12, 0.5068340379253959, 0.0, -0.5310585926382718),\n                        (11, 0.5, 0.0, 0.3744064878335269),\n                        (10, 0.5, 0.0, 0.6981972371211087)],\n                       [(4, 1), (12, 6)],\n                       [(12, 0.5, 0.0, -0.7736945421001282),\n                        (4, 0.4999999999999998, 1.0, 0.14337950343101497),\n                        (1, 0.30408672398469616, 0.0, -0.5000000000000002),\n                        (6, 0.30408672398469616, 0.0, -0.5000000000000002)],\n                       [(11, 12), (10, 4)],\n                       [(4, 0.8040867239846965, 0.0, -0.8566204965689852),\n                        (12, 0.8040867239846965, 0.0, 0.22630545789987178),\n                        (10, 0.0, 0.4659468671857592, 0.0)],\n                       [(4, 1), (12, 6)],\n                       [(4, 0.75, 0.0, -0.8566204965689852),\n                        (12, 0.75, 1.0, 0.22630545789987166),\n                        (1, 0.5, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5)],\n                       [(10, 4), (11, 12)],\n                       [(4, 0.5, 0.0, 0.14337950343101508),\n                        (12, 0.5, 0.0, -0.7736945421001282),\n                        (10, 0.5, 0.0, 0.4467465095138672),\n                        (11, 0.5, 0.0, 0.6064966101833891)],\n                       [(4, 1), (12, 6)],\n                       [(4, 0.25000000000000006, 0.0, -0.8566204965689852),\n                        (12, 0.25000000000000006, 1.0, 0.22630545789987178)],\n                       [(10, 4), (11, 12)],\n                       [(4, 0.5000000000000001, 1.0, 0.14337950343101508),\n                        (12, 0.5, 1.0, -0.7736945421001282),\n                        (10, 0.5, 0.053253490486132815, 0.4467465095138672),\n                        (11, 0.5, -0.10649661018338907, -0.3935033898166107)],\n                       [(4, 1), (12, 6)],\n                       [(1, 1.0, 0.0, 0.5),\n                        (4, 0.5, -0.6433795034310146, -0.8566204965689852),\n                        (6, 1.0, 0.0, 0.5),\n                        (12, 0.5, 0.2736945421001282, 0.22630545789987178)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],\n           \'qubits\': [(3, 4),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (4, 8),\n                      (5, 3),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8)]},\n (13, 5): {\'method\': \'reachable-support count kernels + native scheduling\',\n           \'moments\': [[(0, 0.5, 0.0, 0.5),\n                        (2, 0.4573172924650211, 0.0, 0.5),\n                        (3, 0.5528591303774795, 0.0, -0.5),\n                        (4, 0.6740447590516165, 0.0, -0.5),\n                        (5, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (6, 0.5, 0.0, 0.5),\n                        (7, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (9, 0.9103273151687654, 0.0, 0.5),\n                        (10, 0.4999999999999998, 0.0, -0.721588825990928),\n                        (11, 0.30408672398469655, 0.0, -0.5745192213521528),\n                        (12, 0.30408672398469655, 0.0, -0.5745192213521528)],\n                       [(1, 4), (2, 3), (8, 9)],\n                       [(1, 0.5, 0.0, -0.5),\n                        (2, 0.7821227227391034, 0.0, 0.5),\n                        (3, 0.7856801693438336, 0.0, 0.5),\n                        (4, 0.6740447590516165, 0.0, 0.5),\n                        (8, 0.4213202319080345, 0.0, -0.5000000000000002),\n                        (9, 0.552848538820718, 0.0, -0.5)],\n                       [(3, 8), (4, 9)],\n                       [(3, 0.5, 0.0, 0.5),\n                        (4, 0.49629001056966376, 0.0, 0.5),\n                        (8, 0.6220609163943018, 0.0, 0.5),\n                        (9, 0.3574787763480473, 0.0, -0.5)],\n                       [(3, 4), (8, 9)],\n                       [(1, 4), (3, 0.5, 0.0, 0.5), (8, 0.994035708364498, 0.0, -0.49999999999999956)],\n                       [(2, 3), (4, 0.3823544744598304, 0.0, 0.5), (1, 0.5, 0.0, -0.5)],\n                       [(2, 0.5, 0.0, -0.5), (3, 0.7461587387480209, 0.0, -0.5), (4, 9)],\n                       [(3, 8), (4, 0.88235447445983, 0.0, -0.5), (9, 0.5, 0.0, -0.5)],\n                       [(3, 0.33354763864257064, 0.0, -0.5), (8, 0.5174180205464781, 0.0, -0.5)],\n                       [(3, 4), (8, 9)],\n                       [(3, 0.5, 0.0, -0.5), (4, 0.5, 0.0, -0.5), (8, 0.5, 0.0, 0.5), (9, 0.5, 0.0, 0.5)],\n                       [(2, 3), (4, 9)],\n                       [(1, 4),\n                        (2, 0.4999999999999998, 0.0, 0.1994730230605577),\n                        (3, 0.4999999999999998, 0.0, -0.5000000000000002),\n                        (9, 0.5, 1.1102230246251565e-16, 0.012961324162984567)],\n                       [(3, 8), (4, 0.6083765495574779, 0.0, 0.5064806620814921), (9, 10)],\n                       [(1, 4),\n                        (3, 0.4999999999999998, 0.0, 0.958911095263919),\n                        (8, 0.713234421537507, 0.0, 0.5670727855927733),\n                        (9, 0.7648228951691718, 0.0, 0.7874474740738509),\n                        (10, 0.5225448598685903, 0.0, 0.8911869797251353)],\n                       [(1, 0.9984061930639809, 4.440892098500626e-16, 0.9667891923899257),\n                        (2, 3),\n                        (4, 0.8568537707944928, 0.0, 0.5064806620814926),\n                        (8, 7)],\n                       [(2, 0.5, 0.0, 0.19947302306055725),\n                        (4, 9),\n                        (7, 0.85928593962011, 0.0, 0.07376677451045754),\n                        (8, 0.5, 0.0, -0.8273276816816426)],\n                       [(3, 8),\n                        (4, 0.5, 0.0, -0.49351933791850744),\n                        (7, 2),\n                        (9, 0.930805297572066, 5.551115123125783e-17, 0.17688950057902364)],\n                       [(3, 0.5, 0.0, -0.8394402186131329),\n                        (4, 9),\n                        (7, 0.7485956033393342, 0.0, 0.5737667745104573),\n                        (8, 0.6714165234362537, 0.0, 0.1012562265873651)],\n                       [(2, 3),\n                        (4, 0.5908115752601095, 0.0, -0.4018937021736626),\n                        (8, 7),\n                        (9, 0.5, 0.0, -0.6613984431672932)],\n                       [(1, 4),\n                        (2, 0.0528060550044127, 0.0, -0.6269266331524728),\n                        (3, 0.5, 0.0, -0.8394402186131329),\n                        (7, 0.9961572718865779, 0.0, -0.9262332254895478),\n                        (8, 0.3935294720438233, 0.0, -0.17779088902625606),\n                        (9, 10)],\n                       [(1, 4),\n                        (3, 8),\n                        (7, 2),\n                        (9, 0.20245391911064137, 0.0, 0.3386015568327067),\n                        (10, 0.7201732888048016, 0.0, -0.7118510178536865)],\n                       [(1, 0.0015938069360184048, 0.0, 0.9667891923897574),\n                        (2, 0.24963665794604517, 0.0, -0.10956404824269761),\n                        (4, 9),\n                        (7, 0.5655532211464794, 8.881784197001252e-16, 0.5737667745104462),\n                        (8, 0.31043695253214243, 0.0, -0.7592696386260571)],\n                       [(2, 3),\n                        (4, 0.4054888125911004, 0.0, -0.7837797856754138),\n                        (8, 7),\n                        (9, 0.6373873139241066, 0.0, -0.16139844316729324)],\n                       [(2, 0.32701156232958084, 0.0, 0.49821631046920034),\n                        (7, 0.4999999999999998, 0.0, -0.42623322548955334),\n                        (8, 0.6193705712509703, 0.0, 0.36776467227334736),\n                        (9, 10)],\n                       [(3, 8),\n                        (7, 2),\n                        (9, 0.5, 0.0, -0.16139844316729324),\n                        (10, 0.5503418120493246, 0.0, -0.1702512394470519)],\n                       [(2, 0.5, 1.1102230246251565e-16, -0.2544191580519233),\n                        (4, 9),\n                        (7, 0.5960614607313279, 0.0, 0.8831564130503451),\n                        (8, 0.6256105257587086, 0.0, -0.11550569573628167)],\n                       [(3, 8),\n                        (4, 0.5000000000000003, 0.0, -0.6401167178477545),\n                        (7, 2),\n                        (9, 0.572447838334538, 0.0, -0.7459122467984045)],\n                       [(1, 4),\n                        (2, 0.5, 0.0, 0.245580841948077),\n                        (3, 0.5, 0.0, -0.5183687893056523),\n                        (7, 0.14949404734505523, 0.0, 0.2730671118828738),\n                        (8, 0.9764747842833552, 0.0, -0.7132899378156738),\n                        (9, 10)],\n                       [(1, 0.5, 0.0, -0.06318128417940283),\n                        (2, 3),\n                        (4, 0.7849738353917708, 0.0, -0.1401167178477536),\n                        (8, 7),\n                        (9, 0.5, 0.0, 0.07667066367212794),\n                        (10, 0.4999999999999977, 1.1102230246251565e-16, 0.3151950234033264)],\n                       [(3, 0.5, 0.0, -0.5183687893056512),\n                        (4, 9),\n                        (7, 2),\n                        (8, 0.5, 0.0, -0.9562682209975482)],\n                       [(3, 8),\n                        (4, 0.5517093529747915, 0.0, 0.8598832821522464),\n                        (7, 0.4247761378557593, 1.1102230246251565e-16, -0.022381851578534118),\n                        (9, 0.07810728535625011, 0.0, -0.42332933632787295)],\n                       [(1, 4),\n                        (3, 0.5, 0.0, 0.41444235667571094),\n                        (8, 0.1473187127029063, 4.440892098500626e-16, -0.9562682209975484),\n                        (9, 10)],\n                       [(1, 0.49999999999999967, 0.05994095313865788, 0.936818715820597),\n                        (2, 3),\n                        (8, 7),\n                        (9, 0.5510400458124618, 0.0, -0.7443666929558646),\n                        (10, 0.2219727068374309, 0.0, -0.2540214619058512)],\n                       [(2, 0.4202007516434625, 1.1102230246251565e-16, -0.047856799643425596),\n                        (4, 9),\n                        (7, 0.45008695289946243, 0.0, 0.6634633138266379),\n                        (8, 0.9433458052208397, 4.440892098500626e-16, 0.5437317790024525),\n                        (10, 5)],\n                       [(3, 8),\n                        (5, 0.85928593962011, 0.0, -0.9262332254895425),\n                        (7, 2),\n                        (9, 0.4999999999999993, 0.0, -0.323365634103989),\n                        (10, 0.49999999999999994, 0.0, -0.7540214619058516)],\n                       [(2, 0.5410889047360808, 1.1102230246251565e-16, -0.14305268480415156),\n                        (4, 9),\n                        (7, 0.541088904736081, 0.0, -0.7896451505492768),\n                        (3, 0.5, 0.0, -0.5855576433242891)],\n                       [(2, 0),\n                        (4, 0.5, 0.0, -0.4438840258279413),\n                        (7, 6),\n                        (9, 0.29783770120218545, 0.0, -0.28648077580647424)],\n                       [(0, 0.3040867239846962, 0.0, -0.5000000000000002),\n                        (2, 0.4999999999999998, 0.0, 0.3569473151958489),\n                        (4, 9),\n                        (6, 0.3040867239846962, 0.0, 0.4999999999999998),\n                        (7, 0.5, 0.0, 0.7103548494507239)],\n                       [(3, 2), (4, 0.5, 4.440892098500626e-16, 0.5561159741720587), (8, 7), (9, 10)],\n                       [(2, 0.8040867239846965, 0.0, 0.35694731519584844),\n                        (5, 4),\n                        (7, 0.8040867239846965, 0.0, 0.7103548494507239),\n                        (10, 0.6714165234362537, 0.0, -0.6826053701748582),\n                        (9, 0.5, 0.0, 0.6626303323030966)],\n                       [(2, 0), (5, 0.7485956033393342, 0.0, -0.42623322548954246), (7, 6)],\n                       [(0, 0.5, 0.0, 0.5),\n                        (2, 0.75, 0.0, 0.35694731519584844),\n                        (4, 9),\n                        (6, 0.5, 0.0, 0.5),\n                        (7, 0.75, 0.0, 0.7103548494507239),\n                        (10, 5)],\n                       [(3, 2),\n                        (4, 0.0528060550044127, 4.440892098500626e-16, 0.7297163179590282),\n                        (5, 0.9961572718865779, 0.0, -0.9262332254895478),\n                        (8, 7),\n                        (9, 0.5, 4.440892098500626e-16, 0.6626303323030969),\n                        (10, 0.3935294720438233, 0.0, -0.4035582545612375)],\n                       [(2, 0.5, -2.220446049250313e-16, -0.6430526848041513),\n                        (3, 0.5, 0.0, 0.8938979043076702),\n                        (5, 4),\n                        (7, 0.5, 0.0, -0.2896451505492761),\n                        (8, 0.5, 0.0, -0.9768126733655889),\n                        (9, 10)],\n                       [(2, 0),\n                        (4, 0.24963665794604523, 0.0, -0.7529210971311968),\n                        (5, 0.5655532211464794, 0.0, 0.5737667745104469),\n                        (7, 6),\n                        (10, 0.3104369525321425, 0.0, -0.8220795049614371)],\n                       [(2, 0.25000000000000006, 0.0, 0.35694731519584844),\n                        (4, 9),\n                        (7, 0.25000000000000006, 0.0, 0.7103548494507237),\n                        (10, 5)],\n                       [(3, 2),\n                        (4, 0.3270115623295809, 4.440892098500626e-16, 0.8548592615807016),\n                        (5, 0.4999999999999998, 0.0, 0.5737667745104462),\n                        (8, 7),\n                        (10, 0.6193705712509703, 0.0, 0.050886184139158075)],\n                       [(2, 0.5, -2.220446049250313e-16, -0.6430526848041513),\n                        (3, 0.5, -0.39389790430767047, 0.8938979043076702),\n                        (5, 4),\n                        (7, 0.5, 0.0, -0.2896451505492761),\n                        (8, 0.5, 0.47681267336558864, 0.023187326634411143),\n                        (9, 10)],\n                       [(2, 0),\n                        (4, 0.5, 0.0, 0.1022237930595779),\n                        (5, 0.5960614607313279, -2.220446049250313e-16, -0.11684358694965402),\n                        (7, 6),\n                        (10, 0.6256105257587086, 0.0, -0.46584344785121257)],\n                       [(0, 1.0, 0.0, 0.5),\n                        (2, 0.5, -0.8569473151958489, -0.6430526848041513),\n                        (5, 4),\n                        (6, 1.0, 0.0, 0.5),\n                        (7, 0.4999999999999998, 0.7896451505492763, 0.7103548494507237),\n                        (9, 10)],\n                       [(4, 0.5, 4.440892098500626e-16, 0.6022237930595775),\n                        (5, 0.14949404734505523, 0.0, -0.7269328881171262),\n                        (9, 0.5, 0.0, -0.658441097004385),\n                        (10, 0.9764747842833552, 0.0, -0.8680592057718204)],\n                       [(4, 9), (10, 5)],\n                       [(5, 4), (9, 0.5, 0.0, -0.658441097004385), (10, 0.5, 0.0, -0.6250809225899461)],\n                       [(5, 0.4247761378557593, 0.0, -0.022381851578533896), (9, 10)],\n                       [(9, 0.5, 0.0, -0.5912522429857483),\n                        (10, 0.1473187127029063, 0.0, 0.3749190774100535)],\n                       [(4, 9), (10, 5)],\n                       [(4, 0.4202007516434625, 0.0, 0.3087861514680756),\n                        (5, 0.45008695289946266, 0.0, -0.3365366861733614),\n                        (10, 0.9433458052208397, 0.0, 0.8749190774100513)],\n                       [(5, 4), (9, 10)],\n                       [(4, 0.5, 4.440892098500626e-16, 0.7135902663073499),\n                        (5, 0.5, 0.0, -0.2896451505492772),\n                        (9, 0.20567003520391758, 0.0, -0.09125224298574874),\n                        (10, 0.5, 0.0, -0.33075095779386565)],\n                       [(9, 4), (10, 5)],\n                       [(4, 0.500000000000002, 1.1102230246251565e-16, -0.19774980410048437),\n                        (5, 0.5000000000000022, 0.0, 0.6216949198585577),\n                        (9, 0.4999999999999824, 0.0, 0.4087478133321927),\n                        (10, 0.49999999999998246, 0.0, 0.6692490985240744)],\n                       [(9, 12), (10, 11)],\n                       [(9, 0.5000000000000102, 5.551115123125783e-17, 0.4087478133321926),\n                        (10, 0.5000000000000102, 0.0, 0.6692490985240755),\n                        (11, 0.30408672398469627, 0.0, -0.5745192213521484),\n                        (12, 0.30408672398469627, 0.0, -0.5745192213521484)],\n                       [(9, 4), (10, 5)],\n                       [(4, 0.4999999999999995, 0.0, -0.1977497715853027),\n                        (5, 0.49999999999999956, 0.0, 0.6216948873433754),\n                        (9, 0.75, 0.0, -0.591252154152629),\n                        (10, 0.75, 0.0, 0.669249131039255)],\n                       [(9, 12), (10, 11)],\n                       [(9, 0.500526889421938, 4.440892098500626e-16, 0.9092747819745353),\n                        (10, 0.500526889421938, 0.0, 0.1697760671664179),\n                        (11, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155),\n                        (12, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155)],\n                       [(9, 4), (10, 5)],\n                       [(4, 0.5000000000000004, 0.0, 0.30225022841469773),\n                        (5, 0.5000000000000003, -2.220446049250313e-16, 0.12169488734337541),\n                        (9, 0.5, 4.440892098500626e-16, 0.6592752180479682),\n                        (10, 0.5, 0.0, 0.9197765032398513)],\n                       [(9, 12), (10, 11)],\n                       [(9, 4),\n                        (10, 5),\n                        (11, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (12, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(4, 0.9992548329781763, 0.4715352840133615, -0.1977497715853107),\n                        (5, 0.0007451670218237004, -0.34790937491533347, -0.37830511265661615),\n                        (9, 0.999999967484822, -0.385489750950903, 0.6592752407134563),\n                        (10, 0.999999967484822, 0.35400876387469316, -0.08022337410340041)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],\n           \'qubits\': [(3, 4),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 3),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8),\n                      (6, 6)]},\n (13, 6): {\'method\': \'reachable-support count kernels + native scheduling\',\n           \'moments\': [[(4, 0.9022926806838556, 0.0, 0.5),\n                        (8, 0.570463430758553, 0.0, 0.5),\n                        (3, 0.7780023400091518, 0.0, -0.5),\n                        (9, 0.13692685047306075, 0.0, -0.5000000000000002),\n                        (1, 0.3453030033050204, 0.0, 0.5),\n                        (10, 0.4768011512951635, 0.0, -0.17578877400173032),\n                        (7, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (5, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (6, 0.5, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (12, 0.30408672398469655, 0.0, -0.5745192213521528),\n                        (11, 0.30408672398469655, 0.0, -0.5745192213521528)],\n                       [(1, 4), (8, 9), (2, 3)],\n                       [(9, 0.6465533278056601, 0.0, -0.5),\n                        (4, 0.9532124066841656, 0.0, -0.5),\n                        (8, 0.6059529542072981, 0.0, -0.5),\n                        (3, 0.712446991932818, 0.0, -0.5),\n                        (1, 0.16107414318911473, 0.0, -0.5000000000000002),\n                        (2, 0.5000000000000011, 0.0, -0.5)],\n                       [(3, 8), (4, 9)],\n                       [(4, 0.5, 0.0, 0.5),\n                        (3, 0.9935804708460765, 0.0, 0.4999999999999998),\n                        (8, 0.8687138745151404, 0.0, 0.5),\n                        (9, 0.7660870115906562, 0.0, 0.5)],\n                       [(3, 4), (8, 9)],\n                       [(4, 0.5, 0.0, 0.5),\n                        (2, 3),\n                        (9, 0.7384766399754725, 0.0, 0.5),\n                        (8, 0.25337862340997086, 0.0, 0.5)],\n                       [(1, 4),\n                        (3, 0.16288456946756846, 0.0, 0.5),\n                        (2, 0.4999999999999994, 0.0, -0.5000000000000002)],\n                       [(4, 0.4959990466414421, 0.0, 0.5), (3, 8), (1, 0.4999999999999998, 0.0, 0.5)],\n                       [(4, 9),\n                        (3, 0.19907753991803667, 0.0, 0.5),\n                        (8, 0.3883789600263144, 0.0, 0.4999999999999998)],\n                       [(4, 0.9460040477347941, 0.0, 0.5), (9, 0.5905392139802306, 0.0, -0.5)],\n                       [(3, 4), (8, 9)],\n                       [(3, 0.8715646580878726, 0.0, 0.4999999999999998),\n                        (4, 0.4930323036045407, 0.0, 0.5),\n                        (8, 0.7670902005782116, 0.0, -0.5),\n                        (9, 0.9951094661141681, 0.0, 0.5)],\n                       [(2, 3), (1, 4)],\n                       [(3, 0.8911859054606355, 0.0, -0.5000000000000004),\n                        (4, 0.9447958423234382, 0.0, 0.4999999999999998)],\n                       [(3, 8), (4, 9)],\n                       [(3, 0.9101692727320629, 0.0, -0.5000000000000004),\n                        (4, 0.34823604298102895, 0.0, 0.4999999999999998),\n                        (9, 0.10512449133276025, 0.0, 0.5),\n                        (8, 0.2379821597075702, 0.0, -0.5)],\n                       [(3, 4), (8, 9)],\n                       [(4, 0.5642722205173485, 0.0, 0.5),\n                        (9, 0.4246322820009205, 0.0, -0.5000000000000002),\n                        (3, 0.4515701201931224, 0.0, 0.4999999999999998),\n                        (8, 0.4121186179515961, 0.0, -0.5)],\n                       [(1, 4), (2, 3)],\n                       [(4, 0.8866618222600691, 0.0, -0.5),\n                        (1, 0.5967193824645834, 0.0, -0.3859900904595026),\n                        (3, 0.03415050655983354, 0.0, -0.5000000000000007),\n                        (2, 0.4999999999999998, 0.0, -0.1994730230605577)],\n                       [(4, 9), (3, 8)],\n                       [(9, 0.4964858790790081, 0.0, -0.15243460475233395),\n                        (4, 0.9213692979572716, 0.0, -0.931013012486372),\n                        (3, 0.5410555613213522, 0.0, -0.0016595984875622527),\n                        (8, 0.7288179584097698, 3.3306690738754696e-16, -0.43009055576430666)],\n                       [(9, 10), (1, 4), (2, 3), (8, 7)],\n                       [(9, 0.16399934185718348, 0.0, -0.17181505408613762),\n                        (1, 0.7174704187311103, 0.0, 0.11400990954049761),\n                        (10, 0.6279799493860765, 0.0, -0.770287955286147),\n                        (8, 0.5, 0.0, 0.05279620579869082),\n                        (2, 0.5, 0.0, 0.8005269769394427),\n                        (7, 0.85928593962011, 0.0, -0.9262332254895425)],\n                       [(4, 9), (3, 8), (7, 2)],\n                       [(4, 0.45584673157989886, 0.0, 0.6347668797230908),\n                        (9, 0.2771547609074463, 0.0, 0.5143292658281073),\n                        (3, 0.5, 0.0, -0.7162042008638039),\n                        (7, 0.7485956033393342, 0.0, 0.5737667745104573),\n                        (8, 0.6714165234362537, 0.0, 0.12421229752968327)],\n                       [(9, 10), (1, 4), (2, 3), (8, 7)],\n                       [(10, 0.019663643370415814, 4.440892098500626e-16, 0.5185052910680361),\n                        (1, 0.1940813450363048, 0.0, 0.6140099095404976),\n                        (4, 0.4233573560778924, 0.0, -0.865233120276909),\n                        (9, 0.6057423835093778, 0.0, -0.49627419296985753),\n                        (7, 0.9961572718865779, 0.0, 0.07376677451045216),\n                        (8, 0.3935294720438234, 0.0, 0.4032594131433047),\n                        (3, 0.5, 1.1102230246251565e-16, 0.283795799136196),\n                        (2, 0.05280605500441271, 0.0, 0.6269266331524725)],\n                       [(9, 10), (1, 4), (3, 8), (7, 2)],\n                       [(4, 0.7065221727328495, 8.881784197001252e-16, -0.8372052850885585),\n                        (9, 0.776375637814382, 0.0, -0.489250253584818),\n                        (10, 0.5680691667523844, 0.0, -0.6374006217137276),\n                        (1, 0.574001540707395, 0.0, -0.8859900904595024),\n                        (2, 0.24963665794604498, 0.0, 0.10956404824269711),\n                        (8, 0.3104369525321424, 0.0, -0.015261837256895117),\n                        (7, 0.5655532211464794, 0.0, 0.5737667745104469)],\n                       [(4, 9), (8, 7), (2, 3)],\n                       [(9, 0.7960253457838964, 0.0, -0.45908725462083666),\n                        (4, 0.7550526236278825, 0.0, 0.6709450604318006),\n                        (8, 0.6193705712509703, 0.0, 0.8577038518436999),\n                        (7, 0.4999999999999998, 0.0, 0.5737667745104462),\n                        (2, 0.32701156232958084, 0.0, 0.501783689530799)],\n                       [(1, 4), (9, 10), (3, 8), (7, 2)],\n                       [(4, 0.093170508709036, 0.0, 0.8016281430864909),\n                        (9, 0.5920891677547482, 0.0, 0.04091274537916334),\n                        (10, 0.5767646373955954, 0.0, -0.7141291884531249),\n                        (8, 0.6256105257587086, 0.0, 0.3409742198533303),\n                        (7, 0.5960614607313279, 0.0, 0.8831564130503458),\n                        (2, 0.5, 0.0, -0.7455808419480776)],\n                       [(4, 9), (7, 2), (3, 8)],\n                       [(4, 0.8490154235516789, 0.0, 0.010518848030736772),\n                        (9, 0.8905211577257011, 8.881784197001252e-16, 0.5409127453791627),\n                        (3, 0.5, 0.0, -0.037275630171286434),\n                        (8, 0.9764747842833552, 0.0, -0.0612415380672775),\n                        (2, 0.5, 0.0, 0.7544191580519224),\n                        (7, 0.14949404734505523, 0.0, -0.7269328881171262)],\n                       [(9, 10), (1, 4), (2, 3), (8, 7)],\n                       [(9, 0.623464211234003, 0.0, 0.04091274537916312),\n                        (4, 0.5525232416584167, 0.0, 0.948924160551631),\n                        (10, 0.22565786477821326, 0.0, 0.9040830152802759),\n                        (3, 0.5, 0.0, 0.9627243698287131),\n                        (8, 0.5, 0.0, 0.1817367451145966),\n                        (7, 2)],\n                       [(4, 9), (3, 8), (7, 0.42477613785575913, 0.0, 0.9776181484214663)],\n                       [(4, 0.22560276650113933, -2.220446049250313e-16, 0.11869638514046077),\n                        (9, 0.19755687312566197, 0.0, 0.5409127453791636),\n                        (3, 0.5, 0.0, -0.970086776152649),\n                        (8, 0.14731871270290653, 0.0, -0.8182632548854036)],\n                       [(1, 4), (9, 10), (2, 3), (8, 7)],\n                       [(4, 0.2731956803087589, 0.0, -0.5664247000245206),\n                        (9, 0.5, 0.0, 0.5409127453791636),\n                        (1, 0.5, 0.0, -0.5603573064158267),\n                        (10, 0.08965970809102465, 0.0, 0.42015605383701876),\n                        (2, 0.4202007516434625, 0.0, -0.9521432003565753),\n                        (7, 0.4500869528994624, 0.0, 0.6634633138266381),\n                        (8, 0.9433458052208397, 1.1102230246251565e-16, -0.31826325488540563)],\n                       [(4, 9), (7, 2), (3, 8)],\n                       [(9, 0.6586668818397199, 0.0, 0.9463826119636918),\n                        (4, 0.5, 0.0, 0.06902567085228917),\n                        (2, 0.5410889047360808, 0.0, -0.8569473151958495),\n                        (7, 0.5410889047360808, -2.220446049250313e-16, 0.21035484945072325),\n                        (3, 0.5, 5.551115123125783e-17, 0.0299132238473509)],\n                       [(9, 10), (1, 4), (2, 0), (7, 6)],\n                       [(4, 0.4999999999999998, 0.0, -0.41882447770509135),\n                        (10, 0.35737627229158253, 0.0, 0.47611746869056887),\n                        (9, 0.6178431557297261, 0.0, 0.9423607199204624),\n                        (2, 0.5, 0.0, 0.6430526848041502),\n                        (7, 0.5, 0.0, -0.28964515054927675),\n                        (0, 0.3040867239846962, 0.0, 0.4999999999999998),\n                        (6, 0.3040867239846962, 0.0, 0.4999999999999998),\n                        (1, 0.5, -0.6512655680873509, -0.5603573064158265)],\n                       [(4, 9), (10, 5), (3, 2), (8, 7)],\n                       [(4, 0.5, 0.0, -0.4188244777050909),\n                        (10, 0.5, 0.0, -0.71839631538329),\n                        (5, 0.85928593962011, 0.0, -0.9262332254895425),\n                        (7, 0.8040867239846965, 4.440892098500626e-16, 0.710354849450723),\n                        (2, 0.8040867239846965, 0.0, -0.35694731519585)],\n                       [(5, 4), (9, 10), (2, 0), (7, 6)],\n                       [(5, 0.7485956033393342, 0.0, 0.5737667745104573),\n                        (10, 0.6714165234362537, 5.551115123125783e-17, 0.35301977634770204),\n                        (9, 0.5000000000000001, 0.0, -0.35599059395658905),\n                        (2, 0.75, 0.0, -0.35694731519584977),\n                        (7, 0.7499999999999999, 0.0, 0.710354849450723),\n                        (6, 0.5, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5)],\n                       [(10, 5), (4, 9), (3, 2), (8, 7)],\n                       [(10, 0.3935294720438234, 0.0, 0.6320668919613246),\n                        (4, 0.052806055004412704, 0.0, 0.4075751785079378),\n                        (5, 0.9961572718865779, 0.0, 0.07376677451045216),\n                        (9, 0.5, 0.0, 0.644009406043411),\n                        (7, 0.5, 0.0, 0.710354849450723),\n                        (2, 0.5, 0.0, -0.35694731519585),\n                        (8, 0.5, 0.0, 0.2022811974826353),\n                        (3, 0.5, 4.440892098500626e-16, 0.550457676215391)],\n                       [(9, 10), (5, 4), (7, 6), (2, 0)],\n                       [(10, 0.3104369525321425, 0.0, -0.7864543584388737),\n                        (5, 0.5655532211464794, 0.0, 0.5737667745104469),\n                        (4, 0.24963665794604517, 0.0, -0.10978740640183693),\n                        (2, 0.25, 0.0, 0.6430526848041502),\n                        (7, 0.25, 0.0, -0.28964515054927675)],\n                       [(10, 5), (4, 9), (8, 7), (3, 2)],\n                       [(5, 0.4999999999999998, 0.0, 0.5737667745104462),\n                        (10, 0.6193705712509703, -2.220446049250313e-16, 0.0865113306617209),\n                        (4, 0.3270115623295809, 0.0, -0.7175677651137353),\n                        (2, 0.5, 0.0, 0.6430526848041502),\n                        (7, 0.5, 0.0, -0.28964515054927675),\n                        (8, 0.5, -0.7022811974826342, 0.20228119748263484),\n                        (3, 0.5, 0.949542323784609, 0.550457676215391)],\n                       [(9, 10), (5, 4), (7, 6), (2, 0)],\n                       [(10, 0.6256105257587086, 8.881784197001252e-16, 0.5697816986713502),\n                        (5, 0.5960614607313279, 0.0, 0.8831564130503458),\n                        (4, 0.5, 0.0, 0.03506770340738896),\n                        (7, 0.4999999999999998, 0.7896451505492773, 0.7103548494507228),\n                        (2, 0.4999999999999998, -0.14305268480415023, -0.35694731519584977),\n                        (0, 1.0, 0.0, 0.5),\n                        (6, 1.0, 0.0, 0.5)],\n                       [(9, 10), (5, 4)],\n                       [(10, 0.9764747842833552, 0.0, -0.8324340592492583),\n                        (9, 0.5, 0.0, 0.9650808353508942),\n                        (4, 0.5, 0.0, -0.46493229659261104),\n                        (5, 0.14949404734505523, 0.0, -0.7269328881171262)],\n                       [(10, 5), (4, 9)],\n                       [(9, 0.5, 0.0, -0.03491916464910716), (10, 0.5, 0.0, -0.589455776067384), (5, 4)],\n                       [(9, 10), (5, 0.42477613785575913, 0.0, 0.9776181484214663)],\n                       [(10, 0.1473187127029063, 0.0, -0.589455776067384),\n                        (9, 0.5, 0.0, -0.10210801866774366)],\n                       [(10, 5), (4, 9)],\n                       [(4, 0.4202007516434625, 0.0, -0.1714946550011085),\n                        (10, 0.9433458052208397, 0.0, -0.0894557760673862),\n                        (5, 0.4500869528994624, 0.0, 0.6634633138266381)],\n                       [(5, 4), (9, 10)],\n                       [(10, 0.5, 0.0, -0.29512581127130266),\n                        (9, 0.20567003520391755, 0.0, 0.3978919813322579),\n                        (4, 0.5, 0.0, -0.5762987698403834),\n                        (5, 0.5, 0.0, -0.28964515054927675)],\n                       [(9, 4), (10, 5)],\n                       [(9, 0.4999999999999824, 0.0, 0.8978919250143165),\n                        (10, 0.4999999999999824, 0.0, 0.7048742450466376),\n                        (4, 0.5000000000000019, 0.0, -0.6649586994325487),\n                        (5, 0.5000000000000022, 0.0, 0.6216949198585571)],\n                       [(9, 12), (10, 11)],\n                       [(9, 0.5000000000000102, 0.0, 0.8978919250143165),\n                        (10, 0.5000000000000105, 0.0, 0.7048742450466376),\n                        (12, 0.3040867239846962, 0.0, 0.4254807786478514),\n                        (11, 0.30408672398469627, 0.0, -0.5745192213521484)],\n                       [(9, 4), (10, 5)],\n                       [(9, 0.75, 0.0, 0.8978918924991377),\n                        (10, 0.75, 0.0, 0.704874277561818),\n                        (5, 0.49999999999999956, 0.0, 0.6216948873433756),\n                        (4, 0.49999999999999967, 0.0, 0.33504126805226964)],\n                       [(10, 11), (9, 12)],\n                       [(9, 0.500526889421938, 0.0, -0.6026350436280277),\n                        (10, 0.5005268894219383, 5.551115123125783e-17, 0.20540121368898195),\n                        (12, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155),\n                        (11, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155)],\n                       [(9, 4), (10, 5)],\n                       [(5, 0.5000000000000003, -2.220446049250313e-16, 0.12169488734337541),\n                        (4, 0.5000000000000004, 0.0, 0.8350412680522685),\n                        (9, 0.5, 0.0, 0.6473645202985419),\n                        (10, 0.5, -9.769962616701378e-15, 0.9554016497624245)],\n                       [(9, 12), (10, 11)],\n                       [(9, 4),\n                        (10, 5),\n                        (11, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (12, 0.5, 0.3483047662953691, -0.5745192538673274)],\n                       [(9, 3.251517832758566e-08, 0.6264209921295008, 0.6473644976330524),\n                        (4, 0.0007451670218237008, -0.061255755624225916, -0.664958731947723),\n                        (10, 0.999999967484822, 0.31838381733465093, -0.044598327572097585),\n                        (5, 0.0007451670218237004, -0.34790937491533347, -0.37830511265661615)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],\n           \'qubits\': [(3, 4),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 3),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8),\n                      (6, 6)]},\n (14, 3): {\'method\': \'reachable-support count kernels + native scheduling\',\n           \'moments\': [[(0, 0.500654098647404, 0.0, -0.5),\n                        (5, 0.5460924801300725, 0.0, -0.5),\n                        (1, 0.2952676223543462, 0.0, 0.5),\n                        (6, 0.30754615199876284, 0.0, 0.5),\n                        (10, 0.4118208540116386, 0.0, 0.5),\n                        (11, 0.7990860045271903, 0.0, -0.5),\n                        (2, 0.41218411310679315, 4.973524073736968e-05, -0.5000093746463457),\n                        (3, 0.4535018303190155, 3.2138086280997236e-05, -0.4999866117568501),\n                        (4, 0.6538196916638768, 0.0012975466470504515, 0.4992166768114052),\n                        (7, 0.41218411310679315, 4.973524073736968e-05, -0.5000093746463457),\n                        (8, 0.4535018303190155, 3.2138086280997236e-05, -0.4999866117568501),\n                        (9, 0.6538196916638768, 0.0012975466470504515, 0.4992166768114052),\n                        (12, 0.4135610417387456, 0.0, -0.5),\n                        (13, 0.5, 0.0, 0.5)],\n                       [(0, 5), (1, 6), (10, 11)],\n                       [(0, 0.49546733790847824, 0.0, 0.5),\n                        (1, 0.8558465753189465, 0.0, -0.5),\n                        (5, 0.004485221166433276, 0.0, 0.5),\n                        (10, 0.3479361808073453, 0.0, -0.5),\n                        (6, 0.08992166537861615, 0.0, 0.5),\n                        (11, 0.6026574392905439, 0.0, 0.5)],\n                       [(0, 1), (5, 10), (6, 11)],\n                       [(0, 0.5, 0.0, -0.5),\n                        (1, 0.9531281587107934, 0.0, 0.5),\n                        (5, 1.0, 0.0, 0.5),\n                        (6, 0.29370516922839646, 0.0, -0.5),\n                        (10, 0.9226247843065278, 0.0, -0.5),\n                        (11, 0.7928065501808954, 0.0, -0.5)],\n                       [(0, 1), (5, 6), (10, 11)],\n                       [(0, 0.9999999999999999, 0.0, -0.5000000000000002),\n                        (5, 0.5, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5),\n                        (6, 0.949586832724987, 0.0, 0.5),\n                        (10, 0.2809293943810169, 0.0, 0.5),\n                        (11, 0.30083991650077285, 0.0, 0.5)],\n                       [(0, 5), (1, 6), (10, 11)],\n                       [(0, 0.4203607417436471, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5),\n                        (5, 0.6510554317367806, 0.0, -0.5),\n                        (10, 0.5658284044979385, 0.0, 0.5),\n                        (6, 0.9897711558721369, 0.0, -0.5000000000000002),\n                        (11, 0.9632739550369707, 0.0, -0.5000000000000002)],\n                       [(0, 1), (5, 10), (6, 11)],\n                       [(0, 0.32994625881989303, 0.0, 0.5),\n                        (5, 0.3416961993207158, 0.0, -0.5),\n                        (1, 0.33533707352777786, 0.0, -0.5),\n                        (6, 0.8906281992022789, 0.0, 0.5),\n                        (11, 0.9449650288787829, 0.0, -0.5)],\n                       [(0, 5), (1, 6), (10, 11)],\n                       [(0, 0.49999999999999994, 0.0, 0.5),\n                        (1, 0.7633065265106983, 0.0, -0.5),\n                        (5, 0.5, 0.0, -0.5),\n                        (10, 1.0, 0.0, 0.5),\n                        (6, 0.31746242475777553, 0.0, -0.5),\n                        (11, 0.784375585596912, 0.0, 0.5)],\n                       [(0, 1), (5, 10), (6, 11)],\n                       [(1, 0.23898507145773407, 0.0, 0.5),\n                        (5, 1.0, 0.0, 0.5),\n                        (6, 0.46129367301539215, 0.0, 0.5),\n                        (10, 1.0, 0.0, 0.5),\n                        (11, 0.6256142167541747, 0.0, 0.5)],\n                       [(0, 1), (5, 6), (10, 11)],\n                       [(1, 0.8967343573662824, -1.2208078893793939e-05, 0.5000258701218818),\n                        (0, 1.0, 0.0, 0.49996644615477503),\n                        (6, 0.4870092143809376, 4.609912261643867e-05, -0.5000082531741972),\n                        (5, 1.7235546699866372e-17, -6.710769044948606e-05, 0.5004836298458384),\n                        (11, 0.25219965630460334, 0.0, -0.5000000000000002),\n                        (10, 1.0, 0.0, 0.5)],\n                       [(1, 2), (6, 7), (11, 12)],\n                       [(1, 0.5125153233564552, 6.816602735071342e-05, -0.5000700309596928),\n                        (2, 0.6117890444840384, 0.00023906418654384698, -0.5000335859224758),\n                        (6, 0.5125153233564552, 6.816602735071342e-05, -0.5000700309596928),\n                        (7, 0.6117890444840384, 0.00023906418654384698, -0.5000335859224758),\n                        (11, 0.3819668494768557, 0.0, 0.5),\n                        (12, 0.6338602364006151, 0.0, -0.5)],\n                       [(0, 1), (5, 6), (10, 11)],\n                       [(1, 0.7075435896451812, -0.00015441380199909815, 0.5000167573434053),\n                        (0, 1.0, 0.0, -0.4999840678705958),\n                        (6, 0.7075435896451812, -0.00015441380199909815, 0.5000167573434053),\n                        (5, 1.0, 0.0, -0.4999840678705958),\n                        (11, 0.7376490383100945, 0.0, 0.5),\n                        (10, 1.0, 0.0, 0.5)],\n                       [(1, 2), (6, 7), (11, 12)],\n                       [(1, 0.13376593564282768, -3.928146360454576e-05, 0.5001322711129306),\n                        (2, 0.16507290691754561, -4.4512723523348896e-05, 0.49992633361577),\n                        (6, 0.13376593564282768, -3.928146360454576e-05, 0.5001322711129306),\n                        (7, 0.16507290691754561, -4.4512723523348896e-05, 0.49992633361577),\n                        (11, 0.16018463696253962, 0.0, 0.5),\n                        (12, 0.18023932987586394, 0.0, -0.5000000000000002)],\n                       [(0, 1), (5, 6), (10, 11)],\n                       [(1, 0.4425626628501562, 0.00019430062546132998, 0.49988424829215317),\n                        (0, 0.5, -2.699936250127877e-07, 0.50000374169819),\n                        (6, 0.4425626628501562, 0.00019430062546132998, 0.49988424829215317),\n                        (5, 0.5, -2.699936250127877e-07, 0.50000374169819),\n                        (11, 0.4422317453870271, 0.0, 0.5),\n                        (10, 0.5, 0.0, 0.5)],\n                       [(1, 2), (6, 7), (11, 12)],\n                       [(1, 0.28336083660654154, 6.297169447666562e-05, -0.500078436406141),\n                        (2, 0.874698411133806, -5.5613993600545086e-05, -0.4999329159564543),\n                        (6, 0.28336083660654154, 6.297169447666562e-05, -0.500078436406141),\n                        (7, 0.874698411133806, -5.5613993600545086e-05, -0.4999329159564543),\n                        (11, 0.6666666666666669, 0.0, 0.5),\n                        (12, 0.8040867239846964, 0.0, -0.5000000000000002)],\n                       [(0, 1), (2, 3), (5, 6), (7, 8), (10, 11), (12, 13)],\n                       [(1, 0.9999999999999999, 1.0626804806790346e-05, 0.5000138488409602),\n                        (2, 0.5195218910218442, -2.492251341545959e-05, 0.49999809046902643),\n                        (3, 0.6338602251592093, -4.516549831157235e-05, -0.5000105128978589),\n                        (6, 0.9999999999999999, 1.0626804806790346e-05, 0.5000138488409602),\n                        (7, 0.5195218910218442, -2.492251341545959e-05, 0.49999809046902643),\n                        (8, 0.6338602251592093, -4.516549831157235e-05, -0.5000105128978589),\n                        (11, 1.0, 0.0, 0.5),\n                        (12, 0.49999999999999983, 0.0, -0.5),\n                        (13, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (0, 0.5000000000000002, -9.01601680149966e-05, -0.4999965282954304),\n                        (5, 0.5000000000000002, -9.01601680149966e-05, -0.4999965282954304),\n                        (10, 0.5, 0.0, -0.5)],\n                       [(1, 2), (6, 7), (11, 12)],\n                       [(2, 0.7465546320890828, -6.819337841124096e-05, 0.5000383284289129),\n                        (1, 1.0, 0.0, 0.4999738833942602),\n                        (7, 0.7465546320890828, -6.819337841124096e-05, 0.5000383284289129),\n                        (6, 1.0, 0.0, 0.4999738833942602),\n                        (12, 0.8040867239846964, 0.0, 0.5),\n                        (11, 1.0, 0.0, 0.5)],\n                       [(2, 3), (7, 8), (12, 13)],\n                       [(2, 0.1601846384780278, -4.503868777239184e-05, 0.5000255168882044),\n                        (3, 0.288907983885649, 8.07695182105661e-05, -0.500167316940584),\n                        (7, 0.1601846384780278, -4.503868777239184e-05, 0.5000255168882044),\n                        (8, 0.288907983885649, 8.07695182105661e-05, -0.500167316940584),\n                        (12, 0.75, 0.0, 0.5),\n                        (13, 0.5, 0.0, 0.5)],\n                       [(1, 2), (6, 7), (11, 12)],\n                       [(2, 0.4422317455429899, -3.658965661768043e-05, 0.5000154601436346),\n                        (1, 0.5, -1.2757522787731901e-05, 0.5000119394225819),\n                        (7, 0.4422317455429899, -3.658965661768043e-05, 0.5000154601436346),\n                        (6, 0.5, -1.2757522787731901e-05, 0.5000119394225819),\n                        (12, 0.5000000000000001, 0.0, -0.5),\n                        (11, 0.5, 0.0, 0.5)],\n                       [(2, 3), (7, 8), (12, 13)],\n                       [(2, 0.7312715381493196, -2.9719479089695966e-05, 0.5000244754386074),\n                        (3, 0.9620769699850723, -0.00013852518429713356, 0.500054648470984),\n                        (7, 0.7312715381493196, -2.9719479089695966e-05, 0.5000244754386074),\n                        (8, 0.9620769699850723, -0.00013852518429713356, 0.500054648470984),\n                        (12, 0.25000000000000006, 0.0, 0.5),\n                        (13, 1.0, 0.0, 0.5)],\n                       [(1, 2), (3, 4), (6, 7), (8, 9), (11, 12)],\n                       [(3, 0.37351126548912544, -7.546439919492443e-05, 0.5000773294001128),\n                        (4, 0.3040867239846963, -0.0005776526930145742, -0.49948577654154436),\n                        (2, 5.255863519135624e-17, -1.712579946921977e-05, 0.5012001664600749),\n                        (8, 0.37351126548912544, -7.546439919492443e-05, 0.5000773294001128),\n                        (9, 0.3040867239846963, -0.0005776526930145742, -0.49948577654154436),\n                        (7, 5.255863519135624e-17, -1.712579946921977e-05, 0.5012001664600749),\n                        (12, 0.5, 0.0, -0.5),\n                        (1, 0.5, -4.5164637105132854e-05, -0.5000008181002027),\n                        (6, 0.5, -4.5164637105132854e-05, -0.5000008181002027),\n                        (11, 0.5, 0.0, 0.5)],\n                       [(3, 4), (8, 9), (12, 13)],\n                       [(3, 0.5000000000000004, -1.4577800550252817e-05, -0.4999981349990823),\n                        (4, 0.20227432911242674, -0.00018048654702673694, 0.4999365707654403),\n                        (8, 0.5000000000000004, -1.4577800550252817e-05, -0.4999981349990823),\n                        (9, 0.20227432911242674, -0.00018048654702673694, 0.4999365707654403),\n                        (12, 0.5000000000000001, 0.0, 0.5)],\n                       [(2, 3), (7, 8)],\n                       [(3, 0.25000000000000006, -2.019130418995374e-05, -0.49998019675610816),\n                        (2, 0.5, -3.842212173532644e-06, 0.5000044520738967),\n                        (8, 0.25000000000000006, -2.019130418995374e-05, -0.49998019675610816),\n                        (7, 0.5, -3.842212173532644e-06, 0.5000044520738967)],\n                       [(3, 4), (8, 9)],\n                       [(3, 0.25, 4.345730322263819e-07, 0.4999996119397021),\n                        (4, 0.5000000000000004, 0.00010607496466885635, -0.5002764318251103),\n                        (8, 0.25, 4.345730322263819e-07, 0.4999996119397021),\n                        (9, 0.5000000000000004, 0.00010607496466885635, -0.5002764318251103)],\n                       [(2, 3), (7, 8)],\n                       [(3, 0.5, -4.246260076889996e-06, -0.4999999534872659),\n                        (8, 0.5, -4.246260076889996e-06, -0.4999999534872659),\n                        (2, 0.5, -2.4189174229860555e-05, 0.5000006098617227),\n                        (7, 0.5, -2.4189174229860555e-05, 0.5000006098617227)],\n                       [(3, 4), (8, 9)],\n                       [(3, 0.5000000000000001, -1.937956516417927e-05, 0.4999958002526569),\n                        (4, 6.271301173889318e-17, 0.00014677754793412934, 0.4999292001208716),\n                        (8, 0.5000000000000001, -1.937956516417927e-05, 0.4999958002526569),\n                        (9, 6.271301173889318e-17, 0.00014677754793412934, 0.4999292001208716)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (3, 6),\n                      (3, 7),\n                      (3, 8),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (4, 8),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7)]},\n (14, 4): {\'method\': \'reachable-support count kernels + native scheduling\',\n           \'moments\': [[(3, 0.6181123560327106, 0.0, 0.5),\n                        (4, 0.27294085673093404, 0.0, -0.5),\n                        (5, 0.20109483922963103, 0.0, 0.5),\n                        (10, 0.5, 0.0, 0.5),\n                        (11, 0.5, 0.0, -0.5),\n                        (9, 0.7728144741714948, 0.0, 0.0805732800089835),\n                        (12, 0.7728144741714948, 0.0, 0.0805732800089835),\n                        (8, 0.4999999999999998, 0.0, 0.5937872305331664),\n                        (13, 0.4999999999999998, 0.0, 0.5937872305331664),\n                        (0, 0.5, 0.0, 0.5),\n                        (2, 0.5, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5),\n                        (7, 0.5, 0.0, 0.5)],\n                       [(3, 4), (6, 5), (10, 11)],\n                       [(4, 0.873692139534469, 0.0, 0.5),\n                        (10, 0.4815180274012989, 0.0, -0.5000000000000002),\n                        (5, 0.9811711804907193, 0.0, 0.5),\n                        (11, 0.018481972598701102, 0.0, 0.5),\n                        (3, 0.8292440087952033, 1.0, 0.5),\n                        (6, 0.5, 1.0, 0.5)],\n                       [(4, 10), (5, 11)],\n                       [(4, 0.5, 0.0, -0.5),\n                        (5, 0.5, 0.0, 0.5),\n                        (10, 0.4328808903532817, 0.0, 0.5),\n                        (11, 0.9328808903532817, 0.0, -0.5000000000000002)],\n                       [(4, 5), (10, 11)],\n                       [(3, 4), (6, 5), (10, 0.5, 1.0, 0.5), (11, 0.5, 1.0, -0.5)],\n                       [(4, 0.719230501883114, 0.0, 0.5),\n                        (5, 0.6034942111147568, 0.0, 0.5),\n                        (3, 0.5, 1.0, -0.5),\n                        (6, 0.5, 1.0, 0.5)],\n                       [(4, 10), (5, 11)],\n                       [(4, 0.7961154155528339, 0.0, 0.5),\n                        (5, 0.3965057888852434, 0.0, -0.5),\n                        (11, 0.14938459272268007, 1.0, 0.5)],\n                       [(4, 5), (10, 11)],\n                       [(4, 0.5, 0.0, 0.5), (5, 0.5, 0.0, 0.5), (11, 0.5, 1.0, 0.5)],\n                       [(3, 4), (6, 5)],\n                       [(4, 10), (5, 0.5, 0.0, 0.5), (3, 0.5, 1.0, 0.5), (6, 0.4999999999999998, 1.0, 0.5)],\n                       [(5, 11),\n                        (10, 0.0, -0.9771581978766974, 0.0),\n                        (4, 0.7830718300698545, 0.0, -0.9324672837165302)],\n                       [(10, 9),\n                        (3, 4),\n                        (11, 0.3506154072773198, 0.0, -0.5),\n                        (5, 0.5423492035105278, 0.0, -0.2865947927371191)],\n                       [(11, 12),\n                        (6, 5),\n                        (9, 0.7728144741714948, 0.0, -0.9194267199910164),\n                        (3, 0.5, 0.0, 0.9231677647409811),\n                        (4, 0.2630328282816797, 2.220446049250313e-16, -0.11317608562247905),\n                        (10, 0.791814249190367, 1.0, 0.40998574196175974)],\n                       [(9, 3),\n                        (4, 10),\n                        (12, 0.7728144741714948, 0.0, -0.9194267199910164),\n                        (6, 0.5, 0.0, 0.9231677647409811),\n                        (5, 0.2630328282816796, 0.0, -0.020064248512857263),\n                        (11, 0.791814249190367, -0.9999999999999996, 0.3871439398384573)],\n                       [(12, 6),\n                        (5, 11),\n                        (3, 0.5, 1.0, -0.07683223525901894),\n                        (4, 0.791370059531975, 0.0, -0.12073828251875263),\n                        (10, 0.6714475580169544, 1.0, -0.13135093177758295)],\n                       [(3, 4),\n                        (10, 9),\n                        (6, 0.5, 1.0, -0.07683223525901894),\n                        (5, 0.791370059531975, 0.0, -0.027626445409131284),\n                        (11, 0.6714475580169545, 1.0, -0.1541927339008855)],\n                       [(6, 5),\n                        (11, 12),\n                        (9, 0.5, 0.0, 0.2456489329819702),\n                        (4, 0.6715105414343046, 0.0, 0.943215422614688),\n                        (10, 0.5751637920695032, 1.0, -0.7955426041953662)],\n                       [(9, 3),\n                        (4, 10),\n                        (12, 0.5, 0.0, 0.2456489329819702),\n                        (5, 0.6715105414343046, 0.0, -0.9636727402756897),\n                        (11, 0.5751637920695032, 1.0, -0.8183844063186687)],\n                       [(12, 6),\n                        (5, 11),\n                        (4, 0.8057807413978859, 0.0, 0.10100634445152817),\n                        (10, 0.32404705667898215, 0.9999999999999998, 0.9841253600390274),\n                        (9, 0.5, 0.0, 0.24564893298197044)],\n                       [(3, 4),\n                        (10, 9),\n                        (5, 0.8057807413978859, 1.1102230246251565e-16, 0.19411818156114952),\n                        (11, 0.3240470566789824, 0.9999999999999999, 0.9612835579157248),\n                        (12, 0.5, 0.0, 0.24564893298197044)],\n                       [(6, 5),\n                        (11, 12),\n                        (4, 0.6831600707545022, 0.0, 0.1056808679308689),\n                        (10, 0.5461136927787811, -0.9999999999999996, -0.4590487459652186),\n                        (9, 0.5, 0.0, 0.6159985518275233),\n                        (3, 0.0, 0.35939939477765753, 0.0)],\n                       [(4, 10),\n                        (9, 3),\n                        (5, 0.6831600707545022, 0.0, 0.1987927050404909),\n                        (11, 0.5461136927787811, -0.9999999999999998, -0.48189054808852116),\n                        (12, 0.5, 0.0, 0.6159985518275233),\n                        (6, 0.0, 0.35939939477765753, 0.0)],\n                       [(5, 11),\n                        (12, 6),\n                        (3, 0.5, 1.0, -0.3562259424533867),\n                        (4, 0.5852312538430753, 0.0, -0.7640363113092423),\n                        (10, 0.7893393807194902, 0.9999999999999996, 0.5809027049300293),\n                        (9, 0.5, 0.0, 0.6159985518275233)],\n                       [(3, 4),\n                        (10, 9),\n                        (6, 0.5, 1.0, -0.3562259424533867),\n                        (5, 0.5852312538430753, 0.0, -0.6709244741996208),\n                        (11, 0.7893393807194902, 0.9999999999999998, 0.5580609028067265),\n                        (12, 0.5, 0.0, 0.6159985518275233)],\n                       [(6, 5),\n                        (11, 12),\n                        (4, 0.0, -0.7305296597872475, 0.0),\n                        (10, 0.7114915454728947, 0.0, 0.48993646481577285),\n                        (9, 0.0, -0.9935579880705372, 0.0),\n                        (3, 0.324430705358977, 0.0, -0.8562259424533867)],\n                       [(4, 10),\n                        (9, 3),\n                        (5, 0.0, 0.08324666599350894, 0.0),\n                        (11, 0.7114915454728947, 0.0, 0.46709466269247035),\n                        (12, 0.0, -0.9935579880705372, 0.0),\n                        (6, 0.324430705358977, 0.0, -0.8562259424533867)],\n                       [(5, 11),\n                        (12, 6),\n                        (10, 0.6122002465840692, 0.0, -0.9616363754926063),\n                        (9, 0.5, 0.0, -0.6896229878078779),\n                        (3, 0.5, 0.3712992757836946, 0.6437740575466131)],\n                       [(4, 10),\n                        (9, 8),\n                        (11, 0.6122002465840692, 4.440892098500626e-16, -0.9844781776159088),\n                        (12, 0.5, 0.0, -0.6896229878078779),\n                        (6, 0.5, 0.3712992757836946, 0.6437740575466131)],\n                       [(5, 11),\n                        (12, 13),\n                        (9, 0.9416885504219519, 0.9999999999999998, -0.2404230902197959),\n                        (8, 0.6085354469534892, 8.881784197001252e-16, -0.9311758863092963),\n                        (4, 0.0, 1.0, 0.0)],\n                       [(10, 9),\n                        (12, 0.9416885504219519, 0.9999999999999998, -0.2404230902197959),\n                        (13, 0.6085354469534892, 8.881784197001252e-16, -0.9311758863092963),\n                        (5, 0.0, 1.0, 0.0)],\n                       [(11, 12),\n                        (9, 0.5, 0.0, 0.6868476773632199),\n                        (10, 0.0966640990980218, 0.0, 0.8524388364788609)],\n                       [(9, 8),\n                        (4, 10),\n                        (12, 0.5, 0.0, 0.6868476773632199),\n                        (11, 0.09666409909802187, 0.0, 0.8295970343555588)],\n                       [(12, 13),\n                        (5, 11),\n                        (10, 0.5812745787265814, 0.0, -0.9416591321952976),\n                        (9, 0.5150139304348058, 0.9999999999999998, 0.6868476773632199),\n                        (8, 0.16723314080476154, 0.0, -0.7374666288921621),\n                        (4, 2.220446049250313e-16, 1.0, -0.6633429314122461)],\n                       [(10, 9),\n                        (11, 0.5812745787265814, 0.0, -0.9645009343186002),\n                        (12, 0.5150139304348058, 0.9999999999999998, 0.6868476773632199),\n                        (13, 0.16723314080476154, 0.0, -0.7374666288921621),\n                        (5, 2.220446049250313e-16, 1.0, 0.2435452314781319)],\n                       [(11, 12),\n                        (9, 0.08858950606155513, 0.0, -0.8131523226367803),\n                        (10, 0.2955394861408327, 0.0, 0.08574694958887025)],\n                       [(9, 8),\n                        (4, 10),\n                        (12, 0.08858950606155513, 0.0, -0.8131523226367803),\n                        (11, 0.2955394861408327, 0.0, 0.06290514746556775)],\n                       [(12, 13),\n                        (5, 11),\n                        (10, 0.29430117217170476, 0.0, 0.06353391068151715),\n                        (9, 0.30175052808312813, 0.9999999999999998, 0.6868476773632197),\n                        (8, 0.6303550437629813, 2.220446049250313e-16, -0.18158983536649442),\n                        (4, 0.4999999999999993, -0.9999999999999996, -0.2627750417968002)],\n                       [(10, 9),\n                        (11, 0.2943011721717049, 0.0, 0.04069210855821448),\n                        (12, 0.30175052808312813, 0.9999999999999998, 0.6868476773632197),\n                        (13, 0.6303550437629813, 2.220446049250313e-16, -0.18158983536649442),\n                        (5, 0.49999999999999956, 0.9999999999999999, 0.6441131210935778)],\n                       [(11, 12),\n                        (9, 0.19954896443512302, 0.0, -0.8131523226367807),\n                        (10, 0.30286229487152805, 8.881784197001252e-16, -0.9268607241785529)],\n                       [(9, 8),\n                        (4, 10),\n                        (12, 0.19954896443512302, 0.0, -0.8131523226367807),\n                        (11, 0.30286229487152805, 8.881784197001252e-16, -0.9497025263018555)],\n                       [(12, 13),\n                        (5, 11),\n                        (10, 0.5, 1.1102230246251565e-16, 0.30468860187044855),\n                        (9, 0.22177219480665555, 1.0, -0.3131523226367805),\n                        (8, 0.8315920243625678, 1.1102230246251565e-16, -0.4157702878189209),\n                        (4, 0.33347826094116906, -0.9999999999999996, -0.7627750417968007)],\n                       [(10, 9),\n                        (11, 0.5, 0.0, 0.2818467997471461),\n                        (12, 0.22177219480665555, 1.0, -0.3131523226367805),\n                        (13, 0.8315920243625678, 1.1102230246251565e-16, -0.4157702878189209),\n                        (5, 0.3334782609411692, -0.9999999999999996, 0.1441131210935767)],\n                       [(11, 12),\n                        (9, 0.5, -2.220446049250313e-16, 0.17388514996567728),\n                        (10, 0.5, 0.0, 0.3046886018704491)],\n                       [(9, 8),\n                        (4, 10),\n                        (12, 0.5, -2.220446049250313e-16, 0.17388514996567728),\n                        (11, 0.5, 0.0, 0.2818467997471465)],\n                       [(12, 13),\n                        (5, 11),\n                        (10, 0.5519231616233388, 0.0, -0.9131077146611837),\n                        (9, 0.5, 1.0, 0.17388514996567667),\n                        (8, 0.5, 0.0, 0.46178536847925344),\n                        (4, 0.5276550549544043, 1.0, -0.7627750417968007)],\n                       [(10, 9),\n                        (11, 0.5519231616233388, 0.0, -0.9359495167844865),\n                        (12, 0.5, 1.0, 0.17388514996567667),\n                        (13, 0.5, 0.0, 0.46178536847925344),\n                        (5, 0.5276550549544043, 1.0, 0.14411312109357727)],\n                       [(11, 12), (9, 0.5, 0.0, 0.8744064878335265), (10, 0.5, 0.0, -0.27896096075558896)],\n                       [(9, 8),\n                        (4, 10),\n                        (12, 0.5, 0.0, 0.8744064878335265),\n                        (11, 0.5, 0.0, -0.3018027628788915)],\n                       [(12, 13),\n                        (5, 11),\n                        (8, 0.5, 0.0, -0.038214631520746334),\n                        (4, 0.5, 0.9999999999999999, 0.7372249582031991),\n                        (10, 0.5000000000000001, 0.0, 0.721039039244411)],\n                       [(9, 8),\n                        (4, 10),\n                        (13, 0.5, 0.0, -0.038214631520746334),\n                        (5, 0.5000000000000002, 1.0, -0.3558868789064227),\n                        (11, 0.5, 0.0, 0.6981972371211085)],\n                       [(12, 13),\n                        (5, 11),\n                        (4, 0.4901053274689139, 8.881784197001252e-16, 0.7372484904865106),\n                        (8, 0.5068340379253959, 0.0, -0.5310585926382718),\n                        (9, 0.5, 0.0, 0.3744064878335269),\n                        (10, 0.5, 0.0, 0.7210390392444115)],\n                       [(4, 0),\n                        (8, 2),\n                        (5, 0.4901053274689138, 0.0, -0.3558633466231107),\n                        (13, 0.5068340379253959, 0.0, -0.5310585926382718),\n                        (12, 0.5, 0.0, 0.3744064878335269),\n                        (11, 0.5, 0.0, 0.6981972371211087)],\n                       [(5, 1),\n                        (13, 7),\n                        (8, 0.5, 0.0, -0.7736945421001282),\n                        (4, 0.4999999999999998, -0.9999999999999996, -0.7635086594593636),\n                        (0, 0.30408672398469616, 0.0, -0.5000000000000002),\n                        (2, 0.30408672398469616, 0.0, -0.5000000000000002)],\n                       [(9, 8),\n                        (10, 4),\n                        (13, 0.5, 0.0, -0.7736945421001282),\n                        (5, 0.4999999999999998, 1.0, 0.14337950343101497),\n                        (1, 0.30408672398469616, 0.0, -0.5000000000000002),\n                        (7, 0.30408672398469616, 0.0, -0.5000000000000002)],\n                       [(12, 13),\n                        (11, 5),\n                        (4, 0.8040867239846965, 5.551115123125783e-17, 0.23649134054063653),\n                        (8, 0.8040867239846965, 0.0, 0.22630545789987178),\n                        (10, 0.0, 0.4659468671857592, 0.0)],\n                       [(4, 0),\n                        (8, 2),\n                        (5, 0.8040867239846965, 0.0, -0.8566204965689852),\n                        (13, 0.8040867239846965, 0.0, 0.22630545789987178),\n                        (11, 0.0, 0.4659468671857592, 0.0)],\n                       [(5, 1),\n                        (13, 7),\n                        (4, 0.7499999999999999, 0.0, 0.23649134054063653),\n                        (8, 0.75, 1.0, 0.22630545789987166),\n                        (0, 0.5, 0.0, 0.5),\n                        (2, 0.5, 0.0, 0.5)],\n                       [(10, 4),\n                        (9, 8),\n                        (5, 0.75, 0.0, -0.8566204965689852),\n                        (13, 0.75, 1.0, 0.22630545789987166),\n                        (1, 0.5, 0.0, 0.5),\n                        (7, 0.5, 0.0, 0.5)],\n                       [(11, 5),\n                        (12, 13),\n                        (4, 0.5, 0.0, -0.7635086594593634),\n                        (8, 0.5, 0.0, -0.7736945421001282),\n                        (10, 0.5000000000000002, 0.0, 0.46958831163716974),\n                        (9, 0.5, 0.0, 0.6064966101833891)],\n                       [(4, 0),\n                        (8, 2),\n                        (5, 0.5, 0.0, 0.14337950343101508),\n                        (13, 0.5, 0.0, -0.7736945421001282),\n                        (11, 0.5, 0.0, 0.4467465095138672),\n                        (12, 0.5, 0.0, 0.6064966101833891)],\n                       [(5, 1),\n                        (13, 7),\n                        (4, 0.25000000000000006, 0.0, 0.23649134054063647),\n                        (8, 0.25000000000000006, 1.0, 0.22630545789987178)],\n                       [(10, 4),\n                        (9, 8),\n                        (5, 0.25000000000000006, 0.0, -0.8566204965689852),\n                        (13, 0.25000000000000006, 1.0, 0.22630545789987178)],\n                       [(11, 5),\n                        (12, 13),\n                        (4, 0.5, -0.9999999999999996, -0.7635086594593636),\n                        (8, 0.5, 1.0, -0.7736945421001282),\n                        (10, 0.5, 0.030411688362830258, 0.46958831163716974),\n                        (9, 0.5, -0.10649661018338907, -0.3935033898166107)],\n                       [(4, 0),\n                        (8, 2),\n                        (5, 0.5000000000000001, 1.0, 0.14337950343101508),\n                        (13, 0.5, 1.0, -0.7736945421001282),\n                        (11, 0.5, 0.053253490486132815, 0.4467465095138672),\n                        (12, 0.5, -0.10649661018338907, -0.3935033898166107)],\n                       [(5, 1),\n                        (13, 7),\n                        (0, 1.0, 0.0, 0.5),\n                        (2, 1.0, 0.0, 0.5),\n                        (4, 0.5, 0.2635086594593635, 0.23649134054063647),\n                        (8, 0.5, 0.2736945421001282, 0.22630545789987178)],\n                       [(1, 1.0, 0.0, 0.5),\n                        (5, 0.5, -0.6433795034310146, -0.8566204965689852),\n                        (7, 1.0, 0.0, 0.5),\n                        (13, 0.5, 0.2736945421001282, 0.22630545789987178)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],\n           \'qubits\': [(3, 5),\n                      (3, 6),\n                      (4, 3),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (4, 8),\n                      (5, 3),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8)]},\n (14, 5): {\'method\': \'reachable-support count kernels + native scheduling\',\n           \'moments\': [[(5, 0.6737899123902822, 0.0, 0.5),\n                        (4, 0.5574340700285423, 0.0, -0.5),\n                        (9, 0.837407518247816, 0.0, 0.5),\n                        (1, 4.48305315314812e-06, 0.0, -0.5000000000000002),\n                        (0, 0.4361596319645571, 0.0, 0.5),\n                        (11, 0.4999999999999998, 0.0, -0.721588825990928),\n                        (8, 0.4999999999999998, 0.0, -0.721588825990928),\n                        (6, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (3, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (2, 0.5, 0.0, 0.5),\n                        (13, 0.30408672398469655, 0.0, -0.5745192213521528),\n                        (7, 0.30408672398469655, 0.0, -0.5745192213521528),\n                        (12, 0.30408672398469655, 0.0, -0.5745192213521528)],\n                       [(9, 10), (1, 5), (0, 4)],\n                       [(9, 0.7632552034925755, 0.0, 0.5),\n                        (4, 0.7359966248900482, 0.0, 0.5),\n                        (5, 0.6736886561237059, 0.0, -0.5),\n                        (0, 0.7302021703149366, 0.0, 0.5),\n                        (1, 0.5000023278405843, 0.0, -0.5000000000000002)],\n                       [(4, 9), (5, 10)],\n                       [(5, 0.5001012562805767, 0.0, -0.5),\n                        (4, 0.5, 0.0, -0.5),\n                        (9, 0.8969013295764116, 0.0, -0.5)],\n                       [(4, 5), (9, 10)],\n                       [(4, 0.5, 0.0, -0.5), (1, 5), (9, 0.5024359486831966, 0.0, -0.5)],\n                       [(0, 4),\n                        (5, 0.9848747734228069, 0.0, -0.5000000000000004),\n                        (1, 0.5000038313083655, 0.0, -0.5)],\n                       [(4, 0.7040942510254498, 0.0, -0.5), (5, 10), (0, 0.5, 0.0, -0.5)],\n                       [(4, 9),\n                        (10, 0.4999999999999998, 0.0, -0.5000000000000002),\n                        (5, 0.4848747734228069, 0.0, 0.49999999999999933)],\n                       [(4, 0.2959057489745507, 0.0, 0.5), (9, 0.5, 0.0, -0.5)],\n                       [(9, 10), (4, 5)],\n                       [(4, 0.5, 0.0, -0.5),\n                        (10, 0.5000000000000004, 0.0, -0.5),\n                        (5, 0.5, 0.0, 0.4999999999999994),\n                        (9, 0.5000000000000004, 0.0, -0.5)],\n                       [(1, 5), (0, 4)],\n                       [(5, 10), (4, 0.5, 0.0, -0.5)],\n                       [(5, 0.39162345044252245, 0.0, 0.49351933791850716),\n                        (10, 0.5, 0.0, -0.9870386758370151),\n                        (4, 9)],\n                       [(10, 11),\n                        (1, 5),\n                        (9, 0.5129613241629847, 0.0, 0.0),\n                        (4, 0.6083765495574773, 0.0, -0.5064806620814923)],\n                       [(5, 0.8568537707944928, 0.0, -0.5064806620814928),\n                        (10, 0.7648228951691719, 1.1102230246251565e-16, -0.21255252592614915),\n                        (0, 4),\n                        (9, 8),\n                        (11, 0.5225448598685903, 0.0, 0.8911869797251353),\n                        (1, 0.9984061930639809, 0.0, 0.9667891923899257)],\n                       [(5, 10),\n                        (4, 0.8568537707944928, 0.0, 0.49351933791850744),\n                        (9, 0.7648228951691718, 0.0, -0.27448614991086595),\n                        (8, 0.5225448598685906, 0.0, -0.10881302027486428),\n                        (0, 0.9984061930639809, 8.881784197001252e-16, -0.9667891923899263)],\n                       [(10, 0.930805297572066, 0.0, -0.8231104994209764),\n                        (5, 0.4999999999999997, 0.0, 0.4935193379185074),\n                        (4, 9)],\n                       [(5, 10),\n                        (4, 0.5, 5.551115123125783e-17, 0.49351933791850716),\n                        (9, 0.930805297572066, 0.0, -0.6639281764160385)],\n                       [(10, 0.5, 0.0, 0.33860155683270676),\n                        (4, 9),\n                        (5, 0.5908115752601095, 4.440892098500626e-16, 0.5851449736633523)],\n                       [(10, 11),\n                        (9, 0.4999999999999998, 0.0, -0.8256402326697216),\n                        (4, 0.5908115752601095, 0.0, -0.5981062978263378),\n                        (1, 5)],\n                       [(10, 0.20245391911064137, 0.0, -0.6613984431672928),\n                        (9, 8),\n                        (11, 0.7201732888048016, 0.0, -0.7118510178536865),\n                        (1, 5),\n                        (0, 4)],\n                       [(5, 10),\n                        (9, 0.20245391911064134, 0.0, 0.17435976733027786),\n                        (8, 0.7201732888048016, 0.0, -0.7118510178536865),\n                        (0, 4),\n                        (1, 0.0015938069360184048, 0.0, 0.9667891923897574)],\n                       [(10, 0.6373873139241069, 0.0, 0.8386015568327065),\n                        (4, 9),\n                        (5, 0.4054888125911004, 0.0, 0.20325889016160076),\n                        (0, 0.0015938069360184048, 0.0, -0.9667891923897574)],\n                       [(10, 11),\n                        (9, 0.6373873139241069, 0.0, -0.32564023266972164),\n                        (4, 0.4054888125911004, 0.0, 0.7837797856754125)],\n                       [(10, 0.5, 0.0, 0.8386015568327068),\n                        (9, 8),\n                        (11, 0.5503418120493246, 0.0, -0.1702512394470519)],\n                       [(5, 10),\n                        (9, 0.5, 0.0, -0.32564023266972164),\n                        (8, 0.5503418120493246, 0.0, 0.8297487605529479)],\n                       [(10, 0.5724478383345383, 0.0, 0.2540877532015955),\n                        (5, 0.5000000000000004, 0.0, 0.34692195798925984),\n                        (4, 9)],\n                       [(1, 5),\n                        (10, 11),\n                        (9, 0.572447838334538, 0.0, -0.7411264290386106),\n                        (4, 0.5000000000000004, 0.0, -0.35988328215224596)],\n                       [(10, 0.5, 0.0, -0.9233293363278723),\n                        (5, 0.7849738353917708, 0.0, 0.8469219579892613),\n                        (9, 8),\n                        (0, 4),\n                        (11, 0.4999999999999977, 1.1102230246251565e-16, 0.3151950234033264),\n                        (1, 0.5, 0.0, -0.06318128417940283)],\n                       [(5, 10),\n                        (4, 0.7849738353917708, -2.220446049250313e-16, 0.1401167178477536),\n                        (9, 0.5, 5.551115123125783e-17, 0.4362906604908569),\n                        (8, 0.4999999999999977, 1.1102230246251565e-16, 0.3151950234033264),\n                        (0, 0.5, 0.0, -0.936818715820599)],\n                       [(10, 0.07810728535625011, 0.0, 0.576670663672127),\n                        (4, 9),\n                        (5, 0.5517093529747914, 0.0, -0.1530780420107385)],\n                       [(10, 11),\n                        (9, 0.07810728535625011, 0.0, -0.06370933950914237),\n                        (4, 0.5517093529747914, 0.0, 0.14011671784775276),\n                        (1, 5)],\n                       [(10, 0.5510400458124618, 0.0, 0.25563330704413456),\n                        (9, 8),\n                        (11, 0.2219727068374309, 0.0, -0.2540214619058512),\n                        (0, 4),\n                        (1, 0.49999999999999956, 0.05994095313865877, 0.936818715820597)],\n                       [(5, 10),\n                        (9, 0.5510400458124618, 0.0, 0.2573280171188501),\n                        (11, 6),\n                        (8, 0.2219727068374309, 0.0, 0.7459785380941486),\n                        (0, 0.5000000000000004, -0.06642161522014667, 0.06318128417940105)],\n                       [(10, 0.4999999999999994, 4.440892098500626e-16, 0.6766343658960108),\n                        (4, 9),\n                        (6, 0.85928593962011, 0.0, -0.9262332254895425),\n                        (11, 0.49999999999999994, 0.0, -0.7540214619058516),\n                        (8, 3)],\n                       [(5, 10),\n                        (9, 0.4999999999999993, 0.0, 0.836326958266973),\n                        (8, 0.5, 0.0, 0.24597853809414882),\n                        (3, 0.85928593962011, 0.0, -0.9262332254895425)],\n                       [(5, 0.4999999999999998, 0.0, 0.5431546500090731),\n                        (10, 0.29783770120218545, 0.0, 0.7135192241935249),\n                        (4, 9)],\n                       [(5, 10),\n                        (9, 0.2978377012021855, 0.0, -0.2005579000305402),\n                        (4, 0.5, 0.0, 0.4438840258279413)],\n                       [(5, 0.5, 0.0, -0.45684534999092685), (4, 9), (10, 11)],\n                       [(6, 5),\n                        (10, 0.5, 0.0, -0.3373696676969029),\n                        (4, 0.5, 0.0, 0.4438840258279415),\n                        (11, 0.6714165234362537, 0.0, -0.6826053701748587),\n                        (9, 8)],\n                       [(6, 0.7485956033393342, 0.0, -0.42623322548954246),\n                        (5, 10),\n                        (3, 4),\n                        (9, 0.5, 0.0, -0.14966900814011241),\n                        (8, 0.6714165234362537, 0.0, -0.6826053701748587)],\n                       [(11, 6),\n                        (3, 0.7485956033393342, 0.0, 0.5737667745104573),\n                        (5, 0.0528060550044127, 0.0, -0.283245006203956),\n                        (10, 0.5, 0.0, -0.3373696676969029),\n                        (4, 9)],\n                       [(6, 0.9961572718865779, 0.0, -0.9262332254895478),\n                        (11, 0.3935294720438234, 0.0, -0.4035582545612373),\n                        (8, 3),\n                        (9, 0.5, 0.0, 0.8503309918598878),\n                        (4, 0.052806055004412704, 0.0, -0.7297163179590287)],\n                       [(10, 11),\n                        (6, 5),\n                        (8, 0.3935294720438233, 0.0, -0.4035582545612373),\n                        (3, 0.9961572718865779, 0.0, 0.07376677451045216)],\n                       [(5, 0.24963665794604517, 0.0, 0.23411757870581823),\n                        (6, 0.5655532211464794, 0.0, 0.5737667745104469),\n                        (11, 0.3104369525321425, 0.0, -0.8220795049614371),\n                        (9, 8),\n                        (3, 4)],\n                       [(11, 6),\n                        (5, 10),\n                        (4, 0.24963665794604517, 0.0, 0.7529210971311959),\n                        (8, 0.3104369525321424, 0.0, 0.17792049503856378),\n                        (3, 0.5655532211464794, 0.0, 0.5737667745104469)],\n                       [(5, 0.3270115623295809, 0.0, -0.15810206258228376),\n                        (11, 0.6193705712509703, 0.0, 0.05088618413915785),\n                        (6, 0.4999999999999998, 4.440892098500626e-16, 0.5737667745104462),\n                        (4, 9),\n                        (8, 3)],\n                       [(6, 5),\n                        (10, 11),\n                        (8, 0.6193705712509703, 0.0, -0.9491138158608414),\n                        (3, 0.4999999999999998, 4.440892098500626e-16, 0.5737667745104462),\n                        (4, 0.32701156232958084, 0.0, 0.14514073841929748)],\n                       [(11, 0.6256105257587086, 0.0, -0.46584344785121257),\n                        (6, 0.5960614607313279, -2.220446049250313e-16, -0.11684358694965402),\n                        (5, 0.5, 0.0, -0.9107375311034074),\n                        (3, 4),\n                        (9, 8)],\n                       [(10, 11),\n                        (6, 5),\n                        (8, 0.6256105257587086, 0.0, -0.46584344785121257),\n                        (3, 0.5960614607313279, 0.0, 0.8831564130503458),\n                        (4, 0.5, 0.0, 0.8977762069404214)],\n                       [(5, 0.5, 0.0, -0.4107375311034076),\n                        (11, 0.9764747842833552, 0.0, -0.8680592057718213),\n                        (10, 0.5, 0.0, 0.3415589029956152),\n                        (6, 0.14949404734505523, 0.0, -0.7269328881171262),\n                        (3, 4),\n                        (9, 8)],\n                       [(5, 10),\n                        (11, 6),\n                        (9, 0.5, 0.0, -0.8285975788326287),\n                        (4, 0.5, 0.0, 0.3977762069404216),\n                        (8, 0.9764747842833552, 0.0, 0.13194079422817917),\n                        (3, 0.14949404734505523, 0.0, -0.7269328881171262)],\n                       [(11, 0.5, 0.0, -0.6250809225899472),\n                        (10, 0.5, 0.0, 0.3415589029956152),\n                        (8, 3),\n                        (6, 5),\n                        (4, 9)],\n                       [(10, 11),\n                        (6, 0.4247761378557593, 0.0, -0.022381851578533896),\n                        (8, 0.5, 0.0, 0.3749190774100535),\n                        (9, 0.5, -2.220446049250313e-16, 0.1714024211673711),\n                        (3, 4)],\n                       [(10, 0.5, 5.551115123125783e-17, 0.4087477570142521),\n                        (11, 0.1473187127029063, 0.0, 0.3749190774100535),\n                        (9, 8),\n                        (3, 0.42477613785575913, 0.0, 0.9776181484214663)],\n                       [(5, 10),\n                        (11, 6),\n                        (8, 0.1473187127029063, 0.0, 0.3749190774100535),\n                        (9, 0.5, 5.551115123125783e-17, 0.10421356714873403)],\n                       [(6, 0.45008695289946266, 0.0, -0.3365366861733614),\n                        (5, 0.4202007516434625, 0.0, -0.7041751726949097),\n                        (8, 3),\n                        (4, 9),\n                        (11, 0.9433458052208397, 0.0, 0.8749190774100519)],\n                       [(6, 5),\n                        (8, 0.9433458052208397, 0.0, 0.8749190774100519),\n                        (3, 0.4500869528994624, 0.0, 0.6634633138266381),\n                        (4, 0.4202007516434625, 0.0, 0.6912138485319237),\n                        (10, 11)],\n                       [(6, 0.5410889047360808, -2.220446049250313e-16, 0.21035484945072325),\n                        (3, 4),\n                        (9, 8),\n                        (10, 0.20567003520391755, 0.0, 0.9087477570142521),\n                        (5, 0.5, 0.0, -0.29937105785563567)],\n                       [(6, 2),\n                        (3, 0.5, 0.0, -0.28964515054927675),\n                        (8, 0.5, 0.0, 0.6692490422061348),\n                        (4, 0.5, 0.0, 0.28640973369264877),\n                        (9, 0.20567003520391755, 0.0, 0.6042135671487348),\n                        (10, 5)],\n                       [(6, 0.5, 0.0, -0.2896451505492772),\n                        (9, 4),\n                        (8, 3),\n                        (2, 0.3040867239846962, 0.0, 0.4999999999999998),\n                        (10, 0.49999999999998224, 0.0, -0.5912521866678064),\n                        (5, 0.5000000000000022, 0.0, 0.7892888717365298)],\n                       [(11, 6),\n                        (8, 0.4999999999999824, 0.0, -0.330750901475926),\n                        (9, 0.4999999999999827, 4.440892098500626e-16, -0.8957864891692056),\n                        (10, 13),\n                        (3, 0.5000000000000022, 4.440892098500626e-16, 0.6216949198585571),\n                        (4, 0.5000000000000019, 0.0, 0.19774980410048387)],\n                       [(6, 0.8040867239846965, 0.0, 0.7103548494507228),\n                        (8, 7),\n                        (9, 12),\n                        (10, 0.5000000000000102, 0.0, -0.5912521866678064),\n                        (13, 0.30408672398469627, 0.0, -0.5745192213521486)],\n                       [(6, 2),\n                        (9, 0.5000000000000102, 0.0, -0.8957864891692053),\n                        (8, 0.5000000000000102, 0.0, -0.3307509014759247),\n                        (10, 5),\n                        (12, 0.3040867239846962, 0.0, 0.4254807786478514),\n                        (7, 0.30408672398469627, 0.0, -0.5745192213521484)],\n                       [(6, 0.75, 0.0, 0.7103548494507228),\n                        (9, 4),\n                        (8, 3),\n                        (2, 0.5, 0.0, 0.5),\n                        (10, 0.7499999999999999, 0.0, 0.4087478458473725),\n                        (5, 0.49999999999999956, 0.0, 0.7892889042517117)],\n                       [(11, 6),\n                        (8, 0.75, 0.0, -0.33075086896074435),\n                        (9, 0.75, 0.0, -0.8957865216843854),\n                        (10, 13),\n                        (4, 0.49999999999999956, 0.0, -0.8022502284146986),\n                        (3, 0.4999999999999995, 4.440892098500626e-16, 0.6216948873433756)],\n                       [(6, 0.5, 0.0, 0.710354849450723),\n                        (9, 12),\n                        (8, 7),\n                        (11, 0.5, 0.0, -0.6045364702219087),\n                        (10, 0.5005268894219381, 0.0, -0.0907252180254634),\n                        (13, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155)],\n                       [(6, 2),\n                        (8, 0.500526889421938, 0.0, -0.830223932833581),\n                        (9, 0.5005268894219381, 0.0, -0.3963134578115495),\n                        (10, 5),\n                        (12, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155),\n                        (7, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155)],\n                       [(6, 0.25000000000000006, 0.0, -0.2896451505492772),\n                        (8, 3),\n                        (9, 4),\n                        (5, 0.5000000000000004, 0.0, -0.7107110957482878),\n                        (10, 0.49999999999999994, 0.0, -0.34072478195203093)],\n                       [(11, 6),\n                        (9, 0.5, 0.0, 0.8536861061150187),\n                        (8, 0.5, 0.0, -0.08022349676014806),\n                        (3, 0.5000000000000003, -2.220446049250313e-16, 0.12169488734337541),\n                        (4, 0.5000000000000004, 0.0, -0.30225022841469995),\n                        (10, 13)],\n                       [(6, 0.5, 0.0, -0.2896451505492772),\n                        (8, 3),\n                        (9, 12),\n                        (10, 5),\n                        (11, 0.5, 0.10453647022190982, -0.6045364702219096),\n                        (13, 0.5, -0.6516952337046309, 0.42548074613267234)],\n                       [(6, 2),\n                        (9, 4),\n                        (8, 7),\n                        (3, 0.0007451670218237004, -0.3479093749153339, -0.37830511265661615),\n                        (10, 0.9999999674848217, 0.6145102490490975, -0.34072475928654256),\n                        (5, 0.9992548329781763, -0.5155033918236529, 0.7892889042517031),\n                        (12, 0.5, 0.3483047662953691, -0.5745192538673274)],\n                       [(9, 3.251517832758566e-08, 0.42009940631302456, 0.8536860834495286),\n                        (6, 0.4999999999999998, 0.7896451505492779, 0.7103548494507228),\n                        (8, 0.999999967484822, -0.6459912361253086, 0.9197766258966009),\n                        (2, 1.0, 0.0, 0.5),\n                        (4, 0.0007451670218237004, -0.9239642591572581, 0.1977497715853102),\n                        (7, 0.5, -0.6516952337046309, 0.42548074613267284)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],\n           \'qubits\': [(3, 5),\n                      (3, 6),\n                      (3, 7),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 3),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 5),\n                      (6, 6)]},\n (14, 6): {\'method\': \'reachable-support count kernels + native scheduling\',\n           \'moments\': [[(1, 0.4630759433243212, 0.0, -0.5),\n                        (5, 0.48527820242856157, 0.0, -0.5),\n                        (2, 0.498691251687155, 0.0, -0.5),\n                        (6, 0.5034714935310124, 0.0, -0.5),\n                        (9, 0.33945009469785664, 0.0, -0.5),\n                        (10, 0.5407603083963027, 0.0, -0.5),\n                        (8, 0.4768011512951635, 0.9432287823717029, -0.1757887740017301),\n                        (4, 0.4127792040228902, 0.905374889856748, -0.9262332254895422),\n                        (0, 0.5, 0.0, 0.5),\n                        (12, 0.30408672398469644, 0.48537839930289595, -0.5745192213521528),\n                        (11, 0.4768011512951635, 0.9432287823717029, -0.1757887740017301),\n                        (7, 0.4127792040228902, 0.905374889856748, -0.9262332254895422),\n                        (3, 0.5, 0.0, 0.5),\n                        (13, 0.30408672398469644, 0.48537839930289595, -0.5745192213521528)],\n                       [(1, 5), (2, 6), (9, 10)],\n                       [(5, 0.3789124762180883, 0.0, 0.5),\n                        (9, 0.4058177292404212, 0.0, -0.5000000000000002),\n                        (6, 0.11475317333590987, 0.0, -0.5000000000000002),\n                        (10, 0.8510324075527451, 0.0, 0.5),\n                        (1, 0.6202325463217212, 0.0, 0.5),\n                        (2, 0.614760310033405, 5.551115123125783e-17, 0.49999999999999994)],\n                       [(5, 9), (6, 10)],\n                       [(5, 0.5, 0.0, -0.5),\n                        (6, 1.0, 0.0, 0.5),\n                        (9, 0.03453719968213479, 0.0, 0.5),\n                        (10, 0.628715121481435, 0.0, 0.5)],\n                       [(5, 6), (9, 10)],\n                       [(5, 0.5000000000000001, 0.0, 0.5),\n                        (6, 0.7568820983434257, 0.0, -0.5),\n                        (9, 0.1129617833785391, 0.0, -0.5000000000000002),\n                        (10, 0.6389837719170549, 0.0, -0.5)],\n                       [(1, 5), (2, 6)],\n                       [(5, 0.8697603441988186, 0.0, 0.5),\n                        (6, 0.7568820983434255, 0.0, 0.5),\n                        (1, 0.5, 0.0, -0.5),\n                        (2, 0.9999999999999999, 0.0, -0.5)],\n                       [(5, 9), (6, 10)],\n                       [(5, 0.9947413319518239, 0.0, 0.5),\n                        (6, 0.49999999999999983, 0.0, 0.5),\n                        (9, 0.3527928756243648, 0.0, -0.5),\n                        (10, 0.48152990442961807, 0.0, 0.5)],\n                       [(5, 6), (9, 10)],\n                       [(5, 0.810510574569716, 0.0, 0.5),\n                        (6, 0.25255329759659245, 0.0, -0.5),\n                        (9, 0.5337035612733155, 0.0, 0.5),\n                        (10, 0.038141509743885645, 0.0, 0.5)],\n                       [(1, 5), (2, 6)],\n                       [(5, 0.8105105745697158, 0.0, 0.5),\n                        (6, 0.7525532975965926, 0.0, -0.5),\n                        (1, 0.40328061753541666, 0.19569850482232592, 0.3859900904595022),\n                        (2, 0.40328061753541666, 0.19569850482232592, 0.3859900904595022)],\n                       [(5, 9), (6, 10)],\n                       [(5, 0.5000000000000001, -0.7375342002285916, -0.5767542509936456),\n                        (9, 0.3476072126893276, 0.19982621348950935, -0.005115459796453781),\n                        (6, 0.4999999999999998, -0.891042702215882, 0.576754250993645),\n                        (10, 0.6509298291713228, -0.7227053684541058, -0.022504053294038595)],\n                       [(1, 5), (9, 8), (2, 6), (10, 11)],\n                       [(5, 1.0, 0.0, 0.3248113626984579),\n                        (9, 0.16399934185718348, 0.45575126413524514, -0.3068017700064274),\n                        (1, 0.7174704187311102, -0.4034900130568444, 0.0816885952818282),\n                        (8, 0.6279799493860765, 0.19273066548707757, 0.1729408270855558),\n                        (6, 1.0, 0.0, 0.3248113626984579),\n                        (10, 0.16399934185718348, 0.45575126413524514, -0.3068017700064274),\n                        (2, 0.7174704187311102, -0.4034900130568444, 0.0816885952818282),\n                        (11, 0.6279799493860765, 0.19273066548707757, 0.1729408270855558)],\n                       [(5, 9), (6, 10)],\n                       [(5, 0.4558467315798988, 0.21174351290240845, -0.03079205169629784),\n                        (9, 0.27715476090744623, 0.2424375144678022, -0.16490618595693673),\n                        (6, 0.4558467315798988, 0.21174351290240845, -0.03079205169629784),\n                        (10, 0.27715476090744623, 0.2424375144678022, -0.16490618595693673)],\n                       [(1, 5), (9, 8), (2, 6), (10, 11)],\n                       [(1, 0.19408134503630478, 0.35592919330980477, -0.8218014177750159),\n                        (5, 0.4233573560778924, -0.7407977164480113, -0.3190485387938895),\n                        (9, 0.6057423835093776, -0.08700425013471058, -0.9330721302870997),\n                        (8, 0.01966364337041587, 0.8709790470198683, -0.3455352610731828),\n                        (2, 0.19408134503630478, 0.35592919330980477, -0.8218014177750159),\n                        (6, 0.4233573560778924, -0.7407977164480113, -0.3190485387938895),\n                        (10, 0.6057423835093776, -0.08700425013471058, -0.9330721302870997),\n                        (11, 0.01966364337041587, 0.8709790470198683, -0.3455352610731828)],\n                       [(1, 5), (9, 8), (2, 6), (10, 11)],\n                       [(5, 0.7065221727328495, 0.02949492115582153, 0.9121259095697485),\n                        (9, 0.7763756378143821, 0.02235950876954451, 0.986947558963229),\n                        (1, 0.574001540707395, 0.6757608608683758, -0.9658722244652111),\n                        (8, 0.5680691667523842, 0.6121163096700544, -0.630462126835079),\n                        (6, 0.7065221727328495, 0.02949492115582153, 0.9121259095697485),\n                        (10, 0.7763756378143821, 0.02235950876954451, 0.986947558963229),\n                        (2, 0.574001540707395, 0.6757608608683758, -0.9658722244652111),\n                        (11, 0.5680691667523842, 0.6121163096700544, -0.630462126835079)],\n                       [(5, 9), (6, 10)],\n                       [(5, 0.7550526236278825, 0.9502743195162635, -0.5665295147947895),\n                        (9, 0.7960253457838963, -0.2806238738272744, 0.0394700666967549),\n                        (6, 0.7550526236278825, 0.9502743195162635, -0.5665295147947895),\n                        (10, 0.7960253457838963, -0.2806238738272744, 0.0394700666967549)],\n                       [(1, 5), (9, 8), (2, 6), (10, 11)],\n                       [(5, 0.0931705087090361, -0.516609163737588, 0.25306172206678335),\n                        (9, 0.5920891677547482, 0.9128971050354752, 0.2588461928694804),\n                        (1, 1.0, 0.0, -0.3994351890684793),\n                        (8, 0.5767646373955951, -0.3159383542047276, -0.09507438390442124),\n                        (6, 0.0931705087090361, -0.516609163737588, 0.25306172206678335),\n                        (10, 0.5920891677547482, 0.9128971050354752, 0.2588461928694804),\n                        (2, 1.0, 0.0, -0.3994351890684793),\n                        (11, 0.5767646373955951, -0.3159383542047276, -0.09507438390442124)],\n                       [(5, 9), (6, 10)],\n                       [(5, 0.8490154235516789, -0.7890289379913964, 0.5275618533849498),\n                        (9, 0.8905211577257012, -0.8575924457636419, 0.6717432979049553),\n                        (6, 0.8490154235516789, -0.7890289379913964, 0.5275618533849498),\n                        (10, 0.8905211577257012, -0.8575924457636419, 0.6717432979049553)],\n                       [(1, 5), (9, 8), (2, 6), (10, 11)],\n                       [(5, 0.552523241658417, 0.834668637963333, -0.19987239712734106),\n                        (9, 0.6234642112340029, -0.049859643500711925, -0.6858491478586866),\n                        (1, 1.0, 0.0, -0.3528266328764229),\n                        (8, 0.22565786477821329, 0.7024859196960312, -0.7928005343757489),\n                        (6, 0.552523241658417, 0.834668637963333, -0.19987239712734106),\n                        (10, 0.6234642112340029, -0.049859643500711925, -0.6858491478586866),\n                        (2, 1.0, 0.0, -0.3528266328764229),\n                        (11, 0.22565786477821329, 0.7024859196960312, -0.7928005343757489)],\n                       [(5, 9), (6, 10)],\n                       [(5, 0.22560276650113942, 0.3001342884038919, -0.5349759837528378),\n                        (9, 0.1975568731256621, -0.6369107433613568, 0.7642912086406015),\n                        (6, 0.22560276650113942, 0.3001342884038919, -0.5349759837528378),\n                        (10, 0.1975568731256621, -0.6369107433613568, 0.7642912086406015)],\n                       [(1, 5), (9, 8), (2, 6), (10, 11)],\n                       [(5, 0.27319568030875896, -0.6397906871289019, 0.45027938981603566),\n                        (9, 0.5000000000000001, 0.04596377633835619, 0.12738046527924474),\n                        (1, 0.5000000000000002, 0.07304169278957362, -0.8712614671690471),\n                        (8, 0.08965970809102458, 0.4221266314305334, -0.5742415761229747),\n                        (6, 0.27319568030875896, -0.6397906871289019, 0.45027938981603566),\n                        (10, 0.5000000000000001, 0.04596377633835619, 0.12738046527924474),\n                        (2, 0.5000000000000002, 0.07304169278957362, -0.8712614671690471),\n                        (11, 0.08965970809102458, 0.4221266314305334, -0.5742415761229747)],\n                       [(5, 9), (6, 10)],\n                       [(5, 0.5000000000000002, 0.7179139416966681, -0.8249616681896759),\n                        (9, 0.6586668818397199, 0.14575709279368632, -0.4211858917978708),\n                        (6, 0.5000000000000002, 0.7179139416966681, -0.8249616681896759),\n                        (10, 0.6586668818397199, 0.14575709279368632, -0.4211858917978708)],\n                       [(1, 5), (9, 8), (2, 6), (10, 11)],\n                       [(5, 0.49999999999999983, 0.7467339844822286, 0.38080242206437276),\n                        (9, 0.6178431557297261, -0.7333690348374824, -0.27945069104741327),\n                        (8, 0.3573762722915824, -0.283056487039004, -0.09615352983889114),\n                        (6, 0.49999999999999983, 0.7467339844822286, 0.38080242206437276),\n                        (10, 0.6178431557297261, -0.7333690348374824, -0.27945069104741327),\n                        (11, 0.3573762722915824, -0.283056487039004, -0.09615352983889114),\n                        (1, 0.5000000000000001, 0.586596899876296, 0.2017802256205266),\n                        (2, 0.5000000000000001, 0.586596899876296, 0.2017802256205266)],\n                       [(5, 9), (8, 4), (6, 10), (11, 7)],\n                       [(9, 1.0, 0.0, 0.21461996803249472),\n                        (8, 0.5, 0.7022844924594797, 0.4262761990482463),\n                        (4, 0.85928593962011, -0.4875608597182275, -0.02085833563279449),\n                        (5, 0.5000000000000001, 0.12925995895837872, -0.8724635934533991),\n                        (10, 1.0, 0.0, 0.21461996803249472),\n                        (11, 0.5, 0.7022844924594797, 0.4262761990482463),\n                        (7, 0.85928593962011, -0.4875608597182275, -0.02085833563279449),\n                        (6, 0.5000000000000001, 0.12925995895837872, -0.8724635934533991)],\n                       [(9, 8), (4, 5), (10, 11), (7, 6)],\n                       [(5, 1.0, 0.0, -0.28654568860164664),\n                        (9, 0.5000000000000001, -0.07216288375695012, -0.25958902417306295),\n                        (8, 0.6714165234362537, 0.7430251514267326, -0.8000232167612813),\n                        (4, 0.7485956033393342, -0.10058310030841655, -0.008419195351022202),\n                        (6, 1.0, 0.0, -0.28654568860164664),\n                        (10, 0.5000000000000001, -0.07216288375695012, -0.25958902417306295),\n                        (11, 0.6714165234362537, 0.7430251514267326, -0.8000232167612813),\n                        (7, 0.7485956033393342, -0.10058310030841655, -0.008419195351022202)],\n                       [(5, 9), (8, 4), (6, 10), (11, 7)],\n                       [(9, 0.5000000000000001, -0.47921561052476913, 0.6682480920699869),\n                        (8, 0.39352947204382344, -0.0400166187933757, 0.22204905027907273),\n                        (4, 0.996157271886578, -0.38731562230521455, -0.6090022956594439),\n                        (5, 0.05280605500441269, -0.7140339826902271, -0.0034880864952426816),\n                        (10, 0.5000000000000001, -0.47921561052476913, 0.6682480920699869),\n                        (11, 0.39352947204382344, -0.0400166187933757, 0.22204905027907273),\n                        (7, 0.996157271886578, -0.38731562230521455, -0.6090022956594439),\n                        (6, 0.05280605500441269, -0.7140339826902271, -0.0034880864952426816)],\n                       [(9, 8), (4, 5), (10, 11), (7, 6)],\n                       [(5, 0.24963665794604506, 0.8669932073346845, 0.7651153459047553),\n                        (9, 1.0, 0.0, 0.21763245964208),\n                        (8, 0.3104369525321425, 0.8085371278988395, -0.23648881891450202),\n                        (4, 0.5655532211464792, -0.7603717865514881, -0.49631791796466374),\n                        (6, 0.24963665794604506, 0.8669932073346845, 0.7651153459047553),\n                        (10, 1.0, 0.0, 0.21763245964208),\n                        (11, 0.3104369525321425, 0.8085371278988395, -0.23648881891450202),\n                        (7, 0.5655532211464792, -0.7603717865514881, -0.49631791796466374)],\n                       [(5, 9), (8, 4), (6, 10), (11, 7)],\n                       [(9, 1.3678757607663618e-17, 0.19456334695121486, 0.24274381274574858),\n                        (8, 0.6193705712509703, -0.4502913348689348, -0.5549860019150674),\n                        (4, 0.49999999999999994, 0.914005305656897, 0.7433102954838477),\n                        (5, 0.32701156232958095, 0.6955438578648168, -0.9756718054724585),\n                        (10, 1.3678757607663618e-17, 0.19456334695121486, 0.24274381274574858),\n                        (11, 0.6193705712509703, -0.4502913348689348, -0.5549860019150674),\n                        (7, 0.49999999999999994, 0.914005305656897, 0.7433102954838477),\n                        (6, 0.32701156232958095, 0.6955438578648168, -0.9756718054724585)],\n                       [(9, 8), (4, 5), (10, 11), (7, 6)],\n                       [(9, 1.0, 0.0, 0.3991857392005793),\n                        (8, 0.6256105257587085, 0.9452263271866326, -0.5220069687743731),\n                        (4, 0.5960614607313279, 0.2573629292943688, -0.033294760319355854),\n                        (5, 0.5000000000000001, 0.3922892154072385, 0.472507520913482),\n                        (10, 1.0, 0.0, 0.3991857392005793),\n                        (11, 0.6256105257587085, 0.9452263271866326, -0.5220069687743731),\n                        (7, 0.5960614607313279, 0.2573629292943688, -0.033294760319355854),\n                        (6, 0.5000000000000001, 0.3922892154072385, 0.472507520913482)],\n                       [(9, 8), (4, 5), (10, 11), (7, 6)],\n                       [(5, 0.5, -0.2739726829496021, 0.3647967363207205),\n                        (9, 0.5000000000000001, -0.2267887021096202, 0.036504264403519004),\n                        (8, 0.9764747842833552, 0.005682454151557326, 0.021003600491651175),\n                        (4, 0.14949404734505523, 0.6915804042354776, 0.6139788678075409),\n                        (6, 0.5, -0.2739726829496021, 0.3647967363207205),\n                        (10, 0.5000000000000001, -0.2267887021096202, 0.036504264403519004),\n                        (11, 0.9764747842833552, 0.005682454151557326, 0.021003600491651175),\n                        (7, 0.14949404734505523, 0.6915804042354776, 0.6139788678075409)],\n                       [(5, 9), (8, 4), (6, 10), (11, 7)],\n                       [(9, 0.5, -0.9528264593460201, 0.8097155622938987),\n                        (8, 0.5, 0.47939827321355255, 0.2696643378250829),\n                        (4, 1.0, 0.0, -0.31991326341917814),\n                        (5, 1.0, 0.0, -0.27721006339896803),\n                        (10, 0.5, -0.9528264593460201, 0.8097155622938987),\n                        (11, 0.5, 0.47939827321355255, 0.2696643378250829),\n                        (7, 1.0, 0.0, -0.31991326341917814),\n                        (6, 1.0, 0.0, -0.27721006339896803)],\n                       [(9, 8), (4, 5), (10, 11), (7, 6)],\n                       [(5, 2.6911488363979655e-17, -0.10657650605920965, 0.3199678821356995),\n                        (9, 0.5000000000000001, 0.6065964482612792, -0.07592204303348415),\n                        (8, 0.14731871270290642, 0.47763908808934363, -0.2509373889613644),\n                        (4, 0.42477613785575913, 0.24565143911589105, -0.6499368354199673),\n                        (6, 2.6911488363979655e-17, -0.10657650605920965, 0.3199678821356995),\n                        (10, 0.5000000000000001, 0.6065964482612792, -0.07592204303348415),\n                        (11, 0.14731871270290642, 0.47763908808934363, -0.2509373889613644),\n                        (7, 0.42477613785575913, 0.24565143911589105, -0.6499368354199673)],\n                       [(5, 9), (8, 4), (6, 10), (11, 7)],\n                       [(9, 2.183989817369057e-17, 0.04089633285515237, 0.7520884815410067),\n                        (8, 0.9433458052208397, -0.437164462020319, -0.27329830087202156),\n                        (4, 0.4500869528994625, 0.5858004296037851, -0.09013056170924827),\n                        (5, 0.42020075164346243, -0.13244587453752787, -0.04525832781976602),\n                        (10, 2.183989817369057e-17, 0.04089633285515237, 0.7520884815410067),\n                        (11, 0.9433458052208397, -0.437164462020319, -0.27329830087202156),\n                        (7, 0.4500869528994625, 0.5858004296037851, -0.09013056170924827),\n                        (6, 0.42020075164346243, -0.13244587453752787, -0.04525832781976602)],\n                       [(9, 8), (4, 5), (10, 11), (7, 6)],\n                       [(4, 0.5410889047360808, -0.9487783322704517, -0.05122166772954828),\n                        (8, 5.822447387112414e-17, 0.6899183105243011, -0.1202404731065354),\n                        (9, 0.2056700352039175, 0.10680010097129156, -0.9284292619170531),\n                        (5, 0.5000000000000001, -0.5394914177845007, -0.7729000875180196),\n                        (7, 0.5410889047360808, -0.9487783322704517, -0.05122166772954828),\n                        (11, 5.822447387112414e-17, 0.6899183105243011, -0.1202404731065354),\n                        (10, 0.2056700352039175, 0.10680010097129156, -0.9284292619170531),\n                        (6, 0.5000000000000001, -0.5394914177845007, -0.7729000875180196)],\n                       [(4, 0), (9, 5), (7, 3), (10, 6)],\n                       [(4, 0.49999999999999983, 0.0, -0.5),\n                        (0, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (9, 0.4999999999999824, -0.29919772885013596, -0.32162910462782035),\n                        (5, 0.500000000000002, 0.3758195456300396, -0.2237315757103544),\n                        (7, 0.49999999999999983, 0.0, -0.5),\n                        (3, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (10, 0.4999999999999824, -0.29919772885013596, -0.32162910462782035),\n                        (6, 0.500000000000002, 0.3758195456300396, -0.2237315757103544)],\n                       [(8, 4), (9, 12), (11, 7), (10, 13)],\n                       [(4, 0.8040867239846964, 0.0, 0.5),\n                        (8, 1.0, 0.0, 0.5),\n                        (9, 0.5000000000000104, 0.7374333560946903, -0.6208268334779563),\n                        (12, 0.3040867239846962, 0.6612280619162106, -0.08914082204925244),\n                        (7, 0.8040867239846964, 0.0, 0.5),\n                        (11, 1.0, 0.0, 0.5),\n                        (10, 0.5000000000000104, 0.7374333560946903, -0.6208268334779563),\n                        (13, 0.3040867239846962, 0.6612280619162106, -0.08914082204925244)],\n                       [(4, 0), (9, 5), (7, 3), (10, 6)],\n                       [(4, 0.75, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (9, 0.75, -0.6782507102448094, -0.8833934448680876),\n                        (5, 0.4999999999999995, 0.12674214319260602, 0.15208800243486692),\n                        (7, 0.75, 0.0, 0.5),\n                        (3, 0.5, 0.0, 0.5),\n                        (10, 0.75, -0.6782507102448094, -0.8833934448680876),\n                        (6, 0.4999999999999995, 0.12674214319260602, 0.15208800243486692)],\n                       [(8, 4), (9, 12), (11, 7), (10, 13)],\n                       [(4, 0.5000000000000001, 0.0, -0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (9, 0.5005268894219383, 0.7511005823069626, -0.061117218985732524),\n                        (12, 0.5000000000000102, 0.2800353060019817, -0.4279127601330417),\n                        (7, 0.5000000000000001, 0.0, -0.5),\n                        (11, 0.5, 0.0, 0.5),\n                        (10, 0.5005268894219383, 0.7511005823069626, -0.061117218985732524),\n                        (13, 0.5000000000000102, 0.2800353060019817, -0.4279127601330417)],\n                       [(4, 0), (9, 5), (7, 3), (10, 6)],\n                       [(4, 0.25000000000000006, 0.0, 0.5),\n                        (0, 1.0, 0.0, 0.5),\n                        (9, 0.5000000000000001, -0.3275158685726778, 0.43998379939466253),\n                        (5, 0.5000000000000003, 0.7455961598985248, 0.7788301456274731),\n                        (7, 0.25000000000000006, 0.0, 0.5),\n                        (3, 1.0, 0.0, 0.5),\n                        (10, 0.5000000000000001, -0.3275158685726778, 0.43998379939466253),\n                        (6, 0.5000000000000003, 0.7455961598985248, 0.7788301456274731)],\n                       [(8, 4), (9, 12), (11, 7), (10, 13)],\n                       [(4, 0.5, 0.0, -0.5),\n                        (9, 1.0308214162014585e-14, -0.4910261758928436, 0.61246083991116),\n                        (7, 0.5, 0.0, -0.5),\n                        (10, 1.0308214162014585e-14, -0.4910261758928436, 0.61246083991116),\n                        (8, 0.5, 0.0, 0.5),\n                        (12, 0.5000000000000001, -0.07833700092571938, -0.14787748664623868),\n                        (11, 0.5, 0.0, 0.5),\n                        (13, 0.5000000000000001, -0.07833700092571938, -0.14787748664623868)],\n                       [(4, 0), (9, 5), (7, 3), (10, 6)],\n                       [(4, 0.5000000000000001, 0.0, 0.5),\n                        (9, 0.9999999674848218, 0.6523437121679241, -0.3785582224053705),\n                        (5, 0.9992548329781764, -0.7506407930979386, -0.9755736944740105),\n                        (7, 0.5000000000000001, 0.0, 0.5),\n                        (10, 0.9999999674848218, 0.6523437121679241, -0.3785582224053705),\n                        (6, 0.9992548329781764, -0.7506407930979386, -0.9755736944740105)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (3, 6),\n                      (3, 7),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 5),\n                      (6, 6)]},\n (14, 7): {\'method\': \'reachable-support count kernels + native scheduling\',\n           \'moments\': [[(4, 0.5, 0.0, 0.5),\n                        (8, 0.8238567956974769, 0.0, -0.5),\n                        (9, 0.8615786894865352, 0.0, -0.5),\n                        (5, 0.5013485215354914, 0.0, -0.5),\n                        (1, 0.5, 0.0, -0.5),\n                        (7, 0.151147072532456, 0.0, -0.4549188846524914),\n                        (10, 0.151147072532456, 0.0, -0.4549188846524914),\n                        (3, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (6, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (0, 0.5, 0.0, 0.5),\n                        (13, 0.30408672398469655, 0.0, -0.5745192213521528),\n                        (12, 0.30408672398469655, 0.0, -0.5745192213521528),\n                        (11, 0.30408672398469655, 0.0, -0.5745192213521528)],\n                       [(8, 9), (2, 5), (1, 4)],\n                       [(8, 0.8631826712147866, 0.0, -0.5),\n                        (9, 0.26852113390235993, 0.0, -0.5),\n                        (4, 0.8891976217454292, 0.0, 0.5),\n                        (5, 0.5381346275804277, 0.0, -0.5000000000000002),\n                        (2, 0.4999999999999998, 0.0, -0.5000000000000002),\n                        (1, 0.24294269698947368, 0.0, 0.5)],\n                       [(5, 9), (4, 8)],\n                       [(5, 0.5, 0.0, -0.5),\n                        (4, 0.11558148270486401, 0.0, -0.5000000000000002),\n                        (8, 0.5, 0.0, -0.5),\n                        (9, 0.7024793545619903, 0.0, -0.5)],\n                       [(4, 5), (8, 9)],\n                       [(5, 0.5, 0.0, -0.5), (9, 0.5, 0.0, 0.5), (4, 0.8168108984528897, 0.0, -0.5)],\n                       [(2, 5), (1, 4)],\n                       [(5, 0.5, 0.0, 0.5), (4, 8), (2, 0.4999999999999998, 0.0, -0.5000000000000002)],\n                       [(5, 9), (4, 0.8587961207350768, 0.0, 0.5), (8, 0.5903297495525045, 0.0, 0.5)],\n                       [(5, 0.5, 0.0, 0.5), (9, 0.5, 0.0, 0.5)],\n                       [(4, 5), (8, 9)],\n                       [(5, 0.5, 0.0, 0.4999999999999998), (4, 0.7076353249849605, 0.0, 0.5)],\n                       [(4, 8), (2, 5)],\n                       [(1, 4),\n                        (8, 0.27247163109080774, 0.0, 0.5917820603892103),\n                        (5, 0.5, 0.0, 0.4999999999999998),\n                        (2, 0.9396456562252813, 0.0, -0.5000000000000002)],\n                       [(4, 0.3371480118659349, 0.0, -0.5),\n                        (5, 9),\n                        (1, 0.897812251693201, 0.0, -0.5000000000000004)],\n                       [(4, 8),\n                        (5, 0.3633237216304231, 0.0, -0.5000000000000002),\n                        (9, 0.7577193621661036, 0.0, -0.39921734223686967)],\n                       [(8, 0.9003266480810048, 0.0, 0.5917820603892092),\n                        (4, 0.4999999999999998, 0.0, 0.545168747792844),\n                        (5, 9)],\n                       [(8, 7),\n                        (1, 4),\n                        (9, 0.9003266480810048, 0.0, 0.7360459496662108),\n                        (5, 0.4999999999999998, 0.0, -0.5451687477928442)],\n                       [(8, 0.8238665379044736, 4.440892098500626e-16, 0.5917820603892081),\n                        (4, 0.5, 0.0, 0.5451687477928437),\n                        (9, 10),\n                        (2, 5),\n                        (7, 0.4678935735932473, 0.0, 0.5450811153475075)],\n                       [(4, 8),\n                        (5, 0.5, 0.0, 0.4548312522071565),\n                        (9, 0.8238665379044736, 0.0, 0.7360459496662097),\n                        (10, 0.4678935735932473, 0.0, 0.5450811153475075)],\n                       [(8, 0.25087508170700706, 0.0, -0.4082179396107919), (5, 9), (1, 4)],\n                       [(8, 7), (9, 0.25087508170700706, 0.0, 0.7360459496662097), (2, 5)],\n                       [(8, 0.4411490610015294, 0.0, -0.40821793961079145),\n                        (9, 10),\n                        (7, 0.6252840566860213, 0.0, -0.4549188846524941)],\n                       [(4, 8),\n                        (9, 0.44114906100152934, 0.0, 0.7360459496662097),\n                        (10, 0.6252840566860213, 0.0, -0.4549188846524941)],\n                       [(8, 0.8018172683289451, 4.440892098500626e-16, 0.5917820603892081),\n                        (4, 0.5, 0.0, 0.5461513276753394),\n                        (5, 9)],\n                       [(8, 7),\n                        (1, 4),\n                        (5, 0.4999999999999998, 0.0, -0.54615132767534),\n                        (9, 0.8018172683289451, 4.440892098500626e-16, 0.7360459496662097)],\n                       [(8, 0.034005501067222, 4.440892098500626e-16, 0.5917820603892081),\n                        (4, 0.5, 0.0, 0.5461513276753394),\n                        (9, 10),\n                        (2, 5),\n                        (7, 0.23418985268179596, 4.440892098500626e-16, 0.5450811153475046)],\n                       [(4, 8),\n                        (5, 0.5, 0.0, 0.4538486723246611),\n                        (9, 0.034005501067222, 0.0, 0.7360459496662097),\n                        (10, 0.23418985268179596, 4.440892098500626e-16, 0.5450811153475046)],\n                       [(8, 0.5674840791730928, 4.440892098500626e-16, 0.5917820603892081),\n                        (4, 0.5, 0.0, 0.5575210475770775),\n                        (5, 9)],\n                       [(1, 4),\n                        (8, 7),\n                        (9, 0.5674840791730928, 0.0, -0.2639540503337905),\n                        (5, 0.4999999999999998, 0.0, 0.4424789524229218)],\n                       [(8, 0.5920238735305361, 0.0, -0.40821793961079167),\n                        (4, 0.5, 0.0, -0.44247895242292135),\n                        (2, 5),\n                        (9, 10),\n                        (7, 0.3495759458707011, 4.440892098500626e-16, 0.5450811153475055)],\n                       [(4, 8),\n                        (9, 0.5920238735305361, 0.0, 0.7360459496662097),\n                        (5, 0.5000000000000002, 0.0, 0.4424789524229218),\n                        (10, 0.3495759458707011, 4.440892098500626e-16, 0.5450811153475055)],\n                       [(8, 0.5920238735305361, 4.440892098500626e-16, 0.5917820603892081),\n                        (4, 0.4999999999999993, 0.0, 0.558503627459574),\n                        (5, 9)],\n                       [(8, 7),\n                        (1, 4),\n                        (9, 0.5920238735305361, 0.0, 0.7360459496662097),\n                        (5, 0.4999999999999995, 0.0, -0.558503627459574)],\n                       [(4, 0.7782742815462358, 4.440892098500626e-16, -0.4542339141993159),\n                        (8, 0.14060173868216674, 0.0, 0.5917820603892074),\n                        (9, 10),\n                        (2, 5),\n                        (7, 0.2784873646591348, 0.0, -0.4549188846524954)],\n                       [(4, 8),\n                        (5, 0.7782742815462358, 0.0, -0.5457660858006845),\n                        (9, 0.14060173868216674, 0.0, 0.7360459496662095),\n                        (10, 0.2784873646591348, 0.0, -0.4549188846524954)],\n                       [(4, 0.25, 0.0, 0.5291663156193014), (5, 9), (8, 7)],\n                       [(1, 4),\n                        (8, 0.6270946998774289, 0.0, -0.40821793961079234),\n                        (5, 0.25000000000000006, 0.0, 0.4708336843806986),\n                        (9, 10),\n                        (7, 0.3403378795392227, 0.0, -0.45116623934371125)],\n                       [(4, 0.7499999999999999, 0.0, -0.4708336843806986),\n                        (2, 5),\n                        (9, 0.6270946998774289, 0.0, 0.7360459496662093),\n                        (10, 0.3403378795392227, 0.0, -0.45116623934371125)],\n                       [(4, 8), (5, 0.75, 0.0, 0.4708336843806986)],\n                       [(4, 0.5, 0.0, 0.5291663156193014), (5, 9), (8, 0.5, 0.0, -0.40821793961079234)],\n                       [(1, 4),\n                        (5, 0.5, 0.0, -0.5291663156193014),\n                        (9, 0.5, 0.0, -0.2639540503337907),\n                        (8, 7)],\n                       [(4, 0.13865229719054487, 0.0, -0.4708336843806986),\n                        (2, 5),\n                        (1, 0.5, 0.0, 0.47705448490269675),\n                        (8, 7),\n                        (9, 10)],\n                       [(4, 8),\n                        (5, 0.13865229719054487, 0.0, -0.5291663156193014),\n                        (7, 0.4535551979948596, 0.0, 0.438011157726228),\n                        (2, 0.5, 0.0, 0.5229455150973021),\n                        (9, 10)],\n                       [(4, 0.4999999999999998, 0.0, 0.5291663156193014),\n                        (5, 9),\n                        (8, 0.5611163684243804, 0.0, -0.9704930864486796),\n                        (7, 3),\n                        (10, 0.4535551979948596, 0.0, 0.438011157726228)],\n                       [(1, 4),\n                        (5, 0.4999999999999998, 0.0, 0.4708336843806986),\n                        (3, 0.85928593962011, 0.0, -0.9262332254895425),\n                        (7, 0.5, 0.0, -0.1878391688093657),\n                        (9, 0.5611163684243805, -2.220446049250313e-16, 0.17377080282832358),\n                        (10, 6)],\n                       [(4, 0.5, 0.0, -0.7484151462228379),\n                        (2, 5),\n                        (8, 7),\n                        (6, 0.85928593962011, 0.0, -0.9262332254895425),\n                        (10, 0.5, 0.0, -0.1878391688093657),\n                        (1, 0.5, -0.954108969805394, -0.5229455150973035)],\n                       [(4, 8),\n                        (5, 0.5, 0.0, 0.7484151462228374),\n                        (7, 0.6714165234362537, 0.0, 0.8835769229216266),\n                        (9, 10),\n                        (2, 0.5, 0.0, -0.4770544849026974)],\n                       [(4, 0.5, 0.0, 0.2515848537771621),\n                        (5, 9),\n                        (8, 0.5, 0.0, -0.5248603567479282),\n                        (10, 0.6714165234362537, 0.0, 0.8835769229216266)],\n                       [(3, 4), (5, 0.5, 0.0, -0.25158485377716344), (9, 0.5, 0.0, -0.3805964674709248)],\n                       [(3, 0.7485956033393342, 0.0, 0.5737667745104573), (4, 8), (6, 5)],\n                       [(7, 3),\n                        (6, 0.7485956033393342, 0.0, -0.42623322548954246),\n                        (4, 0.052806055004412704, 0.0, -0.9220154900098112),\n                        (8, 0.5, 5.551115123125783e-17, 0.475139643252072),\n                        (5, 9)],\n                       [(3, 0.9961572718865779, 0.0, 0.07376677451045216),\n                        (7, 0.3935294720438233, 0.0, -0.8373759614647518),\n                        (10, 6),\n                        (5, 0.0528060550044127, 0.0, 0.9220154900098074),\n                        (9, 0.5, 0.0, -0.3805964674709248)],\n                       [(8, 7),\n                        (3, 4),\n                        (6, 0.9961572718865779, 0.0, -0.9262332254895478),\n                        (10, 0.3935294720438233, 0.0, -0.8373759614647518)],\n                       [(4, 0.24963665794604517, 0.0, 0.5606219250804136),\n                        (3, 0.5655532211464794, 0.0, 0.5737667745104469),\n                        (7, 0.31043695253214243, 0.0, -0.2558972118649514),\n                        (6, 5),\n                        (9, 10)],\n                       [(4, 8),\n                        (7, 3),\n                        (10, 0.31043695253214243, 0.0, -0.2558972118649514),\n                        (5, 0.24963665794604525, 0.0, -0.5606219250804185),\n                        (6, 0.5655532211464794, 0.0, 0.5737667745104469)],\n                       [(7, 0.6193705712509703, 4.440892098500626e-16, 0.6170684772356436),\n                        (4, 0.32701156232958084, 0.0, -0.04715843363148409),\n                        (3, 0.4999999999999998, 0.0, 0.5737667745104462),\n                        (5, 9),\n                        (10, 6)],\n                       [(3, 4),\n                        (8, 7),\n                        (5, 0.32701156232958084, 0.0, 0.047158433631480756),\n                        (6, 0.4999999999999998, 0.0, 0.5737667745104462),\n                        (10, 0.6193705712509703, 4.440892098500626e-16, 0.6170684772356436)],\n                       [(3, 0.5960614607313279, 0.0, 0.8831564130503458),\n                        (7, 0.6256105257587086, 0.0, -0.8996611547547264),\n                        (4, 0.5, 0.0, 0.7054770348896398),\n                        (6, 5),\n                        (9, 10)],\n                       [(3, 4),\n                        (8, 7),\n                        (5, 0.5, 0.0, -0.7054770348896429),\n                        (6, 0.5960614607313279, -2.220446049250313e-16, -0.11684358694965402),\n                        (10, 0.6256105257587086, 0.0, -0.8996611547547264)],\n                       [(3, 0.14949404734505523, 0.0, -0.7269328881171262),\n                        (7, 0.9764747842833552, 2.220446049250313e-16, -0.30187691267533445),\n                        (4, 0.5, 0.0, 0.2054770348896397),\n                        (8, 0.5, 4.440892098500626e-16, 0.7962110725595548),\n                        (9, 10),\n                        (6, 5)],\n                       [(4, 8),\n                        (7, 3),\n                        (5, 0.5, 1.1102230246251565e-16, -0.20547703488964464),\n                        (6, 0.14949404734505523, 0.0, -0.7269328881171262),\n                        (10, 0.9764747842833552, 2.220446049250313e-16, -0.30187691267533445),\n                        (9, 0.5, 0.0, 0.9404749618365573)],\n                       [(8, 0.5, 0.0, -0.2037889274404452),\n                        (7, 0.5, 0.0, -0.05889862949346014),\n                        (5, 9),\n                        (10, 6),\n                        (3, 4)],\n                       [(8, 7),\n                        (9, 0.5, 0.0, 0.9404749618365573),\n                        (10, 0.5, 0.0, -0.05889862949346014),\n                        (3, 0.42477613785575913, 0.0, 0.9776181484214663),\n                        (6, 5)],\n                       [(8, 0.5, 0.0, -0.27097778145908213),\n                        (7, 0.14731871270290628, 0.0, -0.058898629493459254),\n                        (9, 10),\n                        (6, 0.4247761378557593, 0.0, -0.022381851578533896)],\n                       [(4, 8),\n                        (7, 3),\n                        (10, 0.14731871270290628, 0.0, -0.058898629493459254),\n                        (9, 0.5, 4.440892098500626e-16, 0.8732861078179197)],\n                       [(4, 0.4202007516434626, 0.0, 0.49891467648114135),\n                        (3, 0.4500869528994624, 0.0, 0.6634633138266381),\n                        (5, 9),\n                        (10, 6),\n                        (7, 0.9433458052208397, 0.0, 0.4411013705065394)],\n                       [(3, 4),\n                        (10, 0.9433458052208397, 0.0, 0.4411013705065394),\n                        (6, 0.45008695289946266, 0.0, -0.3365366861733614),\n                        (5, 0.4202007516434625, 0.0, -0.4989146764811452),\n                        (8, 7)],\n                       [(3, 0.5410889047360808, -2.220446049250313e-16, 0.21035484945072325),\n                        (9, 10),\n                        (6, 5),\n                        (8, 0.2056700352039175, 0.0, 0.2290222185409183),\n                        (4, 0.5, -2.220446049250313e-16, 0.09411056164186743)],\n                       [(3, 0),\n                        (6, 0.5, 0.0, -0.2896451505492772),\n                        (5, 0.5, 0.0, -0.09411056164187137),\n                        (9, 0.20567003520391758, 0.0, 0.37328610781792015),\n                        (10, 0.5, 0.0, 0.23543133530262123),\n                        (8, 4)],\n                       [(3, 0.5, 0.0, -0.28964515054927675),\n                        (10, 6),\n                        (9, 5),\n                        (8, 0.4999999999999827, 0.0, 0.7290221622229764),\n                        (0, 0.3040867239846962, 0.0, 0.4999999999999998),\n                        (4, 0.500000000000002, 0.0, 0.0054506320497014205)],\n                       [(7, 3),\n                        (9, 0.49999999999998224, 0.0, -0.12671394850002127),\n                        (10, 0.4999999999999827, 0.0, -0.7645686083794385),\n                        (8, 12),\n                        (5, 0.500000000000002, 0.0, -0.005450632049704751),\n                        (6, 0.5000000000000022, 0.0, 0.6216949198585577)],\n                       [(3, 0.8040867239846965, 4.440892098500626e-16, 0.710354849450723),\n                        (9, 13),\n                        (10, 11),\n                        (8, 0.5000000000000103, 0.0, 0.7290221622229764),\n                        (12, 0.3040867239846962, 0.0, 0.4254807786478514)],\n                       [(3, 0),\n                        (10, 0.5000000000000102, 0.0, -0.7645686083794381),\n                        (9, 0.5000000000000103, 0.0, -0.12671394850002127),\n                        (8, 4),\n                        (11, 0.30408672398469627, 0.0, -0.5745192213521484),\n                        (13, 0.3040867239846962, 0.0, 0.4254807786478514)],\n                       [(3, 0.7499999999999999, 0.0, 0.710354849450723),\n                        (9, 5),\n                        (10, 6),\n                        (0, 0.5, 0.0, 0.5),\n                        (8, 0.75, 0.0, 0.7290221297077977),\n                        (4, 0.49999999999999967, 0.0, -0.99454940046548)],\n                       [(7, 3),\n                        (9, 0.75, 4.440892098500626e-16, 0.8732860189848002),\n                        (10, 0.75, 4.440892098500626e-16, -0.7645685758642609),\n                        (8, 12),\n                        (6, 0.49999999999999956, 0.0, 0.6216948873433754),\n                        (5, 0.49999999999999967, 0.0, 0.994549400465476)],\n                       [(3, 0.5, 0.0, 0.710354849450723),\n                        (10, 11),\n                        (9, 13),\n                        (8, 0.500526889421938, 0.0, -0.7715048064193661),\n                        (7, 0.5, 0.0, -0.03835417712542144),\n                        (12, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155)],\n                       [(3, 0),\n                        (9, 0.5005268894219383, 0.0, -0.6272409171423645),\n                        (10, 0.500526889421938, 0.0, 0.7359583602629036),\n                        (8, 4),\n                        (11, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155),\n                        (13, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155)],\n                       [(3, 0.25, 0.0, -0.28964515054927675),\n                        (9, 5),\n                        (10, 6),\n                        (8, 0.5000000000000001, 0.0, 0.4784947575072014),\n                        (4, 0.5000000000000004, 0.0, -0.49454940046548)],\n                       [(7, 3),\n                        (9, 0.5, 0.0, -0.37724135321579766),\n                        (10, 0.5, -1.021405182655144e-14, -0.5140412036636528),\n                        (5, 0.5000000000000004, -2.220446049250313e-16, 0.4945494004654756),\n                        (6, 0.5000000000000003, -2.220446049250313e-16, 0.12169488734337541),\n                        (8, 4)],\n                       [(3, 0.5, 0.0, -0.28964515054927675),\n                        (10, 11),\n                        (9, 13),\n                        (8, 12),\n                        (7, 0.5, -0.4616458228745772, -0.03835417712542277),\n                        (4, 0.0007451670218237008, -0.7316650871064769, 0.005450599534528033)],\n                       [(3, 0),\n                        (10, 6),\n                        (9, 5),\n                        (13, 0.5, 0.3483047662953691, -0.5745192538673274),\n                        (12, 0.5, 0.3483047662953691, -0.5745192538673274),\n                        (11, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (8, 3.251517832758566e-08, 0.7952907549208403, 0.478494734841713)],\n                       [(6, 0.0007451670218237004, -0.34790937491533347, -0.37830511265661615),\n                        (10, 0.999999967484822, -0.21217332923926913, 0.48595881900182514),\n                        (0, 1.0, 0.0, 0.5),\n                        (9, 3.251517832758566e-08, 0.6510268656438392, 0.6227586241187144),\n                        (3, 0.4999999999999998, 0.7896451505492773, 0.7103548494507228),\n                        (5, 0.9992548329781763, -0.7207638880374159, 0.9945494004654647)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8),\n                      (6, 5),\n                      (6, 6)]},\n (15, 3): {\'method\': \'reachable-support count kernels + native scheduling\',\n           \'moments\': [[(0, 0.5004931579194968, 0.0, -0.5),\n                        (5, 0.5700348769048169, 0.0, -0.5),\n                        (1, 0.2413858083053559, 0.0, 0.5),\n                        (6, 0.3116198483700336, 0.0, 0.5),\n                        (10, 0.3762212833232641, 0.0, 0.5),\n                        (11, 0.7973888376918397, 0.0, -0.5),\n                        (2, 0.41218411310679315, 4.973524073736968e-05, -0.5000093746463457),\n                        (3, 0.4535018303190155, 3.2138086280997236e-05, -0.4999866117568501),\n                        (4, 0.6538196916638768, 0.0012975466470504515, 0.4992166768114052),\n                        (7, 0.41218411310679315, 4.973524073736968e-05, -0.5000093746463457),\n                        (8, 0.4535018303190155, 3.2138086280997236e-05, -0.4999866117568501),\n                        (9, 0.6538196916638768, 0.0012975466470504515, 0.4992166768114052),\n                        (12, 0.41218411310679315, 4.973524073736968e-05, -0.5000093746463457),\n                        (13, 0.4535018303190155, 3.2138086280997236e-05, -0.4999866117568501),\n                        (14, 0.6538196916638768, 0.0012975466470504515, 0.4992166768114052)],\n                       [(0, 5), (1, 6), (10, 11)],\n                       [(0, 0.49774043690417774, 0.0, 0.5),\n                        (1, 0.8436377750014457, 0.0, -0.5),\n                        (5, 0.00220509059699017, 0.0, 0.5),\n                        (10, 0.3316948803903533, 0.0, -0.5),\n                        (6, 0.1042859540604348, 0.0, 0.5),\n                        (11, 0.6121001089933616, 0.0, 0.5)],\n                       [(0, 1), (5, 10), (6, 11)],\n                       [(0, 0.5, 0.0, -0.5),\n                        (1, 0.9171447574980371, 0.0, 0.5),\n                        (5, 1.0, 0.0, 0.5),\n                        (6, 0.34568332170776594, 0.0, -0.5),\n                        (10, 0.8865593269978381, 0.0, -0.5),\n                        (11, 0.8137793568788922, 0.0, -0.5)],\n                       [(0, 1), (5, 6), (10, 11)],\n                       [(0, 1.0, 0.0, 0.5),\n                        (5, 0.5, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5),\n                        (6, 0.9056685254861114, 0.0, 0.5),\n                        (10, 0.2971845674170174, 0.0, 0.5),\n                        (11, 0.24312932209862245, 0.0, 0.5)],\n                       [(0, 5), (1, 6), (10, 11)],\n                       [(0, 0.4065882770911243, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5),\n                        (5, 0.6594265589016369, 0.0, -0.5),\n                        (10, 0.5135612905851602, 0.0, 0.5),\n                        (6, 0.9866805895423636, 0.0, 0.5),\n                        (11, 0.9185973208306801, 0.0, -0.5000000000000002)],\n                       [(0, 1), (5, 10), (6, 11)],\n                       [(0, 0.3707227555445638, 0.0, 0.5),\n                        (5, 0.3508492030086968, 0.0, -0.5),\n                        (1, 0.329288506053183, 0.0, -0.5),\n                        (6, 0.9117177107952472, 0.0, 0.5),\n                        (11, 0.9874078125129342, 0.0, -0.5)],\n                       [(0, 5), (1, 6), (10, 11)],\n                       [(0, 0.5000000000000001, 0.0, 0.5),\n                        (1, 0.8969972054842089, 0.0, -0.5000000000000002),\n                        (5, 0.49999999999999983, 0.0, -0.5),\n                        (10, 1.0, 0.0, 0.5),\n                        (6, 0.4014516477690309, 0.0, -0.5),\n                        (11, 0.781287702544128, 0.0, 0.5)],\n                       [(0, 1), (5, 10), (6, 11)],\n                       [(1, 0.16304050725178096, 0.0, 0.5),\n                        (5, 1.0, 0.0, 0.5),\n                        (6, 0.4587625645746655, 0.0, 0.5),\n                        (10, 1.0, 0.0, 0.5),\n                        (11, 0.6233735693344237, 0.0, 0.5)],\n                       [(0, 1), (5, 6), (10, 11)],\n                       [(1, 0.9933638140034604, 0.000829234958738656, -0.5003955691202862),\n                        (0, 0.9999999999999999, -0.00021573299038535332, 0.5000743126499676),\n                        (6, 0.5170590993986217, 4.688320552392877e-05, -0.5000082581585381),\n                        (5, 3.3435642680263782e-18, -6.710769044948606e-05, 0.5024930601420385),\n                        (11, 0.26657171199549806, 4.185269792755797e-05, -0.5000110994755456),\n                        (10, 1.0, 0.0, 0.49996644615477503)],\n                       [(1, 2), (6, 7), (11, 12)],\n                       [(1, 0.5125153233564552, 6.816602735071342e-05, -0.5000700309596928),\n                        (2, 0.6117890444840384, 0.00023906418654384698, -0.5000335859224758),\n                        (6, 0.5125153233564552, 6.816602735071342e-05, -0.5000700309596928),\n                        (7, 0.6117890444840384, 0.00023906418654384698, -0.5000335859224758),\n                        (11, 0.5125153233564552, 6.816602735071342e-05, -0.5000700309596928),\n                        (12, 0.6117890444840384, 0.00023906418654384698, -0.5000335859224758)],\n                       [(0, 1), (5, 6), (10, 11)],\n                       [(1, 0.7075435896451812, -0.00015441380199909815, 0.5000167573434053),\n                        (0, 1.0, 0.0, -0.4999840678705958),\n                        (6, 0.7075435896451812, -0.00015441380199909815, 0.5000167573434053),\n                        (5, 1.0, 0.0, -0.4999840678705958),\n                        (11, 0.7075435896451812, -0.00015441380199909815, 0.5000167573434053),\n                        (10, 1.0, 0.0, -0.4999840678705958)],\n                       [(1, 2), (6, 7), (11, 12)],\n                       [(1, 0.13376593564282768, -3.928146360454576e-05, 0.5001322711129306),\n                        (2, 0.16507290691754561, -4.4512723523348896e-05, 0.49992633361577),\n                        (6, 0.13376593564282768, -3.928146360454576e-05, 0.5001322711129306),\n                        (7, 0.16507290691754561, -4.4512723523348896e-05, 0.49992633361577),\n                        (11, 0.13376593564282768, -3.928146360454576e-05, 0.5001322711129306),\n                        (12, 0.16507290691754561, -4.4512723523348896e-05, 0.49992633361577)],\n                       [(0, 1), (5, 6), (10, 11)],\n                       [(1, 0.4425626628501562, 0.00019430062546132998, 0.49988424829215317),\n                        (0, 0.5, -2.699936250127877e-07, 0.50000374169819),\n                        (6, 0.4425626628501562, 0.00019430062546132998, 0.49988424829215317),\n                        (5, 0.5, -2.699936250127877e-07, 0.50000374169819),\n                        (11, 0.4425626628501562, 0.00019430062546132998, 0.49988424829215317),\n                        (10, 0.5, -2.699936250127877e-07, 0.50000374169819)],\n                       [(1, 2), (6, 7), (11, 12)],\n                       [(1, 0.28336083660654154, 6.297169447666562e-05, -0.500078436406141),\n                        (2, 0.874698411133806, -5.5613993600545086e-05, -0.4999329159564543),\n                        (6, 0.28336083660654154, 6.297169447666562e-05, -0.500078436406141),\n                        (7, 0.874698411133806, -5.5613993600545086e-05, -0.4999329159564543),\n                        (11, 0.28336083660654154, 6.297169447666562e-05, -0.500078436406141),\n                        (12, 0.874698411133806, -5.5613993600545086e-05, -0.4999329159564543)],\n                       [(0, 1), (2, 3), (5, 6), (7, 8), (10, 11), (12, 13)],\n                       [(1, 0.9999999999999999, 1.0626804806790346e-05, 0.5000138488409602),\n                        (2, 0.5195218910218442, -2.492251341545959e-05, 0.49999809046902643),\n                        (3, 0.6338602251592093, -4.516549831157235e-05, -0.5000105128978589),\n                        (6, 0.9999999999999999, 1.0626804806790346e-05, 0.5000138488409602),\n                        (7, 0.5195218910218442, -2.492251341545959e-05, 0.49999809046902643),\n                        (8, 0.6338602251592093, -4.516549831157235e-05, -0.5000105128978589),\n                        (11, 0.9999999999999999, 1.0626804806790346e-05, 0.5000138488409602),\n                        (12, 0.5195218910218442, -2.492251341545959e-05, 0.49999809046902643),\n                        (13, 0.6338602251592093, -4.516549831157235e-05, -0.5000105128978589),\n                        (0, 0.5000000000000002, -9.01601680149966e-05, -0.4999965282954304),\n                        (5, 0.5000000000000002, -9.01601680149966e-05, -0.4999965282954304),\n                        (10, 0.5000000000000002, -9.01601680149966e-05, -0.4999965282954304)],\n                       [(1, 2), (6, 7), (11, 12)],\n                       [(2, 0.7465546320890828, -6.819337841124096e-05, 0.5000383284289129),\n                        (1, 1.0, 0.0, 0.4999738833942602),\n                        (7, 0.7465546320890828, -6.819337841124096e-05, 0.5000383284289129),\n                        (6, 1.0, 0.0, 0.4999738833942602),\n                        (12, 0.7465546320890828, -6.819337841124096e-05, 0.5000383284289129),\n                        (11, 1.0, 0.0, 0.4999738833942602)],\n                       [(2, 3), (7, 8), (12, 13)],\n                       [(2, 0.1601846384780278, -4.503868777239184e-05, 0.5000255168882044),\n                        (3, 0.288907983885649, 8.07695182105661e-05, -0.500167316940584),\n                        (7, 0.1601846384780278, -4.503868777239184e-05, 0.5000255168882044),\n                        (8, 0.288907983885649, 8.07695182105661e-05, -0.500167316940584),\n                        (12, 0.1601846384780278, -4.503868777239184e-05, 0.5000255168882044),\n                        (13, 0.288907983885649, 8.07695182105661e-05, -0.500167316940584)],\n                       [(1, 2), (6, 7), (11, 12)],\n                       [(2, 0.4422317455429899, -3.658965661768043e-05, 0.5000154601436346),\n                        (1, 0.5, -1.2757522787731901e-05, 0.5000119394225819),\n                        (7, 0.4422317455429899, -3.658965661768043e-05, 0.5000154601436346),\n                        (6, 0.5, -1.2757522787731901e-05, 0.5000119394225819),\n                        (12, 0.4422317455429899, -3.658965661768043e-05, 0.5000154601436346),\n                        (11, 0.5, -1.2757522787731901e-05, 0.5000119394225819)],\n                       [(2, 3), (7, 8), (12, 13)],\n                       [(2, 0.7312715381493196, -2.9719479089695966e-05, 0.5000244754386074),\n                        (3, 0.9620769699850723, -0.00013852518429713356, 0.500054648470984),\n                        (7, 0.7312715381493196, -2.9719479089695966e-05, 0.5000244754386074),\n                        (8, 0.9620769699850723, -0.00013852518429713356, 0.500054648470984),\n                        (12, 0.7312715381493196, -2.9719479089695966e-05, 0.5000244754386074),\n                        (13, 0.9620769699850723, -0.00013852518429713356, 0.500054648470984)],\n                       [(1, 2), (3, 4), (6, 7), (8, 9), (11, 12), (13, 14)],\n                       [(3, 0.37351126548912544, -7.546439919492443e-05, 0.5000773294001128),\n                        (4, 0.3040867239846963, -0.0005776526930145742, -0.49948577654154436),\n                        (2, 5.255863519135624e-17, -1.712579946921977e-05, 0.5012001664600749),\n                        (8, 0.37351126548912544, -7.546439919492443e-05, 0.5000773294001128),\n                        (9, 0.3040867239846963, -0.0005776526930145742, -0.49948577654154436),\n                        (7, 5.255863519135624e-17, -1.712579946921977e-05, 0.5012001664600749),\n                        (13, 0.37351126548912544, -7.546439919492443e-05, 0.5000773294001128),\n                        (14, 0.3040867239846963, -0.0005776526930145742, -0.49948577654154436),\n                        (12, 5.255863519135624e-17, -1.712579946921977e-05, 0.5012001664600749),\n                        (1, 0.5, -4.5164637105132854e-05, -0.5000008181002027),\n                        (6, 0.5, -4.5164637105132854e-05, -0.5000008181002027),\n                        (11, 0.5, -4.5164637105132854e-05, -0.5000008181002027)],\n                       [(3, 4), (8, 9), (13, 14)],\n                       [(3, 0.5000000000000004, -1.4577800550252817e-05, -0.4999981349990823),\n                        (4, 0.20227432911242674, -0.00018048654702673694, 0.4999365707654403),\n                        (8, 0.5000000000000004, -1.4577800550252817e-05, -0.4999981349990823),\n                        (9, 0.20227432911242674, -0.00018048654702673694, 0.4999365707654403),\n                        (13, 0.5000000000000004, -1.4577800550252817e-05, -0.4999981349990823),\n                        (14, 0.20227432911242674, -0.00018048654702673694, 0.4999365707654403)],\n                       [(2, 3), (7, 8), (12, 13)],\n                       [(3, 0.25000000000000006, -2.019130418995374e-05, -0.49998019675610816),\n                        (2, 0.5, -3.842212173532644e-06, 0.5000044520738967),\n                        (8, 0.25000000000000006, -2.019130418995374e-05, -0.49998019675610816),\n                        (7, 0.5, -3.842212173532644e-06, 0.5000044520738967),\n                        (13, 0.25000000000000006, -2.019130418995374e-05, -0.49998019675610816),\n                        (12, 0.5, -3.842212173532644e-06, 0.5000044520738967)],\n                       [(3, 4), (8, 9), (13, 14)],\n                       [(3, 0.25, 4.345730322263819e-07, 0.4999996119397021),\n                        (4, 0.5000000000000004, 0.00010607496466885635, -0.5002764318251103),\n                        (8, 0.25, 4.345730322263819e-07, 0.4999996119397021),\n                        (9, 0.5000000000000004, 0.00010607496466885635, -0.5002764318251103),\n                        (13, 0.25, 4.345730322263819e-07, 0.4999996119397021),\n                        (14, 0.5000000000000004, 0.00010607496466885635, -0.5002764318251103)],\n                       [(2, 3), (7, 8), (12, 13)],\n                       [(3, 0.5, -4.246260076889996e-06, -0.4999999534872659),\n                        (8, 0.5, -4.246260076889996e-06, -0.4999999534872659),\n                        (13, 0.5, -4.246260076889996e-06, -0.4999999534872659),\n                        (2, 0.5, -2.4189174229860555e-05, 0.5000006098617227),\n                        (7, 0.5, -2.4189174229860555e-05, 0.5000006098617227),\n                        (12, 0.5, -2.4189174229860555e-05, 0.5000006098617227)],\n                       [(3, 4), (8, 9), (13, 14)],\n                       [(3, 0.5000000000000001, -1.937956516417927e-05, 0.4999958002526569),\n                        (4, 6.271301173889318e-17, 0.00014677754793412934, 0.4999292001208716),\n                        (8, 0.5000000000000001, -1.937956516417927e-05, 0.4999958002526569),\n                        (9, 6.271301173889318e-17, 0.00014677754793412934, 0.4999292001208716),\n                        (13, 0.5000000000000001, -1.937956516417927e-05, 0.4999958002526569),\n                        (14, 6.271301173889318e-17, 0.00014677754793412934, 0.4999292001208716)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (3, 6),\n                      (3, 7),\n                      (3, 8),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (4, 8),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8)]},\n (15, 4): {\'method\': \'reachable-support count kernels + native scheduling\',\n           \'moments\': [[(4, 0.5050311694609142, 0.0, 0.5),\n                        (7, 0.7754283788865404, 0.0, -0.5),\n                        (5, 0.31203074057342906, 0.0, -0.5),\n                        (8, 0.6647424953468708, 0.0, -0.5),\n                        (6, 0.4019600678849752, 0.0, 0.5),\n                        (9, 0.9860634487343363, 0.0, 0.5),\n                        (3, 0.42588759890619404, 0.0, 0.5918062908230985),\n                        (14, 0.75, 0.0, -0.07606118876167),\n                        (13, 0.7728144741714948, -0.9999999999999998, 0.0805732800089835),\n                        (1, 0.4999999999999997, 0.9999999999999999, 0.5937872305331664),\n                        (12, 0.4999999999999998, 0.0, 0.5937872305331664),\n                        (0, 0.5, 0.0, 0.5),\n                        (2, 0.5, 0.0, 0.5),\n                        (11, 0.5, 0.0, 0.5),\n                        (10, 0.5, 1.0, 0.5)],\n                       [(4, 7), (5, 8), (6, 9)],\n                       [(4, 0.5066084360344099, 0.0, 0.5),\n                        (5, 0.7976060787857662, 0.0, 0.5),\n                        (7, 0.004284890859710177, 0.0, 0.5),\n                        (8, 0.7535395929271749, 0.0, 0.5),\n                        (6, 0.04960829035671744, 0.0, 0.5),\n                        (9, 0.9957726141502476, 0.0, -0.5000000000000002)],\n                       [(4, 5), (7, 8)],\n                       [(4, 0.3881710437984986, 0.0, 0.5),\n                        (7, 0.8887989454065368, 0.0, 0.5),\n                        (5, 6),\n                        (8, 0.3000520402754841, 0.0, -0.5)],\n                       [(4, 7),\n                        (8, 9),\n                        (5, 0.49999999999999223, 0.0, 0.5),\n                        (6, 0.1993382800735905, 0.0, 0.5)],\n                       [(8, 0.20007974046133922, 0.0, -0.5),\n                        (9, 0.4999999999999998, 0.0, 0.5),\n                        (4, 0.09428404233770227, 0.0, 0.5),\n                        (7, 0.8168684096225483, 0.0, 0.5)],\n                       [(5, 8), (6, 9)],\n                       [(5, 0.6272655967963714, 2.220446049250313e-16, -0.5000000000000002),\n                        (8, 0.139052622364793, 0.0, -0.5000000000000002),\n                        (9, 0.5, 0.0, 0.5),\n                        (6, 0.9921928118257514, 0.0, -0.5)],\n                       [(4, 5), (7, 8)],\n                       [(8, 9), (5, 0.8727344032036237, 0.0, 0.5), (7, 0.5, 1.0, -0.5)],\n                       [(5, 6), (4, 7), (9, 0.8148309319180607, 0.0, 0.5)],\n                       [(5, 0.7233498104809428, 0.0, 0.5),\n                        (6, 0.6959132760153035, 0.0, 0.5),\n                        (7, 0.5, 0.0, -0.5)],\n                       [(5, 8), (6, 9)],\n                       [(7, 8),\n                        (5, 0.0, 1.0, 0.0),\n                        (6, 0.27414986841876443, 0.0, 0.8626655155555705),\n                        (9, 0.5000000000000009, 0.0, -0.5)],\n                       [(4, 5),\n                        (6, 3),\n                        (9, 14),\n                        (7, 0.5777417536294465, 0.0, -0.8698433920623254),\n                        (8, 0.8009489577761006, 0.0, 0.9999999999999998)],\n                       [(7, 8),\n                        (5, 0.7308754207007768, 0.9999999999999999, 0.572080485270047),\n                        (6, 0.7812646514021274, 0.0, -0.26303142154550585),\n                        (9, 0.5, -0.9999999999999996, -0.39810208444779915),\n                        (14, 0.2500000000000002, 0.0, 0.9239388112383322),\n                        (3, 0.34615445441753034, -0.9999999999999998, -0.791259224473966)],\n                       [(4, 5), (8, 9), (7, 0.12935406511725003, 1.0, -0.5318618508996764)],\n                       [(5, 0.26010688228278134, 0.0, -0.3242586920561732),\n                        (4, 0.0, 0.8804502887098096, 0.0),\n                        (9, 0.37023281907812394, 0.0, -0.7314354177811329),\n                        (8, 0.5, 0.0, 0.30454264866352565)],\n                       [(5, 6), (9, 14), (7, 8)],\n                       [(5, 0.37932575112602124, 0.0, 0.09474578424214686),\n                        (6, 3),\n                        (8, 0.949194148682035, 5.551115123125783e-17, 0.30454264866352554),\n                        (9, 0.29863959134576673, 1.0, 0.26856458221886725),\n                        (7, 0.9052420397088912, 0.9999999999999999, 0.9681381491003223)],\n                       [(4, 5),\n                        (6, 0.4999999999999998, 0.0, -0.23110536055840925),\n                        (8, 9),\n                        (3, 0.565849320070786, 0.9999999999999999, 0.8639427624792817)],\n                       [(5, 0.1728700707127321, 1.0, 0.25980625311387395),\n                        (4, 0.4999999999999998, 0.0, -0.011037258340573564),\n                        (9, 0.5, -2.220446049250313e-16, 0.1622120753992698),\n                        (8, 0.5, 0.0, -0.19545735133647457)],\n                       [(5, 6), (9, 14), (7, 8)],\n                       [(5, 0.9245508650264845, 0.0, 0.6066663543154596),\n                        (6, 0.49999999999999845, 0.0, -0.7311053605584099),\n                        (8, 0.5842177387066689, 0.0, -0.6954573513364743),\n                        (9, 2.220446049250313e-16, -0.9999999999999996, 0.5027084528360493),\n                        (7, 0.6063525068195972, 1.0, 0.9681381491003216)],\n                       [(4, 5), (6, 3), (8, 9)],\n                       [(5, 0.5, 1.0, -0.24380250484892163),\n                        (6, 0.4999999999999998, 0.0, -0.033352937607141975),\n                        (4, 0.5, 0.0, 0.9889627416594262),\n                        (9, 0.6297671809218761, 0.0, -0.33778792460073026),\n                        (3, 0.0, 1.0, 0.0),\n                        (8, 0.5, 0.0, 0.30454264866352565)],\n                       [(5, 6), (9, 14), (7, 8)],\n                       [(5, 0.3485609400244275, 0.0, -0.7438025048489216),\n                        (6, 0.5, 0.0, -0.2833529376071433),\n                        (8, 0.8311275895761097, 0.0, -0.6954573513364739),\n                        (9, 0.5, -0.9999999999999998, 0.1622120753992698),\n                        (14, 13),\n                        (7, 0.4999999999999971, -0.1186126279883708, -0.03186185089967952)],\n                       [(4, 5),\n                        (6, 3),\n                        (8, 9),\n                        (14, 0.791814249190367, 0.0, -0.06510304415232726),\n                        (13, 0.7728144741714948, 1.0, -0.9194267199910164)],\n                       [(5, 0.7500000000000004, -2.220446049250313e-16, -0.7438025048489214),\n                        (6, 0.0, 0.30239487438068663, 0.0),\n                        (4, 0.49999999999999956, 0.0, -0.6605683991761879),\n                        (8, 0.5, 0.0, -0.1954573513364739),\n                        (9, 0.9552978770634988, 1.0, -0.7600838396544203)],\n                       [(5, 6), (8, 9)],\n                       [(5, 0.231174319940596, 4.440892098500626e-16, 0.7561974951510784),\n                        (6, 0.620275804748211, 1.0, 0.01904193677354321),\n                        (9, 0.2630328282816797, 1.0, 0.07775947136418593),\n                        (8, 0.5, 0.0, 0.7309004050344949)],\n                       [(4, 5), (6, 3), (9, 14), (13, 8)],\n                       [(5, 0.6245911673988008, 0.0, -0.2438025048489212),\n                        (6, 0.5, -4.440892098500626e-16, -0.48095806322645673),\n                        (3, 0.5, 0.0, -0.5350638469289413),\n                        (14, 0.6714475580169545, 0.0, 0.47623362958701543),\n                        (8, 0.5, 0.0, -0.2690995949655053),\n                        (9, 0.791370059531975, 1.0, 0.08532166826046073),\n                        (4, 0.0, 0.39456853286863147, 0.0)],\n                       [(5, 6), (3, 1), (14, 13), (8, 9)],\n                       [(5, 0.5, 5.551115123125783e-17, 0.25619749515107854),\n                        (6, 0.5000000000000011, 0.0, -0.47377968804878834),\n                        (3, 0.9416885504219519, 0.0, -0.9842637445170233),\n                        (9, 0.6715105414343046, -0.9999999999999993, -0.9786320368729815),\n                        (14, 0.5751637920695032, 0.0, 0.14042530200479847),\n                        (13, 0.5, 0.0, -0.7543510670180298),\n                        (8, 0.0, -0.8733464313908925, 0.0),\n                        (1, 0.6085354469534892, -0.9999999999999996, -0.9311758863092963)],\n                       [(5, 6), (9, 14), (13, 8)],\n                       [(6, 3),\n                        (9, 0.8057807413978859, 0.0, -0.1364229587098209),\n                        (14, 0.3240470566789823, 0.0, 0.360757337770405),\n                        (13, 0.5, 0.0, 0.24564893298197044),\n                        (5, 0.0, 1.0, 0.0)],\n                       [(8, 9),\n                        (14, 13),\n                        (3, 0.5, 8.881784197001252e-16, -0.9115345121000391),\n                        (6, 0.09666409909802184, 4.440892098500626e-16, -0.9255224856791187)],\n                       [(3, 1),\n                        (5, 6),\n                        (13, 0.5, 1.1102230246251565e-16, -0.38400144817247694),\n                        (9, 0.6831600707545022, 0.9999999999999999, 0.8589025178108383),\n                        (14, 0.5461136927787811, 0.0, 0.8039314437746508)],\n                       [(6, 0.5812745787265814, -2.220446049250313e-16, -0.7196204543532774),\n                        (3, 0.5150139304348057, 8.881784197001252e-16, -0.9115345121000391),\n                        (13, 8),\n                        (9, 14),\n                        (1, 0.16723314080476154, 1.0, -0.7374666288921616),\n                        (5, 2.220446049250313e-16, -0.9999999999999996, 0.7234237827167624)],\n                       [(6, 3),\n                        (14, 0.7893393807194902, 0.0, -0.2360200071205969),\n                        (13, 0.5, 0.0, 0.6159985518275235),\n                        (8, 0.5, 0.0, 0.4963470756156263),\n                        (9, 0.5852312538430753, 0.9999999999999999, -0.27138030294905)],\n                       [(14, 13),\n                        (3, 0.08858950606155513, 0.0, -0.4115345121000389),\n                        (6, 0.2955394861408327, 0.0, -0.6922143725691097),\n                        (8, 9)],\n                       [(3, 1),\n                        (5, 6),\n                        (14, 0.7114915454728947, 0.0, 0.8549462329936599),\n                        (13, 0.0, -0.9935579880705372, 0.0),\n                        (8, 0.32443070535897683, 0.0, 0.9963470756156263)],\n                       [(6, 0.29430117217170476, 0.0, -0.7144274114764624),\n                        (3, 0.3017505280831283, 0.0, 0.08846548789996111),\n                        (9, 14),\n                        (13, 8),\n                        (1, 0.6303550437629813, 1.0, -0.18158983536649442),\n                        (5, 0.4999999999999993, 1.0, -0.8760083276677906)],\n                       [(6, 3),\n                        (14, 0.6122002465840692, 0.0, -0.693480926697962),\n                        (13, 0.5, 0.0, -0.6896229878078779),\n                        (8, 0.5, -0.48127374228531905, 0.4963470756156263)],\n                       [(9, 14),\n                        (13, 12),\n                        (3, 0.19954896443512293, 0.0, 0.5884654878999611),\n                        (6, 0.30286229487152805, 0.0, -0.7048220463365322)],\n                       [(3, 1),\n                        (5, 6),\n                        (14, 0.0, -0.24990900502530589, 0.0),\n                        (13, 0.9416885504219519, 0.0, -0.2404230902197959),\n                        (12, 0.6085354469534892, 8.881784197001252e-16, -0.9311758863092963)],\n                       [(6, 0.5, 0.0, 0.5267272797124694),\n                        (3, 0.22177219480665555, 0.0, 0.08846548789996161),\n                        (14, 13),\n                        (1, 0.8315920243625678, 1.0, -0.4157702878189209),\n                        (5, 0.33347826094116917, 0.9999999999999999, 0.6239916723322076)],\n                       [(6, 3),\n                        (13, 0.5, 0.0, 0.6868476773632199),\n                        (14, 0.09666409909802182, -0.9999999999999998, 0.24253485630526422)],\n                       [(13, 12),\n                        (3, 0.5, 8.881784197001252e-16, 0.6014280152975033),\n                        (6, 0.5000000000000001, -2.220446049250313e-16, -0.47327272028753065),\n                        (9, 14)],\n                       [(3, 1),\n                        (5, 6),\n                        (14, 0.5812745787265815, 0.0, 0.036632824979423606),\n                        (13, 0.5150139304348058, 0.0, 0.6868476773632199),\n                        (12, 0.16723314080476154, 0.0, -0.7374666288921621)],\n                       [(6, 0.5519231616233388, 0.0, 0.30893096318083657),\n                        (3, 0.5, 8.881784197001252e-16, 0.6014280152975033),\n                        (14, 13),\n                        (1, 0.5, 0.9999999999999999, 0.46178536847925333),\n                        (5, 0.5276550549544043, 0.9999999999999998, 0.6239916723322081)],\n                       [(6, 3),\n                        (13, 0.08858950606155513, 0.0, -0.8131523226367803),\n                        (14, 0.2955394861408326, 1.0, -0.9907732568047445)],\n                       [(13, 12),\n                        (3, 0.5, 0.0, 0.9009066774296537),\n                        (6, 0.5, 0.0, -0.056922282913569155),\n                        (9, 14)],\n                       [(3, 1),\n                        (5, 6),\n                        (14, 0.2943011721717048, 8.881784197001252e-16, -0.9685602178973916),\n                        (13, 0.3017505280831281, 0.0, 0.6868476773632197),\n                        (12, 0.6303550437629813, 2.220446049250313e-16, -0.18158983536649442),\n                        (9, 0.49999999999999956, 0.0, 0.4968287677512597)],\n                       [(14, 13),\n                        (1, 0.5, 1.0, -0.038214631520746334),\n                        (5, 0.5, 1.0, 0.12399167233220787),\n                        (6, 0.5, 0.0, 0.9430777170864308)],\n                       [(3, 1),\n                        (5, 6),\n                        (13, 0.19954896443512302, 0.0, -0.8131523226367807),\n                        (14, 0.302862294871528, 1.0, 0.021834416962677794)],\n                       [(1, 0.5068340379253959, 0.0, -0.5310585926382718),\n                        (5, 0.4901053274689139, 0.0, 0.12401520461552018),\n                        (13, 12),\n                        (9, 14),\n                        (3, 0.5, 0.0, -0.5990933225703463),\n                        (6, 0.5, 0.0, 0.9430777170864317)],\n                       [(1, 0),\n                        (5, 2),\n                        (14, 0.5, 4.440892098500626e-16, 0.7902850909136756),\n                        (13, 0.22177219480665555, 0.0, -0.3131523226367805),\n                        (12, 0.8315920243625678, 1.1102230246251565e-16, -0.4157702878189209),\n                        (9, 0.33347826094116906, 0.0, 0.9968287677512606)],\n                       [(14, 13),\n                        (1, 0.5, 1.0, -0.7736945421001282),\n                        (5, 0.4999999999999997, 0.9999999999999999, 0.6232580546696456),\n                        (0, 0.30408672398469616, 0.0, -0.5000000000000002),\n                        (2, 0.30408672398469616, 0.0, -0.5000000000000002)],\n                       [(3, 1),\n                        (6, 5),\n                        (13, 0.5, -2.220446049250313e-16, 0.17388514996567728),\n                        (14, 0.5000000000000001, 0.9999999999999999, 0.7902850909136756)],\n                       [(1, 0.8040867239846965, 0.0, 0.22630545789987178),\n                        (5, 0.8040867239846965, 0.0, -0.37674194533035443),\n                        (13, 12),\n                        (9, 14),\n                        (3, 0.0, 0.2389431733857177, 0.0),\n                        (6, 0.0, -0.5340531328142406, 0.0)],\n                       [(1, 0),\n                        (5, 2),\n                        (14, 0.5519231616233389, 1.6653345369377348e-16, 0.00808140744530833),\n                        (13, 0.5, 0.0, 0.17388514996567667),\n                        (12, 0.5, 0.0, 0.46178536847925344),\n                        (9, 0.5276550549544043, 0.0, 0.9968287677512606)],\n                       [(1, 0.7499999999999999, 5.551115123125783e-17, 0.22630545789987172),\n                        (5, 0.75, 0.0, -0.37674194533035443),\n                        (14, 13),\n                        (0, 0.5, 0.0, 0.5),\n                        (2, 0.5, 0.0, 0.5)],\n                       [(3, 1),\n                        (6, 5),\n                        (13, 0.5, 0.0, 0.8744064878335265),\n                        (14, 0.5, 1.0, -0.6260653464602859)],\n                       [(1, 0.5, 0.0, -0.7736945421001282),\n                        (5, 0.5, 0.0, 0.6232580546696456),\n                        (13, 12),\n                        (9, 14),\n                        (3, 0.5, 0.0, -0.592240271534491),\n                        (6, 0.5, 2.220446049250313e-16, -0.30837301052081045)],\n                       [(1, 0),\n                        (5, 2),\n                        (12, 0.5, 0.0, -0.038214631520746334),\n                        (9, 0.5, 0.0, -0.5031712322487385),\n                        (14, 0.5, -0.9999999999999998, 0.37393465353971395)],\n                       [(1, 0.25000000000000006, 5.551115123125783e-17, 0.22630545789987166),\n                        (5, 0.25000000000000006, 0.0, -0.37674194533035443),\n                        (13, 12),\n                        (9, 14)],\n                       [(3, 1),\n                        (6, 5),\n                        (12, 0.5068340379253959, 0.0, -0.5310585926382718),\n                        (9, 0.490105327468914, 0.0, -0.5031947645320511),\n                        (13, 0.5, 0.0, 0.3744064878335269),\n                        (14, 0.5, -0.9999999999999998, 0.37393465353971395)],\n                       [(12, 11),\n                        (1, 0.5, 1.0, -0.7736945421001282),\n                        (5, 0.5, 0.9999999999999999, 0.6232580546696456),\n                        (9, 10),\n                        (3, 0.5, -0.9077597284655088, -0.592240271534491),\n                        (6, 0.5, 0.8083730105208101, -0.30837301052081)],\n                       [(1, 0),\n                        (5, 2),\n                        (12, 0.5, 0.0, -0.7736945421001282),\n                        (9, 0.5, 0.0, 0.9975623854138236),\n                        (11, 0.30408672398469616, 0.0, -0.5000000000000002),\n                        (10, 0.3040867239846962, 1.0, -0.5000000000000002)],\n                       [(13, 12),\n                        (14, 9),\n                        (0, 1.0, 0.0, 0.5),\n                        (1, 0.5, 0.2736945421001282, 0.22630545789987178),\n                        (2, 1.0, 0.0, 0.5),\n                        (5, 0.5, 0.8767419453303544, -0.37674194533035443)],\n                       [(12, 0.8040867239846965, 0.0, 0.22630545789987178),\n                        (9, 0.8040867239846965, 0.0, -0.0024376145861761955)],\n                       [(12, 11), (9, 10)],\n                       [(9, 0.75, 0.0, -0.0024376145861761955),\n                        (12, 0.75, 1.0, 0.22630545789987166),\n                        (11, 0.5, 0.0, 0.5),\n                        (10, 0.5, 1.0, 0.5)],\n                       [(14, 9), (13, 12)],\n                       [(12, 0.5, 0.0, -0.7736945421001282),\n                        (9, 0.5, 0.0, -0.0024376145861770837),\n                        (14, 0.5, 1.1102230246251565e-16, 0.0913322483327147),\n                        (13, 0.5, 0.0, 0.6064966101833891)],\n                       [(12, 11), (9, 10)],\n                       [(9, 0.25000000000000006, 0.0, 0.9975623854138225),\n                        (12, 0.25000000000000006, 1.0, 0.22630545789987178)],\n                       [(14, 9), (13, 12)],\n                       [(9, 0.5000000000000001, 0.0, 0.9975623854138225),\n                        (12, 0.5, 1.0, -0.7736945421001282),\n                        (13, 0.5, -0.10649661018338907, -0.3935033898166107),\n                        (14, 0.5, -0.5913322483327141, 0.0913322483327147)],\n                       [(9, 10), (12, 11)],\n                       [(10, 1.0, 0.0, 0.0),\n                        (11, 1.0, 0.0, 0.5),\n                        (12, 0.5, 0.2736945421001282, 0.22630545789987178),\n                        (9, 0.4999999999999998, -0.4975623854138229, -0.0024376145861770837)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],\n           \'qubits\': [(1, 6),\n                      (2, 6),\n                      (3, 5),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 3),\n                      (6, 4),\n                      (6, 5),\n                      (6, 6)]},\n (15, 5): {\'method\': \'reachable-support count kernels + native scheduling\',\n           \'moments\': [[(3, 0.45753902280850967, 0.0, -0.5),\n                        (8, 0.14099072666039333, 0.0, -0.5000000000000002),\n                        (5, 0.5005458189976442, 0.0, 0.5),\n                        (4, 0.5025848644811362, 0.0, 0.5),\n                        (1, 0.5630190521715605, 0.0, 0.5),\n                        (9, 0.5, 0.0, -0.24745148962808194),\n                        (6, 0.4999999999999998, 0.0, -0.721588825990928),\n                        (13, 0.4999999999999998, 0.0, -0.721588825990928),\n                        (2, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (12, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (0, 0.5, 0.0, 0.5),\n                        (10, 0.30408672398469655, 0.0, -0.5745192213521528),\n                        (14, 0.30408672398469655, 0.0, -0.5745192213521528),\n                        (11, 0.30408672398469655, 0.0, -0.5745192213521528)],\n                       [(5, 4), (1, 3), (7, 8)],\n                       [(7, 0.4191369997162821, 0.0, -0.5000000000000002),\n                        (3, 0.6905771906117342, 0.0, 0.5),\n                        (8, 0.14099072666039333, 0.0, 0.4999999999999998),\n                        (4, 0.9017154857987535, 0.0, -0.5000000000000002),\n                        (5, 0.2527148691845798, 0.0, -0.5),\n                        (1, 0.6863539888389778, 0.0, 0.5)],\n                       [(4, 8), (3, 7)],\n                       [(3, 0.5, 0.0, 0.5),\n                        (4, 0.6544375463289773, 0.0, 0.5),\n                        (7, 0.38291144507086683, 0.0, 0.5)],\n                       [(3, 4), (7, 8)],\n                       [(3, 0.4999999999999998, 0.0, 0.5), (5, 4), (7, 0.9957692972812717, 0.0, 0.5)],\n                       [(1, 3), (4, 0.2947640447824702, 0.0, 0.5), (5, 0.998577127886374, 0.0, -0.5)],\n                       [(3, 0.7483385199514005, 0.0, -0.5), (4, 8), (1, 0.5, 0.0, -0.5)],\n                       [(3, 7), (4, 0.79476404478247, 0.0, -0.5), (8, 0.5, 0.0, -0.5)],\n                       [(7, 0.4838771221091833, 0.0, -0.5000000000000002), (3, 0.3186895926607129, 0.0, 0.5)],\n                       [(3, 4), (7, 8)],\n                       [(4, 0.5, 0.0, -0.5), (8, 0.5, 0.0, 0.5), (3, 0.5, 0.0, -0.5), (7, 0.5, 0.0, 0.5)],\n                       [(5, 4), (8, 9), (1, 3)],\n                       [(4, 8), (5, 0.5, 0.0, 0.38473595982107583), (3, 0.5, 0.0, 0.5)],\n                       [(4, 0.5, 0.0, -0.5838295986345305),\n                        (8, 0.2521698605391784, 0.0, 0.1676591972690611),\n                        (3, 7)],\n                       [(5, 4),\n                        (7, 0.5129613241629847, 0.0, 0.0),\n                        (3, 0.6083765495574773, 0.0, 0.4935193379185076)],\n                       [(5, 0.5, 0.0, -0.6152640401789242), (4, 8), (7, 6), (1, 3)],\n                       [(9, 5),\n                        (8, 0.5557004579686877, 4.440892098500626e-16, 0.6676591972690606),\n                        (7, 0.7648228951691718, 0.0, 0.2744861499108664),\n                        (3, 0.8568537707944928, 0.0, 0.49351933791850744),\n                        (6, 0.5225448598685903, 0.0, 0.8911869797251353),\n                        (1, 0.9984061930639809, 0.0, -0.03321080761007478)],\n                       [(9, 0.22574967231825369, 0.0, 0.33637364812151094),\n                        (5, 0.5, 0.0, 0.08740348058851666),\n                        (3, 7)],\n                       [(5, 4),\n                        (8, 9),\n                        (3, 0.5, 0.0, -0.5064806620814932),\n                        (7, 0.930805297572066, 0.0, 0.6639281764160383)],\n                       [(9, 0.49942055111844974, 0.0, -0.1481155227947728),\n                        (8, 0.993401101423752, 0.0, 0.16765919726906503),\n                        (4, 0.9540075367795029, 2.220446049250313e-16, -0.3562281098137259),\n                        (5, 0.5, 0.0, 0.08740348058851666),\n                        (3, 7)],\n                       [(4, 8),\n                        (9, 5),\n                        (7, 0.4999999999999998, 0.0, 0.8256402326697216),\n                        (3, 0.5908115752601095, 0.0, -0.5981062978263378)],\n                       [(9, 0.42607382518378495, 0.0, -0.4708952936330244),\n                        (8, 0.5808813702856532, 0.0, 0.6676591972690682),\n                        (4, 0.25443599797049227, 0.0, -0.33177874324135015),\n                        (7, 6),\n                        (1, 3)],\n                       [(8, 9),\n                        (5, 4),\n                        (7, 0.20245391911064137, 0.0, -0.17435976733027747),\n                        (6, 0.7201732888048016, 0.0, -0.7118510178536865),\n                        (1, 3)],\n                       [(9, 0.7250211832941115, 0.0, -0.6495276285185723),\n                        (4, 0.7140608751963855, 0.0, -0.5475955962845607),\n                        (8, 0.4515486953156751, 0.0, -0.8323408027309318),\n                        (3, 7),\n                        (1, 0.001593806936018405, 0.0, -0.03321080761024264)],\n                       [(9, 5),\n                        (4, 8),\n                        (7, 0.6373873139241069, 0.0, 0.3256402326697221),\n                        (3, 0.4054888125911002, 0.0, -0.21622021432458727)],\n                       [(8, 0.4314501661314405, 0.0, -0.3323408027309318),\n                        (9, 0.7287196312679844, 8.881784197001252e-16, -0.6442015812643025),\n                        (4, 0.5448381862924326, 4.440892098500626e-16, 0.5207924774217445),\n                        (7, 6)],\n                       [(5, 4),\n                        (8, 9),\n                        (7, 0.5, 0.0, 0.3256402326697221),\n                        (6, 0.5503418120493246, 0.0, -0.1702512394470519)],\n                       [(8, 0.41129107740972154, 0.0, 0.16765919726906814),\n                        (9, 0.9881887823489937, 0.0, 0.10165048895054601),\n                        (4, 0.683530177649277, 0.0, -0.2084549484777245),\n                        (3, 7)],\n                       [(9, 5),\n                        (4, 8),\n                        (7, 0.5724478383345383, 0.0, 0.7411264290386106),\n                        (3, 0.5000000000000004, 0.0, -0.3598832821522451)],\n                       [(4, 0.15105970966010432, 0.0, 0.35819208064544217),\n                        (9, 0.06691772293018138, 0.0, -0.39466187129291863),\n                        (8, 0.0006913190085300335, 0.0, -0.33234080273087807),\n                        (7, 6),\n                        (1, 3)],\n                       [(8, 9),\n                        (5, 4),\n                        (7, 0.5, 0.0, -0.43629066049085674),\n                        (3, 0.7849738353917708, 0.0, -0.8598832821522464),\n                        (6, 0.4999999999999977, 1.1102230246251565e-16, 0.3151950234033264),\n                        (1, 0.5, 0.0, -0.06318128417940283)],\n                       [(8, 0.5, 0.0, 0.6676591972690682),\n                        (4, 0.6991057269115482, 8.881784197001252e-16, 0.5859131698253741),\n                        (9, 0.10622873606974292, 0.0, -0.41042381808958384),\n                        (3, 7)],\n                       [(9, 5),\n                        (4, 8),\n                        (7, 0.07810728535625011, 0.0, 0.06370933950914281),\n                        (3, 0.5517093529747915, 0.0, 0.14011671784775087)],\n                       [(8, 0.8378188644666402, 0.0, -0.8230322509091135),\n                        (9, 0.2924608216157978, 0.0, -0.7366372632486016),\n                        (4, 0.5, 0.0, -0.5667391281695942),\n                        (5, 0.5, 4.440892098500626e-16, -0.8197753589076715),\n                        (7, 6),\n                        (1, 3)],\n                       [(8, 9),\n                        (5, 4),\n                        (7, 0.5510400458124618, 0.0, -0.2573280171188497),\n                        (6, 0.2219727068374309, 0.0, -0.2540214619058512),\n                        (1, 0.4999999999999995, -0.9400590468613435, -0.06318128417940261)],\n                       [(8, 0.1322665856888474, 0.0, 0.49634406008387083),\n                        (9, 0.8453021268939236, 0.0, 0.16769673569719984),\n                        (3, 7),\n                        (6, 2),\n                        (5, 0.5, 0.5278605595904069, -0.819775358907672)],\n                       [(4, 8),\n                        (9, 13),\n                        (7, 0.4999999999999993, 0.0, -0.8363269582669735),\n                        (2, 0.85928593962011, 0.0, -0.9262332254895425),\n                        (6, 0.49999999999999994, 0.0, -0.7540214619058516)],\n                       [(8, 0.8568537707944928, 0.0, -0.3003867606651536),\n                        (9, 0.7648228951691718, 0.0, -0.1067894142136665),\n                        (13, 0.5225448598685906, 0.0, -0.10881302027486428),\n                        (4, 0.9984061930639809, 3.3306690738754696e-16, -0.32544311987678665),\n                        (3, 7)],\n                       [(8, 9),\n                        (7, 0.2978377012021855, 0.0, 0.20055790003054064),\n                        (3, 0.5, 0.0, -0.5561159741720587)],\n                       [(8, 0.5, 0.0, -0.3003867606651536),\n                        (9, 0.930805297572066, 0.0, 0.5037685592811609),\n                        (3, 7)],\n                       [(8, 9), (3, 0.5, 0.0, 0.4438840258279413), (7, 6)],\n                       [(9, 0.5, 0.0, -0.6579434969725222),\n                        (8, 0.5908115752601092, 0.0, 0.7912388750796917),\n                        (2, 3),\n                        (6, 0.6714165234362537, 0.0, -0.6826053701748582),\n                        (7, 0.5, 0.0, 0.1496690081401122)],\n                       [(9, 13), (4, 8), (2, 0.7485956033393342, 0.0, 0.5737667745104573), (3, 7)],\n                       [(9, 0.20245391911064134, 0.0, 0.34205650302747737),\n                        (13, 0.7201732888048016, 0.0, -0.7118510178536865),\n                        (4, 8),\n                        (6, 2),\n                        (3, 0.0528060550044127, 0.0, 0.2702836820409704),\n                        (7, 0.5, -2.220446049250313e-16, -0.8503309918598883)],\n                       [(8, 9),\n                        (4, 0.0015938069360184048, 0.0, -0.32544311987661834),\n                        (6, 0.3935294720438233, 0.0, -0.4035582545612375),\n                        (2, 0.9961572718865779, 0.0, 0.07376677451045216)],\n                       [(9, 0.6373873139241069, 0.0, 0.8420565030274789),\n                        (8, 0.4054888125911004, 0.0, -0.5906472084220589),\n                        (7, 6),\n                        (2, 3)],\n                       [(9, 13),\n                        (6, 0.3104369525321425, 0.0, -0.8220795049614371),\n                        (3, 0.24963665794604523, 0.0, -0.24707890286880363),\n                        (2, 0.5655532211464794, 0.0, 0.5737667745104469)],\n                       [(9, 0.5, 0.0, 0.8420565030274789),\n                        (13, 0.5503418120493246, 0.0, 0.8297487605529479),\n                        (3, 7),\n                        (6, 2)],\n                       [(8, 9),\n                        (3, 0.3270115623295809, 0.0, 0.14514073841929592),\n                        (6, 0.6193705712509703, 0.0, 0.050886184139158075),\n                        (2, 0.4999999999999998, 0.0, 0.5737667745104462)],\n                       [(8, 0.5000000000000004, 4.440892098500626e-16, 0.5530158594055989),\n                        (9, 0.5724478383345383, 0.0, -0.5734296933414109),\n                        (7, 6),\n                        (2, 3)],\n                       [(9, 13),\n                        (4, 8),\n                        (3, 0.5, 0.0, 0.8977762069404194),\n                        (2, 0.5960614607313279, 0.0, 0.8831564130503458),\n                        (6, 0.6256105257587086, 0.0, -0.46584344785121257)],\n                       [(9, 0.5, 0.0, 0.6039873961880566),\n                        (8, 0.7849738353917708, -2.220446049250313e-16, 0.05301585940560122),\n                        (13, 0.4999999999999977, 1.1102230246251565e-16, 0.3151950234033264),\n                        (4, 0.5, 0.0, 0.7045273566925405),\n                        (7, 6),\n                        (2, 3)],\n                       [(8, 9),\n                        (3, 0.5, 0.0, 0.3977762069404212),\n                        (6, 0.9764747842833552, 0.0, -0.8680592057718204),\n                        (2, 0.14949404734505523, 0.0, -0.7269328881171262),\n                        (7, 0.5000000000000002, 0.0, 0.8285975788326296)],\n                       [(9, 0.07810728535625011, 0.0, -0.8960126038119429),\n                        (8, 0.5517093529747914, 0.0, 0.05301585940560111),\n                        (3, 7),\n                        (6, 2)],\n                       [(9, 13),\n                        (4, 8),\n                        (6, 0.5, 0.0, -0.6250809225899461),\n                        (7, 0.5, 0.0, -0.17140242116737037),\n                        (2, 3)],\n                       [(9, 0.5510400458124618, 0.0, -0.5749752471839495),\n                        (13, 0.2219727068374309, 0.0, 0.7459785380941486),\n                        (7, 6),\n                        (2, 0.42477613785575913, 0.0, 0.9776181484214663),\n                        (4, 0.5000000000000003, -0.7077676877332868, 0.7045273566925407)],\n                       [(8, 9),\n                        (13, 12),\n                        (7, 0.5, 0.0, -0.1042135671487332),\n                        (6, 0.1473187127029063, 0.0, 0.3749190774100535)],\n                       [(9, 0.4999999999999994, 0.0, -0.9959763060358278),\n                        (13, 0.5, 0.0, 0.24597853809414882),\n                        (12, 0.85928593962011, 0.0, -0.9262332254895425),\n                        (6, 2),\n                        (3, 7)],\n                       [(8, 9),\n                        (2, 0.4500869528994624, 0.0, 0.6634633138266381),\n                        (3, 0.42020075164346243, 0.0, 0.6912138485319217),\n                        (6, 0.9433458052208397, 0.0, 0.8749190774100513)],\n                       [(9, 0.2978377012021855, 0.0, 0.9671388356666601),\n                        (8, 0.49999999999999983, 0.0, -0.2507514485745881),\n                        (2, 3),\n                        (7, 6)],\n                       [(8, 9),\n                        (2, 0.5410889047360808, -2.220446049250313e-16, 0.21035484945072325),\n                        (3, 0.5, 0.0, 0.2864097336926479),\n                        (7, 0.20567003520391755, 0.0, -0.6042135671487336)],\n                       [(8, 0.5, 2.220446049250313e-16, -0.2507514485745883), (9, 13), (2, 0), (7, 3)],\n                       [(12, 8),\n                        (13, 0.6714165234362537, 0.0, -0.6826053701748587),\n                        (9, 0.5, -2.220446049250313e-16, 0.018027727557087037),\n                        (2, 0.5, 0.0, -0.28964515054927675),\n                        (7, 0.49999999999998224, 4.440892098500626e-16, 0.8957864891692073),\n                        (0, 0.3040867239846962, 0.0, 0.4999999999999998),\n                        (3, 0.5000000000000022, 0.0, -0.8022501958995181)],\n                       [(12, 0.7485956033393342, 0.0, -0.42623322548954246), (8, 9), (6, 2), (7, 11)],\n                       [(13, 12),\n                        (9, 0.5, -2.220446049250313e-16, 0.018027727557087037),\n                        (8, 0.052806055004412704, 0.0, 0.9228488952123819),\n                        (2, 0.8040867239846965, 4.440892098500626e-16, 0.710354849450723),\n                        (7, 0.5000000000000102, 4.440892098500626e-16, 0.8957864891692073),\n                        (11, 0.30408672398469627, 0.0, -0.5745192213521484)],\n                       [(13, 0.3935294720438234, 0.0, -0.4035582545612373),\n                        (12, 0.9961572718865779, 0.0, -0.9262332254895478),\n                        (2, 0),\n                        (7, 3)],\n                       [(12, 8),\n                        (9, 13),\n                        (2, 0.7499999999999999, 0.0, 0.710354849450723),\n                        (7, 0.75, 4.440892098500626e-16, 0.8957865216843865),\n                        (0, 0.5, 0.0, 0.5),\n                        (3, 0.49999999999999967, 0.0, -0.8022502284146997)],\n                       [(13, 0.3104369525321424, 0.0, 0.17792049503856378),\n                        (8, 0.24963665794604517, 0.0, -0.5597885198778432),\n                        (12, 0.5655532211464794, 0.0, 0.5737667745104469),\n                        (6, 2),\n                        (7, 11)],\n                       [(8, 9),\n                        (13, 12),\n                        (2, 0.5, 0.0, 0.710354849450723),\n                        (7, 0.500526889421938, 0.0, 0.3963134578115506),\n                        (6, 0.5, 0.0, -0.604536470221908),\n                        (11, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155)],\n                       [(8, 0.3270115623295809, 0.0, 0.0479918388340545),\n                        (13, 0.6193705712509703, 0.0, -0.9491138158608414),\n                        (12, 0.4999999999999998, 0.0, 0.5737667745104462),\n                        (2, 0),\n                        (7, 3)],\n                       [(12, 8),\n                        (9, 13),\n                        (2, 0.25, 0.0, -0.28964515054927675),\n                        (3, 0.5000000000000004, 0.0, 0.6977497715852994),\n                        (7, 0.5, 0.0, -0.8536861061150169)],\n                       [(13, 0.6256105257587086, 0.0, -0.46584344785121257),\n                        (12, 0.5960614607313279, -2.220446049250313e-16, -0.11684358694965402),\n                        (8, 0.5, 0.0, -0.7046436296870688),\n                        (6, 2),\n                        (7, 3)],\n                       [(12, 8),\n                        (9, 13),\n                        (2, 0.5, 0.0, -0.28964515054927675),\n                        (7, 11),\n                        (3, 0.0007451670218237008, -0.9239642591572572, 0.197749771585308),\n                        (6, 0.5, 0.1045364702219076, -0.6045364702219087)],\n                       [(12, 0.14949404734505523, 0.0, -0.7269328881171262),\n                        (9, 0.5, 0.0, -0.66090084313543),\n                        (8, 0.5, 0.0, -0.20464362968706906),\n                        (13, 0.9764747842833552, -2.220446049250313e-16, 0.13194079422817917),\n                        (2, 0),\n                        (7, 0.999999967484822, 0.12747157321208258, 0.14631391655047077),\n                        (11, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(13, 12),\n                        (8, 9),\n                        (2, 0.4999999999999998, 0.7896451505492773, 0.7103548494507228),\n                        (0, 1.0, 0.0, 0.5)],\n                       [(13, 0.5, 0.0, 0.3749190774100535), (9, 0.5, 0.0, -0.66090084313543), (12, 8)],\n                       [(9, 13), (12, 0.4247761378557593, 0.0, -0.022381851578533896)],\n                       [(13, 0.1473187127029063, 0.0, 0.3749190774100535),\n                        (9, 0.5, 0.0, -0.7280896971540678)],\n                       [(13, 12), (8, 9)],\n                       [(8, 0.4202007516434625, 0.0, -0.4980812712785716),\n                        (12, 0.45008695289946266, 0.0, -0.3365366861733614),\n                        (13, 0.9433458052208397, 0.0, 0.8749190774100519)],\n                       [(9, 13), (12, 8)],\n                       [(8, 0.5, 0.0, -0.09327715643929757),\n                        (9, 0.20567003520391733, 0.0, 0.7719103028459318),\n                        (13, 0.5, 0.0, 0.6692490422061348),\n                        (12, 0.5, 0.0, -0.2896451505492772)],\n                       [(13, 12), (9, 8)],\n                       [(9, 0.49999999999998246, 0.0, 0.27191024652799123),\n                        (13, 0.4999999999999824, 0.0, -0.330750901475926),\n                        (8, 0.500000000000002, 0.0, -0.004617226847131617),\n                        (12, 0.5000000000000022, 0.0, 0.6216949198585577)],\n                       [(9, 10), (13, 14)],\n                       [(13, 0.5000000000000102, 0.0, -0.3307509014759247),\n                        (9, 0.5000000000000104, 0.0, 0.27191024652799145),\n                        (10, 0.3040867239846962, 0.0, 0.4254807786478514),\n                        (14, 0.30408672398469627, 0.0, -0.5745192213521484)],\n                       [(9, 8), (13, 12)],\n                       [(9, 0.75, 0.0, -0.7280897859871851),\n                        (13, 0.75, 0.0, -0.33075086896074435),\n                        (8, 0.49999999999999956, 0.0, 0.99538280566805),\n                        (12, 0.49999999999999956, 0.0, 0.6216948873433754)],\n                       [(13, 14), (9, 10)],\n                       [(13, 0.500526889421938, 0.0, -0.830223932833581),\n                        (9, 0.5005268894219381, 0.0, -0.22861672211434914),\n                        (14, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155),\n                        (10, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155)],\n                       [(9, 8), (13, 12)],\n                       [(8, 0.5000000000000004, 0.0, 0.49538280566805093),\n                        (12, 0.5000000000000003, -2.220446049250313e-16, 0.12169488734337541),\n                        (9, 0.5, 0.0, 0.021382841812215503),\n                        (13, 0.5, 0.0, -0.08022349676014806)],\n                       [(9, 8), (13, 12)],\n                       [(9, 10),\n                        (13, 14),\n                        (12, 0.0007451670218237004, -0.34790937491533347, -0.37830511265661615),\n                        (8, 0.9992548329781763, -0.7215972932399906, 0.9953828056680414)],\n                       [(9, 3.2515178327585665e-08, 0.2524026706158251, -0.9786171808532709),\n                        (10, 0.5, 0.3483047662953691, -0.5745192538673274),\n                        (13, 0.999999967484822, -0.6459910361427852, 0.9197765259053394),\n                        (14, 0.5, -0.6516952337046309, 0.42548074613267284)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8),\n                      (6, 5),\n                      (6, 6),\n                      (6, 7),\n                      (6, 8)]},\n (15, 6): {\'method\': \'reachable-support count kernels + native scheduling\',\n           \'moments\': [[(4, 0.7373539681892943, 0.0, 0.5),\n                        (8, 0.44676443056282406, 0.0, -0.5000000000000002),\n                        (5, 0.1591097040601366, 0.0, 0.5),\n                        (1, 0.9011168519071704, 0.0, -0.5),\n                        (7, 0.7085901148744962, 0.0, 0.5),\n                        (9, 0.5, 0.0, -0.24745148962808194),\n                        (6, 0.4768011512951635, 0.0, -0.17578877400173032),\n                        (13, 0.4768011512951635, 0.0, -0.17578877400173032),\n                        (2, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (12, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (0, 0.5, 0.0, 0.5),\n                        (10, 0.30408672398469655, 0.0, -0.5745192213521528),\n                        (14, 0.30408672398469655, 0.0, -0.5745192213521528),\n                        (11, 0.30408672398469655, 0.0, -0.5745192213521528)],\n                       [(5, 4), (1, 3), (7, 8)],\n                       [(7, 0.5443536300607281, 0.0, -0.5),\n                        (3, 0.6137946531114089, 0.0, 0.5),\n                        (8, 0.21081011765170987, 0.0, -0.5),\n                        (4, 0.7581079035004981, 0.0, -0.5),\n                        (5, 0.38705144949570247, 0.0, -0.5),\n                        (1, 0.40111685190717034, 0.0, 0.4999999999999998)],\n                       [(4, 8), (3, 7)],\n                       [(3, 0.6609494081989717, 0.0, 0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (7, 0.5102068102350223, 0.0, -0.5),\n                        (8, 0.2523672686524676, 0.0, 0.5)],\n                       [(3, 4), (7, 8)],\n                       [(4, 0.5, 0.0, -0.5),\n                        (1, 3),\n                        (8, 0.9456160071264607, 0.0, 0.5),\n                        (7, 0.390303024045044, 0.0, -0.5)],\n                       [(5, 4),\n                        (3, 0.17564872714486615, 0.0, 0.5),\n                        (1, 0.4999999999999998, 0.0, -0.5000000000000002)],\n                       [(4, 0.4215716033272554, 0.0, -0.5), (3, 7), (5, 0.5, 0.0, 0.5)],\n                       [(4, 8), (3, 0.16336727963993836, 0.0, 0.5), (7, 0.5630139738389508, 0.0, 0.5)],\n                       [(4, 0.8306312090125401, 0.0, -0.5), (8, 0.5189686705552345, 0.0, -0.5)],\n                       [(3, 4), (7, 8)],\n                       [(3, 0.8464209044699778, 0.0, 0.5),\n                        (4, 0.31756513752949056, 0.0, 0.5),\n                        (8, 0.812464019304552, 0.0, 0.5),\n                        (7, 0.8097241829115225, 0.0, -0.5)],\n                       [(1, 3), (5, 4)],\n                       [(3, 0.8879545911970581, 0.0, 0.5), (4, 0.9100820590738672, 0.0, 0.5)],\n                       [(3, 7), (4, 8)],\n                       [(4, 0.10550946076945021, 0.0, 0.4999999999999998),\n                        (3, 0.939084250472354, 0.0, 0.5),\n                        (8, 0.16188927506338208, 0.0, 0.5),\n                        (7, 0.30815933398738005, 0.0, 0.5)],\n                       [(3, 4), (7, 8)],\n                       [(4, 0.5010152913918939, 0.0, -0.5),\n                        (8, 0.4963987267193131, 0.0, -0.5),\n                        (3, 0.513539249942093, 0.0, -0.5),\n                        (7, 0.5273673081805759, 0.0, 0.5)],\n                       [(5, 4), (1, 3)],\n                       [(4, 0.9229270144537076, 0.0, 0.5),\n                        (5, 0.5, 0.0, -0.384735959821076),\n                        (3, 0.07826489477032926, 0.0, -0.5000000000000002),\n                        (1, 0.4032806175354167, 0.0, -0.6140099095404981)],\n                       [(4, 8), (3, 7)],\n                       [(8, 0.9113836171241108, 0.0, 0.5),\n                        (4, 0.8249578486829954, 0.0, 0.6659771145786124),\n                        (7, 0.4989731297377558, 0.0, 0.8475312279967941),\n                        (3, 0.9181726713260874, 0.0, 0.8884776335740554)],\n                       [(8, 9), (5, 4), (7, 6), (1, 3)],\n                       [(5, 0.5, 0.0, 0.6152640401789242),\n                        (8, 0.2521698605391784, 0.0, -0.1676591972690611),\n                        (7, 0.16399934185718348, 0.0, 0.8329407363157213),\n                        (1, 0.7174704187311103, 0.0, 0.8859900904595017),\n                        (6, 0.6279799493860765, 0.0, -0.770287955286147)],\n                       [(9, 5), (4, 8), (3, 7)],\n                       [(5, 0.5, 0.0, -0.6820684390536353),\n                        (9, 0.22574967231825369, 0.0, -0.6636263518784891),\n                        (8, 0.5557004579686877, 0.0, 0.3323408027309387),\n                        (3, 0.4558467315798988, 0.0, 0.2681859035801324),\n                        (7, 0.2771547609074463, 0.0, -0.4809149437700333)],\n                       [(5, 4), (8, 9), (1, 3), (7, 6)],\n                       [(5, 0.5, 0.0, 0.31793156094636504),\n                        (8, 0.993401101423752, 0.0, 0.8323408027309351),\n                        (4, 0.9540075367795029, 0.0, -0.4166557838885545),\n                        (9, 0.49942055111844974, 0.0, -0.1481155227947728),\n                        (6, 0.019663643370415814, 4.440892098500626e-16, 0.5185052910680361),\n                        (3, 0.4233573560778924, 0.0, 0.7681859035801324),\n                        (7, 0.6057423835093776, 0.0, 0.5084815974320014),\n                        (1, 0.19408134503630478, 0.0, -0.6140099095404976)],\n                       [(9, 5), (4, 8), (1, 3), (7, 6)],\n                       [(8, 0.5808813702856532, 0.0, -0.6676591972690682),\n                        (9, 0.42607382518378495, 0.0, -0.4708952936330244),\n                        (4, 0.25443599797049243, 0.0, 0.5588948495390698),\n                        (3, 0.7065221727328496, 0.0, 0.7962137387684829),\n                        (7, 0.776375637814382, 0.0, 0.5155055368170409),\n                        (6, 0.5680691667523844, 0.0, -0.6374006217137276),\n                        (1, 0.5740015407073948, 0.0, -0.11400990954049783)],\n                       [(8, 9), (5, 4), (3, 7)],\n                       [(4, 0.7140608751963855, 0.0, 0.7747117025822803),\n                        (9, 0.7250211832941114, 0.0, 0.3504723714814277),\n                        (8, 0.4515486953156751, 0.0, 0.832340802730932),\n                        (7, 0.7960253457838964, 8.881784197001252e-16, 0.5456685357810218),\n                        (3, 0.7550526236278825, -2.220446049250313e-16, 0.3043640842888422)],\n                       [(4, 8), (9, 5), (1, 3), (7, 6)],\n                       [(8, 0.4314501661314405, 0.0, -0.667659197269068),\n                        (4, 0.5448381862924326, 0.0, 0.7063236288759742),\n                        (9, 0.7287196312679844, 0.0, 0.35579841873569795),\n                        (3, 0.093170508709036, 0.0, 0.43504716694353274),\n                        (7, 0.5920891677547482, 0.0, -0.9543314642189782),\n                        (6, 0.5767646373955954, 0.0, -0.7141291884531249)],\n                       [(5, 4), (8, 9), (3, 7)],\n                       [(4, 0.6835301776492771, 0.0, -0.5644289452245554),\n                        (8, 0.4112910774097216, 0.0, 0.8323408027309318),\n                        (9, 0.9881887823489937, 0.0, 0.1016504889505454),\n                        (3, 0.8490154235516789, 0.0, -0.3560621281122216),\n                        (7, 0.8905211577257011, 0.0, -0.45433146421897774)],\n                       [(9, 5), (4, 8), (1, 3), (7, 6)],\n                       [(9, 0.06691772293018138, 0.0, -0.39466187129291863),\n                        (8, 0.0006913190085300335, 0.0, 0.33234080273087235),\n                        (4, 0.1510597096601043, 0.0, -0.13107597434772233),\n                        (3, 0.5525232416584167, 4.440892098500626e-16, 0.5823431844086724),\n                        (7, 0.623464211234003, 8.881784197001252e-16, -0.954331464218978),\n                        (6, 0.22565786477821326, 0.0, 0.9040830152802759)],\n                       [(8, 9), (5, 4), (3, 7)],\n                       [(4, 0.6991057269115482, 0.0, -0.3587970635276547),\n                        (9, 0.10622873606974291, 0.0, 0.5895761819104166),\n                        (8, 0.5, 0.0, -0.667659197269068),\n                        (3, 0.2256027665011393, 0.0, -0.24788459100249738),\n                        (7, 0.19755687312566203, 0.0, -0.45433146421897774)],\n                       [(9, 5), (4, 8), (1, 3), (7, 6)],\n                       [(8, 0.8378188644666402, 0.0, -0.17696774909088653),\n                        (9, 0.2924608216157978, 0.0, 0.26336273675139843),\n                        (5, 0.5, 0.0, 0.4107527214501769),\n                        (4, 0.5, 0.0, -0.2061447655326849),\n                        (3, 0.27319568030875896, 4.440892098500626e-16, -0.9330056761674794),\n                        (7, 0.5, 0.0, -0.45433146421897774),\n                        (6, 0.08965970809102465, 0.0, 0.42015605383701876),\n                        (1, 0.5, 0.0, 0.2116228745031774)],\n                       [(8, 9), (5, 4), (3, 7)],\n                       [(9, 0.6802135982055907, 0.0, 0.1831314084107074),\n                        (8, 0.16058384962817573, 0.0, 0.8137469991417423),\n                        (4, 0.5967193824645834, 0.0, 0.19977994332507792),\n                        (7, 0.6586668818397199, 0.0, -0.048861597634449305),\n                        (3, 0.5, 0.0, -0.2975553052906692),\n                        (5, 0.5, 0.29733247923255846, -0.5892472785498235)],\n                       [(9, 13), (4, 8), (7, 6), (1, 3)],\n                       [(9, 0.16399934185718348, 0.0, -0.3463176265931116),\n                        (4, 0.7174704187311103, 0.0, 0.6997799433250782),\n                        (13, 0.6279799493860765, 0.0, -0.770287955286147),\n                        (7, 0.6178431557297261, 0.0, -0.052883489677677975),\n                        (3, 0.4999999999999998, 0.0, -0.7854054538480497),\n                        (6, 0.35737627229158253, 0.0, 0.47611746869056887),\n                        (1, 0.5, 0.576754250993645, 0.2116228745031774)],\n                       [(8, 9), (6, 2), (3, 7)],\n                       [(8, 0.45584673157989886, 0.0, -0.6915497725428073),\n                        (9, 0.2771547609074463, 5.551115123125783e-17, 0.3398266933211339),\n                        (2, 0.85928593962011, 0.0, -0.9262332254895425),\n                        (6, 0.5, 0.0, -0.71839631538329),\n                        (3, 0.5, 0.0, -0.7854054538480495)],\n                       [(4, 8), (9, 13), (2, 3), (7, 6)],\n                       [(13, 0.019663643370415814, 4.440892098500626e-16, 0.5185052910680361),\n                        (4, 0.19408134503630484, 0.0, -0.8002200566749229),\n                        (9, 0.6057423835093776, 0.0, -0.6707767654768322),\n                        (8, 0.4233573560778924, 0.0, -0.19154977254280725),\n                        (6, 0.6714165234362537, 5.551115123125783e-17, 0.35301977634770204),\n                        (2, 0.7485956033393342, 0.0, 0.5737667745104573),\n                        (7, 0.5000000000000001, 0.0, 0.6487651964452692)],\n                       [(4, 8), (9, 13), (6, 2), (3, 7)],\n                       [(8, 0.7065221727328495, 0.0, -0.16352193735445653),\n                        (9, 0.776375637814382, 0.0, -0.6637528260917924),\n                        (13, 0.5680691667523844, 0.0, -0.6374006217137276),\n                        (4, 0.574001540707395, 0.0, -0.300220056674922),\n                        (7, 0.5, 0.0, -0.351234803554731),\n                        (3, 0.0528060550044127, 0.0, 0.04099420236497986),\n                        (6, 0.3935294720438234, 0.0, 0.6320668919613246),\n                        (2, 0.9961572718865779, 0.0, 0.07376677451045216)],\n                       [(8, 9), (7, 6), (2, 3)],\n                       [(9, 0.7960253457838964, 0.0, -0.63358982712781),\n                        (8, 0.7550526236278825, 0.0, -0.6553715918340974),\n                        (6, 0.3104369525321425, 0.0, -0.7864543584388737),\n                        (2, 0.5655532211464794, 0.0, 0.5737667745104469),\n                        (3, 0.24963665794604523, 0.0, -0.4763683825447953)],\n                       [(9, 13), (4, 8), (6, 2), (3, 7)],\n                       [(8, 0.093170508709036, 0.0, -0.5246885091794071),\n                        (9, 0.5920891677547482, 0.0, -0.13358982712781042),\n                        (13, 0.5767646373955954, 0.0, -0.7141291884531249),\n                        (2, 0.4999999999999998, 0.0, 0.5737667745104462),\n                        (3, 0.32701156232958084, 0.0, 0.9158512587433063),\n                        (6, 0.6193705712509703, -2.220446049250313e-16, 0.0865113306617209)],\n                       [(8, 9), (2, 3), (7, 6)],\n                       [(8, 0.8490154235516789, 0.0, 0.6842021957648388),\n                        (9, 0.8905211577257011, 0.0, 0.3664101728721896),\n                        (2, 0.5960614607313279, 0.0, 0.8831564130503458),\n                        (3, 0.5, 0.0, -0.33151327273556985),\n                        (6, 0.6256105257587086, 8.881784197001252e-16, 0.5697816986713502)],\n                       [(9, 13), (4, 8), (7, 6), (2, 3)],\n                       [(9, 0.623464211234003, 1.1102230246251565e-16, -0.1335898271278113),\n                        (8, 0.5525232416584172, 0.0, -0.37739249171426703),\n                        (13, 0.22565786477821326, 0.0, 0.9040830152802759),\n                        (3, 0.5, 0.0, -0.8315132727355694),\n                        (2, 0.14949404734505523, 0.0, -0.7269328881171262),\n                        (6, 0.9764747842833552, 0.0, -0.8324340592492583),\n                        (7, 0.5, 0.0, -0.030163374247246466)],\n                       [(8, 9), (3, 7), (6, 2)],\n                       [(9, 0.19755687312566203, 0.0, 0.3664101728721898),\n                        (8, 0.2256027665011393, 0.0, 0.7923797328745632),\n                        (6, 0.5, 0.0, -0.589455776067384),\n                        (7, 0.5, 0.0, 0.9698366257527535),\n                        (2, 3)],\n                       [(4, 8), (9, 13), (7, 6), (2, 0.42477613785575913, 0.0, 0.9776181484214663)],\n                       [(8, 0.27319568030875896, 0.0, 0.10725864770958138),\n                        (9, 0.5, 5.551115123125783e-17, 0.3664101728721899),\n                        (13, 0.08965970809102465, 0.0, 0.42015605383701876),\n                        (4, 0.5, 0.0, 0.02541272736875294),\n                        (6, 0.1473187127029063, 0.0, -0.589455776067384),\n                        (7, 0.5, 0.0, 0.9026477717341155)],\n                       [(8, 9), (3, 7), (6, 2)],\n                       [(9, 0.6586668818397199, 0.0, 0.7718800394567182),\n                        (8, 0.5, 0.0, 0.7427090185863905),\n                        (2, 0.4500869528994624, 0.0, 0.6634633138266381),\n                        (3, 0.4202007516434625, 0.0, -0.5380756311440673),\n                        (6, 0.9433458052208397, 0.0, -0.0894557760673862)],\n                       [(4, 8), (9, 13), (2, 3), (7, 6)],\n                       [(8, 0.5, 0.0, 0.2548588700290102),\n                        (9, 0.6178431557297261, 0.0, 0.7678581474134882),\n                        (13, 0.35737627229158253, 0.0, 0.47611746869056887),\n                        (2, 0.5410889047360808, -2.220446049250313e-16, 0.21035484945072325),\n                        (7, 0.20567003520391755, 0.0, -0.5973522282658852),\n                        (3, 0.5, 0.0, -0.9428797459833418),\n                        (4, 0.5, 0.7629643981280698, 0.025412727368752608)],\n                       [(8, 9), (13, 12), (2, 0), (7, 3)],\n                       [(12, 0.85928593962011, 0.0, -0.9262332254895425),\n                        (8, 0.5, 1.1102230246251565e-16, 0.254858870029011),\n                        (13, 0.5, 0.0, -0.71839631538329),\n                        (2, 0.5, 0.0, -0.28964515054927675),\n                        (0, 0.3040867239846962, 0.0, 0.4999999999999998),\n                        (7, 0.49999999999998224, 0.0, -0.0973522845838255),\n                        (3, 0.5000000000000022, 0.0, 0.9684603244244923)],\n                       [(9, 13), (12, 8), (6, 2), (7, 11)],\n                       [(12, 0.7485956033393342, 0.0, 0.5737667745104573),\n                        (9, 0.5, 0.0, -0.5304931664635628),\n                        (13, 0.6714165234362537, 5.551115123125783e-17, 0.35301977634770204),\n                        (2, 0.8040867239846965, 4.440892098500626e-16, 0.710354849450723),\n                        (7, 0.5000000000000103, 0.0, -0.0973522845838255),\n                        (11, 0.3040867239846962, 0.0, 0.4254807786478514)],\n                       [(8, 9), (13, 12), (2, 0), (7, 3)],\n                       [(13, 0.3935294720438234, 0.0, 0.6320668919613246),\n                        (8, 0.05280605500441271, 0.0, -0.9187414737579596),\n                        (12, 0.9961572718865779, 0.0, 0.07376677451045216),\n                        (9, 0.5, 0.0, 0.46950683353643724),\n                        (2, 0.7499999999999999, 0.0, 0.710354849450723),\n                        (7, 0.75, 0.0, -0.09735231709900294),\n                        (0, 0.5, 0.0, 0.5),\n                        (3, 0.49999999999999956, 0.0, -0.031539708090688956)],\n                       [(9, 13), (12, 8), (6, 2), (7, 11)],\n                       [(12, 0.5655532211464794, 0.0, 0.5737667745104469),\n                        (13, 0.3104369525321425, 0.0, -0.7864543584388737),\n                        (8, 0.24963665794604512, 0.0, 0.563895941332266),\n                        (2, 0.5, 0.0, 0.710354849450723),\n                        (7, 0.500526889421938, 5.551115123125783e-17, 0.4021207467738319),\n                        (6, 0.5, 0.0, -0.5689113236993462),\n                        (11, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155)],\n                       [(8, 9), (13, 12), (2, 0), (7, 3)],\n                       [(12, 0.4999999999999998, 0.0, 0.5737667745104462),\n                        (13, 0.6193705712509703, -2.220446049250313e-16, 0.0865113306617209),\n                        (8, 0.3270115623295809, 0.0, -0.04388441737963289),\n                        (2, 0.25, 0.0, -0.28964515054927675),\n                        (3, 0.5000000000000003, 0.0, 0.46846029190931027),\n                        (7, 0.5, 0.0, -0.3478796892995999)],\n                       [(12, 8), (9, 13), (6, 2), (7, 3)],\n                       [(8, 0.5, 0.0, 0.7087510511414921),\n                        (13, 0.6256105257587086, 8.881784197001252e-16, 0.5697816986713502),\n                        (12, 0.5960614607313279, 0.0, 0.8831564130503458),\n                        (2, 0.5, 0.0, -0.28964515054927675),\n                        (7, 11),\n                        (6, 0.5, 0.06891132369934594, -0.5689113236993459),\n                        (3, 0.0007451670218237004, 0.305325220518732, 0.9684602919093186)],\n                       [(9, 13),\n                        (12, 8),\n                        (2, 0),\n                        (7, 3.251517832758566e-08, -0.37833479827235816, -0.34787971196508893),\n                        (11, 0.5, 0.3483047662953691, -0.5745192538673274)],\n                       [(9, 0.5, 0.0, 0.7905782628439184),\n                        (8, 0.5, 0.0, 0.20875105114149006),\n                        (13, 0.9764747842833552, 0.0, -0.8324340592492583),\n                        (12, 0.14949404734505523, 0.0, -0.7269328881171262),\n                        (0, 1.0, 0.0, 0.5),\n                        (2, 0.4999999999999998, 0.7896451505492773, 0.7103548494507228)],\n                       [(13, 12), (8, 9)],\n                       [(13, 0.5, 0.0, -0.589455776067384), (9, 0.5, 0.0, -0.2094217371560818), (12, 8)],\n                       [(9, 13), (12, 0.42477613785575913, 0.0, 0.9776181484214663)],\n                       [(9, 0.5, 0.0, -0.2766105911747161),\n                        (13, 0.1473187127029063, 0.0, -0.589455776067384)],\n                       [(13, 12), (8, 9)],\n                       [(12, 0.4500869528994624, 0.0, 0.6634633138266381),\n                        (13, 0.9433458052208397, 0.0, -0.0894557760673862),\n                        (8, 0.4202007516434625, 0.0, 0.5021886927329928)],\n                       [(9, 13), (12, 8)],\n                       [(12, 0.5, 0.0, -0.28964515054927675),\n                        (8, 0.5, 0.0, 0.09738457789371857),\n                        (13, 0.5, 0.0, -0.29512581127130266),\n                        (9, 0.20567003520391755, 5.551115123125783e-17, 0.2233894088252832)],\n                       [(9, 8), (13, 12)],\n                       [(13, 0.4999999999999824, 0.0, 0.7048742450466376),\n                        (9, 0.4999999999999824, 8.881784197001252e-16, 0.7233893525073412),\n                        (12, 0.5000000000000022, 0.0, 0.6216949198585571),\n                        (8, 0.500000000000002, -2.220446049250313e-16, 0.008724648301553728)],\n                       [(9, 10), (13, 14)],\n                       [(9, 0.5000000000000105, 0.0, 0.7233893525073418),\n                        (13, 0.5000000000000105, 0.0, 0.7048742450466376),\n                        (10, 0.3040867239846962, 0.0, 0.4254807786478514),\n                        (14, 0.30408672398469627, 0.0, -0.5745192213521484)],\n                       [(9, 8), (13, 12)],\n                       [(9, 0.75, 0.0, 0.7233893199921628),\n                        (13, 0.75, 0.0, 0.704874277561818),\n                        (12, 0.49999999999999956, 0.0, 0.6216948873433756),\n                        (8, 0.4999999999999995, 0.0, -0.9912753842136279)],\n                       [(9, 10), (13, 14)],\n                       [(9, 0.500526889421938, 0.0, -0.777137616135001),\n                        (13, 0.5005268894219383, 5.551115123125783e-17, 0.20540121368898195),\n                        (10, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155),\n                        (14, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155)],\n                       [(13, 12), (9, 8)],\n                       [(9, 0.5, 0.0, 0.4728619477915672),\n                        (13, 0.5, -9.769962616701378e-15, 0.9554016497624245),\n                        (8, 0.5000000000000003, 0.0, -0.49127538421362815),\n                        (12, 0.5000000000000003, -2.220446049250313e-16, 0.12169488734337541)],\n                       [(9, 8), (13, 14)],\n                       [(9, 10),\n                        (13, 12),\n                        (8, 0.0007451670218237004, -0.7349391033583288, 0.00872461578637912),\n                        (14, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(9, 3.2515178327585665e-08, 0.8009235646364754, 0.47286192512607883),\n                        (10, 0.5, 0.3483047662953691, -0.5745192538673274),\n                        (12, 0.0007451670218237004, -0.34790937491533347, -0.37830511265661615),\n                        (13, 0.999999967484822, 0.31838381733465093, -0.044598327572097585)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8),\n                      (6, 5),\n                      (6, 6),\n                      (6, 7),\n                      (6, 8)]},\n (15, 7): {\'method\': \'reachable-support count kernels + native scheduling\',\n           \'moments\': [[(4, 0.41348815057431126, 0.0, -0.5000000000000002),\n                        (8, 0.9138626762504471, 0.0, 0.5),\n                        (5, 0.31630918818247244, 0.0, -0.5),\n                        (1, 0.05380912957041928, 0.0, 0.5),\n                        (7, 0.9749914697119216, 0.0, -0.5000000000000002),\n                        (9, 0.5, 0.0, -0.24745148962808194),\n                        (6, 0.151147072532456, 0.0, -0.4549188846524914),\n                        (13, 0.151147072532456, 0.0, -0.4549188846524914),\n                        (2, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (12, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (0, 0.5, 0.0, 0.5),\n                        (10, 0.30408672398469655, 0.0, -0.5745192213521528),\n                        (14, 0.30408672398469655, 0.0, -0.5745192213521528),\n                        (11, 0.30408672398469655, 0.0, -0.5745192213521528)],\n                       [(5, 4), (1, 3), (7, 8)],\n                       [(7, 0.9758980292752844, 2.220446049250313e-16, -0.5000000000000002),\n                        (3, 0.35727615847529887, 0.0, 0.5),\n                        (8, 0.08588433755655986, 0.0, -0.5000000000000002),\n                        (4, 0.6503211074889919, 0.0, -0.5),\n                        (5, 0.3755027911010412, 0.0, -0.5),\n                        (1, 0.4461908704295805, 0.0, 0.5)],\n                       [(4, 8), (3, 7)],\n                       [(3, 0.8589803893445077, 0.0, -0.5),\n                        (4, 0.49999999999999867, 0.0, -0.5000000000000002),\n                        (8, 0.49999999999999956, 0.0, -0.5000000000000002)],\n                       [(3, 4), (7, 8)],\n                       [(4, 0.5000000000000011, 0.0, 0.5),\n                        (1, 3),\n                        (7, 0.8278647016465945, 0.0, -0.5000000000000004),\n                        (8, 0.5000000000000004, 0.0, -0.5)],\n                       [(5, 4), (3, 0.8457372208504039, 0.0, 0.5), (1, 0.4999999999999989, 0.0, 0.5)],\n                       [(4, 0.12844403112522304, 0.0, -0.5), (3, 7), (5, 0.5000000000000016, 0.0, -0.5)],\n                       [(4, 8),\n                        (3, 0.6486931696176113, 0.0, 0.4999999999999998),\n                        (7, 0.5674640172984227, 0.0, 0.4999999999999998)],\n                       [(4, 0.315791130517675, 0.0, -0.5000000000000002), (8, 0.5, 0.0, -0.5)],\n                       [(3, 4), (7, 8)],\n                       [(4, 0.1706789593772302, 0.0, 0.4999999999999998),\n                        (8, 0.5, 0.0, -0.5),\n                        (3, 0.35112917909082464, 0.0, 0.4999999999999996)],\n                       [(5, 4), (8, 9), (1, 3)],\n                       [(4, 0.4478435398279293, 0.0, 0.5),\n                        (5, 0.5, 0.0, 0.38473595982107583),\n                        (3, 0.031841076455933814, 0.0, 0.49999999999999956)],\n                       [(4, 8), (3, 7)],\n                       [(4, 0.9137210455297606, 0.0, 0.9248782235978523),\n                        (8, 0.2521698605391784, 0.0, -0.1676591972690611),\n                        (3, 0.4231123743330681, 0.0, 0.4999999999999998),\n                        (7, 0.27247163109080774, 0.0, -0.4082179396107901)],\n                       [(5, 4), (3, 7)],\n                       [(5, 0.5, 0.0, 0.38473595982107583),\n                        (4, 8),\n                        (7, 0.9003266480810048, 0.0, 0.5917820603892092),\n                        (3, 0.4999999999999998, 0.0, -0.5451687477928442)],\n                       [(9, 5), (8, 0.5557004579686877, 0.0, 0.3323408027309387), (7, 6), (1, 3)],\n                       [(5, 0.5, 0.0, -0.9125965194114836),\n                        (9, 0.22574967231825369, 0.0, -0.6636263518784891),\n                        (7, 0.8238665379044736, 0.0, 0.5917820603892081),\n                        (3, 0.5, 0.0, -0.5451687477928437),\n                        (6, 0.4678935735932473, 0.0, 0.5450811153475075)],\n                       [(8, 9), (5, 4), (3, 7)],\n                       [(8, 0.993401101423752, 0.0, 0.8323408027309351),\n                        (9, 0.49942055111844974, 0.0, -0.1481155227947728),\n                        (4, 0.9540075367795029, 5.551115123125783e-17, 0.11941854995972084),\n                        (5, 0.5, 0.0, 0.08740348058851666),\n                        (7, 0.25087508170700706, 0.0, 0.5917820603892081),\n                        (1, 3)],\n                       [(4, 8), (9, 5), (7, 6)],\n                       [(8, 0.5808813702856532, 0.0, -0.6676591972690682),\n                        (4, 0.25443599797049227, 0.0, -0.905030816612655),\n                        (9, 0.42607382518378495, 0.0, -0.4708952936330244),\n                        (7, 0.4411490610015294, 4.440892098500626e-16, 0.5917820603892083),\n                        (6, 0.6252840566860213, 0.0, -0.4549188846524941)],\n                       [(8, 9), (5, 4), (3, 7)],\n                       [(4, 0.7140608751963855, 0.0, -0.6892139635694441),\n                        (8, 0.4515486953156751, 0.0, 0.832340802730932),\n                        (9, 0.7250211832941114, 0.0, 0.3504723714814277),\n                        (3, 0.5, 0.0, -0.54615132767534),\n                        (7, 0.8018172683289451, 0.0, 0.5917820603892081)],\n                       [(9, 5), (4, 8), (7, 6), (1, 3)],\n                       [(8, 0.4314501661314405, 0.0, -0.667659197269068),\n                        (4, 0.5448381862924326, 0.0, -0.7576020372757506),\n                        (9, 0.7287196312679844, 0.0, 0.35579841873569795),\n                        (7, 0.034005501067222, 0.0, 0.5917820603892083),\n                        (3, 0.5, 0.0, -0.54615132767534),\n                        (6, 0.23418985268179596, 4.440892098500626e-16, 0.5450811153475046)],\n                       [(5, 4), (8, 9), (3, 7)],\n                       [(9, 0.9881887823489937, 0.0, 0.1016504889505454),\n                        (8, 0.4112910774097216, 0.0, 0.8323408027309318),\n                        (4, 0.683530177649277, 0.0, -0.02835461137628026),\n                        (7, 0.5674840791730928, 0.0, -0.4082179396107919),\n                        (3, 0.5, 0.0, -0.5575210475770782)],\n                       [(4, 8), (9, 5), (7, 6), (1, 3)],\n                       [(4, 0.1510597096601043, 5.551115123125783e-17, 0.4049983595005528),\n                        (8, 0.0006913190085300335, 0.0, 0.33234080273087235),\n                        (9, 0.06691772293018138, 0.0, -0.39466187129291863),\n                        (3, 0.5000000000000002, 0.0, 0.4424789524229218),\n                        (7, 0.5920238735305361, 0.0, 0.5917820603892079),\n                        (6, 0.3495759458707011, 4.440892098500626e-16, 0.5450811153475055)],\n                       [(8, 9), (5, 4), (3, 7)],\n                       [(4, 0.6991057269115482, -2.220446049250313e-16, 0.17727727032062002),\n                        (8, 0.5, 0.0, -0.667659197269068),\n                        (9, 0.10622873606974291, 0.0, 0.5895761819104166),\n                        (7, 0.5920238735305361, 4.440892098500626e-16, 0.5917820603892079),\n                        (3, 0.4999999999999995, 0.0, -0.558503627459574)],\n                       [(9, 5), (4, 8), (1, 3), (7, 6)],\n                       [(8, 0.8378188644666402, 0.0, -0.17696774909088653),\n                        (9, 0.2924608216157978, 0.0, 0.26336273675139843),\n                        (4, 0.5, 0.0, 0.32992956831558984),\n                        (3, 0.7782742815462358, 0.0, 0.45423391419931525),\n                        (7, 0.14060173868216674, 0.0, 0.5917820603892074),\n                        (6, 0.2784873646591348, 0.0, -0.4549188846524954),\n                        (5, 0.5, 0.0, 0.18022464109232844)],\n                       [(8, 9), (3, 7), (5, 4)],\n                       [(9, 0.6852297060160111, 0.0, 0.8805625872720799),\n                        (8, 0.5000000000000007, 5.551115123125783e-17, 0.3137469991417418),\n                        (3, 0.24999999999999975, 0.0, -0.5291663156193016),\n                        (7, 6),\n                        (5, 0.5, 0.5278605595904059, -0.8197753589076715)],\n                       [(8, 9),\n                        (1, 3),\n                        (7, 0.6270946998774289, 0.0, 0.5917820603892072),\n                        (6, 0.3403378795392227, 0.0, -0.45116623934371125)],\n                       [(9, 0.9003266480810048, -2.220446049250313e-16, 0.18711485407286027),\n                        (8, 0.4999999999999998, 0.0, 0.18474865271436758),\n                        (3, 0.75, 0.0, 0.4708336843806986)],\n                       [(4, 8), (9, 13), (3, 7)],\n                       [(9, 0.8238665379044736, 0.0, 0.18711485407286155),\n                        (8, 0.5000000000000001, 0.0, -0.8152513472856322),\n                        (13, 0.46789357359324735, 0.0, -0.4549188846524923),\n                        (3, 0.5, 0.0, -0.5291663156193016),\n                        (7, 0.5, 0.0, -0.40821793961079256)],\n                       [(8, 9), (1, 3), (7, 6)],\n                       [(9, 0.25087508170700706, 5.551115123125783e-17, 0.1871148540728615),\n                        (4, 8),\n                        (3, 0.13865229719054487, 0.0, 0.4708336843806973),\n                        (1, 0.5, 0.0, -0.5229455150973021),\n                        (7, 6)],\n                       [(9, 13), (3, 7), (6, 0.4535551979948596, 0.0, 0.438011157726228)],\n                       [(9, 0.44114906100152945, 1.1102230246251565e-16, 0.18711485407286105),\n                        (13, 0.6252840566860214, 2.220446049250313e-16, -0.4549188846524941),\n                        (3, 0.4999999999999998, 0.0, -0.5291663156193016),\n                        (7, 0.5611163684243805, 0.0, 0.02950691355132179),\n                        (6, 2)],\n                       [(8, 9),\n                        (1, 3),\n                        (2, 0.85928593962011, 0.0, -0.9262332254895425),\n                        (6, 0.5, 0.0, -0.1878391688093657)],\n                       [(9, 0.8018172683289451, 5.551115123125783e-17, 0.18711485407286105),\n                        (8, 0.5, 0.0, 0.18376607283187213),\n                        (3, 0.5000000000000002, 0.0, 0.7484151462228374),\n                        (7, 6),\n                        (1, 0.5, -0.9541089698053948, -0.5229455150973024)],\n                       [(9, 13), (4, 8), (3, 7), (6, 0.6714165234362537, 0.0, 0.8835769229216266)],\n                       [(9, 0.034005501067222, 0.0, 0.18711485407286066),\n                        (8, 0.5, 0.0, -0.8162339271681278),\n                        (13, 0.23418985268179598, 0.0, -0.45491888465249497),\n                        (3, 0.5, 0.0, -0.25158485377716344),\n                        (7, 0.5, 0.0, -0.5248603567479269)],\n                       [(8, 9), (2, 3)],\n                       [(8, 0.4999999999999998, 0.0, 0.1723963529301329),\n                        (9, 0.5674840791730928, 0.0, -0.8128851459271387),\n                        (2, 0.7485956033393342, 0.0, -0.42623322548954246),\n                        (3, 7)],\n                       [(4, 8),\n                        (9, 13),\n                        (6, 2),\n                        (7, 0.5, 0.0, -0.5248603567479269),\n                        (3, 0.0528060550044127, 0.0, 0.9220154900098074)],\n                       [(9, 0.5920238735305361, 0.0, 0.18711485407286133),\n                        (8, 0.5000000000000001, 0.0, 0.1723963529301331),\n                        (13, 0.3495759458707011, 4.440892098500626e-16, 0.5450811153475055),\n                        (2, 0.9961572718865779, 0.0, -0.9262332254895478),\n                        (6, 0.3935294720438233, 0.0, -0.8373759614647518)],\n                       [(8, 9), (7, 6), (2, 3)],\n                       [(8, 0.4999999999999994, 0.0, 0.17141377304763722),\n                        (9, 0.5920238735305361, 0.0, 0.18711485407286133),\n                        (2, 0.5655532211464794, 0.0, 0.5737667745104469),\n                        (6, 0.31043695253214243, 0.0, -0.2558972118649514),\n                        (3, 0.24963665794604525, 0.0, -0.5606219250804185)],\n                       [(4, 8), (9, 13), (6, 2), (3, 7)],\n                       [(8, 0.7782742815462358, 5.551115123125783e-17, 0.18415131470652646),\n                        (9, 0.14060173868216674, 0.0, 0.18711485407286155),\n                        (13, 0.27848736465913476, 0.0, 0.5450811153475053),\n                        (2, 0.4999999999999998, 0.0, 0.5737667745104462),\n                        (6, 0.6193705712509703, 4.440892098500626e-16, 0.6170684772356436),\n                        (3, 0.32701156232958084, 0.0, 0.047158433631480756)],\n                       [(8, 9), (2, 3), (7, 6)],\n                       [(8, 0.25000000000000006, 0.0, 0.20075108488790971),\n                        (9, 13),\n                        (6, 0.6256105257587086, 0.0, -0.8996611547547264),\n                        (2, 0.5960614607313279, -2.220446049250313e-16, -0.11684358694965402),\n                        (3, 0.5, 0.0, -0.7054770348896429)],\n                       [(4, 8),\n                        (9, 0.6270946998774289, 0.0, 0.18711485407286194),\n                        (13, 0.34033787953922284, 0.0, -0.45116623934371125),\n                        (2, 3),\n                        (7, 6)],\n                       [(8, 0.7499999999999999, 0.0, 0.20075108488790971),\n                        (6, 0.9764747842833552, 2.220446049250313e-16, -0.30187691267533445),\n                        (7, 0.5, 8.881784197001252e-16, 0.796211072559555),\n                        (3, 0.5, 1.1102230246251565e-16, -0.20547703488964464),\n                        (2, 0.14949404734505523, 0.0, -0.7269328881171262)],\n                       [(8, 9), (3, 7), (6, 2)],\n                       [(8, 0.5, 0.0, 0.2007510848879095),\n                        (9, 0.5, 0.0, -0.8128851459271385),\n                        (6, 0.5, 0.0, -0.05889862949346014),\n                        (7, 0.5, 8.881784197001252e-16, 0.796211072559555),\n                        (2, 3)],\n                       [(4, 8), (9, 13), (7, 6), (2, 0.4247761378557593, 0.0, -0.022381851578533896)],\n                       [(8, 0.1386522971905449, 0.0, 0.2007510848879095),\n                        (4, 0.5, 0.0, -0.35521011726984275),\n                        (9, 13),\n                        (6, 0.14731871270290628, 0.0, -0.058898629493459254),\n                        (7, 0.5, 4.440892098500626e-16, 0.7290222185409181)],\n                       [(8, 9), (13, 0.4535551979948596, 0.0, 0.438011157726228), (6, 2), (3, 7)],\n                       [(8, 0.4999999999999998, -2.220446049250313e-16, 0.20075108488790905),\n                        (9, 0.5611163684243808, 0.0, 0.7493900009107477),\n                        (13, 12),\n                        (2, 0.45008695289946266, 0.0, -0.3365366861733614),\n                        (3, 0.4202007516434625, 0.0, -0.4989146764811452),\n                        (6, 0.9433458052208397, 0.0, 0.4411013705065394)],\n                       [(4, 8),\n                        (13, 0.5, -2.220446049250313e-16, -0.18783916880936635),\n                        (12, 0.85928593962011, 0.0, -0.9262332254895425),\n                        (2, 3),\n                        (7, 6)],\n                       [(8, 0.5, 0.0, 0.47833254673004805),\n                        (9, 13),\n                        (2, 0.5410889047360808, -2.220446049250313e-16, 0.21035484945072325),\n                        (3, 0.5, 0.0, -0.09411056164187137),\n                        (7, 0.2056700352039175, 0.0, 0.2290222185409183),\n                        (4, 0.4999999999999998, 0.8781556323671444, 0.6447898827301577)],\n                       [(8, 9),\n                        (13, 0.6714165234362537, 1.1102230246251565e-16, -0.11642307707837363),\n                        (2, 0),\n                        (7, 3)],\n                       [(8, 0.5, 0.0, 0.4783325467300483),\n                        (9, 0.5, 0.0, -0.6962427287900046),\n                        (2, 0.5, 0.0, -0.2896451505492772),\n                        (7, 0.4999999999999824, 0.0, -0.27097783777702267),\n                        (0, 0.3040867239846962, 0.0, 0.4999999999999998),\n                        (3, 0.500000000000002, 0.0, -0.005450632049704751)],\n                       [(12, 8), (6, 2), (7, 11)],\n                       [(12, 0.7485956033393342, 0.0, -0.42623322548954246),\n                        (8, 9),\n                        (2, 0.8040867239846965, 4.440892098500626e-16, 0.710354849450723),\n                        (7, 0.5000000000000102, 1.1102230246251565e-16, -0.2709778377770229),\n                        (11, 0.3040867239846962, 0.0, 0.4254807786478514)],\n                       [(13, 12),\n                        (9, 0.5, 0.0, -0.6962427287900046),\n                        (8, 0.0528060550044127, 0.0, 0.6519328905170179),\n                        (2, 0),\n                        (7, 3)],\n                       [(13, 0.3935294720438233, 5.551115123125783e-17, 0.16262403853524748),\n                        (12, 0.9961572718865779, 0.0, -0.9262332254895478),\n                        (2, 0.75, 0.0, 0.7103548494507228),\n                        (7, 0.75, 4.440892098500626e-16, 0.7290221297077986),\n                        (0, 0.5, 0.0, 0.5),\n                        (3, 0.49999999999999967, 0.0, 0.994549400465476)],\n                       [(12, 8), (9, 13), (6, 2), (7, 11)],\n                       [(8, 0.24963665794604498, 0.0, -0.8307045245732068),\n                        (13, 0.3104369525321424, 0.0, -0.25589721186495185),\n                        (12, 0.5655532211464794, 0.0, 0.5737667745104469),\n                        (2, 0.5, 4.440892098500626e-16, 0.710354849450723),\n                        (6, 0.5, 0.0, -0.03835417712542144),\n                        (7, 0.500526889421938, 0.0, -0.7715048064193657),\n                        (11, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155)],\n                       [(8, 9), (13, 12), (2, 0), (7, 3)],\n                       [(8, 0.3270115623295809, 0.0, 0.7770758341386912),\n                        (12, 0.4999999999999998, 0.0, 0.5737667745104462),\n                        (13, 0.6193705712509703, 4.440892098500626e-16, 0.6170684772356436),\n                        (2, 0.25000000000000006, 0.0, -0.2896451505492772),\n                        (7, 0.5, 0.0, -0.5215052424927984),\n                        (3, 0.5000000000000004, -2.220446049250313e-16, 0.4945494004654756)],\n                       [(12, 8), (9, 13), (6, 2), (7, 11)],\n                       [(12, 0.5960614607313279, -2.220446049250313e-16, -0.11684358694965402),\n                        (13, 0.6256105257587086, 5.551115123125783e-17, 0.10033884524527337),\n                        (8, 0.5, 5.551115123125783e-17, 0.024440365617567517),\n                        (2, 0.5, 0.0, -0.2896451505492772),\n                        (7, 3),\n                        (11, 0.5, 0.3483047662953691, -0.5745192538673274),\n                        (6, 0.5, -0.4616458228745772, -0.03835417712542277)],\n                       [(12, 8),\n                        (9, 13),\n                        (2, 0),\n                        (3, 0.9992548329781763, -0.7207638880374159, 0.9945494004654647),\n                        (7, 3.251517832758566e-08, 0.7952907549208398, 0.47849473484171356)],\n                       [(9, 0.5, 0.0, -0.017314158097487375),\n                        (12, 0.14949404734505523, 0.0, -0.7269328881171262),\n                        (13, 0.9764747842833552, 2.220446049250313e-16, -0.30187691267533445),\n                        (8, 0.5, 4.440892098500626e-16, 0.5244403656175667),\n                        (0, 1.0, 0.0, 0.5),\n                        (2, 0.4999999999999998, 0.7896451505492776, 0.7103548494507228)],\n                       [(8, 9), (13, 12)],\n                       [(13, 0.49999999999999994, 0.0, -0.05889862949346014),\n                        (9, 0.5, 0.0, -0.017314158097487375),\n                        (12, 8)],\n                       [(9, 13), (12, 0.4247761378557593, 0.0, -0.022381851578533896)],\n                       [(13, 0.14731871270290633, 0.0, 0.9411013705065399),\n                        (9, 0.5, 0.0, 0.04987469592114996)],\n                       [(13, 12), (8, 9)],\n                       [(13, 0.9433458052208397, 0.0, -0.5588986294934615),\n                        (8, 0.4202007516434625, 0.0, 0.2310027240260653),\n                        (12, 0.45008695289946266, 0.0, -0.3365366861733614)],\n                       [(12, 8), (9, 13)],\n                       [(8, 0.5, 8.881784197001252e-16, 0.6358068388653375),\n                        (9, 0.2056700352039173, 0.0, 0.5498746959211498),\n                        (13, 0.5, 0.0, 0.23543133530261962),\n                        (12, 0.5, 0.0, -0.2896451505492772)],\n                       [(13, 12), (9, 8)],\n                       [(9, 0.49999999999998224, 0.0, -0.9501252477609097),\n                        (13, 0.4999999999999827, 0.0, -0.7645686083794385),\n                        (12, 0.5000000000000022, 0.0, 0.6216949198585577),\n                        (8, 0.500000000000002, 0.0, -0.27553323154249565)],\n                       [(13, 14), (9, 10)],\n                       [(13, 0.5000000000000102, 0.0, -0.7645686083794381),\n                        (9, 0.5000000000000102, 0.0, -0.9501252477609101),\n                        (10, 0.30408672398469627, 0.0, -0.5745192213521484),\n                        (14, 0.30408672398469627, 0.0, -0.5745192213521484)],\n                       [(13, 12), (9, 8)],\n                       [(9, 0.75, 0.0, 0.04987478475426799),\n                        (13, 0.7499999999999999, 0.0, -0.7645685758642613),\n                        (8, 0.49999999999999956, 0.0, -0.27553319902731355),\n                        (12, 0.49999999999999956, 0.0, 0.6216948873433754)],\n                       [(9, 10), (13, 14)],\n                       [(13, 0.500526889421938, 4.440892098500626e-16, 0.7359583602629027),\n                        (9, 0.5005268894219381, 2.220446049250313e-16, -0.4495982791185673),\n                        (10, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155),\n                        (14, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155)],\n                       [(9, 8), (13, 12)],\n                       [(13, 0.5, 0.0, -0.5140412036636628),\n                        (8, 0.5000000000000004, 0.0, 0.22446680097268595),\n                        (9, 0.5, 0.0, -0.6995978430451348),\n                        (12, 0.5000000000000003, -2.220446049250313e-16, 0.12169488734337541)],\n                       [(9, 10), (13, 14)],\n                       [(9, 8),\n                        (13, 12),\n                        (14, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (10, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(9, 0.999999967484822, 0.9733833101422, -0.6995978203796469),\n                        (12, 0.0007451670218237004, -0.34790937491533347, -0.37830511265661615),\n                        (13, 0.999999967484822, -0.21217332923927024, 0.48595881900182447),\n                        (8, 0.9992548329781763, 0.549318711455373, -0.27553319902732243)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8),\n                      (6, 5),\n                      (6, 6),\n                      (6, 7),\n                      (6, 8)]}}\n\ndef _round6_unitary(n, k):\n    data = _ROUND6_VERIFIED.get((n, k))\n    if data is None:\n        return None\n    q = [cirq.GridQubit(*v) for v in data[\'qubits\']]\n    moments = []\n    for layer in data[\'moments\']:\n        ops = []\n        for gate in layer:\n            if len(gate) == 2:\n                ops.append(cirq.CZ(q[gate[0]], q[gate[1]]))\n            else:\n                i, x, z, a = gate\n                ops.append(cirq.PhasedXZGate(x_exponent=x, z_exponent=z,\n                                           axis_phase_exponent=a).on(q[i]))\n        moments.append(cirq.Moment(ops))\n    return cirq.Circuit(moments), {q[i] for i in data[\'outputs\']}\n# ROUND 6 VERIFIED NATIVE CIRCUITS END\n\n\n# ROUND 7 VERIFIED NATIVE CIRCUITS START\n# Frozen only after full-state, output-purity and native-coupler checks.\n_ROUND7_VERIFIED = {(10, 3): {\'method\': \'three-arm hub seed + parallel square decoders\',\n           \'moments\': [[(2, 0.5075250292358409, 0.0, -0.5),\n                        (4, 0.5640582364224308, 0.0, -0.5),\n                        (3, 0.21521876042258375, 0.0, 0.5),\n                        (5, 0.4060762532476705, 0.0, 0.5),\n                        (7, 0.3268287043101393, 0.0, 0.5),\n                        (8, 0.538298819430637, 0.0, -0.5),\n                        (0, 0.5390087196324642, -0.8603924452310396, -0.691353199242738),\n                        (1, 0.03900871963245432, -0.1066856530705218, -0.9660439767483449),\n                        (6, 0.5, 0.0, 0.5),\n                        (9, 0.5, 0.0, 0.5)],\n                       [(2, 4), (3, 5), (7, 8)],\n                       [(2, 0.4625211535683372, 5.551115123125783e-17, 0.49999999999999994),\n                        (3, 0.9604308353955838, 0.0, -0.5),\n                        (4, 0.03671905871867653, 0.0, 0.5),\n                        (7, 0.3128234467263022, 0.0, -0.5),\n                        (5, 0.02134966799353345, 0.0, 0.5),\n                        (8, 0.37131330816679287, 0.0, 0.5)],\n                       [(2, 3), (4, 7), (5, 8)],\n                       [(2, 0.5000000000000001, 0.0, -0.5),\n                        (3, 0.7834904832650229, 0.0, 0.5),\n                        (4, 1.0, 0.0, 0.5),\n                        (5, 0.2724041902391714, 0.0, -0.5),\n                        (7, 0.9144066141716028, 0.0, -0.5),\n                        (8, 0.5505652661300525, 0.0, -0.5)],\n                       [(2, 3), (4, 5), (7, 8)],\n                       [(2, 1.0, 0.0, 0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (3, 1.0, 0.0, 0.5),\n                        (5, 0.969751333521469, 0.0, 0.5),\n                        (7, 0.2007184326188422, 0.0, 0.5),\n                        (8, 0.23685822758895508, 0.0, 0.5)],\n                       [(2, 4), (3, 5), (7, 8)],\n                       [(2, 0.5548771086812295, 0.0, 0.5),\n                        (3, 1.0, 0.0, 0.5),\n                        (4, 0.7379536328460687, 0.0, -0.5),\n                        (7, 0.7215353403057394, 0.0, 0.5),\n                        (5, 0.9900120526976681, 2.220446049250313e-16, -0.5000000000000002),\n                        (8, 0.892328487053231, 0.0, 0.5)],\n                       [(2, 3), (4, 7), (5, 8)],\n                       [(2, 0.30279448144254484, 0.0, 0.5),\n                        (4, 0.5229082281297596, 0.0, -0.5),\n                        (3, 0.49205017325274075, 0.0, -0.5),\n                        (5, 0.8763813388802872, 0.0, 0.5),\n                        (8, 0.810070564137761, 0.0, -0.5)],\n                       [(2, 4), (3, 5), (7, 8)],\n                       [(2, 0.4999999999999998, 0.0, 0.5),\n                        (3, 0.8644526864271973, 0.0, 0.5),\n                        (4, 0.5000000000000001, 0.0, -0.5),\n                        (7, 1.0, 0.0, 0.5),\n                        (5, 0.4338494543433774, 0.0, -0.5),\n                        (8, 0.6436424188213695, 0.0, 0.5)],\n                       [(2, 3), (4, 7), (5, 8)],\n                       [(3, 0.040817932460624026, 0.0, -0.5000000000000002),\n                        (4, 1.0, 0.0, 0.5),\n                        (5, 0.6299089726522494, 0.0, 0.5),\n                        (7, 1.0, 0.0, 0.5),\n                        (8, 0.6270991898233952, 0.0, 0.5)],\n                       [(2, 3), (4, 5), (7, 8)],\n                       [(2, 0.9999999999999908, 0.8797721958670468, 0.20390439054162512),\n                        (3, 0.28590078940089186, -0.9029663141725073, -0.5000000000000318),\n                        (5, 0.49696336328397284, 0.0, -0.5),\n                        (8, 0.047368593789405784, 0.0, -0.5000000000000002),\n                        (7, 1.0, 0.0, 0.5)],\n                       [(2, 0), (3, 1), (5, 6), (8, 9)],\n                       [(2, 0.4999999999999988, 0.7855375402055449, 0.26368556094540857),\n                        (3, 3.8567868908521176e-14, -0.7138477097683644, 0.8623864896562614),\n                        (1, 0.9609912803675337, -0.2654416119225229, -0.07272962981882314),\n                        (0, 0.22652038258320545, 0.8444189779561107, -0.006127787872109769),\n                        (5, 0.49999999999999983, 0.0, -0.5),\n                        (6, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (8, 0.49999999999999983, 0.0, -0.5),\n                        (9, 0.3040867239846963, 0.0, -0.5000000000000002)],\n                       [(2, 3), (1, 0), (4, 5), (7, 8)],\n                       [(2, 0.4999999999999983, 0.8421982859273518, -0.9507768988490481),\n                        (0, 0.5000000000003056, -0.7652187161819077, -0.9461330490265849),\n                        (3, 6.745364167227603e-15, -0.5883266847249748, -0.15165003845294356),\n                        (1, 0.7499999999999835, -0.40061767531814807, 0.4381118840480963),\n                        (5, 0.8040867239846964, 0.0, 0.5),\n                        (4, 1.0, 0.0, 0.5),\n                        (8, 0.8040867239846964, 0.0, 0.5),\n                        (7, 1.0, 0.0, 0.5)],\n                       [(2, 0), (3, 1), (5, 6), (8, 9)],\n                       [(2, 0.49999999999999684, -0.6050588866550652, 0.14394699237253755),\n                        (3, 0.5000000000000038, -0.8845692780498959, -0.12117258793593289),\n                        (1, 0.74999999999998, 0.08003009104082975, 0.03749420872993947),\n                        (0, 0.637618967553187, -0.0031023457864229886, 0.47100340349251263),\n                        (5, 0.75, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (8, 0.75, 0.0, 0.5),\n                        (9, 0.5, 0.0, 0.5)],\n                       [(2, 3), (1, 0), (4, 5), (7, 8)],\n                       [(2, 0.5000000000000003, 0.6040035257369907, -0.4611118942825243),\n                        (0, 0.5780174392649265, 0.012788763989812324, -0.7154671565497159),\n                        (3, 0.5000000000000044, 0.20186808770552966, -0.5057418659858346),\n                        (1, 0.9999999999999954, -0.012967077279665062, 0.06339120933111991),\n                        (5, 0.5000000000000001, 0.0, -0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (8, 0.5000000000000001, 0.0, -0.5),\n                        (7, 0.5, 0.0, 0.5)],\n                       [(2, 0), (3, 1), (5, 6), (8, 9)],\n                       [(2, 0.7499999999999958, -0.7659601398809079, -0.8571083685455292),\n                        (3, 0.4999999999999959, -0.27864199383463517, -0.5538737782802994),\n                        (1, 0.7500000000000032, 0.34032278598431986, 0.5696501318346623),\n                        (0, 0.9999999999999996, -0.3648932378899361, -0.9778728802180585),\n                        (5, 0.25000000000000006, 0.0, 0.5),\n                        (6, 1.0, 0.0, 0.5),\n                        (8, 0.25000000000000006, 0.0, 0.5),\n                        (9, 1.0, 0.0, 0.5)],\n                       [(2, 3), (1, 0), (4, 5), (7, 8)],\n                       [(2, 0.6193218529801312, 0.6736152698514805, -0.12306850842643091),\n                        (0, 0.9999999999999993, 0.9339171918431297, -0.5315283251854057),\n                        (3, 0.4999999999999959, -0.0017782357105362578, -0.3325157721149319),\n                        (1, 0.2499999999999959, -0.20581531379718232, 0.9099729178189659),\n                        (5, 0.5, 0.0, -0.5),\n                        (8, 0.5, 0.0, -0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (7, 0.5, 0.0, 0.5)],\n                       [(2, 0), (3, 1), (5, 6), (8, 9)],\n                       [(2, 0.5000000000000003, 0.19870280658969874, -0.44945323857494657),\n                        (3, 0.4999999999999974, 0.4147080617789417, 0.2850278451546754),\n                        (1, 1.4972508298281808e-14, 0.4091233012629609, -0.5399326330322662),\n                        (0, 0.5000000000000108, 0.17203203350261553, 0.9357092063272923),\n                        (5, 0.5000000000000001, 0.0, 0.5),\n                        (8, 0.5000000000000001, 0.0, 0.5)],\n                       [(2, 3), (1, 0)],\n                       [(2, 0.25818731204330203, 0.2147600793762427, 0.24924956801474624),\n                        (0, 0.4999999999999808, 0.17553429973119078, 0.10774123982990619),\n                        (3, 0.7581873120432945, -0.33695871832380564, 0.6997359069336127),\n                        (1, 0.9999999999999805, 0.9453700418180587, -0.5716229425559929)],\n                       [(2, 0), (3, 1)],\n                       [(2, 0.4999999999999981, 0.13136895120953168, -0.035990352609010845),\n                        (3, 0.5000000000000023, -0.7673985900092897, -0.13722281139019699),\n                        (1, 5.015629698628501e-15, 0.30746064351072583, 0.4267597788467477),\n                        (0, 3.3831615324158282e-15, -0.286971644629054, -0.7418832441465113)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],\n           \'qubits\': [(3, 4), (3, 5), (4, 4), (4, 5), (5, 4), (5, 5), (5, 6), (6, 4), (6, 5), (6, 6)]},\n (11, 3): {\'method\': \'three-arm hub seed + parallel square decoders\',\n           \'moments\': [[(2, 0.4977095531171783, 0.0, -0.5),\n                        (4, 0.47999676525351215, 0.0, -0.5),\n                        (3, 0.13289535826303484, 0.0, 0.5),\n                        (5, 0.30529911596742315, 0.0, 0.5),\n                        (7, 0.31211681879788, 0.0, 0.5),\n                        (8, 0.46080081683211016, 0.0, -0.5),\n                        (0, 0.5390087196324642, -0.8603924452310396, -0.691353199242738),\n                        (1, 0.03900871963245432, -0.1066856530705218, -0.9660439767483449),\n                        (6, 0.5, 0.0, 0.5),\n                        (9, 0.5390087196324642, -0.8603924452310396, -0.691353199242738),\n                        (10, 0.03900871963245432, -0.1066856530705218, -0.9660439767483449)],\n                       [(2, 4), (3, 5), (7, 8)],\n                       [(2, 0.4636860383182841, 0.0, 0.5),\n                        (3, 0.8349015140950895, 0.0, -0.5),\n                        (4, 0.03624196952874487, 0.0, 0.5),\n                        (7, 0.2137341063327851, 0.0, -0.5),\n                        (5, 0.22378717952558877, 0.0, 0.5),\n                        (8, 0.4933349703475507, 0.0, 0.5)],\n                       [(2, 3), (4, 7), (5, 8)],\n                       [(2, 0.5000000000000001, 0.0, -0.5),\n                        (3, 0.7311025414991926, 0.0, 0.5),\n                        (4, 1.0, 0.0, 0.5),\n                        (5, 0.32175092945542666, 0.0, -0.5),\n                        (7, 0.9241027775512922, 0.0, -0.5),\n                        (8, 0.4344976034821523, 0.0, -0.5)],\n                       [(2, 3), (4, 5), (7, 8)],\n                       [(2, 1.0, 0.0, 0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (3, 1.0, 0.0, 0.5),\n                        (5, 0.9674838768770302, 0.0, -0.5000000000000002),\n                        (7, 0.3185254465017618, 0.0, 0.5),\n                        (8, 0.17074045794894652, 0.0, 0.5)],\n                       [(2, 4), (3, 5), (7, 8)],\n                       [(2, 0.5881898098972081, 0.0, 0.5),\n                        (3, 1.0, 0.0, 0.5),\n                        (4, 0.7631294770516314, 0.0, -0.5),\n                        (7, 0.6932783553567452, 0.0, 0.5),\n                        (5, 0.9641022893365128, 0.0, 0.5),\n                        (8, 0.7708477377925105, 0.0, 0.5)],\n                       [(2, 3), (4, 7), (5, 8)],\n                       [(2, 0.48176156561702044, 0.0, 0.5),\n                        (4, 0.5996540067178372, 2.220446049250313e-16, -0.5000000000000002),\n                        (3, 0.5044593869341634, 0.0, -0.5),\n                        (5, 0.7543939581801827, 0.0, 0.5),\n                        (8, 0.9081527187719957, 0.0, -0.5)],\n                       [(2, 4), (3, 5), (7, 8)],\n                       [(2, 0.49999999999999994, 0.0, 0.5),\n                        (3, 0.9975529633914461, 0.0, 0.5),\n                        (4, 0.5, 0.0, -0.5),\n                        (7, 1.0, 0.0, 0.5),\n                        (5, 0.526039179709408, 0.0, -0.5),\n                        (8, 0.5470533185096936, 0.0, 0.5)],\n                       [(2, 3), (4, 7), (5, 8)],\n                       [(3, 0.23521863150189462, 0.0, -0.5),\n                        (4, 1.0, 0.0, 0.5),\n                        (5, 0.5728585954068288, 0.0, 0.5),\n                        (7, 1.0, 0.0, 0.5),\n                        (8, 0.7360395305448575, 0.0, 0.5)],\n                       [(2, 3), (4, 5), (7, 8)],\n                       [(2, 0.9999999999999907, 0.8643229550099798, 0.21162901097015868),\n                        (3, 0.43568379158365267, -0.9029663141724991, -0.5000000000000251),\n                        (5, 0.4174153969225333, 0.0, -0.5000000000000002),\n                        (7, 0.9999999999999907, 0.8601627902767732, 0.21370909333676202),\n                        (8, 0.3095168363707791, -0.9029663141725321, 0.5000000000000298)],\n                       [(2, 0), (3, 1), (5, 6), (7, 9), (8, 10)],\n                       [(2, 0.4999999999999988, 0.7855375402055449, 0.26368556094540857),\n                        (3, 3.8567868908521176e-14, -0.7138477097683644, 0.8623864896562614),\n                        (1, 0.9609912803675337, -0.2654416119225229, -0.07272962981882314),\n                        (0, 0.22652038258320545, 0.8444189779561107, -0.006127787872109769),\n                        (5, 0.49999999999999983, 0.0, -0.5),\n                        (6, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (7, 0.4999999999999988, 0.7855375402055449, 0.26368556094540857),\n                        (8, 3.8567868908521176e-14, -0.7138477097683644, 0.8623864896562614),\n                        (10, 0.9609912803675337, -0.2654416119225229, -0.07272962981882314),\n                        (9, 0.22652038258320545, 0.8444189779561107, -0.006127787872109769)],\n                       [(2, 3), (1, 0), (4, 5), (7, 8), (10, 9)],\n                       [(2, 0.4999999999999983, 0.8421982859273518, -0.9507768988490481),\n                        (0, 0.5000000000003056, -0.7652187161819077, -0.9461330490265849),\n                        (3, 6.745364167227603e-15, -0.5883266847249748, -0.15165003845294356),\n                        (1, 0.7499999999999835, -0.40061767531814807, 0.4381118840480963),\n                        (5, 0.8040867239846964, 0.0, 0.5),\n                        (4, 1.0, 0.0, 0.5),\n                        (7, 0.4999999999999983, 0.8421982859273518, -0.9507768988490481),\n                        (9, 0.5000000000003056, -0.7652187161819077, -0.9461330490265849),\n                        (8, 6.745364167227603e-15, -0.5883266847249748, -0.15165003845294356),\n                        (10, 0.7499999999999835, -0.40061767531814807, 0.4381118840480963)],\n                       [(2, 0), (3, 1), (5, 6), (7, 9), (8, 10)],\n                       [(2, 0.49999999999999684, -0.6050588866550652, 0.14394699237253755),\n                        (3, 0.5000000000000038, -0.8845692780498959, -0.12117258793593289),\n                        (1, 0.74999999999998, 0.08003009104082975, 0.03749420872993947),\n                        (0, 0.637618967553187, -0.0031023457864229886, 0.47100340349251263),\n                        (5, 0.75, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (7, 0.49999999999999684, -0.6050588866550652, 0.14394699237253755),\n                        (8, 0.5000000000000038, -0.8845692780498959, -0.12117258793593289),\n                        (10, 0.74999999999998, 0.08003009104082975, 0.03749420872993947),\n                        (9, 0.637618967553187, -0.0031023457864229886, 0.47100340349251263)],\n                       [(2, 3), (1, 0), (4, 5), (7, 8), (10, 9)],\n                       [(2, 0.5000000000000003, 0.6040035257369907, -0.4611118942825243),\n                        (0, 0.5780174392649265, 0.012788763989812324, -0.7154671565497159),\n                        (3, 0.5000000000000044, 0.20186808770552966, -0.5057418659858346),\n                        (1, 0.9999999999999954, -0.012967077279665062, 0.06339120933111991),\n                        (5, 0.5000000000000001, 0.0, -0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (7, 0.5000000000000003, 0.6040035257369907, -0.4611118942825243),\n                        (9, 0.5780174392649265, 0.012788763989812324, -0.7154671565497159),\n                        (8, 0.5000000000000044, 0.20186808770552966, -0.5057418659858346),\n                        (10, 0.9999999999999954, -0.012967077279665062, 0.06339120933111991)],\n                       [(2, 0), (3, 1), (5, 6), (7, 9), (8, 10)],\n                       [(2, 0.7499999999999958, -0.7659601398809079, -0.8571083685455292),\n                        (3, 0.4999999999999959, -0.27864199383463517, -0.5538737782802994),\n                        (1, 0.7500000000000032, 0.34032278598431986, 0.5696501318346623),\n                        (0, 0.9999999999999996, -0.3648932378899361, -0.9778728802180585),\n                        (5, 0.25000000000000006, 0.0, 0.5),\n                        (6, 1.0, 0.0, 0.5),\n                        (7, 0.7499999999999958, -0.7659601398809079, -0.8571083685455292),\n                        (8, 0.4999999999999959, -0.27864199383463517, -0.5538737782802994),\n                        (10, 0.7500000000000032, 0.34032278598431986, 0.5696501318346623),\n                        (9, 0.9999999999999996, -0.3648932378899361, -0.9778728802180585)],\n                       [(2, 3), (1, 0), (4, 5), (7, 8), (10, 9)],\n                       [(2, 0.6193218529801312, 0.6736152698514805, -0.12306850842643091),\n                        (0, 0.9999999999999993, 0.9339171918431297, -0.5315283251854057),\n                        (3, 0.4999999999999959, -0.0017782357105362578, -0.3325157721149319),\n                        (1, 0.2499999999999959, -0.20581531379718232, 0.9099729178189659),\n                        (5, 0.5, 0.0, -0.5),\n                        (7, 0.6193218529801312, 0.6736152698514805, -0.12306850842643091),\n                        (9, 0.9999999999999993, 0.9339171918431297, -0.5315283251854057),\n                        (8, 0.4999999999999959, -0.0017782357105362578, -0.3325157721149319),\n                        (10, 0.2499999999999959, -0.20581531379718232, 0.9099729178189659),\n                        (4, 0.5, 0.0, 0.5)],\n                       [(2, 0), (3, 1), (5, 6), (7, 9), (8, 10)],\n                       [(2, 0.5000000000000003, 0.19870280658969874, -0.44945323857494657),\n                        (3, 0.4999999999999974, 0.4147080617789417, 0.2850278451546754),\n                        (1, 1.4972508298281808e-14, 0.4091233012629609, -0.5399326330322662),\n                        (0, 0.5000000000000108, 0.17203203350261553, 0.9357092063272923),\n                        (7, 0.5000000000000003, 0.19870280658969874, -0.44945323857494657),\n                        (8, 0.4999999999999974, 0.4147080617789417, 0.2850278451546754),\n                        (10, 1.4972508298281808e-14, 0.4091233012629609, -0.5399326330322662),\n                        (9, 0.5000000000000108, 0.17203203350261553, 0.9357092063272923),\n                        (5, 0.5000000000000001, 0.0, 0.5)],\n                       [(2, 3), (1, 0), (7, 8), (10, 9)],\n                       [(2, 0.25818731204330203, 0.2147600793762427, 0.24924956801474624),\n                        (0, 0.4999999999999808, 0.17553429973119078, 0.10774123982990619),\n                        (3, 0.7581873120432945, -0.33695871832380564, 0.6997359069336127),\n                        (1, 0.9999999999999805, 0.9453700418180587, -0.5716229425559929),\n                        (7, 0.25818731204330203, 0.2147600793762427, 0.24924956801474624),\n                        (9, 0.4999999999999808, 0.17553429973119078, 0.10774123982990619),\n                        (8, 0.7581873120432945, -0.33695871832380564, 0.6997359069336127),\n                        (10, 0.9999999999999805, 0.9453700418180587, -0.5716229425559929)],\n                       [(2, 0), (3, 1), (7, 9), (8, 10)],\n                       [(2, 0.4999999999999981, 0.13136895120953168, -0.035990352609010845),\n                        (3, 0.5000000000000023, -0.7673985900092897, -0.13722281139019699),\n                        (1, 5.015629698628501e-15, 0.30746064351072583, 0.4267597788467477),\n                        (0, 3.3831615324158282e-15, -0.286971644629054, -0.7418832441465113),\n                        (7, 0.4999999999999981, 0.13136895120953168, -0.035990352609010845),\n                        (8, 0.5000000000000023, -0.7673985900092897, -0.13722281139019699),\n                        (10, 5.015629698628501e-15, 0.30746064351072583, 0.4267597788467477),\n                        (9, 3.3831615324158282e-15, -0.286971644629054, -0.7418832441465113)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (4, 4),\n                      (4, 5),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (6, 4),\n                      (6, 5),\n                      (7, 4),\n                      (7, 5)]},\n (12, 3): {\'method\': \'three-arm hub seed + parallel square decoders\',\n           \'moments\': [[(2, 0.7788200685577452, -0.7508507123729302, -0.22653708940155637),\n                        (6, 0.22526397327149908, -0.7997119445015537, 0.7738082363977972),\n                        (3, 0.2163099431221632, -0.4185868243287605, 0.10326746393167802),\n                        (7, 0.30508211161021925, -0.09912935990949157, -0.522771351755571),\n                        (4, 0.4759637216047896, -0.3220042360455595, -0.3658823732298373),\n                        (8, 0.5890366802284109, 0.47878475937664344, 0.5524754962750926),\n                        (0, 0.5390087196324642, -0.8603924452310396, -0.691353199242738),\n                        (1, 0.03900871963245432, -0.1066856530705218, -0.9660439767483449),\n                        (10, 0.5390087196324642, -0.8603924452310396, -0.691353199242738),\n                        (11, 0.03900871963245432, -0.1066856530705218, -0.9660439767483449),\n                        (5, 0.5390087196324642, -0.8603924452310396, -0.691353199242738),\n                        (9, 0.03900871963245432, -0.1066856530705218, -0.9660439767483449)],\n                       [(2, 6), (3, 7), (4, 8)],\n                       [(3, 0.39409386583047534, -0.948746575138194, -0.4248651062141926),\n                        (4, 0.7264615133145446, -0.6428799368238742, 0.8132657256962459),\n                        (6, 0.34314612704543035, 0.8341357789668938, -0.8517316032427287),\n                        (7, 0.5741280070063655, -0.3488121594549969, -0.35440409429221864),\n                        (2, 0.17188071295477142, -0.42060383361749865, -0.6717693096779784),\n                        (8, 0.25517010973290954, 0.9786458615592148, 0.24788483067728878)],\n                       [(3, 4), (6, 7)],\n                       [(6, 0.5021506537559169, 0.25080713145718714, 0.015373941534277735),\n                        (3, 0.23277329606553482, 0.45833090210555294, -0.6214671883117859),\n                        (4, 0.3456860273646317, 0.5910302493260993, 0.6453503301336359),\n                        (7, 0.8946592044247962, -0.13952161834585297, -0.030506440729260564)],\n                       [(2, 6), (3, 4), (7, 8)],\n                       [(2, 0.4988840913817213, 0.17993283542147864, 0.7941665852273287),\n                        (6, 0.3584738628825403, -0.15799971462964746, -0.2828858415552644),\n                        (3, 0.7443296695021351, -0.1661608270590671, -0.3253473591824827),\n                        (7, 0.43205710957134846, -0.5074109724436164, 0.14284195083069073),\n                        (4, 0.6165265584396299, 0.46719667652548974, 0.4168996085132504),\n                        (8, 0.777609005398163, 0.5943585260999282, 0.054209239960272215)],\n                       [(2, 6), (3, 7), (4, 8)],\n                       [(2, 9.87774369787353e-16, 0.7797481370198334, -0.7244204198460462),\n                        (3, 0.5703421065405238, 0.8324356577888965, 0.6493597147095036),\n                        (6, 0.1957764380156699, -0.941175157402828, 0.09508556311617589),\n                        (7, 0.6841699392079917, -0.1519368782976902, -0.123821087312175),\n                        (4, 0.5525518623843082, -0.6437024419978223, 0.05991029649006546),\n                        (8, 0.37723358859350964, -0.16128764860877265, -0.37316699844115364)],\n                       [(2, 3), (6, 7), (4, 8)],\n                       [(2, 0.5045233232683171, -0.4344158644741636, 0.2538475576686403),\n                        (6, 0.5193462765801522, -0.7906045721389929, -0.08188914795498503),\n                        (3, 0.5, -0.2813259964690129, -0.9357959427517528),\n                        (4, 0.3107904189763934, -0.7377494263696214, 0.8478691624365837),\n                        (7, 0.4735647734845931, -0.6644288761712425, 0.8997794251014564),\n                        (8, 0.6952784748875307, 0.4932089029607647, -0.6615210923727806)],\n                       [(2, 6), (3, 4), (7, 8)],\n                       [(2, 0.5981612323387153, -0.4967208850192266, -0.6805683068055235),\n                        (3, 0.5, -0.651857967640397, 0.7828780607792343),\n                        (7, 0.9578278956144373, -0.6084975432594704, 0.7523131299879338),\n                        (8, 0.19804234294346126, -0.0019436402934576158, -0.23297646598092459),\n                        (6, 0.10794449395394996, 0.27218527236080536, -0.8700438525379843),\n                        (4, 9.545882748371192e-17, 0.8436943086315054, 0.6411663539878527)],\n                       [(2, 3), (7, 8)],\n                       [(2, 1.6948258748711464e-16, -0.9729616096138198, 0.9413843865476255),\n                        (3, 0.45857265913042705, -0.7555724826850818, 0.17029609976392113),\n                        (7, 0.47628779075776295, -0.6229678773043492, -0.24814914083537465),\n                        (8, 0.5741588829153069, 0.17409647612450152, 0.41811204748885544)],\n                       [(2, 6), (3, 7), (4, 8)],\n                       [(2, 0.4999999999999909, 0.27098254205161076, 0.8497491985614309),\n                        (3, 0.0815173357809869, 0.8351615521593942, 0.12438327706997732),\n                        (6, 0.43701654583088695, -0.15593524784026758, 0.39954360922557075),\n                        (7, 0.43474705370127853, 0.24098947078931354, -0.6447500610643551),\n                        (4, 0.9999999999999907, 0.5098862516632958, 0.564877882699125),\n                        (8, 0.7597561194575178, -0.40327140466614253, 0.9918733596040663)],\n                       [(2, 0), (3, 1), (6, 10), (7, 11), (4, 5), (8, 9)],\n                       [(2, 0.4999999999999988, 0.7855375402055449, 0.26368556094540857),\n                        (3, 3.8567868908521176e-14, -0.7138477097683644, 0.8623864896562614),\n                        (1, 0.9609912803675337, -0.2654416119225229, -0.07272962981882314),\n                        (0, 0.22652038258320545, 0.8444189779561107, -0.006127787872109769),\n                        (6, 0.4999999999999988, 0.7855375402055449, 0.26368556094540857),\n                        (7, 3.8567868908521176e-14, -0.7138477097683644, 0.8623864896562614),\n                        (11, 0.9609912803675337, -0.2654416119225229, -0.07272962981882314),\n                        (10, 0.22652038258320545, 0.8444189779561107, -0.006127787872109769),\n                        (4, 0.4999999999999988, 0.7855375402055449, 0.26368556094540857),\n                        (8, 3.8567868908521176e-14, -0.7138477097683644, 0.8623864896562614),\n                        (9, 0.9609912803675337, -0.2654416119225229, -0.07272962981882314),\n                        (5, 0.22652038258320545, 0.8444189779561107, -0.006127787872109769)],\n                       [(2, 3), (1, 0), (6, 7), (11, 10), (4, 8), (9, 5)],\n                       [(2, 0.4999999999999983, 0.8421982859273518, -0.9507768988490481),\n                        (0, 0.5000000000003056, -0.7652187161819077, -0.9461330490265849),\n                        (3, 6.745364167227603e-15, -0.5883266847249748, -0.15165003845294356),\n                        (1, 0.7499999999999835, -0.40061767531814807, 0.4381118840480963),\n                        (6, 0.4999999999999983, 0.8421982859273518, -0.9507768988490481),\n                        (10, 0.5000000000003056, -0.7652187161819077, -0.9461330490265849),\n                        (7, 6.745364167227603e-15, -0.5883266847249748, -0.15165003845294356),\n                        (11, 0.7499999999999835, -0.40061767531814807, 0.4381118840480963),\n                        (4, 0.4999999999999983, 0.8421982859273518, -0.9507768988490481),\n                        (5, 0.5000000000003056, -0.7652187161819077, -0.9461330490265849),\n                        (8, 6.745364167227603e-15, -0.5883266847249748, -0.15165003845294356),\n                        (9, 0.7499999999999835, -0.40061767531814807, 0.4381118840480963)],\n                       [(2, 0), (3, 1), (6, 10), (7, 11), (4, 5), (8, 9)],\n                       [(2, 0.49999999999999684, -0.6050588866550652, 0.14394699237253755),\n                        (3, 0.5000000000000038, -0.8845692780498959, -0.12117258793593289),\n                        (1, 0.74999999999998, 0.08003009104082975, 0.03749420872993947),\n                        (0, 0.637618967553187, -0.0031023457864229886, 0.47100340349251263),\n                        (6, 0.49999999999999684, -0.6050588866550652, 0.14394699237253755),\n                        (7, 0.5000000000000038, -0.8845692780498959, -0.12117258793593289),\n                        (11, 0.74999999999998, 0.08003009104082975, 0.03749420872993947),\n                        (10, 0.637618967553187, -0.0031023457864229886, 0.47100340349251263),\n                        (4, 0.49999999999999684, -0.6050588866550652, 0.14394699237253755),\n                        (8, 0.5000000000000038, -0.8845692780498959, -0.12117258793593289),\n                        (9, 0.74999999999998, 0.08003009104082975, 0.03749420872993947),\n                        (5, 0.637618967553187, -0.0031023457864229886, 0.47100340349251263)],\n                       [(2, 3), (1, 0), (6, 7), (11, 10), (4, 8), (9, 5)],\n                       [(2, 0.5000000000000003, 0.6040035257369907, -0.4611118942825243),\n                        (0, 0.5780174392649265, 0.012788763989812324, -0.7154671565497159),\n                        (3, 0.5000000000000044, 0.20186808770552966, -0.5057418659858346),\n                        (1, 0.9999999999999954, -0.012967077279665062, 0.06339120933111991),\n                        (6, 0.5000000000000003, 0.6040035257369907, -0.4611118942825243),\n                        (10, 0.5780174392649265, 0.012788763989812324, -0.7154671565497159),\n                        (7, 0.5000000000000044, 0.20186808770552966, -0.5057418659858346),\n                        (11, 0.9999999999999954, -0.012967077279665062, 0.06339120933111991),\n                        (4, 0.5000000000000003, 0.6040035257369907, -0.4611118942825243),\n                        (5, 0.5780174392649265, 0.012788763989812324, -0.7154671565497159),\n                        (8, 0.5000000000000044, 0.20186808770552966, -0.5057418659858346),\n                        (9, 0.9999999999999954, -0.012967077279665062, 0.06339120933111991)],\n                       [(2, 0), (3, 1), (6, 10), (7, 11), (4, 5), (8, 9)],\n                       [(2, 0.7499999999999958, -0.7659601398809079, -0.8571083685455292),\n                        (3, 0.4999999999999959, -0.27864199383463517, -0.5538737782802994),\n                        (1, 0.7500000000000032, 0.34032278598431986, 0.5696501318346623),\n                        (0, 0.9999999999999996, -0.3648932378899361, -0.9778728802180585),\n                        (6, 0.7499999999999958, -0.7659601398809079, -0.8571083685455292),\n                        (7, 0.4999999999999959, -0.27864199383463517, -0.5538737782802994),\n                        (11, 0.7500000000000032, 0.34032278598431986, 0.5696501318346623),\n                        (10, 0.9999999999999996, -0.3648932378899361, -0.9778728802180585),\n                        (4, 0.7499999999999958, -0.7659601398809079, -0.8571083685455292),\n                        (8, 0.4999999999999959, -0.27864199383463517, -0.5538737782802994),\n                        (9, 0.7500000000000032, 0.34032278598431986, 0.5696501318346623),\n                        (5, 0.9999999999999996, -0.3648932378899361, -0.9778728802180585)],\n                       [(2, 3), (1, 0), (6, 7), (11, 10), (4, 8), (9, 5)],\n                       [(2, 0.6193218529801312, 0.6736152698514805, -0.12306850842643091),\n                        (0, 0.9999999999999993, 0.9339171918431297, -0.5315283251854057),\n                        (3, 0.4999999999999959, -0.0017782357105362578, -0.3325157721149319),\n                        (1, 0.2499999999999959, -0.20581531379718232, 0.9099729178189659),\n                        (6, 0.6193218529801312, 0.6736152698514805, -0.12306850842643091),\n                        (10, 0.9999999999999993, 0.9339171918431297, -0.5315283251854057),\n                        (7, 0.4999999999999959, -0.0017782357105362578, -0.3325157721149319),\n                        (11, 0.2499999999999959, -0.20581531379718232, 0.9099729178189659),\n                        (4, 0.6193218529801312, 0.6736152698514805, -0.12306850842643091),\n                        (5, 0.9999999999999993, 0.9339171918431297, -0.5315283251854057),\n                        (8, 0.4999999999999959, -0.0017782357105362578, -0.3325157721149319),\n                        (9, 0.2499999999999959, -0.20581531379718232, 0.9099729178189659)],\n                       [(2, 0), (3, 1), (6, 10), (7, 11), (4, 5), (8, 9)],\n                       [(2, 0.5000000000000003, 0.19870280658969874, -0.44945323857494657),\n                        (3, 0.4999999999999974, 0.4147080617789417, 0.2850278451546754),\n                        (1, 1.4972508298281808e-14, 0.4091233012629609, -0.5399326330322662),\n                        (0, 0.5000000000000108, 0.17203203350261553, 0.9357092063272923),\n                        (6, 0.5000000000000003, 0.19870280658969874, -0.44945323857494657),\n                        (7, 0.4999999999999974, 0.4147080617789417, 0.2850278451546754),\n                        (11, 1.4972508298281808e-14, 0.4091233012629609, -0.5399326330322662),\n                        (10, 0.5000000000000108, 0.17203203350261553, 0.9357092063272923),\n                        (4, 0.5000000000000003, 0.19870280658969874, -0.44945323857494657),\n                        (8, 0.4999999999999974, 0.4147080617789417, 0.2850278451546754),\n                        (9, 1.4972508298281808e-14, 0.4091233012629609, -0.5399326330322662),\n                        (5, 0.5000000000000108, 0.17203203350261553, 0.9357092063272923)],\n                       [(2, 3), (1, 0), (6, 7), (11, 10), (4, 8), (9, 5)],\n                       [(2, 0.25818731204330203, 0.2147600793762427, 0.24924956801474624),\n                        (0, 0.4999999999999808, 0.17553429973119078, 0.10774123982990619),\n                        (3, 0.7581873120432945, -0.33695871832380564, 0.6997359069336127),\n                        (1, 0.9999999999999805, 0.9453700418180587, -0.5716229425559929),\n                        (6, 0.25818731204330203, 0.2147600793762427, 0.24924956801474624),\n                        (10, 0.4999999999999808, 0.17553429973119078, 0.10774123982990619),\n                        (7, 0.7581873120432945, -0.33695871832380564, 0.6997359069336127),\n                        (11, 0.9999999999999805, 0.9453700418180587, -0.5716229425559929),\n                        (4, 0.25818731204330203, 0.2147600793762427, 0.24924956801474624),\n                        (5, 0.4999999999999808, 0.17553429973119078, 0.10774123982990619),\n                        (8, 0.7581873120432945, -0.33695871832380564, 0.6997359069336127),\n                        (9, 0.9999999999999805, 0.9453700418180587, -0.5716229425559929)],\n                       [(2, 0), (3, 1), (6, 10), (7, 11), (4, 5), (8, 9)],\n                       [(2, 0.4999999999999981, 0.13136895120953168, -0.035990352609010845),\n                        (3, 0.5000000000000023, -0.7673985900092897, -0.13722281139019699),\n                        (1, 5.015629698628501e-15, 0.30746064351072583, 0.4267597788467477),\n                        (0, 3.3831615324158282e-15, -0.286971644629054, -0.7418832441465113),\n                        (6, 0.4999999999999981, 0.13136895120953168, -0.035990352609010845),\n                        (7, 0.5000000000000023, -0.7673985900092897, -0.13722281139019699),\n                        (11, 5.015629698628501e-15, 0.30746064351072583, 0.4267597788467477),\n                        (10, 3.3831615324158282e-15, -0.286971644629054, -0.7418832441465113),\n                        (4, 0.4999999999999981, 0.13136895120953168, -0.035990352609010845),\n                        (8, 0.5000000000000023, -0.7673985900092897, -0.13722281139019699),\n                        (9, 5.015629698628501e-15, 0.30746064351072583, 0.4267597788467477),\n                        (5, 3.3831615324158282e-15, -0.286971644629054, -0.7418832441465113)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 4),\n                      (6, 5)]},\n (13, 3): {\'method\': \'three-arm hub seed + parallel square decoders\',\n           \'moments\': [[(3, 0.22639989947272102, 0.0, 0.5),\n                        (5, 0.5648042837558234, 0.0, -0.5),\n                        (2, 0.5011286333212674, 0.0, -0.5),\n                        (9, 0.35843654395441843, 0.0, 0.5),\n                        (6, 0.3790129682707763, 0.0, 0.5),\n                        (10, 0.6115265567555555, 0.0, -0.5),\n                        (7, 0.4135610417387456, 0.0, -0.5),\n                        (11, 0.4135610417387456, 0.0, -0.5),\n                        (0, 0.4217899985400859, 0.0, -0.5001097254686009),\n                        (12, 0.5, 0.0, 0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (4, 0.3578842255793437, 0.0, -0.5503659482326804),\n                        (1, 0.4999999999999983, 0.0, -0.6146222686036573)],\n                       [(9, 10), (3, 6), (2, 5)],\n                       [(9, 0.2081713873978428, 0.0, -0.5),\n                        (10, 0.5109898938650242, 0.0, 0.5),\n                        (5, 0.005466340707054318, 0.0, 0.5),\n                        (6, 0.05896518616556734, 0.0, 0.5),\n                        (2, 0.4944183722198825, 0.0, 0.5),\n                        (3, 0.98879378996015, 2.220446049250313e-16, -0.5000000000000002)],\n                       [(6, 10), (5, 9), (2, 3)],\n                       [(9, 0.9044411812600366, 2.220446049250313e-16, -0.5000000000000002),\n                        (10, 0.5357339946022933, 0.0, -0.5),\n                        (6, 0.2666010547086309, 1.0, -0.5),\n                        (2, 0.5, 0.0, -0.5),\n                        (3, 0.7787066990539395, 0.0, 0.5)],\n                       [(5, 6), (9, 10), (2, 3)],\n                       [(10, 0.22637759711444347, 0.0, 0.5),\n                        (9, 0.20484824042867292, 0.0, 0.5),\n                        (6, 0.9592381571300939, 1.0, 0.5),\n                        (5, 0.5, 0.0, 0.5),\n                        (3, 0.0, 1.0, 0.0)],\n                       [(2, 5), (3, 6), (9, 10)],\n                       [(2, 0.5408256347497621, 0.0, -0.5),\n                        (10, 0.8874597674119915, 0.0, 0.5),\n                        (9, 0.6796033860670321, 1.0, 0.5),\n                        (5, 0.6799453846918433, 0.0, 0.5),\n                        (6, 0.9913262427252412, 0.0, -0.5000000000000002)],\n                       [(6, 10), (5, 9), (2, 3)],\n                       [(6, 0.8492358345397026, 0.0, 0.5),\n                        (10, 0.8351725982640147, 0.0, -0.5),\n                        (5, 0.5122437211025592, 0.0, 0.5),\n                        (3, 0.4769944402767312, 0.0, -0.5),\n                        (2, 0.35408611734683687, 0.0, -0.5),\n                        (9, 0.0, 1.0, 0.0)],\n                       [(9, 10), (2, 5), (3, 6)],\n                       [(3, 0.8130648608503948, 1.0, -0.5000000000000002),\n                        (6, 0.43942608628445434, 0.0, -0.5),\n                        (2, 0.4999999999999994, 0.0, 0.5),\n                        (10, 0.6237193013742914, 0.0, 0.5),\n                        (5, 0.5, 0.0, -0.5)],\n                       [(2, 3), (6, 10), (5, 0.0, 1.0, 0.0)],\n                       [(3, 0.038115023066137795, 1.0, -0.5000000000000002),\n                        (10, 0.6383902508124722, 0.0, 0.5),\n                        (6, 0.6322754934663775, 0.0, 0.5),\n                        (2, 0.0, -0.004280799300016014, 0.0),\n                        (5, 9)],\n                       [(9, 10), (5, 6), (2, 3)],\n                       [(6, 0.5591170679959561, 0.0, -0.5),\n                        (3, 0.7955002753638748, 0.0, 0.34934788086467927),\n                        (10, 0.10047114062159834, 0.0, -0.5000000000000002)],\n                       [(10, 11), (3, 0), (6, 7)],\n                       [(10, 0.3819668494768557, 1.0, 0.5),\n                        (6, 0.3819668494768557, 0.0, 0.5),\n                        (3, 0.8116669186835013, 4.440892098500626e-16, -0.9569736579465713),\n                        (7, 0.633860236400615, 0.0, -0.5),\n                        (11, 0.633860236400615, 0.0, -0.5),\n                        (0, 0.6287848321594933, 0.0, -0.5144920515457643)],\n                       [(2, 3), (5, 6), (9, 10)],\n                       [(10, 0.7376490383100944, 0.0, 0.5),\n                        (3, 0.7186000038451317, 0.0, 0.06472093450165328),\n                        (6, 0.7376490383100944, 0.0, 0.5)],\n                       [(3, 0), (6, 7), (10, 11)],\n                       [(10, 0.16018463696253968, 0.0, 0.5),\n                        (6, 0.16018463696253968, 1.0, 0.5),\n                        (3, 0.18796960844370011, 0.0, 0.28935500851985013),\n                        (7, 0.180239329875864, 0.0, -0.5000000000000002),\n                        (11, 0.180239329875864, 0.0, -0.5000000000000002),\n                        (0, 0.1426534639632415, 0.0, 0.23422019484333445)],\n                       [(9, 10), (5, 6), (2, 3)],\n                       [(10, 0.44223174538702703, 0.0, 0.5),\n                        (3, 0.36815249285725926, 0.0, 0.7413503891189102),\n                        (6, 0.44223174538702703, 0.0, 0.5),\n                        (2, 0.5, 0.0, 0.5028353753199111),\n                        (5, 0.5, 0.0, 0.5),\n                        (9, 0.5, 0.0, 0.5)],\n                       [(10, 11), (3, 0), (6, 7)],\n                       [(11, 0.8040867239846965, 0.0, -0.5000000000000002),\n                        (7, 0.8040867239846965, 0.0, -0.5000000000000002),\n                        (0, 0.5401993801370204, 0.0, -0.5512342235842844),\n                        (3, 0.5762322444517163, 0.0, -0.2894137931753753),\n                        (6, 0.6666666666666669, 1.0, 0.5),\n                        (10, 0.6666666666666669, 0.0, 0.5)],\n                       [(3, 4), (7, 8), (0, 1), (11, 12), (5, 6), (9, 10)],\n                       [(1, 0.3260301916599948, 0.0, 0.43315900579188815),\n                        (11, 0.5, 1.0, -0.5),\n                        (0, 0.15204336199234872, 1.0, 0.9346037967062348),\n                        (4, 0.8053712368962477, 0.0, 0.18888285213015016),\n                        (7, 0.5, 1.0, -0.5),\n                        (8, 0.3040867239846962, 0.0, -0.5000000000000002),\n                        (12, 0.3040867239846962, 0.0, -0.5000000000000002),\n                        (3, 0.0, 0.7024201146828472, 0.0),\n                        (9, 0.5, 0.0, -0.5),\n                        (5, 0.5, 0.0, 0.5)],\n                       [(10, 11), (6, 7), (1, 4), (3, 0)],\n                       [(11, 0.8040867239846965, 5.551115123125783e-17, 0.49999999999999994),\n                        (0, 0.5, 0.0, -0.5653962032937654),\n                        (7, 0.8040867239846965, 5.551115123125783e-17, 0.49999999999999994),\n                        (1, 0.5, 0.0, 0.8430180222691617),\n                        (4, 0.6008134304054109, 0.0, -0.040123990740045024),\n                        (2, 3)],\n                       [(7, 8), (0, 1), (11, 12), (3, 4), (2, 0.5, -0.10090924030729465, 0.5028353753199111)],\n                       [(1, 0.5, 0.0, -0.6569819777308383),\n                        (0, 0.5924401224278535, 1.0, -0.9133528413014171),\n                        (4, 0.3991865695945891, 0.0, 0.959876009259955),\n                        (11, 0.75, 0.0, 0.5),\n                        (7, 0.75, 0.0, 0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (12, 0.5, 0.0, 0.5),\n                        (3, 0.0, 0.8715279661740892, 0.0)],\n                       [(10, 11), (1, 4), (3, 0), (6, 7)],\n                       [(7, 0.5, 0.0, -0.5),\n                        (0, 0.9075598775721465, 0.0, -0.9133528413014158),\n                        (11, 0.5, 0.0, -0.5),\n                        (1, 0.25040914996542357, 0.0, 0.3268926652168218),\n                        (4, 0.1946287631037519, 0.0, 0.2308691663897604),\n                        (6, 0.5, 0.0, 0.5),\n                        (10, 0.5, 0.0, 0.5)],\n                       [(3, 4), (0, 1), (7, 8), (11, 12)],\n                       [(11, 0.25000000000000006, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.33664715869858547),\n                        (7, 0.25000000000000006, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.3497071393883171),\n                        (4, 0.7587162401878453, 4.440892098500626e-16, 0.8228655323944645),\n                        (3, 0.5, 0.0, 0.5858673545226827)],\n                       [(3, 0), (10, 11), (1, 4), (6, 7)],\n                       [(0, 0.5, 0.0, -0.6633528413014143),\n                        (7, 0.5, 1.0, -0.5),\n                        (3, 0.5000000000000002, -0.9999999999999998, 0.0858673545226834),\n                        (11, 0.5, 1.0, -0.5),\n                        (1, 0.5, 1.1102230246251565e-16, -0.15029286061168312),\n                        (6, 0.5, 4.440892098500626e-16, 0.5),\n                        (10, 0.5, 0.0, 0.5)],\n                       [(7, 8), (11, 12), (3, 4), (0, 1)],\n                       [(1, 0.0, 0.378896047164595, 0.0),\n                        (4, 1.0, 0.0, 0.16765763650036608),\n                        (7, 0.5, 4.440892098500626e-16, 0.5),\n                        (12, 1.0, 0.0, 0.5),\n                        (3, 0.5000000000000001, -0.35726416796977123, -0.4141326454773169),\n                        (11, 0.5, 4.440892098500626e-16, 0.5),\n                        (0, 0.5, -0.10804397214567385, -0.663352841301414),\n                        (8, 1.0, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],\n           \'qubits\': [(3, 5),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 4),\n                      (6, 5),\n                      (6, 6),\n                      (6, 7)]},\n (13, 4): {\'method\': \'certified count kernels + parallel coherent decoders\',\n           \'moments\': [[(6, 0.4255771808057458, 0.0, -0.5),\n                        (4, 0.13672592290297383, 0.0, -0.5000000000000002),\n                        (5, 0.2859133439469217, 0.0, 0.5),\n                        (8, 0.7530877427391665, 0.0, 0.5),\n                        (2, 0.4999999999999998, 0.0, 0.5937872305331664),\n                        (11, 0.5, 0.0, -0.07534461897426747),\n                        (10, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (0, 0.5, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5),\n                        (9, 0.3040867239846966, 0.0, -0.5745192213521528),\n                        (12, 0.3040867239846966, 0.0, -0.5745192213521528)],\n                       [(3, 6), (4, 7), (5, 8)],\n                       [(3, 0.994057967253313, 0.0, 0.5),\n                        (4, 0.8632740770970266, 0.0, -0.5),\n                        (6, 0.1318497038270321, 0.0, 0.5),\n                        (7, 0.17750083138414618, 0.0, 0.5),\n                        (5, 0.2676309800973877, 0.0, -0.5),\n                        (8, 0.674589841540707, 0.0, 0.5)],\n                       [(3, 4), (6, 7)],\n                       [(4, 5),\n                        (3, 0.4940579672533128, 0.0, 0.5),\n                        (6, 0.3291523396328094, 0.0, -0.5),\n                        (7, 0.9877837715850091, 0.0, 0.5)],\n                       [(3, 6),\n                        (7, 8),\n                        (4, 0.5, 0.0, -0.5),\n                        (5, 0.00025773139361362507, 0.0, -0.5000000000000002)],\n                       [(4, 7), (8, 0.49999999999999956, 0.0, -0.5000000000000002)],\n                       [(3, 4), (5, 8), (7, 0.18716727776964084, 0.0, -0.5000000000000002)],\n                       [(6, 7),\n                        (3, 0.4995965996049643, 0.0, 0.5),\n                        (4, 0.9995965996049643, 0.0, 0.5000000000000111),\n                        (5, 0.5000000000000004, 0.0, -0.5),\n                        (8, 0.18450505978277265, 0.0, -0.5)],\n                       [(3, 6), (4, 5), (7, 0.18716727776964115, 0.0, 0.5)],\n                       [(7, 8),\n                        (4, 0.9732593325653318, 0.0, 0.5),\n                        (5, 0.4999999999999994, 0.0, -0.5000000000000002),\n                        (6, 0.4990118679928877, 0.0, -0.5),\n                        (3, 0.025933860165469714, 0.0, 0.5)],\n                       [(4, 7), (5, 8)],\n                       [(6, 7),\n                        (4, 0.5008068072691987, 0.0, 0.5),\n                        (5, 0.5197894010103807, 0.0, 0.0),\n                        (8, 0.3478770495896244, 0.0, -0.24064671678968885)],\n                       [(3, 4),\n                        (5, 2),\n                        (8, 11),\n                        (6, 0.5009881320071123, 0.0, -0.5),\n                        (7, 0.8047304592861082, 0.0, -0.37032335839484487)],\n                       [(6, 7),\n                        (3, 0.5, 0.0, 0.5),\n                        (4, 0.9901052994948099, 0.0, -4.440892098500626e-16),\n                        (8, 0.7913327671706831, 0.0, -0.9197304880405697),\n                        (5, 0.9416885504219519, 0.0, -0.05080010241191735),\n                        (11, 0.6321049414225235, 0.0, 0.17090724808476404),\n                        (2, 0.608535446953489, 0.0, -0.9311758863092963)],\n                       [(3, 4), (7, 0.1554805114857343, 0.0, -0.37032335839484465)],\n                       [(7, 8), (4, 5)],\n                       [(8, 11),\n                        (7, 0.7964417643891555, 0.0, 0.6296766416051542),\n                        (4, 0.09666409909802184, 0.0, 0.9517427976303561),\n                        (5, 0.5, 0.0, 0.8764706651710983)],\n                       [(6, 7),\n                        (3, 4),\n                        (5, 2),\n                        (8, 0.5886108001133223, 0.0, -0.9197304880405694),\n                        (11, 0.597350526917102, 0.0, -0.4537132312119436)],\n                       [(7, 0.7964417643891555, 0.0, 0.6296766416051542),\n                        (4, 0.5812745787265814, 0.0, 0.7458407663045148),\n                        (5, 0.5150139304348058, 0.0, 0.8764706651710983),\n                        (2, 0.16723314080476154, 0.0, -0.7374666288921621)],\n                       [(7, 8), (4, 5)],\n                       [(7, 0.5517093529747915, 0.0, 0.6296766416051554),\n                        (8, 0.1795085179856881, 0.0, -0.9197304880405692),\n                        (4, 0.2955394861408327, 0.0, -0.28156531547965313),\n                        (5, 0.08858950606155513, 0.0, -0.6235293348289017)],\n                       [(6, 7), (8, 11), (3, 4), (5, 2)],\n                       [(7, 0.893493814032364, 0.0, 0.6296766416051556),\n                        (8, 0.11878997884016874, 0.0, -0.9197304880405692),\n                        (4, 0.29430117217170476, 0.0, -0.2593522765723),\n                        (5, 0.30175052808312813, 0.0, 0.8764706651710983),\n                        (6, 0.5000000000000009, 0.0, 0.4351616791974199),\n                        (3, 0.49999999999999967, 0.0, 0.7450526497474059),\n                        (2, 0.6303550437629815, 2.220446049250313e-16, -0.18158983536649442)],\n                       [(7, 8), (4, 5)],\n                       [(8, 11),\n                        (7, 0.5, 0.0, -0.3703233583948444),\n                        (4, 0.30286229487152805, 0.0, 0.7310423582877701),\n                        (5, 0.19954896443512293, 0.0, -0.6235293348289019)],\n                       [(6, 7),\n                        (3, 4),\n                        (5, 2),\n                        (11, 0.7780272931625696, 0.0, 0.20475567610558215),\n                        (8, 0.6444528487918006, 0.0, 0.674707506721316)],\n                       [(11, 10),\n                        (7, 0.5, 0.0, 0.8653652977420201),\n                        (4, 0.5, 0.0, -0.5005069677612313),\n                        (5, 0.22177219480665555, 0.0, -0.12352933482890194),\n                        (3, 0.33347826094116906, 0.0, 0.24505264974740465),\n                        (2, 0.8315920243625678, 3.3306690738754696e-16, -0.4157702878189209),\n                        (6, 0.5, 0.12967664160515813, 0.43516167919741966)],\n                       [(7, 8),\n                        (4, 5),\n                        (11, 0.5000000000000001, 0.0, -0.29524432389441824),\n                        (10, 0.85928593962011, 0.0, 0.07376677451045754)],\n                       [(8, 11),\n                        (7, 0.5, 0.0, -0.1346347022579799),\n                        (4, 0.5, 0.0, -0.5005069677612315),\n                        (5, 0.5, 0.0, 0.3635081377735555)],\n                       [(10, 7),\n                        (3, 4),\n                        (5, 2),\n                        (8, 0.5, 4.440892098500626e-16, -0.8331108945921375),\n                        (11, 0.6714165234362537, 3.3306690738754696e-16, -0.36666041562541096)],\n                       [(7, 8),\n                        (10, 0.7485956033393342, 0.0, -0.42623322548954246),\n                        (4, 0.5519231616233389, 0.0, 0.7172893487704008),\n                        (5, 0.5, 0.0, 0.3635081377735554),\n                        (3, 0.5276550549544043, 0.0, 0.2450526497474046),\n                        (2, 0.5, 0.0, 0.46178536847925344)],\n                       [(11, 10),\n                        (4, 5),\n                        (8, 0.5, 0.0, 0.16688910540786278),\n                        (7, 0.0528060550044127, 0.0, -0.3082350460449499)],\n                       [(11, 0.3935294720438233, 0.0, -0.645707531239033),\n                        (10, 0.9961572718865779, 1.1102230246251565e-16, 0.07376677451045255),\n                        (4, 0.5, 0.0, 0.08314259486480591),\n                        (5, 0.5, 0.0, -0.9359705243585945)],\n                       [(8, 11), (10, 7), (3, 4), (5, 2)],\n                       [(7, 0.24963665794604512, 0.0, -0.8255976309547253),\n                        (11, 0.31043695253214243, 0.0, -0.22718628083883408),\n                        (10, 0.5655532211464794, 0.0, 0.5737667745104469),\n                        (3, 0.5000000000000002, 0.0, -0.25494735025259585),\n                        (4, 0.5, 0.0, -0.9168574051351945),\n                        (2, 0.5, 0.0, -0.038214631520746334)],\n                       [(7, 8), (11, 10), (3, 4), (5, 2)],\n                       [(11, 0.6193705712509703, 0.0, 0.8998480300605713),\n                        (10, 0.4999999999999998, 0.0, -0.42623322548955334),\n                        (7, 0.32701156232958084, 0.0, -0.43337798966662344),\n                        (2, 0.5068340379253959, 0.0, -0.5310585926382718),\n                        (3, 0.4901053274689138, 0.0, -0.2549238179692832),\n                        (4, 0.5, 0.0, -0.9168574051351952),\n                        (5, 0.5, 4.440892098500626e-16, 0.5640294756414055)],\n                       [(8, 11), (10, 7), (2, 0), (3, 1)],\n                       [(11, 0.6256105257587086, 0.0, -0.583422337949058),\n                        (10, 0.5960614607313279, 0.0, -0.11684358694965402),\n                        (7, 0.5, 0.0, 0.3192574788545004),\n                        (3, 0.5, 0.0, 0.24431903208484224),\n                        (2, 0.5, 0.0, -0.7736945421001282),\n                        (0, 0.30408672398469605, 0.0, -0.5000000000000002),\n                        (1, 0.30408672398469605, 0.0, -0.5000000000000002)],\n                       [(8, 11), (10, 7), (4, 3), (5, 2)],\n                       [(7, 0.5, 0.0, -0.1807425211454996),\n                        (8, 0.5, 0.0, 0.8458176761003802),\n                        (11, 0.9764747842833552, 0.0, -0.1812065800284497),\n                        (10, 0.14949404734505523, 0.0, 0.2730671118828738),\n                        (2, 0.8040867239846965, 0.0, 0.22630545789987178),\n                        (3, 0.8040867239846965, 0.0, -0.7556809679151582)],\n                       [(7, 8), (11, 10), (2, 0), (3, 1)],\n                       [(10, 7),\n                        (8, 0.5, 0.0, -0.1541823238996205),\n                        (11, 0.5, 0.0, -0.4241848632103249),\n                        (3, 0.7499999999999999, -2.220446049250313e-16, -0.7556809679151579),\n                        (2, 0.7499999999999999, 5.551115123125783e-17, 0.22630545789987172),\n                        (0, 0.5, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5)],\n                       [(8, 11),\n                        (4, 3),\n                        (5, 2),\n                        (10, 0.42477613785575913, 4.440892098500626e-16, 0.977618148421467)],\n                       [(8, 0.5, 0.0, 0.9130065301190169),\n                        (11, 0.14731871270290633, 0.0, 0.5758151367896751),\n                        (2, 0.5, 1.1102230246251565e-16, 0.226305457899871),\n                        (3, 0.5000000000000002, 0.0, -0.7556809679151575),\n                        (4, 0.5, 0.0, -0.19945981034219384),\n                        (5, 0.5, 0.0, 0.7961195979912681)],\n                       [(7, 8), (11, 10), (2, 0), (3, 1)],\n                       [(11, 0.9433458052208397, 0.0, 0.07581513678967644),\n                        (10, 0.4500869528994627, 1.1102230246251565e-16, -0.3365366861733614),\n                        (7, 0.42020075164346243, 5.551115123125783e-17, 0.11269512044600244),\n                        (3, 0.25000000000000006, 0.0, 0.2443190320848428),\n                        (2, 0.25000000000000006, 0.0, -0.7736945421001293)],\n                       [(8, 11), (10, 7), (4, 3), (5, 2)],\n                       [(8, 0.20567003520391758, 0.0, -0.5869934698809824),\n                        (7, 0.5000000000000001, 0.0, -0.2921089943932724),\n                        (11, 0.5, 0.0, -0.7185148280064055),\n                        (10, 0.5, 0.0, 0.7103548494507246),\n                        (2, 0.5, 0.0, -0.7736945421001282),\n                        (3, 0.5, 0.0, 0.24431903208484224),\n                        (4, 0.5, 0.6994598103421938, 0.8005401896578062),\n                        (5, 0.5, -0.2961195979912681, -0.20388040200873192)],\n                       [(8, 7), (11, 10), (2, 0), (3, 1)],\n                       [(8, 0.49999999999998246, 0.0, 0.9130065864369592),\n                        (11, 0.49999999999998246, 0.0, 0.2814851156756535),\n                        (7, 0.500000000000002, 0.0, 0.6192310760145625),\n                        (10, 0.5000000000000022, 8.881784197001252e-16, 0.621694919858556),\n                        (0, 1.0, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5),\n                        (2, 0.5, -0.7263054578998709, -0.7736945421001293),\n                        (3, 0.5, 0.25568096791515726, 0.24431903208484274)],\n                       [(8, 9), (11, 12)],\n                       [(8, 0.5000000000000102, 0.0, 0.913006586436959),\n                        (11, 0.5000000000000102, 1.1102230246251565e-16, 0.28148511567565293),\n                        (9, 0.30408672398469627, 0.0, -0.5745192213521484),\n                        (12, 0.30408672398469605, 0.0, 0.4254807786478514)],\n                       [(8, 7), (11, 10)],\n                       [(8, 0.75, 0.0, 0.9130066189521366),\n                        (11, 0.7499999999999999, 1.1102230246251565e-16, 0.2814850831604748),\n                        (7, 0.49999999999999956, 0.0, 0.6192310434993804),\n                        (10, 0.49999999999999956, 0.0, -0.37830511265662436)],\n                       [(8, 9), (11, 12)],\n                       [(8, 0.500526889421938, 5.551115123125783e-17, 0.41353355507930084),\n                        (11, 0.500526889421938, 0.0, 0.7809581470333107),\n                        (9, 0.5000000000000102, 0.0, 0.4254807786478514),\n                        (12, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515)],\n                       [(8, 7), (11, 10)],\n                       [(8, 0.5000000000000001, 0.0, -0.8364660088472662),\n                        (11, 0.5000000000000001, 0.0, 0.030957710959877205),\n                        (10, 0.5000000000000003, -2.220446049250313e-16, 0.12169488734337536),\n                        (7, 0.5000000000000003, 0.0, 0.11923104349938013)],\n                       [(8, 9), (11, 12)],\n                       [(11, 10),\n                        (8, 7),\n                        (9, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (12, 0.5, 0.3483047662953689, -0.5745192538673269)],\n                       [(10, 0.0007451670218237004, 0.6520906250846661, 0.6216948873433847),\n                        (11, 3.251517832758566e-08, -0.7571721985318351, 0.030957688294388497),\n                        (7, 0.0007451670218237008, -0.3454455310713369, -0.3807689565006118),\n                        (8, 0.999999967484822, 0.11025147594433077, 0.16353401381822158)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],\n           \'qubits\': [(2, 6),\n                      (3, 4),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 5),\n                      (6, 6),\n                      (6, 7)]},\n (13, 5): {\'method\': \'certified count kernels + parallel coherent decoders\',\n           \'moments\': [[(3, 0.569332469519003, 0.0, 0.5),\n                        (4, 0.5893103323535331, 0.0, -0.5),\n                        (5, 0.1695457497514563, 0.0, 0.5),\n                        (8, 0.8851758861264409, 0.0, 0.5),\n                        (11, 0.7010577384351091, 0.0, 0.9439691995767472),\n                        (2, 0.5, 0.0, -0.016627314735808696),\n                        (10, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (1, 0.5, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (9, 0.30408672398469655, 0.0, -0.5745192213521528),\n                        (12, 0.30408672398469655, 0.0, -0.5745192213521528)],\n                       [(4, 7), (3, 6), (5, 8)],\n                       [(4, 0.8562023004789405, 0.0, 0.5),\n                        (3, 0.8644613585144227, 0.0, 0.5),\n                        (5, 0.583132062570996, 0.0, 0.5),\n                        (6, 0.34796383624416055, 0.0, 0.5),\n                        (8, 0.9217652461188455, 0.0, -0.5000000000000002)],\n                       [(3, 4), (6, 7)],\n                       [(4, 0.9241355121124453, 0.0, 0.5),\n                        (6, 0.8479638362441597, 0.0, -0.5000000000000002),\n                        (3, 0.1027320913617671, 0.0, 0.5),\n                        (7, 8)],\n                       [(4, 5), (3, 6), (8, 0.464092919728691, 0.0, -0.5)],\n                       [(4, 0.9406843667808629, 0.0, -0.5),\n                        (6, 0.5000000000000027, 0.0, 0.5),\n                        (5, 0.3728714828738406, 0.0, 0.5)],\n                       [(4, 7), (5, 8), (3, 6)],\n                       [(7, 0.5, 0.0, 0.5),\n                        (8, 0.47680910051200187, 0.0, 0.5),\n                        (4, 0.8264941771064476, 0.0, 0.4999999999999998),\n                        (5, 0.5543622980045821, 0.0, -0.5)],\n                       [(6, 7), (3, 4), (8, 11)],\n                       [(7, 0.5, 0.0, 0.5),\n                        (4, 0.18806105625269726, 0.0, -0.5),\n                        (11, 0.2989422615648911, 4.440892098500626e-16, 0.943969199576747)],\n                       [(7, 8), (4, 5)],\n                       [(7, 0.5, 0.0, -0.5),\n                        (5, 0.5543622980045824, 5.551115123125783e-17, 0.49999999999999994),\n                        (3, 4)],\n                       [(4, 7), (5, 8)],\n                       [(6, 7),\n                        (5, 0.5538900690326477, 0.0, -0.5423932875027972),\n                        (4, 0.5, 0.0, 0.2708918263538407)],\n                       [(7, 0.5997618693438884, 0.0, 0.11028258081031161), (5, 2), (3, 4)],\n                       [(6, 7),\n                        (5, 0.3385028359086418, 0.0, -0.5420543090898613),\n                        (2, 0.6715957554835205, 0.0, -0.16119008792398848)],\n                       [(7, 0.7021281047129817, 0.0, 0.6455671621175141), (4, 5)],\n                       [(7, 8),\n                        (5, 0.5, -2.220446049250313e-16, 0.15330366590650996),\n                        (4, 0.7292475564750502, 0.0, 0.5646366664160829)],\n                       [(7, 0.7118222410987194, 2.220446049250313e-16, -0.3544328378824866),\n                        (8, 0.666200385537906, 0.0, -0.7259619260711769),\n                        (3, 4),\n                        (5, 2)],\n                       [(8, 11),\n                        (6, 7),\n                        (4, 0.8236951914845039, 5.551115123125783e-17, 0.469077579537541),\n                        (5, 0.13681175462965722, 0.0, 0.1533036659065099),\n                        (2, 0.27172923178718184, 0.0, -0.7559786295779003)],\n                       [(8, 0.666200385537906, 0.0, -0.7259619260711769),\n                        (7, 0.9813476461135342, 0.0, -0.35443283788248725),\n                        (11, 0.4198773247206662, 0.0, -0.07074532720037463),\n                        (4, 5)],\n                       [(7, 8),\n                        (4, 0.45188669732231024, 0.0, 0.8324908622148373),\n                        (5, 0.3054490683281037, 0.0, -0.3466963340934899)],\n                       [(8, 0.6135871361834435, 0.0, -0.6490297408388621),\n                        (7, 0.00209965350163821, 0.0, -0.3544328378824817),\n                        (3, 4),\n                        (5, 2)],\n                       [(6, 7),\n                        (8, 11),\n                        (4, 0.5707034495925997, 0.0, 0.773686284461375),\n                        (5, 0.785237728664876, 8.881784197001252e-16, -0.8466963340934905),\n                        (2, 0.47240753175963274, 0.0, 0.657484274013953),\n                        (3, 0.5000000000000036, 0.0, -0.364554086823079)],\n                       [(8, 0.19491599185619116, 0.0, 0.28262019924341786),\n                        (7, 0.1651746478714715, 0.0, 0.6455671621175119),\n                        (11, 0.5000000000000014, 5.551115123125783e-17, 0.375466772284803),\n                        (4, 5)],\n                       [(7, 8),\n                        (5, 0.5858994169251085, 0.0, -0.3466963340934903),\n                        (4, 0.5419474843487609, 0.0, -0.06312269410049831)],\n                       [(7, 0.5846001477399267, 0.0, -0.3544328378824879), (8, 11), (3, 4), (5, 2)],\n                       [(6, 7),\n                        (8, 0.5886108001133223, 0.0, 0.8711660270602777),\n                        (11, 0.48187963875508855, 4.440892098500626e-16, 0.6712802634947104),\n                        (4, 0.5, 0.0, 0.04427543724988775),\n                        (2, 0.643422681037554, 0.0, 0.11543360081664167),\n                        (5, 0.15794180998375126, 0.0, 0.15330366590650923),\n                        (3, 0.5, 0.0, -0.3645540868230759)],\n                       [(7, 0.20938585749425953, 0.0, -0.35443283788248836), (5, 2)],\n                       [(7, 8), (4, 5), (2, 0.6811220971645664, 0.0, -0.34617472538598015)],\n                       [(8, 0.7627370268849369, -2.220446049250313e-16, -0.12883397293972187),\n                        (7, 0.4482906470252086, 0.0, 0.6455671621175119),\n                        (5, 0.5, 0.0, -0.346696334093491),\n                        (4, 0.10589866865514243, 0.0, -0.45572456275011297)],\n                       [(8, 11), (6, 7), (4, 5)],\n                       [(7, 0.893493814032364, 0.0, -0.35443283788248925),\n                        (8, 0.40335427567076765, 0.0, 0.8711660270602788),\n                        (6, 0.5, 0.0, 0.3028118498235144),\n                        (3, 4),\n                        (5, 0.33953119101123347, 0.0, 0.6533036659065095)],\n                       [(7, 8),\n                        (3, 0.49089820890137065, 0.0, -0.7484547709418348),\n                        (4, 0.42589339437843377, 0.0, -0.9557245627501127),\n                        (5, 2)],\n                       [(7, 0.035196301149107656, 0.0, 0.6455671621175108),\n                        (8, 0.8339515505182198, 0.0, 0.8711660270602313),\n                        (3, 4),\n                        (5, 0.5, 0.0, -0.34669633409349077)],\n                       [(8, 11), (6, 7), (4, 0.5, 0.0, -0.9557245627501127)],\n                       [(8, 0.9352064365349756, 5.551115123125783e-17, 0.34290504099508373),\n                        (7, 0.5, 0.0, -0.25109401111953433),\n                        (11, 0.7780272931625696, 0.0, 0.05572810904583947),\n                        (4, 5),\n                        (6, 0.5, -0.6056236996470283, 0.3028118498235144)],\n                       [(7, 8),\n                        (11, 10),\n                        (4, 0.0035868934861584814, 0.0, 0.5442754372498917),\n                        (5, 0.5558460531129372, 0.0, 0.15330366590651012)],\n                       [(7, 0.5, 0.0, 0.7489059888804654),\n                        (10, 0.85928593962011, 0.0, -0.9262332254895425),\n                        (11, 0.5, 0.0, 0.555728109045839),\n                        (3, 4),\n                        (5, 2)],\n                       [(10, 7),\n                        (8, 11),\n                        (2, 0.7708918263538406, 0.0, -0.39597210290977203),\n                        (3, 0.7708918263538402, 0.0, 0.5008243714348846),\n                        (5, 0.4999999999999998, 0.0, -0.3466963340934899),\n                        (4, 0.5, 0.0, -0.9557245627501127)],\n                       [(10, 0.7485956033393342, 0.0, -0.42623322548954246),\n                        (8, 0.5, 0.0, 0.01570019269570283),\n                        (11, 0.6714165234362537, 2.220446049250313e-16, -0.3728557992231676),\n                        (3, 1),\n                        (2, 0)],\n                       [(11, 10),\n                        (7, 8),\n                        (3, 0.5, 0.0, -0.9991756285651157),\n                        (2, 0.5, 0.0, 0.10402789709022792),\n                        (0, 0.3040867239846962, 0.0, 0.4999999999999998),\n                        (1, 0.3040867239846962, 0.0, 0.4999999999999998)],\n                       [(8, 0.5, 0.0, 0.015700192695703663),\n                        (11, 0.3935294720438234, 0.0, -0.09380868360954597),\n                        (7, 0.052806055004412704, 0.0, -0.07749366733256124),\n                        (10, 0.9961572718865779, 0.0, -0.9262332254895478),\n                        (4, 3),\n                        (5, 2)],\n                       [(8, 11),\n                        (10, 7),\n                        (3, 0.8040867239846965, 0.0, -0.9991756285651157),\n                        (2, 0.8040867239846965, 1.1102230246251565e-16, 0.10402789709022814)],\n                       [(7, 0.24963665794604517, 0.0, 0.43986891757721064),\n                        (11, 0.3104369525321424, 0.0, 0.4876700659902544),\n                        (10, 0.5655532211464794, 0.0, 0.5737667745104469),\n                        (2, 0),\n                        (3, 1)],\n                       [(7, 8),\n                        (11, 10),\n                        (2, 0.75, 0.0, 0.10402789709022814),\n                        (3, 0.75, 0.0, -0.9991756285651157),\n                        (1, 0.5, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5)],\n                       [(7, 0.3270115623295809, 8.881784197001252e-16, -0.9523507237108906),\n                        (11, 0.6193705712509703, 0.0, -0.6393642449091499),\n                        (10, 0.4999999999999998, 0.0, 0.5737667745104462),\n                        (5, 2),\n                        (4, 3)],\n                       [(10, 7),\n                        (8, 11),\n                        (3, 0.5, -2.220446049250313e-16, 0.0008243714348843967),\n                        (2, 0.5, 0.0, -0.895972102909772),\n                        (5, 0.5, 0.0, -0.7112504209165704),\n                        (4, 0.5, 0.0, 0.6797213504268071)],\n                       [(11, 0.6256105257587086, 0.0, -0.15609387689952037),\n                        (10, 0.5960614607313279, -2.220446049250313e-16, -0.11684358694965402),\n                        (7, 0.5, 5.551115123125783e-17, 0.2950138077679862),\n                        (3, 1),\n                        (2, 0)],\n                       [(8, 11),\n                        (10, 7),\n                        (3, 0.25000000000000006, 0.0, -0.9991756285651157),\n                        (2, 0.25000000000000006, 0.0, 0.10402789709022825)],\n                       [(7, 0.5, 0.0, 0.7950138077679869),\n                        (8, 0.5, 0.0, -0.6632283779968149),\n                        (10, 0.14949404734505523, 0.0, -0.7269328881171262),\n                        (11, 0.9764747842833552, 0.0, 0.4416903651798709),\n                        (5, 2),\n                        (4, 3)],\n                       [(11, 10),\n                        (7, 8),\n                        (2, 0.5, 0.0, -0.895972102909772),\n                        (3, 0.5, -2.220446049250313e-16, 0.0008243714348843967),\n                        (5, 0.5, 0.21125042091657048, 0.2887495790834295),\n                        (4, 0.5, 0.8202786495731929, -0.32027864957319285)],\n                       [(11, 0.5, 0.0, 0.6846686483617455),\n                        (8, 0.5, 0.0, -0.6632283779968149),\n                        (10, 7),\n                        (3, 1),\n                        (2, 0)],\n                       [(8, 11),\n                        (10, 0.4247761378557593, 0.0, -0.022381851578533896),\n                        (0, 1.0, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5),\n                        (2, 0.4999999999999998, -0.604027897090228, 0.10402789709022792),\n                        (3, 0.4999999999999998, 0.4991756285651152, -0.9991756285651157)],\n                       [(8, 0.5, 0.0, -0.7304172320154523),\n                        (11, 0.14731871270290633, 0.0, 0.684668648361745)],\n                       [(7, 8), (11, 10)],\n                       [(10, 0.45008695289946266, 0.0, -0.3365366861733614),\n                        (11, 0.9433458052208397, 0.0, -0.8153313516382552),\n                        (7, 0.4202007516434625, 8.881784197001252e-16, 0.5015761661764835)],\n                       [(8, 11), (10, 7)],\n                       [(8, 0.20567003520391733, 0.0, 0.7695827679845484),\n                        (10, 0.5, 0.0, -0.2896451505492772),\n                        (11, 0.5, 0.0, 0.9789986131578252),\n                        (7, 0.5, 0.0, 0.9063802810157577)],\n                       [(8, 7), (11, 10)],\n                       [(8, 0.4999999999999824, 5.551115123125783e-17, 0.26958271166660713),\n                        (11, 0.49999999999998246, 0.0, -0.02100133052423292),\n                        (10, 0.5000000000000022, 0.0, 0.6216949198585577),\n                        (7, 0.500000000000002, 0.0, 0.9950402106079221)],\n                       [(8, 9), (11, 12)],\n                       [(11, 0.5000000000000102, 0.0, -0.02100133052423292),\n                        (8, 0.5000000000000103, 0.0, 0.26958271166660713),\n                        (12, 0.30408672398469627, 0.0, -0.5745192213521484),\n                        (9, 0.3040867239846962, 0.0, 0.4254807786478514)],\n                       [(11, 10), (8, 7)],\n                       [(11, 0.75, 0.0, -0.02100129800905437),\n                        (8, 0.75, 0.0, -0.7304173208485718),\n                        (10, 0.49999999999999956, 0.0, 0.6216948873433754),\n                        (7, 0.49999999999999956, 0.0, -0.004959756876894872)],\n                       [(11, 12), (8, 9)],\n                       [(11, 0.500526889421938, 0.0, -0.5204743618818903),\n                        (8, 0.5005268894219381, 0.0, -0.23094425697573628),\n                        (9, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155),\n                        (12, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155)],\n                       [(11, 10), (8, 7)],\n                       [(10, 0.5000000000000003, -2.220446049250313e-16, 0.12169488734337541),\n                        (11, 0.5000000000000002, 5.551115123125783e-17, 0.22952607419154275),\n                        (7, 0.5000000000000003, 0.0, -0.5049597568768931),\n                        (8, 0.5, 0.0, 0.019055306950831685)],\n                       [(11, 10), (8, 7)],\n                       [(11, 12),\n                        (8, 9),\n                        (10, 0.0007451670218237004, -0.34790937491533347, -0.37830511265661615),\n                        (7, 0.9992548329781763, 0.27874526930495214, -0.004959756876901533)],\n                       [(8, 3.251517832758566e-08, 0.2547302054772107, -0.9809447157146567),\n                        (12, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (11, 0.999999967484822, -0.9557406070944765, -0.7704739031429708),\n                        (9, 0.5, 0.3483047662953691, -0.5745192538673274)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],\n           \'qubits\': [(2, 6),\n                      (3, 4),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 5),\n                      (6, 6),\n                      (6, 7)]},\n (14, 3): {\'method\': \'three-arm hub seed + parallel square decoders\',\n           \'moments\': [[(3, 0.22426760152894315, 0.0, 0.4269300027316971),\n                        (8, 0.48843984726151146, 0.0, -0.960980251422612),\n                        (4, 0.268155833004333, 0.0, 0.2676694612408439),\n                        (9, 0.44765872878442003, 0.0, -0.8864668857908022),\n                        (5, 0.579443694978949, 0.0, -0.6188627809687886),\n                        (10, 0.538585551028711, 0.0, 0.8157116108965718),\n                        (1, 0.4217899985400859, 0.0, -0.5001097254686009),\n                        (7, 0.4217899985400859, 0.0, -0.5001097254686009),\n                        (11, 0.4135610417387456, 0.0, -0.5),\n                        (0, 0.4999999999999983, 0.0, -0.6146222686036573),\n                        (12, 0.4999999999999983, 0.0, -0.6146222686036573),\n                        (6, 0.5, 0.0, 0.5),\n                        (2, 0.3578842255793437, 0.0, -0.5503659482326804),\n                        (13, 0.3578842255793437, 0.0, -0.5503659482326804)],\n                       [(3, 8), (4, 9), (5, 10)],\n                       [(4, 0.7553062541204851, 0.0, 0.3964034002344974),\n                        (5, 0.46552097966263467, 0.0, -0.1924811924725438),\n                        (8, 0.15242556582126618, 0.0, 0.604486281078372),\n                        (9, 0.7609028445069608, 0.0, -0.3204521638221274),\n                        (3, 0.42703768281598714, 0.0, -0.7005092272473903),\n                        (10, 0.6458458084723397, 0.0, 0.8157116108965725)],\n                       [(4, 5), (8, 9)],\n                       [(8, 0.9692623502655389, 0.0, 0.9678026483784579),\n                        (5, 0.257159744718654, 0.0, -0.8294720486772631),\n                        (9, 0.7424239904618464, 0.0, -0.7062719507733548)],\n                       [(3, 8), (4, 5), (9, 10)],\n                       [(3, 0.9599557319518159, 0.0, -0.5602565923079164),\n                        (8, 0.23946791673765808, 0.0, -0.650956669193564),\n                        (4, 0.17466807776948218, 0.0, -0.9968702594664394),\n                        (9, 0.8089785382141472, 1.1102230246251565e-16, -0.08683218976362728),\n                        (5, 0.06311104127078737, 0.0, -0.8294720486772642),\n                        (10, 0.5, 0.0, -0.8121850810506395)],\n                       [(3, 8), (4, 9), (5, 10)],\n                       [(3, 0.47435628719668477, 0.0, 0.3102753085931642),\n                        (4, 0.41395063990275915, 0.0, 0.9133476180644173),\n                        (5, 0.3522641339453063, 3.3306690738754696e-16, -0.32947204867726354),\n                        (10, 0.5, 0.0, 0.6878149189493605),\n                        (8, 0.26673531776101456, 0.0, -0.28755878159273385),\n                        (9, 0.38977322735648867, 0.0, -0.42533490349505554)],\n                       [(3, 4), (5, 10), (8, 9)],\n                       [(8, 0.30944179150527346, 0.0, 0.5672673249452769),\n                        (4, 0.4303575245248396, 0.0, -0.5031224909842611),\n                        (5, 0.5799167530749968, 0.0, -0.3294720486772633),\n                        (9, 0.6851059439846782, 0.0, 0.3851425911877515),\n                        (10, 0.994225473631722, 0.0, -0.564112452449893)],\n                       [(3, 8), (4, 5), (9, 10)],\n                       [(3, 4),\n                        (9, 0.7996879170808809, 0.0, -0.28746777203691076),\n                        (10, 0.35127785442647275, 0.0, 0.5431500213632499),\n                        (8, 0.3094417915052792, 0.0, -0.43273267505471935),\n                        (5, 0.49999999999999867, 5.551115123125783e-17, 0.17052795132273607)],\n                       [(9, 10),\n                        (3, 0.15753359009327883, 0.0, 0.43881655812089293),\n                        (4, 0.5, 0.0, 0.6752332273024653)],\n                       [(3, 8),\n                        (9, 0.16262881518327754, 0.0, -0.6257481158496427),\n                        (10, 0.7615496562802844, 0.0, -0.005312094373818077)],\n                       [(4, 9),\n                        (5, 10),\n                        (3, 0.42417551329734837, 0.0, 0.3859106567041417),\n                        (8, 0.5796177256424373, 0.0, 0.5941788886937556)],\n                       [(3, 1),\n                        (8, 7),\n                        (10, 0.7977732920601596, 0.0, 0.16549165681510253),\n                        (4, 0.5, 0.0, 0.6752332273024653)],\n                       [(10, 11),\n                        (3, 0.8116669186835013, 0.0, 0.2523959332050929),\n                        (8, 0.8116669186835013, 0.0, 0.43482034891018007),\n                        (1, 0.6287848321594933, 0.0, -0.5144920515457643),\n                        (7, 0.6287848321594933, 0.0, -0.5144920515457643)],\n                       [(4, 3),\n                        (10, 0.3819668494768556, 0.0, -0.9245745523052049),\n                        (9, 8),\n                        (11, 0.633860236400615, 0.0, -0.5)],\n                       [(5, 10),\n                        (3, 0.7186000038451317, 0.0, 0.27409052565331793),\n                        (8, 0.7186000038451317, 0.0, -0.5434850586415954)],\n                       [(3, 1),\n                        (8, 7),\n                        (10, 0.7376490383100944, 1.1102230246251565e-16, 0.07542544769479498)],\n                       [(10, 11),\n                        (3, 0.18796960844370006, 0.0, 0.49872459967151433),\n                        (8, 0.18796960844370006, 0.0, -0.31885098462339845),\n                        (1, 0.1426534639632415, 0.0, 0.2342201948433344),\n                        (7, 0.1426534639632415, 0.0, 0.2342201948433344)],\n                       [(4, 3),\n                        (10, 0.16018463696253948, 0.0, 0.07542544769479509),\n                        (9, 8),\n                        (11, 0.180239329875864, 0.0, -0.5000000000000002)],\n                       [(5, 10),\n                        (3, 0.36815249285725926, 0.0, -0.04928001972942542),\n                        (8, 0.36815249285725926, 0.0, 0.13314439597566158),\n                        (4, 0.5, 0.0, 0.06766209258747402),\n                        (9, 0.5, 0.0, -0.014910063605600099)],\n                       [(3, 1),\n                        (8, 7),\n                        (10, 0.442231745387027, 0.0, 0.07542544769479509),\n                        (5, 0.5, 8.881784197001252e-16, -0.9296854106424435)],\n                       [(10, 11),\n                        (1, 0.5401993801370204, 0.0, -0.5512342235842844),\n                        (3, 0.5762322444517164, 0.0, 0.9199557979762887),\n                        (7, 0.5401993801370204, 0.0, -0.5512342235842844),\n                        (8, 0.5762322444517164, 0.0, -0.8976197863186242)],\n                       [(1, 0),\n                        (4, 3),\n                        (10, 0.6666666666666669, 5.551115123125783e-17, 0.07542544769479503),\n                        (7, 12),\n                        (9, 8),\n                        (11, 0.8040867239846965, 0.0, -0.5000000000000002)],\n                       [(5, 10),\n                        (11, 6),\n                        (3, 2),\n                        (8, 13),\n                        (1, 0.15204336199234872, 0.0, 0.9346037967062348),\n                        (7, 0.15204336199234872, 0.0, 0.9346037967062348),\n                        (0, 0.3260301916599948, 0.0, 0.43315900579188815),\n                        (12, 0.3260301916599948, 0.0, 0.43315900579188815),\n                        (4, 0.5, 0.3342640424251426, -0.932337907412526),\n                        (9, 0.5, 0.41683619861821736, -0.014910063605599877)],\n                       [(3, 1),\n                        (8, 7),\n                        (11, 0.5, 0.0, -0.5),\n                        (2, 0.8053712368962477, 0.0, 0.18888285213015),\n                        (13, 0.8053712368962477, 0.0, 0.18888285213015),\n                        (6, 0.30408672398469605, 0.0, -0.5000000000000002),\n                        (5, 0.5, -0.5703145893575563, 0.07031458935755674)],\n                       [(10, 11),\n                        (0, 2),\n                        (12, 13),\n                        (1, 0.5, 5.551115123125783e-17, 0.43460379670623456),\n                        (7, 0.5, 5.551115123125783e-17, 0.43460379670623456)],\n                       [(11, 0.8040867239846965, 0.0, -0.5),\n                        (0, 0.5, 0.0, 0.8430180222691617),\n                        (2, 0.6008134304054108, 1.1102230246251565e-16, -0.040123990740045024),\n                        (12, 0.5, 0.0, 0.8430180222691617),\n                        (13, 0.6008134304054108, 1.1102230246251565e-16, -0.040123990740045024)],\n                       [(11, 6), (1, 0), (3, 2), (7, 12), (8, 13)],\n                       [(11, 0.75, 0.0, -0.5),\n                        (0, 0.5, 0.0, -0.6569819777308383),\n                        (2, 0.3991865695945891, 0.0, 0.959876009259955),\n                        (1, 0.5924401224278535, 0.0, 0.08664715869858275),\n                        (7, 0.5924401224278535, 0.0, 0.08664715869858275),\n                        (12, 0.5, 0.0, -0.6569819777308383),\n                        (13, 0.3991865695945891, 0.0, 0.959876009259955),\n                        (6, 0.5, 0.0, 0.5)],\n                       [(10, 11), (0, 2), (3, 1), (8, 7), (12, 13)],\n                       [(11, 0.5, 0.0, 0.5),\n                        (1, 0.9075598775721465, 0.0, -0.9133528413014158),\n                        (0, 0.25040914996542357, 0.0, 0.3268926652168218),\n                        (2, 0.19462876310375188, 0.0, 0.2308691663897604),\n                        (7, 0.9075598775721465, 0.0, -0.9133528413014158),\n                        (12, 0.25040914996542357, 0.0, 0.3268926652168218),\n                        (13, 0.19462876310375188, 0.0, 0.2308691663897604),\n                        (10, 0.5, 0.0, 0.07542544769479531)],\n                       [(11, 6), (1, 0), (3, 2), (7, 12), (8, 13)],\n                       [(11, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (0, 0.5, 0.0, 0.3497071393883171),\n                        (2, 0.7587162401878453, 0.0, 0.8228655323944645),\n                        (3, 0.5, 0.0, -0.7787111351825897),\n                        (1, 0.5, 0.0, 0.33664715869858547),\n                        (8, 0.5, 0.0, 0.40371328052249694),\n                        (7, 0.5, 0.0, 0.33664715869858547),\n                        (12, 0.5, 0.0, 0.3497071393883171),\n                        (13, 0.7587162401878453, 0.0, 0.8228655323944645)],\n                       [(10, 11), (0, 2), (3, 1), (8, 7), (12, 13)],\n                       [(11, 0.5, 0.0, 0.5),\n                        (1, 0.5, 0.0, -0.6633528413014143),\n                        (0, 0.5, 0.0, -0.15029286061168312),\n                        (3, 0.5, 0.0, 0.721288864817411),\n                        (7, 0.5, 0.0, -0.6633528413014143),\n                        (12, 0.5, 0.0, -0.15029286061168312),\n                        (8, 0.49999999999999983, 0.0, -0.096286719477503),\n                        (10, 0.5, 0.4245745523052049, 0.07542544769479509)],\n                       [(11, 6), (1, 0), (3, 2), (7, 12), (8, 13)],\n                       [(6, 1.0, 0.0, 0.5),\n                        (11, 0.5, 4.440892098500626e-16, 0.5),\n                        (0, 0.0, 0.378896047164595, 0.0),\n                        (1, 0.5, -0.10804397214567385, -0.663352841301414),\n                        (2, 1.0, 0.0, 0.16765763650036608),\n                        (3, 0.5000000000000001, 0.0073143217355013945, -0.7787111351825899),\n                        (7, 0.5, -0.10804397214567385, -0.663352841301414),\n                        (8, 0.5000000000000002, 0.8248899060304156, 0.40371328052249655),\n                        (12, 0.0, 0.378896047164595, 0.0),\n                        (13, 1.0, 0.0, 0.16765763650036608)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],\n           \'qubits\': [(3, 3),\n                      (3, 4),\n                      (4, 3),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 3),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 3),\n                      (6, 4)]},\n (14, 4): {\'method\': \'certified count kernels + parallel coherent decoders\',\n           \'moments\': [[(3, 0.1492208351548694, 0.0, 0.5),\n                        (7, 0.49364150917817384, 0.0, -0.5000000000000002),\n                        (4, 0.1304149928257203, 0.0, 0.5),\n                        (8, 0.8109564595691234, 0.0, -0.5000000000000002),\n                        (5, 0.4212522357058561, 1.0, 0.5),\n                        (2, 0.5, 1.0, -0.07534461897426747),\n                        (13, 0.5, 1.0, -0.07534461897426747),\n                        (1, 0.4127792040228902, -0.9999999999999998, -0.9262332254895425),\n                        (12, 0.4127792040228902, -0.9999999999999998, -0.9262332254895425),\n                        (0, 0.5, 1.0, 0.5),\n                        (11, 0.5, 1.0, 0.5),\n                        (6, 0.3040867239846966, 0.0, -0.5745192213521528),\n                        (10, 0.3040867239846966, 0.0, -0.5745192213521528)],\n                       [(3, 7), (4, 8), (5, 9)],\n                       [(3, 0.2324610527908232, 0.0, 0.5),\n                        (4, 0.920204551693893, 0.0, 0.5),\n                        (7, 0.9565608181455243, 0.0, 0.5),\n                        (8, 0.5818292564517651, 0.0, 0.5),\n                        (5, 0.07874776429414387, 0.0, -0.5000000000000002)],\n                       [(3, 4), (7, 8)],\n                       [(3, 0.4458120464876009, 0.0, -0.5),\n                        (7, 0.09583805393818556, 0.0, 0.5),\n                        (4, 0.06033211708887334, 0.0, 0.5),\n                        (8, 0.18282003036844155, 0.0, 0.5)],\n                       [(3, 7), (4, 5), (8, 9)],\n                       [(8, 0.9727870547104276, 0.0, -0.5),\n                        (5, 0.5, 0.0, 0.5),\n                        (9, 0.75, 0.0, -0.5),\n                        (7, 0.8943608728307322, 0.0, 0.5),\n                        (3, 0.8666380306139951, 1.0, 0.5)],\n                       [(4, 8), (5, 9)],\n                       [(8, 0.1956649858835388, 0.0, -0.5000000000000002), (3, 4), (9, 0.75, 0.0, -0.5)],\n                       [(7, 8), (3, 0.4229501978568093, 0.0, -0.5000000000000002), (4, 0.5, 0.0, 0.5)],\n                       [(7, 0.24042206046407322, 0.0, -0.5), (4, 5), (8, 0.4941988651660729, 0.0, 0.5)],\n                       [(3, 7), (8, 9), (4, 0.0, 1.0, 0.0), (5, 0.5, 0.0, -0.5)],\n                       [(4, 8),\n                        (5, 9),\n                        (3, 0.11574275384819255, 0.0, 0.5),\n                        (7, 0.9170234463620259, 1.0, 0.4999999999999999)],\n                       [(3, 4),\n                        (5, 0.7084946987581148, 0.0, -0.19692626900829913),\n                        (7, 8),\n                        (9, 0.34787704958962457, 4.440892098500626e-16, 0.7593532832103107)],\n                       [(5, 2),\n                        (9, 13),\n                        (4, 0.6772758467840048, 1.0, -0.41340381023084216),\n                        (7, 0.6877465543761767, 1.0, 0.5),\n                        (8, 0.8047304592861084, 0.0, 0.6296766416051554)],\n                       [(3, 4),\n                        (7, 8),\n                        (5, 0.7913327671706831, 4.440892098500626e-16, -0.5062564811818511),\n                        (9, 0.7913327671706831, 0.0, 0.08026951195943077),\n                        (2, 0.6321049414225235, 1.0, -0.829092751915236),\n                        (13, 0.6321049414225235, 1.0, -0.829092751915236)],\n                       [(4, 0.155480511485734, 0.9999999999999999, -0.2588288546844242),\n                        (8, 0.1554805114857343, 0.0, -0.370323358394844),\n                        (3, 0.0, -0.00571960596963228, 0.0)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.7964417643891555, 0.0, -0.25882885468442396),\n                        (5, 2),\n                        (8, 0.7964417643891555, 0.0, 0.6296766416051554),\n                        (9, 13)],\n                       [(3, 4),\n                        (7, 8),\n                        (5, 0.5886108001133223, 0.0, 0.4937435188181496),\n                        (9, 0.5886108001133223, 0.0, 0.08026951195943077),\n                        (2, 0.597350526917102, 0.9999999999999999, -0.4537132312119436),\n                        (13, 0.597350526917102, 0.9999999999999999, -0.4537132312119436)],\n                       [(4, 0.7964417643891555, 0.9999999999999999, -0.25882885468442396),\n                        (8, 0.7964417643891555, 0.0, -0.3703233583948444)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.5517093529747913, 4.440892098500626e-16, 0.7411711453155756),\n                        (5, 0.1795085179856881, 0.0, -0.5062564811818504),\n                        (8, 0.5517093529747915, 0.0, -0.37032335839484465),\n                        (9, 0.1795085179856881, 0.0, 0.08026951195943077)],\n                       [(3, 4), (5, 2), (7, 8), (9, 13)],\n                       [(4, 0.893493814032364, -0.9999999999999998, 0.7411711453155756),\n                        (5, 0.11878997884016866, 0.0, -0.5062564811818504),\n                        (8, 0.893493814032364, 0.0, 0.6296766416051556),\n                        (9, 0.11878997884016866, 0.0, 0.08026951195943077),\n                        (3, 0.500000000000001, 5.551115123125783e-17, 0.4294420732277873),\n                        (7, 0.5000000000000009, 0.0, 0.56483832080258)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.5, 0.0, 0.7411711453155756),\n                        (5, 2),\n                        (8, 0.5, 0.0, -0.3703233583948444),\n                        (9, 13)],\n                       [(3, 4),\n                        (7, 8),\n                        (2, 0.7780272931625696, 0.0, -0.7952443238944182),\n                        (5, 0.6444528487918006, 1.1102230246251565e-16, 0.08818151358003495),\n                        (9, 0.6444528487918006, 0.0, -0.3252924932786829),\n                        (13, 0.7780272931625696, 0.0, -0.7952443238944182)],\n                       [(2, 1),\n                        (4, 0.5, 0.9999999999999997, -0.4945175108212885),\n                        (8, 0.5, 0.0, -0.1346347022579799),\n                        (13, 12),\n                        (3, 0.5, -0.864603752425209, -0.5705579267722132),\n                        (7, 0.4999999999999998, -2.6645352591003757e-15, -0.43516167919741977)],\n                       [(4, 5),\n                        (8, 9),\n                        (1, 0.85928593962011, 0.0, -0.9262332254895425),\n                        (2, 0.5000000000000001, 0.0, 0.7047556761055818),\n                        (13, 0.5000000000000001, 0.0, 0.7047556761055818),\n                        (12, 0.85928593962011, 1.0, -0.9262332254895425)],\n                       [(4, 0.5, 0.0, -0.4945175108212887),\n                        (5, 0.0, 0.8908557910557207, 0.0),\n                        (9, 0.0, 0.7178038047731585, 0.0),\n                        (8, 0.5, 0.0, 0.8653652977420201)],\n                       [(1, 4), (5, 2), (9, 13), (12, 8)],\n                       [(2, 0.6714165234362537, 4.440892098500626e-16, 0.6333395843745888),\n                        (1, 0.7485956033393342, 1.0, -0.42623322548954246),\n                        (5, 0.5, 1.0, 0.47121890332230265),\n                        (8, 0.0, -0.8174147036972932, 0.0),\n                        (9, 0.5000000000000001, 8.881784197001252e-16, 0.8846929101810206),\n                        (13, 0.6714165234362537, 4.440892098500626e-16, 0.6333395843745888),\n                        (12, 0.7485956033393342, 1.0, -0.42623322548954246)],\n                       [(2, 1), (4, 5), (8, 9), (13, 12)],\n                       [(1, 0.9961572718865779, 1.0, -0.9262332254895478),\n                        (4, 0.05280605500441269, 0.0, -0.3209171670343185),\n                        (5, 0.5, 1.0, -0.5287810966776971),\n                        (2, 0.3935294720438233, 1.1102230246251565e-16, 0.35429246876096754),\n                        (12, 0.9961572718865779, 0.0, -0.9262332254895478),\n                        (8, 0.05280605500441269, 0.0, -0.12564974974224286),\n                        (9, 0.5, 1.0, -0.11530708981897897),\n                        (13, 0.3935294720438233, 1.1102230246251565e-16, 0.35429246876096754)],\n                       [(1, 4), (5, 2), (12, 8), (9, 13)],\n                       [(4, 0.24963665794604523, 0.0, 0.19644541787545639),\n                        (2, 0.3104369525321424, 0.0, 0.7728137191611664),\n                        (1, 0.5655532211464794, 1.0, -0.4262332254895531),\n                        (8, 0.24963665794604517, 1.0, -0.6430123346520178),\n                        (13, 0.3104369525321424, 0.0, 0.7728137191611664),\n                        (12, 0.5655532211464794, 1.0, -0.4262332254895531)],\n                       [(4, 5), (2, 1), (8, 9), (13, 12)],\n                       [(2, 0.6193705712509703, 0.0, -0.10015196993942821),\n                        (1, 0.4999999999999998, 1.0, -0.4262332254895531),\n                        (4, 0.32701156232958106, 0.0, -0.1957742234126454),\n                        (13, 0.6193705712509703, 0.0, -0.10015196993942821),\n                        (12, 0.4999999999999998, 0.0, -0.42623322548955334),\n                        (8, 0.3270115623295809, 4.440892098500626e-16, 0.7492073066360838)],\n                       [(5, 2), (1, 4), (9, 13), (12, 8)],\n                       [(1, 0.5960614607313279, 1.0, -0.11684358694965402),\n                        (4, 0.5, 8.881784197001252e-16, -0.9484096919337694),\n                        (5, 0.0, 0.9739798652145695, 0.0),\n                        (2, 0.6256105257587086, 1.1102230246251565e-16, 0.4165776620509426),\n                        (12, 0.5960614607313279, -4.440892098500626e-16, -0.11684358694965402),\n                        (8, 0.5000000000000002, 0.0, -0.4981572248427921),\n                        (9, 0.0, -0.02602013478543075, 0.0),\n                        (13, 0.6256105257587086, 1.1102230246251565e-16, 0.4165776620509426)],\n                       [(1, 4), (5, 2), (12, 8), (9, 13)],\n                       [(2, 0.9764747842833552, 4.440892098500626e-16, 0.8187934199715505),\n                        (1, 0.14949404734505523, 1.0, 0.2730671118828738),\n                        (4, 0.5, 0.0, -0.4484096919337692),\n                        (5, 0.5, 1.0, -0.8758726607706111),\n                        (8, 0.5, 0.0, -0.9981572248427919),\n                        (9, 0.5, -4.440892098500626e-16, -0.4623986539118925),\n                        (13, 0.9764747842833552, 4.440892098500626e-16, 0.8187934199715505),\n                        (12, 0.14949404734505523, 1.0, 0.2730671118828738)],\n                       [(2, 1), (4, 5), (8, 9), (13, 12)],\n                       [(4, 0.0, 0.08503202047153685, 0.0),\n                        (5, 0.5, 1.0, 0.12412733922938901),\n                        (2, 0.5, 4.440892098500626e-16, 0.5758151367896762),\n                        (12, 8),\n                        (9, 0.5, 0.9999999999999999, 0.5376013460881071),\n                        (13, 0.5, 4.440892098500626e-16, 0.5758151367896762)],\n                       [(1, 4), (5, 2), (9, 13), (12, 0.42477613785575913, 0.0, -0.022381851578533674)],\n                       [(2, 0.14731871270290633, 0.0, -0.42418486321032356),\n                        (1, 0.4247761378557593, 0.0, 0.9776181484214663),\n                        (5, 0.5, 0.0, 0.1913161932480265),\n                        (13, 0.14731871270290633, 0.0, -0.42418486321032356),\n                        (9, 0.5, 0.9999999999999998, 0.6047902001067444)],\n                       [(2, 1), (4, 5), (13, 12), (8, 9)],\n                       [(1, 0.4500869528994624, 0.0, -0.33653668617336185),\n                        (4, 0.4202007516434625, 1.0, -0.6568153130537349),\n                        (5, 0.0, 1.0, 0.0),\n                        (2, 0.9433458052208397, 0.0, -0.9241848632103227),\n                        (12, 0.45008695289946266, 0.0, 0.6634633138266384),\n                        (8, 0.4202007516434625, 0.0, -0.70471958325129),\n                        (9, 0.0, 1.0, 0.0),\n                        (13, 0.9433458052208397, 0.0, -0.9241848632103227)],\n                       [(1, 4), (5, 2), (12, 8), (9, 13)],\n                       [(5, 0.20567003520391758, 0.0, 0.6913161932480261),\n                        (4, 0.5, 8.881784197001252e-16, 0.7479888017855387),\n                        (1, 0.5410889047360808, 1.1102230246251565e-16, 0.21035484945072336),\n                        (9, 0.20567003520391755, -0.9999999999999996, -0.8952097998932562),\n                        (8, 0.5, 0.0, 0.8904763019094353),\n                        (12, 0.541088904736081, 1.1102230246251565e-16, 0.21035484945072352)],\n                       [(5, 4), (1, 0), (9, 8), (12, 11)],\n                       [(1, 0.5, 0.0, -0.28964515054927653),\n                        (5, 0.49999999999998246, 0.0, -0.8086837504340327),\n                        (9, 0.49999999999998224, 0.0, -0.395209743575315),\n                        (12, 0.5, 0.0, -0.28964515054927653),\n                        (4, 0.5000000000000022, -2.220446049250313e-16, -0.16335126862229465),\n                        (0, 0.3040867239846962, 1.0, -0.5000000000000002),\n                        (8, 0.500000000000002, 0.0, -0.19818362768273),\n                        (11, 0.3040867239846962, 1.0, -0.5000000000000002)],\n                       [(2, 1), (5, 6), (9, 10), (13, 12)],\n                       [(5, 0.5000000000000102, 0.0, -0.8086837504340327),\n                        (1, 0.8040867239846965, 0.0, -0.28964515054927653),\n                        (9, 0.5000000000000102, 1.0, -0.395209743575315),\n                        (12, 0.8040867239846965, 3.3306690738754696e-16, -0.2896451505492763),\n                        (6, 0.30408672398469627, 0.0, -0.5745192213521484),\n                        (2, 0.0, 0.02305897970537224, 0.0),\n                        (10, 0.30408672398469627, 0.0, -0.5745192213521484),\n                        (13, 0.0, 0.02305897970537224, 0.0)],\n                       [(5, 4), (1, 0), (9, 8), (12, 11)],\n                       [(5, 0.75, -2.220446049250313e-16, 0.1913162820811456),\n                        (1, 0.7499999999999999, 1.1102230246251565e-16, -0.28964515054927675),\n                        (9, 0.7499999999999999, 0.0, 0.6047902889398629),\n                        (12, 0.7499999999999999, -4.440892098500626e-16, -0.2896451505492761),\n                        (4, 0.49999999999999956, 0.0, -0.163351236107113),\n                        (0, 0.5, 1.0, 0.5),\n                        (8, 0.49999999999999956, 0.0, -0.19818366019791167),\n                        (11, 0.5, 1.0, 0.5)],\n                       [(5, 6), (2, 1), (9, 10), (13, 12)],\n                       [(5, 0.5005268894219383, 0.0, -0.30815678179169015),\n                        (1, 0.5, 0.0, 0.710354849450723),\n                        (9, 0.500526889421938, 1.0, 0.10531722506702745),\n                        (12, 0.5, 0.0, 0.7103548494507237),\n                        (6, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515),\n                        (2, 0.5, 1.0, 0.5783296641270095),\n                        (10, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515),\n                        (13, 0.5, 1.0, 0.5783296641270095)],\n                       [(5, 4), (1, 0), (9, 8), (12, 11)],\n                       [(5, 0.5, 0.0, -0.5581563457182575),\n                        (1, 0.25000000000000006, 0.0, -0.28964515054927653),\n                        (9, 0.5, 1.1102230246251565e-16, -0.14468233885953996),\n                        (12, 0.25000000000000006, 0.0, -0.2896451505492763),\n                        (4, 0.5000000000000003, 1.1102230246251565e-16, 0.3366487638928869),\n                        (8, 0.5000000000000004, 0.0, -0.6981836601979121)],\n                       [(5, 6), (2, 1), (9, 10), (13, 12)],\n                       [(1, 0.5, 0.0, 0.710354849450723),\n                        (5, 4),\n                        (12, 0.5, 0.0, 0.7103548494507237),\n                        (9, 1.0214051826551442e-14, 0.9999999999999999, 0.35531057022963497),\n                        (2, 0.5, -0.07832966412700948, 0.5783296641270097),\n                        (6, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (10, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (13, 0.5, -0.07832966412700948, 0.5783296641270097)],\n                       [(1, 0),\n                        (12, 11),\n                        (9, 8),\n                        (4, 0.9992548329781763, 0.43713674853517215, -0.16335123610712143),\n                        (5, 0.999999967484822, 0.8319418128153226, -0.5581563230527691)],\n                       [(0, 1.0, 0.0, 0.0),\n                        (1, 0.4999999999999998, -0.21035484945072325, -0.28964515054927675),\n                        (11, 1.0, 0.0, 0.0),\n                        (9, 0.999999967484822, 0.41846780595660493, -0.14468231619405136),\n                        (8, 0.0007451670218237004, 0.47196917262595495, 0.8018163398020959),\n                        (12, 0.4999999999999998, -0.21035484945072325, -0.28964515054927675)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],\n           \'qubits\': [(2, 5),\n                      (3, 5),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 4),\n                      (6, 5),\n                      (6, 6)]},\n (14, 5): {\'method\': \'certified count kernels + parallel coherent decoders\',\n           \'moments\': [[(0, 0.5, 0.0, 0.5),\n                        (1, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (2, 0.7010577384351091, 0.0, 0.9439691995767472),\n                        (3, 0.37780996597847644, 0.0, 0.5),\n                        (4, 0.6649284327846141, 0.0, -0.5),\n                        (5, 0.20369585066947749, 0.0, 0.5),\n                        (6, 0.30408672398469655, 0.0, -0.5745192213521528),\n                        (9, 0.9999404304359252, 0.0, -0.5),\n                        (10, 0.30408672398469655, 0.0, -0.5745192213521528),\n                        (11, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (12, 0.7010577384351091, 0.0, 0.9439691995767472),\n                        (13, 0.30408672398469655, 0.0, -0.5745192213521528)],\n                       [(3, 7), (4, 8), (5, 9)],\n                       [(3, 0.9765810720440469, 0.0, 0.5),\n                        (4, 0.9297740209165868, 0.0, 0.4999999999999998),\n                        (5, 0.5740941455895439, 0.0, 0.5),\n                        (7, 0.37339124773516286, 0.0, 0.5),\n                        (9, 0.9999259074782365, 0.0, 0.5)],\n                       [(3, 4), (7, 8)],\n                       [(3, 0.20093748739029382, 0.0, 0.5),\n                        (4, 0.9999943381762997, 0.0, -0.5000000000000004),\n                        (7, 0.8733912477351629, 0.0, -0.5000000000000002),\n                        (8, 9)],\n                       [(3, 7), (4, 5), (9, 0.4999890307436661, 0.0, -0.5)],\n                       [(4, 0.9999948690424976, 0.0, 0.4999999999999998),\n                        (5, 0.3186839038285761, 0.0, 0.5),\n                        (7, 0.5, 0.0, 0.5)],\n                       [(4, 8), (5, 9)],\n                       [(4, 0.8562952046824337, 0.0, 0.49999999999999956),\n                        (5, 0.49996737843785244, 0.0, -0.5000000000000002),\n                        (8, 0.4999999999999545, 0.0, 0.5),\n                        (9, 0.5000208919712605, 0.0, 0.5)],\n                       [(3, 4), (7, 8), (9, 12)],\n                       [(3, 7),\n                        (4, 0.2308414232966614, 0.0, -0.5000000000000007),\n                        (8, 0.5, 0.0, 0.5),\n                        (12, 0.2989422615648911, 4.440892098500626e-16, 0.943969199576747)],\n                       [(4, 5), (8, 9)],\n                       [(3, 4), (5, 0.4999673784378522, 0.0, 0.5), (8, 0.5, 0.0, -0.5)],\n                       [(4, 8), (5, 9)],\n                       [(4, 0.39529799192523557, 0.0, -0.10562369964702789),\n                        (5, 3.262156214778323e-05, 0.0, 0.49999999999926303),\n                        (7, 8)],\n                       [(3, 4), (5, 2), (8, 0.5997618693438882, 0.0, 0.11028258081031161)],\n                       [(2, 0.29894226156489107, 0.0, -0.05603080042325281),\n                        (4, 0.7021281047129817, -2.220446049250313e-16, -0.105623699647027),\n                        (7, 8)],\n                       [(4, 5), (8, 0.7021281047129817, 0.0, 0.6455671621175141)],\n                       [(4, 0.7118222410987194, 0.0, -0.105623699647027),\n                        (5, 0.6662003855379057, 0.0, 0.7259619260711767),\n                        (8, 9)],\n                       [(3, 4),\n                        (5, 2),\n                        (8, 0.7118222410987194, 2.220446049250313e-16, -0.3544328378824866),\n                        (9, 0.6662003855379057, 4.440892098500626e-16, -0.7259619260711769)],\n                       [(2, 0.4198773247206662, 0.0, -0.07074532720037463),\n                        (4, 0.9813476461135342, 0.0, 0.8943763003529741),\n                        (5, 0.6662003855379061, 4.440892098500626e-16, 0.7259619260711767),\n                        (7, 8),\n                        (9, 12)],\n                       [(4, 5),\n                        (8, 0.9813476461135342, 0.0, -0.35443283788248725),\n                        (9, 0.666200385537906, 0.0, -0.7259619260711767),\n                        (12, 0.4198773247206662, 0.0, -0.07074532720037463)],\n                       [(4, 0.0020996535016382105, 0.0, -0.105623699647035),\n                        (5, 0.6135871361834435, 0.0, -0.35097025916113744),\n                        (8, 9)],\n                       [(3, 4),\n                        (5, 2),\n                        (8, 0.00209965350163821, 0.0, -0.3544328378824817),\n                        (9, 0.6135871361834435, 0.0, -0.6490297408388621)],\n                       [(2, 0.5000000000000011, 0.0, -0.6245332277151969),\n                        (4, 0.1651746478714715, 0.0, -0.10562369964702523),\n                        (5, 0.19491599185619113, 4.440892098500626e-16, 0.7173798007565817),\n                        (7, 8),\n                        (9, 12)],\n                       [(4, 5),\n                        (8, 0.1651746478714715, 0.0, 0.6455671621175119),\n                        (9, 0.19491599185619116, 0.0, 0.2826201992434181),\n                        (12, 0.5000000000000014, 5.551115123125783e-17, 0.375466772284803)],\n                       [(4, 0.5846001477399267, 0.0, -0.10562369964702545), (5, 2), (8, 9)],\n                       [(3, 4),\n                        (5, 0.5886108001133223, 8.881784197001252e-16, -0.8711660270602779),\n                        (8, 0.5846001477399267, 0.0, -0.3544328378824879),\n                        (9, 12),\n                        (2, 0.4818796387550885, 0.0, 0.6712802634947106)],\n                       [(4, 0.20938585749425953, 0.0, 0.8943763003529746),\n                        (7, 8),\n                        (12, 0.48187963875508855, 4.440892098500626e-16, 0.6712802634947104),\n                        (9, 0.5886108001133225, 0.0, 0.8711660270602783)],\n                       [(4, 5), (8, 0.20938585749425953, 0.0, -0.35443283788248836)],\n                       [(4, 0.44829064702520854, 0.0, 0.8943763003529746),\n                        (5, 0.7627370268849369, 0.0, -0.8711660270602786),\n                        (8, 9)],\n                       [(3, 4),\n                        (5, 2),\n                        (8, 0.4482906470252086, 0.0, 0.6455671621175119),\n                        (9, 0.7627370268849369, 0.0, -0.128833972939721)],\n                       [(3, 0.5, 0.0, 0.6971881501764847),\n                        (4, 0.893493814032364, 0.0, 0.8943763003529754),\n                        (5, 0.4033542756707678, 0.0, 0.128833972939721),\n                        (7, 8),\n                        (9, 12)],\n                       [(4, 5),\n                        (7, 0.5, 0.0, 0.3028118498235144),\n                        (8, 0.893493814032364, 0.0, -0.35443283788248925),\n                        (9, 0.4033542756707678, 0.0, 0.8711660270602795)],\n                       [(4, 0.035196301149107656, 0.0, 0.8943763003529754),\n                        (5, 0.8339515505182198, 0.0, -0.8711660270602319),\n                        (8, 9)],\n                       [(3, 4),\n                        (5, 2),\n                        (8, 0.035196301149107656, 0.0, 0.6455671621175108),\n                        (9, 0.8339515505182198, 0.0, 0.8711660270602322)],\n                       [(2, 0.7780272931625696, 0.0, 0.05572810904583947),\n                        (3, 0.4999999999999998, 1.7763568394002505e-15, 0.6971881501764847),\n                        (4, 0.5, 0.0, 0.7910374735900199),\n                        (5, 0.9352064365349756, 0.0, -0.342905040995084),\n                        (7, 8),\n                        (9, 12)],\n                       [(2, 1),\n                        (4, 5),\n                        (7, 0.5, -0.6056236996470279, 0.30281184982351417),\n                        (8, 0.5, 0.0, -0.25109401111953433),\n                        (9, 0.9352064365349756, 0.0, 0.3429050409950847),\n                        (12, 0.7780272931625696, 0.0, 0.05572810904583947)],\n                       [(1, 0.85928593962011, 0.0, -0.9262332254895425),\n                        (2, 0.5, 0.0, 0.555728109045839),\n                        (4, 0.5, 0.0, 0.7910374735900199),\n                        (8, 9),\n                        (12, 11)],\n                       [(1, 4),\n                        (5, 2),\n                        (8, 0.5, 0.0, 0.7489059888804654),\n                        (11, 0.85928593962011, 0.0, -0.9262332254895425),\n                        (12, 0.5, 0.0, 0.555728109045839)],\n                       [(1, 0.7485956033393342, 0.0, 0.5737667745104573),\n                        (2, 0.6714165234362537, 4.440892098500626e-16, 0.6271442007768322),\n                        (5, 0.5, 0.0, 0.9842998073042954),\n                        (9, 12),\n                        (11, 8)],\n                       [(2, 1),\n                        (4, 5),\n                        (9, 0.5, 0.0, 0.015700192695703663),\n                        (11, 0.7485956033393342, 0.0, -0.42623322548954246),\n                        (12, 0.6714165234362537, 2.220446049250313e-16, -0.3728557992231676)],\n                       [(1, 0.9961572718865779, 0.0, 0.07376677451045216),\n                        (2, 0.3935294720438234, 0.0, 0.906191316390454),\n                        (4, 0.0528060550044127, 4.440892098500626e-16, 0.6174371298030485),\n                        (5, 0.5, 1.1102230246251565e-16, -0.015700192695704995),\n                        (8, 9),\n                        (12, 11)],\n                       [(1, 4),\n                        (5, 2),\n                        (8, 0.052806055004412704, 0.0, -0.07749366733256124),\n                        (9, 0.5, 0.0, 0.015700192695703663),\n                        (11, 0.9961572718865779, 0.0, -0.9262332254895478),\n                        (12, 0.3935294720438234, 0.0, -0.09380868360954597)],\n                       [(1, 0.5655532211464794, 0.0, 0.5737667745104469),\n                        (2, 0.3104369525321424, 0.0, 0.48767006599025464),\n                        (4, 0.24963665794604517, -2.220446049250313e-16, 0.10007454489327361),\n                        (9, 12),\n                        (11, 8)],\n                       [(2, 1),\n                        (4, 5),\n                        (8, 0.24963665794604517, 0.0, 0.43986891757721064),\n                        (11, 0.5655532211464794, 0.0, 0.5737667745104469),\n                        (12, 0.3104369525321424, 0.0, 0.4876700659902544)],\n                       [(1, 0.4999999999999998, 0.0, 0.5737667745104462),\n                        (2, 0.6193705712509703, 0.0, -0.6393642449091499),\n                        (4, 0.32701156232958084, 0.0, 0.4922941861813757),\n                        (8, 9),\n                        (12, 11)],\n                       [(1, 4),\n                        (5, 2),\n                        (8, 0.3270115623295809, 8.881784197001252e-16, -0.9523507237108906),\n                        (11, 0.4999999999999998, 0.0, 0.5737667745104462),\n                        (12, 0.6193705712509703, 0.0, -0.6393642449091499)],\n                       [(1, 0.5960614607313279, 0.0, 0.8831564130503458),\n                        (2, 0.6256105257587087, 0.0, 0.8439061231004792),\n                        (4, 0.5, 0.0, -0.7550703452975007),\n                        (9, 12),\n                        (11, 8)],\n                       [(1, 4),\n                        (5, 2),\n                        (8, 0.5, 5.551115123125783e-17, 0.2950138077679862),\n                        (11, 0.5960614607313279, -2.220446049250313e-16, -0.11684358694965402),\n                        (12, 0.6256105257587086, 0.0, -0.15609387689952037)],\n                       [(1, 0.14949404734505523, 0.0, -0.7269328881171262),\n                        (2, 0.9764747842833552, 0.0, 0.4416903651798709),\n                        (4, 0.5, 4.440892098500626e-16, 0.7449296547024993),\n                        (5, 0.5, 0.0, -0.33677162200318644),\n                        (9, 12),\n                        (11, 8)],\n                       [(2, 1),\n                        (4, 5),\n                        (8, 0.5, 0.0, 0.7950138077679869),\n                        (9, 0.5, 0.0, -0.6632283779968149),\n                        (11, 0.14949404734505523, 0.0, -0.7269328881171262),\n                        (12, 0.9764747842833552, 0.0, 0.4416903651798709)],\n                       [(1, 4),\n                        (2, 0.5, 0.0, 0.6846686483617455),\n                        (5, 0.5, 4.440892098500626e-16, 0.6632283779968131),\n                        (8, 9),\n                        (12, 11)],\n                       [(1, 0.42477613785575913, 0.0, 0.9776181484214663),\n                        (5, 2),\n                        (9, 0.5, 0.0, -0.6632283779968149),\n                        (11, 8),\n                        (12, 0.5, 0.0, 0.6846686483617455)],\n                       [(2, 0.14731871270290633, 0.0, -0.31533135163825454),\n                        (5, 0.5, 4.440892098500626e-16, 0.7304172320154503),\n                        (9, 12),\n                        (11, 0.4247761378557593, 0.0, -0.022381851578533896)],\n                       [(2, 1),\n                        (4, 5),\n                        (9, 0.5, 0.0, -0.7304172320154523),\n                        (12, 0.14731871270290633, 0.0, 0.684668648361745)],\n                       [(1, 0.4500869528994624, 0.0, 0.6634633138266381),\n                        (2, 0.9433458052208397, 0.0, 0.18466864836174457),\n                        (4, 0.42020075164346243, 4.440892098500626e-16, -0.9616327037059986),\n                        (8, 9),\n                        (12, 11)],\n                       [(1, 4),\n                        (5, 2),\n                        (8, 0.4202007516434625, 8.881784197001252e-16, 0.5015761661764835),\n                        (11, 0.45008695289946266, 0.0, -0.3365366861733614),\n                        (12, 0.9433458052208397, 0.0, -0.8153313516382552)],\n                       [(1, 0.5410889047360808, -2.220446049250313e-16, 0.21035484945072325),\n                        (4, 0.5, 4.440892098500626e-16, 0.6335631814547269),\n                        (5, 0.20567003520391755, 0.0, 0.23041723201544917),\n                        (9, 12),\n                        (11, 8)],\n                       [(1, 0),\n                        (5, 4),\n                        (8, 0.5, 0.0, 0.9063802810157577),\n                        (9, 0.20567003520391733, 0.0, 0.7695827679845484),\n                        (11, 0.5, 0.0, -0.2896451505492772),\n                        (12, 0.5, 0.0, 0.9789986131578252)],\n                       [(0, 0.3040867239846962, 0.0, 0.4999999999999998),\n                        (1, 0.5, 0.0, -0.28964515054927675),\n                        (4, 0.5000000000000022, 0.0, -0.4550967481374384),\n                        (5, 0.4999999999999827, 0.0, -0.26958271166660897),\n                        (9, 8),\n                        (12, 11)],\n                       [(2, 1),\n                        (5, 6),\n                        (8, 0.500000000000002, 0.0, 0.9950402106079221),\n                        (9, 0.4999999999999824, 5.551115123125783e-17, 0.26958271166660713),\n                        (11, 0.5000000000000022, 0.0, 0.6216949198585577),\n                        (12, 0.49999999999998246, 0.0, -0.02100133052423292)],\n                       [(1, 0.8040867239846965, 4.440892098500626e-16, 0.710354849450723),\n                        (5, 0.5000000000000103, 0.0, -0.26958271166660874),\n                        (6, 0.30408672398469627, 0.0, -0.5745192213521484),\n                        (9, 10),\n                        (12, 13)],\n                       [(1, 0),\n                        (5, 4),\n                        (9, 0.5000000000000103, 0.0, 0.26958271166660713),\n                        (10, 0.3040867239846962, 0.0, 0.4254807786478514),\n                        (12, 0.5000000000000102, 0.0, -0.02100133052423292),\n                        (13, 0.30408672398469627, 0.0, -0.5745192213521484)],\n                       [(0, 0.5, 0.0, 0.5),\n                        (1, 0.7499999999999999, 0.0, 0.710354849450723),\n                        (4, 0.49999999999999956, 0.0, -0.4550967806526205),\n                        (5, 0.75, 0.0, -0.26958267915142997),\n                        (9, 8),\n                        (12, 11)],\n                       [(2, 1),\n                        (5, 6),\n                        (8, 0.49999999999999956, 0.0, -0.004959756876894872),\n                        (9, 0.75, 0.0, -0.7304173208485718),\n                        (11, 0.49999999999999956, 0.0, 0.6216948873433754),\n                        (12, 0.75, 0.0, -0.02100129800905437)],\n                       [(1, 0.5, 0.0, 0.710354849450723),\n                        (2, 0.5, 0.0, 0.7052131007297842),\n                        (5, 0.500526889421938, 0.0, -0.7690557430242662),\n                        (6, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155),\n                        (9, 10),\n                        (12, 13)],\n                       [(1, 0),\n                        (5, 4),\n                        (9, 0.5005268894219381, 0.0, -0.23094425697573628),\n                        (10, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155),\n                        (12, 0.500526889421938, 0.0, -0.5204743618818903),\n                        (13, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155)],\n                       [(1, 0.25, 0.0, -0.28964515054927675),\n                        (4, 0.5000000000000004, 0.0, -0.9550967806526212),\n                        (5, 0.5, 0.0, -0.01905530695083435),\n                        (9, 8),\n                        (12, 11)],\n                       [(2, 1),\n                        (5, 6),\n                        (8, 0.5000000000000003, 0.0, -0.5049597568768931),\n                        (9, 0.5, 0.0, 0.019055306950831685),\n                        (11, 0.5000000000000003, -2.220446049250313e-16, 0.12169488734337541),\n                        (12, 0.5000000000000002, 5.551115123125783e-17, 0.22952607419154275)],\n                       [(1, 0.5, 0.0, -0.28964515054927675),\n                        (2, 0.5, 0.7947868992702162, 0.7052131007297839),\n                        (6, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (9, 8),\n                        (12, 13)],\n                       [(1, 0),\n                        (5, 4),\n                        (8, 0.9992548329781763, 0.27874526930495214, -0.004959756876901533),\n                        (9, 10),\n                        (12, 11),\n                        (13, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(0, 1.0, 0.0, 0.5),\n                        (1, 0.4999999999999998, 0.7896451505492773, 0.7103548494507228),\n                        (4, 0.0007451670218237004, 0.7288822930806633, 0.5449032193473877),\n                        (5, 0.999999967484822, -0.7071592259521, 0.9809447157146545),\n                        (9, 3.251517832758566e-08, 0.2547302054772107, -0.9809447157146567),\n                        (10, 0.5, 0.3483047662953691, -0.5745192538673274),\n                        (11, 0.0007451670218237004, -0.34790937491533347, -0.37830511265661615),\n                        (12, 0.999999967484822, -0.9557408070769995, -0.7704738031517093)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],\n           \'qubits\': [(2, 5),\n                      (3, 5),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 5),\n                      (6, 6),\n                      (6, 7)]},\n (15, 3): {\'method\': \'three-arm hub seed + parallel square decoders\',\n           \'moments\': [[(3, 0.21413835857799723, 5.551115123125783e-17, 0.42691957319401025),\n                        (7, 0.47432814638155835, 0.0, -0.961013903923926),\n                        (5, 0.5182279236984741, 0.0, -0.618866750670523),\n                        (8, 0.4689831206125028, -2.220446049250313e-16, -0.8864191050454908),\n                        (4, 0.26261416185359243, 5.551115123125783e-17, 0.2677251264150761),\n                        (9, 0.5087596650646111, 0.0, 0.8156805681392465),\n                        (1, 0.4217899985400859, 0.0, -0.5001097254686009),\n                        (6, 0.4217899985400859, 0.0, -0.5001097254686009),\n                        (10, 0.4217899985400859, 0.0, -0.5001097254686009),\n                        (12, 0.3578842255793437, 0.0, -0.5503659482326804),\n                        (2, 0.3578842255793437, 0.0, -0.5503659482326804),\n                        (11, 0.4999999999999983, 0.0, -0.6146222686036573),\n                        (14, 0.4999999999999983, 0.0, -0.6146222686036573),\n                        (13, 0.3578842255793437, 0.0, -0.5503659482326804),\n                        (0, 0.4999999999999983, 0.0, -0.6146222686036573)],\n                       [(3, 7), (4, 8), (5, 9)],\n                       [(5, 0.45320025360561456, 0.0, -0.21870871047240859),\n                        (7, 0.1512785056553425, 0.0, 0.5693840621986512),\n                        (8, 0.7493346760757278, 0.0, -0.30482872864648747),\n                        (4, 0.7438230055427686, -2.220446049250313e-16, 0.3764888391835468),\n                        (9, 0.642694968094536, 0.0, 0.8156805681392183),\n                        (3, 0.4160893509934725, 0.0, -0.7031050230544382)],\n                       [(7, 8), (4, 5)],\n                       [(7, 0.9826444630231685, 0.0, 0.9067155315431648),\n                        (8, 0.6929669237054128, 0.0, -0.6971486751298195),\n                        (5, 0.2927870530532774, 1.0, -0.85373931689448)],\n                       [(8, 9), (4, 5), (3, 7)],\n                       [(7, 0.25912088891165563, 0.0, -0.6913800476431471),\n                        (3, 0.9533687963607917, 0.0, -0.5545553536015762),\n                        (8, 0.769672207155006, 1.0, -0.16897011454144661),\n                        (5, 0.08027942489481976, 0.0, 0.14626068310551993),\n                        (9, 0.5, 0.0, -0.7504995866865634),\n                        (4, 0.1758342165277997, 0.0, 0.992479657088305)],\n                       [(4, 8), (5, 9), (3, 7)],\n                       [(7, 0.28739457316555184, 0.0, -0.3192514343207198),\n                        (8, 0.3762663043167722, 0.0, 0.4141758438709737),\n                        (5, 0.39592282276087604, 0.0, 0.64626068310552),\n                        (9, 0.5, 0.0, 0.7495004133134366),\n                        (3, 0.44967481287458944, 1.0, 0.3051032439012417),\n                        (4, 0.4319895940485567, 0.0, 0.8896764600632685)],\n                       [(7, 8), (5, 9), (3, 4)],\n                       [(8, 0.630646241624421, 0.0, -0.6736172475847915),\n                        (9, 0.8856648732943704, 0.0, -0.4148147030619862),\n                        (7, 0.31014531891035224, 8.881784197001252e-16, 0.5606429642274875),\n                        (5, 0.5798003717521301, 0.9999999999999999, 0.64626068310552),\n                        (4, 0.4191540888185723, 0.0, -0.5170420158066475),\n                        (3, 0.0, 1.0, 0.0)],\n                       [(8, 9), (4, 5), (3, 7)],\n                       [(9, 0.40448319627960694, 0.0, 0.7686960448825371),\n                        (8, 0.7881822534297536, 0.0, 0.6310140089685501),\n                        (3, 4),\n                        (7, 0.3101453189103514, 0.0, -0.4393570357725114),\n                        (5, 0.5000000000000135, 0.0, 0.1462606831055186)],\n                       [(8, 9),\n                        (3, 0.15605886499272345, 0.0, 0.38282323868978446),\n                        (4, 0.5, 0.0, 0.6587764778394687)],\n                       [(9, 0.8511868477508192, 0.0, 0.1507411820887678),\n                        (3, 7),\n                        (8, 0.22678291490820102, 1.0, 0.19567492224747884)],\n                       [(5, 9),\n                        (7, 0.5796177256424373, 0.0, 0.5457625889330759),\n                        (3, 0.42379340641987423, 0.0, 0.36503800861797253),\n                        (4, 8)],\n                       [(9, 0.878367579040479, 0.0, 0.33385521888503966),\n                        (3, 1),\n                        (7, 6),\n                        (4, 0.5, 0.0, 0.6587764778394687)],\n                       [(9, 10),\n                        (3, 0.8116669186835013, 0.9999999999999999, 0.21431948711966664),\n                        (7, 0.8116669186835013, -2.220446049250313e-16, 0.3864040491495),\n                        (1, 0.6287848321594933, 0.0, -0.5144920515457643),\n                        (6, 0.6287848321594933, 0.0, -0.5144920515457643)],\n                       [(9, 0.8116669186835013, 1.0, 0.18755127557692447),\n                        (8, 7),\n                        (4, 3),\n                        (10, 0.6287848321594933, 0.0, -0.5144920515457643)],\n                       [(5, 9),\n                        (7, 0.7186000038451317, 0.0, -0.5919013584022754),\n                        (3, 0.7186000038451317, 0.0, -0.7639859204321089)],\n                       [(9, 0.7186000038451317, 0.0, -0.7907541319748512), (7, 6), (3, 1)],\n                       [(9, 10),\n                        (3, 0.1879696084437001, -0.9999999999999998, -0.5393518464139122),\n                        (7, 0.1879696084437001, 0.0, -0.36726728438407874),\n                        (6, 0.1426534639632415, 0.0, 0.23422019484333445),\n                        (1, 0.1426534639632415, 0.0, 0.23422019484333445)],\n                       [(9, 0.1879696084437001, 1.0, -0.5661200579566543),\n                        (8, 7),\n                        (4, 3),\n                        (10, 0.1426534639632415, 0.0, 0.23422019484333445)],\n                       [(5, 9),\n                        (3, 0.36815249285725926, 0.0, -0.08735646581485179),\n                        (7, 0.3681524928572592, 0.0, 0.08472809621498162),\n                        (4, 0.5, 0.0, 0.05851513438451583),\n                        (8, 0.5, 0.0, -0.0748189801695105)],\n                       [(9, 0.36815249285725926, 0.0, -0.11412467735759346),\n                        (3, 1),\n                        (7, 6),\n                        (5, 0.5, 4.440892098500626e-16, -0.963528806937823)],\n                       [(9, 10),\n                        (1, 0.5401993801370204, 0.0, -0.5512342235842844),\n                        (6, 0.5401993801370204, 0.0, -0.5512342235842844),\n                        (3, 0.5762322444517164, 1.0, 0.8818793518908621),\n                        (7, 0.5762322444517165, 0.0, -0.9460360860793045)],\n                       [(9, 0.5762322444517162, 0.9999999999999999, 0.8551111403481202),\n                        (10, 0.5401993801370204, 0.0, -0.5512342235842844),\n                        (7, 12),\n                        (3, 2),\n                        (6, 11),\n                        (1, 0)],\n                       [(9, 13),\n                        (10, 14),\n                        (0, 0.3260301916599948, 0.0, 0.43315900579188815),\n                        (2, 0.8053712368962477, 0.0, 0.18888285213015016),\n                        (12, 0.8053712368962477, 0.0, 0.18888285213015016),\n                        (1, 0.15204336199234872, 1.0, 0.9346037967062348),\n                        (6, 0.15204336199234872, 1.0, 0.9346037967062348),\n                        (11, 0.3260301916599948, 0.0, 0.43315900579188815),\n                        (4, 3),\n                        (7, 0.0, 0.7024201146828472, 0.0)],\n                       [(10, 0.15204336199234872, 1.0, 0.9346037967062348),\n                        (14, 0.3260301916599948, 0.0, 0.43315900579188815),\n                        (13, 0.8053712368962477, 0.0, 0.18888285213015016),\n                        (3, 1),\n                        (0, 2),\n                        (11, 12),\n                        (7, 6),\n                        (9, 0.0, 0.7024201146828472, 0.0),\n                        (4, 0.5, 0.34341100062810126, -0.9414848656154844)],\n                       [(14, 13),\n                        (9, 10),\n                        (6, 0.5, 0.0, -0.5653962032937654),\n                        (0, 0.5, 0.0, 0.8430180222691617),\n                        (11, 0.5, 0.0, 0.8430180222691617),\n                        (1, 0.5, 0.0, -0.5653962032937654),\n                        (2, 0.6008134304054109, 0.0, -0.040123990740045024),\n                        (12, 0.6008134304054109, 0.0, -0.040123990740045024),\n                        (8, 7),\n                        (3, 0.0, 0.7024201146828472, 0.0)],\n                       [(10, 0.5, 0.0, -0.5653962032937654),\n                        (14, 0.5, 0.0, 0.8430180222691617),\n                        (13, 0.6008134304054109, 0.0, -0.040123990740045024),\n                        (6, 11),\n                        (3, 2),\n                        (7, 12),\n                        (1, 0),\n                        (9, 0.0, 0.8715279661740896, 0.0),\n                        (8, 0.4999999999999998, 0.4767451151821269, -0.07481898016951027)],\n                       [(9, 13),\n                        (10, 14),\n                        (1, 0.5924401224278535, 1.0, -0.9133528413014171),\n                        (2, 0.3991865695945891, 0.0, 0.959876009259955),\n                        (11, 0.5, 0.0, -0.6569819777308383),\n                        (0, 0.5, 0.0, -0.6569819777308383),\n                        (6, 0.5924401224278535, 1.0, -0.9133528413014171),\n                        (12, 0.3991865695945891, 0.0, 0.959876009259955),\n                        (7, 0.0, 0.8715279661740896, 0.0),\n                        (3, 0.0, 0.8715279661740896, 0.0)],\n                       [(13, 0.3991865695945891, 0.0, 0.959876009259955),\n                        (10, 0.5924401224278535, 1.0, -0.9133528413014171),\n                        (14, 0.5, 0.0, -0.6569819777308383),\n                        (0, 2),\n                        (11, 12),\n                        (7, 6),\n                        (3, 1),\n                        (5, 9)],\n                       [(14, 13),\n                        (9, 10),\n                        (11, 0.25040914996542357, 0.0, 0.3268926652168218),\n                        (12, 0.1946287631037519, 0.0, 0.2308691663897604),\n                        (0, 0.25040914996542357, 0.0, 0.3268926652168218),\n                        (2, 0.1946287631037519, 0.0, 0.2308691663897604),\n                        (6, 0.9075598775721465, 0.0, -0.9133528413014158),\n                        (1, 0.9075598775721465, 0.0, -0.9133528413014158),\n                        (5, 0.5, -0.63454505804956, 0.03647119306217683)],\n                       [(14, 0.25040914996542357, 0.0, 0.3268926652168218),\n                        (13, 0.1946287631037519, 0.0, 0.2308691663897604),\n                        (10, 0.9075598775721465, 0.0, -0.9133528413014158),\n                        (3, 2),\n                        (7, 12),\n                        (1, 0),\n                        (6, 11)],\n                       [(10, 14),\n                        (9, 13),\n                        (3, 0.5, 0.0, -0.24283950041107927),\n                        (7, 0.5, 0.0, -0.0707549383812458),\n                        (1, 0.5, 0.0, 0.33664715869858547),\n                        (6, 0.5, 0.0, 0.33664715869858547),\n                        (11, 0.5, 0.0, 0.3497071393883171),\n                        (12, 0.7587162401878453, 4.440892098500626e-16, 0.8228655323944645),\n                        (2, 0.7587162401878453, 4.440892098500626e-16, 0.8228655323944645),\n                        (0, 0.5, 0.0, 0.3497071393883171)],\n                       [(9, 0.5, -2.220446049250313e-16, -0.2696077119538214),\n                        (10, 0.5, 0.0, 0.33664715869858547),\n                        (13, 0.7587162401878453, 4.440892098500626e-16, 0.8228655323944645),\n                        (14, 0.5, 0.0, 0.3497071393883171),\n                        (3, 1),\n                        (7, 6),\n                        (0, 2),\n                        (11, 12)],\n                       [(9, 10),\n                        (14, 13),\n                        (0, 0.5, 1.1102230246251565e-16, -0.15029286061168312),\n                        (3, 0.5, 1.0, -0.742839500411079),\n                        (6, 0.5, 0.0, -0.6633528413014143),\n                        (7, 0.5, 1.0, -0.5707549383812458),\n                        (11, 0.5, 1.1102230246251565e-16, -0.15029286061168312),\n                        (1, 0.5, 0.0, -0.6633528413014143)],\n                       [(10, 0.5, 0.0, -0.6633528413014143),\n                        (9, 0.5000000000000001, -0.9999999999999998, -0.7696077119538214),\n                        (14, 0.5, 1.1102230246251565e-16, -0.15029286061168312),\n                        (1, 0),\n                        (7, 12),\n                        (6, 11),\n                        (3, 2)],\n                       [(10, 14),\n                        (9, 13),\n                        (0, 0.0, 0.378896047164595, 0.0),\n                        (2, 1.0, 0.0, 0.16765763650036608),\n                        (6, 0.5, -0.10804397214567385, -0.663352841301414),\n                        (11, 0.0, 0.378896047164595, 0.0),\n                        (12, 1.0, 0.0, 0.16765763650036608),\n                        (1, 0.5, -0.10804397214567385, -0.663352841301414),\n                        (3, 0.5000000000000002, 0.47144268696399116, 0.757160499588921),\n                        (7, 0.5000000000000002, 0.2993581249341575, 0.9292450616187544)],\n                       [(14, 0.0, 0.378896047164595, 0.0),\n                        (9, 0.5000000000000002, 0.4982108985067333, 0.7303922880461784),\n                        (13, 1.0, 0.0, 0.16765763650036608),\n                        (10, 0.5, -0.10804397214567385, -0.663352841301414)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],\n           \'qubits\': [(3, 3),\n                      (3, 4),\n                      (4, 3),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (5, 3),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 3),\n                      (6, 4),\n                      (6, 6),\n                      (6, 7)]},\n (15, 4): {\'method\': \'certified count kernels + parallel coherent decoders\',\n           \'moments\': [[(3, 0.5050311694609142, 0.0, 0.5),\n                        (7, 0.7754283788865404, 0.0, -0.5),\n                        (4, 0.31203074057342906, 0.0, -0.5),\n                        (8, 0.6647424953468708, 0.0, -0.5),\n                        (5, 0.4019600678849752, 0.0, 0.5),\n                        (9, 0.9860634487343363, 0.0, 0.5),\n                        (2, 0.5, 0.0, -0.07534461897426747),\n                        (10, 0.5, 0.0, -0.07535137591189756),\n                        (13, 0.5, 0.0, -0.07534461897426747),\n                        (1, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (12, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (0, 0.5, 0.0, 0.5),\n                        (6, 0.3040867239846966, 0.0, -0.5745192213521528),\n                        (11, 0.3040867239846966, 0.0, -0.5745192213521528),\n                        (14, 0.3040867239846966, 0.0, -0.5745192213521528)],\n                       [(3, 7), (4, 8), (5, 9)],\n                       [(3, 0.5066084360344099, 0.0, 0.5),\n                        (4, 0.7976060787857662, 0.0, 0.5),\n                        (7, 0.004284890859710177, 0.0, 0.5),\n                        (8, 0.7535395929271749, 0.0, 0.5),\n                        (5, 0.04960829035671744, 0.0, 0.5),\n                        (9, 0.9957726141502476, 0.0, -0.5000000000000002)],\n                       [(3, 4), (7, 8)],\n                       [(4, 5),\n                        (3, 0.3881710437984986, 0.0, 0.5),\n                        (7, 0.8887989454065368, 0.0, 0.5),\n                        (8, 0.3000520402754841, 0.0, -0.5)],\n                       [(3, 7),\n                        (8, 9),\n                        (4, 0.49999999999999223, 0.0, 0.5),\n                        (5, 0.1993382800735905, 0.0, 0.5)],\n                       [(8, 0.20007974046133922, 0.0, -0.5),\n                        (9, 0.4999999999999998, 0.0, 0.5),\n                        (3, 0.09428404233770227, 0.0, 0.5),\n                        (7, 0.8168684096225483, 0.0, 0.5)],\n                       [(4, 8), (5, 9)],\n                       [(4, 0.6272655967963714, 0.0, -0.5000000000000002),\n                        (8, 0.139052622364793, 0.0, -0.5000000000000002),\n                        (9, 0.5, 0.0, 0.5),\n                        (5, 0.9921928118257514, 0.0, -0.5)],\n                       [(3, 4), (7, 8)],\n                       [(8, 9), (7, 0.5, 0.0, -0.5), (4, 0.8727344032036237, 0.0, 0.5)],\n                       [(3, 7), (4, 5), (9, 0.8148309319180607, 0.0, 0.5)],\n                       [(4, 0.7233498104809428, 0.0, 0.5),\n                        (5, 0.6959132760153035, 0.0, 0.5),\n                        (7, 0.5, 0.0, 0.5)],\n                       [(4, 8), (5, 9)],\n                       [(7, 8),\n                        (3, 4),\n                        (5, 0.7084946987581148, 0.0, -0.8030737309917009),\n                        (9, 0.692663307285299, 4.440892098500626e-16, -0.7615318252987646)],\n                       [(5, 2),\n                        (9, 10),\n                        (7, 0.8217849400235957, 0.0, -0.5),\n                        (8, 0.3225128449059285, 0.0, 0.41340151052360685),\n                        (4, 0.6772758467840196, 0.0, 0.5865961897691603)],\n                       [(7, 8),\n                        (3, 4),\n                        (9, 0.7797752110042575, 0.0, -0.4263909113599711),\n                        (5, 0.7913327671706831, 0.0, -0.49374351881814893),\n                        (10, 0.6238305862596978, 0.0, -0.8522218234838357),\n                        (2, 0.6321049414225235, 0.0, -0.829092751915236)],\n                       [(8, 0.15491981575607633, 0.0, -0.43215980396583076),\n                        (4, 0.155480511485734, 0.0, 0.7411711453155834)],\n                       [(8, 9), (4, 5)],\n                       [(9, 10),\n                        (5, 2),\n                        (8, 0.803176253409799, 0.0, -0.43215980396583054),\n                        (4, 0.7964417643891555, 0.0, 0.7411711453155836)],\n                       [(7, 8),\n                        (3, 4),\n                        (9, 0.5814154756595525, 0.0, 0.5736090886400287),\n                        (5, 0.5886108001133223, 4.440892098500626e-16, 0.5062564811818508),\n                        (10, 0.6296060223028559, 0.0, -0.486663789960482),\n                        (2, 0.597350526917102, 0.0, -0.4537132312119436)],\n                       [(8, 0.8031762534097988, 0.0, 0.5678401960341692),\n                        (4, 0.7964417643891555, 1.1102230246251565e-16, -0.25882885468441685)],\n                       [(8, 9), (4, 5)],\n                       [(8, 0.5490235858977989, 0.0, -0.43215980396583076),\n                        (9, 0.16666666666666666, 0.0, -0.42639091135997154),\n                        (4, 0.5517093529747915, 0.0, 0.7411711453155838),\n                        (5, 0.1795085179856881, 0.0, -0.49374351881814915)],\n                       [(7, 8), (9, 10), (3, 4), (5, 2)],\n                       [(8, 0.9019528282044029, 0.0, 0.567840196034169),\n                        (9, 0.14193433482139067, 0.0, -0.4263909113599711),\n                        (4, 0.893493814032364, 0.0, -0.25882885468441574),\n                        (5, 0.11878997884016869, 0.0, -0.4937435188181496),\n                        (7, 0.5, 0.0, -0.43524576940924997),\n                        (3, 0.5000000000000009, 0.0, 0.5648383208025802)],\n                       [(8, 9), (4, 5)],\n                       [(8, 0.5, 0.0, 0.567840196034169),\n                        (9, 10),\n                        (4, 0.5, 0.0, -0.2588288546844164),\n                        (5, 2)],\n                       [(7, 8),\n                        (3, 4),\n                        (9, 0.1358581021559238, 0.0, 0.5733752041754423),\n                        (10, 0.3478770495896244, 0.0, -0.4863274291131705),\n                        (2, 0.7780272931625696, -2.220446049250313e-16, 0.20475567610558176),\n                        (5, 0.6444528487918011, 4.440892098500626e-16, 0.9118184864199645)],\n                       [(8, 9),\n                        (10, 13),\n                        (2, 1),\n                        (4, 0.5, 0.0, -0.4945175108212805),\n                        (7, 0.5, 0.0, -0.43524576940925064),\n                        (3, 0.5, 0.9999999999999967, 0.5648383208025809)],\n                       [(4, 5),\n                        (9, 0.155480511485734, 4.440892098500626e-16, 0.5737256218450286),\n                        (10, 0.7913327671706831, 0.0, 0.8345887996359482),\n                        (1, 0.85928593962011, 0.0, 0.07376677451045754),\n                        (2, 0.5, 0.0, -0.29524432389441824),\n                        (13, 0.6321049414225235, 0.0, 0.17090724808476404)],\n                       [(9, 10), (4, 0.5, 0.0, -0.4945175108212809), (5, 2)],\n                       [(1, 4),\n                        (10, 13),\n                        (9, 0.7964417643891555, 1.1102230246251565e-16, -0.42627437815497116),\n                        (2, 0.6714165234362537, 0.0, 0.6333395843745888),\n                        (5, 0.5, 0.0, -0.5803631122665824)],\n                       [(8, 9),\n                        (1, 0.7485956033393342, 0.0, 0.5737667745104575),\n                        (4, 5),\n                        (10, 0.5886108001133223, 0.0, -0.16541120036405133),\n                        (13, 0.597350526917102, 0.0, -0.4537132312119436)],\n                       [(2, 1),\n                        (9, 0.7964417643891555, -4.440892098500626e-16, -0.4262743781549714),\n                        (4, 0.05280605500441269, 0.0, 0.6790828329656893),\n                        (5, 0.5, 0.0, -0.5803631122665824)],\n                       [(9, 10),\n                        (1, 0.9961572718865779, 0.0, -0.9262332254895478),\n                        (2, 0.3935294720438234, 0.0, 0.3542924687609674)],\n                       [(1, 4),\n                        (5, 2),\n                        (9, 0.5517093529747915, 0.0, -0.4262743781549707),\n                        (10, 0.1795085179856881, 0.0, 0.8345887996359487)],\n                       [(8, 9),\n                        (10, 13),\n                        (2, 0.31043695253214243, 0.0, -0.2271862808388332),\n                        (1, 0.5655532211464794, 0.0, 0.5737667745104469),\n                        (4, 0.24963665794604517, 0.0, -0.803554582124536)],\n                       [(2, 1),\n                        (4, 5),\n                        (9, 0.893493814032364, 0.0, -0.4262743781549707),\n                        (10, 0.11878997884016869, 0.0, 0.8345887996359485),\n                        (8, 0.5000000000000009, 0.0, 0.5677561058223375)],\n                       [(9, 10),\n                        (1, 0.4999999999999998, 0.0, -0.4262332254895531),\n                        (4, 0.32701156232958084, 0.0, -0.19577422341263784),\n                        (2, 0.6193705712509703, 0.0, 0.8998480300605716)],\n                       [(1, 4), (5, 2), (10, 13), (9, 0.5, 0.0, 0.5737256218450284)],\n                       [(8, 9),\n                        (1, 0.5960614607313279, 0.0, 0.8831564130503458),\n                        (4, 0.5, 0.0, -0.9484096919337615),\n                        (2, 0.6256105257587086, 1.1102230246251565e-16, 0.4165776620509426),\n                        (13, 0.7780272931625696, 0.0, 0.20475567610558215),\n                        (10, 0.6444528487918006, 0.0, -0.5709732056021655)],\n                       [(1, 4),\n                        (5, 2),\n                        (13, 12),\n                        (9, 0.5, 0.0, -0.6619630342918361),\n                        (8, 0.5, 0.9970822149802403, -0.4322438941776632)],\n                       [(9, 10),\n                        (2, 0.9764747842833552, 0.0, -0.18120658002844947),\n                        (1, 0.14949404734505523, 0.0, 0.2730671118828738),\n                        (4, 0.5, 0.0, -0.4484096919337619),\n                        (5, 0.5, 0.0, -0.2592916829591003),\n                        (13, 0.5000000000000001, 0.0, -0.29524432389441824),\n                        (12, 0.85928593962011, 0.0, 0.07376677451045754)],\n                       [(2, 1), (4, 5), (10, 13), (9, 0.5, 0.0, 0.33803696570816477)],\n                       [(12, 9),\n                        (1, 4),\n                        (5, 0.5, 0.0, -0.2592916829591003),\n                        (2, 0.5, 0.0, -0.42418486321032356),\n                        (10, 0.5, 0.0, 0.9212083930843806),\n                        (13, 0.6714165234362537, 3.3306690738754696e-16, -0.36666041562541096)],\n                       [(5, 2),\n                        (9, 10),\n                        (12, 0.7485956033393342, 0.0, 0.5737667745104575),\n                        (1, 0.4247761378557593, 0.0, -0.02238185157853345)],\n                       [(13, 12),\n                        (2, 0.14731871270290633, 0.0, -0.42418486321032356),\n                        (5, 0.5, 0.0, 0.6735194630222618),\n                        (10, 0.5000000000000001, 4.440892098500626e-16, 0.9212083930843808),\n                        (9, 0.0528060550044127, 4.440892098500626e-16, 0.5116373094951343)],\n                       [(2, 1),\n                        (4, 5),\n                        (13, 0.3935294720438233, 0.0, -0.645707531239033),\n                        (12, 0.9961572718865779, 0.0, -0.9262332254895478)],\n                       [(10, 13),\n                        (12, 9),\n                        (1, 0.4500869528994624, 0.0, 0.6634633138266384),\n                        (4, 0.4202007516434625, 0.0, -0.7418473335252633),\n                        (2, 0.9433458052208397, 0.0, -0.9241848632103227)],\n                       [(1, 4),\n                        (5, 2),\n                        (9, 0.24963665794604517, 0.0, -0.9710001055950901),\n                        (13, 0.31043695253214243, 0.0, -0.22718628083883408),\n                        (12, 0.5655532211464794, 0.0, 0.5737667745104469)],\n                       [(9, 10),\n                        (13, 12),\n                        (1, 0.541088904736081, 0.0, -0.7896451505492761),\n                        (5, 0.20567003520391758, 0.0, -0.8264805369777373),\n                        (4, 0.5, 0.0, -0.3370432186859893)],\n                       [(1, 0),\n                        (5, 4),\n                        (13, 0.6193705712509703, 0.0, 0.8998480300605713),\n                        (12, 0.4999999999999998, 0.0, -0.4262332254895531),\n                        (9, 0.32701156232958106, 0.0, 0.6367802531168079)],\n                       [(10, 13),\n                        (12, 9),\n                        (1, 0.5, 0.0, 0.7103548494507246),\n                        (5, 0.4999999999999824, 0.0, 0.6735194067043218),\n                        (0, 0.3040867239846962, 0.0, 0.4999999999999998),\n                        (4, 0.5000000000000019, 2.220446049250313e-16, -0.24838328909382357)],\n                       [(2, 1),\n                        (5, 6),\n                        (13, 0.6256105257587086, 0.0, -0.583422337949058),\n                        (12, 0.5960614607313279, 0.0, 0.8831564130503458),\n                        (9, 0.5, 0.0, -0.11585521540431598)],\n                       [(10, 13),\n                        (12, 9),\n                        (1, 0.8040867239846965, 0.0, 0.7103548494507246),\n                        (5, 0.5000000000000102, 0.0, 0.6735194067043218),\n                        (6, 0.30408672398469605, 0.0, 0.4254807786478514)],\n                       [(1, 0),\n                        (5, 4),\n                        (9, 0.5, 0.0, 0.384144784595684),\n                        (10, 0.5000000000000001, 0.0, 0.6001369637768981),\n                        (13, 0.9764747842833552, 0.0, -0.1812065800284497),\n                        (12, 0.14949404734505523, 0.0, 0.2730671118828738)],\n                       [(9, 10),\n                        (13, 12),\n                        (1, 0.75, 0.0, 0.7103548494507246),\n                        (5, 0.7499999999999999, 0.0, -0.32648062581085613),\n                        (0, 0.5, 0.0, 0.5),\n                        (4, 0.49999999999999956, 0.0, 0.7516167434213588)],\n                       [(2, 1),\n                        (5, 6),\n                        (10, 0.5, 0.0, 0.6001369637768981),\n                        (13, 0.5, 0.0, -0.4241848632103249),\n                        (12, 9)],\n                       [(10, 13),\n                        (1, 0.5, 0.0, -0.28964515054927675),\n                        (5, 0.500526889421938, 0.0, 0.17299243806197828),\n                        (12, 0.4247761378557593, 0.0, -0.02238185157853345),\n                        (2, 0.5, 0.0, -0.44472931557836404),\n                        (6, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515)],\n                       [(1, 0),\n                        (5, 4),\n                        (10, 0.5, 1.1102230246251565e-16, -0.33267418220446476),\n                        (13, 0.14731871270290633, 0.0, 0.5758151367896751)],\n                       [(9, 10),\n                        (13, 12),\n                        (1, 0.25000000000000006, 0.0, 0.7103548494507246),\n                        (5, 0.5, 0.0, 0.4229920019885469),\n                        (4, 0.5000000000000004, 0.0, 0.2516167434213594)],\n                       [(2, 1),\n                        (5, 6),\n                        (13, 0.9433458052208397, 0.0, 0.07581513678967644),\n                        (12, 0.4500869528994624, 0.0, 0.6634633138266384),\n                        (9, 0.4202007516434625, 0.0, 0.09070714300418126)],\n                       [(5, 4),\n                        (10, 13),\n                        (12, 9),\n                        (1, 0.5, 0.0, -0.28964515054927675),\n                        (2, 0.5, -0.0552706844216353, 0.5552706844216355),\n                        (6, 0.5, 0.3483047662953689, -0.5745192538673269)],\n                       [(1, 0),\n                        (10, 0.20567003520391755, 0.0, -0.8326741822044659),\n                        (9, 0.5, 0.0, 0.49551125784345595),\n                        (13, 0.5, 0.0, -0.7185148280064055),\n                        (12, 0.5, 0.0, 0.7103548494507246),\n                        (4, 0.9992548329781763, -0.47783123099330016, 0.7516167434213521),\n                        (5, 3.251517832758566e-08, -0.14920648956050453, -0.5770080206769421)],\n                       [(10, 9),\n                        (13, 12),\n                        (0, 1.0, 0.0, 0.5),\n                        (1, 0.4999999999999998, 0.7896451505492759, 0.7103548494507241)],\n                       [(10, 0.49999999999998224, 0.0, -0.332674125886524),\n                        (13, 0.49999999999998246, 0.0, 0.2814851156756535),\n                        (9, 0.5000000000000019, 0.0, -0.41582881256437876),\n                        (12, 0.5000000000000022, 0.0, 0.6216949198585568)],\n                       [(10, 11), (13, 14)],\n                       [(10, 0.5000000000000102, 0.0, -0.332674125886524),\n                        (13, 0.5000000000000102, 1.1102230246251565e-16, 0.28148511567565293),\n                        (11, 0.30408672398469627, 0.0, -0.5745192213521484),\n                        (14, 0.30408672398469605, 0.0, 0.4254807786478514)],\n                       [(10, 9), (13, 12)],\n                       [(10, 0.75, 0.0, 0.6673259066286545),\n                        (13, 0.7499999999999999, 1.1102230246251565e-16, 0.2814850831604748),\n                        (9, 0.49999999999999956, 0.0, -0.4158287800491969),\n                        (12, 0.49999999999999956, -4.440892098500626e-16, -0.37830511265662414)],\n                       [(10, 11), (13, 14)],\n                       [(10, 0.500526889421938, 1.6653345369377348e-16, 0.16785284275581847),\n                        (13, 0.500526889421938, 0.0, 0.7809581470333107),\n                        (11, 0.5000000000000102, 0.0, 0.4254807786478514),\n                        (14, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515)],\n                       [(10, 9), (13, 12)],\n                       [(10, 0.5, 0.0, -0.08214672117074828),\n                        (13, 0.5000000000000001, 0.0, 0.030957710959877205),\n                        (9, 0.5000000000000004, 0.0, 0.08417121995080273),\n                        (12, 0.5000000000000003, 0.0, 0.12169488734337536)],\n                       [(10, 11), (13, 14)],\n                       [(10, 9),\n                        (13, 12),\n                        (11, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (14, 0.5, 0.3483047662953689, -0.5745192538673269)],\n                       [(9, 0.9992548329781763, 0.6896142924772553, -0.41582878004920465),\n                        (10, 0.999999967484822, 0.35593218826781414, -0.0821466985052608),\n                        (12, 0.0007451670218237004, 0.6520906250846661, 0.6216948873433847),\n                        (13, 3.251517832758566e-08, -0.7571721985318351, 0.030957688294388497)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],\n           \'qubits\': [(2, 5),\n                      (3, 5),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8),\n                      (6, 6),\n                      (6, 7),\n                      (6, 8)]},\n (15, 5): {\'method\': \'certified count kernels + parallel coherent decoders\',\n           \'moments\': [[(0, 0.5, 0.0, 0.5),\n                        (1, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (2, 0.7010577384351091, 0.0, 0.9439691995767472),\n                        (3, 0.5489789454250179, 0.0, 0.5),\n                        (4, 0.6275002275345876, 0.0, -0.5),\n                        (5, 0.15428447475915785, 0.0, 0.5),\n                        (6, 0.30408672398469655, 0.0, -0.5745192213521528),\n                        (7, 0.16462440927981217, 0.0, -0.5000000000000002),\n                        (9, 0.7654787523830568, 0.0, 0.5),\n                        (10, 0.6965922152469375, 0.0, 0.9441038655014211),\n                        (11, 0.30408672398469655, 0.0, -0.5745192213521528),\n                        (12, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (13, 0.7010577384351091, 0.0, 0.9439691995767472),\n                        (14, 0.30408672398469655, 0.0, -0.5745192213521528)],\n                       [(3, 7), (4, 8), (5, 9)],\n                       [(3, 0.9463453303367797, 0.0, 0.5),\n                        (4, 0.8944734479102134, 0.0, 0.5),\n                        (5, 0.5236972827102715, 0.0, 0.5),\n                        (7, 0.2732330958832385, 0.0, 0.5),\n                        (9, 0.8346328303977621, 0.0, -0.5000000000000002)],\n                       [(3, 4), (7, 8)],\n                       [(4, 0.8480064173628117, 0.0, 0.5), (7, 0.7996673274560251, 0.0, -0.5), (8, 9)],\n                       [(3, 7), (4, 5), (9, 0.41961689080975334, 0.0, -0.5)],\n                       [(4, 0.834181082212214, 0.0, -0.5),\n                        (5, 0.35521166835174856, 0.0, 0.5),\n                        (7, 0.33750642686758914, 0.0, 0.5)],\n                       [(4, 8), (5, 9)],\n                       [(3, 7),\n                        (4, 0.7110369344122653, 0.0, 0.5),\n                        (5, 0.5945881689715725, 0.0, -0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (9, 0.45214815790160445, 0.0, 0.4999999999999998)],\n                       [(3, 4), (7, 8), (9, 10)],\n                       [(4, 0.20908557661949764, 0.0, -0.5),\n                        (8, 0.4999999999999998, 0.0, 0.5),\n                        (10, 0.30340778475306246, 0.0, 0.9441038655014202)],\n                       [(4, 5), (8, 9)],\n                       [(3, 4), (5, 0.5945881689715726, 0.0, 0.5), (8, 0.5, 0.0, -0.5)],\n                       [(5, 9), (7, 8)],\n                       [(4, 8), (5, 0.09458816897157266, 0.0, -0.5000000000000002)],\n                       [(4, 0.39529799192523557, 0.0, -0.10562369964702767),\n                        (8, 0.6033027289115123, -2.220446049250313e-16, 0.09960764295316249)],\n                       [(3, 4), (5, 2), (7, 8)],\n                       [(2, 0.29894226156489107, 0.0, -0.05603080042325281),\n                        (4, 0.7021281047129817, -2.220446049250313e-16, -0.105623699647027),\n                        (8, 0.6987407529165437, 0.0, 0.6323325842192944)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.7118222410987194, 0.0, -0.105623699647027),\n                        (5, 0.6662003855379057, 0.0, 0.7259619260711767),\n                        (8, 0.719448888067144, 2.220446049250313e-16, -0.3676674157807054),\n                        (9, 0.7010945021951457, 0.0, -0.7080336276853447)],\n                       [(3, 4), (5, 2), (7, 8), (9, 10)],\n                       [(2, 0.4198773247206662, 0.0, -0.07074532720037463),\n                        (4, 0.9813476461135342, 0.0, 0.8943763003529741),\n                        (5, 0.6662003855379061, 0.0, 0.7259619260711767),\n                        (8, 0.9875646122406043, 0.0, -0.3676674157807034),\n                        (9, 0.7010945021951457, 0.0, -0.7080336276853445),\n                        (10, 0.4199704507821304, 0.0, -0.04755522240029175)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.0020996535016382105, 0.0, -0.105623699647035),\n                        (5, 0.6135871361834435, 0.0, -0.350970259161139),\n                        (8, 0.011556148971274325, 0.0, 0.6323325842192999),\n                        (9, 0.6150646219686983, 0.0, -0.6406889769585558)],\n                       [(3, 4), (5, 2), (7, 8), (9, 10)],\n                       [(2, 0.5000000000000011, 0.0, -0.6245332277151969),\n                        (4, 0.1651746478714715, 0.0, -0.10562369964702523),\n                        (5, 0.19491599185619113, 4.440892098500626e-16, 0.7173798007565817),\n                        (8, 0.17562795090667677, 0.0, 0.6323325842192973),\n                        (9, 0.19388965723046162, 0.0, 0.2851951825230763),\n                        (10, 0.5, 0.0, 0.3945762649608535)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.5846001477399267, 0.0, -0.10562369964702523),\n                        (5, 2),\n                        (8, 0.5981738766370478, 0.0, -0.3676674157807014)],\n                       [(3, 4),\n                        (5, 0.5886108001133225, 4.440892098500626e-16, -0.8711660270602786),\n                        (7, 8),\n                        (9, 10),\n                        (2, 0.4818796387550885, 0.0, 0.6712802634947106)],\n                       [(4, 0.20938585749425948, 0.0, 0.894376300352975),\n                        (8, 0.1954577893623162, 0.0, -0.36766741578070183),\n                        (9, 0.5814154756595525, 0.0, 0.8808971481500283),\n                        (10, 0.4802737868646165, 4.440892098500626e-16, 0.6921121458286728)],\n                       [(4, 5), (8, 9), (10, 13)],\n                       [(4, 0.4482906470252086, 0.0, 0.894376300352975),\n                        (5, 0.7627370268849369, 0.0, -0.8711660270602788),\n                        (8, 0.45097641410220146, 0.0, 0.6323325842192982),\n                        (9, 0.7821336744650593, 0.0, -0.11910285184997305),\n                        (13, 0.29894226156489107, 0.0, -0.05603080042325281)],\n                       [(3, 4), (5, 2), (7, 8), (9, 10)],\n                       [(3, 0.5, 0.0, 0.6971881501764858),\n                        (4, 0.893493814032364, 0.0, 0.8943763003529754),\n                        (5, 0.4033542756707677, 0.0, 0.12883397293972076),\n                        (7, 0.5, 0.0, 0.30409363494590874),\n                        (8, 0.9019528282044029, 0.0, -0.36766741578070183),\n                        (9, 0.38515105951380035, 0.0, 0.8808971481500263)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.03519630114910766, 0.0, 0.8943763003529761),\n                        (5, 0.8339515505182198, 0.0, -0.8711660270602319),\n                        (9, 0.8527547392502051, 0.0, 0.8808971481500532),\n                        (8, 0.009468002358105965, 0.0, 0.6323325842193022)],\n                       [(3, 4), (5, 2), (9, 10)],\n                       [(2, 0.7780272931625696, 0.0, 0.05572810904583947),\n                        (3, 0.4999999999999998, 8.881784197001252e-16, 0.6971881501764856),\n                        (4, 0.5, 0.0, 0.7910374735900201),\n                        (5, 0.9352064365349756, 2.220446049250313e-16, -0.34290504099508423),\n                        (7, 8),\n                        (9, 0.9168873757132819, 0.0, 0.8714977051481281)],\n                       [(2, 1),\n                        (4, 5),\n                        (7, 0.4999999999999998, -0.6081872698918169, 0.30409363494590863),\n                        (8, 9)],\n                       [(1, 0.85928593962011, 0.0, -0.9262332254895425),\n                        (2, 0.5, 0.0, 0.555728109045839),\n                        (4, 0.5, 0.0, 0.7910374735900203),\n                        (9, 0.7021281047129817, 0.0, 0.8615885497105722)],\n                       [(1, 4), (5, 2), (9, 10)],\n                       [(1, 0.7485956033393342, 0.0, 0.5737667745104573),\n                        (2, 0.6714165234362537, 4.440892098500626e-16, 0.6271442007768322),\n                        (5, 0.5, 4.440892098500626e-16, 0.984299807304295),\n                        (9, 0.7118222410987194, -2.220446049250313e-16, 0.861588549710572),\n                        (10, 0.6662003855379057, 0.0, 0.2162466446835385)],\n                       [(2, 1), (4, 5), (8, 9), (10, 13)],\n                       [(1, 0.9961572718865779, 0.0, 0.07376677451045216),\n                        (2, 0.3935294720438234, 0.0, 0.906191316390454),\n                        (4, 0.0528060550044127, 4.440892098500626e-16, 0.6174371298030485),\n                        (5, 0.5, 1.1102230246251565e-16, -0.015700192695704995),\n                        (9, 0.9813476461135342, -2.220446049250313e-16, -0.13841145028942892),\n                        (10, 0.6662003855379061, 0.0, 0.2162466446835385),\n                        (13, 0.4198773247206662, 0.0, -0.07074532720037463)],\n                       [(1, 4), (5, 2), (9, 10)],\n                       [(1, 0.5655532211464794, 0.0, 0.5737667745104469),\n                        (2, 0.3104369525321424, 0.0, 0.48767006599025464),\n                        (4, 0.24963665794604517, -2.220446049250313e-16, 0.10007454489327361),\n                        (9, 0.00209965350163821, 0.0, 0.8615885497105775),\n                        (10, 0.6135871361834435, 0.0, 0.13931445945122384)],\n                       [(2, 1), (4, 5), (8, 9), (10, 13)],\n                       [(1, 0.4999999999999998, 0.0, 0.5737667745104462),\n                        (2, 0.6193705712509703, 0.0, -0.6393642449091499),\n                        (4, 0.32701156232958084, 0.0, 0.4922941861813757),\n                        (9, 0.16517464787147149, 4.440892098500626e-16, 0.8615885497105704),\n                        (10, 0.19491599185619116, 0.0, -0.7923354806310561),\n                        (13, 0.5000000000000011, 0.0, -0.6245332277151969)],\n                       [(1, 4), (5, 2), (9, 10)],\n                       [(1, 0.5960614607313279, 0.0, 0.8831564130503458),\n                        (2, 0.6256105257587087, 0.0, 0.8439061231004792),\n                        (4, 0.5, 0.0, -0.7550703452975007),\n                        (9, 0.5846001477399267, 4.440892098500626e-16, 0.86158854971057),\n                        (10, 13)],\n                       [(1, 4),\n                        (5, 2),\n                        (8, 9),\n                        (10, 0.5886108001133225, 8.881784197001252e-16, 0.6191186915520834),\n                        (13, 0.4818796387550885, 0.0, 0.6712802634947106)],\n                       [(1, 0.14949404734505523, 0.0, -0.7269328881171262),\n                        (2, 0.9764747842833552, 0.0, 0.4416903651798709),\n                        (4, 0.5, 4.440892098500626e-16, 0.7449296547024993),\n                        (5, 0.5, 0.0, -0.3367716220031869),\n                        (9, 0.20938585749425953, 0.0, -0.13841145028943025)],\n                       [(2, 1), (4, 5), (9, 10)],\n                       [(1, 4),\n                        (2, 0.5, 0.0, 0.6846686483617455),\n                        (5, 0.5, 0.0, 0.6632283779968131),\n                        (9, 0.4482906470252086, 0.0, -0.13841145028943047),\n                        (10, 0.7627370268849369, 0.0, -0.38088130844791723)],\n                       [(1, 0.42477613785575913, 0.0, 0.9776181484214663), (5, 2), (8, 9), (10, 13)],\n                       [(2, 0.14731871270290633, 0.0, -0.31533135163825454),\n                        (5, 0.5, 4.440892098500626e-16, 0.7304172320154503),\n                        (8, 0.5, 0.0, -0.3663856306583084),\n                        (9, 0.893493814032364, 2.220446049250313e-16, -0.13841145028943114),\n                        (10, 0.4033542756707678, 4.440892098500626e-16, 0.6191186915520821)],\n                       [(2, 1), (4, 5), (9, 10)],\n                       [(1, 0.4500869528994624, 0.0, 0.6634633138266381),\n                        (2, 0.9433458052208397, 0.0, 0.18466864836174457),\n                        (4, 0.42020075164346243, 4.440892098500626e-16, -0.9616327037059986),\n                        (9, 0.03519630114910766, 0.0, -0.13841145028943047),\n                        (10, 0.8339515505182198, 4.440892098500626e-16, 0.6191186915521296)],\n                       [(1, 4), (5, 2), (8, 9), (10, 13)],\n                       [(1, 0.5410889047360808, -2.220446049250313e-16, 0.21035484945072325),\n                        (4, 0.5, 4.440892098500626e-16, 0.6335631814547269),\n                        (5, 0.20567003520391755, 0.0, 0.23041723201544917),\n                        (8, 0.4999999999999998, 0.06357378083479492, 0.6336143693416905),\n                        (9, 0.5, 0.0, -0.03507262352647489),\n                        (10, 0.9352064365349756, 0.0, -0.8526203223827222),\n                        (13, 0.7780272931625696, 0.0, 0.05572810904583947)],\n                       [(1, 0), (5, 4), (9, 10), (13, 12)],\n                       [(0, 0.3040867239846962, 0.0, 0.4999999999999998),\n                        (1, 0.5, 0.0, -0.28964515054927675),\n                        (4, 0.5000000000000022, 0.0, -0.4550967481374384),\n                        (5, 0.4999999999999827, 0.0, -0.26958271166660897),\n                        (9, 0.5, 0.0, -0.035072623526474445),\n                        (12, 0.85928593962011, 0.0, -0.9262332254895425),\n                        (13, 0.5, 0.0, 0.555728109045839)],\n                       [(2, 1), (5, 6), (10, 13), (12, 9)],\n                       [(1, 0.8040867239846965, 4.440892098500626e-16, 0.710354849450723),\n                        (5, 0.5000000000000103, 0.0, -0.26958271166660874),\n                        (6, 0.30408672398469627, 0.0, -0.5745192213521484),\n                        (10, 0.5, 0.0, -0.5254154740833425),\n                        (12, 0.7485956033393342, 0.0, -0.42623322548954246),\n                        (13, 0.6714165234362537, 4.440892098500626e-16, 0.6271442007768322)],\n                       [(1, 0), (5, 4), (9, 10), (13, 12)],\n                       [(0, 0.5, 0.0, 0.5),\n                        (1, 0.7499999999999999, 0.0, 0.710354849450723),\n                        (4, 0.49999999999999956, 0.0, -0.4550967806526205),\n                        (5, 0.75, 0.0, -0.26958267915142997),\n                        (9, 0.052806055004412704, 0.0, 0.13852772026049504),\n                        (10, 0.5000000000000001, 0.0, -0.5254154740833412),\n                        (12, 0.9961572718865779, 0.0, -0.9262332254895478),\n                        (13, 0.3935294720438234, 0.0, 0.906191316390454)],\n                       [(2, 1), (5, 6), (10, 13), (12, 9)],\n                       [(1, 0.5, 0.0, 0.710354849450723),\n                        (2, 0.5, 0.0, 0.7052131007297842),\n                        (5, 0.500526889421938, 0.0, -0.7690557430242662),\n                        (6, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155),\n                        (9, 0.24963665794604517, 0.0, 0.6558903051702705),\n                        (12, 0.5655532211464794, 0.0, 0.5737667745104469),\n                        (13, 0.3104369525321424, 0.0, 0.48767006599025464)],\n                       [(1, 0), (5, 4), (9, 10), (13, 12)],\n                       [(1, 0.25, 0.0, -0.28964515054927675),\n                        (4, 0.5000000000000004, 0.0, -0.9550967806526212),\n                        (5, 0.5, 0.0, -0.01905530695083435),\n                        (9, 0.32701156232958095, 0.0, 0.26367066388216737),\n                        (12, 0.4999999999999998, 0.0, 0.5737667745104462),\n                        (13, 0.6193705712509703, 0.0, -0.6393642449091499)],\n                       [(2, 1), (5, 6), (10, 13), (12, 9)],\n                       [(1, 0.5, 0.0, -0.28964515054927675),\n                        (2, 0.5, 0.7947868992702162, 0.7052131007297839),\n                        (5, 4),\n                        (6, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (9, 0.5, 0.0, -0.4889648046389561),\n                        (12, 0.5960614607313279, -2.220446049250313e-16, -0.11684358694965402),\n                        (13, 0.6256105257587087, 0.0, 0.8439061231004792)],\n                       [(1, 0),\n                        (4, 0.0007451670218237004, 0.7288822930806633, 0.5449032193473877),\n                        (10, 13),\n                        (12, 9),\n                        (5, 0.999999967484822, -0.707159425934623, 0.9809448157059157)],\n                       [(0, 1.0, 0.0, 0.5),\n                        (1, 0.4999999999999998, 0.7896451505492773, 0.7103548494507228),\n                        (9, 0.5, 0.0, 0.011035195361044359),\n                        (10, 0.5, 0.0, 0.1535130966091755),\n                        (12, 0.14949404734505523, 0.0, -0.7269328881171262),\n                        (13, 0.9764747842833552, 0.0, 0.4416903651798709)],\n                       [(9, 10), (13, 12)],\n                       [(10, 0.5, 0.0, 0.1535130966091755), (12, 9), (13, 0.5, 0.0, 0.6846686483617455)],\n                       [(10, 13), (12, 0.4247761378557593, 0.0, -0.022381851578533896)],\n                       [(10, 0.5, 0.0, 0.220701950627813),\n                        (13, 0.14731871270290633, 0.0, -0.31533135163825454)],\n                       [(9, 10), (13, 12)],\n                       [(9, 0.4202007516434625, 0.0, -0.2824024462304582),\n                        (12, 0.45008695289946266, 0.0, -0.3365366861733614),\n                        (13, 0.9433458052208397, 0.0, 0.18466864836174457)],\n                       [(10, 13), (12, 9)],\n                       [(9, 0.5, 0.0, 0.12240166860881563),\n                        (10, 0.20567003520391755, 0.0, 0.7207019506278127),\n                        (12, 0.5, 0.0, -0.2896451505492772),\n                        (13, 0.5, 0.0, 0.9789986131578263)],\n                       [(10, 9), (13, 12)],\n                       [(9, 0.5000000000000018, 0.0, -0.7889384017990178),\n                        (10, 0.49999999999998246, 0.0, -0.7792979930542463),\n                        (12, 0.5000000000000022, 0.0, 0.6216949198585577),\n                        (13, 0.49999999999998246, 0.0, -0.02100133052423292)],\n                       [(10, 11), (13, 14)],\n                       [(10, 0.5000000000000102, 0.0, -0.7792979930542463),\n                        (11, 0.30408672398469627, 0.0, -0.5745192213521484),\n                        (13, 0.5000000000000102, 0.0, -0.02100133052423292),\n                        (14, 0.30408672398469627, 0.0, -0.5745192213521484)],\n                       [(10, 9), (13, 12)],\n                       [(9, 0.49999999999999956, 0.0, -0.7889383692838359),\n                        (10, 0.75, 0.0, 0.22070203946093242),\n                        (12, 0.49999999999999956, 0.0, 0.6216948873433754),\n                        (13, 0.75, -2.220446049250313e-16, -0.021001298009053926)],\n                       [(10, 11), (13, 14)],\n                       [(10, 0.500526889421938, 0.0, -0.2787710244119035),\n                        (11, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155),\n                        (13, 0.500526889421938, 0.0, -0.5204743618818899),\n                        (14, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155)],\n                       [(10, 9), (13, 12)],\n                       [(9, 0.5000000000000004, 0.0, -0.2889383692838352),\n                        (10, 0.5, 0.0, -0.528770588338471),\n                        (12, 0.5000000000000003, -2.220446049250313e-16, 0.12169488734337541),\n                        (13, 0.5000000000000002, 5.551115123125783e-17, 0.22952607419154275)],\n                       [(10, 9), (13, 12)],\n                       [(9, 0.9992548329781763, -0.9372761182881044, -0.788938369283845),\n                        (10, 11),\n                        (12, 0.0007451670218237004, -0.34790937491533347, -0.37830511265661615),\n                        (13, 14)],\n                       [(10, 0.999999967484822, 0.8025560554355362, -0.5287705656729824),\n                        (11, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (14, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (13, 0.999999967484822, -0.9557408070770008, -0.770473803151708)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],\n           \'qubits\': [(2, 5),\n                      (3, 5),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8),\n                      (6, 6),\n                      (6, 7),\n                      (6, 8)]}}\n\ndef _round7_unitary(n, k):\n    data = _ROUND7_VERIFIED.get((n, k))\n    if data is None:\n        return None\n    q = [cirq.GridQubit(*v) for v in data[\'qubits\']]\n    moments = []\n    for layer in data[\'moments\']:\n        ops = []\n        for gate in layer:\n            if len(gate) == 2:\n                ops.append(cirq.CZ(q[gate[0]], q[gate[1]]))\n            else:\n                i, x, z, a = gate\n                ops.append(cirq.PhasedXZGate(x_exponent=x, z_exponent=z,\n                                           axis_phase_exponent=a).on(q[i]))\n        moments.append(cirq.Moment(ops))\n    return cirq.Circuit(moments), {q[i] for i in data[\'outputs\']}\n# ROUND 7 VERIFIED NATIVE CIRCUITS END\n\n# ROUND 8 VERIFIED NATIVE CIRCUITS START\n_ROUND8_VERIFIED = {(10, 4): {\'method\': \'round 8 count kernels + parallel coherent decoders\',\n           \'moments\': [[(0, 0.576169350429717, 0.0, 0.5),\n                        (1, 0.24654597436413472, 0.0, -0.5),\n                        (2, 0.33300299859750804, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (7, 0.4999999999999998, 0.0, -0.5000000000000002),\n                        (5, 0.4424705780355341, 0.0, 0.1430563595801586),\n                        (8, 0.442470578035534, 0.0, 0.1430563595801586),\n                        (4, 0.5, 0.0, 0.5),\n                        (9, 0.3040867239846966, 0.0, -0.5745192213521528)],\n                       [(0, 1), (3, 2), (6, 7)],\n                       [(1, 0.9274578370544355, 0.0, 0.5),\n                        (6, 0.5335957289217554, 0.0, -0.5),\n                        (2, 0.8657722407926058, 0.0, 0.5),\n                        (7, 0.03359572892175544, 0.0, -0.5000000000000002),\n                        (0, 0.8952958399493551, 0.0, 0.5),\n                        (3, 0.4999999999999998, 0.0, 0.5)],\n                       [(1, 6), (2, 7)],\n                       [(1, 0.4999999999999998, 0.0, -0.5000000000000002),\n                        (2, 0.5, 0.0, 0.5),\n                        (6, 0.44472707612688994, 0.0, 0.5),\n                        (7, 0.9447270761268902, 0.0, -0.5000000000000002)],\n                       [(1, 2), (6, 7)],\n                       [(0, 1), (3, 2), (6, 0.4999999999999998, 0.0, 0.5), (7, 0.5, 0.0, -0.5)],\n                       [(1, 0.7239752345042922, 0.0, 0.5),\n                        (2, 0.5894561875110336, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (3, 0.5, 0.0, -0.5)],\n                       [(1, 6), (2, 7)],\n                       [(1, 0.8040867239846965, 1.1102230246251565e-16, 0.4999999999999999),\n                        (2, 0.4105438124889665, 0.0, -0.5),\n                        (7, 0.13386023640061523, 0.0, -0.5)],\n                       [(1, 2), (6, 7)],\n                       [(1, 0.5, 0.0, 0.5), (2, 0.5, 0.0, 0.5), (7, 0.5, 0.0, 0.5)],\n                       [(0, 1), (3, 2)],\n                       [(1, 6),\n                        (2, 0.5000000000000003, 0.0, 0.5),\n                        (0, 0.3005478057043407, 0.0, 0.0),\n                        (3, 0.30054780570434086, 0.0, 0.0)],\n                       [(2, 7),\n                        (1, 0.4589097765859911, 0.0, 0.0),\n                        (6, 0.7522115032289235, 0.0, 0.41781955317198194)],\n                       [(0, 1),\n                        (6, 5),\n                        (2, 0.5, 0.0, 0.9589097765859913),\n                        (7, 0.6067381136296596, 0.0, 0.4391987286223504)],\n                       [(3, 2),\n                        (7, 8),\n                        (6, 0.5000000000000003, 0.0, -0.0821804468280185),\n                        (5, 0.8362294663880598, 8.881784197001252e-16, -0.8569436404198416),\n                        (0, 0.5, 0.0, -0.5)],\n                       [(1, 6),\n                        (5, 0),\n                        (7, 0.5000000000000004, 0.0, -0.14018706062631114),\n                        (8, 0.8362294663880598, 0.0, 0.14305635958015883),\n                        (3, 0.5, 0.0, -0.5)],\n                       [(2, 7),\n                        (8, 3),\n                        (1, 0.5, 0.0, 0.2677470455309714),\n                        (6, 0.6556631301548916, 0.0, -0.13749732704777085),\n                        (5, 0.7582170456039754, 0.0, -0.35694364041984095)],\n                       [(0, 1),\n                        (6, 5),\n                        (2, 0.5, 0.0, 0.7266568221169618),\n                        (7, 0.6556631301548916, 0.0, -0.08487018040655814),\n                        (8, 0.7582170456039754, 0.0, 0.643056359580159)],\n                       [(3, 2),\n                        (7, 8),\n                        (1, 0.5, 0.0, 0.26774704553097184),\n                        (6, 0.39779393920402867, 0.0, -0.4466225445456371),\n                        (5, 0.9399029518185915, 1.6653345369377348e-16, 0.14305635958015867),\n                        (0, 0.06063473201948598, 0.0, -0.3331256350418408)],\n                       [(1, 6),\n                        (5, 0),\n                        (2, 0.5, 0.0, 0.7266568221169618),\n                        (7, 0.39779393920402856, 0.0, 0.224255037091308),\n                        (8, 0.9399029518185915, 1.6653345369377348e-16, 0.14305635958015867),\n                        (3, 0.060634732019485986, 0.0, -0.33312563504184056)],\n                       [(2, 7),\n                        (8, 3),\n                        (0, 0.2568381783892426, 0.0, 0.23480140577808023),\n                        (6, 0.32772942681811573, 0.0, -0.0037426714554436025),\n                        (5, 0.5527003573655196, 0.0, -0.35694364041984183)],\n                       [(0, 1),\n                        (6, 5),\n                        (3, 0.2568381783892426, 0.0, 0.23480140577808023),\n                        (7, 0.32772942681811573, 0.0, -0.21862483599888538),\n                        (8, 0.5527003573655196, -2.220446049250313e-16, -0.3569436404198416)],\n                       [(3, 2),\n                        (7, 8),\n                        (6, 0.5673580858862435, 0.0, -0.8860268812526746),\n                        (5, 0.5000000000000003, 0.0, 0.6430563595801579),\n                        (0, 0.3360750009341995, 1.1102230246251565e-16, -0.16976234933941736)],\n                       [(1, 6),\n                        (5, 0),\n                        (7, 0.5673580858862433, 0.0, 0.6636593737983449),\n                        (8, 0.5000000000000004, 0.0, -0.35694364041984183),\n                        (3, 0.33607500093419895, 0.0, -0.16976234933941692)],\n                       [(2, 7),\n                        (8, 3),\n                        (6, 0.6009181180709966, 0.0, -0.3416488858290587),\n                        (5, 0.5896983452724038, 0.0, -0.017519659528558718),\n                        (0, 0.5, 0.0, -0.9213632702772465)],\n                       [(1, 6),\n                        (5, 0),\n                        (7, 0.6009181180709966, 0.0, 0.11928137837472974),\n                        (8, 0.5896983452724036, 0.0, 0.9824803404714411),\n                        (3, 0.5, 0.0, -0.9213632702772458)],\n                       [(2, 7),\n                        (8, 3),\n                        (0, 0.5000000000000002, 0.0, -0.42136327027724607),\n                        (1, 0.5, 0.0, -0.053345871880375206),\n                        (6, 0.9680586457321536, 0.0, 0.04628457890408644),\n                        (5, 0.16802345081794726, 0.0, -0.6221814539667061)],\n                       [(0, 1),\n                        (6, 5),\n                        (3, 0.5000000000000002, 0.0, -0.42136327027724607),\n                        (2, 0.5, 0.0, -0.5944360952943846),\n                        (7, 0.9680586457321536, 3.3306690738754696e-16, -0.268652086358415),\n                        (8, 0.16802345081794726, 0.0, 0.3778185460332937)],\n                       [(3, 2),\n                        (7, 8),\n                        (1, 0.5, 0.0, -0.053345871880374096),\n                        (6, 0.5, 1.1102230246251565e-16, -0.18099802984659563),\n                        (5, 0)],\n                       [(1, 6),\n                        (2, 0.5, 0.0, -0.5944360952943846),\n                        (7, 0.5, 0.0, -0.04136947760773291),\n                        (8, 3),\n                        (5, 0.4054732409466401, 0.0, -0.9211192286933714)],\n                       [(2, 7),\n                        (1, 0.5, 4.440892098500626e-16, -0.9990550652733587),\n                        (6, 0.17213894013095277, 0.0, 0.8190019701534046),\n                        (8, 0.4054732409466402, 0.0, -0.9211192286933716)],\n                       [(0, 1),\n                        (6, 5),\n                        (2, 0.5, 0.0, -0.5401452886873681),\n                        (7, 0.17213894013095277, 0.0, 0.9586305223922666)],\n                       [(3, 2),\n                        (7, 8),\n                        (6, 0.9788981661755423, 1.1102230246251565e-16, 0.3190019701534045),\n                        (5, 0.5343543584033268, 0.0, -0.21858226359651556),\n                        (0, 0.4475286319367211, 0.0, -0.6102077293818242)],\n                       [(1, 6),\n                        (5, 0),\n                        (7, 0.9788981661755423, 0.0, -0.5413694776077322),\n                        (8, 0.5343543584033267, 0.0, 0.7814177364034842),\n                        (3, 0.4475286319367211, 0.0, -0.6102077293818242)],\n                       [(2, 7),\n                        (8, 3),\n                        (0, 0.5, 0.0, -0.2253094187653628),\n                        (1, 0.5205451117070048, 0.0, -0.4990550652733581),\n                        (5, 0.5410902234140091, 0.0, 0.33024257086751563)],\n                       [(0, 1),\n                        (5, 4),\n                        (3, 0.5, 0.0, -0.2253094187653628),\n                        (2, 0.5205451117070048, 0.0, 0.9598547113126312),\n                        (7, 0.5, 0.0, 0.25296114652731494),\n                        (8, 0.4999999999999998, 0.0, 0.830242570867515)],\n                       [(3, 2),\n                        (7, 8),\n                        (0, 0.5, 0.0, 0.31578080464864655),\n                        (5, 0.5, 0.0, -0.16975742913248504),\n                        (4, 0.3040867239846962, 0.0, 0.4999999999999998)],\n                       [(1, 0),\n                        (6, 5),\n                        (3, 0.5, 0.0, 0.31578080464864655),\n                        (7, 0.49999999999998224, 0.0, -0.7470387971547443),\n                        (8, 0.5000000000000022, 0.0, -0.2584186774025792)],\n                       [(2, 3),\n                        (7, 9),\n                        (1, 0.25000000000000006, 4.440892098500626e-16, -0.9990550652733587),\n                        (0, 0.25, 0.0, -0.6842191953513537),\n                        (5, 0.8040867239846965, 1.1102230246251565e-16, -0.16975742913248526)],\n                       [(1, 0),\n                        (5, 4),\n                        (2, 0.25000000000000006, 5.551115123125783e-17, 0.45985471131263156),\n                        (3, 0.25, 0.0, -0.6842191953513537),\n                        (7, 0.5000000000000102, 0.0, -0.7470387971547441),\n                        (9, 0.30408672398469627, 0.0, -0.5745192213521484)],\n                       [(2, 3),\n                        (7, 8),\n                        (5, 0.7499999999999999, 1.1102230246251565e-16, -0.16975742913248526),\n                        (4, 0.5, 0.0, 0.5),\n                        (0, 0.5, -0.8157808046486466, -0.6842191953513534),\n                        (1, 0.5, 0.49905506527335836, 0.0009449347266416441)],\n                       [(6, 5),\n                        (7, 0.75, 0.0, -0.7470387646395655),\n                        (8, 0.49999999999999967, 0.0, -0.25841870991776084),\n                        (2, 0.5, -0.9598547113126319, -0.5401452886873686),\n                        (3, 0.5, -0.8157808046486466, -0.6842191953513534)],\n                       [(7, 9),\n                        (5, 0.5, 0.0, 0.830242570867515),\n                        (6, 0.5000000000000001, 0.0, -0.20154314155360176)],\n                       [(5, 4),\n                        (7, 0.500526889421938, 4.440892098500626e-16, 0.7534881714875985),\n                        (9, 0.5000000000000102, 0.0, 0.4254807786478514)],\n                       [(7, 8), (5, 0.25000000000000006, 0.0, -0.16975742913248526)],\n                       [(6, 5),\n                        (7, 0.5, 0.0, -0.49651139243896836),\n                        (8, 0.5000000000000004, 0.0, -0.7584187099177604)],\n                       [(7, 9),\n                        (5, 0.5, 0.0, 0.830242570867515),\n                        (6, 0.5, -0.2984568584463976, 0.7984568584463976)],\n                       [(5, 4), (7, 8), (9, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(4, 1.0, 0.0, 0.5),\n                        (5, 0.4999999999999998, -0.33024257086751474, -0.16975742913248526),\n                        (7, 0.999999967484822, -0.22970314046396734, 0.5034886302265205),\n                        (8, 0.0007451670218237006, 0.5322042223458032, 0.7415812900822472)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],\n           \'qubits\': [(4, 4), (4, 5), (4, 6), (4, 7), (5, 3), (5, 4), (5, 5), (5, 6), (5, 7), (6, 6)]},\n (11, 4): {\'method\': \'round 8 count kernels + parallel coherent decoders\',\n           \'moments\': [[(1, 0.6804183583624992, 0.0, 0.5),\n                        (2, 0.2715149105420544, 0.0, -0.5000000000000002),\n                        (3, 0.3910132305016676, 0.0, 0.5),\n                        (7, 0.5, 0.0, 0.5),\n                        (8, 0.4999999999999998, 0.0, -0.5000000000000002),\n                        (6, 0.442470578035534, 1.0, 0.1430563595801585),\n                        (9, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (5, 0.5, 1.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (10, 0.5, 1.0, 0.5)],\n                       [(1, 2), (4, 3), (7, 8)],\n                       [(2, 0.8245125290280035, 0.0, 0.5),\n                        (7, 0.3399162669016432, 0.0, -0.5000000000000002),\n                        (3, 0.7367750224499274, 0.0, 0.5),\n                        (8, 0.16008373309835672, 0.0, 0.5),\n                        (1, 0.7552309029870035, 1.0, 0.5),\n                        (4, 0.5, 1.0, 0.5)],\n                       [(2, 7), (3, 8)],\n                       [(2, 0.5, 0.0, -0.5),\n                        (3, 0.5, 0.0, 0.5),\n                        (7, 0.45750956125153186, 0.0, 0.5),\n                        (8, 0.9575095612515319, 0.0, -0.5000000000000002)],\n                       [(2, 3), (7, 8)],\n                       [(1, 2), (4, 3), (7, 0.5, 1.0, 0.5), (8, 0.5, 1.0, -0.5)],\n                       [(2, 0.7356809418135472, 0.0, 0.5),\n                        (3, 0.6015434872071732, 0.0, 0.5),\n                        (1, 0.5, 1.0, -0.5),\n                        (4, 0.5, 1.0, 0.5)],\n                       [(2, 7), (3, 8)],\n                       [(2, 0.7820471084244873, 0.0, 0.5),\n                        (3, 0.3984565127928268, 0.0, -0.5),\n                        (8, 0.17490085614052142, 1.0, 0.5)],\n                       [(2, 3), (7, 8)],\n                       [(2, 0.5, 0.0, 0.5), (3, 0.5, 0.0, 0.5), (8, 0.5, 1.0, 0.5)],\n                       [(1, 2), (4, 3)],\n                       [(2, 7),\n                        (3, 0.4999999999999998, 0.0, 0.5),\n                        (1, 0.3005478057043407, 1.0, 0.0),\n                        (4, 0.3005269769394423, 1.0, 0.0)],\n                       [(3, 8),\n                        (2, 0.4589097765859911, 0.0, 0.0),\n                        (7, 0.7522115032289235, 0.0, -0.5821804468280181)],\n                       [(1, 2),\n                        (7, 6),\n                        (8, 0.5412384261162806, 0.0, -0.5528730953753256),\n                        (3, 0.5, 0.0, 0.958911095263919)],\n                       [(8, 9),\n                        (4, 3),\n                        (6, 0.8362294663880598, 0.0, 0.14305635958015883),\n                        (1, 0.5, 0.0, 0.49999999999999956),\n                        (2, 0.0, 0.40247412455547926, 0.0),\n                        (7, 0.5000000000000003, 0.0, 0.9178195531719815)],\n                       [(6, 1),\n                        (2, 7),\n                        (3, 0.0, -0.4629428026930573, 0.0),\n                        (8, 0.5, 0.0, 0.8766614952223488),\n                        (9, 0.85928593962011, 0.0, -0.9262332254895425),\n                        (4, 0.5, 0.0, 0.49999999999999956)],\n                       [(3, 8),\n                        (9, 4),\n                        (2, 0.5, 0.9999999999999998, 0.6702211700864511),\n                        (7, 0.6556631301548916, 0.0, 0.8625026729522292),\n                        (6, 0.7582170456039754, 0.9999999999999998, 0.643056359580159)],\n                       [(1, 2),\n                        (7, 6),\n                        (8, 0.6714165234362537, 0.0, 0.948077586953342),\n                        (9, 0.7485956033393342, -4.440892098500626e-16, -0.42623322548954246),\n                        (3, 0.5, -0.9999999999999996, 0.29431960644791355)],\n                       [(8, 9),\n                        (4, 3),\n                        (2, 0.5, 1.0, -0.3297788299135487),\n                        (7, 0.39779393920402867, 0.0, 0.5533774554543629),\n                        (6, 0.9399029518185915, 1.0, 0.14305635958015878),\n                        (1, 0.060634732019485986, 0.0, 0.666874364958159)],\n                       [(2, 7),\n                        (6, 1),\n                        (3, 0.5, 0.0, -0.7056803935520863),\n                        (8, 0.3935294720438234, 0.0, -0.7728752974330366),\n                        (9, 0.9961572718865779, 1.0, -0.9262332254895478),\n                        (4, 0.0528060550044127, 0.0, 0.6736003437869695)],\n                       [(3, 8),\n                        (9, 4),\n                        (1, 0.2568381783892426, 0.0, -0.7651985942219197),\n                        (7, 0.3277294268181158, 0.0, 0.9962573285445564),\n                        (6, 0.5527003573655196, 0.9999999999999996, 0.6430563595801584)],\n                       [(1, 2),\n                        (7, 6),\n                        (4, 0.24963665794604517, 0.0, -0.8090370713032553),\n                        (8, 0.3104369525321424, 0.0, 0.8086034521667642),\n                        (9, 0.5655532211464794, 0.0, -0.4262332254895531)],\n                       [(4, 3),\n                        (8, 9),\n                        (7, 0.5673580858862435, 5.551115123125783e-17, 0.1139731187473253),\n                        (6, 0.5000000000000003, 0.9999999999999998, 0.6430563595801582),\n                        (1, 0.33607500093419945, 0.0, 0.8302376506605831)],\n                       [(2, 7),\n                        (6, 1),\n                        (8, 0.6193705712509703, 1.0, -0.31843085873264076),\n                        (9, 0.4999999999999998, 1.0, -0.4262332254895531),\n                        (4, 0.32701156232958106, 0.0, 0.7987432874086426)],\n                       [(3, 8),\n                        (9, 4),\n                        (2, 0.0, 0.325045498277318, 0.0),\n                        (7, 0.6009181180709966, 0.0, 0.6583511141709413),\n                        (6, 0.5896983452724036, 1.0, 0.9824803404714411),\n                        (1, 0.5, -2.220446049250313e-16, 0.0786367297227537)],\n                       [(2, 7),\n                        (6, 1),\n                        (3, 0.0, 0.9739798652145693, 0.0),\n                        (8, 0.6256105257587086, 5.551115123125783e-17, 0.16483950927698848),\n                        (9, 0.5960614607313279, 1.0, -0.11684358694965402),\n                        (4, 0.5, 0.0, 0.04610781888751925)],\n                       [(3, 8),\n                        (9, 4),\n                        (1, 0.5000000000000001, 4.440892098500626e-16, 0.5786367297227535),\n                        (2, 0.5, 0.9999999999999999, 0.6741737509524224),\n                        (7, 0.9680586457321536, -6.661338147750939e-16, -0.9537154210959138),\n                        (6, 0.16802345081794726, 1.0, -0.6221814539667061)],\n                       [(1, 2),\n                        (7, 6),\n                        (8, 0.9764747842833552, 3.3306690738754696e-16, -0.23737624864361972),\n                        (9, 0.14949404734505523, 0.0, 0.2730671118828738),\n                        (4, 0.5, 4.440892098500626e-16, 0.5461078188875186),\n                        (3, 0.5, 0.9999999999999999, 0.9472280423550001)],\n                       [(8, 9),\n                        (4, 3),\n                        (2, 0.5, 1.0, -0.3258262490475774),\n                        (7, 0.5, 0.0, 0.8190019701534039),\n                        (1, 0.0, -0.7558274614571485, 0.0)],\n                       [(2, 7),\n                        (6, 1),\n                        (3, 0.5, 0.0, -0.05277195764500009),\n                        (8, 0.5, 0.0, 0.005602034538254308),\n                        (4, 0.0, 0.08503202047153624, 0.0)],\n                       [(3, 8),\n                        (9, 4),\n                        (2, 0.5, 0.0, -0.2715354424405607),\n                        (7, 0.17213894013095277, 0.0, -0.18099802984659608),\n                        (6, 0.4054732409466402, 0.0, 0.07888077130662807)],\n                       [(1, 2),\n                        (7, 6),\n                        (3, 0.5, 0.0, 0.014416896373636745),\n                        (8, 0.1473187127029063, 1.0, -0.9943979654617454),\n                        (9, 0.4247761378557593, 0.0, 0.9776181484214663)],\n                       [(4, 3),\n                        (8, 9),\n                        (2, 2.2204460492503128e-16, 1.0, -0.8810105324381954),\n                        (7, 0.9788981661755423, 4.440892098500626e-16, -0.6809980298465961),\n                        (6, 0.5343543584033267, 0.0, 0.7814177364034842),\n                        (1, 0.44752863193672116, 1.0, -0.3660351908389725)],\n                       [(2, 7),\n                        (6, 1),\n                        (8, 0.9433458052208397, 0.0, 0.5056020345382535),\n                        (9, 0.45008695289946266, 0.0, 0.6634633138266384),\n                        (4, 0.42020075164346243, -0.9999999999999998, 0.3377021977675534)],\n                       [(3, 8),\n                        (9, 4),\n                        (1, 0.5, 0.0, -0.9811368802225111),\n                        (2, 0.5205451117070048, 0.0, -0.7715354424405612),\n                        (6, 0.5410902234140091, 0.0, -0.669757429132485)],\n                       [(1, 2),\n                        (6, 5),\n                        (4, 0.5410889047360808, 0.0, -0.7574936873931726),\n                        (9, 0.541088904736081, 0.0, -0.7896451505492768),\n                        (3, 0.5000000000000002, 0.0, 0.01441689637363619),\n                        (8, 0.0, 1.0, 0.0)],\n                       [(4, 0),\n                        (9, 10),\n                        (1, 0.5, 0.0, -0.4400466568085022),\n                        (6, 0.5, 0.0, 0.830242570867515),\n                        (5, 0.3040867239846962, 1.0, -0.5000000000000002)],\n                       [(2, 1),\n                        (7, 6),\n                        (4, 0.5000000000000001, 2.220446049250313e-16, -0.25749368739317235),\n                        (9, 0.5, 0.0, 0.710354849450723),\n                        (0, 0.30408672398469616, 0.0, -0.5000000000000002),\n                        (10, 0.3040867239846962, 1.0, -0.5000000000000002)],\n                       [(3, 4),\n                        (8, 9),\n                        (2, 0.25000000000000006, 0.0, 0.7284645575594388),\n                        (1, 0.25000000000000006, 0.0, 0.559953343191498),\n                        (6, 0.8040867239846965, 0.0, 0.8302425708675145),\n                        (7, 0.0, 0.40308628310720346, 0.0)],\n                       [(2, 1),\n                        (6, 5),\n                        (4, 0.8040867239846965, 4.440892098500626e-16, 0.7425063126068276),\n                        (9, 0.8040867239846965, 0.0, -0.28964515054927653)],\n                       [(4, 0),\n                        (9, 10),\n                        (6, 0.7499999999999999, 0.0, 0.8302425708675147),\n                        (5, 0.5, 1.0, 0.5),\n                        (1, 0.5, -0.05995334319149803, 0.559953343191498),\n                        (2, 0.5, 0.7715354424405612, -0.27153544244056116)],\n                       [(7, 6),\n                        (9, 0.7499999999999999, 1.1102230246251565e-16, -0.28964515054927675),\n                        (4, 0.7499999999999999, 0.9999999999999999, 0.7425063126068279),\n                        (0, 0.5, 0.0, 0.5),\n                        (10, 0.5, 1.0, 0.5)],\n                       [(8, 9),\n                        (3, 4),\n                        (6, 0.5, 0.0, -0.16975742913248504),\n                        (7, 0.5, -0.9999999999999996, -0.7984568584463985)],\n                       [(6, 5),\n                        (4, 0.5000000000000002, 0.0, -0.25749368739317235),\n                        (9, 0.5, 0.0, -0.28964515054927653),\n                        (8, 0.5, 0.0, 0.02614648690629323),\n                        (3, 0.5, 0.0, 0.5349613487416773)],\n                       [(4, 0), (9, 10), (6, 0.25000000000000006, 0.0, 0.8302425708675147)],\n                       [(7, 6),\n                        (9, 0.25000000000000006, 0.0, 0.7103548494507228),\n                        (4, 0.25000000000000006, 0.9999999999999998, 0.7425063126068276)],\n                       [(8, 9),\n                        (3, 4),\n                        (6, 0.5, 3.3306690738754696e-16, -0.16975742913248504),\n                        (7, 0.5, -0.7015431415536015, -0.7984568584463985)],\n                       [(6, 5),\n                        (9, 0.5, 0.0, 0.710354849450723),\n                        (4, 0.5, -0.9999999999999996, -0.25749368739317235),\n                        (3, 0.5, -0.034961348741677334, -0.46503865125832267),\n                        (8, 0.5, -0.5261464869062928, 0.026146486906293176)],\n                       [(9, 10),\n                        (4, 0),\n                        (5, 1.0, 0.0, 0.0),\n                        (6, 0.4999999999999998, 0.6697574291324857, 0.8302425708675145)],\n                       [(10, 1.0, 0.0, 0.0),\n                        (0, 1.0, 0.0, 0.5),\n                        (4, 0.5, -0.24250631260682765, 0.7425063126068276),\n                        (9, 0.4999999999999998, -0.21035484945072325, -0.28964515054927675)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],\n           \'qubits\': [(3, 7),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 3),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8)]},\n (11, 5): {\'method\': \'round 8 count kernels + parallel coherent decoders\',\n           \'moments\': [[(1, 0.49070188094737827, 0.0, 0.5),\n                        (2, 0.5550422610798844, 0.0, -0.5),\n                        (4, 0.5008134412882038, 0.0, -0.5),\n                        (3, 0.4930157801733349, 0.0, -0.5),\n                        (7, 1.0, 0.0, 0.5),\n                        (8, 0.7705976301927333, 0.0, -0.5),\n                        (6, 0.5199696142890611, 0.966374362234198, -0.8896751604098314),\n                        (5, 0.5, 0.0, 0.5),\n                        (9, 0.4127792040228902, 0.905374889856748, -0.9262332254895422),\n                        (10, 0.5, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5)],\n                       [(1, 2), (4, 3), (7, 8)],\n                       [(2, 0.9472614134659207, 0.0, 0.5),\n                        (7, 0.42521108825630477, 0.0, -0.5),\n                        (3, 0.3741917205576136, 5.551115123125783e-17, 0.49999999999999994),\n                        (8, 0.2294023698072668, 0.0, 0.5),\n                        (1, 0.9464554303254016, 0.0, 0.5),\n                        (4, 0.39921335209975817, 0.0, -0.5)],\n                       [(2, 7), (3, 8)],\n                       [(2, 0.5, 0.0, -0.5),\n                        (3, 0.4749542747756747, 0.0, 0.5),\n                        (7, 0.5801769884442538, 0.0, 0.5),\n                        (8, 1.0, 0.0, 0.5)],\n                       [(2, 3), (7, 8)],\n                       [(2, 0.49999999999999983, 0.0, -0.5),\n                        (3, 1.0, 0.0, 0.5),\n                        (7, 0.9422205038637325, 0.0, -0.5000000000000002),\n                        (8, 1.0, 0.0, 0.5)],\n                       [(1, 2), (4, 3)],\n                       [(2, 0.7421439232860106, 0.0, 0.5),\n                        (3, 0.29667120537234515, 0.0, 0.5),\n                        (1, 0.5000000000000001, 0.0, -0.5),\n                        (4, 0.0014014702419442263, 0.0, 0.5)],\n                       [(2, 7), (3, 8)],\n                       [(2, 0.3558842447906469, 0.0, -0.5),\n                        (3, 0.7966712053723455, 0.0, 0.5),\n                        (7, 0.5192241398420848, 0.0, -0.5),\n                        (8, 0.5, 0.0, 0.5)],\n                       [(2, 3), (7, 8)],\n                       [(2, 0.5000000000000001, 0.0, 0.5),\n                        (3, 0.5, 0.0, 0.5),\n                        (7, 0.5, 0.0, -0.5),\n                        (8, 0.5000000000000001, 0.0, 0.5)],\n                       [(1, 2), (4, 3)],\n                       [(2, 0.5000000000000001, 0.0, 0.5),\n                        (3, 8),\n                        (1, 0.5, -0.9760163502696884, 0.1994701104915741),\n                        (4, 0.5000000000000002, 0.32700942960715884, -0.1994730230605577)],\n                       [(2, 7),\n                        (3, 0.4999999999999998, 0.5282691788511857, 0.958911095263919),\n                        (8, 0.2219727068374303, -0.49154599147959166, -0.5821778094721621)],\n                       [(2, 0.5, 0.3591968409075692, 0.9589035381440989),\n                        (7, 0.6208336879845766, -0.4649471176858915, -0.5336029193841096),\n                        (4, 3),\n                        (8, 9)],\n                       [(1, 2),\n                        (7, 6),\n                        (3, 1.0, 0.0, 0.21461996803249472),\n                        (8, 0.5, 0.7022844924594797, 0.4262761990482463),\n                        (9, 0.85928593962011, -0.4875608597182275, -0.02085833563279449),\n                        (4, 0.5000000000000001, 0.12925995895837872, -0.8724635934533991)],\n                       [(2, 1.0, 0.0, 0.15136907821349643),\n                        (7, 0.5000000000000001, 0.6584749008440663, 0.4127955417649979),\n                        (6, 0.8325870184220645, -0.5200591664882959, 0.0766992018243659),\n                        (1, 0.5, -0.030723330248449088, -0.7765462397781144),\n                        (3, 8),\n                        (9, 4)],\n                       [(2, 7),\n                        (6, 1),\n                        (4, 1.0, 0.0, -0.28654568860164664),\n                        (3, 0.5000000000000001, -0.07216288375695012, -0.25958902417306295),\n                        (8, 0.6714165234362537, 0.7430251514267326, -0.8000232167612813),\n                        (9, 0.7485956033393342, -0.10058310030841655, -0.008419195351022202)],\n                       [(1, 1.0, 0.0, -0.35237069449328406),\n                        (2, 0.5000000000000001, -0.08183504399310726, -0.25375757867618387),\n                        (7, 0.633295750078832, 0.8024623256494829, -0.7457603550629945),\n                        (6, 0.747846310784871, -0.17134479144473458, 0.05664003533606926),\n                        (4, 3),\n                        (8, 9)],\n                       [(1, 2),\n                        (7, 6),\n                        (3, 0.5000000000000001, -0.47921561052476913, 0.6682480920699869),\n                        (8, 0.39352947204382344, -0.0400166187933757, 0.22204905027907273),\n                        (9, 0.996157271886578, -0.38731562230521455, -0.6090022956594439),\n                        (4, 0.05280605500441269, -0.7140339826902271, -0.0034880864952426816)],\n                       [(2, 0.5, -0.23681199740208614, 0.6644073773307086),\n                        (7, 0.24200275221440573, 0.12003146804240211, 0.21896207919510252),\n                        (6, 0.9580541200199847, -0.5877475103440539, -0.6147047561086649),\n                        (1, 0.036112653164580656, -0.8149062791244521, -0.06549642494681374),\n                        (3, 8),\n                        (9, 4)],\n                       [(2, 7),\n                        (6, 1),\n                        (4, 0.24963665794604506, 0.8669932073346845, 0.7651153459047553),\n                        (3, 1.0, 0.0, 0.21763245964208),\n                        (8, 0.3104369525321425, 0.8085371278988395, -0.23648881891450202),\n                        (9, 0.5655532211464792, -0.7603717865514881, -0.49631791796466374)],\n                       [(1, 0.29557549942386757, -0.8771709074392591, 0.5336267181200474),\n                        (2, 1.0, 0.0, 0.4362658366327108),\n                        (7, 0.26083406388318797, 0.6177144938526578, 0.11150272847709536),\n                        (6, 0.49019895102894673, -0.8569874916299094, -0.7024522664527184),\n                        (4, 3),\n                        (8, 9)],\n                       [(1, 2),\n                        (7, 6),\n                        (3, 1.3678757607663618e-17, 0.19456334695121486, 0.24274381274574858),\n                        (8, 0.6193705712509703, -0.4502913348689348, -0.5549860019150674),\n                        (9, 0.49999999999999994, 0.914005305656897, 0.7433102954838477),\n                        (4, 0.32701156232958095, 0.6955438578648168, -0.9756718054724585)],\n                       [(2, 2.2593636363592025e-17, -0.4110229675034638, -0.5585523672452701),\n                        (7, 0.4922598869063585, -0.44814055334937275, -0.4444150370829989),\n                        (6, 0.49999999999999994, 0.9967146647382108, 0.44056024191737214),\n                        (1, 0.4313085072649695, 0.7499183762094207, -0.8578728931258981),\n                        (3, 8),\n                        (9, 4)],\n                       [(2, 7),\n                        (6, 1),\n                        (3, 1.0, 0.0, 0.3991857392005793),\n                        (8, 0.6256105257587085, 0.9452263271866326, -0.5220069687743731),\n                        (9, 0.5960614607313279, 0.2573629292943688, -0.033294760319355854),\n                        (4, 0.5000000000000001, 0.3922892154072385, 0.472507520913482)],\n                       [(2, 1.0, 0.0, 0.22752939310706877),\n                        (7, 0.6755935676538105, -0.9282460574587303, -0.5056399138381202),\n                        (6, 0.6338347537865279, 0.3223583002431071, -0.09800723979530845),\n                        (1, 0.5000000000000001, 0.40813944694967486, 0.6174374285377566),\n                        (3, 8),\n                        (9, 4)],\n                       [(2, 7),\n                        (6, 1),\n                        (4, 0.5, -0.2739726829496021, 0.3647967363207205),\n                        (3, 0.5000000000000001, -0.2267887021096202, 0.036504264403519004),\n                        (8, 0.9764747842833552, 0.005682454151557326, 0.021003600491651175),\n                        (9, 0.14949404734505523, 0.6915804042354776, 0.6139788678075409)],\n                       [(1, 0.4999999999999998, -0.3029539307227471, 0.5255768754874317),\n                        (2, 0.5, -0.06987507273615945, 0.10006733994442629),\n                        (7, 0.8391622995173471, -0.4666232621508506, 0.12265856248455714),\n                        (6, 0.10478091261206611, 0.6069675174235609, 0.5600579969436181),\n                        (4, 3),\n                        (8, 9)],\n                       [(1, 2),\n                        (7, 6),\n                        (3, 0.5, -0.9528264593460201, 0.8097155622938987),\n                        (8, 0.5, 0.47939827321355255, 0.2696643378250829),\n                        (9, 1.0, 0.0, -0.31991326341917814),\n                        (4, 1.0, 0.0, -0.27721006339896803)],\n                       [(2, 0.5, 0.44888461582760275, -0.9698077327917332),\n                        (7, 0.5000000000000001, -0.8342441162534189, -0.0848749207251096),\n                        (6, 1.0, 0.0, -0.30598384290578995),\n                        (1, 1.0, 0.0, -0.13870845888133387),\n                        (3, 8),\n                        (9, 4)],\n                       [(2, 7),\n                        (6, 1),\n                        (4, 2.6911488363979655e-17, -0.10657650605920965, 0.3199678821356995),\n                        (3, 0.5000000000000001, 0.6065964482612792, -0.07592204303348415),\n                        (8, 0.14731871270290642, 0.47763908808934363, -0.2509373889613644),\n                        (9, 0.42477613785575913, 0.24565143911589105, -0.6499368354199673)],\n                       [(1, 5.1730648837162365e-18, -0.615009442582864, 0.5150410050501057),\n                        (2, 0.5, 0.9359542790463431, -0.4406009729548801),\n                        (7, 0.1802207481101961, 0.3860031568845424, 0.0808809630214713),\n                        (6, 0.411561103136257, -0.027640655655161606, -0.5340368657507382),\n                        (4, 3),\n                        (8, 9)],\n                       [(1, 2),\n                        (7, 6),\n                        (3, 2.183989817369057e-17, 0.04089633285515237, 0.7520884815410067),\n                        (8, 0.9433458052208397, -0.437164462020319, -0.27329830087202156),\n                        (9, 0.4500869528994625, 0.5858004296037851, -0.09013056170924827),\n                        (4, 0.42020075164346243, -0.13244587453752787, -0.04525832781976602)],\n                       [(2, 1.946846003666787e-17, -0.23527072633657475, 0.8386739670881893),\n                        (7, 0.9611848154635414, 0.5482778688470907, 0.9668841199060139),\n                        (6, 0.4752070857908569, -0.8326590813546366, -0.2511029195814165),\n                        (1, 0.4294376327554545, -0.5263063967663268, -0.3336943135481585),\n                        (3, 8),\n                        (9, 4)],\n                       [(2, 7),\n                        (6, 1),\n                        (9, 0.5410889047360808, -0.9487783322704517, -0.05122166772954828),\n                        (8, 5.822447387112414e-17, 0.6899183105243011, -0.1202404731065354),\n                        (4, 0.5410889047360808, 0.2729000875180194, 0.7270999124819801),\n                        (3, 0.5000000000000002, -0.5921151904509876, 0.5715707380829471)],\n                       [(6, 0.5410964618559009, 0.6051656233889696, 0.3948343766110304),\n                        (7, 9.304731065784778e-17, -0.5357102196810557, 0.9333451783572855),\n                        (1, 0.5, 0.4456173037517619, 0.5132862343923374),\n                        (2, 0.5205482309279504, -0.7600825797548882, -0.23991742024511176),\n                        (9, 10),\n                        (4, 0)],\n                       [(6, 5),\n                        (1, 2),\n                        (9, 0.49999999999999983, 0.0, -0.5),\n                        (10, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (4, 0.49999999999999983, 0.0, -0.5),\n                        (0, 0.3040867239846963, 0.0, -0.5000000000000002)],\n                       [(6, 0.49999999999999983, 0.0, -0.5),\n                        (5, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (1, 0.5, 1.0, -0.5),\n                        (8, 9),\n                        (3, 4)],\n                       [(7, 6),\n                        (2, 1),\n                        (9, 0.8040867239846964, 0.0, 0.5),\n                        (8, 1.0, 0.0, 0.5),\n                        (4, 0.8040867239846964, 0.0, 0.5),\n                        (3, 1.0, 0.0, 0.5)],\n                       [(6, 0.8040867239846964, 0.0, 0.5),\n                        (7, 1.0, 0.0, 0.5),\n                        (2, 0.25000000000000006, 0.0, 0.5),\n                        (1, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (9, 10),\n                        (4, 0)],\n                       [(6, 5),\n                        (2, 1),\n                        (9, 0.75, 0.0, 0.5),\n                        (10, 0.5, 0.0, 0.5),\n                        (4, 0.75, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5)],\n                       [(6, 0.75, 0.0, 0.5),\n                        (5, 0.5, 0.0, 0.5),\n                        (8, 9),\n                        (3, 4),\n                        (2, 0.5, 1.0, -0.5),\n                        (1, 0.5, 1.0, -0.5)],\n                       [(7, 6),\n                        (9, 0.5000000000000001, 0.0, -0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (4, 0.5000000000000001, 0.0, -0.5),\n                        (3, 0.5, 0.0, 0.5)],\n                       [(6, 0.5000000000000001, 0.0, -0.5), (7, 0.5, 0.0, 0.5), (9, 10), (4, 0)],\n                       [(6, 5),\n                        (9, 0.25000000000000006, 0.0, 0.5),\n                        (10, 1.0, 0.0, 0.5),\n                        (4, 0.25000000000000006, 0.0, 0.5),\n                        (0, 1.0, 0.0, 0.5)],\n                       [(6, 0.25000000000000006, 0.0, 0.5), (5, 1.0, 0.0, 0.5), (8, 9), (3, 4)],\n                       [(7, 6),\n                        (9, 0.5, 0.0, -0.5),\n                        (4, 0.5, 0.0, -0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (3, 0.5, 0.0, 0.5)],\n                       [(6, 0.5, 0.0, -0.5), (9, 10), (4, 0), (7, 0.5, 0.0, 0.5)],\n                       [(6, 5), (9, 0.5000000000000001, 0.0, 0.5), (4, 0.5000000000000001, 0.0, 0.5)],\n                       [(6, 0.5000000000000001, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],\n           \'qubits\': [(3, 7),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 3),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8)]},\n (13, 4): {\'method\': \'joint count seed and first-bit prefix + parallel decoders\',\n           \'moments\': [[(4, 0.4924853913005282, 0.0, 0.5033305426159476),\n                        (6, 0.846215799391159, 0.0, -0.585965634310784),\n                        (7, 0.519295058483495, 0.0, 0.571562484299075),\n                        (8, 0.8595970149559404, 0.0, -0.5150727690597252),\n                        (5, 0.5092210624521557, 0.0, 0.5077195949417697),\n                        (11, 0.7728144741714895, 0.0, 0.5340030444302604),\n                        (2, 0.4999999999999998, 0.0, 0.5937872305331664),\n                        (10, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (1, 0.5, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (9, 0.30408672398469655, 0.0, -0.5745192213521528),\n                        (12, 0.30408672398469655, 0.0, -0.5745192213521528)],\n                       [(3, 6), (4, 7), (5, 8)],\n                       [(3, 0.5000000000000082, 0.0, -0.38814052101139707),\n                        (4, 0.7966994469363391, 0.0, 0.17924164039288176),\n                        (6, 0.846215799391159, 0.0, 0.4140343656892165),\n                        (7, 0.9656701121264165, 4.440892098500626e-16, 0.9090092197439124),\n                        (5, 0.5892643222216627, 0.0, -0.5024249961634872),\n                        (8, 0.01825127957935591, 0.0, -0.47595562611703457)],\n                       [(3, 4), (6, 7), (8, 11)],\n                       [(4, 0.33655308400709466, 0.0, 0.3292292693648231),\n                        (7, 0.8620124527871776, 5.551115123125783e-17, 0.1803395110035319),\n                        (3, 0.7195346408801285, 0.0, 0.9275067062261011),\n                        (6, 0.4999999999999998, 0.0, 0.5850957101524152),\n                        (11, 0.22718552582851337, 8.881784197001252e-16, -0.4659969555697461)],\n                       [(4, 5)],\n                       [(4, 0.3365530840070935, 0.0, -0.6707707306351736),\n                        (5, 0.5000000000000058, 0.0, 0.7986044123406482)],\n                       [(4, 7), (5, 8)],\n                       [(4, 0.47090840482178287, 0.0, 0.12183117956314454),\n                        (8, 0.4599671026716721, 0.0, -0.15268028985724325),\n                        (7, 0.21735497141644863, 4.440892098500626e-16, 0.5172953700928402),\n                        (5, 0.4999999999999998, 0.0, -0.20139558765935495)],\n                       [(3, 4), (7, 8)],\n                       [(3, 0.5000000000000018, 0.0, 0.8340915998087168),\n                        (7, 0.8131142728774541, 0.0, -0.9681984715613723),\n                        (8, 0.6690344411725309, 0.0, -0.8333355761457926),\n                        (4, 0.5000000000000027, 0.0, 0.7612048890474343)],\n                       [(3, 6), (4, 5)],\n                       [(6, 0.5, 3.3306690738754696e-16, -0.414904289847585),\n                        (4, 0.659345182818458, 0.0, -0.738795110952625),\n                        (5, 0.4999999999999977, 0.0, -0.8680277183870002)],\n                       [(6, 7), (5, 2), (3, 4)],\n                       [(7, 0.3976011891657918, 0.0, -0.2958152209085818),\n                        (4, 0.46288448269279947, 0.0, 0.7816165123610253),\n                        (5, 0.9416885504219519, 0.0, -0.418827820798918),\n                        (2, 0.6085354469534892, 4.440892098500626e-16, -0.9311758863092963)],\n                       [(7, 8), (3, 4)],\n                       [(7, 0.678470455050403, 0.0, 0.8771098395767996),\n                        (8, 0.011596791154027787, 0.0, -0.9876373457060301),\n                        (4, 5)],\n                       [(8, 11),\n                        (6, 7),\n                        (4, 0.09666409909802186, 0.0, 0.6693367583879934),\n                        (5, 0.5, 0.0, -0.4915570532159026)],\n                       [(8, 0.49999999999999845, 0.0, -0.8344617051950733),\n                        (7, 0.7095600142556546, 0.0, 0.8953074152323395),\n                        (6, 0.5, 0.0, 0.8694705207310347),\n                        (3, 4),\n                        (5, 2)],\n                       [(7, 8),\n                        (4, 0.5812745787265813, 0.0, 0.875238789713835),\n                        (5, 0.5150139304348057, 0.0, -0.49155705321590304),\n                        (2, 0.16723314080476157, 0.0, -0.7374666288921621)],\n                       [(7, 0.5000000000000007, 0.0, -0.9569629120925618),\n                        (8, 0.06274699981538136, 0.0, 0.6655382948049202),\n                        (4, 5)],\n                       [(6, 7),\n                        (8, 11),\n                        (4, 0.2955394861408327, 0.0, -0.0973551285019969),\n                        (5, 0.08858950606155515, 0.0, -0.9915570532159022)],\n                       [(8, 0.3773409284157756, 4.440892098500626e-16, 0.5868101190693529),\n                        (7, 0.5000000000000003, 0.0, -0.6687440789251577),\n                        (11, 0.7780272931625737, 0.0, -0.5848034228362193),\n                        (5, 2),\n                        (3, 4),\n                        (6, 0.5000000000000003, -0.6863539727098364, 0.8694705207310336)],\n                       [(11, 10),\n                        (7, 8),\n                        (4, 0.2943011721717048, 0.0, -0.1195681674093505),\n                        (5, 0.3017505280831282, 0.0, 0.5084429467840976),\n                        (2, 0.6303550437629815, 2.220446049250313e-16, -0.18158983536649442),\n                        (3, 0.4999999999999994, 0.0, 0.7480155561090385)],\n                       [(11, 0.5, 0.0, -0.08480342283621578),\n                        (10, 0.85928593962011, 0.0, -0.9262332254895425),\n                        (7, 0.5, 0.0, -0.6687440789251573),\n                        (4, 5)],\n                       [(8, 11),\n                        (10, 7),\n                        (4, 0.302862294871528, 0.0, 0.8900371977305799),\n                        (5, 0.19954896443512293, 0.0, 0.008442946784096905)],\n                       [(10, 0.7485956033393342, 0.0, -0.42623322548954246),\n                        (11, 0.6714165234362537, 1.1102230246251565e-16, -0.013387331105223055),\n                        (8, 0.5, 0.0, 0.0734491018992729),\n                        (5, 2),\n                        (3, 4)],\n                       [(7, 8),\n                        (11, 10),\n                        (5, 0.22177219480665558, 8.881784197001252e-16, 0.5084429467840959),\n                        (4, 0.5, 0.0, 0.12158652377958118),\n                        (3, 0.33347826094116906, 0.0, -0.7519844438909631),\n                        (2, 0.8315920243625678, 2.220446049250313e-16, -0.4157702878189209)],\n                       [(10, 0.9961572718865779, 0.0, -0.9262332254895478),\n                        (7, 0.0528060550044127, 0.0, -0.4951437351381869),\n                        (11, 0.3935294720438233, 5.551115123125783e-17, 0.26565978450839783),\n                        (8, 0.5, 1.1102230246251565e-16, 0.073449101899273),\n                        (4, 5)],\n                       [(10, 7),\n                        (8, 11),\n                        (4, 0.5, 0.0, 0.12158652377958162),\n                        (5, 0.5, 0.0, -0.00451958061344504)],\n                       [(11, 0.3104369525321424, 0.0, -0.15286146589180127),\n                        (10, 0.5655532211464794, 0.0, 0.5737667745104469),\n                        (7, 0.24963665794604523, 0.0, 0.02221884977158739),\n                        (5, 2),\n                        (3, 4)],\n                       [(7, 8),\n                        (11, 10),\n                        (4, 0.5519231616233388, 0.0, 0.9037902072479489),\n                        (5, 0.5000000000000001, 0.0, -0.004519580613445484),\n                        (3, 0.5276550549544043, 0.0, 0.24801555610903708),\n                        (2, 0.5, 0.0, 0.46178536847925344)],\n                       [(10, 0.4999999999999998, 0.0, 0.5737667745104462),\n                        (11, 0.6193705712509704, 4.440892098500626e-16, 0.7201042232087942),\n                        (7, 0.3270115623295809, 0.0, -0.37000079151651466),\n                        (4, 5)],\n                       [(10, 7),\n                        (8, 11),\n                        (4, 0.5, 0.0, -0.4620630388464564),\n                        (5, 0.5, 0.0, -0.30399824274559495)],\n                       [(7, 0.5, 0.0, 0.8773637399623619),\n                        (11, 0.6256105257587086, 1.1102230246251565e-16, 0.20337459121842366),\n                        (10, 0.5960614607313279, -2.220446049250313e-16, -0.11684358694965402),\n                        (5, 2),\n                        (3, 4)],\n                       [(10, 7),\n                        (8, 11),\n                        (2, 0.5, 0.0, -0.038214631520746334),\n                        (3, 0.5, 0.0, 0.7480155561090365),\n                        (4, 0.5, 0.0, 0.5379369611535436)],\n                       [(7, 0.5, 0.0, -0.6226362600376383),\n                        (11, 0.9764747842833552, 2.220446049250313e-16, -0.19884116670218432),\n                        (8, 0.5, 0.0, 0.7523776725917897),\n                        (10, 0.14949404734505523, 0.0, -0.7269328881171262),\n                        (3, 4),\n                        (5, 2)],\n                       [(7, 8),\n                        (11, 10),\n                        (2, 0.5068340379253959, 0.0, -0.5310585926382718),\n                        (3, 0.490105327468914, 1.1102230246251565e-16, -0.2519609116076511),\n                        (4, 0.5, 0.0, 0.5379369611535441),\n                        (5, 0.5, 0.0, -0.8039982427455956)],\n                       [(8, 0.5, 0.0, 0.7523776725917897),\n                        (11, 0.5, 0.0, 0.04413711647968971),\n                        (10, 7),\n                        (3, 1),\n                        (2, 0)],\n                       [(8, 11),\n                        (10, 0.4247761378557593, 0.0, -0.022381851578533896),\n                        (2, 0.5, 0.0, -0.7736945421001282),\n                        (3, 0.4999999999999998, -2.220446049250313e-16, 0.2472819384464749),\n                        (0, 0.3040867239846962, 0.0, -0.5000000000000002),\n                        (1, 0.3040867239846962, 0.0, -0.5000000000000002)],\n                       [(11, 0.14731871270290633, 0.0, -0.95586288352031),\n                        (8, 0.5, 4.440892098500626e-16, 0.8195665266104271),\n                        (5, 2),\n                        (4, 3)],\n                       [(11, 10),\n                        (7, 8),\n                        (2, 0.8040867239846965, 0.0, 0.22630545789987178),\n                        (3, 0.8040867239846965, 0.0, 0.2472819384464755)],\n                       [(7, 0.4202007516434625, 0.0, -0.9160739016291406),\n                        (10, 0.45008695289946266, 0.0, -0.3365366861733614),\n                        (11, 0.9433458052208397, 0.0, -0.45586288352031246),\n                        (2, 0),\n                        (3, 1)],\n                       [(10, 7),\n                        (8, 11),\n                        (3, 0.7499999999999999, 5.551115123125783e-17, 0.24728193844647434),\n                        (2, 0.7499999999999999, 5.551115123125783e-17, 0.22630545789987172),\n                        (0, 0.5, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5)],\n                       [(8, 0.20567003520391758, 0.0, -0.6804334733895732),\n                        (7, 0.5, 0.0, -0.5112697867898672),\n                        (11, 0.5, 5.551115123125783e-17, 0.33846708127576947),\n                        (10, 0.5, 0.0, -0.2896451505492772),\n                        (5, 2),\n                        (4, 3)],\n                       [(8, 7),\n                        (11, 10),\n                        (3, 0.5, 0.0, -0.7527180615535245),\n                        (2, 0.5, 0.0, 0.22630545789987128),\n                        (5, 0.5, 0.0, -0.5719081203957321),\n                        (4, 0.5, 0.0, -0.17946063363945708)],\n                       [(11, 0.4999999999999824, 0.0, -0.6615328624062882),\n                        (8, 0.49999999999998246, 3.3306690738754696e-16, -0.18043341707163219),\n                        (10, 0.5000000000000022, 0.0, 0.6216949198585577),\n                        (7, 0.5000000000000018, 0.0, 0.5773901428022985),\n                        (2, 0),\n                        (3, 1)],\n                       [(11, 12),\n                        (8, 9),\n                        (2, 0.25000000000000006, 0.0, -0.7736945421001284),\n                        (3, 0.25000000000000006, 0.0, 0.2472819384464755)],\n                       [(8, 0.5000000000000104, 0.0, -0.18043341707163196),\n                        (11, 0.5000000000000103, 0.0, -0.6615328624062877),\n                        (9, 0.30408672398469627, 0.0, -0.5745192213521484),\n                        (12, 0.30408672398469627, 0.0, -0.5745192213521484),\n                        (4, 3),\n                        (5, 2)],\n                       [(11, 10),\n                        (8, 7),\n                        (2, 0.5, 0.0, -0.7736945421001282),\n                        (3, 0.5, 0.0, -0.7527180615535245),\n                        (5, 0.5, -0.9280918796042674, 0.42809187960426737),\n                        (4, 0.5, 0.6794606336394573, -0.1794606336394573)],\n                       [(8, 0.75, 0.0, 0.8195666154435466),\n                        (11, 0.7499999999999999, 0.0, -0.6615328298911098),\n                        (7, 0.49999999999999956, 0.0, 0.5773901753174808),\n                        (10, 0.49999999999999956, 0.0, 0.6216948873433754),\n                        (3, 1),\n                        (2, 0)],\n                       [(11, 12),\n                        (8, 9),\n                        (2, 0.5, -0.7263054578998716, -0.7736945421001284),\n                        (0, 1.0, 0.0, 0.5),\n                        (3, 0.5, -0.7472819384464726, -0.7527180615535254),\n                        (1, 1.0, 0.0, 0.5)],\n                       [(8, 0.500526889421938, 0.0, 0.32009355157071084),\n                        (11, 0.5005268894219383, 0.0, 0.8389941062360531),\n                        (9, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155),\n                        (12, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155)],\n                       [(8, 7), (11, 10)],\n                       [(11, 0.5, 0.0, -0.4110054576905129),\n                        (8, 0.5, 0.0, 0.07009398764414332),\n                        (7, 0.5000000000000004, 0.0, -0.9226098246825187),\n                        (10, 0.5000000000000003, -2.220446049250313e-16, 0.12169488734337541)],\n                       [(8, 9), (11, 10)],\n                       [(11, 12),\n                        (8, 7),\n                        (9, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (10, 0.0007451670218237004, -0.34790937491533347, -0.37830511265661615)],\n                       [(12, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (11, 0.999999967484822, -0.31520907521242214, 0.5889945649749744),\n                        (8, 0.999999967484822, 0.20369147945292232, 0.07009401030963147),\n                        (7, 0.9992548329781763, -0.30360466288942156, 0.5773901753174724)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],\n           \'qubits\': [(2, 6),\n                      (3, 4),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 5),\n                      (6, 6),\n                      (6, 7)]},\n (13, 5): {\'method\': \'joint count seed and first-bit prefix + parallel decoders\',\n           \'moments\': [[(3, 0.5124064054678695, 0.0, 0.5091736799778506),\n                        (6, 0.4946072895223379, 0.0, -0.4831371952476968),\n                        (4, 0.9043076593678236, 0.0, -0.543013078287768),\n                        (7, 0.678007576188232, 4.440892098500626e-16, 0.5126813591233705),\n                        (5, 0.5000000000000027, 0.0, -0.40377601235190363),\n                        (8, 0.5000000000000002, 0.0, -0.5805167203878949),\n                        (2, 0.5, 0.0, -0.016627314735808696),\n                        (11, 0.7451246913861498, 0.0, 0.9312608165902445),\n                        (10, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (0, 0.5, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5),\n                        (9, 0.3040867239846966, 0.0, -0.5745192213521528),\n                        (12, 0.3040867239846966, 0.0, -0.5745192213521528)],\n                       [(3, 6), (4, 7), (5, 8)],\n                       [(3, 0.38611867710255277, 0.0, -0.32071832407395195),\n                        (4, 0.23205607932309918, 0.0, -0.2576461913810624),\n                        (6, 0.6570836341264275, 0.0, -0.9079936000382036),\n                        (7, 0.7854141778088372, 0.0, 0.7146652681465788),\n                        (5, 0.529346631847727, 0.0, 0.6137619300034722),\n                        (8, 0.5283502821375806, 0.0, 0.39872321063830113)],\n                       [(3, 4), (6, 7)],\n                       [(4, 5),\n                        (7, 8),\n                        (3, 0.17885376199856862, 0.0, 0.11423001808342464),\n                        (6, 0.3421414331825179, 0.0, -0.7584253247813113)],\n                       [(3, 6),\n                        (4, 0.4999999999999998, 0.0, -0.9962896968029795),\n                        (5, 0.5000000000000002, 0.0, 0.6331760225597092),\n                        (8, 0.5587425874466108, 0.0, -0.6170587279933593)],\n                       [(4, 7),\n                        (5, 8),\n                        (3, 0.9940611459366271, 0.0, -0.3857699819165783),\n                        (6, 0.4999999999999998, 0.0, -0.7584253247813113)],\n                       [(4, 0.5, 4.440892098500626e-16, -0.9962896968029797),\n                        (5, 0.44912129987788507, 0.0, -0.8668239774402897),\n                        (7, 0.9948025841580833, 0.0, -0.37813490493843904),\n                        (8, 0.2569430494789158, 0.0, -0.831128295986729)],\n                       [(3, 4), (5, 2), (6, 7), (8, 11)],\n                       [(3, 0.5000000000000004, 0.0, -0.8857699819165847),\n                        (4, 0.4999999999999998, 0.0, -0.4214032246085946),\n                        (7, 0.7910847695144607, 0.0, 0.4339335938380828),\n                        (8, 0.6818320697830313, -4.440892098500626e-16, 0.13112408462574765),\n                        (11, 0.1622868234878463, 0.0, -0.08013712721211497),\n                        (5, 0.3385028359086418, 0.0, -0.3593007107637525),\n                        (2, 0.6715957554835205, 0.0, -0.16119008792398848)],\n                       [(3, 4), (6, 7), (8, 11)],\n                       [(4, 5),\n                        (7, 0.5212907068281196, 0.0, -0.5658713211836695),\n                        (8, 0.5108572910094389, 0.0, -0.06986820261300952),\n                        (11, 0.24460388882100972, 0.0, -0.35613372205712923)],\n                       [(7, 8),\n                        (4, 0.7292475564750502, 0.0, 0.28485193532916275),\n                        (5, 0.5, 0.0, -0.6639427357673828)],\n                       [(3, 4),\n                        (5, 2),\n                        (7, 0.09618698452819888, 0.0, 0.4377885164601196),\n                        (8, 0.6200378421007051, 0.0, 0.44496440590598985)],\n                       [(6, 7),\n                        (8, 11),\n                        (4, 0.8236951914845039, 0.0, -0.6195889777922952),\n                        (5, 0.13681175462965722, 0.0, -0.663942735767383),\n                        (2, 0.27172923178718184, 0.0, -0.7559786295779003)],\n                       [(4, 5),\n                        (7, 0.39947144700796056, 5.551115123125783e-17, 0.43778851646015404),\n                        (8, 0.5481417770344861, 0.0, -0.5550355940940148),\n                        (11, 0.622144086057159, 0.0, -0.6029891418958397)],\n                       [(7, 8),\n                        (4, 0.4518866973223102, 0.0, -0.9830022604695918),\n                        (5, 0.30544906832810376, 0.0, -0.16394273576738172)],\n                       [(3, 4),\n                        (5, 2),\n                        (8, 11),\n                        (7, 0.8950616894287111, 4.440892098500626e-16, -0.5622114835398313)],\n                       [(6, 7),\n                        (4, 0.5707034495925996, 5.551115123125783e-17, 0.07580231728387055),\n                        (5, 0.785237728664876, 8.881784197001252e-16, -0.6639427357673819),\n                        (8, 0.5886108001133237, 0.0, -0.5550355940940268),\n                        (3, 0.5000000000000036, 0.0, 0.36371652096339746),\n                        (2, 0.4724075317596328, 0.0, 0.657484274013953),\n                        (11, 0.6627749752178567, -2.220446049250313e-16, 0.397010858104165)],\n                       [(4, 5), (7, 0.6234817718997734, 0.0, -0.5622114835398129)],\n                       [(7, 8),\n                        (4, 0.5419474843487612, 0.0, 0.9126112958457437),\n                        (5, 0.5858994169251085, 0.0, 0.8360572642326167)],\n                       [(3, 4),\n                        (5, 2),\n                        (7, 0.10650618596763642, 0.0, 0.43778851646021555),\n                        (8, 0.972660263157114, 0.0, 0.4449644059062949)],\n                       [(6, 7),\n                        (8, 11),\n                        (4, 0.5, 0.0, -0.19478683550464204),\n                        (5, 0.15794180998375124, 0.0, -0.663942735767383),\n                        (2, 0.643422681037554, 0.0, 0.11543360081664167),\n                        (3, 0.5, 0.0, -0.6362834790366056)],\n                       [(4, 5),\n                        (7, 0.551709352974791, 0.0, -0.5622114835398122),\n                        (8, 0.5650108729019683, 0.0, -0.5550355940933844),\n                        (6, 0.5, 0.0, 0.8726849745477665)],\n                       [(5, 2), (7, 8), (4, 0.10589866865514246, 0.0, 0.30521316449535796)],\n                       [(3, 4),\n                        (7, 0.20452064039209134, 0.0, -0.5622114835398158),\n                        (8, 0.8271117191719697, 0.0, 0.44496440590666),\n                        (5, 0.5, 0.0, -0.16394273576738128),\n                        (2, 0.6811220971645664, 0.0, -0.3461747253859806)],\n                       [(6, 7),\n                        (8, 11),\n                        (4, 5),\n                        (3, 0.4908982089013705, 4.440892098500626e-16, 0.7476172050821528)],\n                       [(7, 0.5, 0.0, 0.9627145231845533),\n                        (8, 0.6482311860549647, 0.0, 0.5486360381706115),\n                        (11, 0.7780272931625714, 0.0, -0.4184074515633833),\n                        (4, 0.42589339437843377, 0.0, -0.19478683550464382),\n                        (5, 0.3395311910112335, 0.0, -0.16394273576738128),\n                        (6, 0.4999999999999998, 0.40291599566731184, 0.8726849745477665)],\n                       [(7, 8), (11, 10), (3, 4), (5, 2)],\n                       [(11, 0.5, -2.220446049250313e-16, 0.08159254843661828),\n                        (10, 0.85928593962011, 0.0, -0.9262332254895425),\n                        (7, 0.5, 0.0, 0.9627145231845533),\n                        (4, 0.5, 0.0, 0.8052131644953557),\n                        (5, 0.5, 0.0, 0.836057264232619)],\n                       [(8, 11), (10, 7), (4, 5)],\n                       [(8, 0.5, 0.0, -0.6051067445606602),\n                        (11, 0.6714165234362537, 0.0, 0.15300864016761156),\n                        (10, 0.7485956033393342, 0.0, -0.42623322548954246),\n                        (4, 0.0035868934861584827, 0.0, -0.6947868355046476),\n                        (5, 0.5558460531129374, 0.0, 0.3360572642326196)],\n                       [(7, 8), (11, 10), (3, 4), (5, 2)],\n                       [(8, 0.5, 0.0, -0.6051067445606602),\n                        (11, 0.3935294720438233, 1.1102230246251565e-16, 0.43205575578123195),\n                        (10, 0.9961572718865779, 0.0, -0.9262332254895478),\n                        (7, 0.0528060550044127, 0.0, -0.8636851330284763),\n                        (2, 0.7708918263538406, 0.0, -0.39597210290977203),\n                        (3, 0.7708918263538402, 0.0, -0.5016619372945663),\n                        (4, 0.5000000000000002, 0.0, -0.19478683550464382),\n                        (5, 0.4999999999999998, 0.0, -0.1639427357673804)],\n                       [(8, 11), (10, 7), (2, 0), (3, 1)],\n                       [(7, 0.24963665794604492, 0.0, -0.3463225481187009),\n                        (11, 0.3104369525321424, 0.0, 0.01353450538103318),\n                        (10, 0.5655532211464794, 0.0, 0.5737667745104471),\n                        (3, 0.5, 8.881784197001252e-16, 0.9983380627054332),\n                        (2, 0.5, 0.0, 0.10402789709022792),\n                        (0, 0.3040867239846962, 0.0, 0.4999999999999998),\n                        (1, 0.30408672398469605, 0.0, -0.5000000000000002)],\n                       [(7, 8), (11, 10), (4, 3), (5, 2)],\n                       [(11, 0.6193705712509703, 0.0, 0.8865001944816282),\n                        (10, 0.4999999999999997, 0.0, 0.5737667745104471),\n                        (7, 0.32701156232958084, 0.0, -0.7385421894068038),\n                        (2, 0.8040867239846965, 3.885780586188048e-16, 0.10402789709022786),\n                        (3, 0.8040867239846965, 0.0, -0.001661937294565874)],\n                       [(8, 11), (10, 7), (2, 0), (3, 1)],\n                       [(11, 0.6256105257587086, 0.0, 0.3697705624912577),\n                        (10, 0.5960614607313279, 0.0, -0.11684358694965402),\n                        (7, 0.5, 0.0, 0.5088223420720728),\n                        (3, 0.75, 0.0, -0.001661937294565874),\n                        (2, 0.75, 0.0, 0.10402789709022825),\n                        (0, 0.5, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5)],\n                       [(8, 11), (10, 7), (4, 3), (5, 2)],\n                       [(7, 0.5, 0.0, -0.9911776579279281),\n                        (8, 0.5, 0.0, 0.07382182613185678),\n                        (11, 0.9764747842833552, 0.0, -0.03244519542935098),\n                        (10, 0.14949404734505523, 0.0, -0.7269328881171262),\n                        (2, 0.5, 0.0, -0.895972102909772),\n                        (3, 0.5, 0.0, -0.001661937294565874),\n                        (4, 0.5, 0.0, 0.16976725131843579),\n                        (5, 0.5, 0.0, -0.5284968225904625)],\n                       [(7, 8), (11, 10), (2, 0), (3, 1)],\n                       [(8, 0.5, 0.0, 0.07382182613185678),\n                        (11, 0.5, 0.0, 0.21053308775252355),\n                        (10, 7),\n                        (3, 0.25000000000000006, 0.0, 0.9983380627054343),\n                        (2, 0.25000000000000006, 0.0, 0.10402789709022825)],\n                       [(8, 11), (4, 3), (5, 2), (10, 0.4247761378557593, 0.0, -0.022381851578533674)],\n                       [(8, 0.5, 0.0, 0.14101068015049428),\n                        (11, 0.14731871270290633, 0.0, -0.7894669122474762),\n                        (2, 0.5, 0.0, -0.895972102909772),\n                        (3, 0.5, 0.0, 0.9983380627054343),\n                        (4, 0.5, 0.33023274868156394, -0.8302327486815639),\n                        (5, 0.5, 0.028496822590462267, 0.47150317740953773)],\n                       [(7, 8), (11, 10), (2, 0), (3, 1)],\n                       [(11, 0.9433458052208397, 0.0, -0.2894669122474771),\n                        (10, 0.45008695289946266, 0.0, -0.3365366861733614),\n                        (7, 0.4202007516434625, 0.0, 0.7153847004805705),\n                        (0, 1.0, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5),\n                        (2, 0.4999999999999998, -0.604027897090228, 0.10402789709022808),\n                        (3, 0.5, -0.4983380627054341, 0.9983380627054343)],\n                       [(8, 11), (10, 7)],\n                       [(8, 0.20567003520391755, 8.881784197001252e-16, 0.6410106801504931),\n                        (7, 0.5, 0.0, -0.8798111846801544),\n                        (11, 0.5, 0.0, 0.5048630525486049),\n                        (10, 0.5, 0.0, -0.2896451505492772)],\n                       [(8, 7), (11, 10)],\n                       [(8, 0.4999999999999824, 0.0, -0.8589892635315657),\n                        (11, 0.49999999999998246, 0.0, -0.4951368911334546),\n                        (7, 0.5000000000000018, 0.0, 0.20884874491201083),\n                        (10, 0.5000000000000022, 0.0, 0.6216949198585577)],\n                       [(8, 9), (11, 12)],\n                       [(8, 0.5000000000000102, 4.440892098500626e-16, -0.8589892635315657),\n                        (11, 0.5000000000000103, 0.0, -0.49513689113345416),\n                        (9, 0.30408672398469627, 0.0, -0.5745192213521484),\n                        (12, 0.30408672398469627, 0.0, -0.5745192213521484)],\n                       [(8, 7), (11, 10)],\n                       [(8, 0.75, 5.551115123125783e-17, 0.1410107689836137),\n                        (11, 0.75, 0.0, -0.4951368586182765),\n                        (7, 0.49999999999999956, 0.0, 0.20884877742719293),\n                        (10, 0.49999999999999956, 0.0, 0.6216948873433756)],\n                       [(8, 9), (11, 12)],\n                       [(8, 0.5005268894219383, 0.0, -0.35846229488922177),\n                        (11, 0.5005268894219383, 4.440892098500626e-16, -0.9946099224911118),\n                        (9, 0.5000000000000102, 0.0, 0.4254807786478514),\n                        (12, 0.5000000000000102, 0.0, 0.4254807786478514)],\n                       [(8, 7), (11, 10)],\n                       [(8, 0.5, 0.0, -0.6084618588157897),\n                        (11, 0.5, 0.0, -0.24460948641767866),\n                        (7, 0.5000000000000004, 0.0, 0.7088487774271925),\n                        (10, 0.5000000000000003, 0.0, 0.1216948873433758)],\n                       [(8, 9), (11, 12)],\n                       [(8, 7),\n                        (11, 10),\n                        (9, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (12, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(7, 0.9992548329781763, 0.06493673500086539, 0.208848777427185),\n                        (8, 0.999999967484822, 0.8822473259128545, -0.6084618361503011),\n                        (10, 0.0007451670218237004, -0.34790937491533347, -0.37830511265661615),\n                        (11, 0.999999967484822, -0.4816050464852566, 0.755390536247809)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],\n           \'qubits\': [(2, 6),\n                      (3, 4),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 5),\n                      (6, 6),\n                      (6, 7)]},\n (13, 6): {\'method\': \'round 8 count kernels + parallel coherent decoders\',\n           \'moments\': [[(3, 0.4501939478585324, 0.0, 0.5),\n                        (4, 0.44772204076930416, 0.0, 0.5),\n                        (7, 0.49890701820126315, 0.0, -0.5),\n                        (5, 0.2968525986784217, 0.0, 0.5),\n                        (8, 0.32688298887505485, 0.0, -0.5000000000000002),\n                        (2, 0.5475860233340665, 0.0, -0.3973495931613553),\n                        (11, 0.4768011512951635, 0.0, -0.17578877400173032),\n                        (10, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (0, 0.5, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5),\n                        (9, 0.3040867239846966, 0.0, -0.5745192213521528),\n                        (12, 0.3040867239846966, 0.0, -0.5745192213521528)],\n                       [(3, 6), (4, 7), (5, 8)],\n                       [(3, 0.5498060521414676, 0.0, 0.5),\n                        (4, 0.7514161356970753, 0.0, -0.5),\n                        (7, 0.528747685665836, 0.0, 0.5),\n                        (5, 0.4099536350801986, 0.0, 0.5),\n                        (8, 0.6956068135166766, 0.0, 0.5)],\n                       [(3, 4), (6, 7)],\n                       [(6, 0.9786577811306381, 0.0, 0.5),\n                        (4, 0.4179130342668791, 0.0, -0.5000000000000002),\n                        (7, 0.8442678290297327, 0.0, -0.5)],\n                       [(3, 6), (4, 5), (7, 8)],\n                       [(4, 0.7215147296373279, 0.0, 0.5),\n                        (7, 0.6742556384235301, 0.0, 0.5),\n                        (5, 0.278710499924302, 0.0, 0.5),\n                        (8, 0.45282643258460314, 0.0, 0.5),\n                        (6, 0.47865778113063806, 0.0, 0.5)],\n                       [(4, 7), (5, 8)],\n                       [(4, 0.783830865607837, 0.0, -0.5),\n                        (7, 0.11983442054539162, 0.0, -0.5),\n                        (8, 0.7137922747629629, 0.0, -0.5),\n                        (5, 0.3033248226376897, 0.0, -0.5000000000000002)],\n                       [(3, 4), (6, 7)],\n                       [(7, 8), (3, 0.5, 0.0, 0.5), (6, 0.5, 0.0, -0.5), (4, 0.08774248146714281, 0.0, 0.5)],\n                       [(3, 6), (4, 5), (7, 0.5, 0.0, -0.5), (8, 0.9726170148740435, 0.0, -0.5)],\n                       [(4, 0.5323399770960722, 0.0, -0.5),\n                        (5, 0.4058078462486393, 0.0, 0.5),\n                        (3, 0.5, 0.0, 0.5)],\n                       [(4, 7), (5, 8)],\n                       [(6, 7),\n                        (4, 0.5, 0.0, -0.5),\n                        (5, 0.8698743711571301, 0.0, 0.6703497126193745),\n                        (8, 0.42462669509266576, 0.0, -0.13381503145355977)],\n                       [(3, 4),\n                        (5, 2),\n                        (8, 11),\n                        (6, 0.40328061753541666, 0.0, 0.38599009045950217),\n                        (7, 0.9232457490063548, 0.0, 1.0)],\n                       [(6, 7),\n                        (4, 0.3496081461944575, 0.0, 0.5298336277625533),\n                        (8, 0.1639993418571834, 0.0, -0.30025164825377315),\n                        (5, 0.203817189885253, 0.0, -0.27587420209739144),\n                        (11, 0.6279799493860765, 0.0, -0.770287955286147),\n                        (2, 0.4907843995880299, 0.0, -0.5880277617254166)],\n                       [(3, 4), (7, 8), (6, 0.7174704187311103, 0.0, -0.11400990954049761)],\n                       [(4, 0.8623097487166651, 0.0, 0.5667384100724131),\n                        (7, 0.4558467315798986, 0.0, 0.4947032283154509),\n                        (8, 0.2771547609074463, 0.0, 0.3858926716604727)],\n                       [(4, 5), (6, 7), (8, 11)],\n                       [(4, 0.6133211189678014, 0.0, -0.8962387535137009),\n                        (5, 0.4609426760692442, 0.0, -0.25670740153555194),\n                        (6, 0.19408134503630478, 0.0, 0.3859900904595026),\n                        (7, 0.42335735607789227, 0.0, 0.9947032283154507),\n                        (8, 0.6057423835093776, 0.0, -0.6247107871374926),\n                        (11, 0.019663643370415814, 0.0, 0.5185052910680361)],\n                       [(3, 4), (5, 2), (6, 7), (8, 11)],\n                       [(4, 0.16552235636911306, 0.0, -0.6095321708616528),\n                        (5, 0.8389597068065685, 0.0, 0.8262273337056933),\n                        (7, 0.7065221727328495, 0.0, -0.9772689364961986),\n                        (8, 0.776375637814382, 0.0, -0.6176868477524529),\n                        (2, 0.41925297156500135, 0.0, 0.8910230117351086),\n                        (6, 0.574001540707395, 0.0, 0.8859900904595026),\n                        (11, 0.5680691667523844, 0.0, -0.6374006217137276)],\n                       [(4, 5), (7, 8)],\n                       [(4, 0.5888130859263944, 0.0, -0.44330665331199315),\n                        (5, 0.5974911130186349, 0.0, -0.6425415716211378),\n                        (7, 0.7550526236278825, 0.0, 0.5308814090241605),\n                        (8, 0.7960253457838964, 0.0, -0.587523848788472)],\n                       [(3, 4), (5, 2), (6, 7), (8, 11)],\n                       [(4, 0.98634673875274, 0.0, -0.2612952376844171),\n                        (5, 0.8870191792759949, 0.0, 0.569778841342893),\n                        (7, 0.09317050870903601, 0.0, 0.661564491678851),\n                        (8, 0.5920891677547482, -2.220446049250313e-16, -0.08752384878847197),\n                        (2, 0.3036097100189865, 0.0, -0.41865868665252437),\n                        (11, 0.5767646373955954, 0.0, -0.7141291884531249)],\n                       [(4, 5), (7, 8)],\n                       [(4, 0.3099549190429492, 0.0, -0.2320391062895908),\n                        (5, 0.40168752603487473, 0.0, -0.8480131764389265),\n                        (7, 0.8490154235516789, 0.0, -0.1295448033769031),\n                        (8, 0.8905211577257011, 0.0, 0.4124761512115276)],\n                       [(3, 4), (5, 2), (6, 7), (8, 11)],\n                       [(4, 0.600782897473453, 0.0, -0.3347145198448782),\n                        (5, 0.7259897335585628, 0.0, -0.47079272037710584),\n                        (7, 0.5525232416584167, 0.0, 0.8088605091439913),\n                        (8, 0.623464211234003, 0.0, -0.08752384878847241),\n                        (2, 0.1440642715730953, 0.0, -0.9293010646270876),\n                        (11, 0.22565786477821326, 0.0, 0.9040830152802759)],\n                       [(4, 5), (7, 8)],\n                       [(4, 0.6050322596766913, 0.0, -0.2704588432039663),\n                        (5, 0.4918606380326855, 0.0, -0.7778382627808464),\n                        (7, 0.2256027665011393, 0.0, -0.021367266267178886),\n                        (8, 0.19755687312566206, 0.0, 0.41247615121152825)],\n                       [(3, 4), (5, 2), (6, 7), (8, 11)],\n                       [(4, 0.2930713133531244, 0.0, -0.2722557577346172),\n                        (5, 0.5345099653792809, 0.0, 0.2127689944999971),\n                        (7, 0.27319568030875896, 0.0, -0.7064883514321607),\n                        (8, 0.5, 0.0, 0.41247615121152825),\n                        (2, 0.9747310894860806, 0.0, -0.4482186693900534),\n                        (6, 0.5, 0.0, -0.7883771254968213),\n                        (11, 0.08965970809102466, 0.0, 0.42015605383701876)],\n                       [(4, 5), (7, 8)],\n                       [(4, 0.7602803020313025, 0.0, 0.47748599954342885),\n                        (5, 0.60039080437022, 0.0, -0.04170011862673917),\n                        (7, 0.5, 0.0, -0.07103798055535071),\n                        (8, 0.6586668818397201, 0.0, 0.8179460177960558)],\n                       [(3, 4), (5, 2), (6, 7), (8, 11)],\n                       [(4, 0.5690249633686464, 0.0, -0.11399591378561391),\n                        (5, 0.2634186697710025, 0.0, 0.6421832690773221),\n                        (7, 0.5, 0.0, -0.5588881291127312),\n                        (8, 0.6178431557297261, 0.0, 0.8139241257528269),\n                        (11, 0.3573762722915826, 0.0, 0.4761174686905688),\n                        (2, 0.669270238620916, 0.0, -0.4031683526897307),\n                        (6, 0.5, -0.42324574900635614, -0.7883771254968204)],\n                       [(4, 5), (7, 8), (11, 10)],\n                       [(4, 0.3667835732922082, 0.0, 0.8587869270791131),\n                        (5, 0.7141573878023064, 4.440892098500626e-16, 0.5909663999925776),\n                        (11, 0.5, 0.0, -0.71839631538329),\n                        (10, 0.85928593962011, 0.0, -0.9262332254895425),\n                        (7, 0.5, 0.0, -0.558888129112731)],\n                       [(3, 4), (5, 2), (8, 11), (10, 7)],\n                       [(4, 0.9821865846875268, 0.0, -0.20646460262973854),\n                        (5, 0.16938789829238887, 0.0, 0.7913856983389227),\n                        (8, 0.5, 0.0, -0.48442718812422436),\n                        (11, 0.6714165234362537, 1.1102230246251565e-16, 0.353019776347702),\n                        (10, 0.7485956033393342, 0.0, 0.5737667745104575),\n                        (3, 0.49999999999999956, 0.0, -0.4046051971068747),\n                        (2, 0.4889308684247995, 0.0, -0.990609344979607)],\n                       [(4, 5), (7, 8), (11, 10)],\n                       [(4, 0.06388983453014863, 0.0, -0.024923998616681597),\n                        (5, 0.7210414783752492, 0.0, -0.5187182510065),\n                        (8, 0.5, 4.440892098500626e-16, 0.515572811875775),\n                        (11, 0.3935294720438234, 0.0, 0.6320668919613244),\n                        (10, 0.9961572718865779, 1.1102230246251565e-16, 0.07376677451045255),\n                        (7, 0.05280605500441269, 0.0, 0.26751152710029696)],\n                       [(3, 4), (5, 2), (8, 11), (10, 7)],\n                       [(4, 0.5000000000000001, -2.220446049250313e-16, 0.16996697209468042),\n                        (5, 0.37431403056854723, 0.0, -0.5521964238992452),\n                        (7, 0.24963665794604492, 0.0, -0.2498510578094777),\n                        (11, 0.31043695253214243, 0.0, -0.7864543584388739),\n                        (10, 0.5655532211464794, 0.0, 0.5737667745104471),\n                        (2, 0.3031631789706357, 0.0, -0.6331906474987763),\n                        (3, 0.5, 0.0, 0.5953948028931249)],\n                       [(4, 5), (7, 8), (11, 10)],\n                       [(5, 2),\n                        (4, 0.6424070708696193, 0.0, -0.3300330279053201),\n                        (11, 0.6193705712509703, -2.220446049250313e-16, 0.0865113306617209),\n                        (10, 0.4999999999999998, 0.0, 0.5737667745104469),\n                        (7, 0.32701156232958084, 0.0, -0.8576314165213752)],\n                       [(3, 4), (8, 11), (10, 7), (2, 0.5715285762277693, 0.0, 0.41322246156576803)],\n                       [(2, 0),\n                        (3, 0.38504171397226883, -2.220446049250313e-16, -0.764089462028833),\n                        (11, 0.6256105257587086, 0.0, 0.5697816986713504),\n                        (10, 0.5960614607313279, 0.0, 0.8831564130503462),\n                        (7, 0.5, 0.0, -0.10499594800025225),\n                        (4, 0.5000000000000001, 0.0, -0.8300330279053203)],\n                       [(3, 1),\n                        (8, 11),\n                        (10, 7),\n                        (2, 0.4999999999999997, 0.0, 0.9202562633997626),\n                        (0, 0.30408672398469605, 0.0, -0.5000000000000002)],\n                       [(5, 2),\n                        (3, 0.5, 0.0, 0.85206921155491),\n                        (7, 0.5, 0.0, -0.6049959480002522),\n                        (8, 0.5, 4.440892098500626e-16, 0.8366442411832566),\n                        (11, 0.9764747842833552, 0.0, -0.8324340592492592),\n                        (10, 0.14949404734505523, 0.0, -0.7269328881171258),\n                        (1, 0.30408672398469605, 0.0, -0.5000000000000002)],\n                       [(4, 3), (7, 8), (11, 10), (2, 0.8040867239846965, 0.0, 0.9202562633997626)],\n                       [(2, 0),\n                        (10, 7),\n                        (3, 0.8040867239846965, 1.1102230246251565e-16, -0.14793078844508978),\n                        (8, 0.5, 1.1102230246251565e-16, -0.16335575881674358),\n                        (11, 0.5, 0.0, -0.5894557760673849)],\n                       [(3, 1),\n                        (8, 11),\n                        (2, 0.75, 0.0, 0.9202562633997626),\n                        (10, 0.42477613785575913, 4.440892098500626e-16, 0.9776181484214663),\n                        (0, 0.5, 0.0, 0.5)],\n                       [(5, 2),\n                        (3, 0.75, 0.0, -0.14793078844509022),\n                        (8, 0.5, 0.0, -0.2305446128353803),\n                        (11, 0.1473187127029063, 0.0, -0.589455776067384),\n                        (1, 0.5, 0.0, 0.5)],\n                       [(4, 3),\n                        (7, 8),\n                        (11, 10),\n                        (2, 0.5, 0.0, -0.07974373660023737),\n                        (5, 0.5, 0.0, 0.3156766054324216)],\n                       [(2, 0),\n                        (3, 0.5, 0.0, -0.14793078844508933),\n                        (11, 0.9433458052208397, 0.0, -0.08945577606738642),\n                        (10, 0.4500869528994624, 0.0, 0.6634633138266388),\n                        (7, 0.4202007516434625, 2.220446049250313e-16, -0.3115583064087506),\n                        (4, 0.5, 0.0, -0.8379032263890158)],\n                       [(3, 1), (8, 11), (10, 7), (2, 0.25, 0.0, 0.9202562633997626)],\n                       [(5, 2),\n                        (3, 0.25000000000000006, 0.0, 0.85206921155491),\n                        (8, 0.2056700352039173, 0.0, 0.26945538716462036),\n                        (7, 0.5, 0.0, -0.7163624212480251),\n                        (11, 0.5, 0.0, -0.29512581127130266),\n                        (10, 0.5, 0.0, -0.28964515054927653)],\n                       [(4, 3),\n                        (8, 7),\n                        (11, 10),\n                        (2, 0.5, 0.0, -0.07974373660023737),\n                        (5, 0.5, 0.1843233945675784, 0.3156766054324216)],\n                       [(2, 0),\n                        (3, 0.5, 4.440892098500626e-16, 0.8520692115549107),\n                        (8, 0.49999999999998246, 0.0, 0.7694553308466794),\n                        (11, 0.4999999999999824, 8.881784197001252e-16, 0.7048742450466374),\n                        (7, 0.5000000000000022, 0.0, -0.8050223508401915),\n                        (10, 0.5000000000000022, 0.0, 0.621694919858558),\n                        (4, 0.5, -0.6620967736109833, 0.1620967736109843)],\n                       [(3, 1),\n                        (8, 9),\n                        (11, 12),\n                        (0, 1.0, 0.0, 0.5),\n                        (2, 0.5, 0.5797437366002387, -0.0797437366002387)],\n                       [(8, 0.5000000000000102, 0.0, 0.7694553308466794),\n                        (11, 0.5000000000000102, 0.0, 0.7048742450466383),\n                        (9, 0.30408672398469605, 0.0, 0.4254807786478514),\n                        (12, 0.30408672398469627, 0.0, -0.5745192213521484),\n                        (1, 1.0, 0.0, 0.5),\n                        (3, 0.5, -0.3520692115549109, 0.8520692115549109)],\n                       [(8, 7), (11, 10)],\n                       [(8, 0.75, 0.0, 0.7694552983315002),\n                        (11, 0.75, 0.0, 0.704874277561818),\n                        (7, 0.49999999999999956, 0.0, 0.19497761664462715),\n                        (10, 0.49999999999999956, 0.0, 0.6216948873433763)],\n                       [(8, 9), (11, 12)],\n                       [(8, 0.500526889421938, 0.0, -0.7310716377956639),\n                        (11, 0.5005268894219383, 0.0, 0.2054012136889818),\n                        (9, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515),\n                        (12, 0.5000000000000102, 0.0, 0.4254807786478514)],\n                       [(8, 7), (11, 10)],\n                       [(8, 0.5, 0.0, 0.5189279261309028),\n                        (11, 0.5000000000000001, 0.0, 0.9554016497624125),\n                        (7, 0.5000000000000003, 0.0, 0.6949776166446258),\n                        (10, 0.5000000000000003, 0.0, 0.12169488734337619)],\n                       [(8, 9), (11, 12)],\n                       [(8, 7),\n                        (11, 10),\n                        (9, 0.5, 0.3483047662953689, -0.5745192538673269),\n                        (12, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(7, 0.0007451670218237008, 0.07880789578341663, -0.8050223833553651),\n                        (8, 3.251517832758567e-08, 0.7548575862971392, 0.5189279034654148),\n                        (10, 0.0007451670218237004, -0.34790937491533436, -0.37830511265661526),\n                        (11, 0.999999967484822, 0.31838381733465027, -0.044598327572097585)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],\n           \'qubits\': [(2, 6),\n                      (3, 4),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 5),\n                      (6, 6),\n                      (6, 7)]},\n (14, 4): {\'method\': \'joint count seed and first-bit prefix + parallel decoders\',\n           \'moments\': [[(3, 0.02113622047375885, 0.0, 0.4999902975966264),\n                        (7, 0.4697890246101013, 0.0, -0.5000095675777865),\n                        (4, 0.10516644026849284, 0.0, -0.5000103070374979),\n                        (8, 0.7627919990528385, 0.0, -0.5000096948267472),\n                        (5, 0.42098373622110635, 0.0, 0.49999006607699675),\n                        (2, 0.4999999999999997, 0.0, -0.0748864563157734),\n                        (13, 0.5000000000000036, -2.220446049250313e-16, -0.07551465410399505),\n                        (1, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (12, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (0, 0.5, 1.0, 0.5),\n                        (11, 0.5, 1.0, 0.5),\n                        (6, 0.3040867239846966, 1.0, -0.5745192213521528),\n                        (10, 0.3040867239846966, 1.0, -0.5745192213521528)],\n                       [(3, 7), (4, 8), (5, 9)],\n                       [(3, 0.031408940105505094, 4.440892098500626e-16, 0.505027502617496),\n                        (4, 0.8476571833054374, 0.0, -0.5001200966110222),\n                        (7, 0.9625554616578312, 0.0, -0.4961198626645351),\n                        (8, 0.6822212504053066, 0.0, 0.5016365154557132),\n                        (5, 0.07902133609158167, 0.0, 0.5036721506295523)],\n                       [(3, 4), (7, 8)],\n                       [(3, 0.1553492860618011, 0.0, -0.49707180671307527),\n                        (7, 0.12593406135931628, 0.0, 0.49907326844781),\n                        (4, 0.07539764526086823, 0.0, -0.4935212377034244),\n                        (8, 0.2742330259904474, 8.881784197001252e-16, 0.5033808242671525)],\n                       [(3, 7), (4, 5), (8, 9)],\n                       [(8, 0.87808119707036, 0.0, -0.5010272217087712),\n                        (5, 0.5000000000000111, 0.0, -0.4998970625851644),\n                        (9, 0.75, 0.0, -0.4746025617333578),\n                        (3, 0.8174896377282594, 4.440892098500626e-16, 0.503576534260008),\n                        (7, 0.9660551544293239, 0.0, 0.5061122278510009)],\n                       [(4, 8), (5, 9)],\n                       [(3, 4),\n                        (8, 0.30638256653549734, 0.0, -0.5019463986144597),\n                        (9, 0.7499999999999996, 0.0, -0.4746025617333589)],\n                       [(7, 8),\n                        (3, 0.45594445768420266, 0.0, 0.5043848955978474),\n                        (4, 0.49999999999999395, 0.0, 0.5444680094593588)],\n                       [(7, 0.3011886089570617, 0.0, -0.49813180848517646),\n                        (4, 5),\n                        (8, 0.6199673106667174, 0.0, 0.4954613251919071)],\n                       [(3, 7),\n                        (8, 9),\n                        (4, 1.7682916265693287e-05, 0.0, 0.044468009448721924),\n                        (5, 0.500000000000002, 0.0, 0.49458397801352116)],\n                       [(4, 8),\n                        (5, 9),\n                        (3, 0.28920268780888414, 0.0, -0.49592841848414704),\n                        (7, 0.9933634190056742, 0.0, 0.5036908726276175)],\n                       [(4, 0.3865047784020846, 0.0, 0.5017151885304527),\n                        (5, 0.7915861709875629, 0.0, 0.7220959986702224),\n                        (7, 8),\n                        (9, 0.3478770495896315, 0.0, 0.46369280810462454)],\n                       [(3, 4),\n                        (5, 2),\n                        (9, 13),\n                        (8, 0.6492499478003718, 0.0, 0.45827622636484255),\n                        (7, 0.8594782530133674, 0.0, -0.49481370160197913)],\n                       [(4, 0.5392457749780963, -4.440892098500626e-16, 0.4889614313832653),\n                        (5, 0.7913327671706842, 5.551115123125783e-17, 0.33227143216299115),\n                        (9, 0.7913327671706818, 0.0, -0.21408437533872848),\n                        (2, 0.6144518630659476, -2.220446049250313e-16, 0.17857399537132101),\n                        (13, 0.6311913844927517, 1.1102230246251565e-16, 0.1711472004059677)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.7964417643891573, 0.0, 0.5418919147365457),\n                        (5, 2),\n                        (8, 0.7964417643891528, 0.0, 0.4582762263648177),\n                        (9, 13)],\n                       [(3, 4),\n                        (7, 8),\n                        (5, 0.5886108001133239, 0.0, -0.6677285678370073),\n                        (9, 0.5886108001133206, -2.220446049250313e-16, -0.21408437533873048),\n                        (2, 0.5973505269171038, 0.0, -0.42697227876736576),\n                        (13, 0.5973505269171007, 3.3306690738754696e-16, -0.4524531437903281)],\n                       [(4, 0.7964417643891544, 0.0, 0.5418919147365449),\n                        (8, 0.7964417643891567, 0.0, -0.5417237736351845)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.5517093529747901, 4.440892098500626e-16, 0.5418919147365435),\n                        (5, 0.179508517985687, 0.0, 0.33227143216299115),\n                        (8, 0.551709352974791, 0.0, -0.5417237736351748),\n                        (9, 0.179508517985687, 0.0, -0.21408437533873137)],\n                       [(3, 4), (5, 2), (7, 8), (9, 13)],\n                       [(4, 0.893493814032364, 0.0, 0.541891914736546),\n                        (5, 0.11878997884016983, 0.0, 0.3322714321629907),\n                        (8, 0.8934938140323643, 0.0, 0.45827622636485627),\n                        (9, 0.11878997884016629, 0.0, -0.21408437533872937),\n                        (3, 0.5000000000000027, 0.0, -0.5546093685392881),\n                        (7, 0.500000000000002, 0.0, 0.4464350841233422)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.5000000000000019, 0.0, -0.4581080852634509),\n                        (5, 2),\n                        (8, 0.49999999999999606, 0.0, -0.541723773635109),\n                        (9, 13)],\n                       [(3, 4),\n                        (7, 8),\n                        (2, 0.2219727068374307, 0.0, -0.8618909512172497),\n                        (5, 0.31787713509633825, 0.0, -0.04158383890163364),\n                        (9, 0.4041318051769006, 0.0, 0.34552690179293327),\n                        (13, 0.22197270683742992, 0.0, -0.6772718938792348)],\n                       [(2, 1),\n                        (4, 0.5000000000000004, 0.0, 0.3295501535415365),\n                        (8, 0.49999999999999956, 0.0, 0.7231429681501487),\n                        (13, 12),\n                        (3, 0.5000000000000013, 0.14279458428371772, 0.445390631460711),\n                        (7, 0.4999999999999992, 0.08922515103084283, -0.5535649158766576)],\n                       [(4, 5),\n                        (8, 9),\n                        (1, 0.85928593962011, 0.0, -0.9262332254895425),\n                        (12, 0.85928593962011, 0.0, -0.9262332254895425),\n                        (2, 0.5, 1.0, 0.6381090487827532),\n                        (13, 0.5, 0.9999999999999998, 0.8227281061207667)],\n                       [(4, 0.5000000000000002, 0.0, 0.3295501535415353),\n                        (8, 0.5000000000000002, 0.0, -0.27685703184985266),\n                        (5, 2),\n                        (9, 13)],\n                       [(1, 4),\n                        (12, 8),\n                        (2, 0.6714165234362537, 0.0, 0.7095251405137464),\n                        (5, 0.5, 0.0, 0.45063649264521033),\n                        (9, 0.5, 0.0, 0.8662307728905367),\n                        (13, 0.6714165234362537, 0.0, 0.8941441978517595)],\n                       [(1, 0.7485956033393342, -4.440892098500626e-16, -0.42623322548954246),\n                        (4, 5),\n                        (8, 9),\n                        (12, 0.7485956033393342, -4.440892098500626e-16, -0.42623322548954246)],\n                       [(2, 1),\n                        (13, 12),\n                        (4, 0.052806055004412676, 0.0, -0.49684950267149475),\n                        (5, 0.5, 0.0, 0.45063649264521033),\n                        (9, 0.5, 0.0, 0.8662307728905363),\n                        (8, 0.0528060550044127, 0.0, 0.8967433119371175)],\n                       [(1, 0.9961572718865779, 1.0, -0.9262332254895478),\n                        (2, 0.3935294720438234, 1.0, 0.9885722561273675),\n                        (13, 0.3935294720438234, 1.0, -0.8268086865346198),\n                        (12, 0.9961572718865779, 1.0, -0.9262332254895478)],\n                       [(1, 4), (5, 2), (9, 13), (12, 8)],\n                       [(2, 0.3104369525321425, 0.0, 0.5700510057271686),\n                        (1, 0.5655532211464794, 0.0, -0.4262332254895531),\n                        (13, 0.31043695253214243, 0.0, 0.7546700630651817),\n                        (12, 0.5655532211464794, 0.0, -0.4262332254895531),\n                        (4, 0.24963665794604492, 0.0, 0.020513082238280356),\n                        (5, 0.0, -0.8912180282173903, 0.0),\n                        (8, 0.24963665794604517, 0.0, -0.5858941031531075),\n                        (9, 0.0, -0.8912180282173903, 0.0)],\n                       [(2, 1), (13, 12), (4, 5), (8, 9)],\n                       [(2, 0.6193705712509703, 0.0, -0.5569833051722362),\n                        (13, 0.6193705712509703, 0.0, -0.3723642478342235),\n                        (1, 0.4999999999999998, 1.0, -0.4262332254895531),\n                        (4, 0.32701156232958084, 0.0, 0.6282934409501779),\n                        (12, 0.4999999999999998, 1.0, -0.4262332254895531),\n                        (8, 0.32701156232958106, 0.0, 0.021886255558790424)],\n                       [(5, 2), (9, 13), (1, 4), (12, 8)],\n                       [(1, 0.5960614607313279, 1.0, -0.11684358694965402),\n                        (4, 0.5, 0.0, -0.12434202757094548),\n                        (2, 0.6256105257587086, 1.0, -0.07371293716260774),\n                        (13, 0.6256105257587085, 1.0, 0.11090612017540558),\n                        (12, 0.5960614607313279, 1.0, -0.11684358694965402),\n                        (8, 0.5, 0.0, -0.7307492129623332)],\n                       [(1, 4), (5, 2), (9, 13), (12, 8)],\n                       [(2, 0.9764747842833552, 0.0, -0.47592869508321556),\n                        (1, 0.14949404734505523, 0.0, 0.2730671118828738),\n                        (4, 0.5, 0.0, 0.3756579724290546),\n                        (5, 0.5, 0.0, 0.8804898937353018),\n                        (8, 0.5, 0.0, -0.23074921296233342),\n                        (9, 0.5, 0.0, -0.7039158260193716),\n                        (13, 0.9764747842833552, 0.0, -0.29130963774520224),\n                        (12, 0.14949404734505523, 0.0, 0.2730671118828738)],\n                       [(2, 1), (4, 5), (8, 9), (13, 12)],\n                       [(4, 0.0, -0.9149679795284635, 0.0),\n                        (5, 0.5, 0.0, 0.8804898937353025),\n                        (2, 0.5, 0.9999999999999999, -0.23295041190134125),\n                        (9, 0.5, 0.0, -0.7039158260193716),\n                        (13, 0.5, 0.9999999999999999, -0.048331354563328155),\n                        (8, 0.0, -0.9149679795284635, 0.0)],\n                       [(1, 4), (5, 2), (9, 13), (12, 8)],\n                       [(2, 0.14731871270290633, 1.0, 0.7670495880986579),\n                        (1, 0.4247761378557593, 0.0, 0.9776181484214663),\n                        (4, 0.0, 1.0, 0.0),\n                        (5, 0.5, 4.440892098500626e-16, 0.8133010397166649),\n                        (8, 0.0, 1.0, 0.0),\n                        (9, 0.5, 0.0, -0.771104680038009),\n                        (13, 0.1473187127029063, 1.0, 0.951668645436671),\n                        (12, 0.4247761378557593, 0.0, 0.9776181484214663)],\n                       [(2, 1), (4, 5), (8, 9), (13, 12)],\n                       [(1, 0.45008695289946266, 0.0, 0.6634633138266384),\n                        (4, 0.4202007516434625, 1.0, 0.1672523513090889),\n                        (2, 0.9433458052208397, -0.9999999999999993, 0.2670495880986574),\n                        (13, 0.9433458052208397, 1.0, 0.4516686454366714),\n                        (12, 0.45008695289946266, 0.0, 0.6634633138266384),\n                        (8, 0.4202007516434625, 1.0, -0.43915483408229905)],\n                       [(1, 4), (5, 2), (9, 13), (12, 8)],\n                       [(1, 0.541088904736081, 0.0, -0.7896451505492768),\n                        (5, 0.20567003520391758, 1.1102230246251565e-16, 0.31330103971666545),\n                        (4, 0.5, 1.0, -0.42794353385163686),\n                        (9, 0.20567003520391758, 4.440892098500626e-16, 0.7288953199619914),\n                        (8, 0.5, 1.0, 0.9656492807569754),\n                        (12, 0.541088904736081, 0.0, -0.7896451505492768),\n                        (2, 0.0, 1.0, 0.0),\n                        (13, 0.0, 1.0, 0.0)],\n                       [(1, 0), (5, 4), (9, 8), (12, 11)],\n                       [(1, 0.5, 0.0, 0.710354849450723),\n                        (5, 0.49999999999998246, 2.220446049250313e-16, -0.18669901660127586),\n                        (9, 0.49999999999998246, 0.0, 0.22889526364405044),\n                        (12, 0.5, 0.0, 0.710354849450723),\n                        (0, 0.3040867239846962, 1.0, -0.5000000000000002),\n                        (4, 0.5000000000000022, 0.9999999999999999, 0.6607163957405289),\n                        (8, 0.500000000000002, 1.0, 0.05430921034914116),\n                        (11, 0.3040867239846962, 1.0, -0.5000000000000002)],\n                       [(2, 1), (5, 6), (9, 10), (13, 12)],\n                       [(1, 0.8040867239846965, 0.0, -0.28964515054927653),\n                        (5, 0.5000000000000104, 0.0, -0.18669901660127586),\n                        (9, 0.5000000000000104, 0.0, 0.22889526364405038),\n                        (12, 0.8040867239846965, 0.0, -0.28964515054927653),\n                        (6, 0.30408672398469616, 1.0, -0.5745192213521486),\n                        (10, 0.30408672398469616, 1.0, -0.5745192213521486)],\n                       [(1, 0), (5, 4), (9, 8), (12, 11)],\n                       [(1, 0.7499999999999999, 1.1102230246251565e-16, -0.28964515054927675),\n                        (12, 0.7499999999999999, 1.1102230246251565e-16, -0.28964515054927675),\n                        (5, 0.75, 0.0, 0.8133009508835458),\n                        (9, 0.75, 0.0, -0.7711047688711279),\n                        (0, 0.5, 1.0, 0.5),\n                        (4, 0.49999999999999967, 0.9999999999999999, 0.6607164282557103),\n                        (8, 0.49999999999999956, 1.0, 0.05430924286432287),\n                        (11, 0.5, 1.0, 0.5)],\n                       [(2, 1), (13, 12), (5, 6), (9, 10)],\n                       [(1, 0.5, 0.0, -0.28964515054927653),\n                        (5, 0.500526889421938, 0.0, -0.6872259852436184),\n                        (9, 0.5005268894219381, 0.0, -0.2716317049982926),\n                        (12, 0.5, 0.0, -0.28964515054927653),\n                        (2, 0.5, 0.0, -0.212405959533303),\n                        (13, 0.5, 0.0, -0.027786902195289898),\n                        (6, 0.5000000000000103, 1.0, 0.42548077864785155),\n                        (10, 0.5000000000000103, 1.0, 0.42548077864785155)],\n                       [(1, 0), (5, 4), (9, 8), (12, 11)],\n                       [(1, 0.25000000000000006, 0.0, 0.7103548494507228),\n                        (12, 0.25000000000000006, 0.0, 0.7103548494507228),\n                        (5, 0.5, 1.1102230246251565e-16, -0.43722642131705136),\n                        (9, 0.5, 0.0, -0.021632141071725064),\n                        (4, 0.5000000000000004, 1.0, -0.8392835717442888),\n                        (8, 0.5000000000000004, 0.9999999999999998, 0.5543092428643233)],\n                       [(2, 1), (13, 12), (5, 6), (9, 10)],\n                       [(1, 0.5, 0.0, 0.710354849450723),\n                        (12, 0.5, 0.0, 0.710354849450723),\n                        (5, 4),\n                        (9, 8),\n                        (6, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (10, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (2, 0.5, -0.28759404046669745, -0.212405959533303),\n                        (13, 0.5, -0.47221309780470944, -0.02778690219529012)],\n                       [(1, 0),\n                        (12, 11),\n                        (4, 0.9992548329781763, -0.38693091582765193, 0.6607164282557028),\n                        (8, 0.9992548329781763, 0.21947626956373595, 0.05430924286431488),\n                        (5, 3.251517832758566e-08, 0.7110119337450929, 0.5627735560174607),\n                        (9, 3.251517832758566e-08, 0.2954176534997668, 0.9783678362627866)],\n                       [(0, 1.0, 0.0, 0.0),\n                        (11, 1.0, 0.0, 0.0),\n                        (1, 0.4999999999999998, -0.21035484945072325, -0.28964515054927675),\n                        (12, 0.4999999999999998, -0.21035484945072325, -0.28964515054927675)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],\n           \'qubits\': [(2, 5),\n                      (3, 5),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 4),\n                      (6, 5),\n                      (6, 6)]},\n (14, 5): {\'method\': \'joint small decoder cores + parallel three-site leaves\',\n           \'moments\': [[(3, 0.37780996597847644, 0.0, 0.5),\n                        (4, 0.6649284327846141, 1.0, -0.5),\n                        (5, 0.20369585066947749, 0.0, 0.5),\n                        (9, 0.9999404304359252, 0.0, -0.5),\n                        (2, 0.49999999999999895, 0.0, -0.7088499809721285),\n                        (12, 0.4999999999999989, 1.0, -0.7088499809721285),\n                        (1, 0.48497757661839347, 0.0, -0.834800922923292),\n                        (11, 0.48497757661839347, 0.0, -0.834800922923292),\n                        (0, 0.5, 0.0, 0.5),\n                        (6, 0.30408672398469655, 0.0, -0.5745192213521528),\n                        (13, 0.30408672398469655, 0.0, -0.5745192213521528),\n                        (10, 0.30408672398469655, 1.0, -0.5745192213521528)],\n                       [(3, 7), (4, 8), (5, 9)],\n                       [(3, 0.9765810720440469, 0.0, 0.5),\n                        (4, 0.9297740209165868, 2.220446049250313e-16, -0.5000000000000002),\n                        (7, 0.37339124773516286, 0.0, 0.5),\n                        (5, 0.5740941455895439, 0.0, 0.5),\n                        (9, 0.9999259074782365, 1.0, 0.5)],\n                       [(3, 4), (7, 8)],\n                       [(4, 0.9999943381762997, 0.0, 0.5),\n                        (7, 0.8733912477351629, 0.0, -0.5000000000000002),\n                        (3, 0.20093748739029382, 0.0, 0.5),\n                        (8, 9)],\n                       [(4, 5), (3, 7), (9, 0.4999890307436661, 0.0, 0.5)],\n                       [(4, 0.9999948690424976, 0.0, -0.5),\n                        (7, 0.5, 0.0, 0.5),\n                        (5, 0.3186839038285761, 0.0, 0.5),\n                        (3, 0.0, 0.9283621511353788, 0.0)],\n                       [(4, 8),\n                        (5, 9),\n                        (3, 0.0, -0.31555683986653227, 0.0),\n                        (7, 0.0, 0.2923037808968796, 0.0)],\n                       [(4, 0.8562952046824337, 0.0, -0.5000000000000002),\n                        (8, 0.4999999999999545, 0.0, 0.5),\n                        (5, 0.49996737843785244, 0.0, -0.5000000000000002),\n                        (9, 0.5000208919712607, 0.0, -0.5),\n                        (3, 0.0, 1.0, 0.0),\n                        (7, 0.0, 1.0, 0.0)],\n                       [(7, 8), (3, 4)],\n                       [(4, 0.2308414232966614, 0.0, 0.5), (8, 0.5, 0.0, 0.5), (3, 7)],\n                       [(4, 5), (8, 9)],\n                       [(5, 0.4999673784378522, 1.0, 0.5), (8, 0.5, 1.0, -0.5), (3, 4)],\n                       [(4, 8), (5, 9)],\n                       [(5, 0.4999944718341389, 0.0, 0.054203502216731314),\n                        (4, 0.41174992307908964, 0.0, 0.47289824875204584),\n                        (7, 8),\n                        (9, 0.500000000000001, 0.0, 0.945796497504092)],\n                       [(3, 4),\n                        (5, 2),\n                        (8, 0.9077847323515802, -0.9999999999999998, -0.4075678311410549),\n                        (9, 12)],\n                       [(4, 0.9675670579641595, 0.0, -0.5271017512479375),\n                        (5, 0.4928050422747947, 0.9999999999999999, 0.8828789665933579),\n                        (7, 8),\n                        (9, 0.4928050422747947, 0.0, 0.11715318286617804),\n                        (2, 0.4608483309798826, -2.220446049250313e-16, 0.8309193012950313),\n                        (12, 0.4608483309798824, 0.9999999999999999, 0.8309193012950313)],\n                       [(4, 5), (8, 0.9675670579641595, 0.9999999999999999, 0.6886477534804571)],\n                       [(4, 0.5000000000000004, 0.0, -0.5271017512479204),\n                        (5, 0.5596506416568943, 1.0, 0.7937817676678398),\n                        (8, 9)],\n                       [(4, 5),\n                        (9, 0.5596506416568943, 0.0, 0.20625038179169602),\n                        (8, 0.5000000000000003, 0.9999999999999999, 0.6886477534804405)],\n                       [(4, 0.1275470395921725, 0.0, 0.6252841056811698),\n                        (5, 0.5000000000000004, 0.0, 0.77268059502778),\n                        (8, 9)],\n                       [(3, 4),\n                        (5, 2),\n                        (9, 0.5000000000000003, 0.0, 0.22735155443175586),\n                        (8, 0.1275470395921725, 0.0, -0.4637381034486503)],\n                       [(4, 0.2696276353165254, 0.0, -0.432360298347084),\n                        (5, 0.16304676742985036, 1.0, -0.22731940497221803),\n                        (9, 12),\n                        (7, 8),\n                        (2, 0.8273677055841601, 0.0, 0.011025144178165858)],\n                       [(4, 5),\n                        (9, 0.16304676742985033, 0.0, -0.7726484455682461),\n                        (8, 0.2696276353165252, 1.0, -0.4060936994203965),\n                        (12, 0.8273677055841601, 1.0, 0.011025144178165858)],\n                       [(4, 0.3174136387017352, 8.881784197001252e-16, -0.9904400570522649),\n                        (5, 0.704332640729217, 0.0, -0.7273194049722176),\n                        (8, 9)],\n                       [(5, 2),\n                        (3, 4),\n                        (8, 0.3174136387017352, -0.9999999999999998, 0.15198605928478398),\n                        (9, 0.704332640729217, 0.0, -0.2726484455682465)],\n                       [(5, 0.5, 1.0, -0.7273194049722171),\n                        (4, 0.6276139104334912, 0.0, 0.9136528331529414),\n                        (7, 8),\n                        (9, 12),\n                        (2, 0.727371054243577, 0.0, -0.07318377315388713)],\n                       [(4, 5),\n                        (8, 0.6276139104334912, 1.0, 0.24789316907957826),\n                        (9, 0.5, 0.0, -0.27264844556824697),\n                        (12, 0.727371054243577, 1.0, -0.07318377315388713)],\n                       [(4, 0.4999999999999989, 0.0, -0.19168089960740842),\n                        (5, 0.529090738553442, 0.0, 0.9032770477588372),\n                        (8, 9)],\n                       [(3, 4),\n                        (5, 2),\n                        (9, 0.529090738553442, 0.0, 0.09675510170069884),\n                        (8, 0.4999999999999989, 1.0, -0.6467730981600723)],\n                       [(5, 0.49999999999999956, 1.0, -0.10321942218778468),\n                        (4, 0.6683089837594859, 0.0, -0.6916808996074089),\n                        (9, 12),\n                        (7, 8),\n                        (3, 0.5000000000000004, 0.9999999999999999, -0.4007455643551321),\n                        (2, 0.49999999999999956, 0.0, -0.09727132543917172)],\n                       [(4, 5),\n                        (9, 0.49999999999999956, 0.0, -0.8967484283526794),\n                        (8, 0.6683089837594859, 1.0, -0.14677309816007167),\n                        (12, 0.49999999999999956, 1.0, -0.09727132543917172),\n                        (7, 0.5000000000000003, 0.0, -0.6941453434791436)],\n                       [(4, 0.6578349175114986, 0.0, -0.6916808996074091), (8, 9), (5, 2)],\n                       [(3, 4),\n                        (2, 0.6530017079301533, -2.220446049250313e-16, 0.40272867456082917),\n                        (8, 0.6578349175114986, 1.0, -0.14677309816007167),\n                        (5, 0.5000000000000009, 1.0, -0.10321942218778579),\n                        (9, 12)],\n                       [(4, 0.5, 0.0, -0.1915511931228595),\n                        (7, 8),\n                        (2, 1),\n                        (12, 0.6530017079301533, -2.220446049250313e-16, 0.40272867456082917),\n                        (9, 0.5000000000000009, 0.0, -0.8967484283526783),\n                        (3, 0.4999999999999998, -0.6128053112688461, 0.5992544356448686)],\n                       [(4, 5),\n                        (8, 0.5, -0.9999999999999998, -0.646902804644621),\n                        (1, 0.5605616196317902, 1.0, -0.8348009229232944),\n                        (2, 0.5000000000000003, 0.0, -0.09727132543918615),\n                        (12, 11),\n                        (7, 0.5000000000000001, -0.3194055321448337, -0.6941453434791438)],\n                       [(4, 0.5, 0.0, -0.1915511931228595),\n                        (5, 2),\n                        (8, 9),\n                        (11, 0.5605616196317902, 0.0, -0.8348009229232944),\n                        (12, 0.5000000000000003, 1.0, -0.09727132543918593)],\n                       [(1, 4),\n                        (5, 0.5, 0.0, 0.13351261161596778),\n                        (2, 0.4311009172299638, 0.0, -0.34731902068777165),\n                        (8, 0.5, 0.0, -0.646902804644621),\n                        (9, 12)],\n                       [(1, 0.5818042231825744, 0.0, -0.3348009229233022),\n                        (4, 5),\n                        (11, 8),\n                        (9, 0.5, 0.0, 0.8665195378435682),\n                        (12, 0.4311009172299638, 0.0, -0.34731902068777165)],\n                       [(4, 0.0, 0.30838251308620035, 0.0),\n                        (2, 1),\n                        (11, 0.5818042231825744, 0.0, -0.3348009229233022),\n                        (5, 0.5, 0.0, 0.13351261161596778),\n                        (8, 9)],\n                       [(4, 0.24304696816073573, 4.440892098500626e-16, 0.5226072883186472),\n                        (1, 0.2894984978675621, 0.0, -0.8348009229233004),\n                        (2, 0.03454957128703273, 0.0, 0.8181166636420727),\n                        (12, 11),\n                        (9, 0.5, 0.0, -0.13348046215643183),\n                        (8, 0.24304696816073568, 0.0, -0.05267877299992785)],\n                       [(4, 5),\n                        (2, 1),\n                        (12, 0.03454957128703273, 0.0, 0.8181166636420727),\n                        (11, 0.2894984978675621, 0.0, -0.8348009229233004),\n                        (8, 9)],\n                       [(4, 0.30568317284628654, 0.0, 0.8750503670007201),\n                        (2, 0.9242869536756131, 8.881784197001252e-16, 0.5858061491772157),\n                        (1, 0.87386354985307, 0.0, 0.6651990770766809),\n                        (12, 11),\n                        (8, 0.30568317284628654, 0.0, 0.5948781483179992)],\n                       [(5, 2),\n                        (1, 4),\n                        (11, 0.87386354985307, 0.9999999999999999, 0.6651990770766807),\n                        (12, 0.9242869536756131, -0.9999999999999998, 0.5858061491772157)],\n                       [(1, 0.6888649043845242, 0.0, -0.8348009229233393),\n                        (4, 0.5595532320705698, 0.0, 0.4571480608144438),\n                        (2, 0.23753846872505302, 0.0, 0.11812387065190871),\n                        (5, 0.5, 0.0, 0.6333829051314186),\n                        (9, 12),\n                        (11, 8)],\n                       [(2, 1),\n                        (4, 5),\n                        (11, 0.6888649043845242, 0.0, -0.8348009229233393),\n                        (9, 0.5, 0.0, 0.3666492443281175),\n                        (8, 0.5595532320705698, 0.0, -0.9872195454957242),\n                        (12, 0.23753846872505302, 0.0, 0.11812387065190871)],\n                       [(5, 0.4999999999999998, 0.0, 0.6333829051314182),\n                        (2, 0.6383222145074643, 0.0, 0.17679703477273234),\n                        (1, 0.49999999999999956, 1.0, -0.8348009229233395),\n                        (8, 9),\n                        (12, 11)],\n                       [(1, 4),\n                        (5, 2),\n                        (9, 0.5, 0.0, -0.6333507556718825),\n                        (11, 0.49999999999999956, 0.0, -0.8348009229233395),\n                        (12, 0.6383222145074643, 0.9999999999999998, 0.17679703477273212),\n                        (8, 0.0, 0.6272925464780552, 0.0)],\n                       [(5, 0.5, 1.0, 0.823807406671633),\n                        (2, 0.4999999999999997, 0.0, -0.7484872360058605),\n                        (1, 0.4546007179287068, 0.0, 0.4167387613319282),\n                        (11, 8),\n                        (9, 12)],\n                       [(4, 5),\n                        (2, 1),\n                        (9, 0.5, 0.0, -0.8237752572120971),\n                        (12, 0.4999999999999997, 0.0, -0.7484872360058605),\n                        (11, 0.4546007179287068, 0.0, 0.4167387613319282),\n                        (8, 2.220446049250313e-16, 1.0, 0.3552591646842296)],\n                       [(1, 0.19504058663172444, 1.0, -0.2857088984950029),\n                        (4, 0.2978230143957523, 0.0, 0.812137408728792),\n                        (8, 9),\n                        (12, 11),\n                        (2, 0.8858944659256832, 0.0, 0.2515127639941405)],\n                       [(1, 4),\n                        (8, 0.2978230143957523, 0.0, 0.28508365306798233),\n                        (12, 0.8858944659256832, -0.9999999999999998, 0.25151276399414063),\n                        (11, 0.1950405866317246, 0.0, -0.2857088984950029),\n                        (5, 2)],\n                       [(1, 0.5271017512479523, 0.0, 0.17396795886480326),\n                        (9, 12),\n                        (11, 8),\n                        (4, 0.5000000000000004, 0.0, -0.3904899386819476),\n                        (5, 0.21266361194798075, 1.0, -0.6761925933283681)],\n                       [(1, 0),\n                        (12, 0.5, 1.1102230246251565e-16, 0.03884915204616013),\n                        (11, 0.5000000000000007, 0.0, 0.6739679588648033),\n                        (8, 0.5000000000000004, -0.9999999999999998, -0.5122889995212778),\n                        (9, 0.21266361194798095, 4.440892098500626e-16, 0.6762247427879033),\n                        (5, 4)],\n                       [(1, 0.5, 0.0, 0.6739679588648031),\n                        (9, 8),\n                        (12, 11),\n                        (5, 0.4999999999999824, 0.0, -0.17619253701042648),\n                        (0, 0.3040867239846962, 0.0, -0.5000000000000002),\n                        (4, 0.5000000000000021, 0.0, 0.5348372852140137)],\n                       [(2, 1),\n                        (9, 0.49999999999998257, -2.220446049250313e-16, 0.1762246864699622),\n                        (12, 0.49999999999998246, -2.220446049250313e-16, 0.038849208364101384),\n                        (5, 6),\n                        (11, 0.5000000000000019, 0.0, -0.2513592650311596),\n                        (8, 0.500000000000002, 1.0, 0.5623837765827611)],\n                       [(1, 0.8040867239846965, 2.220446049250313e-16, -0.32603204113519735),\n                        (9, 10),\n                        (12, 13),\n                        (5, 0.5000000000000102, -0.9999999999999998, -0.1761925370104267),\n                        (6, 0.30408672398469627, 0.0, -0.5745192213521484)],\n                       [(1, 0),\n                        (9, 0.5000000000000105, 0.0, 0.17622468646996214),\n                        (12, 0.5000000000000103, 0.0, 0.03884920836410144),\n                        (5, 4),\n                        (10, 0.30408672398469616, 1.0, -0.5745192213521486),\n                        (13, 0.30408672398469627, 0.0, -0.5745192213521484)],\n                       [(1, 0.75, -0.9999999999999998, -0.3260320411351971),\n                        (9, 8),\n                        (12, 11),\n                        (0, 0.5, 0.0, 0.5),\n                        (5, 0.75, 0.0, 0.8238074955047525),\n                        (4, 0.4999999999999995, 0.0, 0.5348372526988321)],\n                       [(2, 1),\n                        (12, 0.75, 4.440892098500626e-16, -0.9611507591207205),\n                        (9, 0.75, 0.0, -0.8237753460452171),\n                        (5, 6),\n                        (8, 0.49999999999999956, 1.0, 0.5623838090979425),\n                        (11, 0.49999999999999956, 0.0, -0.2513592325159779)],\n                       [(1, 0.5000000000000001, 0.0, 0.6739679588648031),\n                        (12, 13),\n                        (9, 10),\n                        (5, 0.500526889421938, -0.9999999999999998, 0.32433443163191633),\n                        (2, 0.5, 0.0, -0.23493636038188193),\n                        (6, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155)],\n                       [(1, 0),\n                        (9, 0.5005268894219381, 0.0, -0.3243022821723809),\n                        (12, 0.500526889421938, 0.0, 0.5393761770064442),\n                        (5, 4),\n                        (10, 0.5000000000000103, 1.0, 0.4254807786478516),\n                        (13, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155)],\n                       [(1, 0.25, 1.0, -0.3260320411351971),\n                        (12, 11),\n                        (9, 8),\n                        (4, 0.5000000000000004, 0.0, 0.03483725269883159),\n                        (5, 0.5, 5.551115123125783e-17, 0.0743348677053492)],\n                       [(2, 1),\n                        (11, 0.5000000000000004, 0.0, 0.24864076748402253),\n                        (8, 0.5000000000000004, 1.0, -0.9376161909020573),\n                        (12, 0.5, 0.0, 0.28937661307987694),\n                        (9, 0.5, 0.0, -0.0743027182458138),\n                        (5, 4)],\n                       [(1, 0.5, 1.0, 0.6739679588648031),\n                        (5, 1.021405182655144e-14, 1.0, 0.5743277767945238),\n                        (9, 8),\n                        (12, 13),\n                        (4, 0.0007451670218237008, -0.26105174027078903, -0.4651627473011599),\n                        (2, 0.5, 0.7349363603818819, 0.7650636396181181)],\n                       [(1, 0),\n                        (5, 6),\n                        (12, 11),\n                        (9, 10),\n                        (8, 0.9992548329781763, -0.28859829666988346, 0.5623838090979341),\n                        (13, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(10, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (12, 0.999999967484822, -0.01559114598281175, 0.2893766357453651),\n                        (1, 0.5, 0.8260320411351971, -0.3260320411351971),\n                        (5, 0.999999967484822, 0.19945059939171644, 0.07433489037083735),\n                        (0, 1.0, 0.0, 0.5),\n                        (6, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (11, 0.9992548329781763, 0.5251447449440367, -0.2513592325159859),\n                        (9, 3.251517832758566e-08, 0.3480882306738562, 0.9256972590886976)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],\n           \'qubits\': [(2, 5),\n                      (3, 5),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 5),\n                      (6, 6),\n                      (6, 7)]},\n (14, 6): {\'method\': \'joint small decoder cores + parallel three-site leaves\',\n           \'moments\': [[(1, 0.4630759433243212, 0.0, -0.5),\n                        (5, 0.48527820242856157, 0.0, -0.5),\n                        (2, 0.498691251687155, 0.0, -0.5),\n                        (6, 0.5034714935310124, 0.0, -0.5),\n                        (9, 0.33945009469785664, 0.0, -0.5),\n                        (10, 0.5407603083963027, 0.0, -0.5),\n                        (8, 0.5000000000000003, 0.8675854111550219, -0.20251959709327183),\n                        (4, 0.5, 0.7692803243874866, -0.377500967254607),\n                        (0, 0.5, 0.0, 0.5),\n                        (12, 0.30408672398469644, 0.48537839930289595, -0.5745192213521528),\n                        (11, 0.5000000000000003, 0.8675854111550219, -0.20251959709327183),\n                        (7, 0.5, 0.7692803243874866, -0.377500967254607),\n                        (3, 0.5, 0.0, 0.5),\n                        (13, 0.30408672398469644, 0.48537839930289595, -0.5745192213521528)],\n                       [(1, 5), (2, 6), (9, 10)],\n                       [(5, 0.3789124762180883, 0.0, 0.5),\n                        (9, 0.4058177292404212, 0.0, -0.5000000000000002),\n                        (6, 0.11475317333590987, 0.0, -0.5000000000000002),\n                        (10, 0.8510324075527451, 0.0, 0.5),\n                        (1, 0.6202325463217212, 0.0, 0.5),\n                        (2, 0.614760310033405, 5.551115123125783e-17, 0.49999999999999994)],\n                       [(5, 9), (6, 10)],\n                       [(5, 0.5, 0.0, -0.5),\n                        (6, 1.0, 0.0, 0.5),\n                        (9, 0.03453719968213479, 0.0, 0.5),\n                        (10, 0.628715121481435, 0.0, 0.5)],\n                       [(5, 6), (9, 10)],\n                       [(5, 0.5000000000000001, 0.0, 0.5),\n                        (6, 0.7568820983434257, 0.0, -0.5),\n                        (9, 0.1129617833785391, 0.0, -0.5000000000000002),\n                        (10, 0.6389837719170549, 0.0, -0.5)],\n                       [(1, 5), (2, 6)],\n                       [(5, 0.8697603441988186, 0.0, 0.5),\n                        (6, 0.7568820983434255, 0.0, 0.5),\n                        (1, 0.5, 0.0, -0.5),\n                        (2, 0.9999999999999999, 0.0, -0.5)],\n                       [(5, 9), (6, 10)],\n                       [(5, 0.9947413319518239, 0.0, 0.5),\n                        (6, 0.49999999999999983, 0.0, 0.5),\n                        (9, 0.3527928756243648, 0.0, -0.5),\n                        (10, 0.48152990442961807, 0.0, 0.5)],\n                       [(5, 6), (9, 10)],\n                       [(5, 0.810510574569716, 0.0, 0.5),\n                        (6, 0.25255329759659245, 0.0, -0.5),\n                        (9, 0.5337035612733155, 0.0, 0.5),\n                        (10, 0.038141509743885645, 0.0, 0.5)],\n                       [(1, 5), (2, 6)],\n                       [(5, 0.8105105745697158, 0.0, 0.5),\n                        (6, 0.7525532975965926, 0.0, -0.5),\n                        (1, 1.824988140266864e-16, 0.0783923935738624, 0.5101750262253448),\n                        (2, 1.9242955340157506e-16, 0.0783923935738624, 0.5096497568920015)],\n                       [(5, 9), (6, 10)],\n                       [(5, 0.5000000000000007, -0.22938550051157458, -0.6802361139219901),\n                        (9, 0.14432012387548546, -0.36186927236083233, -0.07947937457098897),\n                        (6, 0.49999999999999917, -0.5898577283555546, 0.6802361139219899),\n                        (10, 0.8596950642527911, 0.7148510069782518, -0.03240855509554219)],\n                       [(1, 5), (9, 8), (2, 6), (10, 11)],\n                       [(5, 0.9999999999999997, 0.8635656639623304, 0.7449327619111934),\n                        (9, 0.6822968314790298, -0.054913022763340535, 0.1597930542139565),\n                        (1, 0.49999999999999967, -0.36414832513253415, 0.08822007796215076),\n                        (8, 0.25093221684773165, 0.20508143362291859, 0.07265639204621782),\n                        (6, 0.9999999999999997, 0.8635656639623304, 0.7449327619111934),\n                        (10, 0.6822968314790298, -0.054913022763340535, 0.1597930542139565),\n                        (2, 0.49999999999999967, -0.36414832513253415, 0.08822007796215076),\n                        (11, 0.25093221684773165, 0.20508143362291859, 0.07265639204621782)],\n                       [(5, 9), (6, 10)],\n                       [(5, 0.5728573590437842, 0.5670764233438972, -0.1852609831374603),\n                        (9, 0.5, 0.24757260102301026, -0.11736766662282938),\n                        (6, 0.5728573590437842, 0.5670764233438972, -0.1852609831374603),\n                        (10, 0.5, 0.24757260102301026, -0.11736766662282938)],\n                       [(1, 5), (9, 8), (2, 6), (10, 11)],\n                       [(1, 0.4178598011240008, 0.3150889099927735, -0.7759282471703832),\n                        (5, 1.631781156164555e-17, -0.5873919823346907, -0.430570163551661),\n                        (9, 0.2673990199988757, -0.33365272490222253, -0.8697950655998195),\n                        (8, 0.5826592798249117, 0.605774368663364, -0.2967718109189419),\n                        (2, 0.4178598011240008, 0.3150889099927735, -0.7759282471703832),\n                        (6, 1.631781156164555e-17, -0.5873919823346907, -0.430570163551661),\n                        (10, 0.2673990199988757, -0.33365272490222253, -0.8697950655998195),\n                        (11, 0.5826592798249117, 0.605774368663364, -0.2967718109189419)],\n                       [(1, 5), (9, 8), (2, 6), (10, 11)],\n                       [(5, 0.3856735769590187, 0.2125495643690316, 0.928194027274869),\n                        (9, 0.5505347324801628, -0.6988877069042307, -0.703447790502042),\n                        (1, 0.49999999999999983, 0.32399997770662925, 0.0391606628223905),\n                        (8, 0.3689376239568078, 0.8719705090524066, 0.45041832195784437),\n                        (6, 0.3856735769590187, 0.2125495643690316, 0.928194027274869),\n                        (10, 0.5505347324801628, -0.6988877069042307, -0.703447790502042),\n                        (2, 0.49999999999999983, 0.32399997770662925, 0.0391606628223905),\n                        (11, 0.3689376239568078, 0.8719705090524066, 0.45041832195784437)],\n                       [(5, 9), (6, 10)],\n                       [(5, 0.4633349244593254, -0.9212146746287528, 0.14280728474232607),\n                        (9, 0.3213111929430337, -0.8052598942752667, -0.9023354974062725),\n                        (6, 0.4633349244593254, -0.9212146746287528, 0.14280728474232607),\n                        (10, 0.3213111929430337, -0.8052598942752667, -0.9023354974062725)],\n                       [(5, 9), (6, 10)],\n                       [(5, 0.4372106511952734, -0.08072591754993308, 0.05721678538600655),\n                        (9, 0.5666568155244441, -0.9873674527150209, 0.792404608318461),\n                        (6, 0.4372106511952734, -0.08072591754993308, 0.05721678538600655),\n                        (10, 0.5666568155244441, -0.9873674527150209, 0.792404608318461)],\n                       [(1, 5), (9, 8), (2, 6), (10, 11)],\n                       [(5, 0.10399035756228481, 0.6080530598469465, -0.4226079210641438),\n                        (9, 0.58504255233184, 0.20502529185508278, -0.6949628443965601),\n                        (1, 0.5, -0.2877545940293209, -0.28689361812169745),\n                        (8, 0.8311737009895827, -0.5234252742454957, 0.4662977892599787),\n                        (6, 0.10399035756228481, 0.6080530598469465, -0.4226079210641438),\n                        (10, 0.58504255233184, 0.20502529185508278, -0.6949628443965601),\n                        (2, 0.5, -0.2877545940293209, -0.28689361812169745),\n                        (11, 0.8311737009895827, -0.5234252742454957, 0.4662977892599787)],\n                       [(5, 9), (6, 10)],\n                       [(5, 0.4341535955963661, 0.10358814073407496, -0.04622953235841276),\n                        (9, 0.49999999999999956, -0.42257784161141076, -0.48993755254147686),\n                        (6, 0.4341535955963661, 0.10358814073407496, -0.04622953235841276),\n                        (10, 0.49999999999999956, -0.42257784161141076, -0.48993755254147686)],\n                       [(1, 5), (9, 8), (2, 6), (10, 11)],\n                       [(5, 0.4999999999999998, -0.4637680809992448, 0.253475323236589),\n                        (9, 0.5998657277943454, -0.34070259606769393, 0.1873084193903226),\n                        (1, 0.5530742553856173, 0.40067828510028647, -0.07464821215101836),\n                        (8, 0.9999999999999994, -0.9295534250454451, 0.7370827701575497),\n                        (6, 0.4999999999999998, -0.4637680809992448, 0.253475323236589),\n                        (10, 0.5998657277943454, -0.34070259606769393, 0.1873084193903226),\n                        (2, 0.5530742553856173, 0.40067828510028647, -0.07464821215101836),\n                        (11, 0.9999999999999994, -0.9295534250454451, 0.7370827701575497)],\n                       [(5, 9), (6, 10)],\n                       [(5, 0.5000000000000004, 0.8390472565955931, -0.7102927577626557),\n                        (9, 1.2425286741276152e-16, -0.023781567175741092, 0.9210343143636188),\n                        (6, 0.5000000000000004, 0.8390472565955931, -0.7102927577626557),\n                        (10, 1.2425286741276152e-16, -0.023781567175741092, 0.9210343143636188)],\n                       [(1, 5), (9, 8), (2, 6), (10, 11)],\n                       [(5, 0.5000000000000001, 0.9333264666852414, 0.1287544988329375),\n                        (9, 0.5000000000000001, -0.7027147829846936, -0.3527042894436394),\n                        (8, 0.6339658219122923, 0.5865910761309128, -0.3475432792648183),\n                        (6, 0.5000000000000001, 0.9333264666852414, 0.1287544988329375),\n                        (10, 0.5000000000000001, -0.7027147829846936, -0.3527042894436394),\n                        (11, 0.6339658219122923, 0.5865910761309128, -0.3475432792648183),\n                        (1, 0.5000000000000001, 0.01408798401172695, 0.3260300729492681),\n                        (2, 0.5000000000000001, 0.01408798401172695, 0.3260300729492681)],\n                       [(5, 9), (8, 4), (6, 10), (11, 7)],\n                       [(9, 0.5000000000000001, 0.5719198806385787, 0.944580927571667),\n                        (8, 0.9068032572794498, 0.8355238991300334, 0.8728951003320184),\n                        (4, 0.035921075512788087, 0.3723953747889677, -0.10822064286712019),\n                        (5, 0.5991006024208338, 0.5950763379420172, -0.7757192247142704),\n                        (10, 0.5000000000000001, 0.5719198806385787, 0.944580927571667),\n                        (11, 0.9068032572794498, 0.8355238991300334, 0.8728951003320184),\n                        (7, 0.035921075512788087, 0.3723953747889677, -0.10822064286712019),\n                        (6, 0.5991006024208338, 0.5950763379420172, -0.7757192247142704)],\n                       [(9, 8), (4, 5), (10, 11), (7, 6)],\n                       [(5, 0.6983902210776689, -0.3199316016708491, -0.3922285997577968),\n                        (9, 0.9999999999999999, -0.12119478199392009, 0.5210008769771368),\n                        (8, 0.4457229604434564, -0.5208884030065459, -0.07287381382803004),\n                        (4, 0.751811597304503, 0.25358158955187293, -0.23582526807815296),\n                        (6, 0.6983902210776689, -0.3199316016708491, -0.3922285997577968),\n                        (10, 0.9999999999999999, -0.12119478199392009, 0.5210008769771368),\n                        (11, 0.4457229604434564, -0.5208884030065459, -0.07287381382803004),\n                        (7, 0.751811597304503, 0.25358158955187293, -0.23582526807815296)],\n                       [(5, 9), (8, 4), (6, 10), (11, 7)],\n                       [(9, 1.0, 0.0, 0.237031122618504),\n                        (8, 0.7856407379834287, 0.33094556631565303, -0.1674398139457436),\n                        (4, 0.5, 0.8704489163755462, -0.9822436785262807),\n                        (5, 0.6251415065033563, 0.5382515398059069, 0.14022836429623164),\n                        (10, 1.0, 0.0, 0.237031122618504),\n                        (11, 0.7856407379834287, 0.33094556631565303, -0.1674398139457436),\n                        (7, 0.5, 0.8704489163755462, -0.9822436785262807),\n                        (6, 0.6251415065033563, 0.5382515398059069, 0.14022836429623164)],\n                       [(9, 8), (4, 5), (10, 11), (7, 6)],\n                       [(5, 0.9999999999999999, -0.05478476838688007, 0.2728835889646527),\n                        (9, 0.5, -0.2412912313636819, 0.06975608148690038),\n                        (8, 0.4949978217082434, -0.2949680894494069, -0.3364942476300903),\n                        (4, 0.4999999999999998, 0.8377226895572165, -0.6157082354705663),\n                        (6, 0.9999999999999999, -0.05478476838688007, 0.2728835889646527),\n                        (10, 0.5, -0.2412912313636819, 0.06975608148690038),\n                        (11, 0.4949978217082434, -0.2949680894494069, -0.3364942476300903),\n                        (7, 0.4999999999999998, 0.8377226895572165, -0.6157082354705663)],\n                       [(5, 9), (8, 4), (6, 10), (11, 7)],\n                       [(9, 0.5000000000000001, 0.6198553350083378, 0.8284648501232186),\n                        (8, 0.49999999999999983, 0.3764989672509906, -0.09762890551798042),\n                        (4, 0.9999999999999997, 0.8731333921363599, -0.9843865650463126),\n                        (5, 0.9999999999999999, -0.33018465506861583, 0.7464835036541473),\n                        (10, 0.5000000000000001, 0.6198553350083378, 0.8284648501232186),\n                        (11, 0.49999999999999983, 0.3764989672509906, -0.09762890551798042),\n                        (7, 0.9999999999999997, 0.8731333921363599, -0.9843865650463126),\n                        (6, 0.9999999999999999, -0.33018465506861583, 0.7464835036541473)],\n                       [(9, 8), (4, 5), (10, 11), (7, 6)],\n                       [(5, 1.5623957812230537e-17, -0.14022989345186399, 0.26523026428933805),\n                        (9, 0.5, 0.18447894841164114, 0.33237934579314077),\n                        (8, 0.8015319629186126, -0.02961982028751242, 0.2788700617330103),\n                        (4, 0.2887068963335278, -0.3396667872155408, -0.09873965642856897),\n                        (6, 1.5623957812230537e-17, -0.14022989345186399, 0.26523026428933805),\n                        (10, 0.5, 0.18447894841164114, 0.33237934579314077),\n                        (11, 0.8015319629186126, -0.02961982028751242, 0.2788700617330103),\n                        (7, 0.2887068963335278, -0.3396667872155408, -0.09873965642856897)],\n                       [(5, 9), (8, 4), (6, 10), (11, 7)],\n                       [(9, 3.422139113158884e-17, 0.07169782298307648, -0.21328565868549187),\n                        (8, 0.4999999999999993, -0.3630260765874369, -0.2507497585545022),\n                        (4, 0.10822807679614646, 0.8646990468903515, 0.912444371458655),\n                        (5, 0.5000000000000003, 0.6151538833113529, -0.03898869098856084),\n                        (10, 3.422139113158884e-17, 0.07169782298307648, -0.21328565868549187),\n                        (11, 0.4999999999999993, -0.3630260765874369, -0.2507497585545022),\n                        (7, 0.10822807679614646, 0.8646990468903515, 0.912444371458655),\n                        (6, 0.5000000000000003, 0.6151538833113529, -0.03898869098856084)],\n                       [(9, 8), (4, 5), (10, 11), (7, 6)],\n                       [(4, 0.18023611392199, -0.7771434183490045, 0.7771434183490049),\n                        (8, 2.8910412578472565e-16, 0.9538938921029343, -0.5969468920364949),\n                        (9, 0.5663325445329528, 0.0898147218663814, -0.9114438828121421),\n                        (5, 0.4999999999999998, 0.8327683210327581, -0.4238348076772074),\n                        (7, 0.18023611392199, -0.7771434183490045, 0.7771434183490049),\n                        (11, 2.8910412578472565e-16, 0.9538938921029343, -0.5969468920364949),\n                        (10, 0.5663325445329528, 0.0898147218663814, -0.9114438828121421),\n                        (6, 0.4999999999999998, 0.8327683210327581, -0.4238348076772074)],\n                       [(4, 0), (9, 5), (7, 3), (10, 6)],\n                       [(4, 0.49999999999999983, 0.0, -0.5),\n                        (0, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (9, 0.4999999999999824, -0.29919772885013596, -0.32162910462782035),\n                        (5, 0.500000000000002, 0.3758195456300396, -0.2237315757103544),\n                        (7, 0.49999999999999983, 0.0, -0.5),\n                        (3, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (10, 0.4999999999999824, -0.29919772885013596, -0.32162910462782035),\n                        (6, 0.500000000000002, 0.3758195456300396, -0.2237315757103544)],\n                       [(8, 4), (9, 12), (11, 7), (10, 13)],\n                       [(4, 0.8040867239846964, 0.0, 0.5),\n                        (8, 1.0, 0.0, 0.5),\n                        (9, 0.5000000000000104, 0.7374333560946903, -0.6208268334779563),\n                        (12, 0.3040867239846962, 0.6612280619162106, -0.08914082204925244),\n                        (7, 0.8040867239846964, 0.0, 0.5),\n                        (11, 1.0, 0.0, 0.5),\n                        (10, 0.5000000000000104, 0.7374333560946903, -0.6208268334779563),\n                        (13, 0.3040867239846962, 0.6612280619162106, -0.08914082204925244)],\n                       [(4, 0), (9, 5), (7, 3), (10, 6)],\n                       [(4, 0.75, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (9, 0.75, -0.6782507102448094, -0.8833934448680876),\n                        (5, 0.4999999999999995, 0.12674214319260602, 0.15208800243486692),\n                        (7, 0.75, 0.0, 0.5),\n                        (3, 0.5, 0.0, 0.5),\n                        (10, 0.75, -0.6782507102448094, -0.8833934448680876),\n                        (6, 0.4999999999999995, 0.12674214319260602, 0.15208800243486692)],\n                       [(8, 4), (9, 12), (11, 7), (10, 13)],\n                       [(4, 0.5000000000000001, 0.0, -0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (9, 0.5005268894219383, 0.7511005823069626, -0.061117218985732524),\n                        (12, 0.5000000000000102, 0.2800353060019817, -0.4279127601330417),\n                        (7, 0.5000000000000001, 0.0, -0.5),\n                        (11, 0.5, 0.0, 0.5),\n                        (10, 0.5005268894219383, 0.7511005823069626, -0.061117218985732524),\n                        (13, 0.5000000000000102, 0.2800353060019817, -0.4279127601330417)],\n                       [(4, 0), (9, 5), (7, 3), (10, 6)],\n                       [(4, 0.25000000000000006, 0.0, 0.5),\n                        (0, 1.0, 0.0, 0.5),\n                        (9, 0.5000000000000001, -0.3275158685726778, 0.43998379939466253),\n                        (5, 0.5000000000000003, 0.7455961598985248, 0.7788301456274731),\n                        (7, 0.25000000000000006, 0.0, 0.5),\n                        (3, 1.0, 0.0, 0.5),\n                        (10, 0.5000000000000001, -0.3275158685726778, 0.43998379939466253),\n                        (6, 0.5000000000000003, 0.7455961598985248, 0.7788301456274731)],\n                       [(8, 4), (9, 12), (11, 7), (10, 13)],\n                       [(4, 0.5, 0.0, -0.5),\n                        (9, 1.0308214162014585e-14, -0.4910261758928436, 0.61246083991116),\n                        (7, 0.5, 0.0, -0.5),\n                        (10, 1.0308214162014585e-14, -0.4910261758928436, 0.61246083991116),\n                        (8, 0.5, 0.0, 0.5),\n                        (12, 0.5000000000000001, -0.07833700092571938, -0.14787748664623868),\n                        (11, 0.5, 0.0, 0.5),\n                        (13, 0.5000000000000001, -0.07833700092571938, -0.14787748664623868)],\n                       [(4, 0), (9, 5), (7, 3), (10, 6)],\n                       [(4, 0.5000000000000001, 0.0, 0.5),\n                        (9, 0.9999999674848218, 0.6523437121679241, -0.3785582224053705),\n                        (5, 0.9992548329781764, -0.7506407930979386, -0.9755736944740105),\n                        (7, 0.5000000000000001, 0.0, 0.5),\n                        (10, 0.9999999674848218, 0.6523437121679241, -0.3785582224053705),\n                        (6, 0.9992548329781764, -0.7506407930979386, -0.9755736944740105)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (3, 6),\n                      (3, 7),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 5),\n                      (6, 6)]},\n (14, 7): {\'method\': \'joint small decoder cores + parallel three-site leaves\',\n           \'moments\': [[(1, 0.5, 0.0, -0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (5, 0.5013485215354914, 0.0, -0.5),\n                        (8, 0.8238567956974769, 0.0, -0.5),\n                        (9, 0.8615786894865352, 0.0, -0.5),\n                        (7, 0.5, 1.0, -0.20251771408753827),\n                        (10, 0.5, 1.0, -0.20251771408753827),\n                        (3, 0.49999999999999944, 1.0, -0.3649528224832399),\n                        (6, 0.49999999999999944, 1.0, -0.3649528224832399),\n                        (0, 0.5, 1.0, 0.5),\n                        (12, 0.3040867239846966, 1.0, -0.5745192213521528),\n                        (13, 0.3040867239846966, 1.0, -0.5745192213521528),\n                        (11, 0.3040867239846966, 1.0, -0.5745192213521528)],\n                       [(1, 4), (2, 5), (8, 9)],\n                       [(4, 0.8891976217454292, 0.0, 0.5),\n                        (8, 0.8631826712147866, 0.0, -0.5),\n                        (5, 0.5381346275804277, 2.220446049250313e-16, -0.5000000000000002),\n                        (9, 0.26852113390236, 0.0, -0.5),\n                        (1, 0.24294269698947368, 0.0, 0.5),\n                        (2, 0.4999999999999998, 0.0, -0.5000000000000002)],\n                       [(4, 8), (5, 9)],\n                       [(4, 0.11558148270486401, 0.0, -0.5000000000000002),\n                        (5, 0.5, 0.0, -0.5),\n                        (8, 0.5, 0.0, -0.5),\n                        (9, 0.7024793545619903, 0.0, -0.5)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.8168108984528897, 0.0, -0.5), (5, 0.5, 0.0, -0.5), (9, 0.5, 0.0, 0.5)],\n                       [(1, 4), (2, 5)],\n                       [(4, 8), (5, 0.5, 0.0, 0.5), (2, 0.4999999999999998, 0.0, -0.5000000000000002)],\n                       [(5, 9), (4, 0.8587961207350768, 0.0, 0.5), (8, 0.5903297495525045, 0.0, 0.5)],\n                       [(5, 0.5, 1.0, 0.5), (9, 0.5, 1.0, 0.5)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.7076353249849605, 0.0, 0.5), (5, 0.5, 0.0, -0.5)],\n                       [(1, 4), (2, 5)],\n                       [(4, 8),\n                        (5, 0.5, 1.0, -0.5),\n                        (1, 0.10218774830679879, 0.0, 0.5),\n                        (2, 0.06035434377471916, 0.0, 0.5)],\n                       [(5, 9),\n                        (4, 0.23681837113311294, 0.0, 0.7911607475988989),\n                        (8, 0.500000000000002, 0.0, -0.6393986262743319)],\n                       [(1, 4),\n                        (8, 7),\n                        (5, 0.778361677972244, 0.0, -0.8155004040230025),\n                        (9, 0.13939862627433183, 0.0, -0.9999999999999949)],\n                       [(2, 5),\n                        (9, 10),\n                        (4, 0.0, 0.2638235977106111, 0.0),\n                        (8, 0.40250888547893265, 0.0, 0.0458194953410902),\n                        (1, 0.5, 0.0, 0.9877188787148843),\n                        (7, 0.23824984739858224, -0.9999999999999996, 0.04802556671795083)],\n                       [(4, 8),\n                        (9, 0.4025088854789326, 0.0, 0.18521812161542234),\n                        (2, 0.5, 0.9999999999999998, 0.012281121285115215),\n                        (10, 0.23824984739858224, -0.9999999999999996, 0.04802556671795083)],\n                       [(5, 9),\n                        (4, 0.4999999999999993, 1.0, -0.6878117403233299),\n                        (8, 0.5000000000000018, 8.881784197001252e-16, 0.545819495341088)],\n                       [(1, 4),\n                        (8, 7),\n                        (5, 0.4999999999999997, 0.0, -0.5906541367868199),\n                        (9, 0.5000000000000017, 0.0, -0.3147818783845797)],\n                       [(2, 5),\n                        (9, 10),\n                        (1, 0.5000000000000003, 0.0, -0.5122811212851159),\n                        (4, 0.0, 1.0, 0.0),\n                        (8, 0.49999999999999906, 0.0, -0.4541805046589116),\n                        (7, 0.8228387639951986, 1.0, 0.24879131417776962)],\n                       [(1, 4),\n                        (8, 7),\n                        (2, 0.5000000000000004, 0.9999999999999999, -0.4877188787148845),\n                        (9, 0.49999999999999906, 4.440892098500626e-16, 0.6852181216154205),\n                        (10, 0.8228387639951986, 1.0, 0.24879131417776962)],\n                       [(2, 5),\n                        (9, 10),\n                        (4, 0.365320982761632, 1.0, -0.018767313026509136),\n                        (8, 0.4568853839401345, 0.0, -0.9541805046589142),\n                        (1, 0.5, 0.0, 0.9877188787148836),\n                        (7, 0.6082288892935703, 1.0, 0.24879131417776595)],\n                       [(4, 8),\n                        (5, 0.365320982761632, 0.0, 0.7403014359163591),\n                        (9, 0.4568853839401345, 0.0, 0.185218121615418),\n                        (2, 0.5000000000000001, 1.0, 0.012281121285115493),\n                        (10, 0.6082288892935703, 1.0, 0.24879131417776595)],\n                       [(5, 9),\n                        (4, 0.15979316138258426, 1.0, 0.9059060055960126),\n                        (8, 0.5409187240915672, 0.0, 0.5458194953410875)],\n                       [(4, 8),\n                        (5, 0.15979316138258431, 0.0, 0.8156281172938367),\n                        (9, 0.540918724091567, 0.0, 0.6852181216154198)],\n                       [(5, 9),\n                        (4, 0.17667382808391457, 1.0, -0.2437409862059714),\n                        (8, 0.0820855992752405, 0.0, 0.04581949534108709)],\n                       [(1, 4),\n                        (8, 7),\n                        (5, 0.17667382808391455, 0.0, 0.9652751090958214),\n                        (9, 0.0820855992752405, 0.0, -0.814781878384581)],\n                       [(2, 5),\n                        (9, 10),\n                        (4, 0.5000000000000024, 1.0, -0.5722160637211713),\n                        (8, 0.4999999999999983, 0.0, 0.5458194953410869),\n                        (1, 0.5, 0.0, -0.36471258600164846),\n                        (7, 0.2692579739648868, 1.0, -0.7512086858222318)],\n                       [(4, 8),\n                        (5, 0.5000000000000024, 0.0, -0.7062498133889785),\n                        (9, 0.4999999999999983, 0.0, -0.31478187838458105),\n                        (2, 0.5, 1.0, -0.635287413998352),\n                        (10, 0.2692579739648868, 1.0, -0.7512086858222318)],\n                       [(5, 9),\n                        (4, 0.3528413792588168, -0.9999999999999996, -0.5722160637211708),\n                        (8, 0.5000000000000007, 0.0, 0.5458194953410875)],\n                       [(1, 4),\n                        (8, 7),\n                        (5, 0.35284137925881676, 0.0, 0.2937501866110205),\n                        (9, 0.5000000000000003, 0.0, 0.685218121615419)],\n                       [(2, 5),\n                        (9, 10),\n                        (4, 0.5, 1.0, -0.5722160637211744),\n                        (8, 0.2690604471363719, 0.0, -0.9541805046589129),\n                        (1, 0.30168861947752823, 8.881784197001252e-16, -0.8647125860016487)],\n                       [(4, 8),\n                        (5, 0.5, 0.0, 0.2937501866110246),\n                        (9, 0.269060447136372, 0.0, -0.814781878384581),\n                        (2, 0.3016886194775283, 1.0, -0.13528741399835198)],\n                       [(5, 9), (8, 7), (4, 0.5000000000000004, 0.9999999999999998, 0.9277839362788256)],\n                       [(1, 4),\n                        (9, 10),\n                        (5, 0.5000000000000003, 0.0, -0.2062498133889754),\n                        (8, 0.5, 0.0, 0.04581949534108887),\n                        (7, 3)],\n                       [(2, 5),\n                        (4, 0.5, 0.9999999999999998, 0.9277839362788256),\n                        (9, 0.5, 0.0, -0.8147818783845793),\n                        (10, 6),\n                        (3, 0.37658617266632444, 0.0, 0.13504717751675888),\n                        (7, 0.564919347967729, 0.0, -0.8746225131559058),\n                        (1, 0.5000000000000002, -0.7951370705669358, 0.13528741399835237)],\n                       [(4, 8),\n                        (5, 0.5, 0.0, -0.2062498133889754),\n                        (10, 0.564919347967729, 0.0, -0.8746225131559058),\n                        (6, 0.3765861726663244, -0.9999999999999996, 0.13504717751675888),\n                        (2, 0.5, 0.475437757429769, -0.13528741399835242)],\n                       [(5, 9),\n                        (4, 0.5000000000000038, 0.0, -0.07221606372117062),\n                        (8, 0.5, 0.0, 0.04581949534108887)],\n                       [(3, 4),\n                        (8, 7),\n                        (9, 0.5, 0.0, 0.18521812161542078),\n                        (5, 0.5000000000000038, 0.0, -0.20624981338897896)],\n                       [(9, 10),\n                        (6, 5),\n                        (4, 0.6124202497959054, 0.0, -0.9692263187787657),\n                        (8, 0.0, 0.7960999658807593, 0.0),\n                        (7, 0.5000000000000004, -4.440892098500626e-16, -0.3746225131559058),\n                        (3, 0.7664718219180737, 0.9999999999999998, 0.635047177516759)],\n                       [(4, 8),\n                        (7, 3),\n                        (5, 0.6124202497959056, 0.0, 0.690760441668615),\n                        (9, 0.0, -0.4826972866679027, 0.0),\n                        (10, 0.5000000000000004, -4.440892098500626e-16, -0.3746225131559058),\n                        (6, 0.7664718219180737, 0.9999999999999998, 0.635047177516759)],\n                       [(5, 9),\n                        (10, 6),\n                        (3, 0.5000000000000002, 0.0, -0.36495282248324123),\n                        (4, 0.7237608380026965, 0.0, 0.14426333129302504),\n                        (7, 0.8371608839553215, 0.0, 0.17545611282181062)],\n                       [(3, 4),\n                        (8, 7),\n                        (10, 0.8371608839553215, 0.0, 0.17545611282181062),\n                        (6, 0.5000000000000002, 1.0, -0.36495282248324123),\n                        (5, 0.7237608380026965, 0.0, -0.42272920840317596)],\n                       [(9, 10),\n                        (6, 5),\n                        (8, 0.5, 0.0, -0.1580805387781501),\n                        (7, 0.647259141517744, 0.0, 0.3022144361017878),\n                        (3, 0.5000000000000001, 1.0, -0.8649528224832415)],\n                       [(4, 8),\n                        (7, 3),\n                        (5, 0.0, 0.5155754461675257, 0.0),\n                        (9, 0.5, 0.0, -0.29747916505248195),\n                        (10, 0.647259141517744, 0.0, 0.3022144361017878),\n                        (6, 0.5000000000000001, 1.0, -0.8649528224832415)],\n                       [(5, 9),\n                        (10, 6),\n                        (8, 0.5, -2.220446049250313e-16, -0.15808053877815098),\n                        (7, 0.4999999999999998, 0.0, -0.31149386997272455),\n                        (4, 0.0, 0.42309241494576, 0.0)],\n                       [(8, 7),\n                        (3, 4),\n                        (6, 5),\n                        (9, 0.5, 0.0, 0.7025208349475172),\n                        (10, 0.4999999999999998, 0.0, -0.31149386997272455)],\n                       [(9, 10),\n                        (7, 0.6633380081977305, 4.440892098500626e-16, 0.6885061300272752),\n                        (3, 0.5, 0.0, -0.031619489149907754),\n                        (4, 0.0, 1.0, 0.0),\n                        (8, 0.5, 0.0, -0.39585989263665056),\n                        (6, 0.5, 0.0, 0.9683805108500918)],\n                       [(7, 3),\n                        (4, 8),\n                        (9, 0.5, 0.0, 0.4647414810890176),\n                        (10, 0.6633380081977305, 4.440892098500626e-16, 0.6885061300272752)],\n                       [(5, 9),\n                        (10, 6),\n                        (7, 0.5000000000000001, 0.0, 0.18850613002727545),\n                        (3, 0.8248194291522575, 0.0, 0.4683805108500927),\n                        (4, 0.5000000000000004, 0.9999999999999999, -0.07554259123123108)],\n                       [(8, 7),\n                        (3, 4),\n                        (6, 0.8248194291522575, 0.0, -0.5316194891499073),\n                        (5, 0.5000000000000003, 1.1102230246251565e-16, -0.26425542476563413),\n                        (10, 0.5000000000000001, 0.0, 0.18850613002727545)],\n                       [(6, 5),\n                        (9, 10),\n                        (3, 0.18030068686283401, 5.551115123125783e-17, 0.4683805108500939),\n                        (8, 0.5663648310033746, 5.551115123125783e-17, 0.10414010736335022),\n                        (4, 0.5, 1.0, 0.9244574087687691)],\n                       [(3, 0),\n                        (8, 4),\n                        (9, 0.5663648310033746, 0.0, -0.03525851891098153),\n                        (5, 0.5, 0.0, -0.2642554247656346),\n                        (10, 0.5, 0.0, 0.754870961030651),\n                        (6, 0.5000000000000001, 0.0, 0.9683805108500936)],\n                       [(9, 5),\n                        (10, 6),\n                        (3, 0.5, 0.0, 0.9683805108500936),\n                        (8, 0.4999999999999824, -4.440892098500626e-16, -0.39585994895459087),\n                        (0, 0.3040867239846962, 1.0, -0.5000000000000002),\n                        (4, 0.5000000000000019, 1.0, 0.2917277467620205)],\n                       [(7, 3),\n                        (8, 12),\n                        (9, 0.4999999999999824, 0.0, 0.46474142477107705),\n                        (10, 0.49999999999998246, 0.0, -0.24512909528728954),\n                        (5, 0.5000000000000022, 0.0, -0.6315257627588853),\n                        (6, 0.5000000000000022, 0.0, -0.39888982714315757)],\n                       [(9, 13),\n                        (10, 11),\n                        (3, 0.8040867239846965, 0.0, 0.9683805108500936),\n                        (8, 0.5000000000000104, 0.0, -0.3958599489545913),\n                        (7, 0.0, 0.17257469853648622, 0.0),\n                        (12, 0.30408672398469616, 1.0, -0.5745192213521486)],\n                       [(3, 0),\n                        (8, 4),\n                        (9, 0.5000000000000102, 0.0, 0.46474142477107705),\n                        (10, 0.5000000000000102, 3.3306690738754696e-16, -0.24512909528728977),\n                        (13, 0.30408672398469616, 1.0, -0.5745192213521486),\n                        (11, 0.30408672398469616, 1.0, -0.5745192213521486)],\n                       [(9, 5),\n                        (10, 6),\n                        (3, 0.7499999999999999, 0.0, 0.9683805108500936),\n                        (8, 0.75, 0.0, 0.6041400185302308),\n                        (0, 0.5, 1.0, 0.5),\n                        (4, 0.49999999999999956, 1.0, 0.29172777927720167)],\n                       [(7, 3),\n                        (8, 12),\n                        (9, 0.7499999999999999, 5.551115123125783e-17, 0.46474139225589883),\n                        (10, 0.75, 1.1102230246251565e-16, -0.24512912780246965),\n                        (5, 0.49999999999999956, 0.0, 0.3684742047259331),\n                        (6, 0.49999999999999956, 0.0, 0.6011101403416599)],\n                       [(9, 13),\n                        (10, 11),\n                        (3, 0.5, 0.0, -0.03161948914990642),\n                        (8, 0.500526889421938, 4.440892098500626e-16, -0.8963869175969332),\n                        (7, 0.5, -0.9999999999999996, -0.7987688280048215),\n                        (12, 0.5000000000000103, 1.0, 0.42548077864785155)],\n                       [(3, 0),\n                        (8, 4),\n                        (9, 0.500526889421938, 0.0, 0.9642144561287349),\n                        (10, 0.5005268894219383, 0.0, 0.25434393607036654),\n                        (13, 0.5000000000000103, 1.0, 0.42548077864785155),\n                        (11, 0.5000000000000103, 1.0, 0.42548077864785155)],\n                       [(9, 5),\n                        (10, 6),\n                        (3, 0.2500000000000001, 0.0, 0.9683805108500934),\n                        (8, 0.5, 0.0, -0.6463873536703666),\n                        (4, 0.5000000000000003, 0.9999999999999999, 0.7917277792772022)],\n                       [(7, 3),\n                        (8, 12),\n                        (9, 0.5, 0.0, 0.21421402005530155),\n                        (10, 0.5, 0.0, -0.4956565000030668),\n                        (5, 0.5000000000000003, 0.0, 0.8684742047259331),\n                        (6, 0.5000000000000003, 8.881784197001252e-16, -0.8988898596583399)],\n                       [(9, 13),\n                        (10, 11),\n                        (3, 0.5, 0.0, -0.03161948914990642),\n                        (8, 4),\n                        (7, 0.5, -0.7012311719951787, -0.7987688280048215),\n                        (12, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(3, 0),\n                        (9, 5),\n                        (10, 6),\n                        (11, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (13, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (4, 0.9992548329781763, -0.01794226684914335, 0.2917277792771939),\n                        (8, 3.251517832758566e-08, 0.9201728660984068, 0.35361262366414614)],\n                       [(0, 1.0, 0.0, 0.0),\n                        (3, 0.5, 0.5316194891499064, 0.9683805108500934),\n                        (5, 0.0007451670218237004, -0.09468869229788868, -0.6315257952740596),\n                        (6, 0.0007451670218237004, -0.32732462791361616, -0.39888985965833257),\n                        (9, 3.251517832758566e-08, -0.9404285076272605, 0.2142139973898144),\n                        (10, 3.251517832758566e-08, -0.23055798756889212, -0.49565652266855453)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8),\n                      (6, 5),\n                      (6, 6)]},\n (15, 5): {\'method\': \'joint small decoder cores + parallel three-site leaves\',\n           \'moments\': [[(1, 0.5630190521715606, 0.0, 0.5),\n                        (3, 0.45753902280850967, 0.0, -0.5),\n                        (5, 0.5005458189976442, 0.0, 0.5),\n                        (4, 0.5025848644811362, 0.0, 0.5),\n                        (8, 0.14099072666039333, 0.0, -0.5000000000000002),\n                        (6, 0.49999999999999895, 0.0, -0.7088499809721285),\n                        (13, 0.5, 0.0, -0.20307004319295552),\n                        (2, 0.48497757661839347, 0.0, -0.834800922923292),\n                        (0, 0.5, 0.0, 0.5),\n                        (12, 0.49999999999999967, 1.1102230246251565e-16, -0.3779609025165147),\n                        (11, 0.3040867239846966, 0.0, -0.5745192213521528),\n                        (10, 0.3040867239846966, 0.0, -0.5745192213521528),\n                        (14, 0.3040867239846966, 0.0, -0.5745192213521528)],\n                       [(1, 3), (5, 4), (7, 8)],\n                       [(3, 0.6905771906117342, 0.0, 0.5),\n                        (7, 0.4191369997162821, 0.0, -0.5000000000000002),\n                        (4, 0.9017154857987535, 0.0, -0.5000000000000002),\n                        (8, 0.14099072666039333, 0.0, 0.4999999999999998),\n                        (5, 0.25271486918457997, 0.0, -0.5000000000000002),\n                        (1, 0.6863539888389778, 0.0, 0.5)],\n                       [(3, 7), (4, 8)],\n                       [(3, 0.5, 0.0, 0.5),\n                        (4, 0.6544375463289773, 0.0, 0.5),\n                        (7, 0.38291144507086683, 0.0, 0.5)],\n                       [(3, 4), (7, 8)],\n                       [(5, 4),\n                        (3, 0.4999999999999998, 0.0, 0.4999999999999998),\n                        (7, 0.9957692972812717, 0.0, 0.5)],\n                       [(1, 3), (4, 0.2947640447824702, 0.0, 0.5), (5, 0.998577127886374, 0.0, -0.5)],\n                       [(4, 8), (3, 0.7483385199514005, 0.0, -0.5), (1, 0.5, 0.0, -0.5)],\n                       [(3, 7), (4, 0.79476404478247, 0.0, -0.5), (8, 0.5, 0.0, -0.5)],\n                       [(3, 0.3186895926607129, 0.0, 0.5), (7, 0.4838771221091833, 0.0, -0.5000000000000002)],\n                       [(3, 4), (7, 8)],\n                       [(3, 0.5, 0.0, -0.5), (4, 0.5, 0.0, -0.5), (8, 0.5, 0.0, 0.5), (7, 0.5, 0.0, 0.5)],\n                       [(1, 3), (5, 4)],\n                       [(4, 8), (3, 0.5, 0.0, 0.5), (5, 0.7311061266767034, 0.0, 0.08734394500735743)],\n                       [(3, 7),\n                        (9, 5),\n                        (4, 0.5, 0.0, -0.36453629437645896),\n                        (8, 0.5, 0.0, -0.27092741124708164)],\n                       [(4, 8),\n                        (3, 0.5882500769209105, 0.0, 0.4728982487520459),\n                        (7, 0.5542035024959076, 0.0, 0.0),\n                        (5, 0.7311061266767034, 0.0, -0.9126560549926428),\n                        (9, 0.5000000000000002, 1.1102230246251565e-16, -0.21657778195980648)],\n                       [(1, 3),\n                        (7, 6),\n                        (4, 0.24466630114623333, 0.0, 0.8099276012052394),\n                        (8, 0.20853893040407417, 0.0, 0.8163289126443196)],\n                       [(5, 4),\n                        (8, 9),\n                        (3, 0.9675670579641595, 0.0, 0.4728982487520297),\n                        (7, 0.492805042274795, 0.0, -0.3286433146379135),\n                        (6, 0.46084833097988237, 1.1102230246251565e-16, -0.1690806987049689)],\n                       [(3, 7),\n                        (4, 0.9368639593978498, 0.0, 0.7856415522914131),\n                        (8, 0.4999999999999998, 0.0, 0.5200439810017714),\n                        (9, 0.44377939308296455, 0.0, -0.8106655690669697)],\n                       [(4, 8),\n                        (9, 5),\n                        (3, 0.5000000000000004, 0.0, -0.5271017512479872),\n                        (7, 0.5596506416568943, 0.0, -0.23954611571239548)],\n                       [(3, 7),\n                        (4, 0.5540257210242681, 0.0, -0.15648179611313218),\n                        (8, 0.813789773970367, 0.0, -0.4799560189982297),\n                        (9, 0.6046189683150618, 0.0, 0.030029233150676127)],\n                       [(5, 4),\n                        (8, 9),\n                        (3, 0.12754703959217256, 0.0, -0.6794876081770782),\n                        (7, 0.5000000000000004, 0.0, -0.21844494307233564)],\n                       [(1, 3),\n                        (7, 6),\n                        (4, 0.9277113897101394, 0.0, -0.3495232139965765),\n                        (8, 0.020455379119187052, 0.0, 0.020043981001766298),\n                        (9, 0.8066269798399527, 5.551115123125783e-17, 0.2803602528098752)],\n                       [(4, 8),\n                        (9, 5),\n                        (3, 0.2696276353165254, 0.0, 0.37815679585117556),\n                        (7, 0.16304676742985036, 0.0, 0.7815550569276624),\n                        (6, 0.8273677055841601, 0.0, 0.011025144178165858)],\n                       [(3, 7),\n                        (4, 0.32856038849423386, 0.0, -0.8254832466984965),\n                        (8, 0.8798887313457266, 0.0, 0.5200439810017694),\n                        (9, 0.1457061188933495, 0.0, -0.9227041041680091)],\n                       [(4, 8),\n                        (9, 5),\n                        (3, 0.3174136387017352, 0.0, -0.06376344544364332),\n                        (7, 0.704332640729217, 0.0, -0.7184449430723381)],\n                       [(1, 3),\n                        (7, 6),\n                        (4, 0.21091990472113054, 0.0, 0.2956424378627158),\n                        (8, 0.135803336674352, 4.440892098500626e-16, -0.9799560189982319),\n                        (9, 0.38366824201029437, 0.0, -0.2525987027344243)],\n                       [(5, 4),\n                        (8, 9),\n                        (3, 0.6276139104334912, 1.6653345369377348e-16, 0.032143664351150625),\n                        (7, 0.5, 0.0, -0.7184449430723383),\n                        (6, 0.727371054243577, 0.0, 0.9268162268461131)],\n                       [(3, 7),\n                        (4, 0.7172458777721731, 0.0, 0.3404494406165792),\n                        (8, 0.5000000000000003, 0.0, -0.9799560189982317),\n                        (9, 0.049592127870291554, 0.0, 0.2566034070255234)],\n                       [(4, 8),\n                        (9, 5),\n                        (3, 0.4999999999999986, 0.0, 0.13747739711150087),\n                        (7, 0.529090738553442, 0.0, -0.34904139580339244)],\n                       [(1, 3),\n                        (7, 6),\n                        (5, 0.4999999999999997, 0.0, -0.8177318528117712),\n                        (4, 0.5, 0.0, 0.5140374994623187),\n                        (8, 0.5257538498277334, 0.0, -0.8468795165964336),\n                        (9, 0.24366130174103923, 0.0, 0.2668827096100349)],\n                       [(5, 4),\n                        (8, 9),\n                        (3, 0.6683089837594859, 4.440892098500626e-16, 0.6374773971115002),\n                        (7, 0.49999999999999956, 0.0, 0.657455074143229),\n                        (6, 0.49999999999999956, 0.0, -0.09727132543917172),\n                        (1, 0.5000000000000004, 0.0, -0.013550875623977277)],\n                       [(3, 7),\n                        (9, 13),\n                        (4, 8),\n                        (5, 0.5000000000000004, 0.13546370562354226, 0.18226814718822837)],\n                       [(7, 6),\n                        (3, 0.6578349175114986, -4.440892098500626e-16, -0.3625226028884989),\n                        (9, 0.31246956996987313, 0.0, -0.6514681657815511),\n                        (4, 0.5, 0.0, 0.9831144962621796),\n                        (13, 0.11366495197908669, 0.0, 0.12802565437172503)],\n                       [(1, 3),\n                        (6, 0.6530017079301533, 0.0, -0.5972713254391708),\n                        (8, 9),\n                        (7, 0.5000000000000009, 0.0, 0.6574550741432301)],\n                       [(6, 2),\n                        (3, 0.5, 0.0, -0.8626523093730487),\n                        (8, 0.6797328154058815, 0.0, -0.9233139913440256),\n                        (9, 0.5, 0.0, -0.5716419990933859),\n                        (1, 0.4999999999999997, 0.9999999999999993, 0.9864491243760227)],\n                       [(3, 7),\n                        (4, 8),\n                        (6, 0.5000000000000004, 8.881784197001252e-16, 0.9027286745608134),\n                        (9, 13),\n                        (2, 0.5605616196317902, 4.440892098500626e-16, -0.8348009229232944)],\n                       [(7, 6),\n                        (3, 0.5, 0.0, 0.1373476906269518),\n                        (4, 0.23664722798444943, 0.0, 0.48311449626218006),\n                        (9, 0.20756766428084616, 0.0, 0.4283580009066134),\n                        (13, 0.45965848934940423, 0.0, 0.750509805651933)],\n                       [(2, 3),\n                        (7, 0.5, 0.0, 0.42072304033947683),\n                        (4, 8),\n                        (9, 13),\n                        (6, 0.43110091722996385, 0.0, -0.34731902068777165)],\n                       [(3, 7),\n                        (2, 0.5818042231825745, -4.440892098500626e-16, -0.3348009229233022),\n                        (8, 0.5043067323421142, 0.0, 0.44538479622963756),\n                        (9, 0.9139805299674035, 0.0, -0.07164199909338365),\n                        (4, 0.4999999999999998, 0.0, 0.9831144962621796),\n                        (13, 0.5399725193471079, 0.0, -0.09132550086841418)],\n                       [(6, 2),\n                        (3, 0.2430469681607357, 4.440892098500626e-16, 0.7315717222716438),\n                        (7, 0.5, 0.0, -0.5792769596605232),\n                        (8, 9)],\n                       [(3, 7),\n                        (6, 0.03454957128703273, 0.0, 0.8181166636420727),\n                        (2, 0.2894984978675621, 0.0, -0.8348009229233004),\n                        (8, 0.43693748367111007, 1.1102230246251565e-16, 0.3671692521958917),\n                        (9, 0.2080502620770015, 0.0, 0.42835800090661835)],\n                       [(6, 2), (8, 9), (3, 0.30568317284628654, 0.0, -0.620871356410428)],\n                       [(2, 0.87386354985307, 0.0, 0.6651990770766807),\n                        (6, 0.9242869536756131, 4.440892098500626e-16, 0.5858061491772155),\n                        (8, 0.13459349110635355, 0.0, -0.025519519662967838),\n                        (9, 0.7396084787055979, 0.0, 0.9283580009066184)],\n                       [(2, 3), (7, 6), (4, 8), (9, 13)],\n                       [(3, 0.5595532320705698, 0.0, -0.20296905022415124),\n                        (7, 0.5, 0.0, -0.07914725317597382),\n                        (6, 0.23753846872505302, 4.440892098500626e-16, -0.8818761293480915),\n                        (2, 0.6888649043845242, 1.6653345369377348e-16, 0.16519907707666082),\n                        (8, 0.3225108895818005, 0.0, -0.3374062588286648),\n                        (9, 0.7020510034522096, 0.0, -0.5716419990933812),\n                        (4, 0.5, 0.0, -0.04780850693795946),\n                        (13, 0.6530265810282696, 1.1102230246251565e-16, 0.4148579448270263)],\n                       [(3, 7), (6, 2), (8, 9)],\n                       [(2, 0.49999999999999956, 0.0, 0.16519907707666054),\n                        (7, 0.5, 0.0, 0.9208527468240266),\n                        (6, 0.6383222145074643, 4.440892098500626e-16, -0.8232029652272685),\n                        (8, 0.1814139933031107, 0.0, 0.6818786182425496),\n                        (9, 0.5000000000000004, 0.0, -0.5716419990933808)],\n                       [(2, 3), (7, 6), (4, 8), (9, 13)],\n                       [(7, 0.5, 0.0, 0.7304282452838118),\n                        (6, 0.4999999999999997, 0.0, -0.7484872360058605),\n                        (2, 0.4546007179287068, 0.0, -0.5832612386680716),\n                        (8, 0.5, 0.0, 0.6705906102168315),\n                        (9, 0.6180389813777741, 0.0, 0.7327765646792332),\n                        (4, 0.5666836638610826, 0.0, 0.45219149306204054)],\n                       [(3, 7), (6, 2), (8, 9)],\n                       [(2, 0.19504058663172466, 0.0, 0.7142911015049966),\n                        (3, 0.29782301439575226, 0.0, -0.5579583981385003),\n                        (6, 0.8858944659256832, 5.551115123125783e-17, 0.25151276399414046),\n                        (9, 13),\n                        (8, 0.5000000000000007, 0.0, -0.8294093897831682)],\n                       [(2, 3),\n                        (7, 6),\n                        (4, 8),\n                        (13, 0.8203446403177395, 0.0, -0.85533453344054),\n                        (9, 0.5, 0.0, 0.12888011435422375)],\n                       [(2, 0.5271017512479523, 0.0, -0.8260320411351967),\n                        (7, 0.21266361194798097, 0.0, 0.23042824528381345),\n                        (3, 0.5000000000000003, 0.0, 0.6446689492722386),\n                        (13, 12),\n                        (8, 0.5, 4.440892098500626e-16, -0.8294093897831694),\n                        (4, 0.4999999999999998, -0.1344596402502698, 0.45219149306204054)],\n                       [(2, 0),\n                        (7, 3),\n                        (8, 9),\n                        (13, 0.9171590576689481, 0.0, -0.35059406651349767),\n                        (12, 0.22616596014349977, 4.440892098500626e-16, -0.8779609025165152)],\n                       [(2, 0.5, -4.440892098500626e-16, -0.3260320411351967),\n                        (7, 0.49999999999998246, 0.0, -0.26957181103412786),\n                        (9, 0.5000000000000003, 0.0, 0.12888011435422397),\n                        (8, 0.7288253632218438, 1.6653345369377348e-16, 0.006763170987452993),\n                        (0, 0.30408672398469605, 0.0, -0.5000000000000002),\n                        (3, 0.500000000000002, 0.0, 0.7193417253762777)],\n                       [(6, 2), (7, 11), (9, 13), (12, 8)],\n                       [(2, 0.8040867239846965, 0.0, 0.6739679588648031),\n                        (7, 0.5000000000000102, 0.0, -0.26957181103412897),\n                        (8, 0.6880132602922713, 4.440892098500626e-16, 0.5878692613477401),\n                        (13, 0.5200842996825256, 5.551115123125783e-17, 0.044131888056848145),\n                        (12, 0.7517948763488758, 0.0, -0.3779609025165156),\n                        (11, 0.30408672398469605, 0.0, 0.4254807786478514)],\n                       [(2, 0), (7, 3), (8, 9), (13, 12)],\n                       [(2, 0.75, 0.0, 0.6739679588648031),\n                        (7, 0.75, 0.0, 0.7304281564506918),\n                        (13, 0.7871017150857011, 0.0, -0.35639749601148596),\n                        (12, 0.4999999999999998, 0.0, -0.3779609025165165),\n                        (8, 0.6315156078419237, 0.0, -0.2558435891973083),\n                        (0, 0.5, 0.0, 0.5),\n                        (3, 0.49999999999999956, 0.0, -0.2806582421085415)],\n                       [(6, 2), (7, 11), (9, 13), (12, 8)],\n                       [(2, 0.5, 0.0, 0.6739679588648031),\n                        (7, 0.500526889421938, 0.0, -0.7700987796764718),\n                        (9, 0.5, 0.0, -0.8711198856457771),\n                        (13, 0.4929820472032482, 0.0, -0.8563974960114875),\n                        (12, 0.4999999999999998, 0.0, -0.8764349912116867),\n                        (6, 0.5, 4.440892098500626e-16, 0.7650636396181183),\n                        (11, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515)],\n                       [(2, 0), (7, 3), (8, 9), (13, 12)],\n                       [(2, 0.25000000000000006, 0.0, -0.3260320411351971),\n                        (7, 0.5, 0.0, -0.5200992157499043),\n                        (3, 0.5000000000000004, 0.0, -0.780658242108542),\n                        (9, 0.5, 0.0, -0.871119885645776),\n                        (13, 0.5000000000000003, 0.0, 0.6114395689949488),\n                        (12, 8)],\n                       [(6, 2),\n                        (7, 11),\n                        (9, 13),\n                        (12, 0.5750977272438426, 8.881784197001252e-16, 0.9516370188762597)],\n                       [(7, 3),\n                        (2, 0.5, 0.0, -0.3260320411351971),\n                        (9, 0.5, 0.0, -0.2958663322488988),\n                        (13, 0.8015139435996756, 0.0, -0.38856043100505144),\n                        (6, 0.5, -0.2650636396181181, -0.2349363603818817),\n                        (11, 0.5, 0.3483047662953689, -0.5745192538673269)],\n                       [(2, 0),\n                        (8, 9),\n                        (13, 12),\n                        (3, 0.9992548329781763, 0.5544437545365997, -0.2806582421085493),\n                        (7, 3.251517832758566e-08, 0.7938847281779458, 0.4799007615846072)],\n                       [(13, 0.4999999999999998, 0.0, -0.8885604310050528),\n                        (12, 0.10639065047380725, 0.0, 0.407313456556351),\n                        (8, 0.5, 0.0, -0.6868412744947046),\n                        (0, 1.0, 0.0, 0.5),\n                        (2, 0.5, 0.8260320411351971, -0.3260320411351971)],\n                       [(9, 13), (12, 8)],\n                       [(9, 0.5439463403837284, 0.0, 0.2041336677511003),\n                        (8, 0.5, 0.0, -0.6868412744947032),\n                        (13, 0.4999999999999998, 0.0, -0.3446140906213233),\n                        (12, 0.5, 0.0, -0.09268654344364546)],\n                       [(9, 8), (13, 12)],\n                       [(9, 0.49999999999998224, 0.0, -0.2958663885668411),\n                        (13, 0.49999999999998246, 0.0, 0.655385853060735),\n                        (8, 0.500000000000002, 1.1102230246251565e-16, -0.2747339552621597),\n                        (12, 0.5000000000000022, 0.0, 0.4952061373238105)],\n                       [(9, 10), (13, 14)],\n                       [(9, 0.5000000000000103, 1.1102230246251565e-16, -0.2958663885668402),\n                        (13, 0.5000000000000104, 0.0, 0.655385853060735),\n                        (10, 0.30408672398469605, 0.0, 0.4254807786478514),\n                        (14, 0.30408672398469605, 0.0, 0.4254807786478514)],\n                       [(9, 8), (13, 12)],\n                       [(9, 0.7499999999999999, 0.0, 0.7041335789179812),\n                        (13, 0.75, 0.0, 0.6553858205455567),\n                        (8, 0.4999999999999995, 0.0, 0.7252660772530213),\n                        (12, 0.49999999999999944, 0.0, -0.5047938951913713)],\n                       [(9, 10), (13, 14)],\n                       [(9, 0.500526889421938, 0.0, -0.7963933572091826),\n                        (13, 0.500526889421938, 0.0, -0.8451411155816071),\n                        (10, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515),\n                        (14, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515)],\n                       [(9, 8), (13, 12)],\n                       [(9, 0.5, 0.0, -0.5463937932826155),\n                        (13, 0.5, 0.0, 0.4048584483449613),\n                        (8, 0.5000000000000004, 0.0, 0.2252660772530219),\n                        (12, 0.5000000000000004, 0.0, -0.004793895191371966)],\n                       [(9, 10), (13, 14)],\n                       [(9, 8),\n                        (13, 12),\n                        (10, 0.5, 0.3483047662953689, -0.5745192538673269),\n                        (14, 0.5, 0.3483047662953689, -0.5745192538673269)],\n                       [(8, 0.9992548329781763, -0.451480564824962, 0.7252660772530122),\n                        (9, 3.251517832758566e-08, 0.8201793057106572, 0.45360618405189607),\n                        (12, 0.0007451670218237004, 0.778579407619415, 0.49520610480863625),\n                        (13, 3.251517832758566e-08, 0.8689270640830808, 0.40485842567947256)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8),\n                      (6, 5),\n                      (6, 6),\n                      (6, 7),\n                      (6, 8)]},\n (15, 6): {\'method\': \'joint small decoder cores + parallel three-site leaves\',\n           \'moments\': [[(5, 0.15910970406013664, 0.0, 0.5),\n                        (4, 0.7373539681892943, 0.0, 0.5),\n                        (7, 0.7085901148744962, 0.0, 0.5),\n                        (8, 0.4467644305628241, 0.0, -0.5000000000000002),\n                        (1, 0.9011168519071704, 1.0, -0.5),\n                        (6, 0.5000000000000004, 0.0, -0.20251959709327183),\n                        (13, 0.5, 0.9999999999999999, -0.20307004319295552),\n                        (2, 0.5, 1.0, -0.377500967254607),\n                        (0, 0.5, 1.0, 0.5),\n                        (11, 0.3040867239846966, 0.0, -0.5745192213521528),\n                        (12, 0.49999999999999956, 1.0, -0.3779609025165147),\n                        (10, 0.3040867239846966, 1.0, -0.5745192213521528),\n                        (14, 0.3040867239846966, 1.0, -0.5745192213521528)],\n                       [(5, 4), (7, 8), (1, 3)],\n                       [(4, 0.7581079035004981, 0.0, -0.5),\n                        (8, 0.21081011765170987, 0.0, -0.5),\n                        (3, 0.6137946531114089, 0.0, 0.5),\n                        (7, 0.5443536300607281, 1.0, -0.5),\n                        (1, 0.40111685190717034, 0.0, -0.5000000000000002),\n                        (5, 0.38705144949570247, 0.0, -0.5)],\n                       [(4, 8), (3, 7)],\n                       [(7, 0.5102068102350226, 0.0, 0.5),\n                        (8, 0.2523672686524676, 0.0, 0.5),\n                        (3, 0.6609494081989717, 0.0, 0.5),\n                        (4, 0.5, 1.0, 0.5)],\n                       [(7, 8), (3, 4)],\n                       [(1, 3),\n                        (4, 0.5, 0.0, 0.5),\n                        (7, 0.39030302404504397, 0.0, 0.5),\n                        (8, 0.9456160071264607, 0.0, 0.5)],\n                       [(5, 4), (3, 0.17564872714486618, 0.0, 0.5), (1, 0.4999999999999998, 0.0, 0.5)],\n                       [(3, 7), (4, 0.4215716033272554, 0.0, 0.5), (5, 0.5, 0.0, 0.5)],\n                       [(4, 8), (3, 0.16336727963993836, 0.0, 0.5), (7, 0.5630139738389508, 0.0, -0.5)],\n                       [(4, 0.8306312090125401, 0.0, 0.5), (8, 0.5189686705552345, 0.0, -0.5)],\n                       [(3, 4), (7, 8)],\n                       [(3, 0.8464209044699778, 0.0, 0.5),\n                        (4, 0.31756513752949056, 0.0, -0.5),\n                        (7, 0.8097241829115225, 0.0, 0.5),\n                        (8, 0.812464019304552, 0.0, 0.5)],\n                       [(1, 3), (5, 4)],\n                       [(3, 0.8879545911970581, 0.0, 0.5), (4, 0.9100820590738672, 0.0, -0.5)],\n                       [(3, 7), (4, 8)],\n                       [(3, 0.939084250472354, 0.0, 0.5),\n                        (4, 0.10550946076945021, 0.0, -0.5000000000000002),\n                        (7, 0.30815933398738005, 0.0, -0.5),\n                        (8, 0.16188927506338208, 0.0, 0.5)],\n                       [(3, 4), (7, 8)],\n                       [(3, 0.513539249942093, 0.0, -0.5),\n                        (4, 0.5010152913918939, 0.0, 0.5),\n                        (7, 0.5273673081805759, 0.0, -0.5),\n                        (8, 0.4963987267193131, 0.0, -0.5)],\n                       [(1, 3), (5, 4)],\n                       [(3, 0.07826489477032926, 0.0, -0.5000000000000002),\n                        (4, 0.9229270144537076, 0.0, -0.5),\n                        (5, 0.26889387332329684, 0.0, -0.08734394500735743)],\n                       [(3, 7), (4, 8), (9, 5)],\n                       [(3, 0.8177478028805574, 0.0, 0.9552955162279688),\n                        (4, 0.7974347350986499, 4.440892098500626e-16, -0.744456285556562),\n                        (8, 0.566262611941891, 0.0, -0.2647179753448854),\n                        (7, 0.523592635174831, 0.0, -0.36006132039625616),\n                        (5, 0.7311061266767034, 0.0, -0.08734394500735743),\n                        (9, 0.5000000000000002, 1.1102230246251565e-16, -0.21657778195980648)],\n                       [(1, 3), (4, 8), (7, 6)],\n                       [(4, 0.2446663011462331, 0.0, 0.870873798513915),\n                        (8, 0.20853893040407412, 0.0, 0.7072974527217695),\n                        (7, 0.6822968314790296, 0.9999999999999998, 0.6641770602304011),\n                        (6, 0.2509322168477317, 0.0, -0.7949290191088039),\n                        (1, 0.49999999999999956, 1.0, 0.009827684388288294)],\n                       [(5, 4), (8, 9), (3, 7)],\n                       [(4, 0.9368639593978498, 0.0, 0.8951598474277411),\n                        (8, 0.4999999999999997, 8.881784197001252e-16, -0.9964176156356823),\n                        (7, 0.5, 0.0, 0.4419293621569557),\n                        (3, 0.5728573590437845, 0.0, -0.14943365633074968),\n                        (9, 0.44377939308296455, 1.0, 0.1893344309330302)],\n                       [(4, 8), (7, 6), (1, 3), (9, 5)],\n                       [(7, 0.2673990199988756, 0.0, -0.5580706378430444),\n                        (6, 0.5826592798249117, 0.0, 0.6305613443031177),\n                        (8, 0.813789773970367, 0.0, -0.9964176156356812),\n                        (9, 0.6046189683150618, 0.0, 0.030029233150676127),\n                        (1, 0.41785980112400095, 1.0, -0.49017231561171126),\n                        (4, 0.5540257210242681, 0.9999999999999999, 0.8372831958322864)],\n                       [(7, 6), (8, 9), (1, 3), (5, 4)],\n                       [(4, 0.9277113897101394, 0.0, 0.03032461371573092),\n                        (8, 0.020455379119187052, 0.0, 0.5035823843643221),\n                        (9, 0.8066269798399527, 0.0, -0.719639747190125),\n                        (5, 0.0, 0.046777110307263614, 0.0),\n                        (3, 0.3856735769590187, 0.0, 0.7167957742661271),\n                        (7, 0.5505347324801627, -0.9999999999999996, -0.05807063784304445),\n                        (6, 0.3689376239568078, 0.0, 0.7719771085165397),\n                        (1, 0.5000000000000002, 1.0, 0.009827684388288793)],\n                       [(4, 8), (9, 5), (3, 7)],\n                       [(4, 0.3285603884942341, 0.0, -0.4937153535823495),\n                        (8, 0.8798887313457266, 4.440892098500626e-16, -0.9964176156356803),\n                        (9, 0.1457061188933495, 0.0, 0.07729589583199137),\n                        (3, 0.46333492445932545, 3.3306690738754696e-16, -0.28526791883229796),\n                        (7, 0.3213111929430337, 1.0, 0.44192936215695555)],\n                       [(4, 8), (9, 5), (3, 7)],\n                       [(4, 0.2109199047211305, 0.0, -0.6148410381435614),\n                        (7, 0.5666568155244442, 0.0, 0.941929362156956),\n                        (8, 0.13580333667435204, 0.0, -0.496417615635679),\n                        (9, 0.38366824201029437, 0.0, 0.7474012972655757),\n                        (3, 0.43721065119527336, 0.0, 0.8791079058952689)],\n                       [(5, 4), (7, 6), (8, 9), (1, 3)],\n                       [(4, 0.717245877772173, 0.0, -0.659648040897425),\n                        (8, 0.5000000000000004, 0.0, -0.49641761563567943),\n                        (3, 0.1039903575622847, 0.0, -0.7217933052045138),\n                        (7, 0.5850425523318399, 1.0, 0.44192936215695555),\n                        (9, 0.049592127870291554, 1.0, 0.25660340702552353),\n                        (6, 0.8311737009895825, 0.0, -0.0841139332337324),\n                        (1, 0.5, -0.9999999999999996, -0.6402265742624289)],\n                       [(4, 8), (3, 7), (9, 5)],\n                       [(7, 0.49999999999999956, 0.0, 0.4419293621569558),\n                        (8, 0.5257538498277332, 0.0, 0.37050588196252265),\n                        (9, 0.24366130174103914, 0.0, -0.7331172903899656),\n                        (3, 0.43415359559636596, 0.0, -0.4901186340632986),\n                        (5, 0.4999999999999998, 0.0, -0.9456426325192224),\n                        (4, 0.5, 1.0, 0.16676390025683552)],\n                       [(7, 6), (8, 9), (1, 3), (5, 4)],\n                       [(3, 0.4999999999999997, 0.0, -0.6862353489242252),\n                        (7, 0.5998657277943455, 0.9999999999999999, -0.4582468242998341),\n                        (8, 0.0, 1.0, 0.0),\n                        (9, 13),\n                        (1, 0.5530742553856172, 1.0, -0.14022657426242846),\n                        (5, 0.5000000000000004, -0.7366255146690077, -0.9456426325192224)],\n                       [(3, 7),\n                        (4, 8),\n                        (9, 0.31246956996987313, 0.0, -0.6514681657815511),\n                        (13, 0.11366495197908669, 1.0, -0.8719743456282749)],\n                       [(8, 9),\n                        (3, 0.5000000000000004, 0.0, -0.18623534892422544),\n                        (7, 2.220446049250313e-16, 0.9999999999999998, 0.6161816667411557),\n                        (4, 0.5, 0.0, -0.3023130965430252)],\n                       [(1, 3),\n                        (7, 6),\n                        (8, 0.6797328154058815, 2.220446049250313e-16, 0.446940356710112),\n                        (9, 0.5, 0.0, 0.4283580009066142)],\n                       [(4, 8),\n                        (3, 0.5, 1.1102230246251565e-16, -0.18623534892422544),\n                        (7, 0.5000000000000001, 1.0, -0.633775369890361),\n                        (6, 0.6339658219122923, 0.0, 0.8651689462862566),\n                        (9, 13),\n                        (1, 0.5, 0.4803446312234234, -0.14022657426242846)],\n                       [(3, 7),\n                        (6, 2),\n                        (4, 0.23664722798444934, 0.0, -0.802313096543025),\n                        (9, 0.2075676642808464, 0.0, -0.5716419990933865),\n                        (13, 0.45965848934940423, 1.0, 0.7505098056519328)],\n                       [(4, 8),\n                        (2, 0.03592107551278813, 1.0, -0.8775009672546068),\n                        (3, 0.5991006024208342, 0.0, -0.348435158691776),\n                        (7, 0.5, 1.0, -0.6337753698903619),\n                        (6, 0.9068032572794498, 1.1102230246251565e-16, 0.23132164282033318),\n                        (9, 13)],\n                       [(2, 3),\n                        (7, 6),\n                        (8, 0.5043067323421142, 0.0, -0.9217584308635502),\n                        (9, 0.9139805299674035, 0.0, 0.9283580009066161),\n                        (4, 0.5, 0.0, -0.3023130965430254),\n                        (13, 0.5399725193471079, 0.9999999999999999, 0.9086744991315858)],\n                       [(3, 0.6983902210776689, 0.0, -0.13684944570623236),\n                        (8, 9),\n                        (6, 0.4457229604434565, 0.0, 0.01261445611041534),\n                        (2, 0.751811597304503, -0.9999999999999996, 0.6224990327453925)],\n                       [(3, 7),\n                        (6, 2),\n                        (8, 0.43693748367111007, 4.440892098500626e-16, -0.8435428868298047),\n                        (9, 0.2080502620770015, 0.0, 0.42835800090661835)],\n                       [(8, 9),\n                        (2, 0.5, 1.0, -0.3775009672546079),\n                        (3, 0.6251415065033563, 0.0, 0.010761988568890324),\n                        (7, 0.0, -0.3255499447294259, 0.0),\n                        (6, 0.7856407379834285, 0.0, -0.41370794677841705)],\n                       [(2, 3),\n                        (7, 6),\n                        (8, 0.13459349110635357, 0.0, -0.4508541149709444),\n                        (9, 0.7396084787055979, 0.0, -0.07164199909338165)],\n                       [(3, 0.0, 0.2808972619037798, 0.0),\n                        (7, 0.5, 0.0, 0.04067468538021357),\n                        (4, 8),\n                        (6, 0.4949978217082433, -2.220446049250313e-16, 0.08629205322158295),\n                        (2, 0.4999999999999998, 1.0, -0.8814144405744395),\n                        (9, 13)],\n                       [(3, 7),\n                        (6, 2),\n                        (8, 0.3225108895818005, 0.0, 0.8610326241947521),\n                        (9, 0.7020510034522096, 1.1102230246251565e-16, 0.4283580009066189),\n                        (4, 0.5, 0.0, -0.2713900933428861),\n                        (13, 0.6530265810282696, -0.9999999999999996, 0.41485794482702654)],\n                       [(2, 3),\n                        (8, 9),\n                        (7, 0.5, 1.0, -0.9593253146197864),\n                        (6, 0.5, 0.0, -0.4475413783399338)],\n                       [(7, 6),\n                        (8, 0.1814139933031105, 0.0, -0.15825225287646205),\n                        (2, 0.2887068963335278, 0.0, 0.899671023811214),\n                        (9, 0.5000000000000004, 0.0, -0.571641999093381)],\n                       [(4, 8),\n                        (6, 0.8015319629186126, 0.0, -0.4475413783399338),\n                        (7, 0.5, 1.0, -0.07526615395820135),\n                        (9, 13)],\n                       [(6, 2),\n                        (3, 7),\n                        (8, 0.5, 0.0, 0.8530357551492549),\n                        (9, 0.6180389813777741, 0.0, 0.7327765646792332),\n                        (4, 0.5666836638610826, 0.0, 0.22860990665711428)],\n                       [(2, 0.10822807679614659, 0.0, 0.5488202087084497),\n                        (3, 0.5000000000000004, 0.0, 0.5406978948087588),\n                        (8, 9),\n                        (7, 0.0, 1.0, 0.0),\n                        (6, 0.4999999999999993, 8.881784197001252e-16, -0.947541378339934)],\n                       [(2, 3), (7, 6), (8, 0.5000000000000003, 0.0, 0.3530357551492571), (9, 13)],\n                       [(4, 8),\n                        (2, 0.18023611392199, 0.0, 0.548820208708451),\n                        (7, 0.5663325445329528, 1.0, 0.4247338460417982),\n                        (3, 0.4999999999999998, 0.0, 0.540697894808758),\n                        (9, 0.5, 0.0, -0.8711198856457762),\n                        (13, 0.8203446403177395, 0.0, 0.14466546655945978)],\n                       [(2, 0),\n                        (7, 3),\n                        (8, 0.5, 0.0, -0.6469642448507429),\n                        (13, 12),\n                        (4, 0.5, -0.9108780538453436, 0.2286099066571139)],\n                       [(2, 0.5, 0.0, -0.9511797912915487),\n                        (7, 0.4999999999999824, 4.440892098500626e-16, 0.9247339023597374),\n                        (8, 9),\n                        (13, 0.9171590576689481, 4.440892098500626e-16, 0.6494059334865019),\n                        (0, 0.3040867239846962, 1.0, -0.5000000000000002),\n                        (3, 0.5000000000000019, 0.0, -0.8266370161253374),\n                        (12, 0.22616596014349977, -0.9999999999999998, 0.12203909748348507)],\n                       [(6, 2),\n                        (7, 11),\n                        (9, 0.5, 0.0, 0.12888011435422442),\n                        (8, 0.7288253632218438, 0.0, 0.5168631943786348)],\n                       [(9, 13),\n                        (2, 0.8040867239846965, 8.881784197001252e-16, -0.9511797912915487),\n                        (7, 0.5000000000000103, 0.9999999999999999, 0.9247339023597383),\n                        (12, 8),\n                        (11, 0.30408672398469627, 0.0, -0.5745192213521484),\n                        (6, 0.0, 0.6350392865765455, 0.0)],\n                       [(2, 0),\n                        (7, 3),\n                        (8, 0.6880132602922713, 1.1102230246251565e-16, -0.06424289598165345),\n                        (9, 0.0, -0.7230084657212139, 0.0),\n                        (13, 0.5200842996825255, 0.0, -0.9558681119431514),\n                        (12, 0.7517948763488758, -0.9999999999999998, 0.6220390974834844)],\n                       [(7, 0.75, -2.220446049250313e-16, -0.07526606512508316),\n                        (2, 0.7499999999999999, 8.881784197001252e-16, -0.9511797912915487),\n                        (8, 9),\n                        (13, 12),\n                        (0, 0.5, 1.0, 0.5),\n                        (3, 0.49999999999999956, 0.0, -0.8266370486405183)],\n                       [(7, 11),\n                        (6, 2),\n                        (13, 0.7871017150857011, 0.0, 0.6436025039885145),\n                        (12, 0.4999999999999997, 1.0, -0.3779609025165165),\n                        (8, 0.6315156078419237, 0.0, 0.7794699545633959)],\n                       [(9, 13),\n                        (2, 0.5, -2.220446049250313e-16, 0.04882020870845155),\n                        (7, 0.500526889421938, -0.9999999999999996, -0.5747391289979193),\n                        (12, 8),\n                        (11, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515),\n                        (6, 0.5, 1.0, -0.47238403480239244)],\n                       [(2, 0),\n                        (7, 3),\n                        (8, 0.0, 0.5994093898528792, 0.0),\n                        (9, 0.5, 0.0, -0.5941283513669906),\n                        (13, 0.49298204720324823, 0.0, 0.14360250398851315),\n                        (12, 0.4999999999999998, 1.0, -0.8764349912116867)],\n                       [(7, 0.5, 0.0, -0.8247386929244866),\n                        (2, 0.25, 0.0, -0.9511797912915487),\n                        (8, 9),\n                        (13, 12),\n                        (3, 0.5000000000000004, 0.0, 0.673362951359481)],\n                       [(7, 11),\n                        (6, 2),\n                        (9, 0.5, 5.551115123125783e-17, 0.4058716486330099),\n                        (13, 0.5000000000000004, 0.0, -0.3885604310050508),\n                        (12, 8)],\n                       [(2, 0.5, 0.0, 0.04882020870845155),\n                        (9, 13),\n                        (7, 1.021405182655144e-14, 1.0, -0.32474578383531183),\n                        (12, 0.5750977272438422, 0.0, 0.9516370188762604),\n                        (6, 0.5, 0.9723840348023929, -0.4723840348023929),\n                        (11, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(2, 0),\n                        (7, 3),\n                        (9, 0.5, 0.0, 0.9811252020298866),\n                        (13, 0.8015139435996756, 0.0, 0.6114395689949488)],\n                       [(8, 9),\n                        (13, 12),\n                        (0, 1.0, 0.0, 0.0),\n                        (7, 0.999999967484822, -0.9014758399784482, -0.8247386702589985),\n                        (2, 0.4999999999999998, 0.45117979129154895, -0.951179791291549),\n                        (3, 0.0007451670218237008, -0.899577438931439, 0.17336295135948987)],\n                       [(13, 0.4999999999999998, 0.0, 0.11143956899494778),\n                        (12, 0.10639065047380729, 0.0, 0.4073134565563512),\n                        (8, 0.5, 0.0, -0.19012297028633052)],\n                       [(9, 13), (12, 8)],\n                       [(9, 0.5439463403837282, 5.551115123125783e-17, 0.4811252020298865),\n                        (8, 0.4999999999999998, 0.0, -0.1901229702863314),\n                        (13, 0.4999999999999998, 0.0, 0.6553859093786771),\n                        (12, 0.5, 0.0, 0.9073134565563548)],\n                       [(9, 8), (13, 12)],\n                       [(9, 0.49999999999998257, 0.0, 0.9811251457119459),\n                        (13, 0.4999999999999824, 0.0, -0.3446141469392645),\n                        (8, 0.5000000000000022, 0.0, -0.6022302895188751),\n                        (12, 0.5000000000000022, 0.0, -0.5047938626761894)],\n                       [(9, 10), (13, 14)],\n                       [(9, 0.5000000000000103, 0.0, 0.9811251457119459),\n                        (13, 0.5000000000000104, 0.0, -0.3446141469392645),\n                        (10, 0.30408672398469616, 1.0, -0.5745192213521486),\n                        (14, 0.30408672398469616, 1.0, -0.5745192213521486)],\n                       [(9, 8), (13, 12)],\n                       [(9, 0.7499999999999999, 0.0, 0.981125113196768),\n                        (13, 0.7499999999999999, 0.0, -0.3446141794544424),\n                        (8, 0.49999999999999967, 5.551115123125783e-17, 0.39776967796594304),\n                        (12, 0.49999999999999956, 0.0, 0.4952061048086289)],\n                       [(9, 10), (13, 14)],\n                       [(9, 0.5005268894219381, 0.0, -0.5194018229303969),\n                        (13, 0.500526889421938, -2.220446049250313e-16, 0.15485888441839352),\n                        (10, 0.5000000000000103, 1.0, 0.42548077864785155),\n                        (14, 0.5000000000000103, 1.0, 0.42548077864785155)],\n                       [(9, 8), (13, 12)],\n                       [(9, 0.5, 0.0, 0.7305977409961706),\n                        (13, 0.5, 0.0, -0.5951415516550389),\n                        (8, 0.5000000000000004, 4.440892098500626e-16, 0.8977696779659421),\n                        (12, 0.5000000000000003, 0.0, 0.9952061048086287)],\n                       [(9, 10), (13, 14)],\n                       [(9, 8),\n                        (13, 12),\n                        (10, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (14, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(8, 0.0007451670218237004, -0.12398416553789815, -0.6022303220340501),\n                        (9, 3.251517832758566e-08, 0.5431877714318709, 0.7305977183306824),\n                        (12, 0.0007451670218237008, -0.22142059238058542, -0.5047938951913629),\n                        (13, 3.251517832758566e-08, -0.1310729359169187, -0.5951415743205275)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8),\n                      (6, 5),\n                      (6, 6),\n                      (6, 7),\n                      (6, 8)]},\n (15, 7): {\'method\': \'joint small decoder cores + parallel three-site leaves\',\n           \'moments\': [[(5, 0.31630918818247244, 0.0, -0.5),\n                        (4, 0.41348815057431126, 0.0, -0.5000000000000002),\n                        (7, 0.9749914697119216, 0.0, -0.5000000000000002),\n                        (8, 0.9138626762504471, 0.0, 0.5),\n                        (1, 0.05380912957041928, 1.0, 0.5),\n                        (9, 0.3726172094654563, 0.0, -0.24747495433935618),\n                        (6, 0.5, 1.0, -0.20251771408753827),\n                        (13, 0.5000000000000004, 1.0, -0.1910463251434371),\n                        (2, 0.49999999999999944, 1.0, -0.3649528224832399),\n                        (0, 0.5, 1.0, 0.5),\n                        (11, 0.3040867239846966, 1.0, -0.5745192213521528),\n                        (12, 0.5, 1.0, -0.3782002606674788),\n                        (10, 0.3040867239846966, 1.0, -0.5745192213521528),\n                        (14, 0.3040867239846966, 1.0, -0.5745192213521528)],\n                       [(5, 4), (7, 8), (1, 3)],\n                       [(4, 0.6503211074889919, 0.0, -0.5),\n                        (8, 0.08588433755655983, 0.0, -0.5000000000000002),\n                        (3, 0.35727615847529887, 0.0, 0.5),\n                        (7, 0.9758980292752844, 1.0, -0.5),\n                        (1, 0.4461908704295805, 0.0, -0.5),\n                        (5, 0.3755027911010412, 0.0, -0.5)],\n                       [(4, 8), (3, 7)],\n                       [(3, 0.8589803893445077, 0.0, -0.5),\n                        (4, 0.49999999999999867, 1.0, -0.5),\n                        (8, 0.49999999999999956, 1.0, -0.5)],\n                       [(3, 4), (7, 8)],\n                       [(1, 3),\n                        (4, 0.5000000000000011, 0.0, -0.5),\n                        (8, 0.5000000000000003, 0.0, 0.5),\n                        (7, 0.8278647016465945, 0.0, 0.5)],\n                       [(5, 4),\n                        (3, 0.8457372208504039, 1.0, 0.5),\n                        (1, 0.4999999999999989, 0.0, -0.5000000000000002)],\n                       [(4, 0.12844403112522304, 0.0, 0.5), (3, 7), (5, 0.5000000000000016, 0.0, -0.5)],\n                       [(4, 8),\n                        (3, 0.6486931696176113, 2.220446049250313e-16, -0.5000000000000002),\n                        (7, 0.5674640172984227, 0.0, -0.5)],\n                       [(4, 0.315791130517675, 0.0, 0.5), (8, 0.5, 1.0, 0.5)],\n                       [(3, 4), (7, 8)],\n                       [(3, 0.35112917909082464, 0.0, -0.5000000000000002),\n                        (4, 0.1706789593772302, 0.0, -0.5000000000000002),\n                        (8, 0.5, 0.0, -0.5)],\n                       [(1, 3), (5, 4)],\n                       [(4, 0.4478435398279293, 0.0, -0.5),\n                        (3, 0.031841076455933814, 1.0, -0.5000000000000002),\n                        (5, 0.5, 0.0, 0.417035037809574)],\n                       [(4, 8), (3, 7)],\n                       [(3, 0.19434901970930363, 0.0, -0.8854803695531277),\n                        (4, 0.8612176108834231, 0.0, -0.9554622202497913),\n                        (8, 0.1551956192056225, 0.0, 0.7746692152562928),\n                        (7, 0.500000000000002, 0.0, -0.6393986262743319)],\n                       [(1, 3), (5, 4), (8, 9), (7, 6)],\n                       [(7, 0.40250888547893265, 0.0, 0.0458194953410902),\n                        (8, 0.49999999999999956, 0.0, 0.27466921525629456),\n                        (9, 0.31530513919323977, 0.0, 0.752525045660644),\n                        (5, 0.5, 0.0, 0.41703503780957374),\n                        (1, 0.5, -0.9999999999999998, 0.012281121285115715),\n                        (6, 0.23824984739858224, -0.9999999999999996, 0.04802556671795083)],\n                       [(3, 7), (4, 8), (9, 5)],\n                       [(5, 0.5, 0.0, 0.34986898287808277),\n                        (8, 0.5647392099925428, 0.0, 0.5857591318482358),\n                        (9, 0.5000000000000004, 0.0, -0.24747495433935573),\n                        (3, 0.49999999999999956, 0.0, -0.02247740254193875),\n                        (7, 0.5000000000000016, 0.0, -0.4541805046589116)],\n                       [(5, 4), (8, 9), (1, 3), (7, 6)],\n                       [(4, 0.7190546004943026, 0.0, 0.9547007228430036),\n                        (8, 0.5, 0.0, 0.8096714264351825),\n                        (9, 0.7652310577979675, 0.0, 0.637366085625735),\n                        (5, 0.4999999999999998, -2.220446049250313e-16, -0.6501310171219175),\n                        (1, 0.5000000000000003, 1.0, -0.48771887871488406),\n                        (7, 0.499999999999999, 0.0, 0.545819495341088),\n                        (6, 0.8228387639951986, 1.0, 0.24879131417776962)],\n                       [(4, 8), (9, 5), (1, 3), (7, 6)],\n                       [(3, 0.36532098276163216, 0.0, -0.6915218298387598),\n                        (7, 0.4568853839401345, 0.0, 0.04581949534108576),\n                        (4, 0.31013106985866257, 0.0, -0.5273105249898657),\n                        (8, 0.5534917943749377, 4.440892098500626e-16, 0.8096714264351823),\n                        (9, 0.2622259419402657, 0.0, 0.2102955480653782),\n                        (1, 0.5, 1.0, 0.012281121285115937),\n                        (6, 0.6082288892935703, 1.0, 0.24879131417776595)],\n                       [(3, 7), (5, 4), (8, 9)],\n                       [(3, 0.15979316138258426, 0.0, -0.616195148461282),\n                        (7, 0.5409187240915672, 0.0, 0.5458194953410875),\n                        (4, 0.6368964083538908, 0.0, -0.5807040086802473),\n                        (8, 0.0868612228772827, 0.0, -0.6903285735648186),\n                        (9, 0.6066746657954186, 1.0, -0.03501643337464655)],\n                       [(3, 7), (4, 8), (9, 5)],\n                       [(4, 0.5757693861375772, 1.1102230246251565e-16, -0.1464601606157827),\n                        (8, 0.7502067491926145, 8.881784197001252e-16, 0.8096714264351825),\n                        (3, 0.17667382808391466, 0.0, -0.4665481566592975),\n                        (7, 0.0820855992752405, 0.0, -0.9541805046589129),\n                        (9, 0.8545222128357532, 1.0, -0.7124225211626047)],\n                       [(4, 8), (1, 3), (7, 6), (9, 5)],\n                       [(3, 0.5000000000000024, 0.0, -0.1380730791440974),\n                        (7, 0.4999999999999983, -4.440892098500626e-16, -0.4541805046589129),\n                        (8, 0.34344000111504247, 0.0, -0.6903285735648161),\n                        (9, 0.6203487569281176, 0.0, -0.5207403282160519),\n                        (4, 0.010906102888603046, 0.9999999999999999, 0.44817168184996137),\n                        (1, 0.5, 1.0, -0.6352874139983515),\n                        (6, 0.2692579739648868, 1.0, -0.7512086858222318)],\n                       [(3, 7), (8, 9), (5, 4)],\n                       [(4, 0.704704464695499, 0.0, -0.36570044080658937),\n                        (8, 0.5, 0.0, -0.6903285735648159),\n                        (9, 0.10481314833092266, 0.0, -0.5715089930439095),\n                        (5, 0.0, -0.22519417359719496, 0.0),\n                        (3, 0.35284137925881676, 0.0, 0.8619269208559022),\n                        (7, 0.5000000000000007, 0.0, 0.5458194953410875)],\n                       [(4, 8), (9, 5), (1, 3), (7, 6)],\n                       [(3, 0.5, 0.0, 0.8619269208559057),\n                        (7, 0.2690604471363719, 0.0, -0.9541805046589129),\n                        (5, 0.5, 8.881784197001252e-16, 0.860377150904535),\n                        (4, 0.5000000000000001, 1.1102230246251565e-16, 0.038576210459943805),\n                        (8, 0.1629282747669579, 5.551115123125783e-17, 0.24149681274622686),\n                        (9, 0.9365215647402696, 8.881784197001252e-16, 0.9781888605361235),\n                        (1, 0.30168861947752806, 1.0, -0.13528741399835198)],\n                       [(3, 7), (5, 4), (8, 9)],\n                       [(7, 6),\n                        (3, 0.5000000000000004, 0.0, 0.3619269208559056),\n                        (8, 2.220446049250314e-16, -0.9999999999999998, -0.7332766692045596),\n                        (9, 13),\n                        (5, 0.4999999999999998, -0.5417098470904615, -0.13962284909546496)],\n                       [(1, 3),\n                        (4, 8),\n                        (7, 0.5, 0.0, -0.9541805046589111),\n                        (9, 0.23322014968045032, 0.0, -0.4256831654554021),\n                        (6, 2),\n                        (13, 0.37065855880304627, 1.0, 0.7190586046382521)],\n                       [(3, 0.5000000000000001, 0.0, 0.3619269208559056),\n                        (8, 9),\n                        (4, 0.5, 0.0, -0.43763712621360185),\n                        (6, 0.564919347967729, 0.0, -0.8746225131559058),\n                        (2, 0.3765861726663244, -0.9999999999999996, 0.13504717751675888),\n                        (1, 0.5, 0.47543775742976835, -0.13528741399835198)],\n                       [(3, 7),\n                        (8, 0.6005613187689711, 0.0, 0.35898090609705724),\n                        (9, 0.5000000000000003, 0.0, 0.8547588773081138)],\n                       [(4, 8),\n                        (7, 0.5, 0.0, 0.04581949534108887),\n                        (3, 0.5000000000000038, 0.0, 0.36192692085590217),\n                        (9, 13)],\n                       [(7, 6),\n                        (2, 3),\n                        (4, 0.4628051521153966, 0.0, -0.9376371262136016),\n                        (9, 0.176651856143987, 0.0, -0.14524112269188683),\n                        (13, 0.5550552228624257, 1.0, 0.04443005696540403)],\n                       [(4, 8),\n                        (3, 0.6124202497959056, 0.0, -0.7410628240865034),\n                        (7, 0.0, -0.20390003411923963, 0.0),\n                        (6, 0.5000000000000004, -4.440892098500626e-16, -0.3746225131559058),\n                        (2, 0.7664718219180737, 0.9999999999999998, 0.635047177516759),\n                        (9, 13)],\n                       [(3, 7),\n                        (6, 2),\n                        (8, 0.7505624896178791, 0.0, 0.8665968295057813),\n                        (9, 0.9995233959301968, 1.2212453270876722e-14, 0.35475887730809186),\n                        (4, 0.5, 0.0, -0.4376371262136014),\n                        (13, 0.2376105941729456, 0.9999999999999998, 0.5138048538060924)],\n                       [(6, 0.8371608839553215, 0.0, 0.17545611282181062),\n                        (8, 9),\n                        (2, 0.5000000000000002, 1.0, -0.36495282248324123),\n                        (3, 0.7237608380026965, 0.0, 0.14544752584170517)],\n                       [(7, 6),\n                        (2, 3),\n                        (8, 0.5021764573065325, 0.0, 0.4593706457902655),\n                        (9, 0.363932006747824, 0.0, -0.14524112269191702)],\n                       [(8, 9),\n                        (3, 0.0, 0.5155754461675257, 0.0),\n                        (7, 0.5, 0.0, -0.1580805387781501),\n                        (6, 0.647259141517744, 0.0, 0.3022144361017878),\n                        (2, 0.5000000000000001, 1.0, -0.8649528224832415)],\n                       [(3, 7),\n                        (6, 2),\n                        (8, 0.22080971672900618, 0.0, 0.12259049773605746),\n                        (9, 0.6102590262802314, 0.0, 0.354758877308083)],\n                       [(2, 3),\n                        (4, 8),\n                        (7, 0.5, 0.0, 0.841919461221849),\n                        (6, 0.4999999999999998, 0.0, -0.31149386997272455),\n                        (9, 13)],\n                       [(7, 6),\n                        (2, 0.5, 0.0, 0.9683805108500918),\n                        (8, 0.32051363301284425, 0.0, 0.008565802851027371),\n                        (9, 0.44587766664971973, 0.0, 0.8547588773080832),\n                        (4, 0.5, 0.0, -0.4138504628871482),\n                        (13, 0.37762451406506514, 1.0, -0.38570699668816255)],\n                       [(7, 0.5, 4.440892098500626e-16, 0.6041401073633492),\n                        (6, 0.6633380081977305, 4.440892098500626e-16, 0.6885061300272752),\n                        (8, 9)],\n                       [(3, 7),\n                        (6, 2),\n                        (8, 0.3158128697534015, 0.0, -0.8641235779241345),\n                        (9, 0.5, 0.0, -0.14524112269191658)],\n                       [(2, 0.8248194291522575, 0.0, -0.5316194891499073),\n                        (3, 0.5000000000000004, 0.0, 0.303921309479247),\n                        (4, 8),\n                        (6, 0.5000000000000001, 0.0, 0.18850613002727545),\n                        (9, 13)],\n                       [(2, 3),\n                        (7, 6),\n                        (8, 0.4999999999999997, 0.0, 0.5045749364837739),\n                        (9, 0.5541634199581221, 8.881784197001252e-16, 0.5469201838224107),\n                        (4, 0.44468851141909893, 0.0, 0.08614953711285173)],\n                       [(7, 0.5663648310033746, 5.551115123125783e-17, 0.10414010736335022),\n                        (3, 0.5, 0.0, 0.30392130947924656),\n                        (8, 9),\n                        (2, 0.18030068686283401, 5.551115123125783e-17, 0.4683805108500939)],\n                       [(7, 3),\n                        (2, 0),\n                        (8, 0.5000000000000003, 5.551115123125783e-17, 0.004574936483773884),\n                        (9, 13)],\n                       [(4, 8),\n                        (2, 0.5, 0.0, 0.9683805108500936),\n                        (7, 0.49999999999998246, 0.0, 0.6041400510454089),\n                        (3, 0.5000000000000022, -2.220446049250313e-16, -0.06334902851400415),\n                        (9, 0.5, 0.0, -0.9115255164330203),\n                        (0, 0.3040867239846962, 1.0, -0.5000000000000002),\n                        (13, 0.6170361769707173, 0.0, 0.6924350038915881)],\n                       [(6, 2),\n                        (7, 11),\n                        (8, 0.4999999999999998, 0.0, -0.9954250635162261),\n                        (13, 12),\n                        (4, 0.4999999999999998, -0.7674822332987776, 0.08614953711285106)],\n                       [(7, 0.5000000000000105, 0.0, 0.6041400510454089),\n                        (8, 9),\n                        (2, 0.8040867239846965, 0.0, 0.9683805108500936),\n                        (13, 0.8971367959436192, 0.0, -0.29226567734311537),\n                        (6, 0.0, 0.17257469853648622, 0.0),\n                        (11, 0.30408672398469616, 1.0, -0.5745192213521486),\n                        (12, 0.017475024019076853, 1.0, 0.12179973933251803)],\n                       [(7, 3),\n                        (2, 0),\n                        (9, 0.5, 0.0, 0.08847448356697907),\n                        (8, 0.5207317958688105, 0.0, -0.154749232999972)],\n                       [(9, 13),\n                        (2, 0.7499999999999999, 0.0, 0.9683805108500936),\n                        (7, 0.75, 0.0, 0.6041400185302308),\n                        (12, 8),\n                        (3, 0.49999999999999944, 0.0, 0.9366509389708142),\n                        (0, 0.5, 1.0, 0.5)],\n                       [(6, 2),\n                        (7, 11),\n                        (8, 0.748278824232631, 0.0, -0.6336375500593512),\n                        (9, 0.0, -0.6600415756556668, 0.0),\n                        (13, 0.5774545582024434, 2.220446049250313e-16, -0.3333513510078978),\n                        (12, 0.7518188205255605, 0.9999999999999998, 0.6217997393325212)],\n                       [(7, 0.500526889421938, 4.440892098500626e-16, -0.8963869175969332),\n                        (2, 0.5, 0.0, -0.03161948914990642),\n                        (8, 9),\n                        (13, 12),\n                        (6, 0.5, -0.9999999999999996, -0.7987688280048215),\n                        (11, 0.5000000000000103, 1.0, 0.42548077864785155)],\n                       [(7, 3),\n                        (2, 0),\n                        (13, 0.7841078409040306, 0.0, -0.8495265938991565),\n                        (12, 0.49999999999999917, 1.0, -0.37820026066747836),\n                        (8, 0.6029949675741022, 0.0, 0.3974292008207634)],\n                       [(9, 13),\n                        (2, 0.2500000000000001, 0.0, 0.9683805108500934),\n                        (7, 0.5, 0.0, 0.3536126463296333),\n                        (12, 8),\n                        (3, 0.5000000000000004, 0.0, -0.5633490610291858)],\n                       [(6, 2),\n                        (7, 11),\n                        (8, 0.0, 0.08301815506672527, 0.0),\n                        (9, 0.5, 0.0, -0.571567092088686),\n                        (13, 0.5010515534915843, 4.440892098500626e-16, 0.6504734061008433),\n                        (12, 0.5, 1.0, -0.8729260796176377)],\n                       [(2, 0.5, 0.0, -0.03161948914990642),\n                        (7, 3),\n                        (8, 9),\n                        (13, 12),\n                        (6, 0.5, -0.7012311719951787, -0.7987688280048215),\n                        (11, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(2, 0),\n                        (9, 0.5000000000000001, 0.0, 0.42843290791131494),\n                        (13, 0.5, -2.220446049250313e-16, 0.11492166123212405),\n                        (12, 8),\n                        (3, 0.0007451670218237008, -0.6628654265427709, -0.06334906102917826),\n                        (7, 3.251517832758566e-08, 0.9201728660984068, 0.35361262366414614)],\n                       [(9, 13),\n                        (12, 0.8270579430753595, 0.0, 0.5452777397572728),\n                        (0, 1.0, 0.0, 0.0),\n                        (2, 0.4999999999999998, 0.5316194891499064, 0.9683805108500936)],\n                       [(9, 0.4999999999999998, 0.0, 0.7246736831645262),\n                        (13, 0.8015402347134413, 0.0, -0.8850783387678769)],\n                       [(8, 9), (13, 12)],\n                       [(13, 0.4999999999999998, 0.0, 0.6149216612321231),\n                        (12, 0.11016031100871819, 0.0, -0.5495319604596214),\n                        (8, 0.5000000000000004, 0.0, 0.8013665302749651)],\n                       [(9, 13), (12, 8)],\n                       [(9, 0.5448817913860313, 1.1102230246251565e-16, 0.2246736831645268),\n                        (8, 0.4999999999999998, 0.0, 0.801366530274964),\n                        (13, 0.5, 0.0, -0.8401965473818445),\n                        (12, 0.5000000000000004, 0.0, -0.04953196045962027)],\n                       [(9, 8), (13, 12)],\n                       [(9, 0.4999999999999824, 0.0, 0.7246736268465859),\n                        (13, 0.49999999999998246, 0.0, 0.15980339630021412),\n                        (8, 0.5000000000000022, 0.0, 0.3911301130470264),\n                        (12, 0.5000000000000022, 0.0, 0.5402316223124419)],\n                       [(9, 10), (13, 14)],\n                       [(9, 0.5000000000000102, 0.0, 0.724673626846585),\n                        (13, 0.5000000000000102, 0.0, 0.15980339630021412),\n                        (10, 0.30408672398469616, 1.0, -0.5745192213521486),\n                        (14, 0.30408672398469616, 1.0, -0.5745192213521486)],\n                       [(9, 8), (13, 12)],\n                       [(9, 0.75, 0.0, 0.7246735943314073),\n                        (13, 0.75, 0.0, 0.159803363785036),\n                        (8, 0.49999999999999944, 0.0, -0.6088699194681544),\n                        (12, 0.49999999999999956, 0.0, -0.45976841020273973)],\n                       [(9, 10), (13, 14)],\n                       [(9, 0.500526889421938, 0.0, -0.7758533417957569),\n                        (13, 0.500526889421938, 0.0, 0.659276427657872),\n                        (10, 0.5000000000000103, 1.0, 0.42548077864785155),\n                        (14, 0.5000000000000103, 1.0, 0.42548077864785155)],\n                       [(9, 8), (13, 12)],\n                       [(9, 0.5, 0.0, 0.4741462221308108),\n                        (13, 0.5, 0.0, -0.09072400841556139),\n                        (8, 0.5000000000000004, 0.0, -0.10886991946815527),\n                        (12, 0.5000000000000003, 0.0, 0.04023158979726016)],\n                       [(9, 10), (13, 14)],\n                       [(9, 8),\n                        (13, 12),\n                        (10, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (14, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(8, 0.0007451670218237004, 0.8826554318961951, 0.3911300805318544),\n                        (9, 3.251517832758566e-08, 0.7996392902972314, 0.47414619946532205),\n                        (12, 0.0007451670218237004, 0.7335539226307821, 0.5402315897972689),\n                        (13, 3.251517832758566e-08, -0.6354904791563976, -0.09072403108104954)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8),\n                      (6, 5),\n                      (6, 6),\n                      (6, 7),\n                      (6, 8)]}}\n\ndef _round8_unitary(n,k):\n    data = _ROUND8_VERIFIED.get((n,k))\n    if data is None: return None\n    q = [cirq.GridQubit(*v) for v in data[\'qubits\']]\n    moments=[]\n    for layer in data[\'moments\']:\n        ops=[]\n        for gate in layer:\n            if len(gate)==2: ops.append(cirq.CZ(q[gate[0]],q[gate[1]]))\n            else:\n                i,x,z,a=gate\n                ops.append(cirq.PhasedXZGate(x_exponent=x,z_exponent=z,axis_phase_exponent=a).on(q[i]))\n        moments.append(cirq.Moment(ops))\n    return cirq.Circuit(moments),{q[i] for i in data[\'outputs\']}\n# ROUND 8 VERIFIED NATIVE CIRCUITS END\n\n# ROUND 9 VERIFIED NATIVE CIRCUITS START\n_ROUND9_VERIFIED = {(13, 6): {\'method\': \'joint small decoder cores + parallel three-site leaves\',\n           \'moments\': [[(2, 0.500000000000002, 0.0, -0.5000022602591274),\n                        (3, 0.4999999999999983, 0.0, -0.5000010665665804),\n                        (8, 0.49541528500807974, 0.0, -0.5000365774774151),\n                        (9, 0.7681707293706073, -4.440892098500626e-16, -0.4999505102253694),\n                        (4, 0.6149579653158789, -0.9999999999999996, -0.5000109442789036),\n                        (7, 0.4127792040228902, 0.0, -0.9262332254895422),\n                        (10, 0.5000000000000004, 0.0, -0.20251959709327183),\n                        (0, 0.5, 0.0, 0.5),\n                        (6, 0.5, 1.0, 0.5),\n                        (5, 0.5, 1.0, -0.377500967254607),\n                        (12, 0.3040867239846966, 0.0, -0.5745192213521528),\n                        (11, 0.3040867239846966, 1.0, -0.5745192213521528)],\n                       [(2, 3), (8, 9), (1, 4)],\n                       [(3, 0.3765635379368892, 0.0, 0.44713196173534),\n                        (8, 0.511066811346037, 0.0, -0.5567935679104996),\n                        (4, 0.48236800232547455, 0.0, -0.47095374290768666),\n                        (9, 0.37170018891529305, 0.0, 0.20406060098774398),\n                        (1, 0.5000000000000009, 0.0, 0.4913040262154016),\n                        (2, 0.6326231817760049, 5.551115123125783e-17, 0.4215138240609423)],\n                       [(3, 8), (4, 9)],\n                       [(3, 0.5, 0.0, -0.6327521085907859),\n                        (4, 0.4557551338830161, 0.0, -0.4633728734753797),\n                        (8, 0.7753768475063052, 0.0, -0.22420343153139433),\n                        (9, 0.5840258644382185, 5.551115123125783e-17, 0.2710559462840546)],\n                       [(3, 4), (8, 9)],\n                       [(1, 4),\n                        (3, 0.5000000000000002, 0.0, 0.3672478914092139),\n                        (8, 0.20953729491609777, 0.0, -0.21918610840379915),\n                        (9, 0.37153382263801715, 0.0, -0.7259316331787649)],\n                       [(2, 3),\n                        (8, 9),\n                        (4, 0.5000000000000108, 0.0, 0.5407629889040648),\n                        (1, 0.49999999999999556, -2.220446049250313e-16, 0.4913040262154018)],\n                       [(3, 0.5100987536847704, 1.0, -0.3973465950101658),\n                        (8, 0.3557882977813875, -2.220446049250313e-16, -0.16696578789704608),\n                        (9, 0.22883262133009263, 1.0, 0.19127823855873016),\n                        (2, 0.2941398626271703, 0.0, -0.06061150976377183)],\n                       [(3, 4)],\n                       [(3, 8), (4, 0.5000000000000027, 0.0, -0.45923701109593873)],\n                       [(4, 9),\n                        (3, 0.5000000000000004, 0.0, 0.6266021906836161),\n                        (8, 0.8505904906602456, 1.1102230246251565e-16, -0.28721377691605365)],\n                       [(4, 0.5000000000000024, 0.0, -0.6536094574452422),\n                        (2, 3),\n                        (8, 7),\n                        (9, 0.7894437395738274, 0.0, -0.4174526899976423)],\n                       [(1, 4),\n                        (9, 10),\n                        (7, 0.85928593962011, 0.0, -0.9262332254895425),\n                        (2, 0.5000000000000002, 1.1102230246251565e-16, 0.43938849023622717),\n                        (8, 0.5, 1.0, -0.9977179824223505)],\n                       [(4, 0.0, -0.30616287421943533, 0.0),\n                        (9, 0.6822968314790296, 0.0, 0.8413166285485951),\n                        (7, 2),\n                        (3, 8),\n                        (10, 0.2509322168477317, 0.0, -0.7949290191088039),\n                        (1, 0.49999999999999956, 0.0, 0.9768736810382279)],\n                       [(4, 9),\n                        (3, 0.5, 4.440892098500626e-16, 0.8282508768065648),\n                        (8, 0.6714165234362537, 4.440892098500626e-16, -0.926301890691358),\n                        (7, 0.7485956033393342, -4.440892098500626e-16, -0.42623322548954246)],\n                       [(2, 3),\n                        (8, 7),\n                        (9, 0.5, 0.0, 0.6190689304751498),\n                        (4, 0.572857359043784, 1.0, -0.40808611702042263)],\n                       [(9, 10),\n                        (1, 4),\n                        (3, 0.5, 0.0, 0.8282508768065648),\n                        (8, 0.3935294720438233, 1.0, -0.6472547750777364),\n                        (7, 0.9961572718865779, 1.0, -0.9262332254895478),\n                        (2, 0.05280605500441271, 0.0, -0.38701116597680274)],\n                       [(9, 0.2673990199988756, 0.0, -0.3809310695248507),\n                        (10, 0.5826592798249117, 0.0, 0.6305613443031177),\n                        (1, 0.4178598011240009, 0.0, -0.5231263189617721),\n                        (4, 0.0, 1.0, 0.0),\n                        (3, 8),\n                        (7, 2)],\n                       [(9, 10),\n                        (1, 4),\n                        (8, 0.3104369525321424, 0.0, 0.9342239745220648),\n                        (7, 0.5655532211464794, 0.0, -0.4262332254895531),\n                        (2, 0.24963665794604492, 0.0, 0.1303514189329718),\n                        (3, 0.0, -0.8912180282173903, 0.0)],\n                       [(4, 0.3856735769590187, 0.0, 0.7256844523827004),\n                        (9, 0.5505347324801626, 0.0, 0.11906893047514933),\n                        (8, 7),\n                        (2, 3),\n                        (10, 0.3689376239568078, 0.0, 0.7719771085165397),\n                        (1, 0.4999999999999997, 0.0, 0.9768736810382275)],\n                       [(4, 9),\n                        (8, 0.6193705712509705, 0.0, -0.19281033637733969),\n                        (7, 0.4999999999999998, 1.0, -0.4262332254895531),\n                        (2, 0.32701156232958084, 0.0, 0.7381317776448704)],\n                       [(3, 8),\n                        (4, 0.4633349244593255, 0.0, -0.2722518545188741),\n                        (9, 0.3213111929430337, 0.0, 0.6190689304751498),\n                        (7, 2)],\n                       [(4, 9),\n                        (8, 0.6256105257587086, 1.0, 0.2904600316322891),\n                        (7, 0.5960614607313279, 1.0, -0.11684358694965402),\n                        (2, 0.5, -2.220446049250313e-16, -0.014503690876253472)],\n                       [(9, 0.566656815524444, 0.0, -0.8809310695248502),\n                        (4, 0.43721065119527336, 0.9999999999999998, 0.563372320753559),\n                        (3, 8),\n                        (7, 2)],\n                       [(9, 10),\n                        (1, 4),\n                        (2, 0.5, 0.0, 0.485496309123746),\n                        (3, 0.5, 1.3322676295501878e-15, -0.7418957221033438),\n                        (8, 0.9764747842833552, 0.0, -0.11175572628831887),\n                        (7, 0.14949404734505523, 0.0, 0.2730671118828738)],\n                       [(2, 3),\n                        (4, 0.1039903575622847, 0.0, 0.1642735318533416),\n                        (9, 0.58504255233184, 4.440892098500626e-16, 0.6190689304751498),\n                        (8, 7),\n                        (10, 0.8311737009895825, 0.0, -0.0841139332337324),\n                        (1, 0.5, 0.0, -0.37307206031105533)],\n                       [(4, 9),\n                        (3, 0.4999999999999998, 0.0, -0.7418957221033433),\n                        (8, 0.5, 1.0, 0.13122255689355522),\n                        (2, 0.0, -0.9149679795284624, 0.0)],\n                       [(9, 0.49999999999999956, 0.0, 0.61906893047515),\n                        (4, 0.43415359559636596, 1.0, -0.06740113928787395),\n                        (3, 8),\n                        (7, 2)],\n                       [(9, 10),\n                        (1, 4),\n                        (2, 0.0, 1.0, 0.0),\n                        (3, 0.5, 0.0, -0.80908457612198),\n                        (8, 0.14731871270290628, 1.0, -0.8687774431064448),\n                        (7, 0.4247761378557593, 0.0, 0.9776181484214663)],\n                       [(4, 0.4999999999999998, 0.0, 0.1287155755730529),\n                        (9, 0.5998657277943454, 0.0, -0.28110725598163944),\n                        (2, 3),\n                        (8, 7),\n                        (1, 0.5530742553856172, 0.0, -0.8730720603110556)],\n                       [(4, 9),\n                        (8, 0.9433458052208397, 0.9999999999999999, 0.6312225568935546),\n                        (7, 0.45008695289946266, 0.0, 0.6634633138266384),\n                        (2, 0.4202007516434625, 1.0, 0.277090688003781)],\n                       [(4, 0.5000000000000004, 0.9999999999999999, -0.37128442442694687),\n                        (3, 8),\n                        (7, 2),\n                        (9, 2.220446049250313e-16, -0.9999999999999996, 0.7933212350593495)],\n                       [(1, 4),\n                        (9, 10),\n                        (2, 0.541088904736081, 0.0, -0.8181051971569446),\n                        (7, 0.541088904736081, 0.0, -0.7896451505492768),\n                        (3, 0.49999999999999994, 8.881784197001252e-16, -0.80908457612198),\n                        (8, 0.0, 1.0, 0.0)],\n                       [(2, 0),\n                        (4, 0.5, 0.0, -0.37128442442694687),\n                        (9, 0.5, 0.0, -0.45663580157216677),\n                        (7, 6),\n                        (10, 0.6339658219122923, 0.0, 0.8651689462862566),\n                        (1, 0.4999999999999997, -0.7868098827279499, 0.12692793968894484)],\n                       [(4, 9),\n                        (10, 5),\n                        (2, 0.4999999999999997, 1.0, -0.31810519715694374),\n                        (7, 0.5, 0.0, 0.710354849450723),\n                        (0, 0.30408672398469616, 0.0, -0.5000000000000002),\n                        (6, 0.3040867239846962, 1.0, -0.5000000000000002)],\n                       [(5, 0.03592107551278813, 8.881784197001252e-16, -0.8775009672546068),\n                        (4, 0.5991006024208341, 0.0, -0.2090846146593961),\n                        (3, 2),\n                        (8, 7),\n                        (9, 0.5, 1.0, -0.4566358015721663),\n                        (10, 0.9068032572794498, 1.1102230246251565e-16, 0.23132164282033318)],\n                       [(5, 4),\n                        (9, 10),\n                        (2, 0.8040867239846965, 0.0, 0.6818948028430556),\n                        (7, 0.8040867239846965, 0.0, -0.28964515054927653),\n                        (3, 0.0, -0.5498963368207652, 0.0)],\n                       [(2, 0),\n                        (4, 0.6983902210776689, 1.0, -0.42067032764493995),\n                        (7, 6),\n                        (10, 0.4457229604434565, 0.0, 0.01261445611041534),\n                        (5, 0.751811597304503, -0.9999999999999996, 0.6224990327453925)],\n                       [(4, 9),\n                        (10, 5),\n                        (2, 0.75, 0.0, 0.6818948028430556),\n                        (7, 0.7499999999999999, 1.1102230246251565e-16, -0.28964515054927675),\n                        (0, 0.5, 0.0, 0.5),\n                        (6, 0.5, 1.0, 0.5)],\n                       [(3, 2),\n                        (5, 0.5, 1.1102230246251565e-16, -0.3775009672546079),\n                        (4, 0.6251415065033563, 0.0, 0.43171823807993803),\n                        (8, 7),\n                        (9, 0.0, 0.6744500552705741, 0.0),\n                        (10, 0.7856407379834285, 0.0, -0.41370794677841705)],\n                       [(5, 4),\n                        (9, 10),\n                        (2, 0.5, -2.220446049250313e-16, -0.3181051971569442),\n                        (7, 0.5, 0.0, -0.28964515054927653),\n                        (3, 0.5, 5.551115123125783e-17, 0.12047463468921377),\n                        (8, 0.5, 0.0, 0.15176700926159414)],\n                       [(2, 0),\n                        (9, 0.5, 1.0, 0.21781425369840907),\n                        (7, 6),\n                        (10, 0.4949978217082433, -2.220446049250313e-16, 0.08629205322158295),\n                        (5, 0.4999999999999998, 1.0, -0.8814144405744395)],\n                       [(4, 9),\n                        (10, 5),\n                        (2, 0.25000000000000006, 0.0, 0.6818948028430556),\n                        (7, 0.25000000000000006, 0.0, 0.7103548494507228)],\n                       [(3, 2),\n                        (8, 7),\n                        (4, 0.0, 0.05676936615274908, 0.0),\n                        (9, 0.5, -0.9999999999999996, -0.7821857463015918),\n                        (10, 0.5, 0.0, -0.4475413783399338)],\n                       [(5, 4),\n                        (9, 10),\n                        (7, 0.5, 0.0, 0.710354849450723),\n                        (2, 0.5, 1.0, -0.3181051971569442),\n                        (3, 0.5, 0.37952536531078646, 0.12047463468921377),\n                        (8, 0.5, -0.6517670092615937, 0.15176700926159392)],\n                       [(9, 0.5, 0.0, 0.10187341435999242),\n                        (7, 6),\n                        (10, 0.8015319629186126, 0.0, -0.4475413783399338),\n                        (5, 0.2887068963335278, 0.0, -0.100328976188786),\n                        (2, 0)],\n                       [(4, 9),\n                        (10, 5),\n                        (6, 1.0, 0.0, 0.0),\n                        (0, 1.0, 0.0, 0.5),\n                        (2, 0.5, -0.18189480284305581, 0.6818948028430556),\n                        (7, 0.4999999999999998, -0.21035484945072325, -0.28964515054927675)],\n                       [(5, 0.10822807679614657, 0.0, -0.4511797912915503),\n                        (4, 0.5000000000000003, 1.0, 0.23944895989659837),\n                        (9, 0.0, 1.0, 0.0),\n                        (10, 0.4999999999999993, 8.881784197001252e-16, -0.947541378339934)],\n                       [(5, 4), (9, 10)],\n                       [(9, 0.5663325445329528, 4.440892098500626e-16, 0.6018734143599924),\n                        (4, 0.4999999999999997, 0.0, -0.760551040103401),\n                        (10, 0.5, 0.0, 0.6187911661930188),\n                        (5, 0.49999999999999967, 0.0, -0.9511797912915483)],\n                       [(9, 4), (10, 5)],\n                       [(9, 0.4999999999999824, 4.440892098500626e-16, -0.8981265293220673),\n                        (10, 0.49999999999998224, 0.0, -0.38120889012492176),\n                        (4, 0.500000000000002, 0.0, 0.6067838708306934),\n                        (5, 0.5000000000000022, 1.1102230246251565e-16, -0.3185147022256434)],\n                       [(9, 12), (10, 11)],\n                       [(9, 0.5000000000000102, 0.0, -0.8981265293220666),\n                        (10, 0.5000000000000102, 0.0, -0.38120889012492176),\n                        (12, 0.30408672398469627, 0.0, -0.5745192213521484),\n                        (11, 0.30408672398469616, 1.0, -0.5745192213521486)],\n                       [(9, 4), (10, 5)],\n                       [(9, 0.75, -4.440892098500626e-16, 0.10187350319311161),\n                        (10, 0.75, 0.0, -0.3812089226401003),\n                        (4, 0.4999999999999997, 0.0, 0.6067839033458748),\n                        (5, 0.49999999999999956, 0.0, 0.6814852652591754)],\n                       [(9, 12), (10, 11)],\n                       [(9, 0.5005268894219383, 1.1102230246251565e-16, -0.39759956067972446),\n                        (10, 0.5005268894219383, 0.0, 0.1182641412327356),\n                        (12, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515),\n                        (11, 0.5000000000000103, 1.0, 0.42548077864785155)],\n                       [(9, 4), (10, 5)],\n                       [(9, 0.5, 0.0, -0.6475991246062915),\n                        (10, 0.5, 0.0, -0.6317362948406968),\n                        (4, 0.5000000000000004, 0.0, -0.8932160966541245),\n                        (5, 0.5000000000000004, 0.0, -0.8185147347408246)],\n                       [(9, 12), (10, 11)],\n                       [(9, 4),\n                        (10, 5),\n                        (11, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (12, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(4, 0.9992548329781763, -0.332998390917816, 0.6067839033458664),\n                        (9, 0.999999967484822, 0.9213845917033567, -0.6475991019408034),\n                        (5, 0.0007451670218237004, -0.40769975283113236, -0.3185147347408168),\n                        (10, 3.251517832758567e-08, -0.09447819273126123, -0.6317363175061856)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],\n           \'qubits\': [(3, 4),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 3),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8),\n                      (6, 6)]},\n (14, 4): {\'method\': \'re-embedded count cores + four parallel three-site leaves\',\n           \'moments\': [[(1, 0.49999999999998634, 0.0, -0.500026480126484),\n                        (4, 0.49999999999999356, 0.0, -0.5000200091031044),\n                        (5, 0.6036217460678364, 0.0, -0.49999326614665796),\n                        (8, 0.3729697022062186, 0.0, -0.5000283724837651),\n                        (9, 0.8124337756695754, 0.0, -0.4999453471236337),\n                        (7, 0.22718552582850524, 0.0, -0.708982875827169),\n                        (10, 0.22718552582850524, 0.0, -0.708982875827169),\n                        (3, 0.35760927658307967, 0.0, -0.9848107895924711),\n                        (6, 0.35760927658307967, 0.0, -0.9848107895924711),\n                        (0, 0.5, 0.0, 0.5),\n                        (12, 0.3040867239846966, 0.0, -0.5745192213521528),\n                        (13, 0.3040867239846966, 0.0, -0.5745192213521528),\n                        (11, 0.3040867239846966, 0.0, -0.5745192213521528)],\n                       [(1, 4), (2, 5), (8, 9)],\n                       [(4, 0.3643833114082718, 0.0, 0.4639798578267951),\n                        (8, 0.503799087150674, 0.0, -0.6751071912304261),\n                        (5, 0.49199542950196484, 0.0, 0.6390788553564315),\n                        (9, 0.5451817162819368, 5.551115123125783e-17, 0.2208724430855708),\n                        (2, 0.5, 0.0, 0.4918471606308106),\n                        (1, 0.615639734890504, 0.0, 0.6231081777816672)],\n                       [(4, 8), (5, 9)],\n                       [(4, 0.5, 0.0, -0.3864437796005338),\n                        (5, 0.5367280445418148, 0.0, 0.6673281597967822),\n                        (8, 0.7088213817381512, 0.0, -0.2487189312517808),\n                        (9, 0.7796270498253168, 0.0, 0.1532429798663908)],\n                       [(4, 5), (8, 9)],\n                       [(2, 5),\n                        (4, 0.5, 0.0, 0.6135562203994662),\n                        (8, 0.24999999999999822, 0.0, -0.2487189312517808),\n                        (9, 0.3509372957212282, 0.0, -0.8467570201336094)],\n                       [(1, 4),\n                        (8, 9),\n                        (5, 0.5, 0.0, -0.308730229179877),\n                        (2, 0.5, 0.0, -0.5081528393691892)],\n                       [(4, 0.5000000000000402, 0.0, -0.13644377960052845),\n                        (8, 0.34769563521360997, 0.0, -0.04018256758513661),\n                        (9, 0.37787397844673176, 0.0, -0.04497561033344777),\n                        (1, 0.5, 0.0, 0.7038968302516679)],\n                       [(4, 5)],\n                       [(4, 8), (5, 0.5000000000000003, 1.1102230246251565e-16, -0.308730229179877)],\n                       [(5, 9),\n                        (4, 0.8916234504425224, 0.0, 0.20107986135695938),\n                        (8, 0.7715089830054349, 0.0, 0.11503686996070384)],\n                       [(1, 4),\n                        (8, 7),\n                        (5, 0.10837654955747907, 0.0, 0.5287934117776416),\n                        (9, 0.7715089830054349, 0.0, -0.69794358102764)],\n                       [(2, 5),\n                        (9, 10),\n                        (4, 0.5, 0.0, 0.7010798613569571),\n                        (8, 0.4999999999999998, 0.0, 0.0692642659115984),\n                        (7, 0.22718552582850488, 0.0, 0.29101712417283077)],\n                       [(4, 8),\n                        (5, 0.5, -2.220446049250313e-16, 0.028793411777640276),\n                        (9, 0.4999999999999998, 0.0, 0.2562838149233124),\n                        (10, 0.22718552582850488, 0.0, 0.29101712417283077)],\n                       [(5, 9), (8, 7), (1, 4)],\n                       [(2, 5),\n                        (9, 10),\n                        (4, 0.5886108001133223, 0.0, -0.3397935368859397),\n                        (8, 0.680195383285978, 4.440892098500626e-16, -0.9858623762583014),\n                        (7, 0.6354920868801647, 0.0, 0.5482021483143629)],\n                       [(4, 8),\n                        (5, 0.5886108001133226, 0.0, 0.06966681002053571),\n                        (9, 0.680195383285978, 0.0, 0.20115717275341227),\n                        (10, 0.6354920868801647, 0.0, 0.5482021483143629)],\n                       [(5, 9),\n                        (4, 0.5000000000000003, 0.0, 0.26671264908169645),\n                        (8, 0.499999999999999, 0.0, -0.2773840255191038)],\n                       [(1, 4),\n                        (5, 0.5000000000000004, 0.0, -0.5368393759471006),\n                        (8, 7),\n                        (9, 0.4999999999999994, 0.0, -0.0903644765073901)],\n                       [(2, 5),\n                        (9, 10),\n                        (4, 0.5234132252872343, 0.0, 0.4151483323626479),\n                        (8, 0.06708148302124252, 0.0, -0.504075212957263),\n                        (1, 0.5, 0.0, -0.2144167734270348)],\n                       [(4, 8),\n                        (5, 0.5234132252872346, 0.0, 0.3147249407719479),\n                        (9, 0.06708148302124252, 0.0, -0.3170556639455495),\n                        (2, 0.5, 0.0, -0.5975395948775823)],\n                       [(5, 9),\n                        (4, 0.33272914584056146, 0.0, -0.6309984476306965),\n                        (8, 0.059202003835620194, 0.0, -0.004075212957262764)],\n                       [(4, 8),\n                        (5, 0.33272914584056146, 0.0, -0.6391282792347082),\n                        (9, 0.05920200383562026, 0.0, -0.8170556639455491)],\n                       [(5, 9),\n                        (4, 0.2501185405482391, 0.0, 0.8690015523693038),\n                        (8, 0.08917758998520853, 0.0, 0.11983073142660938)],\n                       [(1, 4),\n                        (5, 0.2501185405482391, 0.0, -0.13912827923470905),\n                        (8, 7),\n                        (9, 0.08917758998520853, 0.0, 0.3068502804383222)],\n                       [(2, 5),\n                        (9, 10),\n                        (8, 0.5938461416699012, 0.0, -0.06776760663355841),\n                        (7, 0.021107237782608024, 0.0, -0.9231467397292097),\n                        (1, 0.4999999999999976, 0.32763712475589024, 0.785583226572969)],\n                       [(4, 8),\n                        (9, 0.5938461416699012, 0.0, 0.1192519423781544),\n                        (7, 3),\n                        (10, 0.021107237782608024, 0.0, -0.9231467397292097),\n                        (2, 0.5000000000000024, -0.289240053793554, 0.4024604051224139)],\n                       [(5, 9),\n                        (10, 6),\n                        (3, 0.8402337422326178, 0.0, 0.9498164599690078),\n                        (8, 0.4433968217557545, 0.0, -0.5851488026431948),\n                        (7, 0.3147365041110265, 0.0, -0.7244934550958735)],\n                       [(3, 4),\n                        (6, 0.8402337422326178, 0.0, 0.9498164599690075),\n                        (8, 7),\n                        (9, 0.4433968217557545, 0.0, 0.6018707463685189),\n                        (10, 0.3147365041110265, 0.0, -0.7244934550958735)],\n                       [(6, 5),\n                        (9, 10),\n                        (4, 0.5, 0.0, -0.16168691629918808),\n                        (8, 0.6391921844757946, 0.0, -0.0024666855840544955),\n                        (7, 0.9294528636932586, 4.440892098500626e-16, 0.5567952414001085),\n                        (3, 0.31397581370367167, 0.0, -0.21890167318297582)],\n                       [(4, 8),\n                        (5, 0.5, 0.0, -0.10843981056621677),\n                        (9, 0.6391921844757946, 0.0, -0.815447136572341),\n                        (7, 3),\n                        (10, 0.9294528636932586, 4.440892098500626e-16, 0.5567952414001085),\n                        (6, 0.3139758137036718, 8.881784197001252e-16, 0.7810983268170235)],\n                       [(5, 9),\n                        (10, 6),\n                        (4, 0.5000000000000004, 0.0, -0.16168691629918808),\n                        (8, 0.5, 0.0, -0.21740417840494763),\n                        (7, 0.33108034381799806, 0.0, 0.05679524140011294),\n                        (3, 0.5519644059311344, 0.0, 0.7987523594593542)],\n                       [(4, 8),\n                        (5, 0.5000000000000004, 0.0, -0.10843981056621677),\n                        (9, 0.5, 0.0, -0.030384629393234164),\n                        (7, 3),\n                        (10, 0.33108034381799806, 0.0, 0.05679524140011294),\n                        (6, 0.5519644059311344, 0.0, -0.2012476405406458)],\n                       [(5, 9),\n                        (10, 6),\n                        (3, 0.7831723914001073, 0.0, 0.6373214569840702),\n                        (4, 0.2429421210211464, 0.0, 0.9608570185161058),\n                        (7, 0.8371247201733047, 0.0, -0.8632896185761176)],\n                       [(3, 4),\n                        (6, 0.7831723914001073, 0.0, -0.36267854301592983),\n                        (5, 0.24294212102114646, 0.0, 0.7690162546184884),\n                        (8, 7),\n                        (10, 0.8371247201733047, 0.0, -0.8632896185761176)],\n                       [(6, 5),\n                        (9, 10),\n                        (3, 0.49999999999999956, 0.0, 0.7329995800098472),\n                        (4, 0.36381228346888317, 0.0, 0.1707440945749612),\n                        (8, 0.4960041929285068, 0.0, 0.2825958215950526)],\n                       [(3, 4),\n                        (6, 0.49999999999999906, 0.0, 0.7329995800098474),\n                        (5, 0.3638122834688833, 0.0, -0.4408708214403658),\n                        (8, 7),\n                        (9, 0.4960041929285068, 0.0, -0.5303846293932342)],\n                       [(6, 5),\n                        (9, 10),\n                        (4, 0.022839382910483597, 0.0, -0.3292559054250439),\n                        (8, 0.4999999999999997, 0.0, -0.21740417840494697),\n                        (7, 0.8057964070938461, 0.0, -0.048305353185546185),\n                        (3, 0.7913981257738136, 0.0, 0.7329995800098474)],\n                       [(4, 8),\n                        (5, 0.022839382910483607, 0.0, 0.05912917855963917),\n                        (9, 0.4999999999999998, 0.0, 0.9696153706067667),\n                        (7, 3),\n                        (10, 0.8057964070938461, 0.0, -0.048305353185546185),\n                        (6, 0.7913981257738136, 1.1102230246251565e-16, -0.26700041999015234)],\n                       [(5, 9),\n                        (10, 6),\n                        (3, 0.75, 0.0, 0.7329995800098461),\n                        (4, 0.22716061708951618, 0.0, 0.6707440945749619),\n                        (8, 0.5, 0.0, -0.865348969532548)],\n                       [(3, 4),\n                        (6, 0.75, 0.0, -0.2670004199901532),\n                        (5, 0.22716061708951615, 0.0, -0.9408708214403674),\n                        (8, 7),\n                        (9, 0.5, 0.0, -0.6783294205208339)],\n                       [(6, 5),\n                        (9, 10),\n                        (3, 0.9427632774999226, 0.0, -0.7670004199901537),\n                        (8, 0.20292068705287525, 0.0, -0.36534896953254825),\n                        (4, 0.4999999999999998, 4.440892098500626e-16, 0.6707440945749605)],\n                       [(3, 0),\n                        (8, 4),\n                        (9, 0.20292068705287525, 0.0, -0.1783294205208339),\n                        (5, 0.5, 4.440892098500626e-16, -0.9408708214403663),\n                        (10, 0.5, 1.1102230246251565e-16, -0.16963777986712802),\n                        (6, 0.5, 0.0, -0.2670004199901532)],\n                       [(9, 5),\n                        (10, 6),\n                        (3, 0.4999999999999997, 0.0, -0.26700041999015367),\n                        (8, 0.4999999999999827, 0.0, 0.13465097414951133),\n                        (0, 0.30408672398469605, 0.0, -0.5000000000000002),\n                        (4, 0.500000000000002, 4.440892098500626e-16, 0.6827195027194035)],\n                       [(7, 3),\n                        (8, 12),\n                        (9, 0.49999999999998246, 0.0, -0.6783294768387744),\n                        (10, 0.4999999999999824, 0.0, -0.16963772354918705),\n                        (5, 0.5000000000000022, 0.0, -0.952846229584809),\n                        (6, 0.5000000000000019, 0.0, -0.7766661176341456)],\n                       [(9, 13),\n                        (10, 11),\n                        (3, 0.8040867239846965, 0.0, 0.7329995800098463),\n                        (8, 0.5000000000000103, 0.0, 0.13465097414951094),\n                        (12, 0.30408672398469605, 0.0, 0.4254807786478514)],\n                       [(3, 0),\n                        (8, 4),\n                        (9, 0.5000000000000102, 0.0, -0.6783294768387744),\n                        (10, 0.5000000000000102, 1.1102230246251565e-16, -0.16963772354918616),\n                        (13, 0.30408672398469605, 0.0, 0.4254807786478514),\n                        (11, 0.30408672398469627, 0.0, -0.5745192213521484)],\n                       [(9, 5),\n                        (10, 6),\n                        (3, 0.75, 0.0, 0.7329995800098463),\n                        (8, 0.75, 0.0, -0.8653490583656671),\n                        (0, 0.5, 0.0, 0.5),\n                        (4, 0.4999999999999995, 0.0, -0.3172804647654144)],\n                       [(7, 3),\n                        (8, 12),\n                        (9, 0.7499999999999999, 0.0, -0.6783295093539528),\n                        (10, 0.75, 0.0, 0.8303623089659917),\n                        (5, 0.49999999999999956, 0.0, 0.04715373790001043),\n                        (6, 0.49999999999999956, -2.220446049250313e-16, -0.7766660851189633)],\n                       [(9, 13),\n                        (10, 11),\n                        (3, 0.5000000000000002, 0.0, 0.7329995800098474),\n                        (8, 0.5005268894219383, 2.220446049250313e-16, -0.36587599449283115),\n                        (7, 0.5, -4.440892098500626e-16, -0.44342329229517063),\n                        (12, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515)],\n                       [(3, 0),\n                        (8, 4),\n                        (9, 0.500526889421938, 0.0, -0.17885644548111657),\n                        (10, 0.500526889421938, 0.0, 0.33088924509315576),\n                        (13, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515),\n                        (11, 0.5000000000000102, 0.0, 0.4254807786478514)],\n                       [(9, 5),\n                        (10, 6),\n                        (3, 0.25000000000000006, 0.0, -0.26700041999015234),\n                        (8, 0.5, 0.0, -0.11587643056626407),\n                        (4, 0.5000000000000003, -2.220446049250313e-16, -0.8172804647654144)],\n                       [(7, 3),\n                        (8, 12),\n                        (9, 0.5, 0.0, -0.9288568815545499),\n                        (10, 0.5, 5.551115123125783e-17, 0.08088968116658862),\n                        (5, 0.5000000000000004, 0.0, 0.5471537379000093),\n                        (6, 0.5000000000000003, 1.1102230246251565e-16, -0.27666608511896307)],\n                       [(9, 13),\n                        (10, 11),\n                        (8, 4),\n                        (3, 0.5, 1.1102230246251565e-16, -0.2670004199901541),\n                        (7, 0.5, 0.9434232922951704, 0.5565767077048294),\n                        (12, 0.5, 0.3483047662953689, -0.5745192538673269)],\n                       [(9, 5),\n                        (10, 6),\n                        (3, 0),\n                        (11, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (13, 0.5, 0.3483047662953689, -0.5745192538673269),\n                        (4, 0.9992548329781763, 0.5910659771934725, -0.31728046476542193),\n                        (8, 3.251517832758566e-08, 0.38966194299430557, 0.8841235467682478)],\n                       [(5, 0.0007451670218237008, 0.22663177452803218, -0.9528462620999816),\n                        (6, 0.9992548329781763, -0.9495484024529777, -0.7766660851189717),\n                        (9, 3.251517832758567e-08, 0.20264239398259232, -0.9288569042200383),\n                        (10, 0.999999967484822, 0.19289578593047613, 0.08088970383207611),\n                        (0, 1.0, 0.0, 0.5),\n                        (3, 0.5, 0.7670004199901536, -0.26700041999015234)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8),\n                      (6, 5),\n                      (6, 6)]},\n (14, 5): {\'method\': \'joint small decoder cores + parallel three-site leaves\',\n           \'moments\': [[(0, 0.49999999999999356, 0.0, -0.4999954690927664),\n                        (4, 0.49999999999999334, 0.0, -0.4999960678964577),\n                        (1, 0.5, 0.0, -0.4999901078631839),\n                        (5, 0.5000000000000004, 0.0, -0.49998659960605574),\n                        (9, 0.6533159216672457, 0.0, -0.5000238842365659),\n                        (10, 0.6611590636539217, 0.0, -0.4999486103438322),\n                        (8, 0.49999999999999895, 0.0, -0.7088499809721285),\n                        (11, 0.4999999999999989, 0.9999999999999999, -0.7088499809721285),\n                        (3, 0.48497757661839347, 0.0, -0.834800922923292),\n                        (6, 0.48497757661839347, 0.0, -0.834800922923292),\n                        (2, 0.5, 0.0, 0.5),\n                        (7, 0.3040867239846966, 0.0, -0.5745192213521528),\n                        (12, 0.3040867239846966, 0.0, -0.5745192213521528),\n                        (13, 0.3040867239846966, 1.0, -0.5745192213521528)],\n                       [(0, 4), (1, 5), (9, 10)],\n                       [(4, 0.25124509144124135, 0.0, 0.5524560480887117),\n                        (9, 0.6467743327810145, 0.0, -0.4949283175125294),\n                        (5, 0.5874688511443722, 1.1102230246251565e-16, -0.4894718625997523),\n                        (10, 0.015172117930396967, 0.0, -0.6930394984558987),\n                        (0, 0.2530712542117881, 0.0, 0.5521579304469275),\n                        (1, 0.5880329839967557, 4.440892098500626e-16, 0.5481050123584834)],\n                       [(4, 9), (5, 10)],\n                       [(5, 0.5, 0.0, -0.4403399733737712),\n                        (9, 0.75, -0.9999999999999998, -0.5047156331421747)],\n                       [(4, 5), (9, 10)],\n                       [(4, 0.5, 0.0, 0.13310247619245907),\n                        (5, 0.5, 0.0, 0.5596600266262288),\n                        (9, 0.25000000000000006, 1.0, -0.5047156331421749),\n                        (10, 0.5, 0.0, -0.4918603869859073)],\n                       [(0, 4), (1, 5), (9, 10)],\n                       [(4, 0.35852705142936125, 0.0, -0.3668975238075407),\n                        (5, 0.5000000000000004, 0.0, -0.4403399733737712),\n                        (10, 0.5000000000000004, 0.0, 0.508139613014092),\n                        (1, 0.4999999999999998, 1.0, -0.44889723347396426)],\n                       [(4, 5)],\n                       [(4, 0.4999999999999998, 0.0, -0.8668975238075407), (5, 10)],\n                       [(4, 9),\n                        (5, 0.5882500769209105, 0.0, -0.33351866318604806),\n                        (10, 0.5000000000000009, 0.0, -0.01784973028266501)],\n                       [(4, 0.41174992307908953, 4.440892098500626e-16, -0.9299127206351274),\n                        (9, 0.5000000000000009, 0.0, 0.021273710154581904),\n                        (1, 5),\n                        (10, 11)],\n                       [(0, 4),\n                        (9, 8),\n                        (5, 0.9675670579641595, 0.0, -0.33351866318603207),\n                        (10, 0.49280504227479455, 0.0, -0.8464930449205792),\n                        (11, 0.46084833097988237, 0.9999999999999998, 0.8309193012950313)],\n                       [(4, 0.9675670579641595, 0.0, 0.07008727936485659),\n                        (9, 0.4928050422747947, 0.0, 0.8499170247924959),\n                        (5, 10),\n                        (8, 0.4608483309798826, -2.220446049250313e-16, 0.8309193012950313)],\n                       [(4, 9),\n                        (5, 0.5000000000000003, 0.0, 0.6664813368139852),\n                        (10, 0.5596506416568943, 0.0, 0.24260415400493868)],\n                       [(4, 0.5000000000000004, 0.0, 0.07008727936483972),\n                        (9, 0.5596506416568943, 0.0, 0.7608198258669778),\n                        (5, 10)],\n                       [(4, 9),\n                        (5, 0.1275470395921725, 0.0, 0.8188671937430758),\n                        (10, 0.5000000000000003, 0.0, -0.7362946733550011)],\n                       [(1, 5),\n                        (9, 0.5000000000000004, 0.0, 0.7397186532269178),\n                        (4, 0.12754703959217256, 1.0, 0.9177014224357487),\n                        (10, 11)],\n                       [(9, 8),\n                        (0, 4),\n                        (5, 0.2696276353165252, 0.0, -0.2387772102851784),\n                        (10, 0.16304676742985036, 0.0, 0.26370532664499713),\n                        (1, 0.0, -0.31555683986653227, 0.0),\n                        (11, 0.8273677055841601, 1.0, 0.011025144178165802)],\n                       [(4, 0.2696276353165253, 0.0, 0.9753458264640031),\n                        (9, 0.16304676742985055, 0.0, -0.26028134677308),\n                        (5, 10),\n                        (0, 0.0, -0.31555683986653227, 0.0),\n                        (8, 0.8273677055841601, 0.0, 0.011025144178165858)],\n                       [(4, 9),\n                        (5, 0.3174136387017352, 0.0, 0.20314303100964093),\n                        (10, 0.704332640729217, 2.220446049250313e-16, -0.23629467335500287)],\n                       [(4, 0.3174136387017352, 0.0, -0.4665744148308164),\n                        (9, 0.704332640729217, 4.440892098500626e-16, -0.7602813467730796),\n                        (1, 5),\n                        (10, 11)],\n                       [(0, 4),\n                        (9, 8),\n                        (5, 0.6276139104334912, 1.1102230246251565e-16, 0.10723592121484771),\n                        (10, 0.5, 0.0, -0.23629467335500376),\n                        (1, 0.0, 1.0, 0.0),\n                        (11, 0.727371054243577, 0.9999999999999999, -0.07318377315388713)],\n                       [(4, 0.6276139104334912, 0.0, -0.3706673050360223),\n                        (9, 0.5, 0.0, -0.7602813467730791),\n                        (5, 10),\n                        (8, 0.727371054243577, 0.0, -0.07318377315388713)],\n                       [(4, 9),\n                        (5, 0.4999999999999985, 0.0, 0.0019021884544975776),\n                        (10, 0.5290907385534421, 8.881784197001252e-16, -0.8668911260860583)],\n                       [(4, 0.4999999999999989, 0.0, 0.7346664277243273),\n                        (9, 0.529090738553442, 4.440892098500626e-16, 0.870315105957975),\n                        (1, 5),\n                        (10, 11)],\n                       [(0, 4),\n                        (9, 8),\n                        (5, 0.6683089837594863, 3.3306690738754696e-16, -0.49809781154550326),\n                        (10, 0.4999999999999995, -2.220446049250313e-16, 0.13960534386056356),\n                        (1, 0.5000000000000003, 0.9999999999999998, 0.6121267990399082),\n                        (11, 0.49999999999999956, 1.0, -0.09727132543917172)],\n                       [(4, 0.6683089837594859, 0.0, -0.7653335722756722),\n                        (9, 0.49999999999999967, 1.1102230246251565e-16, -0.1361813639886469),\n                        (5, 10),\n                        (0, 0.5000000000000004, 0.0, -0.04126483078056853),\n                        (8, 0.49999999999999956, 0.0, -0.09727132543917172)],\n                       [(4, 9), (5, 0.6578349175114986, 0.0, 0.5019021884544976), (10, 11)],\n                       [(4, 0.6578349175114986, 0.0, -0.7653335722756722),\n                        (9, 8),\n                        (1, 5),\n                        (10, 0.5000000000000009, 4.440892098500626e-16, -0.8603946561394358),\n                        (11, 0.6530017079301533, -2.220446049250313e-16, 0.4027286745608292)],\n                       [(0, 4),\n                        (9, 0.5000000000000009, -4.440892098500626e-16, -0.13618136398864777),\n                        (5, 0.5, 8.881784197001252e-16, -0.9979681050609541),\n                        (8, 0.6530017079301533, -2.220446049250313e-16, 0.4027286745608292),\n                        (11, 6),\n                        (1, 0.4999999999999998, 0.37432232533611454, -0.38787320096009203)],\n                       [(4, 0.5, 0.0, 0.7345367212397786),\n                        (5, 10),\n                        (8, 3),\n                        (6, 0.5605616196317902, 1.0, -0.8348009229232944),\n                        (11, 0.5000000000000003, 1.0, -0.09727132543918593),\n                        (0, 0.4999999999999998, -0.972286044843409, 0.9587351692194317)],\n                       [(4, 9),\n                        (3, 0.5605616196317902, 0.0, -0.8348009229232944),\n                        (8, 0.5000000000000003, 1.1102230246251565e-16, -0.09727132543918637),\n                        (5, 0.5, 0.0, 0.0020318949390460417),\n                        (10, 11)],\n                       [(4, 0.5, 0.0, 0.7345367212397786),\n                        (9, 8),\n                        (6, 5),\n                        (11, 0.4311009172299638, 0.0, -0.34731902068777165),\n                        (10, 0.5, 0.0, -0.09712668994318818)],\n                       [(3, 4),\n                        (8, 0.4311009172299638, 0.0, -0.34731902068777165),\n                        (6, 0.5818042231825744, -2.220446049250313e-16, -0.3348009229233022),\n                        (9, 0.5, 1.0, 0.10055066981510569),\n                        (5, 0.0, -0.95903976827727, 0.0)],\n                       [(3, 0.5818042231825744, -2.220446049250313e-16, -0.3348009229233022),\n                        (11, 6),\n                        (4, 9),\n                        (5, 10)],\n                       [(8, 3),\n                        (11, 0.03454957128703273, 0.0, 0.8181166636420727),\n                        (6, 0.2894984978675621, 0.0, -0.8348009229233004),\n                        (4, 0.24304696816073568, 0.0, 0.3287607528844717),\n                        (9, 0.5, 1.0, 0.10055066981510569),\n                        (5, 0.24304696816073573, 1.0, 0.4487680950170834),\n                        (10, 0.5, 0.0, -0.09712668994318818)],\n                       [(8, 0.03454957128703273, 0.0, 0.8181166636420727),\n                        (3, 0.2894984978675621, 0.0, -0.8348009229233004),\n                        (11, 6),\n                        (4, 9),\n                        (5, 10)],\n                       [(8, 3),\n                        (6, 0.87386354985307, 0.0, 0.6651990770766809),\n                        (5, 0.30568317284628654, 0.0, 0.8012111736991561),\n                        (4, 0.30568317284628654, 0.0, -0.023682325797601322),\n                        (11, 0.9242869536756131, -0.9999999999999996, 0.5858061491772155)],\n                       [(6, 5),\n                        (8, 0.9242869536756131, 8.881784197001252e-16, 0.5858061491772155),\n                        (3, 0.87386354985307, 0.9999999999999998, 0.6651990770766807),\n                        (10, 11)],\n                       [(9, 8),\n                        (3, 4),\n                        (11, 0.23753846872505302, 0.0, 0.11812387065190871),\n                        (6, 0.6888649043845242, 0.0, -0.8348009229233393),\n                        (5, 0.5595532320705698, 1.0, 0.38330886751287974),\n                        (10, 0.5, 0.0, -0.5969969834586397)],\n                       [(8, 0.23753846872505302, 0.0, 0.11812387065190871),\n                        (3, 0.6888649043845242, 0.0, -0.8348009229233393),\n                        (11, 6),\n                        (4, 0.5595532320705698, 0.0, 0.3942199803886754),\n                        (9, 0.5, 0.9999999999999998, 0.6004209633305568),\n                        (5, 10)],\n                       [(8, 3),\n                        (4, 9),\n                        (6, 0.49999999999999956, 1.0, -0.8348009229233395),\n                        (10, 0.5, 0.0, -0.5969969834586388),\n                        (11, 0.6383222145074643, 0.9999999999999998, 0.17679703477273212)],\n                       [(3, 0.49999999999999956, 0.0, -0.8348009229233395),\n                        (4, 0.0, -0.372707453521945, 0.0),\n                        (9, 0.4999999999999998, 0.0, 0.6004209633305562),\n                        (8, 0.6383222145074643, 0.0, 0.17679703477273234),\n                        (6, 5),\n                        (10, 11)],\n                       [(3, 4),\n                        (9, 8),\n                        (10, 0.5, 0.0, -0.7874214849988537),\n                        (11, 0.4999999999999997, 0.0, -0.7484872360058605),\n                        (6, 0.4546007179287068, 0.0, 0.4167387613319282)],\n                       [(9, 0.5, 0.0, 0.790845464870771),\n                        (5, 10),\n                        (8, 0.4999999999999997, 0.0, -0.7484872360058605),\n                        (3, 0.4546007179287068, 0.0, 0.4167387613319282),\n                        (11, 6)],\n                       [(4, 9),\n                        (8, 3),\n                        (6, 0.19504058663172444, 1.0, -0.2857088984950029),\n                        (5, 0.2978230143957523, 0.0, -0.26170178457277204),\n                        (11, 0.8858944659256832, -0.9999999999999998, 0.25151276399414063)],\n                       [(3, 0.19504058663172466, 0.0, -0.2857088984950029),\n                        (4, 0.2978230143957523, 0.0, -0.333476821047618),\n                        (8, 0.8858944659256832, 5.551115123125783e-17, 0.25151276399414046),\n                        (6, 5),\n                        (10, 11)],\n                       [(3, 4),\n                        (9, 8),\n                        (6, 0.5271017512479523, 5.551115123125783e-17, 0.1739679588648032),\n                        (10, 0.21266361194798108, 0.0, -0.2874214849988519),\n                        (5, 0.5000000000000004, 4.440892098500626e-16, 0.5356708680164881)],\n                       [(6, 2),\n                        (8, 0.5, 1.6653345369377348e-16, 0.038849152046160074),\n                        (3, 0.5000000000000007, 0.0, 0.6739679588648033),\n                        (9, 0.21266361194798092, 0.0, -0.7091545351292301),\n                        (4, 0.5000000000000003, 0.0, 0.8691505263631218),\n                        (10, 5)],\n                       [(8, 3),\n                        (9, 4),\n                        (6, 0.5, 0.0, 0.6739679588648031),\n                        (10, 0.49999999999998224, 0.0, 0.2125784586832058),\n                        (2, 0.30408672398469616, 0.0, -0.5000000000000002),\n                        (5, 0.5000000000000022, 0.0, 0.4609980919124501)],\n                       [(8, 0.4999999999999824, -2.220446049250313e-16, 0.038849208364101384),\n                        (9, 0.4999999999999824, 0.0, -0.20915447881128824),\n                        (11, 6),\n                        (10, 13),\n                        (3, 0.5000000000000019, 0.0, -0.2513592650311596),\n                        (4, 0.5000000000000022, 0.0, -0.056176697532839714)],\n                       [(8, 7),\n                        (9, 12),\n                        (6, 0.8040867239846965, 3.3306690738754696e-16, -0.32603204113519757),\n                        (10, 0.5000000000000102, 0.0, 0.2125784586832058),\n                        (13, 0.30408672398469616, 1.0, -0.5745192213521486)],\n                       [(6, 2),\n                        (8, 0.5000000000000103, 0.0, 0.03884920836410144),\n                        (9, 0.5000000000000103, 0.0, -0.209154478811288),\n                        (10, 5),\n                        (7, 0.30408672398469627, 0.0, -0.5745192213521484),\n                        (12, 0.30408672398469627, 0.0, -0.5745192213521484)],\n                       [(8, 3),\n                        (9, 4),\n                        (10, 0.75, 5.551115123125783e-17, 0.21257842616802763),\n                        (6, 0.75, -0.9999999999999996, -0.32603204113519735),\n                        (2, 0.5, 0.0, 0.5),\n                        (5, 0.49999999999999956, 0.0, -0.5390019406027315)],\n                       [(8, 0.75, 4.440892098500626e-16, -0.9611507591207205),\n                        (9, 0.75, 4.440892098500626e-16, 0.7908455537038903),\n                        (10, 13),\n                        (11, 6),\n                        (3, 0.49999999999999956, 0.0, -0.2513592325159779),\n                        (4, 0.49999999999999944, -2.220446049250313e-16, -0.056176665017658056)],\n                       [(8, 7),\n                        (9, 12),\n                        (6, 0.5000000000000001, 0.0, 0.6739679588648031),\n                        (10, 0.5005268894219383, 0.0, 0.7120514900408637),\n                        (13, 0.5000000000000103, 1.0, 0.42548077864785155),\n                        (11, 0.5, 0.0, -0.23493636038188193)],\n                       [(6, 2),\n                        (8, 0.500526889421938, 0.0, 0.5393761770064442),\n                        (9, 0.500526889421938, 5.551115123125783e-17, 0.29137248983105524),\n                        (10, 5),\n                        (7, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515),\n                        (12, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515)],\n                       [(8, 3),\n                        (9, 4),\n                        (10, 0.5, 0.0, -0.03794894603256971),\n                        (6, 0.25000000000000006, -0.9999999999999998, -0.32603204113519735),\n                        (5, 0.5000000000000003, 1.1102230246251565e-16, -0.03900194060273199)],\n                       [(8, 0.5, 5.551115123125783e-17, 0.28937661307987683),\n                        (9, 0.5000000000000001, 0.0, 0.041372925904488156),\n                        (10, 13),\n                        (11, 6),\n                        (3, 0.5000000000000004, 0.0, 0.24864076748402253),\n                        (4, 0.5000000000000003, 0.0, 0.4438233349823423)],\n                       [(8, 7),\n                        (9, 12),\n                        (10, 5),\n                        (6, 0.5, 0.9999999999999999, 0.6739679588648031),\n                        (13, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (11, 0.5, 0.7349363603818819, 0.7650636396181181)],\n                       [(8, 3),\n                        (9, 4),\n                        (6, 2),\n                        (7, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (12, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (5, 0.0007451670218237004, 0.8127874530307747, 0.4609980593972761),\n                        (10, 3.251517832758567e-08, -0.6882655415393872, -0.03794896869805853)],\n                       [(2, 1.0, 0.0, 0.5),\n                        (3, 0.9992548329781763, 0.5251447449440367, -0.2513592325159859),\n                        (4, 0.9992548329781763, 0.329962177445717, -0.05617666501766605),\n                        (6, 0.5, 0.8260320411351971, -0.3260320411351971),\n                        (8, 0.999999967484822, -0.01559114598281175, 0.2893766357453651),\n                        (9, 0.999999967484822, 0.23241254119257754, 0.04137294856997625)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],\n           \'qubits\': [(3, 5),\n                      (3, 6),\n                      (3, 7),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 3),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 5),\n                      (6, 6)]},\n (14, 6): {\'method\': \'re-embedded count cores + four parallel three-site leaves\',\n           \'moments\': [[(1, 0.5000000000000222, 0.0, -0.5000127089909743),\n                        (4, 0.5000000000000089, 0.0, -0.5000272098567784),\n                        (8, 0.4923777602353679, 0.0, -0.5000109181322072),\n                        (9, 0.7931325315051736, 0.0, -0.4999627409699672),\n                        (5, 0.6153405909564915, 1.0, -0.49999340340948883),\n                        (7, 0.5000000000000004, 0.0, -0.20251959709327183),\n                        (10, 0.5000000000000004, 0.0, -0.20251959709327183),\n                        (3, 0.5, 1.0, -0.377500967254607),\n                        (6, 0.5, 1.0, -0.377500967254607),\n                        (0, 0.5, 1.0, 0.5),\n                        (12, 0.3040867239846966, 0.0, -0.5745192213521528),\n                        (13, 0.3040867239846966, 0.0, -0.5745192213521528),\n                        (11, 0.3040867239846966, 1.0, -0.5745192213521528)],\n                       [(1, 4), (8, 9), (2, 5)],\n                       [(4, 0.3762289699343595, 0.0, 0.4944874827273811),\n                        (8, 0.5526873801036829, 0.0, -0.5812920188006019),\n                        (5, 0.4821610746611873, 0.0, -0.47866053397691966),\n                        (9, 0.28164143645876283, 0.0, 0.20643007632094068),\n                        (1, 0.6174075601975617, 5.551115123125783e-17, 0.37813525292185135),\n                        (2, 0.5, 0.0, 0.49132006815924034)],\n                       [(4, 8), (5, 9)],\n                       [(4, 0.5, 0.0, -0.6401408104079163),\n                        (5, 0.42673338055905957, 1.1102230246251565e-16, -0.47436665137877077),\n                        (8, 0.7025171462389753, 0.0, -0.26039765421675387),\n                        (9, 0.7336925494434161, 5.551115123125783e-17, 0.27807672105552783)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.5, 0.0, 0.35985918959208396),\n                        (2, 5),\n                        (8, 0.26577321596593323, 0.0, -0.24231128170287097),\n                        (9, 0.3148494271028235, 0.0, -0.7058796266375078)],\n                       [(1, 4),\n                        (8, 9),\n                        (5, 0.5000000000000002, 0.0, 0.5296610679801552),\n                        (2, 0.4999999999999998, 0.0, 0.49132006815924034)],\n                       [(4, 0.5000000000000244, 1.0, -0.39014081040791426),\n                        (8, 0.445783672071554, 0.0, -0.22344345479930827),\n                        (9, 0.2595545370930981, 1.0, 0.24325902178973718),\n                        (1, 0.4999999999999998, 0.0, -0.6623412594771172)],\n                       [(4, 5)],\n                       [(4, 8), (5, 0.4999999999999993, 1.1102230246251565e-16, -0.4703389320198452)],\n                       [(5, 9),\n                        (4, 0.5000000000000009, 0.0, 0.9171992223436849),\n                        (8, 0.1916584703469213, 0.0, 0.13356718354563135)],\n                       [(1, 4),\n                        (5, 0.5000000000000003, 0.0, -0.6629988992682665),\n                        (8, 7),\n                        (9, 0.8083415296531204, 1.1102230246251565e-16, -0.36554577562212964)],\n                       [(2, 5),\n                        (9, 10),\n                        (8, 0.6822968314790296, 0.9999999999999996, 0.8580507055866178),\n                        (7, 0.2509322168477317, 0.0, -0.7949290191088039),\n                        (1, 0.4999999999999994, 1.0, -0.14626241929311146)],\n                       [(5, 0.0, -0.3095232681505884, 0.0),\n                        (9, 0.6822968314790296, 0.0, 0.9362841347313369),\n                        (4, 8),\n                        (10, 0.2509322168477317, 0.0, -0.7949290191088039),\n                        (2, 0.49999999999999956, 0.0, 0.9770020858515105)],\n                       [(5, 9),\n                        (8, 0.5, 0.0, 0.6358030075131724),\n                        (4, 0.5728573590437841, 0.0, 0.3655130076994271)],\n                       [(8, 7),\n                        (9, 0.5, 0.0, 0.7140364366578915),\n                        (1, 4),\n                        (5, 0.5728573590437841, 1.0, -0.4208359527745986)],\n                       [(9, 10),\n                        (2, 5),\n                        (8, 0.26739901999887583, 0.0, -0.364196992486828),\n                        (7, 0.5826592798249117, 0.0, 0.6305613443031177),\n                        (1, 0.41785980112400095, 1.0, -0.6462624192931115)],\n                       [(8, 7),\n                        (9, 0.26739901999887583, 0.0, -0.2859635633421087),\n                        (10, 0.5826592798249117, 0.0, 0.6305613443031177),\n                        (1, 4),\n                        (2, 0.41785980112400095, 0.0, -0.52299791414849),\n                        (5, 0.0, 1.0, 0.0)],\n                       [(9, 10),\n                        (2, 5),\n                        (4, 0.3856735769590187, 0.0, -0.768257561703696),\n                        (8, 0.5505347324801627, 1.0, 0.13580300751317187),\n                        (7, 0.3689376239568078, 0.0, 0.7719771085165397),\n                        (1, 0.4999999999999998, 1.0, -0.14626241929311123)],\n                       [(5, 0.3856735769590187, 0.0, 0.7129346166285242),\n                        (9, 0.5505347324801628, 1.6653345369377348e-16, 0.21403643665789113),\n                        (4, 8),\n                        (10, 0.3689376239568078, 0.0, 0.7719771085165397),\n                        (2, 0.4999999999999997, 0.0, 0.9770020858515098)],\n                       [(5, 9),\n                        (4, 0.4633349244593252, 0.0, 0.2296787451978788),\n                        (8, 0.3213111929430338, 0.9999999999999999, 0.6358030075131722)],\n                       [(5, 0.4633349244593255, 0.0, -0.28500169027305033),\n                        (9, 0.3213111929430337, 0.0, 0.7140364366578915),\n                        (4, 8)],\n                       [(5, 9),\n                        (8, 0.5666568155244442, 4.440892098500626e-16, -0.8641969924868276),\n                        (4, 0.43721065119527336, 0.0, -0.6059454300745546)],\n                       [(8, 7),\n                        (9, 0.566656815524444, 0.0, -0.7859635633421085),\n                        (1, 4),\n                        (5, 0.43721065119527336, 0.9999999999999999, 0.5506224849993826)],\n                       [(9, 10),\n                        (2, 5),\n                        (4, 0.1039903575622847, 0.0, -0.20684664117433726),\n                        (8, 0.58504255233184, 0.9999999999999998, 0.6358030075131724),\n                        (7, 0.8311737009895825, 0.0, -0.0841139332337324),\n                        (1, 0.5, 1.0, -0.7963166779438284)],\n                       [(5, 0.1039903575622847, 0.0, 0.1515236960991655),\n                        (9, 0.58504255233184, 0.0, 0.7140364366578913),\n                        (4, 8),\n                        (10, 0.8311737009895825, 0.0, -0.0841139332337324),\n                        (2, 0.5000000000000002, 0.0, -0.37294365549777275)],\n                       [(5, 9),\n                        (8, 0.49999999999999956, 4.440892098500626e-16, 0.6358030075131724),\n                        (4, 0.43415359559636596, 0.0, 0.024828029966878107)],\n                       [(8, 7),\n                        (9, 0.49999999999999956, 4.440892098500626e-16, 0.7140364366578913),\n                        (1, 4),\n                        (5, 0.4341535955963662, 0.9999999999999998, -0.08015097504204993)],\n                       [(9, 10),\n                        (2, 5),\n                        (4, 0.4999999999999998, 0.0, -0.1712886848940487),\n                        (8, 0.5998657277943455, -0.9999999999999996, -0.2643731789436168),\n                        (1, 0.5530742553856172, 1.0, -0.29631667794382843)],\n                       [(5, 0.4999999999999998, 0.0, 0.11596573981887687),\n                        (9, 0.5998657277943455, 1.1102230246251565e-16, -0.18613974979889836),\n                        (4, 8),\n                        (2, 0.5530742553856169, 8.881784197001252e-16, -0.872943655497773)],\n                       [(5, 9),\n                        (4, 0.5000000000000004, 0.0, 0.32871131510595125),\n                        (8, 2.220446049250313e-16, -0.9999999999999996, 0.8100553120973732)],\n                       [(1, 4),\n                        (5, 0.5000000000000003, 0.9999999999999999, -0.3840342601811231),\n                        (8, 7),\n                        (9, 2.220446049250313e-16, 1.0, 0.8882887412420926)],\n                       [(2, 5),\n                        (9, 10),\n                        (4, 0.5, 0.0, 0.32871131510595125),\n                        (8, 0.5, 1.0, -0.4399017245341441),\n                        (7, 0.6339658219122923, 0.0, 0.8651689462862566),\n                        (1, 0.5, 0.6364347349048236, -0.29631667794382843)],\n                       [(5, 0.5, 0.0, -0.3840342601811231),\n                        (9, 0.5, 4.440892098500626e-16, -0.36166829538942524),\n                        (4, 8),\n                        (7, 3),\n                        (10, 0.6339658219122923, 0.0, 0.8651689462862566),\n                        (2, 0.4999999999999997, -0.7869382875412321, 0.12705634450222708)],\n                       [(5, 9),\n                        (10, 6),\n                        (3, 0.03592107551278813, 1.0, -0.8775009672546068),\n                        (4, 0.5991006024208341, 1.6653345369377348e-16, 0.16651150533840015),\n                        (8, 0.5, 1.0, -0.4399017245341441),\n                        (7, 0.9068032572794498, 1.1102230246251565e-16, 0.23132164282033318)],\n                       [(6, 0.03592107551278813, 8.881784197001252e-16, -0.8775009672546068),\n                        (5, 0.5991006024208338, 0.0, -0.22183445041357186),\n                        (3, 4),\n                        (8, 7),\n                        (9, 0.5000000000000001, 1.0, -0.3616682953894248),\n                        (10, 0.9068032572794498, 1.1102230246251565e-16, 0.23132164282033318)],\n                       [(6, 5),\n                        (9, 10),\n                        (4, 0.6983902210776689, 0.0, 0.3780972183239437),\n                        (7, 0.4457229604434565, 0.0, 0.01261445611041534),\n                        (3, 0.751811597304503, -0.9999999999999996, 0.6224990327453925)],\n                       [(4, 8),\n                        (5, 0.6983902210776689, 1.0, -0.4334201633991155),\n                        (7, 3),\n                        (10, 0.4457229604434565, 0.0, 0.01261445611041534),\n                        (6, 0.751811597304503, -0.9999999999999996, 0.6224990327453925)],\n                       [(5, 9),\n                        (10, 6),\n                        (3, 0.5, 1.0, -0.3775009672546079),\n                        (4, 0.6251415065033563, 0.0, 0.5257086525990662),\n                        (8, 0.0, -0.3255499447294259, 0.0),\n                        (7, 0.7856407379834285, 0.0, -0.41370794677841705)],\n                       [(6, 0.5, 1.1102230246251565e-16, -0.3775009672546079),\n                        (5, 0.6251415065033563, 0.0, 0.41896840232576205),\n                        (3, 4),\n                        (8, 7),\n                        (9, 0.0, 0.6744500552705741, 0.0),\n                        (10, 0.7856407379834285, 0.0, -0.41370794677841705)],\n                       [(6, 5),\n                        (9, 10),\n                        (4, 0.0, 0.28089726190378134, 0.0),\n                        (8, 0.5, 0.0, 0.23454833073643133),\n                        (7, 0.4949978217082433, -2.220446049250313e-16, 0.08629205322158295),\n                        (3, 0.4999999999999998, 1.0, -0.8814144405744395)],\n                       [(4, 8),\n                        (9, 0.49999999999999994, 1.0, 0.3127817598811502),\n                        (7, 3),\n                        (10, 0.4949978217082433, -2.220446049250313e-16, 0.08629205322158295),\n                        (6, 0.4999999999999998, 1.0, -0.8814144405744395)],\n                       [(5, 9),\n                        (10, 6),\n                        (3, 4),\n                        (8, 0.5, 1.0, -0.7654516692635693),\n                        (7, 0.5, 0.0, -0.4475413783399338)],\n                       [(5, 0.0, 0.05676936615274908, 0.0),\n                        (8, 7),\n                        (9, 0.5, 1.0, -0.6872182401188511),\n                        (10, 0.5, 0.0, -0.4475413783399338),\n                        (3, 0.2887068963335278, 0.0, 0.899671023811214)],\n                       [(6, 5),\n                        (9, 10),\n                        (7, 0.8015319629186126, 0.0, -0.4475413783399338),\n                        (8, 0.5, 1.0, 0.11860749139801469)],\n                       [(9, 0.5, 1.6653345369377348e-16, 0.1968409205427331),\n                        (7, 3),\n                        (10, 0.8015319629186126, 0.0, -0.4475413783399338),\n                        (6, 0.2887068963335278, 0.0, -0.100328976188786),\n                        (4, 8)],\n                       [(5, 9),\n                        (10, 6),\n                        (3, 0.10822807679614659, 0.0, 0.5488202087084497),\n                        (4, 0.5000000000000004, 0.0, -0.9443554411610635),\n                        (8, 0.0, 1.0, 0.0),\n                        (7, 0.4999999999999993, 8.881784197001252e-16, -0.947541378339934)],\n                       [(3, 4),\n                        (6, 0.10822807679614657, 0.0, -0.4511797912915503),\n                        (5, 0.5000000000000004, 1.0, 0.22669912414242238),\n                        (8, 7),\n                        (9, 0.0, 1.0, 0.0),\n                        (10, 0.4999999999999993, 8.881784197001252e-16, -0.947541378339934)],\n                       [(6, 5),\n                        (9, 10),\n                        (3, 0.18023611392199, 0.0, 0.548820208708451),\n                        (8, 0.5663325445329528, 0.9999999999999999, 0.6186074913980144),\n                        (4, 0.4999999999999998, 0.0, -0.9443554411610642)],\n                       [(9, 0.5663325445329528, 4.440892098500626e-16, 0.6968409205427333),\n                        (5, 0.4999999999999998, 0.0, -0.7733008758575775),\n                        (10, 0.5, 0.0, 0.6187911661930188),\n                        (6, 0.49999999999999967, 0.0, -0.9511797912915483),\n                        (3, 0),\n                        (8, 4)],\n                       [(9, 5),\n                        (10, 6),\n                        (3, 0.5, 0.0, -0.9511797912915487),\n                        (8, 0.49999999999998246, 4.440892098500626e-16, -0.8813924522840457),\n                        (0, 0.3040867239846962, 1.0, -0.5000000000000002),\n                        (4, 0.5000000000000022, 1.1102230246251565e-16, -0.31169035209515883)],\n                       [(7, 3),\n                        (8, 12),\n                        (9, 0.49999999999998246, 4.440892098500626e-16, -0.8031590231393269),\n                        (10, 0.49999999999998224, 0.0, -0.38120889012492176),\n                        (5, 0.500000000000002, 0.0, 0.5940340350765174),\n                        (6, 0.5000000000000022, 1.1102230246251565e-16, -0.3185147022256434)],\n                       [(9, 13),\n                        (10, 11),\n                        (3, 0.8040867239846965, 8.881784197001252e-16, -0.9511797912915487),\n                        (8, 0.5000000000000102, -0.9999999999999993, -0.8813924522840455),\n                        (12, 0.30408672398469627, 0.0, -0.5745192213521484),\n                        (7, 0.0, 0.6350392865765455, 0.0)],\n                       [(9, 0.5000000000000103, 0.0, -0.8031590231393264),\n                        (10, 0.5000000000000102, 0.0, -0.38120889012492176),\n                        (3, 0),\n                        (8, 4),\n                        (13, 0.30408672398469627, 0.0, -0.5745192213521484),\n                        (11, 0.30408672398469616, 1.0, -0.5745192213521486)],\n                       [(9, 5),\n                        (10, 6),\n                        (8, 0.75, 0.0, 0.11860758023113327),\n                        (3, 0.7499999999999999, 8.881784197001252e-16, -0.9511797912915487),\n                        (0, 0.5, 1.0, 0.5),\n                        (4, 0.4999999999999997, 0.0, -0.3116903846103405)],\n                       [(8, 12),\n                        (9, 0.75, 0.0, 0.19684100937585208),\n                        (7, 3),\n                        (10, 0.75, 0.0, -0.3812089226401003),\n                        (5, 0.49999999999999956, 0.0, 0.5940340675916997),\n                        (6, 0.49999999999999956, 0.0, 0.6814852652591754)],\n                       [(9, 13),\n                        (10, 11),\n                        (3, 0.5, -2.220446049250313e-16, 0.04882020870845155),\n                        (8, 0.500526889421938, 1.0, -0.38086548364170314),\n                        (12, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515),\n                        (7, 0.5, 1.0, -0.47238403480239244)],\n                       [(9, 0.5005268894219381, 0.0, -0.30263205449698405),\n                        (10, 0.5005268894219383, 0.0, 0.1182641412327356),\n                        (3, 0),\n                        (8, 4),\n                        (13, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515),\n                        (11, 0.5000000000000103, 1.0, 0.42548077864785155)],\n                       [(9, 5),\n                        (10, 6),\n                        (8, 0.5, 0.0, -0.63086504756827),\n                        (3, 0.25, 0.0, -0.9511797912915487),\n                        (4, 0.5000000000000003, 0.0, -0.8116903846103409)],\n                       [(8, 12),\n                        (9, 0.5, 0.0, -0.5526316184235511),\n                        (7, 3),\n                        (10, 0.5, 0.0, -0.6317362948406968),\n                        (5, 0.5000000000000003, 0.0, -0.9059659324083003),\n                        (6, 0.5000000000000004, 0.0, -0.8185147347408246)],\n                       [(9, 13),\n                        (10, 11),\n                        (3, 0.5, 0.0, 0.04882020870845155),\n                        (8, 1.0214051826551442e-14, 0.9999999999999999, -0.13087213847909496),\n                        (7, 0.5, 0.9723840348023929, -0.4723840348023929),\n                        (12, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(3, 0),\n                        (9, 5),\n                        (10, 6),\n                        (8, 4),\n                        (11, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (13, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(0, 1.0, 0.0, 0.0),\n                        (5, 0.9992548329781763, -0.32024855516364004, 0.5940340675916906),\n                        (8, 0.999999967484822, 0.9046505146653352, -0.6308650249027818),\n                        (9, 0.999999967484822, 0.8264170855206163, -0.552631595758063),\n                        (3, 0.4999999999999998, 0.45117979129154895, -0.951179791291549),\n                        (4, 0.0007451670218237004, 0.5854758970383838, 0.6883096153896671),\n                        (6, 0.0007451670218237004, -0.40769975283113236, -0.3185147347408168),\n                        (10, 3.251517832758567e-08, -0.09447819273126123, -0.6317363175061856)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8),\n                      (6, 5),\n                      (6, 6)]},\n (14, 7): {\'method\': \'joint small decoder cores + parallel three-site leaves\',\n           \'moments\': [[(1, 0.4900654283772908, 0.0, -0.49998884216809203),\n                        (4, 0.49644383901575573, 0.0, -0.4999812163686881),\n                        (2, 0.5, 0.0, -0.4999888028321735),\n                        (5, 0.4999999999999997, 0.0, -0.5000307836334323),\n                        (8, 0.534381188018264, 1.0, -0.500046950022488),\n                        (7, 0.5, 0.0, -0.20251771408753827),\n                        (10, 0.5, 1.0, -0.20251771408753827),\n                        (3, 0.49999999999999944, 1.0, -0.3649528224832399),\n                        (6, 0.49999999999999944, 1.0, -0.3649528224832399),\n                        (0, 0.5, 1.0, 0.5),\n                        (12, 0.3040867239846966, 0.0, -0.5745192213521528),\n                        (13, 0.3040867239846966, 1.0, -0.5745192213521528),\n                        (11, 0.3040867239846966, 1.0, -0.5745192213521528)],\n                       [(1, 4), (2, 5), (8, 9)],\n                       [(4, 0.39056138475274915, 4.440892098500626e-16, 0.5000187836313121),\n                        (8, 0.6126719611275606, 0.0, -0.5317592029707494),\n                        (5, 0.627579978327425, 1.0, -0.4596299023730235),\n                        (1, 0.6036697122798529, 0.0, 0.392760755216313),\n                        (2, 0.19012479142053154, 0.0, 0.5680189215159905)],\n                       [(4, 8), (5, 9)],\n                       [(4, 0.5000000000000067, 0.0, -0.5447472402143156),\n                        (5, 0.9421719950683878, 0.0, -0.43560730830515904),\n                        (8, 0.9292243081139251, -0.9999999999999996, -0.19000320704085372),\n                        (9, 0.7899576746300236, 0.0, 0.6659584803952501)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.4999999999999998, 0.0, 0.2244943849054149),\n                        (5, 0.5, 0.0, 0.5548377857211464),\n                        (8, 2.220446049250314e-16, 1.0, -0.02241907645096841),\n                        (9, 0.4999999999999998, 0.0, -0.3340415196047495)],\n                       [(1, 4), (2, 5), (8, 9)],\n                       [(4, 0.4753784960528104, 1.0, -0.2755056150945854),\n                        (8, 0.4999999999999998, 0.0, -0.2551686991202362),\n                        (9, 0.21549645867193612, 0.0, 0.749957277729336),\n                        (1, 0.5000000000000141, 0.0, -0.6427672855265485),\n                        (2, 0.7692416251197113, 1.0, -0.9776259174815849)],\n                       [(4, 5)],\n                       [(4, 8), (5, 0.5, 0.0, 0.5548377857211464)],\n                       [(5, 9),\n                        (4, 0.49999999999999484, 0.0, 0.961386650375732),\n                        (8, 0.15876855287750913, 0.0, 0.24483130087975974)],\n                       [(1, 4),\n                        (8, 7),\n                        (5, 0.5000000000000046, 0.0, -0.7082699488085362),\n                        (9, 0.8412314471224911, 5.551115123125783e-17, 0.18388621172263508)],\n                       [(2, 5),\n                        (9, 10),\n                        (8, 0.40250888547893265, 1.0, -0.9403868207356585),\n                        (7, 0.2382498473985822, 0.0, 0.048025566717950996),\n                        (1, 0.5, 1.0, -0.35655934735313455)],\n                       [(4, 8),\n                        (5, 0.0, -0.1163697347903907, 0.0),\n                        (9, 0.40250888547893265, 0.0, 0.3691043333380537),\n                        (2, 0.5, 1.1102230246251565e-16, -0.19141797930816917),\n                        (10, 0.23824984739858224, -0.9999999999999996, 0.04802556671795083)],\n                       [(5, 9),\n                        (8, 0.5000000000000013, 0.0, 0.5596131792643433),\n                        (4, 0.49999999999999956, 0.0, 0.46138665037573556)],\n                       [(5, 0.4999999999999992, 0.0, -0.3246396835989289),\n                        (8, 7),\n                        (1, 4),\n                        (9, 0.5000000000000013, 0.0, 0.8691043333380519)],\n                       [(2, 5),\n                        (9, 10),\n                        (8, 0.4999999999999994, 0.0, -0.4403868207356567),\n                        (7, 0.8228387639951986, 1.1102230246251565e-16, 0.24879131417776945),\n                        (1, 0.5000000000000004, -0.9999999999999998, -0.8565593473531345)],\n                       [(2, 0.5000000000000003, -2.220446049250313e-16, -0.691417979308169),\n                        (8, 7),\n                        (1, 4),\n                        (9, 0.49999999999999895, -2.220446049250313e-16, -0.13089566666194807),\n                        (10, 0.8228387639951986, 1.0, 0.24879131417776962)],\n                       [(2, 5),\n                        (9, 10),\n                        (4, 0.36532098276163216, 0.0, -0.20765777692108545),\n                        (8, 0.45688538394013417, -0.9999999999999996, 0.05961317926434584),\n                        (7, 0.6082288892935703, 0.0, 0.24879131417776595),\n                        (1, 0.5000000000000001, 1.0, -0.3565593473531341)],\n                       [(4, 8),\n                        (5, 0.3653209827616323, 1.0, 0.3444047436978924),\n                        (9, 0.4568853839401342, 0.0, -0.6308956666619507),\n                        (2, 0.5, 1.1102230246251565e-16, -0.19141797930816917),\n                        (10, 0.6082288892935703, 1.0, 0.24879131417776595)],\n                       [(5, 9),\n                        (4, 0.15979316138258426, 0.0, 0.8676689044563926),\n                        (8, 0.540918724091567, 0.9999999999999997, 0.5596131792643442)],\n                       [(4, 8),\n                        (5, 0.15979316138258431, -0.9999999999999998, -0.7309219376795857),\n                        (9, 0.540918724091567, 0.0, 0.869104333338051)],\n                       [(5, 9),\n                        (8, 0.0820855992752405, 4.440892098500626e-16, -0.9403868207356547),\n                        (4, 0.17667382808391466, 0.0, 0.017315896258376928)],\n                       [(5, 0.17667382808391466, 1.1102230246251565e-16, 0.11943107051842994),\n                        (8, 7),\n                        (1, 4),\n                        (9, 0.0820855992752405, 0.0, 0.36910433333805015)],\n                       [(2, 5),\n                        (9, 10),\n                        (4, 0.5000000000000027, 0.0, 0.34579097377357715),\n                        (8, 0.4999999999999983, 0.9999999999999999, 0.5596131792643444),\n                        (7, 0.2692579739648868, 0.0, -0.751208685822232),\n                        (1, 0.5, -0.9999999999999998, 0.9958721173633978)],\n                       [(4, 8),\n                        (5, 0.5000000000000024, 1.0, -0.20904400699677006),\n                        (9, 0.4999999999999983, 0.0, 0.8691043333380502),\n                        (2, 0.5, 0.0, -0.8389865145916364),\n                        (10, 0.2692579739648868, 1.0, -0.7512086858222318)],\n                       [(5, 9),\n                        (8, 0.5000000000000007, 0.0, 0.5596131792643444),\n                        (4, 0.3528413792588168, 0.0, 0.34579097377357626)],\n                       [(5, 0.35284137925881687, 0.0, -0.20904400699676895),\n                        (8, 7),\n                        (1, 4),\n                        (9, 0.5000000000000007, 0.0, 0.8691043333380502)],\n                       [(2, 5),\n                        (9, 10),\n                        (4, 0.5, 1.1102230246251565e-16, 0.3457909737735799),\n                        (8, 0.269060447136372, -0.9999999999999998, 0.05961317926434412),\n                        (1, 0.301688619477528, -0.9999999999999996, -0.5041278826366022)],\n                       [(4, 8),\n                        (5, 0.5000000000000001, 1.0, -0.20904400699677317),\n                        (9, 0.2690604471363719, 0.0, -0.6308956666619496),\n                        (2, 0.30168861947752806, 0.0, -0.33898651459163665)],\n                       [(5, 9), (4, 0.5000000000000004, 0.0, 0.84579097377358), (8, 0.0, 1.0, 0.0)],\n                       [(5, 0.5000000000000004, 0.0, -0.709044006996773), (9, 10), (1, 4), (8, 7)],\n                       [(2, 5),\n                        (4, 0.5, 0.0, 0.8457909737735798),\n                        (8, 0.5, -0.9999999999999998, 0.0596131792643424),\n                        (9, 0.5, 0.0, 0.369104333338052),\n                        (7, 3),\n                        (10, 6),\n                        (1, 0.5, 0.844278226068019, -0.5041278826366022)],\n                       [(4, 8),\n                        (5, 0.5, -0.9999999999999998, -0.7090440069967727),\n                        (6, 0.37658617266632444, 0.0, 0.13504717751675888),\n                        (10, 0.564919347967729, 0.0, -0.8746225131559058),\n                        (3, 0.3765861726663244, -0.9999999999999996, 0.13504717751675888),\n                        (7, 0.564919347967729, 5.551115123125783e-17, 0.1253774868440941),\n                        (2, 0.5, 0.6791368580230537, -0.33898651459163687)],\n                       [(5, 9),\n                        (4, 0.5000000000000038, 0.0, -0.15420902622642396),\n                        (8, 0.5, 1.0, 0.05961317926434162)],\n                       [(5, 0.5000000000000038, 0.0, 0.29095599300323105),\n                        (9, 0.5, 0.0, 0.36910433333805237),\n                        (3, 4),\n                        (8, 7)],\n                       [(6, 5),\n                        (9, 10),\n                        (4, 0.6124202497959054, 0.0, 0.7428012288311705),\n                        (7, 0.5000000000000004, -4.440892098500626e-16, -0.3746225131559058),\n                        (3, 0.7664718219180737, 0.9999999999999998, 0.635047177516759)],\n                       [(4, 8),\n                        (5, 0.6124202497959054, 0.0, -0.6060542620543636),\n                        (9, 0.0, -0.8503977523568442, 0.0),\n                        (7, 3),\n                        (10, 0.5000000000000004, -4.440892098500626e-16, -0.3746225131559058),\n                        (6, 0.7664718219180737, 0.9999999999999998, 0.635047177516759)],\n                       [(5, 9),\n                        (10, 6),\n                        (3, 0.5000000000000002, 1.0, -0.36495282248324123),\n                        (4, 0.7237608380026965, 0.0, 0.6293115787593799),\n                        (8, 0.0, -0.4467550240632503, 0.0),\n                        (7, 0.8371608839553215, 0.0, 0.17545611282181062)],\n                       [(6, 0.5000000000000002, 0.0, -0.36495282248324123),\n                        (5, 0.7237608380026965, 0.0, 0.5074353880174274),\n                        (10, 0.8371608839553215, 0.0, 0.17545611282181062),\n                        (3, 4),\n                        (8, 7)],\n                       [(6, 5),\n                        (9, 10),\n                        (4, 0.0, 0.5155754461675253, 0.0),\n                        (8, 0.5, 0.0, 0.612858155201091),\n                        (7, 0.647259141517744, 0.0, -0.6977855638982122),\n                        (3, 0.5000000000000001, 1.0, -0.8649528224832415)],\n                       [(4, 8),\n                        (9, 0.5, 0.0, 0.5187065809812095),\n                        (7, 3),\n                        (10, 0.647259141517744, 0.0, 0.3022144361017878),\n                        (6, 0.5000000000000001, 1.0, -0.8649528224832415)],\n                       [(5, 9),\n                        (10, 6),\n                        (3, 4),\n                        (8, 0.5, 1.0, -0.3871418447989079),\n                        (7, 0.49999999999999956, 0.0, 0.6885061300272752)],\n                       [(9, 0.5, 0.0, 0.5187065809812095),\n                        (10, 0.4999999999999998, 0.0, -0.31149386997272455),\n                        (5, 0.0, 0.42309241494576, 0.0),\n                        (8, 7),\n                        (3, 0.5, 0.0, 0.9683805108500918)],\n                       [(9, 10),\n                        (6, 5),\n                        (7, 0.6633380081977305, 0.0, 0.6885061300272752),\n                        (8, 0.5, 0.9999999999999998, 0.8506375090595912)],\n                       [(7, 3),\n                        (10, 0.6633380081977305, 4.440892098500626e-16, 0.6885061300272752),\n                        (6, 0.5, 0.0, -0.031619489149907754),\n                        (4, 8),\n                        (5, 0.0, 1.0, 0.0),\n                        (9, 0.5, 0.0, 0.28092722712270923)],\n                       [(10, 6),\n                        (5, 9),\n                        (3, 0.8248194291522575, 0.0, -0.5316194891499073),\n                        (4, 0.5000000000000003, 0.0, 0.7877853623969213),\n                        (8, 0.0, 1.0, 0.0),\n                        (7, 0.5000000000000001, 0.0, 0.18850613002727545)],\n                       [(3, 4),\n                        (10, 0.5000000000000001, 0.0, 0.18850613002727545),\n                        (6, 0.8248194291522575, 0.0, 0.4683805108500927),\n                        (5, 0.5000000000000004, 1.0, 0.28762946549317153),\n                        (8, 7)],\n                       [(9, 10),\n                        (6, 5),\n                        (3, 0.18030068686283401, 5.551115123125783e-17, 0.4683805108500939),\n                        (8, 0.5663648310033745, 1.0, -0.6493624909404092),\n                        (4, 0.5, 0.0, 0.7877853623969207)],\n                       [(10, 0.5, 0.0, 0.754870961030651),\n                        (6, 0.5000000000000001, 0.0, 0.9683805108500936),\n                        (3, 0),\n                        (8, 4),\n                        (9, 0.5663648310033746, 0.0, 0.7809272271227101),\n                        (5, 0.5, 1.0, -0.7123705345068287)],\n                       [(10, 6),\n                        (9, 5),\n                        (3, 0.5, 0.0, 0.9683805108500936),\n                        (8, 0.49999999999998257, -2.220446049250313e-16, -0.14936243462246823),\n                        (0, 0.3040867239846962, 1.0, -0.5000000000000002),\n                        (4, 0.5000000000000022, 0.0, -0.5794849755963298)],\n                       [(7, 3),\n                        (8, 12),\n                        (9, 0.49999999999998246, 0.0, 0.28092717080476914),\n                        (10, 0.49999999999998246, 0.0, -0.24512909528728954),\n                        (6, 0.5000000000000022, 0.0, -0.39888982714315757),\n                        (5, 0.5000000000000019, 0.9999999999999999, 0.6548998034864224)],\n                       [(9, 13),\n                        (10, 11),\n                        (3, 0.8040867239846965, 0.0, 0.9683805108500936),\n                        (8, 0.5000000000000102, 1.0, -0.1493624346224678),\n                        (12, 0.30408672398469627, 0.0, -0.5745192213521484),\n                        (7, 0.0, -0.8274253014635145, 0.0)],\n                       [(10, 0.5000000000000102, 3.3306690738754696e-16, -0.24512909528728977),\n                        (3, 0),\n                        (8, 4),\n                        (9, 0.5000000000000102, 0.0, 0.2809271708047687),\n                        (13, 0.30408672398469616, 1.0, -0.5745192213521486),\n                        (11, 0.30408672398469616, 1.0, -0.5745192213521486)],\n                       [(10, 6),\n                        (9, 5),\n                        (8, 0.75, 0.0, 0.8506375978927101),\n                        (3, 0.7499999999999999, 0.0, 0.9683805108500936),\n                        (0, 0.5, 1.0, 0.5),\n                        (4, 0.49999999999999956, 0.0, -0.5794850081115106)],\n                       [(8, 12),\n                        (7, 3),\n                        (9, 0.75, 0.0, -0.7190728617104094),\n                        (10, 0.75, 1.1102230246251565e-16, -0.24512912780246965),\n                        (6, 0.49999999999999956, 0.0, 0.6011101403416608),\n                        (5, 0.49999999999999967, 0.9999999999999999, 0.6548998360016038)],\n                       [(9, 13),\n                        (10, 11),\n                        (3, 0.5, 0.0, -0.03161948914990642),\n                        (8, 0.5005268894219383, -0.9999999999999996, 0.35116453401987424),\n                        (12, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515),\n                        (7, 0.5, -0.9999999999999996, -0.7987688280048215)],\n                       [(10, 0.5005268894219383, 0.0, 0.25434393607036654),\n                        (3, 0),\n                        (8, 4),\n                        (9, 0.500526889421938, 1.1102230246251565e-16, -0.21959979783757366),\n                        (13, 0.5000000000000103, 1.0, 0.42548077864785155),\n                        (11, 0.5000000000000103, 1.0, 0.42548077864785155)],\n                       [(10, 6),\n                        (9, 5),\n                        (8, 0.5, 1.1102230246251565e-16, 0.10116497009330677),\n                        (3, 0.2500000000000001, 0.0, 0.9683805108500934),\n                        (4, 0.5000000000000004, 4.440892098500626e-16, 0.9205149918884887)],\n                       [(8, 12),\n                        (7, 3),\n                        (9, 0.5, 0.0, 0.030399766088993585),\n                        (10, 0.5, 0.0, -0.4956565000030668),\n                        (6, 0.5000000000000003, 8.881784197001252e-16, -0.8988898596583399),\n                        (5, 0.5000000000000003, 0.9999999999999998, -0.8451001639983955)],\n                       [(9, 13),\n                        (10, 11),\n                        (3, 0.5, 0.0, -0.03161948914990642),\n                        (8, 1.0214051826551442e-14, 0.9999999999999998, 0.6011578791824821),\n                        (7, 0.5, -0.7012311719951787, -0.7987688280048215),\n                        (12, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(3, 0),\n                        (10, 6),\n                        (8, 4),\n                        (9, 5),\n                        (11, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (13, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(0, 1.0, 0.0, 0.0),\n                        (5, 0.9992548329781763, -0.3811143235735446, 0.6548998360015954),\n                        (8, 0.999999967484822, 0.17262049700375842, 0.10116499275879492),\n                        (3, 0.4999999999999998, 0.5316194891499064, 0.9683805108500936),\n                        (4, 0.0007451670218237004, 0.8532705205395539, 0.42051499188849695),\n                        (6, 0.0007451670218237006, -0.3273246279136166, -0.3988898596583321),\n                        (9, 3.251517832758566e-08, 0.24338574633904742, -0.9696002565764943),\n                        (10, 3.251517832758566e-08, -0.23055798756889212, -0.49565652266855453)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8),\n                      (6, 5),\n                      (6, 6)]},\n (15, 4): {\'method\': \'re-embedded count cores + four parallel three-site leaves\',\n           \'moments\': [[(1, 0.5000000000000004, 0.0, -0.5000075367107908),\n                        (3, 0.5, 0.0, -0.5000048204691958),\n                        (4, 0.5398760879493478, 0.0, -0.49997652083158384),\n                        (7, 0.3506396248117343, 0.0, -0.5000366846791133),\n                        (8, 0.8971135525850333, 0.0, -0.49994432091071594),\n                        (6, 0.22718552582850524, 0.0, -0.708982875827169),\n                        (13, 0.49999999999999956, 0.0, -0.14711776354248007),\n                        (2, 0.35760927658307967, 0.0, -0.9848107895924711),\n                        (12, 0.7139623524129013, 0.0, -0.350985519246513),\n                        (0, 0.5, 0.0, 0.5),\n                        (11, 0.3040867239846966, 0.0, -0.5745192213521528),\n                        (10, 0.3040867239846966, 0.0, -0.5745192213521528),\n                        (14, 0.3040867239846966, 0.0, -0.5745192213521528)],\n                       [(1, 3), (5, 4), (7, 8)],\n                       [(3, 0.3645435900006674, 0.0, 0.517085591690893),\n                        (7, 0.4630154240149502, 0.0, -0.7078970920608687),\n                        (4, 0.5489951360758489, 0.0, 0.6598275546835288),\n                        (8, 0.6998610612586851, -2.220446049250313e-16, 0.17733547366764674),\n                        (5, 0.5000000000000022, 0.0, 0.4921667148472513),\n                        (1, 0.629369581579009, 0.0, 0.6468020657901101)],\n                       [(3, 7), (4, 8)],\n                       [(3, 0.5, 0.0, -0.3142288378277136),\n                        (4, 0.47334781214404265, 0.0, 0.7508823428995606),\n                        (7, 0.5956561507711625, 1.1102230246251565e-16, -0.0778388321203689),\n                        (8, 0.6538194785636017, 0.0, 0.050497260912362174)],\n                       [(3, 4), (7, 8)],\n                       [(5, 4),\n                        (3, 0.5, 0.0, 0.6857711621722866),\n                        (7, 0.2752731114452889, 0.0, -0.07824749849266244),\n                        (8, 0.2922950755832589, 0.0, -0.9689922384609875)],\n                       [(1, 3),\n                        (7, 8),\n                        (5, 0.4423456032214794, 0.0, -0.4326555930335607),\n                        (4, 0.4999999999999998, 0.0, -0.2125399483659629)],\n                       [(9, 5),\n                        (3, 0.5077877108807936, 1.1102230246251565e-16, -0.051341187037494684),\n                        (7, 0.37676215574847927, 0.0, 0.08311327773161148),\n                        (8, 0.486970738579104, 0.0, -0.16395232680703198),\n                        (1, 0.5000000000000007, 0.0, 0.6914958931777739)],\n                       [(3, 4),\n                        (5, 0.905588427626018, 1.1102230246251565e-16, 0.271868791269443),\n                        (9, 0.5, 0.0, 0.7861872525800169)],\n                       [(3, 7), (4, 0.5000000000000013, 0.0, -0.21253994836596313)],\n                       [(4, 8),\n                        (3, 0.8916234504425208, 0.0, 0.2496375320711528),\n                        (7, 0.8340673682739812, 0.0, 0.49213628813828414)],\n                       [(1, 3),\n                        (7, 6),\n                        (4, 0.499999999999999, 0.0, 0.5478309368438334),\n                        (8, 0.6834279337560494, 0.0, 0.2826269819520144)],\n                       [(4, 8),\n                        (3, 0.5, 0.0, 0.7496375320711548),\n                        (7, 0.4999999999999998, 0.0, 0.44379762698283826),\n                        (6, 0.22718552582850488, 0.0, 0.29101712417283077)],\n                       [(3, 7),\n                        (4, 0.12117854642700412, 0.0, 0.7843638346889745),\n                        (8, 0.11328910811143535, 0.0, -0.9521049451558721)],\n                       [(5, 4), (7, 6), (8, 9), (1, 3)],\n                       [(4, 0.9095387369494694, 0.0, -0.11922348817086648),\n                        (8, 0.5, 0.0, -0.7687116405426444),\n                        (9, 0.6409740898841031, 0.0, 0.715408387939233),\n                        (3, 0.5886108001133226, 0.0, -0.2912358661717409),\n                        (7, 0.680195383285978, 0.0, -0.6113290151870618),\n                        (6, 0.6354920868801647, 0.0, 0.5482021483143629)],\n                       [(4, 8), (9, 5), (3, 7)],\n                       [(4, 0.445722612083208, 0.0, 0.28794268048147265),\n                        (8, 0.9189865165454063, 0.0, -0.7687116405426466),\n                        (9, 0.3856965147986098, 0.0, -0.7872867860364003),\n                        (3, 0.5000000000000004, 0.0, 0.3152703197958946),\n                        (7, 0.49999999999999906, 0.0, 0.09714933555213556)],\n                       [(4, 8), (9, 5), (1, 3), (7, 6)],\n                       [(4, 0.7501046918877674, 0.0, 0.5084490443764489),\n                        (8, 0.048498527884762506, 0.0, -0.2687116405426475),\n                        (9, 0.8708750414451267, 0.0, -0.7112479214414928),\n                        (3, 0.5234132252872342, 0.0, 0.46370600307684556),\n                        (7, 0.06708148302124252, 0.0, -0.12954185188602363),\n                        (1, 0.5, 1.1102230246251565e-16, -0.24256508796187415)],\n                       [(5, 4), (8, 9), (3, 7)],\n                       [(4, 0.4661685750334878, 0.0, -0.5201800901234552),\n                        (8, 0.4999999999999998, 0.0, -0.26871164054264707),\n                        (9, 0.4769399325118889, 0.0, -0.13493967999280887),\n                        (3, 0.33272914584056146, 0.0, -0.5824407769164972),\n                        (7, 0.05920200383562026, 0.0, 0.3704581481139763)],\n                       [(4, 8), (9, 5), (3, 7)],\n                       [(5, 0.4999999999999998, 0.0, 0.829712846053523),\n                        (4, 0.4999999999999997, 0.0, 0.5765539160317323),\n                        (8, 0.5135786749521977, 8.881784197001252e-16, 0.7360632533922762),\n                        (9, 0.11601310423962068, 0.0, 0.36689942887836846),\n                        (3, 0.25011854054823907, 4.440892098500626e-16, 0.9175592230835028),\n                        (7, 0.08917758998520853, 0.0, 0.49436409249784846)],\n                       [(5, 4), (8, 9), (1, 3), (7, 6)],\n                       [(9, 13),\n                        (4, 8),\n                        (7, 0.5938461416699012, 5.551115123125783e-17, 0.30676575443768067),\n                        (6, 0.021107237782608024, 0.0, -0.9231467397292097),\n                        (5, 0.49999999999999645, -0.9470955625045279, 0.8297128460535219),\n                        (1, 0.49999999999999767, 0.35578543929073003, 0.7574349120381294)],\n                       [(3, 7),\n                        (6, 2),\n                        (9, 0.6459206838777178, 0.0, -0.5457898759412201),\n                        (4, 0.5, 0.0, 0.5597861780310811),\n                        (13, 0.27784540982768524, 0.0, -0.26970665398215354)],\n                       [(8, 9),\n                        (2, 0.8402337422326178, 0.0, 0.9498164599690078),\n                        (7, 0.4433968217557545, 0.0, -0.21061544157195566),\n                        (6, 0.3147365041110265, 0.0, -0.7244934550958735)],\n                       [(2, 3),\n                        (7, 6),\n                        (8, 0.6878823673938599, 0.0, -0.41769517846087956),\n                        (9, 0.49999999999999944, 0.0, 0.04524259464472258)],\n                       [(4, 8),\n                        (9, 13),\n                        (3, 0.5, 0.0, -0.11312924558498949),\n                        (7, 0.6391921844757946, 0.0, 0.3720666754871851),\n                        (6, 0.9294528636932586, 4.440892098500626e-16, 0.5567952414001085),\n                        (2, 0.31397581370367167, 0.0, -0.21890167318297582)],\n                       [(3, 7),\n                        (6, 2),\n                        (4, 0.36093889170991433, 0.0, 0.059786178031081494),\n                        (9, 0.6807543702293102, 1.1102230246251565e-16, 0.045242594644722245),\n                        (13, 0.6615779164321718, 0.0, -0.24486873569361656)],\n                       [(4, 8),\n                        (9, 13),\n                        (3, 0.5000000000000004, -4.440892098500626e-16, -0.11312924558498949),\n                        (7, 0.5, 0.0, 0.1571291826662919),\n                        (6, 0.33108034381799806, 0.0, 0.05679524140011294),\n                        (2, 0.5519644059311344, 0.0, 0.7987523594593542)],\n                       [(3, 7),\n                        (6, 2),\n                        (8, 0.21771790206209457, 0.0, -0.7051747412902187),\n                        (9, 0.9140597740875678, 0.0, -0.45475740535527787),\n                        (4, 0.01676773800065057, 8.881784197001252e-16, -0.940213821968918),\n                        (13, 0.3835336580321995, 0.0, 0.9186547735247841)],\n                       [(8, 9),\n                        (2, 0.7831723914001073, 0.0, 0.6373214569840702),\n                        (3, 0.24294212102114646, 0.0, -0.9905853107696949),\n                        (6, 0.8371247201733047, 0.0, -0.8632896185761176)],\n                       [(2, 3),\n                        (7, 6),\n                        (8, 0.2960864908121443, 0.0, -0.3678973707892821),\n                        (9, 0.4999999999999998, 0.0, -0.454757405355277)],\n                       [(4, 8),\n                        (9, 13),\n                        (2, 0.49999999999999956, 0.0, 0.7329995800098472),\n                        (3, 0.36381228346888317, 0.0, 0.21930176528916023),\n                        (7, 0.4960041929285068, 0.0, 0.6571291826662926)],\n                       [(2, 3),\n                        (7, 6),\n                        (8, 0.49999999999999956, 0.0, -0.6310957421154475),\n                        (9, 0.47784932123795026, 0.0, -0.5789171174356127),\n                        (4, 0.2537023786592818, 0.0, 0.05978617803108155)],\n                       [(8, 9),\n                        (3, 0.022839382910483597, 0.0, -0.28069823471084465),\n                        (7, 0.5, 0.0, 0.15712918266629305),\n                        (6, 0.8057964070938461, 0.0, -0.048305353185546185),\n                        (2, 0.7913981257738136, 0.0, 0.7329995800098474)],\n                       [(3, 7), (6, 2), (9, 13), (8, 0.5, 0.0, -0.1310957421154466)],\n                       [(4, 8),\n                        (2, 0.75, 0.0, 0.7329995800098461),\n                        (3, 0.22716061708951643, 0.0, 0.7193017652891598),\n                        (7, 0.5, 0.0, -0.49081560846130823),\n                        (13, 0.41077394080097146, 0.0, 0.719870699894825),\n                        (9, 0.5, 8.881784197001252e-16, 0.8859059144395667)],\n                       [(2, 3),\n                        (7, 6),\n                        (13, 12),\n                        (8, 0.4999999999999998, 0.0, -0.1310957421154466),\n                        (4, 0.5000000000000003, 0.8228311055179143, 0.059786178031080495)],\n                       [(8, 9),\n                        (2, 0.9427632774999226, 0.0, -0.7670004199901537),\n                        (7, 0.2029206870528753, 0.0, 0.009184391538691994),\n                        (3, 0.4999999999999998, 0.0, 0.7193017652891596),\n                        (13, 0.9209651411768475, 0.0, -0.17265368755422283),\n                        (12, 0.18797344272617975, 0.0, 0.1490144807534874)],\n                       [(2, 0),\n                        (7, 3),\n                        (9, 0.5000000000000004, 0.0, 0.8859059144395673),\n                        (8, 0.3532263144177301, 0.0, 0.9365262571081256)],\n                       [(9, 13),\n                        (12, 8),\n                        (2, 0.4999999999999997, 0.0, -0.26700041999015367),\n                        (7, 0.49999999999998246, 0.0, 0.5091843352207515),\n                        (0, 0.30408672398469605, 0.0, -0.5000000000000002),\n                        (3, 0.500000000000002, 0.0, 0.7312771734336028)],\n                       [(6, 2),\n                        (7, 11),\n                        (13, 0.4958095421256964, 0.0, 0.545240320885148),\n                        (12, 0.10063705686025855, 4.440892098500626e-16, 0.6490144807534872),\n                        (8, 0.6082923684818491, 0.0, -0.29866349698139905)],\n                       [(9, 13),\n                        (12, 8),\n                        (2, 0.8040867239846965, 0.0, 0.7329995800098463),\n                        (7, 0.5000000000000103, 0.0, 0.5091843352207508),\n                        (11, 0.30408672398469605, 0.0, 0.4254807786478514)],\n                       [(2, 0),\n                        (7, 3),\n                        (8, 0.4168165496090295, 0.0, -0.2517776791446398),\n                        (9, 0.4999999999999998, 0.0, 0.38590591443956845),\n                        (13, 0.47929006645163197, 0.0, -0.11094542842509725),\n                        (12, 0.06227689441408989, 0.0, 0.14901448075348972)],\n                       [(8, 9),\n                        (13, 12),\n                        (2, 0.75, 0.0, 0.7329995800098463),\n                        (7, 0.7499999999999999, 1.1102230246251565e-16, -0.4908156972944271),\n                        (0, 0.5, 0.0, 0.5),\n                        (3, 0.4999999999999998, 0.0, -0.26872279405121624)],\n                       [(6, 2),\n                        (7, 11),\n                        (9, 0.5, 0.0, -0.6140940855604311),\n                        (13, 0.5809659768784905, 0.0, 0.8161200751094453),\n                        (12, 0.8037128700384213, 0.0, 0.6490144807534877)],\n                       [(9, 13),\n                        (12, 8),\n                        (2, 0.5000000000000002, 0.0, 0.7329995800098474),\n                        (7, 0.500526889421938, 1.1102230246251565e-16, 0.008657366578408765),\n                        (6, 0.5, -4.440892098500626e-16, -0.44342329229517063),\n                        (11, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515)],\n                       [(2, 0),\n                        (7, 3),\n                        (9, 0.5, 0.0, -0.049180488064326955),\n                        (12, 0.045748023543320304, 0.0, 0.14901448075348656)],\n                       [(8, 9),\n                        (13, 12),\n                        (2, 0.25000000000000006, 0.0, -0.26700041999015234),\n                        (7, 0.5, 0.0, 0.25865693050497596),\n                        (3, 0.5000000000000004, 0.0, -0.7687227940512158)],\n                       [(6, 2),\n                        (7, 11),\n                        (13, 0.4999999999999998, 0.0, -0.04165914141316063),\n                        (12, 0.22604969255933055, 0.0, -0.35098551924651455),\n                        (8, 0.5, 0.0, -0.5239211150480099)],\n                       [(7, 3),\n                        (9, 13),\n                        (12, 8),\n                        (2, 0.5, 1.1102230246251565e-16, -0.2670004199901541),\n                        (6, 0.5, 0.9434232922951704, 0.5565767077048294),\n                        (11, 0.5, 0.3483047662953689, -0.5745192538673269)],\n                       [(2, 0),\n                        (9, 0.6088317711209519, 0.0, 0.4508195119356727),\n                        (8, 0.22616755706827293, 2.220446049250313e-16, -0.023921115048009867),\n                        (13, 0.4999999999999998, 0.0, 0.8495090874658884),\n                        (12, 0.20697756857097405, 0.0, -0.35098551924651344),\n                        (3, 0.9992548329781763, 0.5425083064792755, -0.2687227940512251),\n                        (7, 3.251517832758567e-08, 0.015128581923066431, -0.7413430921605122)],\n                       [(9, 8),\n                        (13, 12),\n                        (0, 1.0, 0.0, 0.5),\n                        (2, 0.5, 0.7670004199901536, -0.26700041999015234)],\n                       [(9, 0.4999999999999824, 0.0, 0.9508194556177323),\n                        (13, 0.49999999999998246, 4.440892098500626e-16, 0.849509143783828),\n                        (8, 0.5000000000000022, 0.0, -0.8062575728061063),\n                        (12, 0.5000000000000018, 5.551115123125783e-17, 0.4313509385115835)],\n                       [(9, 10), (13, 14)],\n                       [(9, 0.5000000000000107, 0.0, 0.9508194556177323),\n                        (13, 0.5000000000000102, 0.0, 0.8495091437838282),\n                        (10, 0.30408672398469605, 0.0, 0.4254807786478514),\n                        (14, 0.30408672398469627, 0.0, -0.5745192213521484)],\n                       [(9, 8), (13, 12)],\n                       [(9, 0.75, 0.0, 0.9508194231025549),\n                        (13, 0.75, -2.220446049250313e-16, -0.1504908237009932),\n                        (8, 0.49999999999999956, -2.220446049250313e-16, 0.19374239467871185),\n                        (12, 0.49999999999999956, 0.0, 0.43135097102676556)],\n                       [(9, 10), (13, 14)],\n                       [(9, 0.500526889421938, 0.0, -0.5497075130246087),\n                        (13, 0.5005268894219381, 4.440892098500626e-16, -0.6499638875738283),\n                        (10, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515),\n                        (14, 0.5000000000000102, 0.0, 0.4254807786478514)],\n                       [(9, 8), (13, 12)],\n                       [(9, 0.5, 0.0, 0.7002920509019579),\n                        (13, 0.5, 4.440892098500626e-16, -0.8999634515003958),\n                        (8, 0.5000000000000004, 0.0, 0.6937423946787113),\n                        (12, 0.5000000000000004, 0.0, 0.9313509710267658)],\n                       [(9, 10), (13, 14)],\n                       [(9, 8),\n                        (13, 12),\n                        (10, 0.5, 0.3483047662953689, -0.5745192538673269),\n                        (14, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(8, 0.0007451670218237006, 0.08004311774933237, -0.8062576053212804),\n                        (9, 3.251517832758567e-08, 0.5734934615260858, 0.7002920282364686),\n                        (12, 0.9992548329781763, -0.15756545859870608, 0.43135097102675724),\n                        (13, 0.999999967484822, -0.826251081402539, -0.8999634288349077)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8),\n                      (6, 5),\n                      (6, 6),\n                      (6, 7),\n                      (6, 8)]},\n (15, 5): {\'method\': \'joint small decoder cores + parallel three-site leaves\',\n           \'moments\': [[(1, 0.4132089683491019, 0.0, 0.49997884705812984),\n                        (3, 0.5802824054563749, 0.0, -0.4999990850181959),\n                        (5, 0.5013042540503503, 3.3306690738754696e-16, -0.49995613306162046),\n                        (4, 0.5018038413865794, 0.0, 0.5000353368095434),\n                        (7, 0.9967991537583112, 1.0, -0.49998759899452705),\n                        (6, 0.49999999999999895, 0.0, -0.7088499809721285),\n                        (13, 0.5, 1.0, -0.2656833070307263),\n                        (2, 0.48497757661839347, 0.0, -0.834800922923292),\n                        (12, 0.4153663251971116, -0.9999999999999996, -0.4155947556675157),\n                        (0, 0.5, 0.0, 0.5),\n                        (11, 0.3040867239846966, 0.0, -0.5745192213521528),\n                        (10, 0.3040867239846966, 1.0, -0.5745192213521528),\n                        (14, 0.3040867239846966, 1.0, -0.5745192213521528)],\n                       [(1, 3), (5, 4), (7, 8)],\n                       [(3, 0.742998384089613, 0.0, 0.5000009149818057),\n                        (7, 0.8860638226358781, 0.0, -0.4154890493425054),\n                        (4, 0.7005550416029226, 1.0, -0.5084074327980057),\n                        (1, 0.7319156006638878, 1.1102230246251565e-16, 0.49608291586590425),\n                        (5, 0.6992712642148322, 0.0, -0.49995613306163555)],\n                       [(3, 7), (4, 8)],\n                       [(3, 0.4999999999999989, 0.0, -0.50417698927985),\n                        (4, 0.49865952234007743, 5.551115123125783e-17, 0.48664609854968516),\n                        (7, 0.7820488647322601, 0.0, -0.42501406412548426),\n                        (8, 0.0, 0.0001633994887711432, 0.0)],\n                       [(3, 4), (7, 8)],\n                       [(3, 0.5000000000000011, 0.0, -0.5041769892798682),\n                        (4, 0.5, 1.1102230246251565e-16, 0.45535175093357727),\n                        (7, 0.7981507785417876, -4.440892098500626e-16, -0.46776197017808707)],\n                       [(1, 3), (5, 4)],\n                       [(3, 0.2521202898334676, 0.0, 0.47883770266637965),\n                        (5, 0.7284764219286508, 5.551115123125783e-17, 0.19290768103388672),\n                        (4, 0.7791526022324156, 1.0, -0.44639741482765505),\n                        (1, 0.5, 0.0, 0.4934888779964566)],\n                       [(3, 7), (9, 5), (4, 8)],\n                       [(3, 0.18131040733928783, 0.0, -0.497003344931797),\n                        (4, 0.24913515449947915, 0.0, 0.4597365523823118),\n                        (7, 0.5161228778908168, 0.9999999999999999, 0.5018487005733345),\n                        (8, 0.5, 4.440892098500626e-16, 0.5000410171042737),\n                        (5, 0.728476421928667, 0.0, -0.8070923189661237),\n                        (9, 0.5, 0.9999999999999999, -0.1805896763080077)],\n                       [(3, 4), (7, 8)],\n                       [(7, 0.5, 0.0, -0.15783650755877865),\n                        (4, 0.10091200530617515, 1.0, 0.12771185178299171),\n                        (8, 0.7006923750136328, -2.220446049250313e-16, 4.1017104273510796e-05)],\n                       [(3, 7), (4, 8)],\n                       [(3, 0.5882500769209105, 0.0, 0.5720496838619722),\n                        (4, 0.28536701088763433, 0.0, -0.6483937593240281),\n                        (7, 0.703340369879063, 0.0, 0.34216349244122046),\n                        (8, 0.1068742790842334, 0.0, -0.36209557644195556)],\n                       [(1, 3), (5, 4), (7, 6), (8, 9)],\n                       [(3, 0.9675670579641595, 0.0, 0.5720496838619564),\n                        (7, 0.49280504227479494, 0.0, 0.6708068070791349),\n                        (4, 0.9108537925987485, 1.0, 0.06160374353493514),\n                        (8, 0.5, 0.0, -0.6769760088641037),\n                        (9, 0.46352713151159636, 1.0, 0.2640454433674886),\n                        (6, 0.4608483309798826, -2.220446049250313e-16, 0.8309193012950313)],\n                       [(3, 7), (4, 8), (9, 5)],\n                       [(3, 0.5000000000000003, 0.0, 0.5720496838619391),\n                        (7, 0.5596506416568943, 0.0, 0.5817096081536164),\n                        (4, 0.5377801164087214, 1.0, 0.5510320743777812),\n                        (8, 0.8172104746408917, 0.0, -0.6769760088641037),\n                        (9, 0.6058193373299144, 1.0, 0.09006893310335501)],\n                       [(3, 7), (5, 4), (8, 9)],\n                       [(7, 0.5000000000000004, 0.0, 0.560608435513557),\n                        (9, 0.3263740628486297, 0.0, 0.2755902782462125),\n                        (5, 0.0, 0.05164172914714532, 0.0),\n                        (3, 0.12754703959217256, 1.0, -0.5803361730671512),\n                        (4, 0.8266664524069582, -0.9999999999999996, -0.15240547099621793),\n                        (8, 0.045931842050171756, 0.0, -0.17697600886410347)],\n                       [(7, 6), (9, 5), (1, 3), (4, 8)],\n                       [(3, 0.2696276353165252, 0.0, -0.5226917690388972),\n                        (7, 0.16304676742985036, 0.0, -0.439391564486441),\n                        (9, 0.11904903998422628, 0.0, 0.6792811142119086),\n                        (4, 0.4273399417518991, -0.9999999999999996, -0.5459242847272363),\n                        (8, 0.9539104356517047, 4.440892098500626e-16, -0.6769760088641046),\n                        (1, 0.0, -0.31555683986653227, 0.0),\n                        (6, 0.8273677055841601, 0.0, 0.011025144178165858)],\n                       [(3, 7), (9, 5), (4, 8)],\n                       [(3, 0.3174136387017352, 0.0, 0.035387989666283426),\n                        (4, 0.25349168400187955, 0.0, -0.8580202494864657),\n                        (7, 0.704332640729217, 4.440892098500626e-16, -0.9393915644864408),\n                        (8, 0.15644213008117777, 0.0, 0.8230239911358932),\n                        (9, 0.6917633216559813, 0.9999999999999996, 0.8660475973984443)],\n                       [(1, 3), (5, 4), (7, 6), (8, 9)],\n                       [(3, 0.6276139104334912, 0.0, 0.13129509946107754),\n                        (7, 0.5, 8.881784197001252e-16, -0.9393915644864408),\n                        (4, 0.562586548020497, 0.9999999999999999, -0.16532168782477896),\n                        (8, 0.5, 0.0, 0.8230239911358943),\n                        (9, 0.03151537942096394, 1.0, -0.7185699907363319),\n                        (6, 0.727371054243577, 0.0, -0.07318377315388713)],\n                       [(3, 7), (4, 8), (9, 5)],\n                       [(3, 0.4999999999999989, 0.0, -0.7633711677785731),\n                        (7, 0.529090738553442, 0.0, 0.6912048882446138),\n                        (5, 0.5, -2.220446049250313e-16, 0.3228407545700467),\n                        (4, 0.5, 1.0, -0.9783293652273826),\n                        (8, 0.4546464932122305, 2.7755575615628914e-16, 0.23912733304908762),\n                        (9, 0.4392960432426437, 1.0, -0.28535129208653487)],\n                       [(1, 3), (7, 6), (5, 4), (8, 9)],\n                       [(3, 0.6683089837594859, 1.1102230246251565e-16, -0.2633711677785726),\n                        (7, 0.49999999999999956, 0.0, -0.3152915817020081),\n                        (4, 8),\n                        (9, 13),\n                        (1, 0.5000000000000004, 0.0, -0.4044909326685464),\n                        (6, 0.49999999999999956, 0.0, -0.09727132543917172),\n                        (5, 0.4999999999999998, -0.0006509335106938963, 0.32284075457004624)],\n                       [(3, 7),\n                        (8, 0.0, 0.37004598770015207, 0.0),\n                        (9, 0.3778861449265366, 5.551115123125783e-17, 0.21456099605698642),\n                        (4, 0.4999999999999994, 0.0, -0.43850981925979937),\n                        (13, 0.06834101983091842, -0.9999999999999996, 0.7789857130606195)],\n                       [(3, 0.6578349175114986, 0.0, -0.2633711677785726), (7, 6), (8, 9)],\n                       [(1, 3),\n                        (7, 0.5000000000000009, 0.0, -0.315291581702009),\n                        (6, 0.6530017079301533, -2.220446049250313e-16, 0.4027286745608292),\n                        (8, 0.41331991606738616, -0.9999999999999998, 0.08236417735614388),\n                        (9, 0.4999999999999993, 0.0, 0.07857198636443818)],\n                       [(3, 0.5, 0.0, -0.7635008742631222),\n                        (6, 2),\n                        (4, 8),\n                        (9, 13),\n                        (1, 0.4999999999999998, -0.6090599429554304, 0.5955090673314531)],\n                       [(3, 7),\n                        (2, 0.5605616196317902, 0.0, -0.8348009229232944),\n                        (6, 0.5000000000000003, 1.1102230246251565e-16, -0.09727132543918637),\n                        (4, 0.11253431722702034, 0.0, -0.9385098192597996),\n                        (8, 0.0, 1.0, 0.0),\n                        (9, 0.6865025359633383, 0.0, -0.9214280136355613),\n                        (13, 0.5297395290041438, 0.9999999999999999, -0.4250491790947357)],\n                       [(3, 0.5, 0.0, -0.7635008742631222), (7, 6), (4, 8), (9, 13)],\n                       [(2, 3),\n                        (6, 0.4311009172299638, 0.0, -0.34731902068777165),\n                        (7, 0.5, 1.0, -0.07855954789825548),\n                        (8, 0.4609737351550911, 0.9999999999999999, -0.2312964982908472),\n                        (9, 0.960833440065644, 0.0, 0.5785719863644316),\n                        (4, 0.03981954596758298, 0.0, -0.9385098192597925),\n                        (13, 0.801538355530847, 1.0, -0.05577781524071823)],\n                       [(2, 0.5818042231825744, -2.220446049250313e-16, -0.3348009229233022), (3, 7), (8, 9)],\n                       [(6, 2),\n                        (3, 0.2430469681607357, 0.0, 0.830723157381571),\n                        (7, 0.5, 1.0, -0.07855954789825548),\n                        (8, 0.38627649394841107, 1.0, -0.473492956746846),\n                        (9, 0.16629816714536846, 0.0, 0.07857198636442414)],\n                       [(6, 0.03454957128703273, 0.0, 0.8181166636420727),\n                        (2, 0.2894984978675621, 0.0, -0.8348009229233004),\n                        (3, 7),\n                        (8, 9)],\n                       [(6, 2),\n                        (3, 0.30568317284628654, 5.551115123125783e-17, 0.47828007869949823),\n                        (8, 0.5824624691266849, 0.9999999999999999, 0.854529223462319),\n                        (9, 0.5, 0.0, 0.07857198636442386)],\n                       [(6, 0.9242869536756131, 8.881784197001252e-16, 0.5858061491772155),\n                        (2, 0.87386354985307, 0.9999999999999998, 0.6651990770766807),\n                        (4, 8),\n                        (9, 13)],\n                       [(7, 6),\n                        (2, 3),\n                        (8, 0.5, 1.0, 0.43982511498840604),\n                        (9, 0.38819804894301435, 0.0, -0.8273080168536269),\n                        (4, 0.599987907871844, 5.551115123125783e-17, 0.061490180740200795)],\n                       [(6, 0.23753846872505302, 0.0, 0.11812387065190871),\n                        (2, 0.6888649043845242, 0.0, -0.8348009229233393),\n                        (3, 0.5595532320705698, 4.440892098500626e-16, 0.8961823848857746),\n                        (7, 0.4999999999999997, -0.9999999999999998, 0.42131074561719534),\n                        (8, 9)],\n                       [(6, 2),\n                        (3, 7),\n                        (9, 13),\n                        (8, 0.5000000000000003, 0.9999999999999999, -0.06017488501159418)],\n                       [(2, 0.49999999999999956, 0.0, -0.8348009229233395),\n                        (3, 0.0, -0.372707453521945, 0.0),\n                        (7, 0.4999999999999998, 0.0, 0.4213107456171951),\n                        (6, 0.6383222145074643, 0.0, 0.17679703477273234),\n                        (4, 8),\n                        (9, 0.5, 0.0, -0.6257067766658577),\n                        (13, 0.07014815572408309, 0.0, -0.08481842695879838)],\n                       [(2, 3),\n                        (7, 6),\n                        (8, 0.5, 1.0, -0.06017488501159418),\n                        (13, 12),\n                        (4, 0.5, -0.23930035968084695, -0.9385098192597998)],\n                       [(7, 0.5, 0.0, 0.6117352471574098),\n                        (6, 0.4999999999999997, 0.0, -0.7484872360058605),\n                        (2, 0.4546007179287068, 0.0, 0.4167387613319282),\n                        (8, 9),\n                        (13, 0.5000000000000007, 1.1102230246251565e-16, 0.3764843351646857),\n                        (12, 0.5, 0.0, 0.08440524433248409)],\n                       [(3, 7),\n                        (6, 2),\n                        (9, 0.5, 0.0, -0.6257067766658575),\n                        (8, 0.457677335555713, 0.0, -0.4563339738034573)],\n                       [(2, 0.19504058663172466, 0.0, -0.2857088984950029),\n                        (3, 0.29782301439575226, 0.0, 0.16848558344948122),\n                        (6, 0.8858944659256832, 5.551115123125783e-17, 0.25151276399414046),\n                        (9, 13),\n                        (12, 8)],\n                       [(2, 3),\n                        (7, 6),\n                        (8, 0.6037949133843687, 0.0, 0.40194811801047736),\n                        (9, 0.0, -0.3483327506753857, 0.0),\n                        (13, 0.3478558516819395, 0.0, 0.8764843351646865),\n                        (12, 0.23682201779168552, 1.0, 0.5844052443324841)],\n                       [(2, 0.5271017512479523, 5.551115123125783e-17, 0.1739679588648032),\n                        (7, 0.21266361194798095, 0.0, -0.8882647528425918),\n                        (3, 0.5000000000000003, 0.0, -0.6288870691397788),\n                        (8, 9),\n                        (13, 12)],\n                       [(2, 0),\n                        (7, 3),\n                        (13, 0.8070141267910333, 0.0, -0.7408387912329149),\n                        (12, 0.49999999999999944, 0.0, -0.41559475566751614),\n                        (8, 0.7459460754308767, 0.0, -0.6454357089961718)],\n                       [(2, 0.5, 0.0, 0.6739679588648031),\n                        (7, 0.49999999999998257, 0.0, -0.3882646965246508),\n                        (9, 13),\n                        (12, 8),\n                        (0, 0.30408672398469616, 0.0, -0.5000000000000002),\n                        (3, 0.5000000000000019, 5.551115123125783e-17, 0.4457857069642596)],\n                       [(6, 2),\n                        (7, 11),\n                        (9, 0.5, 4.440892098500626e-16, 0.5259604726587561),\n                        (13, 0.5587687851709019, 0.0, 0.9955029279481928),\n                        (12, 0.5, -0.9999999999999996, -0.9155947556675159)],\n                       [(2, 0.8040867239846965, 3.3306690738754696e-16, -0.32603204113519757),\n                        (7, 0.5000000000000102, 1.1102230246251565e-16, -0.3882646965246499),\n                        (8, 9),\n                        (13, 12),\n                        (11, 0.30408672398469627, 0.0, -0.5745192213521484)],\n                       [(2, 0),\n                        (7, 3),\n                        (9, 0.5, 0.0, 0.5259604726587566),\n                        (13, 0.5, 0.0, -0.5618299159145819),\n                        (8, 0.0, 0.5943823240658705, 0.0)],\n                       [(7, 0.7499999999999999, 0.0, 0.6117353359905289),\n                        (9, 13),\n                        (2, 0.75, -0.9999999999999996, -0.32603204113519735),\n                        (12, 8),\n                        (0, 0.5, 0.0, 0.5),\n                        (3, 0.49999999999999956, 0.0, 0.44578573947944156)],\n                       [(7, 11),\n                        (6, 2),\n                        (13, 0.7515732421974963, 0.0, 0.43817008408541896),\n                        (12, 0.5, 0.0, -0.08226142233418265),\n                        (8, 0.0, 1.0, 0.0),\n                        (9, 0.4999999999999997, 0.0, 0.3850511215461382)],\n                       [(2, 0.5000000000000001, 0.0, 0.6739679588648031),\n                        (7, 0.5005268894219383, 0.0, 0.11226227211769357),\n                        (13, 12),\n                        (8, 9),\n                        (11, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515),\n                        (6, 0.5, 0.0, -0.23493636038188193)],\n                       [(2, 0),\n                        (7, 3),\n                        (13, 0.5, -2.220446049250313e-16, -0.06182991591458098),\n                        (12, 0.7945263425651934, 0.0, -0.5822614223341844),\n                        (8, 0.4999999999999998, 1.0, -0.9603519410711705)],\n                       [(7, 0.5, 0.0, -0.13773729180887417),\n                        (9, 13),\n                        (2, 0.25000000000000006, -0.9999999999999998, -0.32603204113519735),\n                        (12, 8),\n                        (3, 0.5000000000000004, 4.440892098500626e-16, 0.9457857394794418)],\n                       [(7, 11),\n                        (6, 2),\n                        (13, 0.5, 0.0, 0.4865743927167295),\n                        (12, 0.5, 0.0, 0.9177385776658147),\n                        (9, 0.5484043086313106, 0.0, 0.8850511215461376),\n                        (8, 0.5, -0.9999999999999998, 0.039648058928830165)],\n                       [(7, 3),\n                        (13, 12),\n                        (2, 0.5, 0.9999999999999999, 0.6739679588648031),\n                        (9, 8),\n                        (11, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (6, 0.5, 0.7349363603818819, 0.7650636396181181)],\n                       [(2, 0),\n                        (9, 0.49999999999998246, -2.220446049250313e-16, 0.3850510652281978),\n                        (13, 0.49999999999998246, 0.0, -0.5134256636012113),\n                        (12, 0.5000000000000019, 0.0, -0.48545280507156496),\n                        (8, 0.5000000000000022, -0.9999999999999996, -0.5571605583337913),\n                        (3, 0.9992548329781763, -0.17200022705138274, 0.4457857394794335),\n                        (7, 0.999999967484822, 0.4115227589059395, -0.13773726914338624)],\n                       [(9, 10),\n                        (13, 14),\n                        (0, 1.0, 0.0, 0.5),\n                        (2, 0.5, 0.8260320411351971, -0.3260320411351971)],\n                       [(13, 0.5000000000000104, 0.0, -0.5134256636012111),\n                        (9, 0.5000000000000102, 0.0, 0.3850510652281978),\n                        (10, 0.30408672398469616, 1.0, -0.5745192213521486),\n                        (14, 0.30408672398469616, 1.0, -0.5745192213521486)],\n                       [(13, 12), (9, 8)],\n                       [(9, 0.75, 0.0, -0.6149489672869812),\n                        (13, 0.75, 0.0, -0.5134256961163892),\n                        (12, 0.49999999999999956, 0.0, 0.5145471624132525),\n                        (8, 0.49999999999999956, 1.0, -0.5571605258186088)],\n                       [(9, 10), (13, 14)],\n                       [(13, 0.500526889421938, 0.0, -0.013952632243553253),\n                        (9, 0.500526889421938, 0.0, -0.11547590341414526),\n                        (10, 0.5000000000000103, 1.0, 0.42548077864785155),\n                        (14, 0.5000000000000103, 1.0, 0.42548077864785155)],\n                       [(13, 12), (9, 8)],\n                       [(9, 0.5, 0.0, 0.13452366051242226),\n                        (13, 0.5, 0.0, -0.7639530683169866),\n                        (12, 0.5000000000000004, 8.881784197001252e-16, -0.9854528375867473),\n                        (8, 0.5000000000000003, 0.9999999999999998, -0.057160525818608754)],\n                       [(9, 10), (13, 14)],\n                       [(13, 12),\n                        (9, 8),\n                        (10, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (14, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(8, 0.9992548329781763, 0.830946038246668, -0.5571605258186167),\n                        (9, 3.251517832758566e-08, 0.13926185191561968, -0.8654763621530666),\n                        (12, 0.0007451670218237004, -0.24076164998520877, -0.48545283758673996),\n                        (13, 3.251517832758566e-08, 0.037738580745028116, -0.7639530909824748)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8),\n                      (6, 5),\n                      (6, 6),\n                      (6, 7),\n                      (6, 8)]},\n (15, 6): {\'method\': \'joint small decoder cores + parallel three-site leaves\',\n           \'moments\': [[(1, 0.5000000000000382, 0.0, -0.5000019337012178),\n                        (3, 0.5000000000000147, 0.0, -0.5000007658083554),\n                        (7, 0.5132067564082936, 0.0, -0.5000362261889171),\n                        (8, 0.7584162415732909, 0.0, -0.4999501908947448),\n                        (4, 0.6230533557941094, 1.0, -0.5000106170491638),\n                        (9, 2.220446049250313e-16, 1.0, -0.2348302905697659),\n                        (6, 0.5000000000000004, 0.0, -0.20251959709327183),\n                        (13, 0.5, 1.0, -0.2656833070307263),\n                        (2, 0.5, 1.0, -0.377500967254607),\n                        (12, 0.4153663251971116, -0.9999999999999996, -0.4155947556675157),\n                        (0, 0.5, 1.0, 0.5),\n                        (11, 0.3040867239846966, 0.0, -0.5745192213521528),\n                        (10, 0.3040867239846966, 1.0, -0.5745192213521528),\n                        (14, 0.3040867239846966, 1.0, -0.5745192213521528)],\n                       [(1, 3), (7, 8), (5, 4)],\n                       [(3, 0.37557409562953725, 0.0, 0.504109668990262),\n                        (7, 0.5265163766591239, 0.0, -0.5457184504550758),\n                        (4, 0.47429358945762257, 0.0, -0.4717737970570637),\n                        (8, 0.29360061860502845, 0.0, 0.2067343314266062),\n                        (1, 0.6183921134981585, 0.0, 0.4049421398506312),\n                        (5, 0.5000000000000001, 0.0, 0.4913009692061716)],\n                       [(3, 7), (4, 8)],\n                       [(3, 0.5, 0.0, -0.6016534624534482),\n                        (4, 0.42618789837524673, 0.0, -0.46603718758518453),\n                        (7, 0.7735304898884182, 1.1102230246251565e-16, -0.2187930900865176),\n                        (8, 0.618670663843772, 0.0, 0.26240664866901664)],\n                       [(3, 4), (7, 8)],\n                       [(3, 0.5000000000000093, 0.0, 0.39834653754654137),\n                        (5, 4),\n                        (7, 0.22609155517551918, 0.0, -0.2060701906402631),\n                        (8, 0.37095638547642845, 0.0, -0.7289142218977207)],\n                       [(1, 3),\n                        (7, 8),\n                        (4, 0.49999999999999895, 0.0, 0.5379994514639419),\n                        (5, 0.42226058550096274, 3.3306690738754696e-16, -0.24577787632214076)],\n                       [(3, 0.5086756775644807, 0.9999999999999999, -0.3632594697163327),\n                        (9, 5),\n                        (7, 0.3418454539690408, 0.0, -0.17313667320303727),\n                        (8, 0.26128653503356075, -0.9999999999999996, 0.21840090406351087),\n                        (1, 0.49999999999999906, 0.0, -0.6344725485666272)],\n                       [(3, 4),\n                        (5, 0.728476421928667, 0.0, 0.3240411515620809),\n                        (9, 0.5, 0.0, -0.1805896763080077)],\n                       [(3, 7), (4, 0.5000000000000078, -4.440892098500626e-16, -0.46200054853605677)],\n                       [(4, 8),\n                        (3, 0.5000000000000036, 4.440892098500626e-16, 0.9356518684993973),\n                        (7, 0.20153995331528243, 0.0, 0.15027161362888286)],\n                       [(1, 3),\n                        (7, 6),\n                        (4, 0.5000000000000018, 0.0, -0.6253216226694649),\n                        (8, 0.7398919760248123, 1.0, -0.3420442310691145)],\n                       [(4, 8),\n                        (7, 0.6822968314790296, 0.9999999999999999, 0.8968837321902037),\n                        (6, 0.2509322168477317, 0.0, -0.7949290191088039),\n                        (1, 0.49999999999999944, 1.0, -0.11838123509915977)],\n                       [(4, 0.2853670108876343, 0.0, 0.6507839884375588),\n                        (8, 0.1068742790842334, 8.881784197001252e-16, 0.8320563951668203),\n                        (3, 7)],\n                       [(5, 4),\n                        (8, 9),\n                        (7, 0.5, 0.0, 0.6746360341167585),\n                        (3, 0.5728573590437842, 0.0, 0.383965653855137)],\n                       [(7, 6),\n                        (9, 0.46352713151159636, 0.0, 0.2640454433674886),\n                        (5, 0.0, -0.1323202671766246, 0.0),\n                        (1, 3),\n                        (4, 0.9108537925987485, 0.0, 0.9407864855785952),\n                        (8, 0.5, 1.0, -0.8530631724110307)],\n                       [(9, 5),\n                        (4, 8),\n                        (7, 0.2673990199988756, 0.0, -0.3253639658832419),\n                        (6, 0.5826592798249117, 0.0, 0.6305613443031177),\n                        (1, 0.41785980112400095, 1.0, -0.6183812350991598)],\n                       [(7, 6),\n                        (8, 0.8172104746408917, 0.0, -0.8530631724110305),\n                        (9, 0.6058193373299144, 0.0, 0.09006893310335506),\n                        (1, 3),\n                        (5, 0.0, 1.0, 0.0),\n                        (4, 0.5377801164087216, 5.551115123125783e-17, 0.45135815473574914)],\n                       [(8, 9),\n                        (5, 4),\n                        (3, 0.3856735769590187, 0.0, -0.7498049155479862),\n                        (7, 0.5505347324801626, 0.9999999999999998, 0.17463603411675815),\n                        (6, 0.3689376239568078, 0.0, 0.7719771085165397),\n                        (1, 0.4999999999999998, 0.9999999999999998, -0.11838123509915954)],\n                       [(3, 7),\n                        (4, 0.8266664524069582, 8.881784197001252e-16, -0.8452042998902514),\n                        (8, 0.045931842050171756, 0.9999999999999999, 0.6469368275889693),\n                        (9, 0.3263740628486297, 1.0, 0.2755902782462124)],\n                       [(4, 8),\n                        (9, 5),\n                        (3, 0.4633349244593254, 0.0, 0.2481313913535887),\n                        (7, 0.3213111929430337, 0.9999999999999998, 0.6746360341167583)],\n                       [(3, 7),\n                        (4, 0.4273399417518991, 0.0, -0.4516854861592332),\n                        (8, 0.9539104356517047, 1.0, -0.8530631724110302),\n                        (9, 0.11904903998422632, 1.0, 0.6792811142119082)],\n                       [(4, 8),\n                        (9, 5),\n                        (7, 0.566656815524444, 0.0, -0.8253639658832417),\n                        (3, 0.43721065119527336, 0.0, -0.5874927839188446)],\n                       [(4, 0.25349168400187955, 0.0, -0.13958952140000358),\n                        (7, 6),\n                        (8, 0.15644213008117777, 0.0, -0.3530631724110278),\n                        (9, 0.6917633216559813, 0.0, 0.8660475973984443),\n                        (1, 3)],\n                       [(5, 4),\n                        (8, 9),\n                        (3, 0.1039903575622847, 0.0, -0.1883939950186273),\n                        (7, 0.5850425523318398, 0.9999999999999998, 0.6746360341167583),\n                        (6, 0.8311737009895825, 0.0, -0.0841139332337324),\n                        (1, 0.5, 1.0, -0.7684354937498763)],\n                       [(9, 0.03151537942096394, 0.0, -0.7185699907363328),\n                        (5, 0.0, 0.8771962368439331, 0.0),\n                        (3, 7),\n                        (4, 0.562586548020497, 0.0, 0.16771191693830984),\n                        (8, 0.5000000000000001, 1.0, -0.35306317241102914)],\n                       [(9, 5),\n                        (4, 8),\n                        (7, 0.49999999999999956, 0.0, 0.6746360341167585),\n                        (3, 0.43415359559636596, 0.0, 0.04328067612258829)],\n                       [(7, 6),\n                        (8, 0.4546464932122306, 0.0, 0.23083348567577783),\n                        (9, 0.4392960432426437, 0.0, -0.28535129208653487),\n                        (1, 3),\n                        (5, 0.5, 1.0, -0.009374223159635875),\n                        (4, 0.5, 0.0, 0.9807195943409128)],\n                       [(8, 9),\n                        (5, 4),\n                        (3, 0.4999999999999998, 0.0, -0.1528360387383385),\n                        (7, 0.5998657277943455, 1.0, -0.22554015234003089),\n                        (1, 0.553074255385617, 1.0, -0.2684354937498765)],\n                       [(4, 0.0, -0.783243063064585, 0.0),\n                        (3, 7),\n                        (9, 13),\n                        (5, 0.5, 0.3315640442189882, 0.9906257768403643)],\n                       [(4, 8),\n                        (9, 0.3778861449265366, 5.551115123125783e-17, 0.21456099605698642),\n                        (3, 0.5000000000000003, 0.0, 0.34716396126166105),\n                        (7, 2.220446049250313e-16, 1.0, 0.8488883387009589),\n                        (13, 0.06834101983091842, -0.9999999999999996, 0.7789857130606195)],\n                       [(8, 9),\n                        (1, 3),\n                        (7, 6),\n                        (4, 0.49999999999999956, 0.9999999999999998, 0.6576569853087455)],\n                       [(3, 0.5000000000000001, 0.0, 0.34716396126166127),\n                        (7, 0.5, 0.9999999999999999, -0.4010686979305582),\n                        (8, 0.4133199160673859, 0.0, -0.24235737093112597),\n                        (6, 0.6339658219122923, 0.0, 0.8651689462862566),\n                        (9, 0.4999999999999994, 0.0, -0.9214280136355619),\n                        (1, 0.5, 0.6085535507108716, -0.26843549374987674)],\n                       [(3, 7), (4, 8), (6, 2), (9, 13)],\n                       [(2, 0.03592107551278813, 1.0, -0.8775009672546068),\n                        (3, 0.5991006024208341, 0.0, 0.1849641514941105),\n                        (4, 0.11253431722702033, 1.0, -0.8423430146912545),\n                        (7, 0.5, 1.0, -0.4010686979305591),\n                        (6, 0.9068032572794498, 1.1102230246251565e-16, 0.23132164282033318),\n                        (9, 0.6865025359633383, 1.1102230246251565e-16, 0.07857198636443863),\n                        (13, 0.5297395290041438, 0.9999999999999999, -0.4250491790947357)],\n                       [(2, 3), (4, 8), (7, 6), (9, 13)],\n                       [(3, 0.6983902210776689, 0.0, 0.39654986447965435),\n                        (8, 0.4609737351550911, 0.0, 0.07130330471586532),\n                        (9, 0.960833440065644, 0.0, -0.42142801363556837),\n                        (6, 0.4457229604434565, 0.0, 0.01261445611041534),\n                        (2, 0.751811597304503, -0.9999999999999996, 0.6224990327453925),\n                        (4, 0.03981954596758296, 1.0, -0.8423430146912616),\n                        (13, 0.801538355530847, 1.0, -0.05577781524071823)],\n                       [(3, 7), (8, 9), (6, 2)],\n                       [(8, 0.38627649394841107, 0.0, -0.6865002368281361),\n                        (9, 0.16629816714536846, 0.0, 0.07857198636442414),\n                        (2, 0.5, 1.0, -0.3775009672546079),\n                        (3, 0.6251415065033563, 4.440892098500626e-16, 0.5441612987547761),\n                        (7, 0.0, -0.3255499447294259, 0.0),\n                        (6, 0.7856407379834285, 0.0, -0.41370794677841705)],\n                       [(8, 9), (2, 3), (7, 6)],\n                       [(3, 0.0, 0.28089726190378134, 0.0),\n                        (7, 0.5, 0.0, 0.27338135734001656),\n                        (8, 0.5824624691266849, 0.0, 0.9854775829626989),\n                        (6, 0.4949978217082433, -2.220446049250313e-16, 0.08629205322158295),\n                        (2, 0.4999999999999998, 1.0, -0.8814144405744395),\n                        (9, 0.5, 4.440892098500626e-16, -0.9214280136355764)],\n                       [(3, 7), (4, 8), (6, 2), (9, 13)],\n                       [(2, 3),\n                        (8, 0.5, 0.0, -0.5998183085633881),\n                        (9, 0.38819804894301435, 0.0, 0.17269198314637357),\n                        (7, 0.5, 1.0, -0.7266186426599841),\n                        (6, 0.5, 0.0, -0.4475413783399338),\n                        (4, 0.599987907871844, -0.9999999999999998, 0.15765698530874528)],\n                       [(8, 9), (7, 6), (2, 0.2887068963335278, 0.0, 0.899671023811214)],\n                       [(6, 0.8015319629186126, 0.0, -0.4475413783399338),\n                        (9, 13),\n                        (7, 0.5, 1.0, 0.15744051800159986),\n                        (8, 0.5000000000000004, 0.0, 0.9001816914366119)],\n                       [(6, 2),\n                        (3, 7),\n                        (4, 8),\n                        (9, 0.5, 0.0, -0.6257067766658579),\n                        (13, 0.07014815572408309, 0.0, -0.08481842695879838)],\n                       [(2, 0.10822807679614659, 0.0, 0.5488202087084497),\n                        (3, 0.5000000000000004, 4.440892098500626e-16, -0.9259027950053544),\n                        (8, 0.5, 0.0, 0.9001816914366121),\n                        (7, 0.0, 1.0, 0.0),\n                        (6, 0.4999999999999993, 8.881784197001252e-16, -0.947541378339934),\n                        (13, 12),\n                        (4, 0.5, -0.33546716424939227, 0.15765698530874528)],\n                       [(2, 3),\n                        (8, 9),\n                        (7, 6),\n                        (13, 0.5000000000000007, 1.1102230246251565e-16, 0.3764843351646857),\n                        (12, 0.5, 0.9999999999999998, 0.08440524433248403)],\n                       [(9, 0.5, 0.0, 0.37429322333414206),\n                        (2, 0.18023611392199, 0.0, 0.548820208708451),\n                        (7, 0.5663325445329528, 0.9999999999999998, 0.6574405180016001),\n                        (3, 0.4999999999999998, 0.0, -0.9259027950053547),\n                        (8, 0.4576773355557129, 0.0, 0.29634078022847543)],\n                       [(9, 13), (2, 0), (7, 3), (12, 8)],\n                       [(2, 0.5, 0.0, -0.9511797912915487),\n                        (7, 0.4999999999999824, 8.881784197001252e-16, -0.8425594256804598),\n                        (8, 0.6037949133843687, 0.0, -0.5619413115854588),\n                        (9, 0.0, -0.3483327506753857, 0.0),\n                        (13, 0.3478558516819395, 0.0, 0.8764843351646865),\n                        (12, 0.23682201779168552, 1.0, 0.5844052443324841),\n                        (0, 0.3040867239846962, 1.0, -0.5000000000000002),\n                        (3, 0.5000000000000022, 0.0, -0.29323770593944953)],\n                       [(6, 2), (7, 11), (8, 9), (13, 12)],\n                       [(13, 0.8070141267910333, 0.0, -0.7408387912329149),\n                        (2, 0.8040867239846965, 8.881784197001252e-16, -0.9511797912915487),\n                        (7, 0.5000000000000104, 1.0, -0.8425594256804596),\n                        (12, 0.49999999999999944, 1.0, -0.41559475566751614),\n                        (8, 0.7459460754308767, 0.0, 0.48544251542118966),\n                        (11, 0.30408672398469627, 0.0, -0.5745192213521484),\n                        (6, 0.0, 0.6350392865765455, 0.0)],\n                       [(9, 13), (2, 0), (7, 3), (12, 8)],\n                       [(7, 0.75, 0.0, 0.15744060683471867),\n                        (2, 0.7499999999999999, 8.881784197001252e-16, -0.9511797912915487),\n                        (8, 0.0, 0.6416837974414842, 0.0),\n                        (9, 0.5, 4.440892098500626e-16, 0.5259604726587561),\n                        (13, 0.5587687851709019, 0.0, 0.9955029279481928),\n                        (12, 0.5, -0.9999999999999996, -0.9155947556675159),\n                        (0, 0.5, 1.0, 0.5),\n                        (3, 0.49999999999999956, 0.0, -0.2932377384546312)],\n                       [(7, 11), (6, 2), (8, 9), (13, 12)],\n                       [(9, 0.5, 0.0, -0.47403952734124344),\n                        (13, 0.5, 0.0, -0.5618299159145819),\n                        (12, 8),\n                        (2, 0.5, -2.220446049250313e-16, 0.04882020870845155),\n                        (7, 0.500526889421938, 1.0, -0.3420324570381177),\n                        (11, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515),\n                        (6, 0.5, 1.0, -0.47238403480239244)],\n                       [(9, 13), (2, 0), (7, 3), (12, 0.5, 0.0, 0.9177385776658173)],\n                       [(7, 0.5, 0.0, -0.5920320209646845),\n                        (9, 0.4999999999999998, 0.0, -0.6149488784538617),\n                        (13, 0.7515732421974963, 0.0, 0.43817008408541896),\n                        (2, 0.25, 0.0, -0.9511797912915487),\n                        (3, 0.5000000000000003, 0.0, -0.7932377384546314)],\n                       [(7, 11), (8, 9), (13, 12), (6, 2)],\n                       [(2, 0.5, 0.0, 0.04882020870845155),\n                        (13, 0.5, -2.220446049250313e-16, -0.06182991591458098),\n                        (12, 0.7945263425651934, 0.0, 0.41773857766581557),\n                        (8, 0.4999999999999998, 0.0, -0.9635751309964571),\n                        (7, 1.0214051826551437e-14, 1.0, -0.09203911187550995),\n                        (6, 0.5, 0.9723840348023929, -0.4723840348023929),\n                        (11, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(2, 0), (9, 13), (12, 8), (7, 3)],\n                       [(9, 0.5484043086313106, 0.0, 0.8850511215461376),\n                        (8, 0.4999999999999997, 0.0, -0.9635751309964573),\n                        (13, 0.5, 0.0, 0.4865743927167295),\n                        (12, 0.5, 0.0, 0.9177385776658147),\n                        (0, 1.0, 0.0, 0.0),\n                        (7, 0.999999967484822, 0.86581748806175, -0.5920319982991964),\n                        (2, 0.4999999999999998, 0.45117979129154895, -0.951179791291549),\n                        (3, 0.0007451670218237004, 0.5670232508826754, 0.7067622615453761)],\n                       [(9, 8), (13, 12)],\n                       [(9, 0.49999999999998246, 0.0, -0.6149489347718022),\n                        (13, 0.49999999999998246, 0.0, -0.5134256636012113),\n                        (8, 0.5000000000000022, 0.0, 0.6332334862661635),\n                        (12, 0.5000000000000022, 0.0, -0.4854528050715654)],\n                       [(9, 10), (13, 14)],\n                       [(9, 0.5000000000000103, 0.0, -0.6149489347718022),\n                        (13, 0.5000000000000104, 0.0, -0.5134256636012111),\n                        (10, 0.30408672398469616, 1.0, -0.5745192213521486),\n                        (14, 0.30408672398469616, 1.0, -0.5745192213521486)],\n                       [(9, 8), (13, 12)],\n                       [(9, 0.75, 0.0, -0.6149489672869812),\n                        (13, 0.75, 0.0, -0.5134256961163892),\n                        (8, 0.49999999999999944, 1.1102230246251565e-16, -0.36676654624901817),\n                        (12, 0.49999999999999956, 0.0, 0.5145471624132525)],\n                       [(9, 10), (13, 14)],\n                       [(9, 0.500526889421938, 0.0, -0.11547590341414526),\n                        (13, 0.500526889421938, 0.0, -0.013952632243553253),\n                        (10, 0.5000000000000103, 1.0, 0.42548077864785155),\n                        (14, 0.5000000000000103, 1.0, 0.42548077864785155)],\n                       [(9, 8), (13, 12)],\n                       [(9, 0.5, 8.881784197001252e-16, -0.865476339487578),\n                        (13, 0.5, 0.0, -0.7639530683169866),\n                        (8, 0.5000000000000004, 0.0, 0.13323345375098133),\n                        (12, 0.5000000000000004, 8.881784197001252e-16, -0.9854528375867473)],\n                       [(9, 10), (13, 14)],\n                       [(9, 8),\n                        (13, 12),\n                        (10, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (14, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(8, 0.0007451670218237004, 0.6405520586770601, 0.6332334537509905),\n                        (9, 3.251517832758566e-08, 0.13926185191561968, -0.8654763621530666),\n                        (12, 0.0007451670218237004, -0.24076164998520877, -0.48545283758673996),\n                        (13, 3.251517832758566e-08, 0.037738580745028116, -0.7639530909824748)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8),\n                      (6, 5),\n                      (6, 6),\n                      (6, 7),\n                      (6, 8)]},\n (15, 7): {\'method\': \'joint small decoder cores + parallel three-site leaves\',\n           \'moments\': [[(1, 0.4988189621527233, 4.440892098500626e-16, 0.5000132890801281),\n                        (3, 0.49873408182877055, 0.0, -0.4999842579431477),\n                        (5, 0.370281875930452, 0.0, -0.4999633932626457),\n                        (4, 0.593781864502583, 0.0, 0.5000268807909329),\n                        (7, 0.8986079470749518, -0.9999999999999998, -0.4999992457736355),\n                        (9, 0.4527182707706126, 1.0, -0.24751666824061802),\n                        (6, 0.5, 1.0, -0.20251771408753827),\n                        (13, 0.5000000000000013, 1.0, -0.17371849805590767),\n                        (2, 0.49999999999999944, 1.0, -0.3649528224832399),\n                        (0, 0.5, 1.0, 0.5),\n                        (12, 0.4727334744730367, 1.0, -0.3928638400947648),\n                        (11, 0.3040867239846966, 1.0, -0.5745192213521528),\n                        (10, 0.3040867239846966, 1.0, -0.5745192213521528),\n                        (14, 0.3040867239846966, 1.0, -0.5745192213521528)],\n                       [(1, 3), (5, 4), (7, 8)],\n                       [(3, 0.7410621194120419, 0.0, 0.5026315957156993),\n                        (7, 0.8986079470749517, 0.0, -0.4999992457736355),\n                        (4, 0.707998139485833, 0.9999999999999999, -0.4999731192090664),\n                        (8, 0.02486224694339967, 0.0, 0.5026625968514971),\n                        (1, 0.7389639984597631, 4.440892098500626e-16, 0.5000132890801436),\n                        (5, 0.6884836038452212, 0.0, -0.49864821530723846)],\n                       [(3, 7), (4, 8)],\n                       [(3, 0.5021057977311296, 0.0, -0.49558393080413765),\n                        (4, 0.5000000000000013, 0.0, 0.4973617077789517),\n                        (8, 0.9476399747011979, 8.881784197001252e-16, 0.524802191730708)],\n                       [(3, 4), (7, 8)],\n                       [(3, 0.49999999999999867, 0.0, -0.48626162633137593),\n                        (4, 0.4999999999999983, 0.0, 0.49736170777895117),\n                        (7, 0.7832003174946012, 4.440892098500626e-16, 0.6577522061234371),\n                        (8, 0.4577081401521621, 1.1102230246251565e-16, -0.45618851185746223)],\n                       [(1, 3), (5, 4)],\n                       [(3, 0.5000000000000007, 0.9999999999999999, 0.5137383736686241),\n                        (4, 0.9290282784717401, 1.0, -0.5153703277761099),\n                        (5, 0.9291397888772785, 0.0, 0.0020858368975522623)],\n                       [(3, 7), (4, 8)],\n                       [(3, 0.4999999999999927, 0.0, -0.5281794010626368),\n                        (4, 0.19555838049871357, 0.0, 0.47157482966131425),\n                        (7, 0.5457235976529733, 0.0, -0.7851348671771659),\n                        (8, 0.5903722393270099, 0.0, -0.4594239333836514)],\n                       [(3, 4), (7, 8)],\n                       [(4, 0.49999999999999556, 0.0, -0.5340954322728788),\n                        (3, 0.5000000000000004, 1.0, 0.4718205989373625),\n                        (7, 0.5, 4.440892098500626e-16, -0.9978144890807483),\n                        (8, 0.045371944551592325, 8.881784197001252e-16, 0.5722096808492829)],\n                       [(5, 4), (3, 7), (8, 9)],\n                       [(9, 0.26368442749281984, 0.0, 0.7524833317593826),\n                        (5, 0.5, 0.0, -0.4979141631024311),\n                        (3, 0.5000000000000091, 0.9999999999999998, 0.5109158203331519),\n                        (4, 0.0, 1.0, 0.0),\n                        (8, 0.5, 0.0, 0.07220968084928536),\n                        (7, 0.5000000000000007, 0.0, 0.5600828633462807)],\n                       [(9, 5), (1, 3), (4, 8), (7, 6)],\n                       [(7, 0.40250888547893265, 0.0, -0.7546990150382955),\n                        (5, 0.5, 3.3306690738754696e-16, -0.41877879155744013),\n                        (8, 0.7110861852691608, 0.0, -0.06933083235557369),\n                        (9, 0.5000000000000003, 1.0, -0.24751666824061846),\n                        (1, 0.5, 0.0, 0.9572434519413031),\n                        (6, 0.23824984739858224, -0.9999999999999996, 0.04802556671795083)],\n                       [(3, 7), (5, 4), (8, 9)],\n                       [(3, 0.49999999999999967, -2.220446049250313e-16, 0.010915820333153137),\n                        (9, 0.8512987536381686, 0.0, 0.46838204771103825),\n                        (5, 0.4999999999999998, 0.0, 0.5812212084425605),\n                        (4, 0.6924966344652026, -0.9999999999999998, 0.8993204749982835),\n                        (8, 0.5, 0.0, -0.05528203938544429),\n                        (7, 0.5000000000000018, 0.0, 0.7453009849617027)],\n                       [(1, 3), (9, 5), (4, 8), (7, 6)],\n                       [(1, 0.5000000000000004, 0.0, 0.4572434519413029),\n                        (4, 0.3805885884955774, 0.0, -0.559931565529574),\n                        (7, 0.49999999999999906, 0.0, -0.2546990150382973),\n                        (6, 0.8228387639951986, 1.0, 0.24879131417776962),\n                        (8, 0.5525901244753028, 1.1102230246251565e-16, -0.05528203938544385),\n                        (9, 0.1797198598247371, 1.0, -0.39092807885471936)],\n                       [(1, 3), (5, 4), (7, 6), (8, 9)],\n                       [(3, 0.36532098276163216, 0.0, -0.6581286069636674),\n                        (7, 0.4568853839401346, 0.0, -0.7546990150383),\n                        (4, 0.5119952026399054, 1.0, -0.5804601895274484),\n                        (8, 0.0003506775695956146, 0.0, -0.5552820393853861),\n                        (9, 0.5130535017498933, 1.0, -0.24237337735644848),\n                        (1, 0.5, 0.0, 0.9572434519413029),\n                        (6, 0.6082288892935703, 1.0, 0.24879131417776595)],\n                       [(3, 7), (4, 8), (9, 5)],\n                       [(3, 0.15979316138258426, 0.0, -0.5828019255861889),\n                        (7, 0.5409187240915672, 0.0, -0.25469901503829817),\n                        (4, 0.5298022113453075, -0.9999999999999998, -0.2130136574961603),\n                        (8, 0.7694888139508351, 0.0, -0.05528203938544429),\n                        (9, 0.9314514367511142, 1.0, 0.9547885656839177)],\n                       [(3, 7), (4, 8), (9, 5)],\n                       [(3, 0.17667382808391469, 0.0, -0.43315493378420467),\n                        (4, 0.1389801686104124, 1.0, -0.6991122817811923),\n                        (7, 0.0820855992752405, 0.0, 0.24530098496170083),\n                        (8, 0.4171451942563298, 0.0, -0.555282039385445),\n                        (9, 0.6036471329534792, 1.0, -0.9213870733171643)],\n                       [(1, 3), (5, 4), (7, 6), (8, 9)],\n                       [(3, 0.5000000000000024, 0.0, 0.895320143730995),\n                        (7, 0.4999999999999982, 0.0, 0.7453009849617007),\n                        (9, 0.20617180405202437, 0.0, 0.9335533390268933),\n                        (5, 0.0, -0.16980719486940687, 0.0),\n                        (4, 0.5961764807855426, 1.0, -0.3599814607223144),\n                        (8, 0.5, 0.0, -0.5552820393854452),\n                        (1, 0.5, 4.440892098500626e-16, 0.6048119872247699),\n                        (6, 0.2692579739648868, 1.0, -0.7512086858222318)],\n                       [(3, 7), (9, 5), (4, 8)],\n                       [(3, 0.35284137925881676, 0.0, -0.104679856269005),\n                        (5, 0.5, 0.0, 0.11660946277210149),\n                        (4, 0.5000000000000002, 0.0, 0.003521231949387804),\n                        (7, 0.5000000000000003, 0.0, -0.25469901503829817),\n                        (8, 0.18521437083459932, 0.0, 0.45699178098013193),\n                        (9, 0.5460931463447297, 1.0, 0.47505274170880973)],\n                       [(1, 3), (5, 4), (7, 6), (8, 9)],\n                       [(3, 0.5, 0.0, 0.8953201437309981),\n                        (7, 0.269060447136372, 1.1102230246251565e-16, 0.24530098496170083),\n                        (4, 8),\n                        (9, 13),\n                        (1, 0.3016886194775281, 4.440892098500626e-16, -0.8951880127752294),\n                        (5, 0.5, 0.2011581389943249, -0.8833905372278987)],\n                       [(3, 7),\n                        (8, 0.0, 0.17567641768561693, 0.0),\n                        (9, 0.21483673072595866, 0.0, -0.27262733750663903),\n                        (4, 0.0, 0.41426486019210396, 0.0),\n                        (13, 0.386575549221946, 1.0, -0.7541754089658816)],\n                       [(3, 0.5000000000000004, 0.0, 0.3953201437309985), (7, 6), (8, 9)],\n                       [(1, 3),\n                        (7, 0.5, 0.0, 0.24530098496170183),\n                        (8, 0.4750788623520885, 0.0, -0.36550647055235164),\n                        (6, 2),\n                        (9, 0.4999999999999998, 0.0, -0.9852131610618484)],\n                       [(3, 0.5, 0.0, -0.6046798562690014),\n                        (4, 8),\n                        (9, 13),\n                        (6, 0.564919347967729, 0.0, -0.8746225131559058),\n                        (2, 0.3765861726663244, -0.9999999999999996, 0.13504717751675888),\n                        (1, 0.5, 0.23533835620664556, -0.8951880127752292)],\n                       [(3, 7),\n                        (8, 0.9762240977328398, 1.0, -0.6487801426090742),\n                        (9, 0.4034398069250414, 0.0, -0.9852131610618478),\n                        (13, 0.39686191691086053, 0.9999999999999998, 0.5866748822893892)],\n                       [(7, 0.5, 4.440892098500626e-16, -0.7546990150382982),\n                        (3, 0.5000000000000038, 0.0, -0.604679856269005),\n                        (8, 9)],\n                       [(7, 6),\n                        (2, 3),\n                        (8, 0.5218195191679285, 0.9999999999999998, 0.3182118161882392),\n                        (9, 0.4125773033740703, 0.0, -0.4852131610618464)],\n                       [(3, 0.6124202497959055, 0.0, 0.29233039878858946),\n                        (7, 0.0, -0.6242513880498217, 0.0),\n                        (6, 0.5000000000000004, -4.440892098500626e-16, -0.3746225131559058),\n                        (2, 0.7664718219180737, 0.9999999999999998, 0.635047177516759),\n                        (8, 9)],\n                       [(3, 7),\n                        (6, 2),\n                        (8, 0.2539342848410182, 0.0, -0.14528696254026796),\n                        (9, 0.15286141079706939, 4.440892098500626e-16, -0.9852131610618473)],\n                       [(4, 8),\n                        (6, 0.8371608839553215, 0.0, 0.17545611282181062),\n                        (2, 0.5000000000000002, 1.0, -0.36495282248324123),\n                        (3, 0.7237608380026965, 0.0, -0.821159251283202),\n                        (9, 13)],\n                       [(7, 6),\n                        (2, 3),\n                        (8, 0.4017684760043936, 1.0, 0.7929749903225811),\n                        (9, 0.5210000735134734, 3.3306690738754696e-16, -0.4852131610618473),\n                        (4, 0.5, 1.1102230246251565e-16, 0.4177860921414921),\n                        (13, 0.6771418776405094, -0.9999999999999996, 0.3355741136763307)],\n                       [(3, 0.0, -0.48442455383247607, 0.0),\n                        (7, 0.5, 4.440892098500626e-16, 0.6210495969118799),\n                        (6, 0.647259141517744, 0.0, 0.3022144361017878),\n                        (2, 0.5000000000000001, 1.0, -0.8649528224832415),\n                        (8, 9)],\n                       [(3, 7),\n                        (6, 2),\n                        (8, 0.347095495162088, 0.0, 0.729842737944892),\n                        (9, 0.5000000000000003, 1.1102230246251565e-16, -0.48521316106184686)],\n                       [(2, 3),\n                        (4, 8),\n                        (7, 0.5, 2.220446049250313e-16, -0.3789504030881212),\n                        (6, 0.4999999999999998, 0.0, -0.31149386997272455),\n                        (9, 13)],\n                       [(7, 6),\n                        (2, 0.5, 0.0, 0.9683805108500918),\n                        (8, 0.5, 1.0, 0.41981185270567284),\n                        (9, 0.6257532079246091, 0.0, 0.105662581990142),\n                        (4, 0.48905598751615315, 0.0, 0.9177860921414922)],\n                       [(7, 0.5, 0.0, -0.6167297569466204),\n                        (6, 0.6633380081977305, 4.440892098500626e-16, 0.6885061300272752),\n                        (8, 9)],\n                       [(3, 7), (6, 2), (8, 0.5, 0.0, -0.0801881472943271), (9, 13)],\n                       [(2, 0.8248194291522575, 0.0, -0.5316194891499073),\n                        (3, 0.5000000000000004, 0.0, 0.33731453235433884),\n                        (4, 8),\n                        (6, 0.5000000000000001, 0.0, 0.18850613002727545),\n                        (9, 0.4999999999999998, 0.0, -0.3453781411600303),\n                        (13, 0.4316343725850984, 0.0, -0.23820612346952252)],\n                       [(2, 3),\n                        (7, 6),\n                        (8, 0.5, 1.0, -0.0801881472943271),\n                        (13, 12),\n                        (4, 0.5000000000000001, -0.6000184903750672, 0.9177860921414922)],\n                       [(7, 0.5663648310033746, 0.0, 0.8832702430533805),\n                        (3, 0.5, 0.0, 0.3373145323543385),\n                        (2, 0.18030068686283401, 5.551115123125783e-17, 0.4683805108500939),\n                        (8, 9),\n                        (13, 0.801165141780831, 0.0, -0.7892978164059836),\n                        (12, 0.500000000000002, 0.0, -0.8928638400947646)],\n                       [(7, 3),\n                        (2, 0),\n                        (9, 0.5, 0.0, -0.34537814116003007),\n                        (8, 0.5391668189464952, 0.0, -0.13887512237013855)],\n                       [(2, 0.5, 0.0, 0.9683805108500936),\n                        (9, 13),\n                        (12, 8),\n                        (7, 0.49999999999998224, 0.0, -0.6167298132645609),\n                        (3, 0.5000000000000022, 0.0, -0.029955805638912203),\n                        (0, 0.3040867239846962, 1.0, -0.5000000000000002)],\n                       [(6, 2),\n                        (7, 11),\n                        (8, 0.7160350962932711, 2.220446049250313e-16, -0.02095066378824839),\n                        (9, 0.0, -0.026668701851910503, 0.0),\n                        (13, 0.8221427976501903, 0.0, -0.7272472749755741),\n                        (12, 0.718842061451741, 0.9999999999999999, 0.6071361599052354)],\n                       [(7, 0.5000000000000103, 0.0, -0.6167298132645609),\n                        (8, 9),\n                        (2, 0.8040867239846965, 0.0, 0.9683805108500936),\n                        (13, 12),\n                        (6, 0.0, 0.17257469853648622, 0.0),\n                        (11, 0.30408672398469616, 1.0, -0.5745192213521486)],\n                       [(7, 3),\n                        (2, 0),\n                        (13, 0.6929081336817443, 0.0, 0.8048476615380329),\n                        (12, 0.4999999999999998, 0.0, -0.3928638400947646),\n                        (8, 0.6159237840503593, 0.0, 0.8527343271001073)],\n                       [(9, 13),\n                        (12, 8),\n                        (2, 0.7499999999999999, 0.0, 0.9683805108500936),\n                        (7, 0.75, 0.0, -0.616729845779739),\n                        (3, 0.49999999999999956, 0.0, 0.9700441618459059),\n                        (0, 0.5, 1.0, 0.5)],\n                       [(6, 2),\n                        (7, 11),\n                        (9, 0.5, 0.0, 0.6279531569880601),\n                        (13, 0.5090895496123653, 0.0, 0.3048476615380329),\n                        (12, 0.4999999999999998, 1.0, -0.9527477008638054)],\n                       [(7, 0.500526889421938, 0.0, -0.11725678190690392),\n                        (8, 9),\n                        (2, 0.5, 0.0, -0.03161948914990642),\n                        (13, 12),\n                        (6, 0.5, -0.9999999999999996, -0.7987688280048215),\n                        (11, 0.5000000000000103, 1.0, 0.42548077864785155)],\n                       [(7, 3),\n                        (2, 0),\n                        (9, 0.5, 0.0, 0.6279531569880601),\n                        (13, 0.5000000000000003, 0.0, -0.3968018388516166),\n                        (8, 0.0, 0.7658919006045153, 0.0)],\n                       [(9, 13),\n                        (2, 0.2500000000000001, 0.0, 0.9683805108500934),\n                        (7, 0.5, 0.0, -0.8672572179803364),\n                        (12, 8),\n                        (3, 0.5000000000000003, 0.0, -0.5299558381540947)],\n                       [(6, 2),\n                        (7, 11),\n                        (13, 0.7426364901337341, 0.0, 0.6031981611483839),\n                        (12, 0.8302927422861927, 0.0, -0.39531858476709036),\n                        (8, 0.0, 1.0, 0.0),\n                        (9, 0.4999999999999998, 0.0, 0.7892534928835118)],\n                       [(2, 0.5, 0.0, -0.03161948914990642),\n                        (7, 3),\n                        (13, 12),\n                        (8, 9),\n                        (6, 0.5, -0.7012311719951787, -0.7987688280048215),\n                        (11, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(2, 0),\n                        (13, 0.5, 0.0, 0.10319816114838531),\n                        (12, 0.5321581041534849, 0.0, 0.6711169572607618),\n                        (8, 0.5, 1.0, 0.34813113914715776),\n                        (3, 0.0007451670218237004, -0.6962586494178638, -0.02995583815408631),\n                        (7, 3.251517832758566e-08, 0.14104273040837878, -0.8672572406458252)],\n                       [(9, 13),\n                        (12, 8),\n                        (0, 1.0, 0.0, 0.0),\n                        (2, 0.4999999999999998, 0.5316194891499064, 0.9683805108500936)],\n                       [(13, 0.5, 0.0, 0.6471802504867687),\n                        (12, 0.5000000000000004, 0.0, 0.17111695726076182),\n                        (9, 0.5439820893383833, 0.0, -0.7107465071164878),\n                        (8, 0.4999999999999998, 1.0, -0.6518688608528422)],\n                       [(13, 12), (9, 8)],\n                       [(9, 0.49999999999998224, 0.0, 0.7892534365655712),\n                        (13, 0.4999999999999826, 0.0, -0.3528198058311718),\n                        (12, 0.5000000000000022, 0.0, 0.7590811359375276),\n                        (8, 0.500000000000002, 0.9999999999999999, 0.7601669604703909)],\n                       [(9, 10), (13, 14)],\n                       [(13, 0.5000000000000103, 0.0, -0.3528198058311718),\n                        (9, 0.5000000000000102, 0.0, 0.7892534365655712),\n                        (10, 0.30408672398469616, 1.0, -0.5745192213521486),\n                        (14, 0.30408672398469616, 1.0, -0.5745192213521486)],\n                       [(13, 12), (9, 8)],\n                       [(9, 0.75, 0.0, -0.21074659594960732),\n                        (13, 0.7499999999999999, 0.0, -0.3528198383463499),\n                        (12, 0.49999999999999944, 0.0, -0.24091889657765364),\n                        (8, 0.49999999999999956, 0.9999999999999999, 0.7601669929855734)],\n                       [(9, 10), (13, 14)],\n                       [(13, 0.500526889421938, 0.0, 0.1466532255264852),\n                        (9, 0.500526889421938, 1.1102230246251565e-16, 0.2887264679232281),\n                        (10, 0.5000000000000103, 1.0, 0.42548077864785155),\n                        (14, 0.5000000000000103, 1.0, 0.42548077864785155)],\n                       [(13, 12), (9, 8)],\n                       [(9, 0.5, 0.0, 0.5387260318497957),\n                        (13, 0.5, 0.0, -0.6033472105469482),\n                        (12, 0.5000000000000003, 1.1102230246251565e-16, 0.25908110342234625),\n                        (8, 0.5000000000000003, -0.9999999999999996, -0.7398330070144274)],\n                       [(9, 10), (13, 14)],\n                       [(13, 12),\n                        (9, 8),\n                        (10, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (14, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(8, 0.9992548329781763, -0.4863814805575142, 0.7601669929855652),\n                        (9, 3.251517832758566e-08, -0.26494051942175356, -0.46127399081569265),\n                        (12, 0.0007451670218237004, 0.5147044090056969, 0.7590811034223541),\n                        (13, 3.251517832758566e-08, -0.12286727702500921, -0.603347233212437)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8),\n                      (6, 5),\n                      (6, 6),\n                      (6, 7),\n                      (6, 8)]}}\n\ndef _round9_unitary(n,k):\n    data = _ROUND9_VERIFIED.get((n,k))\n    if data is None: return None\n    q = [cirq.GridQubit(*v) for v in data[\'qubits\']]\n    moments=[]\n    for layer in data[\'moments\']:\n        ops=[]\n        for gate in layer:\n            if len(gate)==2: ops.append(cirq.CZ(q[gate[0]],q[gate[1]]))\n            else:\n                i,x,z,a=gate\n                ops.append(cirq.PhasedXZGate(x_exponent=x,z_exponent=z,axis_phase_exponent=a).on(q[i]))\n        moments.append(cirq.Moment(ops))\n    return cirq.Circuit(moments),{q[i] for i in data[\'outputs\']}\n# ROUND 9 VERIFIED NATIVE CIRCUITS END\n\n# ROUND 10 VERIFIED NATIVE CIRCUITS START\n_ROUND10_VERIFIED = {(11, 5): {\'method\': \'transferred short count seed + parallel coherent decoders\',\n           \'moments\': [[(1, 0.019053050798816838, -0.0513562762694062, 0.3251567256036215),\n                        (2, 0.5135085657647106, -0.41914370100178844, -0.6838343714621595),\n                        (3, 0.27456511347169266, 0.8544594926503115, 0.9901070313805884),\n                        (8, 0.770879073173055, 0.06313080969914164, 0.61408391458213),\n                        (7, 0.6007821695036887, 0.6674093305266244, -0.8672263848998099),\n                        (4, 0.5000000000000004, 0.6603321162200121, -0.019548280462959422),\n                        (6, 0.5199696142890611, 0.966374362234198, -0.8896751604098314),\n                        (5, 0.5, 0.0, 0.5),\n                        (9, 0.4127792040228902, 0.905374889856748, -0.9262332254895422),\n                        (10, 0.5, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5)],\n                       [(1, 2), (3, 8)],\n                       [(2, 0.20926000199663888, -0.5305760209550308, -0.5854282673619697),\n                        (3, 0.4610661296978037, 0.7938327505250322, 0.40159366099829596),\n                        (8, 0.538092094217944, 0.8988682761575443, 0.7270875650575477),\n                        (1, 0.4992167453476743, 0.12307307595125527, -0.6452197531912627)],\n                       [(2, 3), (7, 8)],\n                       [(2, 0.5, 0.5779614089581346, 0.36183841126334393),\n                        (7, 0.8216946454418361, -0.9668065243026382, -0.06434250046797407),\n                        (8, 0.8436143405757461, -0.94482710626367, 0.2446414171816892),\n                        (3, 1.0533785535973878e-16, 0.5740876391056153, -0.8606464345330231)],\n                       [(1, 2), (7, 8), (4, 3)],\n                       [(2, 0.5, -0.3723217361245814, 0.9397998202214768),\n                        (3, 2.3289058342757918e-17, 0.5862961341457424, -0.5616249750902196),\n                        (7, 0.8728974913945616, 0.19727173285961397, 0.26390571368523463),\n                        (8, 0.38134300015623357, -0.3400763658197792, -0.00027569473025712554),\n                        (4, 0.5000000000000001, 0.5533550240412688, -0.3592161642429481),\n                        (1, 0.6407382389690576, 0.7458041147779152, -0.02235035455601575)],\n                       [(2, 3), (7, 8)],\n                       [(2, 0.01280233017413947, 0.4135860020062133, 0.4174375533742921),\n                        (7, 0.5450437072158779, 0.5893085453695355, -0.2633887526230856),\n                        (3, 0.5, 0.6271025193347309, 0.1836119986958712),\n                        (8, 0.2564807509088814, 0.20112807499078678, -0.3598189731677146)],\n                       [(2, 7), (4, 3)],\n                       [(2, 0.5000000000000001, 0.10089828273510282, 0.9659243946074898),\n                        (3, 6.683300056343812e-17, 0.8871965634002833, 0.13806696658163087),\n                        (7, 0.5000000000000001, 0.9966659465040273, -0.585609114409642),\n                        (4, 0.49999999999999994, -0.10452047447486512, 0.23205688102146632)],\n                       [(2, 3), (7, 8)],\n                       [(2, 0.5000000000000001, 0.98618545807613, -0.6680850790244617),\n                        (7, 0.468031688821072, 0.20107700972863718, -0.7101995438610371),\n                        (3, 0.2632147621115001, 0.7892691926842185, -0.8020889185691145),\n                        (8, 0.22197270683743042, -0.3193789389887365, -0.7543448619630173)],\n                       [(1, 2), (7, 6), (4, 3), (8, 9)],\n                       [(2, 1.0, 0.0, 0.15136907821349643),\n                        (7, 0.5000000000000001, 0.6584749008440663, 0.4127955417649979),\n                        (6, 0.8325870184220645, -0.5200591664882959, 0.0766992018243659),\n                        (1, 0.5, -0.030723330248449088, -0.7765462397781144),\n                        (3, 1.0, 0.0, 0.21461996803249472),\n                        (8, 0.5, 0.7022844924594797, 0.4262761990482463),\n                        (9, 0.85928593962011, -0.4875608597182275, -0.02085833563279449),\n                        (4, 0.5000000000000001, 0.12925995895837872, -0.8724635934533991)],\n                       [(2, 7), (6, 1), (3, 8), (9, 4)],\n                       [(1, 1.0, 0.0, -0.35237069449328406),\n                        (2, 0.5000000000000001, -0.08183504399310726, -0.25375757867618387),\n                        (7, 0.633295750078832, 0.8024623256494829, -0.7457603550629945),\n                        (6, 0.747846310784871, -0.17134479144473458, 0.05664003533606926),\n                        (4, 1.0, 0.0, -0.28654568860164664),\n                        (3, 0.5000000000000001, -0.07216288375695012, -0.25958902417306295),\n                        (8, 0.6714165234362537, 0.7430251514267326, -0.8000232167612813),\n                        (9, 0.7485956033393342, -0.10058310030841655, -0.008419195351022202)],\n                       [(1, 2), (7, 6), (4, 3), (8, 9)],\n                       [(2, 0.5, -0.23681199740208614, 0.6644073773307086),\n                        (7, 0.24200275221440573, 0.12003146804240211, 0.21896207919510252),\n                        (6, 0.9580541200199847, -0.5877475103440539, -0.6147047561086649),\n                        (1, 0.036112653164580656, -0.8149062791244521, -0.06549642494681374),\n                        (3, 0.5000000000000001, -0.47921561052476913, 0.6682480920699869),\n                        (8, 0.39352947204382344, -0.0400166187933757, 0.22204905027907273),\n                        (9, 0.996157271886578, -0.38731562230521455, -0.6090022956594439),\n                        (4, 0.05280605500441269, -0.7140339826902271, -0.0034880864952426816)],\n                       [(2, 7), (6, 1), (3, 8), (9, 4)],\n                       [(1, 0.29557549942386757, -0.8771709074392591, 0.5336267181200474),\n                        (2, 1.0, 0.0, 0.4362658366327108),\n                        (7, 0.26083406388318797, 0.6177144938526578, 0.11150272847709536),\n                        (6, 0.49019895102894673, -0.8569874916299094, -0.7024522664527184),\n                        (4, 0.24963665794604506, 0.8669932073346845, 0.7651153459047553),\n                        (3, 1.0, 0.0, 0.21763245964208),\n                        (8, 0.3104369525321425, 0.8085371278988395, -0.23648881891450202),\n                        (9, 0.5655532211464792, -0.7603717865514881, -0.49631791796466374)],\n                       [(1, 2), (7, 6), (4, 3), (8, 9)],\n                       [(2, 2.2593636363592025e-17, -0.4110229675034638, -0.5585523672452701),\n                        (7, 0.4922598869063585, -0.44814055334937275, -0.4444150370829989),\n                        (6, 0.49999999999999994, 0.9967146647382108, 0.44056024191737214),\n                        (1, 0.4313085072649695, 0.7499183762094207, -0.8578728931258981),\n                        (3, 1.3678757607663618e-17, 0.19456334695121486, 0.24274381274574858),\n                        (8, 0.6193705712509703, -0.4502913348689348, -0.5549860019150674),\n                        (9, 0.49999999999999994, 0.914005305656897, 0.7433102954838477),\n                        (4, 0.32701156232958095, 0.6955438578648168, -0.9756718054724585)],\n                       [(2, 7), (6, 1), (3, 8), (9, 4)],\n                       [(2, 1.0, 0.0, 0.22752939310706877),\n                        (7, 0.6755935676538105, -0.9282460574587303, -0.5056399138381202),\n                        (6, 0.6338347537865279, 0.3223583002431071, -0.09800723979530845),\n                        (1, 0.5000000000000001, 0.40813944694967486, 0.6174374285377566),\n                        (3, 1.0, 0.0, 0.3991857392005793),\n                        (8, 0.6256105257587085, 0.9452263271866326, -0.5220069687743731),\n                        (9, 0.5960614607313279, 0.2573629292943688, -0.033294760319355854),\n                        (4, 0.5000000000000001, 0.3922892154072385, 0.472507520913482)],\n                       [(2, 7), (6, 1), (3, 8), (9, 4)],\n                       [(1, 0.4999999999999998, -0.3029539307227471, 0.5255768754874317),\n                        (2, 0.5, -0.06987507273615945, 0.10006733994442629),\n                        (7, 0.8391622995173471, -0.4666232621508506, 0.12265856248455714),\n                        (6, 0.10478091261206611, 0.6069675174235609, 0.5600579969436181),\n                        (4, 0.5, -0.2739726829496021, 0.3647967363207205),\n                        (3, 0.5000000000000001, -0.2267887021096202, 0.036504264403519004),\n                        (8, 0.9764747842833552, 0.005682454151557326, 0.021003600491651175),\n                        (9, 0.14949404734505523, 0.6915804042354776, 0.6139788678075409)],\n                       [(1, 2), (7, 6), (4, 3), (8, 9)],\n                       [(2, 0.5, 0.44888461582760275, -0.9698077327917332),\n                        (7, 0.5000000000000001, -0.8342441162534189, -0.0848749207251096),\n                        (6, 1.0, 0.0, -0.30598384290578995),\n                        (1, 1.0, 0.0, -0.13870845888133387),\n                        (3, 0.5, -0.9528264593460201, 0.8097155622938987),\n                        (8, 0.5, 0.47939827321355255, 0.2696643378250829),\n                        (9, 1.0, 0.0, -0.31991326341917814),\n                        (4, 1.0, 0.0, -0.27721006339896803)],\n                       [(2, 7), (6, 1), (3, 8), (9, 4)],\n                       [(1, 5.1730648837162365e-18, -0.615009442582864, 0.5150410050501057),\n                        (2, 0.5, 0.9359542790463431, -0.4406009729548801),\n                        (7, 0.1802207481101961, 0.3860031568845424, 0.0808809630214713),\n                        (6, 0.411561103136257, -0.027640655655161606, -0.5340368657507382),\n                        (4, 2.6911488363979655e-17, -0.10657650605920965, 0.3199678821356995),\n                        (3, 0.5000000000000001, 0.6065964482612792, -0.07592204303348415),\n                        (8, 0.14731871270290642, 0.47763908808934363, -0.2509373889613644),\n                        (9, 0.42477613785575913, 0.24565143911589105, -0.6499368354199673)],\n                       [(1, 2), (7, 6), (4, 3), (8, 9)],\n                       [(2, 1.946846003666787e-17, -0.23527072633657475, 0.8386739670881893),\n                        (7, 0.9611848154635414, 0.5482778688470907, 0.9668841199060139),\n                        (6, 0.4752070857908569, -0.8326590813546366, -0.2511029195814165),\n                        (1, 0.4294376327554545, -0.5263063967663268, -0.3336943135481585),\n                        (3, 2.183989817369057e-17, 0.04089633285515237, 0.7520884815410067),\n                        (8, 0.9433458052208397, -0.437164462020319, -0.27329830087202156),\n                        (9, 0.4500869528994625, 0.5858004296037851, -0.09013056170924827),\n                        (4, 0.42020075164346243, -0.13244587453752787, -0.04525832781976602)],\n                       [(2, 7), (6, 1), (3, 8), (9, 4)],\n                       [(6, 0.5410964618559009, 0.6051656233889696, 0.3948343766110304),\n                        (7, 9.304731065784778e-17, -0.5357102196810557, 0.9333451783572855),\n                        (1, 0.5, 0.4456173037517619, 0.5132862343923374),\n                        (2, 0.5205482309279504, -0.7600825797548882, -0.23991742024511176),\n                        (9, 0.5410889047360808, -0.9487783322704517, -0.05122166772954828),\n                        (8, 5.822447387112414e-17, 0.6899183105243011, -0.1202404731065354),\n                        (4, 0.5410889047360808, 0.2729000875180194, 0.7270999124819801),\n                        (3, 0.5000000000000002, -0.5921151904509876, 0.5715707380829471)],\n                       [(6, 5), (1, 2), (9, 10), (4, 0)],\n                       [(6, 0.49999999999999983, 0.0, -0.5),\n                        (5, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (1, 0.5, 1.0, -0.5),\n                        (9, 0.49999999999999983, 0.0, -0.5),\n                        (10, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (4, 0.49999999999999983, 0.0, -0.5),\n                        (0, 0.3040867239846963, 0.0, -0.5000000000000002)],\n                       [(7, 6), (2, 1), (8, 9), (3, 4)],\n                       [(6, 0.8040867239846964, 0.0, 0.5),\n                        (7, 1.0, 0.0, 0.5),\n                        (2, 0.25000000000000006, 0.0, 0.5),\n                        (1, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (9, 0.8040867239846964, 0.0, 0.5),\n                        (8, 1.0, 0.0, 0.5),\n                        (4, 0.8040867239846964, 0.0, 0.5),\n                        (3, 1.0, 0.0, 0.5)],\n                       [(6, 5), (2, 1), (9, 10), (4, 0)],\n                       [(6, 0.75, 0.0, 0.5),\n                        (5, 0.5, 0.0, 0.5),\n                        (9, 0.75, 0.0, 0.5),\n                        (10, 0.5, 0.0, 0.5),\n                        (4, 0.75, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (2, 0.5, 1.0, -0.5),\n                        (1, 0.5, 1.0, -0.5)],\n                       [(7, 6), (8, 9), (3, 4)],\n                       [(6, 0.5000000000000001, 0.0, -0.5),\n                        (7, 0.5, 0.0, 0.5),\n                        (9, 0.5000000000000001, 0.0, -0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (4, 0.5000000000000001, 0.0, -0.5),\n                        (3, 0.5, 0.0, 0.5)],\n                       [(6, 5), (9, 10), (4, 0)],\n                       [(6, 0.25000000000000006, 0.0, 0.5),\n                        (5, 1.0, 0.0, 0.5),\n                        (9, 0.25000000000000006, 0.0, 0.5),\n                        (10, 1.0, 0.0, 0.5),\n                        (4, 0.25000000000000006, 0.0, 0.5),\n                        (0, 1.0, 0.0, 0.5)],\n                       [(7, 6), (8, 9), (3, 4)],\n                       [(6, 0.5, 0.0, -0.5),\n                        (9, 0.5, 0.0, -0.5),\n                        (4, 0.5, 0.0, -0.5),\n                        (7, 0.5, 0.0, 0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (3, 0.5, 0.0, 0.5)],\n                       [(6, 5), (9, 10), (4, 0)],\n                       [(6, 0.5000000000000001, 0.0, 0.5),\n                        (9, 0.5000000000000001, 0.0, 0.5),\n                        (4, 0.5000000000000001, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],\n           \'qubits\': [(3, 7),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 3),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8)]},\n (12, 4): {\'method\': \'full-ladder count seed + joint decoder cores\',\n           \'moments\': [[(3, 0.9933648087803231, 0.9461982568675674, 0.8975899207683933),\n                        (4, 2.7841837543632823e-16, 0.6122009934036537, -0.8328465887086485),\n                        (8, 0.49529242920096594, 0.5589613626840377, -0.3287628524027575),\n                        (9, 0.4915790135309922, -0.578720530630787, 0.615563398863773),\n                        (5, 0.5000000000000067, 0.5002212061185203, -0.8794783908786035),\n                        (10, 0.5000000000000056, -0.7752506292037618, -0.1263015289687539),\n                        (2, 0.49999999999999994, 0.5787597999008574, 0.5937872305331664),\n                        (1, 0.5, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (11, 0.49999999999999994, 0.5787597999008574, 0.5937872305331664),\n                        (7, 0.5, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5)],\n                       [(3, 4), (8, 9), (5, 10)],\n                       [(3, 0.43370334220597884, -0.25282820444733023, 0.057596550469559304),\n                        (8, 0.6245135074010438, 0.7092387343138077, -0.3438705380512568),\n                        (4, 0.4999999999999998, 0.7315937567662979, -0.08578844968793264),\n                        (9, 0.3378859057788629, -0.5730810846973431, 0.024179210789726624),\n                        (5, 0.4480071007089909, -0.19227132536931757, -0.35267091359510516),\n                        (10, 0.9416579811003823, -0.4478655060429464, -0.050710561144686306)],\n                       [(3, 8), (4, 9)],\n                       [(4, 0.48236654496476655, 0.22066439457745857, -0.24409968294424278),\n                        (9, 0.11144213027334705, 0.7732834693107491, -0.0943345023037847),\n                        (3, 0.49999999999999944, 0.6302373878550374, 0.3090037144231131),\n                        (8, 0.5000000000000003, 0.008394657718208132, -0.3116538143597256)],\n                       [(4, 5), (9, 10), (3, 8)],\n                       [(4, 0.7500000000000004, -0.7897879800757033, 0.5250931499741711),\n                        (9, 0.7499999999999973, 0.17242912253105835, 0.6789489670069642),\n                        (5, 0.7500000000000011, -0.9401364948454591, 0.019579957197285724),\n                        (10, 0.2500000000000012, 0.24434478249948244, 0.7855659281658398),\n                        (3, 0.5000000000000001, 0.2680666363988804, 0.9392411022781519),\n                        (8, 0.4285153692282758, 0.5985672746232784, -0.803259156641517)],\n                       [(4, 9), (5, 10)],\n                       [(4, 0.12370339211414162, -0.49854534521875316, -0.7646948301015308),\n                        (5, 0.747406784228284, 0.6340485914268337, 0.5794434623518261),\n                        (9, 0.12370339211414173, 0.38538173514169216, 0.35137808953802285),\n                        (10, 0.252593215771716, -0.31641865688666315, 0.5299107106653225)],\n                       [(3, 4), (5, 2), (8, 9), (10, 11)],\n                       [(4, 1.0, 0.0, 0.1691930686148133),\n                        (5, 0.9416885504219519, -0.7966896157630077, 0.16269195136674125),\n                        (3, 3.9340371463309e-17, 0.6414807162476399, 0.2906690114531043),\n                        (2, 0.608535446953489, -0.05921722784353989, -0.35241608640843847),\n                        (9, 1.0, 0.0, 0.1691930686148133),\n                        (10, 0.9416885504219519, -0.7966896157630077, 0.16269195136674125),\n                        (8, 3.9340371463309e-17, 0.6414807162476399, 0.2906690114531043),\n                        (11, 0.608535446953489, -0.05921722784353989, -0.35241608640843847)],\n                       [(4, 5), (9, 10)],\n                       [(4, 0.0966640990980218, 0.5634347839619753, 0.553369110180242),\n                        (5, 0.5, -0.7038779488919584, 0.29327310318674915),\n                        (9, 0.0966640990980218, 0.5634347839619753, 0.553369110180242),\n                        (10, 0.5, -0.7038779488919584, 0.29327310318674915)],\n                       [(3, 4), (5, 2), (8, 9), (10, 11)],\n                       [(4, 0.5812745787265815, -0.5231014844999589, 0.9109018628163763),\n                        (5, 0.5150139304348057, 0.757857138986012, -0.41060484570520916),\n                        (3, 1.3964893704413632e-16, 0.9929370915236234, 0.9765964277471055),\n                        (2, 0.16723314080476154, -0.013430024769138349, -0.2179240568348444),\n                        (9, 0.5812745787265815, -0.5231014844999589, 0.9109018628163763),\n                        (10, 0.5150139304348057, 0.757857138986012, -0.41060484570520916),\n                        (8, 1.3964893704413632e-16, 0.9929370915236234, 0.9765964277471055),\n                        (11, 0.16723314080476154, -0.013430024769138349, -0.2179240568348444)],\n                       [(4, 5), (9, 10)],\n                       [(4, 0.2955394861408327, 0.3578853008257912, -0.6396057034677505),\n                        (5, 0.08858950606155516, 0.19518014609201906, 0.8472522932808026),\n                        (9, 0.2955394861408327, 0.3578853008257912, -0.6396057034677505),\n                        (10, 0.08858950606155516, 0.19518014609201906, 0.8472522932808026)],\n                       [(3, 4), (5, 2), (8, 9), (10, 11)],\n                       [(4, 0.29430117217170476, 0.30940417700783807, -0.25950736373460614),\n                        (5, 0.3017505280831282, 0.716557647237992, 0.5424324393728217),\n                        (3, 0.49999999999999944, -0.3206716363078259, 0.370101408886175),\n                        (2, 0.6303550437629812, 0.9417119920777328, 0.32452271192168486),\n                        (9, 0.29430117217170476, 0.30940417700783807, -0.25950736373460614),\n                        (10, 0.3017505280831282, 0.716557647237992, 0.5424324393728217),\n                        (8, 0.49999999999999944, -0.3206716363078259, 0.370101408886175),\n                        (11, 0.6303550437629812, 0.9417119920777328, 0.32452271192168486)],\n                       [(4, 5), (9, 10)],\n                       [(4, 0.30286229487152805, -0.017751310223818262, -0.9597085518666981),\n                        (5, 0.19954896443512302, 0.16657505632770442, -0.24100991338918654),\n                        (9, 0.30286229487152805, -0.017751310223818262, -0.9597085518666981),\n                        (10, 0.19954896443512302, 0.16657505632770442, -0.24100991338918654)],\n                       [(3, 4), (5, 2), (8, 9), (10, 11)],\n                       [(4, 0.5000000000000001, -0.7247105199628492, -0.20900918813951774),\n                        (5, 0.22177219480665553, 0.8267460770192709, 0.4255651429385177),\n                        (3, 0.33347826094116906, -0.9085789967480333, -0.4505702274216521),\n                        (2, 0.8315920243625678, -0.7670400114327012, -0.9679457484530092),\n                        (9, 0.5000000000000001, -0.7247105199628492, -0.20900918813951774),\n                        (10, 0.22177219480665553, 0.8267460770192709, 0.4255651429385177),\n                        (8, 0.33347826094116906, -0.9085789967480333, -0.4505702274216521),\n                        (11, 0.8315920243625678, -0.7670400114327012, -0.9679457484530092)],\n                       [(4, 5), (9, 10)],\n                       [(4, 0.5, -0.03872592450091705, -0.9337197081023676),\n                        (5, 0.5000000000000001, 0.36752926291371363, -0.26065130743975384),\n                        (9, 0.5, -0.03872592450091705, -0.9337197081023676),\n                        (10, 0.5000000000000001, 0.36752926291371363, -0.26065130743975384)],\n                       [(3, 4), (5, 2), (8, 9), (10, 11)],\n                       [(4, 0.5519231616233385, 0.8578787303175794, 0.24535068392834808),\n                        (5, 0.5000000000000001, 0.8314434388409561, 0.10687795547395951),\n                        (3, 0.5276550549544043, -0.08390205901888415, 0.6408507758303144),\n                        (2, 0.5, 0.4279854302193228, -0.8574301035875358),\n                        (9, 0.5519231616233385, 0.8578787303175794, 0.24535068392834808),\n                        (10, 0.5000000000000001, 0.8314434388409561, 0.10687795547395951),\n                        (8, 0.5276550549544043, -0.08390205901888415, 0.6408507758303144),\n                        (11, 0.5, 0.4279854302193228, -0.8574301035875358)],\n                       [(4, 5), (9, 10)],\n                       [(4, 0.5, -0.6086747371118502, 0.4690826603403329),\n                        (5, 0.5, -0.17246289212507904, -0.3611572678172341),\n                        (9, 0.5, -0.6086747371118502, 0.4690826603403329),\n                        (10, 0.5, -0.17246289212507904, -0.3611572678172341)],\n                       [(3, 4), (5, 2), (8, 9), (10, 11)],\n                       [(3, 0.5000000000000001, 0.9531778878973458, 0.05694871681142982),\n                        (4, 0.5, 0.4283637736626025, 0.8604079232284827),\n                        (5, 6.08618156565454e-17, 0.24248973224243925, -0.19143556071220313),\n                        (2, 0.5, 0.4069435061415536, -0.9294446733682129),\n                        (8, 0.5000000000000001, 0.9531778878973458, 0.05694871681142982),\n                        (9, 0.5, 0.4283637736626025, 0.8604079232284827),\n                        (10, 6.08618156565454e-17, 0.24248973224243925, -0.19143556071220313),\n                        (11, 0.5, 0.4069435061415536, -0.9294446733682129)],\n                       [(3, 4), (5, 2), (8, 9), (10, 11)],\n                       [(3, 0.4901053274689139, -0.009392987046213497, -0.989849863007912),\n                        (4, 0.5, 0.49383070831591425, -0.7112283031089153),\n                        (2, 0.5068340379253959, 0.7579810778060412, 0.9846548716558152),\n                        (5, 0.5, -0.9409596946499887, -0.7911304276998739),\n                        (8, 0.4901053274689139, -0.009392987046213497, -0.989849863007912),\n                        (9, 0.5, 0.49383070831591425, -0.7112283031089153),\n                        (11, 0.5068340379253959, 0.7579810778060412, 0.9846548716558152),\n                        (10, 0.5, -0.9409596946499887, -0.7911304276998739)],\n                       [(3, 1), (2, 0), (8, 7), (11, 6)],\n                       [(3, 0.49999999999999983, 0.0, -0.5),\n                        (1, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (2, 0.49999999999999983, 0.0, -0.5),\n                        (0, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (8, 0.49999999999999983, 0.0, -0.5),\n                        (7, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (11, 0.49999999999999983, 0.0, -0.5),\n                        (6, 0.3040867239846963, 0.0, -0.5000000000000002)],\n                       [(4, 3), (5, 2), (9, 8), (10, 11)],\n                       [(3, 0.8040867239846964, 0.0, 0.5),\n                        (4, 1.0, 0.0, 0.5),\n                        (2, 0.8040867239846964, 0.0, 0.5),\n                        (5, 1.0, 0.0, 0.5),\n                        (8, 0.8040867239846964, 0.0, 0.5),\n                        (9, 1.0, 0.0, 0.5),\n                        (11, 0.8040867239846964, 0.0, 0.5),\n                        (10, 1.0, 0.0, 0.5)],\n                       [(3, 1), (2, 0), (8, 7), (11, 6)],\n                       [(3, 0.75, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5),\n                        (2, 0.75, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (8, 0.75, 0.0, 0.5),\n                        (7, 0.5, 0.0, 0.5),\n                        (11, 0.75, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5)],\n                       [(4, 3), (5, 2), (9, 8), (10, 11)],\n                       [(3, 0.5000000000000001, 0.0, -0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (2, 0.5000000000000001, 0.0, -0.5),\n                        (5, 0.5, 0.0, 0.5),\n                        (8, 0.5000000000000001, 0.0, -0.5),\n                        (9, 0.5, 0.0, 0.5),\n                        (11, 0.5000000000000001, 0.0, -0.5),\n                        (10, 0.5, 0.0, 0.5)],\n                       [(3, 1), (2, 0), (8, 7), (11, 6)],\n                       [(3, 0.25000000000000006, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5),\n                        (2, 0.25000000000000006, 0.0, 0.5),\n                        (0, 1.0, 0.0, 0.5),\n                        (8, 0.25000000000000006, 0.0, 0.5),\n                        (7, 1.0, 0.0, 0.5),\n                        (11, 0.25000000000000006, 0.0, 0.5),\n                        (6, 1.0, 0.0, 0.5)],\n                       [(4, 3), (5, 2), (9, 8), (10, 11)],\n                       [(3, 0.5, 0.0, -0.5),\n                        (2, 0.5, 0.0, -0.5),\n                        (8, 0.5, 0.0, -0.5),\n                        (11, 0.5, 0.0, -0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (5, 0.5, 0.0, 0.5),\n                        (9, 0.5, 0.0, 0.5),\n                        (10, 0.5, 0.0, 0.5)],\n                       [(3, 1), (2, 0), (8, 7), (11, 6)],\n                       [(3, 0.5000000000000001, 0.0, 0.5),\n                        (2, 0.5000000000000001, 0.0, 0.5),\n                        (8, 0.5000000000000001, 0.0, 0.5),\n                        (11, 0.5000000000000001, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],\n           \'qubits\': [(2, 6),\n                      (3, 4),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 3),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7)]},\n (12, 5): {\'method\': \'transferred short count seed + parallel coherent decoders\',\n           \'moments\': [[(2, 0.019044757770101953, -0.051356184361657675, 0.3251617103003935),\n                        (3, 0.5135300564410764, -0.4191546188854791, -0.6838354675521687),\n                        (4, 0.27265272059682893, 0.8569212596882951, 0.9901137776625968),\n                        (9, 0.7785768126708875, 0.06827987460334928, 0.6140883725838775),\n                        (8, 0.6052333268801525, 0.698015306169861, -0.8703861916436315),\n                        (5, 0.5000000000000019, 0.6603408230111939, -0.0195436365722097),\n                        (7, 0.4127792040228902, 0.905374889856748, -0.9262332254895422),\n                        (6, 0.5, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (10, 0.4127792040228902, 0.905374889856748, -0.9262332254895422),\n                        (11, 0.5, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5)],\n                       [(2, 3), (4, 9)],\n                       [(3, 0.20924191377148105, -0.5305841152987563, -0.5854102596087551),\n                        (4, 0.45440571358656495, 0.7961786336475795, 0.39911182635923004),\n                        (9, 0.542924724859561, 0.8983200421595123, 0.7219363959502656),\n                        (2, 0.4992158391199147, 0.12318600314285177, -0.645219648869686)],\n                       [(3, 4), (8, 9)],\n                       [(3, 0.49999999999999983, 0.5779898179552568, 0.3618114372718674),\n                        (8, 0.75233673194592, -0.94283869574036, -0.07362533867048748),\n                        (9, 0.8478591636701596, -0.9148004373080596, 0.2503227328072589),\n                        (4, 1.0626967249141569e-16, 0.5738375504667869, -0.8604962218001142)],\n                       [(2, 3), (8, 9), (5, 4)],\n                       [(3, 0.5000000000000001, -0.3723355038602789, 0.9398012552271247),\n                        (4, 4.7054410845933996e-17, 0.5860012280658864, -0.5615299157158871),\n                        (8, 0.873419788243788, 0.1905851500276533, 0.24924281983333813),\n                        (9, 0.395271066077429, -0.3119593674882015, -0.03597642395189249),\n                        (5, 0.4999999999999997, 0.5530640250745447, -0.35920281356101613),\n                        (2, 0.6402039461142681, 0.6497739521666086, -0.022237545619990584)],\n                       [(3, 4), (8, 9)],\n                       [(3, 0.012797835765225385, 0.4135876198008517, 0.41744091564958696),\n                        (8, 0.5566034695127617, 0.5634057830159116, -0.24203883049936015),\n                        (4, 0.5, 0.6268987443869024, 0.18380075616413716),\n                        (9, 0.3107096622470824, 0.20661973409662104, -0.3674475881572792)],\n                       [(3, 8), (5, 4)],\n                       [(3, 0.5000000000000001, 0.09630310513057871, 0.9659212382827824),\n                        (4, 2.109701558630658e-15, 0.8872130789231938, -0.8619539406718177),\n                        (8, 0.5000000000000001, 0.9921095098390575, -0.5810443023216156),\n                        (5, 0.5000000000000002, -0.10480712606901754, 0.23234353261561852)],\n                       [(3, 4), (8, 9)],\n                       [(3, 0.5000000000000001, -0.8538140735416668, -0.659005652343229),\n                        (8, 0.4438276130925125, 0.29939793599862163, -0.804844242458399),\n                        (4, 0.258648674616029, 0.7892676946408771, -0.8020874205257655),\n                        (9, 0.22197270683743042, -0.33632612978749776, -0.737397671164256)],\n                       [(2, 3), (8, 7), (5, 4), (9, 10)],\n                       [(3, 1.0, 0.0, 0.21461996803249472),\n                        (8, 0.5, 0.7022844924594797, 0.4262761990482463),\n                        (7, 0.85928593962011, -0.4875608597182275, -0.02085833563279449),\n                        (2, 0.5000000000000001, 0.12925995895837872, -0.8724635934533991),\n                        (4, 1.0, 0.0, 0.21461996803249472),\n                        (9, 0.5, 0.7022844924594797, 0.4262761990482463),\n                        (10, 0.85928593962011, -0.4875608597182275, -0.02085833563279449),\n                        (5, 0.5000000000000001, 0.12925995895837872, -0.8724635934533991)],\n                       [(3, 8), (7, 2), (4, 9), (10, 5)],\n                       [(2, 1.0, 0.0, -0.28654568860164664),\n                        (3, 0.5000000000000001, -0.07216288375695012, -0.25958902417306295),\n                        (8, 0.6714165234362537, 0.7430251514267326, -0.8000232167612813),\n                        (7, 0.7485956033393342, -0.10058310030841655, -0.008419195351022202),\n                        (5, 1.0, 0.0, -0.28654568860164664),\n                        (4, 0.5000000000000001, -0.07216288375695012, -0.25958902417306295),\n                        (9, 0.6714165234362537, 0.7430251514267326, -0.8000232167612813),\n                        (10, 0.7485956033393342, -0.10058310030841655, -0.008419195351022202)],\n                       [(2, 3), (8, 7), (5, 4), (9, 10)],\n                       [(3, 0.5000000000000001, -0.47921561052476913, 0.6682480920699869),\n                        (8, 0.39352947204382344, -0.0400166187933757, 0.22204905027907273),\n                        (7, 0.996157271886578, -0.38731562230521455, -0.6090022956594439),\n                        (2, 0.05280605500441269, -0.7140339826902271, -0.0034880864952426816),\n                        (4, 0.5000000000000001, -0.47921561052476913, 0.6682480920699869),\n                        (9, 0.39352947204382344, -0.0400166187933757, 0.22204905027907273),\n                        (10, 0.996157271886578, -0.38731562230521455, -0.6090022956594439),\n                        (5, 0.05280605500441269, -0.7140339826902271, -0.0034880864952426816)],\n                       [(3, 8), (7, 2), (4, 9), (10, 5)],\n                       [(2, 0.24963665794604506, 0.8669932073346845, 0.7651153459047553),\n                        (3, 1.0, 0.0, 0.21763245964208),\n                        (8, 0.3104369525321425, 0.8085371278988395, -0.23648881891450202),\n                        (7, 0.5655532211464792, -0.7603717865514881, -0.49631791796466374),\n                        (5, 0.24963665794604506, 0.8669932073346845, 0.7651153459047553),\n                        (4, 1.0, 0.0, 0.21763245964208),\n                        (9, 0.3104369525321425, 0.8085371278988395, -0.23648881891450202),\n                        (10, 0.5655532211464792, -0.7603717865514881, -0.49631791796466374)],\n                       [(2, 3), (8, 7), (5, 4), (9, 10)],\n                       [(3, 1.3678757607663618e-17, 0.19456334695121486, 0.24274381274574858),\n                        (8, 0.6193705712509703, -0.4502913348689348, -0.5549860019150674),\n                        (7, 0.49999999999999994, 0.914005305656897, 0.7433102954838477),\n                        (2, 0.32701156232958095, 0.6955438578648168, -0.9756718054724585),\n                        (4, 1.3678757607663618e-17, 0.19456334695121486, 0.24274381274574858),\n                        (9, 0.6193705712509703, -0.4502913348689348, -0.5549860019150674),\n                        (10, 0.49999999999999994, 0.914005305656897, 0.7433102954838477),\n                        (5, 0.32701156232958095, 0.6955438578648168, -0.9756718054724585)],\n                       [(3, 8), (7, 2), (4, 9), (10, 5)],\n                       [(3, 1.0, 0.0, 0.3991857392005793),\n                        (8, 0.6256105257587085, 0.9452263271866326, -0.5220069687743731),\n                        (7, 0.5960614607313279, 0.2573629292943688, -0.033294760319355854),\n                        (2, 0.5000000000000001, 0.3922892154072385, 0.472507520913482),\n                        (4, 1.0, 0.0, 0.3991857392005793),\n                        (9, 0.6256105257587085, 0.9452263271866326, -0.5220069687743731),\n                        (10, 0.5960614607313279, 0.2573629292943688, -0.033294760319355854),\n                        (5, 0.5000000000000001, 0.3922892154072385, 0.472507520913482)],\n                       [(3, 8), (7, 2), (4, 9), (10, 5)],\n                       [(2, 0.5, -0.2739726829496021, 0.3647967363207205),\n                        (3, 0.5000000000000001, -0.2267887021096202, 0.036504264403519004),\n                        (8, 0.9764747842833552, 0.005682454151557326, 0.021003600491651175),\n                        (7, 0.14949404734505523, 0.6915804042354776, 0.6139788678075409),\n                        (5, 0.5, -0.2739726829496021, 0.3647967363207205),\n                        (4, 0.5000000000000001, -0.2267887021096202, 0.036504264403519004),\n                        (9, 0.9764747842833552, 0.005682454151557326, 0.021003600491651175),\n                        (10, 0.14949404734505523, 0.6915804042354776, 0.6139788678075409)],\n                       [(2, 3), (8, 7), (5, 4), (9, 10)],\n                       [(3, 0.5, -0.9528264593460201, 0.8097155622938987),\n                        (8, 0.5, 0.47939827321355255, 0.2696643378250829),\n                        (7, 1.0, 0.0, -0.31991326341917814),\n                        (2, 1.0, 0.0, -0.27721006339896803),\n                        (4, 0.5, -0.9528264593460201, 0.8097155622938987),\n                        (9, 0.5, 0.47939827321355255, 0.2696643378250829),\n                        (10, 1.0, 0.0, -0.31991326341917814),\n                        (5, 1.0, 0.0, -0.27721006339896803)],\n                       [(3, 8), (7, 2), (4, 9), (10, 5)],\n                       [(2, 2.6911488363979655e-17, -0.10657650605920965, 0.3199678821356995),\n                        (3, 0.5000000000000001, 0.6065964482612792, -0.07592204303348415),\n                        (8, 0.14731871270290642, 0.47763908808934363, -0.2509373889613644),\n                        (7, 0.42477613785575913, 0.24565143911589105, -0.6499368354199673),\n                        (5, 2.6911488363979655e-17, -0.10657650605920965, 0.3199678821356995),\n                        (4, 0.5000000000000001, 0.6065964482612792, -0.07592204303348415),\n                        (9, 0.14731871270290642, 0.47763908808934363, -0.2509373889613644),\n                        (10, 0.42477613785575913, 0.24565143911589105, -0.6499368354199673)],\n                       [(2, 3), (8, 7), (5, 4), (9, 10)],\n                       [(3, 2.183989817369057e-17, 0.04089633285515237, 0.7520884815410067),\n                        (8, 0.9433458052208397, -0.437164462020319, -0.27329830087202156),\n                        (7, 0.4500869528994625, 0.5858004296037851, -0.09013056170924827),\n                        (2, 0.42020075164346243, -0.13244587453752787, -0.04525832781976602),\n                        (4, 2.183989817369057e-17, 0.04089633285515237, 0.7520884815410067),\n                        (9, 0.9433458052208397, -0.437164462020319, -0.27329830087202156),\n                        (10, 0.4500869528994625, 0.5858004296037851, -0.09013056170924827),\n                        (5, 0.42020075164346243, -0.13244587453752787, -0.04525832781976602)],\n                       [(3, 8), (7, 2), (4, 9), (10, 5)],\n                       [(7, 0.5410889047360808, -0.9487783322704517, -0.05122166772954828),\n                        (8, 5.822447387112414e-17, 0.6899183105243011, -0.1202404731065354),\n                        (2, 0.5410889047360808, 0.2729000875180194, 0.7270999124819801),\n                        (3, 0.5000000000000002, -0.5921151904509876, 0.5715707380829471),\n                        (10, 0.5410889047360808, -0.9487783322704517, -0.05122166772954828),\n                        (9, 5.822447387112414e-17, 0.6899183105243011, -0.1202404731065354),\n                        (5, 0.5410889047360808, 0.2729000875180194, 0.7270999124819801),\n                        (4, 0.5000000000000002, -0.5921151904509876, 0.5715707380829471)],\n                       [(7, 6), (2, 0), (10, 11), (5, 1)],\n                       [(7, 0.49999999999999983, 0.0, -0.5),\n                        (6, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (2, 0.49999999999999983, 0.0, -0.5),\n                        (0, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (10, 0.49999999999999983, 0.0, -0.5),\n                        (11, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (5, 0.49999999999999983, 0.0, -0.5),\n                        (1, 0.3040867239846963, 0.0, -0.5000000000000002)],\n                       [(8, 7), (3, 2), (9, 10), (4, 5)],\n                       [(7, 0.8040867239846964, 0.0, 0.5),\n                        (8, 1.0, 0.0, 0.5),\n                        (2, 0.8040867239846964, 0.0, 0.5),\n                        (3, 1.0, 0.0, 0.5),\n                        (10, 0.8040867239846964, 0.0, 0.5),\n                        (9, 1.0, 0.0, 0.5),\n                        (5, 0.8040867239846964, 0.0, 0.5),\n                        (4, 1.0, 0.0, 0.5)],\n                       [(7, 6), (2, 0), (10, 11), (5, 1)],\n                       [(7, 0.75, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (2, 0.75, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (10, 0.75, 0.0, 0.5),\n                        (11, 0.5, 0.0, 0.5),\n                        (5, 0.75, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5)],\n                       [(8, 7), (3, 2), (9, 10), (4, 5)],\n                       [(7, 0.5000000000000001, 0.0, -0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (2, 0.5000000000000001, 0.0, -0.5),\n                        (3, 0.5, 0.0, 0.5),\n                        (10, 0.5000000000000001, 0.0, -0.5),\n                        (9, 0.5, 0.0, 0.5),\n                        (5, 0.5000000000000001, 0.0, -0.5),\n                        (4, 0.5, 0.0, 0.5)],\n                       [(7, 6), (2, 0), (10, 11), (5, 1)],\n                       [(7, 0.25000000000000006, 0.0, 0.5),\n                        (6, 1.0, 0.0, 0.5),\n                        (2, 0.25000000000000006, 0.0, 0.5),\n                        (0, 1.0, 0.0, 0.5),\n                        (10, 0.25000000000000006, 0.0, 0.5),\n                        (11, 1.0, 0.0, 0.5),\n                        (5, 0.25000000000000006, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5)],\n                       [(8, 7), (3, 2), (9, 10), (4, 5)],\n                       [(7, 0.5, 0.0, -0.5),\n                        (2, 0.5, 0.0, -0.5),\n                        (10, 0.5, 0.0, -0.5),\n                        (5, 0.5, 0.0, -0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (3, 0.5, 0.0, 0.5),\n                        (9, 0.5, 0.0, 0.5),\n                        (4, 0.5, 0.0, 0.5)],\n                       [(7, 6), (2, 0), (10, 11), (5, 1)],\n                       [(7, 0.5000000000000001, 0.0, 0.5),\n                        (2, 0.5000000000000001, 0.0, 0.5),\n                        (10, 0.5000000000000001, 0.0, 0.5),\n                        (5, 0.5000000000000001, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],\n           \'qubits\': [(3, 4),\n                      (3, 7),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 3),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8)]},\n (12, 6): {\'method\': \'transferred short count seed + parallel coherent decoders\',\n           \'moments\': [[(2, 0.4999999999999875, 0.0608830126659885, -0.5000046339505302),\n                        (3, 0.4999999999999948, 0.0029504280164456276, -0.49998770718291174),\n                        (5, 1.0, 0.0, 0.49784649423013483),\n                        (4, 0.6055568218179003, -0.01036599982073394, -0.5000133980254566),\n                        (8, 0.48919898476516693, 0.0379995859464064, -0.5000322203013847),\n                        (9, 0.8141464902330346, 0.14450708760430497, -0.49994612846401587),\n                        (7, 0.4127792040228902, 0.905374889856748, -0.9262332254895422),\n                        (6, 0.5, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (10, 0.4127792040228902, 0.905374889856748, -0.9262332254895422),\n                        (11, 0.5, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5)],\n                       [(2, 3), (5, 4), (8, 9)],\n                       [(3, 0.3764097293463915, 0.07029015634149782, 0.4969734711424112),\n                        (8, 0.5473484901822152, -0.12252863106037903, -0.5379782354903577),\n                        (4, 0.49189864787915555, -0.013398241587176063, -0.4896279421194034),\n                        (9, 0.29591814236896186, 0.11468018258378426, 0.35535719960561496),\n                        (2, 0.61741343591263, 0.09398289390440234, 0.43910625175538215),\n                        (5, 0.49999999999999994, -0.004192850931005854, 0.5043963362191022)],\n                       [(3, 8), (4, 9)],\n                       [(3, 0.5, 0.06727345447739008, -0.5672484293602065),\n                        (4, 0.43530490587970394, -0.003995007209391943, -0.4970357867118156),\n                        (8, 0.7048730173606251, -0.16409712431833734, -0.33955524818512517),\n                        (9, 0.7513533539297025, -0.023519456074684975, 0.5299317756638773)],\n                       [(3, 4), (8, 9)],\n                       [(3, 0.4999999999999998, -0.12503877383883233, 0.5000250251171834),\n                        (4, 0.9999999999999999, -3.966953275424956e-05, -0.4990070639032398),\n                        (8, 0.2588147559466226, 0.0022783902687134594, -0.49367041480632246),\n                        (9, 0.3291964326030524, 0.023256532462736956, -0.4851512792519048)],\n                       [(2, 3), (5, 4), (8, 9)],\n                       [(3, 0.49999999999998646, -0.052184768374203605, -0.3750137487216485),\n                        (4, 0.5, 0.0009575672597872398, -0.5009621795266965),\n                        (8, 0.4717300643619974, -0.04842342918472209, -0.459941776404746),\n                        (9, 0.2216794887688361, -0.013678308280409723, 0.47409762049936865),\n                        (2, 0.2943797872669513, 0.6343947924313633, -0.006858385884761731),\n                        (5, 0.2961121785108624, 0.627332921258505, 0.00020348528809610578)],\n                       [(3, 4)],\n                       [(3, 4.3431655460055873e-17, 0.29452429563584326, -0.7303393581814506),\n                        (4, 0.5000000000000001, -0.008214339885839417, 0.4999953877330905)],\n                       [(3, 8), (4, 9)],\n                       [(3, 0.5000000000000003, -0.41012570545270366, 0.8973059795678078),\n                        (8, 0.813132379260486, -0.3045811682419053, -0.592248860736089),\n                        (4, 0.49999999999999994, 0.5253794272956771, 0.9618008468194275),\n                        (9, 0.3971819999211597, -0.5844824297420252, 0.6104727833183987)],\n                       [(2, 3), (8, 7), (5, 4), (9, 10)],\n                       [(3, 1.0, 0.0, 0.21461996803249472),\n                        (8, 0.5, 0.7022844924594797, 0.4262761990482463),\n                        (7, 0.85928593962011, -0.4875608597182275, -0.02085833563279449),\n                        (2, 0.5000000000000001, 0.12925995895837872, -0.8724635934533991),\n                        (4, 1.0, 0.0, 0.21461996803249472),\n                        (9, 0.5, 0.7022844924594797, 0.4262761990482463),\n                        (10, 0.85928593962011, -0.4875608597182275, -0.02085833563279449),\n                        (5, 0.5000000000000001, 0.12925995895837872, -0.8724635934533991)],\n                       [(3, 8), (7, 2), (4, 9), (10, 5)],\n                       [(2, 1.0, 0.0, -0.28654568860164664),\n                        (3, 0.5000000000000001, -0.07216288375695012, -0.25958902417306295),\n                        (8, 0.6714165234362537, 0.7430251514267326, -0.8000232167612813),\n                        (7, 0.7485956033393342, -0.10058310030841655, -0.008419195351022202),\n                        (5, 1.0, 0.0, -0.28654568860164664),\n                        (4, 0.5000000000000001, -0.07216288375695012, -0.25958902417306295),\n                        (9, 0.6714165234362537, 0.7430251514267326, -0.8000232167612813),\n                        (10, 0.7485956033393342, -0.10058310030841655, -0.008419195351022202)],\n                       [(2, 3), (8, 7), (5, 4), (9, 10)],\n                       [(3, 0.5000000000000001, -0.47921561052476913, 0.6682480920699869),\n                        (8, 0.39352947204382344, -0.0400166187933757, 0.22204905027907273),\n                        (7, 0.996157271886578, -0.38731562230521455, -0.6090022956594439),\n                        (2, 0.05280605500441269, -0.7140339826902271, -0.0034880864952426816),\n                        (4, 0.5000000000000001, -0.47921561052476913, 0.6682480920699869),\n                        (9, 0.39352947204382344, -0.0400166187933757, 0.22204905027907273),\n                        (10, 0.996157271886578, -0.38731562230521455, -0.6090022956594439),\n                        (5, 0.05280605500441269, -0.7140339826902271, -0.0034880864952426816)],\n                       [(3, 8), (7, 2), (4, 9), (10, 5)],\n                       [(2, 0.24963665794604506, 0.8669932073346845, 0.7651153459047553),\n                        (3, 1.0, 0.0, 0.21763245964208),\n                        (8, 0.3104369525321425, 0.8085371278988395, -0.23648881891450202),\n                        (7, 0.5655532211464792, -0.7603717865514881, -0.49631791796466374),\n                        (5, 0.24963665794604506, 0.8669932073346845, 0.7651153459047553),\n                        (4, 1.0, 0.0, 0.21763245964208),\n                        (9, 0.3104369525321425, 0.8085371278988395, -0.23648881891450202),\n                        (10, 0.5655532211464792, -0.7603717865514881, -0.49631791796466374)],\n                       [(2, 3), (8, 7), (5, 4), (9, 10)],\n                       [(3, 1.3678757607663618e-17, 0.19456334695121486, 0.24274381274574858),\n                        (8, 0.6193705712509703, -0.4502913348689348, -0.5549860019150674),\n                        (7, 0.49999999999999994, 0.914005305656897, 0.7433102954838477),\n                        (2, 0.32701156232958095, 0.6955438578648168, -0.9756718054724585),\n                        (4, 1.3678757607663618e-17, 0.19456334695121486, 0.24274381274574858),\n                        (9, 0.6193705712509703, -0.4502913348689348, -0.5549860019150674),\n                        (10, 0.49999999999999994, 0.914005305656897, 0.7433102954838477),\n                        (5, 0.32701156232958095, 0.6955438578648168, -0.9756718054724585)],\n                       [(3, 8), (7, 2), (4, 9), (10, 5)],\n                       [(3, 1.0, 0.0, 0.3991857392005793),\n                        (8, 0.6256105257587085, 0.9452263271866326, -0.5220069687743731),\n                        (7, 0.5960614607313279, 0.2573629292943688, -0.033294760319355854),\n                        (2, 0.5000000000000001, 0.3922892154072385, 0.472507520913482),\n                        (4, 1.0, 0.0, 0.3991857392005793),\n                        (9, 0.6256105257587085, 0.9452263271866326, -0.5220069687743731),\n                        (10, 0.5960614607313279, 0.2573629292943688, -0.033294760319355854),\n                        (5, 0.5000000000000001, 0.3922892154072385, 0.472507520913482)],\n                       [(3, 8), (7, 2), (4, 9), (10, 5)],\n                       [(2, 0.5, -0.2739726829496021, 0.3647967363207205),\n                        (3, 0.5000000000000001, -0.2267887021096202, 0.036504264403519004),\n                        (8, 0.9764747842833552, 0.005682454151557326, 0.021003600491651175),\n                        (7, 0.14949404734505523, 0.6915804042354776, 0.6139788678075409),\n                        (5, 0.5, -0.2739726829496021, 0.3647967363207205),\n                        (4, 0.5000000000000001, -0.2267887021096202, 0.036504264403519004),\n                        (9, 0.9764747842833552, 0.005682454151557326, 0.021003600491651175),\n                        (10, 0.14949404734505523, 0.6915804042354776, 0.6139788678075409)],\n                       [(2, 3), (8, 7), (5, 4), (9, 10)],\n                       [(3, 0.5, -0.9528264593460201, 0.8097155622938987),\n                        (8, 0.5, 0.47939827321355255, 0.2696643378250829),\n                        (7, 1.0, 0.0, -0.31991326341917814),\n                        (2, 1.0, 0.0, -0.27721006339896803),\n                        (4, 0.5, -0.9528264593460201, 0.8097155622938987),\n                        (9, 0.5, 0.47939827321355255, 0.2696643378250829),\n                        (10, 1.0, 0.0, -0.31991326341917814),\n                        (5, 1.0, 0.0, -0.27721006339896803)],\n                       [(3, 8), (7, 2), (4, 9), (10, 5)],\n                       [(2, 2.6911488363979655e-17, -0.10657650605920965, 0.3199678821356995),\n                        (3, 0.5000000000000001, 0.6065964482612792, -0.07592204303348415),\n                        (8, 0.14731871270290642, 0.47763908808934363, -0.2509373889613644),\n                        (7, 0.42477613785575913, 0.24565143911589105, -0.6499368354199673),\n                        (5, 2.6911488363979655e-17, -0.10657650605920965, 0.3199678821356995),\n                        (4, 0.5000000000000001, 0.6065964482612792, -0.07592204303348415),\n                        (9, 0.14731871270290642, 0.47763908808934363, -0.2509373889613644),\n                        (10, 0.42477613785575913, 0.24565143911589105, -0.6499368354199673)],\n                       [(2, 3), (8, 7), (5, 4), (9, 10)],\n                       [(3, 2.183989817369057e-17, 0.04089633285515237, 0.7520884815410067),\n                        (8, 0.9433458052208397, -0.437164462020319, -0.27329830087202156),\n                        (7, 0.4500869528994625, 0.5858004296037851, -0.09013056170924827),\n                        (2, 0.42020075164346243, -0.13244587453752787, -0.04525832781976602),\n                        (4, 2.183989817369057e-17, 0.04089633285515237, 0.7520884815410067),\n                        (9, 0.9433458052208397, -0.437164462020319, -0.27329830087202156),\n                        (10, 0.4500869528994625, 0.5858004296037851, -0.09013056170924827),\n                        (5, 0.42020075164346243, -0.13244587453752787, -0.04525832781976602)],\n                       [(3, 8), (7, 2), (4, 9), (10, 5)],\n                       [(7, 0.5410889047360808, -0.9487783322704517, -0.05122166772954828),\n                        (8, 5.822447387112414e-17, 0.6899183105243011, -0.1202404731065354),\n                        (2, 0.5410889047360808, 0.2729000875180194, 0.7270999124819801),\n                        (3, 0.5000000000000002, -0.5921151904509876, 0.5715707380829471),\n                        (10, 0.5410889047360808, -0.9487783322704517, -0.05122166772954828),\n                        (9, 5.822447387112414e-17, 0.6899183105243011, -0.1202404731065354),\n                        (5, 0.5410889047360808, 0.2729000875180194, 0.7270999124819801),\n                        (4, 0.5000000000000002, -0.5921151904509876, 0.5715707380829471)],\n                       [(7, 6), (2, 0), (10, 11), (5, 1)],\n                       [(7, 0.49999999999999983, 0.0, -0.5),\n                        (6, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (2, 0.49999999999999983, 0.0, -0.5),\n                        (0, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (10, 0.49999999999999983, 0.0, -0.5),\n                        (11, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (5, 0.49999999999999983, 0.0, -0.5),\n                        (1, 0.3040867239846963, 0.0, -0.5000000000000002)],\n                       [(8, 7), (3, 2), (9, 10), (4, 5)],\n                       [(7, 0.8040867239846964, 0.0, 0.5),\n                        (8, 1.0, 0.0, 0.5),\n                        (2, 0.8040867239846964, 0.0, 0.5),\n                        (3, 1.0, 0.0, 0.5),\n                        (10, 0.8040867239846964, 0.0, 0.5),\n                        (9, 1.0, 0.0, 0.5),\n                        (5, 0.8040867239846964, 0.0, 0.5),\n                        (4, 1.0, 0.0, 0.5)],\n                       [(7, 6), (2, 0), (10, 11), (5, 1)],\n                       [(7, 0.75, 0.0, 0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (2, 0.75, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (10, 0.75, 0.0, 0.5),\n                        (11, 0.5, 0.0, 0.5),\n                        (5, 0.75, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5)],\n                       [(8, 7), (3, 2), (9, 10), (4, 5)],\n                       [(7, 0.5000000000000001, 0.0, -0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (2, 0.5000000000000001, 0.0, -0.5),\n                        (3, 0.5, 0.0, 0.5),\n                        (10, 0.5000000000000001, 0.0, -0.5),\n                        (9, 0.5, 0.0, 0.5),\n                        (5, 0.5000000000000001, 0.0, -0.5),\n                        (4, 0.5, 0.0, 0.5)],\n                       [(7, 6), (2, 0), (10, 11), (5, 1)],\n                       [(7, 0.25000000000000006, 0.0, 0.5),\n                        (6, 1.0, 0.0, 0.5),\n                        (2, 0.25000000000000006, 0.0, 0.5),\n                        (0, 1.0, 0.0, 0.5),\n                        (10, 0.25000000000000006, 0.0, 0.5),\n                        (11, 1.0, 0.0, 0.5),\n                        (5, 0.25000000000000006, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5)],\n                       [(8, 7), (3, 2), (9, 10), (4, 5)],\n                       [(7, 0.5, 0.0, -0.5),\n                        (2, 0.5, 0.0, -0.5),\n                        (10, 0.5, 0.0, -0.5),\n                        (5, 0.5, 0.0, -0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (3, 0.5, 0.0, 0.5),\n                        (9, 0.5, 0.0, 0.5),\n                        (4, 0.5, 0.0, 0.5)],\n                       [(7, 6), (2, 0), (10, 11), (5, 1)],\n                       [(7, 0.5000000000000001, 0.0, 0.5),\n                        (2, 0.5000000000000001, 0.0, 0.5),\n                        (10, 0.5000000000000001, 0.0, 0.5),\n                        (5, 0.5000000000000001, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],\n           \'qubits\': [(3, 4),\n                      (3, 7),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 3),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8)]},\n (13, 4): {\'method\': \'full-ladder count seed + joint decoder cores\',\n           \'moments\': [[(4, 0.557811869395386, 0.0, 0.6821647699135944),\n                        (5, 0.4999999999999998, 0.0, -0.33433375954457456),\n                        (8, 0.4999999999999998, 0.0, -0.13349690514626866),\n                        (6, 0.6983944273829072, 0.0, -0.20306145252121333),\n                        (7, 0.19507456211685237, 0.0, -0.8816043374388429),\n                        (2, 0.4999999999999998, 0.0, 0.5937872305331664),\n                        (11, 0.22718552582850524, 0.0, -0.708982875827169),\n                        (10, 0.35760927658307967, 0.0, -0.9848107895924711),\n                        (0, 0.5, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5),\n                        (9, 0.3040867239846966, 0.0, -0.5745192213521528),\n                        (12, 0.3040867239846966, 0.0, -0.5745192213521528)],\n                       [(3, 4), (5, 8), (6, 7)],\n                       [(6, 0.8668403300628493, 0.0, -0.13651985034855918),\n                        (4, 0.8622833270652166, 0.0, -0.948007586857661),\n                        (7, 0.4149890398343652, 0.0, -0.14934172598271456),\n                        (5, 0.46919390730617927, 0.0, -0.09048185217903315),\n                        (8, 0.24539624746491118, 0.0, -0.5892127560771983)],\n                       [(3, 6), (4, 7)],\n                       [(4, 0.36302133982994794, 0.0, -0.8554906968751519),\n                        (7, 0.7088872865866075, 0.0, -0.22324871790719847),\n                        (6, 0.7619987654996385, 0.0, -0.45955920966719077)],\n                       [(4, 5), (7, 8)],\n                       [(4, 0.275461446092466, 8.881784197001252e-16, -0.990183657133825),\n                        (5, 0.31549494021723395, 0.0, -0.8539018581929556),\n                        (8, 0.8154949402172265, 0.0, -0.8844956611839363),\n                        (7, 0.4999999999999997, 0.0, -0.7232487179071991)],\n                       [(3, 4), (5, 8), (6, 7)],\n                       [(3, 0.5, 0.0, 0.6823759809265937),\n                        (4, 0.06140013208754747, 0.0, 0.5182466412505562),\n                        (7, 0.7056101508781536, 0.0, -0.8013812580662698),\n                        (5, 0.7486428139376007, 0.0, 0.6460981418070424),\n                        (8, 0.5, 4.440892098500626e-16, -0.8844956611839336)],\n                       [(3, 6), (4, 7), (5, 2), (8, 11)],\n                       [(3, 0.5, 0.0, 0.6823759809265937),\n                        (4, 0.6243214069688002, 0.0, -0.5413060358056618),\n                        (7, 0.4695858070438934, 0.0, -0.8227301483558893),\n                        (5, 0.9416885504219519, 0.0, -0.4047019606048754),\n                        (8, 0.4999999999999997, 0.0, 0.7999535291835189),\n                        (2, 0.608535446953489, 0.0, -0.9311758863092963),\n                        (11, 0.22718552582850488, 0.0, 0.29101712417283077)],\n                       [(3, 4), (6, 7)],\n                       [(4, 5), (7, 0.5000000000000002, 0.0, -0.7355313944136224)],\n                       [(7, 8),\n                        (4, 0.09666409909802184, 0.0, 0.5069511665640078),\n                        (5, 0.5000000000000001, -4.440892098500626e-16, -0.47743119302185955)],\n                       [(3, 4), (5, 2), (8, 11), (6, 7)],\n                       [(4, 0.5812745787265814, 0.0, 0.7128531978898494),\n                        (5, 0.5150139304348058, 0.0, -0.47743119302186),\n                        (7, 0.5886108001133223, 0.0, 0.2235952073434811),\n                        (8, 0.6801953832859778, 0.0, -0.255173112986381),\n                        (2, 0.16723314080476154, 0.0, -0.7374666288921621),\n                        (11, 0.6354920868801647, 0.0, 0.5482021483143629)],\n                       [(4, 5), (7, 8)],\n                       [(4, 0.2955394861408327, 0.0, -0.25974072032598294),\n                        (5, 0.08858950606155516, 0.0, -0.9774311930218602),\n                        (7, 0.5000000000000003, 0.0, 0.830101393311117),\n                        (8, 0.49999999999999906, 0.0, 0.4533052377528164)],\n                       [(3, 4), (5, 2), (6, 7), (8, 11)],\n                       [(4, 0.2943011721717048, 0.0, -0.28195375923333676),\n                        (5, 0.30175052808312813, 0.0, 0.5225688069781396),\n                        (7, 0.5234132252872342, 0.0, 0.978537076592068),\n                        (8, 0.06708148302124252, 0.0, 0.2266140503146572),\n                        (3, 0.49999999999999956, 0.0, -0.5533214710126013),\n                        (2, 0.6303550437629815, 2.220446049250313e-16, -0.18158983536649442),\n                        (6, 0.5, 0.0, -0.8041825318856919)],\n                       [(4, 5), (7, 8)],\n                       [(4, 0.30286229487152805, 4.440892098500626e-16, 0.7276516059065934),\n                        (5, 0.19954896443512293, 0.0, 0.022568806978139233),\n                        (7, 0.33272914584056146, 0.0, -0.06760970340127592),\n                        (8, 0.05920200383562026, 0.0, 0.7266140503146572)],\n                       [(3, 4), (5, 2), (7, 8)],\n                       [(4, 0.5, 0.0, -0.04079906804440503),\n                        (5, 0.22177219480665558, 0.0, 0.5225688069781396),\n                        (7, 0.25011854054823907, 0.0, -0.5676097034012753),\n                        (8, 0.08917758998520853, 0.0, 0.8505199946985289),\n                        (3, 0.33347826094116917, 0.0, -0.05332147101260265),\n                        (2, 0.8315920243625678, 3.3306690738754696e-16, -0.4157702878189209)],\n                       [(4, 5), (6, 7), (8, 11)],\n                       [(4, 0.5000000000000002, 0.0, -0.04079906804440481),\n                        (5, 0.5, 0.0, 0.009606279580597121),\n                        (8, 0.5938461416699012, 0.0, 0.6629216566383613),\n                        (11, 0.021107237782608024, 0.0, -0.9231467397292097),\n                        (6, 0.4999999999999976, 0.9174028832145481, 0.19581746811431178)],\n                       [(3, 4), (5, 2), (7, 8), (11, 10)],\n                       [(4, 0.5519231616233388, 0.0, 0.7414046154239624),\n                        (5, 0.5, 0.0, 0.009606279580596677),\n                        (8, 0.4433968217557545, 0.0, 0.1455404606287254),\n                        (11, 0.3147365041110265, 0.0, -0.7244934550958735),\n                        (10, 0.8402337422326178, 0.0, 0.9498164599690078),\n                        (3, 0.5276550549544043, 4.440892098500626e-16, 0.9466785289873973),\n                        (2, 0.5, 0.0, 0.46178536847925344)],\n                       [(4, 5), (8, 11), (10, 7)],\n                       [(4, 0.5, 0.0, -0.6244486306704427),\n                        (5, 0.5, 0.0, -0.289872382551553),\n                        (7, 0.5, 0.0, 0.4017018279302333),\n                        (8, 0.6391921844757946, 0.0, 0.7282225776878652),\n                        (11, 0.9294528636932586, 4.440892098500626e-16, 0.5567952414001085),\n                        (10, 0.31397581370367167, 0.0, -0.21890167318297582)],\n                       [(3, 4), (5, 2), (7, 8), (11, 10)],\n                       [(3, 0.5, 0.0, -0.5533214710126035),\n                        (4, 0.5, 0.0, 0.37555136932955735),\n                        (2, 0.5, 0.0, -0.038214631520746334),\n                        (7, 0.5000000000000004, 0.0, 0.4017018279302333),\n                        (8, 0.5, 0.0, 0.5132850848669728),\n                        (11, 0.33108034381799806, 0.0, 0.05679524140011294),\n                        (10, 0.5519644059311344, 0.0, 0.7987523594593542)],\n                       [(3, 4), (5, 2), (7, 8), (11, 10)],\n                       [(2, 0.5068340379253959, 0.0, -0.5310585926382718),\n                        (3, 0.490105327468914, 0.0, 0.44670206127071),\n                        (11, 0.8371247201733047, 0.0, -0.8632896185761176),\n                        (10, 0.7831723914001073, 0.0, 0.6373214569840702),\n                        (7, 0.2429421210211464, 0.0, -0.47575423725447186),\n                        (4, 0.5, 0.0, 0.375551369329558),\n                        (5, 0.49999999999999983, 0.0, -0.7898723825515532)],\n                       [(2, 0), (3, 1), (8, 11), (10, 7)],\n                       [(3, 0.4999999999999998, 0.0, 0.9459449113248355),\n                        (2, 0.5, 0.0, -0.7736945421001282),\n                        (8, 0.4960041929285068, 0.0, -0.9867149151330268),\n                        (10, 0.49999999999999956, 0.0, 0.7329995800098472),\n                        (7, 0.3638122834688833, 0.0, 0.7341328388043822),\n                        (0, 0.30408672398469605, 0.0, -0.5000000000000002),\n                        (1, 0.30408672398469605, 0.0, -0.5000000000000002)],\n                       [(4, 3), (5, 2), (8, 11), (10, 7)],\n                       [(2, 0.8040867239846965, 0.0, 0.22630545789987178),\n                        (3, 0.8040867239846965, 0.0, 0.9459449113248355),\n                        (7, 0.022839382910483597, 0.0, 0.23413283880437818),\n                        (8, 0.5, 4.440892098500626e-16, 0.513285084866973),\n                        (11, 0.8057964070938461, 0.0, -0.048305353185546185),\n                        (10, 0.7913981257738136, 0.0, 0.7329995800098474)],\n                       [(2, 0), (3, 1), (7, 8), (11, 10)],\n                       [(3, 0.75, 0.0, 0.9459449113248355),\n                        (2, 0.7499999999999999, 5.551115123125783e-17, 0.22630545789987172),\n                        (8, 0.5, 0.0, -0.1346597062606274),\n                        (10, 0.75, 0.0, 0.7329995800098461),\n                        (7, 0.22716061708951643, -2.220446049250313e-16, -0.7658671611956172),\n                        (0, 0.5, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5)],\n                       [(4, 3), (5, 2), (8, 11), (10, 7)],\n                       [(2, 0.5, 1.1102230246251565e-16, 0.226305457899871),\n                        (3, 0.5, 0.0, -0.05405508867516451),\n                        (8, 0.20292068705287522, 0.0, 0.36534029373937255),\n                        (7, 0.5000000000000001, -2.220446049250313e-16, -0.7658671611956172),\n                        (11, 0.5, 1.1102230246251565e-16, -0.16963777986712802),\n                        (10, 0.5, 1.1102230246251565e-16, -0.2670004199901541),\n                        (4, 0.5, 0.0, -0.3418462254634438),\n                        (5, 0.5, 0.0, -0.5577822602016909)],\n                       [(2, 0), (3, 1), (8, 7), (11, 10)],\n                       [(3, 0.25, 0.0, 0.9459449113248355),\n                        (2, 0.25000000000000006, 0.0, -0.7736945421001293),\n                        (8, 0.49999999999998235, 4.440892098500626e-16, 0.8653402374214318),\n                        (11, 0.4999999999999824, 0.0, -0.16963772354918705),\n                        (7, 0.5000000000000022, 0.0, -0.7538917530511764),\n                        (10, 0.5000000000000022, 0.0, -0.776666117634147)],\n                       [(4, 3), (5, 2), (8, 9), (11, 12)],\n                       [(2, 0.5, 0.0, -0.7736945421001282),\n                        (3, 0.5, 0.0, -0.05405508867516451),\n                        (8, 0.5000000000000102, 0.0, 0.8653402374214312),\n                        (11, 0.5000000000000102, 1.1102230246251565e-16, -0.16963772354918616),\n                        (9, 0.30408672398469605, 0.0, 0.4254807786478514),\n                        (12, 0.30408672398469627, 0.0, -0.5745192213521484),\n                        (4, 0.5, 0.8418462254634442, -0.3418462254634438),\n                        (5, 0.5, -0.9422177397983091, 0.44221773979830914)],\n                       [(2, 0), (3, 1), (8, 7), (11, 10)],\n                       [(8, 0.75, 1.1102230246251565e-16, -0.13465979509374648),\n                        (11, 0.75, 0.0, 0.8303623089659917),\n                        (7, 0.49999999999999956, -2.220446049250313e-16, 0.2461082794640057),\n                        (10, 0.49999999999999956, 0.0, -0.7766660851189651),\n                        (0, 1.0, 0.0, 0.5),\n                        (1, 1.0, 0.0, 0.5),\n                        (2, 0.5, -0.7263054578998709, -0.7736945421001293),\n                        (3, 0.5, 0.554055088675165, -0.05405508867516495)],\n                       [(8, 9), (11, 12)],\n                       [(8, 0.5005268894219381, 1.1102230246251565e-16, 0.3648132687790894),\n                        (11, 0.500526889421938, 0.0, 0.33088924509315576),\n                        (9, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515),\n                        (12, 0.5000000000000102, 0.0, 0.4254807786478514)],\n                       [(8, 7), (11, 10)],\n                       [(8, 0.5, 0.0, 0.6148128327056561),\n                        (11, 0.5, 5.551115123125783e-17, 0.08088968116658862),\n                        (7, 0.5000000000000004, 0.0, -0.25389172053599385),\n                        (10, 0.5000000000000004, 1.1102230246251565e-16, -0.27666608511896484)],\n                       [(8, 9), (11, 12)],\n                       [(8, 7),\n                        (11, 10),\n                        (9, 0.5, 0.3483047662953689, -0.5745192538673269),\n                        (12, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(7, 0.9992548329781763, 0.027677232964052445, 0.24610827946399794),\n                        (8, 3.251517832758566e-08, -0.3410273202776142, -0.38518718995983203),\n                        (10, 0.9992548329781763, -0.9495484024529759, -0.7766660851189735),\n                        (11, 0.999999967484822, 0.19289578593047613, 0.08088970383207611)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],\n           \'qubits\': [(2, 6),\n                      (3, 4),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 5),\n                      (6, 6),\n                      (6, 7)]},\n (13, 5): {\'method\': \'full-ladder count seed + joint decoder cores\',\n           \'moments\': [[(4, 0.8138849477842469, 0.0, 0.451451068435639),\n                        (5, 0.19771575564063265, 0.0, -0.6452734339569066),\n                        (7, 0.48773319825590145, 0.0, 0.2464534846669834),\n                        (8, 0.16362931622318963, 0.0, -0.5151103878879049),\n                        (3, 0.3016935522109139, 0.0, 0.5907166803683126),\n                        (2, 0.5, 0.0, -0.016627314735808696),\n                        (11, 0.49999999999999895, 0.0, -0.7088499809721285),\n                        (10, 0.48497757661839347, 0.0, -0.834800922923292),\n                        (0, 0.5, 1.0, 0.5),\n                        (1, 0.5, 1.0, 0.5),\n                        (9, 0.3040867239846966, 0.0, -0.5745192213521528),\n                        (12, 0.3040867239846966, 0.0, -0.5745192213521528)],\n                       [(4, 5), (7, 8)],\n                       [(4, 0.31710801470135797, 0.0, 0.5836562720220191),\n                        (7, 0.6213140367376081, -0.9999999999999996, -0.22915813972099075),\n                        (5, 0.0903120792991998, 0.0, -0.6866556269143476),\n                        (8, 0.497757816199472, 8.881784197001252e-16, -0.9101593233018268)],\n                       [(3, 4), (6, 7)],\n                       [(4, 0.599507429572071, 0.0, -0.2862704437338366),\n                        (7, 0.9680618203879026, 8.881784197001252e-16, -0.8961202268826416),\n                        (3, 0.7451823945884855, 0.9999999999999998, 0.7353116398967556),\n                        (6, 0.5, 0.0, -0.4027498548272366)],\n                       [(4, 5), (7, 8), (3, 6)],\n                       [(4, 0.28206690710260524, 0.0, -0.19980090984097387),\n                        (7, 0.7304617567414211, 0.0, -0.3646599967156221),\n                        (5, 0.7768913438727099, 0.0, -0.5164373669451634),\n                        (8, 0.18166268916206096, 0.0, 0.40902191094969637),\n                        (6, 0.4999999999999997, 0.0, 0.5972501451727632)],\n                       [(4, 7), (5, 8)],\n                       [(4, 0.5, 0.0, -0.2550619266328904),\n                        (7, 0.4999999999999998, 0.0, -0.7995241848994177),\n                        (8, 0.4999999999999998, 0.0, 0.38097551672644414)],\n                       [(4, 5), (7, 8)],\n                       [(5, 0.5000000000000001, 0.0, -0.19750501965420053),\n                        (8, 0.5515483609988416, 0.0, 0.8809755167264448),\n                        (4, 0.6229151979565117, 1.0, -0.7550619266328904),\n                        (7, 0.10075495914883417, 1.0, -0.14152692312500625)],\n                       [(5, 2), (8, 11), (3, 4), (6, 7)],\n                       [(7, 0.9675670579641595, 2.220446049250313e-16, -0.306409619912273),\n                        (8, 0.492805042274795, 5.551115123125783e-17, 0.20961883136435938),\n                        (5, 0.3385028359086418, 1.0, -0.1899817529776633),\n                        (2, 0.6715957554835205, 0.0, -0.16119008792398848),\n                        (6, 0.0, -0.3433965861636681, 0.0),\n                        (11, 0.4608483309798826, -2.220446049250313e-16, 0.8309193012950313)],\n                       [(7, 8), (4, 5)],\n                       [(4, 0.7292475564750505, 0.0, -0.5488067666951333),\n                        (5, 0.5, 0.0, 0.5053762220187079),\n                        (7, 0.5000000000000004, 0.0, -0.3064096199122899),\n                        (8, 0.5596506416568943, -4.440892098500626e-16, 0.12052163243884151)],\n                       [(3, 4), (5, 2), (7, 8)],\n                       [(7, 0.1275470395921725, 0.0, 0.5412045231586196),\n                        (8, 0.5000000000000003, 0.0, 0.0994204597987815),\n                        (4, 0.8236951914845039, 1.1102230246251565e-16, -0.4532476798165912),\n                        (5, 0.13681175462965722, 1.0, 0.5053762220187072),\n                        (2, 0.27172923178718184, 0.0, -0.7559786295779003),\n                        (3, 0.0, -0.6145493576814927, 0.0)],\n                       [(6, 7), (8, 11), (4, 5)],\n                       [(5, 0.3054490683281037, 0.0, 0.005376222018707688),\n                        (7, 0.2696276353165253, 4.440892098500626e-16, 0.5988489271868733),\n                        (8, 0.16304676742985036, 0.0, -0.9005795402012164),\n                        (4, 0.4518866973223102, 0.0, -0.8166609624938876),\n                        (11, 0.8273677055841601, 0.0, 0.011025144178165858)],\n                       [(5, 2), (7, 8), (3, 4)],\n                       [(8, 0.704332640729217, 0.0, 0.5994204597987838),\n                        (4, 0.5707034495925994, 0.0, -0.7578563847404256),\n                        (5, 0.785237728664876, 0.9999999999999999, -0.49462377798129253),\n                        (7, 0.3174136387017352, 1.0, -0.8430713141079453),\n                        (2, 0.4724075317596328, 0.0, 0.657484274013953),\n                        (3, 0.5000000000000036, 1.0, 0.1822716674435978)],\n                       [(8, 11), (4, 5), (6, 7)],\n                       [(5, 0.5858994169251086, 0.0, 0.005376222018707688),\n                        (7, 0.6276139104334912, 0.0, -0.7471642043131517),\n                        (8, 0.5, 0.0, 0.5994204597987844),\n                        (4, 0.5419474843487613, 5.551115123125783e-17, 0.07895259382144804),\n                        (11, 0.727371054243577, 0.0, -0.07318377315388713)],\n                       [(5, 2), (7, 8), (3, 4)],\n                       [(8, 0.529090738553442, 0.0, 0.2300169125298383),\n                        (4, 0.5, 0.0, -0.028445537528938303),\n                        (5, 0.15794180998375124, -0.9999999999999996, 0.5053762220187068),\n                        (7, 0.4999999999999989, 1.0, 0.3581695284471982),\n                        (3, 0.5000000000000002, 1.0, 0.1822716674436009),\n                        (2, 0.643422681037554, 1.0, 0.11543360081664161)],\n                       [(8, 11), (4, 5), (6, 7)],\n                       [(7, 0.6683089837594859, 4.440892098500626e-16, 0.8581695284471988),\n                        (8, 0.49999999999999944, 4.440892098500626e-16, -0.7764795574167833),\n                        (4, 0.10589866865514248, 0.0, 0.4715544624710617),\n                        (5, 2),\n                        (11, 0.49999999999999956, 0.0, -0.09727132543917172),\n                        (6, 0.5000000000000003, 0.0, 0.9470831233180523)],\n                       [(7, 8),\n                        (3, 4),\n                        (5, 0.5, 5.551115123125783e-17, 0.005376222018707355),\n                        (2, 0.6811220971645664, -0.9999999999999998, 0.6538252746140196)],\n                       [(8, 11),\n                        (4, 0.0, 0.6173893956701, 0.0),\n                        (7, 0.6578349175114986, 0.9999999999999998, 0.8581695284471988),\n                        (3, 0.49089820890137076, 1.1102230246251565e-16, -0.2016290166751582)],\n                       [(4, 5),\n                        (6, 7),\n                        (8, 0.5000000000000009, 0.0, -0.7764795574167849),\n                        (11, 0.6530017079301533, -2.220446049250313e-16, 0.4027286745608292)],\n                       [(4, 0.42589339437843365, 0.0, -0.41105614185883765),\n                        (7, 0.5, 0.0, 0.35803982196264905),\n                        (11, 10),\n                        (5, 0.3395311910112335, 0.0, 0.0053762220187074106),\n                        (6, 0.5000000000000004, 0.03936600105797039, 0.9470831233180523)],\n                       [(3, 4),\n                        (7, 8),\n                        (5, 2),\n                        (11, 0.5000000000000003, 1.1102230246251565e-16, -0.09727132543918637),\n                        (10, 0.5605616196317902, 0.0, -0.8348009229232944)],\n                       [(8, 11),\n                        (7, 0.5, 0.0, 0.35803982196264905),\n                        (4, 0.5, 1.0, -0.4110561418588379),\n                        (5, 0.5, 0.0, -0.9946237779812921)],\n                       [(10, 7),\n                        (4, 5),\n                        (11, 0.4311009172299638, 0.0, -0.34731902068777165),\n                        (8, 0.5, -0.9999999999999996, -0.5397475236130314)],\n                       [(5, 0.5558460531129376, 0.0, 0.5053762220187079),\n                        (10, 0.5818042231825744, -2.220446049250313e-16, -0.3348009229233022),\n                        (4, 0.0035868934861584827, 1.0, -0.9110561418588412),\n                        (7, 8)],\n                       [(5, 2),\n                        (11, 10),\n                        (3, 4),\n                        (7, 0.2430469681607357, 0.0, -0.04773614639265755),\n                        (8, 0.5, -0.9999999999999996, -0.5397475236130314)],\n                       [(11, 0.03454957128703273, 0.0, 0.8181166636420727),\n                        (10, 0.2894984978675621, 0.0, -0.8348009229233004),\n                        (2, 0.7708918263538406, 0.0, -0.39597210290977203),\n                        (3, 0.7708918263538402, 0.0, -0.9523498742984386),\n                        (7, 8),\n                        (5, 0.4999999999999998, 0.0, 0.005376222018708354),\n                        (4, 0.5000000000000001, 1.0, -0.4110561418588379)],\n                       [(11, 10), (2, 0), (3, 1), (7, 0.3056831728462866, 0.0, -0.40017922507473047)],\n                       [(2, 0.5, 0.0, 0.10402789709022792),\n                        (11, 0.9242869536756131, 8.881784197001252e-16, 0.5858061491772155),\n                        (3, 0.5, 0.0, -0.45234987429843887),\n                        (10, 0.87386354985307, 0.9999999999999998, 0.6651990770766807),\n                        (0, 0.3040867239846962, 1.0, -0.5000000000000002),\n                        (1, 0.3040867239846962, 1.0, -0.5000000000000002)],\n                       [(5, 2), (8, 11), (4, 3), (10, 7)],\n                       [(11, 0.23753846872505302, 0.0, 0.11812387065190871),\n                        (10, 0.6888649043845242, 0.0, -0.8348009229233393),\n                        (2, 0.8040867239846965, 1.6653345369377348e-16, 0.10402789709022808),\n                        (3, 0.8040867239846965, 0.0, 0.5476501257015611),\n                        (7, 0.5595532320705697, 0.0, 0.017723081111546024),\n                        (8, 0.5, 0.9999999999999999, -0.03987723009757982),\n                        (5, 0.0, -0.5431422476211072, 0.0)],\n                       [(11, 10), (2, 0), (3, 1), (7, 8)],\n                       [(3, 0.75, 0.0, 0.5476501257015611),\n                        (8, 0.5000000000000002, -2.220446049250313e-16, -0.039877230097580485),\n                        (11, 0.6383222145074643, 0.0, 0.17679703477273234),\n                        (10, 0.49999999999999956, 0.0, -0.8348009229233395),\n                        (7, 0.0, -0.37270745352194545, 0.0),\n                        (2, 0.75, 0.0, 0.10402789709022808),\n                        (0, 0.5, 1.0, 0.5),\n                        (1, 0.5, 1.0, 0.5)],\n                       [(4, 3), (8, 11), (10, 7), (5, 2)],\n                       [(8, 0.5, 0.0, 0.15054727144263436),\n                        (11, 0.4999999999999997, 0.0, -0.7484872360058605),\n                        (10, 0.4546007179287068, 0.0, 0.4167387613319282),\n                        (2, 0.5, 0.0, -0.895972102909772),\n                        (3, 0.5, 0.0, 0.5476501257015611),\n                        (4, 0.5, 0.0, 0.953497944964242),\n                        (5, 0.5, 1.0, -0.9023201124254785)],\n                       [(7, 8), (11, 10), (2, 0), (3, 1)],\n                       [(3, 0.25000000000000006, 0.0, -0.4523498742984391),\n                        (11, 0.8858944659256832, 5.551115123125783e-17, 0.25151276399414046),\n                        (10, 0.19504058663172466, 0.0, -0.2857088984950029),\n                        (7, 0.2978230143957522, 0.0, -0.7099737203247478),\n                        (2, 0.25000000000000006, 0.0, 0.10402789709022825)],\n                       [(4, 3), (8, 11), (10, 7), (5, 2)],\n                       [(2, 0.5, 0.0, -0.895972102909772),\n                        (3, 0.5, 0.0, -0.4523498742984391),\n                        (8, 0.21266361194798097, 0.0, 0.6505472714426326),\n                        (7, 0.5000000000000003, 0.0, 0.4926536270859925),\n                        (11, 0.5, 1.6653345369377348e-16, 0.038849152046160074),\n                        (10, 0.5000000000000007, 0.0, 0.6739679588648033),\n                        (5, 0.5, -0.5976798875745217, -0.9023201124254785),\n                        (4, 0.5, 0.546502055035758, 0.953497944964242)],\n                       [(2, 0), (3, 1), (8, 7), (11, 10)],\n                       [(8, 0.4999999999999824, 4.440892098500626e-16, -0.849452672239426),\n                        (11, 0.4999999999999824, -2.220446049250313e-16, 0.038849208364101384),\n                        (7, 0.500000000000002, 0.0, -0.4326735968099693),\n                        (10, 0.5000000000000019, 0.0, -0.2513592650311596),\n                        (0, 1.0, 0.0, 0.0),\n                        (1, 1.0, 0.0, 0.0),\n                        (2, 0.4999999999999998, -0.6040278970902282, 0.1040278970902283),\n                        (3, 0.4999999999999998, 0.9523498742984396, 0.5476501257015605)],\n                       [(8, 9), (11, 12)],\n                       [(8, 0.5000000000000103, 0.0, -0.8494526722394253),\n                        (11, 0.5000000000000103, 0.0, 0.03884920836410144),\n                        (9, 0.30408672398469627, 0.0, -0.5745192213521484),\n                        (12, 0.30408672398469627, 0.0, -0.5745192213521484)],\n                       [(8, 7), (11, 10)],\n                       [(8, 0.75, 0.0, 0.15054736027575344),\n                        (11, 0.75, 4.440892098500626e-16, -0.9611507591207205),\n                        (7, 0.49999999999999956, 0.0, -0.43267356429478787),\n                        (10, 0.49999999999999956, 0.0, -0.2513592325159779)],\n                       [(8, 9), (11, 12)],\n                       [(8, 0.500526889421938, 0.0, -0.34892570359708186),\n                        (11, 0.500526889421938, 0.0, 0.5393761770064442),\n                        (9, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515),\n                        (12, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515)],\n                       [(8, 7), (11, 10)],\n                       [(8, 0.5, 0.0, -0.5989252675236496),\n                        (11, 0.5, 5.551115123125783e-17, 0.28937661307987683),\n                        (7, 0.5000000000000004, 0.0, 0.0673264357052128),\n                        (10, 0.5000000000000004, 0.0, 0.24864076748402253)],\n                       [(8, 9), (11, 12)],\n                       [(8, 7),\n                        (11, 10),\n                        (9, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (12, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(7, 0.9992548329781763, 0.7064590767228467, -0.43267356429479564),\n                        (8, 0.999999967484822, 0.8727107346207148, -0.5989252448581615),\n                        (10, 0.9992548329781763, 0.5251447449440367, -0.2513592325159859),\n                        (11, 0.999999967484822, -0.01559114598281175, 0.2893766357453651)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],\n           \'qubits\': [(2, 6),\n                      (3, 4),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 5),\n                      (6, 6),\n                      (6, 7)]},\n (14, 4): {\'method\': \'full-ladder count seed + joint decoder cores\',\n           \'moments\': [[(0, 0.5, 0.0, 0.5),\n                        (1, 0.35760927658307967, 0.0, -0.9848107895924711),\n                        (2, 0.2271855258285052, 0.0, -0.708982875827169),\n                        (4, 0.5000000000000004, 0.0, 0.41486515710664773),\n                        (5, 0.5000000000000009, 0.0, -0.6071766448746729),\n                        (6, 0.30408672398469655, 0.0, -0.5745192213521528),\n                        (7, 0.05843249485107593, 8.881784197001252e-16, -0.95137724379125),\n                        (8, 0.5, 0.0, 0.561138072557031),\n                        (9, 0.5000000000000019, 0.0, 0.7904207099825034),\n                        (10, 0.30408672398469655, 0.0, -0.5745192213521528),\n                        (11, 0.5, 0.0, 0.5),\n                        (12, 0.35760927658307967, 0.0, -0.9848107895924711),\n                        (13, 0.2271855258285052, 0.0, -0.708982875827169)],\n                       [(3, 7), (4, 8), (5, 9)],\n                       [(3, 0.49332168028256174, 4.440892098500626e-16, 0.6336983243665104),\n                        (4, 0.7245717802304248, 0.0, -0.9113440205462175),\n                        (5, 0.3371168074051336, 0.0, 0.37215328785234725),\n                        (7, 0.05843249485107573, 0.0, 0.04862275620875023),\n                        (8, 0.04815700339732576, 0.0, -0.019820365552267738),\n                        (9, 0.16407381447172714, 0.0, -0.24619611272460395)],\n                       [(3, 4), (7, 8)],\n                       [(4, 0.12068227110890642, 0.0, -0.11488744919650928),\n                        (7, 0.49999999999999595, 0.0, 0.38702674264002646),\n                        (8, 0.4889448848971045, 1.1102230246251565e-16, -0.25040975145735467)],\n                       [(3, 7), (4, 5), (8, 9)],\n                       [(4, 0.7500000000000058, 0.0, 0.88511255080349),\n                        (5, 0.2499999999999969, 0.0, -0.8118656185066513),\n                        (8, 0.7499999999999981, 0.0, -0.05679745541643855),\n                        (9, 0.7499999999999969, 0.0, 0.9798484592894459),\n                        (7, 0.5, 0.0, -0.6129732573599684)],\n                       [(4, 8), (5, 9)],\n                       [(4, 0.47358199645116533, 0.0, 0.9903417702722856),\n                        (5, 0.5000000000000011, -2.220446049250313e-16, -0.811865618506652),\n                        (8, 0.5264180035488338, 0.0, -0.16202667488523703),\n                        (9, 0.5, 0.0, -0.020151540710554583)],\n                       [(3, 4), (5, 2), (7, 8), (9, 13)],\n                       [(2, 0.22718552582850488, 0.0, 0.2910171241728308),\n                        (4, 0.5, 0.0, -0.9342121461279094),\n                        (5, 0.4999999999999998, 0.0, -0.1521546545231216),\n                        (8, 0.5000000000000002, 0.0, 0.9134194087145677),\n                        (9, 0.4999999999999998, 0.0, -0.36044057672702445),\n                        (13, 0.22718552582850488, 0.0, 0.2910171241728308)],\n                       [(4, 5), (8, 9)],\n                       [(3, 4), (5, 2), (7, 8), (9, 13)],\n                       [(5, 0.680195383285978, 4.440892098500626e-16, 0.792718703306978),\n                        (9, 0.680195383285978, 0.0, 0.5844327811030758),\n                        (2, 0.6354920868801655, -8.881784197001252e-16, 0.5482021483143635),\n                        (4, 0.5886108001133228, -1.3322676295501878e-15, 0.0249144556291947),\n                        (8, 0.5886108001133228, -1.3322676295501878e-15, -0.1274539895283291),\n                        (13, 0.6354920868801655, -8.881784197001252e-16, 0.5482021483143635)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.5000000000000004, 4.440892098500626e-16, 0.6314206415968306),\n                        (5, 0.4999999999999994, 0.0, -0.49880294595382435),\n                        (8, 0.5000000000000003, 0.0, 0.4790521964393077),\n                        (9, 0.4999999999999993, 0.0, -0.707088868157727)],\n                       [(3, 4), (5, 2), (7, 8), (9, 13)],\n                       [(3, 0.5, 0.0, -0.3329764617331561),\n                        (4, 0.5234132252872344, 0.0, 0.7798563248777819),\n                        (5, 0.06708148302124252, 0.0, -0.7254941333919833),\n                        (7, 0.5, 0.0, -0.0664429892685352),\n                        (8, 0.5234132252872344, 0.0, 0.6274878797202588),\n                        (9, 0.06708148302124252, 0.0, -0.9337800555958857)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.3327291458405614, 0.0, -0.26629045511556204),\n                        (5, 0.059202003835620236, 0.0, -0.22549413339198332),\n                        (8, 0.33272914584056146, 0.0, -0.4186589002730863),\n                        (9, 0.059202003835620236, 0.0, -0.4337800555958857)],\n                       [(4, 5), (8, 9)],\n                       [(5, 0.08917758998520853, 0.0, -0.1015881890081114),\n                        (8, 0.25011854054823907, 0.0, -0.9186589002730847),\n                        (9, 0.08917758998520853, 0.0, -0.3098741112120138),\n                        (4, 0.25011854054823907, 0.0, -0.7662904551155616)],\n                       [(5, 2), (7, 8), (9, 13)],\n                       [(2, 0.021107237782608028, 0.0, -0.9231467397292097),\n                        (3, 4),\n                        (5, 0.5938461416699012, 0.0, -0.2891865270682792),\n                        (7, 0.4999999999999976, 0.1796633405973913, 0.9335570107314686),\n                        (9, 0.5938461416699012, 0.0, -0.4974724492721816),\n                        (13, 0.021107237782608028, 0.0, -0.9231467397292097)],\n                       [(2, 1),\n                        (3, 0.4999999999999976, 0.446196813062012, 0.6670235382668477),\n                        (4, 5),\n                        (8, 9),\n                        (13, 12)],\n                       [(1, 0.8402337422326178, 0.0, 0.9498164599690078),\n                        (2, 0.3147365041110266, 0.0, -0.7244934550958735),\n                        (5, 0.4433968217557545, 0.0, -0.8065677230779151),\n                        (9, 0.4433968217557545, 0.0, 0.9851463547181822),\n                        (12, 0.8402337422326178, 0.0, 0.9498164599690078),\n                        (13, 0.3147365041110266, 0.0, -0.7244934550958735)],\n                       [(1, 4), (5, 2), (9, 13), (12, 8)],\n                       [(1, 0.31397581370367184, 0.0, -0.21890167318297582),\n                        (2, 0.9294528636932586, 8.881784197001252e-16, 0.5567952414001085),\n                        (4, 0.5, 5.551115123125783e-17, 0.20302107621594673),\n                        (5, 0.6391921844757946, 0.0, -0.22388560601877483),\n                        (8, 0.5, -2.220446049250313e-16, 0.050652631058421715),\n                        (9, 0.6391921844757946, 0.0, -0.43217152822267746),\n                        (12, 0.31397581370367184, 0.0, -0.21890167318297582),\n                        (13, 0.9294528636932586, 8.881784197001252e-16, 0.5567952414001085)],\n                       [(2, 1), (4, 5), (8, 9), (13, 12)],\n                       [(1, 0.5519644059311344, 0.0, 0.7987523594593542),\n                        (2, 0.33108034381799806, 0.0, 0.05679524140011294),\n                        (4, 0.5000000000000004, 0.0, 0.20302107621594678),\n                        (5, 0.5, 0.0, -0.4388230988396682),\n                        (8, 0.5000000000000003, -2.220446049250313e-16, 0.050652631058421715),\n                        (9, 0.5000000000000001, 0.0, -0.647109021043571),\n                        (12, 0.5519644059311344, 0.0, 0.7987523594593542),\n                        (13, 0.33108034381799806, 0.0, 0.05679524140011294)],\n                       [(2, 1), (4, 5), (8, 9), (13, 12)],\n                       [(1, 0.7831723914001073, 0.0, 0.6373214569840702),\n                        (2, 0.8371247201733047, 0.0, -0.8632896185761176),\n                        (4, 0.24294212102114646, 0.0, -0.6744349889687595),\n                        (8, 0.24294212102114637, 0.0, -0.8268034341262835),\n                        (12, 0.7831723914001073, 0.0, 0.6373214569840702),\n                        (13, 0.8371247201733047, 0.0, -0.8632896185761176)],\n                       [(1, 4), (5, 2), (9, 13), (12, 8)],\n                       [(1, 0.49999999999999944, 4.440892098500626e-16, 0.7329995800098472),\n                        (4, 0.3638122834688833, 0.0, 0.5354520870900956),\n                        (5, 0.4960041929285068, 5.551115123125783e-17, 0.0611769011603322),\n                        (8, 0.3638122834688833, 0.0, 0.38308364193257094),\n                        (9, 0.4960041929285068, 0.0, -0.14710902104356993),\n                        (12, 0.49999999999999944, 4.440892098500626e-16, 0.7329995800098472)],\n                       [(1, 4), (5, 2), (9, 13), (12, 8)],\n                       [(1, 0.7913981257738136, 0.0, 0.7329995800098474),\n                        (2, 0.8057964070938461, 0.0, -0.04830535318554707),\n                        (4, 0.0228393829104836, 0.0, 0.0354520870900894),\n                        (5, 0.4999999999999997, 0.0, -0.4388230988396673),\n                        (8, 0.022839382910483597, 0.0, -0.11691635806743483),\n                        (9, 0.4999999999999998, 0.0, -0.6471090210435695),\n                        (12, 0.7913981257738136, 0.0, 0.7329995800098474),\n                        (13, 0.8057964070938461, 0.0, -0.04830535318554707)],\n                       [(2, 1), (4, 5), (8, 9), (13, 12)],\n                       [(1, 0.75, 0.0, 0.7329995800098463),\n                        (4, 0.22716061708951618, 0.0, -0.9645479129099039),\n                        (5, 0.5, 0.0, 0.9132321100327316),\n                        (8, 0.22716061708951624, 0.0, 0.883083641932572),\n                        (9, 0.5, 0.0, 0.704946187828829),\n                        (12, 0.75, 0.0, 0.7329995800098463)],\n                       [(1, 4), (5, 2), (9, 13), (12, 8)],\n                       [(1, 0.9427632774999226, 0.0, -0.7670004199901534),\n                        (4, 0.4999999999999998, 0.0, -0.9645479129099062),\n                        (5, 0.20292068705287525, 0.0, -0.5867678899672684),\n                        (8, 0.4999999999999997, 0.0, 0.8830836419325698),\n                        (9, 0.20292068705287525, 0.0, -0.795053812171171),\n                        (12, 0.9427632774999226, 0.0, -0.7670004199901534)],\n                       [(1, 0), (5, 4), (9, 8), (12, 11)],\n                       [(0, 0.3040867239846962, 0.0, -0.5000000000000002),\n                        (1, 0.4999999999999998, 0.0, -0.2670004199901532),\n                        (4, 0.500000000000002, 0.0, -0.9525725047654634),\n                        (5, 0.49999999999998246, 0.0, -0.0867679462852089),\n                        (8, 0.5000000000000022, 0.0, 0.8950590500770128),\n                        (9, 0.49999999999998246, 0.0, -0.2950538684891111),\n                        (11, 0.3040867239846962, 0.0, -0.5000000000000002),\n                        (12, 0.4999999999999998, 0.0, -0.2670004199901532)],\n                       [(2, 1), (5, 6), (9, 10), (13, 12)],\n                       [(1, 0.8040867239846965, 0.0, 0.7329995800098466),\n                        (5, 0.5000000000000102, 0.0, -0.08676794628520934),\n                        (6, 0.3040867239846962, 0.0, 0.4254807786478514),\n                        (9, 0.5000000000000103, 0.0, -0.2950538684891111),\n                        (10, 0.3040867239846962, 0.0, 0.4254807786478514),\n                        (12, 0.8040867239846965, 0.0, 0.7329995800098466)],\n                       [(1, 0), (5, 4), (9, 8), (12, 11)],\n                       [(0, 0.5, 0.0, 0.5),\n                        (1, 0.75, 0.0, 0.7329995800098466),\n                        (4, 0.4999999999999995, 0.0, 0.04742752774971759),\n                        (5, 0.75, 0.0, 0.9132320211996126),\n                        (8, 0.49999999999999956, 0.0, -0.10494091740780642),\n                        (9, 0.75, 4.440892098500626e-16, 0.7049460989957104),\n                        (11, 0.5, 0.0, 0.5),\n                        (12, 0.75, 0.0, 0.7329995800098466)],\n                       [(2, 1), (5, 6), (9, 10), (13, 12)],\n                       [(1, 0.5, 0.0, 0.7329995800098474),\n                        (2, 0.5, 2.220446049250313e-16, -0.44342329229517063),\n                        (5, 0.500526889421938, 0.0, -0.5872949149275515),\n                        (6, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155),\n                        (9, 0.5005268894219381, 0.0, -0.7955808371314532),\n                        (10, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155),\n                        (12, 0.5, 0.0, 0.7329995800098474),\n                        (13, 0.5, 2.220446049250313e-16, -0.44342329229517063)],\n                       [(1, 0), (5, 4), (9, 8), (12, 11)],\n                       [(1, 0.25000000000000006, 0.0, -0.26700041999015234),\n                        (4, 0.5000000000000004, 0.0, -0.45257247225028174),\n                        (5, 0.5, 0.0, -0.3372953510009846),\n                        (8, 0.5000000000000004, 0.0, -0.604940917407806),\n                        (9, 0.5, 0.0, -0.5455812732048857),\n                        (12, 0.25000000000000006, 0.0, -0.26700041999015234)],\n                       [(2, 1), (5, 6), (9, 8), (13, 12)],\n                       [(1, 0.5, 1.1102230246251565e-16, -0.26700041999015367),\n                        (2, 0.5, 0.9434232922951704, 0.5565767077048294),\n                        (5, 4),\n                        (6, 0.5, 0.3483047662953691, -0.5745192538673274),\n                        (8, 0.9992548329781763, 0.37872642983586546, -0.1049409174078133),\n                        (9, 10),\n                        (12, 0.5, 1.1102230246251565e-16, -0.26700041999015367),\n                        (13, 0.5, 0.9434232922951704, 0.5565767077048294)],\n                       [(1, 0),\n                        (4, 0.9992548329781763, 0.22635798467834123, 0.04742752774971093),\n                        (5, 3.2515178327585665e-08, 0.6110808634290263, 0.6627046263335277),\n                        (9, 3.251517832758566e-08, 0.8193667856329276, 0.4544187041296257),\n                        (10, 0.5, 0.3483047662953691, -0.5745192538673274),\n                        (12, 11)],\n                       [(0, 1.0, 0.0, 0.5),\n                        (1, 0.5, 0.7670004199901537, -0.26700041999015234),\n                        (11, 1.0, 0.0, 0.5),\n                        (12, 0.5, 0.7670004199901537, -0.26700041999015234)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],\n           \'qubits\': [(2, 5),\n                      (3, 5),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 4),\n                      (6, 5),\n                      (6, 6)]},\n (14, 5): {\'method\': \'full-ladder count seed + joint decoder cores\',\n           \'moments\': [[(3, 0.4291258049338973, -0.33619259345427, 0.27888300541374644),\n                        (4, 0.8029022216991971, 0.17967409411742574, -0.6795402904419618),\n                        (7, 0.4461007598529419, -0.9889714909990381, 0.811154843560373),\n                        (8, 0.5475750903028308, -0.6509235226001127, 0.44189632417956765),\n                        (5, 0.20961770999891505, -0.5551951289528794, -0.09325449435175726),\n                        (9, 0.4999999999999994, -0.8268773312843654, 0.3843727146867719),\n                        (2, 0.4999999999999989, -0.355968954837675, -0.7088499809721285),\n                        (1, 0.48497757661839347, 0.7879716185413642, -0.834800922923292),\n                        (6, 0.30408672398469644, 0.48537839930289595, -0.5745192213521528),\n                        (0, 0.5, 0.0, 0.5),\n                        (13, 0.4999999999999989, -0.355968954837675, -0.7088499809721285),\n                        (12, 0.48497757661839347, 0.7879716185413642, -0.834800922923292),\n                        (10, 0.30408672398469644, 0.48537839930289595, -0.5745192213521528),\n                        (11, 0.5, 0.0, 0.5)],\n                       [(3, 4), (7, 8)],\n                       [(4, 0.09275945293094852, 0.8483293112870449, 0.2271722753726409),\n                        (8, 0.7668857957035857, 0.1768904302906315, -0.17349813922391233),\n                        (3, 0.5915601888091553, -0.1697296791878915, 0.8008945589409266),\n                        (7, 0.7719184238060086, 0.060310051522143304, 0.8221833525613349)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.5839140775751577, -0.34607659150223036, 0.38370733179805117),\n                        (8, 5.804421932165967e-17, 0.5454239453279757, 0.46674560341825394),\n                        (5, 0.6794758505275434, 0.41601170232978646, 0.5580084241339129),\n                        (9, 0.7609429609419834, 0.18266219582910237, -0.6181457832967272)],\n                       [(3, 4), (7, 8), (5, 9)],\n                       [(3, 0.4267197981978191, -0.8010665396478638, 0.9533401235731631),\n                        (7, 0.8334102992827598, 0.47748639417477656, -0.5111674232892016),\n                        (4, 0.6479378857123176, -0.586143910684032, 0.8915154335008122),\n                        (8, 0.8025962924367844, -0.5637503116250122, -0.5640570561756553),\n                        (5, 1.0, 0.0, -0.1615932262882147),\n                        (9, 0.5, 0.06593593715609103, 0.8449409682549109)],\n                       [(3, 7), (4, 8)],\n                       [(4, 0.5000000000000001, -0.7746421317492975, 0.579925715243609),\n                        (8, 0.5, -0.21198827861584846, 0.8766846551095366),\n                        (3, 5.311744531340066e-16, -0.5085340436454824, -0.4547902864013973),\n                        (7, 0.49999999999999867, -0.12921612430237528, 0.6138779928829261)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.5789905350453514, 0.24908828994214288, 0.3963998896284909),\n                        (5, 0.4999999999999993, 0.5063968720587635, -0.44187576750681634),\n                        (8, 0.7539715276231561, -0.7737760672371199, 0.5364788470486759),\n                        (9, 0.355072881972375, -0.8463558008590539, 0.4108769054110007)],\n                       [(3, 4), (5, 2), (7, 8), (9, 13)],\n                       [(4, 0.9675670579641595, -0.16122952726528128, -0.8315400945967895),\n                        (5, 0.49280504227479477, 0.9075635894024535, -0.10683558081013844),\n                        (3, 0.9999999999999991, -0.5436467934198348, -0.9605731563991813),\n                        (2, 0.4608483309798825, -0.09915742697491803, 0.47495034645735623),\n                        (8, 0.9675670579641595, -0.16122952726528128, -0.8315400945967895),\n                        (9, 0.49280504227479477, 0.9075635894024535, -0.10683558081013844),\n                        (7, 0.9999999999999991, -0.5436467934198348, -0.9605731563991813),\n                        (13, 0.4608483309798825, -0.09915742697491803, 0.47495034645735623)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.5000000000000003, -0.7586424053170859, -0.9927696218620878),\n                        (5, 0.5596506416568944, 0.24188869310330485, 0.7116308096667971),\n                        (8, 0.5000000000000003, -0.7586424053170859, -0.9927696218620878),\n                        (9, 0.5596506416568944, 0.24188869310330485, 0.7116308096667971)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.12754703959217242, -0.36481970302095323, -0.9037978841082646),\n                        (5, 0.5000000000000006, 0.9492195332378222, 0.9324183301300419),\n                        (8, 0.12754703959217242, -0.36481970302095323, -0.9037978841082646),\n                        (9, 0.5000000000000006, 0.9492195332378222, 0.9324183301300419)],\n                       [(3, 4), (5, 2), (7, 8), (9, 13)],\n                       [(4, 0.2696276353165252, -0.5495993141445261, 0.789026816899036),\n                        (5, 0.16304676742985044, -0.24342097404763607, 0.8816378633678659),\n                        (3, 1.0, 0.0, 0.33800163024771757),\n                        (2, 0.8273677055841603, 0.5873346958909589, -0.4441012376344271),\n                        (8, 0.2696276353165252, -0.5495993141445261, 0.789026816899036),\n                        (9, 0.16304676742985044, -0.24342097404763607, 0.8816378633678659),\n                        (7, 1.0, 0.0, 0.33800163024771757),\n                        (13, 0.8273677055841603, 0.5873346958909589, -0.4441012376344271)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.3174136387017352, 0.6930417686718258, 0.7975072614596905),\n                        (5, 0.704332640729217, 0.301171029514539, 0.13821688932023013),\n                        (8, 0.3174136387017352, 0.6930417686718258, 0.7975072614596905),\n                        (9, 0.704332640729217, 0.301171029514539, 0.13821688932023013)],\n                       [(3, 4), (5, 2), (7, 8), (9, 13)],\n                       [(4, 0.6276139104334912, -0.7712944540396096, -0.4135438600736898),\n                        (5, 0.5, -0.4399356237180161, 0.4393879188347696),\n                        (3, 4.6653006315616715e-17, 0.43188418124781147, -0.003472632906682538),\n                        (2, 0.7273710542435771, -0.15147124437161352, 0.059024540924478786),\n                        (8, 0.6276139104334912, -0.7712944540396096, -0.4135438600736898),\n                        (9, 0.5, -0.4399356237180161, 0.4393879188347696),\n                        (7, 4.6653006315616715e-17, 0.43188418124781147, -0.003472632906682538),\n                        (13, 0.7273710542435771, -0.15147124437161352, 0.059024540924478786)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.4999999999999989, -0.1357827491349317, -0.0795045813529498),\n                        (5, 0.529090738553442, -0.8807713572410005, -0.3699512521521924),\n                        (8, 0.4999999999999989, -0.1357827491349317, -0.0795045813529498),\n                        (9, 0.529090738553442, -0.8807713572410005, -0.3699512521521924)],\n                       [(3, 4), (5, 2), (7, 8), (9, 13)],\n                       [(4, 0.6683089837594861, -0.9159285507569996, 0.28471266951211915),\n                        (5, 0.49999999999999944, 0.7314449591021075, -0.2572190793398146),\n                        (3, 0.5000000000000004, 0.7760642706230433, 0.8750535406396822),\n                        (2, 0.49999999999999967, -0.2608811138803624, -0.11653425573241938),\n                        (8, 0.6683089837594861, -0.9159285507569996, 0.28471266951211915),\n                        (9, 0.49999999999999944, 0.7314449591021075, -0.2572190793398146),\n                        (7, 0.5000000000000004, 0.7760642706230433, 0.8750535406396822),\n                        (13, 0.49999999999999967, -0.2608811138803624, -0.11653425573241938)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.6578349175114987, 0.9219224176101205, -0.6312158812448805),\n                        (5, 2.2586248823393435e-14, 0.6784772766603563, -0.012237680117800354),\n                        (8, 0.6578349175114987, 0.9219224176101205, -0.6312158812448805),\n                        (9, 2.2586248823393435e-14, 0.6784772766603563, -0.012237680117800354)],\n                       [(3, 4), (5, 2), (7, 8), (9, 13)],\n                       [(4, 0.5, 0.7931943159212288, -0.20942317011930944),\n                        (5, 0.5000000000000009, 0.8692668947781764, -0.8472968435773516),\n                        (2, 0.6530017079301534, 0.7854030106156598, 0.12258463038721912),\n                        (8, 0.5, 0.7931943159212288, -0.20942317011930944),\n                        (9, 0.5000000000000009, 0.8692668947781764, -0.8472968435773516),\n                        (13, 0.6530017079301534, 0.7854030106156598, 0.12258463038721912),\n                        (3, 0.4999999999999998, -0.6646686868867029, 0.6511178112627256),\n                        (7, 0.4999999999999998, -0.6646686868867029, 0.6511178112627256)],\n                       [(4, 5), (2, 1), (8, 9), (13, 12)],\n                       [(5, 8.859013786781178e-17, -0.55927478731788, -0.8230859054287967),\n                        (2, 0.5000000000000007, -0.9781657902510845, 0.4079876410028638),\n                        (1, 0.5605616196317903, -0.48158863111488426, -0.046829304381930115),\n                        (4, 0.5, -0.3409383911073316, 0.5837711458019192),\n                        (9, 8.859013786781178e-17, -0.55927478731788, -0.8230859054287967),\n                        (13, 0.5000000000000007, -0.9781657902510845, 0.4079876410028638),\n                        (12, 0.5605616196317903, -0.48158863111488426, -0.046829304381930115),\n                        (8, 0.5, -0.3409383911073316, 0.5837711458019192)],\n                       [(5, 2), (1, 4), (9, 13), (12, 8)],\n                       [(4, 1.0, 0.0, -0.2945271818851718),\n                        (5, 0.5, 0.02229733440238324, -0.3005727023133016),\n                        (2, 0.4311009172299637, -0.7940071962495523, -0.820225844496806),\n                        (1, 0.5818042231825744, 0.22571420434272071, -0.028417935496822366),\n                        (8, 1.0, 0.0, -0.2945271818851718),\n                        (9, 0.5, 0.02229733440238324, -0.3005727023133016),\n                        (13, 0.4311009172299637, -0.7940071962495523, -0.820225844496806),\n                        (12, 0.5818042231825744, 0.22571420434272071, -0.028417935496822366)],\n                       [(4, 5), (2, 1), (8, 9), (13, 12)],\n                       [(4, 0.2430469681607356, -0.3869185186975477, -0.4261111501096244),\n                        (5, 0.5, -0.43764950070618913, -0.27827536791091845),\n                        (2, 0.034549571287032656, -0.7922338222925074, -0.4487973564165144),\n                        (1, 0.2894984978675621, -0.21954933440100288, -0.30270373115409965),\n                        (8, 0.2430469681607356, -0.3869185186975477, -0.4261111501096244),\n                        (9, 0.5, -0.43764950070618913, -0.27827536791091845),\n                        (13, 0.034549571287032656, -0.7922338222925074, -0.4487973564165144),\n                        (12, 0.2894984978675621, -0.21954933440100288, -0.30270373115409965)],\n                       [(4, 5), (2, 1), (8, 9), (13, 12)],\n                       [(5, 5.961468152386983e-17, 0.06207043471344065, 0.2906552214721423),\n                        (2, 0.924286953675613, -0.1210255993169338, 0.5266583068261215),\n                        (1, 0.87386354985307, 0.606274243274356, 0.9777469344448786),\n                        (4, 0.30568317284628654, 0.47615442362120625, -0.46058659012509917),\n                        (9, 5.961468152386983e-17, 0.06207043471344065, 0.2906552214721423),\n                        (13, 0.924286953675613, -0.1210255993169338, 0.5266583068261215),\n                        (12, 0.87386354985307, 0.606274243274356, 0.9777469344448786),\n                        (8, 0.30568317284628654, 0.47615442362120625, -0.46058659012509917)],\n                       [(5, 2), (1, 4), (9, 13), (12, 8)],\n                       [(4, 0.5595532320705697, -0.9382131809591647, -0.4023344726901694),\n                        (5, 0.49999999999999994, 0.9579714106467603, -0.15398414038821606),\n                        (2, 0.23753846872505302, 0.34674553885222426, -0.0620495710161193),\n                        (1, 0.6888649043845242, -0.5060248221136647, 0.08402117771921447),\n                        (8, 0.5595532320705697, -0.9382131809591647, -0.4023344726901694),\n                        (9, 0.49999999999999994, 0.9579714106467603, -0.15398414038821606),\n                        (13, 0.23753846872505302, 0.34674553885222426, -0.0620495710161193),\n                        (12, 0.6888649043845242, -0.5060248221136647, 0.08402117771921447)],\n                       [(4, 5), (2, 1), (8, 9), (13, 12)],\n                       [(5, 0.49999999999999994, 0.9575369521221349, 0.803987270258544),\n                        (2, 0.6383222145074643, 0.33638096074046486, 0.3433691319569282),\n                        (1, 0.49999999999999944, 0.17442555331704923, -0.4220036443944504),\n                        (4, 1.0, 0.0, -0.3298581846816502),\n                        (9, 0.49999999999999994, 0.9575369521221349, 0.803987270258544),\n                        (13, 0.6383222145074643, 0.33638096074046486, 0.3433691319569282),\n                        (12, 0.49999999999999944, 0.17442555331704923, -0.4220036443944504),\n                        (8, 1.0, 0.0, -0.3298581846816502)],\n                       [(5, 2), (1, 4), (9, 13), (12, 8)],\n                       [(4, 1.485031169328526e-16, -0.25837047674816205, 0.39601744798793204),\n                        (5, 0.5, 0.5353934994993023, -0.04805127607910631),\n                        (2, 0.49999999999999967, 0.7435078383024374, -0.2455341780811997),\n                        (1, 0.4546007179287068, 0.6136707732082303, -0.9960384068221335),\n                        (8, 1.485031169328526e-16, -0.25837047674816205, 0.39601744798793204),\n                        (9, 0.5, 0.5353934994993023, -0.04805127607910631),\n                        (13, 0.49999999999999967, 0.7435078383024374, -0.2455341780811997),\n                        (12, 0.4546007179287068, 0.6136707732082303, -0.9960384068221335)],\n                       [(4, 5), (2, 1), (8, 9), (13, 12)],\n                       [(5, 1.1856887587474516e-16, 0.009773085015903238, 0.8004265897893221),\n                        (2, 0.8858944659256832, 0.202434171219939, -0.5020263397787612),\n                        (1, 0.19504058663172452, 0.5914907098316629, 0.9151847065591654),\n                        (4, 0.29782301439575243, -0.20754376111225703, 0.06747145962352313),\n                        (9, 1.1856887587474516e-16, 0.009773085015903238, 0.8004265897893221),\n                        (13, 0.8858944659256832, 0.202434171219939, -0.5020263397787612),\n                        (12, 0.19504058663172452, 0.5914907098316629, 0.9151847065591654),\n                        (8, 0.29782301439575243, -0.20754376111225703, 0.06747145962352313)],\n                       [(5, 2), (1, 4), (9, 13), (12, 8)],\n                       [(5, 0.21266361194798092, 0.18125553061813982, 0.997115308436098),\n                        (4, 0.5000000000000004, -0.3609593977363983, -0.9374449540779946),\n                        (1, 0.5271017512479523, -0.966352273750634, -0.03364772624936552),\n                        (2, 4.3497884521439305e-16, -0.7139587070651552, -0.7829653673588395),\n                        (9, 0.21266361194798092, 0.18125553061813982, 0.997115308436098),\n                        (8, 0.5000000000000004, -0.3609593977363983, -0.9374449540779946),\n                        (12, 0.5271017512479523, -0.966352273750634, -0.03364772624936552),\n                        (13, 4.3497884521439305e-16, -0.7139587070651552, -0.7829653673588395)],\n                       [(5, 4), (1, 0), (9, 8), (12, 11)],\n                       [(5, 0.4999999999999824, -0.29919772885013596, -0.32162910462782035),\n                        (4, 0.500000000000002, 0.3758195456300396, -0.2237315757103544),\n                        (1, 0.49999999999999983, 0.0, -0.5),\n                        (0, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (9, 0.4999999999999824, -0.29919772885013596, -0.32162910462782035),\n                        (8, 0.500000000000002, 0.3758195456300396, -0.2237315757103544),\n                        (12, 0.49999999999999983, 0.0, -0.5),\n                        (11, 0.3040867239846963, 0.0, -0.5000000000000002)],\n                       [(5, 6), (2, 1), (9, 10), (13, 12)],\n                       [(5, 0.5000000000000104, 0.7374333560946903, -0.6208268334779563),\n                        (6, 0.3040867239846962, 0.6612280619162106, -0.08914082204925244),\n                        (1, 0.8040867239846964, 0.0, 0.5),\n                        (2, 1.0, 0.0, 0.5),\n                        (9, 0.5000000000000104, 0.7374333560946903, -0.6208268334779563),\n                        (10, 0.3040867239846962, 0.6612280619162106, -0.08914082204925244),\n                        (12, 0.8040867239846964, 0.0, 0.5),\n                        (13, 1.0, 0.0, 0.5)],\n                       [(5, 4), (1, 0), (9, 8), (12, 11)],\n                       [(5, 0.75, -0.6782507102448094, -0.8833934448680876),\n                        (4, 0.4999999999999995, 0.12674214319260602, 0.15208800243486692),\n                        (1, 0.75, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (9, 0.75, -0.6782507102448094, -0.8833934448680876),\n                        (8, 0.4999999999999995, 0.12674214319260602, 0.15208800243486692),\n                        (12, 0.75, 0.0, 0.5),\n                        (11, 0.5, 0.0, 0.5)],\n                       [(5, 6), (2, 1), (9, 10), (13, 12)],\n                       [(5, 0.5005268894219383, 0.7511005823069626, -0.061117218985732524),\n                        (6, 0.5000000000000102, 0.2800353060019817, -0.4279127601330417),\n                        (1, 0.5000000000000001, 0.0, -0.5),\n                        (2, 0.5, 0.0, 0.5),\n                        (9, 0.5005268894219383, 0.7511005823069626, -0.061117218985732524),\n                        (10, 0.5000000000000102, 0.2800353060019817, -0.4279127601330417),\n                        (12, 0.5000000000000001, 0.0, -0.5),\n                        (13, 0.5, 0.0, 0.5)],\n                       [(5, 4), (1, 0), (9, 8), (12, 11)],\n                       [(5, 0.5000000000000001, -0.3275158685726778, 0.43998379939466253),\n                        (4, 0.5000000000000003, 0.7455961598985248, 0.7788301456274731),\n                        (1, 0.25000000000000006, 0.0, 0.5),\n                        (0, 1.0, 0.0, 0.5),\n                        (9, 0.5000000000000001, -0.3275158685726778, 0.43998379939466253),\n                        (8, 0.5000000000000003, 0.7455961598985248, 0.7788301456274731),\n                        (12, 0.25000000000000006, 0.0, 0.5),\n                        (11, 1.0, 0.0, 0.5)],\n                       [(5, 6), (2, 1), (9, 10), (13, 12)],\n                       [(5, 1.0308214162014585e-14, -0.4910261758928436, 0.61246083991116),\n                        (1, 0.5, 0.0, -0.5),\n                        (9, 1.0308214162014585e-14, -0.4910261758928436, 0.61246083991116),\n                        (12, 0.5, 0.0, -0.5),\n                        (6, 0.5000000000000001, -0.07833700092571938, -0.14787748664623868),\n                        (2, 0.5, 0.0, 0.5),\n                        (10, 0.5000000000000001, -0.07833700092571938, -0.14787748664623868),\n                        (13, 0.5, 0.0, 0.5)],\n                       [(5, 4), (1, 0), (9, 8), (12, 11)],\n                       [(5, 0.9999999674848218, 0.6523437121679241, -0.3785582224053705),\n                        (4, 0.9992548329781764, -0.7506407930979386, -0.9755736944740105),\n                        (1, 0.5000000000000001, 0.0, 0.5),\n                        (9, 0.9999999674848218, 0.6523437121679241, -0.3785582224053705),\n                        (8, 0.9992548329781764, -0.7506407930979386, -0.9755736944740105),\n                        (12, 0.5000000000000001, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],\n           \'qubits\': [(2, 5),\n                      (3, 5),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 4),\n                      (6, 5),\n                      (6, 6)]},\n (14, 6): {\'method\': \'full-ladder count seed + joint decoder cores\',\n           \'moments\': [[(4, 0.4693683757935015, 0.0, -0.3727443053643906),\n                        (5, 0.26161164497266604, 0.0, 0.050637160456765695),\n                        (8, 0.5508652978889468, 0.0, 0.2392663068429361),\n                        (9, 0.3407473804350728, 0.0, 0.8207344818745221),\n                        (7, 0.5314957750505971, 1.0, 0.36859409172974755),\n                        (2, 0.5000000000000004, 0.0, -0.20251959709327183),\n                        (13, 0.5000000000000004, 0.0, -0.20251959709327183),\n                        (1, 0.5, 1.0, -0.377500967254607),\n                        (12, 0.5, 1.0, -0.377500967254607),\n                        (0, 0.5, 1.0, 0.5),\n                        (11, 0.5, 1.0, 0.5),\n                        (6, 0.3040867239846966, 0.0, -0.5745192213521528),\n                        (10, 0.3040867239846966, 0.0, -0.5745192213521528)],\n                       [(4, 5), (8, 9), (3, 7)],\n                       [(4, 0.24586190582054246, 0.0, 0.9981579246220915),\n                        (8, 0.5161703940433506, 8.881784197001252e-16, 0.5096558652447831),\n                        (5, 0.23202847318659447, 0.0, -0.46476357163737436),\n                        (9, 0.15041455533187123, 0.0, 0.36459738820484944),\n                        (7, 0.5314957750505966, 0.0, 0.36859409172974655),\n                        (3, 0.4999999999999988, 0.0, -0.7846103652883083)],\n                       [(4, 8), (5, 9)],\n                       [(4, 0.1629718389756887, 0.0, 0.36790709826803214),\n                        (5, 0.7273102506721383, 0.0, 0.2744193570582145),\n                        (8, 0.9740360915568254, 0.0, -0.36603420341614834),\n                        (9, 0.9009894494923816, 0.0, -0.618554377115297)],\n                       [(4, 5), (8, 9)],\n                       [(5, 0.26255073903045506, 0.0, 0.7465020220716543),\n                        (9, 0.9051126063656172, 5.551115123125783e-17, 0.3469811997275872),\n                        (8, 0.9663707424185195, 0.0, -0.1695782148497118),\n                        (4, 0.4430130717642777, 1.0, -0.5688365244967677)],\n                       [(5, 9), (7, 8), (3, 4)],\n                       [(4, 0.4999999999999998, 0.0, -0.7156213447310924),\n                        (8, 0.007380759634877609, 0.0, 0.49936091673989924),\n                        (5, 0.3855378563150733, 0.0, -0.10519068660272435),\n                        (9, 0.49928891967004163, 0.0, 0.057646941776138994),\n                        (3, 0.5000000000000022, 0.0, 0.21538963471169303),\n                        (7, 0.5000000000000002, 0.9999999999999999, 0.5281412032422539)],\n                       [(4, 8), (5, 2), (9, 13), (3, 7)],\n                       [(7, 0.5, 0.0, -0.47185879675774633),\n                        (8, 0.49999999999999917, 0.0, 0.29591720099806995),\n                        (4, 0.4153279096772517, 1.0, 0.7843786552689074),\n                        (5, 0.6822968314790296, 1.0, -0.9580208466092155),\n                        (9, 0.6822968314790296, 1.0, 0.0733624345025693),\n                        (2, 0.2509322168477317, 0.0, -0.7949290191088039),\n                        (13, 0.2509322168477317, 0.0, -0.7949290191088039)],\n                       [(7, 8), (3, 4)],\n                       [(4, 5),\n                        (8, 9),\n                        (3, 0.49999999999999944, 0.0, 0.5764691935089019),\n                        (7, 0.49999999999999956, 0.9999999999999999, 0.8135078318461024)],\n                       [(4, 0.5728573590437841, 0.0, 0.7326924406246498),\n                        (5, 0.5000000000000001, 0.0, 0.8197314553173387),\n                        (9, 0.5, 0.0, -0.14888526357087617),\n                        (8, 0.5728573590437841, 0.0, 0.7442309863538124)],\n                       [(3, 4), (5, 2), (9, 13), (7, 8)],\n                       [(3, 0.4178598011240009, 0.0, 0.07646919350890169),\n                        (5, 0.2673990199988756, 0.0, -0.1802685446826615),\n                        (2, 0.5826592798249117, 0.0, 0.6305613443031177),\n                        (9, 0.2673990199988756, 0.0, 0.8511147364291234),\n                        (13, 0.5826592798249117, 0.0, 0.6305613443031177),\n                        (7, 0.4178598011240006, 1.0, 0.3135078318461026)],\n                       [(3, 4), (5, 2), (9, 13), (7, 8)],\n                       [(4, 0.3856735769590187, 0.0, -0.40107812877847326),\n                        (5, 0.5505347324801626, 1.0, 0.3197314553173387),\n                        (8, 0.3856735769590187, 0.0, -0.38953958304931047),\n                        (9, 0.5505347324801628, 1.0, -0.6488852635708764),\n                        (3, 0.4999999999999997, 0.0, 0.5764691935089017),\n                        (2, 0.3689376239568078, 0.0, 0.7719771085165397),\n                        (13, 0.3689376239568078, 0.0, 0.7719771085165397),\n                        (7, 0.4999999999999997, 0.9999999999999998, 0.8135078318461031)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.46333492445932517, 0.0, 0.5968581781231013),\n                        (5, 0.3213111929430338, 1.0, 0.8197314553173383),\n                        (8, 0.46333492445932517, 4.440892098500626e-16, 0.6083967238522641),\n                        (9, 0.3213111929430338, 1.0, -0.14888526357087617)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.4372106511952734, 0.0, -0.2387659971493319),\n                        (5, 0.566656815524444, 0.0, -0.6802685446826611),\n                        (9, 0.5666568155244442, 1.1102230246251565e-16, 0.3511147364291237),\n                        (8, 0.43721065119527336, 0.0, -0.22722745142016887)],\n                       [(3, 4), (5, 2), (9, 13), (7, 8)],\n                       [(4, 0.1039903575622847, 0.0, -0.8396672082491146),\n                        (5, 0.5850425523318399, 1.0, 0.8197314553173387),\n                        (8, 0.1039903575622847, 0.0, 0.1718713374800483),\n                        (9, 0.5850425523318398, 0.9999999999999998, -0.14888526357087617),\n                        (3, 0.5, 1.1102230246251565e-16, 0.22652345215961833),\n                        (2, 0.8311737009895825, 0.0, -0.0841139332337324),\n                        (13, 0.8311737009895825, 0.0, -0.0841139332337324),\n                        (7, 0.5, 1.0, 0.16345357319538595)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.43415359559636596, 0.0, -0.6079925371078989),\n                        (5, 0.49999999999999956, 0.0, 0.8197314553173394),\n                        (9, 0.49999999999999967, 1.1102230246251565e-16, -0.14888526357087617),\n                        (8, 0.43415359559636596, 5.551115123125783e-17, 0.40354600862126355)],\n                       [(3, 4), (5, 2), (9, 13), (7, 8)],\n                       [(4, 0.4999999999999997, 0.0, 0.195890748031175),\n                        (5, 0.5998657277943455, 1.0, -0.08044473113945028),\n                        (8, 0.4999999999999998, 0.0, 0.20742929376033714),\n                        (9, 0.5998657277943454, 1.0, 0.9509385499723342),\n                        (3, 0.5530742553856172, 0.0, 0.726523452159618),\n                        (7, 0.5530742553856176, 0.9999999999999998, 0.6634535731953859)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.5000000000000004, 0.0, 0.6958907480311751),\n                        (5, 2.2204460492503123e-16, 0.9999999999999999, 0.9939837599015393),\n                        (8, 0.5000000000000004, 0.0, 0.7074292937603368),\n                        (9, 2.220446049250313e-16, 1.0, 0.025367041013324987)],\n                       [(3, 4), (5, 2), (7, 8), (9, 13)],\n                       [(2, 0.6339658219122923, 0.0, 0.8651689462862566),\n                        (4, 0.5, 0.0, -0.3041092519688249),\n                        (5, 0.5, 1.0, -0.2559732767299776),\n                        (8, 0.5, 0.0, 0.707429293760337),\n                        (9, 0.5000000000000002, -0.9999999999999998, 0.7754100043818073),\n                        (13, 0.6339658219122923, 0.0, 0.8651689462862566),\n                        (7, 0.5, -0.3233355162343905, 0.6634535731953859),\n                        (3, 0.5, 0.613594604801377, 0.7265234521596187)],\n                       [(2, 1), (4, 5), (8, 9), (13, 12)],\n                       [(1, 0.03592107551278813, 1.0, -0.8775009672546068),\n                        (4, 0.5991006024208342, 0.0, -0.46630906173637676),\n                        (5, 0.5, 0.9999999999999999, -0.25597327672997805),\n                        (2, 0.9068032572794498, 1.1102230246251565e-16, 0.23132164282033318),\n                        (9, 0.5, 0.9999999999999998, 0.7754100043818077),\n                        (13, 0.9068032572794498, 1.1102230246251565e-16, 0.23132164282033318),\n                        (12, 0.03592107551278813, 1.0, -0.8775009672546068),\n                        (8, 0.5991006024208342, 4.440892098500626e-16, 0.5452294839927856)],\n                       [(1, 4), (5, 2), (9, 13), (12, 8)],\n                       [(4, 0.6983902210776689, 1.1102230246251565e-16, -0.25472334875083313),\n                        (8, 0.6983902210776689, 0.0, 0.7568151969783297),\n                        (2, 0.4457229604434565, 0.0, 0.01261445611041534),\n                        (1, 0.751811597304503, -0.9999999999999996, 0.6224990327453925),\n                        (13, 0.4457229604434565, 0.0, 0.01261445611041534),\n                        (12, 0.751811597304503, -0.9999999999999996, 0.6224990327453925)],\n                       [(4, 5), (8, 9), (2, 1), (13, 12)],\n                       [(1, 0.5, 1.0, -0.3775009672546079),\n                        (4, 0.6251415065033563, 0.0, -0.10711191447571),\n                        (5, 0.0, -0.3255499447294259, 0.0),\n                        (2, 0.7856407379834285, 0.0, -0.41370794677841705),\n                        (9, 0.0, -0.3255499447294259, 0.0),\n                        (13, 0.7856407379834285, 0.0, -0.41370794677841705),\n                        (12, 0.5, 1.0, -0.3775009672546079),\n                        (8, 0.6251415065033563, 0.0, 0.9044266312534524)],\n                       [(1, 4), (5, 2), (9, 13), (12, 8)],\n                       [(4, 0.0, -0.7191027380962189, 0.0),\n                        (5, 0.5, 0.0, 0.4184767785405973),\n                        (8, 0.0, 0.2808972619037806, 0.0),\n                        (9, 0.5, 0.0, -0.5501399403476175),\n                        (2, 0.4949978217082433, -2.220446049250313e-16, 0.08629205322158295),\n                        (1, 0.4999999999999998, 1.0, -0.8814144405744395),\n                        (13, 0.4949978217082433, -2.220446049250313e-16, 0.08629205322158295),\n                        (12, 0.4999999999999998, 1.0, -0.8814144405744395)],\n                       [(4, 5), (8, 9), (2, 1), (13, 12)],\n                       [(1, 4),\n                        (12, 8),\n                        (5, 0.5, -0.9999999999999996, -0.5815232214594028),\n                        (2, 0.5, 0.0, -0.4475413783399338),\n                        (9, 0.5, -0.9999999999999998, 0.44986005965238246),\n                        (13, 0.5, 0.0, -0.4475413783399338)],\n                       [(5, 2),\n                        (9, 13),\n                        (1, 0.2887068963335278, 0.0, 0.899671023811214),\n                        (12, 0.2887068963335278, 0.0, 0.899671023811214)],\n                       [(2, 0.8015319629186126, 0.0, -0.4475413783399338),\n                        (13, 0.8015319629186126, 0.0, -0.4475413783399338),\n                        (5, 0.5, 1.0, 0.3025359392021816),\n                        (9, 0.5, 1.0, -0.6660807796860331)],\n                       [(2, 1), (13, 12), (4, 5), (8, 9)],\n                       [(1, 0.10822807679614659, 0.0, 0.5488202087084497),\n                        (4, 0.5000000000000003, 0.0, -0.5771760082358401),\n                        (12, 0.10822807679614659, 0.0, 0.5488202087084497),\n                        (8, 0.5000000000000004, 0.0, -0.5656374625066771),\n                        (5, 0.0, 1.0, 0.0),\n                        (2, 0.4999999999999993, 8.881784197001252e-16, -0.947541378339934),\n                        (9, 0.0, 1.0, 0.0),\n                        (13, 0.4999999999999993, 8.881784197001252e-16, -0.947541378339934)],\n                       [(1, 4), (12, 8), (5, 2), (9, 13)],\n                       [(1, 0.18023611392199, 0.0, 0.548820208708451),\n                        (5, 0.5663325445329529, 0.9999999999999998, 0.8025359392021811),\n                        (4, 0.4999999999999998, 0.0, -0.577176008235841),\n                        (9, 0.566332544532953, 0.9999999999999999, -0.16608077968603308),\n                        (8, 0.4999999999999998, 0.0, -0.565637462506678),\n                        (12, 0.18023611392199, 0.0, 0.548820208708451)],\n                       [(1, 0), (5, 4), (9, 8), (12, 11)],\n                       [(1, 0.5, 0.0, -0.9511797912915487),\n                        (5, 0.49999999999998246, 0.0, -0.6974640044798788),\n                        (9, 0.4999999999999824, 1.1102230246251565e-16, 0.3339192766319067),\n                        (12, 0.5, 0.0, -0.9511797912915487),\n                        (0, 0.3040867239846962, 1.0, -0.5000000000000002),\n                        (4, 0.5000000000000022, 0.0, 0.0554890808300641),\n                        (8, 0.5000000000000022, 0.0, 0.06702762655922712),\n                        (11, 0.3040867239846962, 1.0, -0.5000000000000002)],\n                       [(2, 1), (5, 6), (9, 10), (13, 12)],\n                       [(1, 0.8040867239846965, 8.881784197001252e-16, -0.9511797912915487),\n                        (5, 0.5000000000000102, 1.0, -0.6974640044798788),\n                        (9, 0.5000000000000102, 0.9999999999999999, 0.33391927663190757),\n                        (12, 0.8040867239846965, 8.881784197001252e-16, -0.9511797912915487),\n                        (6, 0.30408672398469627, 0.0, -0.5745192213521484),\n                        (10, 0.30408672398469627, 0.0, -0.5745192213521484),\n                        (2, 0.0, 0.6350392865765455, 0.0),\n                        (13, 0.0, 0.6350392865765455, 0.0)],\n                       [(1, 0), (5, 4), (9, 8), (12, 11)],\n                       [(5, 0.75, -2.220446049250313e-16, 0.3025360280352998),\n                        (9, 0.75, 0.0, -0.6660806908529147),\n                        (1, 0.7499999999999999, 8.881784197001252e-16, -0.9511797912915487),\n                        (12, 0.7499999999999999, 8.881784197001252e-16, -0.9511797912915487),\n                        (0, 0.5, 1.0, 0.5),\n                        (4, 0.4999999999999995, 0.0, 0.055489048314883216),\n                        (8, 0.49999999999999944, 0.0, 0.0670275940440464),\n                        (11, 0.5, 1.0, 0.5)],\n                       [(5, 6), (9, 10), (2, 1), (13, 12)],\n                       [(1, 0.5, -2.220446049250313e-16, 0.04882020870845155),\n                        (5, 0.5005268894219381, 1.0, -0.19693703583753597),\n                        (9, 0.500526889421938, 0.9999999999999998, 0.8344462452742494),\n                        (12, 0.5, -2.220446049250313e-16, 0.04882020870845155),\n                        (6, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515),\n                        (10, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515),\n                        (2, 0.5, 1.0, -0.47238403480239244),\n                        (13, 0.5, 1.0, -0.47238403480239244)],\n                       [(1, 0), (5, 4), (9, 8), (12, 11)],\n                       [(5, 0.5, 0.0, -0.4469365997641035),\n                        (9, 0.5, 0.0, 0.5844466813476823),\n                        (1, 0.25, 0.0, -0.9511797912915487),\n                        (12, 0.25, 0.0, -0.9511797912915487),\n                        (4, 0.5000000000000004, -2.220446049250313e-16, -0.44451095168511756),\n                        (8, 0.5000000000000004, 1.1102230246251565e-16, -0.43297240595595454)],\n                       [(5, 6), (9, 10), (2, 1), (13, 12)],\n                       [(1, 0.5, 0.0, 0.04882020870845155),\n                        (12, 0.5, 0.0, 0.04882020870845155),\n                        (5, 1.0214051826551437e-14, 1.0, 0.05305630932507166),\n                        (9, 1.0214051826551442e-14, 1.0, -0.9155604095631429),\n                        (2, 0.5, 0.9723840348023929, -0.4723840348023929),\n                        (6, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (10, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (13, 0.5, 0.9723840348023929, -0.4723840348023929)],\n                       [(1, 0), (12, 11), (5, 4), (9, 8)],\n                       [(0, 1.0, 0.0, 0.0),\n                        (11, 1.0, 0.0, 0.0),\n                        (5, 0.999999967484822, 0.7207220668611685, -0.4469365770986149),\n                        (9, 0.999999967484822, -0.3106612142506173, 0.5844467040131704),\n                        (1, 0.4999999999999998, 0.45117979129154895, -0.951179791291549),\n                        (4, 0.0007451670218237006, 0.21829646411315906, -0.9445109516851093),\n                        (8, 0.0007451670218237004, 0.20675791838399604, -0.9329724059559461),\n                        (12, 0.4999999999999998, 0.45117979129154895, -0.951179791291549)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],\n           \'qubits\': [(2, 5),\n                      (3, 5),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 4),\n                      (6, 5),\n                      (6, 6)]},\n (14, 7): {\'method\': \'full-ladder count seed + joint decoder cores\',\n           \'moments\': [[(3, 0.17822086994158945, 0.0, -0.3580439563386024),\n                        (7, 0.7887543778507795, 5.551115123125783e-17, 0.3988902970543859),\n                        (8, 0.8295385773488477, 0.0, 0.23567945389510814),\n                        (4, 0.8271972559865568, 0.0, 0.900675622191125),\n                        (5, 0.4784749461845292, 0.0, -0.9421760829658345),\n                        (13, 0.5, 0.0, -0.20251771408753827),\n                        (2, 0.5, 0.0, -0.20251771408753827),\n                        (12, 0.4999999999999993, 0.0, -0.3649528224832399),\n                        (1, 0.4999999999999993, 0.0, -0.3649528224832399),\n                        (0, 0.5, 0.0, 0.5),\n                        (11, 0.5, 0.0, 0.5),\n                        (10, 0.30408672398469655, 0.0, -0.5745192213521528),\n                        (6, 0.30408672398469655, 0.0, -0.5745192213521528)],\n                       [(7, 8), (3, 4)],\n                       [(8, 0.482214901062441, 0.0, -0.5730493408354187),\n                        (4, 0.4763451894165355, 0.0, 0.19292052814170585),\n                        (3, 0.8658604633376181, 0.0, 0.29942355761653205),\n                        (7, 0.20911492682496283, 0.0, -0.622129504711858)],\n                       [(8, 9), (4, 5)],\n                       [(9, 0.49999999999999906, 0.0, -0.6259387316945164),\n                        (8, 0.9368237683926718, 0.0, 0.347616964624885),\n                        (4, 0.5972118199777512, 5.551115123125783e-17, 0.0723326692006156),\n                        (5, 0.8968037881267981, 0.0, 0.05782391703416567)],\n                       [(3, 4), (5, 9), (7, 8)],\n                       [(9, 0.028576985383623073, 0.0, -0.1259387316945415),\n                        (7, 0.47132528349926955, 0.0, -0.3698864070276562),\n                        (8, 0.8443366831748911, 0.0, -0.4465082890519403),\n                        (3, 0.47214430111529504, 0.0, -0.31442780633034406),\n                        (5, 0.4999999999999977, 0.0, -0.9707530683494574)],\n                       [(5, 2), (4, 8), (9, 13), (3, 7)],\n                       [(7, 0.5000000000000049, 0.0, -0.47049834920843203),\n                        (8, 0.6858364037898694, 0.0, 0.3967025069783072),\n                        (5, 0.4025088854789329, 0.0, -0.655971189964879),\n                        (4, 0.5000000000000004, 0.0, 0.3864962654107464),\n                        (9, 0.40250888547893265, 0.0, 0.6888431466900595),\n                        (13, 0.23824984739858215, 0.0, 0.048025566717950996),\n                        (2, 0.23824984739858215, 0.0, 0.048025566717950996)],\n                       [(3, 4), (7, 8)],\n                       [(8, 9),\n                        (4, 5),\n                        (3, 0.5, 4.440892098500626e-16, 0.7762128674505466),\n                        (7, 0.5, 0.0, -0.5611390229893229)],\n                       [(9, 0.5000000000000018, 0.0, -0.8111568533099385),\n                        (4, 0.49999999999999956, 0.0, -0.11350373458925267),\n                        (8, 0.49999999999999956, 0.0, -0.6032974930216866),\n                        (5, 0.5000000000000016, 0.0, 0.8440288100351228)],\n                       [(9, 13), (5, 2), (3, 4), (7, 8)],\n                       [(9, 0.4999999999999993, 0.0, 0.1888431466900617),\n                        (13, 0.8228387639951986, 5.551115123125783e-17, 0.2487913141777695),\n                        (5, 0.4999999999999994, 0.0, -0.15597118996487724),\n                        (3, 0.5000000000000003, 0.0, -0.7237871325494534),\n                        (7, 0.5000000000000004, -2.220446049250313e-16, -0.06113902298932272),\n                        (2, 0.8228387639951986, 5.551115123125783e-17, 0.2487913141777695)],\n                       [(5, 2), (7, 8), (9, 13), (3, 4)],\n                       [(5, 0.4568853839401341, 5.551115123125783e-17, 0.3440288100351253),\n                        (8, 0.36532098276163205, 4.440892098500626e-16, 0.7276580796814924),\n                        (4, 0.36532098276163205, 4.440892098500626e-16, 0.5555406927075683),\n                        (9, 0.4568853839401341, 0.0, 0.6888431466900642),\n                        (7, 0.5, 0.0, -0.5611390229893227),\n                        (13, 0.6082288892935703, 0.0, 0.24879131417776595),\n                        (2, 0.6082288892935703, 0.0, 0.24879131417776595),\n                        (3, 0.5, 0.0, 0.7762128674505466)],\n                       [(8, 9), (4, 5)],\n                       [(9, 0.5409187240915672, 0.0, 0.1888431466900628),\n                        (4, 0.1597931613825843, 0.0, -0.5197859886699094),\n                        (5, 0.540918724091567, 0.0, 0.8440288100351236),\n                        (8, 0.15979316138258426, 0.0, -0.19701523894103)],\n                       [(4, 5), (8, 9)],\n                       [(9, 0.08208559927524052, 0.0, -0.31115685330993603),\n                        (8, 0.17667382808391463, 0.0, 0.9526317528609547),\n                        (4, 0.17667382808391466, 0.0, 0.33056701952810563),\n                        (5, 0.0820855992752405, 0.0, -0.6559711899648766)],\n                       [(5, 2), (9, 13), (7, 8), (3, 4)],\n                       [(5, 0.4999999999999983, 0.0, 0.8440288100351239),\n                        (4, 0.5000000000000024, 4.440892098500626e-16, -0.9979080579870945),\n                        (8, 0.5000000000000024, 0.0, -0.7188931696238452),\n                        (9, 0.4999999999999983, 0.0, -0.8111568533099378),\n                        (3, 0.5, 0.0, -0.5762185972659861),\n                        (7, 0.5, 0.0, -0.2087075582727902),\n                        (2, 0.2692579739648868, 0.0, -0.751208685822232),\n                        (13, 0.2692579739648868, 0.0, -0.751208685822232)],\n                       [(8, 9), (4, 5), (2, 1), (13, 12)],\n                       [(5, 0.5000000000000007, 0.0, 0.8440288100351234),\n                        (8, 0.35284137925881676, 0.0, -0.7188931696238459),\n                        (9, 0.5000000000000007, 0.0, 0.1888431466900628),\n                        (4, 0.35284137925881676, 4.440892098500626e-16, -0.997908057987094),\n                        (1, 0.3765861726663244, 0.0, -0.864952822483241),\n                        (12, 0.3765861726663244, 0.0, -0.864952822483241)],\n                       [(3, 4), (7, 8), (5, 2), (9, 13)],\n                       [(8, 0.5, 0.0, -0.7188931696238421),\n                        (5, 0.269060447136372, 0.0, 0.3440288100351241),\n                        (4, 0.5, 0.0, 0.0020919420129021926),\n                        (9, 0.269060447136372, 0.0, -0.3111568533099367),\n                        (3, 0.3016886194775282, 0.0, 0.9237814027340143),\n                        (7, 0.3016886194775281, -2.220446049250313e-16, -0.7087075582727904)],\n                       [(4, 5), (8, 9)],\n                       [(4, 0.5000000000000003, 0.0, -0.4979080579870978),\n                        (8, 0.5000000000000004, 0.0, -0.21889316962384275),\n                        (5, 2),\n                        (9, 13)],\n                       [(3, 4),\n                        (7, 8),\n                        (9, 0.5, 0.0, -0.31115685330993914),\n                        (5, 0.5, 0.0, -0.6559711899648781),\n                        (13, 0.564919347967729, 0.0, 0.12537748684409417),\n                        (2, 0.564919347967729, 0.0, 0.12537748684409417)],\n                       [(8, 0.5, -2.220446049250313e-16, -0.21889316962384253),\n                        (4, 0.5, 0.0, 0.5020919420129029),\n                        (3, 0.5, -0.5836310593025975, -0.07621859726598545),\n                        (7, 0.5, 0.048857901704206785, 0.2912924417272096)],\n                       [(8, 9), (4, 5)],\n                       [(5, 0.5, 0.0, -0.6559711899648781),\n                        (9, 0.5, 0.0, 0.6888431466900611),\n                        (8, 0.5000000000000038, 0.0, 0.7811068303761546),\n                        (4, 0.5000000000000038, 0.0, -0.49790805798709314)],\n                       [(9, 13), (1, 4), (5, 2), (12, 8)],\n                       [(1, 0.7664718219180737, 0.0, -0.36495282248324123),\n                        (2, 0.5000000000000004, -2.220446049250313e-16, -0.3746225131559058),\n                        (8, 0.6124202497959055, 0.0, -0.32188291456625073),\n                        (12, 0.7664718219180737, 0.0, 0.635047177516759),\n                        (13, 0.5000000000000004, -2.220446049250313e-16, -0.3746225131559058),\n                        (4, 0.6124202497959057, 0.0, 0.6050816869553111)],\n                       [(8, 9), (4, 5), (13, 12), (2, 1)],\n                       [(1, 0.5, 0.0, -0.36495282248324123),\n                        (8, 0.7237608380026965, 0.0, -0.4353725646380413),\n                        (4, 0.7237608380026964, 0.0, 0.7185713370271023),\n                        (13, 0.8371608839553215, 0.0, 0.17545611282181062),\n                        (2, 0.8371608839553215, 0.0, 0.17545611282181062),\n                        (12, 0.5, 4.440892098500626e-16, 0.6350471775167583)],\n                       [(1, 4), (5, 2), (12, 8), (9, 13)],\n                       [(1, 0.5, 0.0, -0.8649528224832415),\n                        (12, 0.5, 0.0, -0.8649528224832415),\n                        (5, 0.5, 0.0, 0.3440288100351202),\n                        (13, 0.647259141517744, 0.0, -0.6977855638982122),\n                        (2, 0.647259141517744, 0.0, -0.6977855638982122),\n                        (9, 0.5, 8.881784197001252e-16, 0.6888431466900604)],\n                       [(8, 9), (2, 1), (13, 12), (4, 5)],\n                       [(9, 0.5, 2.220446049250313e-16, -0.31115685330994025),\n                        (13, 0.49999999999999956, 0.0, 0.6885061300272752),\n                        (5, 0.5, 0.0, 0.3440288100351221),\n                        (2, 0.49999999999999956, 0.0, 0.6885061300272752),\n                        (12, 8),\n                        (1, 4)],\n                       [(5, 2),\n                        (9, 13),\n                        (12, 0.5, 0.0, -0.03161948914990864),\n                        (1, 0.5, 4.440892098500626e-16, 0.9683805108500918)],\n                       [(9, 0.4999999999999998, 0.0, -0.07337749945143934),\n                        (13, 0.6633380081977305, 0.0, 0.6885061300272752),\n                        (5, 0.4999999999999998, 4.440892098500626e-16, 0.5818081638936208),\n                        (2, 0.6633380081977305, 0.0, 0.6885061300272752)],\n                       [(13, 12), (4, 5), (8, 9), (2, 1)],\n                       [(4, 0.5000000000000004, 0.0, -0.9243270004429134),\n                        (12, 0.8248194291522575, 0.0, 0.46838051085009247),\n                        (8, 0.5000000000000004, 0.0, -0.7924742271680252),\n                        (1, 0.8248194291522575, 0.0, -0.5316194891499075),\n                        (2, 0.5000000000000001, 0.0, 0.18850613002727545),\n                        (13, 0.5000000000000001, 0.0, 0.18850613002727545)],\n                       [(1, 4), (12, 8), (9, 13), (5, 2)],\n                       [(12, 0.18030068686283407, 0.0, -0.531619489149906),\n                        (1, 0.18030068686283407, 0.0, -0.5316194891499064),\n                        (5, 0.5663648310033746, 0.0, 0.08180816389362078),\n                        (8, 0.5, -2.220446049250313e-16, -0.7924742271680252),\n                        (4, 0.5, 0.0, -0.9243270004429134),\n                        (9, 0.5663648310033745, 5.551115123125783e-17, 0.4266225005485586)],\n                       [(12, 11), (1, 0), (9, 8), (5, 4)],\n                       [(12, 0.5, 0.0, -0.03161948914990642),\n                        (1, 0.5, 0.0, -0.03161948914990642),\n                        (11, 0.3040867239846962, 0.0, 0.4999999999999998),\n                        (5, 0.4999999999999826, 0.0, 0.581808220211562),\n                        (0, 0.3040867239846962, 0.0, 0.4999999999999998),\n                        (9, 0.49999999999998246, 0.0, -0.07337744313349992),\n                        (8, 0.5000000000000022, 0.0, -0.15974456516127633),\n                        (4, 0.5000000000000022, 0.0, 0.4429433375503393)],\n                       [(13, 12), (2, 1), (5, 6), (9, 10)],\n                       [(12, 0.8040867239846965, 0.0, -0.03161948914990642),\n                        (1, 0.8040867239846965, 0.0, -0.03161948914990642),\n                        (5, 0.5000000000000102, 0.0, 0.581808220211562),\n                        (9, 0.5000000000000102, 0.0, -0.0733774431334997),\n                        (10, 0.30408672398469627, 0.0, -0.5745192213521484),\n                        (6, 0.30408672398469627, 0.0, -0.5745192213521484)],\n                       [(1, 0), (12, 11), (9, 8), (5, 4)],\n                       [(12, 0.75, 0.0, -0.03161948914990642),\n                        (1, 0.75, 0.0, -0.03161948914990642),\n                        (9, 0.75, 0.0, -0.07337741061832004),\n                        (11, 0.5, 0.0, 0.5),\n                        (5, 0.75, 0.0, -0.4181917472732597),\n                        (0, 0.5, 0.0, 0.5),\n                        (4, 0.49999999999999956, 0.0, 0.442943370065519),\n                        (8, 0.49999999999999956, 0.0, -0.159744597676458)],\n                       [(13, 12), (2, 1), (5, 6), (9, 10)],\n                       [(12, 0.5, 0.0, 0.9683805108500936),\n                        (1, 0.5, 0.0, 0.9683805108500936),\n                        (9, 0.5005268894219383, 0.0, -0.572850474491156),\n                        (13, 0.5, 0.0, 0.028656473458693432),\n                        (5, 0.5005268894219383, 0.0, -0.9176648111460959),\n                        (2, 0.5, 0.0, 0.028656473458693432),\n                        (10, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155),\n                        (6, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155)],\n                       [(1, 0), (12, 11), (5, 4), (9, 8)],\n                       [(12, 0.25000000000000006, 0.0, -0.03161948914990642),\n                        (1, 0.25000000000000006, 0.0, -0.03161948914990642),\n                        (4, 0.5000000000000004, 0.0, 0.9429433700655194),\n                        (5, 0.5, 0.0, 0.8323356249273368),\n                        (9, 0.5, 0.0, 0.17714996158227603),\n                        (8, 0.5000000000000004, 0.0, -0.6597445976764584)],\n                       [(13, 12), (2, 1), (9, 10), (5, 6)],\n                       [(1, 0.5, 0.0, 0.9683805108500936),\n                        (12, 0.5, 0.0, 0.9683805108500936),\n                        (9, 8),\n                        (5, 4),\n                        (2, 0.5, -0.5286564734586934, -0.9713435265413066),\n                        (6, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (13, 0.5, -0.5286564734586934, -0.9713435265413066),\n                        (10, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(1, 0),\n                        (12, 11),\n                        (8, 0.0007451670218237006, 0.43353011010449993, 0.8402554023235496),\n                        (9, 0.999999967484822, -0.9033644944852117, -0.8228500157522343),\n                        (5, 0.999999967484822, -0.5585501578302723, 0.8323356475928256),\n                        (4, 0.9992548329781763, -0.16915785763746127, 0.442943370065511)],\n                       [(12, 0.4999999999999998, -0.4683805108500936, -0.03161948914990642),\n                        (1, 0.5, -0.46838051085009313, -0.03161948914990642),\n                        (0, 1.0, 0.0, 0.5),\n                        (11, 1.0, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],\n           \'qubits\': [(2, 5),\n                      (3, 5),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 4),\n                      (6, 5),\n                      (6, 6)]},\n (15, 4): {\'method\': \'joint small decoder cores + parallel three-site leaves\',\n           \'moments\': [[(5, 0.8209310735358963, 0.0, 0.9908477378181435),\n                        (4, 0.5125741823407423, 0.0, 0.6428343724342964),\n                        (3, 0.4334386871630289, 0.0, -0.7253989113723505),\n                        (7, 0.14285447272611318, 0.0, -0.9023769229096481),\n                        (8, 0.5731522811129205, 0.0, -0.3854608149558829),\n                        (9, 0.49999999999999645, 0.0, 0.2196996590986673),\n                        (10, 0.5, 0.0, -0.07535137591189756),\n                        (2, 0.2271855258285052, 0.0, -0.708982875827169),\n                        (13, 0.2271855258285052, 0.0, -0.708982875827169),\n                        (1, 0.35760927658307967, 0.0, -0.9848107895924711),\n                        (12, 0.35760927658307967, 0.0, -0.9848107895924711),\n                        (0, 0.5, 0.0, 0.5),\n                        (11, 0.30408672398469655, 0.0, -0.5745192213521528),\n                        (14, 0.30408672398469655, 0.0, -0.5745192213521528),\n                        (6, 0.30408672398469655, 0.0, -0.5745192213521528)],\n                       [(4, 5), (7, 8)],\n                       [(4, 0.7175087811197525, -2.220446049250313e-16, 0.17755400160575885),\n                        (8, 0.6561976856126213, 5.551115123125783e-17, 0.08513670081751118),\n                        (5, 0.49361662395985273, 0.0, -0.8072303613859402),\n                        (7, 0.9674297140873604, 0.0, 0.051499784792820635)],\n                       [(3, 4), (7, 8), (5, 9)],\n                       [(4, 0.25953568718845843, 0.0, 0.7811100846210635),\n                        (3, 0.27418031666840825, 0.0, -0.005502650600590853),\n                        (5, 0.3040867239847014, 0.0, -0.2259204745284824),\n                        (7, 0.638514096502743, 0.0, 0.6902604076193861),\n                        (9, 0.45075003958650495, 0.0, -0.9707033835806471),\n                        (8, 0.9288536731195594, 0.0, -0.0505162503472687)],\n                       [(3, 4), (8, 9)],\n                       [(3, 0.525942052306256, 0.0, -0.8786197023281634),\n                        (4, 0.42881632564510985, 4.440892098500626e-16, 0.5113942875895379),\n                        (8, 9)],\n                       [(4, 8), (3, 7), (5, 9)],\n                       [(8, 0.6436038121118814, 0.0, 0.26012168732655105),\n                        (9, 0.7784033208810368, 0.0, -0.7034574540601111),\n                        (7, 0.5000000000000044, 3.3306690738754696e-16, -0.29860229084791445),\n                        (4, 0.49999999999999956, -2.220446049250313e-16, 0.31805172622903144),\n                        (5, 0.500000000000011, 0.0, 0.7740795254715105)],\n                       [(9, 10), (7, 8), (3, 4), (5, 2)],\n                       [(8, 0.15491981575607627, 0.0, 0.0012937704257105498),\n                        (9, 0.7797752110042575, 0.0, 0.4162415620365465),\n                        (10, 0.6238305862596978, 0.0, 0.1477781765161641),\n                        (4, 0.49936650567575985, 0.0, -0.573573503612794),\n                        (5, 0.4999999999999998, 0.0, 0.34487272663196356),\n                        (2, 0.22718552582850488, 0.0, 0.2910171241728308)],\n                       [(8, 9), (3, 4), (5, 2)],\n                       [(8, 0.803176253409799, 0.0, -0.9987062295742897),\n                        (9, 10),\n                        (4, 0.5, 5.551115123125783e-17, 0.4282146693560128),\n                        (2, 0.6354920868801655, -8.881784197001252e-16, 0.5482021483143635)],\n                       [(7, 8),\n                        (9, 0.5814154756595525, 0.0, -0.5837584379634531),\n                        (10, 0.6296060223028559, 0.0, -0.486663789960482),\n                        (3, 4)],\n                       [(8, 0.8031762534097988, 0.0, 0.0012937704257096616), (10, 13), (4, 5)],\n                       [(8, 9),\n                        (13, 0.22718552582850488, 0.0, 0.2910171241728308),\n                        (4, 0.5886108001133223, 0.0, 0.38734127111311634),\n                        (5, 0.680195383285978, 0.0, -0.7102539155379364)],\n                       [(9, 0.16666666666666669, 0.0, 0.41624156203654716),\n                        (8, 0.5490235858977985, 0.0, 0.0012937704257103277),\n                        (4, 5)],\n                       [(9, 10),\n                        (7, 8),\n                        (4, 0.5000000000000004, 0.0, 0.9938474570807516),\n                        (5, 0.499999999999999, 0.0, -0.001775564798738749)],\n                       [(9, 0.14193433482139065, 0.0, 0.41624156203654694),\n                        (8, 0.9019528282044029, 0.0, -0.9987062295742908),\n                        (3, 4),\n                        (5, 2),\n                        (7, 0.5, 1.1102230246251565e-16, -0.15626398788343265)],\n                       [(9, 10),\n                        (5, 0.06708148302124252, 0.0, -0.22846675223689816),\n                        (4, 0.5234132252872342, 5.551115123125783e-17, 0.14228314036170348),\n                        (3, 0.5, 0.0, -0.7721277380113178)],\n                       [(8, 9), (10, 0.4999999999999998, 0.0, -0.7927993070080479), (4, 5)],\n                       [(8, 0.5, 0.0, 0.0012937704257096616),\n                        (9, 0.435202759783167, 0.0, -0.6163584387280552),\n                        (10, 13),\n                        (5, 0.059202003835620236, 0.0, 0.27153324776310184),\n                        (4, 0.33272914584056146, 0.0, -0.9038636396316402)],\n                       [(8, 9), (13, 0.6354920868801655, -8.881784197001252e-16, 0.5482021483143635), (4, 5)],\n                       [(9, 0.5, 0.0, 0.7985752844500635),\n                        (5, 0.08917758998520853, 0.0, 0.39543919214697393),\n                        (4, 0.2501185405482391, 0.0, 0.5961363603683605),\n                        (7, 8)],\n                       [(9, 10), (5, 2), (3, 4), (7, 0.5, -0.2789817815258173, -0.15626398788343265)],\n                       [(8, 9),\n                        (10, 0.680195383285978, 0.0, -0.8479259491779476),\n                        (5, 0.5938461416699012, 0.0, 0.20784085408680597),\n                        (2, 0.021107237782608028, 0.0, -0.9231467397292097),\n                        (3, 0.5000000000000024, 0.8853480893401815, -0.7721277380113218)],\n                       [(9, 0.5886108001133225, 0.0, 0.8394486826929599), (4, 5), (2, 1)],\n                       [(9, 10),\n                        (5, 0.4433968217557545, 0.0, -0.3095403419228302),\n                        (1, 0.8402337422326178, 0.0, 0.9498164599690078),\n                        (2, 0.3147365041110266, 0.0, -0.7244934550958735)],\n                       [(10, 0.4999999999999994, 0.0, 0.8605524015612502),\n                        (9, 0.5000000000000004, 0.0, 0.23294249672532352),\n                        (1, 4),\n                        (5, 2)],\n                       [(8, 9),\n                        (10, 13),\n                        (4, 0.5, 0.0, 0.5654478916998691),\n                        (2, 0.9294528636932586, 8.881784197001252e-16, 0.5567952414001085),\n                        (1, 0.31397581370367184, 0.0, -0.21890167318297582),\n                        (5, 0.6391921844757946, 0.0, 0.2731417751363101)],\n                       [(10, 0.0670814830212425, 0.0, 0.6338612141230897),\n                        (9, 0.5234132252872342, 0.0, -0.9154931865556264),\n                        (8, 0.5, 0.0, -0.04717235031239975),\n                        (4, 5),\n                        (2, 1)],\n                       [(9, 10),\n                        (4, 0.5000000000000004, 0.0, 0.5654478916998675),\n                        (5, 0.5, 5.551115123125783e-17, 0.05820428231541691),\n                        (1, 0.5519644059311344, 0.0, 0.7987523594593542),\n                        (2, 0.33108034381799806, 0.0, 0.05679524140011294)],\n                       [(10, 0.059202003835620236, 0.0, 0.1338612141230912),\n                        (9, 0.33272914584056146, 0.0, 0.1306535934377172),\n                        (4, 5),\n                        (2, 1)],\n                       [(9, 10),\n                        (2, 0.8371247201733047, 0.0, -0.8632896185761176),\n                        (4, 0.24294212102114646, 0.0, -0.31200817348483834),\n                        (1, 0.7831723914001073, 0.0, 0.6373214569840702)],\n                       [(10, 0.08917758998520853, 0.0, -0.742232841493037),\n                        (9, 0.2501185405482391, 4.440892098500626e-16, 0.630653593437716),\n                        (1, 4),\n                        (5, 2)],\n                       [(10, 13),\n                        (8, 9),\n                        (4, 0.3638122834688833, 0.0, 0.8978789025740177),\n                        (5, 0.4960041929285068, 0.0, 0.558204282315417),\n                        (1, 0.49999999999999944, 4.440892098500626e-16, 0.7329995800098472)],\n                       [(10, 0.5938461416699012, 0.0, -0.9298311795532059),\n                        (13, 0.021107237782608028, 0.0, -0.9231467397292097),\n                        (1, 4),\n                        (5, 2),\n                        (8, 0.5000000000000027, -0.8396072983587364, 0.9528276496875958)],\n                       [(13, 12),\n                        (9, 10),\n                        (5, 0.4999999999999998, 0.0, 0.05820428231541758),\n                        (1, 0.7913981257738136, 0.0, 0.7329995800098474),\n                        (4, 0.022839382910483597, 0.0, 0.39787890257401143),\n                        (2, 0.8057964070938461, 0.0, -0.04830535318554707)],\n                       [(12, 0.8402337422326178, 0.0, 0.9498164599690075),\n                        (13, 0.3147365041110266, 0.0, -0.7244934550958735),\n                        (10, 0.4433968217557545, 0.0, -0.4472123755628419),\n                        (4, 5),\n                        (2, 1)],\n                       [(12, 9),\n                        (10, 13),\n                        (4, 0.22716061708951618, 0.0, -0.602121097425983),\n                        (1, 0.75, 0.0, 0.7329995800098463),\n                        (5, 0.5, 0.0, -0.5897405088121834)],\n                       [(12, 0.3139758137036717, 0.0, 0.7810983268170237),\n                        (13, 0.9294528636932586, 8.881784197001252e-16, 0.5567952414001085),\n                        (9, 0.5, 0.0, 0.6613420621062078),\n                        (10, 0.6391921844757946, 0.0, 0.1354697414962982),\n                        (1, 4),\n                        (5, 2)],\n                       [(9, 10),\n                        (13, 12),\n                        (1, 0.9427632774999226, 0.0, -0.7670004199901534),\n                        (5, 0.20292068705287525, 0.0, -0.08974050881218343),\n                        (4, 0.4999999999999997, 0.0, -0.602121097425983)],\n                       [(12, 0.5519644059311344, 0.0, -0.20124764054064626),\n                        (10, 0.5, 0.0, 0.9205322486754051),\n                        (9, 0.5000000000000004, 0.0, 0.6613420621062078),\n                        (13, 0.33108034381799806, 0.0, 0.05679524140011294),\n                        (1, 0),\n                        (5, 4)],\n                       [(9, 10),\n                        (13, 12),\n                        (1, 0.4999999999999998, 0.0, -0.2670004199901532),\n                        (0, 0.3040867239846962, 0.0, -0.5000000000000002),\n                        (5, 0.49999999999998224, 5.551115123125783e-17, 0.410259434869876),\n                        (4, 0.5000000000000018, 0.0, -0.59014568928154)],\n                       [(9, 0.24294212102114648, 0.0, -0.46120187270908697),\n                        (12, 0.7831723914001073, 1.1102230246251565e-16, -0.36267854301593005),\n                        (13, 0.8371247201733047, 0.0, -0.8632896185761176),\n                        (2, 1),\n                        (5, 6)],\n                       [(10, 13),\n                        (12, 9),\n                        (1, 0.8040867239846965, 0.0, 0.7329995800098466),\n                        (5, 0.5000000000000107, 5.551115123125783e-17, 0.4102594348698756),\n                        (6, 0.3040867239846962, 0.0, 0.4254807786478514)],\n                       [(12, 0.49999999999999944, 4.440892098500626e-16, 0.7329995800098472),\n                        (10, 0.4960041929285068, 0.0, 0.42053224867540506),\n                        (9, 0.3638122834688832, 0.0, 0.32891105123205855),\n                        (1, 0),\n                        (5, 4)],\n                       [(10, 13),\n                        (12, 9),\n                        (1, 0.75, 0.0, 0.7329995800098466),\n                        (5, 0.75, 0.0, -0.5897405976453025),\n                        (0, 0.5, 0.0, 0.5),\n                        (4, 0.49999999999999967, 0.0, 0.4098543432336421)],\n                       [(12, 0.7913981257738136, 0.0, -0.26700041999015234),\n                        (13, 0.8057964070938461, 0.0, -0.04830535318554707),\n                        (10, 0.5, 0.0, -0.07946775132459383),\n                        (9, 0.022839382910483597, 0.0, 0.8289110512320632),\n                        (2, 1),\n                        (5, 6)],\n                       [(9, 10),\n                        (13, 12),\n                        (1, 0.5, 0.0, 0.7329995800098474),\n                        (2, 0.5, 2.220446049250313e-16, -0.44342329229517063),\n                        (5, 0.500526889421938, 0.0, -0.09026753377246655),\n                        (6, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155)],\n                       [(9, 0.22716061708951618, 0.0, -0.17108894876794256),\n                        (10, 0.5, 0.0, 0.2725874575478038),\n                        (12, 0.75, 0.0, -0.2670004199901532),\n                        (1, 0),\n                        (5, 4)],\n                       [(10, 13),\n                        (12, 9),\n                        (1, 0.25000000000000006, 0.0, -0.26700041999015234),\n                        (5, 0.5, 0.0, 0.15973203015410053),\n                        (4, 0.5000000000000004, 0.0, -0.09014565676635855)],\n                       [(9, 0.4999999999999998, 0.0, -0.17108894876794167),\n                        (10, 0.20292068705287525, 0.0, 0.7725874575478036),\n                        (13, 0.5, -4.440892098500626e-16, -0.16963777986712802),\n                        (12, 0.5, 0.0, -0.26700041999015367),\n                        (2, 1),\n                        (5, 6)],\n                       [(10, 9),\n                        (13, 12),\n                        (1, 0.5, 1.1102230246251565e-16, -0.26700041999015367),\n                        (5, 4),\n                        (2, 0.5, 0.9434232922951704, 0.5565767077048294),\n                        (6, 0.5, 0.3483047662953691, -0.5745192538673274)],\n                       [(13, 0.4999999999999824, 0.0, -0.16963772354918794),\n                        (10, 0.4999999999999824, 0.0, 0.27258740122986325),\n                        (12, 0.5000000000000022, 0.0, -0.7766661176341456),\n                        (9, 0.5000000000000021, 0.0, -0.1830643569123842),\n                        (1, 0),\n                        (4, 0.9992548329781763, -0.1360688308055824, 0.40985434323363296),\n                        (5, 3.251517832758566e-08, 0.11405348227394096, -0.8402679925113872)],\n                       [(13, 14),\n                        (10, 11),\n                        (1, 0.5, 0.7670004199901537, -0.26700041999015234),\n                        (0, 1.0, 0.0, 0.5)],\n                       [(13, 0.5000000000000102, 1.1102230246251565e-16, -0.16963772354918705),\n                        (10, 0.5000000000000104, 0.0, 0.27258740122986325),\n                        (11, 0.3040867239846962, 0.0, 0.4254807786478514),\n                        (14, 0.30408672398469627, 0.0, -0.5745192213521484)],\n                       [(13, 12), (10, 9)],\n                       [(13, 0.75, 0.0, 0.8303623089659911),\n                        (10, 0.7499999999999999, 0.0, 0.27258736871468464),\n                        (12, 0.4999999999999995, 0.0, -0.7766660851189646),\n                        (9, 0.49999999999999956, 4.440892098500626e-16, 0.816935610572433)],\n                       [(13, 14), (10, 11)],\n                       [(13, 0.500526889421938, 5.551115123125783e-17, 0.33088924509315526),\n                        (10, 0.500526889421938, 0.0, 0.7720604325875211),\n                        (14, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155),\n                        (11, 0.5000000000000102, 5.551115123125783e-17, 0.42548077864785155)],\n                       [(13, 12), (10, 9)],\n                       [(12, 0.5000000000000003, 0.0, -0.27666608511896307),\n                        (9, 0.5000000000000004, 0.0, -0.6830643894275656),\n                        (13, 0.5, -2.220446049250313e-16, 0.08088968116658896),\n                        (10, 0.5, 0.0, 0.0220599965140878)],\n                       [(10, 9), (13, 14)],\n                       [(10, 11),\n                        (13, 12),\n                        (14, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (9, 0.0007451670218237004, -0.5431500981443909, -0.18306438942755787)],\n                       [(10, 3.251517832758566e-08, -0.7482744840860471, 0.022059973848600478),\n                        (12, 0.9992548329781763, -0.9495484024529768, -0.7766660851189726),\n                        (13, 0.999999967484822, 0.19289578593047718, 0.08088970383207572),\n                        (11, 0.5, 0.3483047662953691, -0.5745192538673274)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],\n           \'qubits\': [(2, 5),\n                      (3, 5),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8),\n                      (6, 6),\n                      (6, 7),\n                      (6, 8)]},\n (15, 5): {\'method\': \'joint small decoder cores + parallel three-site leaves\',\n           \'moments\': [[(3, 0.8186973709050703, 0.0, 0.11403812651524681),\n                        (7, 0.35489675432422335, 1.1102230246251565e-16, -0.08029623950639175),\n                        (5, 0.9819640186706098, 0.0, 0.9951578389697471),\n                        (4, 0.6519792949687725, 0.0, -0.33160968696791926),\n                        (8, 0.6135654383257683, 0.0, -0.9899145128841333),\n                        (1, 0.5000000000000004, 0.0, -0.5933937249479115),\n                        (9, 0.5, 1.1102230246251565e-16, -0.177549935141174),\n                        (6, 0.49999999999999895, 0.0, -0.7088499809721285),\n                        (13, 0.5000000000000003, -0.9999999999999996, -0.29375365603915826),\n                        (2, 0.48497757661839347, 0.0, -0.834800922923292),\n                        (12, 0.4249991593149187, 1.0, -0.4160175847282592),\n                        (0, 0.5, 0.0, 0.5),\n                        (11, 0.3040867239846966, 0.0, -0.5745192213521528),\n                        (10, 0.3040867239846966, 1.0, -0.5745192213521528),\n                        (14, 0.3040867239846966, 1.0, -0.5745192213521528)],\n                       [(3, 7), (5, 4)],\n                       [(3, 0.4504225954403706, 0.0, -0.5627118148043007),\n                        (4, 0.7714725038539685, 1.1102230246251565e-16, 0.37056929785118153),\n                        (7, 0.9864550287211351, -4.440892098500626e-16, -0.41120747292333837),\n                        (5, 0.4983635353823899, 0.0, 0.9178056044121616)],\n                       [(3, 4), (7, 8)],\n                       [(1, 3),\n                        (5, 4),\n                        (7, 0.3673050849619639, 0.0, -0.3001307948400589),\n                        (8, 0.17156066914016166, 0.0, 0.8707464832696235)],\n                       [(7, 8),\n                        (3, 0.5, 0.0, 0.54925761103062),\n                        (5, 0.1622415371566619, 0.0, 0.3998304590823016),\n                        (4, 0.4999999999999998, 0.0, 0.45512124094450024),\n                        (1, 0.5000000000000002, 0.0, -0.5933937249479122)],\n                       [(7, 0.53340749132248, 0.0, 0.38128661855910173),\n                        (5, 4),\n                        (8, 0.3835542028856178, 0.0, 0.6890127977929112)],\n                       [(3, 7),\n                        (4, 0.4464829302910055, 0.0, -0.5471149844612948),\n                        (5, 0.4999999999999986, 0.0, 0.4227005465637681)],\n                       [(3, 0.5, 0.0, -0.45074238896938046),\n                        (7, 0.5000000000000004, 0.0, -0.1523111179895551)],\n                       [(3, 4), (7, 8)],\n                       [(3, 0.5, 0.0, 0.5728432243770509),\n                        (4, 0.26626650422028253, 1.1102230246251565e-16, 0.4662409272974104),\n                        (8, 0.5000000000000004, 0.0, -0.41834257086596915),\n                        (7, 0.49999999999999944, 0.0, 0.8476888820104447)],\n                       [(1, 3), (5, 4), (8, 9)],\n                       [(3, 0.4999999999999998, 0.0, 0.5728432243770509),\n                        (8, 0.5750947397234332, 1.0, -0.9773389068046463),\n                        (9, 0.5388320632192303, 1.0, 0.5619412969211144)],\n                       [(3, 7), (4, 8), (9, 5)],\n                       [(3, 0.41174992307908953, 0.0, -0.261046201750851),\n                        (4, 0.25799913242764916, 0.0, 0.20867920002696339),\n                        (7, 0.5000000000000009, 0.0, -0.6502978749550874),\n                        (8, 0.49999999999999906, 0.0, -0.48426088250961774),\n                        (9, 0.6303834566684253, 0.0, 0.49277545208375345)],\n                       [(1, 3), (5, 4), (7, 6), (8, 9)],\n                       [(3, 0.9675670579641595, 0.0, 0.7389537982491325),\n                        (7, 0.4928050422747947, 0.0, 0.17834543968282673),\n                        (9, 0.41798215431769514, 0.0, 0.6067568601922761),\n                        (5, 0.0, -0.9737187863292855, 0.0),\n                        (4, 0.4007325538142628, 1.1102230246251565e-16, 0.3532846448549235),\n                        (8, 0.15819627402434763, 1.0, 0.5157391174903827),\n                        (6, 0.4608483309798826, -2.220446049250313e-16, 0.8309193012950313)],\n                       [(3, 7), (9, 5), (4, 8)],\n                       [(3, 0.5000000000000004, 0.0, 0.7389537982491154),\n                        (7, 0.5596506416568943, 0.0, 0.08924824075730847),\n                        (9, 0.3489180583921312, 0.0, 0.03273134652395049),\n                        (4, 0.6167758176827268, 0.0, 0.6371322082448254),\n                        (8, 0.894326291264272, 1.0, 0.01573911749037954)],\n                       [(3, 7), (9, 5), (4, 8)],\n                       [(7, 0.5000000000000004, 0.0, 0.06814706811724891),\n                        (8, 0.0023237737136505556, 0.0, -0.4842608825097894),\n                        (9, 0.8475447656731139, 0.0, -0.5467426956589625),\n                        (3, 0.12754703959217256, 1.0, -0.41343205867997534),\n                        (5, 2.220446049250313e-16, 1.0, 0.6703956944329073),\n                        (4, 0.32373355791164515, 0.0, 0.7941155189049776)],\n                       [(7, 6), (8, 9), (1, 3), (5, 4)],\n                       [(3, 0.2696276353165252, 0.0, -0.3557876546517209),\n                        (7, 0.16304676742985036, 0.0, -0.9318529318827491),\n                        (4, 0.7790217319903623, 0.0, 0.8191833078881556),\n                        (8, 0.4999999999999998, 0.9999999999999999, 0.5157391174903769),\n                        (9, 0.06802765123714914, 1.0, -0.13349798984654093),\n                        (1, 0.0, -0.31555683986653227, 0.0),\n                        (6, 0.8273677055841601, 0.0, 0.011025144178165858)],\n                       [(3, 7), (4, 8), (9, 5)],\n                       [(3, 0.3174136387017352, 0.0, 0.20229210405345976),\n                        (5, 0.5, 0.0, -0.8275169686047597),\n                        (4, 0.5000000000000001, 0.0, -0.11245629742359009),\n                        (7, 0.704332640729217, 0.0, 0.5681470681172509),\n                        (8, 0.4117206649156042, 0.0, -0.4115518628622936),\n                        (9, 0.6779434348903984, 0.0, 0.31830555961609885)],\n                       [(1, 3), (5, 4), (7, 6), (8, 9)],\n                       [(3, 0.6276139104334912, 0.0, 0.29819921384825365),\n                        (7, 0.5, 4.440892098500626e-16, 0.5681470681172516),\n                        (4, 0.0, -0.16866101281274037, 0.0),\n                        (9, 13),\n                        (6, 0.727371054243577, 0.0, -0.07318377315388713),\n                        (5, 0.4999999999999997, 0.1647349211400116, 0.17248303139523963)],\n                       [(3, 7),\n                        (4, 8),\n                        (9, 0.16814561073940362, 0.0, 0.9089524723729223),\n                        (13, 0.26236271930206223, 0.9999999999999999, 0.8767572090207052)],\n                       [(3, 0.49999999999999895, 0.0, -0.5964670533913965),\n                        (7, 0.529090738553442, 0.0, 0.1987435208483057),\n                        (8, 9)],\n                       [(1, 3),\n                        (7, 6),\n                        (8, 0.527160237449889, 0.0, 0.648329448252023),\n                        (9, 0.49999999999999956, 0.0, 0.38619806600984763)],\n                       [(3, 0.6683089837594859, 0.0, -0.09646705339139605),\n                        (7, 0.49999999999999956, 0.0, -0.807752949098316),\n                        (4, 8),\n                        (9, 13),\n                        (1, 0.5000000000000004, 0.0, -0.3358383687213544),\n                        (6, 0.49999999999999956, 0.0, -0.09727132543917172)],\n                       [(3, 7),\n                        (8, 0.46199100412343963, 0.0, 0.5393832849386844),\n                        (9, 0.05611225401408456, 0.0, -0.6138019339901597),\n                        (4, 0.5, 4.440892098500626e-16, -0.28111731023633),\n                        (13, 0.41148713642119633, 0.9999999999999999, 0.9576745302205008)],\n                       [(3, 0.6578349175114985, 0.0, -0.09646705339139672), (7, 6), (8, 9)],\n                       [(1, 3),\n                        (7, 0.5000000000000009, 0.0, -0.8077529490983171),\n                        (6, 0.6530017079301533, -2.220446049250313e-16, 0.4027286745608292),\n                        (8, 0.21357675869785495, 0.0, 0.07760744671387909),\n                        (9, 0.6201712438034157, 4.440892098500626e-16, 0.8861980660098474)],\n                       [(3, 0.5, 0.0, -0.5965967598759463),\n                        (6, 2),\n                        (8, 9),\n                        (1, 0.4999999999999998, -0.6777125069026235, 0.664161631278646)],\n                       [(3, 7),\n                        (2, 0.5605616196317902, 0.0, -0.8348009229232944),\n                        (8, 0.6381841379922086, 0.0, -0.341297678630621),\n                        (6, 0.5000000000000003, 1.1102230246251565e-16, -0.09727132543918637),\n                        (9, 0.5000000000000013, 0.0, -0.11380193399015237)],\n                       [(3, 0.5, 0.0, -0.5965967598759463), (4, 8), (7, 6), (9, 13)],\n                       [(2, 3),\n                        (6, 0.4311009172299638, 0.0, -0.34731902068777165),\n                        (8, 0.5000000000000004, 0.0, -0.5466392246905043),\n                        (9, 0.35762932882362675, 0.0, 0.7800008567129855),\n                        (7, 0.5, 1.0, -0.5710209152945636),\n                        (4, 0.7056575226810609, 0.0, 0.2188826897636691)],\n                       [(2, 0.5818042231825744, -2.220446049250313e-16, -0.3348009229233022), (8, 9), (3, 7)],\n                       [(6, 2),\n                        (8, 0.49999999999999956, 0.0, 0.9533607753094966),\n                        (9, 13),\n                        (3, 0.2430469681607357, 4.440892098500626e-16, 0.9976272717687469),\n                        (7, 0.5, 1.0, -0.5710209152945636)],\n                       [(4, 8),\n                        (6, 0.03454957128703273, 0.0, 0.8181166636420727),\n                        (2, 0.2894984978675621, 0.0, -0.8348009229233004),\n                        (3, 7),\n                        (9, 0.5, 0.0, -0.08960609517985496),\n                        (13, 0.6867642766408553, 0.0, 0.9231892660394194)],\n                       [(6, 2),\n                        (8, 0.4999999999999997, 0.0, -0.04663922469050341),\n                        (3, 0.3056831728462866, 0.0, 0.6451841930866737),\n                        (13, 12),\n                        (4, 0.4999999999999997, 0.6183352627715819, 0.2188826897636697)],\n                       [(6, 0.9242869536756131, 8.881784197001252e-16, 0.5858061491772155),\n                        (8, 9),\n                        (2, 0.87386354985307, 0.9999999999999998, 0.6651990770766807),\n                        (13, 0.4999999999982148, 0.0, -0.46085812004732873),\n                        (12, 0.4999999999993539, 1.0, 0.08398241527174072)],\n                       [(7, 6),\n                        (2, 3),\n                        (9, 0.5, 0.0, 0.9103939048201459),\n                        (8, 0.8033952045865456, 0.0, -0.6223167683803963)],\n                       [(6, 0.23753846872505302, 0.0, 0.11812387065190871),\n                        (2, 0.6888649043845242, 0.0, -0.8348009229233393),\n                        (9, 13),\n                        (3, 0.5595532320705698, 4.440892098500626e-16, -0.9369135007270495),\n                        (7, 0.4999999999999998, 1.0, -0.07115062177911269),\n                        (12, 8)],\n                       [(6, 2),\n                        (3, 7),\n                        (8, 0.604127332642834, 0.0, -0.6812716691470246),\n                        (9, 0.0, -0.10647507320209026, 0.0),\n                        (13, 0.39194079872774695, 0.0, 0.03914187995248669),\n                        (12, 0.2549082754107861, -0.9999999999999996, 0.5839824152717399)],\n                       [(2, 0.49999999999999956, 0.0, -0.8348009229233395),\n                        (3, 0.0, -0.37270745352194434, 0.0),\n                        (7, 0.5, 0.0, -0.07115062177911291),\n                        (6, 0.6383222145074643, 0.0, 0.17679703477273234),\n                        (8, 9),\n                        (13, 12)],\n                       [(2, 3),\n                        (7, 6),\n                        (13, 0.7588561382101471, 1.1102230246251565e-16, 0.41246845114232455),\n                        (12, 0.49999999999999944, 1.0, -0.4160175847282588),\n                        (8, 0.7449926306166219, 0.0, 0.37125730635508525)],\n                       [(7, 0.5, 5.551115123125783e-17, 0.11927387976110193),\n                        (6, 0.4999999999999997, 0.0, -0.7484872360058605),\n                        (2, 0.4546007179287068, 0.0, 0.4167387613319282),\n                        (9, 13),\n                        (12, 8)],\n                       [(3, 7),\n                        (6, 2),\n                        (8, 0.0, -0.22180782431639257, 0.0),\n                        (9, 0.5, 4.440892098500626e-16, -0.6960811683819472),\n                        (13, 0.5944883203675292, 0.0, 0.11795781417882178),\n                        (12, 0.5000000000000248, 1.0, -0.9160175847284338)],\n                       [(2, 0.19504058663172466, 0.0, -0.2857088984950029),\n                        (3, 0.2978230143957523, 0.0, 0.33538969783665773),\n                        (6, 0.8858944659256832, 5.551115123125783e-17, 0.25151276399414046),\n                        (8, 9),\n                        (13, 12)],\n                       [(2, 3),\n                        (7, 6),\n                        (9, 0.5, 0.0, 0.303918831618053),\n                        (13, 0.5000000000000007, 0.0, 0.5433088941315738),\n                        (12, 8)],\n                       [(2, 0.5271017512479523, 5.551115123125783e-17, 0.1739679588648032),\n                        (7, 0.21266361194798095, 0.0, 0.6192738797611008),\n                        (3, 0.5000000000000003, -4.440892098500626e-16, -0.4619829547526033),\n                        (9, 13),\n                        (12, 0.4999999999988204, 0.0, 0.9173155857195314)],\n                       [(2, 0),\n                        (7, 3),\n                        (9, 0.5, 0.0, 0.165974168600953),\n                        (13, 0.7512924912067074, 0.0, -0.45669110586842443)],\n                       [(2, 0.5, 0.0, 0.6739679588648031),\n                        (7, 0.4999999999999824, 0.0, -0.8807260639209578),\n                        (8, 9),\n                        (13, 12),\n                        (0, 0.30408672398469616, 0.0, -0.5000000000000002),\n                        (3, 0.500000000000002, 0.0, 0.6126898213514362)],\n                       [(6, 2),\n                        (7, 11),\n                        (13, 0.499999999999814, 8.881784197001252e-16, -0.9566911058684238),\n                        (12, 0.7168080708097687, 0.0, 0.417315101508664),\n                        (8, 0.49999962383286367, 5.551115123125783e-17, 0.0671504761089094)],\n                       [(2, 0.8040867239846965, 3.3306690738754696e-16, -0.32603204113519757),\n                        (7, 0.5000000000000102, 0.0, -0.8807260639209578),\n                        (9, 13),\n                        (12, 8),\n                        (11, 0.30408672398469627, 0.0, -0.5745192213521484)],\n                       [(2, 0),\n                        (7, 3),\n                        (9, 0.563432440107209, 0.0, -0.33402583139904607),\n                        (8, 0.5000000000000007, 0.0, 0.06715047610866964),\n                        (13, 0.5, 0.0, -0.3932586657612145),\n                        (12, 0.4999999999999998, 0.0, 0.9173147966175437)],\n                       [(7, 0.75, 0.0, 0.11927396859422035),\n                        (9, 8),\n                        (13, 12),\n                        (2, 0.75, -0.9999999999999996, -0.32603204113519735),\n                        (0, 0.5, 0.0, 0.5),\n                        (3, 0.49999999999999956, 0.0, 0.612689853866617)],\n                       [(7, 11),\n                        (6, 2),\n                        (9, 0.49999999999998235, 5.551115123125783e-17, 0.16597411228301334),\n                        (13, 0.49999999999998246, 4.440892098500626e-16, 0.6067412779208448),\n                        (8, 0.5000000000000022, 0.0, -0.305984643676914),\n                        (12, 0.5000000000000022, 0.0, -0.4558203231680382)],\n                       [(2, 0.5000000000000001, 0.0, 0.6739679588648031),\n                        (7, 0.5005268894219383, 4.440892098500626e-16, -0.3801990952786154),\n                        (9, 10),\n                        (13, 14),\n                        (11, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515),\n                        (6, 0.5, 0.0, -0.23493636038188193)],\n                       [(2, 0),\n                        (7, 3),\n                        (9, 0.5000000000000102, 5.551115123125783e-17, 0.1659741122830124),\n                        (13, 0.5000000000000102, 4.440892098500626e-16, 0.6067412779208448),\n                        (10, 0.30408672398469616, 1.0, -0.5745192213521486),\n                        (14, 0.30408672398469616, 1.0, -0.5745192213521486)],\n                       [(7, 0.5, 0.0, -0.6301986592051827),\n                        (9, 8),\n                        (13, 12),\n                        (2, 0.25000000000000006, -0.9999999999999998, -0.32603204113519735),\n                        (3, 0.5000000000000003, 0.0, -0.8873101461333819)],\n                       [(7, 11),\n                        (6, 2),\n                        (9, 0.75, -2.220446049250313e-16, 0.16597407976783535),\n                        (13, 0.75, 0.0, 0.6067412454056667),\n                        (8, 0.49999999999999956, 0.0, 0.694015323807905),\n                        (12, 0.49999999999999956, 0.0, 0.5441796443167801)],\n                       [(7, 3),\n                        (2, 0.5, 0.9999999999999999, 0.6739679588648031),\n                        (9, 10),\n                        (13, 14),\n                        (11, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (6, 0.5, 0.7349363603818819, 0.7650636396181181)],\n                       [(2, 0),\n                        (9, 0.500526889421938, 0.0, 0.6654471436406704),\n                        (13, 0.500526889421938, 0.0, -0.8937856907214983),\n                        (10, 0.5000000000000103, 1.0, 0.42548077864785155),\n                        (14, 0.5000000000000103, 1.0, 0.42548077864785155),\n                        (3, 0.9992548329781763, -0.3389043414385582, 0.6126898538666086),\n                        (7, 0.999999967484822, 0.9039841263022477, -0.6301986365396945)],\n                       [(9, 8),\n                        (13, 12),\n                        (0, 1.0, 0.0, 0.5),\n                        (2, 0.5, 0.8260320411351971, -0.3260320411351971)],\n                       [(9, 0.5, -2.220446049250313e-16, -0.0845532924327621),\n                        (13, 0.5, 0.0, 0.3562138732050695),\n                        (8, 0.5000000000000004, 0.0, -0.8059846761920946),\n                        (12, 0.5000000000000003, 0.0, -0.9558203556832208)],\n                       [(9, 10), (13, 14)],\n                       [(9, 8),\n                        (13, 12),\n                        (10, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (14, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(8, 0.0007451670218237004, -0.42022981137986193, -0.30598467619208813),\n                        (9, 3.251517832758566e-08, -0.6416611951391955, -0.0845533150982507),\n                        (12, 0.0007451670218237004, -0.27039413188873684, -0.4558203556832119),\n                        (13, 3.251517832758566e-08, 0.9175716392229736, 0.35621385053957977)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8),\n                      (6, 5),\n                      (6, 6),\n                      (6, 7),\n                      (6, 8)]},\n (15, 7): {\'method\': \'joint small decoder cores + parallel three-site leaves\',\n           \'moments\': [[(3, 0.49826870226158543, 0.0, 0.5315948742077734),\n                        (4, 0.5000000000000004, 0.0, -0.32062613310631694),\n                        (7, 0.9467403485941701, 0.0, -0.659891619232204),\n                        (8, 0.4999999999999972, -2.220446049250313e-16, -0.46842942620674144),\n                        (5, 0.4999999999999989, 0.0, 0.30275883571657247),\n                        (1, 0.8246604806561132, 0.0, -0.6467565288357977),\n                        (9, 0.4999999999999998, 0.0, -0.24927020326150684),\n                        (6, 0.5, 0.0, -0.20251771408753827),\n                        (13, 0.5000000000000018, 1.0, -0.05961715220065633),\n                        (2, 0.49999999999999944, 1.0, -0.3649528224832399),\n                        (12, 0.5132372001223594, 1.0, -0.3901571463071045),\n                        (0, 0.5, 1.0, 0.5),\n                        (11, 0.3040867239846966, 0.0, -0.5745192213521528),\n                        (10, 0.3040867239846966, 1.0, -0.5745192213521528),\n                        (14, 0.3040867239846966, 1.0, -0.5745192213521528)],\n                       [(3, 4), (7, 8)],\n                       [(3, 7),\n                        (4, 0.9316176020145068, 5.551115123125783e-17, 0.17937386689344853),\n                        (8, 0.4999999999999977, 0.0, -0.4684294262067361)],\n                       [(5, 4),\n                        (3, 0.5, 0.0, 0.866566658754937),\n                        (7, 0.5000000000000018, 0.0, 0.5872798439201941)],\n                       [(1, 3),\n                        (5, 0.6413290988859992, 0.0, 0.8805469187946222),\n                        (4, 0.5, 0.0, 0.5342851748417405),\n                        (7, 8)],\n                       [(5, 4),\n                        (1, 0.4999999999999998, 4.440892098500626e-16, 0.8532434711642047),\n                        (3, 0.5, 0.0, 0.8665666587549357),\n                        (8, 0.5451649362849962, 0.0, 0.07199820008846003),\n                        (7, 0.5526631368745091, 0.0, -0.9127201560798066)],\n                       [(1, 3),\n                        (4, 0.4027756110228036, 0.0, -0.3042483384350261),\n                        (5, 0.7428353455108296, 0.0, -0.5854869764262778)],\n                       [(4, 8),\n                        (3, 0.32477005048394103, 0.0, -0.6209276779886619),\n                        (1, 0.4999999999999995, 0.0, 0.8532434711642052)],\n                       [(4, 0.14184065006773824, 5.551115123125783e-17, 0.35870125356161064),\n                        (8, 0.5404252536846117, 4.440892098500626e-16, -0.9295908209423183)],\n                       [(3, 4), (7, 8)],\n                       [(3, 0.49999999999999956, 0.0, 0.37252798402052734),\n                        (7, 0.5, 0.0, 0.5872798439201923),\n                        (4, 0.4999999999999972, 0.0, -0.8028568414626689),\n                        (8, 0.8092888888091392, 0.0, -0.7865244341966551)],\n                       [(3, 7), (5, 4), (8, 9)],\n                       [(3, 0.5000000000000027, 0.0, -0.654473772675858),\n                        (8, 0.9654590361209281, 4.440892098500626e-16, 0.7134755658033336),\n                        (7, 0.49999999999999906, 0.0, 0.876708733821767),\n                        (5, 0.5, 0.0, -0.085486976426276)],\n                       [(1, 3), (4, 8), (7, 6), (9, 5)],\n                       [(3, 0.04647198093454085, 0.0, -0.15447377267584628),\n                        (5, 0.5, 0.0, -0.08964531064215042),\n                        (8, 0.8200940364446669, 0.0, -0.7865244341966673),\n                        (9, 0.21994685508514106, 0.0, 0.10628303654897175),\n                        (7, 0.4025088854789326, 1.0, -0.8085093877936544),\n                        (6, 0.2382498473985822, 0.0, 0.048025566717950996)],\n                       [(1, 3), (5, 4), (8, 9)],\n                       [(4, 0.8460005307855218, 0.0, 0.9324494917434116),\n                        (8, 0.6890163301428918, 0.0, 0.7134755658033327),\n                        (9, 0.6096668132999816, 0.0, -0.7383581411174034),\n                        (5, 0.5, 0.0, 0.91035468935785),\n                        (3, 7),\n                        (1, 0.5, 1.0, 0.07627797069572789)],\n                       [(4, 8),\n                        (9, 5),\n                        (7, 0.5000000000000016, 0.0, 0.6914906122063473),\n                        (3, 0.4999999999999997, 0.0, 0.8455262273241408)],\n                       [(4, 0.5845169958340231, 0.0, 0.2978543093193007),\n                        (7, 6),\n                        (8, 0.8232473481276812, 0.0, 0.21347556580333227),\n                        (9, 0.2521331218198125, 0.0, -0.7107117784653785),\n                        (1, 3)],\n                       [(5, 4),\n                        (8, 9),\n                        (7, 0.4999999999999993, 2.220446049250313e-16, -0.30850938779365267),\n                        (6, 0.8228387639951986, 1.1102230246251565e-16, 0.24879131417776945),\n                        (1, 0.5000000000000003, 1.0, -0.4237220293042716)],\n                       [(4, 0.4254286984202271, 0.0, 0.26852653159932993),\n                        (8, 0.2066825275444964, 8.881784197001252e-16, 0.7134755658033305),\n                        (7, 6),\n                        (1, 3),\n                        (9, 0.7262013565878536, 1.0, -0.3915949387156321)],\n                       [(4, 8),\n                        (9, 5),\n                        (3, 0.36532098276163216, 0.0, 0.17648180002731978),\n                        (7, 0.45688538394013445, 1.0, 0.19149061220635022),\n                        (6, 0.6082288892935703, 0.0, 0.24879131417776595),\n                        (1, 0.5000000000000001, 1.0, 0.07627797069572811)],\n                       [(4, 0.3813125022538922, 0.0, 0.3391936143831713),\n                        (8, 0.8293202449371231, 0.0, 0.2134755658033321),\n                        (3, 7),\n                        (9, 0.8716444748905248, 1.0, 0.2716325670282217)],\n                       [(4, 8),\n                        (9, 5),\n                        (3, 0.15979316138258426, 0.0, -0.7481915185952028),\n                        (7, 0.540918724091567, 0.9999999999999998, 0.6914906122063482)],\n                       [(8, 0.018606848196365704, 0.0, 0.71347556580334),\n                        (9, 0.8275802982908016, -4.440892098500626e-16, -0.23450174161127824),\n                        (3, 7),\n                        (4, 0.15044849274722297, 1.0, -0.9677765916498695)],\n                       [(8, 9),\n                        (5, 4),\n                        (7, 0.0820855992752405, 0.0, -0.8085093877936513),\n                        (3, 0.17667382808391466, 0.0, 0.4014554732067821)],\n                       [(4, 0.7397154449607974, 1.1102230246251565e-16, -0.2214694392998764),\n                        (8, 0.4999999999999998, 0.0, 0.7134755658033323),\n                        (7, 6),\n                        (9, 0.2477347359606612, 0.0, -0.1674837153451536),\n                        (5, 0.0, -0.1633336348559169, 0.0),\n                        (1, 3)],\n                       [(4, 8),\n                        (9, 5),\n                        (3, 0.5000000000000024, 0.0, 0.7299305507219822),\n                        (7, 0.4999999999999982, -0.9999999999999998, 0.6914906122063484),\n                        (6, 0.2692579739648868, 0.0, -0.751208685822232),\n                        (1, 0.5000000000000002, -0.9999999999999996, -0.5712905645877397)],\n                       [(5, 0.5, 0.0, 0.4439099981331701),\n                        (4, 0.5000000000000004, 0.0, -0.0396057990089389),\n                        (8, 0.4564046063066609, 0.0, -0.5129225181422381),\n                        (9, 0.6344506588693033, 0.0, 0.2650315767394704),\n                        (3, 7)],\n                       [(5, 4),\n                        (8, 9),\n                        (7, 0.5000000000000007, 0.0, 0.6914906122063484),\n                        (3, 0.35284137925881676, 0.0, 0.7299305507219813)],\n                       [(7, 6),\n                        (1, 3),\n                        (8, 0.0, 1.0, 0.0),\n                        (9, 13),\n                        (5, 0.5, -0.6242254674313306, -0.5560900018668296)],\n                       [(4, 8),\n                        (9, 0.7357804847159248, 0.0, 0.799839990315014),\n                        (3, 0.5, 0.0, 0.7299305507219853),\n                        (7, 0.2690604471363719, 1.0, 0.19149061220634872),\n                        (1, 0.30168861947752823, 1.0, -0.07129056458773952),\n                        (13, 0.14987790312208762, 1.0, -0.9184345745726195)],\n                       [(8, 9), (3, 7), (4, 0.0, -0.8114653152534659, 0.0)],\n                       [(3, 0.5000000000000004, 0.0, -0.7700694492780153),\n                        (7, 0.0, 1.0, 0.0),\n                        (9, 0.5, 0.0, 0.8006237570902135),\n                        (8, 0.6471132918108689, 0.0, -0.01866219118981416)],\n                       [(1, 3), (7, 6), (9, 13)],\n                       [(9, 0.8776478197236328, 0.0, 0.800623757090213),\n                        (3, 0.5, 4.440892098500626e-16, -0.7700694492780153),\n                        (7, 0.5, 1.0, 0.19149061220634717),\n                        (6, 2),\n                        (13, 0.6739840728188655, 1.0, -0.9510004559079259),\n                        (1, 0.5, 0.41144090801915634, -0.07129056458773975)],\n                       [(8, 9),\n                        (3, 7),\n                        (2, 0.3765861726663244, -0.9999999999999996, 0.13504717751675888),\n                        (6, 0.564919347967729, 5.551115123125783e-17, 0.1253774868440941)],\n                       [(8, 0.3396187696500643, 0.0, 0.9947548401311082),\n                        (9, 0.18946358304197863, 1.1102230246251565e-16, 0.3006237570902125),\n                        (3, 0.5000000000000038, 0.0, 0.22993055072198132),\n                        (7, 0.5, 1.0, 0.1914906122063464)],\n                       [(8, 9), (2, 3), (7, 6)],\n                       [(3, 0.6124202497959054, 8.881784197001252e-16, -0.8730591942204244),\n                        (8, 0.2951538153860544, 0.0, 0.05197736515696061),\n                        (6, 0.5000000000000004, -4.440892098500626e-16, -0.3746225131559058),\n                        (2, 0.7664718219180737, 0.9999999999999998, 0.635047177516759),\n                        (9, 0.645709372678986, 0.0, 0.8006237570902124)],\n                       [(3, 7), (4, 8), (6, 2), (9, 13)],\n                       [(8, 0.2762392233376908, 0.0, -0.1638213875295369),\n                        (9, 0.5149002647850929, 0.0, -0.6993762429097865),\n                        (2, 0.5000000000000002, 1.0, -0.36495282248324123),\n                        (3, 0.7237608380026964, 0.0, -0.9865488442922143),\n                        (7, 0.0, -0.44675502406324963, 0.0),\n                        (6, 0.8371608839553215, 0.0, 0.17545611282181062),\n                        (4, 0.5, 1.0, -0.851071114262405),\n                        (13, 0.4684758799110338, -0.9999999999999998, 0.28875187722571316)],\n                       [(8, 9), (2, 3), (7, 6)],\n                       [(3, 0.0, 0.5155754461675244, 0.0),\n                        (7, 0.5, 0.0, 0.7447355881430946),\n                        (8, 0.45893015895032185, 0.0, 0.6779225398198683),\n                        (6, 0.647259141517744, 0.0, -0.6977855638982122),\n                        (2, 0.5000000000000001, 1.0, -0.8649528224832415),\n                        (9, 0.5000000000000003, 0.0, 0.3006237570902126)],\n                       [(3, 7), (4, 8), (6, 2), (9, 13)],\n                       [(2, 3),\n                        (8, 0.5, 0.0, 0.8400502521580435),\n                        (9, 0.34547154120847123, 0.0, 0.18110029569729924),\n                        (7, 0.5, 1.0, -0.2552644118569045),\n                        (6, 0.49999999999999956, 0.0, 0.6885061300272752),\n                        (4, 0.4999999999999997, 1.0, -0.351071114262405)],\n                       [(8, 9), (7, 6), (2, 0.5, 0.0, 0.9683805108500918)],\n                       [(6, 0.6633380081977305, 0.0, 0.6885061300272752),\n                        (9, 13),\n                        (7, 0.5, -0.9999999999999996, 0.9825149420015955),\n                        (8, 0.5, 0.0, 0.3400502521580435)],\n                       [(6, 2),\n                        (3, 7),\n                        (4, 8),\n                        (9, 0.5, 0.0, 0.9043829633279703),\n                        (13, 0.06012004713910613, 0.0, 0.785989060922849)],\n                       [(2, 0.8248194291522575, 0.0, -0.5316194891499073),\n                        (3, 0.5000000000000004, 4.440892098500626e-16, -0.8280750606546727),\n                        (8, 0.5, 0.0, 0.3400502521580435),\n                        (7, 0.0, 1.0, 0.0),\n                        (6, 0.5000000000000001, 0.0, 0.18850613002727545),\n                        (13, 12),\n                        (4, 0.5000000000000004, 0.6707556449642436, -0.35107111426240434)],\n                       [(2, 3),\n                        (8, 9),\n                        (7, 6),\n                        (13, 0.7495430201170181, 0.0, 0.7859890609228533),\n                        (12, 0.5000000000000036, 0.9999999999999998, 0.10984285369289531)],\n                       [(9, 0.4999999999999998, 0.0, -0.0956170366720297),\n                        (2, 0.18030068686283401, 5.551115123125783e-17, 0.4683805108500939),\n                        (7, 0.5663648310033746, 1.0, -0.5174850579984049),\n                        (3, 0.5, 4.440892098500626e-16, -0.8280750606546727),\n                        (8, 0.5000000000000007, 0.0, -0.6599497478419556)],\n                       [(9, 13), (2, 0), (7, 3), (12, 8)],\n                       [(2, 0.5, 0.0, 0.9683805108500936),\n                        (7, 0.49999999999998246, 0.0, -0.017485001680463963),\n                        (8, 0.6996509454823032, 0.0, 0.6468389529408194),\n                        (9, 0.0, 0.19270439133078582, 0.0),\n                        (13, 0.7096334797826299, 0.0, 0.7859890609228533),\n                        (12, 0.6353215710838273, 0.9999999999999998, 0.6098428536928953),\n                        (0, 0.3040867239846962, 1.0, -0.5000000000000002),\n                        (3, 0.5000000000000022, 0.0, -0.19534539864792322)],\n                       [(6, 2), (7, 11), (8, 9), (13, 12)],\n                       [(13, 0.8327721342201915, 0.0, 0.28598906092285303),\n                        (2, 0.8040867239846965, 0.0, 0.9683805108500936),\n                        (7, 0.5000000000000102, 1.0, -0.017485001680463963),\n                        (12, 0.5, 1.0, -0.3901571463071043),\n                        (8, 0.6244194866951238, 0.0, -0.4995436859977147),\n                        (11, 0.30408672398469627, 0.0, -0.5745192213521484),\n                        (6, 0.0, -0.8274253014635145, 0.0)],\n                       [(9, 13), (2, 0), (7, 3), (12, 8)],\n                       [(7, 0.75, 0.0, 0.9825150308347146),\n                        (2, 0.7499999999999999, 0.0, 0.9683805108500936),\n                        (8, 0.0, -0.6089568169567903, 0.0),\n                        (9, 0.5, 0.0, 0.09708735465875673),\n                        (13, 0.4999999999999998, 0.0, -0.2140109390771472),\n                        (12, 0.5000000000000003, 0.9999999999999999, 0.7537853447666714),\n                        (0, 0.5, 1.0, 0.5),\n                        (3, 0.49999999999999956, 0.0, -0.1953454311631051)],\n                       [(7, 11), (6, 2), (8, 9), (13, 12)],\n                       [(9, 0.5000000000000002, 4.440892098500626e-16, -0.9029126453412426),\n                        (13, 0.5, 0.0, 0.7756473953900564),\n                        (12, 8),\n                        (2, 0.5, 0.0, -0.03161948914990642),\n                        (7, 0.500526889421938, 1.0, 0.4830419669618785),\n                        (11, 0.5000000000000102, 1.1102230246251565e-16, 0.4254807786478515),\n                        (6, 0.5, -0.9999999999999996, -0.7987688280048215)],\n                       [(9, 13),\n                        (2, 0),\n                        (7, 3),\n                        (12, 0.8333333333333335, 1.1102230246251565e-16, 0.2537853447666717)],\n                       [(7, 0.5, 0.0, 0.2330424030353116),\n                        (9, 0.5, 0.0, -0.35524852001198326),\n                        (13, 0.44763595031804465, 0.0, -0.22435260460994355),\n                        (2, 0.2500000000000001, 0.0, 0.9683805108500934),\n                        (3, 0.5000000000000004, 0.0, -0.6953454311631049)],\n                       [(7, 11), (8, 9), (13, 12), (6, 2)],\n                       [(2, 0.5, 0.0, -0.03161948914990642),\n                        (13, 0.4999999999999998, 0.0, -0.724352604609944),\n                        (12, 0.8544579937822567, -2.220446049250313e-16, -0.7462146552333284),\n                        (8, 0.5000000000000004, 0.0, -0.6924422265981729),\n                        (7, 1.0214051826551437e-14, -0.9999999999999998, 0.7330353121244857),\n                        (6, 0.5, -0.7012311719951787, -0.7987688280048215),\n                        (11, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(2, 0), (9, 13), (12, 8), (7, 3)],\n                       [(9, 0.5458990182737975, 4.440892098500626e-16, -0.8552485200119835),\n                        (8, 0.4999999999999998, 0.0, -0.6924422265981729),\n                        (13, 0.5, 0.0, -0.17845358633614605),\n                        (12, 0.5, 0.0, -0.2462146552333282),\n                        (0, 1.0, 0.0, 0.0),\n                        (7, 0.999999967484822, 0.04074306406175365, 0.2330424257007997),\n                        (2, 0.4999999999999998, 0.5316194891499064, 0.9683805108500936),\n                        (3, 0.0007451670218237004, 0.46913094359114815, 0.8046545688369025)],\n                       [(9, 8), (13, 12)],\n                       [(9, 0.4999999999999824, 0.0, -0.3552485763299238),\n                        (13, 0.49999999999998246, 0.0, 0.8215463573459125),\n                        (8, 0.5000000000000022, 0.0, 0.8993558099494203),\n                        (12, 0.5000000000000022, 0.0, 0.34558338131426586)],\n                       [(9, 10), (13, 14)],\n                       [(9, 0.5000000000000102, 0.0, -0.3552485763299238),\n                        (13, 0.5000000000000102, 0.0, 0.8215463573459125),\n                        (10, 0.30408672398469616, 1.0, -0.5745192213521486),\n                        (14, 0.30408672398469616, 1.0, -0.5745192213521486)],\n                       [(9, 8), (13, 12)],\n                       [(9, 0.75, 0.0, -0.3552486088451019),\n                        (13, 0.75, 0.0, 0.8215463248307342),\n                        (8, 0.49999999999999956, 0.0, -0.10064422256576133),\n                        (12, 0.49999999999999956, 0.0, -0.6544166512009157)],\n                       [(9, 10), (13, 14)],\n                       [(9, 0.500526889421938, 1.1102230246251565e-16, 0.144224455027734),\n                        (13, 0.500526889421938, 0.0, -0.6789806112964296),\n                        (10, 0.5000000000000103, 1.0, 0.42548077864785155),\n                        (14, 0.5000000000000103, 1.0, 0.42548077864785155)],\n                       [(9, 8), (13, 12)],\n                       [(9, 0.5, 0.0, -0.6057759810456993),\n                        (13, 0.5000000000000001, 0.0, 0.5710189526301364),\n                        (8, 0.5000000000000004, 0.0, 0.3993557774342378),\n                        (12, 0.5000000000000004, 0.0, -0.15441665120091663)],\n                       [(9, 10), (13, 14)],\n                       [(9, 8),\n                        (13, 12),\n                        (10, 0.5, -0.6516952337046309, 0.42548074613267284),\n                        (14, 0.5, -0.6516952337046309, 0.42548074613267284)],\n                       [(8, 0.0007451670218237004, 0.37442973499380416, 0.8993557774342469),\n                        (9, 3.251517832758567e-08, -0.12043850652625987, -0.605776003711187),\n                        (12, 0.0007451670218237004, 0.9282021636289586, 0.3455833487990924),\n                        (13, 3.251517832758566e-08, 0.7027665597979045, 0.5710189299646491)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8),\n                      (6, 5),\n                      (6, 6),\n                      (6, 7),\n                      (6, 8)]}}\n\ndef _round10_unitary(n,k):\n    data = _ROUND10_VERIFIED.get((n,k))\n    if data is None: return None\n    q = [cirq.GridQubit(*v) for v in data[\'qubits\']]\n    moments=[]\n    for layer in data[\'moments\']:\n        ops=[]\n        for gate in layer:\n            if len(gate)==2: ops.append(cirq.CZ(q[gate[0]],q[gate[1]]))\n            else:\n                i,x,z,a=gate\n                ops.append(cirq.PhasedXZGate(x_exponent=x,z_exponent=z,axis_phase_exponent=a).on(q[i]))\n        moments.append(cirq.Moment(ops))\n    return cirq.Circuit(moments),{q[i] for i in data[\'outputs\']}\n# ROUND 10 VERIFIED NATIVE CIRCUITS END\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n# ROUND 11 VERIFIED NATIVE CIRCUITS START\n_ROUND11_VERIFIED = {(5, 2): {\'method\': \'direct low-weight state synthesis on a native grid\',\n          \'moments\': [[(0, 0.7559708302047685, -0.7733086023947533, 0.320969318803232),\n                       (1, 0.8043783088735437, 0.6515170907733827, 0.8159513083684549),\n                       (3, 0.4999999999999998, -0.5584149856733831, -0.6886988302124197),\n                       (4, 0.49999999999999994, 0.0522068075235933, 0.9454544440520358),\n                       (2, 0.5, -0.5441281184864035, 0.8654852752243574)],\n                      [(0, 1), (3, 4)],\n                      [(0, 0.7640425092546823, -0.7021420611459614, 0.037764963556376874),\n                       (3, 0.8404269858700166, 0.20835461538394462, -0.757774650718011),\n                       (1, 0.35633990510033997, 0.7688316360267597, 0.7028519599554299),\n                       (4, 0.3796598828425095, 0.6689530215511548, 0.7427083039774838)],\n                      [(0, 3), (1, 4)],\n                      [(0, 0.36242246794415295, 0.8819637508692, -0.44282228160989634),\n                       (3, 0.4269051475961739, 0.9224883840287371, 0.31481565925388344),\n                       (1, 0.39758361765043343, -0.8511245801814411, -0.8896734454479434),\n                       (4, 0.8597811009962283, 0.25747989373888236, 0.41259734329829)],\n                      [(0, 3), (1, 2)],\n                      [(0, 0.9524924195304101, 0.8878700870004521, -0.060858530740695915),\n                       (1, 0.5000000000000001, -0.39716675709977634, -0.7469854453224467),\n                       (3, 0.9377537051604905, 0.2984306318552603, 0.5500389637478837),\n                       (2, 0.4999999999999812, 0.43796891982541664, 0.3213571567379665)],\n                      [(0, 1), (3, 4)],\n                      [(0, 0.5, -0.17675239063724413, -0.6729884437402438),\n                       (3, 0.5000000000000001, -0.8275458650649554, -0.7663737351094986),\n                       (1, 5.790041403928687e-17, -0.4180243592456583, -0.6188655918573518),\n                       (4, 0.5, 0.11207341156084594, -0.5799227629628274)],\n                      [(0, 3), (1, 4)],\n                      [(0, 0.5, 0.10287949124781015, -0.8497408343774884),\n                       (1, 0.5000000000000001, -0.6846847814617965, 0.4378234383321186),\n                       (3, 1.0, 0.0, -0.4203904716520661),\n                       (4, 4.672238078773624e-17, 0.7209880082723035, 0.13896407401301308)]],\n          \'outputs\': [0, 1, 2, 3, 4],\n          \'qubits\': [(4, 4), (4, 5), (4, 6), (5, 4), (5, 5)]},\n (6, 1): {\'method\': \'joint spatial W tree and native boundary optimization\',\n          \'moments\': [[(3, 0.5, 0.0, 0.5),\n                       (4, 0.3040867239846965, 1.0, -0.5),\n                       (1, 0.5, 0.0, -0.5),\n                       (0, 0.5, 0.0, -0.5),\n                       (2, 0.5, 0.0, 0.5),\n                       (5, 0.75, 0.0, -0.5)],\n                      [(0, 1), (3, 4)],\n                      [(3, 0), (4, 0.3040867239846965, 1.0, 0.5), (1, 0.3040867239846965, 0.0, 0.5)],\n                      [(3, 0.5, 0.0, 0.5), (0, 0.3040867239846965, 0.0, -0.5), (1, 2), (4, 5)],\n                      [(3, 4),\n                       (0, 1),\n                       (2, 0.25000000000000006, 0.0, -0.5000000000000002),\n                       (5, 0.25000000000000006, 0.0, -0.5000000000000002)],\n                      [(4, 0.5, 0.0, -0.5), (1, 0.75, 0.0, 0.5), (0, 0.5, 0.0, 0.5), (3, 0.5, 1.0, 0.5)],\n                      [(4, 5), (1, 2)],\n                      [(1, 0.5, 0.0, -0.5), (2, 0.5, 0.0, 0.5), (4, 0.5, 0.0, -0.5), (5, 1.0, 0.0, 0.5)]],\n          \'outputs\': [0, 1, 2, 3, 4, 5],\n          \'qubits\': [(5, 7), (5, 8), (5, 9), (6, 7), (7, 7), (7, 8)]},\n (6, 3): {\'method\': \'signed Dicke seed with a final native CZ correction\',\n          \'moments\': [[(0, 0.5000000000000002, -0.42428321319834006, -0.615868865238284),\n                       (1, 0.5, -0.2519173839487392, -0.9675350218670773),\n                       (2, 0.8363999696779808, 0.9379785511573107, 0.8936283737483495),\n                       (5, 0.526754374176265, -0.6473078521102427, 0.6795413764203446),\n                       (3, 0.5, -0.32366876754882634, -0.3279065170779405),\n                       (4, 0.5000000000000001, 0.4869912831230203, -0.5153197970854857)],\n                      [(0, 1), (2, 5), (3, 4)],\n                      [(0, 0.4955064169794595, -0.2813818073889578, -0.33444259534337095),\n                       (3, 0.24463915542810818, -0.5587349151831664, -0.5969028639308964),\n                       (1, 0.010003221723666326, -0.26271879056656555, 0.21685523912695337),\n                       (2, 0.11512317675386498, 0.415360048319528, -0.12481465861592378),\n                       (4, 0.7401412009846919, 0.028153773436543794, 0.9195409303186741),\n                       (5, 0.8887386629788069, 0.8334394756523826, 0.9199752393668572)],\n                      [(0, 3), (1, 2), (4, 5)],\n                      [(0, 1.0, 0.0, -0.4317166375775292),\n                       (3, 0.5, -0.6998212618013269, 0.5192680789034021),\n                       (1, 0.30408672398469644, 0.32002980482159193, -0.9781039968447103),\n                       (4, 0.5000000000000001, -0.6086116085100326, -0.3025758952518387),\n                       (2, 0.75, 0.3308662961451064, -0.21463088277254672),\n                       (5, 1.0, 0.0, -0.439474007216587)],\n                      [(0, 3), (1, 4), (2, 5)],\n                      [(1, 0.5, -0.45710839983832985, 0.5919258079768814),\n                       (2, 0.7499999999999999, -0.8330105879569543, -0.8837645866274411),\n                       (3, 3.751202460924933e-17, 0.22051625658456508, -0.7338543553075545),\n                       (4, 0.5, -0.5773089777696865, -0.9111875037618715),\n                       (0, 0.4999999999999993, -0.7812753304252786, 0.0031338183045893087),\n                       (5, 0.499999999999999, -0.9962644412395125, 0.48255059324622723)],\n                      [(1, 2), (3, 4)],\n                      [(3, 0.5000000000000001, 0.818151696386199, -0.46003692631335946),\n                       (1, 6.59212284199981e-18, -0.6117376076694867, 0.4498212304204001),\n                       (4, 6.917714808702173e-17, 0.6840708319916903, -0.014757362583317502),\n                       (2, 0.5000000000000001, 0.7145604980856157, -0.7167751745843953)],\n                      [(0, 3), (1, 4), (2, 5)],\n                      [(0, 1.0, 0.0, -0.01492614010732729),\n                       (1, 5.582691693024814e-17, -0.27479056856303075, -0.8539042309766744),\n                       (2, 1.0, 0.0, -0.37696272229637273),\n                       (3, 0.5000000000000001, 0.3901744618331948, 0.8581147700728393),\n                       (4, 0.5000000000000001, 0.05271488144590175, -0.3044256495398676),\n                       (5, 0.5, 0.7620030798993191, 0.48628615200671493)],\n                      [(3, 4)]],\n          \'outputs\': [0, 1, 2, 3, 4, 5],\n          \'qubits\': [(4, 4), (4, 5), (4, 6), (5, 4), (5, 5), (5, 6)]},\n (10, 1): {\'method\': \'reweighted neighbour transfer of verified W tree\',\n           \'moments\': [[(5, 0.5, 0.0, -0.5),\n                        (1, 0.5, 0.0, 0.5),\n                        (6, 0.5, 0.0, -0.5),\n                        (0, 0.2820471084244876, 0.0, -0.5),\n                        (7, 0.5, 0.0, 0.5),\n                        (4, 0.3040867239846965, 0.0, -0.5),\n                        (3, 0.5, 0.0, -0.5),\n                        (2, 0.5, 0.0, -0.5),\n                        (8, 0.25000000000000006, 0.0, 0.5),\n                        (9, 0.5, 0.0, -0.5)],\n                       [(5, 6), (1, 0)],\n                       [(1, 5), (0, 0.2820471084244876, 0.0, -0.5), (6, 0.2820471084244876, 0.0, 0.5)],\n                       [(5, 0.2820471084244876, 0.0, -0.5), (1, 0.5, 0.0, 0.5), (6, 7), (0, 4)],\n                       [(5, 6),\n                        (0, 1),\n                        (4, 0.3040867239846965, 0.0, 0.5),\n                        (7, 0.3040867239846965, 0.0, -0.5)],\n                       [(0, 0.5, 0.0, -0.5),\n                        (6, 0.6959132760153035, 0.0, 0.5),\n                        (1, 2),\n                        (7, 3),\n                        (5, 9),\n                        (4, 8)],\n                       [(4, 0),\n                        (6, 7),\n                        (2, 0.25000000000000006, 0.0, 0.5),\n                        (8, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (3, 0.25000000000000006, 0.0, 0.5),\n                        (1, 0.75, 0.0, 0.5),\n                        (9, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (5, 0.75, 0.0, -0.5)],\n                       [(4, 0.5, 0.0, 0.5),\n                        (7, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (5, 9),\n                        (1, 2),\n                        (6, 0.5, 0.0, -0.5),\n                        (0, 0.5, 0.0, 0.5)],\n                       [(7, 3),\n                        (4, 8),\n                        (2, 0.5, 0.0, -0.5),\n                        (5, 0.5, 0.0, -0.5),\n                        (1, 0.5, 1.0, -0.5),\n                        (9, 0.5, 0.0, -0.5)],\n                       [(7, 0.5, 0.0, 0.5), (4, 0.5, 0.0, -0.5), (3, 0.5, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],\n           \'qubits\': [(6, 6), (6, 7), (6, 8), (6, 9), (7, 6), (7, 7), (7, 8), (7, 9), (8, 6), (8, 7)]},\n (10, 4): {\'method\': \'transferred short count seed + parallel coherent decoders\',\n           \'moments\': [[(1, 0.5141693595374417, -0.5766337513833917, 0.1347811824712168),\n                        (2, 0.4945985514881317, 0.6827630404837524, 0.6991180434310782),\n                        (6, 0.12633994766573645, -0.435319496143201, 0.35190237270422703),\n                        (7, 0.9882194941978002, 0.2946312065682317, -0.929113956061357),\n                        (0, 0.3811595417885368, 0.9697950820412474, 0.3084267893801399),\n                        (3, 0.4999999999999984, -0.9259203825459004, 0.19943454840273128),\n                        (5, 0.442470578035534, -0.28102121072478137, 0.1430563595801585),\n                        (4, 0.5, 0.0, 0.5),\n                        (8, 0.442470578035534, -0.28102121072478137, 0.1430563595801585),\n                        (9, 0.5, 0.0, 0.5)],\n                       [(1, 2), (6, 7)],\n                       [(1, 0.028943285570018903, -0.995496667877797, 0.9523399051309158),\n                        (2, 0.6159738324588147, 0.13476787882917207, 0.38188108391483055),\n                        (7, 0.2730211946178439, 0.2637018167621763, -0.7651859536045416),\n                        (6, 0.8731451735984569, -0.6842599774402556, 0.5541567308881872)],\n                       [(0, 1), (2, 7)],\n                       [(0, 0.4999999999999998, 0.5954024372067718, 0.27822187142138766),\n                        (1, 0.7254816490767371, -0.685988155731418, -0.37208354342781336),\n                        (2, 1.0, 0.0, -0.01631083567385283),\n                        (7, 0.7356864836971903, -0.8477723159627966, -0.49366864645486697)],\n                       [(0, 1), (3, 2), (6, 7)],\n                       [(1, 0.25009745461738947, -0.8722121169829311, 0.9307901790914714),\n                        (6, 0.8660682106437447, -0.024273693302521826, -0.8009823684080328),\n                        (2, 1.0, 0.0, 0.48500150565644673),\n                        (7, 0.7501918424734006, -0.5008129898275633, 0.658559037582336),\n                        (0, 0.4837607224615599, 0.10635084417906715, -0.5993484093959505),\n                        (3, 0.953453779603521, -0.594691521849988, 0.7735141658568285)],\n                       [(1, 6), (2, 7)],\n                       [(1, 0.5, 0.6509024572625783, 0.06645309975595176),\n                        (2, 0.5, 0.9164310682677745, -0.5356384024167988),\n                        (6, 0.49999999999999994, 0.5327552050430002, 0.8770307780259043),\n                        (7, 0.7515047581965919, -0.4231447972809146, -0.8579541347808359)],\n                       [(1, 2), (6, 7)],\n                       [(1, 0.5, -0.8682998318147113, 0.06691244382977968),\n                        (6, 0.7477260579261715, 0.4649572346042072, 0.40352570716161923),\n                        (2, 0.6425656643754104, -0.18218005383590707, 0.8807926658509755),\n                        (7, 0.24778849677107653, 0.14711992006968844, 0.7124469069099815)],\n                       [(0, 1), (6, 5), (3, 2), (7, 8)],\n                       [(1, 1.0, 0.0, -0.10015032570719207),\n                        (6, 0.5000000000000003, -0.5985907290172183, 0.35956682697967),\n                        (5, 0.8362294663880598, 0.6599160233076027, -0.13796485114462254),\n                        (0, 0.5000000000000001, 0.976965698949942, -0.3211773559931612),\n                        (2, 1.0, 0.0, -0.10015032570719207),\n                        (7, 0.5000000000000003, -0.5985907290172183, 0.35956682697967),\n                        (8, 0.8362294663880598, 0.6599160233076027, -0.13796485114462254),\n                        (3, 0.5000000000000001, 0.976965698949942, -0.3211773559931612)],\n                       [(1, 6), (5, 0), (2, 7), (8, 3)],\n                       [(0, 1.0, 0.0, -0.31434674519867767),\n                        (1, 0.5, 0.5337999423852557, -0.6311662178984807),\n                        (6, 0.6556631301548916, 0.8610335052305682, -0.18370702181779563),\n                        (5, 0.7582170456039754, 0.24497674682826176, -0.9780488278370199),\n                        (3, 1.0, 0.0, -0.31434674519867767),\n                        (2, 0.5, 0.5337999423852557, -0.6311662178984807),\n                        (7, 0.6556631301548916, 0.8610335052305682, -0.18370702181779563),\n                        (8, 0.7582170456039754, 0.24497674682826176, -0.9780488278370199)],\n                       [(0, 1), (6, 5), (3, 2), (7, 8)],\n                       [(1, 0.5, 0.12660884439905296, 0.902633724486775),\n                        (6, 0.39779393920402867, 0.4913476881548977, 0.9864517009106388),\n                        (5, 0.9399029518185916, -0.6124748890555916, 0.7669279189912419),\n                        (0, 0.06063473201948603, -0.4963768337778258, 0.5486438016877042),\n                        (2, 0.5, 0.12660884439905296, 0.902633724486775),\n                        (7, 0.39779393920402867, 0.4913476881548977, 0.9864517009106388),\n                        (8, 0.9399029518185916, -0.6124748890555916, 0.7669279189912419),\n                        (3, 0.06063473201948603, -0.4963768337778258, 0.5486438016877042)],\n                       [(1, 6), (5, 0), (2, 7), (8, 3)],\n                       [(0, 0.2568381783892426, -0.9163451042010391, -0.5156600729100429),\n                        (1, 1.0, 0.0, 0.06504238523536776),\n                        (6, 0.3277294268181158, -0.2575493796249244, -0.9650804840246567),\n                        (5, 0.5527003573655196, -0.8452155307097831, 0.6544530299356497),\n                        (3, 0.2568381783892426, -0.9163451042010391, -0.5156600729100429),\n                        (2, 1.0, 0.0, 0.06504238523536776),\n                        (7, 0.3277294268181158, -0.2575493796249244, -0.9650804840246567),\n                        (8, 0.5527003573655196, -0.8452155307097831, 0.6544530299356497)],\n                       [(0, 1), (6, 5), (3, 2), (7, 8)],\n                       [(1, 1.525866550641284e-17, -0.864432526760778, 0.0973871639073946),\n                        (6, 0.5673580858862431, 0.9415745135213562, -0.34034565385235016),\n                        (5, 0.5000000000000003, -0.10285799929589734, -0.1907625007741336),\n                        (0, 0.3360750009341992, 0.12772468562361616, -0.02744142199358457),\n                        (2, 1.525866550641284e-17, -0.864432526760778, 0.0973871639073946),\n                        (7, 0.5673580858862431, 0.9415745135213562, -0.34034565385235016),\n                        (8, 0.5000000000000003, -0.10285799929589734, -0.1907625007741336),\n                        (3, 0.3360750009341992, 0.12772468562361616, -0.02744142199358457)],\n                       [(1, 6), (5, 0), (2, 7), (8, 3)],\n                       [(1, 1.0, 0.0, -0.13686739238675116),\n                        (6, 0.6009181180709966, -0.37590323618597976, -0.94314913575461),\n                        (5, 0.589698345272404, -0.9717330369198909, 0.04580348082125213),\n                        (0, 0.5, -0.8985247372087768, 0.8518841845678609),\n                        (2, 1.0, 0.0, -0.13686739238675116),\n                        (7, 0.6009181180709966, -0.37590323618597976, -0.94314913575461),\n                        (8, 0.589698345272404, -0.9717330369198909, 0.04580348082125213),\n                        (3, 0.5, -0.8985247372087768, 0.8518841845678609)],\n                       [(1, 6), (5, 0), (2, 7), (8, 3)],\n                       [(0, 0.5000000000000001, 0.24603694537177956, -0.5466405526409159),\n                        (1, 0.5, 0.9661915441444264, -0.8312373770089783),\n                        (6, 0.9680586457321536, 0.40824001833190493, 0.29301416332626573),\n                        (5, 0.16802345081794726, 0.9565664897859618, -0.5305913505367861),\n                        (3, 0.5000000000000001, 0.24603694537177956, -0.5466405526409159),\n                        (2, 0.5, 0.9661915441444264, -0.8312373770089783),\n                        (7, 0.9680586457321536, 0.40824001833190493, 0.29301416332626573),\n                        (8, 0.16802345081794726, 0.9565664897859618, -0.5305913505367861)],\n                       [(0, 1), (6, 5), (3, 2), (7, 8)],\n                       [(1, 0.5000000000000001, 0.0879217546736264, -0.8650458328645521),\n                        (6, 0.5, -0.025899135106421278, 0.9285367904088526),\n                        (5, 1.0, 0.0, -0.18010489955936437),\n                        (0, 1.0, 0.0, 0.3702544798805023),\n                        (2, 0.5000000000000001, 0.0879217546736264, -0.8650458328645521),\n                        (7, 0.5, -0.025899135106421278, 0.9285367904088526),\n                        (8, 1.0, 0.0, -0.18010489955936437),\n                        (3, 1.0, 0.0, 0.3702544798805023)],\n                       [(1, 6), (5, 0), (2, 7), (8, 3)],\n                       [(0, 2.07697835290096e-17, -0.3529944660081865, 0.07600736747582898),\n                        (1, 0.5, -0.800969845204833, -0.7228332715839092),\n                        (6, 0.17213894013095282, -0.8259170422327864, -0.09736234469756888),\n                        (5, 0.4054732409466402, 0.40623186260012156, 0.5127528363587612),\n                        (3, 2.07697835290096e-17, -0.3529944660081865, 0.07600736747582898),\n                        (2, 0.5, -0.800969845204833, -0.7228332715839092),\n                        (7, 0.17213894013095282, -0.8259170422327864, -0.09736234469756888),\n                        (8, 0.4054732409466402, 0.40623186260012156, 0.5127528363587612)],\n                       [(0, 1), (6, 5), (3, 2), (7, 8)],\n                       [(1, 1.4972000079457492e-16, -0.4847813462063919, -0.1332782067863767),\n                        (6, 0.9788981661755423, 0.9154541845329569, 0.5767206130696452),\n                        (5, 0.5343543584033268, 0.9858519541162853, -0.783552266137973),\n                        (0, 0.4475286319367211, 0.13879837860263777, -0.5007263580826236),\n                        (2, 1.4972000079457492e-16, -0.4847813462063919, -0.1332782067863767),\n                        (7, 0.9788981661755423, 0.9154541845329569, 0.5767206130696452),\n                        (8, 0.5343543584033268, 0.9858519541162853, -0.783552266137973),\n                        (3, 0.4475286319367211, 0.13879837860263777, -0.5007263580826236)],\n                       [(1, 6), (5, 0), (2, 7), (8, 3)],\n                       [(5, 0.5410902234140091, -0.6534748535142816, -0.34652514648571886),\n                        (6, 7.430673758610534e-17, 0.48728009069039224, -0.02959093189001516),\n                        (0, 0.5, -0.0640605545504842, -0.9770296688635247),\n                        (1, 0.5205451117070048, -0.4914155370048656, -0.5085844629951344),\n                        (8, 0.5410902234140091, -0.6534748535142816, -0.34652514648571886),\n                        (7, 7.430673758610534e-17, 0.48728009069039224, -0.02959093189001516),\n                        (3, 0.5, -0.0640605545504842, -0.9770296688635247),\n                        (2, 0.5205451117070048, -0.4914155370048656, -0.5085844629951344)],\n                       [(5, 4), (0, 1), (8, 9), (3, 2)],\n                       [(5, 0.49999999999999983, 0.0, -0.5),\n                        (4, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (0, 0.5, 1.0, -0.5),\n                        (8, 0.49999999999999983, 0.0, -0.5),\n                        (9, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (3, 0.5, 1.0, -0.5)],\n                       [(6, 5), (1, 0), (7, 8), (2, 3)],\n                       [(5, 0.8040867239846964, 0.0, 0.5),\n                        (6, 1.0, 0.0, 0.5),\n                        (1, 0.25000000000000006, 0.0, 0.5),\n                        (0, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (8, 0.8040867239846964, 0.0, 0.5),\n                        (7, 1.0, 0.0, 0.5),\n                        (2, 0.25000000000000006, 0.0, 0.5),\n                        (3, 0.25000000000000006, 0.0, -0.5000000000000002)],\n                       [(5, 4), (1, 0), (8, 9), (2, 3)],\n                       [(5, 0.75, 0.0, 0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (8, 0.75, 0.0, 0.5),\n                        (9, 0.5, 0.0, 0.5),\n                        (1, 0.5, 1.0, -0.5),\n                        (0, 0.5, 1.0, -0.5),\n                        (2, 0.5, 1.0, -0.5),\n                        (3, 0.5, 1.0, -0.5)],\n                       [(6, 5), (7, 8)],\n                       [(5, 0.5000000000000001, 0.0, -0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (8, 0.5000000000000001, 0.0, -0.5),\n                        (7, 0.5, 0.0, 0.5)],\n                       [(5, 4), (8, 9)],\n                       [(5, 0.25000000000000006, 0.0, 0.5),\n                        (4, 1.0, 0.0, 0.5),\n                        (8, 0.25000000000000006, 0.0, 0.5),\n                        (9, 1.0, 0.0, 0.5)],\n                       [(6, 5), (7, 8)],\n                       [(5, 0.5, 0.0, -0.5), (8, 0.5, 0.0, -0.5), (6, 0.5, 0.0, 0.5), (7, 0.5, 0.0, 0.5)],\n                       [(5, 4), (8, 9)],\n                       [(5, 0.5000000000000001, 0.0, 0.5), (8, 0.5000000000000001, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],\n           \'qubits\': [(4, 4), (4, 5), (4, 6), (4, 7), (5, 3), (5, 4), (5, 5), (5, 6), (5, 7), (5, 8)]},\n (11, 2): {\'method\': \'phase-relaxed three-arm seed with exact local phase correction\',\n           \'moments\': [[(2, 0.4557615490547753, 2.206217718292436e-06, -0.5000170641315411),\n                        (4, 0.39647087794134084, 2.5285751402592815e-05, -0.5000136436612144),\n                        (3, 0.71635816643173, -6.213296563162274e-05, -0.4999484778599419),\n                        (5, 0.4882498920953619, 4.349047638485537e-05, -0.5000372429586861),\n                        (7, 0.5000000000000001, 4.484277495442335e-05, 0.49997327275728976),\n                        (8, 0.5000000000000007, -3.079062871558236e-05, 0.5000371070259482),\n                        (1, 0.5000000000000001, 0.6385429473035269, -0.3100090298255287),\n                        (0, 0.5000000000000001, -0.6436503183081559, 0.8364418775619731),\n                        (6, 0.8040867239846964, 0.0, 0.5),\n                        (10, 0.5000000000000001, 0.6385429473035269, -0.3100090298255287),\n                        (9, 0.5000000000000001, -0.6436503183081559, 0.8364418775619731)],\n                       [(2, 4), (3, 5), (7, 8), (1, 0), (10, 9)],\n                       [(2, 0.06266681994258239, -2.6253817492660758e-05, 0.5000351543442945),\n                        (3, 0.9854744291641051, 1.23991103570531e-05, 0.4999893891744264),\n                        (4, 0.8759274384518551, 5.935745732932318e-06, -0.49998317798429825),\n                        (7, 0.09537532256601006, -6.247882046839237e-06, 0.5000159394368624),\n                        (5, 0.9813194918136614, 1.707218795410359e-05, -0.5000017190236441),\n                        (8, 0.7404528653710281, -6.1814674987559215e-06, 0.4999876709542109),\n                        (0, 0.7998718716147821, 0.890455176120845, 0.387859482353943),\n                        (1, 0.6216286712361211, 0.8060091866595659, 0.31230732769357783),\n                        (9, 0.7998718716147821, 0.890455176120845, 0.387859482353943),\n                        (10, 0.6216286712361211, 0.8060091866595659, 0.31230732769357783)],\n                       [(2, 3), (4, 7), (5, 8)],\n                       [(2, 0.7334807844810332, 2.0065414599590436e-05, 0.4999560243339107),\n                        (4, 0.5, -6.01428603852483e-06, -0.499994536598948),\n                        (3, 0.9999999999999999, 3.398902833151585e-05, 0.49997089486762114),\n                        (5, 1.0, 0.0, -0.4999977471796391),\n                        (7, 0.8358281884034311, 4.663390635317732e-06, -0.5000041969720577),\n                        (8, 3.987176112355756e-17, 2.6671289874657944e-05, -0.5000113185320616)],\n                       [(2, 4), (3, 5), (7, 8)],\n                       [(2, 7.897992968066795e-17, 1.3975586627246184e-05, 0.4999863802791145),\n                        (3, 0.19591327601501213, 2.598422301058889e-05, -0.5000264767816907),\n                        (4, 0.7728144741700669, -1.1515919953186682e-06, -0.49999952618405397),\n                        (7, 0.49999999999999944, -5.2530211367418644e-06, 0.5000004664185773),\n                        (5, 0.8040867239846561, 3.2024893785176367e-06, 0.49999691440885),\n                        (8, 0.1959132760153036, 1.2921213192518266e-05, 0.499987078786805)],\n                       [(2, 3), (4, 7), (5, 8)],\n                       [(2, 0.5, 9.934664860544729e-06, -0.5000099346648625),\n                        (4, 0.7271855258285053, -7.048858295277682e-06, 0.5000006777758679),\n                        (3, 0.7800614770004511, -4.400068196019902e-06, -0.4999995078998687),\n                        (5, 0.5000000000000014, -3.083337176357226e-05, 0.4999998831017616),\n                        (7, 0.9193905068961589, -4.786602553430441e-06, -0.49999521339744657),\n                        (8, 0.5, 1.4850524667209264e-06, 0.4999999999999976)],\n                       [(2, 4), (3, 5), (7, 8)],\n                       [(2, 0.5, -1.1566576945121199e-05, 0.4999999999999982),\n                        (3, 1.7313894484761872e-17, -5.708405599103017e-06, -0.49999609203192885),\n                        (4, 1.0, 0.0, -0.4999999999999991),\n                        (5, 3.47195095968923e-16, -4.712604885437344e-06, 0.5000309502699913),\n                        (7, 0.49999999999999983, 0.0, -0.5000000000000002),\n                        (8, 1.8014580033997582e-16, -1.6283088521973355e-07, -0.5000014850524646)],\n                       [(2, 3), (4, 5), (7, 8)],\n                       [(2, 0.5, -0.264214544122521, -0.7003509383216537),\n                        (3, 1.0, 0.0, 0.4561028372864193),\n                        (5, 1.3763823146121172e-16, -0.16410769149579707, -0.45542744324518036),\n                        (4, 0.8358447735207343, 0.9999936289175705, -0.9999936289175706),\n                        (7, 0.21390940690927968, -0.4381972885329142, -0.8945309765264866),\n                        (8, 7.220050238071246e-17, -0.08779636329404972, -0.623153814248042)],\n                       [(2, 3), (5, 6), (7, 8)],\n                       [(2, 0.5147343406546259, -0.544923949304474, 0.5354345175558253),\n                        (3, 1.0, 0.0, 0.002402555022007813),\n                        (5, 0.5, 0.0, -0.5),\n                        (6, 0.8040867239846964, 0.0, 0.5),\n                        (7, 0.5147343406546259, -0.544923949304474, 0.5354345175558253),\n                        (8, 1.0, 0.0, 0.002402555022007813)],\n                       [(2, 0), (3, 1), (4, 5), (7, 9), (8, 10)],\n                       [(2, 0.10241638234956676, 0.9457975847243225, -0.009489431748648691),\n                        (3, 0.6880806378478918, 0.8000338194427294, 0.263461268346828),\n                        (1, 0.866139763599385, 0.020128350303798004, 0.4971821495681795),\n                        (0, 0.5000000000000001, -0.5540071245883458, 0.16782978861775605),\n                        (5, 0.8040867239846964, 0.0, -0.5),\n                        (4, 0.6959132760153036, 0.0, 0.5),\n                        (7, 0.10241638234956676, 0.9457975847243225, -0.009489431748648691),\n                        (8, 0.6880806378478918, 0.8000338194427294, 0.263461268346828),\n                        (10, 0.866139763599385, 0.020128350303798004, 0.4971821495681795),\n                        (9, 0.5000000000000001, -0.5540071245883458, 0.16782978861775605)],\n                       [(2, 3), (1, 0), (5, 6), (7, 8), (10, 9)],\n                       [(2, 0.2966429086516353, 0.059657267082854126, 0.9363081529756734),\n                        (0, 1.0, 0.0, 0.12682399545097972),\n                        (3, 0.5000000000000002, -0.10295850378913207, 0.5634950877895573),\n                        (1, 5.0729768938045475e-17, -0.2871650987669918, 0.2647611481229777),\n                        (5, 0.5, 0.0, 0.5),\n                        (6, 0.5, 1.0, -0.5),\n                        (7, 0.2966429086516353, 0.059657267082854126, 0.9363081529756734),\n                        (9, 1.0, 0.0, 0.12682399545097972),\n                        (8, 0.5000000000000002, -0.10295850378913207, 0.5634950877895573),\n                        (10, 5.0729768938045475e-17, -0.2871650987669918, 0.2647611481229777)],\n                       [(2, 0), (3, 1), (4, 5), (7, 9), (8, 10)],\n                       [(2, 1.0, 0.0, -0.4465813274792114),\n                        (3, 0.5000000000000001, -0.28511399229663636, -0.039463415999574813),\n                        (1, 0.2677204728012302, 0.22338963539295975, -0.5650218341958806),\n                        (0, 0.5000000000000001, -0.07363506793441332, 0.6398253268725491),\n                        (5, 2.2816711776645322e-17, 1.0, 0.5),\n                        (7, 1.0, 0.0, -0.4465813274792114),\n                        (8, 0.5000000000000001, -0.28511399229663636, -0.039463415999574813),\n                        (10, 0.2677204728012302, 0.22338963539295975, -0.5650218341958806),\n                        (9, 0.5000000000000001, -0.07363506793441332, 0.6398253268725491),\n                        (4, 0.5, 0.0, -0.5)],\n                       [(2, 3), (1, 0), (5, 6), (7, 8), (10, 9)],\n                       [(2, 0.25000000000000006, 0.11105162785252942, 0.11087192498305015),\n                        (0, 0.25000000000000006, 0.787777488519148, 0.5661902589381358),\n                        (3, 0.12888105840915656, 0.4681194071763806, 0.17542259170378877),\n                        (1, 0.5, -0.3466673767462274, 0.7607841835466458),\n                        (5, 0.25000000000000006, 0.0, 0.5),\n                        (6, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (7, 0.25000000000000006, 0.11105162785252942, 0.11087192498305015),\n                        (9, 0.25000000000000006, 0.787777488519148, 0.5661902589381358),\n                        (8, 0.12888105840915656, 0.4681194071763806, 0.17542259170378877),\n                        (10, 0.5, -0.3466673767462274, 0.7607841835466458)],\n                       [(2, 0), (3, 1), (5, 6), (7, 9), (8, 10)],\n                       [(2, 0.5000000000000001, -0.872163295266233, -0.7780764471644204),\n                        (3, 0.5000000000000001, -0.793781741310823, 0.6435419988801694),\n                        (1, 0.5, 0.9356434507689282, 0.9141168068004182),\n                        (0, 0.49999999999999994, -0.004207489887937399, 0.35396774745728393),\n                        (5, 0.5, 1.0, -0.5),\n                        (6, 0.5, 1.0, -0.5),\n                        (7, 0.5000000000000001, -0.872163295266233, -0.7780764471644204),\n                        (8, 0.5000000000000001, -0.793781741310823, 0.6435419988801694),\n                        (10, 0.5, 0.9356434507689282, 0.9141168068004182),\n                        (9, 0.49999999999999994, -0.004207489887937399, 0.35396774745728393)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],\n           \'qubits\': [(3, 4),\n                      (3, 5),\n                      (4, 4),\n                      (4, 5),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (6, 4),\n                      (6, 5),\n                      (7, 4),\n                      (7, 5)]},\n (11, 4): {\'method\': \'transferred short count seed + parallel coherent decoders\',\n           \'moments\': [[(2, 0.5090585006619261, -0.5767209768639374, 0.1347810768372748),\n                        (3, 0.4966810234728208, 0.6827610492180689, 0.6991192988990056),\n                        (7, 0.13136732649011545, -0.4150541099163969, 0.35190321079000514),\n                        (8, 0.9650839235558885, 0.29802257176940516, -0.9291158659068195),\n                        (1, 0.3554214735996391, 0.9742853766721036, 0.3040897385107064),\n                        (4, 0.49999999999999994, -0.9259099121342174, 0.19942700615327347),\n                        (6, 0.442470578035534, -0.28102121072478137, 0.1430563595801585),\n                        (5, 0.5, 0.0, 0.5),\n                        (9, 0.4127792040228902, 0.905374889856748, -0.9262332254895422),\n                        (10, 0.5, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5)],\n                       [(2, 3), (7, 8)],\n                       [(2, 0.027658917775452607, -0.9953427338405545, 0.9524267180339823),\n                        (3, 0.6118093291523629, 0.13461402807370482, 0.38188034811707516),\n                        (8, 0.22149398043953314, 0.2664826476350983, -0.7685765613482478),\n                        (7, 0.8423481164117017, -0.46237353659978986, 0.43580718145864134)],\n                       [(1, 2), (3, 8)],\n                       [(1, 0.5, 0.5949694905316372, 0.27837511518281016),\n                        (2, 0.7613423740926692, -0.6879903676052783, -0.3723263148238838),\n                        (3, 1.0, 0.0, -0.016233747411281252),\n                        (8, 0.8035994154650444, -0.84838280955764, -0.4930581528600233)],\n                       [(1, 2), (4, 3), (7, 8)],\n                       [(2, 0.29024892335769925, -0.8725550979261085, 0.9330352135209878),\n                        (7, 0.7980201802080572, -0.0605290669318741, -0.790980600558008),\n                        (3, 1.0, 0.0, 0.4850025803696101),\n                        (8, 0.8347775209415229, -0.4849334032023256, 0.6585590375823367),\n                        (1, 0.5595921920530222, -0.6075535708322097, 0.4649871321169995),\n                        (4, 0.03092206383813562, 0.8540193125275457, 0.7735170940190594)],\n                       [(2, 7), (3, 8)],\n                       [(2, 0.5000000000000001, 0.6528038778150518, 0.06455104626961278),\n                        (3, 0.5, 0.9164321066438346, -0.5356394407928589),\n                        (7, 0.5, 0.5065015992636726, 0.9032843838052325),\n                        (8, 0.8386918872147733, -0.42748640553710704, -0.8738337214061112)],\n                       [(2, 3), (7, 8)],\n                       [(2, 0.5000000000000001, -0.7165354209428032, -0.0848519670421286),\n                        (7, 0.5737816774266398, -0.888360053985527, 0.1709967636194623),\n                        (3, 0.7969494981376857, 0.10638760826412952, 0.8807926658509753),\n                        (8, 0.22197270683743045, 0.6231124235664489, 0.303163775481797)],\n                       [(1, 2), (7, 6), (4, 3), (8, 9)],\n                       [(2, 1.0, 0.0, -0.10015032570719207),\n                        (7, 0.5000000000000003, -0.5985907290172183, 0.35956682697967),\n                        (6, 0.8362294663880598, 0.6599160233076027, -0.13796485114462254),\n                        (1, 0.5000000000000001, 0.976965698949942, -0.3211773559931612),\n                        (3, 1.0, 0.0, 0.21461996803249472),\n                        (8, 0.5, 0.7022844924594797, 0.4262761990482463),\n                        (9, 0.85928593962011, -0.4875608597182275, -0.02085833563279449),\n                        (4, 0.5000000000000001, 0.12925995895837872, -0.8724635934533991)],\n                       [(2, 7), (6, 1), (3, 8), (9, 4)],\n                       [(1, 1.0, 0.0, -0.31434674519867767),\n                        (2, 0.5, 0.5337999423852557, -0.6311662178984807),\n                        (7, 0.6556631301548916, 0.8610335052305682, -0.18370702181779563),\n                        (6, 0.7582170456039754, 0.24497674682826176, -0.9780488278370199),\n                        (4, 1.0, 0.0, -0.28654568860164664),\n                        (3, 0.5000000000000001, -0.07216288375695012, -0.25958902417306295),\n                        (8, 0.6714165234362537, 0.7430251514267326, -0.8000232167612813),\n                        (9, 0.7485956033393342, -0.10058310030841655, -0.008419195351022202)],\n                       [(1, 2), (7, 6), (4, 3), (8, 9)],\n                       [(2, 0.5, 0.12660884439905296, 0.902633724486775),\n                        (7, 0.39779393920402867, 0.4913476881548977, 0.9864517009106388),\n                        (6, 0.9399029518185916, -0.6124748890555916, 0.7669279189912419),\n                        (1, 0.06063473201948603, -0.4963768337778258, 0.5486438016877042),\n                        (3, 0.5000000000000001, -0.47921561052476913, 0.6682480920699869),\n                        (8, 0.39352947204382344, -0.0400166187933757, 0.22204905027907273),\n                        (9, 0.996157271886578, -0.38731562230521455, -0.6090022956594439),\n                        (4, 0.05280605500441269, -0.7140339826902271, -0.0034880864952426816)],\n                       [(2, 7), (6, 1), (3, 8), (9, 4)],\n                       [(1, 0.2568381783892426, -0.9163451042010391, -0.5156600729100429),\n                        (2, 1.0, 0.0, 0.06504238523536776),\n                        (7, 0.3277294268181158, -0.2575493796249244, -0.9650804840246567),\n                        (6, 0.5527003573655196, -0.8452155307097831, 0.6544530299356497),\n                        (4, 0.24963665794604506, 0.8669932073346845, 0.7651153459047553),\n                        (3, 1.0, 0.0, 0.21763245964208),\n                        (8, 0.3104369525321425, 0.8085371278988395, -0.23648881891450202),\n                        (9, 0.5655532211464792, -0.7603717865514881, -0.49631791796466374)],\n                       [(1, 2), (7, 6), (4, 3), (8, 9)],\n                       [(2, 1.525866550641284e-17, -0.864432526760778, 0.0973871639073946),\n                        (7, 0.5673580858862431, 0.9415745135213562, -0.34034565385235016),\n                        (6, 0.5000000000000003, -0.10285799929589734, -0.1907625007741336),\n                        (1, 0.3360750009341992, 0.12772468562361616, -0.02744142199358457),\n                        (3, 1.3678757607663618e-17, 0.19456334695121486, 0.24274381274574858),\n                        (8, 0.6193705712509703, -0.4502913348689348, -0.5549860019150674),\n                        (9, 0.49999999999999994, 0.914005305656897, 0.7433102954838477),\n                        (4, 0.32701156232958095, 0.6955438578648168, -0.9756718054724585)],\n                       [(2, 7), (6, 1), (3, 8), (9, 4)],\n                       [(2, 1.0, 0.0, -0.13686739238675116),\n                        (7, 0.6009181180709966, -0.37590323618597976, -0.94314913575461),\n                        (6, 0.589698345272404, -0.9717330369198909, 0.04580348082125213),\n                        (1, 0.5, -0.8985247372087768, 0.8518841845678609),\n                        (3, 1.0, 0.0, 0.3991857392005793),\n                        (8, 0.6256105257587085, 0.9452263271866326, -0.5220069687743731),\n                        (9, 0.5960614607313279, 0.2573629292943688, -0.033294760319355854),\n                        (4, 0.5000000000000001, 0.3922892154072385, 0.472507520913482)],\n                       [(2, 7), (6, 1), (3, 8), (9, 4)],\n                       [(1, 0.5000000000000001, 0.24603694537177956, -0.5466405526409159),\n                        (2, 0.5, 0.9661915441444264, -0.8312373770089783),\n                        (7, 0.9680586457321536, 0.40824001833190493, 0.29301416332626573),\n                        (6, 0.16802345081794726, 0.9565664897859618, -0.5305913505367861),\n                        (4, 0.5, -0.2739726829496021, 0.3647967363207205),\n                        (3, 0.5000000000000001, -0.2267887021096202, 0.036504264403519004),\n                        (8, 0.9764747842833552, 0.005682454151557326, 0.021003600491651175),\n                        (9, 0.14949404734505523, 0.6915804042354776, 0.6139788678075409)],\n                       [(1, 2), (7, 6), (4, 3), (8, 9)],\n                       [(2, 0.5000000000000001, 0.0879217546736264, -0.8650458328645521),\n                        (7, 0.5, -0.025899135106421278, 0.9285367904088526),\n                        (6, 1.0, 0.0, -0.18010489955936437),\n                        (1, 1.0, 0.0, 0.3702544798805023),\n                        (3, 0.5, -0.9528264593460201, 0.8097155622938987),\n                        (8, 0.5, 0.47939827321355255, 0.2696643378250829),\n                        (9, 1.0, 0.0, -0.31991326341917814),\n                        (4, 1.0, 0.0, -0.27721006339896803)],\n                       [(2, 7), (6, 1), (3, 8), (9, 4)],\n                       [(1, 2.07697835290096e-17, -0.3529944660081865, 0.07600736747582898),\n                        (2, 0.5, -0.800969845204833, -0.7228332715839092),\n                        (7, 0.17213894013095282, -0.8259170422327864, -0.09736234469756888),\n                        (6, 0.4054732409466402, 0.40623186260012156, 0.5127528363587612),\n                        (4, 2.6911488363979655e-17, -0.10657650605920965, 0.3199678821356995),\n                        (3, 0.5000000000000001, 0.6065964482612792, -0.07592204303348415),\n                        (8, 0.14731871270290642, 0.47763908808934363, -0.2509373889613644),\n                        (9, 0.42477613785575913, 0.24565143911589105, -0.6499368354199673)],\n                       [(1, 2), (7, 6), (4, 3), (8, 9)],\n                       [(2, 1.4972000079457492e-16, -0.4847813462063919, -0.1332782067863767),\n                        (7, 0.9788981661755423, 0.9154541845329569, 0.5767206130696452),\n                        (6, 0.5343543584033268, 0.9858519541162853, -0.783552266137973),\n                        (1, 0.4475286319367211, 0.13879837860263777, -0.5007263580826236),\n                        (3, 2.183989817369057e-17, 0.04089633285515237, 0.7520884815410067),\n                        (8, 0.9433458052208397, -0.437164462020319, -0.27329830087202156),\n                        (9, 0.4500869528994625, 0.5858004296037851, -0.09013056170924827),\n                        (4, 0.42020075164346243, -0.13244587453752787, -0.04525832781976602)],\n                       [(2, 7), (6, 1), (3, 8), (9, 4)],\n                       [(6, 0.5410902234140091, -0.6534748535142816, -0.34652514648571886),\n                        (7, 7.430673758610534e-17, 0.48728009069039224, -0.02959093189001516),\n                        (1, 0.5, -0.0640605545504842, -0.9770296688635247),\n                        (2, 0.5205451117070048, -0.4914155370048656, -0.5085844629951344),\n                        (9, 0.5410889047360808, -0.9487783322704517, -0.05122166772954828),\n                        (8, 5.822447387112414e-17, 0.6899183105243011, -0.1202404731065354),\n                        (4, 0.5410889047360808, 0.2729000875180194, 0.7270999124819801),\n                        (3, 0.5000000000000002, -0.5921151904509876, 0.5715707380829471)],\n                       [(6, 5), (1, 2), (9, 10), (4, 0)],\n                       [(6, 0.49999999999999983, 0.0, -0.5),\n                        (5, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (1, 0.5, 1.0, -0.5),\n                        (9, 0.49999999999999983, 0.0, -0.5),\n                        (10, 0.3040867239846963, 0.0, -0.5000000000000002),\n                        (4, 0.49999999999999983, 0.0, -0.5),\n                        (0, 0.3040867239846963, 0.0, -0.5000000000000002)],\n                       [(7, 6), (2, 1), (8, 9), (3, 4)],\n                       [(6, 0.8040867239846964, 0.0, 0.5),\n                        (7, 1.0, 0.0, 0.5),\n                        (2, 0.25000000000000006, 0.0, 0.5),\n                        (1, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (9, 0.8040867239846964, 0.0, 0.5),\n                        (8, 1.0, 0.0, 0.5),\n                        (4, 0.8040867239846964, 0.0, 0.5),\n                        (3, 1.0, 0.0, 0.5)],\n                       [(6, 5), (2, 1), (9, 10), (4, 0)],\n                       [(6, 0.75, 0.0, 0.5),\n                        (5, 0.5, 0.0, 0.5),\n                        (9, 0.75, 0.0, 0.5),\n                        (10, 0.5, 0.0, 0.5),\n                        (4, 0.75, 0.0, 0.5),\n                        (0, 0.5, 0.0, 0.5),\n                        (2, 0.5, 1.0, -0.5),\n                        (1, 0.5, 1.0, -0.5)],\n                       [(7, 6), (8, 9), (3, 4)],\n                       [(6, 0.5000000000000001, 0.0, -0.5),\n                        (7, 0.5, 0.0, 0.5),\n                        (9, 0.5000000000000001, 0.0, -0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (4, 0.5000000000000001, 0.0, -0.5),\n                        (3, 0.5, 0.0, 0.5)],\n                       [(6, 5), (9, 10), (4, 0)],\n                       [(6, 0.25000000000000006, 0.0, 0.5),\n                        (5, 1.0, 0.0, 0.5),\n                        (9, 0.25000000000000006, 0.0, 0.5),\n                        (10, 1.0, 0.0, 0.5),\n                        (4, 0.25000000000000006, 0.0, 0.5),\n                        (0, 1.0, 0.0, 0.5)],\n                       [(7, 6), (8, 9), (3, 4)],\n                       [(6, 0.5, 0.0, -0.5),\n                        (9, 0.5, 0.0, -0.5),\n                        (4, 0.5, 0.0, -0.5),\n                        (7, 0.5, 0.0, 0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (3, 0.5, 0.0, 0.5)],\n                       [(6, 5), (9, 10), (4, 0)],\n                       [(6, 0.5000000000000001, 0.0, 0.5),\n                        (9, 0.5000000000000001, 0.0, 0.5),\n                        (4, 0.5000000000000001, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],\n           \'qubits\': [(3, 7),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (4, 7),\n                      (5, 3),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8)]},\n (12, 1): {\'method\': \'joint spatial W tree and native boundary optimization\',\n           \'moments\': [[(4, 0.5, 1.0, -0.5),\n                        (5, 0.5, 0.0, -0.5),\n                        (6, 0.5, 0.0, 0.5),\n                        (10, 0.3040867239846965, 1.0, -0.5),\n                        (11, 0.25000000000000006, 0.0, 0.5),\n                        (1, 0.5, 0.0, -0.5),\n                        (0, 0.5, 0.0, -0.5),\n                        (3, 0.5, 1.0, 0.5),\n                        (7, 0.25000000000000006, 0.0, 0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (2, 0.5, 0.0, 0.5),\n                        (9, 0.5, 0.0, -0.5)],\n                       [(5, 4), (6, 10)],\n                       [(6, 5), (10, 0.3040867239846965, 1.0, 0.5), (4, 0.3040867239846965, 1.0, 0.5)],\n                       [(6, 0.5, 0.0, 0.5), (5, 0.6959132760153035, 0.0, -0.5), (10, 11), (4, 1)],\n                       [(6, 10),\n                        (5, 4),\n                        (11, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (1, 0.25000000000000006, 0.0, 0.5)],\n                       [(4, 0.75, 0.0, -0.5), (10, 0.5, 0.0, 0.5), (11, 7), (6, 3), (5, 2), (1, 0)],\n                       [(10, 11),\n                        (4, 8),\n                        (5, 0.0, 1.0, 0.0),\n                        (2, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (3, 0.25000000000000006, 1.0, 0.5),\n                        (6, 0.25000000000000006, 0.0, -0.5),\n                        (7, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (0, 0.25000000000000006, 0.0, 0.5)],\n                       [(4, 1),\n                        (10, 9),\n                        (5, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (8, 0.25000000000000006, 0.0, 0.5),\n                        (11, 0.5, 0.0, 0.5),\n                        (6, 3)],\n                       [(4, 0.25000000000000006, 0.0, 0.5),\n                        (9, 0.25000000000000006, 0.0, 0.5),\n                        (1, 0.75, 0.0, -0.5),\n                        (10, 0.75, 0.0, -0.5),\n                        (11, 7),\n                        (5, 2),\n                        (6, 0.5, 1.0, 0.5),\n                        (3, 0.5, 0.0, -0.5)],\n                       [(10, 9),\n                        (1, 0),\n                        (4, 8),\n                        (11, 0.5, 0.0, -0.5),\n                        (5, 0.5, 1.0, 0.5),\n                        (2, 0.5, 0.0, -0.5)],\n                       [(4, 0.5, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5),\n                        (8, 0.5, 0.0, -0.5),\n                        (10, 0.5, 0.0, 0.5),\n                        (9, 0.5, 0.0, -0.5),\n                        (0, 0.5, 0.0, -0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],\n           \'qubits\': [(4, 5),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 5),\n                      (6, 6),\n                      (6, 7),\n                      (6, 8),\n                      (7, 5),\n                      (7, 6),\n                      (7, 7),\n                      (7, 8)]},\n (14, 1): {\'method\': \'joint spatial W tree and native boundary optimization\',\n           \'moments\': [[(3, 0.25000000000000006, 0.0, 0.5),\n                        (7, 0.45437105165701, 0.0, 0.5),\n                        (8, 0.5, 0.0, -0.5),\n                        (4, 0.5, 0.0, -0.5),\n                        (5, 0.5, 0.0, 0.5),\n                        (1, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (11, 0.5, 0.0, -0.5),\n                        (10, 0.5, 0.0, 0.5),\n                        (12, 0.5, 0.0, 0.5),\n                        (2, 0.75, 0.0, 0.5),\n                        (9, 0.75, 0.0, 0.5),\n                        (13, 0.5, 0.0, -0.5),\n                        (6, 0.5, 0.0, -0.5),\n                        (0, 0.25000000000000006, 0.0, 0.5)],\n                       [(8, 4), (7, 3)],\n                       [(7, 8), (3, 0.25000000000000006, 0.0, 0.5), (4, 0.25000000000000006, 0.0, 0.5)],\n                       [(7, 0.5, 0.0, -0.5),\n                        (8, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (4, 5),\n                        (3, 1)],\n                       [(7, 3),\n                        (8, 4),\n                        (5, 0.3040867239846965, 0.0, -0.5),\n                        (1, 0.25000000000000006, 0.0, 0.5)],\n                       [(3, 0.5, 0.0, -0.5), (4, 0.6959132760153035, 0.0, 0.5), (7, 10), (8, 11), (1, 0)],\n                       [(4, 5),\n                        (1, 3),\n                        (8, 0.3040867239846965, 0.0, -0.5),\n                        (11, 0.3040867239846965, 0.0, 0.5),\n                        (10, 0.25000000000000006, 0.0, -0.5),\n                        (7, 0.75, 0.0, -0.5),\n                        (0, 0.25000000000000006, 0.0, -0.5000000000000002)],\n                       [(3, 0.5, 0.0, -0.5),\n                        (5, 0.5, 0.0, -0.5),\n                        (7, 6),\n                        (8, 11),\n                        (10, 13),\n                        (1, 0.5, 0.0, 0.5),\n                        (4, 0.5, 0.0, -0.5)],\n                       [(3, 2),\n                        (7, 10),\n                        (5, 9),\n                        (11, 12),\n                        (6, 0.25000000000000006, 0.0, 0.5),\n                        (13, 0.25000000000000006, 0.0, -0.5),\n                        (1, 0),\n                        (8, 0.5, 0.0, 0.5)],\n                       [(2, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (3, 0.5, 0.0, 0.5),\n                        (11, 0.25000000000000006, 0.0, 0.5),\n                        (10, 0.75, 0.0, -0.5),\n                        (7, 0.75, 0.0, 0.5),\n                        (9, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (12, 0.25000000000000006, 0.0, 0.5),\n                        (5, 0.5, 0.0, 0.5),\n                        (1, 0.5, 0.0, -0.5)],\n                       [(10, 13), (7, 6), (11, 12), (3, 2), (5, 9)],\n                       [(7, 0.5, 1.0, -0.5),\n                        (9, 1.0, 0.0, 0.5),\n                        (3, 0.5, 0.0, -0.5),\n                        (6, 0.5, 0.0, -0.5),\n                        (5, 0.5, 0.0, -0.5),\n                        (2, 1.0, 0.0, 0.5),\n                        (11, 0.5, 0.0, 0.5),\n                        (12, 0.5, 0.0, -0.5),\n                        (10, 0.5, 0.0, -0.5),\n                        (13, 0.5, 0.0, -0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],\n           \'qubits\': [(3, 6),\n                      (4, 6),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (5, 8),\n                      (6, 5),\n                      (6, 6),\n                      (6, 7),\n                      (6, 8),\n                      (7, 6),\n                      (7, 7),\n                      (7, 8),\n                      (8, 6)]},\n (14, 2): {\'method\': \'phase-relaxed three-arm seed with exact local phase correction\',\n           \'moments\': [[(7, 0.43136113001545656, 0.0, -0.4999975513771948),\n                        (10, 0.5000000000000001, 0.0, 0.49999188355695523),\n                        (5, 0.7277083635319732, 0.0, -0.5000036200039524),\n                        (8, 0.48765457944230856, 0.0, -0.4999925395676783),\n                        (4, 0.47179568489943596, 0.0, -0.5000003709679939),\n                        (11, 0.4999999999999998, 0.0, 0.5000048327277871),\n                        (9, 0.7179528915755126, 1.0, 0.5),\n                        (2, 0.7179528915755126, 0.0, 0.5),\n                        (12, 0.75, 0.0, 0.5),\n                        (1, 0.75, 1.0, 0.5),\n                        (6, 0.75, 1.0, 0.5),\n                        (0, 0.8040867239846965, 1.0, 0.49999999999999994),\n                        (13, 0.8040867239846965, 1.0, 0.49999999999999994),\n                        (3, 0.8040867239846965, 1.0, 0.49999999999999994)],\n                       [(5, 8), (4, 7), (10, 11)],\n                       [(10, 0.08766936986901919, 0.0, 0.49999338434401314),\n                        (5, 0.9858049410111958, 0.0, 0.4999963799960474),\n                        (7, 0.8777874307483884, 0.0, -0.49999948976899966),\n                        (4, 0.06029711583124752, 0.0, 0.4999902837174788),\n                        (8, 0.9811902139406308, 0.0, -0.49999465191032644),\n                        (11, 0.7328221121462921, 0.0, 0.5000044300947526)],\n                       [(7, 10), (4, 5), (8, 11)],\n                       [(4, 0.7493054797605098, 0.0, 0.49999964587671153),\n                        (7, 0.5, -2.220446049250313e-16, -0.4999923660379406),\n                        (10, 0.8204914820143119, 0.0, -0.5000081562942609),\n                        (5, 8)],\n                       [(10, 11),\n                        (4, 7),\n                        (5, 0.17950851798567607, 0.0, 0.499996504747414),\n                        (8, 0.7097846883716525, 0.0, -0.49999614674510706)],\n                       [(10, 0.5, 0.0, 0.49999184370573896),\n                        (7, 0.7499999999999347, 0.0, -0.4999925697935903),\n                        (4, 5),\n                        (11, 0.2902153116275834, 0.9999999999999999, 0.5000052470086038)],\n                       [(4, 0.0, 1.0, 0.0),\n                        (7, 10),\n                        (8, 11),\n                        (5, 0.7831771523714535, 0.0, 0.49999670939415575)],\n                       [(7, 0.75, 4.440892098500626e-16, 0.5000071420524064),\n                        (4, 0.5, 0.0, -0.5000003541232887),\n                        (8, 0.5, 0.0, -0.49999514437986026),\n                        (5, 0.0, -0.15585425947624998, 0.0),\n                        (11, 0.5, 0.0, 0.500005247008604),\n                        (10, 0.9193721487969426, 0.0, -0.5000118879076834)],\n                       [(4, 7), (5, 8), (10, 11)],\n                       [(4, 0.5, -0.9999999999999998, 0.49999964587671136),\n                        (7, 8),\n                        (5, 2),\n                        (10, 0.5000000000000004, 0.0, -0.5000118879076834),\n                        (11, 12)],\n                       [(4, 5),\n                        (8, 9),\n                        (7, 0.8441513265891005, 0.0, -0.9999928579475936),\n                        (2, 0.7179528915755126, 0.0, 0.5),\n                        (10, 11),\n                        (12, 0.75, 0.0, 0.5)],\n                       [(8, 0.5, 0.0, 0.3441488315703465),\n                        (5, 0.5, 0.0, -0.5000087176940158),\n                        (4, 0.4999999999999998, 0.0, 0.655848319287611),\n                        (9, 0.7179528915755126, 1.0, 0.5),\n                        (2, 1),\n                        (11, 0.5, 0.0, -0.3441498111939185),\n                        (10, 0.8260374972254332, 0.0, 0.8600764938561405),\n                        (12, 13)],\n                       [(7, 8),\n                        (4, 5),\n                        (9, 6),\n                        (10, 11),\n                        (1, 0.75, 1.0, 0.5),\n                        (13, 0.8040867239846965, 1.0, 0.49999999999999994)],\n                       [(8, 0.8524163823495667, 0.0, -0.6558511684296535),\n                        (5, 0.8524163823495667, 4.440892098500626e-16, -0.500008717694016),\n                        (11, 0.8333333333333333, 0.0, -0.3441498111939185),\n                        (6, 0.75, 1.0, 0.5),\n                        (1, 0),\n                        (4, 0.6475836176504333, 0.0, 0.655848319287611),\n                        (7, 0.6475836176504333, 0.0, -0.49999285794759407),\n                        (10, 0.6666666666666669, 0.0, -0.800080403359914)],\n                       [(8, 9), (5, 2), (6, 3), (11, 12), (0, 0.8040867239846965, 1.0, 0.49999999999999994)],\n                       [(9, 0.5, 0.0, -0.5),\n                        (2, 0.5, 0.0, -0.5),\n                        (5, 0.5, 0.0, 0.4999912823059842),\n                        (8, 0.5, 0.0, 0.3441488315703465),\n                        (11, 0.5, 0.0, 0.6558501888060815),\n                        (12, 0.5, 0.0, -0.5),\n                        (3, 0.8040867239846965, 1.0, 0.49999999999999994)],\n                       [(8, 9), (5, 2), (11, 12)],\n                       [(9, 0.8333333333333333, 0.0, 0.5),\n                        (2, 0.8333333333333333, 0.0, -0.5),\n                        (4, 5),\n                        (7, 8),\n                        (12, 0.8040867239846965, 2.220446049250313e-16, -0.5000000000000002),\n                        (10, 11)],\n                       [(2, 1),\n                        (9, 6),\n                        (5, 0.0, 1.0, 0.0),\n                        (8, 0.6666666666666667, 5.551115123125783e-17, 0.3441488315703464),\n                        (11, 0.0, 1.0, 0.0),\n                        (12, 13),\n                        (10, 0.5, -0.699919596640086, 0.19991959664008596),\n                        (4, 0.5, -0.155848319287611, -0.344151680712389),\n                        (7, 0.5, -7.1420524052623335e-06, 0.5000071420524055)],\n                       [(2, 0.5, 0.0, 0.5),\n                        (1, 0.5, 0.0, -0.5),\n                        (9, 0.5, 0.0, -0.5),\n                        (6, 0.5, 0.0, -0.5),\n                        (5, 0.6666666666666669, -0.9999999999999998, 0.49999128230598416),\n                        (13, 0.5, 1.0, -0.5),\n                        (11, 0.6959132760153035, 1.0, 0.6558501888060815),\n                        (12, 0.5, 0.0, 0.5)],\n                       [(2, 1), (9, 6), (11, 12)],\n                       [(6, 0.8040867239846965, 5.551115123125783e-17, 0.49999999999999994),\n                        (1, 0.8040867239846965, 5.551115123125783e-17, 0.49999999999999994),\n                        (5, 2),\n                        (8, 9),\n                        (12, 13),\n                        (11, 0.5, -0.15585018880608104, -0.3441498111939185)],\n                       [(6, 3),\n                        (1, 0),\n                        (2, 0.6959132760153035, 0.0, -0.5),\n                        (9, 0.6959132760153035, 0.0, -0.5),\n                        (13, 0.25000000000000006, 1.0, 0.5),\n                        (12, 0.25000000000000006, 0.0, -0.5),\n                        (8, 0.5, -0.8441488315703456, -0.6558511684296546),\n                        (5, 0.5, 8.717694015558664e-06, -0.5000087176940158)],\n                       [(6, 0.5, 0.0, -0.5),\n                        (0, 0.5, 1.0, -0.5),\n                        (3, 0.5, 1.0, -0.5),\n                        (1, 0.5, 0.0, -0.5),\n                        (12, 13)],\n                       [(1, 0), (9, 6), (13, 0.5, 0.0, 0.5), (12, 0.5, 0.0, -0.5)],\n                       [(2, 1), (6, 3), (0, 0.25000000000000006, 1.0, 0.5), (9, 0.5, 0.0, 0.5)],\n                       [(1, 0.25000000000000006, 0.0, -0.5),\n                        (3, 0.25000000000000006, 1.0, 0.5),\n                        (6, 0.25000000000000006, 0.0, -0.5),\n                        (2, 0.5, 0.0, 0.5)],\n                       [(1, 0), (6, 3)],\n                       [(3, 0.5, 0.0, 0.5), (6, 0.5, 0.0, -0.5), (0, 0.5, 0.0, 0.5), (1, 0.5, 0.0, -0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],\n           \'qubits\': [(1, 5),\n                      (2, 5),\n                      (3, 5),\n                      (3, 6),\n                      (4, 4),\n                      (4, 5),\n                      (4, 6),\n                      (5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (6, 4),\n                      (6, 5),\n                      (6, 6),\n                      (6, 7)]},\n (15, 1): {\'method\': \'joint spatial W tree and native boundary optimization\',\n           \'moments\': [[(9, 0.5, 0.0, 0.5),\n                        (5, 0.5, 0.0, -0.5),\n                        (6, 0.5212364096070798, 0.0, 0.5),\n                        (13, 0.5, 0.0, -0.5),\n                        (10, 0.7728144741714953, 0.0, -0.5),\n                        (14, 0.8040867239846965, 5.551115123125783e-17, 0.49999999999999994),\n                        (2, 0.5, 0.0, 0.5),\n                        (1, 0.5, 0.0, 0.5),\n                        (8, 0.5, 0.0, 0.5),\n                        (0, 0.5, 0.0, -0.5),\n                        (3, 0.5, 0.0, 0.5),\n                        (11, 0.5, 0.0, -0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (12, 0.75, 0.0, -0.5),\n                        (7, 0.5, 0.0, 0.5)],\n                       [(5, 9), (6, 10)],\n                       [(6, 5),\n                        (9, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (10, 0.22718552582850488, 0.0, 0.5)],\n                       [(5, 0.75, 0.0, 0.5), (9, 13), (6, 0.5, 0.0, 0.5), (10, 14)],\n                       [(5, 9),\n                        (13, 0.25000000000000006, 0.0, 0.5),\n                        (6, 2),\n                        (14, 0.1959132760153035, 0.0, -0.5000000000000002)],\n                       [(9, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (6, 10),\n                        (5, 1),\n                        (2, 0.25000000000000006, 0.0, -0.5)],\n                       [(9, 13),\n                        (6, 0.75, 0.0, -0.5),\n                        (1, 0.25000000000000006, 0.0, 0.5),\n                        (5, 0.75, 0.0, 0.5),\n                        (10, 0.5, 0.0, 0.5),\n                        (2, 3)],\n                       [(13, 0.5, 0.0, 0.5),\n                        (1, 0),\n                        (10, 11),\n                        (9, 8),\n                        (6, 7),\n                        (5, 4),\n                        (3, 0.25000000000000006, 0.0, -0.5000000000000002)],\n                       [(10, 14),\n                        (5, 1),\n                        (13, 12),\n                        (6, 2),\n                        (9, 0.75, 0.0, 0.5),\n                        (11, 0.25000000000000006, 0.0, 0.5),\n                        (7, 0.25000000000000006, 0.0, 0.5),\n                        (8, 0.25000000000000006, 0.0, -0.5),\n                        (0, 0.25000000000000006, 0.0, -0.5),\n                        (4, 0.25000000000000006, 0.0, -0.5)],\n                       [(6, 0.75, 0.0, -0.5),\n                        (10, 0.75, 0.0, -0.5),\n                        (5, 0.75, 0.0, 0.5),\n                        (12, 0.25000000000000006, 0.0, -0.5000000000000002),\n                        (13, 0.5, 0.0, -0.5),\n                        (2, 0.75, 0.0, 0.5),\n                        (1, 0.75, 0.0, -0.5),\n                        (9, 8),\n                        (14, 1.0, 0.0, 0.5)],\n                       [(10, 11),\n                        (1, 0),\n                        (5, 4),\n                        (2, 3),\n                        (13, 12),\n                        (6, 7),\n                        (9, 0.5, 0.0, -0.5),\n                        (8, 0.5, 0.0, 0.5)],\n                       [(3, 0.5, 0.0, 0.5),\n                        (4, 0.5, 0.0, 0.5),\n                        (10, 0.5, 0.0, 0.5),\n                        (13, 0.5, 0.0, -0.5),\n                        (1, 0.5, 0.0, -0.5),\n                        (2, 0.5, 0.0, -0.5),\n                        (11, 0.5, 0.0, -0.5),\n                        (0, 0.5, 0.0, -0.5),\n                        (7, 0.5, 0.0, 0.5),\n                        (5, 0.5, 0.0, -0.5),\n                        (6, 0.5, 1.0, -0.5),\n                        (12, 1.0, 0.0, 0.5)]],\n           \'outputs\': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],\n           \'qubits\': [(5, 4),\n                      (5, 5),\n                      (5, 6),\n                      (5, 7),\n                      (6, 4),\n                      (6, 5),\n                      (6, 6),\n                      (6, 7),\n                      (7, 4),\n                      (7, 5),\n                      (7, 6),\n                      (7, 7),\n                      (8, 4),\n                      (8, 5),\n                      (8, 6)]}}\n\ndef _round11_native(n,k):\n    data = _ROUND11_VERIFIED.get((n,k))\n    if data is None: return None\n    q = [cirq.GridQubit(*v) for v in data[\'qubits\']]\n    moments=[]\n    for layer in data[\'moments\']:\n        ops=[]\n        for gate in layer:\n            if gate[0]==\'json\': ops.append(cirq.read_json(json_text=gate[1]))\n            elif len(gate)==2: ops.append(cirq.CZ(q[gate[0]],q[gate[1]]))\n            else:\n                i,x,z,a=gate\n                ops.append(cirq.PhasedXZGate(x_exponent=x,z_exponent=z,axis_phase_exponent=a).on(q[i]))\n        moments.append(cirq.Moment(ops))\n    return cirq.Circuit(moments),{q[i] for i in data[\'outputs\']}\n\ndef _round11_unitary(n,k):\n    value=_round11_native(n,k)\n    return value if value is not None and not value[0].has_measurements() else None\n# ROUND 11 VERIFIED NATIVE CIRCUITS END\n'
_bonus_mod = _types.ModuleType('dicke_bonus')
exec(compile(_BONUS_SRC, 'dicke_bonus', 'exec'), _bonus_mod.__dict__)
BRICK = {(8, 4): {'cells': [[0, 0], [0, 1], [0, 2], [0, 3], [1, 0], [1, 1], [1, 2], [1, 3]], 'classes': [[[0, 1], [2, 3], [4, 5], [6, 7]], [[0, 4], [1, 5], [2, 6], [3, 7]], [[1, 2], [5, 6]]], 'seq': [0, 1, 2, 0, 1, 2, 0, 1, 2, 0, 1], 'params': [[[3.068792367559651], [-8.378156144171875e-08], [1.4487405269184614], [-1.8858008832311792], [3.1415926581705595], [-1.814379060005788], [-2.4127720212550945], [1.105311309458161]], [[0.9527751675367783], [0.4422354293677706], [-0.07250802704499605], [3.245748352564454], [-2.819288759361134], [-1.814379064305216], [-2.277846587044213], [0.11115276867414421]], [[0.04732279059598456], [-3.0299624489382646], [0.7971701143296829], [0.38999847358178197], [1.9732488663782528], [3.1415926651837607], [0.25857403299578363], [-1.3705111062096536]], [[-0.05423265503141149], [0.5092226722342936], [-1.9361584378695451], [-0.2925216654636982], [1.1717763194284276], [3.1415927108970814], [0.07192894594052314], [1.8577255059668611]], [[3.1415926461952237], [0.044018464209175365], [1.1493381755441425], [-2.2111147877263693], [-0.006287860063873581], [5.0395765376930586e-08], [-3.9023000325667123], [-2.4371743513693316]], [[3.1184355026881025], [-0.04401845825770144], [-3.3196050656148635], [2.085753453686407], [-1.5134129241608667], [-2.903069982842026e-09], [-0.5147761928146509], [2.6635950758852487]], [[-2.1631188598709956], [-0.8529107524041943], [-0.7142634546749544], [0.6458881109873241], [0.2667470309766852], [4.71238898642339], [1.3134826005852103], [2.465368851302781]], [[1.4510097746161654], [-2.3561944856055166], [2.4835526675345125], [1.508106119489801], [-2.8224344404218185], [0.31915821254808635], [3.379595468505202], [-3.4909121151902576]], [[1.7716318436596694], [-1.570796316454206], [-1.0578325132389985], [1.9085390308588661], [1.7444575494433743], [-1.570796309165627], [-0.1256921500899725], [1.1140353592224528]], [[-1.7716318419789454], [-1.570796330525948], [1.5707963019435092], [0.10015461129668764], [-0.1736612173594073], [-9.527756688525287e-09], [1.5707963267736638], [0.042124907636033194]], [[1.5707963439159565], [-1.5707963247103272], [-3.056486702398206e-08], [-1.570796339112287], [-8.453105341366282e-09], [3.1415926628612745], [-3.1415926542605535], [1.5707963236725173]], [[3.1415926460098196], [1.5707963284167983], [-1.5707963277785904], [1.5707963285358981], [-1.5707963355575942], [9.802396439378823e-09], [1.8054870447721946e-09], [3.5216234160297626e-09]]], 'mode': 'real', 'd': 11, 'N': 8, 'n': 4, 'anc': False}}


def _brick_circuit(d):
    import cirq_google
    dev = cirq_google.Willow105
    qset = set(dev.metadata.qubit_set)
    pairs = set(frozenset(p) for p in dev.metadata.qubit_pairs)
    for r0 in range(-2, 14):
        for c0 in range(-2, 14):
            q = [cirq.GridQubit(r0 + r, c0 + c) for r, c in d['cells']]
            if all(x in qset for x in q) and all(frozenset((q[a], q[b])) in pairs for cl in d['classes'] for a, b in cl):
                break
        else:
            continue
        break
    P = d['params']
    def layer(L):
        ops = []
        for i in range(len(q)):
            if d['mode'] == 'real':
                U = ry(P[L][i][0])
            else:
                a_, t_, b_ = P[L][i]
                U = rz(b_) @ ry(t_) @ rz(a_)
            ops.append(cirq.PhasedXZGate.from_matrix(U).on(q[i]))
        return cirq.Moment(ops)
    moments = [layer(0)]
    for t, c in enumerate(d['seq']):
        moments.append(cirq.Moment([cirq.CZ(q[a], q[b]) for a, b in d['classes'][c]]))
        moments.append(layer(t + 1))
    return cirq.Circuit(moments), set(q[:d['N']])

For small $N$, you may check that your circuit produces the correct state using the following function:

In [ ]:
def create_dicke_statevector(num_qubits: int, num_excitations: int) -> np.ndarray:
    psi = np.zeros(2**num_qubits)
    normalization = 1 / np.sqrt(math.comb(num_qubits, num_excitations))
    qubits = range(num_qubits)
    for excited_qubits in itertools.combinations(qubits, num_excitations):
        idx = sum(2**qubit_number for qubit_number in excited_qubits)
        psi[idx] = normalization
    return psi


def test_circuit(circuit: cirq.Circuit, num_excitations: int, qubits: set[cirq.GridQubit], sampler = cirq.Simulator()):
    all_qubits = circuit.all_qubits()
    ancilla_qubits = all_qubits.difference(qubits)
    ordered_qubits = sorted(qubits) + sorted(ancilla_qubits)
    keep_indices = range(qubits)
    num_qubits = len(qubits)
    psi_circuit_full = sampler.simulate(circuit, qubit_order=ordered_qubits).final_state_vector
    psi_circuit = cirq.sub_state_vector(
        state_vector=psi_circuit_full,
        keep_indices=keep_indices
    ) if len(all_qubits) > len(qubits) else psi_circuit_full
    assert np.isclose(np.abs(psi_circuit @ create_dicke_statevector(num_qubits, num_excitations)) ** 2, 1.0)

### Requirements:
* The resulting circuit is allowed to contain only
  * CZ gates between pairs of qubits included in `cz_pauli_error_rate`, which will be a subset (and most) of the connected pairs in the [spec sheet](https://quantumai.google/static/site-assets/downloads/willow-spec-sheet.pdf).
  * Arbitrary single-qubit unitaries (`cirq.PhasedXZGate`) on qubits included in `sq_pauli_error_rate`, which will be most of the qubits in the spec sheet.
  * Optionally, mid-circuit reset gates.
* Classically-conditioned gates are not supported.
* In the absence of noise, the circuit must either produce perfect Dicke states of the requested $N$ and $n$ or throw a `NotImplementedError`.
* If your circuit construction code takes longer than a minute to return a circuit, that is counted the same as returning a `NotImplementedError`.
* Any circuits containing more than 1750 moments will not be run.

# Scoring

We prepare the Dicke state on actual quantum hardware and measure it in both the Z basis and the X basis. We use the following `overall_score` function to compare against the expected distributions.

In [ ]:
def check_z_bitstring_distribution(
    bitstrings: np.ndarray, num_excitations: int
) -> float:
    """Measure how close the observed Z-basis distribution of bitstrings is to
    the ideal one. The lower the score, the better.

    Args:
        bitstrings: An array of shape (n_shots, n_qubits) containing the n_shots Z-basis bitstrings.
        num_excitations: The weight, n, of the Dicke state.

    Returns:
        A measure of how close the distribution is to the ideal one.
    """
    # first, let's see how many bitstrings have one excitation
    n_shots, n_qubits = bitstrings.shape
    measured_excitations = np.sum(bitstrings, axis=1)
    fraction_in_subspace = np.mean(measured_excitations == num_excitations)

    # now, let's check how uniform the distribution is over states with n excitations
    chi2 = 0
    ideal_prob = 1 / math.comb(n_qubits, num_excitations)
    for indices in itertools.combinations(range(n_qubits), num_excitations):
        ideal_state = np.zeros(n_qubits)
        ideal_state[np.array(indices)] = 1
        match_count = np.sum(np.all(bitstrings == ideal_state, axis=1))
        expected_count = n_shots * fraction_in_subspace * ideal_prob
        chi2 += (match_count - expected_count) ** 2 / expected_count

    return ideal_prob * chi2 / fraction_in_subspace


def get_correlator_average(bitstrings: np.ndarray, correlator_size: int) -> float:
    """Get the average of all weight-correlator_size correlators in `bitstrings`.

    Args:
        bitstrings: The input bitstrings, shape (n_shots, n_qubits).
        correlator_size: Compute all correlators of this weight.

    Returns:
        The average of all weight-k correlators.
    """
    correlator_sum = 0.0
    n_shots, n_qubits = bitstrings.shape
    for indices in itertools.combinations(range(n_qubits), correlator_size):
        correlator_sum += 1 - 2 * np.mean(np.sum(bitstrings[:, indices], axis=1) % 2)
    return correlator_sum / math.comb(n_qubits, correlator_size)


def check_x_bitstring_distribution(
    bitstrings: np.ndarray, num_excitations: int, max_correlator_size: int
) -> float:
    """Check expected correlations in the X basis. The lower the score, the better.

    Args:
        bitstrings: An array of shape (n_shots, n_qubits) containing the n_shots X-basis bitstrings.
        max_correlator_size: The maximum weight of correlators to consider.
        num_excitations: The weight, n, of the Dicke state.

    Returns:
        A measure of how the correlations in the X-basis measurements compare to expectation.
    """
    comb = lambda n, k: math.comb(n, k) if (k >= 0 and n >= k) else 0
    n_shots, n_qubits = bitstrings.shape
    score = 0.0
    for correlator_size in range(1, max_correlator_size + 1):
        expected_correlator = (
            comb(correlator_size, correlator_size // 2)
            * comb(n_qubits - correlator_size, num_excitations - correlator_size // 2)
            / comb(n_qubits, num_excitations)
            if correlator_size % 2 == 0
            else 0.0
        )
        measured_correlator = get_correlator_average(bitstrings, correlator_size)
        score += (
            (expected_correlator - measured_correlator) ** 2 / expected_correlator**2
            if expected_correlator != 0
            else (expected_correlator - measured_correlator) ** 2
        )

    return score


def overall_score(
    z_bitstrings: np.ndarray,
    x_bitstrings: np.ndarray,
    num_excitations: int,
    max_correlator_size: int,
) -> float:
    """An overall score that checks how the Z and X basis bitstrings compare to their ideal
    distributions. The higher the score, the better.

    Args:
        z_bitstrings: The bitstrings measured in the Z basis
        x_bitstrings: The bitstrings measured in the X basis
        num_excitations: The weight, n, of the Dicke state.
        max_correlator_size: The maximum weight of correlators to consider

    Returns:
        A measure of how similar the bitstrings are to their ideal distributions.
    """
    return 1 / (
        check_z_bitstring_distribution(z_bitstrings, num_excitations)
        + check_x_bitstring_distribution(
            x_bitstrings, num_excitations, max_correlator_size
        )
    )


### Judging criteria:
* We will run circuits generated by your code for various choices of `num_qubits` and `num_excitations` on real quantum hardware, measuring in the Z basis as well as in the X basis, each for 10,000 shots.
* For each choice of `num_qubits` and `num_excitations`, we use the `overall_score` function to score the performance. We then sum this over the choices of `num_qubits`.
* The solution with the highest score wins.

### Things to consider:
* It is permitted to use AI to answer these questions, but please include any prompts you gave to the AI in your submission.
* This will be run on real quantum hardware, so techniques like dynamical decoupling are important.


You may use the following functions to generate fake error rates to test your algorithm.

In [ ]:
def pick_single_qubit_dropouts(
    dropout_rate: float,
    rng: np.random.Generator = np.random.default_rng()
) -> set[cirq.GridQubit]:
    """Pick some qubits to be broken.

    Args:
        dropout_rate: The probability that a qubit is broken.
        rng: The pseudo-random number generator

    Returns:
        The broken qubits.
    """
    all_qubits = np.array(list(cirq_google.Willow105.metadata.qubit_set))
    is_dropout = rng.random(len(all_qubits)) < dropout_rate
    return set(all_qubits[is_dropout])


def generate_qubit_error_rates(
    median: float,
    spread: float,
    dropouts: set[cirq.GridQubit],
    uncertainty: float,
    rng: np.random.Generator = np.random.default_rng(),
) -> dict[cirq.GridQubit, tuple[float, float]]:
    """Generate fake single-qubit error rates.

    Args:
        median: The median error rate.
        spread: The spread of the lognormal distribution
        dropouts: Qubits to exclude
        uncertainty: The uncertainty of the measured error rates.
        rng: The pseudo-random number generator

    Returns:
        A dictionary from qubits to the measured error rate and statistical uncertainty.
    """

    all_qubits = np.array(list(cirq_google.Willow105.metadata.qubit_set))
    is_working = np.array([q not in dropouts for q in all_qubits])
    working_qubits = all_qubits[is_working]
    error_rate = rng.lognormal(np.log(median), spread, len(working_qubits))
    return dict(
        zip(working_qubits, zip(error_rate, uncertainty * np.ones(len(working_qubits))))
    )


def generate_two_qubit_pauli_error_rates(
    median: float = 2.5e-3,
    spread: float = 0.5,
    dropout_rate: float = 0.01,
    uncertainty: float = 5e-4,
    rng: np.random.Generator = np.random.default_rng(),
    single_qubit_dropouts: set[cirq.GridQubit] = None,
) -> dict[tuple[cirq.GridQubit, cirq.GridQubit], tuple[float, float]]:
    """Generate fake two-qubit error rates.

    Args:
        median: The median error rate.
        spread: The spread of the lognormal distribution
        dropout_rate: The probability of a broken coupler.
        uncertainty: The uncertainty of the measured error rates.
        rng: The pseudo-random number generator
        single_qubit_dropouts: The single-qubit dropouts to skip.

    Returns:
        A dictionary from qubits to the measured error rate and statistical uncertainty.
    """
    if single_qubit_dropouts is None:
        single_qubit_dropouts = set()

    all_pairs = np.array(list(cirq_google.Willow105.metadata.qubit_pairs))
    no_qubit_dropouts = np.array(
        [np.all([q not in single_qubit_dropouts for q in pair]) for pair in all_pairs]
    )
    is_working = (rng.random(len(all_pairs)) > dropout_rate) * no_qubit_dropouts
    working_pairs = [tuple(pair) for pair in all_pairs[is_working]]
    error_rate = rng.lognormal(np.log(median), spread, len(working_pairs))
    return dict(
        zip(working_pairs, zip(error_rate, uncertainty * np.ones(len(working_pairs))))
    )


In [ ]:
single_qubit_dropouts = pick_single_qubit_dropouts(0.01)
sq_pauli_error_rate = generate_qubit_error_rates(4e-4, 0.5, single_qubit_dropouts, 2e-4)
cz_pauli_error_rate = generate_two_qubit_pauli_error_rates(
    single_qubit_dropouts=single_qubit_dropouts
)
zero_errors = generate_qubit_error_rates(5e-3, 0.5, single_qubit_dropouts, 2e-4)
one_errors = generate_qubit_error_rates(1e-2, 0.5, single_qubit_dropouts, 3e-4)

# Bonus question:
Now suppose we allow for adaptive circuits, i.e. those with mid-circuit measurements and subsequent gates conditioned on the outcomes of those measurements. See if you can construct shallower circuits to prepare Dicke states given this capability. We will not run these circuits on the hardware.

For all $(N, n)$, with $2 \leq N \leq 15$, and $1 \leq n \leq \lfloor N/2 \rfloor$, we will compare the depth of the circuits produced by the various submissions and assign a point to the one with the shallowest depth, measured by the number of moments in the circuit.

Specifically, your task is to complete the following function:

In [ ]:
# Uses _bonus_mod, BRICK and _brick_circuit defined in the create_dicke_state cell above.
def create_dicke_state_adaptive(
    num_qubits: int, num_excitations: int
) -> tuple[cirq.Circuit, set[cirq.GridQubit]]:
    """Create a Cirq circuit that produces a Dicke state, allowing for adaptive circuits.

    Args:
        num_qubits: The number, N, of qubits in the Dicke state.
        num_excitations: The number, n, of 1s in the state.

    Returns:
        A Cirq circuit to produce Dicke state and a set indicating which qubits the resulting state lives on.

    Raises:
        NotImplementedError: The requested (N, n) combination is not implemented.
    """
    N, n = int(num_qubits), int(num_excitations)
    cands = []
    try:
        cands.append(_bonus_mod.create_dicke_state_adaptive(N, n))
    except NotImplementedError:
        pass
    if (N, n) in BRICK:
        cands.append(_brick_circuit(BRICK[(N, n)]))
    if not cands:
        raise NotImplementedError
    return min(cands, key=lambda cq: len(cq[0]))

## Scoring

We will use the following function to score submissions:

In [ ]:
def score_solution() -> float:
    score = 0
    for num_qubits in range(2, 21):
        for num_excitations in range(1, num_qubits//2+1):
            try:
                circuit, qubits = create_dicke_state_adaptive(num_qubits, num_excitations)
            except NotImplementedError:
                continue
            validate_circuit(circuit, num_excitations, qubits) # checks that the circuit is valid
            score += num_qubits * num_excitations / len(circuit)
    return score

Notes and constraints:
* We will verify classically that the circuits produce the desired Dicke state up to a global phase.
* Using adaptive circuits is not required, so you may reuse your solution from the previous question.
* Although we do not provide error rates for this question, it is still necessary to respect the connectivity of the [Willow device](https://quantumai.google/static/site-assets/downloads/willow-spec-sheet.pdf), i.e. you can only use qubits on that device and CZ gates between adjacent qubits on the device. You may use `cirq_google.Willow105.validate_circuit(circuit)` to validate that your circuits are correctly routed to such a device.
* On real hardware, qubits along a diagonal such that row # + column # = const share a readout line and measuring one qubit dephases the other qubits on that readout line. Therefore, mid-circuit measurements of some but not all of the qubits on a readout line are not allowed. (This constraint is not checked by `cirq_google.Willow105.validate_circuit(circuit)`.)
* As before, the code must execute in under a minute for each $(N, n)$.